In [47]:
from pathlib import Path
import json
import pandas as pd

from politdata.api import (
    DEFAULT_BASE_URL,
    fetch_all_parties,
    fetch_party_account,
)

from politdata.discovery import (
    build_organization_manifest,
    compare_manifests,
    save_discovery_snapshot,
    save_committed_manifest,
)

from politdata.change_detection import (
    organization_content_hash,
    deep_diff,
    classify_record_change,
)

from politdata.sync import (
    run_organization_sync,
)
from politdata.refresh import (
    initialize_refresh_state,
    select_rolling_refresh_candidates,
    update_refresh_state,
)
from politdata.reports import (
    fetch_all_reports,
    reports_to_manifest,
    add_periodicity_flags,
    discover_reports,
)
from politdata.report_discovery import (
    initialize_report_discovery_state,
    run_report_discovery_batch,
    build_report_manifest_from_snapshots,
)
from politdata.report_details import (
    initialize_report_detail_state,
    run_report_detail_batch,
    report_detail_content_hash,
)

print("Notebook environment ready.")

Notebook environment ready.


In [1]:
import sys
print(sys.executable)

C:\Users\Igor\miniforge3\envs\work\python.exe


In [2]:
%pip list

Package                   Version
------------------------- -----------
anyio                     4.12.0
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.0.5
attrs                     25.4.0
babel                     2.17.0
backports.zstd            1.2.0
beautifulsoup4            4.14.3
bleach                    6.3.0
Brotli                    1.2.0
cached-property           1.5.2
certifi                   2025.11.12
cffi                      2.0.0
charset-normalizer        3.4.4
colorama                  0.4.6
comm                      0.2.3
debugpy                   1.8.19
decorator                 5.2.1
defusedxml                0.7.1
et_xmlfile                2.0.0
exceptiongroup            1.3.1
executing                 2.2.1
fastjsonschema            2.21.2
fqdn                      1.5.1
h11                       0.16.0
h2                        4.3.0
hpack             

In [3]:
%pip install polars duckdb

   ---------------------------------------- 0.0/865.2 kB ? eta -:--:--
   ------------------------------------ --- 786.4/865.2 kB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 865.2/865.2 kB 5.5 MB/s  0:00:00
   ---------------------------------------- 0.0/53.7 MB ? eta -:--:--
   - -------------------------------------- 2.4/53.7 MB 11.2 MB/s eta 0:00:05
   -- ------------------------------------- 3.7/53.7 MB 9.5 MB/s eta 0:00:06
   --- ------------------------------------ 5.0/53.7 MB 8.0 MB/s eta 0:00:07
   ----- ---------------------------------- 7.1/53.7 MB 8.4 MB/s eta 0:00:06
   ------ --------------------------------- 8.7/53.7 MB 8.3 MB/s eta 0:00:06
   ------- -------------------------------- 10.5/53.7 MB 8.2 MB/s eta 0:00:06
   --------- ------------------------------ 12.6/53.7 MB 8.5 MB/s eta 0:00:05
   ---------- ----------------------------- 14.7/53.7 MB 8.6 MB/s eta 0:00:05
   ------------ --------------------------- 16.5/53.7 MB 8.6 MB/s eta 0:00:05
   -

In [4]:
import requests
import pandas as pd
import polars as pl
import pyarrow
import duckdb

print("requests:", requests.__version__)
print("pandas:", pd.__version__)
print("polars:", pl.__version__)
print("pyarrow:", pyarrow.__version__)
print("duckdb:", duckdb.__version__)

requests: 2.32.5
pandas: 2.3.3
polars: 1.44.1
pyarrow: 22.0.0
duckdb: 1.5.5


In [5]:
import requests

SWAGGER_URL = "https://politdata.nazk.gov.ua/swagger/api/"

response = requests.get(SWAGGER_URL, timeout=30)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print("Length:", len(response.text))

Status: 200
Content-Type: text/html
Length: 734


In [6]:
print(response.text[:3000])

<!-- HTML for static distribution bundle build -->
<!DOCTYPE html>
<html lang="en">
  <head>
    <meta charset="UTF-8">
    <title>Swagger UI</title>
    <link rel="stylesheet" type="text/css" href="./swagger-ui.css" />
    <link rel="stylesheet" type="text/css" href="index.css" />
    <link rel="icon" type="image/png" href="./favicon-32x32.png" sizes="32x32" />
    <link rel="icon" type="image/png" href="./favicon-16x16.png" sizes="16x16" />
  </head>

  <body>
    <div id="swagger-ui"></div>
    <script src="./swagger-ui-bundle.js" charset="UTF-8"> </script>
    <script src="./swagger-ui-standalone-preset.js" charset="UTF-8"> </script>
    <script src="./swagger-initializer.js" charset="UTF-8"> </script>
  </body>
</html>



In [7]:
initializer_url = "https://politdata.nazk.gov.ua/swagger/api/swagger-initializer.js"

r = requests.get(initializer_url, timeout=30)

print("Status:", r.status_code)
print(r.text[:5000])

Status: 200
window.onload = function() {
  
      //<editor-fold desc="Changeable Configuration Block">
      window.ui = SwaggerUIBundle({
        "dom_id": "#swagger-ui",
        deepLinking: true,
        presets: [
          SwaggerUIBundle.presets.apis,
          SwaggerUIStandalonePreset
        ],
        plugins: [
          SwaggerUIBundle.plugins.DownloadUrl
        ],
        layout: "StandaloneLayout",
        queryConfigEnabled: false,
        url: "/themes/nazk/swagger/public-api.yaml",
      })
      
      //</editor-fold>

};



In [8]:
import yaml
import requests

OPENAPI_URL = "https://politdata.nazk.gov.ua/themes/nazk/swagger/public-api.yaml"

r = requests.get(OPENAPI_URL, timeout=30)
print("Status:", r.status_code)
print("Content-Type:", r.headers.get("content-type"))
print("Length:", len(r.text))

api_spec = yaml.safe_load(r.text)

print("OpenAPI version:", api_spec.get("openapi"))
print("API title:", api_spec.get("info", {}).get("title"))
print("API version:", api_spec.get("info", {}).get("version"))

Status: 200
Content-Type: application/octet-stream
Length: 15141
OpenAPI version: 3.0.3
API title: Публічний API порталу політичних фінансів
API version: 2.0.0


In [9]:
paths = api_spec.get("paths", {})

print("Number of paths:", len(paths))

for path in list(paths.keys())[:30]:
    print(path)

Number of paths: 13
/parties
/party/{id}
/party/{id}/reports
/party/report/{id}
/party/report/{id}/intangible
/party/report/{id}/money
/party/report/{id}/movable
/party/report/{id}/paper
/party/report/{id}/realty
/party/report/{id}/transport
/party/report/{id}/obligations
/party/report/{id}/payments
/party/report/{id}/payments/{type}


In [10]:
import pandas as pd

rows = []

for path, methods in api_spec["paths"].items():
    for method, details in methods.items():
        if method.lower() not in {"get", "post", "put", "delete", "patch"}:
            continue

        params = []

        for p in details.get("parameters", []):
            params.append({
                "name": p.get("name"),
                "in": p.get("in"),
                "required": p.get("required", False),
            })

        rows.append({
            "method": method.upper(),
            "path": path,
            "summary": details.get("summary"),
            "description": details.get("description"),
            "parameters": params,
        })

endpoints_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 150)

endpoints_df

,method,path,summary,description,parameters
0,POST,/parties,Отримання переліку політичних партій та регіональних осередків,None,[]
1,GET,/party/{id},Отримання облікового запису політичної партії/регіонального осередку,None,"[{'name': 'id', 'in': 'path', 'required': True}]"
2,POST,/party/{id}/reports,Отримання списку звітів політичної партії/регіонального осередку,None,"[{'name': 'id', 'in': 'path', 'required': True}]"
3,GET,/party/report/{id},Отримання деталізації по звіту політичної партії,None,"[{'name': 'id', 'in': 'path', 'required': True}]"
4,POST,/party/report/{id}/intangible,"Отримання списку нематеріальних активів, зазначених у звіті політичної партії",None,"[{'name': 'id', 'in': 'path', 'required': True}]"
5,POST,/party/report/{id}/money,"Отримання списку фінансових операцій, зазначених у звіті політичної партії",None,"[{'name': 'id', 'in': 'path', 'required': True}]"
6,POST,/party/report/{id}/movable,"Отримання списку рухомого майна, зазначені у звіті політичної партії",None,"[{'name': 'id', 'in': 'path', 'required': True}]"
7,POST,/party/report/{id}/paper,"Отримання списку цінних паперів, зазначених у звіті політичної партії",None,"[{'name': 'id', 'in': 'path', 'required': True}]"
8,POST,/party/report/{id}/realty,"Отримання списку нерухомого майна, зазначених у звіті політичної партії",None,"[{'name': 'id', 'in': 'path', 'required': True}]"
9,POST,/party/report/{id}/transport,"Отримання списку транспортних засобів, зазначених у звіті політичної партії",None,"[{'name': 'id', 'in': 'path', 'required': True}]"


In [11]:
endpoints_df[
    endpoints_df["path"] == "/parties"
].T

,0
method,POST
path,/parties
summary,Отримання переліку політичних партій та регіональних осередків
description,None
parameters,[]


In [12]:
parties_op = api_spec["paths"]["/parties"]["post"]

print("Summary:")
print(parties_op.get("summary"))

print("\nRequest body:")
print(parties_op.get("requestBody"))

Summary:
Отримання переліку політичних партій та регіональних осередків

Request body:
{'content': {'application/json': {'schema': {'$ref': '#/components/schemas/DataTableRequest'}}}}


In [13]:
import pprint

pprint.pp(
    parties_op.get("requestBody", {})
)

{'content': {'application/json': {'schema': {'$ref': '#/components/schemas/DataTableRequest'}}}}


In [14]:
data_table_schema = api_spec["components"]["schemas"]["DataTableRequest"]

import pprint
pprint.pp(data_table_schema)

{'type': 'object',
 'properties': {'filters': {'oneOf': [{'$ref': '#/components/schemas/SimpleFilters'},
                                      {'$ref': '#/components/schemas/AdvancedFilters'}]},
                'order': {'type': 'object',
                          'additionalProperties': {'enum': ['asc', 'desc']},
                          'example': None},
                'pager': {'$ref': '#/components/schemas/Pager'}}}


In [15]:
schemas = api_spec["components"]["schemas"]

print("PAGER")
pprint.pp(schemas["Pager"])

print("\nSIMPLE FILTERS")
pprint.pp(schemas["SimpleFilters"])

PAGER
{'type': 'object',
 'properties': {'page': {'type': 'integer', 'minimum': 1},
                'size': {'type': 'integer', 'minimum': 10, 'maximum': 100}},
 'example': {'page': 1, 'size': 10}}

SIMPLE FILTERS
{'description': 'Масив фільтрів у форматі: ключ => значення',
 'type': 'object',
 'additionalProperties': {'oneOf': [{'type': 'number'},
                                    {'type': 'string'},
                                    {'type': 'boolean'},
                                    {'type': 'array',
                                     'items': {'oneOf': [{'type': 'number'},
                                                         {'type': 'string'}]}}]},
 'example': None}


In [16]:
print("ADVANCED FILTERS")
pprint.pp(schemas["AdvancedFilters"])

ADVANCED FILTERS
{'type': 'array', 'items': {'$ref': '#/components/schemas/AdvancedFilter'}}


In [17]:
api_spec.get("servers")

[{'url': 'https://politdata.nazk.gov.ua/api/v2'}]

In [18]:
pprint.pp(schemas["AdvancedFilters"])

{'type': 'array', 'items': {'$ref': '#/components/schemas/AdvancedFilter'}}


In [19]:
BASE_URL = "https://politdata.nazk.gov.ua/api/v2"

payload = {
    "filters": {},
    "order": {},
    "pager": {
        "page": 1,
        "size": 10
    }
}

response = requests.post(
    f"{BASE_URL}/parties",
    json=payload,
    timeout=30
)

print("Status:", response.status_code)
print("Content-Type:", response.headers.get("content-type"))
print(response.text[:5000])

Status: 200
Content-Type: application/json; charset=UTF-8
{"results":{"list":[{"id":"d0f1e224-e3de-45a2-8d12-43e4e9faee92","is_active":true,"code":"40392899","name":"ПОЛІТИЧНА ПАРТІЯ «ОБ'ЄДНАНА КРАЇНА»","web_site_url":null,"actual_address_same_register":true,"created_at":"2021-07-30 15:44:19.728","updated_at":"2026-04-01 18:52:40.656931","email":"admin@pp-rozvytok.com","phone":"+380968833388","head_info":{"head_last_name":"Хомідов","head_first_name":"Руслан","head_middle_name":"Володимирович"},"register_address":{"country":"Україна","post_index":"01042","region":"Київ","district":"","city":"","street":"Іоанна Павла ІІ (Печерський район)","building":"4/6","apartments":"","common":"","building_part_num":"","address_uk":null,"address_en":null,"post_index_not_provided":null,"district_not_provided":null,"common_not_provided":null,"city_not_provided":null,"region_not_provided":null,"street_not_provided":null,"building_not_provided":null,"building_part_num_not_provided":null,"apartments_not_p

In [20]:
data = response.json()

print("Top-level keys:")
print(data.keys())

print("\nresults keys:")
print(data["results"].keys())

Top-level keys:
dict_keys(['results'])

results keys:
dict_keys(['list', 'count'])


In [21]:
for key, value in data["results"].items():
    if key == "list":
        print(f"{key}: list with {len(value)} records")
    else:
        print(f"{key}: {value}")

list: list with 10 records
count: 321


In [22]:
first_party = data["results"]["list"][0]

for key, value in first_party.items():
    print(f"{key:30} {type(value).__name__:12} {str(value)[:150]}")

id                             str          d0f1e224-e3de-45a2-8d12-43e4e9faee92
is_active                      bool         True
code                           str          40392899
name                           str          ПОЛІТИЧНА ПАРТІЯ «ОБ'ЄДНАНА КРАЇНА»
web_site_url                   NoneType     None
actual_address_same_register   bool         True
created_at                     str          2021-07-30 15:44:19.728
updated_at                     str          2026-04-01 18:52:40.656931
email                          str          admin@pp-rozvytok.com
phone                          str          +380968833388
head_info                      dict         {'head_last_name': 'Хомідов', 'head_first_name': 'Руслан', 'head_middle_name': 'Володимирович'}
register_address               dict         {'country': 'Україна', 'post_index': '01042', 'region': 'Київ', 'district': '', 'city': '', 'street': 'Іоанна Павла ІІ (Печерський район)', 'building'
actual_address                 NoneType  

In [23]:
import math

total_count = data["results"]["count"]
page_size = 100
total_pages = math.ceil(total_count / page_size)

print("Total records:", total_count)
print("Page size:", page_size)
print("Total pages:", total_pages)

Total records: 321
Page size: 100
Total pages: 4


In [24]:
payload_last = {
    "filters": {},
    "order": {},
    "pager": {
        "page": 4,
        "size": 100
    }
}

response_last = requests.post(
    f"{BASE_URL}/parties",
    json=payload_last,
    timeout=30
)

data_last = response_last.json()

print("Status:", response_last.status_code)
print("Records on page 4:", len(data_last["results"]["list"]))
print("Total count:", data_last["results"]["count"])

Status: 200
Records on page 4: 21
Total count: 321


In [25]:
from math import ceil
from pathlib import Path
from datetime import datetime
import json
import requests


def download_all_parties(base_url, page_size=100, timeout=30):
    endpoint = f"{base_url}/parties"

    # Перший запит — щоб дізнатися загальну кількість записів
    first_payload = {
        "filters": {},
        "order": {},
        "pager": {
            "page": 1,
            "size": page_size
        }
    }

    response = requests.post(
        endpoint,
        json=first_payload,
        timeout=timeout
    )
    response.raise_for_status()

    first_data = response.json()
    results = first_data["results"]

    total_count = results["count"]
    total_pages = ceil(total_count / page_size)

    all_records = list(results["list"])

    print(f"Total records reported by API: {total_count}")
    print(f"Total pages: {total_pages}")
    print(f"Page 1: {len(results['list'])} records")

    # Решта сторінок
    for page in range(2, total_pages + 1):
        payload = {
            "filters": {},
            "order": {},
            "pager": {
                "page": page,
                "size": page_size
            }
        }

        response = requests.post(
            endpoint,
            json=payload,
            timeout=timeout
        )
        response.raise_for_status()

        page_data = response.json()
        page_records = page_data["results"]["list"]

        all_records.extend(page_records)

        print(f"Page {page}: {len(page_records)} records")

    # Контроль кількості
    if len(all_records) != total_count:
        raise ValueError(
            f"Downloaded {len(all_records)} records, "
            f"but API reported {total_count}"
        )

    # Контроль унікальності ID
    ids = [record["id"] for record in all_records]
    unique_ids = set(ids)

    if len(ids) != len(unique_ids):
        raise ValueError(
            f"Duplicate IDs detected: "
            f"{len(ids) - len(unique_ids)} duplicates"
        )

    print()
    print("Download completed successfully.")
    print(f"Downloaded records: {len(all_records)}")
    print(f"Unique IDs: {len(unique_ids)}")

    return all_records

In [26]:
parties_raw = download_all_parties(BASE_URL)

Total records reported by API: 321
Total pages: 4
Page 1: 100 records
Page 2: 100 records
Page 3: 100 records
Page 4: 21 records

Download completed successfully.
Downloaded records: 321
Unique IDs: 321


In [27]:
from pathlib import Path
from datetime import datetime, timezone
import json

raw_dir = Path("data/raw/parties")
raw_dir.mkdir(parents=True, exist_ok=True)

downloaded_at = datetime.now(timezone.utc)

snapshot = {
    "source": "PolitData",
    "endpoint": "/parties",
    "base_url": BASE_URL,
    "downloaded_at_utc": downloaded_at.isoformat(),
    "record_count": len(parties_raw),
    "records": parties_raw,
}

filename = (
    f"parties_"
    f"{downloaded_at.strftime('%Y-%m-%d_%H-%M-%S')}_UTC.json"
)

output_path = raw_dir / filename

with output_path.open("w", encoding="utf-8") as f:
    json.dump(
        snapshot,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Saved:", output_path)
print("Records:", len(parties_raw))
print("Size:", round(output_path.stat().st_size / 1024, 1), "KB")

Saved: data\raw\parties\parties_2026-08-28_14-29-24_UTC.json
Records: 321
Size: 3399.6 KB


In [28]:
with output_path.open("r", encoding="utf-8") as f:
    check = json.load(f)

print("Source:", check["source"])
print("Endpoint:", check["endpoint"])
print("Declared records:", check["record_count"])
print("Loaded records:", len(check["records"]))
print("First ID:", check["records"][0]["id"])
print("Last ID:", check["records"][-1]["id"])

Source: PolitData
Endpoint: /parties
Declared records: 321
Loaded records: 321
First ID: d0f1e224-e3de-45a2-8d12-43e4e9faee92
Last ID: e75abd72-368e-4963-a386-103f702ed7bc


In [29]:
import hashlib

def sha256_file(path):
    hasher = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)

    return hasher.hexdigest()


snapshot_sha256 = sha256_file(output_path)

print("File:", output_path)
print("SHA-256:", snapshot_sha256)

File: data\raw\parties\parties_2026-08-28_14-29-24_UTC.json
SHA-256: 3bc25978d3bf68df523b566e989720c7ed3a097267ec044ab9f684d881ec9d0e


In [30]:
metadata_path = output_path.with_suffix(".metadata.json")

metadata = {
    "source": "PolitData",
    "endpoint": "/parties",
    "downloaded_at_utc": downloaded_at.isoformat(),
    "record_count": len(parties_raw),
    "snapshot_file": output_path.name,
    "sha256": snapshot_sha256,
}

with metadata_path.open("w", encoding="utf-8") as f:
    json.dump(
        metadata,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Metadata saved:", metadata_path)

Metadata saved: data\raw\parties\parties_2026-08-28_14-29-24_UTC.metadata.json


In [31]:
import pandas as pd
from collections import Counter

print("Total records:", len(parties_raw))

# Типи вкладених полів
for field in ["head_info", "register_address", "actual_address", "parent", "regional_offices"]:
    counts = Counter(type(x.get(field)).__name__ for x in parties_raw)
    print(f"{field}: {dict(counts)}")

Total records: 321
head_info: {'dict': 253, 'list': 68}
register_address: {'dict': 311, 'NoneType': 10}
actual_address: {'NoneType': 198, 'dict': 123}
parent: {'list': 321}
regional_offices: {'list': 321}


In [32]:
print("Parent length distribution:")
print(Counter(len(x.get("parent") or []) for x in parties_raw))

print("\nRegional offices length distribution:")
print(Counter(len(x.get("regional_offices") or []) for x in parties_raw))

Parent length distribution:
Counter({0: 321})

Regional offices length distribution:
Counter({0: 107, 1: 40, 2: 12, 3: 9, 4: 8, 28: 6, 20: 6, 14: 6, 13: 5, 24: 5, 31: 4, 21: 4, 16: 4, 26: 3, 11: 3, 19: 3, 10: 3, 5: 3, 34: 3, 7: 3, 17: 3, 32: 3, 49: 3, 9: 3, 30: 2, 25: 2, 77: 2, 18: 2, 58: 2, 103: 2, 40: 2, 65: 2, 27: 2, 33: 2, 22: 2, 146: 2, 8: 1, 696: 1, 37: 1, 71: 1, 276: 1, 44: 1, 694: 1, 51: 1, 75: 1, 67: 1, 47: 1, 431: 1, 320: 1, 78: 1, 191: 1, 76: 1, 64: 1, 43: 1, 122: 1, 90: 1, 82: 1, 57: 1, 36: 1, 306: 1, 86: 1, 72: 1, 6: 1, 152: 1, 41: 1, 12: 1, 38: 1, 63: 1, 207: 1, 120: 1, 205: 1, 153: 1, 138: 1, 576: 1, 59: 1, 192: 1, 66: 1, 60: 1, 83: 1, 15: 1, 46: 1, 85: 1, 239: 1, 23: 1})


In [33]:
records_with_parent = [
    x for x in parties_raw
    if x.get("parent")
]

print("Records with parent:", len(records_with_parent))

for x in records_with_parent[:10]:
    print("\nNAME:", x.get("name"))
    print("CODE:", x.get("code"))
    print("ID:", x.get("id"))
    print("PARENT:", x.get("parent"))

Records with parent: 0


In [34]:
# Загальна кількість вкладених регіональних/місцевих осередків
total_offices = sum(
    len(party.get("regional_offices") or [])
    for party in parties_raw
)

print("Top-level party records:", len(parties_raw))
print("Nested regional offices:", total_offices)

Top-level party records: 321
Nested regional offices: 9596


In [35]:
# Знаходимо першу партію, яка має хоча б один осередок
party_with_office = next(
    party for party in parties_raw
    if party.get("regional_offices")
)

first_office = party_with_office["regional_offices"][0]

print("PARTY:")
print(party_with_office["name"])
print(party_with_office["id"])

print("\nFIRST OFFICE:")
for key, value in first_office.items():
    print(
        f"{key:30} "
        f"{type(value).__name__:12} "
        f"{str(value)[:200]}"
    )

PARTY:
ПОЛІТИЧНА ПАРТІЯ "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ "ФАКЕЛ"
5966f24e-8b67-428d-817f-9b48eb335077

FIRST OFFICE:
id                             str          1cf95166-a8c5-4363-a268-36ab0d276b87
is_active                      bool         True
code                           str          43754413
name                           str          БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ "ФАКЕЛ"


In [36]:
office_ids = []

for party in parties_raw:
    for office in party.get("regional_offices") or []:
        office_ids.append(office.get("id"))

print("Office records:", len(office_ids))
print("Unique office IDs:", len(set(office_ids)))
print("Duplicate office IDs:", len(office_ids) - len(set(office_ids)))

Office records: 9596
Unique office IDs: 9596
Duplicate office IDs: 0


In [37]:
office_counts = [
    {
        "party_id": party["id"],
        "code": party.get("code"),
        "name": party.get("name"),
        "regional_offices_count": len(
            party.get("regional_offices") or []
        )
    }
    for party in parties_raw
]

office_counts_df = (
    pd.DataFrame(office_counts)
    .sort_values(
        "regional_offices_count",
        ascending=False
    )
)

office_counts_df.head(20)

,party_id,code,name,regional_offices_count
26,17e3d21a-4e0a-4f54-9ce7-1abf4485fbbb,20073933,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ РЕГІОНІВ»,696
59,1e52ae20-1244-42ac-927d-7e014c80fc14,20069956,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»,694
267,155c3deb-caa3-488b-8ecf-5e54d674a3f6,00013215,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»,576
84,171b4fa0-5b63-4c2f-982c-a11a2b5ecabf,21710533,ПОЛІТИЧНА ПАРТІЯ «УКРАЇНСЬКА НАРОДНА ПАРТІЯ»,431
91,35579664-92f4-4b98-a229-637aa78b52af,00013209,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,320
172,b83037f6-6302-4b90-abc1-642a95d68171,00013267,ПОЛІТИЧНА ПАРТІЯ «НАРОДНО-ДЕМОКРАТИЧНА ПАРТІЯ (НДП)»,306
35,01eddd1d-e3ec-41ad-b1f9-5a52a4d6d2fe,00062923,ПОЛІТИЧНА ПАРТІЯ ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «ГРОМАДА»,276
307,e347584f-4b99-4b65-bb8e-3b290974416d,26296630,"ПОЛІТИЧНА ПАРТІЯ ""ЗАВЖДИ СИЛЬНІ УКРАЇНЦІ""",239
235,659b917b-0eb0-4dbd-bb94-56a886675f31,21716145,ПОЛІТИЧНА ПАРТІЯ «СПРАВЕДЛИВІСТЬ»,207
258,0d8460ee-fe65-427e-8efa-ec488676f345,34763401,АГРАРНА ПАРТІЯ УКРАЇНИ,205


In [38]:
party_with_office = next(
    party for party in parties_raw
    if party.get("regional_offices")
)

first_office = party_with_office["regional_offices"][0]

print("PARTY:")
print(party_with_office["name"])
print(party_with_office["id"])

print("\nFIRST OFFICE:")
for key, value in first_office.items():
    print(
        f"{key:30} "
        f"{type(value).__name__:12} "
        f"{str(value)[:200]}"
    )

PARTY:
ПОЛІТИЧНА ПАРТІЯ "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ "ФАКЕЛ"
5966f24e-8b67-428d-817f-9b48eb335077

FIRST OFFICE:
id                             str          1cf95166-a8c5-4363-a268-36ab0d276b87
is_active                      bool         True
code                           str          43754413
name                           str          БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ "ФАКЕЛ"


In [39]:
office_counts = [
    {
        "party_id": party["id"],
        "code": party.get("code"),
        "name": party.get("name"),
        "regional_offices_count": len(
            party.get("regional_offices") or []
        )
    }
    for party in parties_raw
]

office_counts_df = (
    pd.DataFrame(office_counts)
    .sort_values("regional_offices_count", ascending=False)
)

office_counts_df.head(20)

,party_id,code,name,regional_offices_count
26,17e3d21a-4e0a-4f54-9ce7-1abf4485fbbb,20073933,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ РЕГІОНІВ»,696
59,1e52ae20-1244-42ac-927d-7e014c80fc14,20069956,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»,694
267,155c3deb-caa3-488b-8ecf-5e54d674a3f6,00013215,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»,576
84,171b4fa0-5b63-4c2f-982c-a11a2b5ecabf,21710533,ПОЛІТИЧНА ПАРТІЯ «УКРАЇНСЬКА НАРОДНА ПАРТІЯ»,431
91,35579664-92f4-4b98-a229-637aa78b52af,00013209,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,320
172,b83037f6-6302-4b90-abc1-642a95d68171,00013267,ПОЛІТИЧНА ПАРТІЯ «НАРОДНО-ДЕМОКРАТИЧНА ПАРТІЯ (НДП)»,306
35,01eddd1d-e3ec-41ad-b1f9-5a52a4d6d2fe,00062923,ПОЛІТИЧНА ПАРТІЯ ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «ГРОМАДА»,276
307,e347584f-4b99-4b65-bb8e-3b290974416d,26296630,"ПОЛІТИЧНА ПАРТІЯ ""ЗАВЖДИ СИЛЬНІ УКРАЇНЦІ""",239
235,659b917b-0eb0-4dbd-bb94-56a886675f31,21716145,ПОЛІТИЧНА ПАРТІЯ «СПРАВЕДЛИВІСТЬ»,207
258,0d8460ee-fe65-427e-8efa-ec488676f345,34763401,АГРАРНА ПАРТІЯ УКРАЇНИ,205


In [40]:
party_detail_op = api_spec["paths"]["/party/{id}"]["get"]

print("Summary:")
print(party_detail_op.get("summary"))

print("\nParameters:")
pprint.pp(party_detail_op.get("parameters"))

print("\nResponses:")
pprint.pp(party_detail_op.get("responses"))

Summary:
Отримання облікового запису політичної партії/регіонального осередку

Parameters:
[{'name': 'id',
  'description': 'Ідентифікатор політичної партії або регіонального осередку',
  'schema': {'type': 'string'},
  'in': 'path',
  'required': True}]

Responses:
{'200': {'description': 'OK',
         'content': {'application/json': {'schema': {'allOf': [{'$ref': '#/components/schemas/ApiResponse'},
                                                               {'type': 'object',
                                                                'properties': {'results': {'$ref': '#/components/schemas/PartyModel'}}}]}}}}}


In [41]:
office_id = first_office["id"]

office_response = requests.get(
    f"{BASE_URL}/party/{office_id}",
    timeout=30
)

print("Status:", office_response.status_code)
print("Content-Type:", office_response.headers.get("content-type"))
print(office_response.text[:5000])

Status: 200
Content-Type: application/json; charset=UTF-8
{"code":0,"results":{"id":"1cf95166-a8c5-4363-a268-36ab0d276b87","is_active":true,"code":"43754413","name":"БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ \"ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ \"ФАКЕЛ\"","web_site_url":null,"actual_address_same_register":true,"created_at":"2025-07-01 08:47:31.153957","updated_at":"2025-07-01 08:47:31.153957","email":null,"phone":null,"head_info":{"head_last_name":"Роговий ","head_first_name":"Андрій","head_middle_name":"Борисович"},"register_address":{"country":"Україна","post_index":"09100","region":"Київська обл.","district":"Білоцерківський р-н","city":"Біла Церква","street":"Ярмаркова","building":"1/28","apartments":null,"common":null,"building_part_num":null,"address_uk":null,"address_en":null,"post_index_not_provided":null,"district_not_provided":null,"common_not_provided":null,"city_not_provided":null,"region_not_provided":null,"street_not_provided":null,"building_not_provided":null,"building_p

In [42]:
office_detail = office_response.json()["results"]

print("OFFICE:")
print(office_detail["name"])

print("\nPARENT:")
print(office_detail.get("parent"))

print("\nCHILDREN:")
print("Count:", len(office_detail.get("regional_offices") or []))

for child in (office_detail.get("regional_offices") or [])[:10]:
    print(child)

OFFICE:
БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ "ФАКЕЛ"

PARENT:
{'id': '5966f24e-8b67-428d-817f-9b48eb335077', 'is_active': True, 'code': '40347432', 'name': 'ПОЛІТИЧНА ПАРТІЯ "ВСЕУКРАЇНСЬКЕ ОБ\'ЄДНАННЯ "ФАКЕЛ"'}

CHILDREN:
Count: 0


In [43]:
batkivshchyna = next(
    p for p in parties_raw
    if "БАТЬКІВЩИНА" in p["name"].upper()
)

print(batkivshchyna["name"])
print("Party ID:", batkivshchyna["id"])
print("Offices:", len(batkivshchyna["regional_offices"]))

ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»
Party ID: 1e52ae20-1244-42ac-927d-7e014c80fc14
Offices: 694


In [44]:
sample_offices = batkivshchyna["regional_offices"][:20]

sample_results = []

for office in sample_offices:
    r = requests.get(
        f"{BASE_URL}/party/{office['id']}",
        timeout=30
    )
    r.raise_for_status()

    detail = r.json()["results"]

    sample_results.append({
        "id": detail["id"],
        "name": detail["name"],
        "parent_id": (
            detail.get("parent", {}).get("id")
            if isinstance(detail.get("parent"), dict)
            else None
        ),
        "children_count": len(
            detail.get("regional_offices") or []
        )
    })

sample_df = pd.DataFrame(sample_results)

sample_df[
    ["name", "parent_id", "children_count"]
]

,name,parent_id,children_count
0,"ЧЕРНІГІВСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОГО ОБ'ЄДНАННЯ ""БАТЬКІВЩИНА""",1e52ae20-1244-42ac-927d-7e014c80fc14,0
1,"Бердянська міська організація політичної партії ""Всеукраїнське об'єднання ""Батьківщина""",1e52ae20-1244-42ac-927d-7e014c80fc14,0
2,"Енергодарська міська організація політичної партії ""Всеукраїнське об'єднання ""Батьківщина""",1e52ae20-1244-42ac-927d-7e014c80fc14,0
3,"Приморська районна партійна організація Всеукраїнського об'єднання ""Батьківщина""",1e52ae20-1244-42ac-927d-7e014c80fc14,0
4,"Чернігівська районна партійна організація Всеукраїнського об'єднання ""Батьківщина"" Запорізької області",1e52ae20-1244-42ac-927d-7e014c80fc14,0
5,"ТОКМАЦЬКА РАЙОННА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ""ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ""БАТЬКІВЩИНА""",1e52ae20-1244-42ac-927d-7e014c80fc14,0
6,"Якимівська районна партійна організація Всеукраїнського об'єднання ""Батьківщина"" Запорізької області",1e52ae20-1244-42ac-927d-7e014c80fc14,0
7,"Новокаховська міська організація партії ""Всеукраїнське об'єднання ""Батьківщина""",1e52ae20-1244-42ac-927d-7e014c80fc14,0
8,"Генічеська районна організація Всеукраїнського об'єднання ""Батьківщина""",1e52ae20-1244-42ac-927d-7e014c80fc14,0
9,"Каховська районна організація партії ""Всеукраїнське об'єднання ""Батьківщина""",1e52ae20-1244-42ac-927d-7e014c80fc14,0


In [45]:
print(
    "Sample offices with children:",
    (sample_df["children_count"] > 0).sum()
)

sample_df[
    sample_df["children_count"] > 0
].sort_values(
    "children_count",
    ascending=False
)

Sample offices with children: 0


,id,name,parent_id,children_count


In [46]:
office_samples = []

for party in parties_raw:
    offices = party.get("regional_offices") or []

    if offices:
        office_samples.append({
            "root_party_id": party["id"],
            "root_party_name": party["name"],
            "office_id": offices[0]["id"],
            "office_name_index": offices[0]["name"],
        })

print("Parties with offices:", len(office_samples))

Parties with offices: 214


In [47]:
import time
from tqdm.auto import tqdm

session = requests.Session()

hierarchy_check = []
errors = []

for item in tqdm(office_samples):
    try:
        r = session.get(
            f"{BASE_URL}/party/{item['office_id']}",
            timeout=30
        )

        if r.status_code != 200:
            errors.append({
                "office_id": item["office_id"],
                "status": r.status_code
            })
            continue

        detail = r.json()["results"]

        parent = detail.get("parent")

        if isinstance(parent, dict):
            parent_id = parent.get("id")
        else:
            parent_id = None

        children = detail.get("regional_offices") or []

        hierarchy_check.append({
            "root_party_id": item["root_party_id"],
            "root_party_name": item["root_party_name"],
            "office_id": detail["id"],
            "office_name": detail["name"],
            "parent_id": parent_id,
            "parent_matches_root": parent_id == item["root_party_id"],
            "children_count": len(children),
        })

        time.sleep(0.15)

    except Exception as e:
        errors.append({
            "office_id": item["office_id"],
            "error": str(e)
        })

  0%|          | 0/214 [00:00<?, ?it/s]

In [48]:
hierarchy_df = pd.DataFrame(hierarchy_check)

print("Checked:", len(hierarchy_df))
print("Errors:", len(errors))

print("\nParent matches root:")
print(hierarchy_df["parent_matches_root"].value_counts(dropna=False))

print("\nChildren count:")
print(hierarchy_df["children_count"].value_counts().sort_index())

Checked: 214
Errors: 0

Parent matches root:
parent_matches_root
True    214
Name: count, dtype: int64

Children count:
children_count
0    214
Name: count, dtype: int64


In [49]:
anomalies = hierarchy_df[
    (~hierarchy_df["parent_matches_root"]) |
    (hierarchy_df["children_count"] > 0)
]

print("Hierarchy anomalies:", len(anomalies))

anomalies[
    [
        "root_party_name",
        "office_name",
        "parent_id",
        "parent_matches_root",
        "children_count"
    ]
]

Hierarchy anomalies: 0


,root_party_name,office_name,parent_id,parent_matches_root,children_count


In [50]:
organization_index = []

for party in parties_raw:
    # Центральна партія
    organization_index.append({
        "organization_id": party["id"],
        "root_party_id": party["id"],
        "parent_id": None,
        "entity_type": "party",
        "code": party.get("code"),
        "name": party.get("name"),
    })

    # Осередки
    for office in party.get("regional_offices") or []:
        organization_index.append({
            "organization_id": office["id"],
            "root_party_id": party["id"],
            "parent_id": party["id"],
            "entity_type": "office",
            "code": office.get("code"),
            "name": office.get("name"),
        })

organization_index_df = pd.DataFrame(organization_index)

print("Organizations:", len(organization_index_df))
print()
print(
    organization_index_df["entity_type"]
    .value_counts()
)

Organizations: 9917

entity_type
office    9596
party      321
Name: count, dtype: int64


In [51]:
print(
    "Unique IDs:",
    organization_index_df["organization_id"].nunique()
)

print(
    "Duplicate IDs:",
    organization_index_df["organization_id"].duplicated().sum()
)

print(
    "Missing IDs:",
    organization_index_df["organization_id"].isna().sum()
)

Unique IDs: 9917
Duplicate IDs: 0
Missing IDs: 0


In [52]:
from pathlib import Path

manifest_dir = Path("data/interim")
manifest_dir.mkdir(parents=True, exist_ok=True)

manifest_path = manifest_dir / "organization_manifest.parquet"

organization_index_df.to_parquet(
    manifest_path,
    index=False
)

print("Saved:", manifest_path)
print("Rows:", len(organization_index_df))

Saved: data\interim\organization_manifest.parquet
Rows: 9917


In [53]:
manifest_check = pd.read_parquet(manifest_path)

print("Rows:", len(manifest_check))
print("Unique IDs:", manifest_check["organization_id"].nunique())
print(manifest_check["entity_type"].value_counts())

Rows: 9917
Unique IDs: 9917
entity_type
office    9596
party      321
Name: count, dtype: int64


In [54]:
import requests
import time

session = requests.Session()

def download_organization_detail(
    organization_id,
    base_url=BASE_URL,
    timeout=30,
    max_retries=3
):
    url = f"{base_url}/party/{organization_id}"

    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                url,
                timeout=timeout
            )

            response.raise_for_status()

            data = response.json()

            if "results" not in data:
                raise ValueError(
                    f"No 'results' in response for {organization_id}"
                )

            return data["results"]

        except Exception as e:
            if attempt == max_retries:
                raise

            print(
                f"Attempt {attempt} failed for "
                f"{organization_id}: {e}"
            )

            time.sleep(attempt * 2)

In [55]:
test_id = organization_index_df.iloc[321]["organization_id"]

test_detail = download_organization_detail(test_id)

print(test_detail["id"])
print(test_detail["name"])
print(test_detail.get("parent"))

95f6fcaf-3ff1-4d47-afd3-b4d3ff4eb24b
РОКИТНІВСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПАРТІЇ РЕГІОНІВ
{'id': '17e3d21a-4e0a-4f54-9ce7-1abf4485fbbb', 'is_active': True, 'code': '20073933', 'name': 'ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ РЕГІОНІВ»'}


In [56]:
from pathlib import Path
import json
from tqdm.auto import tqdm

details_dir = Path("data/raw/organization_details")
details_dir.mkdir(parents=True, exist_ok=True)

def save_organization_detail(detail, output_dir=details_dir):
    organization_id = detail["id"]
    path = output_dir / f"{organization_id}.json"

    with path.open("w", encoding="utf-8") as f:
        json.dump(
            detail,
            f,
            ensure_ascii=False,
            indent=2
        )

    return path

In [57]:
def download_organization_batch(
    manifest_df,
    limit=None,
    output_dir=details_dir
):
    rows = manifest_df if limit is None else manifest_df.head(limit)

    downloaded = 0
    skipped = 0
    failed = []

    for _, row in tqdm(rows.iterrows(), total=len(rows)):
        organization_id = row["organization_id"]
        output_path = output_dir / f"{organization_id}.json"

        # Resume: якщо файл уже є, не качаємо повторно
        if output_path.exists():
            skipped += 1
            continue

        try:
            detail = download_organization_detail(
                organization_id
            )

            save_organization_detail(
                detail,
                output_dir=output_dir
            )

            downloaded += 1

        except Exception as e:
            failed.append({
                "organization_id": organization_id,
                "error": str(e)
            })

    print()
    print("Downloaded:", downloaded)
    print("Skipped:", skipped)
    print("Failed:", len(failed))

    return failed

In [58]:
failed_test = download_organization_batch(
    organization_index_df,
    limit=20
)

  0%|          | 0/20 [00:00<?, ?it/s]


Downloaded: 20
Skipped: 0
Failed: 0


In [59]:
failed_test = download_organization_batch(
    organization_index_df,
    limit=20
)

  0%|          | 0/20 [00:00<?, ?it/s]


Downloaded: 0
Skipped: 20
Failed: 0


In [60]:
from pathlib import Path

accounts_dir = Path("data/raw/party_accounts")
accounts_dir.mkdir(parents=True, exist_ok=True)

print(accounts_dir)

data\raw\party_accounts


In [61]:
import requests
import time

session = requests.Session()

def fetch_party_account(
    organization_id,
    base_url=BASE_URL,
    timeout=30,
    max_retries=4
):
    url = f"{base_url}/party/{organization_id}"

    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = session.get(
                url,
                timeout=timeout
            )

            response.raise_for_status()

            data = response.json()

            if "results" not in data:
                raise ValueError(
                    f"No 'results' field for {organization_id}"
                )

            returned_id = data["results"].get("id")

            if returned_id != organization_id:
                raise ValueError(
                    f"Requested {organization_id}, "
                    f"but API returned {returned_id}"
                )

            return data

        except Exception as e:
            last_error = e

            if attempt < max_retries:
                time.sleep(attempt * 2)

    raise last_error

In [62]:
import json

def save_raw_party_account(
    api_response,
    output_dir=accounts_dir
):
    organization_id = api_response["results"]["id"]

    output_path = (
        output_dir /
        f"{organization_id}.json"
    )

    with output_path.open(
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            api_response,
            f,
            ensure_ascii=False,
            indent=2
        )

    return output_path

In [63]:
from tqdm.auto import tqdm

def download_party_accounts(
    manifest_df,
    limit=None,
    output_dir=accounts_dir,
    request_delay=0.15
):
    if limit is None:
        rows = manifest_df
    else:
        rows = manifest_df.head(limit)

    downloaded = 0
    skipped = 0
    failed = []

    for _, row in tqdm(
        rows.iterrows(),
        total=len(rows)
    ):
        organization_id = row["organization_id"]

        output_path = (
            output_dir /
            f"{organization_id}.json"
        )

        # Resume
        if output_path.exists():
            skipped += 1
            continue

        try:
            api_response = fetch_party_account(
                organization_id
            )

            save_raw_party_account(
                api_response,
                output_dir=output_dir
            )

            downloaded += 1

        except Exception as e:
            failed.append({
                "organization_id": organization_id,
                "entity_type": row["entity_type"],
                "name": row["name"],
                "error": str(e)
            })

        time.sleep(request_delay)

    print()
    print("Downloaded:", downloaded)
    print("Skipped:", skipped)
    print("Failed:", len(failed))

    return failed

In [64]:
failed_test_v2 = download_party_accounts(
    organization_index_df,
    limit=20
)

  0%|          | 0/20 [00:00<?, ?it/s]


Downloaded: 20
Skipped: 0
Failed: 0


In [65]:
failed_test_v2 = download_party_accounts(
    organization_index_df,
    limit=20
)

  0%|          | 0/20 [00:00<?, ?it/s]


Downloaded: 0
Skipped: 20
Failed: 0


In [66]:
test_file = next(accounts_dir.glob("*.json"))

with test_file.open(
    "r",
    encoding="utf-8"
) as f:
    test_raw = json.load(f)

print("Top-level keys:", test_raw.keys())
print("API code:", test_raw.get("code"))
print("ID:", test_raw["results"]["id"])
print("Name:", test_raw["results"]["name"])

Top-level keys: dict_keys(['code', 'results'])
API code: 0
ID: 011c6969-8c97-4009-8fce-37e7b64576a4
Name: РЕНІЙСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПАРТІЇ "РОДИНА"


In [67]:
failed_full = download_party_accounts(
    organization_index_df,
    limit=None
)

  0%|          | 0/9917 [00:00<?, ?it/s]


Downloaded: 9897
Skipped: 20
Failed: 0


In [68]:
from datetime import datetime, timezone
import json

qa_dir = Path("logs")
qa_dir.mkdir(parents=True, exist_ok=True)

failed_path = qa_dir / "party_accounts_failed.json"

with failed_path.open("w", encoding="utf-8") as f:
    json.dump(
        failed_full,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Failed requests:", len(failed_full))
print("Saved:", failed_path)

Failed requests: 0
Saved: logs\party_accounts_failed.json


In [69]:
from pathlib import Path
import json
import pandas as pd

accounts_dir = Path("data/raw/party_accounts")

manifest_ids = set(
    organization_index_df["organization_id"].astype(str)
)

raw_files = list(accounts_dir.glob("*.json"))

print("Manifest organizations:", len(manifest_ids))
print("RAW files:", len(raw_files))

Manifest organizations: 9917
RAW files: 9917


In [70]:
qa_results = []

for path in raw_files:
    filename_id = path.stem

    try:
        with path.open("r", encoding="utf-8") as f:
            raw = json.load(f)

        api_code = raw.get("code")
        results = raw.get("results")

        if isinstance(results, dict):
            content_id = results.get("id")
        else:
            content_id = None

        qa_results.append({
            "filename": path.name,
            "filename_id": filename_id,
            "content_id": content_id,
            "api_code": api_code,
            "valid_json": True,
            "has_results": isinstance(results, dict),
            "id_matches_filename": content_id == filename_id,
            "in_manifest": filename_id in manifest_ids,
            "error": None,
        })

    except Exception as e:
        qa_results.append({
            "filename": path.name,
            "filename_id": filename_id,
            "content_id": None,
            "api_code": None,
            "valid_json": False,
            "has_results": False,
            "id_matches_filename": False,
            "in_manifest": filename_id in manifest_ids,
            "error": str(e),
        })

qa_df = pd.DataFrame(qa_results)

In [71]:
print("Files checked:", len(qa_df))
print("Valid JSON:", qa_df["valid_json"].sum())
print("Has results:", qa_df["has_results"].sum())
print("ID matches filename:", qa_df["id_matches_filename"].sum())
print("In manifest:", qa_df["in_manifest"].sum())

print(
    "Unique content IDs:",
    qa_df["content_id"].nunique(dropna=True)
)

Files checked: 9917
Valid JSON: 9917
Has results: 9917
ID matches filename: 9917
In manifest: 9917
Unique content IDs: 9917


In [72]:
file_ids = set(qa_df["filename_id"])
content_ids = set(
    qa_df["content_id"].dropna().astype(str)
)

missing_files = manifest_ids - file_ids
unexpected_files = file_ids - manifest_ids
missing_content_ids = manifest_ids - content_ids

print("Missing files:", len(missing_files))
print("Unexpected files:", len(unexpected_files))
print("Missing content IDs:", len(missing_content_ids))

Missing files: 0
Unexpected files: 0
Missing content IDs: 0


In [73]:
anomalies = qa_df[
    (~qa_df["valid_json"]) |
    (~qa_df["has_results"]) |
    (~qa_df["id_matches_filename"]) |
    (~qa_df["in_manifest"]) |
    (qa_df["api_code"] != 0)
]

print("QA anomalies:", len(anomalies))

anomalies.head(20)

QA anomalies: 0


,filename,filename_id,content_id,api_code,valid_json,has_results,id_matches_filename,in_manifest,error


In [74]:
from collections import Counter, defaultdict

field_presence = Counter()
field_types = defaultdict(Counter)

for path in raw_files:
    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)

    record = raw["results"]

    for key, value in record.items():
        field_presence[key] += 1
        field_types[key][type(value).__name__] += 1

print("FIELDS FOUND:\n")

for field in sorted(field_presence):
    print(
        f"{field:30} "
        f"present={field_presence[field]:5} "
        f"types={dict(field_types[field])}"
    )

FIELDS FOUND:

actual_address                 present= 9917 types={'dict': 2066, 'NoneType': 7851}
actual_address_same_register   present= 9917 types={'bool': 9917}
code                           present= 9917 types={'str': 9917}
created_at                     present= 9917 types={'str': 9917}
email                          present= 9917 types={'NoneType': 9817, 'str': 100}
head_info                      present= 9917 types={'dict': 6471, 'list': 3446}
id                             present= 9917 types={'str': 9917}
is_active                      present= 9917 types={'bool': 9917}
name                           present= 9917 types={'str': 9917}
parent                         present= 9917 types={'dict': 9596, 'list': 321}
phone                          present= 9917 types={'NoneType': 9842, 'str': 75}
regional_offices               present= 9917 types={'list': 9917}
register_address               present= 9917 types={'dict': 9563, 'NoneType': 354}
updated_at                     present

In [75]:
nested_fields = [
    "head_info",
    "register_address",
    "actual_address",
    "parent",
    "regional_offices",
]

for field in nested_fields:
    print("\n" + "=" * 70)
    print(field.upper())

    types = Counter()
    lengths = Counter()

    for path in raw_files:
        with path.open("r", encoding="utf-8") as f:
            record = json.load(f)["results"]

        value = record.get(field)

        types[type(value).__name__] += 1

        if isinstance(value, (list, dict)):
            lengths[len(value)] += 1

    print("Types:", dict(types))
    print("Lengths:", dict(lengths))


HEAD_INFO
Types: {'dict': 6471, 'list': 3446}
Lengths: {3: 6345, 0: 3446, 2: 121, 1: 5}

REGISTER_ADDRESS
Types: {'dict': 9563, 'NoneType': 354}
Lengths: {23: 9563}

ACTUAL_ADDRESS
Types: {'dict': 2066, 'NoneType': 7851}
Lengths: {23: 2066}

PARENT
Types: {'dict': 9596, 'list': 321}
Lengths: {4: 9596, 0: 321}

REGIONAL_OFFICES
Types: {'list': 9917}
Lengths: {0: 9703, 276: 1, 3: 9, 4: 8, 64: 1, 13: 5, 205: 1, 1: 40, 5: 3, 20: 6, 34: 3, 32: 3, 40: 2, 576: 1, 2: 12, 431: 1, 696: 1, 153: 1, 694: 1, 33: 2, 28: 6, 16: 4, 67: 1, 21: 4, 14: 6, 31: 4, 320: 1, 9: 3, 76: 1, 7: 3, 26: 3, 24: 5, 38: 1, 191: 1, 17: 3, 27: 2, 59: 1, 44: 1, 49: 3, 51: 1, 10: 3, 57: 1, 19: 3, 18: 2, 207: 1, 65: 2, 23: 1, 71: 1, 66: 1, 25: 2, 41: 1, 22: 2, 72: 1, 30: 2, 46: 1, 85: 1, 15: 1, 78: 1, 152: 1, 37: 1, 83: 1, 47: 1, 36: 1, 138: 1, 58: 2, 75: 1, 103: 2, 90: 1, 77: 2, 122: 1, 306: 1, 60: 1, 82: 1, 11: 3, 192: 1, 146: 2, 239: 1, 120: 1, 86: 1, 8: 1, 6: 1, 12: 1, 43: 1, 63: 1}


In [76]:
codes = []

for path in raw_files:
    with path.open("r", encoding="utf-8") as f:
        record = json.load(f)["results"]

    codes.append(record.get("code"))

codes_series = pd.Series(codes, dtype="string")

print("Records:", len(codes_series))
print("Missing:", codes_series.isna().sum())
print("Unique:", codes_series.nunique(dropna=True))

print("\nLength distribution:")
print(
    codes_series
    .dropna()
    .str.len()
    .value_counts()
    .sort_index()
)

print("\nNon-digit examples:")
print(
    codes_series[
        codes_series.notna()
        & ~codes_series.str.fullmatch(r"\d+")
    ].head(20)
)

Records: 9917
Missing: 0
Unique: 9917

Length distribution:
7       1
8    9916
Name: count, dtype: Int64

Non-digit examples:
Series([], dtype: string)


In [77]:
from collections import Counter

head_key_sets = Counter()

for path in raw_files:
    with path.open("r", encoding="utf-8") as f:
        record = json.load(f)["results"]

    head = record.get("head_info")

    if isinstance(head, dict):
        head_key_sets[tuple(sorted(head.keys()))] += 1
    else:
        head_key_sets[()] += 1

for keys, count in head_key_sets.most_common():
    print(count, keys)

6345 ('head_first_name', 'head_last_name', 'head_middle_name')
3446 ()
115 ('head_first_name', 'head_last_name')
5 ('head_last_name', 'head_middle_name')
3 ('head_last_name',)
1 ('head_first_name',)
1 ('head_middle_name',)
1 ('head_first_name', 'head_middle_name')


In [78]:
head_field_presence = Counter()

for path in raw_files:
    with path.open("r", encoding="utf-8") as f:
        head = json.load(f)["results"].get("head_info")

    if isinstance(head, dict):
        for key, value in head.items():
            if value not in (None, ""):
                head_field_presence[key] += 1

print(head_field_presence)

Counter({'head_last_name': 6468, 'head_first_name': 6462, 'head_middle_name': 6352})


In [79]:
def profile_dict_field(field_name):
    key_presence = Counter()
    nonempty_presence = Counter()

    for path in raw_files:
        with path.open("r", encoding="utf-8") as f:
            value = json.load(f)["results"].get(field_name)

        if not isinstance(value, dict):
            continue

        for key, item in value.items():
            key_presence[key] += 1

            if item not in (None, ""):
                nonempty_presence[key] += 1

    result = pd.DataFrame({
        "present": pd.Series(key_presence),
        "nonempty": pd.Series(nonempty_presence)
    }).fillna(0).astype(int)

    result["empty_or_null"] = (
        result["present"] - result["nonempty"]
    )

    return result.sort_index()

In [80]:
register_address_profile = profile_dict_field(
    "register_address"
)

register_address_profile

,present,nonempty,empty_or_null
address_en,9563,0,9563
address_en_not_provided,9563,0,9563
address_uk,9563,0,9563
address_uk_not_provided,9563,0,9563
apartments,9563,2955,6608
apartments_not_provided,9563,23,9540
building,9563,8528,1035
building_not_provided,9563,11,9552
building_part_num,9563,312,9251
building_part_num_not_provided,9563,27,9536


In [81]:
actual_address_profile = profile_dict_field(
    "actual_address"
)

actual_address_profile

,present,nonempty,empty_or_null
address_en,2066,0,2066
address_en_not_provided,2066,0,2066
address_uk,2066,0,2066
address_uk_not_provided,2066,0,2066
apartments,2066,146,1920
apartments_not_provided,2066,0,2066
building,2066,258,1808
building_not_provided,2066,0,2066
building_part_num,2066,16,2050
building_part_num_not_provided,2066,2,2064


In [82]:
seven_digit_codes = []

for path in raw_files:
    with path.open("r", encoding="utf-8") as f:
        record = json.load(f)["results"]

    if len(record["code"]) != 8:
        seven_digit_codes.append({
            "id": record["id"],
            "code": record["code"],
            "name": record["name"],
            "parent": record.get("parent")
        })

pd.DataFrame(seven_digit_codes)

,id,code,name,parent
0,ed413385-2c34-407c-9080-5a6ab3143e84,3000002,Печерська районна у місті Києві організація Аграрної партії України,"{'id': '0d8460ee-fe65-427e-8efa-ec488676f345', 'is_active': True, 'code': '34763401', 'name': 'АГРАРНА ПАРТІЯ УКРАЇНИ'}"


In [83]:
timestamps = []

for path in raw_files:
    with path.open("r", encoding="utf-8") as f:
        record = json.load(f)["results"]

    timestamps.append({
        "id": record["id"],
        "created_at": record["created_at"],
        "updated_at": record["updated_at"]
    })

timestamps_df = pd.DataFrame(timestamps)

timestamps_df["created_parsed"] = pd.to_datetime(
    timestamps_df["created_at"],
    errors="coerce"
)

timestamps_df["updated_parsed"] = pd.to_datetime(
    timestamps_df["updated_at"],
    errors="coerce"
)

print(
    "Invalid created_at:",
    timestamps_df["created_parsed"].isna().sum()
)

print(
    "Invalid updated_at:",
    timestamps_df["updated_parsed"].isna().sum()
)

print(
    "Earliest created:",
    timestamps_df["created_parsed"].min()
)

print(
    "Latest created:",
    timestamps_df["created_parsed"].max()
)

print(
    "Earliest updated:",
    timestamps_df["updated_parsed"].min()
)

print(
    "Latest updated:",
    timestamps_df["updated_parsed"].max()
)

Invalid created_at: 0
Invalid updated_at: 0
Earliest created: 2021-05-13 12:02:34.582000
Latest created: 2026-08-09 12:45:48.796402
Earliest updated: 2025-07-01 08:47:24.964675
Latest updated: 2026-08-28 18:33:01.181341


In [84]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd


normalized_dir = Path("data/processed/normalized_v0_1")
normalized_dir.mkdir(parents=True, exist_ok=True)


address_fields = [
    "country",
    "post_index",
    "region",
    "district",
    "city",
    "street",
    "building",
    "apartments",
    "common",
    "building_part_num",
    "address_uk",
    "address_en",
    "post_index_not_provided",
    "district_not_provided",
    "common_not_provided",
    "city_not_provided",
    "region_not_provided",
    "street_not_provided",
    "building_not_provided",
    "building_part_num_not_provided",
    "apartments_not_provided",
    "address_en_not_provided",
    "address_uk_not_provided",
]


organization_rows = []
head_rows = []
address_rows = []


for path in raw_files:

    with path.open("r", encoding="utf-8") as f:
        api_response = json.load(f)

    record = api_response["results"]

    organization_id = record["id"]
    parent = record.get("parent")

    # Визначаємо тип і зв'язки
    if isinstance(parent, dict):
        entity_type = "office"
        parent_id = parent.get("id")
        root_party_id = parent_id
    else:
        entity_type = "party"
        parent_id = None
        root_party_id = organization_id

    # -------------------------
    # organizations
    # -------------------------

    organization_rows.append({
        "organization_id": organization_id,
        "root_party_id": root_party_id,
        "parent_id": parent_id,
        "entity_type": entity_type,

        "code": record.get("code"),
        "name": record.get("name"),
        "is_active": record.get("is_active"),

        "created_at": record.get("created_at"),
        "updated_at": record.get("updated_at"),

        "web_site_url": record.get("web_site_url"),
        "email": record.get("email"),
        "phone": record.get("phone"),

        "actual_address_same_register":
            record.get("actual_address_same_register"),
    })

    # -------------------------
    # organization_heads
    # -------------------------

    head = record.get("head_info")

    if isinstance(head, dict):

        head_rows.append({
            "organization_id": organization_id,
            "head_last_name":
                head.get("head_last_name"),
            "head_first_name":
                head.get("head_first_name"),
            "head_middle_name":
                head.get("head_middle_name"),
        })

    # -------------------------
    # organization_addresses
    # -------------------------

    for address_type, source_field in [
        ("register", "register_address"),
        ("actual", "actual_address"),
    ]:

        address = record.get(source_field)

        if not isinstance(address, dict):
            continue

        row = {
            "organization_id": organization_id,
            "address_type": address_type,
        }

        for field in address_fields:
            row[field] = address.get(field)

        address_rows.append(row)

In [85]:
organizations_df = pd.DataFrame(organization_rows)
heads_df = pd.DataFrame(head_rows)
addresses_df = pd.DataFrame(address_rows)

print("Organizations:", len(organizations_df))
print("Heads:", len(heads_df))
print("Addresses:", len(addresses_df))

print("\nEntity types:")
print(organizations_df["entity_type"].value_counts())

print("\nAddress types:")
print(addresses_df["address_type"].value_counts())

Organizations: 9917
Heads: 6471
Addresses: 11629

Entity types:
entity_type
office    9596
party      321
Name: count, dtype: int64

Address types:
address_type
register    9563
actual      2066
Name: count, dtype: int64


In [86]:
string_columns = [
    "organization_id",
    "root_party_id",
    "parent_id",
    "entity_type",
    "code",
    "name",
    "web_site_url",
    "email",
    "phone",
]

for col in string_columns:
    organizations_df[col] = (
        organizations_df[col].astype("string")
    )

organizations_df["is_active"] = (
    organizations_df["is_active"].astype("boolean")
)

organizations_df["actual_address_same_register"] = (
    organizations_df["actual_address_same_register"]
    .astype("boolean")
)

organizations_df["created_at"] = pd.to_datetime(
    organizations_df["created_at"],
    errors="raise"
)

organizations_df["updated_at"] = pd.to_datetime(
    organizations_df["updated_at"],
    errors="raise"
)

In [87]:
for col in [
    "organization_id",
    "head_last_name",
    "head_first_name",
    "head_middle_name",
]:
    heads_df[col] = heads_df[col].astype("string")

In [88]:
for col in addresses_df.columns:
    print(
        f"{col:35}",
        addresses_df[col]
        .map(type)
        .value_counts()
        .to_dict()
    )

organization_id                     {<class 'str'>: 11629}
address_type                        {<class 'str'>: 11629}
country                             {<class 'str'>: 11626, <class 'NoneType'>: 3}
post_index                          {<class 'str'>: 8743, <class 'NoneType'>: 2886}
region                              {<class 'str'>: 9732, <class 'NoneType'>: 1897}
district                            {<class 'str'>: 9668, <class 'NoneType'>: 1961}
city                                {<class 'str'>: 7866, <class 'NoneType'>: 3763}
street                              {<class 'str'>: 9699, <class 'NoneType'>: 1930}
building                            {<class 'str'>: 8913, <class 'NoneType'>: 2716}
apartments                          {<class 'NoneType'>: 7293, <class 'str'>: 4336}
common                              {<class 'NoneType'>: 9424, <class 'str'>: 2205}
building_part_num                   {<class 'NoneType'>: 9251, <class 'str'>: 2378}
address_uk                          {<class 

In [89]:
address_string_columns = [
    "organization_id",
    "address_type",
    "country",
    "post_index",
    "region",
    "district",
    "city",
    "street",
    "building",
    "apartments",
    "common",
    "building_part_num",
    "address_uk",
    "address_en",
]

address_boolean_columns = [
    "post_index_not_provided",
    "district_not_provided",
    "common_not_provided",
    "city_not_provided",
    "region_not_provided",
    "street_not_provided",
    "building_not_provided",
    "building_part_num_not_provided",
    "apartments_not_provided",
    "address_en_not_provided",
    "address_uk_not_provided",
]

for col in address_string_columns:
    addresses_df[col] = addresses_df[col].astype("string")

for col in address_boolean_columns:
    addresses_df[col] = addresses_df[col].astype("boolean")

In [90]:
print("ORGANIZATIONS")
print("Rows:", len(organizations_df))
print("Unique IDs:", organizations_df["organization_id"].nunique())
print("Missing IDs:", organizations_df["organization_id"].isna().sum())
print("Duplicate IDs:", organizations_df["organization_id"].duplicated().sum())

print("\nHEADS")
print("Rows:", len(heads_df))
print("Unique organization IDs:", heads_df["organization_id"].nunique())
print("Duplicate organization IDs:", heads_df["organization_id"].duplicated().sum())

print("\nADDRESSES")
print("Rows:", len(addresses_df))
print(
    addresses_df["address_type"]
    .value_counts(dropna=False)
)

ORGANIZATIONS
Rows: 9917
Unique IDs: 9917
Missing IDs: 0
Duplicate IDs: 0

HEADS
Rows: 6471
Unique organization IDs: 6471
Duplicate organization IDs: 0

ADDRESSES
Rows: 11629
address_type
register    9563
actual      2066
Name: count, dtype: Int64


In [91]:
organization_ids = set(
    organizations_df["organization_id"].dropna()
)

head_ids = set(
    heads_df["organization_id"].dropna()
)

address_ids = set(
    addresses_df["organization_id"].dropna()
)

print(
    "Head IDs missing from organizations:",
    len(head_ids - organization_ids)
)

print(
    "Address IDs missing from organizations:",
    len(address_ids - organization_ids)
)

office_rows = organizations_df[
    organizations_df["entity_type"] == "office"
]

party_rows = organizations_df[
    organizations_df["entity_type"] == "party"
]

print(
    "Office parent IDs missing from organizations:",
    len(
        set(office_rows["parent_id"].dropna())
        - organization_ids
    )
)

print(
    "Party rows with parent_id:",
    party_rows["parent_id"].notna().sum()
)

Head IDs missing from organizations: 0
Address IDs missing from organizations: 0
Office parent IDs missing from organizations: 0
Party rows with parent_id: 0


In [92]:
print(
    "Office rows:",
    len(office_rows)
)

print(
    "Office root_party_id == parent_id:",
    (
        office_rows["root_party_id"]
        == office_rows["parent_id"]
    ).sum()
)

print(
    "Party root_party_id == organization_id:",
    (
        party_rows["root_party_id"]
        == party_rows["organization_id"]
    ).sum()
)

Office rows: 9596
Office root_party_id == parent_id: 9596
Party root_party_id == organization_id: 321


In [93]:
organizations_path = (
    normalized_dir / "organizations.parquet"
)

heads_path = (
    normalized_dir / "organization_heads.parquet"
)

addresses_path = (
    normalized_dir / "organization_addresses.parquet"
)

organizations_df.to_parquet(
    organizations_path,
    index=False
)

heads_df.to_parquet(
    heads_path,
    index=False
)

addresses_df.to_parquet(
    addresses_path,
    index=False
)

print("Saved:")
print(organizations_path)
print(heads_path)
print(addresses_path)

Saved:
data\processed\normalized_v0_1\organizations.parquet
data\processed\normalized_v0_1\organization_heads.parquet
data\processed\normalized_v0_1\organization_addresses.parquet


In [94]:
organizations_check = pd.read_parquet(
    normalized_dir / "organizations.parquet"
)

heads_check = pd.read_parquet(
    normalized_dir / "organization_heads.parquet"
)

addresses_check = pd.read_parquet(
    normalized_dir / "organization_addresses.parquet"
)

print("Organizations:", organizations_check.shape)
print("Heads:", heads_check.shape)
print("Addresses:", addresses_check.shape)

Organizations: (9917, 13)
Heads: (6471, 4)
Addresses: (11629, 25)


In [95]:
print("ORGANIZATIONS DTYPES")
print(organizations_check.dtypes)

print("\nHEADS DTYPES")
print(heads_check.dtypes)

print("\nADDRESSES DTYPES")
print(addresses_check.dtypes)

ORGANIZATIONS DTYPES
organization_id                 string[python]
root_party_id                   string[python]
parent_id                       string[python]
entity_type                     string[python]
code                            string[python]
name                            string[python]
is_active                              boolean
created_at                      datetime64[ns]
updated_at                      datetime64[ns]
web_site_url                    string[python]
email                           string[python]
phone                           string[python]
actual_address_same_register           boolean
dtype: object

HEADS DTYPES
organization_id     string[python]
head_last_name      string[python]
head_first_name     string[python]
head_middle_name    string[python]
dtype: object

ADDRESSES DTYPES
organization_id                   string[python]
address_type                      string[python]
country                           string[python]
post_index           

In [96]:
print(
    "Organization IDs unique:",
    organizations_check["organization_id"].nunique()
)

print(
    "Office rows:",
    (organizations_check["entity_type"] == "office").sum()
)

print(
    "Party rows:",
    (organizations_check["entity_type"] == "party").sum()
)

print(
    "Heads:",
    len(heads_check)
)

print(
    "Register addresses:",
    (addresses_check["address_type"] == "register").sum()
)

print(
    "Actual addresses:",
    (addresses_check["address_type"] == "actual").sum()
)

Organization IDs unique: 9917
Office rows: 9596
Party rows: 321
Heads: 6471
Register addresses: 9563
Actual addresses: 2066


In [97]:
from pathlib import Path

pyproject = """[build-system]
requires = ["setuptools>=68"]
build-backend = "setuptools.build_meta"

[project]
name = "politdata-pipeline"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "requests>=2.32",
    "pandas>=2.3",
    "pyarrow>=22",
    "tqdm>=4.67",
]

[tool.setuptools]
package-dir = {"" = "src"}

[tool.setuptools.packages.find]
where = ["src"]
"""

Path("pyproject.toml").write_text(
    pyproject,
    encoding="utf-8"
)

print("Created pyproject.toml")

Created pyproject.toml


In [98]:
%pip install -e .

Obtaining file:///C:/Users/Igor/Desktop/Projects/politdata-pipeline
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for politdata-pipeline (pyproject.toml): started
  Building editable for politdata-pipeline (pyproject.toml): finished with status 'done'
  Created wheel for politdata-pipeline: filename=politdata_pipeline-0.1.0-0.editable-py3-none-any.whl size=1381 sha256=c6b9676c5f8ce8f2a7e403ff06be4c667a5a55803609e4a4f974d91978f0a587
  Stored in directory: C:\Users\Igor\AppData\Local\Temp\pip-ephem-wheel-cache-fadqtr3p\wheels\

In [99]:
%%writefile src/politdata/api.py

from math import ceil
import time

import requests


DEFAULT_BASE_URL = "https://politdata.nazk.gov.ua/api/v2"


def fetch_all_parties(
    base_url=DEFAULT_BASE_URL,
    page_size=100,
    timeout=30,
):
    """
    Download the current /parties index from PolitData.

    Important:
    This function performs fresh discovery every time it runs.
    It does not rely on a previously saved list of party IDs.
    """

    endpoint = f"{base_url}/parties"

    session = requests.Session()

    def fetch_page(page):
        payload = {
            "filters": {},
            "order": {},
            "pager": {
                "page": page,
                "size": page_size,
            },
        }

        response = session.post(
            endpoint,
            json=payload,
            timeout=timeout,
        )

        response.raise_for_status()

        return response.json()["results"]

    first_page = fetch_page(1)

    total_count = first_page["count"]
    total_pages = ceil(total_count / page_size)

    records = list(first_page["list"])

    for page in range(2, total_pages + 1):
        page_results = fetch_page(page)
        records.extend(page_results["list"])

    if len(records) != total_count:
        raise ValueError(
            f"PolitData reported {total_count} records, "
            f"but {len(records)} were downloaded."
        )

    ids = [record["id"] for record in records]

    if len(ids) != len(set(ids)):
        raise ValueError(
            "Duplicate organization IDs returned by /parties."
        )

    return records


def fetch_party_account(
    organization_id,
    base_url=DEFAULT_BASE_URL,
    timeout=30,
    max_retries=4,
):
    """
    Download the full PolitData account for one party
    or regional/local party organization.
    """

    url = f"{base_url}/party/{organization_id}"

    session = requests.Session()

    last_error = None

    for attempt in range(1, max_retries + 1):

        try:
            response = session.get(
                url,
                timeout=timeout,
            )

            response.raise_for_status()

            data = response.json()

            if "results" not in data:
                raise ValueError(
                    f"No 'results' field for {organization_id}"
                )

            returned_id = data["results"].get("id")

            if returned_id != organization_id:
                raise ValueError(
                    f"Requested {organization_id}, "
                    f"but API returned {returned_id}"
                )

            return data

        except Exception as exc:
            last_error = exc

            if attempt < max_retries:
                time.sleep(attempt * 2)

    raise last_error

Writing src/politdata/api.py


In [1]:
from politdata.api import (
    DEFAULT_BASE_URL,
    fetch_all_parties,
    fetch_party_account,
)

print(DEFAULT_BASE_URL)

https://politdata.nazk.gov.ua/api/v2


In [2]:
fresh_parties = fetch_all_parties()

print("Current records:", len(fresh_parties))
print("First:", fresh_parties[0]["name"])

Current records: 321
First: ПОЛІТИЧНА ПАРТІЯ «ОБ'ЄДНАНА КРАЇНА»


In [3]:
%%writefile src/politdata/discovery.py

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


COMPARE_COLUMNS = [
    "root_party_id",
    "parent_id",
    "entity_type",
    "code",
    "name",
    "is_active",
    "source_updated_at",
]


def build_organization_manifest(parties, discovered_at_utc=None):
    """
    Build a fresh organization manifest from the current /parties response.

    The manifest is rebuilt from scratch on every discovery run.
    """

    if discovered_at_utc is None:
        discovered_at_utc = datetime.now(timezone.utc).isoformat()

    rows = []

    for party in parties:
        party_id = party["id"]

        # Central party
        rows.append({
            "organization_id": party_id,
            "root_party_id": party_id,
            "parent_id": None,
            "entity_type": "party",
            "code": party.get("code"),
            "name": party.get("name"),
            "is_active": party.get("is_active"),
            "source_created_at": party.get("created_at"),
            "source_updated_at": party.get("updated_at"),
            "discovered_at_utc": discovered_at_utc,
        })

        # Regional/local organizations
        for office in party.get("regional_offices") or []:
            rows.append({
                "organization_id": office["id"],
                "root_party_id": party_id,
                "parent_id": party_id,
                "entity_type": "office",
                "code": office.get("code"),
                "name": office.get("name"),
                "is_active": office.get("is_active"),
                "source_created_at": None,
                "source_updated_at": None,
                "discovered_at_utc": discovered_at_utc,
            })

    df = pd.DataFrame(rows)

    if df["organization_id"].isna().any():
        raise ValueError("Manifest contains missing organization IDs.")

    if df["organization_id"].duplicated().any():
        duplicates = df.loc[
            df["organization_id"].duplicated(),
            "organization_id"
        ].tolist()

        raise ValueError(
            f"Duplicate organization IDs in manifest: {duplicates[:10]}"
        )

    return df


def compare_manifests(current, previous):
    """
    Compare current discovery with a previous manifest.

    Returns:
        summary
        new_df
        disappeared_df
        changed_df
    """

    current = current.copy()
    previous = previous.copy()

    current_ids = set(current["organization_id"])
    previous_ids = set(previous["organization_id"])

    new_ids = current_ids - previous_ids
    disappeared_ids = previous_ids - current_ids
    common_ids = current_ids & previous_ids

    new_df = current[
        current["organization_id"].isin(new_ids)
    ].copy()

    disappeared_df = previous[
        previous["organization_id"].isin(disappeared_ids)
    ].copy()

    current_indexed = current.set_index("organization_id")
    previous_indexed = previous.set_index("organization_id")

    # Only compare fields available in both manifest versions.
    compare_columns = [
        col
        for col in COMPARE_COLUMNS
        if col in current.columns and col in previous.columns
    ]

    changed_rows = []

    for organization_id in common_ids:
        changed_fields = []

        for col in compare_columns:
            current_value = current_indexed.at[
                organization_id, col
            ]
            previous_value = previous_indexed.at[
                organization_id, col
            ]

            current_missing = pd.isna(current_value)
            previous_missing = pd.isna(previous_value)

            if current_missing and previous_missing:
                continue

            if current_missing != previous_missing:
                changed_fields.append(col)
                continue

            if current_value != previous_value:
                changed_fields.append(col)

        if changed_fields:
            row = current_indexed.loc[
                organization_id
            ].to_dict()

            row["organization_id"] = organization_id
            row["changed_fields"] = changed_fields

            changed_rows.append(row)

    changed_df = pd.DataFrame(changed_rows)

    summary = {
        "current": len(current_ids),
        "previous": len(previous_ids),
        "new": len(new_ids),
        "disappeared": len(disappeared_ids),
        "existing": len(common_ids),
        "changed": len(changed_rows),
        "compared_columns": compare_columns,
    }

    return summary, new_df, disappeared_df, changed_df


def save_manifest_snapshot(
    manifest,
    output_dir="data/interim/manifests",
):
    """
    Save a timestamped manifest snapshot and update latest.parquet.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now(timezone.utc).strftime(
        "%Y-%m-%d_%H-%M-%S_UTC"
    )

    snapshot_path = (
        output_dir /
        f"organization_manifest_{timestamp}.parquet"
    )

    latest_path = (
        output_dir /
        "organization_manifest_latest.parquet"
    )

    manifest.to_parquet(
        snapshot_path,
        index=False
    )

    manifest.to_parquet(
        latest_path,
        index=False
    )

    return snapshot_path, latest_path

Writing src/politdata/discovery.py


In [1]:
from politdata.api import fetch_all_parties
from politdata.discovery import (
    build_organization_manifest,
    compare_manifests,
    save_manifest_snapshot,
)

In [2]:
fresh_parties = fetch_all_parties()

current_manifest = build_organization_manifest(
    fresh_parties
)

print("Fresh parties:", len(fresh_parties))
print("Manifest organizations:", len(current_manifest))

print()
print(
    current_manifest["entity_type"]
    .value_counts()
)

Fresh parties: 321
Manifest organizations: 9917

entity_type
office    9596
party      321
Name: count, dtype: int64


In [4]:
import pandas as pd

In [5]:
previous_manifest = pd.read_parquet(
    "data/interim/organization_manifest.parquet"
)

summary, new_orgs, disappeared_orgs, changed_orgs = (
    compare_manifests(
        current_manifest,
        previous_manifest
    )
)

summary

{'current': 9917,
 'previous': 9917,
 'new': 0,
 'disappeared': 0,
 'existing': 9917,
 'changed': 0,
 'compared_columns': ['root_party_id',
  'parent_id',
  'entity_type',
  'code',
  'name']}

In [6]:
snapshot_path, latest_path = save_manifest_snapshot(
    current_manifest
)

print("Snapshot:", snapshot_path)
print("Latest:", latest_path)

Snapshot: data\interim\manifests\organization_manifest_2026-08-29_07-26-26_UTC.parquet
Latest: data\interim\manifests\organization_manifest_latest.parquet


In [7]:
previous_manifest_v2 = pd.read_parquet(
    "data/interim/manifests/organization_manifest_latest.parquet"
)

fresh_parties_v2 = fetch_all_parties()

current_manifest_v2 = build_organization_manifest(
    fresh_parties_v2
)

summary_v2, new_v2, disappeared_v2, changed_v2 = (
    compare_manifests(
        current_manifest_v2,
        previous_manifest_v2
    )
)

summary_v2

{'current': 9917,
 'previous': 9917,
 'new': 0,
 'disappeared': 0,
 'existing': 9917,
 'changed': 1,
 'compared_columns': ['root_party_id',
  'parent_id',
  'entity_type',
  'code',
  'name',
  'is_active',
  'source_updated_at']}

In [8]:
changed_v2[
    [
        "organization_id",
        "entity_type",
        "code",
        "name",
        "is_active",
        "source_updated_at",
        "changed_fields",
    ]
]

,organization_id,entity_type,code,name,is_active,source_updated_at,changed_fields
0,b10b0b92-76fd-47ee-87a7-31b0765e99a9,party,37932584,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ УКРАЇНСЬКОГО НАРОДУ»,False,2026-08-29 10:26:01.25834,[source_updated_at]


In [9]:
changed_id = changed_v2.iloc[0]["organization_id"]

old_row = (
    previous_manifest_v2
    .set_index("organization_id")
    .loc[changed_id]
)

new_row = (
    current_manifest_v2
    .set_index("organization_id")
    .loc[changed_id]
)

print("ORGANIZATION:")
print(new_row["name"])
print()

for col in summary_v2["compared_columns"]:
    old_value = old_row[col]
    new_value = new_row[col]

    if (
        (pd.isna(old_value) and pd.isna(new_value))
        or old_value == new_value
    ):
        continue

    print(col)
    print("OLD:", old_value)
    print("NEW:", new_value)
    print()

ORGANIZATION:
ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ УКРАЇНСЬКОГО НАРОДУ»

source_updated_at
OLD: 2026-08-29 10:22:01.273636
NEW: 2026-08-29 10:26:01.25834



In [13]:
from pathlib import Path
import json

In [16]:
changed_id = changed_v2.iloc[0]["organization_id"]

old_path = (
    Path("data/raw/party_accounts")
    / f"{changed_id}.json"
)

with old_path.open("r", encoding="utf-8") as f:
    old_full = json.load(f)

old_detail = old_full["results"]

new_full = fetch_party_account(changed_id)
new_detail = new_full["results"]

print("OLD updated_at:", old_detail["updated_at"])
print("NEW updated_at:", new_detail["updated_at"])

OLD updated_at: 2026-08-28 18:33:01.181341
NEW updated_at: 2026-08-29 10:34:01.208653


In [17]:
def deep_diff(old, new, path=""):
    differences = []

    if type(old) != type(new):
        differences.append({
            "field": path,
            "old": old,
            "new": new,
        })
        return differences

    if isinstance(old, dict):
        keys = set(old) | set(new)

        for key in sorted(keys):
            new_path = f"{path}.{key}" if path else key

            if key not in old:
                differences.append({
                    "field": new_path,
                    "old": "<MISSING>",
                    "new": new[key],
                })

            elif key not in new:
                differences.append({
                    "field": new_path,
                    "old": old[key],
                    "new": "<MISSING>",
                })

            else:
                differences.extend(
                    deep_diff(
                        old[key],
                        new[key],
                        new_path,
                    )
                )

    elif isinstance(old, list):
        if old != new:
            differences.append({
                "field": path,
                "old": old,
                "new": new,
            })

    else:
        if old != new:
            differences.append({
                "field": path,
                "old": old,
                "new": new,
            })

    return differences

In [18]:
differences = deep_diff(
    old_detail,
    new_detail
)

print("Changed fields:", len(differences))

for diff in differences:
    print("\nFIELD:", diff["field"])
    print("OLD:", diff["old"])
    print("NEW:", diff["new"])

Changed fields: 1

FIELD: updated_at
OLD: 2026-08-28 18:33:01.181341
NEW: 2026-08-29 10:34:01.208653


In [19]:
import hashlib
import json
from copy import deepcopy


def organization_content_hash(record):
    clean = deepcopy(record)

    # updated_at не вважаємо змістовною частиною картки
    clean.pop("updated_at", None)

    canonical = json.dumps(
        clean,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical.encode("utf-8")
    ).hexdigest()

In [20]:
old_hash = organization_content_hash(old_detail)
new_hash = organization_content_hash(new_detail)

print("Old content hash:", old_hash)
print("New content hash:", new_hash)
print("Content identical:", old_hash == new_hash)

Old content hash: 2b8f6d075a39c3b6dd57654f558f522825fbb2808cdbecd15f4b99ecc45bd94b
New content hash: 2b8f6d075a39c3b6dd57654f558f522825fbb2808cdbecd15f4b99ecc45bd94b
Content identical: True


In [21]:
%%writefile src/politdata/change_detection.py

from copy import deepcopy
import hashlib
import json


DEFAULT_IGNORED_FIELDS = {
    "updated_at",
}


def canonicalize_record(
    record,
    ignored_fields=None,
):
    """
    Return a canonical copy of a PolitData organization record
    for meaningful-content comparison.

    Technical fields such as updated_at may be excluded.
    """

    if ignored_fields is None:
        ignored_fields = DEFAULT_IGNORED_FIELDS

    clean = deepcopy(record)

    for field in ignored_fields:
        clean.pop(field, None)

    return clean


def organization_content_hash(
    record,
    ignored_fields=None,
):
    """
    Calculate a stable SHA-256 hash of meaningful organization content.
    """

    clean = canonicalize_record(
        record,
        ignored_fields=ignored_fields,
    )

    canonical_json = json.dumps(
        clean,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        canonical_json.encode("utf-8")
    ).hexdigest()


def deep_diff(old, new, path=""):
    """
    Recursively compare two JSON-compatible objects.
    """

    differences = []

    if type(old) != type(new):
        differences.append({
            "field": path,
            "old": old,
            "new": new,
        })
        return differences

    if isinstance(old, dict):

        keys = set(old) | set(new)

        for key in sorted(keys):

            new_path = (
                f"{path}.{key}"
                if path
                else key
            )

            if key not in old:
                differences.append({
                    "field": new_path,
                    "old": "<MISSING>",
                    "new": new[key],
                })

            elif key not in new:
                differences.append({
                    "field": new_path,
                    "old": old[key],
                    "new": "<MISSING>",
                })

            else:
                differences.extend(
                    deep_diff(
                        old[key],
                        new[key],
                        new_path,
                    )
                )

    elif isinstance(old, list):

        if old != new:
            differences.append({
                "field": path,
                "old": old,
                "new": new,
            })

    else:

        if old != new:
            differences.append({
                "field": path,
                "old": old,
                "new": new,
            })

    return differences


def classify_record_change(
    old_record,
    new_record,
    ignored_fields=None,
):
    """
    Classify whether a fetched organization record changed meaningfully.

    Returns both the full diff and the content hash comparison.
    """

    if ignored_fields is None:
        ignored_fields = DEFAULT_IGNORED_FIELDS

    all_differences = deep_diff(
        old_record,
        new_record,
    )

    old_hash = organization_content_hash(
        old_record,
        ignored_fields=ignored_fields,
    )

    new_hash = organization_content_hash(
        new_record,
        ignored_fields=ignored_fields,
    )

    content_changed = old_hash != new_hash

    return {
        "content_changed": content_changed,
        "old_content_hash": old_hash,
        "new_content_hash": new_hash,
        "changed_fields": [
            diff["field"]
            for diff in all_differences
        ],
        "differences": all_differences,
    }

Writing src/politdata/change_detection.py


In [1]:
from politdata.change_detection import (
    organization_content_hash,
    deep_diff,
    classify_record_change,
)

In [5]:
previous_manifest = pd.read_parquet(
    "data/interim/manifests/organization_manifest_latest.parquet"
)

fresh_parties = fetch_all_parties()

current_manifest = build_organization_manifest(
    fresh_parties
)

summary, new_orgs, disappeared_orgs, changed_orgs = (
    compare_manifests(
        current_manifest,
        previous_manifest
    )
)

print(summary)

{'current': 9917, 'previous': 9917, 'new': 0, 'disappeared': 0, 'existing': 9917, 'changed': 1, 'compared_columns': ['root_party_id', 'parent_id', 'entity_type', 'code', 'name', 'is_active', 'source_updated_at']}


In [6]:
print("Changed organizations:", len(changed_orgs))

if len(changed_orgs) > 0:
    display(
        changed_orgs[
            [
                "organization_id",
                "entity_type",
                "code",
                "name",
                "source_updated_at",
                "changed_fields",
            ]
        ]
    )

Changed organizations: 1


,organization_id,entity_type,code,name,source_updated_at,changed_fields
0,b10b0b92-76fd-47ee-87a7-31b0765e99a9,party,37932584,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ УКРАЇНСЬКОГО НАРОДУ»,2026-08-29 10:44:01.433021,[source_updated_at]


In [7]:
changed_id = changed_orgs.iloc[0]["organization_id"]

print("Changed ID:", changed_id)
print("Name:", changed_orgs.iloc[0]["name"])

Changed ID: b10b0b92-76fd-47ee-87a7-31b0765e99a9
Name: ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ УКРАЇНСЬКОГО НАРОДУ»


In [8]:
old_path = (
    Path("data/raw/party_accounts")
    / f"{changed_id}.json"
)

with old_path.open("r", encoding="utf-8") as f:
    old_full = json.load(f)

old_detail = old_full["results"]

new_full = fetch_party_account(changed_id)
new_detail = new_full["results"]

print("Organization:", new_detail["name"])
print("OLD updated_at:", old_detail["updated_at"])
print("NEW updated_at:", new_detail["updated_at"])

Organization: ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ УКРАЇНСЬКОГО НАРОДУ»
OLD updated_at: 2026-08-28 18:33:01.181341
NEW updated_at: 2026-08-29 10:45:01.208136


In [9]:
change_test = classify_record_change(
    old_detail,
    new_detail
)

print(
    "Content changed:",
    change_test["content_changed"]
)

print(
    "Changed fields:",
    change_test["changed_fields"]
)

print(
    "Old hash:",
    change_test["old_content_hash"]
)

print(
    "New hash:",
    change_test["new_content_hash"]
)

Content changed: False
Changed fields: ['updated_at']
Old hash: 2b8f6d075a39c3b6dd57654f558f522825fbb2808cdbecd15f4b99ecc45bd94b
New hash: 2b8f6d075a39c3b6dd57654f558f522825fbb2808cdbecd15f4b99ecc45bd94b


In [10]:
%%writefile src/politdata/discovery.py

from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


INDEX_COMPARE_COLUMNS = [
    "root_party_id",
    "parent_id",
    "entity_type",
    "code",
    "name",
    "is_active",
]

REFRESH_TRIGGER_COLUMNS = [
    "source_updated_at",
]


def build_organization_manifest(
    parties,
    discovered_at_utc=None,
):
    """
    Build a fresh organization manifest from the current /parties response.

    The manifest is rebuilt from scratch on every discovery run.
    """

    if discovered_at_utc is None:
        discovered_at_utc = datetime.now(
            timezone.utc
        ).isoformat()

    rows = []

    for party in parties:
        party_id = party["id"]

        # Central party
        rows.append({
            "organization_id": party_id,
            "root_party_id": party_id,
            "parent_id": None,
            "entity_type": "party",
            "code": party.get("code"),
            "name": party.get("name"),
            "is_active": party.get("is_active"),
            "source_created_at": party.get("created_at"),
            "source_updated_at": party.get("updated_at"),
            "discovered_at_utc": discovered_at_utc,
        })

        # Regional/local organizations
        for office in party.get("regional_offices") or []:
            rows.append({
                "organization_id": office["id"],
                "root_party_id": party_id,
                "parent_id": party_id,
                "entity_type": "office",
                "code": office.get("code"),
                "name": office.get("name"),
                "is_active": office.get("is_active"),

                # /parties does not expose these for offices
                "source_created_at": None,
                "source_updated_at": None,

                "discovered_at_utc": discovered_at_utc,
            })

    df = pd.DataFrame(rows)

    if df["organization_id"].isna().any():
        raise ValueError(
            "Manifest contains missing organization IDs."
        )

    if df["organization_id"].duplicated().any():
        duplicates = df.loc[
            df["organization_id"].duplicated(),
            "organization_id",
        ].tolist()

        raise ValueError(
            f"Duplicate organization IDs: {duplicates[:10]}"
        )

    return df


def _values_different(old_value, new_value):
    old_missing = pd.isna(old_value)
    new_missing = pd.isna(new_value)

    if old_missing and new_missing:
        return False

    if old_missing != new_missing:
        return True

    return old_value != new_value


def compare_manifests(
    current,
    previous,
):
    """
    Compare fresh discovery with a previous committed manifest.

    Separates:
    - new organizations
    - disappeared organizations
    - meaningful changes visible in /parties
    - refresh candidates triggered by technical/source fields
    """

    current = current.copy()
    previous = previous.copy()

    current_ids = set(
        current["organization_id"]
    )

    previous_ids = set(
        previous["organization_id"]
    )

    new_ids = current_ids - previous_ids
    disappeared_ids = previous_ids - current_ids
    common_ids = current_ids & previous_ids

    new_df = current[
        current["organization_id"].isin(new_ids)
    ].copy()

    disappeared_df = previous[
        previous["organization_id"].isin(
            disappeared_ids
        )
    ].copy()

    current_indexed = current.set_index(
        "organization_id"
    )

    previous_indexed = previous.set_index(
        "organization_id"
    )

    index_columns = [
        col
        for col in INDEX_COMPARE_COLUMNS
        if (
            col in current.columns
            and col in previous.columns
        )
    ]

    refresh_columns = [
        col
        for col in REFRESH_TRIGGER_COLUMNS
        if (
            col in current.columns
            and col in previous.columns
        )
    ]

    index_changed_rows = []
    refresh_candidate_rows = []

    for organization_id in sorted(common_ids):

        index_changed_fields = []
        refresh_trigger_fields = []

        for col in index_columns:
            old_value = previous_indexed.at[
                organization_id, col
            ]
            new_value = current_indexed.at[
                organization_id, col
            ]

            if _values_different(
                old_value,
                new_value,
            ):
                index_changed_fields.append(col)

        for col in refresh_columns:
            old_value = previous_indexed.at[
                organization_id, col
            ]
            new_value = current_indexed.at[
                organization_id, col
            ]

            if _values_different(
                old_value,
                new_value,
            ):
                refresh_trigger_fields.append(col)

        if index_changed_fields:
            row = current_indexed.loc[
                organization_id
            ].to_dict()

            row["organization_id"] = (
                organization_id
            )

            row["changed_fields"] = (
                index_changed_fields
            )

            index_changed_rows.append(row)

        if (
            index_changed_fields
            or refresh_trigger_fields
        ):
            row = current_indexed.loc[
                organization_id
            ].to_dict()

            row["organization_id"] = (
                organization_id
            )

            row["index_changed_fields"] = (
                index_changed_fields
            )

            row["refresh_trigger_fields"] = (
                refresh_trigger_fields
            )

            refresh_candidate_rows.append(row)

    index_changed_df = pd.DataFrame(
        index_changed_rows
    )

    refresh_candidates_df = pd.DataFrame(
        refresh_candidate_rows
    )

    summary = {
        "current": len(current_ids),
        "previous": len(previous_ids),
        "new": len(new_ids),
        "disappeared": len(disappeared_ids),
        "existing": len(common_ids),
        "index_changed": len(index_changed_rows),
        "refresh_candidates": len(
            refresh_candidate_rows
        ),
        "index_compared_columns": index_columns,
        "refresh_trigger_columns": refresh_columns,
    }

    return (
        summary,
        new_df,
        disappeared_df,
        index_changed_df,
        refresh_candidates_df,
    )


def save_discovery_snapshot(
    manifest,
    output_dir="data/interim/manifests",
):
    """
    Save a timestamped manifest produced by discovery.

    Does NOT change the committed baseline.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    timestamp = datetime.now(
        timezone.utc
    ).strftime(
        "%Y-%m-%d_%H-%M-%S_UTC"
    )

    snapshot_path = (
        output_dir
        / f"organization_manifest_{timestamp}.parquet"
    )

    manifest.to_parquet(
        snapshot_path,
        index=False,
    )

    return snapshot_path


def save_committed_manifest(
    manifest,
    output_dir="data/interim/manifests",
):
    """
    Update the committed baseline only after
    synchronization has completed successfully.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    committed_path = (
        output_dir
        / "organization_manifest_committed.parquet"
    )

    manifest.to_parquet(
        committed_path,
        index=False,
    )

    return committed_path

Overwriting src/politdata/discovery.py


In [2]:
previous_manifest = pd.read_parquet(
    "data/interim/manifests/organization_manifest_latest.parquet"
)

fresh_parties = fetch_all_parties()

current_manifest = build_organization_manifest(
    fresh_parties
)

(
    summary,
    new_orgs,
    disappeared_orgs,
    index_changed_orgs,
    refresh_candidates,
) = compare_manifests(
    current_manifest,
    previous_manifest,
)

summary

{'current': 9917,
 'previous': 9917,
 'new': 0,
 'disappeared': 0,
 'existing': 9917,
 'index_changed': 0,
 'refresh_candidates': 1,
 'index_compared_columns': ['root_party_id',
  'parent_id',
  'entity_type',
  'code',
  'name',
  'is_active'],
 'refresh_trigger_columns': ['source_updated_at']}

In [3]:
print("Refresh candidates:", len(refresh_candidates))

if len(refresh_candidates) > 0:
    display(
        refresh_candidates[
            [
                "organization_id",
                "entity_type",
                "code",
                "name",
                "source_updated_at",
                "index_changed_fields",
                "refresh_trigger_fields",
            ]
        ]
    )

Refresh candidates: 1


,organization_id,entity_type,code,name,source_updated_at,index_changed_fields,refresh_trigger_fields
0,b10b0b92-76fd-47ee-87a7-31b0765e99a9,party,37932584,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ УКРАЇНСЬКОГО НАРОДУ»,2026-08-29 10:47:01.275225,[],[source_updated_at]


In [4]:
previous_manifest = pd.read_parquet(
    "data/interim/manifests/organization_manifest_latest.parquet"
)

committed_path = save_committed_manifest(
    previous_manifest
)

print("Committed baseline:", committed_path)
print("Rows:", len(previous_manifest))

Committed baseline: data\interim\manifests\organization_manifest_committed.parquet
Rows: 9917


In [5]:
%%writefile src/politdata/sync.py

from datetime import datetime, timezone
from pathlib import Path
import json
import shutil

import pandas as pd

from .api import (
    fetch_all_parties,
    fetch_party_account,
)

from .discovery import (
    build_organization_manifest,
    compare_manifests,
    save_discovery_snapshot,
    save_committed_manifest,
)

from .change_detection import (
    classify_record_change,
)


DEFAULT_COMMITTED_MANIFEST = Path(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

DEFAULT_CURRENT_RAW_DIR = Path(
    "data/raw/party_accounts"
)

DEFAULT_VERSION_DIR = Path(
    "data/raw/party_account_versions"
)

DEFAULT_LOG_DIR = Path(
    "logs/sync_runs"
)


def _write_json(path, data):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temp_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

    temp_path.replace(path)


def _read_json(path):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def _archive_baseline_if_needed(
    organization_id,
    current_raw_path,
    version_dir,
):
    """
    Preserve our original pre-versioning RAW file
    before it is overwritten for the first time.
    """

    if not current_raw_path.exists():
        return

    org_version_dir = (
        version_dir / organization_id
    )

    org_version_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    baseline_path = (
        org_version_dir
        / "baseline_initial.json"
    )

    if not baseline_path.exists():
        shutil.copy2(
            current_raw_path,
            baseline_path,
        )


def _save_fetched_version(
    organization_id,
    api_response,
    retrieved_at,
    version_dir,
):
    org_version_dir = (
        version_dir / organization_id
    )

    org_version_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    timestamp = retrieved_at.strftime(
        "%Y-%m-%d_%H-%M-%S_%f_UTC"
    )

    version_path = (
        org_version_dir
        / f"{timestamp}.json"
    )

    _write_json(
        version_path,
        api_response,
    )

    return version_path


def run_organization_sync(
    committed_manifest_path=DEFAULT_COMMITTED_MANIFEST,
    current_raw_dir=DEFAULT_CURRENT_RAW_DIR,
    version_dir=DEFAULT_VERSION_DIR,
    log_dir=DEFAULT_LOG_DIR,
):
    """
    Run one incremental synchronization cycle.

    The committed manifest is updated ONLY if all required
    organization fetches succeed.
    """

    run_started = datetime.now(
        timezone.utc
    )

    current_raw_dir = Path(
        current_raw_dir
    )

    version_dir = Path(
        version_dir
    )

    log_dir = Path(
        log_dir
    )

    log_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not Path(
        committed_manifest_path
    ).exists():
        raise FileNotFoundError(
            "Committed manifest not found: "
            f"{committed_manifest_path}"
        )

    previous_manifest = pd.read_parquet(
        committed_manifest_path
    )

    # --------------------------------
    # DISCOVERY
    # --------------------------------

    fresh_parties = fetch_all_parties()

    current_manifest = (
        build_organization_manifest(
            fresh_parties
        )
    )

    discovery_snapshot = (
        save_discovery_snapshot(
            current_manifest
        )
    )

    (
        discovery_summary,
        new_orgs,
        disappeared_orgs,
        index_changed_orgs,
        refresh_candidates,
    ) = compare_manifests(
        current_manifest,
        previous_manifest,
    )

    # NEW always need full fetch.
    candidate_ids = set(
        new_orgs["organization_id"].tolist()
    )

    # Index/source changes also need verification.
    if not refresh_candidates.empty:
        candidate_ids.update(
            refresh_candidates[
                "organization_id"
            ].tolist()
        )

    candidate_ids = sorted(candidate_ids)

    # --------------------------------
    # FETCH + CONTENT COMPARISON
    # --------------------------------

    results = []
    failures = []

    for organization_id in candidate_ids:

        current_raw_path = (
            current_raw_dir
            / f"{organization_id}.json"
        )

        try:
            old_response = None

            if current_raw_path.exists():
                old_response = _read_json(
                    current_raw_path
                )

            new_response = (
                fetch_party_account(
                    organization_id
                )
            )

            retrieved_at = datetime.now(
                timezone.utc
            )

            # Preserve original baseline before
            # overwriting current snapshot.
            _archive_baseline_if_needed(
                organization_id,
                current_raw_path,
                version_dir,
            )

            version_path = (
                _save_fetched_version(
                    organization_id,
                    new_response,
                    retrieved_at,
                    version_dir,
                )
            )

            if old_response is None:

                status = "new"

                change_info = {
                    "content_changed": True,
                    "changed_fields": [],
                    "old_content_hash": None,
                    "new_content_hash": None,
                }

            else:

                change_info = (
                    classify_record_change(
                        old_response["results"],
                        new_response["results"],
                    )
                )

                if change_info[
                    "content_changed"
                ]:
                    status = (
                        "meaningful_change"
                    )
                else:
                    status = (
                        "technical_refresh"
                    )

            # Current snapshot always becomes
            # the latest successfully fetched response.
            _write_json(
                current_raw_path,
                new_response,
            )

            results.append({
                "organization_id":
                    organization_id,

                "name":
                    new_response[
                        "results"
                    ].get("name"),

                "status":
                    status,

                "retrieved_at_utc":
                    retrieved_at.isoformat(),

                "version_path":
                    str(version_path),

                "changed_fields":
                    change_info.get(
                        "changed_fields",
                        []
                    ),

                "old_content_hash":
                    change_info.get(
                        "old_content_hash"
                    ),

                "new_content_hash":
                    change_info.get(
                        "new_content_hash"
                    ),
            })

        except Exception as exc:

            failures.append({
                "organization_id":
                    organization_id,

                "error":
                    str(exc),
            })

    # --------------------------------
    # COMMIT ONLY AFTER SUCCESS
    # --------------------------------

    committed = False

    if not failures:

        save_committed_manifest(
            current_manifest
        )

        committed = True

    run_finished = datetime.now(
        timezone.utc
    )

    status_counts = {}

    for item in results:
        status = item["status"]

        status_counts[status] = (
            status_counts.get(
                status, 0
            ) + 1
        )

    run_log = {
        "run_started_at_utc":
            run_started.isoformat(),

        "run_finished_at_utc":
            run_finished.isoformat(),

        "committed":
            committed,

        "discovery_snapshot":
            str(discovery_snapshot),

        "discovery":
            discovery_summary,

        "fetch_candidates":
            len(candidate_ids),

        "status_counts":
            status_counts,

        "disappeared_ids":
            disappeared_orgs[
                "organization_id"
            ].tolist(),

        "index_changed_ids":
            (
                index_changed_orgs[
                    "organization_id"
                ].tolist()
                if not index_changed_orgs.empty
                else []
            ),

        "results":
            results,

        "failures":
            failures,
    }

    log_timestamp = (
        run_started.strftime(
            "%Y-%m-%d_%H-%M-%S_UTC"
        )
    )

    log_path = (
        log_dir
        / f"sync_{log_timestamp}.json"
    )

    _write_json(
        log_path,
        run_log,
    )

    return run_log

Writing src/politdata/sync.py


In [3]:
sync_result = run_organization_sync()

print(
    "Committed:",
    sync_result["committed"]
)

print(
    "Discovery:",
    sync_result["discovery"]
)

print(
    "Fetch candidates:",
    sync_result["fetch_candidates"]
)

print(
    "Status counts:",
    sync_result["status_counts"]
)

print(
    "Failures:",
    len(sync_result["failures"])
)

Committed: True
Discovery: {'current': 9917, 'previous': 9917, 'new': 0, 'disappeared': 0, 'existing': 9917, 'index_changed': 0, 'refresh_candidates': 1, 'index_compared_columns': ['root_party_id', 'parent_id', 'entity_type', 'code', 'name', 'is_active'], 'refresh_trigger_columns': ['source_updated_at']}
Fetch candidates: 1
Status counts: {'technical_refresh': 1}
Failures: 0


In [4]:
sync_result_2 = run_organization_sync()

print(
    "Committed:",
    sync_result_2["committed"]
)

print(
    "Discovery:",
    sync_result_2["discovery"]
)

print(
    "Fetch candidates:",
    sync_result_2["fetch_candidates"]
)

print(
    "Status counts:",
    sync_result_2["status_counts"]
)

print(
    "Failures:",
    len(sync_result_2["failures"])
)

Committed: True
Discovery: {'current': 9917, 'previous': 9917, 'new': 0, 'disappeared': 0, 'existing': 9917, 'index_changed': 0, 'refresh_candidates': 1, 'index_compared_columns': ['root_party_id', 'parent_id', 'entity_type', 'code', 'name', 'is_active'], 'refresh_trigger_columns': ['source_updated_at']}
Fetch candidates: 1
Status counts: {'technical_refresh': 1}
Failures: 0


In [5]:
%%writefile src/politdata/sync.py

from datetime import datetime, timezone
from pathlib import Path
import json
import shutil

import pandas as pd

from .api import (
    fetch_all_parties,
    fetch_party_account,
)

from .discovery import (
    build_organization_manifest,
    compare_manifests,
    save_discovery_snapshot,
    save_committed_manifest,
)

from .change_detection import (
    classify_record_change,
    organization_content_hash,
)


DEFAULT_COMMITTED_MANIFEST = Path(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

DEFAULT_CURRENT_RAW_DIR = Path(
    "data/raw/party_accounts"
)

DEFAULT_VERSION_DIR = Path(
    "data/raw/party_account_versions"
)

DEFAULT_LOG_DIR = Path(
    "logs/sync_runs"
)


def _write_json(path, data):
    """
    Safely write JSON through a temporary file.
    """

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temp_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

    temp_path.replace(path)


def _read_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def _archive_baseline_if_needed(
    organization_id,
    current_raw_path,
    version_dir,
):
    """
    Preserve the original RAW file from before
    version tracking was introduced.

    This happens only once per organization.
    """

    current_raw_path = Path(
        current_raw_path
    )

    if not current_raw_path.exists():
        return None

    org_version_dir = (
        Path(version_dir)
        / organization_id
    )

    org_version_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    baseline_path = (
        org_version_dir
        / "baseline_initial.json"
    )

    if not baseline_path.exists():
        shutil.copy2(
            current_raw_path,
            baseline_path,
        )

    return baseline_path


def _save_fetched_version(
    organization_id,
    api_response,
    retrieved_at,
    version_dir,
):
    """
    Save a historical RAW version.

    This should be used only for:
    - new organizations
    - meaningful content changes
    """

    org_version_dir = (
        Path(version_dir)
        / organization_id
    )

    org_version_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    timestamp = retrieved_at.strftime(
        "%Y-%m-%d_%H-%M-%S_%f_UTC"
    )

    version_path = (
        org_version_dir
        / f"{timestamp}.json"
    )

    _write_json(
        version_path,
        api_response,
    )

    return version_path


def run_organization_sync(
    committed_manifest_path=DEFAULT_COMMITTED_MANIFEST,
    current_raw_dir=DEFAULT_CURRENT_RAW_DIR,
    version_dir=DEFAULT_VERSION_DIR,
    log_dir=DEFAULT_LOG_DIR,
):
    """
    Run one incremental PolitData organization sync.

    Flow:
    1. Rebuild discovery manifest from /parties.
    2. Compare against committed manifest.
    3. Fetch new or potentially changed organizations.
    4. Compare full content.
    5. Store historical versions only for meaningful changes.
    6. Update current RAW snapshot.
    7. Commit manifest only if all required fetches succeed.
    """

    run_started = datetime.now(
        timezone.utc
    )

    committed_manifest_path = Path(
        committed_manifest_path
    )

    current_raw_dir = Path(
        current_raw_dir
    )

    version_dir = Path(
        version_dir
    )

    log_dir = Path(
        log_dir
    )

    log_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not committed_manifest_path.exists():
        raise FileNotFoundError(
            "Committed manifest not found: "
            f"{committed_manifest_path}"
        )

    previous_manifest = pd.read_parquet(
        committed_manifest_path
    )

    # -------------------------------------------------
    # DISCOVERY
    # -------------------------------------------------

    fresh_parties = fetch_all_parties()

    current_manifest = (
        build_organization_manifest(
            fresh_parties
        )
    )

    discovery_snapshot = (
        save_discovery_snapshot(
            current_manifest
        )
    )

    (
        discovery_summary,
        new_orgs,
        disappeared_orgs,
        index_changed_orgs,
        refresh_candidates,
    ) = compare_manifests(
        current_manifest,
        previous_manifest,
    )

    candidate_ids = set()

    if not new_orgs.empty:
        candidate_ids.update(
            new_orgs[
                "organization_id"
            ].tolist()
        )

    if not refresh_candidates.empty:
        candidate_ids.update(
            refresh_candidates[
                "organization_id"
            ].tolist()
        )

    candidate_ids = sorted(
        candidate_ids
    )

    # -------------------------------------------------
    # FETCH + CONTENT COMPARISON
    # -------------------------------------------------

    results = []
    failures = []

    for organization_id in candidate_ids:

        current_raw_path = (
            current_raw_dir
            / f"{organization_id}.json"
        )

        try:

            old_response = None

            if current_raw_path.exists():
                old_response = _read_json(
                    current_raw_path
                )

            new_response = (
                fetch_party_account(
                    organization_id
                )
            )

            retrieved_at = datetime.now(
                timezone.utc
            )

            new_detail = new_response[
                "results"
            ]

            old_detail = (
                old_response["results"]
                if old_response is not None
                else None
            )

            old_updated_at = (
                old_detail.get("updated_at")
                if old_detail is not None
                else None
            )

            new_updated_at = (
                new_detail.get("updated_at")
            )

            # Preserve our pre-versioning baseline once.
            if old_response is not None:
                _archive_baseline_if_needed(
                    organization_id,
                    current_raw_path,
                    version_dir,
                )

            # -----------------------------------------
            # CLASSIFY
            # -----------------------------------------

            if old_detail is None:

                status = "new"

                old_content_hash = None

                new_content_hash = (
                    organization_content_hash(
                        new_detail
                    )
                )

                changed_fields = []

            else:

                change_info = (
                    classify_record_change(
                        old_detail,
                        new_detail,
                    )
                )

                old_content_hash = (
                    change_info[
                        "old_content_hash"
                    ]
                )

                new_content_hash = (
                    change_info[
                        "new_content_hash"
                    ]
                )

                changed_fields = (
                    change_info[
                        "changed_fields"
                    ]
                )

                if change_info[
                    "content_changed"
                ]:
                    status = (
                        "meaningful_change"
                    )
                else:
                    status = (
                        "technical_refresh"
                    )

            # -----------------------------------------
            # VERSION HISTORY
            # -----------------------------------------

            version_path = None

            if status in {
                "new",
                "meaningful_change",
            }:

                version_path = (
                    _save_fetched_version(
                        organization_id,
                        new_response,
                        retrieved_at,
                        version_dir,
                    )
                )

            # technical_refresh is NOT saved
            # as a separate historical content version.

            # -----------------------------------------
            # CURRENT RAW SNAPSHOT
            # -----------------------------------------

            # Always retain the latest actual API response,
            # including the newest updated_at value.
            _write_json(
                current_raw_path,
                new_response,
            )

            results.append({
                "organization_id":
                    organization_id,

                "name":
                    new_detail.get("name"),

                "status":
                    status,

                "retrieved_at_utc":
                    retrieved_at.isoformat(),

                "old_updated_at":
                    old_updated_at,

                "new_updated_at":
                    new_updated_at,

                "version_path":
                    (
                        str(version_path)
                        if version_path
                        else None
                    ),

                "changed_fields":
                    changed_fields,

                "old_content_hash":
                    old_content_hash,

                "new_content_hash":
                    new_content_hash,
            })

        except Exception as exc:

            failures.append({
                "organization_id":
                    organization_id,

                "error":
                    repr(exc),
            })

    # -------------------------------------------------
    # COMMIT
    # -------------------------------------------------

    committed = False

    if not failures:

        save_committed_manifest(
            current_manifest
        )

        committed = True

    # -------------------------------------------------
    # RUN LOG
    # -------------------------------------------------

    run_finished = datetime.now(
        timezone.utc
    )

    status_counts = {}

    for item in results:

        status = item["status"]

        status_counts[status] = (
            status_counts.get(
                status,
                0,
            )
            + 1
        )

    run_log = {
        "run_started_at_utc":
            run_started.isoformat(),

        "run_finished_at_utc":
            run_finished.isoformat(),

        "committed":
            committed,

        "discovery_snapshot":
            str(discovery_snapshot),

        "discovery":
            discovery_summary,

        "fetch_candidates":
            len(candidate_ids),

        "status_counts":
            status_counts,

        "disappeared_ids":
            (
                disappeared_orgs[
                    "organization_id"
                ].tolist()
                if not disappeared_orgs.empty
                else []
            ),

        "index_changed_ids":
            (
                index_changed_orgs[
                    "organization_id"
                ].tolist()
                if not index_changed_orgs.empty
                else []
            ),

        "results":
            results,

        "failures":
            failures,
    }

    log_timestamp = (
        run_started.strftime(
            "%Y-%m-%d_%H-%M-%S_UTC"
        )
    )

    log_path = (
        log_dir
        / f"sync_{log_timestamp}.json"
    )

    _write_json(
        log_path,
        run_log,
    )

    return run_log

Overwriting src/politdata/sync.py


In [2]:
sync_result_3 = run_organization_sync()

print(
    "Committed:",
    sync_result_3["committed"]
)

print(
    "Discovery:",
    sync_result_3["discovery"]
)

print(
    "Fetch candidates:",
    sync_result_3["fetch_candidates"]
)

print(
    "Status counts:",
    sync_result_3["status_counts"]
)

print(
    "Failures:",
    len(sync_result_3["failures"])
)

Committed: True
Discovery: {'current': 9917, 'previous': 9917, 'new': 0, 'disappeared': 0, 'existing': 9917, 'index_changed': 0, 'refresh_candidates': 1, 'index_compared_columns': ['root_party_id', 'parent_id', 'entity_type', 'code', 'name', 'is_active'], 'refresh_trigger_columns': ['source_updated_at']}
Fetch candidates: 1
Status counts: {'technical_refresh': 1}
Failures: 0


In [3]:
for item in sync_result_3["results"]:
    print()
    print("Organization:", item["name"])
    print("Status:", item["status"])
    print("OLD updated_at:", item["old_updated_at"])
    print("NEW updated_at:", item["new_updated_at"])
    print("Changed fields:", item["changed_fields"])
    print("Version path:", item["version_path"])


Organization: ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ УКРАЇНСЬКОГО НАРОДУ»
Status: technical_refresh
OLD updated_at: 2026-08-29 10:53:00.952348
NEW updated_at: 2026-08-29 10:56:00.635349
Changed fields: ['updated_at']
Version path: None


In [4]:
%%writefile src/politdata/refresh.py

from datetime import datetime, timezone, timedelta
from pathlib import Path
import json

import pandas as pd

from .change_detection import organization_content_hash


DEFAULT_REFRESH_STATE_PATH = Path(
    "data/interim/state/organization_refresh_state.parquet"
)

DEFAULT_RAW_DIR = Path(
    "data/raw/party_accounts"
)


def save_refresh_state(
    state,
    state_path=DEFAULT_REFRESH_STATE_PATH,
):
    """
    Safely save operational refresh state.
    """

    state_path = Path(state_path)

    state_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = state_path.with_suffix(
        ".tmp.parquet"
    )

    state.to_parquet(
        temp_path,
        index=False,
    )

    temp_path.replace(state_path)

    return state_path


def initialize_refresh_state(
    manifest,
    raw_dir=DEFAULT_RAW_DIR,
    state_path=DEFAULT_REFRESH_STATE_PATH,
    overwrite=False,
):
    """
    Bootstrap refresh state from existing RAW files.

    File modification time is used only as an initial
    approximation of when the organization was last fetched.
    After bootstrap, the pipeline will maintain explicit
    last_checked_at_utc values itself.
    """

    state_path = Path(state_path)
    raw_dir = Path(raw_dir)

    if state_path.exists() and not overwrite:
        return pd.read_parquet(
            state_path
        )

    rows = []

    for row in manifest.itertuples(
        index=False
    ):
        organization_id = (
            row.organization_id
        )

        raw_path = (
            raw_dir
            / f"{organization_id}.json"
        )

        last_checked_at_utc = None
        content_hash = None

        if raw_path.exists():

            modified_at = datetime.fromtimestamp(
                raw_path.stat().st_mtime,
                tz=timezone.utc,
            )

            last_checked_at_utc = (
                modified_at.isoformat()
            )

            try:
                with raw_path.open(
                    "r",
                    encoding="utf-8",
                ) as f:
                    api_response = json.load(f)

                detail = api_response.get(
                    "results"
                )

                if detail is not None:
                    content_hash = (
                        organization_content_hash(
                            detail
                        )
                    )

            except Exception:
                content_hash = None

        rows.append({
            "organization_id":
                organization_id,

            "entity_type":
                row.entity_type,

            "last_checked_at_utc":
                last_checked_at_utc,

            "last_status":
                "bootstrap"
                if raw_path.exists()
                else None,

            "last_content_hash":
                content_hash,

            "last_meaningful_change_at_utc":
                None,
        })

    state = pd.DataFrame(rows)

    save_refresh_state(
        state,
        state_path=state_path,
    )

    return state


def select_rolling_refresh_candidates(
    manifest,
    state,
    refresh_interval_days=7,
    limit=1400,
    entity_type="office",
    now_utc=None,
):
    """
    Select the stalest organizations that are due
    for a periodic full-card refresh.

    Missing last_checked values are considered most urgent.
    """

    if now_utc is None:
        now_utc = datetime.now(
            timezone.utc
        )

    cutoff = (
        now_utc
        - timedelta(
            days=refresh_interval_days
        )
    )

    state_small = state[
        [
            "organization_id",
            "last_checked_at_utc",
        ]
    ].copy()

    merged = manifest.merge(
        state_small,
        on="organization_id",
        how="left",
    )

    candidates = merged[
        merged["entity_type"]
        == entity_type
    ].copy()

    candidates[
        "last_checked_parsed"
    ] = pd.to_datetime(
        candidates[
            "last_checked_at_utc"
        ],
        utc=True,
        errors="coerce",
    )

    due_mask = (
        candidates[
            "last_checked_parsed"
        ].isna()
        |
        (
            candidates[
                "last_checked_parsed"
            ]
            <= cutoff
        )
    )

    candidates = candidates[
        due_mask
    ].copy()

    candidates[
        "_missing_check"
    ] = candidates[
        "last_checked_parsed"
    ].isna()

    candidates = candidates.sort_values(
        by=[
            "_missing_check",
            "last_checked_parsed",
            "organization_id",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )

    if limit is not None:
        candidates = candidates.head(
            limit
        )

    return candidates.drop(
        columns=[
            "_missing_check",
            "last_checked_parsed",
        ]
    ).reset_index(
        drop=True
    )


def update_refresh_state(
    state,
    updates,
    state_path=DEFAULT_REFRESH_STATE_PATH,
):
    """
    Apply successful refresh results to operational state.

    Failed fetches should not be passed here, so they
    remain due for a later retry.
    """

    state = state.copy()

    state = state.set_index(
        "organization_id"
    )

    for update in updates:

        organization_id = update[
            "organization_id"
        ]

        if organization_id not in state.index:
            state.loc[
                organization_id,
                "entity_type",
            ] = update.get(
                "entity_type"
            )

        state.loc[
            organization_id,
            "last_checked_at_utc",
        ] = update.get(
            "last_checked_at_utc"
        )

        state.loc[
            organization_id,
            "last_status",
        ] = update.get(
            "last_status"
        )

        state.loc[
            organization_id,
            "last_content_hash",
        ] = update.get(
            "last_content_hash"
        )

        meaningful_at = update.get(
            "last_meaningful_change_at_utc"
        )

        if meaningful_at is not None:
            state.loc[
                organization_id,
                "last_meaningful_change_at_utc",
            ] = meaningful_at

    state = state.reset_index()

    save_refresh_state(
        state,
        state_path=state_path,
    )

    return state

Writing src/politdata/refresh.py


In [3]:
committed_manifest = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

refresh_state = initialize_refresh_state(
    committed_manifest
)

print(
    "Refresh state rows:",
    len(refresh_state)
)

print(
    refresh_state[
        "entity_type"
    ].value_counts()
)

print(
    "Missing last_checked:",
    refresh_state[
        "last_checked_at_utc"
    ].isna().sum()
)

print(
    "Missing content hash:",
    refresh_state[
        "last_content_hash"
    ].isna().sum()
)

Refresh state rows: 9917
entity_type
office    9596
party      321
Name: count, dtype: int64
Missing last_checked: 0
Missing content hash: 0


In [4]:
due_offices = (
    select_rolling_refresh_candidates(
        committed_manifest,
        refresh_state,
        refresh_interval_days=7,
        limit=1400,
    )
)

print(
    "Offices due now:",
    len(due_offices)
)

Offices due now: 0


In [5]:
test_rolling = (
    select_rolling_refresh_candidates(
        committed_manifest,
        refresh_state,
        refresh_interval_days=0,
        limit=20,
    )
)

print(
    "Test rolling candidates:",
    len(test_rolling)
)

display(
    test_rolling[
        [
            "organization_id",
            "root_party_id",
            "code",
            "name",
            "last_checked_at_utc",
        ]
    ]
)

Test rolling candidates: 20


,organization_id,root_party_id,code,name,last_checked_at_utc
0,1cf95166-a8c5-4363-a268-36ab0d276b87,5966f24e-8b67-428d-817f-9b48eb335077,43754413,БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,2026-08-28T14:59:01.177680+00:00
1,fe8d5200-4709-4715-b316-6535ad348ec5,9df5225a-d90d-4ce6-8bd3-bfc9f35cf320,36806991,Київська обласна організація політичної партії...,2026-08-28T14:59:02.010419+00:00
2,c53485c4-fb53-423f-a978-a96b32f7436a,adc9e573-86f9-4ec3-8331-70c99ec46699,43764332,КИЇВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ...,2026-08-28T14:59:02.432868+00:00
3,f90ec33b-a925-4435-985c-d55e86d3b64b,f0398186-80ea-4b4f-8e6b-d9890cb51b2c,36821737,"КРИМСЬКА РЕСПУБЛІКАНСЬКА ОРГАНІЗАЦІЯ ПАРТІЇ ""Р...",2026-08-28T14:59:02.870272+00:00
4,011c6969-8c97-4009-8fce-37e7b64576a4,f0398186-80ea-4b4f-8e6b-d9890cb51b2c,37010276,"РЕНІЙСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПАРТІЇ ""РОДИНА""",2026-08-28T14:59:03.080432+00:00
5,942a39f5-97ac-4445-8150-9a4fb1b5e83c,f0398186-80ea-4b4f-8e6b-d9890cb51b2c,36501338,ПОЛІТИЧНА ПАРТІЯ ОДЕСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПА...,2026-08-28T14:59:03.292824+00:00
6,de5de4e3-3510-42bf-b1f7-73964a681743,f0398186-80ea-4b4f-8e6b-d9890cb51b2c,36714340,"ТАТАРБУНАРСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПАРТІЇ ""РОД...",2026-08-28T14:59:03.500185+00:00
7,64ecdb81-ec90-4419-81e5-6fb4f98912fd,f0398186-80ea-4b4f-8e6b-d9890cb51b2c,37246919,СЕВАСТОПОЛЬСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ ПА...,2026-08-28T14:59:03.702271+00:00
8,b18cbcd0-f14b-4790-b91a-f9bc5fc33c70,f0398186-80ea-4b4f-8e6b-d9890cb51b2c,36343882,"ОДЕСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПАРТІЇ ""РОДИНА""",2026-08-28T14:59:03.910472+00:00
9,6e0c8b9c-8427-4664-83c9-273b80962cba,f0398186-80ea-4b4f-8e6b-d9890cb51b2c,36562652,ЛЬВІВСЬКА ОБЛАСНА ПАРТОРГАНІЗАЦІЯ ПРОГРЕСИВНО-...,2026-08-28T14:59:04.114498+00:00


In [6]:
%%writefile src/politdata/sync.py

from datetime import datetime, timezone
from pathlib import Path
import json
import shutil

import pandas as pd

from .api import (
    fetch_all_parties,
    fetch_party_account,
)

from .discovery import (
    build_organization_manifest,
    compare_manifests,
    save_discovery_snapshot,
    save_committed_manifest,
)

from .change_detection import (
    classify_record_change,
    organization_content_hash,
)

from .refresh import (
    DEFAULT_REFRESH_STATE_PATH,
    initialize_refresh_state,
    select_rolling_refresh_candidates,
    update_refresh_state,
)


DEFAULT_COMMITTED_MANIFEST = Path(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

DEFAULT_CURRENT_RAW_DIR = Path(
    "data/raw/party_accounts"
)

DEFAULT_VERSION_DIR = Path(
    "data/raw/party_account_versions"
)

DEFAULT_LOG_DIR = Path(
    "logs/sync_runs"
)


def _write_json(path, data):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temp_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

    temp_path.replace(path)


def _read_json(path):
    with Path(path).open(
        "r",
        encoding="utf-8",
    ) as f:
        return json.load(f)


def _archive_baseline_if_needed(
    organization_id,
    current_raw_path,
    version_dir,
):
    """
    Preserve the pre-versioning RAW state once.
    """

    current_raw_path = Path(
        current_raw_path
    )

    if not current_raw_path.exists():
        return None

    org_version_dir = (
        Path(version_dir)
        / organization_id
    )

    org_version_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    baseline_path = (
        org_version_dir
        / "baseline_initial.json"
    )

    if not baseline_path.exists():
        shutil.copy2(
            current_raw_path,
            baseline_path,
        )

    return baseline_path


def _save_fetched_version(
    organization_id,
    api_response,
    retrieved_at,
    version_dir,
):
    """
    Save a historical version only when content
    is new or meaningfully changed.
    """

    org_version_dir = (
        Path(version_dir)
        / organization_id
    )

    org_version_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    timestamp = retrieved_at.strftime(
        "%Y-%m-%d_%H-%M-%S_%f_UTC"
    )

    version_path = (
        org_version_dir
        / f"{timestamp}.json"
    )

    _write_json(
        version_path,
        api_response,
    )

    return version_path


def run_organization_sync(
    committed_manifest_path=DEFAULT_COMMITTED_MANIFEST,
    current_raw_dir=DEFAULT_CURRENT_RAW_DIR,
    version_dir=DEFAULT_VERSION_DIR,
    log_dir=DEFAULT_LOG_DIR,
    refresh_state_path=DEFAULT_REFRESH_STATE_PATH,
    enable_rolling_refresh=True,
    rolling_refresh_interval_days=7,
    rolling_refresh_limit=1400,
):
    """
    Run one organization synchronization cycle.

    Discovery candidates:
    - new organizations
    - index changes
    - source_updated_at changes

    Rolling candidates:
    - offices whose full cards have not been checked
      within rolling_refresh_interval_days

    Historical versions are saved only for:
    - new organizations
    - meaningful content changes

    A failure of a discovery-critical fetch blocks
    committing the new manifest.

    A failure of a rolling-only fetch does NOT block
    committing the manifest; that organization remains
    due for another rolling refresh.
    """

    run_started = datetime.now(
        timezone.utc
    )

    committed_manifest_path = Path(
        committed_manifest_path
    )

    current_raw_dir = Path(
        current_raw_dir
    )

    version_dir = Path(
        version_dir
    )

    log_dir = Path(
        log_dir
    )

    log_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if not committed_manifest_path.exists():
        raise FileNotFoundError(
            "Committed manifest not found: "
            f"{committed_manifest_path}"
        )

    previous_manifest = pd.read_parquet(
        committed_manifest_path
    )

    # -------------------------------------------------
    # DISCOVERY
    # -------------------------------------------------

    fresh_parties = fetch_all_parties()

    current_manifest = (
        build_organization_manifest(
            fresh_parties
        )
    )

    discovery_snapshot = (
        save_discovery_snapshot(
            current_manifest
        )
    )

    (
        discovery_summary,
        new_orgs,
        disappeared_orgs,
        index_changed_orgs,
        refresh_candidates,
    ) = compare_manifests(
        current_manifest,
        previous_manifest,
    )

    # -------------------------------------------------
    # REFRESH STATE
    # -------------------------------------------------

    refresh_state = initialize_refresh_state(
        current_manifest,
        raw_dir=current_raw_dir,
        state_path=refresh_state_path,
    )

    if enable_rolling_refresh:
        rolling_candidates = (
            select_rolling_refresh_candidates(
                current_manifest,
                refresh_state,
                refresh_interval_days=(
                    rolling_refresh_interval_days
                ),
                limit=rolling_refresh_limit,
                entity_type="office",
            )
        )
    else:
        rolling_candidates = pd.DataFrame()

    # -------------------------------------------------
    # BUILD CANDIDATE SET + REASONS
    # -------------------------------------------------

    candidate_reasons = {}

    def add_reason(
        organization_ids,
        reason,
    ):
        for organization_id in organization_ids:
            candidate_reasons.setdefault(
                organization_id,
                set(),
            ).add(reason)

    if not new_orgs.empty:
        add_reason(
            new_orgs[
                "organization_id"
            ].tolist(),
            "new",
        )

    if not index_changed_orgs.empty:
        add_reason(
            index_changed_orgs[
                "organization_id"
            ].tolist(),
            "index_changed",
        )

    if not refresh_candidates.empty:
        add_reason(
            refresh_candidates[
                "organization_id"
            ].tolist(),
            "discovery_refresh",
        )

    if not rolling_candidates.empty:
        add_reason(
            rolling_candidates[
                "organization_id"
            ].tolist(),
            "rolling_refresh",
        )

    candidate_ids = sorted(
        candidate_reasons.keys()
    )

    # These fetches must succeed before
    # discovery may be committed.
    critical_candidate_ids = set()

    if not new_orgs.empty:
        critical_candidate_ids.update(
            new_orgs[
                "organization_id"
            ].tolist()
        )

    if not refresh_candidates.empty:
        critical_candidate_ids.update(
            refresh_candidates[
                "organization_id"
            ].tolist()
        )

    # Used for entity_type lookup.
    manifest_indexed = (
        current_manifest.set_index(
            "organization_id"
        )
    )

    # -------------------------------------------------
    # FETCH + COMPARE
    # -------------------------------------------------

    results = []
    failures = []
    refresh_state_updates = []

    for organization_id in candidate_ids:

        reasons = sorted(
            candidate_reasons[
                organization_id
            ]
        )

        is_critical = (
            organization_id
            in critical_candidate_ids
        )

        current_raw_path = (
            current_raw_dir
            / f"{organization_id}.json"
        )

        try:

            old_response = None

            if current_raw_path.exists():
                old_response = _read_json(
                    current_raw_path
                )

            new_response = (
                fetch_party_account(
                    organization_id
                )
            )

            retrieved_at = datetime.now(
                timezone.utc
            )

            new_detail = (
                new_response["results"]
            )

            old_detail = (
                old_response["results"]
                if old_response is not None
                else None
            )

            old_updated_at = (
                old_detail.get("updated_at")
                if old_detail is not None
                else None
            )

            new_updated_at = (
                new_detail.get(
                    "updated_at"
                )
            )

            if old_response is not None:
                _archive_baseline_if_needed(
                    organization_id,
                    current_raw_path,
                    version_dir,
                )

            # -----------------------------------------
            # CLASSIFY CONTENT
            # -----------------------------------------

            if old_detail is None:

                status = "new"

                old_content_hash = None

                new_content_hash = (
                    organization_content_hash(
                        new_detail
                    )
                )

                changed_fields = []

            else:

                change_info = (
                    classify_record_change(
                        old_detail,
                        new_detail,
                    )
                )

                old_content_hash = (
                    change_info[
                        "old_content_hash"
                    ]
                )

                new_content_hash = (
                    change_info[
                        "new_content_hash"
                    ]
                )

                changed_fields = (
                    change_info[
                        "changed_fields"
                    ]
                )

                if change_info[
                    "content_changed"
                ]:
                    status = (
                        "meaningful_change"
                    )

                elif changed_fields:
                    status = (
                        "technical_refresh"
                    )

                else:
                    status = (
                        "unchanged"
                    )

            # -----------------------------------------
            # HISTORICAL VERSION
            # -----------------------------------------

            version_path = None

            if status in {
                "new",
                "meaningful_change",
            }:

                version_path = (
                    _save_fetched_version(
                        organization_id,
                        new_response,
                        retrieved_at,
                        version_dir,
                    )
                )

            # -----------------------------------------
            # CURRENT RAW
            # -----------------------------------------

            _write_json(
                current_raw_path,
                new_response,
            )

            entity_type = (
                manifest_indexed.at[
                    organization_id,
                    "entity_type",
                ]
            )

            meaningful_change_at = None

            if status == "meaningful_change":
                meaningful_change_at = (
                    retrieved_at.isoformat()
                )

            refresh_state_updates.append({
                "organization_id":
                    organization_id,

                "entity_type":
                    entity_type,

                "last_checked_at_utc":
                    retrieved_at.isoformat(),

                "last_status":
                    status,

                "last_content_hash":
                    new_content_hash,

                "last_meaningful_change_at_utc":
                    meaningful_change_at,
            })

            results.append({
                "organization_id":
                    organization_id,

                "entity_type":
                    entity_type,

                "name":
                    new_detail.get("name"),

                "candidate_reasons":
                    reasons,

                "critical":
                    is_critical,

                "status":
                    status,

                "retrieved_at_utc":
                    retrieved_at.isoformat(),

                "old_updated_at":
                    old_updated_at,

                "new_updated_at":
                    new_updated_at,

                "version_path":
                    (
                        str(version_path)
                        if version_path
                        else None
                    ),

                "changed_fields":
                    changed_fields,

                "old_content_hash":
                    old_content_hash,

                "new_content_hash":
                    new_content_hash,
            })

        except Exception as exc:

            failures.append({
                "organization_id":
                    organization_id,

                "candidate_reasons":
                    reasons,

                "critical":
                    is_critical,

                "error":
                    repr(exc),
            })

    # -------------------------------------------------
    # UPDATE REFRESH STATE FOR SUCCESSFUL FETCHES
    # -------------------------------------------------

    if refresh_state_updates:

        refresh_state = (
            update_refresh_state(
                refresh_state,
                refresh_state_updates,
                state_path=refresh_state_path,
            )
        )

    # -------------------------------------------------
    # COMMIT DISCOVERY
    # -------------------------------------------------

    critical_failures = [
        failure
        for failure in failures
        if failure["critical"]
    ]

    rolling_only_failures = [
        failure
        for failure in failures
        if not failure["critical"]
    ]

    committed = False

    if not critical_failures:

        save_committed_manifest(
            current_manifest
        )

        committed = True

    # -------------------------------------------------
    # RUN SUMMARY
    # -------------------------------------------------

    run_finished = datetime.now(
        timezone.utc
    )

    status_counts = {}

    for item in results:

        status = item["status"]

        status_counts[status] = (
            status_counts.get(
                status,
                0,
            )
            + 1
        )

    run_log = {
        "run_started_at_utc":
            run_started.isoformat(),

        "run_finished_at_utc":
            run_finished.isoformat(),

        "committed":
            committed,

        "discovery_snapshot":
            str(discovery_snapshot),

        "discovery":
            discovery_summary,

        "rolling_refresh": {
            "enabled":
                enable_rolling_refresh,

            "interval_days":
                rolling_refresh_interval_days,

            "limit":
                rolling_refresh_limit,

            "selected":
                len(
                    rolling_candidates
                ),
        },

        "fetch_candidates":
            len(candidate_ids),

        "critical_candidates":
            len(
                critical_candidate_ids
            ),

        "status_counts":
            status_counts,

        "disappeared_ids":
            (
                disappeared_orgs[
                    "organization_id"
                ].tolist()
                if not disappeared_orgs.empty
                else []
            ),

        "index_changed_ids":
            (
                index_changed_orgs[
                    "organization_id"
                ].tolist()
                if not index_changed_orgs.empty
                else []
            ),

        "results":
            results,

        "failures":
            failures,

        "critical_failures":
            critical_failures,

        "rolling_only_failures":
            rolling_only_failures,
    }

    log_timestamp = (
        run_started.strftime(
            "%Y-%m-%d_%H-%M-%S_UTC"
        )
    )

    log_path = (
        log_dir
        / f"sync_{log_timestamp}.json"
    )

    _write_json(
        log_path,
        run_log,
    )

    return run_log

Overwriting src/politdata/sync.py


In [2]:
sync_rolling_test = run_organization_sync(
    rolling_refresh_interval_days=0,
    rolling_refresh_limit=20,
)

print(
    "Committed:",
    sync_rolling_test["committed"]
)

print(
    "Discovery:",
    sync_rolling_test["discovery"]
)

print(
    "Rolling selected:",
    sync_rolling_test[
        "rolling_refresh"
    ]["selected"]
)

print(
    "Fetch candidates:",
    sync_rolling_test[
        "fetch_candidates"
    ]
)

print(
    "Status counts:",
    sync_rolling_test[
        "status_counts"
    ]
)

print(
    "Failures:",
    len(
        sync_rolling_test[
            "failures"
        ]
    )
)

print(
    "Critical failures:",
    len(
        sync_rolling_test[
            "critical_failures"
        ]
    )
)

Committed: True
Discovery: {'current': 9917, 'previous': 9917, 'new': 0, 'disappeared': 0, 'existing': 9917, 'index_changed': 0, 'refresh_candidates': 1, 'index_compared_columns': ['root_party_id', 'parent_id', 'entity_type', 'code', 'name', 'is_active'], 'refresh_trigger_columns': ['source_updated_at']}
Rolling selected: 20
Fetch candidates: 21
Status counts: {'unchanged': 20, 'technical_refresh': 1}
Failures: 0
Critical failures: 0


In [3]:
rolling_results = [
    item
    for item in sync_rolling_test["results"]
    if "rolling_refresh"
    in item["candidate_reasons"]
]

print(
    "Rolling results:",
    len(rolling_results)
)

for item in rolling_results:
    print(
        item["status"],
        "|",
        item["name"]
    )

Rolling results: 20
unchanged | РЕНІЙСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПАРТІЇ "РОДИНА"
unchanged | КИЇВСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "УКРАЇНЦІ"
unchanged | ЛУГАНСЬКА ОБЛАСНА КРАЙОВА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ВЕЛИКА УКРАЇНА"
unchanged | БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ "ФАКЕЛ"
unchanged | ЧЕРКАСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "УКРАЇНЦІ"
unchanged | ХАРКІВСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "УКРАЇНЦІ"
unchanged | ЖИТОМИРСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "НОВА ГЕНЕРАЦІЯ УКРАЇНИ"
unchanged | СЕВАСТОПОЛЬСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ ПАРТІЇ "РОДИНА"
unchanged | ЛЬВІВСЬКА ОБЛАСНА ПАРТОРГАНІЗАЦІЯ ПРОГРЕСИВНО-ДЕМОКРАТИЧНОЇ ПАРТІЇ УКРАЇНИ"
unchanged | КРАЙОВА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ВЕЛИКА УКРАЇНА" У МІСТІ КИЄВІ
unchanged | ВІННИЦЬКА ОБЛАСНА КРАЙОВА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ВЕЛИКА УКРАЇНА"
unchanged | ПОЛІТИЧНА ПАРТІЯ ОДЕСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПАРТІЇ "РОДИНА"
unchan

In [5]:
# Read the latest committed manifest from disk
committed_manifest_after = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

# Read the refresh state updated by the last sync
refresh_state_after = pd.read_parquet(
    "data/interim/state/"
    "organization_refresh_state.parquet"
)

# Select the next 20 stalest offices.
# interval_days=0 is ONLY for this test.
next_rolling = select_rolling_refresh_candidates(
    committed_manifest_after,
    refresh_state_after,
    refresh_interval_days=0,
    limit=20,
)

print(
    "Committed manifest rows:",
    len(committed_manifest_after)
)

print(
    "Refresh state rows:",
    len(refresh_state_after)
)

print(
    "Next rolling candidates:",
    len(next_rolling)
)

display(
    next_rolling[
        [
            "organization_id",
            "name",
            "last_checked_at_utc",
        ]
    ]
)

Committed manifest rows: 9917
Refresh state rows: 9917
Next rolling candidates: 20


,organization_id,name,last_checked_at_utc
0,2ab998da-786c-4009-93fb-2ee2b1e006e9,ЧЕРНІГІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПА...,2026-08-28T15:00:06.278340+00:00
1,d0e840e8-0f09-4a15-bd6f-b04abe49926d,ДНІПРОПЕТРОВСЬКА ОБЛАСНА РЕГІОНАЛЬНА ПАРТОРГАН...,2026-08-28T15:00:06.482524+00:00
2,9adced86-d052-4896-9a49-862f5b030e0f,ТЕРНОПІЛЬСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,2026-08-28T15:00:06.680960+00:00
3,2043aaa8-c985-4c63-b045-ffbe2087ece7,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,2026-08-28T15:00:06.884579+00:00
4,41720afd-582e-4824-b0e4-7bbcc91495b8,ЖИТОМИРСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАР...,2026-08-28T15:00:07.084799+00:00
5,2f5c7b56-d605-4d8e-8646-afbf3abbb908,ПОЛІТИЧНА ПАРТІЯ ІВАНО-ФРАНКІВСЬКА ОБЛАСНА ОРГ...,2026-08-28T15:00:07.294421+00:00
6,9f5e38ad-2ca1-4d55-9f4e-6342fedaa559,ЛЬВІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2026-08-28T15:00:07.501224+00:00
7,882256ea-b2a7-4a20-b7a9-459fc20041a6,КІРОВОГРАДСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ...,2026-08-28T15:00:07.714349+00:00
8,a650812f-bad7-4abe-a9d0-21e35b9032a1,ОДЕСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,2026-08-28T15:00:07.922193+00:00
9,7f9b5c93-41b7-422d-8229-20a2407d8852,РЕСПУБЛІКАНСЬКА В АВТОНОМНІЙ РЕСПУБЛІЦІ КРИМ О...,2026-08-28T15:00:08.128535+00:00


In [6]:
print(
    "Rolling selected:",
    sync_rolling_test["rolling_refresh"]["selected"]
)

print(
    "Status counts:",
    sync_rolling_test["status_counts"]
)

print(
    "Failures:",
    len(sync_rolling_test["failures"])
)

print(
    "Critical failures:",
    len(sync_rolling_test["critical_failures"])
)

print(
    "Rolling-only failures:",
    len(sync_rolling_test["rolling_only_failures"])
)

Rolling selected: 20
Status counts: {'unchanged': 20, 'technical_refresh': 1}
Failures: 0
Critical failures: 0
Rolling-only failures: 0


In [9]:
import requests
import json
import pandas as pd


# -----------------------------
# 1. Find test party
# -----------------------------

manifest_reports_test = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

golos_candidates = manifest_reports_test[
    (
        manifest_reports_test["entity_type"] == "party"
    )
    &
    (
        manifest_reports_test["name"]
        .str.contains(
            "ГОЛОС",
            case=False,
            na=False,
        )
    )
].copy()

display(
    golos_candidates[
        [
            "organization_id",
            "code",
            "name",
        ]
    ]
)

test_party_id = (
    golos_candidates.iloc[0][
        "organization_id"
    ]
)

test_party_name = (
    golos_candidates.iloc[0][
        "name"
    ]
)

print()
print("Testing party:")
print(test_party_name)
print(test_party_id)


# -----------------------------
# 2. POST /party/{id}/reports
# -----------------------------

url = (
    f"{DEFAULT_BASE_URL}/party/"
    f"{test_party_id}/reports"
)

# PolitData uses this DataTable-style payload
# for list endpoints.
payload = {
    "filters": {},
    "order": {},
    "pager": {
        "page": 1,
        "size": 100,
    },
}

response = requests.post(
    url,
    json=payload,
    timeout=30,
)

print()
print("HTTP status:", response.status_code)
print(
    "Content-Type:",
    response.headers.get("content-type")
)

# Do not raise yet:
# we want to see the API response even if it rejects the payload.
try:
    reports_probe = response.json()
except ValueError:
    reports_probe = response.text

print()
print("Response:")
print(
    json.dumps(
        reports_probe,
        ensure_ascii=False,
        indent=2,
    )[:10000]
    if isinstance(reports_probe, (dict, list))
    else reports_probe[:10000]
)

,organization_id,code,name
1341,b11839f3-bffe-4f93-ba10-3b877ed181b9,39651598,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»
4383,997341ae-5d03-4a1e-919a-b91d7bbafa08,40280750,"ПОЛІТИЧНА ПАРТІЯ ""НАШ ГОЛОСІЇВ"""



Testing party:
ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»
b11839f3-bffe-4f93-ba10-3b877ed181b9

HTTP status: 200
Content-Type: application/json; charset=UTF-8

Response:
{
  "results": {
    "list": [
      {
        "id": "e1b43970-54c5-11ef-8b5a-af0533ccdcca",
        "schema_version": 1,
        "report_type": "main",
        "year": 2024,
        "quarter": 2,
        "party_id": "b11839f3-bffe-4f93-ba10-3b877ed181b9",
        "is_party_office": false,
        "signed_date": "2024-08-07 17:25:44",
        "created_date": "2024-08-07 17:25:47",
        "signatory_id": null,
        "special_status": null,
        "public_summary": {
          "v": 2,
          "expenses": {
            "refunds_of_inflows": {
              "sum": 0,
              "row_count": 0
            },
            "payments_other_accounts": {
              "sum": 71137,
              "row_count": 24
            },
            "other_refunds_and_transfers": {
              "sum": 0,
              "row_count": 0
            },


In [10]:
# ---------------------------------
# PROFILE CENTRAL PARTY REPORT LIST
# ---------------------------------

results_obj = reports_probe["results"]
reports_list = results_obj.get("list", [])

print("Results keys:", list(results_obj.keys()))
print("Reports returned:", len(reports_list))

for key, value in results_obj.items():
    if key != "list":
        print(f"{key}: {value}")


# Flat table without public_summary
reports_df = pd.DataFrame(
    [
        {
            key: value
            for key, value in report.items()
            if key != "public_summary"
        }
        for report in reports_list
    ]
)

# Preserve original quarter exactly as PolitData provides it,
# but derive semantic period type.
reports_df["period_type"] = (
    reports_df["quarter"]
    .map({
        1: "quarterly",
        2: "quarterly",
        3: "quarterly",
        4: "quarterly",
        5: "annual",
    })
    .fillna("other")
)

print()
print("Report columns:")
print(reports_df.columns.tolist())

print()
print("Years:")
print(
    reports_df["year"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("Quarter values:")
print(
    reports_df["quarter"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("Period types:")
print(
    reports_df["period_type"]
    .value_counts(dropna=False)
)

print()
print("Report types:")
print(
    reports_df["report_type"]
    .value_counts(dropna=False)
)

print()
print("Schema versions:")
print(
    reports_df["schema_version"]
    .value_counts(dropna=False)
)

print()
print("is_party_office:")
print(
    reports_df["is_party_office"]
    .value_counts(dropna=False)
)

print()
print("Special statuses:")
print(
    reports_df["special_status"]
    .value_counts(dropna=False)
)


# ---------------------------------
# ORGANIZATION × YEAR PERIODICITY
# ---------------------------------

periodicity_by_year = (
    reports_df
    .groupby(
        ["party_id", "year"],
        dropna=False,
    )
    .agg(
        has_annual_report=(
            "period_type",
            lambda x: (x == "annual").any()
        ),
        has_quarterly_reports=(
            "period_type",
            lambda x: (x == "quarterly").any()
        ),
        annual_report_count=(
            "period_type",
            lambda x: (x == "annual").sum()
        ),
        quarterly_report_count=(
            "period_type",
            lambda x: (x == "quarterly").sum()
        ),
        total_report_count=(
            "id",
            "count"
        ),
    )
    .reset_index()
)

periodicity_by_year[
    "has_mixed_periodicity"
] = (
    periodicity_by_year[
        "has_annual_report"
    ]
    &
    periodicity_by_year[
        "has_quarterly_reports"
    ]
)

print()
print("Periodicity by year:")
display(periodicity_by_year)


print()
print("Mixed annual + quarterly years:")
display(
    periodicity_by_year[
        periodicity_by_year[
            "has_mixed_periodicity"
        ]
    ]
)


print()
print("All reports:")
display(
    reports_df.sort_values(
        [
            "year",
            "quarter",
            "created_date",
        ]
    )
)

Results keys: ['list', 'count']
Reports returned: 16
count: 16

Report columns:
['id', 'schema_version', 'report_type', 'year', 'quarter', 'party_id', 'is_party_office', 'signed_date', 'created_date', 'signatory_id', 'special_status', 'period_type']

Years:
year
2021    4
2022    1
2023    1
2024    4
2025    4
2026    2
Name: count, dtype: int64

Quarter values:
quarter
1    4
2    4
3    3
4    3
5    2
Name: count, dtype: int64

Period types:
period_type
quarterly    14
annual        2
Name: count, dtype: int64

Report types:
report_type
main    16
Name: count, dtype: int64

Schema versions:
schema_version
1    16
Name: count, dtype: int64

is_party_office:
is_party_office
False    16
Name: count, dtype: int64

Special statuses:
special_status
None    16
Name: count, dtype: int64

Periodicity by year:


,party_id,year,has_annual_report,has_quarterly_reports,annual_report_count,quarterly_report_count,total_report_count,has_mixed_periodicity
0,b11839f3-bffe-4f93-ba10-3b877ed181b9,2021,False,True,0,4,4,False
1,b11839f3-bffe-4f93-ba10-3b877ed181b9,2022,True,False,1,0,1,False
2,b11839f3-bffe-4f93-ba10-3b877ed181b9,2023,True,False,1,0,1,False
3,b11839f3-bffe-4f93-ba10-3b877ed181b9,2024,False,True,0,4,4,False
4,b11839f3-bffe-4f93-ba10-3b877ed181b9,2025,False,True,0,4,4,False
5,b11839f3-bffe-4f93-ba10-3b877ed181b9,2026,False,True,0,2,2,False



Mixed annual + quarterly years:


,party_id,year,has_annual_report,has_quarterly_reports,annual_report_count,quarterly_report_count,total_report_count,has_mixed_periodicity



All reports:


,id,schema_version,report_type,year,quarter,party_id,is_party_office,signed_date,created_date,signatory_id,special_status,period_type
4,2d9642f0-eab0-11ee-98d2-6d05851fad7e,1,main,2021,1,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-03-25 16:02:29,2024-03-25 16:02:31,None,None,quarterly
5,2394f660-eab1-11ee-96f1-37a2ca81244c,1,main,2021,2,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-03-25 16:09:12,2024-03-25 16:09:14,None,None,quarterly
6,7f7d6040-eab4-11ee-96f1-37a2ca81244c,1,main,2021,3,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-03-25 16:33:19,2024-03-25 16:33:21,None,None,quarterly
7,b9ad52d0-eab8-11ee-823a-73af3dbabf67,1,main,2021,4,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-03-25 17:03:43,2024-03-25 17:03:45,None,None,quarterly
8,c7f18e50-eab9-11ee-8a6c-27bc4724684c,1,main,2022,5,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-03-25 17:10:57,2024-03-25 17:10:59,None,None,annual
3,d83040d0-e876-11ee-a478-6126d3bbe2ef,1,main,2023,5,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-03-22 20:07:59,2024-03-22 20:08:01,None,None,annual
9,fe0ede20-0e04-11ef-823a-73af3dbabf67,1,main,2024,1,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-05-09 16:08:18,2024-05-09 16:08:21,None,None,quarterly
0,e1b43970-54c5-11ef-8b5a-af0533ccdcca,1,main,2024,2,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-08-07 17:25:44,2024-08-07 17:25:47,None,None,quarterly
1,248d0cc0-a031-11ef-8b5a-af0533ccdcca,1,main,2024,3,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2024-11-11 15:40:20,2024-11-11 15:40:29,None,None,quarterly
2,fb71f0e0-e7b5-11ef-82bf-3b31b6f46948,1,main,2024,4,b11839f3-bffe-4f93-ba10-3b877ed181b9,False,2025-02-10 15:57:45,2025-02-10 15:58:07,None,None,quarterly


In [11]:
# ---------------------------------
# PROBE ONE REGIONAL OFFICE
# ---------------------------------

golos_offices = manifest_reports_test[
    (
        manifest_reports_test["entity_type"]
        == "office"
    )
    &
    (
        manifest_reports_test["root_party_id"]
        == test_party_id
    )
].copy()

print(
    "Golos offices in manifest:",
    len(golos_offices)
)

test_office = golos_offices.iloc[0]

test_office_id = test_office[
    "organization_id"
]

print()
print("Testing office:")
print(test_office["name"])
print(test_office_id)


office_url = (
    f"{DEFAULT_BASE_URL}/party/"
    f"{test_office_id}/reports"
)

office_payload = {
    "filters": {},
    "order": {},
    "pager": {
        "page": 1,
        "size": 100,
    },
}

office_response = requests.post(
    office_url,
    json=office_payload,
    timeout=30,
)

print()
print(
    "Office HTTP status:",
    office_response.status_code
)

try:
    office_probe = office_response.json()
except ValueError:
    office_probe = office_response.text

if not isinstance(office_probe, dict):
    print(office_probe)

else:
    office_results = office_probe.get(
        "results",
        {}
    )

    office_reports = office_results.get(
        "list",
        []
    )

    print(
        "Office results keys:",
        list(office_results.keys())
    )

    print(
        "Office reports returned:",
        len(office_reports)
    )

    for key, value in office_results.items():
        if key != "list":
            print(f"{key}: {value}")

    if office_reports:
        print()
        print("First office report:")

        first_office_report = (
            office_reports[0]
        )

        for field in [
            "id",
            "schema_version",
            "report_type",
            "year",
            "quarter",
            "party_id",
            "is_party_office",
            "signed_date",
            "created_date",
            "special_status",
        ]:
            print(
                f"{field}:",
                first_office_report.get(field)
            )

Golos offices in manifest: 25

Testing office:
КИЇВСЬКА ОБЛАСНА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "ГОЛОС"
09d9a7c1-4391-4a40-82fe-b7c45c34f10f

Office HTTP status: 200
Office results keys: ['list', 'count']
Office reports returned: 16
count: 16

First office report:
id: 8ffda3c0-e76d-11ee-8847-316909bdf249
schema_version: 1
report_type: main
year: 2023
quarter: 5
party_id: 09d9a7c1-4391-4a40-82fe-b7c45c34f10f
is_party_office: True
signed_date: 2024-03-22 17:39:04
created_date: 2024-03-22 20:06:02
special_status: None


In [12]:
%%writefile src/politdata/reports.py

from datetime import datetime, timezone
from math import ceil
import time

import pandas as pd
import requests

from .api import DEFAULT_BASE_URL


def fetch_reports_page(
    organization_id,
    page=1,
    page_size=100,
    base_url=DEFAULT_BASE_URL,
    timeout=30,
    session=None,
):
    """
    Fetch one page of reports for a party or regional office.
    """

    owns_session = session is None

    if session is None:
        session = requests.Session()

    url = (
        f"{base_url}/party/"
        f"{organization_id}/reports"
    )

    payload = {
        "filters": {},
        "order": {},
        "pager": {
            "page": page,
            "size": page_size,
        },
    }

    try:
        response = session.post(
            url,
            json=payload,
            timeout=timeout,
        )

        response.raise_for_status()

        data = response.json()

        results = data.get("results")

        if not isinstance(results, dict):
            raise ValueError(
                "Reports response has no valid 'results' object."
            )

        report_list = results.get("list")

        if not isinstance(report_list, list):
            raise ValueError(
                "Reports response has no valid 'results.list'."
            )

        count = results.get("count")

        if count is None:
            raise ValueError(
                "Reports response has no 'results.count'."
            )

        return {
            "list": report_list,
            "count": int(count),
        }

    finally:
        if owns_session:
            session.close()


def fetch_all_reports(
    organization_id,
    page_size=100,
    base_url=DEFAULT_BASE_URL,
    timeout=30,
):
    """
    Fetch all reports for one organization,
    handling PolitData pagination automatically.
    """

    session = requests.Session()

    try:
        first = fetch_reports_page(
            organization_id,
            page=1,
            page_size=page_size,
            base_url=base_url,
            timeout=timeout,
            session=session,
        )

        reports = list(first["list"])
        count = first["count"]

        total_pages = max(
            1,
            ceil(count / page_size),
        )

        for page in range(
            2,
            total_pages + 1,
        ):
            page_result = fetch_reports_page(
                organization_id,
                page=page,
                page_size=page_size,
                base_url=base_url,
                timeout=timeout,
                session=session,
            )

            reports.extend(
                page_result["list"]
            )

        if len(reports) != count:
            raise ValueError(
                f"Report count mismatch for "
                f"{organization_id}: "
                f"expected {count}, got {len(reports)}"
            )

        report_ids = [
            report.get("id")
            for report in reports
        ]

        non_null_ids = [
            report_id
            for report_id in report_ids
            if report_id is not None
        ]

        if len(non_null_ids) != len(
            set(non_null_ids)
        ):
            raise ValueError(
                "Duplicate report IDs returned for "
                f"organization {organization_id}."
            )

        return reports

    finally:
        session.close()


def classify_period_type(quarter):
    """
    Interpret PolitData quarter codes without altering
    the original quarter value.

    1-4 = quarterly report
    5   = annual report
    """

    if quarter in {1, 2, 3, 4}:
        return "quarterly"

    if quarter == 5:
        return "annual"

    if quarter is None:
        return "missing"

    return "other"


def reports_to_manifest(
    organization_row,
    reports,
    discovered_at_utc=None,
):
    """
    Convert report-list records for one organization
    into lightweight report-manifest rows.
    """

    if discovered_at_utc is None:
        discovered_at_utc = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

    rows = []

    organization_id = (
        organization_row[
            "organization_id"
        ]
    )

    root_party_id = (
        organization_row[
            "root_party_id"
        ]
    )

    entity_type = (
        organization_row[
            "entity_type"
        ]
    )

    for report in reports:

        public_summary = (
            report.get(
                "public_summary"
            )
            or {}
        )

        rows.append({
            "report_id":
                report.get("id"),

            "organization_id":
                organization_id,

            "root_party_id":
                root_party_id,

            "entity_type":
                entity_type,

            "source_party_id":
                report.get("party_id"),

            "party_id_matches_organization":
                (
                    report.get("party_id")
                    == organization_id
                ),

            "is_party_office":
                report.get(
                    "is_party_office"
                ),

            "schema_version":
                report.get(
                    "schema_version"
                ),

            "report_type":
                report.get(
                    "report_type"
                ),

            "year":
                report.get("year"),

            # Preserve source field exactly.
            "quarter":
                report.get("quarter"),

            # Derived semantic field.
            "period_type":
                classify_period_type(
                    report.get("quarter")
                ),

            "signed_date":
                report.get(
                    "signed_date"
                ),

            "created_date":
                report.get(
                    "created_date"
                ),

            "signatory_id":
                report.get(
                    "signatory_id"
                ),

            "special_status":
                report.get(
                    "special_status"
                ),

            "public_summary_version":
                public_summary.get("v"),

            "public_summary_generated_at":
                public_summary.get(
                    "generated_at"
                ),

            "discovered_at_utc":
                discovered_at_utc,
        })

    return pd.DataFrame(rows)


def add_periodicity_flags(
    reports_df,
):
    """
    Add organization-year periodicity diagnostics.

    Also provides a preliminary annual-preference flag
    for annualized analysis.

    IMPORTANT:
    This does NOT yet resolve multiple report versions
    (main / corrected / other report_type values).
    """

    df = reports_df.copy()

    if df.empty:
        return df

    periodicity = (
        df.groupby(
            [
                "organization_id",
                "year",
            ],
            dropna=False,
        )
        .agg(
            has_annual_report=(
                "period_type",
                lambda x:
                    (x == "annual").any(),
            ),
            has_quarterly_reports=(
                "period_type",
                lambda x:
                    (x == "quarterly").any(),
            ),
            annual_report_count=(
                "period_type",
                lambda x:
                    (x == "annual").sum(),
            ),
            quarterly_report_count=(
                "period_type",
                lambda x:
                    (x == "quarterly").sum(),
            ),
            report_count=(
                "report_id",
                "count",
            ),
        )
        .reset_index()
    )

    periodicity[
        "has_mixed_periodicity"
    ] = (
        periodicity[
            "has_annual_report"
        ]
        &
        periodicity[
            "has_quarterly_reports"
        ]
    )

    df = df.merge(
        periodicity,
        on=[
            "organization_id",
            "year",
        ],
        how="left",
    )

    # User-defined analytical preference:
    #
    # if annual exists for organization + year:
    #     annual reports are preferred;
    #     quarterly reports are excluded from
    #     annualized aggregation.
    #
    # if annual does not exist:
    #     quarterly reports remain eligible.
    #
    # Version duplicates are NOT resolved here.

    df[
        "include_by_annual_preference"
    ] = False

    annual_exists = (
        df["has_annual_report"]
    )

    df.loc[
        annual_exists
        &
        (
            df["period_type"]
            == "annual"
        ),
        "include_by_annual_preference",
    ] = True

    df.loc[
        (~annual_exists)
        &
        (
            df["period_type"]
            == "quarterly"
        ),
        "include_by_annual_preference",
    ] = True

    return df


def discover_reports(
    organization_manifest,
    organization_ids=None,
    request_delay=0.15,
):
    """
    Discover reports for a selected set of organizations.

    Returns:
    - report manifest DataFrame
    - errors list

    This function does not yet persist RAW responses.
    """

    if organization_ids is None:
        selected = (
            organization_manifest.copy()
        )
    else:
        selected = (
            organization_manifest[
                organization_manifest[
                    "organization_id"
                ].isin(
                    organization_ids
                )
            ].copy()
        )

    discovered_at_utc = (
        datetime.now(
            timezone.utc
        ).isoformat()
    )

    frames = []
    errors = []

    for row in selected.to_dict(
        orient="records"
    ):

        organization_id = row[
            "organization_id"
        ]

        try:
            reports = fetch_all_reports(
                organization_id
            )

            if reports:
                frame = reports_to_manifest(
                    row,
                    reports,
                    discovered_at_utc=(
                        discovered_at_utc
                    ),
                )

                frames.append(frame)

        except Exception as exc:
            errors.append({
                "organization_id":
                    organization_id,

                "name":
                    row.get("name"),

                "error":
                    repr(exc),
            })

        if request_delay:
            time.sleep(
                request_delay
            )

    if frames:
        reports_df = pd.concat(
            frames,
            ignore_index=True,
        )
    else:
        reports_df = pd.DataFrame()

    if not reports_df.empty:

        if reports_df[
            "report_id"
        ].isna().any():
            raise ValueError(
                "Discovered report without report_id."
            )

        reports_df = (
            add_periodicity_flags(
                reports_df
            )
        )

    return reports_df, errors

Writing src/politdata/reports.py


In [2]:
manifest_test = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

golos_party = manifest_test[
    (
        manifest_test["entity_type"]
        == "party"
    )
    &
    (
        manifest_test["name"]
        == "ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»"
    )
].iloc[0]

golos_party_id = (
    golos_party[
        "organization_id"
    ]
)

golos_offices = manifest_test[
    (
        manifest_test["entity_type"]
        == "office"
    )
    &
    (
        manifest_test[
            "root_party_id"
        ]
        == golos_party_id
    )
]

test_office_id = (
    golos_offices.iloc[0][
        "organization_id"
    ]
)

test_ids = [
    golos_party_id,
    test_office_id,
]

test_reports, test_errors = (
    discover_reports(
        manifest_test,
        organization_ids=test_ids,
        request_delay=0,
    )
)

print(
    "Reports discovered:",
    len(test_reports)
)

print(
    "Errors:",
    len(test_errors)
)

print()
print("By entity type:")
print(
    test_reports[
        "entity_type"
    ].value_counts(
        dropna=False
    )
)

print()
print("Party ID mismatches:")
print(
    (
        ~test_reports[
            "party_id_matches_organization"
        ]
    ).sum()
)

print()
print("Period types:")
print(
    test_reports[
        "period_type"
    ].value_counts(
        dropna=False
    )
)

print()
print("Report types:")
print(
    test_reports[
        "report_type"
    ].value_counts(
        dropna=False
    )
)

Reports discovered: 32
Errors: 0

By entity type:
entity_type
party     16
office    16
Name: count, dtype: int64

Party ID mismatches:
0

Period types:
period_type
quarterly    28
annual        4
Name: count, dtype: int64

Report types:
report_type
main    32
Name: count, dtype: int64


In [3]:
periodicity_test = (
    test_reports[
        [
            "organization_id",
            "entity_type",
            "year",
            "has_annual_report",
            "has_quarterly_reports",
            "annual_report_count",
            "quarterly_report_count",
            "report_count",
            "has_mixed_periodicity",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "organization_id",
            "year",
        ]
    )
)

display(periodicity_test)

,organization_id,entity_type,year,has_annual_report,has_quarterly_reports,annual_report_count,quarterly_report_count,report_count,has_mixed_periodicity
19,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,office,2021,False,True,0,4,4,False
25,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,office,2022,True,False,1,0,1,False
16,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,office,2023,True,False,1,0,1,False
17,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,office,2024,False,True,0,4,4,False
18,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,office,2025,False,True,0,4,4,False
21,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,office,2026,False,True,0,2,2,False
4,b11839f3-bffe-4f93-ba10-3b877ed181b9,party,2021,False,True,0,4,4,False
8,b11839f3-bffe-4f93-ba10-3b877ed181b9,party,2022,True,False,1,0,1,False
3,b11839f3-bffe-4f93-ba10-3b877ed181b9,party,2023,True,False,1,0,1,False
0,b11839f3-bffe-4f93-ba10-3b877ed181b9,party,2024,False,True,0,4,4,False


In [4]:
mixed_test = (
    periodicity_test[
        periodicity_test[
            "has_mixed_periodicity"
        ]
    ]
)

print(
    "Organization-years with "
    "annual + quarterly reports:",
    len(mixed_test)
)

display(mixed_test)

Organization-years with annual + quarterly reports: 0


,organization_id,entity_type,year,has_annual_report,has_quarterly_reports,annual_report_count,quarterly_report_count,report_count,has_mixed_periodicity


In [5]:
display(
    test_reports[
        [
            "organization_id",
            "year",
            "quarter",
            "period_type",
            "report_type",
            "include_by_annual_preference",
        ]
    ]
    .sort_values(
        [
            "organization_id",
            "year",
            "quarter",
        ]
    )
)

,organization_id,year,quarter,period_type,report_type,include_by_annual_preference
19,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2021,1,quarterly,main,True
29,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2021,2,quarterly,main,True
24,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2021,3,quarterly,main,True
30,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2021,4,quarterly,main,True
25,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2022,5,annual,main,True
16,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2023,5,annual,main,True
17,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2024,1,quarterly,main,True
28,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2024,2,quarterly,main,True
23,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2024,3,quarterly,main,True
22,09d9a7c1-4391-4a40-82fe-b7c45c34f10f,2024,4,quarterly,main,True


In [6]:
# ---------------------------------
# DISCOVER REPORTS FOR ALL
# CENTRAL PARTIES
# ---------------------------------

manifest_current = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

central_party_ids = (
    manifest_current.loc[
        manifest_current["entity_type"]
        == "party",
        "organization_id",
    ]
    .tolist()
)

print(
    "Central parties to check:",
    len(central_party_ids)
)

central_reports, central_report_errors = (
    discover_reports(
        manifest_current,
        organization_ids=central_party_ids,
        request_delay=0.15,
    )
)

print()
print(
    "Reports discovered:",
    len(central_reports)
)

print(
    "Organizations with reports:",
    central_reports[
        "organization_id"
    ].nunique()
)

print(
    "Errors:",
    len(central_report_errors)
)

Central parties to check: 321

Reports discovered: 3639
Organizations with reports: 297
Errors: 12


In [7]:
if central_report_errors:
    display(
        pd.DataFrame(
            central_report_errors
        )
    )
else:
    print(
        "No report discovery errors."
    )

,organization_id,name,error
0,033a1696-ab48-4362-b579-699cc48aefbe,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДЯНСЬКЕ СУСПІЛЬСТВО»,ValueError('Report count mismatch for 033a1696...
1,9c883297-8c91-4bf6-8144-6304336d7178,ПОЛІТИЧНА ПАРТІЯ «ВЕЛИКА КРАЇНА»,ValueError('Report count mismatch for 9c883297...
2,f785a0c5-6fb8-4774-965b-793d3842fc49,ПОЛІТИЧНА ПАРТІЯ «ОШУКАНІ УКРАЇНЦІ»,"ReadTimeout(ReadTimeoutError(""HTTPSConnectionP..."
3,1388ae5f-2819-4153-84de-5463a956d3b4,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,ValueError('Report count mismatch for 1388ae5f...
4,626abb7f-6b12-487f-85c6-b6604080c95c,ПОЛІТИЧНА ПАРТІЯ «НАШЕ МІСТО»,ValueError('Report count mismatch for 626abb7f...
5,87ebdf18-4c0c-404f-bb45-3b354c77a212,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «УК...,ValueError('Report count mismatch for 87ebdf18...
6,40178e4b-d02d-4747-ae16-5e6385099f56,ПОЛІТИЧНА ПАРТІЯ «НАРОДНІ ІНІЦІАТИВИ ОЛЕКСАНДР...,ValueError('Report count mismatch for 40178e4b...
7,874bc096-934e-4077-b83f-5f8582d08aa1,"ПОЛІТИЧНА ПАРТІЯ ""НАША УКРАЇНА""","ReadTimeout(ReadTimeoutError(""HTTPSConnectionP..."
8,3e063c7d-ff7d-4a98-922c-c1f38c17df34,"ПОЛІТИЧНА ПАРТІЯ ""РЕАЛЬНІ СПРАВИ""","ReadTimeout(ReadTimeoutError(""HTTPSConnectionP..."
9,6bd4cca1-e368-490c-bd5b-243d66d22470,ПОЛІТИЧНА ПАРТІЯ «УСПІШНЕ МАЙБУТНЄ»,"ReadTimeout(ReadTimeoutError(""HTTPSConnectionP..."


In [8]:
print("Report types:")

display(
    central_reports[
        "report_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "report_type"
    )
    .reset_index(
        name="count"
    )
)

Report types:


,report_type,count
0,main,3639


In [9]:
print("Report types:")

display(
    central_reports[
        "report_type"
    ]
    .value_counts(
        dropna=False
    )
    .rename_axis(
        "report_type"
    )
    .reset_index(
        name="count"
    )
)

Report types:


,report_type,count
0,main,3639


In [10]:
print("Quarter codes:")

display(
    central_reports[
        "quarter"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .rename_axis(
        "quarter"
    )
    .reset_index(
        name="count"
    )
)

Quarter codes:


,quarter,count
0,1,866
1,2,902
2,3,630
3,4,641
4,5,600


In [11]:
central_periodicity = (
    central_reports[
        [
            "organization_id",
            "year",
            "has_annual_report",
            "has_quarterly_reports",
            "annual_report_count",
            "quarterly_report_count",
            "report_count",
            "has_mixed_periodicity",
        ]
    ]
    .drop_duplicates()
)

mixed_central = (
    central_periodicity[
        central_periodicity[
            "has_mixed_periodicity"
        ]
    ]
    .copy()
)

print(
    "Mixed organization-years:",
    len(mixed_central)
)

print(
    "Organizations with mixed periodicity:",
    mixed_central[
        "organization_id"
    ].nunique()
)

display(
    mixed_central.sort_values(
        [
            "year",
            "organization_id",
        ]
    )
)

Mixed organization-years: 46
Organizations with mixed periodicity: 28


,organization_id,year,has_annual_report,has_quarterly_reports,annual_report_count,quarterly_report_count,report_count,has_mixed_periodicity
399,01eddd1d-e3ec-41ad-b1f9-5a52a4d6d2fe,2021,True,True,1,1,2,True
2701,2dc08b7f-2a1e-49ae-a5d0-34041fd7165c,2021,True,True,1,2,3,True
1965,341777e6-d135-45ab-8c09-658a2c4abec3,2021,True,True,1,2,3,True
1418,3751e0fa-2389-45ac-93dc-a0bf6cb4e544,2021,True,True,1,1,2,True
1199,3f9de41c-dad8-46a7-9d68-c3141ca4f6ce,2021,True,True,1,4,5,True
2830,5be3df62-ca1d-472f-9df5-b4bad77d2bd8,2021,True,True,1,4,5,True
2732,659b917b-0eb0-4dbd-bb94-56a886675f31,2021,True,True,1,1,2,True
1634,6c002f48-99ab-4f45-ab2f-0b1a18fec286,2021,True,True,1,2,3,True
509,940be6f3-c7fa-471e-8d18-bca7fd9ab006,2021,True,True,1,1,2,True
736,993c5a91-04d5-49c4-8a37-a8a8872c9c2d,2021,True,True,1,3,4,True


In [12]:
mixed_central_named = (
    mixed_central.merge(
        manifest_current[
            [
                "organization_id",
                "name",
                "code",
            ]
        ],
        on="organization_id",
        how="left",
    )
)

display(
    mixed_central_named[
        [
            "organization_id",
            "code",
            "name",
            "year",
            "annual_report_count",
            "quarterly_report_count",
            "report_count",
        ]
    ]
    .sort_values(
        [
            "year",
            "name",
        ]
    )
)

,organization_id,code,name,year,annual_report_count,quarterly_report_count,report_count
8,993c5a91-04d5-49c4-8a37-a8a8872c9c2d,33438054,«КМКС» ПАРТІЯ УГОРЦІВ УКРАЇНИ,2021,1,3,4
17,6c002f48-99ab-4f45-ab2f-0b1a18fec286,39587072,"ПОЛІТИЧНА ПАРТІЯ ""ГРОМАДИ ЗАКАРПАТТЯ""",2021,1,2,3
42,5be3df62-ca1d-472f-9df5-b4bad77d2bd8,40201270,"ПОЛІТИЧНА ПАРТІЯ ""НОВІ РУБЕЖІ""",2021,1,4,5
31,d13251de-6917-4f9a-bdd3-d095fdae9c05,43781583,"ПОЛІТИЧНА ПАРТІЯ ""РІДНА ГАЛИЧИНА""",2021,1,4,5
11,ce8321cb-d148-4b94-a98a-1557138e78be,40508224,"ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТКУ І УКРАЇНСЬКОЇ МРІЇ""",2021,1,3,4
16,3751e0fa-2389-45ac-93dc-a0bf6cb4e544,33291360,"ПОЛІТИЧНА ПАРТІЯ ""СОЦІАЛІСТИЧНА УКРАЇНА""",2021,1,1,2
7,fbb473b0-2839-4fb3-9dff-a9235e3ae9c7,39742784,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДЯНСЬКИЙ РУХ «СПІЛЬНА С...,2021,1,1,2
38,ce516376-dfbb-445a-a15d-e2a00ead39b1,33476152,ПОЛІТИЧНА ПАРТІЯ «ДЕМОКРАТИЧНА ПАРТІЯ УГОРЦІВ ...,2021,1,4,5
13,3f9de41c-dad8-46a7-9d68-c3141ca4f6ce,40211383,ПОЛІТИЧНА ПАРТІЯ «ЖИТТЯ»,2021,1,4,5
3,f5334c01-faf3-44f4-ae43-6bfceffe8b5e,40212900,ПОЛІТИЧНА ПАРТІЯ «ЗА ОДЕЩИНУ»,2021,1,3,4


In [13]:
period_duplicates = (
    central_reports
    .groupby(
        [
            "organization_id",
            "year",
            "quarter",
        ],
        dropna=False,
    )
    .agg(
        report_count=(
            "report_id",
            "count"
        ),
        report_types=(
            "report_type",
            lambda x: sorted(
                set(
                    str(v)
                    for v in x
                )
            )
        ),
    )
    .reset_index()
)

period_duplicates = (
    period_duplicates[
        period_duplicates[
            "report_count"
        ] > 1
    ]
    .copy()
)

print(
    "Organization-periods with "
    "multiple reports:",
    len(period_duplicates)
)

display(
    period_duplicates
)

Organization-periods with multiple reports: 0


,organization_id,year,quarter,report_count,report_types


In [14]:
print("Total errors:", len(central_report_errors))
print()

for i, error in enumerate(
    central_report_errors,
    start=1,
):
    print("=" * 80)
    print("ERROR", i)
    print("Organization ID:", error["organization_id"])
    print("Name:", error["name"])
    print("Error:", error["error"])

Total errors: 12

ERROR 1
Organization ID: 033a1696-ab48-4362-b579-699cc48aefbe
Name: ПОЛІТИЧНА ПАРТІЯ «ГРОМАДЯНСЬКЕ СУСПІЛЬСТВО»
Error: ValueError('Report count mismatch for 033a1696-ab48-4362-b579-699cc48aefbe: expected 18, got 17')
ERROR 2
Organization ID: 9c883297-8c91-4bf6-8144-6304336d7178
Name: ПОЛІТИЧНА ПАРТІЯ «ВЕЛИКА КРАЇНА»
Error: ValueError('Report count mismatch for 9c883297-8c91-4bf6-8144-6304336d7178: expected 22, got 21')
ERROR 3
Organization ID: f785a0c5-6fb8-4774-965b-793d3842fc49
Name: ПОЛІТИЧНА ПАРТІЯ «ОШУКАНІ УКРАЇНЦІ»
Error: ReadTimeout(ReadTimeoutError("HTTPSConnectionPool(host='politdata.nazk.gov.ua', port=443): Read timed out. (read timeout=30)"))
ERROR 4
Organization ID: 1388ae5f-2819-4153-84de-5463a956d3b4
Name: ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»
Error: ValueError('Report count mismatch for 1388ae5f-2819-4153-84de-5463a956d3b4: expected 15, got 14')
ERROR 5
Organization ID: 626abb7f-6b12-487f-85c6-b6604080c95c
Name: ПОЛІТИЧНА ПАРТІЯ «НАШЕ МІСТО»
Error: ValueError('R

In [15]:
import math
import requests


diagnostic_id = (
    "033a1696-ab48-4362-b579-699cc48aefbe"
)

diagnostic_url = (
    f"{DEFAULT_BASE_URL}/party/"
    f"{diagnostic_id}/reports"
)


def probe_reports_page(
    page,
    size,
):
    payload = {
        "filters": {},
        "order": {},
        "pager": {
            "page": page,
            "size": size,
        },
    }

    response = requests.post(
        diagnostic_url,
        json=payload,
        timeout=60,
    )

    print(
        f"page={page}, size={size}, "
        f"HTTP={response.status_code}"
    )

    response.raise_for_status()

    data = response.json()

    results = data["results"]

    items = results.get("list", [])
    count = results.get("count")

    print(
        "declared count:",
        count,
        "| returned rows:",
        len(items),
    )

    return items, count

In [16]:
for size in [10, 25, 50, 100]:
    print()
    print("-" * 60)

    items, count = probe_reports_page(
        page=1,
        size=size,
    )


------------------------------------------------------------
page=1, size=10, HTTP=200
declared count: 18 | returned rows: 10

------------------------------------------------------------
page=1, size=25, HTTP=200
declared count: 18 | returned rows: 17

------------------------------------------------------------
page=1, size=50, HTTP=200
declared count: 18 | returned rows: 17

------------------------------------------------------------
page=1, size=100, HTTP=200
declared count: 18 | returned rows: 17


In [17]:
page_size = 10

first_items, declared_count = (
    probe_reports_page(
        page=1,
        size=page_size,
    )
)

pages = math.ceil(
    declared_count / page_size
)

all_items = list(first_items)

print()
print(
    "Declared total:",
    declared_count
)

print(
    "Expected pages:",
    pages
)

for page in range(
    2,
    pages + 1,
):
    print()

    items, page_count = (
        probe_reports_page(
            page=page,
            size=page_size,
        )
    )

    all_items.extend(items)


all_ids = [
    item.get("id")
    for item in all_items
]

print()
print("=" * 60)

print(
    "Declared count:",
    declared_count
)

print(
    "Collected rows:",
    len(all_items)
)

print(
    "Unique report IDs:",
    len(set(all_ids))
)

print(
    "Duplicate IDs:",
    len(all_ids)
    - len(set(all_ids))
)

page=1, size=10, HTTP=200
declared count: 18 | returned rows: 10

Declared total: 18
Expected pages: 2

page=2, size=10, HTTP=200
declared count: 18 | returned rows: 7

Declared count: 18
Collected rows: 17
Unique report IDs: 17
Duplicate IDs: 0


In [18]:
%%writefile src/politdata/reports.py

from datetime import datetime, timezone
import time

import pandas as pd
import requests

from .api import DEFAULT_BASE_URL


RETRYABLE_STATUS_CODES = {
    408,
    429,
    500,
    502,
    503,
    504,
}


def fetch_reports_page(
    organization_id,
    page=1,
    page_size=100,
    base_url=DEFAULT_BASE_URL,
    timeout=60,
    max_retries=4,
    retry_backoff=1.0,
    session=None,
):
    """
    Fetch one page of reports for a party or regional office.

    Retries temporary network/server failures.
    """

    owns_session = session is None

    if session is None:
        session = requests.Session()

    url = (
        f"{base_url}/party/"
        f"{organization_id}/reports"
    )

    payload = {
        "filters": {},
        "order": {},
        "pager": {
            "page": page,
            "size": page_size,
        },
    }

    try:

        for attempt in range(max_retries):

            try:

                response = session.post(
                    url,
                    json=payload,
                    timeout=timeout,
                )

                if (
                    response.status_code
                    in RETRYABLE_STATUS_CODES
                ):
                    response.raise_for_status()

                response.raise_for_status()

                data = response.json()

                results = data.get("results")

                if not isinstance(
                    results,
                    dict,
                ):
                    raise ValueError(
                        "Reports response has no "
                        "valid 'results' object."
                    )

                report_list = results.get(
                    "list"
                )

                if not isinstance(
                    report_list,
                    list,
                ):
                    raise ValueError(
                        "Reports response has no "
                        "valid 'results.list'."
                    )

                count = results.get(
                    "count"
                )

                if count is None:
                    raise ValueError(
                        "Reports response has no "
                        "'results.count'."
                    )

                return {
                    "list": report_list,
                    "count": int(count),
                }

            except (
                requests.exceptions.ReadTimeout,
                requests.exceptions.ConnectTimeout,
                requests.exceptions.ConnectionError,
            ):

                if attempt == max_retries - 1:
                    raise

                sleep_seconds = (
                    retry_backoff
                    * (2 ** attempt)
                )

                time.sleep(
                    sleep_seconds
                )

            except requests.exceptions.HTTPError:

                status_code = (
                    response.status_code
                )

                if (
                    status_code
                    not in RETRYABLE_STATUS_CODES
                    or attempt
                    == max_retries - 1
                ):
                    raise

                sleep_seconds = (
                    retry_backoff
                    * (2 ** attempt)
                )

                time.sleep(
                    sleep_seconds
                )

    finally:

        if owns_session:
            session.close()


def fetch_all_reports(
    organization_id,
    page_size=100,
    base_url=DEFAULT_BASE_URL,
    timeout=60,
    max_retries=4,
    retry_backoff=1.0,
    max_pages=100,
    return_metadata=False,
):
    """
    Fetch all available reports for one organization.

    Important:
    PolitData's declared `count` may be larger than the
    number of reports actually returned by the API.

    Therefore:
    - count mismatch is recorded as metadata;
    - available reports are NOT discarded;
    - pagination stops based on returned page length,
      not declared count alone.
    """

    session = requests.Session()

    try:

        all_rows = []
        declared_counts = []

        page = 1

        while True:

            page_result = fetch_reports_page(
                organization_id,
                page=page,
                page_size=page_size,
                base_url=base_url,
                timeout=timeout,
                max_retries=max_retries,
                retry_backoff=retry_backoff,
                session=session,
            )

            page_rows = (
                page_result["list"]
            )

            declared_counts.append(
                page_result["count"]
            )

            all_rows.extend(
                page_rows
            )

            # Stop when API returns a partial page.
            # This is safer than trusting `count`.
            if len(page_rows) < page_size:
                break

            page += 1

            if page > max_pages:
                raise RuntimeError(
                    "Maximum report pages exceeded "
                    f"for {organization_id}."
                )

        declared_count = (
            declared_counts[0]
            if declared_counts
            else 0
        )

        count_changed_during_fetch = (
            len(
                set(
                    declared_counts
                )
            )
            > 1
        )

        # ---------------------------------
        # Validate IDs + remove duplicates
        # ---------------------------------

        unique_reports = []
        seen_ids = set()
        duplicate_ids = []

        for report in all_rows:

            report_id = report.get(
                "id"
            )

            if not report_id:
                raise ValueError(
                    "Report without ID returned for "
                    f"{organization_id}."
                )

            if report_id in seen_ids:

                duplicate_ids.append(
                    report_id
                )

                continue

            seen_ids.add(
                report_id
            )

            unique_reports.append(
                report
            )

        fetched_count = len(
            unique_reports
        )

        metadata = {
            "organization_id":
                organization_id,

            "declared_count":
                declared_count,

            "raw_fetched_rows":
                len(all_rows),

            "fetched_count":
                fetched_count,

            "count_mismatch":
                (
                    declared_count
                    != fetched_count
                ),

            "count_difference":
                (
                    declared_count
                    - fetched_count
                ),

            "pages_requested":
                page,

            "page_size":
                page_size,

            "duplicate_report_count":
                len(
                    duplicate_ids
                ),

            "duplicate_report_ids":
                sorted(
                    set(
                        duplicate_ids
                    )
                ),

            "count_changed_during_fetch":
                count_changed_during_fetch,
        }

        if return_metadata:

            return (
                unique_reports,
                metadata,
            )

        return unique_reports

    finally:

        session.close()


def classify_period_type(
    quarter
):
    """
    Interpret PolitData quarter codes.

    Source value is preserved separately.

    1-4 = quarterly
    5   = annual
    """

    if quarter in {
        1,
        2,
        3,
        4,
    }:
        return "quarterly"

    if quarter == 5:
        return "annual"

    if quarter is None:
        return "missing"

    return "other"


def reports_to_manifest(
    organization_row,
    reports,
    discovered_at_utc=None,
):
    """
    Convert report-list records into lightweight
    report-manifest rows.
    """

    if discovered_at_utc is None:

        discovered_at_utc = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

    rows = []

    organization_id = (
        organization_row[
            "organization_id"
        ]
    )

    root_party_id = (
        organization_row[
            "root_party_id"
        ]
    )

    entity_type = (
        organization_row[
            "entity_type"
        ]
    )

    for report in reports:

        public_summary = (
            report.get(
                "public_summary"
            )
            or {}
        )

        rows.append({
            "report_id":
                report.get("id"),

            "organization_id":
                organization_id,

            "root_party_id":
                root_party_id,

            "entity_type":
                entity_type,

            "source_party_id":
                report.get(
                    "party_id"
                ),

            "party_id_matches_organization":
                (
                    report.get(
                        "party_id"
                    )
                    == organization_id
                ),

            "is_party_office":
                report.get(
                    "is_party_office"
                ),

            "schema_version":
                report.get(
                    "schema_version"
                ),

            "report_type":
                report.get(
                    "report_type"
                ),

            "year":
                report.get(
                    "year"
                ),

            # Original PolitData value.
            "quarter":
                report.get(
                    "quarter"
                ),

            # Derived interpretation.
            "period_type":
                classify_period_type(
                    report.get(
                        "quarter"
                    )
                ),

            "signed_date":
                report.get(
                    "signed_date"
                ),

            "created_date":
                report.get(
                    "created_date"
                ),

            "signatory_id":
                report.get(
                    "signatory_id"
                ),

            "special_status":
                report.get(
                    "special_status"
                ),

            "public_summary_version":
                public_summary.get(
                    "v"
                ),

            "public_summary_generated_at":
                public_summary.get(
                    "generated_at"
                ),

            "discovered_at_utc":
                discovered_at_utc,
        })

    return pd.DataFrame(
        rows
    )


def add_periodicity_flags(
    reports_df,
):
    """
    Add organization-year periodicity diagnostics.

    Annual-preference rule:

    If an annual report exists for an organization-year,
    quarterly reports from that organization-year are
    excluded from annualized aggregation.

    Nothing is physically deleted.
    """

    df = reports_df.copy()

    if df.empty:
        return df

    periodicity = (
        df.groupby(
            [
                "organization_id",
                "year",
            ],
            dropna=False,
        )
        .agg(
            has_annual_report=(
                "period_type",
                lambda x:
                    (
                        x
                        == "annual"
                    ).any(),
            ),

            has_quarterly_reports=(
                "period_type",
                lambda x:
                    (
                        x
                        == "quarterly"
                    ).any(),
            ),

            annual_report_count=(
                "period_type",
                lambda x:
                    (
                        x
                        == "annual"
                    ).sum(),
            ),

            quarterly_report_count=(
                "period_type",
                lambda x:
                    (
                        x
                        == "quarterly"
                    ).sum(),
            ),

            report_count=(
                "report_id",
                "count",
            ),
        )
        .reset_index()
    )

    periodicity[
        "has_mixed_periodicity"
    ] = (
        periodicity[
            "has_annual_report"
        ]
        &
        periodicity[
            "has_quarterly_reports"
        ]
    )

    df = df.merge(
        periodicity,
        on=[
            "organization_id",
            "year",
        ],
        how="left",
    )

    df[
        "include_by_annual_preference"
    ] = False

    annual_exists = (
        df[
            "has_annual_report"
        ]
    )

    # Annual exists -> prefer annual.
    df.loc[
        annual_exists
        &
        (
            df[
                "period_type"
            ]
            == "annual"
        ),
        "include_by_annual_preference",
    ] = True

    # No annual -> retain quarterlies.
    df.loc[
        (~annual_exists)
        &
        (
            df[
                "period_type"
            ]
            == "quarterly"
        ),
        "include_by_annual_preference",
    ] = True

    return df


def discover_reports(
    organization_manifest,
    organization_ids=None,
    request_delay=0.15,
    timeout=60,
    max_retries=4,
    return_diagnostics=False,
):
    """
    Discover reports for selected organizations.

    Count mismatches are diagnostics, not fatal errors.

    Actual network/API failures remain errors.
    """

    if organization_ids is None:

        selected = (
            organization_manifest.copy()
        )

    else:

        selected = (
            organization_manifest[
                organization_manifest[
                    "organization_id"
                ].isin(
                    organization_ids
                )
            ].copy()
        )

    discovered_at_utc = (
        datetime.now(
            timezone.utc
        ).isoformat()
    )

    frames = []
    errors = []
    diagnostics = []

    for row in selected.to_dict(
        orient="records"
    ):

        organization_id = row[
            "organization_id"
        ]

        try:

            (
                reports,
                fetch_metadata,
            ) = fetch_all_reports(
                organization_id,
                timeout=timeout,
                max_retries=max_retries,
                return_metadata=True,
            )

            diagnostics.append({
                "organization_id":
                    organization_id,

                "entity_type":
                    row.get(
                        "entity_type"
                    ),

                "name":
                    row.get(
                        "name"
                    ),

                **fetch_metadata,
            })

            if reports:

                frame = reports_to_manifest(
                    row,
                    reports,
                    discovered_at_utc=(
                        discovered_at_utc
                    ),
                )

                frames.append(
                    frame
                )

        except Exception as exc:

            errors.append({
                "organization_id":
                    organization_id,

                "name":
                    row.get(
                        "name"
                    ),

                "error":
                    repr(
                        exc
                    ),
            })

        if request_delay:

            time.sleep(
                request_delay
            )

    if frames:

        reports_df = pd.concat(
            frames,
            ignore_index=True,
        )

    else:

        reports_df = (
            pd.DataFrame()
        )

    if not reports_df.empty:

        if reports_df[
            "report_id"
        ].isna().any():

            raise ValueError(
                "Discovered report "
                "without report_id."
            )

        reports_df = (
            add_periodicity_flags(
                reports_df
            )
        )

    diagnostics_df = (
        pd.DataFrame(
            diagnostics
        )
    )

    if return_diagnostics:

        return (
            reports_df,
            errors,
            diagnostics_df,
        )

    return (
        reports_df,
        errors,
    )

Overwriting src/politdata/reports.py


In [2]:
problem_party_ids = [
    "033a1696-ab48-4362-b579-699cc48aefbe",
    "9c883297-8c91-4bf6-8144-6304336d7178",
    "f785a0c5-6fb8-4774-965b-793d3842fc49",
    "1388ae5f-2819-4153-84de-5463a956d3b4",
    "626abb7f-6b12-487f-85c6-b6604080c95c",
    "87ebdf18-4c0c-404f-bb45-3b354c77a212",
    "40178e4b-d02d-4747-ae16-5e6385099f56",
    "874bc096-934e-4077-b83f-5f8582d08aa1",
    "3e063c7d-ff7d-4a98-922c-c1f38c17df34",
    "6bd4cca1-e368-490c-bd5b-243d66d22470",
    "79cb7831-508b-466a-8db0-db9c5f4080b2",
    "725c7a38-0a5d-4f1c-a5d0-f4024a06fb27",
]

manifest_problem_test = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

(
    problem_reports,
    problem_errors,
    problem_diagnostics,
) = discover_reports(
    manifest_problem_test,
    organization_ids=problem_party_ids,
    request_delay=0.15,
    timeout=60,
    max_retries=4,
    return_diagnostics=True,
)

print(
    "Organizations tested:",
    len(problem_party_ids)
)

print(
    "Reports recovered:",
    len(problem_reports)
)

print(
    "Errors:",
    len(problem_errors)
)

print(
    "Count mismatches:",
    problem_diagnostics[
        "count_mismatch"
    ].sum()
)

Organizations tested: 12
Reports recovered: 189
Errors: 0
Count mismatches: 7


In [3]:
display(
    problem_diagnostics[
        [
            "organization_id",
            "name",
            "declared_count",
            "fetched_count",
            "count_difference",
            "count_mismatch",
            "pages_requested",
            "duplicate_report_count",
            "count_changed_during_fetch",
        ]
    ]
    .sort_values(
        [
            "count_mismatch",
            "name",
        ],
        ascending=[
            False,
            True,
        ]
    )
)

,organization_id,name,declared_count,fetched_count,count_difference,count_mismatch,pages_requested,duplicate_report_count,count_changed_during_fetch
1,9c883297-8c91-4bf6-8144-6304336d7178,ПОЛІТИЧНА ПАРТІЯ «ВЕЛИКА КРАЇНА»,22,21,1,True,1,0,False
5,87ebdf18-4c0c-404f-bb45-3b354c77a212,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «УК...,14,13,1,True,1,0,False
0,033a1696-ab48-4362-b579-699cc48aefbe,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДЯНСЬКЕ СУСПІЛЬСТВО»,18,17,1,True,1,0,False
6,40178e4b-d02d-4747-ae16-5e6385099f56,ПОЛІТИЧНА ПАРТІЯ «НАРОДНІ ІНІЦІАТИВИ ОЛЕКСАНДР...,19,18,1,True,1,0,False
4,626abb7f-6b12-487f-85c6-b6604080c95c,ПОЛІТИЧНА ПАРТІЯ «НАШЕ МІСТО»,15,13,2,True,1,0,False
3,1388ae5f-2819-4153-84de-5463a956d3b4,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,15,14,1,True,1,0,False
10,79cb7831-508b-466a-8db0-db9c5f4080b2,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ВОЛОДИМИРА БУРЯКА «ЄД...,22,21,1,True,1,0,False
7,874bc096-934e-4077-b83f-5f8582d08aa1,"ПОЛІТИЧНА ПАРТІЯ ""НАША УКРАЇНА""",7,7,0,False,1,0,False
8,3e063c7d-ff7d-4a98-922c-c1f38c17df34,"ПОЛІТИЧНА ПАРТІЯ ""РЕАЛЬНІ СПРАВИ""",13,13,0,False,1,0,False
11,725c7a38-0a5d-4f1c-a5d0-f4024a06fb27,"ПОЛІТИЧНА ПАРТІЯ ""ТРИБУНАЛ""",15,15,0,False,1,0,False


In [4]:
if problem_errors:
    display(
        pd.DataFrame(
            problem_errors
        )
    )
else:
    print(
        "No hard errors."
    )

No hard errors.


In [5]:
manifest_current = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

central_party_ids = (
    manifest_current.loc[
        manifest_current["entity_type"] == "party",
        "organization_id",
    ]
    .tolist()
)

(
    central_reports_v2,
    central_errors_v2,
    central_diagnostics_v2,
) = discover_reports(
    manifest_current,
    organization_ids=central_party_ids,
    request_delay=0.15,
    timeout=60,
    max_retries=4,
    return_diagnostics=True,
)

print("Central parties checked:", len(central_party_ids))
print("Reports discovered:", len(central_reports_v2))

print(
    "Organizations with reports:",
    central_reports_v2[
        "organization_id"
    ].nunique()
)

print("Hard errors:", len(central_errors_v2))

print(
    "Count mismatches:",
    central_diagnostics_v2[
        "count_mismatch"
    ].sum()
)

Central parties checked: 321
Reports discovered: 3828
Organizations with reports: 309
Hard errors: 0
Count mismatches: 7


In [6]:
from pathlib import Path

reports_output_dir = Path(
    "data/interim/reports"
)

reports_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

central_reports_path = (
    reports_output_dir
    / "central_party_reports_manifest.parquet"
)

central_diagnostics_path = (
    reports_output_dir
    / "central_party_reports_diagnostics.parquet"
)

central_reports_v2.to_parquet(
    central_reports_path,
    index=False,
)

central_diagnostics_v2.to_parquet(
    central_diagnostics_path,
    index=False,
)

print("Saved reports:")
print(central_reports_path)

print()
print("Saved diagnostics:")
print(central_diagnostics_path)

Saved reports:
data\interim\reports\central_party_reports_manifest.parquet

Saved diagnostics:
data\interim\reports\central_party_reports_diagnostics.parquet


In [7]:
print("REPORT MANIFEST QA")
print("-" * 50)

print(
    "Rows:",
    len(central_reports_v2)
)

print(
    "Unique report IDs:",
    central_reports_v2[
        "report_id"
    ].nunique()
)

print(
    "Duplicate report IDs:",
    central_reports_v2[
        "report_id"
    ].duplicated().sum()
)

print(
    "Missing report IDs:",
    central_reports_v2[
        "report_id"
    ].isna().sum()
)

print(
    "Party ID mismatches:",
    (
        ~central_reports_v2[
            "party_id_matches_organization"
        ]
    ).sum()
)

print()
print("Period types:")
print(
    central_reports_v2[
        "period_type"
    ].value_counts(
        dropna=False
    )
)

print()
print("Report types:")
print(
    central_reports_v2[
        "report_type"
    ].value_counts(
        dropna=False
    )
)

print()
print("Quarter codes:")
print(
    central_reports_v2[
        "quarter"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)

REPORT MANIFEST QA
--------------------------------------------------
Rows: 3828
Unique report IDs: 3828
Duplicate report IDs: 0
Missing report IDs: 0
Party ID mismatches: 0

Period types:
period_type
quarterly    3203
annual        625
Name: count, dtype: int64

Report types:
report_type
main    3828
Name: count, dtype: int64

Quarter codes:
quarter
1    911
2    950
3    665
4    677
5    625
Name: count, dtype: int64


In [8]:
central_periodicity_v2 = (
    central_reports_v2[
        [
            "organization_id",
            "year",
            "has_annual_report",
            "has_quarterly_reports",
            "annual_report_count",
            "quarterly_report_count",
            "report_count",
            "has_mixed_periodicity",
        ]
    ]
    .drop_duplicates()
)

mixed_central_v2 = (
    central_periodicity_v2[
        central_periodicity_v2[
            "has_mixed_periodicity"
        ]
    ]
    .copy()
)

print(
    "Mixed organization-years:",
    len(mixed_central_v2)
)

print(
    "Parties with mixed periodicity:",
    mixed_central_v2[
        "organization_id"
    ].nunique()
)

Mixed organization-years: 51
Parties with mixed periodicity: 31


In [9]:
count_mismatch_cases = (
    central_diagnostics_v2[
        central_diagnostics_v2[
            "count_mismatch"
        ]
    ]
    .copy()
)

count_mismatch_path = (
    reports_output_dir
    / "central_party_report_count_anomalies.parquet"
)

count_mismatch_cases.to_parquet(
    count_mismatch_path,
    index=False,
)

print(
    "Count mismatch cases:",
    len(count_mismatch_cases)
)

display(
    count_mismatch_cases[
        [
            "organization_id",
            "name",
            "declared_count",
            "fetched_count",
            "count_difference",
        ]
    ]
)

Count mismatch cases: 7


,organization_id,name,declared_count,fetched_count,count_difference
23,033a1696-ab48-4362-b579-699cc48aefbe,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДЯНСЬКЕ СУСПІЛЬСТВО»,18,17,1
27,9c883297-8c91-4bf6-8144-6304336d7178,ПОЛІТИЧНА ПАРТІЯ «ВЕЛИКА КРАЇНА»,22,21,1
127,1388ae5f-2819-4153-84de-5463a956d3b4,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,15,14,1
128,626abb7f-6b12-487f-85c6-b6604080c95c,ПОЛІТИЧНА ПАРТІЯ «НАШЕ МІСТО»,15,13,2
153,87ebdf18-4c0c-404f-bb45-3b354c77a212,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «УК...,14,13,1
194,40178e4b-d02d-4747-ae16-5e6385099f56,ПОЛІТИЧНА ПАРТІЯ «НАРОДНІ ІНІЦІАТИВИ ОЛЕКСАНДР...,19,18,1
233,79cb7831-508b-466a-8db0-db9c5f4080b2,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ВОЛОДИМИРА БУРЯКА «ЄД...,22,21,1


In [10]:
%%writefile src/politdata/report_discovery.py

from datetime import datetime, timezone
from pathlib import Path
import json

import pandas as pd
from tqdm.auto import tqdm

from .reports import (
    fetch_all_reports,
    reports_to_manifest,
    add_periodicity_flags,
)


DEFAULT_STATE_PATH = Path(
    "data/interim/state/report_discovery_state.parquet"
)

DEFAULT_SNAPSHOT_DIR = Path(
    "data/raw/report_lists"
)


def _write_json_atomic(path, data):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    with temp_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

    temp_path.replace(path)


def save_report_discovery_state(
    state,
    state_path=DEFAULT_STATE_PATH,
):
    """
    Safely persist discovery progress.
    """

    state_path = Path(state_path)

    state_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = state_path.with_suffix(
        ".tmp.parquet"
    )

    state.to_parquet(
        temp_path,
        index=False,
    )

    temp_path.replace(
        state_path
    )

    return state_path


def initialize_report_discovery_state(
    organization_manifest,
    state_path=DEFAULT_STATE_PATH,
):
    """
    Initialize or update report-discovery state.

    Existing successful/error states are preserved.
    Newly discovered organizations start as pending.
    """

    state_path = Path(state_path)

    manifest = (
        organization_manifest[
            [
                "organization_id",
                "root_party_id",
                "entity_type",
                "name",
            ]
        ]
        .copy()
    )

    if state_path.exists():

        old_state = pd.read_parquet(
            state_path
        )

        old_state = old_state.set_index(
            "organization_id"
        )

        rows = []

        for row in manifest.to_dict(
            orient="records"
        ):

            organization_id = row[
                "organization_id"
            ]

            if organization_id in old_state.index:

                old = old_state.loc[
                    organization_id
                ].to_dict()

                old.update({
                    "organization_id":
                        organization_id,

                    "root_party_id":
                        row["root_party_id"],

                    "entity_type":
                        row["entity_type"],

                    "name":
                        row["name"],
                })

                rows.append(old)

            else:

                rows.append({
                    **row,
                    "status": "pending",
                    "last_checked_at_utc": None,
                    "declared_count": None,
                    "fetched_count": None,
                    "count_difference": None,
                    "count_mismatch": None,
                    "snapshot_path": None,
                    "error": None,
                })

        state = pd.DataFrame(
            rows
        )

    else:

        state = manifest.copy()

        state["status"] = "pending"
        state["last_checked_at_utc"] = None
        state["declared_count"] = None
        state["fetched_count"] = None
        state["count_difference"] = None
        state["count_mismatch"] = None
        state["snapshot_path"] = None
        state["error"] = None

    save_report_discovery_state(
        state,
        state_path=state_path,
    )

    return state


def run_report_discovery_batch(
    organization_manifest,
    organization_ids=None,
    entity_type=None,
    limit=None,
    state_path=DEFAULT_STATE_PATH,
    snapshot_dir=DEFAULT_SNAPSHOT_DIR,
    timeout=60,
    max_retries=4,
    retry_errors=True,
):
    """
    Discover report lists with durable resume support.

    Successful organizations are skipped on future runs.

    Errors remain retryable.

    State is saved after EACH organization, so an interrupted
    run can resume without repeating completed work.
    """

    snapshot_dir = Path(
        snapshot_dir
    )

    snapshot_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    state = initialize_report_discovery_state(
        organization_manifest,
        state_path=state_path,
    )

    candidates = state.copy()

    if organization_ids is not None:

        organization_ids = set(
            organization_ids
        )

        candidates = candidates[
            candidates[
                "organization_id"
            ].isin(
                organization_ids
            )
        ]

    if entity_type is not None:

        candidates = candidates[
            candidates[
                "entity_type"
            ]
            == entity_type
        ]

    # Successful organizations do not need
    # to be downloaded again.
    candidates = candidates[
        candidates["status"]
        != "success"
    ]

    if not retry_errors:

        candidates = candidates[
            candidates["status"]
            != "error"
        ]

    if limit is not None:

        candidates = candidates.head(
            limit
        )

    selected_ids = candidates[
        "organization_id"
    ].tolist()

    state = state.set_index(
        "organization_id"
    )

    successful = 0
    failed = 0
    reports_fetched = 0

    for organization_id in tqdm(
        selected_ids,
        desc="Report discovery",
    ):

        retrieved_at = datetime.now(
            timezone.utc
        )

        try:

            (
                reports,
                metadata,
            ) = fetch_all_reports(
                organization_id,
                timeout=timeout,
                max_retries=max_retries,
                return_metadata=True,
            )

            snapshot_path = (
                snapshot_dir
                / f"{organization_id}.json"
            )

            # Report objects themselves are stored
            # unchanged as returned by PolitData.
            snapshot = {
                "source": "PolitData",
                "endpoint":
                    f"/party/{organization_id}/reports",

                "organization_id":
                    organization_id,

                "retrieved_at_utc":
                    retrieved_at.isoformat(),

                "fetch_metadata":
                    metadata,

                "reports":
                    reports,
            }

            _write_json_atomic(
                snapshot_path,
                snapshot,
            )

            state.loc[
                organization_id,
                "status",
            ] = "success"

            state.loc[
                organization_id,
                "last_checked_at_utc",
            ] = retrieved_at.isoformat()

            state.loc[
                organization_id,
                "declared_count",
            ] = metadata[
                "declared_count"
            ]

            state.loc[
                organization_id,
                "fetched_count",
            ] = metadata[
                "fetched_count"
            ]

            state.loc[
                organization_id,
                "count_difference",
            ] = metadata[
                "count_difference"
            ]

            state.loc[
                organization_id,
                "count_mismatch",
            ] = metadata[
                "count_mismatch"
            ]

            state.loc[
                organization_id,
                "snapshot_path",
            ] = str(
                snapshot_path
            )

            state.loc[
                organization_id,
                "error",
            ] = None

            successful += 1

            reports_fetched += len(
                reports
            )

        except Exception as exc:

            state.loc[
                organization_id,
                "status",
            ] = "error"

            state.loc[
                organization_id,
                "last_checked_at_utc",
            ] = retrieved_at.isoformat()

            state.loc[
                organization_id,
                "error",
            ] = repr(
                exc
            )

            failed += 1

        # CRITICAL:
        # save progress after every organization.
        current_state = (
            state
            .reset_index()
        )

        save_report_discovery_state(
            current_state,
            state_path=state_path,
        )

    final_state = (
        state.reset_index()
    )

    summary = {
        "selected": len(
            selected_ids
        ),

        "successful":
            successful,

        "failed":
            failed,

        "reports_fetched":
            reports_fetched,

        "total_success_in_state":
            (
                final_state[
                    "status"
                ]
                == "success"
            ).sum(),

        "total_errors_in_state":
            (
                final_state[
                    "status"
                ]
                == "error"
            ).sum(),

        "total_pending_in_state":
            (
                final_state[
                    "status"
                ]
                == "pending"
            ).sum(),
    }

    return (
        summary,
        final_state,
    )


def build_report_manifest_from_snapshots(
    organization_manifest,
    snapshot_dir=DEFAULT_SNAPSHOT_DIR,
):
    """
    Rebuild one unified report manifest from saved
    per-organization report-list snapshots.
    """

    snapshot_dir = Path(
        snapshot_dir
    )

    manifest_indexed = (
        organization_manifest
        .set_index(
            "organization_id"
        )
    )

    frames = []
    missing_organizations = []

    for snapshot_path in sorted(
        snapshot_dir.glob(
            "*.json"
        )
    ):

        with snapshot_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            snapshot = json.load(
                f
            )

        organization_id = (
            snapshot[
                "organization_id"
            ]
        )

        if (
            organization_id
            not in manifest_indexed.index
        ):

            missing_organizations.append(
                organization_id
            )

            continue

        reports = snapshot.get(
            "reports",
            []
        )

        if not reports:
            continue

        organization_row = (
            manifest_indexed.loc[
                organization_id
            ].to_dict()
        )

        organization_row[
            "organization_id"
        ] = organization_id

        frame = reports_to_manifest(
            organization_row,
            reports,
            discovered_at_utc=(
                snapshot.get(
                    "retrieved_at_utc"
                )
            ),
        )

        frames.append(
            frame
        )

    if frames:

        reports_df = pd.concat(
            frames,
            ignore_index=True,
        )

        reports_df = (
            add_periodicity_flags(
                reports_df
            )
        )

    else:

        reports_df = pd.DataFrame()

    qa = {
        "rows":
            len(reports_df),

        "unique_report_ids":
            (
                reports_df[
                    "report_id"
                ].nunique()
                if not reports_df.empty
                else 0
            ),

        "duplicate_report_ids":
            (
                reports_df[
                    "report_id"
                ].duplicated().sum()
                if not reports_df.empty
                else 0
            ),

        "missing_report_ids":
            (
                reports_df[
                    "report_id"
                ].isna().sum()
                if not reports_df.empty
                else 0
            ),

        "party_id_mismatches":
            (
                (
                    ~reports_df[
                        "party_id_matches_organization"
                    ]
                ).sum()
                if not reports_df.empty
                else 0
            ),

        "snapshot_orgs_not_in_manifest":
            len(
                missing_organizations
            ),
    }

    return (
        reports_df,
        qa,
    )

Writing src/politdata/report_discovery.py


In [2]:
manifest_reports_batch = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

test_office_ids = (
    manifest_reports_batch.loc[
        manifest_reports_batch[
            "entity_type"
        ] == "office",
        "organization_id",
    ]
    .head(20)
    .tolist()
)

batch_summary_1, report_state_1 = (
    run_report_discovery_batch(
        manifest_reports_batch,
        organization_ids=test_office_ids,
        timeout=60,
        max_retries=4,
    )
)

print(batch_summary_1)

Report discovery:   0%|          | 0/20 [00:00<?, ?it/s]

{'selected': 20, 'successful': 20, 'failed': 0, 'reports_fetched': 238, 'total_success_in_state': np.int64(20), 'total_errors_in_state': np.int64(0), 'total_pending_in_state': np.int64(9897)}


In [3]:
batch_summary_2, report_state_2 = (
    run_report_discovery_batch(
        manifest_reports_batch,
        organization_ids=test_office_ids,
        timeout=60,
        max_retries=4,
    )
)

print(batch_summary_2)

Report discovery: 0it [00:00, ?it/s]

{'selected': 0, 'successful': 0, 'failed': 0, 'reports_fetched': 0, 'total_success_in_state': np.int64(20), 'total_errors_in_state': np.int64(0), 'total_pending_in_state': np.int64(9897)}


In [4]:
display(
    report_state_2[
        report_state_2[
            "organization_id"
        ].isin(
            test_office_ids
        )
    ][
        [
            "organization_id",
            "name",
            "status",
            "declared_count",
            "fetched_count",
            "count_mismatch",
            "snapshot_path",
            "error",
        ]
    ]
)

,organization_id,name,status,declared_count,fetched_count,count_mismatch,snapshot_path,error
2,1cf95166-a8c5-4363-a268-36ab0d276b87,БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,success,13.0,13.0,False,data\raw\report_lists\1cf95166-a8c5-4363-a268-...,None
6,fe8d5200-4709-4715-b316-6535ad348ec5,Київська обласна організація політичної партії...,success,0.0,0.0,False,data\raw\report_lists\fe8d5200-4709-4715-b316-...,None
8,c53485c4-fb53-423f-a978-a96b32f7436a,КИЇВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ...,success,12.0,12.0,False,data\raw\report_lists\c53485c4-fb53-423f-a978-...,None
10,f90ec33b-a925-4435-985c-d55e86d3b64b,"КРИМСЬКА РЕСПУБЛІКАНСЬКА ОРГАНІЗАЦІЯ ПАРТІЇ ""Р...",success,14.0,14.0,False,data\raw\report_lists\f90ec33b-a925-4435-985c-...,None
11,011c6969-8c97-4009-8fce-37e7b64576a4,"РЕНІЙСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПАРТІЇ ""РОДИНА""",success,14.0,14.0,False,data\raw\report_lists\011c6969-8c97-4009-8fce-...,None
12,942a39f5-97ac-4445-8150-9a4fb1b5e83c,ПОЛІТИЧНА ПАРТІЯ ОДЕСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПА...,success,14.0,14.0,False,data\raw\report_lists\942a39f5-97ac-4445-8150-...,None
13,de5de4e3-3510-42bf-b1f7-73964a681743,"ТАТАРБУНАРСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПАРТІЇ ""РОД...",success,14.0,14.0,False,data\raw\report_lists\de5de4e3-3510-42bf-b1f7-...,None
14,64ecdb81-ec90-4419-81e5-6fb4f98912fd,СЕВАСТОПОЛЬСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ ПА...,success,14.0,14.0,False,data\raw\report_lists\64ecdb81-ec90-4419-81e5-...,None
15,b18cbcd0-f14b-4790-b91a-f9bc5fc33c70,"ОДЕСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПАРТІЇ ""РОДИНА""",success,14.0,14.0,False,data\raw\report_lists\b18cbcd0-f14b-4790-b91a-...,None
16,6e0c8b9c-8427-4664-83c9-273b80962cba,ЛЬВІВСЬКА ОБЛАСНА ПАРТОРГАНІЗАЦІЯ ПРОГРЕСИВНО-...,success,13.0,13.0,False,data\raw\report_lists\6e0c8b9c-8427-4664-83c9-...,None


In [5]:
manifest_reports_full = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

central_batch_summary, report_state_after_parties = (
    run_report_discovery_batch(
        manifest_reports_full,
        entity_type="party",
        timeout=60,
        max_retries=4,
    )
)

print(central_batch_summary)

Report discovery:   0%|          | 0/321 [00:00<?, ?it/s]

{'selected': 321, 'successful': 321, 'failed': 0, 'reports_fetched': 3828, 'total_success_in_state': np.int64(341), 'total_errors_in_state': np.int64(0), 'total_pending_in_state': np.int64(9576)}


In [6]:
print(
    report_state_after_parties[
        "status"
    ].value_counts()
)

print()

print(
    pd.crosstab(
        report_state_after_parties[
            "entity_type"
        ],
        report_state_after_parties[
            "status"
        ],
    )
)

status
pending    9576
success     341
Name: count, dtype: int64

status       pending  success
entity_type                  
office          9576       20
party              0      321


In [7]:
report_count_anomalies = (
    report_state_after_parties[
        report_state_after_parties[
            "count_mismatch"
        ].fillna(False)
    ]
    .copy()
)

print(
    "Count mismatch organizations:",
    len(report_count_anomalies)
)

display(
    report_count_anomalies[
        [
            "organization_id",
            "entity_type",
            "name",
            "declared_count",
            "fetched_count",
            "count_difference",
        ]
    ]
)

Count mismatch organizations: 7


C:\Users\Igor\AppData\Local\Temp\ipykernel_14832\1121222632.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ].fillna(False)


,organization_id,entity_type,name,declared_count,fetched_count,count_difference
153,033a1696-ab48-4362-b579-699cc48aefbe,party,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДЯНСЬКЕ СУСПІЛЬСТВО»,18.0,17.0,1.0
853,9c883297-8c91-4bf6-8144-6304336d7178,party,ПОЛІТИЧНА ПАРТІЯ «ВЕЛИКА КРАЇНА»,22.0,21.0,1.0
4309,1388ae5f-2819-4153-84de-5463a956d3b4,party,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,15.0,14.0,1.0
4350,626abb7f-6b12-487f-85c6-b6604080c95c,party,ПОЛІТИЧНА ПАРТІЯ «НАШЕ МІСТО»,15.0,13.0,2.0
4943,87ebdf18-4c0c-404f-bb45-3b354c77a212,party,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «УК...,14.0,13.0,1.0
6009,40178e4b-d02d-4747-ae16-5e6385099f56,party,ПОЛІТИЧНА ПАРТІЯ «НАРОДНІ ІНІЦІАТИВИ ОЛЕКСАНДР...,19.0,18.0,1.0
6485,79cb7831-508b-466a-8db0-db9c5f4080b2,party,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ВОЛОДИМИРА БУРЯКА «ЄД...,22.0,21.0,1.0


In [8]:
office_batch_500, report_state_500 = (
    run_report_discovery_batch(
        manifest_reports_full,
        entity_type="office",
        limit=500,
        timeout=60,
        max_retries=4,
    )
)

print(office_batch_500)

Report discovery:   0%|          | 0/500 [00:00<?, ?it/s]

{'selected': 500, 'successful': 500, 'failed': 0, 'reports_fetched': 1733, 'total_success_in_state': np.int64(841), 'total_errors_in_state': np.int64(0), 'total_pending_in_state': np.int64(9076)}


In [9]:
print(
    pd.crosstab(
        report_state_500[
            "entity_type"
        ],
        report_state_500[
            "status"
        ],
    )
)

print()

office_errors = report_state_500[
    (
        report_state_500[
            "entity_type"
        ] == "office"
    )
    &
    (
        report_state_500[
            "status"
        ] == "error"
    )
]

print(
    "Office errors:",
    len(office_errors)
)

if len(office_errors):
    display(
        office_errors[
            [
                "organization_id",
                "name",
                "error",
            ]
        ]
    )

status       pending  success
entity_type                  
office          9076      520
party              0      321

Office errors: 0


In [10]:
office_full_run, report_state_full = (
    run_report_discovery_batch(
        manifest_reports_full,
        entity_type="office",
        timeout=60,
        max_retries=4,
    )
)

print(office_full_run)

Report discovery:   0%|          | 0/9076 [00:00<?, ?it/s]

PermissionError: [WinError 5] Access is denied: 'data\\interim\\state\\report_discovery_state.tmp.parquet' -> 'data\\interim\\state\\report_discovery_state.parquet'

In [11]:
from pathlib import Path
import pandas as pd

state_path = Path(
    "data/interim/state/"
    "report_discovery_state.parquet"
)

tmp_path = Path(
    "data/interim/state/"
    "report_discovery_state.tmp.parquet"
)

snapshot_dir = Path(
    "data/raw/report_lists"
)

print(
    "State exists:",
    state_path.exists()
)

print(
    "Temporary state exists:",
    tmp_path.exists()
)

print(
    "RAW report-list snapshots:",
    len(list(snapshot_dir.glob("*.json")))
)

if state_path.exists():

    state_before_recovery = pd.read_parquet(
        state_path
    )

    print()
    print("Saved state:")
    print(
        pd.crosstab(
            state_before_recovery[
                "entity_type"
            ],
            state_before_recovery[
                "status"
            ],
        )
    )

State exists: True
Temporary state exists: True
RAW report-list snapshots: 1383

Saved state:
status       pending  success
entity_type                  
office          8535     1061
party              0      321


In [12]:
%%writefile src/politdata/report_discovery.py

from datetime import datetime, timezone
from pathlib import Path
import json
import os
import time
import uuid

import pandas as pd
from tqdm.auto import tqdm

from .reports import (
    fetch_all_reports,
    reports_to_manifest,
    add_periodicity_flags,
)


DEFAULT_STATE_PATH = Path(
    "data/interim/state/"
    "report_discovery_state.parquet"
)

DEFAULT_SNAPSHOT_DIR = Path(
    "data/raw/report_lists"
)


def _write_json_atomic(path, data):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_name(
        f"{path.stem}.{uuid.uuid4().hex}.tmp"
        f"{path.suffix}"
    )

    try:

        with temp_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False,
                indent=2,
            )

        os.replace(
            temp_path,
            path,
        )

    finally:

        if temp_path.exists():

            try:
                temp_path.unlink()
            except OSError:
                pass


def save_report_discovery_state(
    state,
    state_path=DEFAULT_STATE_PATH,
    max_retries=20,
    retry_delay=0.25,
):
    """
    Safely persist discovery state.

    Windows can temporarily lock a Parquet file
    during replacement. We therefore retry the
    atomic replacement several times.
    """

    state_path = Path(
        state_path
    )

    state_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = state_path.with_name(
        f"{state_path.stem}."
        f"{uuid.uuid4().hex}.tmp"
        f"{state_path.suffix}"
    )

    state.to_parquet(
        temp_path,
        index=False,
    )

    try:

        for attempt in range(
            1,
            max_retries + 1,
        ):

            try:

                os.replace(
                    temp_path,
                    state_path,
                )

                return state_path

            except PermissionError:

                if attempt == max_retries:
                    raise

                time.sleep(
                    retry_delay
                    * attempt
                )

    finally:

        if temp_path.exists():

            try:
                temp_path.unlink()
            except OSError:
                pass


def _try_save_state(
    state,
    state_path,
):
    """
    State persistence failure must not destroy
    a long discovery run.

    RAW snapshots remain recoverable and will
    be reconciled on the next run.
    """

    try:

        save_report_discovery_state(
            state,
            state_path=state_path,
        )

        return True

    except PermissionError as exc:

        print()
        print(
            "WARNING: state checkpoint "
            "could not be written."
        )

        print(
            "RAW snapshots remain safe."
        )

        print(
            "Windows error:",
            repr(exc)
        )

        return False


def initialize_report_discovery_state(
    organization_manifest,
    state_path=DEFAULT_STATE_PATH,
):
    """
    Initialize or update discovery state while
    preserving existing statuses.
    """

    state_path = Path(
        state_path
    )

    manifest = (
        organization_manifest[
            [
                "organization_id",
                "root_party_id",
                "entity_type",
                "name",
            ]
        ]
        .copy()
    )

    if state_path.exists():

        old_state = pd.read_parquet(
            state_path
        )

        old_state = old_state.set_index(
            "organization_id"
        )

        rows = []

        for row in manifest.to_dict(
            orient="records"
        ):

            organization_id = (
                row["organization_id"]
            )

            if (
                organization_id
                in old_state.index
            ):

                old = (
                    old_state.loc[
                        organization_id
                    ].to_dict()
                )

                old.update({
                    "organization_id":
                        organization_id,

                    "root_party_id":
                        row[
                            "root_party_id"
                        ],

                    "entity_type":
                        row[
                            "entity_type"
                        ],

                    "name":
                        row["name"],
                })

                rows.append(
                    old
                )

            else:

                rows.append({
                    **row,
                    "status":
                        "pending",

                    "last_checked_at_utc":
                        None,

                    "declared_count":
                        None,

                    "fetched_count":
                        None,

                    "count_difference":
                        None,

                    "count_mismatch":
                        None,

                    "snapshot_path":
                        None,

                    "error":
                        None,
                })

        state = pd.DataFrame(
            rows
        )

    else:

        state = manifest.copy()

        state[
            "status"
        ] = "pending"

        state[
            "last_checked_at_utc"
        ] = None

        state[
            "declared_count"
        ] = None

        state[
            "fetched_count"
        ] = None

        state[
            "count_difference"
        ] = None

        state[
            "count_mismatch"
        ] = None

        state[
            "snapshot_path"
        ] = None

        state[
            "error"
        ] = None

    return state


def reconcile_state_from_snapshots(
    state,
    snapshot_dir=DEFAULT_SNAPSHOT_DIR,
):
    """
    Recover successful state from RAW snapshots.

    This handles the case where:
    1. API response was successfully saved to JSON,
    2. process failed before Parquet state was updated.

    Therefore a valid snapshot is authoritative evidence
    that discovery for that organization succeeded.
    """

    snapshot_dir = Path(
        snapshot_dir
    )

    state = state.copy()

    state = state.set_index(
        "organization_id"
    )

    recovered = 0

    for organization_id in state.index:

        if (
            state.at[
                organization_id,
                "status",
            ]
            == "success"
        ):
            continue

        snapshot_path = (
            snapshot_dir
            / f"{organization_id}.json"
        )

        if not snapshot_path.exists():
            continue

        try:

            with snapshot_path.open(
                "r",
                encoding="utf-8",
            ) as f:

                snapshot = json.load(
                    f
                )

            if (
                snapshot.get(
                    "organization_id"
                )
                != organization_id
            ):
                continue

            metadata = snapshot.get(
                "fetch_metadata",
                {}
            )

            reports = snapshot.get(
                "reports"
            )

            if not isinstance(
                reports,
                list,
            ):
                continue

            state.at[
                organization_id,
                "status",
            ] = "success"

            state.at[
                organization_id,
                "last_checked_at_utc",
            ] = snapshot.get(
                "retrieved_at_utc"
            )

            state.at[
                organization_id,
                "declared_count",
            ] = metadata.get(
                "declared_count"
            )

            state.at[
                organization_id,
                "fetched_count",
            ] = metadata.get(
                "fetched_count",
                len(reports),
            )

            state.at[
                organization_id,
                "count_difference",
            ] = metadata.get(
                "count_difference"
            )

            state.at[
                organization_id,
                "count_mismatch",
            ] = metadata.get(
                "count_mismatch"
            )

            state.at[
                organization_id,
                "snapshot_path",
            ] = str(
                snapshot_path
            )

            state.at[
                organization_id,
                "error",
            ] = None

            recovered += 1

        except Exception:
            # Invalid/incomplete snapshot is not
            # considered successful.
            continue

    return (
        state.reset_index(),
        recovered,
    )


def run_report_discovery_batch(
    organization_manifest,
    organization_ids=None,
    entity_type=None,
    limit=None,
    state_path=DEFAULT_STATE_PATH,
    snapshot_dir=DEFAULT_SNAPSHOT_DIR,
    timeout=60,
    max_retries=4,
    retry_errors=True,
    checkpoint_every=25,
):
    """
    Resumable report-list discovery.

    RAW snapshots are written per organization.

    Shared Parquet state is only a checkpoint layer;
    snapshots can recover progress after an interrupted
    or failed state write.
    """

    snapshot_dir = Path(
        snapshot_dir
    )

    snapshot_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    state = (
        initialize_report_discovery_state(
            organization_manifest,
            state_path=state_path,
        )
    )

    # Recover any snapshots written after
    # the last successful state checkpoint.
    (
        state,
        recovered_from_snapshots,
    ) = reconcile_state_from_snapshots(
        state,
        snapshot_dir=snapshot_dir,
    )

    if recovered_from_snapshots:

        print(
            "Recovered from RAW snapshots:",
            recovered_from_snapshots
        )

    # Try to checkpoint recovered state,
    # but do not abort if Windows locks it.
    _try_save_state(
        state,
        state_path,
    )

    candidates = state.copy()

    if organization_ids is not None:

        organization_ids = set(
            organization_ids
        )

        candidates = candidates[
            candidates[
                "organization_id"
            ].isin(
                organization_ids
            )
        ]

    if entity_type is not None:

        candidates = candidates[
            candidates[
                "entity_type"
            ]
            == entity_type
        ]

    candidates = candidates[
        candidates[
            "status"
        ]
        != "success"
    ]

    if not retry_errors:

        candidates = candidates[
            candidates[
                "status"
            ]
            != "error"
        ]

    if limit is not None:

        candidates = (
            candidates.head(
                limit
            )
        )

    selected_ids = (
        candidates[
            "organization_id"
        ].tolist()
    )

    state = state.set_index(
        "organization_id"
    )

    successful = 0
    failed = 0
    reports_fetched = 0
    attempts_since_checkpoint = 0

    for organization_id in tqdm(
        selected_ids,
        desc="Report discovery",
    ):

        retrieved_at = datetime.now(
            timezone.utc
        )

        try:

            (
                reports,
                metadata,
            ) = fetch_all_reports(
                organization_id,
                timeout=timeout,
                max_retries=max_retries,
                return_metadata=True,
            )

            snapshot_path = (
                snapshot_dir
                / f"{organization_id}.json"
            )

            snapshot = {
                "source":
                    "PolitData",

                "endpoint":
                    (
                        f"/party/"
                        f"{organization_id}"
                        f"/reports"
                    ),

                "organization_id":
                    organization_id,

                "retrieved_at_utc":
                    retrieved_at.isoformat(),

                "fetch_metadata":
                    metadata,

                "reports":
                    reports,
            }

            # RAW snapshot FIRST.
            _write_json_atomic(
                snapshot_path,
                snapshot,
            )

            state.at[
                organization_id,
                "status",
            ] = "success"

            state.at[
                organization_id,
                "last_checked_at_utc",
            ] = retrieved_at.isoformat()

            state.at[
                organization_id,
                "declared_count",
            ] = metadata[
                "declared_count"
            ]

            state.at[
                organization_id,
                "fetched_count",
            ] = metadata[
                "fetched_count"
            ]

            state.at[
                organization_id,
                "count_difference",
            ] = metadata[
                "count_difference"
            ]

            state.at[
                organization_id,
                "count_mismatch",
            ] = metadata[
                "count_mismatch"
            ]

            state.at[
                organization_id,
                "snapshot_path",
            ] = str(
                snapshot_path
            )

            state.at[
                organization_id,
                "error",
            ] = None

            successful += 1

            reports_fetched += len(
                reports
            )

        except Exception as exc:

            state.at[
                organization_id,
                "status",
            ] = "error"

            state.at[
                organization_id,
                "last_checked_at_utc",
            ] = retrieved_at.isoformat()

            state.at[
                organization_id,
                "error",
            ] = repr(
                exc
            )

            failed += 1

        attempts_since_checkpoint += 1

        # Do not rewrite the shared Parquet
        # thousands of times.
        if (
            attempts_since_checkpoint
            >= checkpoint_every
        ):

            current_state = (
                state.reset_index()
            )

            _try_save_state(
                current_state,
                state_path,
            )

            attempts_since_checkpoint = 0

    final_state = (
        state.reset_index()
    )

    # Final checkpoint.
    _try_save_state(
        final_state,
        state_path,
    )

    summary = {
        "selected":
            len(selected_ids),

        "successful":
            successful,

        "failed":
            failed,

        "reports_fetched":
            reports_fetched,

        "recovered_from_snapshots":
            recovered_from_snapshots,

        "total_success_in_state":
            int(
                (
                    final_state[
                        "status"
                    ]
                    == "success"
                ).sum()
            ),

        "total_errors_in_state":
            int(
                (
                    final_state[
                        "status"
                    ]
                    == "error"
                ).sum()
            ),

        "total_pending_in_state":
            int(
                (
                    final_state[
                        "status"
                    ]
                    == "pending"
                ).sum()
            ),
    }

    return (
        summary,
        final_state,
    )


def build_report_manifest_from_snapshots(
    organization_manifest,
    snapshot_dir=DEFAULT_SNAPSHOT_DIR,
):
    """
    Rebuild unified report manifest from saved
    per-organization RAW report-list snapshots.
    """

    snapshot_dir = Path(
        snapshot_dir
    )

    manifest_indexed = (
        organization_manifest
        .set_index(
            "organization_id"
        )
    )

    frames = []
    missing_organizations = []

    for snapshot_path in sorted(
        snapshot_dir.glob(
            "*.json"
        )
    ):

        with snapshot_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            snapshot = json.load(
                f
            )

        organization_id = (
            snapshot[
                "organization_id"
            ]
        )

        if (
            organization_id
            not in manifest_indexed.index
        ):

            missing_organizations.append(
                organization_id
            )

            continue

        reports = snapshot.get(
            "reports",
            []
        )

        if not reports:
            continue

        organization_row = (
            manifest_indexed.loc[
                organization_id
            ].to_dict()
        )

        organization_row[
            "organization_id"
        ] = organization_id

        frame = reports_to_manifest(
            organization_row,
            reports,
            discovered_at_utc=(
                snapshot.get(
                    "retrieved_at_utc"
                )
            ),
        )

        frames.append(
            frame
        )

    if frames:

        reports_df = pd.concat(
            frames,
            ignore_index=True,
        )

        reports_df = (
            add_periodicity_flags(
                reports_df
            )
        )

    else:

        reports_df = pd.DataFrame()

    qa = {
        "rows":
            len(reports_df),

        "unique_report_ids":
            (
                reports_df[
                    "report_id"
                ].nunique()
                if not reports_df.empty
                else 0
            ),

        "duplicate_report_ids":
            (
                reports_df[
                    "report_id"
                ].duplicated().sum()
                if not reports_df.empty
                else 0
            ),

        "missing_report_ids":
            (
                reports_df[
                    "report_id"
                ].isna().sum()
                if not reports_df.empty
                else 0
            ),

        "party_id_mismatches":
            (
                (
                    ~reports_df[
                        "party_id_matches_organization"
                    ]
                ).sum()
                if not reports_df.empty
                else 0
            ),

        "snapshot_orgs_not_in_manifest":
            len(
                missing_organizations
            ),
    }

    return (
        reports_df,
        qa,
    )

Overwriting src/politdata/report_discovery.py


In [2]:
manifest_reports_resume = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

recovery_summary, recovered_state = (
    run_report_discovery_batch(
        manifest_reports_resume,
        entity_type="office",
        limit=0,
        timeout=60,
        max_retries=4,
    )
)

print(recovery_summary)

print()

print(
    pd.crosstab(
        recovered_state[
            "entity_type"
        ],
        recovered_state[
            "status"
        ],
    )
)

Recovered from RAW snapshots: 1


Report discovery: 0it [00:00, ?it/s]

{'selected': 0, 'successful': 0, 'failed': 0, 'reports_fetched': 0, 'recovered_from_snapshots': 1, 'total_success_in_state': 1383, 'total_errors_in_state': 0, 'total_pending_in_state': 8534}

status       pending  success
entity_type                  
office          8534     1062
party              0      321


In [7]:
office_batch_next, report_state_next = (
    run_report_discovery_batch(
        manifest_reports_resume,
        entity_type="office",
        limit=1000,
        timeout=60,
        max_retries=4,
        checkpoint_every=25,
    )
)

print(office_batch_next)

print()

print(
    pd.crosstab(
        report_state_next[
            "entity_type"
        ],
        report_state_next[
            "status"
        ],
    )
)

Report discovery:   0%|          | 0/1000 [00:00<?, ?it/s]

{'selected': 1000, 'successful': 1000, 'failed': 0, 'reports_fetched': 7954, 'recovered_from_snapshots': 0, 'total_success_in_state': 6383, 'total_errors_in_state': 0, 'total_pending_in_state': 3534}

status       pending  success
entity_type                  
office          3534     6062
party              0      321


In [8]:
import time
import pandas as pd


BATCH_SIZE = 1000

while True:

    batch_summary, report_state_auto = (
        run_report_discovery_batch(
            manifest_reports_resume,
            entity_type="office",
            limit=BATCH_SIZE,
            timeout=60,
            max_retries=4,
            checkpoint_every=25,
        )
    )

    office_state = report_state_auto[
        report_state_auto["entity_type"] == "office"
    ]

    success_count = (
        office_state["status"] == "success"
    ).sum()

    pending_count = (
        office_state["status"] == "pending"
    ).sum()

    error_count = (
        office_state["status"] == "error"
    ).sum()

    print()
    print("=" * 60)
    print("BATCH FINISHED")
    print("Success:", success_count)
    print("Pending:", pending_count)
    print("Errors:", error_count)
    print("=" * 60)

    # Everything processed successfully
    if pending_count == 0 and error_count == 0:
        print()
        print("ALL OFFICES FINISHED.")
        break

    # No pending left, but some errors remain.
    # They will be retried automatically.
    if pending_count == 0 and error_count > 0:
        print()
        print(
            f"Retrying {error_count} failed organizations..."
        )

    # Small pause before the next batch
    time.sleep(5)

Report discovery:   0%|          | 0/1000 [00:00<?, ?it/s]


BATCH FINISHED
Success: 7062
Pending: 2534
Errors: 0


Report discovery:   0%|          | 0/1000 [00:00<?, ?it/s]


BATCH FINISHED
Success: 8062
Pending: 1534
Errors: 0


Report discovery:   0%|          | 0/1000 [00:00<?, ?it/s]


BATCH FINISHED
Success: 9062
Pending: 534
Errors: 0


Report discovery:   0%|          | 0/534 [00:00<?, ?it/s]


BATCH FINISHED
Success: 9596
Pending: 0
Errors: 0

ALL OFFICES FINISHED.


In [9]:
from pathlib import Path
import pandas as pd

manifest_final = pd.read_parquet(
    "data/interim/manifests/"
    "organization_manifest_committed.parquet"
)

report_state_final = pd.read_parquet(
    "data/interim/state/"
    "report_discovery_state.parquet"
)

print("DISCOVERY STATE")
print(
    pd.crosstab(
        report_state_final["entity_type"],
        report_state_final["status"],
    )
)

pending_count = (
    report_state_final["status"]
    == "pending"
).sum()

error_count = (
    report_state_final["status"]
    == "error"
).sum()

print()
print("Pending:", pending_count)
print("Errors:", error_count)

if pending_count == 0 and error_count == 0:

    (
        all_reports_manifest,
        all_reports_qa,
    ) = build_report_manifest_from_snapshots(
        manifest_final
    )

    output_dir = Path(
        "data/interim/reports"
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    all_reports_path = (
        output_dir
        / "all_reports_manifest.parquet"
    )

    all_reports_manifest.to_parquet(
        all_reports_path,
        index=False,
    )

    print()
    print("REPORT MANIFEST QA")
    print(all_reports_qa)

    print()
    print(
        "Total reports:",
        len(all_reports_manifest)
    )

    print(
        "Organizations with reports:",
        all_reports_manifest[
            "organization_id"
        ].nunique()
    )

    print()
    print("Entity types:")
    print(
        all_reports_manifest[
            "entity_type"
        ].value_counts()
    )

    print()
    print("Report types:")
    print(
        all_reports_manifest[
            "report_type"
        ].value_counts(
            dropna=False
        )
    )

    print()
    print("Schema versions:")
    print(
        all_reports_manifest[
            "schema_version"
        ].value_counts(
            dropna=False
        )
    )

    print()
    print("Quarter codes:")
    print(
        all_reports_manifest[
            "quarter"
        ]
        .value_counts(
            dropna=False
        )
        .sort_index()
    )

    print()
    print("Saved:")
    print(all_reports_path)

else:

    print()
    print(
        "Discovery is not complete yet."
    )

DISCOVERY STATE
status       success
entity_type         
office          9596
party            321

Pending: 0
Errors: 0


C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\report_discovery.py:909: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  reports_df = pd.concat(



REPORT MANIFEST QA
{'rows': 79004, 'unique_report_ids': 79004, 'duplicate_report_ids': np.int64(0), 'missing_report_ids': np.int64(0), 'party_id_mismatches': np.int64(0), 'snapshot_orgs_not_in_manifest': 0}

Total reports: 79004
Organizations with reports: 8516

Entity types:
entity_type
office    75176
party      3828
Name: count, dtype: int64

Report types:
report_type
main    79004
Name: count, dtype: int64

Schema versions:
schema_version
1    79004
Name: count, dtype: int64

Quarter codes:
quarter
1    19367
2    20494
3    13661
4    14418
5    11064
Name: count, dtype: int64

Saved:
data\interim\reports\all_reports_manifest.parquet


In [10]:
from pathlib import Path
import pandas as pd

reports_dir = Path(
    "data/interim/reports"
)

all_reports = pd.read_parquet(
    reports_dir
    / "all_reports_manifest.parquet"
)

# ---------------------------------
# ORGANIZATION × YEAR REPORTING
# ---------------------------------

organization_year_reporting = (
    all_reports[
        [
            "organization_id",
            "root_party_id",
            "entity_type",
            "year",
            "has_annual_report",
            "has_quarterly_reports",
            "annual_report_count",
            "quarterly_report_count",
            "report_count",
            "has_mixed_periodicity",
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "organization_id",
            "year",
        ]
    )
)

organization_year_reporting.to_parquet(
    reports_dir
    / "organization_year_reporting.parquet",
    index=False,
)


# ---------------------------------
# ANNUAL + QUARTERLY SAME YEAR
# ---------------------------------

mixed_periodicity_cases = (
    organization_year_reporting[
        organization_year_reporting[
            "has_mixed_periodicity"
        ]
    ]
    .copy()
)

mixed_periodicity_cases.to_parquet(
    reports_dir
    / "mixed_periodicity_cases.parquet",
    index=False,
)


# ---------------------------------
# MULTIPLE REPORTS SAME PERIOD
# ---------------------------------

report_period_duplicates = (
    all_reports
    .groupby(
        [
            "organization_id",
            "year",
            "quarter",
        ],
        dropna=False,
    )
    .agg(
        report_count=(
            "report_id",
            "count",
        ),
        report_types=(
            "report_type",
            lambda x: sorted(
                set(
                    str(v)
                    for v in x
                )
            ),
        ),
    )
    .reset_index()
)

report_period_duplicates = (
    report_period_duplicates[
        report_period_duplicates[
            "report_count"
        ] > 1
    ]
    .copy()
)

report_period_duplicates.to_parquet(
    reports_dir
    / "report_period_duplicates.parquet",
    index=False,
)


print(
    "Organization-years:",
    len(organization_year_reporting)
)

print(
    "Mixed organization-years:",
    len(mixed_periodicity_cases)
)

print(
    "Organizations with mixed periodicity:",
    mixed_periodicity_cases[
        "organization_id"
    ].nunique()
)

print(
    "Duplicate organization-periods:",
    len(report_period_duplicates)
)

Organization-years: 33400
Mixed organization-years: 487
Organizations with mixed periodicity: 440
Duplicate organization-periods: 137


In [11]:
if len(report_period_duplicates):

    display(
        report_period_duplicates
        .sort_values(
            [
                "report_count",
                "year",
            ],
            ascending=[
                False,
                True,
            ]
        )
        .head(50)
    )

else:

    print(
        "No organization-year-quarter duplicates."
    )

,organization_id,year,quarter,report_count,report_types
26676,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,2025,2,7,[main]
58653,bfdeea3c-8942-4caa-a8bd-edcd81753f48,2025,2,7,[main]
58966,c10e970a-e75e-40ab-abf0-3b56584807c8,2025,2,7,[main]
647,01feeadd-7267-436f-9971-16000e03486f,2025,2,5,[main]
2046,069312d2-6aec-4d10-b4ab-e63af8132db8,2025,2,5,[main]
2160,06e7599d-6409-4a11-9cf3-8faa45cf78f6,2025,2,5,[main]
5356,11473023-f857-432b-9f53-fa85883ff010,2025,2,5,[main]
30846,669739dd-a445-4e01-8c18-b6cd36bfe16b,2025,2,5,[main]
41575,88547f57-fd1b-4a77-a721-9f6a68f9d231,2025,2,5,[main]
63977,d0af1c66-e2e1-4c3f-b210-133940cab80b,2025,2,5,[main]


In [12]:
# ---------------------------------
# PROFILE MULTIPLE REPORT INSTANCES
# ---------------------------------

duplicate_distribution = (
    report_period_duplicates
    .groupby(
        [
            "year",
            "quarter",
        ],
        dropna=False,
    )
    .agg(
        duplicate_periods=(
            "organization_id",
            "count",
        ),
        max_reports_in_period=(
            "report_count",
            "max",
        ),
        total_report_instances=(
            "report_count",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "year",
            "quarter",
        ]
    )
)

display(duplicate_distribution)

,year,quarter,duplicate_periods,max_reports_in_period,total_report_instances
0,2023,5,1,2,2
1,2024,4,1,2,2
2,2025,1,2,2,4
3,2025,2,128,7,323
4,2025,3,5,2,10


In [13]:
largest_duplicate = (
    report_period_duplicates
    .sort_values(
        "report_count",
        ascending=False,
    )
    .iloc[0]
)

dup_org_id = (
    largest_duplicate[
        "organization_id"
    ]
)

dup_year = int(
    largest_duplicate["year"]
)

dup_quarter = int(
    largest_duplicate["quarter"]
)

print("Organization:", dup_org_id)
print("Year:", dup_year)
print("Quarter:", dup_quarter)
print(
    "Report instances:",
    largest_duplicate[
        "report_count"
    ]
)

duplicate_instances = (
    all_reports[
        (
            all_reports[
                "organization_id"
            ] == dup_org_id
        )
        &
        (
            all_reports["year"]
            == dup_year
        )
        &
        (
            all_reports["quarter"]
            == dup_quarter
        )
    ]
    .copy()
    .sort_values(
        "created_date"
    )
)

display(
    duplicate_instances[
        [
            "report_id",
            "organization_id",
            "entity_type",
            "year",
            "quarter",
            "report_type",
            "signed_date",
            "created_date",
            "special_status",
            "public_summary_generated_at",
        ]
    ]
)

Organization: 58a7e1e0-2f73-4b27-bd4c-c37b5518b1da
Year: 2025
Quarter: 2
Report instances: 7


,report_id,organization_id,entity_type,year,quarter,report_type,signed_date,created_date,special_status,public_summary_generated_at
26750,00085e3b-2753-429a-a8ae-29c63a2c7a2a,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,office,2025,2,main,None,2025-07-02 07:56:25.1357,None,None
26748,2e435d08-e718-48b2-8044-d3b29a6b36ec,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,office,2025,2,main,None,2025-07-02 07:58:59.944149,None,None
26753,e77d4376-cc57-450c-9de4-a4ad35d19110,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,office,2025,2,main,None,2025-07-02 09:57:35.327534,None,None
26745,9e143cf9-37bf-4d43-84fa-6d423ffba4da,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,office,2025,2,main,None,2025-07-14 12:55:34.633632,None,None
26751,e6fcb550-2880-4792-9db3-c1ad67185386,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,office,2025,2,main,None,2025-07-25 13:55:29.383372,None,None
26752,d62929d3-0021-439f-a94d-1a972cae3adb,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,office,2025,2,main,None,2025-07-25 14:23:55.913887,None,None
26757,00dbd4cb-96d7-497a-880b-2cf3bda509c6,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,office,2025,2,main,2025-07-27 11:29:13.95929,2025-07-26 18:47:41.826247,None,None


In [14]:
organization_info = (
    manifest_final[
        manifest_final[
            "organization_id"
        ] == dup_org_id
    ]
)

display(
    organization_info[
        [
            "organization_id",
            "root_party_id",
            "entity_type",
            "code",
            "name",
        ]
    ]
)

,organization_id,root_party_id,entity_type,code,name
4115,58a7e1e0-2f73-4b27-bd4c-c37b5518b1da,304fb45c-b6f2-471e-b01f-09e7c0e5ced2,office,33557105,АПОСТОЛІВСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...


In [15]:
from pathlib import Path
import json

snapshot_path = (
    Path("data/raw/report_lists")
    / f"{dup_org_id}.json"
)

with snapshot_path.open(
    "r",
    encoding="utf-8",
) as f:
    report_list_snapshot = json.load(f)

raw_duplicate_reports = [
    report
    for report
    in report_list_snapshot["reports"]
    if (
        report.get("year") == dup_year
        and
        report.get("quarter")
        == dup_quarter
    )
]

print(
    "RAW duplicate reports:",
    len(raw_duplicate_reports)
)

for report in sorted(
    raw_duplicate_reports,
    key=lambda x:
        x.get("created_date") or "",
):
    print()
    print("=" * 70)
    print("ID:", report.get("id"))
    print(
        "report_type:",
        report.get("report_type")
    )
    print(
        "signed_date:",
        report.get("signed_date")
    )
    print(
        "created_date:",
        report.get("created_date")
    )
    print(
        "special_status:",
        report.get("special_status")
    )

    public_summary = (
        report.get("public_summary")
        or {}
    )

    print(
        "summary generated:",
        public_summary.get(
            "generated_at"
        )
    )

RAW duplicate reports: 7

ID: 00085e3b-2753-429a-a8ae-29c63a2c7a2a
report_type: main
signed_date: None
created_date: 2025-07-02 07:56:25.1357
special_status: None
summary generated: None

ID: 2e435d08-e718-48b2-8044-d3b29a6b36ec
report_type: main
signed_date: None
created_date: 2025-07-02 07:58:59.944149
special_status: None
summary generated: None

ID: e77d4376-cc57-450c-9de4-a4ad35d19110
report_type: main
signed_date: None
created_date: 2025-07-02 09:57:35.327534
special_status: None
summary generated: None

ID: 9e143cf9-37bf-4d43-84fa-6d423ffba4da
report_type: main
signed_date: None
created_date: 2025-07-14 12:55:34.633632
special_status: None
summary generated: None

ID: e6fcb550-2880-4792-9db3-c1ad67185386
report_type: main
signed_date: None
created_date: 2025-07-25 13:55:29.383372
special_status: None
summary generated: None

ID: d62929d3-0021-439f-a94d-1a972cae3adb
report_type: main
signed_date: None
created_date: 2025-07-25 14:23:55.913887
special_status: None
summary generated

In [16]:
import pandas as pd


GROUP_KEYS = [
    "organization_id",
    "year",
    "quarter",
]


# ---------------------------------
# ALL REPORT INSTANCES THAT BELONG
# TO DUPLICATE REPORTING PERIODS
# ---------------------------------

duplicate_keys = (
    report_period_duplicates[
        GROUP_KEYS
    ]
    .drop_duplicates()
)

duplicate_instances = (
    all_reports
    .merge(
        duplicate_keys,
        on=GROUP_KEYS,
        how="inner",
    )
    .copy()
)

duplicate_instances[
    "is_signed"
] = duplicate_instances[
    "signed_date"
].notna()


# ---------------------------------
# GROUP-LEVEL STATISTICS
# ---------------------------------

duplicate_group_profile = (
    duplicate_instances
    .groupby(
        GROUP_KEYS,
        dropna=False,
    )
    .agg(
        report_instances=(
            "report_id",
            "size",
        ),
        signed_count=(
            "is_signed",
            "sum",
        ),
        first_created=(
            "created_date",
            "min",
        ),
        last_created=(
            "created_date",
            "max",
        ),
    )
    .reset_index()
)

duplicate_group_profile[
    "unsigned_count"
] = (
    duplicate_group_profile[
        "report_instances"
    ]
    -
    duplicate_group_profile[
        "signed_count"
    ]
)


# ---------------------------------
# FIND LATEST INSTANCE BY CREATED_DATE
# ---------------------------------

latest_instances = (
    duplicate_instances
    .sort_values(
        GROUP_KEYS
        + ["created_date"]
    )
    .groupby(
        GROUP_KEYS,
        dropna=False,
    )
    .tail(1)
    [
        GROUP_KEYS
        + [
            "report_id",
            "is_signed",
            "signed_date",
            "created_date",
        ]
    ]
    .rename(
        columns={
            "report_id":
                "latest_report_id",
            "is_signed":
                "latest_is_signed",
            "signed_date":
                "latest_signed_date",
            "created_date":
                "latest_created_date",
        }
    )
)

duplicate_group_profile = (
    duplicate_group_profile
    .merge(
        latest_instances,
        on=GROUP_KEYS,
        how="left",
    )
)


# ---------------------------------
# CLASSIFY DUPLICATE PATTERN
# ---------------------------------

def classify_duplicate_group(row):

    if row["signed_count"] == 0:
        return "no_signed_instance"

    if row["signed_count"] == 1:

        if row["latest_is_signed"]:
            return "one_signed_and_latest"

        return "one_signed_but_not_latest"

    return "multiple_signed_instances"


duplicate_group_profile[
    "duplicate_pattern"
] = duplicate_group_profile.apply(
    classify_duplicate_group,
    axis=1,
)


print("Duplicate reporting periods:")
print(
    len(
        duplicate_group_profile
    )
)

print()
print("PATTERNS")
print(
    duplicate_group_profile[
        "duplicate_pattern"
    ]
    .value_counts(
        dropna=False
    )
)

print()
print("SIGNED COUNT")
print(
    duplicate_group_profile[
        "signed_count"
    ]
    .value_counts()
    .sort_index()
)

print()
print(
    "Total report instances:",
    duplicate_group_profile[
        "report_instances"
    ].sum()
)

print(
    "Signed instances:",
    duplicate_group_profile[
        "signed_count"
    ].sum()
)

print(
    "Unsigned instances:",
    duplicate_group_profile[
        "unsigned_count"
    ].sum()
)

Duplicate reporting periods:
137

PATTERNS
duplicate_pattern
one_signed_and_latest        133
multiple_signed_instances      3
no_signed_instance             1
Name: count, dtype: int64

SIGNED COUNT
signed_count
0      1
1    133
2      3
Name: count, dtype: int64

Total report instances: 341
Signed instances: 139
Unsigned instances: 202


In [17]:
duplicate_anomalies = (
    duplicate_group_profile[
        duplicate_group_profile[
            "duplicate_pattern"
        ]
        != "one_signed_and_latest"
    ]
    .copy()
)

print(
    "Groups not matching "
    "'one signed and latest':",
    len(duplicate_anomalies)
)

display(
    duplicate_anomalies
    .sort_values(
        [
            "year",
            "quarter",
            "organization_id",
        ]
    )
    .head(100)
)

Groups not matching 'one signed and latest': 4


,organization_id,year,quarter,report_instances,signed_count,first_created,last_created,unsigned_count,latest_report_id,latest_is_signed,latest_signed_date,latest_created_date,duplicate_pattern
15,1388ae5f-2819-4153-84de-5463a956d3b4,2023,5,2,2,2024-03-24 10:27:46,2024-03-25 14:48:01,0,de3d0b30-eaa5-11ee-92b7-3b3b09128e69,True,2024-03-25 14:47:59,2024-03-25 14:48:01,multiple_signed_instances
109,c79788a0-19c4-46d8-9c58-0a57c7974dca,2024,4,2,2,2025-01-30 14:30:43,2025-02-09 19:51:46,0,855d9410-e533-11ef-9743-8991a53f54af,True,2025-02-07 11:13:34,2025-02-09 19:51:46,multiple_signed_instances
110,c79788a0-19c4-46d8-9c58-0a57c7974dca,2025,1,2,2,2025-05-06 14:56:58,2025-05-08 21:51:03,0,16693f50-27dc-11f0-8c75-21cbf53902ff,True,2025-05-03 08:04:49,2025-05-08 21:51:03,multiple_signed_instances
115,d0af1c66-e2e1-4c3f-b210-133940cab80b,2025,2,5,0,2025-07-24 14:57:07.539129,2025-07-29 14:20:39.408477,5,9f7c1a84-9816-4801-be34-52fd9139141c,False,None,2025-07-29 14:20:39.408477,no_signed_instance


In [18]:
# ---------------------------------
# SIGNATURE STATUS — FULL CORPUS
# ---------------------------------

all_reports_signature_profile = (
    all_reports
    .assign(
        is_signed=
            all_reports[
                "signed_date"
            ].notna()
    )
)

print("TOTAL REPORT INSTANCES")
print(len(all_reports_signature_profile))

print()
print("SIGNED STATUS")
print(
    all_reports_signature_profile[
        "is_signed"
    ].value_counts()
)

print()
print("SIGNED STATUS BY ENTITY TYPE")
print(
    pd.crosstab(
        all_reports_signature_profile[
            "entity_type"
        ],
        all_reports_signature_profile[
            "is_signed"
        ],
    )
)

print()
print("SIGNED STATUS BY YEAR")
print(
    pd.crosstab(
        all_reports_signature_profile[
            "year"
        ],
        all_reports_signature_profile[
            "is_signed"
        ],
    )
)

print()
print("SIGNED STATUS BY YEAR / QUARTER")
print(
    all_reports_signature_profile
    .groupby(
        [
            "year",
            "quarter",
        ]
    )
    .agg(
        report_instances=(
            "report_id",
            "size",
        ),
        signed=(
            "is_signed",
            "sum",
        ),
    )
    .assign(
        unsigned=lambda df:
            df["report_instances"]
            - df["signed"]
    )
)

TOTAL REPORT INSTANCES
79004

SIGNED STATUS
is_signed
True     78794
False      210
Name: count, dtype: int64

SIGNED STATUS BY ENTITY TYPE
is_signed    False  True 
entity_type              
office         210  74966
party            0   3828

SIGNED STATUS BY YEAR
is_signed  False  True 
year                   
2020           0      3
2021           0   9644
2022           0   4925
2023           0   4311
2024           0  20381
2025         210  25553
2026           0  13977

SIGNED STATUS BY YEAR / QUARTER
              report_instances  signed  unsigned
year quarter                                    
2020 2                       1       1         0
     3                       1       1         0
     4                       1       1         0
2021 1                    1586    1586         0
     2                    1901    1901         0
     3                    1578    1578         0
     4                    1344    1344         0
     5                    3235    3235     

In [19]:
# ---------------------------------
# SINGLETON UNSIGNED REPORT PERIODS
# ---------------------------------

period_counts = (
    all_reports
    .groupby(
        [
            "organization_id",
            "year",
            "quarter",
        ],
        dropna=False,
    )
    .agg(
        report_instance_count=(
            "report_id",
            "size",
        )
    )
    .reset_index()
)

reports_with_period_count = (
    all_reports_signature_profile
    .merge(
        period_counts,
        on=[
            "organization_id",
            "year",
            "quarter",
        ],
        how="left",
    )
)

singleton_unsigned = (
    reports_with_period_count[
        (
            reports_with_period_count[
                "report_instance_count"
            ] == 1
        )
        &
        (
            ~reports_with_period_count[
                "is_signed"
            ]
        )
    ]
    .copy()
)

print(
    "Singleton unsigned report periods:",
    len(singleton_unsigned)
)

print(
    "Organizations:",
    singleton_unsigned[
        "organization_id"
    ].nunique()
)

print()
print("By year:")
print(
    singleton_unsigned[
        "year"
    ]
    .value_counts()
    .sort_index()
)

print()
print("By quarter:")
print(
    singleton_unsigned[
        "quarter"
    ]
    .value_counts()
    .sort_index()
)

Singleton unsigned report periods: 8
Organizations: 8

By year:
year
2025    8
Name: count, dtype: int64

By quarter:
quarter
2    5
3    3
Name: count, dtype: int64


In [21]:
from pathlib import Path
import pandas as pd

reports_dir = Path(
    "data/interim/reports"
)

REPORT_KEY = [
    "organization_id",
    "year",
    "quarter",
]

reports_selection = (
    all_reports
    .copy()
)

reports_selection[
    "is_signed"
] = reports_selection[
    "signed_date"
].notna()


# ---------------------------------
# PERIOD-LEVEL PROFILE
# ---------------------------------

report_period_selection = (
    reports_selection
    .groupby(
        REPORT_KEY,
        dropna=False,
    )
    .agg(
        report_instance_count=(
            "report_id",
            "size",
        ),
        signed_count=(
            "is_signed",
            "sum",
        ),
    )
    .reset_index()
)

report_period_selection[
    "unsigned_count"
] = (
    report_period_selection[
        "report_instance_count"
    ]
    -
    report_period_selection[
        "signed_count"
    ]
)


def classify_period(row):

    if row["signed_count"] == 1:
        return "unique_signed"

    if row["signed_count"] > 1:
        return "multiple_signed"

    return "no_signed"


report_period_selection[
    "selection_status"
] = (
    report_period_selection.apply(
        classify_period,
        axis=1,
    )
)


# ---------------------------------
# FIND SELECTED REPORT
# ONLY WHEN EXACTLY ONE IS SIGNED
# ---------------------------------

unique_signed_ids = (
    reports_selection[
        reports_selection[
            "is_signed"
        ]
    ]
    .merge(
        report_period_selection[
            REPORT_KEY
            + [
                "selection_status",
            ]
        ],
        on=REPORT_KEY,
        how="inner",
    )
)

unique_signed_ids = (
    unique_signed_ids[
        unique_signed_ids[
            "selection_status"
        ]
        == "unique_signed"
    ]
    [
        REPORT_KEY
        + ["report_id"]
    ]
    .rename(
        columns={
            "report_id":
                "selected_report_id"
        }
    )
)

report_period_selection = (
    report_period_selection
    .merge(
        unique_signed_ids,
        on=REPORT_KEY,
        how="left",
    )
)


# ---------------------------------
# ADD PERIOD SELECTION BACK
# TO REPORT INSTANCES
# ---------------------------------

report_instances_selection = (
    reports_selection
    .merge(
        report_period_selection,
        on=REPORT_KEY,
        how="left",
    )
)

report_instances_selection[
    "is_selected_report"
] = (
    report_instances_selection[
        "report_id"
    ]
    ==
    report_instances_selection[
        "selected_report_id"
    ]
)


def classify_instance(row):

    status = row[
        "selection_status"
    ]

    if status == "unique_signed":

        if row["is_selected_report"]:
            return "selected"

        return "nonselected_unsigned_instance"

    if status == "multiple_signed":

        if row["is_signed"]:
            return "ambiguous_signed_candidate"

        return "nonselected_unsigned_instance"

    return "unsigned_unresolved"


report_instances_selection[
    "instance_selection_role"
] = (
    report_instances_selection.apply(
        classify_instance,
        axis=1,
    )
)


# ---------------------------------
# SAVE
# ---------------------------------

report_period_selection.to_parquet(
    reports_dir
    / "report_period_selection.parquet",
    index=False,
)

report_instances_selection.to_parquet(
    reports_dir
    / "report_instances_with_selection.parquet",
    index=False,
)


print("Logical reporting periods:")
print(
    len(report_period_selection)
)

print()
print("Selection status:")
print(
    report_period_selection[
        "selection_status"
    ]
    .value_counts()
)

print()
print("Selected report IDs:")
print(
    report_period_selection[
        "selected_report_id"
    ]
    .notna()
    .sum()
)

Logical reporting periods:
78800

Selection status:
selection_status
unique_signed      78788
no_signed              9
multiple_signed        3
Name: count, dtype: int64

Selected report IDs:
78788


In [22]:
unresolved_periods = (
    report_period_selection[
        report_period_selection[
            "selection_status"
        ]
        != "unique_signed"
    ]
    .copy()
)

unresolved_instances = (
    report_instances_selection[
        report_instances_selection[
            "selection_status"
        ]
        != "unique_signed"
    ]
    .merge(
        manifest_final[
            [
                "organization_id",
                "root_party_id",
                "entity_type",
                "code",
                "name",
            ]
        ],
        on=[
            "organization_id",
            "root_party_id",
            "entity_type",
        ],
        how="left",
    )
    .sort_values(
        [
            "selection_status",
            "organization_id",
            "year",
            "quarter",
            "created_date",
        ]
    )
)

print(
    "Unresolved periods:",
    len(unresolved_periods)
)

print(
    "Report instances involved:",
    len(unresolved_instances)
)

display(
    unresolved_instances[
        [
            "selection_status",
            "name",
            "code",
            "organization_id",
            "root_party_id",
            "year",
            "quarter",
            "report_id",
            "is_signed",
            "signed_date",
            "created_date",
            "special_status",
        ]
    ]
)

Unresolved periods: 12
Report instances involved: 19


,selection_status,name,code,organization_id,root_party_id,year,quarter,report_id,is_signed,signed_date,created_date,special_status
1,multiple_signed,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,21709406,1388ae5f-2819-4153-84de-5463a956d3b4,1388ae5f-2819-4153-84de-5463a956d3b4,2023,5,58d8edd0-e9b8-11ee-92b7-3b3b09128e69,True,2024-03-24 10:27:44,2024-03-24 10:27:46,None
0,multiple_signed,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,21709406,1388ae5f-2819-4153-84de-5463a956d3b4,1388ae5f-2819-4153-84de-5463a956d3b4,2023,5,de3d0b30-eaa5-11ee-92b7-3b3b09128e69,True,2024-03-25 14:47:59,2024-03-25 14:48:01,None
8,multiple_signed,ПОЛІТИЧНА ПАРТІЯ НОВОРОЗДІЛЬСЬКА МІСЬКА ОРГАНІ...,26057546,c79788a0-19c4-46d8-9c58-0a57c7974dca,171b4fa0-5b63-4c2f-982c-a11a2b5ecabf,2024,4,12273290-cdb8-11ef-ac2d-b9ec46358bc3,True,2025-01-08 14:00:23,2025-01-30 14:30:43,None
9,multiple_signed,ПОЛІТИЧНА ПАРТІЯ НОВОРОЗДІЛЬСЬКА МІСЬКА ОРГАНІ...,26057546,c79788a0-19c4-46d8-9c58-0a57c7974dca,171b4fa0-5b63-4c2f-982c-a11a2b5ecabf,2024,4,855d9410-e533-11ef-9743-8991a53f54af,True,2025-02-07 11:13:34,2025-02-09 19:51:46,None
7,multiple_signed,ПОЛІТИЧНА ПАРТІЯ НОВОРОЗДІЛЬСЬКА МІСЬКА ОРГАНІ...,26057546,c79788a0-19c4-46d8-9c58-0a57c7974dca,171b4fa0-5b63-4c2f-982c-a11a2b5ecabf,2025,1,d137cb30-1535-11f0-8f85-d95649015426,True,2025-04-09 14:29:26,2025-05-06 14:56:58,None
6,multiple_signed,ПОЛІТИЧНА ПАРТІЯ НОВОРОЗДІЛЬСЬКА МІСЬКА ОРГАНІ...,26057546,c79788a0-19c4-46d8-9c58-0a57c7974dca,171b4fa0-5b63-4c2f-982c-a11a2b5ecabf,2025,1,16693f50-27dc-11f0-8c75-21cbf53902ff,True,2025-05-03 08:04:49,2025-05-08 21:51:03,None
2,no_signed,Артемівська міська організація політичної парт...,25894197,1e93504a-3560-4597-8b71-e434c7030b90,1e52ae20-1244-42ac-927d-7e014c80fc14,2025,2,5287f407-4433-4325-b4a0-af76af441707,False,None,2025-07-03 12:40:23.210504,None
3,no_signed,Погребищенська районна організація Єдиний Центр,35322765,5f5d330e-1a62-432e-8d85-488af6e20ab1,91e8da6e-0219-4fc3-a7bb-37bfdb2a4ec2,2025,2,fff2968d-f24a-4a49-9013-8adc9c5048d3,False,None,2025-07-11 10:01:41.567272,None
4,no_signed,ХМЕЛЬНИЦЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ДЕМОКРАТИЧНОЇ ...,23651728,6a9e9bdb-8137-42ce-9e2b-371d4e8fb0ed,9b260a75-7212-49e8-8f1f-2bcc8a91da74,2025,2,86f9ea93-ba9c-42bf-9e35-e9dc8b60959d,False,None,2025-08-05 15:23:03.364496,None
5,no_signed,"ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПАРТІЇ ""ЗА ПРАВ...",36944630,8e9c72f5-6974-4fdb-a2ba-27ca9136ebb4,e8ebdbde-4e91-4232-9ef1-0e7fc2cc68da,2025,3,b23ac9a4-75c7-4b67-a3c6-7fd1f603a939,False,None,2025-10-01 08:55:24.712455,None


In [23]:
from pathlib import Path
import json
import time
import requests


probe_dir = Path(
    "data/raw/report_detail_probes/unresolved"
)

probe_dir.mkdir(
    parents=True,
    exist_ok=True,
)


unresolved_report_ids = (
    unresolved_instances[
        "report_id"
    ]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(
    "Reports to probe:",
    len(unresolved_report_ids)
)


probe_results = []

for i, report_id in enumerate(
    unresolved_report_ids,
    start=1,
):

    url = (
        f"{DEFAULT_BASE_URL}"
        f"/party/report/{report_id}"
    )

    try:

        response = requests.get(
            url,
            timeout=60,
        )

        status_code = (
            response.status_code
        )

        response.raise_for_status()

        payload = response.json()

        output_path = (
            probe_dir
            / f"{report_id}.json"
        )

        with output_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                payload,
                f,
                ensure_ascii=False,
                indent=2,
            )

        probe_results.append(
            {
                "report_id":
                    report_id,
                "status":
                    "success",
                "http_status":
                    status_code,
                "path":
                    str(output_path),
            }
        )

    except Exception as exc:

        probe_results.append(
            {
                "report_id":
                    report_id,
                "status":
                    "error",
                "http_status":
                    getattr(
                        response,
                        "status_code",
                        None,
                    )
                    if "response"
                    in locals()
                    else None,
                "error":
                    repr(exc),
            }
        )

    print(
        f"{i}/{len(unresolved_report_ids)}",
        report_id,
        probe_results[-1][
            "status"
        ],
    )

    time.sleep(0.3)

Reports to probe: 19
1/19 58d8edd0-e9b8-11ee-92b7-3b3b09128e69 success
2/19 de3d0b30-eaa5-11ee-92b7-3b3b09128e69 success
3/19 12273290-cdb8-11ef-ac2d-b9ec46358bc3 success
4/19 855d9410-e533-11ef-9743-8991a53f54af success
5/19 d137cb30-1535-11f0-8f85-d95649015426 success
6/19 16693f50-27dc-11f0-8c75-21cbf53902ff success
7/19 5287f407-4433-4325-b4a0-af76af441707 success
8/19 fff2968d-f24a-4a49-9013-8adc9c5048d3 success
9/19 86f9ea93-ba9c-42bf-9e35-e9dc8b60959d success
10/19 b23ac9a4-75c7-4b67-a3c6-7fd1f603a939 success
11/19 de8daf97-2452-4749-b6c2-dd00a5ce89e9 success
12/19 e7624d43-98a6-4b92-a764-712a1e419996 success
13/19 30579c6f-455e-4b1d-8c84-c5457eb4435e success
14/19 63d166df-8a62-45fb-9141-3242495d36b3 success
15/19 9f7c1a84-9816-4801-be34-52fd9139141c success
16/19 f76511e0-ead6-4aed-9041-56ab4c8f997c success
17/19 64ba85ab-74ed-4a6f-92e8-d5ea8caf66d9 success
18/19 9c57c568-bcea-49e8-9a7a-c307a5354cee success
19/19 315e99ab-039b-46f0-92e4-9c53142a3ace success


In [24]:
from collections import Counter


top_level_keys = Counter()

result_keys = Counter()

for report_id in unresolved_report_ids:

    path = (
        probe_dir
        / f"{report_id}.json"
    )

    if not path.exists():
        continue

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        payload = json.load(f)

    top_level_keys.update(
        payload.keys()
    )

    results = payload.get(
        "results"
    )

    if isinstance(
        results,
        dict,
    ):
        result_keys.update(
            results.keys()
        )


print(
    "TOP-LEVEL KEYS"
)

for key, count in (
    top_level_keys
    .most_common()
):
    print(
        count,
        key,
    )


print()
print(
    "RESULT KEYS"
)

for key, count in (
    result_keys
    .most_common()
):
    print(
        count,
        key,
    )

TOP-LEVEL KEYS
19 code
19 results

RESULT KEYS
19 id
19 schema_version
19 report_type
19 year
19 quarter
19 party_id
19 is_party_office
19 signed_date
19 created_date
19 signatory_id
19 special_status
19 public_summary
19 head_info
19 employees_by_civil_contract
19 employees_by_employment_contract
19 organizations
19 regional_offices
19 properties
19 payment_info
19 obligations
19 conclusion


In [25]:
example_report_id = (
    unresolved_instances[
        unresolved_instances[
            "selection_status"
        ]
        == "multiple_signed"
    ]
    .iloc[0][
        "report_id"
    ]
)

example_path = (
    probe_dir
    / f"{example_report_id}.json"
)

with example_path.open(
    "r",
    encoding="utf-8",
) as f:
    example_detail = (
        json.load(f)
    )

print(
    json.dumps(
        example_detail,
        ensure_ascii=False,
        indent=2,
    )[:20000]
)

{
  "code": 0,
  "results": {
    "id": "58d8edd0-e9b8-11ee-92b7-3b3b09128e69",
    "schema_version": 1,
    "report_type": "main",
    "year": 2023,
    "quarter": 5,
    "party_id": "1388ae5f-2819-4153-84de-5463a956d3b4",
    "is_party_office": false,
    "signed_date": "2024-03-24 10:27:44",
    "created_date": "2024-03-24 10:27:46",
    "signatory_id": null,
    "special_status": null,
    "public_summary": {
      "v": 2,
      "expenses": {
        "refunds_of_inflows": {
          "sum": 0,
          "row_count": 0
        },
        "payments_other_accounts": {
          "sum": 0,
          "row_count": 0
        },
        "other_refunds_and_transfers": {
          "sum": 0,
          "row_count": 0
        },
        "payments_separate_budget_account": {
          "sum": 0,
          "row_count": 0
        }
      },
      "receipts": {
        "other_receipts": {
          "sum": 0,
          "row_count": 0
        },
        "other_contributions": {
          "sum": 5400,
 

In [26]:
from pathlib import Path
import json
import pandas as pd

probe_dir = Path(
    "data/raw/report_detail_probes/unresolved"
)

selection = pd.read_parquet(
    "data/interim/reports/"
    "report_instances_with_selection.parquet"
)

unresolved = selection[
    selection["selection_status"]
    != "unique_signed"
].copy()


def safe_len(value):
    return len(value) if isinstance(value, list) else 0


rows = []

for path in probe_dir.glob("*.json"):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        payload = json.load(f)

    r = payload.get("results", {})

    payment_info = (
        r.get("payment_info")
        or {}
    )

    incoming = (
        payment_info.get("incoming")
        or {}
    )

    outgoing = (
        payment_info.get("outgoing")
        or {}
    )

    properties = (
        r.get("properties")
        or {}
    )

    rows.append(
        {
            "report_id":
                r.get("id"),

            "year":
                r.get("year"),

            "quarter":
                r.get("quarter"),

            "signed_date":
                r.get("signed_date"),

            "created_date":
                r.get("created_date"),

            "conclusion":
                r.get("conclusion"),

            "monetary_contributions":
                safe_len(
                    incoming.get(
                        "monetary_contributions"
                    )
                ),

            "other_contributions":
                safe_len(
                    incoming.get(
                        "other_contributions"
                    )
                ),

            "state_funding":
                safe_len(
                    incoming.get(
                        "state_funding"
                    )
                ),

            "other_incomes":
                safe_len(
                    incoming.get(
                        "other_incomes"
                    )
                ),

            "budget_expenses":
                safe_len(
                    outgoing.get(
                        "budget_expenses"
                    )
                ),

            "outgoing_expenses":
                safe_len(
                    outgoing.get(
                        "outgoing_expenses"
                    )
                ),

            "return_expenses":
                safe_len(
                    outgoing.get(
                        "return_expenses"
                    )
                ),

            "transfer_expenses":
                safe_len(
                    outgoing.get(
                        "transfer_expenses"
                    )
                ),

            "real_estate":
                safe_len(
                    properties.get(
                        "property_object"
                    )
                ),

            "money":
                safe_len(
                    properties.get(
                        "property_moneys"
                    )
                ),

            "movable":
                safe_len(
                    properties.get(
                        "property_movable"
                    )
                ),

            "transport":
                safe_len(
                    properties.get(
                        "property_transport"
                    )
                ),

            "securities":
                safe_len(
                    properties.get(
                        "property_paper"
                    )
                ),

            "intangible":
                safe_len(
                    properties.get(
                        "property_intangible_asset"
                    )
                ),

            "obligations":
                safe_len(
                    r.get("obligations")
                ),
        }
    )


detail_profile = pd.DataFrame(rows)

detail_profile = (
    detail_profile
    .merge(
        unresolved[
            [
                "report_id",
                "organization_id",
                "selection_status",
                "is_signed",
            ]
        ],
        on="report_id",
        how="left",
    )
    .sort_values(
        [
            "selection_status",
            "organization_id",
            "year",
            "quarter",
            "created_date",
        ]
    )
)

display(detail_profile)

,report_id,year,quarter,signed_date,created_date,conclusion,monetary_contributions,other_contributions,state_funding,other_incomes,...,real_estate,money,movable,transport,securities,intangible,obligations,organization_id,selection_status,is_signed
5,58d8edd0-e9b8-11ee-92b7-3b3b09128e69,2023,5,2024-03-24 10:27:44,2024-03-24 10:27:46,exist,0,2,0,0,...,1,0,0,0,0,0,0,1388ae5f-2819-4153-84de-5463a956d3b4,multiple_signed,True
14,de3d0b30-eaa5-11ee-92b7-3b3b09128e69,2023,5,2024-03-25 14:47:59,2024-03-25 14:48:01,exist,0,2,0,0,...,1,0,0,0,0,0,0,1388ae5f-2819-4153-84de-5463a956d3b4,multiple_signed,True
0,12273290-cdb8-11ef-ac2d-b9ec46358bc3,2024,4,2025-01-08 14:00:23,2025-01-30 14:30:43,missing,0,0,0,0,...,0,0,0,0,0,0,0,c79788a0-19c4-46d8-9c58-0a57c7974dca,multiple_signed,True
8,855d9410-e533-11ef-9743-8991a53f54af,2024,4,2025-02-07 11:13:34,2025-02-09 19:51:46,missing,0,0,0,0,...,0,0,0,0,0,0,0,c79788a0-19c4-46d8-9c58-0a57c7974dca,multiple_signed,True
13,d137cb30-1535-11f0-8f85-d95649015426,2025,1,2025-04-09 14:29:26,2025-05-06 14:56:58,missing,0,0,0,0,...,0,0,0,0,0,0,0,c79788a0-19c4-46d8-9c58-0a57c7974dca,multiple_signed,True
1,16693f50-27dc-11f0-8c75-21cbf53902ff,2025,1,2025-05-03 08:04:49,2025-05-08 21:51:03,missing,0,0,0,0,...,0,0,0,0,0,0,0,c79788a0-19c4-46d8-9c58-0a57c7974dca,multiple_signed,True
4,5287f407-4433-4325-b4a0-af76af441707,2025,2,None,2025-07-03 12:40:23.210504,missing,0,0,0,0,...,0,0,0,0,0,0,0,1e93504a-3560-4597-8b71-e434c7030b90,no_signed,False
18,fff2968d-f24a-4a49-9013-8adc9c5048d3,2025,2,None,2025-07-11 10:01:41.567272,missing,0,0,0,0,...,0,0,0,0,0,0,0,5f5d330e-1a62-432e-8d85-488af6e20ab1,no_signed,False
9,86f9ea93-ba9c-42bf-9e35-e9dc8b60959d,2025,2,None,2025-08-05 15:23:03.364496,missing,0,0,0,0,...,0,0,0,0,0,0,0,6a9e9bdb-8137-42ce-9e2b-371d4e8fb0ed,no_signed,False
12,b23ac9a4-75c7-4b67-a3c6-7fd1f603a939,2025,3,None,2025-10-01 08:55:24.712455,missing,0,0,0,0,...,0,0,0,0,0,0,0,8e9c72f5-6974-4fdb-a2ba-27ca9136ebb4,no_signed,False


In [27]:
comparison_rows = []

for path in probe_dir.glob("*.json"):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        payload = json.load(f)

    r = payload["results"]

    payment_info = (
        r.get("payment_info")
        or {}
    )

    incoming = (
        payment_info.get("incoming")
        or {}
    )

    outgoing = (
        payment_info.get("outgoing")
        or {}
    )

    summary = (
        r.get("public_summary")
        or {}
    )

    receipts = (
        summary.get("receipts")
        or {}
    )

    expenses = (
        summary.get("expenses")
        or {}
    )


    def summary_count(
        section,
        key,
        count_field,
    ):
        value = (
            section.get(key)
            or {}
        )

        return value.get(
            count_field
        )


    comparison_rows.append(
        {
            "report_id":
                r["id"],

            # Incoming
            "monetary_rows":
                safe_len(
                    incoming.get(
                        "monetary_contributions"
                    )
                ),

            "monetary_summary":
                summary_count(
                    receipts,
                    "monetary_contributions",
                    "operation_count",
                ),

            "other_contribution_rows":
                safe_len(
                    incoming.get(
                        "other_contributions"
                    )
                ),

            "other_contribution_summary":
                summary_count(
                    receipts,
                    "other_contributions",
                    "operation_count",
                ),

            "state_funding_rows":
                safe_len(
                    incoming.get(
                        "state_funding"
                    )
                ),

            "state_funding_summary":
                summary_count(
                    receipts,
                    "state_budget_funding",
                    "row_count",
                ),

            "other_income_rows":
                safe_len(
                    incoming.get(
                        "other_incomes"
                    )
                ),

            "other_income_summary":
                summary_count(
                    receipts,
                    "other_receipts",
                    "row_count",
                ),

            # Outgoing
            "budget_expense_rows":
                safe_len(
                    outgoing.get(
                        "budget_expenses"
                    )
                ),

            "budget_expense_summary":
                summary_count(
                    expenses,
                    "payments_separate_budget_account",
                    "row_count",
                ),

            "outgoing_expense_rows":
                safe_len(
                    outgoing.get(
                        "outgoing_expenses"
                    )
                ),

            "outgoing_expense_summary":
                summary_count(
                    expenses,
                    "payments_other_accounts",
                    "row_count",
                ),

            "return_expense_rows":
                safe_len(
                    outgoing.get(
                        "return_expenses"
                    )
                ),

            "return_expense_summary":
                summary_count(
                    expenses,
                    "refunds_of_inflows",
                    "row_count",
                ),

            "transfer_expense_rows":
                safe_len(
                    outgoing.get(
                        "transfer_expenses"
                    )
                ),

            "transfer_expense_summary":
                summary_count(
                    expenses,
                    "other_refunds_and_transfers",
                    "row_count",
                ),
        }
    )


payment_completeness = (
    pd.DataFrame(
        comparison_rows
    )
)

pairs = [
    (
        "monetary_rows",
        "monetary_summary",
    ),
    (
        "other_contribution_rows",
        "other_contribution_summary",
    ),
    (
        "state_funding_rows",
        "state_funding_summary",
    ),
    (
        "other_income_rows",
        "other_income_summary",
    ),
    (
        "budget_expense_rows",
        "budget_expense_summary",
    ),
    (
        "outgoing_expense_rows",
        "outgoing_expense_summary",
    ),
    (
        "return_expense_rows",
        "return_expense_summary",
    ),
    (
        "transfer_expense_rows",
        "transfer_expense_summary",
    ),
]


for rows_col, summary_col in pairs:

    payment_completeness[
        rows_col + "_matches"
    ] = (
        payment_completeness[
            summary_col
        ].isna()
        |
        (
            payment_completeness[
                rows_col
            ]
            ==
            payment_completeness[
                summary_col
            ]
        )
    )


match_cols = [
    col
    for col
    in payment_completeness.columns
    if col.endswith("_matches")
]

payment_completeness[
    "all_counts_match"
] = (
    payment_completeness[
        match_cols
    ]
    .all(axis=1)
)


print(
    "Reports checked:",
    len(payment_completeness)
)

print()

print(
    payment_completeness[
        "all_counts_match"
    ].value_counts(
        dropna=False
    )
)

display(
    payment_completeness[
        ~payment_completeness[
            "all_counts_match"
        ]
    ]
)

Reports checked: 19

all_counts_match
True    19
Name: count, dtype: int64


,report_id,monetary_rows,monetary_summary,other_contribution_rows,other_contribution_summary,state_funding_rows,state_funding_summary,other_income_rows,other_income_summary,budget_expense_rows,...,transfer_expense_summary,monetary_rows_matches,other_contribution_rows_matches,state_funding_rows_matches,other_income_rows_matches,budget_expense_rows_matches,outgoing_expense_rows_matches,return_expense_rows_matches,transfer_expense_rows_matches,all_counts_match


In [28]:
import hashlib


TECHNICAL_KEYS = {
    "id",
    "report_id",
    "created_at",
    "updated_at",
}


CONTENT_SECTIONS = [
    "head_info",
    "employees_by_civil_contract",
    "employees_by_employment_contract",
    "organizations",
    "regional_offices",
    "properties",
    "payment_info",
    "obligations",
    "conclusion",
]


def normalize_for_comparison(value):

    if isinstance(value, dict):

        return {
            key:
                normalize_for_comparison(
                    child
                )
            for key, child
            in sorted(
                value.items()
            )
            if key
            not in TECHNICAL_KEYS
        }

    if isinstance(value, list):

        normalized = [
            normalize_for_comparison(
                item
            )
            for item in value
        ]

        # Sort deterministically
        return sorted(
            normalized,
            key=lambda x:
                json.dumps(
                    x,
                    ensure_ascii=False,
                    sort_keys=True,
                    default=str,
                )
        )

    return value


def section_hash(value):

    normalized = (
        normalize_for_comparison(
            value
        )
    )

    encoded = json.dumps(
        normalized,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(
        encoded
    ).hexdigest()


multiple_signed = (
    unresolved[
        unresolved[
            "selection_status"
        ]
        == "multiple_signed"
    ]
    .copy()
)


comparison = []

for (
    organization_id,
    year,
    quarter
), group in multiple_signed.groupby(
    [
        "organization_id",
        "year",
        "quarter",
    ]
):

    report_ids = (
        group[
            "report_id"
        ]
        .tolist()
    )

    payloads = {}

    for report_id in report_ids:

        path = (
            probe_dir
            / f"{report_id}.json"
        )

        with path.open(
            "r",
            encoding="utf-8",
        ) as f:

            payloads[
                report_id
            ] = (
                json.load(f)[
                    "results"
                ]
            )

    hashes = {}

    for report_id, result in (
        payloads.items()
    ):

        hashes[
            report_id
        ] = {
            section:
                section_hash(
                    result.get(
                        section
                    )
                )
            for section
            in CONTENT_SECTIONS
        }

    ids = list(
        hashes.keys()
    )

    first_id = ids[0]
    second_id = ids[1]

    row = {
        "organization_id":
            organization_id,

        "year":
            year,

        "quarter":
            quarter,

        "report_1":
            first_id,

        "report_2":
            second_id,
    }

    for section in CONTENT_SECTIONS:

        row[
            section + "_same"
        ] = (
            hashes[first_id][
                section
            ]
            ==
            hashes[second_id][
                section
            ]
        )

    comparison.append(row)


signed_pair_comparison = (
    pd.DataFrame(comparison)
)

display(
    signed_pair_comparison
)

,organization_id,year,quarter,report_1,report_2,head_info_same,employees_by_civil_contract_same,employees_by_employment_contract_same,organizations_same,regional_offices_same,properties_same,payment_info_same,obligations_same,conclusion_same
0,1388ae5f-2819-4153-84de-5463a956d3b4,2023,5,de3d0b30-eaa5-11ee-92b7-3b3b09128e69,58d8edd0-e9b8-11ee-92b7-3b3b09128e69,True,True,True,True,False,False,False,True,True
1,c79788a0-19c4-46d8-9c58-0a57c7974dca,2024,4,12273290-cdb8-11ef-ac2d-b9ec46358bc3,855d9410-e533-11ef-9743-8991a53f54af,False,True,True,True,True,True,True,True,True
2,c79788a0-19c4-46d8-9c58-0a57c7974dca,2025,1,16693f50-27dc-11f0-8c75-21cbf53902ff,d137cb30-1535-11f0-8f85-d95649015426,False,True,True,True,True,True,True,True,True


In [29]:
from pathlib import Path
import json
import pandas as pd

probe_dir = Path(
    "data/raw/report_detail_probes/unresolved"
)

TECHNICAL_KEYS = {
    "id",
    "report_id",
    "created_at",
    "updated_at",
}

CONTENT_SECTIONS = [
    "head_info",
    "employees_by_civil_contract",
    "employees_by_employment_contract",
    "organizations",
    "regional_offices",
    "properties",
    "payment_info",
    "obligations",
    "conclusion",
]


def normalize_content(value):

    if isinstance(value, dict):

        return {
            key: normalize_content(child)
            for key, child
            in sorted(value.items())
            if key not in TECHNICAL_KEYS
        }

    if isinstance(value, list):

        values = [
            normalize_content(item)
            for item in value
        ]

        return sorted(
            values,
            key=lambda x: json.dumps(
                x,
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            )
        )

    return value


multiple_signed_groups = (
    report_instances_selection[
        report_instances_selection[
            "selection_status"
        ] == "multiple_signed"
    ]
    .copy()
)

multiple_signed_groups[
    "signed_date"
] = pd.to_datetime(
    multiple_signed_groups[
        "signed_date"
    ],
    errors="coerce",
)

for (
    organization_id,
    year,
    quarter
), group in multiple_signed_groups.groupby(
    [
        "organization_id",
        "year",
        "quarter",
    ]
):

    group = group.sort_values(
        [
            "signed_date",
            "created_date",
        ]
    )

    old_id = group.iloc[0]["report_id"]
    new_id = group.iloc[-1]["report_id"]

    with (
        probe_dir
        / f"{old_id}.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:
        old_report = json.load(f)["results"]

    with (
        probe_dir
        / f"{new_id}.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:
        new_report = json.load(f)["results"]

    print()
    print("=" * 80)
    print(
        organization_id,
        year,
        quarter,
    )
    print("=" * 80)

    print(
        "OLDER:",
        old_id,
        old_report.get("signed_date"),
    )

    print(
        "NEWER:",
        new_id,
        new_report.get("signed_date"),
    )

    changed_sections = []

    for section in CONTENT_SECTIONS:

        old_value = normalize_content(
            old_report.get(section)
        )

        new_value = normalize_content(
            new_report.get(section)
        )

        if old_value != new_value:
            changed_sections.append(
                section
            )

            print()
            print(
                f"CHANGED SECTION: {section}"
            )

            print()
            print("OLD:")
            print(
                json.dumps(
                    old_value,
                    ensure_ascii=False,
                    indent=2,
                    default=str,
                )[:15000]
            )

            print()
            print("NEW:")
            print(
                json.dumps(
                    new_value,
                    ensure_ascii=False,
                    indent=2,
                    default=str,
                )[:15000]
            )

    print()
    print(
        "Changed sections:",
        changed_sections,
    )


1388ae5f-2819-4153-84de-5463a956d3b4 2023 5
OLDER: 58d8edd0-e9b8-11ee-92b7-3b3b09128e69 2024-03-24 10:27:44
NEWER: de3d0b30-eaa5-11ee-92b7-3b3b09128e69 2024-03-25 14:47:59

CHANGED SECTION: regional_offices

OLD:
[]

NEW:
[
  {
    "code": "25786987",
    "name": "ПОЛІТИЧНА ПАРТІЯ СОСНИЦЬКА РАЙОННА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОЇ ПАРТІЇ \"НОВА СИЛА\" "
  },
  {
    "code": "25863653",
    "name": "ПОЛІТИЧНА ПАРТІЯ ХАРКІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОЇ ПАРТІЇ \"НОВА СИЛА\" "
  },
  {
    "code": "25919849",
    "name": "ПОЛІТИЧНА ПАРТІЯ ЗАПОРІЗЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОЇ ПАРТІЇ \"НОВА СИЛА\" "
  },
  {
    "code": "26028734",
    "name": "ПОЛІТИЧНА ПАРТІЯ МИКОЛАЇВСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОЇ ПАРТІЇ \"НОВА СИЛА\" "
  },
  {
    "code": "26030079",
    "name": "ПОЛІТИЧНА ПАРТІЯ ОЧАКІВСЬКА РАЙОННА ПАРТІЙНА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОЇ ПАРТІЇ \"НОВА СИЛА\" "
  },
  {
    "code": "26033847",
    "name": "ПОЛІТИЧНА ПАРТІЯ ГОРОДНЯНСЬКА РАЙОННА ПАРТІЙНА ОРГАНІЗА

In [31]:
from pathlib import Path
import pandas as pd

reports_dir = Path(
    "data/interim/reports"
)

reports = pd.read_parquet(
    reports_dir
    / "all_reports_manifest.parquet"
)

REPORT_KEY = [
    "organization_id",
    "year",
    "quarter",
]


# ---------------------------------
# ROBUST DATETIME PARSING
# ---------------------------------

reports["signed_date_dt"] = pd.to_datetime(
    reports["signed_date"],
    format="mixed",
    errors="coerce",
)

reports["created_date_dt"] = pd.to_datetime(
    reports["created_date"],
    format="mixed",
    errors="coerce",
)


# ---------------------------------
# SANITY CHECK BEFORE SELECTION
# ---------------------------------

reports["is_signed"] = (
    reports["signed_date"].notna()
)

signed_source_count = int(
    reports["signed_date"]
    .notna()
    .sum()
)

signed_parsed_count = int(
    reports["signed_date_dt"]
    .notna()
    .sum()
)

print(
    "Signed in source:",
    signed_source_count,
)

print(
    "Signed successfully parsed:",
    signed_parsed_count,
)

if (
    signed_source_count
    != signed_parsed_count
):
    raise ValueError(
        "Some non-null signed_date values "
        "failed datetime parsing."
    )


# ---------------------------------
# PERIOD PROFILE
# ---------------------------------

period_selection = (
    reports
    .groupby(
        REPORT_KEY,
        dropna=False,
    )
    .agg(
        report_instance_count=(
            "report_id",
            "size",
        ),
        signed_instance_count=(
            "is_signed",
            "sum",
        ),
    )
    .reset_index()
)

period_selection[
    "unsigned_instance_count"
] = (
    period_selection[
        "report_instance_count"
    ]
    -
    period_selection[
        "signed_instance_count"
    ]
)


# ---------------------------------
# SELECT LATEST SIGNED INSTANCE
# ---------------------------------

signed_reports = (
    reports[
        reports["is_signed"]
    ]
    .copy()
)

signed_reports = (
    signed_reports
    .sort_values(
        REPORT_KEY
        + [
            "signed_date_dt",
            "created_date_dt",
            "report_id",
        ]
    )
)

selected_signed = (
    signed_reports
    .groupby(
        REPORT_KEY,
        dropna=False,
    )
    .tail(1)
    [
        REPORT_KEY
        + [
            "report_id",
            "signed_date_dt",
            "created_date_dt",
        ]
    ]
    .rename(
        columns={
            "report_id":
                "selected_report_id",
            "signed_date_dt":
                "selected_signed_date",
            "created_date_dt":
                "selected_created_date",
        }
    )
)

period_selection = (
    period_selection
    .merge(
        selected_signed,
        on=REPORT_KEY,
        how="left",
    )
)


# ---------------------------------
# SELECTION METHOD
# ---------------------------------

def classify_selection(row):

    n = int(
        row["signed_instance_count"]
    )

    if n == 0:
        return "no_signed_report"

    if n == 1:
        return "unique_signed"

    return "latest_of_multiple_signed"


period_selection[
    "selection_method"
] = (
    period_selection.apply(
        classify_selection,
        axis=1,
    )
)


# ---------------------------------
# ADD SELECTION BACK TO INSTANCES
# ---------------------------------

report_instances_final = (
    reports
    .merge(
        period_selection[
            REPORT_KEY
            + [
                "report_instance_count",
                "signed_instance_count",
                "unsigned_instance_count",
                "selected_report_id",
                "selection_method",
            ]
        ],
        on=REPORT_KEY,
        how="left",
    )
)

report_instances_final[
    "is_selected_report"
] = (
    report_instances_final[
        "report_id"
    ]
    ==
    report_instances_final[
        "selected_report_id"
    ]
)


# ---------------------------------
# SELECTED ANALYTICAL REPORTS
# ---------------------------------

selected_reports = (
    report_instances_final[
        report_instances_final[
            "is_selected_report"
        ]
    ]
    .copy()
)


# ---------------------------------
# FINAL QA
# ---------------------------------

print()
print(
    "Logical reporting periods:",
    len(period_selection)
)

print()
print("Selection methods:")

print(
    period_selection[
        "selection_method"
    ]
    .value_counts()
)

print()

print(
    "Selected reports:",
    len(selected_reports)
)

print(
    "Periods without signed report:",
    int(
        period_selection[
            "selected_report_id"
        ]
        .isna()
        .sum()
    )
)


# ---------------------------------
# HARD EXPECTED QA
# ---------------------------------

assert len(reports) == 79004

assert (
    reports["is_signed"].sum()
    == 78794
)

assert len(
    period_selection
) == 78800

assert (
    period_selection[
        "selection_method"
    ]
    .value_counts()
    .get(
        "unique_signed",
        0,
    )
    == 78788
)

assert (
    period_selection[
        "selection_method"
    ]
    .value_counts()
    .get(
        "latest_of_multiple_signed",
        0,
    )
    == 3
)

assert (
    period_selection[
        "selection_method"
    ]
    .value_counts()
    .get(
        "no_signed_report",
        0,
    )
    == 9
)

assert (
    len(selected_reports)
    == 78791
)


# ---------------------------------
# SAVE ONLY AFTER QA PASSES
# ---------------------------------

period_selection.to_parquet(
    reports_dir
    / "report_period_selection_final.parquet",
    index=False,
)

report_instances_final.to_parquet(
    reports_dir
    / "report_instances_with_final_selection.parquet",
    index=False,
)

selected_reports.to_parquet(
    reports_dir
    / "selected_reports_manifest.parquet",
    index=False,
)

print()
print(
    "QA passed. Corrected files saved."
)

Signed in source: 78794
Signed successfully parsed: 78794

Logical reporting periods: 78800

Selection methods:
selection_method
unique_signed                78788
no_signed_report                 9
latest_of_multiple_signed        3
Name: count, dtype: int64

Selected reports: 78791
Periods without signed report: 9

QA passed. Corrected files saved.


In [32]:
from pathlib import Path
import pandas as pd

reports_dir = Path(
    "data/interim/reports"
)

selected_reports = pd.read_parquet(
    reports_dir
    / "selected_reports_manifest.parquet"
)


# ---------------------------------
# ORGANIZATION × YEAR PERIODICITY
# ---------------------------------

selected_year_flags = (
    selected_reports
    .groupby(
        [
            "organization_id",
            "year",
        ],
        dropna=False,
    )
    .agg(
        has_annual_report=(
            "quarter",
            lambda x:
                (x == 5).any(),
        ),

        has_quarterly_reports=(
            "quarter",
            lambda x:
                x.isin(
                    [1, 2, 3, 4]
                ).any(),
        ),

        annual_report_count=(
            "quarter",
            lambda x:
                (x == 5).sum(),
        ),

        quarterly_report_count=(
            "quarter",
            lambda x:
                x.isin(
                    [1, 2, 3, 4]
                ).sum(),
        ),

        selected_report_count=(
            "report_id",
            "size",
        ),
    )
    .reset_index()
)


selected_year_flags[
    "has_mixed_periodicity"
] = (
    selected_year_flags[
        "has_annual_report"
    ]
    &
    selected_year_flags[
        "has_quarterly_reports"
    ]
)


# ---------------------------------
# ADD FLAGS TO REPORTS
# ---------------------------------

selected_reports = (
    selected_reports
    .drop(
        columns=[
            "has_annual_report",
            "has_quarterly_reports",
            "annual_report_count",
            "quarterly_report_count",
            "report_count",
            "has_mixed_periodicity",
            "include_by_annual_preference",
            "include_in_annualized_analysis",
        ],
        errors="ignore",
    )
    .merge(
        selected_year_flags,
        on=[
            "organization_id",
            "year",
        ],
        how="left",
    )
)


# ---------------------------------
# ANNUALIZATION RULE
#
# If annual exists:
#     use annual only.
#
# If annual does not exist:
#     use available quarterlies.
# ---------------------------------

selected_reports[
    "include_in_annualized_analysis"
] = (
    (
        selected_reports[
            "has_annual_report"
        ]
        &
        (
            selected_reports[
                "quarter"
            ] == 5
        )
    )
    |
    (
        ~selected_reports[
            "has_annual_report"
        ]
        &
        selected_reports[
            "quarter"
        ].isin(
            [1, 2, 3, 4]
        )
    )
)


# ---------------------------------
# QA
# ---------------------------------

print(
    "Selected signed reports:",
    len(selected_reports)
)

print(
    "Organization-years:",
    len(selected_year_flags)
)

print(
    "Mixed organization-years:",
    int(
        selected_year_flags[
            "has_mixed_periodicity"
        ].sum()
    )
)

print(
    "Organizations with mixed periodicity:",
    selected_year_flags.loc[
        selected_year_flags[
            "has_mixed_periodicity"
        ],
        "organization_id",
    ].nunique()
)

print(
    "Reports included in annualized analysis:",
    int(
        selected_reports[
            "include_in_annualized_analysis"
        ].sum()
    )
)


# ---------------------------------
# IMPORTANT QA:
# NO ORGANIZATION-YEAR MAY HAVE
# ANNUAL + QUARTERLY SELECTED TOGETHER
# ---------------------------------

annualized = (
    selected_reports[
        selected_reports[
            "include_in_annualized_analysis"
        ]
    ]
)

annualized_check = (
    annualized
    .groupby(
        [
            "organization_id",
            "year",
        ]
    )
    .agg(
        has_annual=(
            "quarter",
            lambda x:
                (x == 5).any(),
        ),

        has_quarterly=(
            "quarter",
            lambda x:
                x.isin(
                    [1, 2, 3, 4]
                ).any(),
        ),
    )
)

bad_annualized_groups = (
    annualized_check[
        annualized_check[
            "has_annual"
        ]
        &
        annualized_check[
            "has_quarterly"
        ]
    ]
)

print(
    "Annualized groups with "
    "annual + quarterly together:",
    len(bad_annualized_groups)
)

assert len(
    bad_annualized_groups
) == 0


# ---------------------------------
# SAVE
# ---------------------------------

selected_year_flags.to_parquet(
    reports_dir
    / "selected_organization_year_reporting.parquet",
    index=False,
)

selected_reports.to_parquet(
    reports_dir
    / "selected_reports_manifest.parquet",
    index=False,
)

print()
print("Annual preference QA passed.")

Selected signed reports: 78791
Organization-years: 33397
Mixed organization-years: 487
Organizations with mixed periodicity: 440
Reports included in annualized analysis: 78099
Annualized groups with annual + quarterly together: 0

Annual preference QA passed.


In [33]:
summary_columns = [
    col
    for col
    in selected_reports.columns
    if any(
        token in col.lower()
        for token in [
            "summary",
            "contribution",
            "funding",
            "receipt",
            "expense",
            "payment",
            "refund",
            "transfer",
        ]
    )
]

print(
    "\n".join(
        summary_columns
    )
)

public_summary_version
public_summary_generated_at


In [34]:
from pathlib import Path
import json
import pandas as pd

reports_dir = Path(
    "data/interim/reports"
)

raw_lists_dir = Path(
    "data/raw/report_lists"
)

selected_reports = pd.read_parquet(
    reports_dir
    / "selected_reports_manifest.parquet"
)

selected_ids = set(
    selected_reports[
        "report_id"
    ].astype(str)
)


# ---------------------------------
# PUBLIC SUMMARY MAPPING
# ---------------------------------

SUMMARY_SPECS = {

    "monetary_contributions": {
        "section": "receipts",
        "key": "monetary_contributions",
        "count": "operation_count",
    },

    "other_contributions": {
        "section": "receipts",
        "key": "other_contributions",
        "count": "operation_count",
    },

    "state_funding": {
        "section": "receipts",
        "key": "state_budget_funding",
        "count": "row_count",
    },

    "other_incomes": {
        "section": "receipts",
        "key": "other_receipts",
        "count": "row_count",
    },

    "budget_expenses": {
        "section": "expenses",
        "key": "payments_separate_budget_account",
        "count": "row_count",
    },

    "outgoing_expenses": {
        "section": "expenses",
        "key": "payments_other_accounts",
        "count": "row_count",
    },

    "return_expenses": {
        "section": "expenses",
        "key": "refunds_of_inflows",
        "count": "row_count",
    },

    "transfer_expenses": {
        "section": "expenses",
        "key": "other_refunds_and_transfers",
        "count": "row_count",
    },
}


def get_summary_metric(
    public_summary,
    spec,
):

    if not isinstance(
        public_summary,
        dict,
    ):
        return None, None

    section = (
        public_summary.get(
            spec["section"]
        )
        or {}
    )

    metric = (
        section.get(
            spec["key"]
        )
        or {}
    )

    return (
        metric.get(
            spec["count"]
        ),
        metric.get("sum"),
    )


rows = []

snapshot_files = list(
    raw_lists_dir.glob("*.json")
)

print(
    "Report-list snapshots:",
    len(snapshot_files)
)


for i, path in enumerate(
    snapshot_files,
    start=1,
):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        snapshot = json.load(f)

    for report in (
        snapshot.get("reports")
        or []
    ):

        report_id = str(
            report.get("id")
        )

        if (
            report_id
            not in selected_ids
        ):
            continue

        public_summary = (
            report.get(
                "public_summary"
            )
        )

        row = {
            "report_id":
                report_id,

            "public_summary_present":
                isinstance(
                    public_summary,
                    dict,
                ),
        }

        if isinstance(
            public_summary,
            dict,
        ):

            row[
                "public_summary_version"
            ] = public_summary.get(
                "v"
            )

            row[
                "public_summary_generated_at"
            ] = public_summary.get(
                "generated_at"
            )

        else:

            row[
                "public_summary_version"
            ] = None

            row[
                "public_summary_generated_at"
            ] = None

        for (
            category,
            spec
        ) in SUMMARY_SPECS.items():

            count, amount = (
                get_summary_metric(
                    public_summary,
                    spec,
                )
            )

            row[
                f"{category}_count"
            ] = count

            row[
                f"{category}_sum"
            ] = amount

        rows.append(row)

    if (
        i % 1000 == 0
        or i == len(snapshot_files)
    ):
        print(
            f"{i}/{len(snapshot_files)} "
            "snapshots scanned"
        )


summary_profile = (
    pd.DataFrame(rows)
)


# ---------------------------------
# QA BEFORE MERGE
# ---------------------------------

print()
print(
    "Profile rows:",
    len(summary_profile)
)

print(
    "Unique report IDs:",
    summary_profile[
        "report_id"
    ].nunique()
)

print(
    "Duplicate IDs:",
    summary_profile[
        "report_id"
    ].duplicated().sum()
)


missing_ids = (
    selected_ids
    -
    set(
        summary_profile[
            "report_id"
        ]
    )
)

print(
    "Selected IDs missing "
    "from RAW snapshots:",
    len(missing_ids)
)

assert len(missing_ids) == 0
assert (
    summary_profile[
        "report_id"
    ].duplicated().sum()
    == 0
)


# ---------------------------------
# ADD REPORT METADATA
# ---------------------------------

metadata_columns = [
    "report_id",
    "organization_id",
    "root_party_id",
    "entity_type",
    "year",
    "quarter",
    "selection_method",
    "include_in_annualized_analysis",
]

summary_profile = (
    selected_reports[
        metadata_columns
    ]
    .merge(
        summary_profile,
        on="report_id",
        how="left",
    )
)


summary_profile.to_parquet(
    reports_dir
    / "selected_reports_public_summary_profile.parquet",
    index=False,
)


print()
print(
    "Selected reports:",
    len(selected_reports)
)

print(
    "Summary profile rows:",
    len(summary_profile)
)

print(
    "Reports with public_summary:",
    int(
        summary_profile[
            "public_summary_present"
        ].sum()
    )
)

print()
print(
    "Saved:",
    reports_dir
    / "selected_reports_public_summary_profile.parquet"
)

Report-list snapshots: 9917
1000/9917 snapshots scanned
2000/9917 snapshots scanned
3000/9917 snapshots scanned
4000/9917 snapshots scanned
5000/9917 snapshots scanned
6000/9917 snapshots scanned
7000/9917 snapshots scanned
8000/9917 snapshots scanned
9000/9917 snapshots scanned
9917/9917 snapshots scanned

Profile rows: 78791
Unique report IDs: 78791
Duplicate IDs: 0
Selected IDs missing from RAW snapshots: 0

Selected reports: 78791
Summary profile rows: 78791
Reports with public_summary: 11845

Saved: data\interim\reports\selected_reports_public_summary_profile.parquet


In [35]:
FINANCIAL_CATEGORIES = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


candidate_rows = []

for category in (
    FINANCIAL_CATEGORIES
):

    count_col = (
        f"{category}_count"
    )

    top = (
        summary_profile[
            summary_profile[
                count_col
            ].notna()
        ]
        .sort_values(
            count_col,
            ascending=False,
        )
        .head(5)
        .copy()
    )

    top[
        "stress_category"
    ] = category

    top[
        "stress_row_count"
    ] = top[
        count_col
    ]

    top[
        "stress_rank"
    ] = range(
        1,
        len(top) + 1,
    )

    candidate_rows.append(
        top[
            [
                "stress_category",
                "stress_rank",
                "stress_row_count",
                "report_id",
                "organization_id",
                "root_party_id",
                "entity_type",
                "year",
                "quarter",
            ]
        ]
    )


stress_candidates = (
    pd.concat(
        candidate_rows,
        ignore_index=True,
    )
)

stress_candidates.to_parquet(
    reports_dir
    / "report_detail_stress_candidates.parquet",
    index=False,
)


print(
    "Candidate rows:",
    len(stress_candidates)
)

print(
    "Unique reports:",
    stress_candidates[
        "report_id"
    ].nunique()
)

display(
    stress_candidates
)

Candidate rows: 40
Unique reports: 28


,stress_category,stress_rank,stress_row_count,report_id,organization_id,root_party_id,entity_type,year,quarter
0,monetary_contributions,1,1655.0,e77ef3f0-0401-11ef-823a-73af3dbabf67,de1e8dd6-56aa-4c71-90d9-5ca008438089,de1e8dd6-56aa-4c71-90d9-5ca008438089,party,2022,5
1,monetary_contributions,2,1536.0,f1b84070-0400-11ef-8a6c-27bc4724684c,de1e8dd6-56aa-4c71-90d9-5ca008438089,de1e8dd6-56aa-4c71-90d9-5ca008438089,party,2021,5
2,monetary_contributions,3,1490.0,eb178e90-0402-11ef-92b7-3b3b09128e69,de1e8dd6-56aa-4c71-90d9-5ca008438089,de1e8dd6-56aa-4c71-90d9-5ca008438089,party,2023,5
3,monetary_contributions,4,1082.0,c48713f0-05e2-4fd5-ae9e-7eb1a95f81c6,de1e8dd6-56aa-4c71-90d9-5ca008438089,de1e8dd6-56aa-4c71-90d9-5ca008438089,party,2026,2
4,monetary_contributions,5,1065.0,0b84b35c-f98f-474e-b48f-a1103045d83a,de1e8dd6-56aa-4c71-90d9-5ca008438089,de1e8dd6-56aa-4c71-90d9-5ca008438089,party,2026,1
5,other_contributions,1,326.0,e582f580-e6b8-11ee-96d4-258361b278a8,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,party,2021,1
6,other_contributions,2,180.0,9c24f0c0-e6c0-11ee-8d4b-3dd92aa40d65,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,party,2021,2
7,other_contributions,3,157.0,915612b0-e87c-11ee-8847-316909bdf249,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,party,2022,5
8,other_contributions,4,141.0,f0ba4bb0-03f0-11ef-98d2-6d05851fad7e,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,party,2021,4
9,other_contributions,5,109.0,5ea80b00-e90a-11ee-89f8-172c7818e8e6,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,party,2023,5


In [36]:
from pathlib import Path
import json
import time
import requests

stress_dir = Path(
    "data/raw/report_detail_probes/stress"
)

stress_dir.mkdir(
    parents=True,
    exist_ok=True,
)

stress_report_ids = (
    stress_candidates[
        "report_id"
    ]
    .drop_duplicates()
    .tolist()
)


def fetch_report_detail_probe(
    report_id,
    timeout=180,
    max_retries=4,
):

    url = (
        f"{DEFAULT_BASE_URL}"
        f"/party/report/{report_id}"
    )

    last_error = None

    for attempt in range(
        1,
        max_retries + 1,
    ):

        try:

            response = requests.get(
                url,
                timeout=timeout,
            )

            response.raise_for_status()

            return response.json()

        except Exception as exc:

            last_error = exc

            if attempt < max_retries:

                sleep_seconds = (
                    2 ** (attempt - 1)
                )

                print(
                    f"Retry {attempt}/"
                    f"{max_retries} "
                    f"after {sleep_seconds}s"
                )

                time.sleep(
                    sleep_seconds
                )

    raise last_error


print(
    "Unique stress reports:",
    len(stress_report_ids)
)


for i, report_id in enumerate(
    stress_report_ids,
    start=1,
):

    output_path = (
        stress_dir
        / f"{report_id}.json"
    )

    if output_path.exists():

        print(
            f"{i}/{len(stress_report_ids)}",
            report_id,
            "already saved",
        )

        continue

    try:

        payload = (
            fetch_report_detail_probe(
                report_id
            )
        )

        with output_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                payload,
                f,
                ensure_ascii=False,
                indent=2,
            )

        print(
            f"{i}/{len(stress_report_ids)}",
            report_id,
            "success",
        )

    except Exception as exc:

        print(
            f"{i}/{len(stress_report_ids)}",
            report_id,
            "ERROR",
            repr(exc),
        )

    time.sleep(0.3)

Unique stress reports: 28
1/28 e77ef3f0-0401-11ef-823a-73af3dbabf67 success
2/28 f1b84070-0400-11ef-8a6c-27bc4724684c success
3/28 eb178e90-0402-11ef-92b7-3b3b09128e69 success
4/28 c48713f0-05e2-4fd5-ae9e-7eb1a95f81c6 success
5/28 0b84b35c-f98f-474e-b48f-a1103045d83a success
6/28 e582f580-e6b8-11ee-96d4-258361b278a8 success
7/28 9c24f0c0-e6c0-11ee-8d4b-3dd92aa40d65 success
8/28 915612b0-e87c-11ee-8847-316909bdf249 success
9/28 f0ba4bb0-03f0-11ef-98d2-6d05851fad7e success
10/28 5ea80b00-e90a-11ee-89f8-172c7818e8e6 success
11/28 2be74ca0-eb6f-11ee-823a-73af3dbabf67 success
12/28 818a7a10-eacf-11ee-823a-73af3dbabf67 success
13/28 f2ef12d0-ead2-11ee-96f1-37a2ca81244c success
14/28 88fb9720-ead4-11ee-92b7-3b3b09128e69 success
15/28 fe19a420-02ee-11ef-98d2-6d05851fad7e success
16/28 29fcd30a-8e54-494b-9c34-125d3f3c14c8 success
17/28 433bcd2d-3248-40f5-af1f-0894f8c7b3de success
18/28 87664a89-d2f0-41f8-86fa-8a2982ada76d success
19/28 6d30d130-9d13-11ef-b4cc-1f31c784e768 success
20/28 df634440

In [37]:
from pathlib import Path
import json
import pandas as pd


reports_dir = Path(
    "data/interim/reports"
)

stress_dir = Path(
    "data/raw/report_detail_probes/stress"
)


summary_profile = pd.read_parquet(
    reports_dir
    / "selected_reports_public_summary_profile.parquet"
)

stress_candidates = pd.read_parquet(
    reports_dir
    / "report_detail_stress_candidates.parquet"
)


FINANCIAL_CATEGORIES = {
    "monetary_contributions": (
        "incoming",
        "monetary_contributions",
    ),
    "other_contributions": (
        "incoming",
        "other_contributions",
    ),
    "state_funding": (
        "incoming",
        "state_funding",
    ),
    "other_incomes": (
        "incoming",
        "other_incomes",
    ),
    "budget_expenses": (
        "outgoing",
        "budget_expenses",
    ),
    "outgoing_expenses": (
        "outgoing",
        "outgoing_expenses",
    ),
    "return_expenses": (
        "outgoing",
        "return_expenses",
    ),
    "transfer_expenses": (
        "outgoing",
        "transfer_expenses",
    ),
}


stress_report_ids = (
    stress_candidates[
        "report_id"
    ]
    .drop_duplicates()
    .tolist()
)


rows = []

for report_id in stress_report_ids:

    path = (
        stress_dir
        / f"{report_id}.json"
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        payload = json.load(f)

    result = (
        payload.get("results")
        or {}
    )

    payment_info = (
        result.get("payment_info")
        or {}
    )

    row = {
        "report_id":
            report_id,

        "detail_file_mb":
            path.stat().st_size
            / 1024
            / 1024,
    }

    for (
        category,
        (
            direction,
            detail_key,
        )
    ) in FINANCIAL_CATEGORIES.items():

        direction_data = (
            payment_info.get(direction)
            or {}
        )

        values = (
            direction_data.get(
                detail_key
            )
            or []
        )

        row[
            f"{category}_detail_count"
        ] = len(values)

    rows.append(row)


detail_counts = pd.DataFrame(
    rows
)


# ---------------------------------
# EXPECTED COUNTS FROM PUBLIC SUMMARY
# ---------------------------------

expected_columns = [
    "report_id",
]

for category in (
    FINANCIAL_CATEGORIES
):

    expected_columns.append(
        f"{category}_count"
    )


comparison = (
    detail_counts
    .merge(
        summary_profile[
            expected_columns
        ],
        on="report_id",
        how="left",
    )
)


# ---------------------------------
# COMPARE EACH CATEGORY
# ---------------------------------

match_columns = []

for category in (
    FINANCIAL_CATEGORIES
):

    detail_col = (
        f"{category}_detail_count"
    )

    summary_col = (
        f"{category}_count"
    )

    match_col = (
        f"{category}_matches"
    )

    comparison[
        match_col
    ] = (
        comparison[
            summary_col
        ].isna()
        |
        (
            comparison[
                detail_col
            ]
            ==
            comparison[
                summary_col
            ]
        )
    )

    match_columns.append(
        match_col
    )


comparison[
    "all_counts_match"
] = (
    comparison[
        match_columns
    ]
    .all(axis=1)
)


# ---------------------------------
# QA
# ---------------------------------

print(
    "Stress reports checked:",
    len(comparison)
)

print()

print(
    "All category counts match:"
)

print(
    comparison[
        "all_counts_match"
    ].value_counts(
        dropna=False
    )
)


print()
print(
    "Largest detail file (MB):",
    round(
        comparison[
            "detail_file_mb"
        ].max(),
        2,
    )
)

print(
    "Median detail file (MB):",
    round(
        comparison[
            "detail_file_mb"
        ].median(),
        2,
    )
)


print()
print(
    "MAX EMBEDDED ROW COUNTS"
)

for category in (
    FINANCIAL_CATEGORIES
):

    print(
        category,
        int(
            comparison[
                f"{category}_detail_count"
            ].max()
        ),
    )


# ---------------------------------
# ONLY MISMATCHES
# ---------------------------------

mismatches = (
    comparison[
        ~comparison[
            "all_counts_match"
        ]
    ]
    .copy()
)

print()
print(
    "Reports with mismatches:",
    len(mismatches)
)

display(
    mismatches
)

Stress reports checked: 28

All category counts match:
all_counts_match
False    22
True      6
Name: count, dtype: int64

Largest detail file (MB): 4.45
Median detail file (MB): 1.78

MAX EMBEDDED ROW COUNTS
monetary_contributions 1612
other_contributions 13
state_funding 8
other_incomes 64
budget_expenses 1625
outgoing_expenses 918
return_expenses 10
transfer_expenses 2

Reports with mismatches: 22


,report_id,detail_file_mb,monetary_contributions_detail_count,other_contributions_detail_count,state_funding_detail_count,other_incomes_detail_count,budget_expenses_detail_count,outgoing_expenses_detail_count,return_expenses_detail_count,transfer_expenses_detail_count,...,transfer_expenses_count,monetary_contributions_matches,other_contributions_matches,state_funding_matches,other_incomes_matches,budget_expenses_matches,outgoing_expenses_matches,return_expenses_matches,transfer_expenses_matches,all_counts_match
0,e77ef3f0-0401-11ef-823a-73af3dbabf67,3.774614,1612,0,0,10,0,343,0,0,...,0.0,False,True,True,True,True,False,True,True,False
1,f1b84070-0400-11ef-8a6c-27bc4724684c,3.398303,1460,0,0,17,0,292,0,0,...,0.0,False,True,True,False,True,False,True,True,False
2,eb178e90-0402-11ef-92b7-3b3b09128e69,3.241547,1471,0,0,2,0,210,0,0,...,0.0,False,True,True,True,True,False,True,True,False
3,c48713f0-05e2-4fd5-ae9e-7eb1a95f81c6,2.222556,1079,0,0,0,0,73,0,0,...,0.0,False,True,True,True,True,False,True,True,False
4,0b84b35c-f98f-474e-b48f-a1103045d83a,2.212497,1065,0,0,0,0,82,0,0,...,0.0,True,True,True,False,True,False,True,True,False
5,e582f580-e6b8-11ee-96d4-258361b278a8,0.761496,0,3,1,17,200,69,0,0,...,0.0,False,False,True,False,True,False,False,True,False
6,9c24f0c0-e6c0-11ee-8d4b-3dd92aa40d65,0.912901,0,0,1,3,366,1,0,0,...,0.0,False,False,True,False,True,False,True,True,False
7,915612b0-e87c-11ee-8847-316909bdf249,3.419683,1,0,4,27,1625,39,0,0,...,0.0,False,False,True,False,True,False,True,True,False
8,f0ba4bb0-03f0-11ef-98d2-6d05851fad7e,0.220939,7,1,0,1,0,92,0,0,...,0.0,False,False,True,True,True,False,True,True,False
9,5ea80b00-e90a-11ee-89f8-172c7818e8e6,3.478523,9,0,4,14,1502,203,0,0,...,0.0,False,False,True,False,True,False,True,True,False


In [38]:
import requests
import yaml
import json


SWAGGER_URL = (
    "https://politdata.nazk.gov.ua/"
    "themes/nazk/swagger/public-api.yaml"
)

swagger_response = requests.get(
    SWAGGER_URL,
    timeout=60,
)

swagger_response.raise_for_status()

swagger_spec = yaml.safe_load(
    swagger_response.text
)


# ---------------------------------
# HELPERS
# ---------------------------------

def resolve_ref(spec, obj):

    if not isinstance(obj, dict):
        return obj

    ref = obj.get("$ref")

    if not ref:
        return obj

    if not ref.startswith("#/"):
        return obj

    current = spec

    for part in ref[2:].split("/"):
        current = current[part]

    return current


endpoint_path = (
    "/party/report/{id}/payments/{type}"
)

path_spec = swagger_spec[
    "paths"
][endpoint_path]

operation = path_spec["post"]


print("=" * 80)
print("ENDPOINT")
print("=" * 80)
print(endpoint_path)

print()
print("SUMMARY:")
print(
    operation.get("summary")
)

print()
print("DESCRIPTION:")
print(
    operation.get("description")
)


# ---------------------------------
# PARAMETERS
# ---------------------------------

parameters = (
    path_spec.get(
        "parameters",
        []
    )
    +
    operation.get(
        "parameters",
        []
    )
)

print()
print("=" * 80)
print("PARAMETERS")
print("=" * 80)

payment_type_values = None

for parameter in parameters:

    parameter = resolve_ref(
        swagger_spec,
        parameter,
    )

    name = parameter.get(
        "name"
    )

    print()
    print("-" * 60)
    print("NAME:", name)
    print(
        "IN:",
        parameter.get("in")
    )
    print(
        "REQUIRED:",
        parameter.get(
            "required"
        )
    )
    print(
        "DESCRIPTION:",
        parameter.get(
            "description"
        )
    )

    schema = resolve_ref(
        swagger_spec,
        parameter.get(
            "schema",
            {}
        ),
    )

    print(
        "SCHEMA:"
    )

    print(
        json.dumps(
            schema,
            ensure_ascii=False,
            indent=2,
        )
    )

    if name == "type":

        payment_type_values = (
            schema.get("enum")
        )

        if payment_type_values:

            print()
            print(
                "ALLOWED TYPE VALUES:"
            )

            for value in (
                payment_type_values
            ):
                print(
                    repr(value)
                )


# ---------------------------------
# REQUEST BODY
# ---------------------------------

print()
print("=" * 80)
print("REQUEST BODY")
print("=" * 80)

request_body = resolve_ref(
    swagger_spec,
    operation.get(
        "requestBody",
        {},
    ),
)

print(
    json.dumps(
        request_body,
        ensure_ascii=False,
        indent=2,
    )[:15000]
)


# ---------------------------------
# RESPONSES
# ---------------------------------

print()
print("=" * 80)
print("RESPONSES")
print("=" * 80)

print(
    json.dumps(
        operation.get(
            "responses",
            {}
        ),
        ensure_ascii=False,
        indent=2,
    )[:15000]
)


print()
print("=" * 80)
print("RESULT")
print("=" * 80)

print(
    "payment_type_values =",
    payment_type_values,
)

ENDPOINT
/party/report/{id}/payments/{type}

SUMMARY:
Отримання списку фінансових надходжень, витрат та повернень, зазначених у звіті політичної партії

DESCRIPTION:
None

PARAMETERS

------------------------------------------------------------
NAME: id
IN: path
REQUIRED: True
DESCRIPTION: Ідентифікатор звіту політичної партії
SCHEMA:
{
  "type": "string"
}

------------------------------------------------------------
NAME: type
IN: path
REQUIRED: True
DESCRIPTION: Тип платіжних операцій
SCHEMA:
{
  "type": "string",
  "enum": [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses"
  ]
}

ALLOWED TYPE VALUES:
'monetary_contributions'
'other_contributions'
'state_funding'
'other_incomes'
'budget_expenses'
'outgoing_expenses'
'return_expenses'
'transfer_expenses'

REQUEST BODY
{}

RESPONSES
{
  "200": {
    "description": "OK",
    "content": {
     

In [39]:
import requests
import json

TEST_REPORT_ID = (
    "e77ef3f0-0401-11ef-823a-73af3dbabf67"
)

TEST_PAYMENT_TYPE = (
    "monetary_contributions"
)

url = (
    f"{DEFAULT_BASE_URL}"
    f"/party/report/{TEST_REPORT_ID}"
    f"/payments/{TEST_PAYMENT_TYPE}"
)

response = requests.post(
    url,
    json={},
    timeout=180,
)

print(
    "HTTP:",
    response.status_code,
)

response.raise_for_status()

payload = response.json()

print()
print(
    "TOP-LEVEL TYPE:",
    type(payload).__name__,
)

if isinstance(payload, dict):

    print(
        "TOP-LEVEL KEYS:",
        list(payload.keys()),
    )

    results = payload.get(
        "results"
    )

    print()
    print(
        "RESULTS TYPE:",
        type(results).__name__,
    )

    if isinstance(results, dict):

        print(
            "RESULTS KEYS:",
            list(results.keys()),
        )

        for key, value in (
            results.items()
        ):

            if isinstance(value, list):

                print(
                    f"{key}: list, "
                    f"{len(value)} rows"
                )

            else:

                print(
                    f"{key}: "
                    f"{type(value).__name__}"
                )

    elif isinstance(results, list):

        print(
            "RESULT ROWS:",
            len(results),
        )


print()
print("=" * 80)
print("FIRST SMALL SAMPLE")
print("=" * 80)

# Don't print thousands of rows.
if isinstance(
    payload.get("results"),
    list,
):

    sample = payload[
        "results"
    ][:2]

elif isinstance(
    payload.get("results"),
    dict,
):

    # Replace large lists with first 2 rows
    sample = {}

    for key, value in (
        payload["results"].items()
    ):

        if isinstance(value, list):

            sample[key] = (
                value[:2]
            )

        else:

            sample[key] = value

else:

    sample = payload


print(
    json.dumps(
        sample,
        ensure_ascii=False,
        indent=2,
        default=str,
    )[:12000]
)

HTTP: 200

TOP-LEVEL TYPE: dict
TOP-LEVEL KEYS: ['results']

RESULTS TYPE: dict
RESULTS KEYS: ['list', 'count']
list: list, 1612 rows
count: int

FIRST SMALL SAMPLE
{
  "list": [
    {
      "id": "7dcf65be-cfff-40c6-b78d-fa0a62f61152",
      "report_id": "e77ef3f0-0401-11ef-823a-73af3dbabf67",
      "group_code": "3_1",
      "payment_type": null,
      "payment_code": null,
      "payment_number": null,
      "payment_amount": 994,
      "payment_currency": null,
      "payment_reason": null,
      "payment_purpose": null,
      "payment_operation_date": "2022-07-26",
      "payment_instruction_date": null,
      "payment_description": null,
      "refund_date": null,
      "refund_amount": null,
      "refund_budget_amount": null,
      "refund_reason": null,
      "refund_purpose": null,
      "refund_description": null,
      "payer_type": "Фізична особа",
      "payer_name": "Ютовець Олександр",
      "payer_code": "[конфіденційна інформація]",
      "payer_birthday": "[конфіденц

In [40]:
from pathlib import Path
import json
import time
import requests
import pandas as pd


stress_dir = Path(
    "data/raw/report_detail_probes/stress"
)

payments_probe_dir = Path(
    "data/raw/payment_endpoint_probes/stress"
)

payments_probe_dir.mkdir(
    parents=True,
    exist_ok=True,
)


PAYMENT_TYPES = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


stress_report_ids = sorted(
    path.stem
    for path in stress_dir.glob("*.json")
)

print(
    "Stress reports:",
    len(stress_report_ids)
)

print(
    "Expected endpoint checks:",
    len(stress_report_ids)
    * len(PAYMENT_TYPES)
)


def fetch_payment_endpoint(
    report_id,
    payment_type,
    timeout=180,
    max_retries=4,
):

    url = (
        f"{DEFAULT_BASE_URL}"
        f"/party/report/{report_id}"
        f"/payments/{payment_type}"
    )

    last_error = None

    for attempt in range(
        1,
        max_retries + 1,
    ):

        try:

            response = requests.post(
                url,
                json={},
                timeout=timeout,
            )

            response.raise_for_status()

            payload = response.json()

            results = (
                payload.get("results")
                or {}
            )

            rows = (
                results.get("list")
                or []
            )

            count = results.get(
                "count"
            )

            return {
                "payload": payload,
                "rows": rows,
                "count": count,
            }

        except Exception as exc:

            last_error = exc

            if attempt < max_retries:

                sleep_seconds = (
                    2 ** (attempt - 1)
                )

                print(
                    "retry",
                    report_id,
                    payment_type,
                    attempt,
                    f"after {sleep_seconds}s",
                )

                time.sleep(
                    sleep_seconds
                )

    raise last_error


comparison_rows = []


for report_no, report_id in enumerate(
    stress_report_ids,
    start=1,
):

    detail_path = (
        stress_dir
        / f"{report_id}.json"
    )

    with detail_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = (
            json.load(f)[
                "results"
            ]
        )

    payment_info = (
        detail.get("payment_info")
        or {}
    )

    print()
    print(
        f"REPORT "
        f"{report_no}/{len(stress_report_ids)}"
    )

    for payment_type in PAYMENT_TYPES:

        direction, key = (
            PAYMENT_PATHS[
                payment_type
            ]
        )

        embedded_rows = (
            (
                payment_info.get(
                    direction
                )
                or {}
            )
            .get(key)
            or []
        )

        raw_dir = (
            payments_probe_dir
            / report_id
        )

        raw_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        raw_path = (
            raw_dir
            / f"{payment_type}.json"
        )


        # --------------------------
        # USE EXISTING RAW IF SAVED
        # --------------------------

        if raw_path.exists():

            with raw_path.open(
                "r",
                encoding="utf-8",
            ) as f:

                endpoint_payload = (
                    json.load(f)
                )

            results = (
                endpoint_payload.get(
                    "results"
                )
                or {}
            )

            endpoint_rows = (
                results.get("list")
                or []
            )

            declared_count = (
                results.get("count")
            )

            fetch_status = (
                "existing"
            )

        else:

            try:

                fetched = (
                    fetch_payment_endpoint(
                        report_id,
                        payment_type,
                    )
                )

                endpoint_payload = (
                    fetched["payload"]
                )

                endpoint_rows = (
                    fetched["rows"]
                )

                declared_count = (
                    fetched["count"]
                )

                with raw_path.open(
                    "w",
                    encoding="utf-8",
                ) as f:

                    json.dump(
                        endpoint_payload,
                        f,
                        ensure_ascii=False,
                        indent=2,
                    )

                fetch_status = (
                    "success"
                )

            except Exception as exc:

                comparison_rows.append(
                    {
                        "report_id":
                            report_id,

                        "payment_type":
                            payment_type,

                        "fetch_status":
                            "error",

                        "error":
                            repr(exc),
                    }
                )

                print(
                    payment_type,
                    "ERROR",
                )

                continue


        # --------------------------
        # ID COMPARISON
        # --------------------------

        embedded_ids = {
            str(row.get("id"))
            for row in embedded_rows
            if isinstance(row, dict)
            and row.get("id")
        }

        endpoint_ids = {
            str(row.get("id"))
            for row in endpoint_rows
            if isinstance(row, dict)
            and row.get("id")
        }


        only_embedded = (
            embedded_ids
            -
            endpoint_ids
        )

        only_endpoint = (
            endpoint_ids
            -
            embedded_ids
        )

        intersection = (
            embedded_ids
            &
            endpoint_ids
        )


        comparison_rows.append(
            {
                "report_id":
                    report_id,

                "payment_type":
                    payment_type,

                "fetch_status":
                    fetch_status,

                "embedded_count":
                    len(embedded_rows),

                "endpoint_list_count":
                    len(endpoint_rows),

                "endpoint_declared_count":
                    declared_count,

                "embedded_unique_ids":
                    len(embedded_ids),

                "endpoint_unique_ids":
                    len(endpoint_ids),

                "intersection_count":
                    len(intersection),

                "only_in_embedded_count":
                    len(only_embedded),

                "only_in_endpoint_count":
                    len(only_endpoint),

                "same_ids":
                    (
                        embedded_ids
                        ==
                        endpoint_ids
                    ),

                "endpoint_count_matches_list":
                    (
                        declared_count
                        ==
                        len(endpoint_rows)
                    )
                    if declared_count
                    is not None
                    else None,
            }
        )

        print(
            f"{payment_type}: "
            f"detail={len(embedded_rows)}, "
            f"endpoint={len(endpoint_rows)}, "
            f"same_ids="
            f"{embedded_ids == endpoint_ids}"
        )

        time.sleep(0.2)


payment_source_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)


output_path = (
    Path(
        "data/interim/reports"
    )
    / "payment_source_comparison_stress.parquet"
)

payment_source_comparison.to_parquet(
    output_path,
    index=False,
)


print()
print("=" * 70)
print("FINAL QA")
print("=" * 70)

print(
    "Checks:",
    len(
        payment_source_comparison
    )
)

print()

print(
    "Fetch status:"
)

print(
    payment_source_comparison[
        "fetch_status"
    ].value_counts(
        dropna=False
    )
)

print()

print(
    "ID SET MATCH:"
)

print(
    payment_source_comparison[
        "same_ids"
    ].value_counts(
        dropna=False
    )
)

print()

print(
    "Endpoint count matches "
    "returned list:"
)

print(
    payment_source_comparison[
        "endpoint_count_matches_list"
    ].value_counts(
        dropna=False
    )
)

print()

print(
    "Checks with ID mismatch:"
)

id_mismatches = (
    payment_source_comparison[
        payment_source_comparison[
            "same_ids"
        ] == False
    ]
)

print(
    len(id_mismatches)
)

display(
    id_mismatches
)

Stress reports: 28
Expected endpoint checks: 224

REPORT 1/28
monetary_contributions: detail=1065, endpoint=1065, same_ids=True
other_contributions: detail=0, endpoint=0, same_ids=True
state_funding: detail=0, endpoint=0, same_ids=True
other_incomes: detail=0, endpoint=0, same_ids=True
budget_expenses: detail=0, endpoint=0, same_ids=True
outgoing_expenses: detail=82, endpoint=82, same_ids=True
return_expenses: detail=0, endpoint=0, same_ids=True
transfer_expenses: detail=0, endpoint=0, same_ids=True

REPORT 2/28
monetary_contributions: detail=156, endpoint=156, same_ids=True
other_contributions: detail=0, endpoint=0, same_ids=True
state_funding: detail=0, endpoint=0, same_ids=True
other_incomes: detail=7, endpoint=7, same_ids=True
budget_expenses: detail=359, endpoint=359, same_ids=True
outgoing_expenses: detail=136, endpoint=136, same_ids=True
return_expenses: detail=0, endpoint=0, same_ids=True
transfer_expenses: detail=0, endpoint=0, same_ids=True

REPORT 3/28
monetary_contributions

,report_id,payment_type,fetch_status,embedded_count,endpoint_list_count,endpoint_declared_count,embedded_unique_ids,endpoint_unique_ids,intersection_count,only_in_embedded_count,only_in_endpoint_count,same_ids,endpoint_count_matches_list


In [41]:
from pathlib import Path
import json
import time
import requests
import pandas as pd


stress_dir = Path(
    "data/raw/report_detail_probes/stress"
)

specialized_dir = Path(
    "data/raw/report_section_probes/stress"
)

specialized_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# endpoint -> location inside report detail
SECTION_SPECS = {
    "realty": {
        "detail_group": "properties",
        "detail_key": "property_object",
    },

    "money": {
        "detail_group": "properties",
        "detail_key": "property_moneys",
    },

    "movable": {
        "detail_group": "properties",
        "detail_key": "property_movable",
    },

    "paper": {
        "detail_group": "properties",
        "detail_key": "property_paper",
    },

    "transport": {
        "detail_group": "properties",
        "detail_key": "property_transport",
    },

    "intangible": {
        "detail_group": "properties",
        "detail_key": "property_intangible_asset",
    },

    "obligations": {
        "detail_group": None,
        "detail_key": "obligations",
    },
}


stress_report_ids = sorted(
    path.stem
    for path
    in stress_dir.glob("*.json")
)

print(
    "Stress reports:",
    len(stress_report_ids)
)

print(
    "Expected endpoint checks:",
    len(stress_report_ids)
    * len(SECTION_SPECS)
)


def get_detail_rows(
    report_detail,
    spec,
):

    if spec["detail_group"] is None:

        return (
            report_detail.get(
                spec["detail_key"]
            )
            or []
        )

    group = (
        report_detail.get(
            spec["detail_group"]
        )
        or {}
    )

    return (
        group.get(
            spec["detail_key"]
        )
        or []
    )


def extract_endpoint_rows(
    payload,
):
    """
    Support the response shapes we may encounter.
    """

    results = payload.get(
        "results"
    )

    if isinstance(
        results,
        list,
    ):
        return (
            results,
            len(results),
        )

    if isinstance(
        results,
        dict,
    ):

        if isinstance(
            results.get("list"),
            list,
        ):
            return (
                results["list"],
                results.get("count"),
            )

    return (
        [],
        None,
    )


def fetch_section(
    report_id,
    endpoint_name,
    timeout=180,
    max_retries=4,
):

    url = (
        f"{DEFAULT_BASE_URL}"
        f"/party/report/{report_id}"
        f"/{endpoint_name}"
    )

    last_error = None

    for attempt in range(
        1,
        max_retries + 1,
    ):

        try:

            response = requests.post(
                url,
                json={},
                timeout=timeout,
            )

            response.raise_for_status()

            return response.json()

        except Exception as exc:

            last_error = exc

            if attempt < max_retries:

                sleep_seconds = (
                    2 ** (attempt - 1)
                )

                print(
                    "retry",
                    report_id,
                    endpoint_name,
                    attempt,
                    f"after {sleep_seconds}s",
                )

                time.sleep(
                    sleep_seconds
                )

    raise last_error


def records_by_id(rows):

    return {
        str(row["id"]): row
        for row in rows
        if (
            isinstance(row, dict)
            and row.get("id")
        )
    }


comparison_rows = []


for report_no, report_id in enumerate(
    stress_report_ids,
    start=1,
):

    detail_path = (
        stress_dir
        / f"{report_id}.json"
    )

    with detail_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        report_detail = (
            json.load(f)[
                "results"
            ]
        )

    print()
    print(
        f"REPORT "
        f"{report_no}/"
        f"{len(stress_report_ids)}"
    )

    for (
        endpoint_name,
        spec
    ) in SECTION_SPECS.items():

        detail_rows = (
            get_detail_rows(
                report_detail,
                spec,
            )
        )

        report_raw_dir = (
            specialized_dir
            / report_id
        )

        report_raw_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        raw_path = (
            report_raw_dir
            / f"{endpoint_name}.json"
        )


        # --------------------------
        # FETCH OR REUSE RAW
        # --------------------------

        if raw_path.exists():

            with raw_path.open(
                "r",
                encoding="utf-8",
            ) as f:

                endpoint_payload = (
                    json.load(f)
                )

            fetch_status = (
                "existing"
            )

        else:

            try:

                endpoint_payload = (
                    fetch_section(
                        report_id,
                        endpoint_name,
                    )
                )

                with raw_path.open(
                    "w",
                    encoding="utf-8",
                ) as f:

                    json.dump(
                        endpoint_payload,
                        f,
                        ensure_ascii=False,
                        indent=2,
                    )

                fetch_status = (
                    "success"
                )

            except Exception as exc:

                comparison_rows.append(
                    {
                        "report_id":
                            report_id,

                        "section":
                            endpoint_name,

                        "fetch_status":
                            "error",

                        "error":
                            repr(exc),
                    }
                )

                print(
                    endpoint_name,
                    "ERROR",
                )

                continue


        endpoint_rows, declared_count = (
            extract_endpoint_rows(
                endpoint_payload
            )
        )


        # --------------------------
        # ID COMPARISON
        # --------------------------

        detail_map = (
            records_by_id(
                detail_rows
            )
        )

        endpoint_map = (
            records_by_id(
                endpoint_rows
            )
        )

        detail_ids = set(
            detail_map
        )

        endpoint_ids = set(
            endpoint_map
        )

        common_ids = (
            detail_ids
            &
            endpoint_ids
        )

        only_detail = (
            detail_ids
            -
            endpoint_ids
        )

        only_endpoint = (
            endpoint_ids
            -
            detail_ids
        )


        # --------------------------
        # FULL RECORD COMPARISON
        # FOR MATCHED IDs
        # --------------------------

        content_mismatch_ids = [
            row_id
            for row_id
            in common_ids
            if (
                detail_map[row_id]
                !=
                endpoint_map[row_id]
            )
        ]


        comparison_rows.append(
            {
                "report_id":
                    report_id,

                "section":
                    endpoint_name,

                "fetch_status":
                    fetch_status,

                "detail_count":
                    len(detail_rows),

                "endpoint_list_count":
                    len(endpoint_rows),

                "endpoint_declared_count":
                    declared_count,

                "detail_unique_ids":
                    len(detail_ids),

                "endpoint_unique_ids":
                    len(endpoint_ids),

                "intersection_count":
                    len(common_ids),

                "only_in_detail_count":
                    len(only_detail),

                "only_in_endpoint_count":
                    len(only_endpoint),

                "same_ids":
                    (
                        detail_ids
                        ==
                        endpoint_ids
                    ),

                "content_mismatch_count":
                    len(
                        content_mismatch_ids
                    ),

                "same_content":
                    (
                        len(
                            content_mismatch_ids
                        )
                        == 0
                    ),

                "endpoint_count_matches_list":
                    (
                        declared_count
                        ==
                        len(endpoint_rows)
                    )
                    if declared_count
                    is not None
                    else None,
            }
        )

        print(
            f"{endpoint_name}: "
            f"detail={len(detail_rows)}, "
            f"endpoint={len(endpoint_rows)}, "
            f"same_ids="
            f"{detail_ids == endpoint_ids}, "
            f"content_diff="
            f"{len(content_mismatch_ids)}"
        )

        time.sleep(0.2)


section_source_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)


output_path = (
    Path(
        "data/interim/reports"
    )
    / "report_section_source_comparison_stress.parquet"
)

section_source_comparison.to_parquet(
    output_path,
    index=False,
)


# ---------------------------------
# FINAL QA
# ---------------------------------

print()
print("=" * 70)
print("FINAL QA")
print("=" * 70)

print(
    "Checks:",
    len(
        section_source_comparison
    )
)

print()
print("Fetch status:")

print(
    section_source_comparison[
        "fetch_status"
    ].value_counts(
        dropna=False
    )
)

print()
print("ID SET MATCH:")

print(
    section_source_comparison[
        "same_ids"
    ].value_counts(
        dropna=False
    )
)

print()
print("CONTENT MATCH:")

print(
    section_source_comparison[
        "same_content"
    ].value_counts(
        dropna=False
    )
)

print()
print(
    "Endpoint count matches list:"
)

print(
    section_source_comparison[
        "endpoint_count_matches_list"
    ].value_counts(
        dropna=False
    )
)


problem_checks = (
    section_source_comparison[
        (
            section_source_comparison[
                "same_ids"
            ] == False
        )
        |
        (
            section_source_comparison[
                "same_content"
            ] == False
        )
        |
        (
            section_source_comparison[
                "fetch_status"
            ] == "error"
        )
    ]
    .copy()
)


print()
print(
    "Checks with any problem:",
    len(problem_checks)
)

display(
    problem_checks
)

Stress reports: 28
Expected endpoint checks: 196

REPORT 1/28
realty: detail=1, endpoint=1, same_ids=True, content_diff=1
money: detail=1, endpoint=1, same_ids=True, content_diff=1
movable: detail=0, endpoint=0, same_ids=True, content_diff=0
paper: detail=0, endpoint=0, same_ids=True, content_diff=0
transport: detail=0, endpoint=0, same_ids=True, content_diff=0
intangible: detail=1, endpoint=1, same_ids=True, content_diff=1
obligations: detail=0, endpoint=0, same_ids=True, content_diff=0

REPORT 2/28
realty: detail=5, endpoint=5, same_ids=True, content_diff=5
money: detail=7, endpoint=7, same_ids=True, content_diff=7
movable: detail=0, endpoint=0, same_ids=True, content_diff=0
paper: detail=0, endpoint=0, same_ids=True, content_diff=0
transport: detail=0, endpoint=0, same_ids=True, content_diff=0
intangible: detail=9, endpoint=9, same_ids=True, content_diff=9
obligations: detail=19, endpoint=19, same_ids=True, content_diff=19

REPORT 3/28
realty: detail=1, endpoint=1, same_ids=True, co

,report_id,section,fetch_status,detail_count,endpoint_list_count,endpoint_declared_count,detail_unique_ids,endpoint_unique_ids,intersection_count,only_in_detail_count,only_in_endpoint_count,same_ids,content_mismatch_count,same_content,endpoint_count_matches_list
0,0b84b35c-f98f-474e-b48f-a1103045d83a,realty,success,1,1,1,1,1,1,0,0,True,1,False,True
1,0b84b35c-f98f-474e-b48f-a1103045d83a,money,success,1,1,1,1,1,1,0,0,True,1,False,True
5,0b84b35c-f98f-474e-b48f-a1103045d83a,intangible,success,1,1,1,1,1,1,0,0,True,1,False,True
7,2394f660-eab1-11ee-96f1-37a2ca81244c,realty,success,5,5,5,5,5,5,0,0,True,5,False,True
8,2394f660-eab1-11ee-96f1-37a2ca81244c,money,success,7,7,7,7,7,7,0,0,True,7,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
187,f2ef12d0-ead2-11ee-96f1-37a2ca81244c,intangible,success,91,91,91,91,91,91,0,0,True,91,False,True
188,f2ef12d0-ead2-11ee-96f1-37a2ca81244c,obligations,success,5,5,5,5,5,5,0,0,True,5,False,True
189,fe19a420-02ee-11ef-98d2-6d05851fad7e,realty,success,2,2,2,2,2,2,0,0,True,2,False,True
190,fe19a420-02ee-11ef-98d2-6d05851fad7e,money,success,2,2,2,2,2,2,0,0,True,2,False,True


In [42]:
from collections import Counter, defaultdict
from pathlib import Path
import json
import pandas as pd


stress_dir = Path(
    "data/raw/report_detail_probes/stress"
)

specialized_dir = Path(
    "data/raw/report_section_probes/stress"
)


SECTION_SPECS = {
    "realty": {
        "detail_group": "properties",
        "detail_key": "property_object",
    },
    "money": {
        "detail_group": "properties",
        "detail_key": "property_moneys",
    },
    "movable": {
        "detail_group": "properties",
        "detail_key": "property_movable",
    },
    "paper": {
        "detail_group": "properties",
        "detail_key": "property_paper",
    },
    "transport": {
        "detail_group": "properties",
        "detail_key": "property_transport",
    },
    "intangible": {
        "detail_group": "properties",
        "detail_key": "property_intangible_asset",
    },
    "obligations": {
        "detail_group": None,
        "detail_key": "obligations",
    },
}


def get_detail_rows(
    report_detail,
    spec,
):

    if spec["detail_group"] is None:

        return (
            report_detail.get(
                spec["detail_key"]
            )
            or []
        )

    group = (
        report_detail.get(
            spec["detail_group"]
        )
        or {}
    )

    return (
        group.get(
            spec["detail_key"]
        )
        or []
    )


def get_endpoint_rows(
    payload,
):

    results = payload.get(
        "results"
    )

    if isinstance(
        results,
        list,
    ):
        return results

    if isinstance(
        results,
        dict,
    ):
        return (
            results.get("list")
            or []
        )

    return []


def index_by_id(rows):

    return {
        str(row["id"]): row
        for row in rows
        if (
            isinstance(row, dict)
            and row.get("id")
        )
    }


# ---------------------------------
# COLLECT FIELD-LEVEL DIFFERENCES
# ---------------------------------

field_diff_counter = Counter()

missing_in_detail_counter = Counter()

missing_in_endpoint_counter = Counter()

value_diff_counter = Counter()

section_field_diff = defaultdict(
    Counter
)

examples = {}


for report_dir in (
    specialized_dir.iterdir()
):

    if not report_dir.is_dir():
        continue

    report_id = (
        report_dir.name
    )

    detail_path = (
        stress_dir
        / f"{report_id}.json"
    )

    if not detail_path.exists():
        continue

    with detail_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        report_detail = (
            json.load(f)[
                "results"
            ]
        )


    for (
        section,
        spec
    ) in SECTION_SPECS.items():

        endpoint_path = (
            report_dir
            / f"{section}.json"
        )

        if not endpoint_path.exists():
            continue

        with endpoint_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            endpoint_payload = (
                json.load(f)
            )

        detail_rows = (
            get_detail_rows(
                report_detail,
                spec,
            )
        )

        endpoint_rows = (
            get_endpoint_rows(
                endpoint_payload
            )
        )

        detail_map = (
            index_by_id(
                detail_rows
            )
        )

        endpoint_map = (
            index_by_id(
                endpoint_rows
            )
        )


        for row_id in (
            set(detail_map)
            &
            set(endpoint_map)
        ):

            d = detail_map[
                row_id
            ]

            e = endpoint_map[
                row_id
            ]

            all_keys = (
                set(d)
                |
                set(e)
            )

            for key in all_keys:

                if key not in d:

                    missing_in_detail_counter[
                        key
                    ] += 1

                    field_diff_counter[
                        key
                    ] += 1

                    section_field_diff[
                        section
                    ][
                        key
                    ] += 1

                    examples.setdefault(
                        (
                            section,
                            key,
                            "missing_in_detail",
                        ),
                        {
                            "report_id":
                                report_id,
                            "row_id":
                                row_id,
                            "detail":
                                None,
                            "endpoint":
                                e.get(key),
                        },
                    )

                    continue


                if key not in e:

                    missing_in_endpoint_counter[
                        key
                    ] += 1

                    field_diff_counter[
                        key
                    ] += 1

                    section_field_diff[
                        section
                    ][
                        key
                    ] += 1

                    examples.setdefault(
                        (
                            section,
                            key,
                            "missing_in_endpoint",
                        ),
                        {
                            "report_id":
                                report_id,
                            "row_id":
                                row_id,
                            "detail":
                                d.get(key),
                            "endpoint":
                                None,
                        },
                    )

                    continue


                if d.get(key) != e.get(key):

                    value_diff_counter[
                        key
                    ] += 1

                    field_diff_counter[
                        key
                    ] += 1

                    section_field_diff[
                        section
                    ][
                        key
                    ] += 1

                    examples.setdefault(
                        (
                            section,
                            key,
                            "different_value",
                        ),
                        {
                            "report_id":
                                report_id,
                            "row_id":
                                row_id,
                            "detail":
                                d.get(key),
                            "endpoint":
                                e.get(key),
                        },
                    )


print("=" * 70)
print("MOST COMMON DIFFERING FIELDS")
print("=" * 70)

for key, count in (
    field_diff_counter
    .most_common(50)
):

    print(
        f"{key}: {count}"
    )


print()
print("=" * 70)
print("MISSING IN DETAIL")
print("=" * 70)

for key, count in (
    missing_in_detail_counter
    .most_common()
):

    print(
        f"{key}: {count}"
    )


print()
print("=" * 70)
print("MISSING IN ENDPOINT")
print("=" * 70)

for key, count in (
    missing_in_endpoint_counter
    .most_common()
):

    print(
        f"{key}: {count}"
    )


print()
print("=" * 70)
print("DIFFERENT VALUES")
print("=" * 70)

for key, count in (
    value_diff_counter
    .most_common()
):

    print(
        f"{key}: {count}"
    )


print()
print("=" * 70)
print("BY SECTION")
print("=" * 70)

for section in SECTION_SPECS:

    print()
    print(section)

    for key, count in (
        section_field_diff[
            section
        ]
        .most_common(20)
    ):

        print(
            f"  {key}: {count}"
        )

MOST COMMON DIFFERING FIELDS
party_id: 1229
office_id: 1229
owning_subject_id: 625
object_type_id: 448
transport_type_id: 177
object_rights_id: 177
party_report_id: 177

MISSING IN DETAIL
owning_subject_id: 625
object_type_id: 448
transport_type_id: 177
object_rights_id: 177
party_report_id: 177

MISSING IN ENDPOINT
party_id: 1229
office_id: 1229

DIFFERENT VALUES

BY SECTION

realty
  party_id: 82
  office_id: 82

money
  party_id: 97
  office_id: 97

movable
  party_id: 37
  office_id: 37

paper

transport
  transport_type_id: 177
  owning_subject_id: 177
  object_rights_id: 177
  party_report_id: 177

intangible
  party_id: 565
  office_id: 565

obligations
  party_id: 448
  owning_subject_id: 448
  office_id: 448
  object_type_id: 448


In [43]:
example_rows = []

for (
    section,
    key,
    difference_type
), example in examples.items():

    example_rows.append(
        {
            "section":
                section,

            "field":
                key,

            "difference_type":
                difference_type,

            "report_id":
                example[
                    "report_id"
                ],

            "row_id":
                example[
                    "row_id"
                ],

            "detail_value":
                example[
                    "detail"
                ],

            "endpoint_value":
                example[
                    "endpoint"
                ],
        }
    )


field_diff_examples = (
    pd.DataFrame(
        example_rows
    )
    .sort_values(
        [
            "section",
            "field",
            "difference_type",
        ]
    )
)

display(
    field_diff_examples
)

,section,field,difference_type,report_id,row_id,detail_value,endpoint_value
5,intangible,office_id,missing_in_endpoint,0b84b35c-f98f-474e-b48f-a1103045d83a,05b9cd2b-d107-486f-85a6-1b516b45ce8a,None,None
4,intangible,party_id,missing_in_endpoint,0b84b35c-f98f-474e-b48f-a1103045d83a,05b9cd2b-d107-486f-85a6-1b516b45ce8a,de1e8dd6-56aa-4c71-90d9-5ca008438089,None
3,money,office_id,missing_in_endpoint,0b84b35c-f98f-474e-b48f-a1103045d83a,c79bf5be-49a0-43ba-9656-0779694d8ac2,None,None
2,money,party_id,missing_in_endpoint,0b84b35c-f98f-474e-b48f-a1103045d83a,c79bf5be-49a0-43ba-9656-0779694d8ac2,de1e8dd6-56aa-4c71-90d9-5ca008438089,None
11,movable,office_id,missing_in_endpoint,29fcd30a-8e54-494b-9c34-125d3f3c14c8,88f69d4d-ebd4-4d1c-bf60-01a338a093c2,None,None
10,movable,party_id,missing_in_endpoint,29fcd30a-8e54-494b-9c34-125d3f3c14c8,88f69d4d-ebd4-4d1c-bf60-01a338a093c2,7d9a4591-6cea-451f-8545-b211f5e7b9bd,None
9,obligations,object_type_id,missing_in_detail,2394f660-eab1-11ee-96f1-37a2ca81244c,2497661c-d4e7-4ea4-ba18-84e65251505f,None,None
8,obligations,office_id,missing_in_endpoint,2394f660-eab1-11ee-96f1-37a2ca81244c,2497661c-d4e7-4ea4-ba18-84e65251505f,None,None
7,obligations,owning_subject_id,missing_in_detail,2394f660-eab1-11ee-96f1-37a2ca81244c,2497661c-d4e7-4ea4-ba18-84e65251505f,None,None
6,obligations,party_id,missing_in_endpoint,2394f660-eab1-11ee-96f1-37a2ca81244c,2497661c-d4e7-4ea4-ba18-84e65251505f,b11839f3-bffe-4f93-ba10-3b877ed181b9,None


In [44]:
from collections import Counter, defaultdict
from pathlib import Path
import json


stress_dir = Path(
    "data/raw/report_detail_probes/stress"
)

payments_dir = Path(
    "data/raw/payment_endpoint_probes/stress"
)


PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


missing_in_detail = Counter()
missing_in_endpoint = Counter()
different_values = Counter()

different_by_type = defaultdict(
    Counter
)


def by_id(rows):

    return {
        str(row["id"]): row
        for row in rows
        if (
            isinstance(row, dict)
            and row.get("id")
        )
    }


for report_dir in payments_dir.iterdir():

    if not report_dir.is_dir():
        continue

    report_id = report_dir.name

    detail_path = (
        stress_dir
        / f"{report_id}.json"
    )

    with detail_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = (
            json.load(f)[
                "results"
            ]
        )

    payment_info = (
        detail.get("payment_info")
        or {}
    )

    for (
        payment_type,
        (
            direction,
            key,
        )
    ) in PAYMENT_PATHS.items():

        endpoint_path = (
            report_dir
            / f"{payment_type}.json"
        )

        if not endpoint_path.exists():
            continue

        with endpoint_path.open(
            "r",
            encoding="utf-8",
        ) as f:

            endpoint_payload = (
                json.load(f)
            )

        detail_rows = (
            (
                payment_info.get(
                    direction
                )
                or {}
            )
            .get(key)
            or []
        )

        endpoint_rows = (
            (
                endpoint_payload.get(
                    "results"
                )
                or {}
            )
            .get("list")
            or []
        )

        detail_map = by_id(
            detail_rows
        )

        endpoint_map = by_id(
            endpoint_rows
        )

        assert (
            set(detail_map)
            ==
            set(endpoint_map)
        )

        for row_id in detail_map:

            d = detail_map[
                row_id
            ]

            e = endpoint_map[
                row_id
            ]

            for field in (
                set(d)
                |
                set(e)
            ):

                if field not in d:

                    missing_in_detail[
                        field
                    ] += 1

                    continue

                if field not in e:

                    missing_in_endpoint[
                        field
                    ] += 1

                    continue

                if d[field] != e[field]:

                    different_values[
                        field
                    ] += 1

                    different_by_type[
                        payment_type
                    ][
                        field
                    ] += 1


print("MISSING IN DETAIL")
print(missing_in_detail)

print()

print("MISSING IN ENDPOINT")
print(missing_in_endpoint)

print()

print("ACTUAL VALUE DIFFERENCES")
print(different_values)

print()

print("VALUE DIFFERENCES BY TYPE")

for payment_type in PAYMENT_PATHS:

    print(
        payment_type,
        dict(
            different_by_type[
                payment_type
            ]
        ),
    )

MISSING IN DETAIL
Counter()

MISSING IN ENDPOINT
Counter({'office_id': 25005, 'party_id': 25005})

ACTUAL VALUE DIFFERENCES
Counter()

VALUE DIFFERENCES BY TYPE
monetary_contributions {}
other_contributions {}
state_funding {}
other_incomes {}
budget_expenses {}
outgoing_expenses {}
return_expenses {}
transfer_expenses {}


In [45]:
from pathlib import Path
import json
import pandas as pd

reports_dir = Path(
    "data/interim/reports"
)

raw_lists_dir = Path(
    "data/raw/report_lists"
)

selected_reports = pd.read_parquet(
    reports_dir
    / "selected_reports_manifest.parquet"
)

selected_ids = set(
    selected_reports[
        "report_id"
    ].astype(str)
)

rows = []

for path in raw_lists_dir.glob("*.json"):

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        snapshot = json.load(f)

    for report in (
        snapshot.get("reports")
        or []
    ):

        report_id = str(
            report.get("id")
        )

        if report_id not in selected_ids:
            continue

        public_summary = (
            report.get("public_summary")
            or {}
        )

        property_summary = (
            public_summary.get(
                "property_and_intangible"
            )
            or {}
        )

        securities = (
            property_summary.get(
                "securities"
            )
            or {}
        )

        row_count = (
            securities.get(
                "row_count"
            )
        )

        if (
            row_count is not None
            and row_count > 0
        ):

            rows.append(
                {
                    "report_id":
                        report_id,
                    "securities_hint_count":
                        row_count,
                }
            )


paper_candidates = (
    pd.DataFrame(rows)
    .drop_duplicates(
        subset=["report_id"]
    )
    .merge(
        selected_reports[
            [
                "report_id",
                "organization_id",
                "root_party_id",
                "entity_type",
                "year",
                "quarter",
            ]
        ],
        on="report_id",
        how="left",
    )
    .sort_values(
        "securities_hint_count",
        ascending=False,
    )
)

print(
    "Selected reports with "
    "securities hint:",
    len(paper_candidates)
)

display(
    paper_candidates.head(10)
)

KeyError: 'report_id'

In [46]:
%%writefile src/politdata/report_details.py

from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import copy
import hashlib
import json
import os
import time
import uuid

import pandas as pd
import requests


DEFAULT_BASE_URL = (
    "https://politdata.nazk.gov.ua/api/v2"
)

DEFAULT_RAW_DIR = Path(
    "data/raw/report_details"
)

DEFAULT_STATE_PATH = Path(
    "data/interim/state/"
    "report_detail_state.parquet"
)

RETRYABLE_STATUS_CODES = {
    408,
    429,
    500,
    502,
    503,
    504,
}


METADATA_COLUMNS = [
    "report_id",
    "organization_id",
    "root_party_id",
    "entity_type",
    "year",
    "quarter",
    "selection_method",
]


STATE_COLUMNS = [
    "status",
    "attempts",
    "last_checked_at_utc",
    "retrieved_at_utc",
    "raw_path",
    "raw_payload_hash",
    "content_hash",
    "file_size_bytes",
    "property_paper_count",
    "last_error",
]


# ============================================================
# TIME
# ============================================================

def utc_now_iso():
    return (
        datetime.now(
            timezone.utc
        )
        .isoformat()
    )


def file_mtime_utc(path):
    return (
        datetime.fromtimestamp(
            path.stat().st_mtime,
            tz=timezone.utc,
        )
        .isoformat()
    )


# ============================================================
# HASHING
# ============================================================

def _canonical_json_bytes(value):
    return json.dumps(
        value,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")


def raw_payload_hash(payload):
    """
    Exact hash of the entire API payload.

    public_summary IS included here because this hash is
    intended only to describe the exact stored response.
    """

    return hashlib.sha256(
        _canonical_json_bytes(
            payload
        )
    ).hexdigest()


def report_detail_content_hash(payload):
    """
    Semantic hash of source report detail.

    public_summary is explicitly excluded because it is
    a derived NACP layer and must not determine whether
    the underlying report content changed.
    """

    semantic = copy.deepcopy(
        payload
    )

    results = semantic.get(
        "results"
    )

    if isinstance(
        results,
        dict,
    ):
        results.pop(
            "public_summary",
            None,
        )

    return hashlib.sha256(
        _canonical_json_bytes(
            semantic
        )
    ).hexdigest()


# ============================================================
# VALIDATION
# ============================================================

def validate_report_detail_payload(
    payload,
    expected_report_id,
):
    if not isinstance(
        payload,
        dict,
    ):
        raise ValueError(
            "Report detail response "
            "is not a dictionary."
        )

    code = payload.get(
        "code"
    )

    if (
        code is not None
        and code != 0
    ):
        raise ValueError(
            f"Unexpected API code: {code}"
        )

    results = payload.get(
        "results"
    )

    if not isinstance(
        results,
        dict,
    ):
        raise ValueError(
            "Missing results object."
        )

    actual_id = results.get(
        "id"
    )

    if str(actual_id) != str(
        expected_report_id
    ):
        raise ValueError(
            "Report ID mismatch: "
            f"expected={expected_report_id}, "
            f"actual={actual_id}"
        )

    return results


def property_paper_count(
    payload,
):
    results = payload.get(
        "results"
    ) or {}

    properties = results.get(
        "properties"
    ) or {}

    rows = properties.get(
        "property_paper"
    ) or []

    return len(rows)


# ============================================================
# ATOMIC FILE WRITES
# ============================================================

def _atomic_replace(
    temp_path,
    final_path,
    max_retries=20,
):

    final_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    last_error = None

    for attempt in range(
        1,
        max_retries + 1,
    ):
        try:

            os.replace(
                temp_path,
                final_path,
            )

            return

        except PermissionError as exc:

            last_error = exc

            if attempt == max_retries:
                break

            time.sleep(
                0.25 * attempt
            )

    raise last_error


def atomic_write_json(
    path,
    payload,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_name(
        path.name
        + ".tmp."
        + uuid.uuid4().hex
    )

    try:

        with temp_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            # Compact JSON materially reduces storage
            # for tens of thousands of reports.
            json.dump(
                payload,
                f,
                ensure_ascii=False,
                separators=(",", ":"),
            )

            f.flush()

            os.fsync(
                f.fileno()
            )

        _atomic_replace(
            temp_path,
            path,
        )

    finally:

        if temp_path.exists():
            try:
                temp_path.unlink()
            except OSError:
                pass


def atomic_write_parquet(
    df,
    path,
):
    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp_path = path.with_name(
        path.stem
        + ".tmp."
        + uuid.uuid4().hex
        + path.suffix
    )

    try:

        df.to_parquet(
            temp_path,
            index=False,
        )

        _atomic_replace(
            temp_path,
            path,
        )

    finally:

        if temp_path.exists():
            try:
                temp_path.unlink()
            except OSError:
                pass


def try_save_state(
    state,
    state_path,
):
    try:

        atomic_write_parquet(
            state,
            state_path,
        )

        return True

    except PermissionError as exc:

        print(
            "WARNING: state checkpoint "
            "could not be saved:"
        )

        print(
            repr(exc)
        )

        print(
            "RAW files already written "
            "remain recoverable."
        )

        return False


# ============================================================
# STATE
# ============================================================

def initialize_report_detail_state(
    selected_reports,
    state_path=DEFAULT_STATE_PATH,
):

    missing = [
        col
        for col
        in METADATA_COLUMNS
        if col
        not in selected_reports.columns
    ]

    if missing:
        raise ValueError(
            "Missing selected-report columns: "
            + ", ".join(missing)
        )

    base = (
        selected_reports[
            METADATA_COLUMNS
        ]
        .drop_duplicates(
            subset=[
                "report_id"
            ]
        )
        .copy()
    )

    base[
        "report_id"
    ] = base[
        "report_id"
    ].astype(str)

    state_path = Path(
        state_path
    )

    if state_path.exists():

        existing = pd.read_parquet(
            state_path
        )

        existing[
            "report_id"
        ] = existing[
            "report_id"
        ].astype(str)

        for col in STATE_COLUMNS:

            if col not in (
                existing.columns
            ):
                existing[col] = None

        existing_ops = (
            existing[
                [
                    "report_id"
                ]
                + STATE_COLUMNS
            ]
            .drop_duplicates(
                subset=[
                    "report_id"
                ]
            )
        )

        state = base.merge(
            existing_ops,
            on="report_id",
            how="left",
        )

    else:

        state = base.copy()

        for col in STATE_COLUMNS:
            state[col] = None


    state[
        "status"
    ] = state[
        "status"
    ].fillna(
        "pending"
    )

    state[
        "attempts"
    ] = (
        pd.to_numeric(
            state[
                "attempts"
            ],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
    )

    return state


# ============================================================
# RAW RECOVERY
# ============================================================

def reconcile_state_from_raw(
    state,
    raw_dir=DEFAULT_RAW_DIR,
):

    raw_dir = Path(
        raw_dir
    )

    recovered = 0

    state = state.copy()

    candidate_mask = (
        state[
            "status"
        ]
        != "success"
    )

    candidate_indices = (
        state.index[
            candidate_mask
        ]
        .tolist()
    )

    for idx in candidate_indices:

        report_id = str(
            state.at[
                idx,
                "report_id",
            ]
        )

        path = (
            raw_dir
            / f"{report_id}.json"
        )

        if not path.exists():
            continue

        try:

            with path.open(
                "r",
                encoding="utf-8",
            ) as f:

                payload = json.load(f)

            validate_report_detail_payload(
                payload,
                report_id,
            )

            state.at[
                idx,
                "status",
            ] = "success"

            state.at[
                idx,
                "raw_path",
            ] = str(path)

            state.at[
                idx,
                "raw_payload_hash",
            ] = raw_payload_hash(
                payload
            )

            state.at[
                idx,
                "content_hash",
            ] = (
                report_detail_content_hash(
                    payload
                )
            )

            state.at[
                idx,
                "file_size_bytes",
            ] = path.stat().st_size

            state.at[
                idx,
                "property_paper_count",
            ] = property_paper_count(
                payload
            )

            state.at[
                idx,
                "retrieved_at_utc",
            ] = file_mtime_utc(
                path
            )

            state.at[
                idx,
                "last_error",
            ] = None

            recovered += 1

        except Exception:
            # Existing file is not trusted if it
            # cannot be parsed and validated.
            continue

    return state, recovered


# ============================================================
# API FETCH
# ============================================================

def fetch_report_detail(
    report_id,
    base_url=DEFAULT_BASE_URL,
    timeout=180,
    max_retries=4,
    session=None,
):

    if session is None:
        session = requests.Session()

    url = (
        f"{base_url}"
        f"/party/report/{report_id}"
    )

    last_error = None

    for attempt in range(
        1,
        max_retries + 1,
    ):

        try:

            response = session.get(
                url,
                timeout=timeout,
            )

            if (
                response.status_code
                in RETRYABLE_STATUS_CODES
            ):

                raise requests.HTTPError(
                    "Retryable HTTP status "
                    f"{response.status_code}",
                    response=response,
                )

            response.raise_for_status()

            payload = response.json()

            validate_report_detail_payload(
                payload,
                report_id,
            )

            return payload

        except (
            requests.RequestException,
            ValueError,
            json.JSONDecodeError,
        ) as exc:

            last_error = exc

            retryable = True

            if isinstance(
                exc,
                requests.HTTPError,
            ):

                response = getattr(
                    exc,
                    "response",
                    None,
                )

                if (
                    response is not None
                    and response.status_code
                    not in RETRYABLE_STATUS_CODES
                ):
                    retryable = False

            if (
                not retryable
                or attempt
                == max_retries
            ):
                break

            sleep_seconds = (
                2 ** (attempt - 1)
            )

            time.sleep(
                sleep_seconds
            )

    raise last_error


# ============================================================
# MAIN BATCH RUNNER
# ============================================================

def run_report_detail_batch(
    selected_reports,
    limit=None,
    retry_errors=True,
    checkpoint_every=25,
    timeout=180,
    max_retries=4,
    base_url=DEFAULT_BASE_URL,
    raw_dir=DEFAULT_RAW_DIR,
    state_path=DEFAULT_STATE_PATH,
):

    raw_dir = Path(
        raw_dir
    )

    state_path = Path(
        state_path
    )

    raw_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    state = (
        initialize_report_detail_state(
            selected_reports,
            state_path=state_path,
        )
    )

    # Recover RAW written after the most recent
    # successful state checkpoint.
    state, recovered = (
        reconcile_state_from_raw(
            state,
            raw_dir=raw_dir,
        )
    )

    if recovered:

        print(
            "Recovered from RAW:",
            recovered
        )

        try_save_state(
            state,
            state_path,
        )


    candidate_statuses = [
        "pending",
    ]

    if retry_errors:
        candidate_statuses.append(
            "error"
        )

    candidates = state[
        state[
            "status"
        ].isin(
            candidate_statuses
        )
    ]

    if limit is not None:

        candidates = (
            candidates.head(
                int(limit)
            )
        )

    report_ids = (
        candidates[
            "report_id"
        ]
        .astype(str)
        .tolist()
    )

    print(
        "Selected for download:",
        len(report_ids)
    )

    if not report_ids:

        return (
            {
                "selected": 0,
                "successful": 0,
                "failed": 0,
                "recovered_from_raw":
                    recovered,
                "bytes_written":
                    0,
                "total_success_in_state":
                    int(
                        (
                            state[
                                "status"
                            ]
                            == "success"
                        ).sum()
                    ),
                "total_errors_in_state":
                    int(
                        (
                            state[
                                "status"
                            ]
                            == "error"
                        ).sum()
                    ),
                "total_pending_in_state":
                    int(
                        (
                            state[
                                "status"
                            ]
                            == "pending"
                        ).sum()
                    ),
            },
            state,
        )


    # Efficient state updates by report_id.
    state = state.set_index(
        "report_id",
        drop=False,
    )

    session = requests.Session()

    successful = 0
    failed = 0
    bytes_written = 0

    paper_reports = []

    try:

        for n, report_id in enumerate(
            report_ids,
            start=1,
        ):

            checked_at = (
                utc_now_iso()
            )

            state.at[
                report_id,
                "attempts",
            ] = int(
                state.at[
                    report_id,
                    "attempts",
                ]
            ) + 1

            state.at[
                report_id,
                "last_checked_at_utc",
            ] = checked_at

            try:

                payload = (
                    fetch_report_detail(
                        report_id,
                        base_url=base_url,
                        timeout=timeout,
                        max_retries=max_retries,
                        session=session,
                    )
                )

                output_path = (
                    raw_dir
                    / f"{report_id}.json"
                )

                # RAW first, state second.
                atomic_write_json(
                    output_path,
                    payload,
                )

                retrieved_at = (
                    utc_now_iso()
                )

                size_bytes = (
                    output_path
                    .stat()
                    .st_size
                )

                paper_count = (
                    property_paper_count(
                        payload
                    )
                )

                state.at[
                    report_id,
                    "status",
                ] = "success"

                state.at[
                    report_id,
                    "retrieved_at_utc",
                ] = retrieved_at

                state.at[
                    report_id,
                    "raw_path",
                ] = str(
                    output_path
                )

                state.at[
                    report_id,
                    "raw_payload_hash",
                ] = raw_payload_hash(
                    payload
                )

                state.at[
                    report_id,
                    "content_hash",
                ] = (
                    report_detail_content_hash(
                        payload
                    )
                )

                state.at[
                    report_id,
                    "file_size_bytes",
                ] = size_bytes

                state.at[
                    report_id,
                    "property_paper_count",
                ] = paper_count

                state.at[
                    report_id,
                    "last_error",
                ] = None

                successful += 1

                bytes_written += (
                    size_bytes
                )

                if paper_count > 0:

                    paper_reports.append(
                        (
                            report_id,
                            paper_count,
                        )
                    )

            except Exception as exc:

                state.at[
                    report_id,
                    "status",
                ] = "error"

                state.at[
                    report_id,
                    "last_error",
                ] = repr(exc)

                failed += 1

                print()
                print(
                    "ERROR:",
                    report_id,
                )

                print(
                    repr(exc)
                )


            if (
                n % checkpoint_every
                == 0
                or n
                == len(report_ids)
            ):

                state_for_save = (
                    state.reset_index(
                        drop=True
                    )
                )

                try_save_state(
                    state_for_save,
                    state_path,
                )

                print(
                    f"{n}/{len(report_ids)} "
                    f"processed | "
                    f"success={successful} | "
                    f"errors={failed}"
                )

    finally:

        session.close()


    state = state.reset_index(
        drop=True
    )

    try_save_state(
        state,
        state_path,
    )


    if paper_reports:

        print()
        print(
            "NON-EMPTY property_paper "
            "FOUND:"
        )

        for (
            report_id,
            count
        ) in paper_reports[:10]:

            print(
                report_id,
                count,
            )


    summary = {
        "selected":
            len(report_ids),

        "successful":
            successful,

        "failed":
            failed,

        "recovered_from_raw":
            recovered,

        "bytes_written":
            bytes_written,

        "total_success_in_state":
            int(
                (
                    state[
                        "status"
                    ]
                    == "success"
                ).sum()
            ),

        "total_errors_in_state":
            int(
                (
                    state[
                        "status"
                    ]
                    == "error"
                ).sum()
            ),

        "total_pending_in_state":
            int(
                (
                    state[
                        "status"
                    ]
                    == "pending"
                ).sum()
            ),

        "reports_with_paper_this_run":
            len(
                paper_reports
            ),
    }

    return (
        summary,
        state,
    )

Writing src/politdata/report_details.py


In [48]:
from pathlib import Path
import pandas as pd

selected_reports = pd.read_parquet(
    "data/interim/reports/"
    "selected_reports_manifest.parquet"
)

print(
    "Selected reports:",
    len(selected_reports)
)

detail_test_summary, detail_state = (
    run_report_detail_batch(
        selected_reports,
        limit=20,
        timeout=180,
        max_retries=4,
        checkpoint_every=5,
    )
)

print()
print(
    detail_test_summary
)

print()
print(
    pd.crosstab(
        detail_state[
            "entity_type"
        ],
        detail_state[
            "status"
        ],
    )
)

Selected reports: 78791
Selected for download: 20
5/20 processed | success=5 | errors=0
10/20 processed | success=10 | errors=0
15/20 processed | success=15 | errors=0
20/20 processed | success=20 | errors=0

{'selected': 20, 'successful': 20, 'failed': 0, 'recovered_from_raw': 0, 'bytes_written': 125047, 'total_success_in_state': 20, 'total_errors_in_state': 0, 'total_pending_in_state': 78771, 'reports_with_paper_this_run': 0}

status       pending  success
entity_type                  
office         74944       20
party           3827        0


In [49]:
from pathlib import Path
import pandas as pd

selected_reports = pd.read_parquet(
    "data/interim/reports/"
    "selected_reports_manifest.parquet"
)

print(
    "Selected reports:",
    len(selected_reports)
)

detail_test_summary, detail_state = (
    run_report_detail_batch(
        selected_reports,
        limit=20,
        timeout=180,
        max_retries=4,
        checkpoint_every=5,
    )
)

print()
print(
    detail_test_summary
)

print()
print(
    pd.crosstab(
        detail_state[
            "entity_type"
        ],
        detail_state[
            "status"
        ],
    )
)

Selected reports: 78791
Selected for download: 20
5/20 processed | success=5 | errors=0
10/20 processed | success=10 | errors=0
15/20 processed | success=15 | errors=0
20/20 processed | success=20 | errors=0

{'selected': 20, 'successful': 20, 'failed': 0, 'recovered_from_raw': 0, 'bytes_written': 25047, 'total_success_in_state': 40, 'total_errors_in_state': 0, 'total_pending_in_state': 78751, 'reports_with_paper_this_run': 0}

status       pending  success
entity_type                  
office         74924       40
party           3827        0


In [52]:
from pathlib import Path
import importlib
import textwrap

module_path = Path(
    "src/politdata/report_details.py"
)

source = module_path.read_text(
    encoding="utf-8"
)

marker = "\ndef run_report_detail_batch("

if marker not in source:
    raise RuntimeError(
        "run_report_detail_batch() not found "
        "in report_details.py"
    )

prefix = source.split(
    marker,
    1,
)[0]


new_function = r'''
def run_report_detail_batch(
    selected_reports,
    limit=None,
    retry_errors=True,
    checkpoint_every=25,
    timeout=180,
    max_retries=4,
    base_url=DEFAULT_BASE_URL,
    raw_dir=DEFAULT_RAW_DIR,
    state_path=DEFAULT_STATE_PATH,
):

    from tqdm.auto import tqdm

    raw_dir = Path(raw_dir)
    state_path = Path(state_path)

    raw_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    state = initialize_report_detail_state(
        selected_reports,
        state_path=state_path,
    )

    # Recover valid RAW files that may exist
    # after an interrupted run.
    state, recovered = reconcile_state_from_raw(
        state,
        raw_dir=raw_dir,
    )

    if recovered:

        try_save_state(
            state,
            state_path,
        )

    candidate_statuses = [
        "pending",
    ]

    if retry_errors:
        candidate_statuses.append(
            "error"
        )

    candidates = state[
        state["status"].isin(
            candidate_statuses
        )
    ]

    if limit is not None:

        candidates = candidates.head(
            int(limit)
        )

    report_ids = (
        candidates["report_id"]
        .astype(str)
        .tolist()
    )

    print(
        "Selected for download:",
        len(report_ids)
    )

    if not report_ids:

        summary = {
            "selected": 0,
            "successful": 0,
            "failed": 0,
            "recovered_from_raw":
                recovered,
            "bytes_written": 0,
            "total_success_in_state":
                int(
                    (
                        state["status"]
                        == "success"
                    ).sum()
                ),
            "total_errors_in_state":
                int(
                    (
                        state["status"]
                        == "error"
                    ).sum()
                ),
            "total_pending_in_state":
                int(
                    (
                        state["status"]
                        == "pending"
                    ).sum()
                ),
            "reports_with_paper_this_run":
                0,
        }

        return (
            summary,
            state,
        )

    state = state.set_index(
        "report_id",
        drop=False,
    )

    session = requests.Session()

    successful = 0
    failed = 0
    bytes_written = 0

    paper_reports = []

    progress = tqdm(
        report_ids,
        total=len(report_ids),
        desc="Report details",
        unit="report",
        dynamic_ncols=True,
        mininterval=1.0,
        smoothing=0.1,
    )

    try:

        for n, report_id in enumerate(
            progress,
            start=1,
        ):

            checked_at = utc_now_iso()

            state.at[
                report_id,
                "attempts",
            ] = (
                int(
                    state.at[
                        report_id,
                        "attempts",
                    ]
                )
                + 1
            )

            state.at[
                report_id,
                "last_checked_at_utc",
            ] = checked_at

            try:

                payload = fetch_report_detail(
                    report_id,
                    base_url=base_url,
                    timeout=timeout,
                    max_retries=max_retries,
                    session=session,
                )

                output_path = (
                    raw_dir
                    / f"{report_id}.json"
                )

                # RAW first, state second.
                atomic_write_json(
                    output_path,
                    payload,
                )

                retrieved_at = utc_now_iso()

                size_bytes = (
                    output_path
                    .stat()
                    .st_size
                )

                paper_count = (
                    property_paper_count(
                        payload
                    )
                )

                state.at[
                    report_id,
                    "status",
                ] = "success"

                state.at[
                    report_id,
                    "retrieved_at_utc",
                ] = retrieved_at

                state.at[
                    report_id,
                    "raw_path",
                ] = str(
                    output_path
                )

                state.at[
                    report_id,
                    "raw_payload_hash",
                ] = raw_payload_hash(
                    payload
                )

                state.at[
                    report_id,
                    "content_hash",
                ] = (
                    report_detail_content_hash(
                        payload
                    )
                )

                state.at[
                    report_id,
                    "file_size_bytes",
                ] = size_bytes

                state.at[
                    report_id,
                    "property_paper_count",
                ] = paper_count

                state.at[
                    report_id,
                    "last_error",
                ] = None

                successful += 1
                bytes_written += size_bytes

                if paper_count > 0:

                    paper_reports.append(
                        (
                            report_id,
                            paper_count,
                        )
                    )

            except Exception as exc:

                state.at[
                    report_id,
                    "status",
                ] = "error"

                state.at[
                    report_id,
                    "last_error",
                ] = repr(exc)

                failed += 1

            # Save state silently.
            if (
                n % checkpoint_every == 0
                or n == len(report_ids)
            ):

                state_for_save = (
                    state.reset_index(
                        drop=True
                    )
                )

                try_save_state(
                    state_for_save,
                    state_path,
                )

            # Update only the existing progress bar.
            if (
                n % 25 == 0
                or n == len(report_ids)
            ):

                progress.set_postfix(
                    success=successful,
                    errors=failed,
                    refresh=False,
                )

    finally:

        progress.close()
        session.close()

    state = state.reset_index(
        drop=True
    )

    try_save_state(
        state,
        state_path,
    )

    if paper_reports:

        print(
            "Non-empty property_paper:",
            len(paper_reports),
        )

        print(
            "First examples:",
            paper_reports[:5],
        )

    summary = {
        "selected":
            len(report_ids),

        "successful":
            successful,

        "failed":
            failed,

        "recovered_from_raw":
            recovered,

        "bytes_written":
            bytes_written,

        "total_success_in_state":
            int(
                (
                    state["status"]
                    == "success"
                ).sum()
            ),

        "total_errors_in_state":
            int(
                (
                    state["status"]
                    == "error"
                ).sum()
            ),

        "total_pending_in_state":
            int(
                (
                    state["status"]
                    == "pending"
                ).sum()
            ),

        "reports_with_paper_this_run":
            len(paper_reports),
    }

    return (
        summary,
        state,
    )
'''


module_path.write_text(
    prefix
    + "\n"
    + textwrap.dedent(
        new_function
    ).lstrip(),
    encoding="utf-8",
)


# Reload edited module
import politdata.report_details as report_details

importlib.reload(
    report_details
)

run_report_detail_batch = (
    report_details.run_report_detail_batch
)

print(
    "report_details.py updated and reloaded."
)

report_details.py updated and reloaded.


In [53]:
detail_summary, detail_state = (
    run_report_detail_batch(
        selected_reports,
        limit=None,
        timeout=180,
        max_retries=4,
        checkpoint_every=25,
    )
)

Selected for download: 77895


Report details:   0%|                                                                    | 0/77895 [00:00<?, ?…

In [54]:
from pathlib import Path
import pandas as pd
import numpy as np


REPORTS_DIR = Path(
    "data/interim/reports"
)

STATE_DIR = Path(
    "data/interim/state"
)


all_reports = pd.read_parquet(
    REPORTS_DIR
    / "all_reports_manifest.parquet"
)

selected_reports = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
)


# ----------------------------------------------------------
# BASIC QA
# ----------------------------------------------------------

all_reports["report_id"] = (
    all_reports["report_id"]
    .astype(str)
)

selected_reports["report_id"] = (
    selected_reports["report_id"]
    .astype(str)
)

selected_ids = set(
    selected_reports["report_id"]
)

excluded = (
    all_reports[
        ~all_reports[
            "report_id"
        ].isin(selected_ids)
    ]
    .copy()
)

print(
    "All report instances:",
    len(all_reports)
)

print(
    "Selected reports:",
    len(selected_reports)
)

print(
    "Excluded report instances:",
    len(excluded)
)

assert len(all_reports) == 79004
assert len(selected_reports) == 78791
assert len(excluded) == 213


# ----------------------------------------------------------
# FIND SELECTED COUNTERPART IN SAME LOGICAL PERIOD
# ----------------------------------------------------------

PERIOD_COLS = [
    "organization_id",
    "year",
    "quarter",
]

for col in PERIOD_COLS:

    assert col in all_reports.columns, (
        f"Missing in all_reports: {col}"
    )

    assert col in selected_reports.columns, (
        f"Missing in selected_reports: {col}"
    )


selected_periods = (
    selected_reports[
        PERIOD_COLS
        + ["report_id"]
    ]
    .rename(
        columns={
            "report_id":
                "selected_counterpart_report_id"
        }
    )
)


excluded = excluded.merge(
    selected_periods,
    on=PERIOD_COLS,
    how="left",
)


excluded[
    "excluded_reason"
] = np.where(
    excluded[
        "selected_counterpart_report_id"
    ].notna(),

    "nonselected_instance_in_selected_period",

    "period_without_selected_signed_report",
)


print()
print(
    excluded[
        "excluded_reason"
    ].value_counts()
)


# ----------------------------------------------------------
# MAKE MANIFEST COMPATIBLE WITH REPORT DETAIL DOWNLOADER
# ----------------------------------------------------------

# Normally these are already present.
# If not, restore them from organizations.parquet.

organizations = pd.read_parquet(
    "data/processed/normalized_v0_1/"
    "organizations.parquet"
)

org_meta = (
    organizations[
        [
            "organization_id",
            "root_party_id",
            "entity_type",
        ]
    ]
    .drop_duplicates(
        subset=[
            "organization_id"
        ]
    )
)


for col in [
    "root_party_id",
    "entity_type",
]:

    if col not in excluded.columns:

        excluded = excluded.merge(
            org_meta[
                [
                    "organization_id",
                    col,
                ]
            ],
            on="organization_id",
            how="left",
        )


# Downloader requires this column.
# Here it records WHY this source instance was not
# in our primary selected-report set.

excluded[
    "selection_method"
] = excluded[
    "excluded_reason"
]


excluded_manifest_path = (
    REPORTS_DIR
    / "excluded_report_instances_manifest.parquet"
)

excluded.to_parquet(
    excluded_manifest_path,
    index=False,
)


print()
print(
    "Saved:",
    excluded_manifest_path
)


# ----------------------------------------------------------
# DOWNLOAD ALL 213
# ----------------------------------------------------------

excluded_summary, excluded_state = (
    run_report_detail_batch(
        excluded,
        limit=None,

        # SAME RAW corpus:
        raw_dir=(
            "data/raw/report_details"
        ),

        # SEPARATE state so we do not overwrite
        # the 78,791-report ingestion state:
        state_path=(
            STATE_DIR
            / "report_detail_excluded_state.parquet"
        ),

        timeout=180,
        max_retries=4,
        checkpoint_every=25,
    )
)


print()
print(
    excluded_summary
)

print()
print(
    excluded_state[
        "status"
    ].value_counts(
        dropna=False
    )
)

All report instances: 79004
Selected reports: 78791
Excluded report instances: 213

excluded_reason
nonselected_instance_in_selected_period    200
period_without_selected_signed_report       13
Name: count, dtype: int64

Saved: data\interim\reports\excluded_report_instances_manifest.parquet
Selected for download: 213


Report details:   0%|                                                                      | 0/213 [00:00<?, ?…


{'selected': 213, 'successful': 213, 'failed': 0, 'recovered_from_raw': 0, 'bytes_written': 1039867, 'total_success_in_state': 213, 'total_errors_in_state': 0, 'total_pending_in_state': 0, 'reports_with_paper_this_run': 0}

status
success    213
Name: count, dtype: int64


In [55]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import hashlib
import json

import pandas as pd


RAW_DIR = Path(
    "data/raw/report_details"
)

OUT_DIR = Path(
    "data/interim/reports"
)


PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


# ==========================================================
# HELPERS
# ==========================================================

def to_decimal(value):

    if value is None:
        return Decimal("0")

    if isinstance(
        value,
        bool,
    ):
        return Decimal("0")

    try:

        return Decimal(
            str(value)
        )

    except (
        InvalidOperation,
        ValueError,
        TypeError,
    ):

        return Decimal("0")


def load_report_detail(
    report_id,
):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        payload = json.load(f)

    return payload[
        "results"
    ]


def payment_rows(
    detail,
    payment_type,
):

    direction, key = (
        PAYMENT_PATHS[
            payment_type
        ]
    )

    payment_info = (
        detail.get(
            "payment_info"
        )
        or {}
    )

    return (
        (
            payment_info.get(
                direction
            )
            or {}
        )
        .get(key)
        or []
    )


def sum_field(
    rows,
    field,
):

    return sum(
        (
            to_decimal(
                row.get(field)
            )
            for row in rows
            if isinstance(
                row,
                dict,
            )
        ),
        Decimal("0"),
    )


def effective_return_sum(
    rows,
):
    """
    For return rows:
    prefer refund_amount when present;
    otherwise fall back to payment_amount.

    We do NOT add both fields from the same row.
    """

    total = Decimal("0")

    for row in rows:

        if not isinstance(
            row,
            dict,
        ):
            continue

        if row.get(
            "refund_amount"
        ) is not None:

            total += to_decimal(
                row.get(
                    "refund_amount"
                )
            )

        else:

            total += to_decimal(
                row.get(
                    "payment_amount"
                )
            )

    return total


def normalized_payment_info(
    detail,
):
    """
    Build substantive financial representation.

    created_at / updated_at are excluded because
    we want to detect changes in reported finance,
    not API technical timestamps.
    """

    normalized = {}

    for payment_type in PAYMENT_PATHS:

        rows = payment_rows(
            detail,
            payment_type,
        )

        clean_rows = []

        for row in rows:

            if not isinstance(
                row,
                dict,
            ):
                continue

            clean = {
                k: v
                for k, v
                in row.items()
                if k not in {
                    "created_at",
                    "updated_at",
                }
            }

            clean_rows.append(
                clean
            )

        clean_rows.sort(
            key=lambda x: str(
                x.get("id", "")
            )
        )

        normalized[
            payment_type
        ] = clean_rows

    return normalized


def financial_content_hash(
    detail,
):

    canonical = json.dumps(
        normalized_payment_info(
            detail
        ),
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )

    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


def summarize_finance(
    detail,
):

    result = {}

    total_rows = 0

    category_amounts = {}

    for payment_type in PAYMENT_PATHS:

        rows = payment_rows(
            detail,
            payment_type,
        )

        row_count = len(rows)

        total_rows += row_count

        payment_amount = (
            sum_field(
                rows,
                "payment_amount",
            )
        )

        refund_amount = (
            sum_field(
                rows,
                "refund_amount",
            )
        )

        refund_budget_amount = (
            sum_field(
                rows,
                "refund_budget_amount",
            )
        )

        result[
            f"{payment_type}_rows"
        ] = row_count

        result[
            f"{payment_type}_payment_amount"
        ] = float(
            payment_amount
        )

        result[
            f"{payment_type}_refund_amount"
        ] = float(
            refund_amount
        )

        result[
            f"{payment_type}_refund_budget_amount"
        ] = float(
            refund_budget_amount
        )

        category_amounts[
            payment_type
        ] = payment_amount


    # ----------------------------------------------
    # ANALYTICAL SCREENING TOTALS
    # ----------------------------------------------

    cash_inflow = (
        category_amounts[
            "monetary_contributions"
        ]
        +
        category_amounts[
            "state_funding"
        ]
        +
        category_amounts[
            "other_incomes"
        ]
    )

    nonmonetary = (
        category_amounts[
            "other_contributions"
        ]
    )

    spending = (
        category_amounts[
            "budget_expenses"
        ]
        +
        category_amounts[
            "outgoing_expenses"
        ]
    )

    return_rows = payment_rows(
        detail,
        "return_expenses",
    )

    returns = (
        effective_return_sum(
            return_rows
        )
    )

    transfers = (
        category_amounts[
            "transfer_expenses"
        ]
    )


    # This is an ACTIVITY INDICATOR,
    # not party-level net cash flow.
    #
    # Transfers/refunds can represent circulation rather
    # than new economic inflow/outflow.

    gross_cash_activity = (
        abs(cash_inflow)
        +
        abs(spending)
        +
        abs(returns)
        +
        abs(transfers)
    )


    result.update(
        {
            "financial_rows_total":
                total_rows,

            "cash_inflow_total":
                float(cash_inflow),

            "nonmonetary_contributions_total":
                float(nonmonetary),

            "spending_total":
                float(spending),

            "returns_total":
                float(returns),

            "transfers_total":
                float(transfers),

            "gross_cash_activity_indicator":
                float(
                    gross_cash_activity
                ),

            "has_financial_rows":
                total_rows > 0,

            "has_nonzero_cash_activity":
                gross_cash_activity != 0,

            "financial_content_hash":
                financial_content_hash(
                    detail
                ),
        }
    )

    return result


# ==========================================================
# REPORT-LEVEL ANALYSIS
# ==========================================================

excluded = pd.read_parquet(
    OUT_DIR
    / "excluded_report_instances_manifest.parquet"
)

rows = []


for n, report in excluded.iterrows():

    report_id = str(
        report[
            "report_id"
        ]
    )

    detail = load_report_detail(
        report_id
    )

    finance = summarize_finance(
        detail
    )

    row = {
        "report_id":
            report_id,

        "organization_id":
            report.get(
                "organization_id"
            ),

        "root_party_id":
            report.get(
                "root_party_id"
            ),

        "entity_type":
            report.get(
                "entity_type"
            ),

        "year":
            report.get(
                "year"
            ),

        "quarter":
            report.get(
                "quarter"
            ),

        "signed_date":
            report.get(
                "signed_date"
            ),

        "created_date":
            report.get(
                "created_date"
            ),

        "excluded_reason":
            report.get(
                "excluded_reason"
            ),

        "selected_counterpart_report_id":
            report.get(
                "selected_counterpart_report_id"
            ),
    }

    row.update(
        finance
    )


    # ----------------------------------------------
    # COMPARE TO SELECTED VERSION OF SAME PERIOD
    # ----------------------------------------------

    counterpart_id = row[
        "selected_counterpart_report_id"
    ]

    if (
        pd.notna(
            counterpart_id
        )
        and str(
            counterpart_id
        )
        not in {
            "",
            "nan",
            "None",
        }
    ):

        counterpart_id = str(
            counterpart_id
        )

        counterpart_detail = (
            load_report_detail(
                counterpart_id
            )
        )

        counterpart_finance = (
            summarize_finance(
                counterpart_detail
            )
        )

        row[
            "same_financial_content_as_selected"
        ] = (
            finance[
                "financial_content_hash"
            ]
            ==
            counterpart_finance[
                "financial_content_hash"
            ]
        )

        row[
            "selected_cash_inflow_total"
        ] = counterpart_finance[
            "cash_inflow_total"
        ]

        row[
            "selected_spending_total"
        ] = counterpart_finance[
            "spending_total"
        ]

        row[
            "selected_gross_cash_activity_indicator"
        ] = counterpart_finance[
            "gross_cash_activity_indicator"
        ]

        row[
            "cash_inflow_difference_vs_selected"
        ] = (
            finance[
                "cash_inflow_total"
            ]
            -
            counterpart_finance[
                "cash_inflow_total"
            ]
        )

        row[
            "spending_difference_vs_selected"
        ] = (
            finance[
                "spending_total"
            ]
            -
            counterpart_finance[
                "spending_total"
            ]
        )

    else:

        row[
            "same_financial_content_as_selected"
        ] = None


    rows.append(
        row
    )


excluded_finance = (
    pd.DataFrame(
        rows
    )
)


# ==========================================================
# PARTY NAMES
# ==========================================================

organizations = pd.read_parquet(
    "data/processed/normalized_v0_1/"
    "organizations.parquet"
)


NAME_CANDIDATES = [
    "name",
    "organization_name",
    "party_name",
    "full_name",
]

name_col = next(
    (
        col
        for col
        in NAME_CANDIDATES
        if col
        in organizations.columns
    ),
    None,
)


if name_col is not None:

    org_names = (
        organizations[
            [
                "organization_id",
                name_col,
            ]
        ]
        .drop_duplicates(
            subset=[
                "organization_id"
            ]
        )
        .set_index(
            "organization_id"
        )[
            name_col
        ]
    )

    excluded_finance[
        "organization_name"
    ] = excluded_finance[
        "organization_id"
    ].map(
        org_names
    )

    excluded_finance[
        "party_name"
    ] = excluded_finance[
        "root_party_id"
    ].map(
        org_names
    )

else:

    excluded_finance[
        "organization_name"
    ] = None

    excluded_finance[
        "party_name"
    ] = None


# ==========================================================
# SAVE REPORT LEVEL
# ==========================================================

excluded_finance = (
    excluded_finance.sort_values(
        [
            "gross_cash_activity_indicator",
            "financial_rows_total",
        ],
        ascending=[
            False,
            False,
        ],
    )
)


report_output = (
    OUT_DIR
    / "excluded_report_financial_impact.parquet"
)

excluded_finance.to_parquet(
    report_output,
    index=False,
)


csv_output = (
    OUT_DIR
    / "excluded_report_financial_impact.csv"
)

excluded_finance.to_csv(
    csv_output,
    index=False,
    encoding="utf-8-sig",
)


# ==========================================================
# PARTY LEVEL
# ==========================================================

party_finance = (
    excluded_finance
    .groupby(
        [
            "root_party_id",
            "party_name",
        ],
        dropna=False,
    )
    .agg(
        excluded_report_instances=(
            "report_id",
            "count",
        ),

        reports_with_financial_rows=(
            "has_financial_rows",
            "sum",
        ),

        reports_with_nonzero_cash_activity=(
            "has_nonzero_cash_activity",
            "sum",
        ),

        cash_inflow_total=(
            "cash_inflow_total",
            "sum",
        ),

        nonmonetary_contributions_total=(
            "nonmonetary_contributions_total",
            "sum",
        ),

        spending_total=(
            "spending_total",
            "sum",
        ),

        returns_total=(
            "returns_total",
            "sum",
        ),

        transfers_total=(
            "transfers_total",
            "sum",
        ),

        gross_cash_activity_indicator=(
            "gross_cash_activity_indicator",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        "gross_cash_activity_indicator",
        ascending=False,
    )
)


party_output = (
    OUT_DIR
    / "excluded_party_financial_impact.parquet"
)

party_finance.to_parquet(
    party_output,
    index=False,
)


party_csv = (
    OUT_DIR
    / "excluded_party_financial_impact.csv"
)

party_finance.to_csv(
    party_csv,
    index=False,
    encoding="utf-8-sig",
)


# ==========================================================
# FINAL QA / SCREENING
# ==========================================================

print("=" * 70)
print("EXCLUDED REPORT FINANCIAL IMPACT")
print("=" * 70)

print(
    "Excluded instances:",
    len(
        excluded_finance
    )
)

print(
    "With financial rows:",
    int(
        excluded_finance[
            "has_financial_rows"
        ].sum()
    )
)

print(
    "With non-zero cash activity:",
    int(
        excluded_finance[
            "has_nonzero_cash_activity"
        ].sum()
    )
)

print()

print(
    "Financial comparison to selected:"
)

print(
    excluded_finance[
        "same_financial_content_as_selected"
    ].value_counts(
        dropna=False
    )
)

print()

print(
    "Total cash inflow:",
    excluded_finance[
        "cash_inflow_total"
    ].sum()
)

print(
    "Total spending:",
    excluded_finance[
        "spending_total"
    ].sum()
)

print(
    "Total returns:",
    excluded_finance[
        "returns_total"
    ].sum()
)

print(
    "Total transfers:",
    excluded_finance[
        "transfers_total"
    ].sum()
)

print()

print("=" * 70)
print("TOP EXCLUDED REPORTS BY CASH ACTIVITY")
print("=" * 70)

display(
    excluded_finance[
        [
            "party_name",
            "organization_name",
            "report_id",
            "year",
            "quarter",
            "excluded_reason",
            "financial_rows_total",
            "cash_inflow_total",
            "spending_total",
            "returns_total",
            "transfers_total",
            "gross_cash_activity_indicator",
            "same_financial_content_as_selected",
        ]
    ]
    .head(30)
)


print()
print("=" * 70)
print("PARTIES WITH LARGEST EXCLUDED ACTIVITY")
print("=" * 70)

display(
    party_finance.head(30)
)

EXCLUDED REPORT FINANCIAL IMPACT
Excluded instances: 213
With financial rows: 6
With non-zero cash activity: 2

Financial comparison to selected:
same_financial_content_as_selected
True     169
False     31
None      13
Name: count, dtype: int64

Total cash inflow: 2100000.0
Total spending: 2155994.42
Total returns: 0.0
Total transfers: 0.0

TOP EXCLUDED REPORTS BY CASH ACTIVITY


,party_name,organization_name,report_id,year,quarter,excluded_reason,financial_rows_total,cash_inflow_total,spending_total,returns_total,transfers_total,gross_cash_activity_indicator,same_financial_content_as_selected
81,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,2025,2,nonselected_instance_in_selected_period,250,1050000.0,1077997.21,0.0,0.0,2127997.21,False
82,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,c0c83821-6cea-4b1e-8b6d-989b7b92ed1f,2025,2,nonselected_instance_in_selected_period,250,1050000.0,1077997.21,0.0,0.0,2127997.21,False
129,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,ДНІПРОПЕТРОВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНО...,63ccb8ae-f6d4-4ede-8a16-b0ee9062d1b7,2025,2,nonselected_instance_in_selected_period,3,0.0,0.00,0.0,0.0,0.00,False
28,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,58d8edd0-e9b8-11ee-92b7-3b3b09128e69,2023,5,nonselected_instance_in_selected_period,2,0.0,0.00,0.0,0.0,0.00,False
41,ПОЛІТИЧНА ПАРТІЯ «БЛОК ВІЛКУЛА «УКРАЇНСЬКА ПЕР...,КІРОВОГРАДСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,9b7ae41c-f4e6-4c75-bf5e-719804ef2796,2025,2,nonselected_instance_in_selected_period,2,0.0,0.00,0.0,0.0,0.00,False
177,ПОЛІТИЧНА ПАРТІЯ «СОЦІАЛЬНА РЕКОНСТРУКЦІЯ»,ДНІПРОПЕТРОВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНО...,5d98002c-a312-4758-8a5a-23a6e28bd5ac,2025,2,nonselected_instance_in_selected_period,1,0.0,0.00,0.0,0.0,0.00,False
0,ПОЛІТИЧНА ПАРТІЯ «КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛ...,ВІННИЦЬКА ОБЛАСНА ОРГАНІЗАЦІЯ КОНГРЕСУ УКРАЇНС...,2e7283e0-6204-49e9-a44f-f7c688dc8a3f,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.0,0.0,0.00,True
1,ПОЛІТИЧНА ПАРТІЯ «КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛ...,ВІННИЦЬКА ОБЛАСНА ОРГАНІЗАЦІЯ КОНГРЕСУ УКРАЇНС...,721f2576-f40c-4b7e-bc5b-82d2db152cb3,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.0,0.0,0.00,True
2,ПОЛІТИЧНА ПАРТІЯ «КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛ...,ВІННИЦЬКА ОБЛАСНА ОРГАНІЗАЦІЯ КОНГРЕСУ УКРАЇНС...,95fdadc5-62c7-4c12-bb5d-b0ffe97a9429,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.0,0.0,0.00,True
3,ПОЛІТИЧНА ПАРТІЯ «КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛ...,ВІННИЦЬКА ОБЛАСНА ОРГАНІЗАЦІЯ КОНГРЕСУ УКРАЇНС...,16bb8c3f-b12e-44e7-83bb-94d1bc3e4409,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.0,0.0,0.00,True



PARTIES WITH LARGEST EXCLUDED ACTIVITY


,root_party_id,party_name,excluded_report_instances,reports_with_financial_rows,reports_with_nonzero_cash_activity,cash_inflow_total,nonmonetary_contributions_total,spending_total,returns_total,transfers_total,gross_cash_activity_indicator
27,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,6,2,2,2100000.0,0.0,2155994.42,0.0,0.0,4255994.42
1,0927ab46-3ba4-4c63-85b9-4e58d74c95d4,"Політична партія ""НАРОД""",3,0,0,0.0,0.0,0.00,0.0,0.0,0.00
0,01eddd1d-e3ec-41ad-b1f9-5a52a4d6d2fe,ПОЛІТИЧНА ПАРТІЯ ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «ГРО...,2,0,0,0.0,0.0,0.00,0.0,0.0,0.00
3,1388ae5f-2819-4153-84de-5463a956d3b4,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,1,1,0,0.0,5400.0,0.00,0.0,0.0,0.00
4,171b4fa0-5b63-4c2f-982c-a11a2b5ecabf,ПОЛІТИЧНА ПАРТІЯ «УКРАЇНСЬКА НАРОДНА ПАРТІЯ»,3,0,0,0.0,0.0,0.00,0.0,0.0,0.00
5,1e52ae20-1244-42ac-927d-7e014c80fc14,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БА...,6,0,0,0.0,0.0,0.00,0.0,0.0,0.00
6,21599f9d-983a-4889-be0b-03943d7f86f6,ПОЛІТИЧНА ПАРТІЯ «ВОЛЯ»,5,0,0,0.0,0.0,0.00,0.0,0.0,0.00
7,304fb45c-b6f2-471e-b01f-09e7c0e5ced2,"ПОЛІТИЧНА ПАРТІЯ ""УКРАЇНЦІ РАЗОМ""",7,0,0,0.0,0.0,0.00,0.0,0.0,0.00
8,35579664-92f4-4b98-a229-637aa78b52af,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,34,0,0,0.0,0.0,0.00,0.0,0.0,0.00
9,396111ed-b32a-4448-afbf-213bd8d196ad,"ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""",1,0,0,0.0,0.0,0.00,0.0,0.0,0.00


In [56]:
sn = (
    excluded_finance[
        excluded_finance[
            "party_name"
        ].str.contains(
            "СЛУГА НАРОДУ",
            case=False,
            na=False,
        )
    ]
    .copy()
)

display(
    sn[
        [
            "party_name",
            "organization_name",
            "report_id",
            "year",
            "quarter",
            "excluded_reason",
            "financial_rows_total",
            "cash_inflow_total",
            "spending_total",
            "gross_cash_activity_indicator",
            "selected_counterpart_report_id",
            "same_financial_content_as_selected",
            "cash_inflow_difference_vs_selected",
            "spending_difference_vs_selected",
            "financial_content_hash",
        ]
    ]
    .sort_values(
        [
            "year",
            "quarter",
            "report_id",
        ]
    )
)

,party_name,organization_name,report_id,year,quarter,excluded_reason,financial_rows_total,cash_inflow_total,spending_total,gross_cash_activity_indicator,selected_counterpart_report_id,same_financial_content_as_selected,cash_inflow_difference_vs_selected,spending_difference_vs_selected,financial_content_hash
81,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,2025,2,nonselected_instance_in_selected_period,250,1050000.0,1077997.21,2127997.21,764a691c-3237-4044-9430-44313fbf739f,False,1050000.00,1077997.21,7beae3d78011b13502b98218c749f6cdcda9dfab175cb1...
102,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,45c5faf0-ec85-469a-8b19-03aec57be9c9,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,84185feb-e42f-4dc6-a96c-61f3cbbe24a5,False,-3610000.00,-3749205.42,73a73af585b780c54995fcb9b93487e844123aa314adf2...
69,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЗАПОРІЗЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,720aab74-67a8-44eb-87bf-2d73fa12a03f,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,12687709-301b-48f2-8891-bbaf9a962d74,False,-1000000.00,-967762.20,73a73af585b780c54995fcb9b93487e844123aa314adf2...
101,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,8939ffbf-fd03-4eae-8ef2-b7afe70d0d48,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,84185feb-e42f-4dc6-a96c-61f3cbbe24a5,False,-3610000.00,-3749205.42,73a73af585b780c54995fcb9b93487e844123aa314adf2...
82,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,c0c83821-6cea-4b1e-8b6d-989b7b92ed1f,2025,2,nonselected_instance_in_selected_period,250,1050000.0,1077997.21,2127997.21,764a691c-3237-4044-9430-44313fbf739f,False,1050000.00,1077997.21,65e4adc749f77de41231c3cca589d93b016dc11e941cbc...
188,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ПОЛТАВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,fcbabc2c-6d21-4bf9-8295-41fc6e2d7531,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,975a129a-4865-43d4-9002-c2876600ead4,False,-4339510.56,-4197689.19,73a73af585b780c54995fcb9b93487e844123aa314adf2...


In [57]:
sn_active = (
    sn[
        sn[
            "has_nonzero_cash_activity"
        ]
    ]
    .copy()
)

print(
    "Active excluded SN reports:",
    len(sn_active)
)

print()

print(
    "Unique financial hashes:",
    sn_active[
        "financial_content_hash"
    ].nunique()
)

print()

display(
    sn_active[
        [
            "report_id",
            "organization_name",
            "year",
            "quarter",
            "financial_rows_total",
            "cash_inflow_total",
            "spending_total",
            "financial_content_hash",
            "selected_counterpart_report_id",
        ]
    ]
)

Active excluded SN reports: 2

Unique financial hashes: 2



,report_id,organization_name,year,quarter,financial_rows_total,cash_inflow_total,spending_total,financial_content_hash,selected_counterpart_report_id
81,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,2,250,1050000.0,1077997.21,7beae3d78011b13502b98218c749f6cdcda9dfab175cb1...,764a691c-3237-4044-9430-44313fbf739f
82,c0c83821-6cea-4b1e-8b6d-989b7b92ed1f,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,2,250,1050000.0,1077997.21,65e4adc749f77de41231c3cca589d93b016dc11e941cbc...,764a691c-3237-4044-9430-44313fbf739f


In [58]:
from collections import Counter
from pathlib import Path
import json
import pandas as pd

from politdata.report_details import fetch_report_detail


RAW_DIR = Path(
    "data/raw/report_details"
)

REPORT_IDS = {
    "excluded_A":
        "317a1c90-a5a3-43e5-8e62-e205cc84fb1e",

    "excluded_B":
        "c0c83821-6cea-4b1e-8b6d-989b7b92ed1f",

    "selected":
        "764a691c-3237-4044-9430-44313fbf739f",
}


PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


# These fields identify the API/database record,
# not the substantive financial operation.
TECHNICAL_FIELDS = {
    "id",
    "report_id",
    "party_id",
    "office_id",
    "created_at",
    "updated_at",
}


def load_detail(report_id):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if path.exists():

        with path.open(
            "r",
            encoding="utf-8",
        ) as f:

            return json.load(f)[
                "results"
            ]

    # Fallback if this particular report
    # has not yet reached the RAW downloader.
    print(
        "RAW missing, fetching:",
        report_id
    )

    return fetch_report_detail(
        report_id
    )["results"]


def payment_rows(
    detail,
    payment_type,
):

    direction, key = (
        PAYMENT_PATHS[
            payment_type
        ]
    )

    return (
        (
            (
                detail.get(
                    "payment_info"
                )
                or {}
            )
            .get(direction)
            or {}
        )
        .get(key)
        or []
    )


def substantive_row(row):

    return {
        key: value
        for key, value
        in row.items()
        if key
        not in TECHNICAL_FIELDS
    }


def row_signature(row):

    return json.dumps(
        substantive_row(row),
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )


def signature_counter(rows):

    return Counter(
        row_signature(row)
        for row in rows
    )


def amount_sum(
    rows,
    field="payment_amount",
):

    total = 0.0

    for row in rows:

        value = row.get(
            field
        )

        if value is not None:

            try:
                total += float(value)
            except (
                TypeError,
                ValueError,
            ):
                pass

    return total


details = {
    label: load_detail(
        report_id
    )
    for label, report_id
    in REPORT_IDS.items()
}


# ============================================================
# BASIC METADATA
# ============================================================

print("=" * 80)
print("REPORT VERSIONS")
print("=" * 80)

metadata_rows = []

for label, detail in details.items():

    metadata_rows.append(
        {
            "version":
                label,

            "report_id":
                detail.get("id"),

            "year":
                detail.get("year"),

            "quarter":
                detail.get("quarter"),

            "created_date":
                detail.get("created_date"),

            "signed_date":
                detail.get("signed_date"),

            "is_party_office":
                detail.get("is_party_office"),
        }
    )

display(
    pd.DataFrame(
        metadata_rows
    )
)


# ============================================================
# PAIRWISE SUBSTANTIVE COMPARISON
# ============================================================

pairs = [
    (
        "excluded_A",
        "excluded_B",
    ),
    (
        "excluded_A",
        "selected",
    ),
    (
        "excluded_B",
        "selected",
    ),
]


comparison_rows = []

difference_examples = []


for left_label, right_label in pairs:

    left_detail = details[
        left_label
    ]

    right_detail = details[
        right_label
    ]

    for payment_type in PAYMENT_PATHS:

        left_rows = payment_rows(
            left_detail,
            payment_type,
        )

        right_rows = payment_rows(
            right_detail,
            payment_type,
        )

        left_counter = (
            signature_counter(
                left_rows
            )
        )

        right_counter = (
            signature_counter(
                right_rows
            )
        )

        common_counter = (
            left_counter
            &
            right_counter
        )

        only_left_counter = (
            left_counter
            -
            right_counter
        )

        only_right_counter = (
            right_counter
            -
            left_counter
        )

        common_rows = sum(
            common_counter.values()
        )

        only_left = sum(
            only_left_counter.values()
        )

        only_right = sum(
            only_right_counter.values()
        )


        left_ids = {
            str(row.get("id"))
            for row in left_rows
            if row.get("id")
        }

        right_ids = {
            str(row.get("id"))
            for row in right_rows
            if row.get("id")
        }


        comparison_rows.append(
            {
                "comparison":
                    f"{left_label} → {right_label}",

                "payment_type":
                    payment_type,

                "left_rows":
                    len(left_rows),

                "right_rows":
                    len(right_rows),

                "substantive_overlap":
                    common_rows,

                "only_left":
                    only_left,

                "only_right":
                    only_right,

                "same_substantive_content":
                    (
                        left_counter
                        ==
                        right_counter
                    ),

                "source_id_overlap":
                    len(
                        left_ids
                        &
                        right_ids
                    ),

                "left_payment_amount":
                    amount_sum(
                        left_rows
                    ),

                "right_payment_amount":
                    amount_sum(
                        right_rows
                    ),

                "payment_amount_difference":
                    (
                        amount_sum(
                            right_rows
                        )
                        -
                        amount_sum(
                            left_rows
                        )
                    ),
            }
        )


        # Store a few actual differing records
        # for inspection if needed.
        for signature, count in list(
            only_left_counter.items()
        )[:3]:

            difference_examples.append(
                {
                    "comparison":
                        f"{left_label} → {right_label}",

                    "payment_type":
                        payment_type,

                    "side":
                        "only_left",

                    "count":
                        count,

                    "row":
                        json.loads(
                            signature
                        ),
                }
            )


        for signature, count in list(
            only_right_counter.items()
        )[:3]:

            difference_examples.append(
                {
                    "comparison":
                        f"{left_label} → {right_label}",

                    "payment_type":
                        payment_type,

                    "side":
                        "only_right",

                    "count":
                        count,

                    "row":
                        json.loads(
                            signature
                        ),
                }
            )


comparison = pd.DataFrame(
    comparison_rows
)


# Only categories that contain records
# in at least one version.
active_comparison = (
    comparison[
        (
            comparison[
                "left_rows"
            ]
            > 0
        )
        |
        (
            comparison[
                "right_rows"
            ]
            > 0
        )
    ]
    .copy()
)


print()
print("=" * 80)
print("FINANCIAL CONTENT COMPARISON")
print("=" * 80)

display(
    active_comparison
)


print()
print("=" * 80)
print("PAIR SUMMARY")
print("=" * 80)

pair_summary = (
    comparison
    .groupby(
        "comparison"
    )
    .agg(
        left_rows=(
            "left_rows",
            "sum",
        ),

        right_rows=(
            "right_rows",
            "sum",
        ),

        substantive_overlap=(
            "substantive_overlap",
            "sum",
        ),

        only_left=(
            "only_left",
            "sum",
        ),

        only_right=(
            "only_right",
            "sum",
        ),

        categories_different=(
            "same_substantive_content",
            lambda x: int(
                (~x).sum()
            ),
        ),

        left_payment_amount=(
            "left_payment_amount",
            "sum",
        ),

        right_payment_amount=(
            "right_payment_amount",
            "sum",
        ),
    )
    .reset_index()
)

pair_summary[
    "same_financial_content"
] = (
    (
        pair_summary[
            "only_left"
        ]
        == 0
    )
    &
    (
        pair_summary[
            "only_right"
        ]
        == 0
    )
)


display(
    pair_summary
)


print()
print("=" * 80)
print("DIFFERENCE EXAMPLES")
print("=" * 80)

difference_examples_df = (
    pd.DataFrame(
        difference_examples
    )
)

display(
    difference_examples_df
)

REPORT VERSIONS


,version,report_id,year,quarter,created_date,signed_date,is_party_office
0,excluded_A,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,2025,2,2025-07-11 17:38:23.040509,None,True
1,excluded_B,c0c83821-6cea-4b1e-8b6d-989b7b92ed1f,2025,2,2025-07-11 13:31:08.170797,None,True
2,selected,764a691c-3237-4044-9430-44313fbf739f,2025,2,2025-08-07 22:08:10.331902,2025-08-07 22:15:52.146213,True



FINANCIAL CONTENT COMPARISON


,comparison,payment_type,left_rows,right_rows,substantive_overlap,only_left,only_right,same_substantive_content,source_id_overlap,left_payment_amount,right_payment_amount,payment_amount_difference
3,excluded_A → excluded_B,other_incomes,3,3,3,0,0,True,0,1050000.00,1050000.00,0.000000e+00
5,excluded_A → excluded_B,outgoing_expenses,247,247,247,0,0,True,0,1077997.21,1077997.21,2.328306e-10
11,excluded_A → selected,other_incomes,3,0,0,3,0,False,0,1050000.00,0.00,-1.050000e+06
13,excluded_A → selected,outgoing_expenses,247,0,0,247,0,False,0,1077997.21,0.00,-1.077997e+06
19,excluded_B → selected,other_incomes,3,0,0,3,0,False,0,1050000.00,0.00,-1.050000e+06
21,excluded_B → selected,outgoing_expenses,247,0,0,247,0,False,0,1077997.21,0.00,-1.077997e+06



PAIR SUMMARY


,comparison,left_rows,right_rows,substantive_overlap,only_left,only_right,categories_different,left_payment_amount,right_payment_amount,same_financial_content
0,excluded_A → excluded_B,250,250,250,0,0,0,2127997.21,2127997.21,True
1,excluded_A → selected,250,0,0,250,0,2,2127997.21,0.00,False
2,excluded_B → selected,250,0,0,250,0,2,2127997.21,0.00,False



DIFFERENCE EXAMPLES


,comparison,payment_type,side,count,row
0,excluded_A → selected,other_incomes,only_left,1,"{'group_code': '3_4', 'payer_account_iban': No..."
1,excluded_A → selected,other_incomes,only_left,1,"{'group_code': '3_4', 'payer_account_iban': No..."
2,excluded_A → selected,other_incomes,only_left,1,"{'group_code': '3_4', 'payer_account_iban': No..."
3,excluded_A → selected,outgoing_expenses,only_left,12,"{'group_code': '4_2', 'payer_account_iban': No..."
4,excluded_A → selected,outgoing_expenses,only_left,1,"{'group_code': '4_2', 'payer_account_iban': No..."
5,excluded_A → selected,outgoing_expenses,only_left,1,"{'group_code': '4_2', 'payer_account_iban': No..."
6,excluded_B → selected,other_incomes,only_left,1,"{'group_code': '3_4', 'payer_account_iban': No..."
7,excluded_B → selected,other_incomes,only_left,1,"{'group_code': '3_4', 'payer_account_iban': No..."
8,excluded_B → selected,other_incomes,only_left,1,"{'group_code': '3_4', 'payer_account_iban': No..."
9,excluded_B → selected,outgoing_expenses,only_left,1,"{'group_code': '4_2', 'payer_account_iban': No..."


In [59]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd


RAW_DIR = Path(
    "data/raw/report_details"
)

OUT_DIR = Path(
    "data/interim/reports"
)


PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


TECHNICAL_FIELDS = {
    "id",
    "report_id",
    "party_id",
    "office_id",
    "created_at",
    "updated_at",
}


def load_detail(report_id):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(f)[
            "results"
        ]


def get_payment_rows(
    detail,
    payment_type,
):

    direction, key = (
        PAYMENT_PATHS[
            payment_type
        ]
    )

    return (
        (
            (
                detail.get(
                    "payment_info"
                )
                or {}
            )
            .get(direction)
            or {}
        )
        .get(key)
        or []
    )


def substantive_payment_content(
    detail,
):

    result = {}

    for payment_type in PAYMENT_PATHS:

        rows = get_payment_rows(
            detail,
            payment_type,
        )

        normalized_rows = []

        for row in rows:

            normalized = {
                key: value
                for key, value
                in row.items()
                if key
                not in TECHNICAL_FIELDS
            }

            normalized_rows.append(
                normalized
            )

        normalized_rows.sort(
            key=lambda row:
                json.dumps(
                    row,
                    ensure_ascii=False,
                    sort_keys=True,
                    separators=(",", ":"),
                    default=str,
                )
        )

        result[
            payment_type
        ] = normalized_rows

    return result


def substantive_financial_hash(
    detail,
):

    content = (
        substantive_payment_content(
            detail
        )
    )

    canonical = json.dumps(
        content,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    )

    return hashlib.sha256(
        canonical.encode(
            "utf-8"
        )
    ).hexdigest()


# ---------------------------------------------------------
# LOAD PREVIOUS IMPACT TABLE
# ---------------------------------------------------------

impact = pd.read_parquet(
    OUT_DIR
    / "excluded_report_financial_impact.parquet"
)

impact[
    "report_id"
] = impact[
    "report_id"
].astype(str)


# ---------------------------------------------------------
# CORRECT SUBSTANTIVE HASHES
# ---------------------------------------------------------

new_hashes = {}

for report_id in (
    impact[
        "report_id"
    ]
    .unique()
):

    detail = load_detail(
        report_id
    )

    new_hashes[
        report_id
    ] = (
        substantive_financial_hash(
            detail
        )
    )


impact[
    "substantive_financial_hash"
] = impact[
    "report_id"
].map(
    new_hashes
)


# ---------------------------------------------------------
# COMPARE AGAINST SELECTED COUNTERPART
# ---------------------------------------------------------

selected_hash_cache = {}

same_as_selected = []


for _, row in impact.iterrows():

    counterpart = row.get(
        "selected_counterpart_report_id"
    )

    if (
        pd.isna(counterpart)
        or str(counterpart)
        in {
            "",
            "None",
            "nan",
        }
    ):

        same_as_selected.append(
            None
        )

        continue


    counterpart = str(
        counterpart
    )

    if counterpart not in (
        selected_hash_cache
    ):

        selected_detail = load_detail(
            counterpart
        )

        selected_hash_cache[
            counterpart
        ] = (
            substantive_financial_hash(
                selected_detail
            )
        )


    same_as_selected.append(
        row[
            "substantive_financial_hash"
        ]
        ==
        selected_hash_cache[
            counterpart
        ]
    )


impact[
    "same_substantive_financial_content_as_selected"
] = same_as_selected


# ---------------------------------------------------------
# IDENTIFY DUPLICATE EXCLUDED FINANCIAL VERSIONS
# ---------------------------------------------------------

VERSION_GROUP_COLS = [
    "organization_id",
    "year",
    "quarter",
    "substantive_financial_hash",
]


impact[
    "excluded_financial_version_number"
] = (
    impact
    .groupby(
        VERSION_GROUP_COLS,
        dropna=False,
    )
    .cumcount()
    + 1
)


impact[
    "is_first_instance_of_financial_version"
] = (
    impact[
        "excluded_financial_version_number"
    ]
    == 1
)


# ---------------------------------------------------------
# DISTINCT EXCLUDED FINANCIAL VERSIONS
# ---------------------------------------------------------

distinct_versions = (
    impact[
        impact[
            "is_first_instance_of_financial_version"
        ]
    ]
    .copy()
)


# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------

impact.to_parquet(
    OUT_DIR
    / "excluded_report_financial_impact_substantive.parquet",
    index=False,
)

impact.to_csv(
    OUT_DIR
    / "excluded_report_financial_impact_substantive.csv",
    index=False,
    encoding="utf-8-sig",
)

distinct_versions.to_parquet(
    OUT_DIR
    / "excluded_distinct_financial_versions.parquet",
    index=False,
)

distinct_versions.to_csv(
    OUT_DIR
    / "excluded_distinct_financial_versions.csv",
    index=False,
    encoding="utf-8-sig",
)


# ---------------------------------------------------------
# QA
# ---------------------------------------------------------

print("=" * 70)
print("CORRECTED SUBSTANTIVE COMPARISON")
print("=" * 70)

print(
    impact[
        "same_substantive_financial_content_as_selected"
    ].value_counts(
        dropna=False
    )
)

print()

print(
    "Excluded report instances:",
    len(impact)
)

print(
    "Distinct excluded financial versions:",
    len(distinct_versions)
)

print()

print(
    "Distinct versions with non-zero cash activity:",
    int(
        distinct_versions[
            "has_nonzero_cash_activity"
        ].sum()
    )
)


print()
print("=" * 70)
print("SUBSTANTIVELY DIFFERENT FROM SELECTED")
print("=" * 70)

different = (
    distinct_versions[
        distinct_versions[
            "same_substantive_financial_content_as_selected"
        ] == False
    ]
    .sort_values(
        "gross_cash_activity_indicator",
        ascending=False,
    )
)

display(
    different[
        [
            "party_name",
            "organization_name",
            "report_id",
            "year",
            "quarter",
            "excluded_reason",
            "financial_rows_total",
            "cash_inflow_total",
            "nonmonetary_contributions_total",
            "spending_total",
            "gross_cash_activity_indicator",
            "selected_counterpart_report_id",
        ]
    ]
)

CORRECTED SUBSTANTIVE COMPARISON
same_substantive_financial_content_as_selected
True     170
False     30
None      13
Name: count, dtype: int64

Excluded report instances: 213
Distinct excluded financial versions: 145

Distinct versions with non-zero cash activity: 1

SUBSTANTIVELY DIFFERENT FROM SELECTED


,party_name,organization_name,report_id,year,quarter,excluded_reason,financial_rows_total,cash_inflow_total,nonmonetary_contributions_total,spending_total,gross_cash_activity_indicator,selected_counterpart_report_id
0,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,2025,2,nonselected_instance_in_selected_period,250,1050000.0,0.00,1077997.21,2127997.21,764a691c-3237-4044-9430-44313fbf739f
2,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,ДНІПРОПЕТРОВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНО...,63ccb8ae-f6d4-4ede-8a16-b0ee9062d1b7,2025,2,nonselected_instance_in_selected_period,3,0.0,16495.11,0.00,0.00,6730e708-b247-4bb0-9ece-75855f64ab99
3,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,ПОЛІТИЧНА ПАРТІЯ «НОВА СИЛА»,58d8edd0-e9b8-11ee-92b7-3b3b09128e69,2023,5,nonselected_instance_in_selected_period,2,0.0,5400.00,0.00,0.00,de3d0b30-eaa5-11ee-92b7-3b3b09128e69
4,ПОЛІТИЧНА ПАРТІЯ «БЛОК ВІЛКУЛА «УКРАЇНСЬКА ПЕР...,КІРОВОГРАДСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,9b7ae41c-f4e6-4c75-bf5e-719804ef2796,2025,2,nonselected_instance_in_selected_period,2,0.0,3410.00,0.00,0.00,0a49ccc2-d006-4fbf-a40d-4e506e677bc9
10,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БА...,Оболонська районна партійна організація Всеукр...,618e9957-c77d-416e-bd58-92c124d50822,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,0.00,f4d73669-71ce-44be-8975-b98549fc3d1b
27,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,РІВНЕНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,a5c2b95a-ed3a-439f-81b9-88201bc378d2,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,0.00,9ad81916-4471-405c-a98b-935b99656113
34,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,СУМСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,2770909f-fc6e-487e-aa4f-cad3a45b1d5a,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,0.00,5243ce47-84e4-4331-80f1-24833756dbb3
53,ПОЛІТИЧНА ПАРТІЯ «КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА П...,Житомирська обласна організація політичної пар...,76a622a6-f879-401a-b40c-bf8e35c6035a,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,0.00,aabebdd0-2ff8-493f-b5c9-54ed52dd909f
58,ПОЛІТИЧНА ПАРТІЯ «БЛОК ВІЛКУЛА «УКРАЇНСЬКА ПЕР...,ДНІПРОПЕТРОВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНО...,504365ea-eae6-46b2-a732-36dd741a8ec2,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,0.00,eb9f868e-62ca-4368-9ba8-d7c6eba4c0b5
60,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,КІРОВОГРАДСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ...,a1793e3d-c453-460e-ad15-8dbc26b7d2aa,2025,2,nonselected_instance_in_selected_period,0,0.0,0.00,0.00,0.00,e48609d7-eb09-4ca7-8fbb-983336aae999


In [60]:
from pathlib import Path
import json
import pandas as pd


RAW_DIR = Path(
    "data/raw/report_details"
)

OUT_DIR = Path(
    "data/interim/reports"
)


def load_detail(report_id):

    with (
        RAW_DIR
        / f"{report_id}.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(f)["results"]


def summarize_counterpart(detail):

    payment_info = (
        detail.get("payment_info")
        or {}
    )

    incoming = (
        payment_info.get("incoming")
        or {}
    )

    outgoing = (
        payment_info.get("outgoing")
        or {}
    )

    def rows(group, key):
        return (
            group.get(key)
            or []
        )

    monetary = rows(
        incoming,
        "monetary_contributions",
    )

    other_contrib = rows(
        incoming,
        "other_contributions",
    )

    state = rows(
        incoming,
        "state_funding",
    )

    other_income = rows(
        incoming,
        "other_incomes",
    )

    budget = rows(
        outgoing,
        "budget_expenses",
    )

    expense = rows(
        outgoing,
        "outgoing_expenses",
    )

    returns = rows(
        outgoing,
        "return_expenses",
    )

    transfers = rows(
        outgoing,
        "transfer_expenses",
    )

    all_rows = (
        monetary
        + other_contrib
        + state
        + other_income
        + budget
        + expense
        + returns
        + transfers
    )

    def amount_sum(items):

        result = 0.0

        for row in items:

            value = row.get(
                "payment_amount"
            )

            if value is not None:
                result += float(value)

        return result


    cash_inflow = (
        amount_sum(monetary)
        + amount_sum(state)
        + amount_sum(other_income)
    )

    spending = (
        amount_sum(budget)
        + amount_sum(expense)
    )

    nonmonetary = (
        amount_sum(other_contrib)
    )

    return {
        "selected_financial_rows":
            len(all_rows),

        "selected_cash_inflow":
            cash_inflow,

        "selected_spending":
            spending,

        "selected_nonmonetary":
            nonmonetary,
    }


impact = pd.read_parquet(
    OUT_DIR
    / "excluded_report_financial_impact_substantive.parquet"
)


different = (
    impact[
        impact[
            "same_substantive_financial_content_as_selected"
        ] == False
    ]
    .copy()
)


selected_cache = {}

rows = []


for _, row in different.iterrows():

    counterpart = str(
        row[
            "selected_counterpart_report_id"
        ]
    )

    if counterpart not in selected_cache:

        selected_cache[
            counterpart
        ] = summarize_counterpart(
            load_detail(
                counterpart
            )
        )

    selected = (
        selected_cache[
            counterpart
        ]
    )

    result = row.to_dict()

    result.update(
        selected
    )


    excluded_rows = int(
        row[
            "financial_rows_total"
        ]
    )

    selected_rows = int(
        selected[
            "selected_financial_rows"
        ]
    )


    if (
        excluded_rows > 0
        and selected_rows == 0
    ):

        change_type = (
            "financial_content_removed"
        )

    elif (
        excluded_rows == 0
        and selected_rows > 0
    ):

        change_type = (
            "financial_content_added"
        )

    elif (
        excluded_rows > 0
        and selected_rows > 0
    ):

        change_type = (
            "financial_content_modified"
        )

    else:

        change_type = (
            "non_financial_or_zero_value_change"
        )


    result[
        "financial_change_type"
    ] = change_type

    result[
        "cash_inflow_change"
    ] = (
        selected[
            "selected_cash_inflow"
        ]
        -
        float(
            row[
                "cash_inflow_total"
            ]
        )
    )

    result[
        "spending_change"
    ] = (
        selected[
            "selected_spending"
        ]
        -
        float(
            row[
                "spending_total"
            ]
        )
    )

    rows.append(
        result
    )


change_analysis = pd.DataFrame(
    rows
)


print("=" * 70)
print("CHANGE TYPES")
print("=" * 70)

print(
    change_analysis[
        "financial_change_type"
    ].value_counts()
)


print()
print("=" * 70)
print("MATERIAL VERSION CHANGES")
print("=" * 70)

display(
    change_analysis[
        [
            "party_name",
            "organization_name",
            "report_id",
            "year",
            "quarter",
            "financial_change_type",

            "financial_rows_total",
            "selected_financial_rows",

            "cash_inflow_total",
            "selected_cash_inflow",
            "cash_inflow_change",

            "nonmonetary_contributions_total",
            "selected_nonmonetary",

            "spending_total",
            "selected_spending",
            "spending_change",

            "selected_counterpart_report_id",
        ]
    ]
    .sort_values(
        [
            "financial_change_type",
            "cash_inflow_change",
            "spending_change",
        ],
        ascending=[
            True,
            False,
            False,
        ],
    )
)


change_analysis.to_parquet(
    OUT_DIR
    / "excluded_report_financial_change_analysis.parquet",
    index=False,
)

change_analysis.to_csv(
    OUT_DIR
    / "excluded_report_financial_change_analysis.csv",
    index=False,
    encoding="utf-8-sig",
)

CHANGE TYPES
financial_change_type
financial_content_added       25
financial_content_modified     3
financial_content_removed      2
Name: count, dtype: int64

MATERIAL VERSION CHANGES


,party_name,organization_name,report_id,year,quarter,financial_change_type,financial_rows_total,selected_financial_rows,cash_inflow_total,selected_cash_inflow,cash_inflow_change,nonmonetary_contributions_total,selected_nonmonetary,spending_total,selected_spending,spending_change,selected_counterpart_report_id
26,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ПОЛТАВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,fcbabc2c-6d21-4bf9-8295-41fc6e2d7531,2025,2,financial_content_added,0,725,0.0,4339510.56,4339510.56,0.00,0.00,0.00,4197689.19,4197689.19,975a129a-4865-43d4-9002-c2876600ead4
16,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,8939ffbf-fd03-4eae-8ef2-b7afe70d0d48,2025,2,financial_content_added,0,824,0.0,3610000.00,3610000.00,0.00,0.00,0.00,3749205.42,3749205.42,84185feb-e42f-4dc6-a96c-61f3cbbe24a5
17,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,45c5faf0-ec85-469a-8b19-03aec57be9c9,2025,2,financial_content_added,0,824,0.0,3610000.00,3610000.00,0.00,0.00,0.00,3749205.42,3749205.42,84185feb-e42f-4dc6-a96c-61f3cbbe24a5
14,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЗАПОРІЗЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,720aab74-67a8-44eb-87bf-2d73fa12a03f,2025,2,financial_content_added,0,174,0.0,1000000.00,1000000.00,0.00,0.00,0.00,967762.20,967762.20,12687709-301b-48f2-8891-bbaf9a962d74
21,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛЬВІВСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,e4612672-5728-4058-ba9c-c4a0aaf0f4d8,2025,2,financial_content_added,0,121,0.0,802189.41,802189.41,0.00,12339.60,0.00,779148.58,779148.58,53e13e78-f470-4589-a9b0-060dba1f8ae9
22,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛЬВІВСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,b779e0ff-583d-428f-ae7b-c6c37e8588d6,2025,2,financial_content_added,0,121,0.0,802189.41,802189.41,0.00,12339.60,0.00,779148.58,779148.58,53e13e78-f470-4589-a9b0-060dba1f8ae9
7,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,СУМСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,2770909f-fc6e-487e-aa4f-cad3a45b1d5a,2025,2,financial_content_added,0,133,0.0,736814.58,736814.58,0.00,0.00,0.00,737086.33,737086.33,5243ce47-84e4-4331-80f1-24833756dbb3
18,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ТЕРНОПІЛЬСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИ...,49d1e7d2-d8e6-4e7b-9b6d-fe2882af64f2,2025,2,financial_content_added,0,83,0.0,571575.59,571575.59,0.00,0.00,0.00,494661.37,494661.37,08561cc9-4614-455d-9a3c-6068518688bc
19,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ТЕРНОПІЛЬСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИ...,f58f113b-3941-4365-a571-b0b754ea51a1,2025,2,financial_content_added,0,83,0.0,571575.59,571575.59,0.00,0.00,0.00,494661.37,494661.37,08561cc9-4614-455d-9a3c-6068518688bc
15,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БА...,ЗАКАРПАТСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКО...,c76e7c7b-19a4-4289-af32-32d0eb982018,2025,2,financial_content_added,0,17,0.0,76140.00,76140.00,0.00,0.00,0.00,76696.32,76696.32,fe2b11a3-7f44-4d01-9690-6046c2e1eb1d


In [61]:
from pathlib import Path
import pandas as pd

OUT_DIR = Path(
    "data/interim/reports"
)


# ============================================================
# 1. DISTINCT VERSION-CHANGE CASES
# ============================================================

# change_analysis already contains the 30 instances
# classified as added / modified / removed.

distinct_change_cases = (
    change_analysis
    .sort_values(
        [
            "organization_id",
            "year",
            "quarter",
            "report_id",
        ]
    )
    .drop_duplicates(
        subset=[
            "organization_id",
            "year",
            "quarter",
            "substantive_financial_hash",
            "selected_counterpart_report_id",
            "financial_change_type",
        ]
    )
    .copy()
)


print("=" * 70)
print("DISTINCT VERSION-CHANGE CASES")
print("=" * 70)

print(
    distinct_change_cases[
        "financial_change_type"
    ].value_counts()
)

print()

print(
    "Instances before dedup:",
    len(change_analysis)
)

print(
    "Distinct transitions:",
    len(distinct_change_cases)
)


# ============================================================
# 2. MATERIAL CASH CHANGES
# ============================================================

material_cash_changes = (
    distinct_change_cases[
        (
            distinct_change_cases[
                "cash_inflow_change"
            ].abs() > 0
        )
        |
        (
            distinct_change_cases[
                "spending_change"
            ].abs() > 0
        )
    ]
    .copy()
)


print()
print("=" * 70)
print("DISTINCT CASES WITH CASH DIFFERENCE")
print("=" * 70)

print(
    len(material_cash_changes)
)

display(
    material_cash_changes[
        [
            "party_name",
            "organization_name",
            "report_id",
            "year",
            "quarter",
            "financial_change_type",

            "financial_rows_total",
            "selected_financial_rows",

            "cash_inflow_total",
            "selected_cash_inflow",
            "cash_inflow_change",

            "spending_total",
            "selected_spending",
            "spending_change",

            "selected_counterpart_report_id",
        ]
    ]
)


# ============================================================
# 3. PERIODS WITHOUT ANY SELECTED SIGNED REPORT
# ============================================================

impact = pd.read_parquet(
    OUT_DIR
    / "excluded_report_financial_impact_substantive.parquet"
)


no_selected = (
    impact[
        impact[
            "excluded_reason"
        ]
        ==
        "period_without_selected_signed_report"
    ]
    .copy()
)


# Deduplicate identical financial versions inside
# the same logical period.
no_selected_distinct = (
    no_selected
    .drop_duplicates(
        subset=[
            "organization_id",
            "year",
            "quarter",
            "substantive_financial_hash",
        ]
    )
    .copy()
)


print()
print("=" * 70)
print("PERIODS WITHOUT SELECTED SIGNED REPORT")
print("=" * 70)

print(
    "Source instances:",
    len(no_selected)
)

print(
    "Logical periods:",
    no_selected[
        [
            "organization_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Distinct financial versions:",
    len(no_selected_distinct)
)

print(
    "With financial rows:",
    int(
        no_selected_distinct[
            "has_financial_rows"
        ].sum()
    )
)

print(
    "With non-zero cash activity:",
    int(
        no_selected_distinct[
            "has_nonzero_cash_activity"
        ].sum()
    )
)


display(
    no_selected_distinct[
        [
            "party_name",
            "organization_name",
            "report_id",
            "year",
            "quarter",
            "financial_rows_total",
            "cash_inflow_total",
            "nonmonetary_contributions_total",
            "spending_total",
            "returns_total",
            "transfers_total",
        ]
    ]
    .sort_values(
        [
            "financial_rows_total",
            "cash_inflow_total",
        ],
        ascending=False,
    )
)


# ============================================================
# 4. SAVE FINAL AUDIT REGISTERS
# ============================================================

distinct_change_cases.to_parquet(
    OUT_DIR
    / "report_version_financial_changes.parquet",
    index=False,
)

distinct_change_cases.to_csv(
    OUT_DIR
    / "report_version_financial_changes.csv",
    index=False,
    encoding="utf-8-sig",
)

no_selected_distinct.to_parquet(
    OUT_DIR
    / "report_periods_without_signed_financial_profile.parquet",
    index=False,
)

print()
print("Saved final version-history audit tables.")

DISTINCT VERSION-CHANGE CASES
financial_change_type
financial_content_added       20
financial_content_modified     3
financial_content_removed      1
Name: count, dtype: int64

Instances before dedup: 30
Distinct transitions: 24

DISTINCT CASES WITH CASH DIFFERENCE
13


,party_name,organization_name,report_id,year,quarter,financial_change_type,financial_rows_total,selected_financial_rows,cash_inflow_total,selected_cash_inflow,cash_inflow_change,spending_total,selected_spending,spending_change,selected_counterpart_report_id
5,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БА...,Оболонська районна партійна організація Всеукр...,618e9957-c77d-416e-bd58-92c124d50822,2025,2,financial_content_added,0,11,0.0,38000.00,38000.00,0.00,48083.28,48083.28,f4d73669-71ce-44be-8975-b98549fc3d1b
7,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,СУМСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,2770909f-fc6e-487e-aa4f-cad3a45b1d5a,2025,2,financial_content_added,0,133,0.0,736814.58,736814.58,0.00,737086.33,737086.33,5243ce47-84e4-4331-80f1-24833756dbb3
14,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЗАПОРІЗЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,720aab74-67a8-44eb-87bf-2d73fa12a03f,2025,2,financial_content_added,0,174,0.0,1000000.00,1000000.00,0.00,967762.20,967762.20,12687709-301b-48f2-8891-bbaf9a962d74
0,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,2025,2,financial_content_removed,250,0,1050000.0,0.00,-1050000.00,1077997.21,0.00,-1077997.21,764a691c-3237-4044-9430-44313fbf739f
15,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БА...,ЗАКАРПАТСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКО...,c76e7c7b-19a4-4289-af32-32d0eb982018,2025,2,financial_content_added,0,17,0.0,76140.00,76140.00,0.00,76696.32,76696.32,fe2b11a3-7f44-4d01-9690-6046c2e1eb1d
17,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,45c5faf0-ec85-469a-8b19-03aec57be9c9,2025,2,financial_content_added,0,824,0.0,3610000.00,3610000.00,0.00,3749205.42,3749205.42,84185feb-e42f-4dc6-a96c-61f3cbbe24a5
18,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ТЕРНОПІЛЬСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИ...,49d1e7d2-d8e6-4e7b-9b6d-fe2882af64f2,2025,2,financial_content_added,0,83,0.0,571575.59,571575.59,0.00,494661.37,494661.37,08561cc9-4614-455d-9a3c-6068518688bc
20,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БА...,Кіровоградська обласна організація політичної ...,f9aa8972-6153-4c02-b6ee-a97b07272eb3,2025,2,financial_content_added,0,28,0.0,10000.00,10000.00,0.00,8064.87,8064.87,876bb8ff-8379-47ab-9102-969211d9cab9
22,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛЬВІВСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,b779e0ff-583d-428f-ae7b-c6c37e8588d6,2025,2,financial_content_added,0,121,0.0,802189.41,802189.41,0.00,779148.58,779148.58,53e13e78-f470-4589-a9b0-060dba1f8ae9
23,ПОЛІТИЧНА ПАРТІЯ «КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА П...,ВІННИЦЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,0e2dcc6c-e6b8-4752-8ff7-08eb20b55ea5,2025,2,financial_content_added,0,1,0.0,0.00,0.00,0.00,225.00,225.00,9e1e9c0c-56bf-4790-a88d-d51f78338574



PERIODS WITHOUT SELECTED SIGNED REPORT
Source instances: 13
Logical periods: 9
Distinct financial versions: 9
With financial rows: 0
With non-zero cash activity: 0


,party_name,organization_name,report_id,year,quarter,financial_rows_total,cash_inflow_total,nonmonetary_contributions_total,spending_total,returns_total,transfers_total
45,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БА...,Артемівська міська організація політичної парт...,5287f407-4433-4325-b4a0-af76af441707,2025,2,0,0.0,0.0,0.0,0.0,0.0
85,ПОЛІТИЧНА ПАРТІЯ «КОМАНДА АНДРІЯ БАЛОГИ»,Погребищенська районна організація Єдиний Центр,fff2968d-f24a-4a49-9013-8adc9c5048d3,2025,2,0,0.0,0.0,0.0,0.0,0.0
98,"ПОЛІТИЧНА ПАРТІЯ ""ДЕМОКРАТИЧНА ПАРТІЯ УКРАЇНИ""",ХМЕЛЬНИЦЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ДЕМОКРАТИЧНОЇ ...,86f9ea93-ba9c-42bf-9e35-e9dc8b60959d,2025,2,0,0.0,0.0,0.0,0.0,0.0
119,ПОЛІТИЧНА ПАРТІЯ «ЗА ПРАВА ЛЮДИНИ»,"ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПАРТІЇ ""ЗА ПРАВ...",b23ac9a4-75c7-4b67-a3c6-7fd1f603a939,2025,3,0,0.0,0.0,0.0,0.0,0.0
178,"ПОЛІТИЧНА ПАРТІЯ ""ДЕМОКРАТИЧНА ПАРТІЯ УКРАЇНИ""",ЗАПОРІЗЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ДЕМОКРАТИЧНОЇ П...,63d166df-8a62-45fb-9141-3242495d36b3,2025,2,0,0.0,0.0,0.0,0.0,0.0
186,ПОЛІТИЧНА ПАРТІЯ «СПРАВА»,СЕВАСТОПОЛЬСКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,f76511e0-ead6-4aed-9041-56ab4c8f997c,2025,2,0,0.0,0.0,0.0,0.0,0.0
190,ПАРТІЯ «СИЛЬНА УКРАЇНА»,ЯСИНУВАТСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІТ...,64ba85ab-74ed-4a6f-92e8-d5ea8caf66d9,2025,3,0,0.0,0.0,0.0,0.0,0.0
195,АГРАРНА ПАРТІЯ УКРАЇНИ,Печерська районна у місті Києві організація Аг...,9c57c568-bcea-49e8-9a7a-c307a5354cee,2025,2,0,0.0,0.0,0.0,0.0,0.0
210,ПОЛІТИЧНА ПАРТІЯ «ВІЛЬНІ ТА ВІРНІ»,ПОЛІТИЧНА ПАРТІЯ ДОБРОПІЛЬСЬКА МІСЬКА ОРГАНІЗА...,315e99ab-039b-46f0-92e4-9c53142a3ace,2025,3,0,0.0,0.0,0.0,0.0,0.0



Saved final version-history audit tables.


In [62]:
from pathlib import Path
import json
import pandas as pd


RAW_DIR = Path(
    "data/raw/report_details"
)

EXCLUDED_REPORT_ID = (
    "317a1c90-a5a3-43e5-8e62-e205cc84fb1e"
)

SIGNED_REPORT_ID = (
    "764a691c-3237-4044-9430-44313fbf739f"
)


def load_detail(report_id):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        return json.load(f)["results"]


excluded = load_detail(
    EXCLUDED_REPORT_ID
)

signed = load_detail(
    SIGNED_REPORT_ID
)


# ============================================================
# EXTRACT FINANCIAL ROWS
# ============================================================

payment_info = (
    excluded.get("payment_info")
    or {}
)

incoming = (
    payment_info.get("incoming")
    or {}
)

outgoing = (
    payment_info.get("outgoing")
    or {}
)


rows = []


# -------------------------
# OTHER INCOMES
# -------------------------

for row in (
    incoming.get("other_incomes")
    or []
):

    rows.append(
        {
            "direction":
                "incoming",

            "payment_category":
                "other_incomes",

            **row,
        }
    )


# -------------------------
# OUTGOING EXPENSES
# -------------------------

for row in (
    outgoing.get("outgoing_expenses")
    or []
):

    rows.append(
        {
            "direction":
                "outgoing",

            "payment_category":
                "outgoing_expenses",

            **row,
        }
    )


transactions = pd.DataFrame(
    rows
)


# ============================================================
# SELECT USEFUL COLUMNS
# ============================================================

preferred_columns = [
    "direction",
    "payment_category",

    "payment_operation_date",
    "payment_instruction_date",

    "payment_amount",

    "payment_number",
    "payment_reason",
    "payment_purpose",
    "payment_description",

    "payer_type",
    "payer_name",
    "payer_code",
    "payer_account_type",
    "payer_account_iban",
    "payer_bank_name",

    "receiver_type",
    "receiver_name",
    "receiver_code",
    "receiver_account_type",
    "receiver_account_iban",
    "receiver_bank_name",

    "group_code",
    "id",
]


display_columns = [
    col
    for col in preferred_columns
    if col in transactions.columns
]


transactions_view = (
    transactions[
        display_columns
    ]
    .sort_values(
        [
            "direction",
            "payment_operation_date",
            "payment_amount",
        ],
        ascending=[
            True,
            True,
            False,
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# QA
# ============================================================

print(
    "Excluded report:",
    EXCLUDED_REPORT_ID
)

print(
    "Signed report:",
    SIGNED_REPORT_ID
)

print()

print(
    "Transactions absent from signed report:",
    len(transactions_view)
)

print()

print(
    transactions_view[
        "payment_category"
    ].value_counts()
)

print()

print(
    "Incoming total:",
    transactions_view.loc[
        transactions_view[
            "direction"
        ] == "incoming",
        "payment_amount",
    ].sum()
)

print(
    "Outgoing total:",
    transactions_view.loc[
        transactions_view[
            "direction"
        ] == "outgoing",
        "payment_amount",
    ].sum()
)


# ============================================================
# DISPLAY ALL 250
# ============================================================

pd.set_option(
    "display.max_rows",
    300
)

pd.set_option(
    "display.max_columns",
    50
)

display(
    transactions_view
)


# ============================================================
# SAVE
# ============================================================

output_path = Path(
    "data/interim/reports/"
    "sluga_narodu_luhansk_2025_q2_"
    "transactions_removed_before_signed_report.csv"
)

transactions_view.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig",
)

print()
print(
    "Saved:",
    output_path
)

Excluded report: 317a1c90-a5a3-43e5-8e62-e205cc84fb1e
Signed report: 764a691c-3237-4044-9430-44313fbf739f

Transactions absent from signed report: 250

payment_category
outgoing_expenses    247
other_incomes          3
Name: count, dtype: int64

Incoming total: 1050000.0
Outgoing total: 1077997.21


,direction,payment_category,payment_operation_date,payment_instruction_date,payment_amount,payment_number,payment_reason,payment_purpose,payment_description,payer_type,payer_name,payer_code,payer_account_type,payer_account_iban,payer_bank_name,receiver_type,receiver_name,receiver_code,receiver_account_type,receiver_account_iban,receiver_bank_name,group_code,id
0,incoming,other_incomes,2025-04-10,None,330000.00,None,None,None,"Фінансування обласної організації, згідно бюдж...",Юридична особа,"ПОЛІТИЧНА ПАРТІЯ ""СЛУГА НАРОДУ""",40422142,None,None,None,None,None,None,None,UA263223130000026004000045109,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",3_4,ab93c286-db1c-44bf-b447-01daf55d89ca
1,incoming,other_incomes,2025-05-09,None,390000.00,None,None,None,"Фінансування обласної організації, згідно бюдж...",Юридична особа,"ПОЛІТИЧНА ПАРТІЯ ""СЛУГА НАРОДУ""",40422142,None,None,None,None,None,None,None,UA263223130000026004000045109,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",3_4,aeaf8792-8126-4d46-a321-89b58e3f396f
2,incoming,other_incomes,2025-06-10,None,330000.00,None,None,None,"Фінансування обласної організації, згідно бюдж...",Юридична особа,"ПОЛІТИЧНА ПАРТІЯ ""СЛУГА НАРОДУ""",40422142,None,None,None,None,None,None,None,UA263223130000026004000045109,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",3_4,69c6af44-679f-4a08-9028-933f95d60b05
3,outgoing,outgoing_expenses,2025-04-01,None,4550.00,None,за внесення змін в ЄДР,*** Плата за скорочення термінів надання послу...,None,None,None,None,None,None,None,Юридична особа,ГОЛОВНЕ УПРАВЛІННЯ ДЕРЖАВНОЇ КАЗНАЧЕЙСЬКОЇ СЛУ...,37993783,None,UA263223130000026004000045109,None,4_2,e56b2ec2-dd9a-4887-b440-7454e494a1ca
4,outgoing,outgoing_expenses,2025-04-01,None,4300.00,None,Рахунок № 2 від 01.04.2025,За вчинення нотаріальних дій згідно рахунку-фа...,None,None,None,None,None,None,None,Фізична особа/ФОП,ПРИВАТНИЙ НОТАРІУС ЗАБЛОЦЬКА ЛАРИСА БОРИСІВНА,[конфіденційна інформація],None,UA263223130000026004000045109,None,4_2,d6707c29-f492-4679-aa17-78fc41bb6004
5,outgoing,outgoing_expenses,2025-04-01,None,910.00,None,за внесення змін в ЄДР,*** Адміністративний збір за проведення держав...,None,None,None,None,None,None,None,Юридична особа,ГОЛОВНЕ УПРАВЛІННЯ ДЕРЖАВНОЇ КАЗНАЧЕЙСЬКОЇ СЛУ...,37993783,None,UA263223130000026004000045109,None,4_2,84d00fc4-5c4d-486d-b1e2-153c4c2f7538
6,outgoing,outgoing_expenses,2025-04-01,None,350.00,None,Послуги банку (РКО),Плата за розрахункове обслуговування поточного...,None,None,None,None,None,None,None,Юридична особа,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",00032112,None,UA263223130000026004000045109,None,4_2,30df1d15-5316-4b5a-b90b-593215474939
7,outgoing,outgoing_expenses,2025-04-01,None,3.50,None,Послуги банку (РКО),Плата за переказ коштів в нац.валюті з поточн....,None,None,None,None,None,None,None,Юридична особа,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",00032112,None,UA263223130000026004000045109,None,4_2,1e52f943-6145-4410-b1ff-900de83fc90a
8,outgoing,outgoing_expenses,2025-04-01,None,3.50,None,Послуги банку (РКО),Плата за переказ коштів в нац.валюті з поточн....,None,None,None,None,None,None,None,Юридична особа,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",00032112,None,UA263223130000026004000045109,None,4_2,22d70207-8b7f-4028-b523-000786ba6dfb
9,outgoing,outgoing_expenses,2025-04-01,None,3.50,None,Послуги банку (РКО),Плата за переказ коштів в нац.валюті з поточн....,None,None,None,None,None,None,None,Юридична особа,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",00032112,None,UA263223130000026004000045109,None,4_2,834ab908-3490-427f-abf0-8b14ab6178a6



Saved: data\interim\reports\sluga_narodu_luhansk_2025_q2_transactions_removed_before_signed_report.csv


In [63]:
from pathlib import Path
import json
import pandas as pd

from politdata.report_details import (
    fetch_report_detail,
    atomic_write_json,
)


RAW_DIR = Path(
    "data/raw/report_details"
)

REPORTS_DIR = Path(
    "data/interim/reports"
)

Q2_DRAFT_ID = (
    "317a1c90-a5a3-43e5-8e62-e205cc84fb1e"
)


# ============================================================
# LOAD / FETCH
# ============================================================

def load_or_fetch(report_id):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if path.exists():

        with path.open(
            "r",
            encoding="utf-8",
        ) as f:

            return json.load(f)["results"]

    payload = fetch_report_detail(
        report_id
    )

    atomic_write_json(
        path,
        payload,
    )

    return payload["results"]


# ============================================================
# IDENTIFY LUHANSK ORGANIZATION
# ============================================================

excluded_manifest = pd.read_parquet(
    REPORTS_DIR
    / "excluded_report_instances_manifest.parquet"
)

draft_row = excluded_manifest[
    excluded_manifest["report_id"].astype(str)
    == Q2_DRAFT_ID
].iloc[0]

organization_id = str(
    draft_row["organization_id"]
)

print(
    "Organization ID:",
    organization_id
)


# ============================================================
# FIND SELECTED Q1 / Q2 / Q3
# ============================================================

selected_reports = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
)

selected_reports[
    "organization_id"
] = selected_reports[
    "organization_id"
].astype(str)

period_reports = (
    selected_reports[
        (
            selected_reports[
                "organization_id"
            ]
            == organization_id
        )
        &
        (
            selected_reports[
                "year"
            ]
            == 2025
        )
        &
        (
            selected_reports[
                "quarter"
            ].isin(
                [1, 2, 3]
            )
        )
    ]
    .sort_values(
        "quarter"
    )
    .copy()
)

print()
print("SELECTED REPORTS")

display(
    period_reports[
        [
            "report_id",
            "year",
            "quarter",
            "signed_date",
        ]
    ]
)


# ============================================================
# LOAD REPORT DETAILS
# ============================================================

details = {}

for _, row in (
    period_reports.iterrows()
):

    quarter = int(
        row["quarter"]
    )

    report_id = str(
        row["report_id"]
    )

    details[
        f"Q{quarter}_signed"
    ] = load_or_fetch(
        report_id
    )


details[
    "Q2_draft"
] = load_or_fetch(
    Q2_DRAFT_ID
)


# ============================================================
# FINANCIAL TRANSACTION SUMMARY
# ============================================================

PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


def get_rows(
    detail,
    payment_type,
):

    direction, key = (
        PAYMENT_PATHS[
            payment_type
        ]
    )

    return (
        (
            (
                detail.get(
                    "payment_info"
                )
                or {}
            )
            .get(direction)
            or {}
        )
        .get(key)
        or []
    )


summary_rows = []


for version, detail in details.items():

    for payment_type in PAYMENT_PATHS:

        rows = get_rows(
            detail,
            payment_type,
        )

        amount = sum(
            float(
                row.get(
                    "payment_amount"
                )
                or 0
            )
            for row in rows
        )

        summary_rows.append(
            {
                "version":
                    version,

                "payment_type":
                    payment_type,

                "rows":
                    len(rows),

                "amount":
                    amount,
            }
        )


payment_summary = (
    pd.DataFrame(
        summary_rows
    )
)


print()
print("=" * 80)
print("PAYMENT SUMMARY")
print("=" * 80)

display(
    payment_summary[
        (
            payment_summary[
                "rows"
            ] > 0
        )
        |
        (
            payment_summary[
                "amount"
            ] != 0
        )
    ]
)


# ============================================================
# PROPERTY_MONEYS / ACCOUNT BALANCES
# ============================================================

money_frames = []


for version, detail in details.items():

    money_rows = (
        (
            detail.get(
                "properties"
            )
            or {}
        )
        .get(
            "property_moneys"
        )
        or []
    )

    if money_rows:

        df = pd.DataFrame(
            money_rows
        )

        df.insert(
            0,
            "version",
            version,
        )

        money_frames.append(
            df
        )


if money_frames:

    money = pd.concat(
        money_frames,
        ignore_index=True,
        sort=False,
    )

else:

    money = pd.DataFrame()


print()
print("=" * 80)
print("PROPERTY_MONEYS — ALL AVAILABLE FIELDS")
print("=" * 80)

print(
    list(
        money.columns
    )
)

display(
    money
)


# ============================================================
# PARTY-ACCOUNT TRANSACTION FLOW BY IBAN
# ============================================================

account_rows = []


for version, detail in details.items():

    for payment_type in PAYMENT_PATHS:

        rows = get_rows(
            detail,
            payment_type,
        )

        direction = (
            PAYMENT_PATHS[
                payment_type
            ][0]
        )

        for row in rows:

            amount = float(
                row.get(
                    "payment_amount"
                )
                or 0
            )

            if direction == "incoming":

                party_iban = (
                    row.get(
                        "receiver_account_iban"
                    )
                )

                signed_amount = amount

            else:

                party_iban = (
                    row.get(
                        "payer_account_iban"
                    )
                )

                signed_amount = (
                    -amount
                )


            account_rows.append(
                {
                    "version":
                        version,

                    "payment_type":
                        payment_type,

                    "party_iban":
                        party_iban,

                    "date":
                        row.get(
                            "payment_operation_date"
                        ),

                    "amount":
                        amount,

                    "signed_amount":
                        signed_amount,

                    "payer_name":
                        row.get(
                            "payer_name"
                        ),

                    "receiver_name":
                        row.get(
                            "receiver_name"
                        ),
                }
            )


account_transactions = (
    pd.DataFrame(
        account_rows
    )
)


account_flow = (
    account_transactions
    .groupby(
        [
            "version",
            "party_iban",
        ],
        dropna=False,
    )
    .agg(
        transaction_rows=(
            "amount",
            "size",
        ),

        inflow=(
            "signed_amount",
            lambda s:
                s[
                    s > 0
                ].sum(),
        ),

        outflow=(
            "signed_amount",
            lambda s:
                -s[
                    s < 0
                ].sum(),
        ),

        net_flow=(
            "signed_amount",
            "sum",
        ),
    )
    .reset_index()
)


print()
print("=" * 80)
print("ACCOUNT-LEVEL FLOW")
print("=" * 80)

display(
    account_flow.sort_values(
        [
            "party_iban",
            "version",
        ]
    )
)


# ============================================================
# Q2 DRAFT — QUICK CHECK
# ============================================================

q2draft = account_flow[
    account_flow[
        "version"
    ]
    == "Q2_draft"
]


print()
print("=" * 80)
print("Q2 DRAFT TOTAL")
print("=" * 80)

print(
    "Inflows:",
    q2draft[
        "inflow"
    ].sum()
)

print(
    "Outflows:",
    q2draft[
        "outflow"
    ].sum()
)

print(
    "Net:",
    q2draft[
        "net_flow"
    ].sum()
)

Organization ID: 5f508361-50a4-4464-880d-01f1843cbe76

SELECTED REPORTS


,report_id,year,quarter,signed_date
28677,d861fd70-2ca2-11f0-b8ea-758045529e07,2025,1,2025-05-09 10:12:07
28675,764a691c-3237-4044-9430-44313fbf739f,2025,2,2025-08-07 22:15:52.146213
28682,f5a60c98-bd12-4795-9aef-fb0d9def5471,2025,3,2025-11-06 11:36:41.599296



PAYMENT SUMMARY


,version,payment_type,rows,amount
3,Q1_signed,other_incomes,5,1053359.00
5,Q1_signed,outgoing_expenses,256,1054888.48
19,Q3_signed,other_incomes,4,1056049.06
21,Q3_signed,outgoing_expenses,249,998054.95
27,Q2_draft,other_incomes,3,1050000.00
29,Q2_draft,outgoing_expenses,247,1077997.21



PROPERTY_MONEYS — ALL AVAILABLE FIELDS
['version', 'id', 'party_id', 'office_id', 'report_status', 'account_type', 'account_number', 'account_holder', 'account_holder_code', 'begin_period_balance', 'end_period_balance', 'report_period_income', 'report_period_used_funds', 'created_at']


,version,id,party_id,office_id,report_status,account_type,account_number,account_holder,account_holder_code,begin_period_balance,end_period_balance,report_period_income,report_period_used_funds,created_at
0,Q1_signed,90f8422f-8b29-456f-95c2-1840fb3d96ec,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA263223130000026004000045109,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,32694.44,31164.96,1053359,1054888.48,2025-05-09 10:12:07
1,Q1_signed,9c1ed991-af51-4ac7-9056-ce7af0e80a65,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA723223130000026040000010659,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,0,0,0,0,2025-05-09 10:12:07
2,Q1_signed,30937686-87b6-47aa-991f-f302b888f2f4,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA263223130000026004000045109,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,32694.44,31164.96,1053359,1054888.48,2026-02-12 11:50:10
3,Q1_signed,98d7d78e-d4ad-4a9f-9ab6-924fb487898b,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA723223130000026040000010659,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,0,0,0,0,2026-02-12 11:50:10
4,Q3_signed,aacd4d90-c54d-4523-8002-1d9aa1030fd6,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA263223130000026004000045109,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,3167.75,61161.86,1055000,997005.89,2025-11-06 11:36:42
5,Q3_signed,5efdc73e-0996-4f0e-a817-6467365e5849,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA723223130000026040000010659,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,0,0,1049.06,1049.06,2025-11-06 11:36:42
6,Q2_draft,68f3d610-9bca-49ed-b9be-d06ed22378fa,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA263223130000026004000045109,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,31164.96,3167.75,1050000,1077997.21,2025-08-08 04:33:36
7,Q2_draft,8b5ddf4f-5f76-459b-8076-5a45860aec94,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,5f508361-50a4-4464-880d-01f1843cbe76,2,Поточний рахунок,UA723223130000026040000010659,"АКЦІОНЕРНЕ ТОВАРИСТВО ""ДЕРЖАВНИЙ ЕКСПОРТНО-ІМП...",None,0,0,0,0,2025-08-08 04:33:36



ACCOUNT-LEVEL FLOW


,version,party_iban,transaction_rows,inflow,outflow,net_flow
0,Q1_signed,UA263223130000026004000045109,5,1053359.00,-0.00,1053359.00
2,Q2_draft,UA263223130000026004000045109,3,1050000.00,-0.00,1050000.00
4,Q3_signed,UA263223130000026004000045109,3,1055000.00,-0.00,1055000.00
5,Q3_signed,UA723223130000026040000010659,1,1049.06,-0.00,1049.06
1,Q1_signed,NaN,256,0.00,1054888.48,-1054888.48
3,Q2_draft,NaN,247,0.00,1077997.21,-1077997.21
6,Q3_signed,NaN,249,0.00,998054.95,-998054.95



Q2 DRAFT TOTAL
Inflows: 1050000.0
Outflows: 1077997.21
Net: -27997.209999999963


In [64]:
q2_transactions = account_transactions[
    account_transactions["version"]
    == "Q2_draft"
].copy()

q2_transactions["date"] = pd.to_datetime(
    q2_transactions["date"],
    errors="coerce",
)

print(
    "Transactions:",
    len(q2_transactions)
)

print(
    "Earliest:",
    q2_transactions["date"].min()
)

print(
    "Latest:",
    q2_transactions["date"].max()
)

outside_q2 = q2_transactions[
    ~q2_transactions["date"].between(
        "2025-04-01",
        "2025-06-30",
        inclusive="both",
    )
]

print(
    "Outside 2025 Q2:",
    len(outside_q2)
)

display(outside_q2)

Transactions: 250
Earliest: 2025-04-01 00:00:00
Latest: 2025-06-30 00:00:00
Outside 2025 Q2: 0


,version,payment_type,party_iban,date,amount,signed_amount,payer_name,receiver_name


In [65]:
from pathlib import Path
import pandas as pd


REPORTS_DIR = Path(
    "data/interim/reports"
)


# ============================================================
# 1. ANALYTICAL OVERRIDES
# ============================================================

analysis_overrides = pd.DataFrame(
    [
        {
            "organization_id":
                "5f508361-50a4-4464-880d-01f1843cbe76",

            "year":
                2025,

            "quarter":
                2,

            "official_selected_report_id":
                "764a691c-3237-4044-9430-44313fbf739f",

            "analysis_selected_report_id":
                "317a1c90-a5a3-43e5-8e62-e205cc84fb1e",

            "analysis_selection_method":
                "manual_financial_continuity_override",

            "override_reason":
                (
                    "Unsigned report contains 250 Q2 transactions; "
                    "opening balance exactly matches signed Q1 closing "
                    "balance and closing balance exactly matches signed "
                    "Q3 opening balance. Signed Q2 financial section is empty."
                ),

            "continuity_exact":
                True,

            "analysis_override":
                True,
        }
    ]
)


override_path = (
    REPORTS_DIR
    / "analysis_report_overrides.parquet"
)

analysis_overrides.to_parquet(
    override_path,
    index=False,
)

print(
    "Saved:",
    override_path
)


# ============================================================
# 2. BUILD ANALYTICAL REPORT MANIFEST
# ============================================================

selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()


selected["report_id"] = (
    selected["report_id"]
    .astype(str)
)

selected["organization_id"] = (
    selected["organization_id"]
    .astype(str)
)


# Preserve official source selection explicitly.
selected[
    "official_selected_report_id"
] = selected[
    "report_id"
]

selected[
    "analysis_selected_report_id"
] = selected[
    "report_id"
]

selected[
    "analysis_selection_method"
] = (
    "official_selected_signed_report"
)

selected[
    "analysis_override"
] = False

selected[
    "override_reason"
] = None

selected[
    "continuity_exact"
] = None


# ============================================================
# 3. APPLY OVERRIDES
# ============================================================

for _, override in (
    analysis_overrides.iterrows()
):

    mask = (
        (
            selected[
                "organization_id"
            ]
            ==
            str(
                override[
                    "organization_id"
                ]
            )
        )
        &
        (
            selected[
                "year"
            ]
            ==
            override[
                "year"
            ]
        )
        &
        (
            selected[
                "quarter"
            ]
            ==
            override[
                "quarter"
            ]
        )
    )


    matches = int(
        mask.sum()
    )

    if matches != 1:

        raise RuntimeError(
            "Expected exactly one official selected report "
            f"for override period, found {matches}"
        )


    actual_official_id = str(
        selected.loc[
            mask,
            "report_id",
        ].iloc[0]
    )

    expected_official_id = str(
        override[
            "official_selected_report_id"
        ]
    )

    if (
        actual_official_id
        != expected_official_id
    ):

        raise RuntimeError(
            "Official report ID does not match override definition: "
            f"{actual_official_id} != {expected_official_id}"
        )


    selected.loc[
        mask,
        "analysis_selected_report_id"
    ] = str(
        override[
            "analysis_selected_report_id"
        ]
    )

    selected.loc[
        mask,
        "analysis_selection_method"
    ] = override[
        "analysis_selection_method"
    ]

    selected.loc[
        mask,
        "analysis_override"
    ] = True

    selected.loc[
        mask,
        "override_reason"
    ] = override[
        "override_reason"
    ]

    selected.loc[
        mask,
        "continuity_exact"
    ] = override[
        "continuity_exact"
    ]


# ============================================================
# 4. QA
# ============================================================

print(
    "Official selected reports:",
    len(selected)
)

print(
    "Analytical overrides:",
    int(
        selected[
            "analysis_override"
        ].sum()
    )
)

assert len(selected) == 78791

assert (
    selected[
        "analysis_selected_report_id"
    ]
    .notna()
    .all()
)


display(
    selected[
        selected[
            "analysis_override"
        ]
    ][
        [
            "organization_id",
            "year",
            "quarter",
            "official_selected_report_id",
            "analysis_selected_report_id",
            "analysis_selection_method",
            "continuity_exact",
            "override_reason",
        ]
    ]
)


# ============================================================
# 5. SAVE
# ============================================================

analysis_manifest_path = (
    REPORTS_DIR
    / "analysis_selected_reports_manifest.parquet"
)

selected.to_parquet(
    analysis_manifest_path,
    index=False,
)

print()
print(
    "Saved:",
    analysis_manifest_path
)

Saved: data\interim\reports\analysis_report_overrides.parquet
Official selected reports: 78791
Analytical overrides: 1


,organization_id,year,quarter,official_selected_report_id,analysis_selected_report_id,analysis_selection_method,continuity_exact,override_reason
28675,5f508361-50a4-4464-880d-01f1843cbe76,2025,2,764a691c-3237-4044-9430-44313fbf739f,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,manual_financial_continuity_override,True,Unsigned report contains 250 Q2 transactions; ...



Saved: data\interim\reports\analysis_selected_reports_manifest.parquet


In [66]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
from collections import defaultdict
import json

import pandas as pd
from tqdm.auto import tqdm


RAW_DIR = Path(
    "data/raw/report_details"
)

REPORTS_DIR = Path(
    "data/interim/reports"
)

OUT_DIR = Path(
    "data/interim/audit"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# SOURCE MANIFEST
#
# IMPORTANT:
# audit OFFICIAL selected signed reports,
# not analytical overrides
# ============================================================

selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()

selected["report_id"] = (
    selected["report_id"]
    .astype(str)
)

selected["organization_id"] = (
    selected["organization_id"]
    .astype(str)
)

selected["root_party_id"] = (
    selected["root_party_id"]
    .astype(str)
)


# ============================================================
# PARTY NAMES
# ============================================================

organizations = pd.read_parquet(
    "data/processed/normalized_v0_1/"
    "organizations.parquet"
)

NAME_CANDIDATES = [
    "name",
    "organization_name",
    "party_name",
    "full_name",
]

name_col = next(
    (
        col
        for col in NAME_CANDIDATES
        if col in organizations.columns
    ),
    None,
)

if name_col is not None:

    org_names = (
        organizations[
            [
                "organization_id",
                name_col,
            ]
        ]
        .drop_duplicates(
            "organization_id"
        )
        .set_index(
            "organization_id"
        )[name_col]
    )

else:

    org_names = {}


# ============================================================
# NUMERIC HELPERS
# ============================================================

def dec(value):

    if value is None:
        return None

    try:

        return Decimal(
            str(value)
        )

    except (
        InvalidOperation,
        ValueError,
        TypeError,
    ):

        return None


def amount_sum(rows):

    total = Decimal("0")

    for row in rows:

        value = dec(
            row.get(
                "payment_amount"
            )
        )

        if value is not None:
            total += value

    return total


def money_equal(
    left,
    right,
    tolerance=Decimal("0.01"),
):

    if (
        left is None
        or right is None
    ):
        return None

    return (
        abs(left - right)
        <= tolerance
    )


# ============================================================
# STORAGE
# ============================================================

snapshot_rows = []
report_finance_rows = []

missing_raw = []


# ============================================================
# EXTRACT ACCOUNT SNAPSHOTS + STATE FUNDING FLAGS
# ============================================================

for report in tqdm(
    selected.itertuples(
        index=False
    ),
    total=len(selected),
    desc="Audit extraction",
    unit="report",
    dynamic_ncols=True,
):

    report_id = str(
        report.report_id
    )

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if not path.exists():

        missing_raw.append(
            report_id
        )

        continue


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = (
            json.load(f)[
                "results"
            ]
        )


    # --------------------------------------------------------
    # DIRECT SOURCE INDICATORS OF STATE-FUNDING EXPOSURE
    # --------------------------------------------------------

    payment_info = (
        detail.get(
            "payment_info"
        )
        or {}
    )

    incoming = (
        payment_info.get(
            "incoming"
        )
        or {}
    )

    outgoing = (
        payment_info.get(
            "outgoing"
        )
        or {}
    )

    state_rows = (
        incoming.get(
            "state_funding"
        )
        or []
    )

    budget_rows = (
        outgoing.get(
            "budget_expenses"
        )
        or []
    )

    state_amount = (
        amount_sum(
            state_rows
        )
    )

    budget_amount = (
        amount_sum(
            budget_rows
        )
    )


    report_finance_rows.append(
        {
            "report_id":
                report_id,

            "organization_id":
                str(
                    report.organization_id
                ),

            "root_party_id":
                str(
                    report.root_party_id
                ),

            "year":
                int(report.year),

            "quarter":
                int(report.quarter),

            "state_funding_rows":
                len(state_rows),

            "state_funding_amount":
                float(state_amount),

            "budget_expenses_rows":
                len(budget_rows),

            "budget_expenses_amount":
                float(budget_amount),

            "has_direct_state_funding":
                len(state_rows) > 0,

            "has_budget_account_expenses":
                len(budget_rows) > 0,
        }
    )


    # --------------------------------------------------------
    # PROPERTY_MONEYS
    # --------------------------------------------------------

    properties = (
        detail.get(
            "properties"
        )
        or {}
    )

    money_rows = (
        properties.get(
            "property_moneys"
        )
        or []
    )


    for row in money_rows:

        begin = dec(
            row.get(
                "begin_period_balance"
            )
        )

        end = dec(
            row.get(
                "end_period_balance"
            )
        )

        income = dec(
            row.get(
                "report_period_income"
            )
        )

        used = dec(
            row.get(
                "report_period_used_funds"
            )
        )


        # Internal arithmetic:
        #
        # begin + income - used = end

        if None in {
            begin,
            end,
            income,
            used,
        }:

            math_delta = None
            math_ok = None

        else:

            calculated_end = (
                begin
                + income
                - used
            )

            math_delta = (
                end
                - calculated_end
            )

            math_ok = money_equal(
                end,
                calculated_end,
            )


        snapshot_rows.append(
            {
                "report_id":
                    report_id,

                "organization_id":
                    str(
                        report.organization_id
                    ),

                "root_party_id":
                    str(
                        report.root_party_id
                    ),

                "year":
                    int(report.year),

                "quarter":
                    int(report.quarter),

                "signed_date":
                    getattr(
                        report,
                        "signed_date",
                        None,
                    ),

                "source_row_id":
                    row.get("id"),

                "account_type":
                    row.get(
                        "account_type"
                    ),

                "account_number":
                    row.get(
                        "account_number"
                    ),

                "account_holder":
                    row.get(
                        "account_holder"
                    ),

                "account_holder_code":
                    row.get(
                        "account_holder_code"
                    ),

                "begin_period_balance":
                    float(begin)
                    if begin is not None
                    else None,

                "end_period_balance":
                    float(end)
                    if end is not None
                    else None,

                "report_period_income":
                    float(income)
                    if income is not None
                    else None,

                "report_period_used_funds":
                    float(used)
                    if used is not None
                    else None,

                "internal_math_delta":
                    float(math_delta)
                    if math_delta is not None
                    else None,

                "internal_math_ok":
                    math_ok,

                "created_at":
                    row.get(
                        "created_at"
                    ),
            }
        )


print()
print(
    "Missing RAW:",
    len(missing_raw)
)

if missing_raw:

    print(
        "First missing IDs:",
        missing_raw[:10]
    )


# ============================================================
# DATAFRAMES
# ============================================================

snapshots_raw = pd.DataFrame(
    snapshot_rows
)

report_finance = pd.DataFrame(
    report_finance_rows
)


# ============================================================
# DEDUPLICATE PROPERTY_MONEYS
#
# We already observed exact substantive duplicates whose only
# differences are API row id / created_at.
# ============================================================

SUBSTANTIVE_ACCOUNT_FIELDS = [
    "report_id",
    "account_type",
    "account_number",
    "account_holder",
    "account_holder_code",
    "begin_period_balance",
    "end_period_balance",
    "report_period_income",
    "report_period_used_funds",
]


snapshots = (
    snapshots_raw
    .drop_duplicates(
        subset=SUBSTANTIVE_ACCOUNT_FIELDS
    )
    .copy()
)


print()
print(
    "Raw property_moneys rows:",
    len(snapshots_raw)
)

print(
    "After substantive dedup:",
    len(snapshots)
)

print(
    "Exact technical duplicates removed:",
    len(snapshots_raw)
    - len(snapshots)
)


# ============================================================
# DETECT MULTIPLE DIFFERENT SNAPSHOTS FOR SAME ACCOUNT/REPORT
# ============================================================

account_variant_counts = (
    snapshots
    .groupby(
        [
            "report_id",
            "account_number",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "account_variant_count"
    )
    .reset_index()
)


snapshots = snapshots.merge(
    account_variant_counts,
    on=[
        "report_id",
        "account_number",
    ],
    how="left",
)


account_conflicts = (
    snapshots[
        snapshots[
            "account_variant_count"
        ] > 1
    ]
    .copy()
)


# ============================================================
# INTERNAL MATH ISSUES
# ============================================================

internal_math_issues = (
    snapshots[
        snapshots[
            "internal_math_ok"
        ] == False
    ]
    .copy()
)


# ============================================================
# ONLY UNAMBIGUOUS ACCOUNT SNAPSHOTS FOR CONTINUITY
# ============================================================

continuity_source = (
    snapshots[
        (
            snapshots[
                "account_variant_count"
            ] == 1
        )
        &
        (
            snapshots[
                "account_number"
            ].notna()
        )
        &
        (
            snapshots[
                "quarter"
            ].isin(
                [1, 2, 3, 4]
            )
        )
    ]
    .copy()
)


# Quarter index allows:
#
# Q1 -> Q2
# Q2 -> Q3
# Q3 -> Q4
# Q4 -> next-year Q1

continuity_source[
    "period_index"
] = (
    continuity_source[
        "year"
    ] * 4
    +
    continuity_source[
        "quarter"
    ]
    - 1
)


# ============================================================
# BUILD REPORT-PERIOD INDEX
# ============================================================

quarterly_reports = (
    selected[
        selected[
            "quarter"
        ].isin(
            [1, 2, 3, 4]
        )
    ][
        [
            "report_id",
            "organization_id",
            "root_party_id",
            "year",
            "quarter",
            "signed_date",
        ]
    ]
    .copy()
)

quarterly_reports[
    "period_index"
] = (
    quarterly_reports[
        "year"
    ] * 4
    +
    quarterly_reports[
        "quarter"
    ]
    - 1
)


# ============================================================
# COMPARE CONSECUTIVE QUARTERS
# ============================================================

continuity_rows = []


for organization_id, org_reports in tqdm(
    quarterly_reports.groupby(
        "organization_id"
    ),
    desc="Quarter continuity",
    unit="org",
    dynamic_ncols=True,
):

    org_reports = (
        org_reports
        .sort_values(
            "period_index"
        )
        .reset_index(
            drop=True
        )
    )


    for i in range(
        len(org_reports) - 1
    ):

        prev = org_reports.iloc[i]
        nxt = org_reports.iloc[i + 1]


        # Only truly consecutive quarters.
        if (
            int(
                nxt[
                    "period_index"
                ]
            )
            -
            int(
                prev[
                    "period_index"
                ]
            )
            != 1
        ):

            continue


        prev_accounts = (
            continuity_source[
                continuity_source[
                    "report_id"
                ]
                ==
                str(
                    prev[
                        "report_id"
                    ]
                )
            ]
            .set_index(
                "account_number"
            )
        )


        next_accounts = (
            continuity_source[
                continuity_source[
                    "report_id"
                ]
                ==
                str(
                    nxt[
                        "report_id"
                    ]
                )
            ]
            .set_index(
                "account_number"
            )
        )


        account_numbers = (
            set(
                prev_accounts.index
            )
            |
            set(
                next_accounts.index
            )
        )


        for account_number in (
            account_numbers
        ):

            prev_exists = (
                account_number
                in prev_accounts.index
            )

            next_exists = (
                account_number
                in next_accounts.index
            )


            prev_end = None
            next_begin = None


            if prev_exists:

                value = (
                    prev_accounts
                    .loc[
                        account_number,
                        "end_period_balance",
                    ]
                )

                prev_end = dec(
                    value
                )


            if next_exists:

                value = (
                    next_accounts
                    .loc[
                        account_number,
                        "begin_period_balance",
                    ]
                )

                next_begin = dec(
                    value
                )


            # ----------------------------------------------
            # CLASSIFY
            # ----------------------------------------------

            if (
                prev_exists
                and next_exists
            ):

                exact = money_equal(
                    prev_end,
                    next_begin,
                )

                if exact is None:

                    status = (
                        "missing_balance_value"
                    )

                elif exact:

                    status = (
                        "continuous"
                    )

                else:

                    status = (
                        "balance_mismatch"
                    )


            elif prev_exists:

                if (
                    prev_end is not None
                    and money_equal(
                        prev_end,
                        Decimal("0"),
                    )
                ):

                    status = (
                        "account_closed_at_zero"
                    )

                else:

                    status = (
                        "account_missing_next_"
                        "with_nonzero_end"
                    )


            else:

                if (
                    next_begin is not None
                    and money_equal(
                        next_begin,
                        Decimal("0"),
                    )
                ):

                    status = (
                        "new_account_with_zero_begin"
                    )

                else:

                    status = (
                        "new_account_with_"
                        "nonzero_begin"
                    )


            delta = None

            if (
                prev_end is not None
                and next_begin is not None
            ):

                delta = (
                    next_begin
                    - prev_end
                )


            continuity_rows.append(
                {
                    "organization_id":
                        str(
                            organization_id
                        ),

                    "root_party_id":
                        str(
                            prev[
                                "root_party_id"
                            ]
                        ),

                    "account_number":
                        account_number,

                    "previous_report_id":
                        str(
                            prev[
                                "report_id"
                            ]
                        ),

                    "previous_year":
                        int(
                            prev["year"]
                        ),

                    "previous_quarter":
                        int(
                            prev["quarter"]
                        ),

                    "next_report_id":
                        str(
                            nxt[
                                "report_id"
                            ]
                        ),

                    "next_year":
                        int(
                            nxt["year"]
                        ),

                    "next_quarter":
                        int(
                            nxt["quarter"]
                        ),

                    "previous_end_balance":
                        float(prev_end)
                        if prev_end is not None
                        else None,

                    "next_begin_balance":
                        float(next_begin)
                        if next_begin is not None
                        else None,

                    "balance_delta":
                        float(delta)
                        if delta is not None
                        else None,

                    "continuity_status":
                        status,
                }
            )


continuity = pd.DataFrame(
    continuity_rows
)


# ============================================================
# STATE-FUNDING PARTY FLAGS
#
# IMPORTANT:
# this means "party network has direct evidence of
# state-funding activity somewhere in selected reports".
#
# It does NOT mean every expense by every office is attributed
# to state financing.
# ============================================================

party_state_exposure = (
    report_finance
    .groupby(
        "root_party_id",
        as_index=False,
    )
    .agg(
        direct_state_funding_rows=(
            "state_funding_rows",
            "sum",
        ),

        direct_state_funding_amount=(
            "state_funding_amount",
            "sum",
        ),

        budget_expense_rows=(
            "budget_expenses_rows",
            "sum",
        ),

        budget_expense_amount=(
            "budget_expenses_amount",
            "sum",
        ),

        reports_with_direct_state_funding=(
            "has_direct_state_funding",
            "sum",
        ),

        reports_with_budget_expenses=(
            "has_budget_account_expenses",
            "sum",
        ),
    )
)


party_state_exposure[
    "party_state_funding_exposure"
] = (
    (
        party_state_exposure[
            "direct_state_funding_rows"
        ] > 0
    )
    |
    (
        party_state_exposure[
            "budget_expense_rows"
        ] > 0
    )
)


if name_col is not None:

    party_state_exposure[
        "party_name"
    ] = party_state_exposure[
        "root_party_id"
    ].map(
        org_names
    )


# ============================================================
# ADD PARTY FLAGS TO CONTINUITY TABLE
# ============================================================

continuity = continuity.merge(
    party_state_exposure[
        [
            "root_party_id",
            "party_state_funding_exposure",
            "direct_state_funding_amount",
            "budget_expense_amount",
        ]
    ],
    on="root_party_id",
    how="left",
)


if name_col is not None:

    continuity[
        "party_name"
    ] = continuity[
        "root_party_id"
    ].map(
        org_names
    )

    continuity[
        "organization_name"
    ] = continuity[
        "organization_id"
    ].map(
        org_names
    )


# ============================================================
# BREAKS
# ============================================================

NORMAL_STATUSES = {
    "continuous",
    "account_closed_at_zero",
    "new_account_with_zero_begin",
}


continuity_breaks = (
    continuity[
        ~continuity[
            "continuity_status"
        ].isin(
            NORMAL_STATUSES
        )
    ]
    .copy()
)


state_funding_breaks = (
    continuity_breaks[
        continuity_breaks[
            "party_state_funding_exposure"
        ] == True
    ]
    .copy()
)


# ============================================================
# SAVE
# ============================================================

snapshots.to_parquet(
    OUT_DIR
    / "selected_account_snapshots.parquet",
    index=False,
)

internal_math_issues.to_parquet(
    OUT_DIR
    / "selected_account_internal_math_issues.parquet",
    index=False,
)

account_conflicts.to_parquet(
    OUT_DIR
    / "selected_account_snapshot_conflicts.parquet",
    index=False,
)

continuity.to_parquet(
    OUT_DIR
    / "selected_account_continuity.parquet",
    index=False,
)

continuity_breaks.to_parquet(
    OUT_DIR
    / "selected_account_continuity_breaks.parquet",
    index=False,
)

state_funding_breaks.to_parquet(
    OUT_DIR
    / "selected_account_continuity_breaks_state_funding_parties.parquet",
    index=False,
)

party_state_exposure.to_parquet(
    OUT_DIR
    / "party_state_funding_exposure.parquet",
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 72)
print("ACCOUNT INTEGRITY AUDIT")
print("=" * 72)

print(
    "Selected reports:",
    len(selected)
)

print(
    "Reports missing RAW:",
    len(missing_raw)
)

print(
    "Account snapshots:",
    len(snapshots)
)

print(
    "Internal math issues:",
    len(internal_math_issues)
)

print(
    "Conflicting account snapshots:",
    len(account_conflicts)
)

print()

print("CONTINUITY STATUS")

print(
    continuity[
        "continuity_status"
    ].value_counts()
)

print()

print(
    "Total continuity breaks:",
    len(continuity_breaks)
)

print(
    "Breaks in state-funding party networks:",
    len(state_funding_breaks)
)

print()

print(
    "Parties with state-funding exposure:",
    int(
        party_state_exposure[
            "party_state_funding_exposure"
        ].sum()
    )
)


print()
print("=" * 72)
print("STATE-FUNDING PARTIES — CONTINUITY BREAKS")
print("=" * 72)

display(
    state_funding_breaks[
        [
            col
            for col in [
                "party_name",
                "organization_name",
                "account_number",

                "previous_year",
                "previous_quarter",
                "previous_end_balance",

                "next_year",
                "next_quarter",
                "next_begin_balance",

                "balance_delta",
                "continuity_status",

                "previous_report_id",
                "next_report_id",

                "direct_state_funding_amount",
                "budget_expense_amount",
            ]
            if col
            in state_funding_breaks.columns
        ]
    ]
    .sort_values(
        "balance_delta",
        key=lambda s: s.abs(),
        ascending=False,
    )
    .head(100)
)

Audit extraction:   0%|                                                                  | 0/78791 [00:00<?, ?…


Missing RAW: 0

Raw property_moneys rows: 19138
After substantive dedup: 12588
Exact technical duplicates removed: 6550


Quarter continuity:   0%|                                                                    | 0/8432 [00:00<?…


ACCOUNT INTEGRITY AUDIT
Selected reports: 78791
Reports missing RAW: 0
Account snapshots: 12588
Internal math issues: 0
Conflicting account snapshots: 8

CONTINUITY STATUS
continuity_status
continuous                               8425
account_closed_at_zero                    680
new_account_with_zero_begin               461
new_account_with_nonzero_begin            142
account_missing_next_with_nonzero_end     131
balance_mismatch                           84
Name: count, dtype: int64

Total continuity breaks: 357
Breaks in state-funding party networks: 109

Parties with state-funding exposure: 17

STATE-FUNDING PARTIES — CONTINUITY BREAKS


,party_name,organization_name,account_number,previous_year,previous_quarter,previous_end_balance,next_year,next_quarter,next_begin_balance,balance_delta,continuity_status,previous_report_id,next_report_id,direct_state_funding_amount,budget_expense_amount
7788,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Чернігівська обласна організація ПП ""Рідне місто""",UA053052990000026006006302776,2024,3,76004.06,2024,4,3284.06,-72720.00,balance_mismatch,48854fd0-a00d-11ef-9197-0fbca96b9f8a,9d7aa980-e618-11ef-9743-8991a53f54af,0.000000e+00,4.616434e+04
7830,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Новомосковська міська організація Партії Зелен...,UA373052990000026005050311231,2021,1,74175.00,2021,2,11492.00,-62683.00,balance_mismatch,5c70bc00-7d15-11ec-ba2b-a5f020a204de,64ea2fd0-8295-11ec-80e8-cd54133e5d2c,0.000000e+00,5.906700e+04
9466,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,Івано-Франківська територіальна організація По...,UA133204780000026003924420903,2026,1,0.00,2026,2,36585.46,36585.46,balance_mismatch,3f67abe2-d9ad-4e29-8301-8c3c2944a604,085c0cfa-899d-4d2b-bf78-5cd344e65eb6,7.272993e+08,8.315782e+08
9467,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,Івано-Франківська територіальна організація По...,UA593805820000026004010333684,2026,1,36585.46,2026,2,0.00,-36585.46,balance_mismatch,3f67abe2-d9ad-4e29-8301-8c3c2944a604,085c0cfa-899d-4d2b-bf78-5cd344e65eb6,7.272993e+08,8.315782e+08
8810,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»,"Кам""янець-Подільська міська організація Всеукр...",UA593052990000026005036010222,2025,3,0.00,2025,4,8649.79,8649.79,balance_mismatch,a182fe66-ff4c-4f69-9812-596e9ad15035,713b908f-e97d-41f8-9c35-9e53bfb58056,0.000000e+00,4.245476e+04
7895,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,UA433375460000026007055029277,2021,1,475745.28,2021,2,472857.28,-2888.00,balance_mismatch,05291170-87e9-11ec-87ad-bf13a104ce8a,ede95260-fc92-11ee-92b7-3b3b09128e69,0.000000e+00,1.153100e+04
5062,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,ХАРКІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,UA103515330000026000052222703,2023,3,41.36,2023,4,1863.88,1822.52,balance_mismatch,6579f330-fe4a-11ee-8a6c-27bc4724684c,ea23f9c0-ea7a-11ee-8a6c-27bc4724684c,0.000000e+00,1.259139e+05
5063,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,ХАРКІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,UA103515330000026000052222703,2023,4,1863.88,2024,1,41.36,-1822.52,balance_mismatch,ea23f9c0-ea7a-11ee-8a6c-27bc4724684c,e5e0b740-0d57-11ef-823a-73af3dbabf67,0.000000e+00,1.259139e+05
1680,ПОЛІТИЧНА ПАРТІЯ «КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА П...,Житомирська обласна організація політичної пар...,UA593052990000026003016402904,2024,2,119363.00,2024,3,117563.35,-1799.65,balance_mismatch,6b119230-3f72-11ef-b4cc-1f31c784e768,adfc2580-887c-11ef-ac2d-b9ec46358bc3,0.000000e+00,2.500750e+04
9556,ПОЛІТИЧНА ПАРТІЯ «ЗА ОДЕЩИНУ»,ПОЛІТИЧНА ПАРТІЯ «ЗА ОДЕЩИНУ»,UA463282090000026009000019813,2025,2,8655.59,2025,3,7405.59,-1250.00,balance_mismatch,0ac17bfe-c08d-4a4b-81a1-de1d4dbe9362,7314c0f8-0028-40de-aee7-220446674e7e,0.000000e+00,3.300000e+03


In [67]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import re
import unicodedata

import pandas as pd
from tqdm.auto import tqdm


REPORTS_DIR = Path(
    "data/interim/reports"
)

AUDIT_DIR = Path(
    "data/interim/audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD PREVIOUSLY EXTRACTED ACCOUNT SNAPSHOTS
# ============================================================

snapshots_path = (
    AUDIT_DIR
    / "selected_account_snapshots.parquet"
)

if not snapshots_path.exists():
    raise FileNotFoundError(
        "selected_account_snapshots.parquet not found. "
        "Let the previous audit finish first."
    )

snapshots = pd.read_parquet(
    snapshots_path
).copy()


selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()


selected["report_id"] = (
    selected["report_id"]
    .astype(str)
)

selected["organization_id"] = (
    selected["organization_id"]
    .astype(str)
)

selected["root_party_id"] = (
    selected["root_party_id"]
    .astype(str)
)


# ============================================================
# PARTY / ORGANIZATION NAMES
# ============================================================

organizations = pd.read_parquet(
    "data/processed/normalized_v0_1/"
    "organizations.parquet"
)

NAME_CANDIDATES = [
    "name",
    "organization_name",
    "party_name",
    "full_name",
]

name_col = next(
    (
        col
        for col
        in NAME_CANDIDATES
        if col in organizations.columns
    ),
    None,
)


if name_col is not None:

    org_names = (
        organizations[
            [
                "organization_id",
                name_col,
            ]
        ]
        .drop_duplicates(
            "organization_id"
        )
        .set_index(
            "organization_id"
        )[name_col]
    )

else:

    org_names = {}


# ============================================================
# MONEY HELPERS
# ============================================================

def dec(value):

    if value is None:
        return None

    try:

        if pd.isna(value):
            return None

    except Exception:
        pass

    try:

        return Decimal(
            str(value)
        )

    except (
        InvalidOperation,
        ValueError,
        TypeError,
    ):

        return None


def money_equal(
    left,
    right,
    tolerance=Decimal("0.01"),
):

    if (
        left is None
        or right is None
    ):
        return None

    return (
        abs(left - right)
        <= tolerance
    )


# ============================================================
# ACCOUNT NUMBER NORMALIZATION
# ============================================================

ZERO_WIDTH_CHARS = {
    "\u200b",
    "\u200c",
    "\u200d",
    "\u2060",
    "\ufeff",
}


def normalize_account_number(value):
    """
    Conservative canonicalization.

    Always:
      - Unicode NFKC
      - remove zero-width chars
      - uppercase
      - remove all whitespace

    Additionally:
      - remove common separators (- . / \\ _)
        ONLY when doing so produces a valid-looking
        Ukrainian IBAN: UA + 27 digits.

    Raw value is never overwritten.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    raw = str(value)

    raw = unicodedata.normalize(
        "NFKC",
        raw,
    )

    raw = "".join(
        ch
        for ch in raw
        if ch not in ZERO_WIDTH_CHARS
    )

    raw = raw.upper()

    # Remove every Unicode whitespace character.
    whitespace_clean = "".join(
        ch
        for ch in raw
        if not ch.isspace()
    )

    if not whitespace_clean:
        return None


    # First: already standard after whitespace cleanup.
    if re.fullmatch(
        r"UA\d{27}",
        whitespace_clean,
    ):
        return whitespace_clean


    # Second: cautiously try common visual separators.
    separator_clean = re.sub(
        r"[-._/\\]",
        "",
        whitespace_clean,
    )

    if re.fullmatch(
        r"UA\d{27}",
        separator_clean,
    ):
        return separator_clean


    # Non-standard identifier:
    # preserve conservative normalized form,
    # do not destroy punctuation.
    return whitespace_clean


snapshots[
    "account_number_raw"
] = snapshots[
    "account_number"
]


snapshots[
    "account_number_normalized"
] = snapshots[
    "account_number_raw"
].map(
    normalize_account_number
)


snapshots[
    "is_standard_ua_iban"
] = snapshots[
    "account_number_normalized"
].str.fullmatch(
    r"UA\d{27}",
    na=False,
)


# ============================================================
# ACCOUNT SPELLING QA
# ============================================================

spelling_variants = (
    snapshots[
        snapshots[
            "account_number_normalized"
        ].notna()
    ]
    .groupby(
        "account_number_normalized",
        dropna=False,
    )
    .agg(
        raw_variant_count=(
            "account_number_raw",
            "nunique",
        ),

        raw_examples=(
            "account_number_raw",
            lambda s:
                " | ".join(
                    list(
                        dict.fromkeys(
                            str(x)
                            for x in s
                            if pd.notna(x)
                        )
                    )[:10]
                ),
        ),

        report_count=(
            "report_id",
            "nunique",
        ),

        organization_count=(
            "organization_id",
            "nunique",
        ),
    )
    .reset_index()
)


multiple_spellings = (
    spelling_variants[
        spelling_variants[
            "raw_variant_count"
        ] > 1
    ]
    .copy()
)


nonstandard_accounts = (
    snapshots[
        (
            snapshots[
                "account_number_normalized"
            ].notna()
        )
        &
        (
            ~snapshots[
                "is_standard_ua_iban"
            ]
        )
    ][
        [
            "account_number_raw",
            "account_number_normalized",
            "organization_id",
            "report_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
    .copy()
)


# ============================================================
# DEDUP AGAIN — THIS TIME BY NORMALIZED ACCOUNT
# ============================================================

NORMALIZED_SUBSTANTIVE_FIELDS = [
    "report_id",
    "account_type",
    "account_number_normalized",
    "account_holder",
    "account_holder_code",
    "begin_period_balance",
    "end_period_balance",
    "report_period_income",
    "report_period_used_funds",
]


normalized_snapshots = (
    snapshots
    .drop_duplicates(
        subset=NORMALIZED_SUBSTANTIVE_FIELDS
    )
    .copy()
)


# ============================================================
# DETECT CONFLICTING SNAPSHOTS FOR SAME NORMALIZED ACCOUNT
# ============================================================

normalized_variant_counts = (
    normalized_snapshots[
        normalized_snapshots[
            "account_number_normalized"
        ].notna()
    ]
    .groupby(
        [
            "report_id",
            "account_number_normalized",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "normalized_account_variant_count"
    )
    .reset_index()
)


normalized_snapshots = (
    normalized_snapshots.merge(
        normalized_variant_counts,
        on=[
            "report_id",
            "account_number_normalized",
        ],
        how="left",
    )
)


normalized_conflicts = (
    normalized_snapshots[
        normalized_snapshots[
            "normalized_account_variant_count"
        ] > 1
    ]
    .copy()
)


# ============================================================
# RECHECK INTERNAL ARITHMETIC
# ============================================================

internal_math_ok = []
internal_math_delta = []


for row in (
    normalized_snapshots.itertuples(
        index=False
    )
):

    begin = dec(
        row.begin_period_balance
    )

    end = dec(
        row.end_period_balance
    )

    income = dec(
        row.report_period_income
    )

    used = dec(
        row.report_period_used_funds
    )


    if None in {
        begin,
        end,
        income,
        used,
    }:

        internal_math_ok.append(
            None
        )

        internal_math_delta.append(
            None
        )

        continue


    calculated_end = (
        begin
        + income
        - used
    )

    delta = (
        end
        - calculated_end
    )

    internal_math_delta.append(
        float(delta)
    )

    internal_math_ok.append(
        money_equal(
            end,
            calculated_end,
        )
    )


normalized_snapshots[
    "internal_math_ok_normalized"
] = internal_math_ok

normalized_snapshots[
    "internal_math_delta_normalized"
] = internal_math_delta


normalized_math_issues = (
    normalized_snapshots[
        normalized_snapshots[
            "internal_math_ok_normalized"
        ] == False
    ]
    .copy()
)


# ============================================================
# CONTINUITY SOURCE
#
# Exclude ambiguous report/account combinations.
# ============================================================

continuity_source = (
    normalized_snapshots[
        (
            normalized_snapshots[
                "normalized_account_variant_count"
            ] == 1
        )
        &
        (
            normalized_snapshots[
                "account_number_normalized"
            ].notna()
        )
        &
        (
            normalized_snapshots[
                "quarter"
            ].isin(
                [1, 2, 3, 4]
            )
        )
    ]
    .copy()
)


# ============================================================
# QUARTERLY REPORT INDEX
# ============================================================

quarterly_reports = (
    selected[
        selected[
            "quarter"
        ].isin(
            [1, 2, 3, 4]
        )
    ][
        [
            "report_id",
            "organization_id",
            "root_party_id",
            "year",
            "quarter",
            "signed_date",
        ]
    ]
    .copy()
)


quarterly_reports[
    "period_index"
] = (
    quarterly_reports[
        "year"
    ] * 4
    +
    quarterly_reports[
        "quarter"
    ]
    - 1
)


# ============================================================
# INDEX ACCOUNT SNAPSHOTS BY REPORT
#
# Avoid repeatedly filtering the full dataframe.
# ============================================================

accounts_by_report = {}


for report_id, df in (
    continuity_source.groupby(
        "report_id"
    )
):

    accounts_by_report[
        str(report_id)
    ] = (
        df.set_index(
            "account_number_normalized"
        )
    )


# ============================================================
# RECOMPUTE QUARTER-TO-QUARTER CONTINUITY
# ============================================================

continuity_rows = []


org_groups = list(
    quarterly_reports.groupby(
        "organization_id"
    )
)


for organization_id, org_reports in tqdm(
    org_groups,
    desc="Normalized continuity",
    unit="org",
    dynamic_ncols=True,
):

    org_reports = (
        org_reports
        .sort_values(
            "period_index"
        )
        .reset_index(
            drop=True
        )
    )


    for i in range(
        len(org_reports) - 1
    ):

        prev = org_reports.iloc[i]
        nxt = org_reports.iloc[i + 1]


        # Only consecutive quarters.
        if (
            int(
                nxt[
                    "period_index"
                ]
            )
            -
            int(
                prev[
                    "period_index"
                ]
            )
            != 1
        ):

            continue


        prev_report_id = str(
            prev["report_id"]
        )

        next_report_id = str(
            nxt["report_id"]
        )


        prev_accounts = (
            accounts_by_report.get(
                prev_report_id,
                pd.DataFrame(),
            )
        )

        next_accounts = (
            accounts_by_report.get(
                next_report_id,
                pd.DataFrame(),
            )
        )


        if len(prev_accounts):

            prev_numbers = set(
                prev_accounts.index
            )

        else:

            prev_numbers = set()


        if len(next_accounts):

            next_numbers = set(
                next_accounts.index
            )

        else:

            next_numbers = set()


        account_numbers = (
            prev_numbers
            |
            next_numbers
        )


        for account_number in (
            account_numbers
        ):

            prev_exists = (
                account_number
                in prev_numbers
            )

            next_exists = (
                account_number
                in next_numbers
            )


            prev_end = None
            next_begin = None

            prev_raw = None
            next_raw = None


            if prev_exists:

                prev_row = (
                    prev_accounts.loc[
                        account_number
                    ]
                )

                prev_end = dec(
                    prev_row[
                        "end_period_balance"
                    ]
                )

                prev_raw = prev_row[
                    "account_number_raw"
                ]


            if next_exists:

                next_row = (
                    next_accounts.loc[
                        account_number
                    ]
                )

                next_begin = dec(
                    next_row[
                        "begin_period_balance"
                    ]
                )

                next_raw = next_row[
                    "account_number_raw"
                ]


            # ----------------------------------------------
            # CLASSIFICATION
            # ----------------------------------------------

            if (
                prev_exists
                and next_exists
            ):

                exact = money_equal(
                    prev_end,
                    next_begin,
                )

                if exact is None:

                    status = (
                        "missing_balance_value"
                    )

                elif exact:

                    status = (
                        "continuous"
                    )

                else:

                    status = (
                        "balance_mismatch"
                    )


            elif prev_exists:

                if (
                    prev_end is not None
                    and money_equal(
                        prev_end,
                        Decimal("0"),
                    )
                ):

                    status = (
                        "account_closed_at_zero"
                    )

                else:

                    status = (
                        "account_missing_next_"
                        "with_nonzero_end"
                    )


            else:

                if (
                    next_begin is not None
                    and money_equal(
                        next_begin,
                        Decimal("0"),
                    )
                ):

                    status = (
                        "new_account_with_zero_begin"
                    )

                else:

                    status = (
                        "new_account_with_"
                        "nonzero_begin"
                    )


            if (
                prev_end is not None
                and next_begin is not None
            ):

                delta = (
                    next_begin
                    - prev_end
                )

            else:

                delta = None


            continuity_rows.append(
                {
                    "organization_id":
                        str(
                            organization_id
                        ),

                    "root_party_id":
                        str(
                            prev[
                                "root_party_id"
                            ]
                        ),

                    "account_number_normalized":
                        account_number,

                    "previous_account_number_raw":
                        prev_raw,

                    "next_account_number_raw":
                        next_raw,

                    "raw_spelling_changed":
                        (
                            prev_raw
                            != next_raw
                        )
                        if (
                            prev_raw is not None
                            and next_raw is not None
                        )
                        else None,

                    "previous_report_id":
                        prev_report_id,

                    "previous_year":
                        int(
                            prev[
                                "year"
                            ]
                        ),

                    "previous_quarter":
                        int(
                            prev[
                                "quarter"
                            ]
                        ),

                    "previous_end_balance":
                        float(
                            prev_end
                        )
                        if prev_end is not None
                        else None,

                    "next_report_id":
                        next_report_id,

                    "next_year":
                        int(
                            nxt[
                                "year"
                            ]
                        ),

                    "next_quarter":
                        int(
                            nxt[
                                "quarter"
                            ]
                        ),

                    "next_begin_balance":
                        float(
                            next_begin
                        )
                        if next_begin is not None
                        else None,

                    "balance_delta":
                        float(
                            delta
                        )
                        if delta is not None
                        else None,

                    "continuity_status":
                        status,

                    "is_standard_ua_iban":
                        bool(
                            re.fullmatch(
                                r"UA\d{27}",
                                str(
                                    account_number
                                ),
                            )
                        ),
                }
            )


continuity = pd.DataFrame(
    continuity_rows
)


# ============================================================
# STATE-FUNDING EXPOSURE FROM PREVIOUS AUDIT
# ============================================================

state_path = (
    AUDIT_DIR
    / "party_state_funding_exposure.parquet"
)


if state_path.exists():

    party_state_exposure = (
        pd.read_parquet(
            state_path
        )
    )

    party_state_exposure[
        "root_party_id"
    ] = party_state_exposure[
        "root_party_id"
    ].astype(str)


    continuity = continuity.merge(
        party_state_exposure[
            [
                "root_party_id",
                "party_state_funding_exposure",
                "direct_state_funding_amount",
                "budget_expense_amount",
            ]
        ],
        on="root_party_id",
        how="left",
    )

else:

    print(
        "WARNING: party_state_funding_exposure.parquet "
        "not found."
    )

    continuity[
        "party_state_funding_exposure"
    ] = False

    continuity[
        "direct_state_funding_amount"
    ] = None

    continuity[
        "budget_expense_amount"
    ] = None


# ============================================================
# NAMES
# ============================================================

if name_col is not None:

    continuity[
        "party_name"
    ] = continuity[
        "root_party_id"
    ].map(
        org_names
    )

    continuity[
        "organization_name"
    ] = continuity[
        "organization_id"
    ].map(
        org_names
    )


# ============================================================
# BREAKS
# ============================================================

NORMAL_STATUSES = {
    "continuous",
    "account_closed_at_zero",
    "new_account_with_zero_begin",
}


continuity_breaks = (
    continuity[
        ~continuity[
            "continuity_status"
        ].isin(
            NORMAL_STATUSES
        )
    ]
    .copy()
)


state_funding_breaks = (
    continuity_breaks[
        continuity_breaks[
            "party_state_funding_exposure"
        ] == True
    ]
    .copy()
)


# ============================================================
# SAVE NEW NORMALIZED AUDIT
# ============================================================

normalized_snapshots.to_parquet(
    AUDIT_DIR
    / "selected_account_snapshots_normalized.parquet",
    index=False,
)

spelling_variants.to_parquet(
    AUDIT_DIR
    / "account_number_spelling_variants.parquet",
    index=False,
)

multiple_spellings.to_parquet(
    AUDIT_DIR
    / "account_number_multiple_spellings.parquet",
    index=False,
)

nonstandard_accounts.to_parquet(
    AUDIT_DIR
    / "account_number_nonstandard_values.parquet",
    index=False,
)

normalized_conflicts.to_parquet(
    AUDIT_DIR
    / "selected_account_snapshot_conflicts_normalized.parquet",
    index=False,
)

normalized_math_issues.to_parquet(
    AUDIT_DIR
    / "selected_account_internal_math_issues_normalized.parquet",
    index=False,
)

continuity.to_parquet(
    AUDIT_DIR
    / "selected_account_continuity_normalized.parquet",
    index=False,
)

continuity_breaks.to_parquet(
    AUDIT_DIR
    / "selected_account_continuity_breaks_normalized.parquet",
    index=False,
)

state_funding_breaks.to_parquet(
    AUDIT_DIR
    / "selected_account_continuity_breaks_state_funding_parties_normalized.parquet",
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 72)
print("ACCOUNT NUMBER NORMALIZATION QA")
print("=" * 72)

print(
    "Account snapshots before normalized dedup:",
    len(snapshots)
)

print(
    "After normalized dedup:",
    len(normalized_snapshots)
)

print(
    "Additional duplicates collapsed:",
    len(snapshots)
    - len(normalized_snapshots)
)

print()

print(
    "Unique raw account spellings:",
    snapshots[
        "account_number_raw"
    ].nunique(
        dropna=True
    )
)

print(
    "Unique normalized account IDs:",
    snapshots[
        "account_number_normalized"
    ].nunique(
        dropna=True
    )
)

print(
    "Normalized accounts with >1 raw spelling:",
    len(multiple_spellings)
)

print(
    "Non-standard account identifiers:",
    nonstandard_accounts[
        "account_number_normalized"
    ].nunique(
        dropna=True
    )
)

print(
    "Conflicting snapshots after normalization:",
    len(normalized_conflicts)
)

print(
    "Internal math issues:",
    len(normalized_math_issues)
)


print()
print("=" * 72)
print("NORMALIZED CONTINUITY STATUS")
print("=" * 72)

print(
    continuity[
        "continuity_status"
    ].value_counts()
)


print()
print(
    "Total normalized continuity breaks:",
    len(continuity_breaks)
)

print(
    "Breaks in state-funding party networks:",
    len(state_funding_breaks)
)


print()
print("=" * 72)
print("RAW SPELLING VARIANTS")
print("=" * 72)

display(
    multiple_spellings
    .sort_values(
        [
            "raw_variant_count",
            "report_count",
        ],
        ascending=False,
    )
    .head(50)
)


print()
print("=" * 72)
print("STATE-FUNDING PARTIES — NORMALIZED BREAKS")
print("=" * 72)

display(
    state_funding_breaks[
        [
            col
            for col in [
                "party_name",
                "organization_name",

                "account_number_normalized",
                "previous_account_number_raw",
                "next_account_number_raw",
                "raw_spelling_changed",

                "previous_year",
                "previous_quarter",
                "previous_end_balance",

                "next_year",
                "next_quarter",
                "next_begin_balance",

                "balance_delta",
                "continuity_status",

                "previous_report_id",
                "next_report_id",
            ]
            if col
            in state_funding_breaks.columns
        ]
    ]
    .sort_values(
        "balance_delta",
        key=lambda s: s.abs(),
        ascending=False,
        na_position="last",
    )
    .head(100)
)

Normalized continuity:   0%|                                                                 | 0/8432 [00:00<?…


ACCOUNT NUMBER NORMALIZATION QA
Account snapshots before normalized dedup: 12588
After normalized dedup: 12588
Additional duplicates collapsed: 0

Unique raw account spellings: 1805
Unique normalized account IDs: 1679
Normalized accounts with >1 raw spelling: 116
Non-standard account identifiers: 17
Conflicting snapshots after normalization: 8
Internal math issues: 0

NORMALIZED CONTINUITY STATUS
continuity_status
continuous                               8563
account_closed_at_zero                    603
new_account_with_zero_begin               384
balance_mismatch                           88
new_account_with_nonzero_begin             77
account_missing_next_with_nonzero_end      66
Name: count, dtype: int64

Total normalized continuity breaks: 231
Breaks in state-funding party networks: 60

RAW SPELLING VARIANTS


,account_number_normalized,raw_variant_count,raw_examples,report_count,organization_count
411,UA243005280000026008202335986,3,UA243005280000026008202335986 | UA243005280000...,22,1
939,UA553282090000026000000012585,3,UA553282090000026000000012585 | UA553282090000...,19,1
1157,UA673282090000026040010004725,3,UA 67 328209 0000026040010004725 | UA673282090...,18,1
198,UA113805820000026007010333670,3,UA 11 380582 0000026007010333670\n | UA 11 380...,10,1
246,UA143052990000026001045034884,3,UA143052990000026001045034884\n | UA1430529900...,8,1
846,UA503348510000000026008137708,3,UA503348510000000026008137708 | UA 50334851000...,7,1
1352,UA793052990000026006023307969,3,UA793052990000026006023307969 | UA793052990000...,7,1
1606,UA943223130000026009020030177,3,UA943223130000026009020030177 | \tUA9432231300...,6,1
520,UA303808380000026008700622652,3,UA303808380000026008700622652 | UA3038083800...,5,1
1069,UA623516290000000002600426373,3,UA623516290000000002600426373 | UA62351629 000...,5,1



STATE-FUNDING PARTIES — NORMALIZED BREAKS


,party_name,organization_name,account_number_normalized,previous_account_number_raw,next_account_number_raw,raw_spelling_changed,previous_year,previous_quarter,previous_end_balance,next_year,next_quarter,next_begin_balance,balance_delta,continuity_status,previous_report_id,next_report_id
7669,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Чернігівська обласна організація ПП ""Рідне місто""",UA053052990000026006006302776,UA053052990000026006006302776,UA053052990000026006006302776,False,2024,3,76004.06,2024,4,3284.06,-72720.00,balance_mismatch,48854fd0-a00d-11ef-9197-0fbca96b9f8a,9d7aa980-e618-11ef-9743-8991a53f54af
7711,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Новомосковська міська організація Партії Зелен...,UA373052990000026005050311231,UA373052990000026005050311231,UA373052990000026005050311231,False,2021,1,74175.00,2021,2,11492.00,-62683.00,balance_mismatch,5c70bc00-7d15-11ec-ba2b-a5f020a204de,64ea2fd0-8295-11ec-80e8-cd54133e5d2c
9329,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,Івано-Франківська територіальна організація По...,UA593805820000026004010333684,UA593805820000026004010333684,UA593805820000026004010333684,False,2026,1,36585.46,2026,2,0.00,-36585.46,balance_mismatch,3f67abe2-d9ad-4e29-8301-8c3c2944a604,085c0cfa-899d-4d2b-bf78-5cd344e65eb6
9328,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,Івано-Франківська територіальна організація По...,UA133204780000026003924420903,UA133204780000026003924420903,UA133204780000026003924420903,False,2026,1,0.00,2026,2,36585.46,36585.46,balance_mismatch,3f67abe2-d9ad-4e29-8301-8c3c2944a604,085c0cfa-899d-4d2b-bf78-5cd344e65eb6
8682,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»,"Кам""янець-Подільська міська організація Всеукр...",UA593052990000026005036010222,UA593052990000026005036010222,UA593052990000026005036010222,False,2025,3,0.00,2025,4,8649.79,8649.79,balance_mismatch,a182fe66-ff4c-4f69-9812-596e9ad15035,713b908f-e97d-41f8-9c35-9e53bfb58056
7776,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,UA433375460000026007055029277,UA433375460000026007055029277,UA433375460000026007055029277,False,2021,1,475745.28,2021,2,472857.28,-2888.00,balance_mismatch,05291170-87e9-11ec-87ad-bf13a104ce8a,ede95260-fc92-11ee-92b7-3b3b09128e69
4991,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,ХАРКІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,UA103515330000026000052222703,UA103515330000026000052222703,UA103515330000026000052222703,False,2023,4,1863.88,2024,1,41.36,-1822.52,balance_mismatch,ea23f9c0-ea7a-11ee-8a6c-27bc4724684c,e5e0b740-0d57-11ef-823a-73af3dbabf67
4990,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,ХАРКІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,UA103515330000026000052222703,UA103515330000026000052222703,UA103515330000026000052222703,False,2023,3,41.36,2023,4,1863.88,1822.52,balance_mismatch,6579f330-fe4a-11ee-8a6c-27bc4724684c,ea23f9c0-ea7a-11ee-8a6c-27bc4724684c
1659,ПОЛІТИЧНА ПАРТІЯ «КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА П...,Житомирська обласна організація політичної пар...,UA593052990000026003016402904,UA593052990000026003016402904,UA593052990000026003016402904,False,2024,2,119363.00,2024,3,117563.35,-1799.65,balance_mismatch,6b119230-3f72-11ef-b4cc-1f31c784e768,adfc2580-887c-11ef-ac2d-b9ec46358bc3
9418,ПОЛІТИЧНА ПАРТІЯ «ЗА ОДЕЩИНУ»,ПОЛІТИЧНА ПАРТІЯ «ЗА ОДЕЩИНУ»,UA463282090000026009000019813,UA463282090000026009000019813,UA463282090000026009000019813,False,2025,2,8655.59,2025,3,7405.59,-1250.00,balance_mismatch,0ac17bfe-c08d-4a4b-81a1-de1d4dbe9362,7314c0f8-0028-40de-aee7-220446674e7e


In [69]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import re
import unicodedata

import pandas as pd
from tqdm.auto import tqdm


REPORTS_DIR = Path(
    "data/interim/reports"
)

AUDIT_DIR = Path(
    "data/interim/audit"
)

AUDIT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD
# ============================================================

snapshots = pd.read_parquet(
    AUDIT_DIR
    / "selected_account_snapshots.parquet"
).copy()


selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()


for col in [
    "report_id",
    "organization_id",
    "root_party_id",
]:
    selected[col] = selected[col].astype(str)


for col in [
    "report_id",
    "organization_id",
    "root_party_id",
]:
    snapshots[col] = snapshots[col].astype(str)


# ============================================================
# ORGANIZATION / PARTY NAMES
# ============================================================

organizations = pd.read_parquet(
    "data/processed/normalized_v0_1/"
    "organizations.parquet"
)

NAME_CANDIDATES = [
    "name",
    "organization_name",
    "party_name",
    "full_name",
]

name_col = next(
    (
        col
        for col in NAME_CANDIDATES
        if col in organizations.columns
    ),
    None,
)


if name_col is not None:

    org_names = (
        organizations[
            [
                "organization_id",
                name_col,
            ]
        ]
        .drop_duplicates(
            "organization_id"
        )
        .set_index(
            "organization_id"
        )[name_col]
    )

else:

    org_names = {}


# ============================================================
# MONEY HELPERS
# ============================================================

def dec(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        return Decimal(
            str(value)
        )

    except (
        InvalidOperation,
        ValueError,
        TypeError,
    ):
        return None


def money_equal(
    left,
    right,
    tolerance=Decimal("0.01"),
):

    if (
        left is None
        or right is None
    ):
        return None

    return (
        abs(left - right)
        <= tolerance
    )


# ============================================================
# IBAN VALIDATION
# ============================================================

def is_valid_ua_iban(iban):
    """
    Ukrainian IBAN:
      UA + 27 digits = 29 characters total.

    Also validates standard IBAN MOD-97 checksum.
    """

    if not isinstance(
        iban,
        str,
    ):
        return False


    if not re.fullmatch(
        r"UA\d{27}",
        iban,
    ):
        return False


    # Standard IBAN checksum:
    # move first 4 chars to the end,
    # letters -> A=10 ... Z=35,
    # result mod 97 must equal 1.

    rearranged = (
        iban[4:]
        + iban[:4]
    )

    remainder = 0

    for ch in rearranged:

        if ch.isdigit():

            digits = ch

        else:

            digits = str(
                ord(ch) - 55
            )


        for digit in digits:

            remainder = (
                remainder * 10
                + int(digit)
            ) % 97


    return remainder == 1


# ============================================================
# IBAN NORMALIZATION
# ============================================================

ZERO_WIDTH_CHARS = {
    "\u200b",
    "\u200c",
    "\u200d",
    "\u2060",
    "\ufeff",
}


SAFE_INTERNAL_SEPARATORS = {
    " ",
    "\t",
    "\n",
    "\r",
    "-",
    ".",
    "_",
    "/",
    "\\",
    ":",
}


def unicode_clean(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass


    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = "".join(
        ch
        for ch in text
        if ch not in ZERO_WIDTH_CHARS
    )

    return text.upper()


def conservative_nonstandard_key(text):
    """
    For values where a valid IBAN cannot be proven.

    We still:
      - normalize Unicode/case
      - remove zero-width
      - remove whitespace
      - remove everything before first UA

    But DO NOT delete punctuation inside the remaining string.
    """

    if text is None:
        return None


    text = "".join(
        ch
        for ch in text
        if not ch.isspace()
    )


    pos = text.find("UA")

    if pos >= 0:
        text = text[pos:]


    if not text:
        return None


    return (
        "NONSTANDARD::"
        + text
    )


def extract_valid_iban_candidates(text):
    """
    Look for UA anywhere in the field.

    Everything before UA is irrelevant.

    After UA:
      - collect exactly 27 digits
      - allow whitespace / safe separators
      - stop at unexpected characters

    Candidate is accepted only with valid MOD-97 checksum.
    """

    if text is None:
        return []


    starts = [
        match.start()
        for match in re.finditer(
            "UA",
            text,
        )
    ]


    candidates = []


    for start in starts:

        rest = text[
            start + 2:
        ]

        digits = []

        had_internal_formatting = False

        end_position = None


        for pos, ch in enumerate(rest):

            if (
                "0"
                <= ch
                <= "9"
            ):

                digits.append(ch)

                if len(digits) == 27:

                    end_position = pos
                    break


            elif (
                ch.isspace()
                or ch
                in SAFE_INTERNAL_SEPARATORS
            ):

                had_internal_formatting = True
                continue


            else:

                # Unexpected character inside candidate.
                break


        if len(digits) != 27:
            continue


        iban = (
            "UA"
            + "".join(digits)
        )


        if not is_valid_ua_iban(
            iban
        ):
            continue


        # Record whether there was text before UA.
        prefix_removed = (
            start > 0
        )


        # Check trailing material after the 27th digit.
        trailing = (
            rest[
                end_position + 1:
            ]
            if end_position is not None
            else ""
        )

        trailing_nonformat = "".join(
            ch
            for ch in trailing
            if (
                not ch.isspace()
                and ch
                not in SAFE_INTERNAL_SEPARATORS
            )
        )


        candidates.append(
            {
                "iban":
                    iban,

                "prefix_removed":
                    prefix_removed,

                "internal_formatting_removed":
                    had_internal_formatting,

                "trailing_material":
                    bool(
                        trailing_nonformat
                    ),
            }
        )


    return candidates


def normalize_account_number_v2(value):

    text = unicode_clean(
        value
    )


    if text is None:

        return {
            "canonical_iban":
                None,

            "account_match_key":
                None,

            "iban_checksum_valid":
                False,

            "normalization_status":
                "missing",

            "prefix_removed":
                False,

            "internal_formatting_removed":
                False,

            "trailing_material":
                False,

            "valid_candidate_count":
                0,
        }


    candidates = (
        extract_valid_iban_candidates(
            text
        )
    )


    unique_ibans = sorted(
        set(
            item["iban"]
            for item in candidates
        )
    )


    # --------------------------------------------------------
    # EXACTLY ONE UNIQUE VALID IBAN
    # --------------------------------------------------------

    if len(unique_ibans) == 1:

        iban = unique_ibans[0]

        relevant = [
            item
            for item in candidates
            if item["iban"] == iban
        ]


        prefix_removed = any(
            item[
                "prefix_removed"
            ]
            for item in relevant
        )

        formatting_removed = any(
            item[
                "internal_formatting_removed"
            ]
            for item in relevant
        )

        trailing_material = any(
            item[
                "trailing_material"
            ]
            for item in relevant
        )


        if (
            not prefix_removed
            and not formatting_removed
            and not trailing_material
            and text == iban
        ):

            status = (
                "valid_exact"
            )


        elif prefix_removed:

            status = (
                "valid_after_prefix_removal"
            )


        elif formatting_removed:

            status = (
                "valid_after_formatting_cleanup"
            )


        elif trailing_material:

            status = (
                "valid_with_trailing_material"
            )


        else:

            status = (
                "valid_normalized"
            )


        return {
            "canonical_iban":
                iban,

            "account_match_key":
                iban,

            "iban_checksum_valid":
                True,

            "normalization_status":
                status,

            "prefix_removed":
                prefix_removed,

            "internal_formatting_removed":
                formatting_removed,

            "trailing_material":
                trailing_material,

            "valid_candidate_count":
                1,
        }


    # --------------------------------------------------------
    # MULTIPLE DIFFERENT VALID IBANS
    # --------------------------------------------------------

    if len(unique_ibans) > 1:

        return {
            "canonical_iban":
                None,

            "account_match_key":
                None,

            "iban_checksum_valid":
                False,

            "normalization_status":
                "ambiguous_multiple_valid_ibans",

            "prefix_removed":
                False,

            "internal_formatting_removed":
                False,

            "trailing_material":
                True,

            "valid_candidate_count":
                len(
                    unique_ibans
                ),
        }


    # --------------------------------------------------------
    # NO PROVABLY VALID IBAN
    #
    # Preserve conservative identity.
    # --------------------------------------------------------

    fallback = (
        conservative_nonstandard_key(
            text
        )
    )


    return {
        "canonical_iban":
            None,

        "account_match_key":
            fallback,

        "iban_checksum_valid":
            False,

        "normalization_status":
            "nonstandard_or_invalid_iban",

        "prefix_removed":
            (
                text.find("UA") > 0
            ),

        "internal_formatting_removed":
            False,

        "trailing_material":
            False,

        "valid_candidate_count":
            0,
    }


# ============================================================
# APPLY NORMALIZATION
# ============================================================

snapshots[
    "account_number_raw"
] = snapshots[
    "account_number"
]


normalization_records = (
    snapshots[
        "account_number_raw"
    ]
    .map(
        normalize_account_number_v2
    )
)


normalization_df = pd.DataFrame(
    normalization_records.tolist(),
    index=snapshots.index,
)


for col in (
    normalization_df.columns
):

    snapshots[col] = (
        normalization_df[col]
    )


# ============================================================
# QA — NORMALIZATION METHODS
# ============================================================

normalization_summary = (
    snapshots[
        "normalization_status"
    ]
    .value_counts(
        dropna=False
    )
)


valid_iban_rows = (
    snapshots[
        snapshots[
            "canonical_iban"
        ].notna()
    ]
    .copy()
)


nonstandard_rows = (
    snapshots[
        snapshots[
            "normalization_status"
        ]
        ==
        "nonstandard_or_invalid_iban"
    ]
    .copy()
)


ambiguous_rows = (
    snapshots[
        snapshots[
            "normalization_status"
        ]
        ==
        "ambiguous_multiple_valid_ibans"
    ]
    .copy()
)


# ============================================================
# RAW SPELLING VARIANTS OF VALID IBANS
# ============================================================

iban_spelling_variants = (
    valid_iban_rows
    .groupby(
        "canonical_iban",
        dropna=False,
    )
    .agg(
        raw_variant_count=(
            "account_number_raw",
            "nunique",
        ),

        raw_examples=(
            "account_number_raw",
            lambda s:
                " | ".join(
                    list(
                        dict.fromkeys(
                            str(x)
                            for x in s
                            if pd.notna(x)
                        )
                    )[:10]
                ),
        ),

        report_count=(
            "report_id",
            "nunique",
        ),

        organization_count=(
            "organization_id",
            "nunique",
        ),
    )
    .reset_index()
)


multiple_spellings = (
    iban_spelling_variants[
        iban_spelling_variants[
            "raw_variant_count"
        ] > 1
    ]
    .copy()
)


# ============================================================
# DEDUP BY PROVEN ACCOUNT IDENTITY
# ============================================================

SUBSTANTIVE_FIELDS = [
    "report_id",
    "account_type",
    "account_match_key",
    "account_holder",
    "account_holder_code",
    "begin_period_balance",
    "end_period_balance",
    "report_period_income",
    "report_period_used_funds",
]


normalized_snapshots = (
    snapshots
    .drop_duplicates(
        subset=SUBSTANTIVE_FIELDS
    )
    .copy()
)


# ============================================================
# CONFLICTING FINANCIAL STATES
#
# Same report + same proven/fallback account identity,
# but more than one substantive financial snapshot.
#
# Important:
# selected_account_snapshots.parquet may already contain
# account_variant_count from an earlier audit. Remove those
# derived columns before calculating the new V2 version.
# ============================================================

STALE_VARIANT_COLUMNS = [
    "account_variant_count",
    "account_variant_count_x",
    "account_variant_count_y",
    "normalized_account_variant_count",
]

normalized_snapshots = (
    normalized_snapshots.drop(
        columns=[
            col
            for col in STALE_VARIANT_COLUMNS
            if col in normalized_snapshots.columns
        ],
        errors="ignore",
    )
)


variant_counts = (
    normalized_snapshots[
        normalized_snapshots[
            "account_match_key"
        ].notna()
    ]
    .groupby(
        [
            "report_id",
            "account_match_key",
        ],
        dropna=False,
    )
    .size()
    .rename(
        "account_variant_count"
    )
    .reset_index()
)


normalized_snapshots = (
    normalized_snapshots.merge(
        variant_counts,
        on=[
            "report_id",
            "account_match_key",
        ],
        how="left",
        validate="many_to_one",
    )
)


# Rows without a usable match key cannot have a calculated
# variant count.
normalized_snapshots[
    "account_variant_count"
] = normalized_snapshots[
    "account_variant_count"
].fillna(0)


conflicts = (
    normalized_snapshots[
        normalized_snapshots[
            "account_variant_count"
        ] > 1
    ]
    .copy()
)


print(
    "Conflicting report/account snapshots:",
    len(conflicts)
)

# ============================================================
# INTERNAL ARITHMETIC
# ============================================================

math_ok = []
math_delta = []


for row in (
    normalized_snapshots.itertuples(
        index=False
    )
):

    begin = dec(
        row.begin_period_balance
    )

    end = dec(
        row.end_period_balance
    )

    income = dec(
        row.report_period_income
    )

    used = dec(
        row.report_period_used_funds
    )


    if None in {
        begin,
        end,
        income,
        used,
    }:

        math_ok.append(
            None
        )

        math_delta.append(
            None
        )

        continue


    calculated = (
        begin
        + income
        - used
    )

    delta = (
        end
        - calculated
    )


    math_ok.append(
        money_equal(
            end,
            calculated,
        )
    )

    math_delta.append(
        float(delta)
    )


normalized_snapshots[
    "internal_math_ok_v2"
] = math_ok

normalized_snapshots[
    "internal_math_delta_v2"
] = math_delta


math_issues = (
    normalized_snapshots[
        normalized_snapshots[
            "internal_math_ok_v2"
        ] == False
    ]
    .copy()
)


# ============================================================
# USABLE ACCOUNT SNAPSHOTS
# ============================================================

continuity_source = (
    normalized_snapshots[
        (
            normalized_snapshots[
                "account_match_key"
            ].notna()
        )
        &
        (
            normalized_snapshots[
                "account_variant_count"
            ] == 1
        )
        &
        (
            normalized_snapshots[
                "quarter"
            ].isin(
                [1, 2, 3, 4]
            )
        )
    ]
    .copy()
)


# ============================================================
# REPORT-LEVEL ACCOUNT IDENTITY QA
# ============================================================

report_identity_qa = (
    normalized_snapshots
    .groupby(
        "report_id",
        as_index=False,
    )
    .agg(
        snapshot_rows=(
            "account_match_key",
            "size",
        ),

        unresolved_account_rows=(
            "account_match_key",
            lambda s:
                int(
                    s.isna().sum()
                ),
        ),

        conflicting_account_rows=(
            "account_variant_count",
            lambda s:
                int(
                    (
                        s.fillna(0)
                        > 1
                    ).sum()
                ),
        ),
    )
)


report_identity_qa[
    "account_identity_complete"
] = (
    (
        report_identity_qa[
            "unresolved_account_rows"
        ] == 0
    )
    &
    (
        report_identity_qa[
            "conflicting_account_rows"
        ] == 0
    )
)


# ============================================================
# QUARTERLY REPORT INDEX
# ============================================================

quarterly_reports = (
    selected[
        selected[
            "quarter"
        ].isin(
            [1, 2, 3, 4]
        )
    ][
        [
            "report_id",
            "organization_id",
            "root_party_id",
            "year",
            "quarter",
            "signed_date",
        ]
    ]
    .copy()
)


quarterly_reports[
    "period_index"
] = (
    quarterly_reports[
        "year"
    ] * 4
    +
    quarterly_reports[
        "quarter"
    ]
    - 1
)


# ============================================================
# FAST ACCOUNT INDEX
# ============================================================

accounts_by_report = {}


for report_id, df in (
    continuity_source.groupby(
        "report_id"
    )
):

    accounts_by_report[
        str(report_id)
    ] = (
        df.set_index(
            "account_match_key"
        )
    )


# ============================================================
# ACCOUNT-LEVEL CONTINUITY
# ============================================================

continuity_rows = []


for organization_id, org_reports in tqdm(
    quarterly_reports.groupby(
        "organization_id"
    ),
    total=quarterly_reports[
        "organization_id"
    ].nunique(),
    desc="IBAN-v2 account continuity",
    unit="org",
    dynamic_ncols=True,
):

    org_reports = (
        org_reports
        .sort_values(
            "period_index"
        )
        .reset_index(
            drop=True
        )
    )


    for i in range(
        len(org_reports) - 1
    ):

        prev = org_reports.iloc[i]
        nxt = org_reports.iloc[
            i + 1
        ]


        if (
            int(
                nxt["period_index"]
            )
            -
            int(
                prev["period_index"]
            )
            != 1
        ):

            continue


        prev_id = str(
            prev["report_id"]
        )

        next_id = str(
            nxt["report_id"]
        )


        prev_accounts = (
            accounts_by_report.get(
                prev_id
            )
        )

        next_accounts = (
            accounts_by_report.get(
                next_id
            )
        )


        prev_keys = (
            set(
                prev_accounts.index
            )
            if prev_accounts
            is not None
            else set()
        )

        next_keys = (
            set(
                next_accounts.index
            )
            if next_accounts
            is not None
            else set()
        )


        for key in (
            prev_keys
            | next_keys
        ):

            prev_exists = (
                key in prev_keys
            )

            next_exists = (
                key in next_keys
            )


            prev_end = None
            next_begin = None

            prev_raw = None
            next_raw = None

            canonical_iban = None


            if prev_exists:

                row = (
                    prev_accounts.loc[
                        key
                    ]
                )

                prev_end = dec(
                    row[
                        "end_period_balance"
                    ]
                )

                prev_raw = row[
                    "account_number_raw"
                ]

                canonical_iban = row[
                    "canonical_iban"
                ]


            if next_exists:

                row = (
                    next_accounts.loc[
                        key
                    ]
                )

                next_begin = dec(
                    row[
                        "begin_period_balance"
                    ]
                )

                next_raw = row[
                    "account_number_raw"
                ]

                if canonical_iban is None:

                    canonical_iban = row[
                        "canonical_iban"
                    ]


            # ------------------------------------------------
            # CLASSIFY
            # ------------------------------------------------

            if (
                prev_exists
                and next_exists
            ):

                exact = money_equal(
                    prev_end,
                    next_begin,
                )

                if exact is None:

                    status = (
                        "missing_balance_value"
                    )

                elif exact:

                    status = (
                        "continuous"
                    )

                else:

                    status = (
                        "balance_mismatch"
                    )


            elif prev_exists:

                if (
                    prev_end is not None
                    and money_equal(
                        prev_end,
                        Decimal("0"),
                    )
                ):

                    status = (
                        "account_closed_at_zero"
                    )

                else:

                    status = (
                        "account_missing_next_"
                        "with_nonzero_end"
                    )


            else:

                if (
                    next_begin is not None
                    and money_equal(
                        next_begin,
                        Decimal("0"),
                    )
                ):

                    status = (
                        "new_account_with_zero_begin"
                    )

                else:

                    status = (
                        "new_account_with_"
                        "nonzero_begin"
                    )


            if (
                prev_end is not None
                and next_begin is not None
            ):

                delta = (
                    next_begin
                    - prev_end
                )

            else:

                delta = None


            continuity_rows.append(
                {
                    "organization_id":
                        str(
                            organization_id
                        ),

                    "root_party_id":
                        str(
                            prev[
                                "root_party_id"
                            ]
                        ),

                    "account_match_key":
                        key,

                    "canonical_iban":
                        canonical_iban,

                    "previous_account_number_raw":
                        prev_raw,

                    "next_account_number_raw":
                        next_raw,

                    "previous_report_id":
                        prev_id,

                    "previous_year":
                        int(
                            prev["year"]
                        ),

                    "previous_quarter":
                        int(
                            prev["quarter"]
                        ),

                    "previous_end_balance":
                        float(
                            prev_end
                        )
                        if prev_end is not None
                        else None,

                    "next_report_id":
                        next_id,

                    "next_year":
                        int(
                            nxt["year"]
                        ),

                    "next_quarter":
                        int(
                            nxt["quarter"]
                        ),

                    "next_begin_balance":
                        float(
                            next_begin
                        )
                        if next_begin is not None
                        else None,

                    "balance_delta":
                        float(
                            delta
                        )
                        if delta is not None
                        else None,

                    "continuity_status":
                        status,
                }
            )


account_continuity = pd.DataFrame(
    continuity_rows
)


# ============================================================
# ORGANIZATION-LEVEL BALANCE TOTALS
#
# Important second layer:
#
# Sum of all end balances in Qn
# vs
# sum of all begin balances in Qn+1
#
# This distinguishes:
#
#   account changed / funds moved between accounts
#
# from
#
#   total money continuity actually broke
# ============================================================

usable_for_totals = (
    continuity_source.copy()
)


report_totals = (
    usable_for_totals
    .groupby(
        "report_id",
        as_index=False,
    )
    .agg(
        total_begin_balance=(
            "begin_period_balance",
            "sum",
        ),

        total_end_balance=(
            "end_period_balance",
            "sum",
        ),

        total_report_income=(
            "report_period_income",
            "sum",
        ),

        total_report_used_funds=(
            "report_period_used_funds",
            "sum",
        ),

        usable_account_count=(
            "account_match_key",
            "nunique",
        ),
    )
)


report_totals = (
    quarterly_reports.merge(
        report_totals,
        on="report_id",
        how="left",
    )
)


report_totals = (
    report_totals.merge(
        report_identity_qa[
            [
                "report_id",
                "account_identity_complete",
                "unresolved_account_rows",
                "conflicting_account_rows",
            ]
        ],
        on="report_id",
        how="left",
    )
)


report_totals[
    "usable_account_count"
] = (
    report_totals[
        "usable_account_count"
    ]
    .fillna(0)
    .astype(int)
)


report_totals[
    "account_identity_complete"
] = (
    report_totals[
        "account_identity_complete"
    ]
    .fillna(True)
)


# Empty property_moneys is not itself an identity problem.
# We need to preserve the fact that there are zero accounts.


organization_continuity_rows = []


for organization_id, org_reports in (
    report_totals.groupby(
        "organization_id"
    )
):

    org_reports = (
        org_reports
        .sort_values(
            "period_index"
        )
        .reset_index(
            drop=True
        )
    )


    for i in range(
        len(org_reports) - 1
    ):

        prev = org_reports.iloc[i]
        nxt = org_reports.iloc[
            i + 1
        ]


        if (
            int(
                nxt["period_index"]
            )
            -
            int(
                prev["period_index"]
            )
            != 1
        ):

            continue


        prev_end = dec(
            prev[
                "total_end_balance"
            ]
        )

        next_begin = dec(
            nxt[
                "total_begin_balance"
            ]
        )


        prev_count = int(
            prev[
                "usable_account_count"
            ]
        )

        next_count = int(
            nxt[
                "usable_account_count"
            ]
        )


        prev_complete = bool(
            prev[
                "account_identity_complete"
            ]
        )

        next_complete = bool(
            nxt[
                "account_identity_complete"
            ]
        )


        # pandas sum over no rows is NaN after merge.
        # Conceptually an empty account section = no reported
        # account balance, i.e. zero reported total.
        if prev_count == 0:
            prev_end = Decimal("0")

        if next_count == 0:
            next_begin = Decimal("0")


        if (
            not prev_complete
            or not next_complete
        ):

            status = (
                "account_identity_incomplete"
            )

            delta = None


        else:

            delta = (
                next_begin
                - prev_end
            )


            if money_equal(
                prev_end,
                next_begin,
            ):

                status = (
                    "organization_balance_continuous"
                )

            else:

                status = (
                    "organization_balance_mismatch"
                )


        organization_continuity_rows.append(
            {
                "organization_id":
                    str(
                        organization_id
                    ),

                "root_party_id":
                    str(
                        prev[
                            "root_party_id"
                        ]
                    ),

                "previous_report_id":
                    str(
                        prev[
                            "report_id"
                        ]
                    ),

                "previous_year":
                    int(
                        prev["year"]
                    ),

                "previous_quarter":
                    int(
                        prev["quarter"]
                    ),

                "previous_account_count":
                    prev_count,

                "previous_total_end_balance":
                    float(
                        prev_end
                    )
                    if prev_end is not None
                    else None,

                "next_report_id":
                    str(
                        nxt[
                            "report_id"
                        ]
                    ),

                "next_year":
                    int(
                        nxt["year"]
                    ),

                "next_quarter":
                    int(
                        nxt["quarter"]
                    ),

                "next_account_count":
                    next_count,

                "next_total_begin_balance":
                    float(
                        next_begin
                    )
                    if next_begin is not None
                    else None,

                "organization_balance_delta":
                    float(
                        delta
                    )
                    if delta is not None
                    else None,

                "organization_continuity_status":
                    status,
            }
        )


organization_continuity = pd.DataFrame(
    organization_continuity_rows
)


# ============================================================
# STATE-FUNDING EXPOSURE
# ============================================================

state_path = (
    AUDIT_DIR
    / "party_state_funding_exposure.parquet"
)


if state_path.exists():

    party_state_exposure = (
        pd.read_parquet(
            state_path
        )
    )

    party_state_exposure[
        "root_party_id"
    ] = party_state_exposure[
        "root_party_id"
    ].astype(str)


    state_cols = [
        "root_party_id",
        "party_state_funding_exposure",
        "direct_state_funding_amount",
        "budget_expense_amount",
    ]


    account_continuity = (
        account_continuity.merge(
            party_state_exposure[
                state_cols
            ],
            on="root_party_id",
            how="left",
        )
    )


    organization_continuity = (
        organization_continuity.merge(
            party_state_exposure[
                state_cols
            ],
            on="root_party_id",
            how="left",
        )
    )


else:

    account_continuity[
        "party_state_funding_exposure"
    ] = False

    organization_continuity[
        "party_state_funding_exposure"
    ] = False


# ============================================================
# ADD NAMES
# ============================================================

if name_col is not None:

    for df in [
        account_continuity,
        organization_continuity,
    ]:

        df[
            "party_name"
        ] = df[
            "root_party_id"
        ].map(
            org_names
        )

        df[
            "organization_name"
        ] = df[
            "organization_id"
        ].map(
            org_names
        )


# ============================================================
# ACCOUNT BREAKS
# ============================================================

NORMAL_ACCOUNT_STATUSES = {
    "continuous",
    "account_closed_at_zero",
    "new_account_with_zero_begin",
}


account_breaks = (
    account_continuity[
        ~account_continuity[
            "continuity_status"
        ].isin(
            NORMAL_ACCOUNT_STATUSES
        )
    ]
    .copy()
)


# ============================================================
# ORGANIZATION BREAKS
# ============================================================

organization_breaks = (
    organization_continuity[
        organization_continuity[
            "organization_continuity_status"
        ]
        ==
        "organization_balance_mismatch"
    ]
    .copy()
)


state_organization_breaks = (
    organization_breaks[
        organization_breaks[
            "party_state_funding_exposure"
        ] == True
    ]
    .copy()
)


# ============================================================
# SAVE V2
# ============================================================

snapshots.to_parquet(
    AUDIT_DIR
    / "selected_account_snapshots_iban_v2.parquet",
    index=False,
)

multiple_spellings.to_parquet(
    AUDIT_DIR
    / "account_number_multiple_spellings_iban_v2.parquet",
    index=False,
)

nonstandard_rows.to_parquet(
    AUDIT_DIR
    / "account_number_nonstandard_iban_v2.parquet",
    index=False,
)

ambiguous_rows.to_parquet(
    AUDIT_DIR
    / "account_number_ambiguous_iban_v2.parquet",
    index=False,
)

conflicts.to_parquet(
    AUDIT_DIR
    / "selected_account_conflicts_iban_v2.parquet",
    index=False,
)

math_issues.to_parquet(
    AUDIT_DIR
    / "selected_account_internal_math_issues_iban_v2.parquet",
    index=False,
)

account_continuity.to_parquet(
    AUDIT_DIR
    / "selected_account_continuity_iban_v2.parquet",
    index=False,
)

account_breaks.to_parquet(
    AUDIT_DIR
    / "selected_account_continuity_breaks_iban_v2.parquet",
    index=False,
)

organization_continuity.to_parquet(
    AUDIT_DIR
    / "selected_organization_balance_continuity_iban_v2.parquet",
    index=False,
)

organization_breaks.to_parquet(
    AUDIT_DIR
    / "selected_organization_balance_breaks_iban_v2.parquet",
    index=False,
)

state_organization_breaks.to_parquet(
    AUDIT_DIR
    / "selected_organization_balance_breaks_state_funding_iban_v2.parquet",
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 76)
print("IBAN V2 NORMALIZATION")
print("=" * 76)

print(
    normalization_summary
)

print()

print(
    "Raw account spellings:",
    snapshots[
        "account_number_raw"
    ].nunique(
        dropna=True
    )
)

print(
    "Valid canonical IBANs:",
    snapshots[
        "canonical_iban"
    ].nunique(
        dropna=True
    )
)

print(
    "Valid IBANs with >1 raw spelling:",
    len(
        multiple_spellings
    )
)

print(
    "Non-standard / checksum-invalid rows:",
    len(
        nonstandard_rows
    )
)

print(
    "Ambiguous multi-IBAN rows:",
    len(
        ambiguous_rows
    )
)

print(
    "Conflicting financial snapshots:",
    len(
        conflicts
    )
)

print(
    "Internal math issues:",
    len(
        math_issues
    )
)


print()
print("=" * 76)
print("ACCOUNT-LEVEL CONTINUITY")
print("=" * 76)

print(
    account_continuity[
        "continuity_status"
    ].value_counts()
)

print()

print(
    "Account-level break candidates:",
    len(
        account_breaks
    )
)


print()
print("=" * 76)
print("ORGANIZATION-LEVEL BALANCE CONTINUITY")
print("=" * 76)

print(
    organization_continuity[
        "organization_continuity_status"
    ].value_counts()
)

print()

print(
    "Organization-level monetary breaks:",
    len(
        organization_breaks
    )
)

print(
    "Organization-level breaks in state-funding party networks:",
    len(
        state_organization_breaks
    )
)


print()
print("=" * 76)
print("PREFIX / FORMAT CLEANUPS — EXAMPLES")
print("=" * 76)

display(
    snapshots[
        snapshots[
            "normalization_status"
        ].isin(
            [
                "valid_after_prefix_removal",
                "valid_after_formatting_cleanup",
                "valid_with_trailing_material",
            ]
        )
    ][
        [
            "account_number_raw",
            "canonical_iban",
            "normalization_status",
            "organization_id",
            "report_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
    .head(100)
)


print()
print("=" * 76)
print("NON-STANDARD / INVALID — REVIEW")
print("=" * 76)

display(
    nonstandard_rows[
        [
            "account_number_raw",
            "account_match_key",
            "organization_id",
            "report_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
    .head(100)
)


print()
print("=" * 76)
print("STATE-FUNDING NETWORKS — ORGANIZATION BALANCE BREAKS")
print("=" * 76)

display(
    state_organization_breaks[
        [
            col
            for col in [
                "party_name",
                "organization_name",

                "previous_year",
                "previous_quarter",
                "previous_account_count",
                "previous_total_end_balance",

                "next_year",
                "next_quarter",
                "next_account_count",
                "next_total_begin_balance",

                "organization_balance_delta",

                "previous_report_id",
                "next_report_id",
            ]
            if col
            in state_organization_breaks.columns
        ]
    ]
    .sort_values(
        "organization_balance_delta",
        key=lambda s:
            s.abs(),
        ascending=False,
        na_position="last",
    )
    .head(100)
)

Conflicting report/account snapshots: 8


IBAN-v2 account continuity:   0%|                                                            | 0/8432 [00:00<?…

C:\Users\Igor\AppData\Local\Temp\ipykernel_30228\749337584.py:1609: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(True)



IBAN V2 NORMALIZATION
normalization_status
valid_exact                       12057
valid_after_formatting_cleanup      287
valid_normalized                    135
valid_after_prefix_removal          109
Name: count, dtype: int64

Raw account spellings: 1805
Valid canonical IBANs: 1664
Valid IBANs with >1 raw spelling: 128
Non-standard / checksum-invalid rows: 0
Ambiguous multi-IBAN rows: 0
Conflicting financial snapshots: 8
Internal math issues: 0

ACCOUNT-LEVEL CONTINUITY
continuity_status
continuous                               8583
account_closed_at_zero                    590
new_account_with_zero_begin               371
balance_mismatch                           88
new_account_with_nonzero_begin             70
account_missing_next_with_nonzero_end      59
Name: count, dtype: int64

Account-level break candidates: 217

ORGANIZATION-LEVEL BALANCE CONTINUITY
organization_continuity_status
organization_balance_continuous    57204
organization_balance_mismatch        163
account_iden

,account_number_raw,canonical_iban,normalization_status,organization_id,report_id,year,quarter
147,UA72305299000002903605020\n4734,UA723052990000029036050204734,valid_after_formatting_cleanup,04e1d8c5-e95e-4232-a175-726dacb56a1d,a6dfa4d0-1f98-11f0-a300-eb47575da91a,2025,1
160,UA72305299000002903605020\n4734,UA723052990000029036050204734,valid_after_formatting_cleanup,04e1d8c5-e95e-4232-a175-726dacb56a1d,bb7a3920-de64-11ef-bde1-41cc07d44e1f,2024,4
544,UA073052990000026009011802150,UA073052990000026009011802150,valid_after_prefix_removal,0c8c7235-2171-4877-832b-6d6c1843d9b7,92a7abc4-31d7-4fb4-bde5-522ebb5d0e51,2026,2
549,UA073052990000026009011802150,UA073052990000026009011802150,valid_after_prefix_removal,0c8c7235-2171-4877-832b-6d6c1843d9b7,16f0d5ed-e07e-4758-b7f5-9335a1dc6256,2026,1
712,UA883805820000026003010318\n161,UA883805820000026003010318161,valid_after_formatting_cleanup,0eba1f24-199e-4882-9a6c-5e2ff071284f,097d7a40-e9fd-11ee-96f1-37a2ca81244c,2022,4
809,:UA493052990000026044050219770,UA493052990000026044050219770,valid_after_prefix_removal,0f3812ac-d6b0-4496-8b73-cc9d560adb6c,dc526030-5577-11ef-9197-0fbca96b9f8a,2024,2
816,№UA303052990000026002043302142,UA303052990000026002043302142,valid_after_prefix_removal,0f4ea977-251d-4c2e-b8bc-bd095bf67650,f4cf26a0-2cd8-11f0-8f85-d95649015426,2025,1
859,UA 703052990000026008021205685,UA703052990000026008021205685,valid_after_formatting_cleanup,10149245-06a0-4705-af87-f89dcb120779,023fa9a0-87e2-11ef-b4cc-1f31c784e768,2024,3
860,UA 703052990000026008021205685,UA703052990000026008021205685,valid_after_formatting_cleanup,10149245-06a0-4705-af87-f89dcb120779,78c985e0-0c73-11ef-823a-73af3dbabf67,2024,1
861,UA 703052990000026008021205685,UA703052990000026008021205685,valid_after_formatting_cleanup,10149245-06a0-4705-af87-f89dcb120779,71851ae0-da42-11ef-9743-8991a53f54af,2024,4



NON-STANDARD / INVALID — REVIEW


,account_number_raw,account_match_key,organization_id,report_id,year,quarter



STATE-FUNDING NETWORKS — ORGANIZATION BALANCE BREAKS


,party_name,organization_name,previous_year,previous_quarter,previous_account_count,previous_total_end_balance,next_year,next_quarter,next_account_count,next_total_begin_balance,organization_balance_delta,previous_report_id,next_report_id
46803,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Чернігівська обласна організація ПП ""Рідне місто""",2024,3,1,76004.06,2024,4,1,3284.06,-72720.00,48854fd0-a00d-11ef-9197-0fbca96b9f8a,9d7aa980-e618-11ef-9743-8991a53f54af
47043,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Новомосковська міська організація Партії Зелен...,2021,1,1,74175.00,2021,2,1,11492.00,-62683.00,5c70bc00-7d15-11ec-ba2b-a5f020a204de,64ea2fd0-8295-11ec-80e8-cd54133e5d2c
20879,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,2,31164.96,2025,2,0,0.00,-31164.96,d861fd70-2ca2-11f0-b8ea-758045529e07,764a691c-3237-4044-9430-44313fbf739f
44285,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,2026,1,2,28554.16,2026,2,2,793.00,-27761.16,5b761b27-5a83-4b64-a34f-b3ae18edbca7,04e257ab-b312-4de7-833a-f75b88623d19
56629,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,2025,2,1,0.00,2025,3,1,14812.94,14812.94,fa433233-c5b4-4e47-8e1e-bd6787b70abe,3f5bd744-4c7f-4ae6-9034-c2ad2e8f9b9d
51917,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»,"Кам""янець-Подільська міська організація Всеукр...",2025,3,1,0.00,2025,4,1,8649.79,8649.79,a182fe66-ff4c-4f69-9812-596e9ad15035,713b908f-e97d-41f8-9c35-9e53bfb58056
20880,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,2,0,0.00,2025,3,2,3167.75,3167.75,764a691c-3237-4044-9430-44313fbf739f,f5a60c98-bd12-4795-9aef-fb0d9def5471
47846,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,2021,1,1,475745.28,2021,2,1,472857.28,-2888.00,05291170-87e9-11ec-87ad-bf13a104ce8a,ede95260-fc92-11ee-92b7-3b3b09128e69
9345,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ЛЬВІВСЬКА ОБЛАСНА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПО...,2024,2,2,2043.82,2024,3,1,0.00,-2043.82,a9ad4470-53dc-11ef-b4cc-1f31c784e768,6082acb0-9b6a-11ef-9197-0fbca96b9f8a
30905,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧ...,ХАРКІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,2023,3,1,41.36,2023,4,1,1863.88,1822.52,6579f330-fe4a-11ee-8a6c-27bc4724684c,ea23f9c0-ea7a-11ee-8a6c-27bc4724684c


In [70]:
from pathlib import Path
from decimal import Decimal
import json

import pandas as pd


AUDIT_DIR = Path(
    "data/interim/audit"
)

REPORTS_DIR = Path(
    "data/interim/reports"
)


# ============================================================
# LOAD CLEAN ORGANIZATION-LEVEL BREAKS
# ============================================================

breaks = pd.read_parquet(
    AUDIT_DIR
    / "selected_organization_balance_breaks_iban_v2.parquet"
).copy()


selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()


for col in [
    "organization_id",
    "root_party_id",
    "previous_report_id",
    "next_report_id",
]:
    if col in breaks.columns:
        breaks[col] = breaks[col].astype(str)


for col in [
    "report_id",
    "organization_id",
    "root_party_id",
]:
    selected[col] = selected[col].astype(str)


# ============================================================
# PERIOD INDEX
# ============================================================

breaks[
    "previous_period_index"
] = (
    breaks[
        "previous_year"
    ] * 4
    +
    breaks[
        "previous_quarter"
    ]
    - 1
)

breaks[
    "next_period_index"
] = (
    breaks[
        "next_year"
    ] * 4
    +
    breaks[
        "next_quarter"
    ]
    - 1
)


breaks = (
    breaks
    .sort_values(
        [
            "organization_id",
            "previous_period_index",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# BUILD CONNECTED BREAK EPISODES
#
# Example:
#
# Q3 -> Q4   break
# Q4 -> Q1   break
#
# These share Q4, so they are ONE episode.
#
# If:
#
# Q1 -> Q2 break
# Q2 -> Q3 continuous
# Q3 -> Q4 break
#
# these are TWO separate episodes.
# ============================================================

episode_ids = []

episode_counter = 0

last_org = None
last_next_report_id = None


for row in breaks.itertuples(
    index=False
):

    current_org = str(
        row.organization_id
    )

    current_previous = str(
        row.previous_report_id
    )


    same_episode = (
        current_org == last_org
        and
        current_previous
        == last_next_report_id
    )


    if not same_episode:

        episode_counter += 1


    episode_ids.append(
        episode_counter
    )


    last_org = current_org

    last_next_report_id = str(
        row.next_report_id
    )


breaks[
    "anomaly_episode_id"
] = episode_ids


# ============================================================
# BUILD REPORT NODE TABLE
#
# We want one record for every report participating
# in a break episode.
# ============================================================

node_rows = []


for row in breaks.itertuples(
    index=False
):

    # previous report
    node_rows.append(
        {
            "anomaly_episode_id":
                row.anomaly_episode_id,

            "organization_id":
                row.organization_id,

            "root_party_id":
                row.root_party_id,

            "report_id":
                row.previous_report_id,

            "year":
                int(
                    row.previous_year
                ),

            "quarter":
                int(
                    row.previous_quarter
                ),

            "reported_account_count":
                int(
                    row.previous_account_count
                ),

            "reported_total_balance":
                float(
                    row.previous_total_end_balance
                ),

            "balance_role":
                "end",
        }
    )


    # next report
    node_rows.append(
        {
            "anomaly_episode_id":
                row.anomaly_episode_id,

            "organization_id":
                row.organization_id,

            "root_party_id":
                row.root_party_id,

            "report_id":
                row.next_report_id,

            "year":
                int(
                    row.next_year
                ),

            "quarter":
                int(
                    row.next_quarter
                ),

            "reported_account_count":
                int(
                    row.next_account_count
                ),

            "reported_total_balance":
                float(
                    row.next_total_begin_balance
                ),

            "balance_role":
                "begin",
        }
    )


nodes = pd.DataFrame(
    node_rows
)


# Same report may occur twice inside one episode:
#
# previous edge -> as "next"
# next edge     -> as "previous"
#
# Collapse to one report node, while keeping both observations.
# ============================================================

report_nodes = (
    nodes
    .groupby(
        [
            "anomaly_episode_id",
            "organization_id",
            "root_party_id",
            "report_id",
            "year",
            "quarter",
        ],
        as_index=False,
    )
    .agg(
        reported_account_count_min=(
            "reported_account_count",
            "min",
        ),

        reported_account_count_max=(
            "reported_account_count",
            "max",
        ),

        observed_balance_min=(
            "reported_total_balance",
            "min",
        ),

        observed_balance_max=(
            "reported_total_balance",
            "max",
        ),

        balance_roles=(
            "balance_role",
            lambda s:
                ",".join(
                    sorted(
                        set(s)
                    )
                ),
        ),
    )
)


report_nodes[
    "is_empty_financial_report"
] = (
    report_nodes[
        "reported_account_count_max"
    ] == 0
)


# ============================================================
# ADD SIGNED / REPORT METADATA
# ============================================================

selected_meta_cols = [
    col
    for col in [
        "report_id",
        "signed_date",
        "created_date",
        "is_party_office",
        "entity_type",
    ]
    if col in selected.columns
]


report_nodes = report_nodes.merge(
    selected[
        selected_meta_cols
    ].drop_duplicates(
        "report_id"
    ),
    on="report_id",
    how="left",
)


# ============================================================
# EPISODE CLASSIFICATION
# ============================================================

def money_close(
    value,
    target=0,
    tolerance=0.01,
):

    if value is None:
        return False

    if pd.isna(value):
        return False

    return (
        abs(
            float(value)
            - float(target)
        )
        <= tolerance
    )


episode_rows = []


for episode_id, df in (
    breaks.groupby(
        "anomaly_episode_id"
    )
):

    df = (
        df.sort_values(
            "previous_period_index"
        )
        .copy()
    )


    episode_nodes = (
        report_nodes[
            report_nodes[
                "anomaly_episode_id"
            ]
            == episode_id
        ]
        .sort_values(
            [
                "year",
                "quarter",
            ]
        )
        .copy()
    )


    deltas = (
        df[
            "organization_balance_delta"
        ]
        .astype(float)
        .tolist()
    )


    break_count = len(df)

    report_count = (
        episode_nodes[
            "report_id"
        ].nunique()
    )

    empty_reports = (
        episode_nodes[
            episode_nodes[
                "is_empty_financial_report"
            ]
        ]
    )

    empty_report_count = (
        empty_reports[
            "report_id"
        ].nunique()
    )


    # --------------------------------------------------------
    # BASIC SHAPE
    # --------------------------------------------------------

    if break_count == 1:

        episode_shape = (
            "single_transition_break"
        )


    elif (
        break_count == 2
        and money_close(
            deltas[0]
            + deltas[1]
        )
    ):

        episode_shape = (
            "two_sided_exact_reversal"
        )


    elif break_count == 2:

        episode_shape = (
            "two_sided_nonreversing"
        )


    else:

        episode_shape = (
            "multi_transition_episode"
        )


    # --------------------------------------------------------
    # SPECIAL FLAGS
    # --------------------------------------------------------

    has_empty_report = (
        empty_report_count > 0
    )


    # Strong candidate resembling Luhansk:
    #
    # non-empty → empty → non-empty,
    # with two adjacent breaks around the empty report.
    #
    missing_financial_section_candidate = False


    if (
        break_count == 2
        and report_count == 3
        and empty_report_count == 1
    ):

        ordered = (
            episode_nodes
            .sort_values(
                [
                    "year",
                    "quarter",
                ]
            )
            .reset_index(
                drop=True
            )
        )


        if len(ordered) == 3:

            middle_empty = bool(
                ordered.loc[
                    1,
                    "is_empty_financial_report"
                ]
            )

            outer_nonempty = (
                not bool(
                    ordered.loc[
                        0,
                        "is_empty_financial_report"
                    ]
                )
                and
                not bool(
                    ordered.loc[
                        2,
                        "is_empty_financial_report"
                    ]
                )
            )


            missing_financial_section_candidate = (
                middle_empty
                and outer_nonempty
            )


    # --------------------------------------------------------
    # PRIORITY
    # --------------------------------------------------------

    max_abs_delta = max(
        abs(x)
        for x in deltas
    )


    if missing_financial_section_candidate:

        audit_priority = (
            "critical_candidate"
        )


    elif (
        bool(
            df[
                "party_state_funding_exposure"
            ].fillna(False).any()
        )
        and max_abs_delta >= 10000
    ):

        audit_priority = (
            "high"
        )


    elif max_abs_delta >= 10000:

        audit_priority = (
            "medium_high"
        )


    elif max_abs_delta >= 1000:

        audit_priority = (
            "medium"
        )


    else:

        audit_priority = (
            "low"
        )


    # --------------------------------------------------------
    # SERIALIZE REPORT IDS FOR EASY REVIEW
    # --------------------------------------------------------

    report_ids = (
        episode_nodes[
            "report_id"
        ]
        .astype(str)
        .tolist()
    )


    empty_report_ids = (
        empty_reports[
            "report_id"
        ]
        .astype(str)
        .tolist()
    )


    first = df.iloc[0]
    last = df.iloc[-1]


    episode_rows.append(
        {
            "anomaly_episode_id":
                int(
                    episode_id
                ),

            "organization_id":
                str(
                    first[
                        "organization_id"
                    ]
                ),

            "root_party_id":
                str(
                    first[
                        "root_party_id"
                    ]
                ),

            "party_name":
                first.get(
                    "party_name"
                ),

            "organization_name":
                first.get(
                    "organization_name"
                ),

            "start_year":
                int(
                    first[
                        "previous_year"
                    ]
                ),

            "start_quarter":
                int(
                    first[
                        "previous_quarter"
                    ]
                ),

            "end_year":
                int(
                    last[
                        "next_year"
                    ]
                ),

            "end_quarter":
                int(
                    last[
                        "next_quarter"
                    ]
                ),

            "break_count":
                break_count,

            "report_count":
                report_count,

            "episode_shape":
                episode_shape,

            "has_empty_financial_report":
                has_empty_report,

            "empty_report_count":
                empty_report_count,

            "missing_financial_section_candidate":
                missing_financial_section_candidate,

            "max_abs_balance_delta":
                max_abs_delta,

            "sum_transition_deltas":
                sum(
                    deltas
                ),

            "party_state_funding_exposure":
                bool(
                    df[
                        "party_state_funding_exposure"
                    ]
                    .fillna(False)
                    .any()
                ),

            "audit_priority":
                audit_priority,

            "report_ids_json":
                json.dumps(
                    report_ids,
                    ensure_ascii=False,
                ),

            "empty_report_ids_json":
                json.dumps(
                    empty_report_ids,
                    ensure_ascii=False,
                ),
        }
    )


episodes = pd.DataFrame(
    episode_rows
)


# ============================================================
# SAVE
# ============================================================

breaks.to_parquet(
    AUDIT_DIR
    / "organization_balance_breaks_with_episode_id.parquet",
    index=False,
)

report_nodes.to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_episode_reports.parquet",
    index=False,
)

episodes.to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_episodes.parquet",
    index=False,
)


# Easy CSV review too.
episodes.to_csv(
    AUDIT_DIR
    / "organization_balance_anomaly_episodes.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# SUMMARY
# ============================================================

print(
    "Broken transitions:",
    len(breaks)
)

print(
    "Distinct anomaly episodes:",
    len(episodes)
)

print()


print(
    "EPISODE SHAPES"
)

print(
    episodes[
        "episode_shape"
    ].value_counts()
)

print()


print(
    "AUDIT PRIORITY"
)

print(
    episodes[
        "audit_priority"
    ].value_counts()
)

print()


print(
    "Episodes in state-funding party networks:",
    int(
        episodes[
            "party_state_funding_exposure"
        ].sum()
    )
)

print(
    "Missing-financial-section candidates:",
    int(
        episodes[
            "missing_financial_section_candidate"
        ].sum()
    )
)


# ============================================================
# TOP PRIORITY EPISODES
# ============================================================

priority_rank = {
    "critical_candidate": 0,
    "high": 1,
    "medium_high": 2,
    "medium": 3,
    "low": 4,
}


episodes[
    "_priority_rank"
] = episodes[
    "audit_priority"
].map(
    priority_rank
)


display(
    episodes
    .sort_values(
        [
            "_priority_rank",
            "max_abs_balance_delta",
        ],
        ascending=[
            True,
            False,
        ],
    )
    [
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "start_year",
            "start_quarter",
            "end_year",
            "end_quarter",

            "break_count",
            "report_count",

            "episode_shape",

            "has_empty_financial_report",
            "missing_financial_section_candidate",

            "max_abs_balance_delta",
            "sum_transition_deltas",

            "party_state_funding_exposure",
            "audit_priority",

            "report_ids_json",
            "empty_report_ids_json",
        ]
    ]
    .head(100)
)

Broken transitions: 163
Distinct anomaly episodes: 131

EPISODE SHAPES
episode_shape
single_transition_break     106
two_sided_exact_reversal     13
two_sided_nonreversing        6
multi_transition_episode      6
Name: count, dtype: int64

AUDIT PRIORITY
audit_priority
low                   76
medium                37
medium_high           11
high                   4
critical_candidate     3
Name: count, dtype: int64

Episodes in state-funding party networks: 26
Missing-financial-section candidates: 3


,anomaly_episode_id,party_name,organization_name,start_year,start_quarter,end_year,end_quarter,break_count,report_count,episode_shape,has_empty_financial_report,missing_financial_section_candidate,max_abs_balance_delta,sum_transition_deltas,party_state_funding_exposure,audit_priority,report_ids_json,empty_report_ids_json
47,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,2025,3,2,3,two_sided_nonreversing,True,True,31164.96,-27997.21,True,critical_candidate,"[""d861fd70-2ca2-11f0-b8ea-758045529e07"", ""764a...","[""764a691c-3237-4044-9430-44313fbf739f""]"
83,84,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,2025,4,2026,2,2,3,two_sided_nonreversing,True,True,14535.58,-182.00,False,critical_candidate,"[""1afcb219-2cfd-4cc2-92d2-af47a49d7ab5"", ""0404...","[""04040916-0d94-46af-a0f6-6194262b612b""]"
39,40,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,Полтавська крайова організація Народного Руху ...,2025,1,2025,3,2,3,two_sided_exact_reversal,True,True,117.83,0.00,False,critical_candidate,"[""9cd3b8f0-2cd6-11f0-8f85-d95649015426"", ""8e30...","[""8e30a6c9-4cc5-41ba-909d-935b4ce0f204""]"
104,105,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Чернігівська обласна організація ПП ""Рідне місто""",2024,3,2024,4,1,2,single_transition_break,False,False,72720.00,-72720.00,True,high,"[""48854fd0-a00d-11ef-9197-0fbca96b9f8a"", ""9d7a...",[]
105,106,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Новомосковська міська організація Партії Зелен...,2021,1,2021,2,1,2,single_transition_break,False,False,62683.00,-62683.00,True,high,"[""5c70bc00-7d15-11ec-ba2b-a5f020a204de"", ""64ea...",[]
100,101,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,2026,1,2026,2,1,2,single_transition_break,False,False,27761.16,-27761.16,True,high,"[""5b761b27-5a83-4b64-a34f-b3ae18edbca7"", ""04e2...",[]
128,129,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,2025,2,2025,3,1,2,single_transition_break,False,False,14812.94,14812.94,True,high,"[""fa433233-c5b4-4e47-8e1e-bd6787b70abe"", ""3f5b...",[]
29,30,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,ЧЕРНІГІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПА...,2024,4,2025,1,1,2,single_transition_break,False,False,585171.00,-585171.00,False,medium_high,"[""90fb5f50-e060-11ef-bde1-41cc07d44e1f"", ""0b3d...",[]
101,102,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,2024,4,2025,1,1,2,single_transition_break,True,False,366538.98,-366538.98,False,medium_high,"[""9d9f4dc0-e060-11ef-a9cb-c3967e5ec0ca"", ""df02...","[""df023360-29dd-11f0-8c75-21cbf53902ff""]"
23,24,"ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""","ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""",2024,2,2024,3,1,2,single_transition_break,True,False,315740.00,315740.00,False,medium_high,"[""d84b7780-5633-11ef-9f3c-219e709335ad"", ""c110...","[""d84b7780-5633-11ef-9f3c-219e709335ad""]"


In [71]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import json
import re
import unicodedata

import pandas as pd
from tqdm.auto import tqdm


RAW_DIR = Path(
    "data/raw/report_details"
)

REPORTS_DIR = Path(
    "data/interim/reports"
)

AUDIT_DIR = Path(
    "data/interim/audit"
)


# ============================================================
# LOAD
# ============================================================

episodes = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_episodes.parquet"
)

episode_reports = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_episode_reports.parquet"
)

all_reports = pd.read_parquet(
    REPORTS_DIR
    / "all_reports_manifest.parquet"
).copy()

selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()


for df in [
    all_reports,
    selected,
    episode_reports,
]:

    for col in [
        "report_id",
        "organization_id",
        "root_party_id",
    ]:

        if col in df.columns:
            df[col] = df[col].astype(str)


# ============================================================
# HELPERS
# ============================================================

def dec(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        return Decimal(str(value))

    except (
        InvalidOperation,
        TypeError,
        ValueError,
    ):
        return None


def close_money(
    left,
    right,
    tolerance=Decimal("0.01"),
):

    if left is None or right is None:
        return None

    return (
        abs(left - right)
        <= tolerance
    )


def valid_ua_iban(iban):

    if not isinstance(iban, str):
        return False

    if not re.fullmatch(
        r"UA\d{27}",
        iban,
    ):
        return False

    rearranged = (
        iban[4:]
        + iban[:4]
    )

    remainder = 0

    for ch in rearranged:

        if ch.isdigit():
            digits = ch
        else:
            digits = str(
                ord(ch) - 55
            )

        for digit in digits:

            remainder = (
                remainder * 10
                + int(digit)
            ) % 97

    return remainder == 1


def account_key(value):
    """
    Conservative account identity for deduplication.

    If exactly one valid Ukrainian IBAN can be recovered,
    use it.

    Otherwise preserve cleaned RAW identity.
    """

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass


    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).upper()


    text = (
        text
        .replace("\u200b", "")
        .replace("\u200c", "")
        .replace("\u200d", "")
        .replace("\u2060", "")
        .replace("\ufeff", "")
    )


    starts = [
        m.start()
        for m in re.finditer(
            "UA",
            text,
        )
    ]

    candidates = set()


    for start in starts:

        digits = []

        for ch in text[
            start + 2:
        ]:

            if ch.isdigit():

                digits.append(ch)

                if len(digits) == 27:
                    break

            elif (
                ch.isspace()
                or ch in "-._/\\:"
            ):
                continue

            else:
                break


        if len(digits) == 27:

            candidate = (
                "UA"
                + "".join(digits)
            )

            if valid_ua_iban(
                candidate
            ):
                candidates.add(
                    candidate
                )


    if len(candidates) == 1:
        return next(
            iter(candidates)
        )


    # Conservative fallback.
    text = "".join(
        ch
        for ch in text
        if not ch.isspace()
    )

    pos = text.find("UA")

    if pos >= 0:
        text = text[pos:]

    if not text:
        return None

    return (
        "NONSTANDARD::"
        + text
    )


# ============================================================
# EXTRACT REPORT FINANCIAL SNAPSHOT
# ============================================================

snapshot_cache = {}


def get_report_snapshot(report_id):

    report_id = str(
        report_id
    )

    if report_id in snapshot_cache:
        return snapshot_cache[
            report_id
        ]


    path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    if not path.exists():

        result = {
            "raw_available":
                False,

            "account_count":
                None,

            "total_begin":
                None,

            "total_end":
                None,

            "total_income":
                None,

            "total_used":
                None,

            "internal_math_ok":
                None,

            "account_conflict":
                None,
        }

        snapshot_cache[
            report_id
        ] = result

        return result


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = (
            json.load(f)[
                "results"
            ]
        )


    rows = (
        (
            detail.get(
                "properties"
            )
            or {}
        )
        .get(
            "property_moneys"
        )
        or []
    )


    cleaned = []


    for row in rows:

        cleaned.append(
            {
                "account_key":
                    account_key(
                        row.get(
                            "account_number"
                        )
                    ),

                "account_type":
                    row.get(
                        "account_type"
                    ),

                "account_holder":
                    row.get(
                        "account_holder"
                    ),

                "account_holder_code":
                    row.get(
                        "account_holder_code"
                    ),

                "begin":
                    dec(
                        row.get(
                            "begin_period_balance"
                        )
                    ),

                "end":
                    dec(
                        row.get(
                            "end_period_balance"
                        )
                    ),

                "income":
                    dec(
                        row.get(
                            "report_period_income"
                        )
                    ),

                "used":
                    dec(
                        row.get(
                            "report_period_used_funds"
                        )
                    ),
            }
        )


    if not cleaned:

        result = {
            "raw_available":
                True,

            "account_count":
                0,

            "total_begin":
                Decimal("0"),

            "total_end":
                Decimal("0"),

            "total_income":
                Decimal("0"),

            "total_used":
                Decimal("0"),

            "internal_math_ok":
                True,

            "account_conflict":
                False,
        }

        snapshot_cache[
            report_id
        ] = result

        return result


    df = pd.DataFrame(
        cleaned
    )


    # ----------------------------------------------
    # Exact substantive technical duplicates
    # ----------------------------------------------

    dedup_cols = [
        "account_key",
        "account_type",
        "account_holder",
        "account_holder_code",
        "begin",
        "end",
        "income",
        "used",
    ]


    df = df.drop_duplicates(
        subset=dedup_cols
    )


    # ----------------------------------------------
    # Same account but different financial snapshots
    # => do not silently choose one.
    # ----------------------------------------------

    account_conflict = bool(
        (
            df[
                df[
                    "account_key"
                ].notna()
            ]
            .groupby(
                "account_key"
            )
            .size()
            > 1
        ).any()
    )


    if account_conflict:

        result = {
            "raw_available":
                True,

            "account_count":
                int(
                    df[
                        "account_key"
                    ].nunique(
                        dropna=True
                    )
                ),

            "total_begin":
                None,

            "total_end":
                None,

            "total_income":
                None,

            "total_used":
                None,

            "internal_math_ok":
                None,

            "account_conflict":
                True,
        }

        snapshot_cache[
            report_id
        ] = result

        return result


    # ----------------------------------------------
    # Check row arithmetic
    # ----------------------------------------------

    math_flags = []


    for row in (
        df.itertuples(
            index=False
        )
    ):

        if None in {
            row.begin,
            row.end,
            row.income,
            row.used,
        }:

            math_flags.append(
                False
            )

        else:

            math_flags.append(
                close_money(
                    row.begin
                    + row.income
                    - row.used,

                    row.end,
                )
            )


    internal_math_ok = all(
        x is True
        for x in math_flags
    )


    # If values are missing, don't manufacture totals.
    required_cols = [
        "begin",
        "end",
        "income",
        "used",
    ]


    complete = all(
        df[col].notna().all()
        for col in required_cols
    )


    if not complete:

        total_begin = None
        total_end = None
        total_income = None
        total_used = None

    else:

        total_begin = sum(
            df["begin"],
            Decimal("0"),
        )

        total_end = sum(
            df["end"],
            Decimal("0"),
        )

        total_income = sum(
            df["income"],
            Decimal("0"),
        )

        total_used = sum(
            df["used"],
            Decimal("0"),
        )


    result = {
        "raw_available":
            True,

        "account_count":
            int(
                df[
                    "account_key"
                ].nunique(
                    dropna=True
                )
            ),

        "total_begin":
            total_begin,

        "total_end":
            total_end,

        "total_income":
            total_income,

        "total_used":
            total_used,

        "internal_math_ok":
            internal_math_ok,

        "account_conflict":
            False,
    }


    snapshot_cache[
        report_id
    ] = result

    return result


# ============================================================
# SELECTED QUARTER INDEX
# ============================================================

quarterly_selected = (
    selected[
        selected[
            "quarter"
        ].isin(
            [1, 2, 3, 4]
        )
    ]
    .copy()
)


quarterly_selected[
    "period_index"
] = (
    quarterly_selected[
        "year"
    ] * 4
    +
    quarterly_selected[
        "quarter"
    ]
    - 1
)


selected_period_lookup = {}


for row in (
    quarterly_selected.itertuples(
        index=False
    )
):

    selected_period_lookup[
        (
            str(
                row.organization_id
            ),
            int(
                row.period_index
            ),
        )
    ] = str(
        row.report_id
    )


# ============================================================
# ONLY PERIODS PARTICIPATING IN ANOMALY EPISODES
# ============================================================

problem_periods = (
    episode_reports[
        [
            "anomaly_episode_id",
            "organization_id",
            "root_party_id",
            "report_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
    .copy()
)


problem_periods[
    "period_index"
] = (
    problem_periods[
        "year"
    ] * 4
    +
    problem_periods[
        "quarter"
    ]
    - 1
)


# ============================================================
# ALL SOURCE INSTANCES BY PERIOD
# ============================================================

period_instances = {}


for (
    organization_id,
    year,
    quarter,
), df in all_reports.groupby(
    [
        "organization_id",
        "year",
        "quarter",
    ]
):

    period_instances[
        (
            str(
                organization_id
            ),
            int(year),
            int(quarter),
        )
    ] = df.copy()


# ============================================================
# TEST EACH SOURCE VERSION AGAINST ITS SIGNED NEIGHBOURS
# ============================================================

test_rows = []


for period in tqdm(
    problem_periods.itertuples(
        index=False
    ),
    total=len(
        problem_periods
    ),
    desc="Testing alternative versions",
    unit="period",
):

    organization_id = str(
        period.organization_id
    )

    year = int(
        period.year
    )

    quarter = int(
        period.quarter
    )

    period_index = int(
        period.period_index
    )

    official_report_id = str(
        period.report_id
    )


    instances = period_instances.get(
        (
            organization_id,
            year,
            quarter,
        )
    )


    if instances is None:
        continue


    prev_report_id = (
        selected_period_lookup.get(
            (
                organization_id,
                period_index - 1,
            )
        )
    )

    next_report_id = (
        selected_period_lookup.get(
            (
                organization_id,
                period_index + 1,
            )
        )
    )


    prev_snapshot = (
        get_report_snapshot(
            prev_report_id
        )
        if prev_report_id
        else None
    )

    next_snapshot = (
        get_report_snapshot(
            next_report_id
        )
        if next_report_id
        else None
    )


    for candidate in (
        instances.itertuples(
            index=False
        )
    ):

        candidate_id = str(
            candidate.report_id
        )

        snap = get_report_snapshot(
            candidate_id
        )


        if not snap[
            "raw_available"
        ]:

            continue


        if snap[
            "account_conflict"
        ]:

            valid_for_scoring = False

        elif not snap[
            "internal_math_ok"
        ]:

            valid_for_scoring = False

        else:

            valid_for_scoring = True


        left_delta = None
        right_delta = None


        # ----------------------------------------------
        # Previous signed report END
        # vs candidate BEGIN
        # ----------------------------------------------

        if (
            valid_for_scoring
            and prev_snapshot
            and prev_snapshot[
                "total_end"
            ] is not None
            and snap[
                "total_begin"
            ] is not None
        ):

            left_delta = (
                snap[
                    "total_begin"
                ]
                -
                prev_snapshot[
                    "total_end"
                ]
            )


        # ----------------------------------------------
        # Candidate END
        # vs next signed report BEGIN
        # ----------------------------------------------

        if (
            valid_for_scoring
            and next_snapshot
            and next_snapshot[
                "total_begin"
            ] is not None
            and snap[
                "total_end"
            ] is not None
        ):

            right_delta = (
                next_snapshot[
                    "total_begin"
                ]
                -
                snap[
                    "total_end"
                ]
            )


        available_boundaries = sum(
            x is not None
            for x in [
                left_delta,
                right_delta,
            ]
        )


        if available_boundaries:

            continuity_score = sum(
                abs(x)
                for x in [
                    left_delta,
                    right_delta,
                ]
                if x is not None
            )

        else:

            continuity_score = None


        left_exact = (
            close_money(
                left_delta,
                Decimal("0"),
            )
            if left_delta
            is not None
            else None
        )

        right_exact = (
            close_money(
                right_delta,
                Decimal("0"),
            )
            if right_delta
            is not None
            else None
        )


        exact_bridge = (
            available_boundaries == 2
            and left_exact is True
            and right_exact is True
        )


        signed_date = getattr(
            candidate,
            "signed_date",
            None,
        )

        is_signed = bool(
            pd.notna(
                signed_date
            )
        )


        test_rows.append(
            {
                "anomaly_episode_id":
                    int(
                        period.anomaly_episode_id
                    ),

                "organization_id":
                    organization_id,

                "root_party_id":
                    str(
                        period.root_party_id
                    ),

                "year":
                    year,

                "quarter":
                    quarter,

                "official_selected_report_id":
                    official_report_id,

                "candidate_report_id":
                    candidate_id,

                "candidate_is_official_selected":
                    candidate_id
                    == official_report_id,

                "candidate_signed":
                    is_signed,

                "candidate_signed_date":
                    signed_date,

                "candidate_created_date":
                    getattr(
                        candidate,
                        "created_date",
                        None,
                    ),

                "candidate_account_count":
                    snap[
                        "account_count"
                    ],

                "candidate_total_begin":
                    float(
                        snap[
                            "total_begin"
                        ]
                    )
                    if snap[
                        "total_begin"
                    ]
                    is not None
                    else None,

                "candidate_total_end":
                    float(
                        snap[
                            "total_end"
                        ]
                    )
                    if snap[
                        "total_end"
                    ]
                    is not None
                    else None,

                "candidate_total_income":
                    float(
                        snap[
                            "total_income"
                        ]
                    )
                    if snap[
                        "total_income"
                    ]
                    is not None
                    else None,

                "candidate_total_used":
                    float(
                        snap[
                            "total_used"
                        ]
                    )
                    if snap[
                        "total_used"
                    ]
                    is not None
                    else None,

                "candidate_internal_math_ok":
                    snap[
                        "internal_math_ok"
                    ],

                "candidate_account_conflict":
                    snap[
                        "account_conflict"
                    ],

                "previous_selected_report_id":
                    prev_report_id,

                "next_selected_report_id":
                    next_report_id,

                "left_balance_delta":
                    float(
                        left_delta
                    )
                    if left_delta
                    is not None
                    else None,

                "right_balance_delta":
                    float(
                        right_delta
                    )
                    if right_delta
                    is not None
                    else None,

                "left_exact":
                    left_exact,

                "right_exact":
                    right_exact,

                "available_boundaries":
                    available_boundaries,

                "continuity_score":
                    float(
                        continuity_score
                    )
                    if continuity_score
                    is not None
                    else None,

                "exact_bridge":
                    exact_bridge,
            }
        )


tests = pd.DataFrame(
    test_rows
)


# ============================================================
# COMPARE ALTERNATIVES WITH OFFICIAL SELECTED VERSION
# ============================================================

official_scores = (
    tests[
        tests[
            "candidate_is_official_selected"
        ]
    ][
        [
            "anomaly_episode_id",
            "organization_id",
            "year",
            "quarter",
            "continuity_score",
        ]
    ]
    .rename(
        columns={
            "continuity_score":
                "official_continuity_score"
        }
    )
)


tests = tests.merge(
    official_scores,
    on=[
        "anomaly_episode_id",
        "organization_id",
        "year",
        "quarter",
    ],
    how="left",
)


tests[
    "continuity_improvement"
] = (
    tests[
        "official_continuity_score"
    ]
    -
    tests[
        "continuity_score"
    ]
)


tests[
    "improves_on_official"
] = (
    (
        ~tests[
            "candidate_is_official_selected"
        ]
    )
    &
    (
        tests[
            "continuity_improvement"
        ] > 0.01
    )
)


# ============================================================
# BEST ALTERNATIVE FOR EACH PERIOD
# ============================================================

alternatives = (
    tests[
        ~tests[
            "candidate_is_official_selected"
        ]
    ]
    .copy()
)


scorable_alternatives = (
    alternatives[
        alternatives[
            "continuity_score"
        ].notna()
    ]
    .copy()
)


best_alternatives = (
    scorable_alternatives
    .sort_values(
        [
            "anomaly_episode_id",
            "organization_id",
            "year",
            "quarter",
            "continuity_score",
            "candidate_signed",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            True,
            False,
        ],
    )
    .groupby(
        [
            "anomaly_episode_id",
            "organization_id",
            "year",
            "quarter",
        ],
        as_index=False,
    )
    .first()
)


# Add episode metadata.
episode_meta = (
    episodes[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",
            "episode_shape",
            "missing_financial_section_candidate",
            "party_state_funding_exposure",
            "audit_priority",
        ]
    ]
)


best_alternatives = (
    best_alternatives.merge(
        episode_meta,
        on="anomaly_episode_id",
        how="left",
    )
)


# ============================================================
# HIGH-VALUE RESULTS
# ============================================================

restoring = (
    alternatives[
        alternatives[
            "exact_bridge"
        ] == True
    ]
    .merge(
        episode_meta,
        on="anomaly_episode_id",
        how="left",
    )
    .copy()
)


improving = (
    best_alternatives[
        best_alternatives[
            "improves_on_official"
        ] == True
    ]
    .copy()
)


# ============================================================
# SAVE
# ============================================================

tests.to_parquet(
    AUDIT_DIR
    / "anomaly_report_version_continuity_tests.parquet",
    index=False,
)

tests.to_csv(
    AUDIT_DIR
    / "anomaly_report_version_continuity_tests.csv",
    index=False,
    encoding="utf-8-sig",
)


best_alternatives.to_parquet(
    AUDIT_DIR
    / "anomaly_report_version_best_alternatives.parquet",
    index=False,
)

best_alternatives.to_csv(
    AUDIT_DIR
    / "anomaly_report_version_best_alternatives.csv",
    index=False,
    encoding="utf-8-sig",
)


restoring.to_parquet(
    AUDIT_DIR
    / "anomaly_report_versions_exactly_restoring_continuity.parquet",
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 76)
print("ALTERNATIVE REPORT VERSION TEST")
print("=" * 76)

print(
    "Anomaly periods tested:",
    tests[
        [
            "organization_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "All source report instances tested:",
    len(tests)
)

print(
    "Non-selected alternatives tested:",
    len(alternatives)
)

print(
    "Periods with at least one alternative:",
    alternatives[
        [
            "organization_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "Best alternatives improving continuity:",
    len(improving)
)

print(
    "Alternative versions restoring BOTH boundaries exactly:",
    len(restoring)
)


print()
print("=" * 76)
print("EXACT CONTINUITY RESTORATIONS")
print("=" * 76)

display(
    restoring[
        [
            col
            for col in [
                "anomaly_episode_id",
                "party_name",
                "organization_name",

                "year",
                "quarter",

                "official_selected_report_id",
                "candidate_report_id",

                "candidate_signed",
                "candidate_signed_date",

                "candidate_account_count",

                "candidate_total_begin",
                "candidate_total_income",
                "candidate_total_used",
                "candidate_total_end",

                "left_balance_delta",
                "right_balance_delta",

                "official_continuity_score",
                "continuity_score",
                "continuity_improvement",

                "party_state_funding_exposure",
                "audit_priority",
            ]
            if col in restoring.columns
        ]
    ]
    .sort_values(
        [
            "party_state_funding_exposure",
            "continuity_improvement",
        ],
        ascending=[
            False,
            False,
        ],
    )
)


print()
print("=" * 76)
print("BEST IMPROVING ALTERNATIVES")
print("=" * 76)

display(
    improving[
        [
            col
            for col in [
                "anomaly_episode_id",
                "party_name",
                "organization_name",

                "year",
                "quarter",

                "official_selected_report_id",
                "candidate_report_id",

                "candidate_signed",

                "official_continuity_score",
                "continuity_score",
                "continuity_improvement",

                "left_balance_delta",
                "right_balance_delta",
                "exact_bridge",

                "party_state_funding_exposure",
                "audit_priority",
            ]
            if col in improving.columns
        ]
    ]
    .sort_values(
        "continuity_improvement",
        ascending=False,
    )
    .head(100)
)

Testing alternative versions:   0%|          | 0/294 [00:00<?, ?period/s]


ALTERNATIVE REPORT VERSION TEST
Anomaly periods tested: 294
All source report instances tested: 296
Non-selected alternatives tested: 2
Periods with at least one alternative: 1
Best alternatives improving continuity: 1
Alternative versions restoring BOTH boundaries exactly: 2

EXACT CONTINUITY RESTORATIONS


,anomaly_episode_id,party_name,organization_name,year,quarter,official_selected_report_id,candidate_report_id,candidate_signed,candidate_signed_date,candidate_account_count,candidate_total_begin,candidate_total_income,candidate_total_used,candidate_total_end,left_balance_delta,right_balance_delta,official_continuity_score,continuity_score,continuity_improvement,party_state_funding_exposure,audit_priority
0,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,2,764a691c-3237-4044-9430-44313fbf739f,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,False,None,2,31164.96,1050000.0,1077997.21,3167.75,0.0,0.0,34332.71,0.0,34332.71,True,critical_candidate
1,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,2,764a691c-3237-4044-9430-44313fbf739f,c0c83821-6cea-4b1e-8b6d-989b7b92ed1f,False,None,2,31164.96,1050000.0,1077997.21,3167.75,0.0,0.0,34332.71,0.0,34332.71,True,critical_candidate



BEST IMPROVING ALTERNATIVES


,anomaly_episode_id,party_name,organization_name,year,quarter,official_selected_report_id,candidate_report_id,candidate_signed,official_continuity_score,continuity_score,continuity_improvement,left_balance_delta,right_balance_delta,exact_bridge,party_state_funding_exposure,audit_priority
0,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,2,764a691c-3237-4044-9430-44313fbf739f,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,False,34332.71,0.0,34332.71,0.0,0.0,True,True,critical_candidate


In [72]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import json

import pandas as pd


RAW_DIR = Path(
    "data/raw/report_details"
)

AUDIT_DIR = Path(
    "data/interim/audit"
)

REPORTS_DIR = Path(
    "data/interim/reports"
)


# ============================================================
# LOAD
# ============================================================

episodes = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_episodes.parquet"
)

episode_reports = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_episode_reports.parquet"
)

all_reports = pd.read_parquet(
    REPORTS_DIR
    / "all_reports_manifest.parquet"
).copy()


all_reports["report_id"] = (
    all_reports["report_id"]
    .astype(str)
)


# ============================================================
# THREE CRITICAL CANDIDATES
# ============================================================

critical = (
    episodes[
        episodes[
            "missing_financial_section_candidate"
        ] == True
    ]
    .copy()
)

print(
    "Critical candidates:",
    len(critical)
)

display(
    critical[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",
            "start_year",
            "start_quarter",
            "end_year",
            "end_quarter",
            "max_abs_balance_delta",
            "sum_transition_deltas",
            "party_state_funding_exposure",
        ]
    ]
)


# ============================================================
# PAYMENT STRUCTURE
# ============================================================

PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


CASH_INFLOW_TYPES = {
    "monetary_contributions",
    "state_funding",
    "other_incomes",
}


CASH_OUTFLOW_TYPES = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


def dec(value):

    if value is None:
        return Decimal("0")

    try:
        if pd.isna(value):
            return Decimal("0")
    except Exception:
        pass

    try:
        return Decimal(
            str(value)
        )

    except (
        InvalidOperation,
        ValueError,
        TypeError,
    ):
        return Decimal("0")


def get_payment_rows(
    detail,
    payment_type,
):

    direction, key = (
        PAYMENT_PATHS[
            payment_type
        ]
    )

    return (
        (
            (
                detail.get(
                    "payment_info"
                )
                or {}
            )
            .get(direction)
            or {}
        )
        .get(key)
        or []
    )


# ============================================================
# REPORTS IN THE THREE EPISODES
# ============================================================

critical_ids = set(
    critical[
        "anomaly_episode_id"
    ]
)


reports = (
    episode_reports[
        episode_reports[
            "anomaly_episode_id"
        ].isin(
            critical_ids
        )
    ]
    .drop_duplicates(
        [
            "anomaly_episode_id",
            "report_id",
        ]
    )
    .sort_values(
        [
            "anomaly_episode_id",
            "year",
            "quarter",
        ]
    )
    .copy()
)


summary_rows = []
category_rows = []


for row in (
    reports.itertuples(
        index=False
    )
):

    report_id = str(
        row.report_id
    )

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if not path.exists():

        print(
            "MISSING RAW:",
            report_id
        )

        continue


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = (
            json.load(f)[
                "results"
            ]
        )


    # --------------------------------------------------------
    # PROPERTY_MONEYS
    # --------------------------------------------------------

    money_rows = (
        (
            detail.get(
                "properties"
            )
            or {}
        )
        .get(
            "property_moneys"
        )
        or []
    )


    # Dedup exact substantive duplicates.
    money_seen = set()
    money_clean = []


    for m in money_rows:

        key = (
            m.get(
                "account_number"
            ),
            m.get(
                "begin_period_balance"
            ),
            m.get(
                "end_period_balance"
            ),
            m.get(
                "report_period_income"
            ),
            m.get(
                "report_period_used_funds"
            ),
        )

        if key in money_seen:
            continue

        money_seen.add(
            key
        )

        money_clean.append(
            m
        )


    begin_total = sum(
        (
            dec(
                m.get(
                    "begin_period_balance"
                )
            )
            for m in money_clean
        ),
        Decimal("0"),
    )

    end_total = sum(
        (
            dec(
                m.get(
                    "end_period_balance"
                )
            )
            for m in money_clean
        ),
        Decimal("0"),
    )

    reported_income = sum(
        (
            dec(
                m.get(
                    "report_period_income"
                )
            )
            for m in money_clean
        ),
        Decimal("0"),
    )

    reported_used = sum(
        (
            dec(
                m.get(
                    "report_period_used_funds"
                )
            )
            for m in money_clean
        ),
        Decimal("0"),
    )


    # --------------------------------------------------------
    # PAYMENTS
    # --------------------------------------------------------

    cash_inflow = Decimal("0")
    cash_outflow = Decimal("0")

    total_payment_rows = 0


    for payment_type in (
        PAYMENT_PATHS
    ):

        payment_rows = (
            get_payment_rows(
                detail,
                payment_type,
            )
        )

        amount = sum(
            (
                dec(
                    p.get(
                        "payment_amount"
                    )
                )
                for p in payment_rows
            ),
            Decimal("0"),
        )


        total_payment_rows += len(
            payment_rows
        )


        if (
            payment_type
            in CASH_INFLOW_TYPES
        ):

            cash_inflow += amount


        if (
            payment_type
            in CASH_OUTFLOW_TYPES
        ):

            cash_outflow += amount


        category_rows.append(
            {
                "anomaly_episode_id":
                    row.anomaly_episode_id,

                "party_name":
                    critical.loc[
                        critical[
                            "anomaly_episode_id"
                        ]
                        ==
                        row.anomaly_episode_id,
                        "party_name",
                    ].iloc[0],

                "organization_name":
                    critical.loc[
                        critical[
                            "anomaly_episode_id"
                        ]
                        ==
                        row.anomaly_episode_id,
                        "organization_name",
                    ].iloc[0],

                "year":
                    int(row.year),

                "quarter":
                    int(row.quarter),

                "report_id":
                    report_id,

                "payment_type":
                    payment_type,

                "rows":
                    len(
                        payment_rows
                    ),

                "amount":
                    float(
                        amount
                    ),
            }
        )


    # --------------------------------------------------------
    # VERSION COUNTS FOR THIS PERIOD
    # --------------------------------------------------------

    source_instances = (
        all_reports[
            (
                all_reports[
                    "organization_id"
                ].astype(str)
                ==
                str(
                    row.organization_id
                )
            )
            &
            (
                all_reports[
                    "year"
                ]
                ==
                int(row.year)
            )
            &
            (
                all_reports[
                    "quarter"
                ]
                ==
                int(row.quarter)
            )
        ]
    )


    summary_rows.append(
        {
            "anomaly_episode_id":
                row.anomaly_episode_id,

            "party_name":
                critical.loc[
                    critical[
                        "anomaly_episode_id"
                    ]
                    ==
                    row.anomaly_episode_id,
                    "party_name",
                ].iloc[0],

            "organization_name":
                critical.loc[
                    critical[
                        "anomaly_episode_id"
                    ]
                    ==
                    row.anomaly_episode_id,
                    "organization_name",
                ].iloc[0],

            "year":
                int(row.year),

            "quarter":
                int(row.quarter),

            "report_id":
                report_id,

            "signed_date":
                detail.get(
                    "signed_date"
                ),

            "source_instance_count":
                len(
                    source_instances
                ),

            "account_count":
                len(
                    money_clean
                ),

            "begin_balance":
                float(
                    begin_total
                ),

            "reported_income":
                float(
                    reported_income
                ),

            "reported_used":
                float(
                    reported_used
                ),

            "end_balance":
                float(
                    end_total
                ),

            "payment_rows":
                total_payment_rows,

            "cash_inflow_from_transactions":
                float(
                    cash_inflow
                ),

            "cash_outflow_from_transactions":
                float(
                    cash_outflow
                ),

            "transaction_net_flow":
                float(
                    cash_inflow
                    - cash_outflow
                ),

            "has_account_data":
                len(
                    money_clean
                ) > 0,

            "has_transaction_data":
                total_payment_rows > 0,
        }
    )


summary = pd.DataFrame(
    summary_rows
)

categories = pd.DataFrame(
    category_rows
)


# ============================================================
# SAVE
# ============================================================

summary.to_parquet(
    AUDIT_DIR
    / "critical_missing_balance_section_report_diagnostics.parquet",
    index=False,
)

categories.to_parquet(
    AUDIT_DIR
    / "critical_missing_balance_section_payment_categories.parquet",
    index=False,
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 80)
print("CRITICAL REPORT SEQUENCES")
print("=" * 80)

display(
    summary[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "year",
            "quarter",

            "report_id",
            "signed_date",
            "source_instance_count",

            "account_count",
            "begin_balance",
            "reported_income",
            "reported_used",
            "end_balance",

            "payment_rows",
            "cash_inflow_from_transactions",
            "cash_outflow_from_transactions",
            "transaction_net_flow",

            "has_account_data",
            "has_transaction_data",
        ]
    ]
    .sort_values(
        [
            "anomaly_episode_id",
            "year",
            "quarter",
        ]
    )
)


print()
print("=" * 80)
print("NON-ZERO PAYMENT CATEGORIES")
print("=" * 80)

display(
    categories[
        (
            categories[
                "rows"
            ] > 0
        )
        |
        (
            categories[
                "amount"
            ] != 0
        )
    ]
    .sort_values(
        [
            "anomaly_episode_id",
            "year",
            "quarter",
            "payment_type",
        ]
    )
)

Critical candidates: 3


,anomaly_episode_id,party_name,organization_name,start_year,start_quarter,end_year,end_quarter,max_abs_balance_delta,sum_transition_deltas,party_state_funding_exposure
39,40,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,Полтавська крайова організація Народного Руху ...,2025,1,2025,3,117.83,0.00,False
47,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,2025,3,31164.96,-27997.21,True
83,84,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,2025,4,2026,2,14535.58,-182.00,False



CRITICAL REPORT SEQUENCES


,anomaly_episode_id,party_name,organization_name,year,quarter,report_id,signed_date,source_instance_count,account_count,begin_balance,reported_income,reported_used,end_balance,payment_rows,cash_inflow_from_transactions,cash_outflow_from_transactions,transaction_net_flow,has_account_data,has_transaction_data
0,40,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,Полтавська крайова організація Народного Руху ...,2025,1,9cd3b8f0-2cd6-11f0-8f85-d95649015426,2025-05-09 16:08:40,1,1,117.83,0.00,0.00,117.83,0,0.00,0.00,0.00,True,False
1,40,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,Полтавська крайова організація Народного Руху ...,2025,2,8e30a6c9-4cc5-41ba-909d-935b4ce0f204,2025-08-08 21:05:38.500704,1,0,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,False,False
2,40,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,Полтавська крайова організація Народного Руху ...,2025,3,90ca0e4e-e6a3-4d34-8d92-d9c3032808ee,2025-11-05 12:41:54.032102,1,1,117.83,0.00,117.83,0.00,1,0.00,117.83,-117.83,True,True
3,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,d861fd70-2ca2-11f0-b8ea-758045529e07,2025-05-09 10:12:07,1,2,32694.44,1053359.00,1054888.48,31164.96,261,1053359.00,1054888.48,-1529.48,True,True
4,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,2,764a691c-3237-4044-9430-44313fbf739f,2025-08-07 22:15:52.146213,3,0,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,False,False
5,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,3,f5a60c98-bd12-4795-9aef-fb0d9def5471,2025-11-06 11:36:41.599296,1,2,3167.75,1056049.06,998054.95,61161.86,253,1056049.06,998054.95,57994.11,True,True
6,84,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,2025,4,1afcb219-2cfd-4cc2-92d2-af47a49d7ab5,2026-03-12 09:37:40.459089,1,1,14535.58,0.00,0.00,14535.58,0,0.00,0.00,0.00,True,False
7,84,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,2026,1,04040916-0d94-46af-a0f6-6194262b612b,2026-04-24 12:36:59.470336,1,0,0.00,0.00,0.00,0.00,0,0.00,0.00,0.00,False,False
8,84,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,2026,2,484fb507-be99-42cd-94f9-be36cac58bb5,2026-07-03 14:52:14.193405,1,1,14353.58,0.00,0.00,14353.58,0,0.00,0.00,0.00,True,False



NON-ZERO PAYMENT CATEGORIES


,anomaly_episode_id,party_name,organization_name,year,quarter,report_id,payment_type,rows,amount
21,40,Політична партія «НАРОДНИЙ РУХ УКРАЇНИ»,Полтавська крайова організація Народного Руху ...,2025,3,90ca0e4e-e6a3-4d34-8d92-d9c3032808ee,outgoing_expenses,1,117.83
27,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,d861fd70-2ca2-11f0-b8ea-758045529e07,other_incomes,5,1053359.00
29,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,d861fd70-2ca2-11f0-b8ea-758045529e07,outgoing_expenses,256,1054888.48
43,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,3,f5a60c98-bd12-4795-9aef-fb0d9def5471,other_incomes,4,1056049.06
45,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,3,f5a60c98-bd12-4795-9aef-fb0d9def5471,outgoing_expenses,249,998054.95


In [73]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import json

import pandas as pd
from tqdm.auto import tqdm


AUDIT_DIR = Path("data/interim/audit")
REPORTS_DIR = Path("data/interim/reports")
RAW_DIR = Path("data/raw/report_details")


# ============================================================
# LOAD
# ============================================================

episodes = pd.read_parquet(
    AUDIT_DIR / "organization_balance_anomaly_episodes.parquet"
).copy()

breaks = pd.read_parquet(
    AUDIT_DIR / "organization_balance_breaks_with_episode_id.parquet"
).copy()

episode_reports = pd.read_parquet(
    AUDIT_DIR / "organization_balance_anomaly_episode_reports.parquet"
).copy()

all_reports = pd.read_parquet(
    REPORTS_DIR / "all_reports_manifest.parquet"
).copy()


ALT_TEST_PATH = (
    AUDIT_DIR
    / "anomaly_report_version_continuity_tests.parquet"
)

if ALT_TEST_PATH.exists():
    alt_tests = pd.read_parquet(
        ALT_TEST_PATH
    ).copy()
else:
    alt_tests = pd.DataFrame()


# Normalize IDs.
for df in [
    episodes,
    breaks,
    episode_reports,
    all_reports,
    alt_tests,
]:
    for col in [
        "report_id",
        "candidate_report_id",
        "official_selected_report_id",
        "previous_report_id",
        "next_report_id",
        "organization_id",
        "root_party_id",
    ]:
        if col in df.columns:
            df[col] = df[col].astype(str)


# ============================================================
# MONEY HELPERS
# ============================================================

def dec(value):

    if value is None:
        return Decimal("0")

    try:
        if pd.isna(value):
            return Decimal("0")
    except Exception:
        pass

    try:
        return Decimal(
            str(value)
        )

    except (
        InvalidOperation,
        ValueError,
        TypeError,
    ):
        return Decimal("0")


def close_zero(
    value,
    tolerance=Decimal("0.01"),
):

    if value is None:
        return False

    try:
        if pd.isna(value):
            return False
    except Exception:
        pass

    return (
        abs(
            Decimal(str(value))
        )
        <= tolerance
    )


# ============================================================
# PAYMENT STRUCTURE
# ============================================================

PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


CASH_INFLOW_TYPES = {
    "monetary_contributions",
    "state_funding",
    "other_incomes",
}


CASH_OUTFLOW_TYPES = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


# ============================================================
# REPORT-LEVEL FINANCIAL DIAGNOSTICS
# ============================================================

report_ids = sorted(
    set(
        episode_reports[
            "report_id"
        ].astype(str)
    )
)


diagnostic_rows = []


for report_id in tqdm(
    report_ids,
    desc="Episode financial diagnostics",
    unit="report",
):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    if not path.exists():

        diagnostic_rows.append(
            {
                "report_id":
                    report_id,

                "raw_available":
                    False,
            }
        )

        continue


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = json.load(f)[
            "results"
        ]


    # --------------------------------------------------------
    # ACCOUNT / BALANCE SECTION
    # --------------------------------------------------------

    property_moneys = (
        (
            detail.get(
                "properties"
            )
            or {}
        )
        .get(
            "property_moneys"
        )
        or []
    )


    # Dedup exact financial snapshots.
    seen = set()
    money_rows = []


    for row in property_moneys:

        key = (
            row.get(
                "account_number"
            ),
            row.get(
                "begin_period_balance"
            ),
            row.get(
                "end_period_balance"
            ),
            row.get(
                "report_period_income"
            ),
            row.get(
                "report_period_used_funds"
            ),
        )


        if key in seen:
            continue

        seen.add(key)
        money_rows.append(row)


    account_count = len(
        money_rows
    )


    total_begin = sum(
        (
            dec(
                r.get(
                    "begin_period_balance"
                )
            )
            for r in money_rows
        ),
        Decimal("0"),
    )


    total_end = sum(
        (
            dec(
                r.get(
                    "end_period_balance"
                )
            )
            for r in money_rows
        ),
        Decimal("0"),
    )


    total_income = sum(
        (
            dec(
                r.get(
                    "report_period_income"
                )
            )
            for r in money_rows
        ),
        Decimal("0"),
    )


    total_used = sum(
        (
            dec(
                r.get(
                    "report_period_used_funds"
                )
            )
            for r in money_rows
        ),
        Decimal("0"),
    )


    # --------------------------------------------------------
    # TRANSACTIONS
    # --------------------------------------------------------

    payment_info = (
        detail.get(
            "payment_info"
        )
        or {}
    )


    payment_rows_total = 0

    cash_inflow = Decimal("0")
    cash_outflow = Decimal("0")

    state_funding_rows = 0
    state_funding_amount = Decimal("0")

    budget_expense_rows = 0
    budget_expense_amount = Decimal("0")


    for payment_type, (
        direction,
        key,
    ) in PAYMENT_PATHS.items():

        rows = (
            (
                payment_info.get(
                    direction
                )
                or {}
            )
            .get(key)
            or []
        )


        amount = sum(
            (
                dec(
                    row.get(
                        "payment_amount"
                    )
                )
                for row in rows
            ),
            Decimal("0"),
        )


        payment_rows_total += len(
            rows
        )


        if (
            payment_type
            in CASH_INFLOW_TYPES
        ):
            cash_inflow += amount


        if (
            payment_type
            in CASH_OUTFLOW_TYPES
        ):
            cash_outflow += amount


        if (
            payment_type
            == "state_funding"
        ):

            state_funding_rows = len(
                rows
            )

            state_funding_amount = (
                amount
            )


        if (
            payment_type
            == "budget_expenses"
        ):

            budget_expense_rows = len(
                rows
            )

            budget_expense_amount = (
                amount
            )


    diagnostic_rows.append(
        {
            "report_id":
                report_id,

            "raw_available":
                True,

            "account_count":
                account_count,

            "total_begin_balance":
                float(
                    total_begin
                ),

            "total_end_balance":
                float(
                    total_end
                ),

            "reported_income":
                float(
                    total_income
                ),

            "reported_used":
                float(
                    total_used
                ),

            "payment_rows":
                payment_rows_total,

            "cash_inflow":
                float(
                    cash_inflow
                ),

            "cash_outflow":
                float(
                    cash_outflow
                ),

            "transaction_net_flow":
                float(
                    cash_inflow
                    - cash_outflow
                ),

            "state_funding_rows":
                state_funding_rows,

            "state_funding_amount":
                float(
                    state_funding_amount
                ),

            "budget_expense_rows":
                budget_expense_rows,

            "budget_expense_amount":
                float(
                    budget_expense_amount
                ),

            "has_balance_data":
                account_count > 0,

            "has_transaction_data":
                payment_rows_total > 0,

            "financial_block_empty":
                (
                    account_count == 0
                    and
                    payment_rows_total == 0
                ),

            "balance_section_empty_but_transactions_present":
                (
                    account_count == 0
                    and
                    payment_rows_total > 0
                ),
        }
    )


report_diagnostics = pd.DataFrame(
    diagnostic_rows
)


# ============================================================
# ADD REPORT/PERIOD METADATA
# ============================================================

report_meta = (
    episode_reports[
        [
            "anomaly_episode_id",
            "organization_id",
            "root_party_id",
            "report_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates()
)


report_diag = (
    report_meta.merge(
        report_diagnostics,
        on="report_id",
        how="left",
    )
)


report_diag[
    "period_index"
] = (
    report_diag[
        "year"
    ] * 4
    +
    report_diag[
        "quarter"
    ]
    - 1
)


# ============================================================
# SOURCE INSTANCE COUNTS
# ============================================================

source_instance_counts = (
    all_reports
    .groupby(
        [
            "organization_id",
            "year",
            "quarter",
        ]
    )
    .size()
    .rename(
        "source_instance_count"
    )
    .reset_index()
)


report_diag = (
    report_diag.merge(
        source_instance_counts,
        on=[
            "organization_id",
            "year",
            "quarter",
        ],
        how="left",
    )
)


# ============================================================
# ALTERNATIVE VERSION EVIDENCE
# ============================================================

alternative_episode_rows = []


if len(alt_tests):

    alternatives = (
        alt_tests[
            ~alt_tests[
                "candidate_is_official_selected"
            ]
        ]
        .copy()
    )


    for episode_id, df in (
        alternatives.groupby(
            "anomaly_episode_id"
        )
    ):

        # ----------------------------------------------
        # Distinct financial states, not distinct report IDs.
        #
        # This collapses the two identical unsigned
        # Luhansk versions into one substantive state.
        # ----------------------------------------------

        financial_state_cols = [
            "candidate_account_count",
            "candidate_total_begin",
            "candidate_total_income",
            "candidate_total_used",
            "candidate_total_end",
        ]


        distinct_states = (
            df[
                financial_state_cols
            ]
            .drop_duplicates()
        )


        exact = (
            df[
                df[
                    "exact_bridge"
                ] == True
            ]
            .copy()
        )


        exact_distinct_states = (
            exact[
                financial_state_cols
            ]
            .drop_duplicates()
        )


        improving = (
            df[
                df[
                    "improves_on_official"
                ] == True
            ]
            .copy()
        )


        alternative_episode_rows.append(
            {
                "anomaly_episode_id":
                    int(
                        episode_id
                    ),

                "alternative_report_count":
                    len(df),

                "alternative_financial_state_count":
                    len(
                        distinct_states
                    ),

                "improving_alternative_report_count":
                    len(
                        improving
                    ),

                "exact_restoring_report_count":
                    len(
                        exact
                    ),

                "exact_restoring_financial_state_count":
                    len(
                        exact_distinct_states
                    ),

                "has_improving_alternative":
                    len(
                        improving
                    ) > 0,

                "has_exact_restoring_alternative":
                    len(
                        exact
                    ) > 0,

                "alternative_report_ids_json":
                    json.dumps(
                        sorted(
                            set(
                                df[
                                    "candidate_report_id"
                                ].astype(str)
                            )
                        ),
                        ensure_ascii=False,
                    ),

                "exact_restoring_report_ids_json":
                    json.dumps(
                        sorted(
                            set(
                                exact[
                                    "candidate_report_id"
                                ].astype(str)
                            )
                        ),
                        ensure_ascii=False,
                    ),

                "best_continuity_improvement":
                    (
                        float(
                            improving[
                                "continuity_improvement"
                            ].max()
                        )
                        if len(
                            improving
                        )
                        else 0.0
                    ),
            }
        )


alternative_episode_summary = pd.DataFrame(
    alternative_episode_rows
)


# ============================================================
# BUILD FINAL EPISODE CLASSIFICATION
# ============================================================

classification_rows = []


for episode in episodes.itertuples(
    index=False
):

    episode_id = int(
        episode.anomaly_episode_id
    )


    br = (
        breaks[
            breaks[
                "anomaly_episode_id"
            ] == episode_id
        ]
        .sort_values(
            [
                "previous_year",
                "previous_quarter",
            ]
        )
        .copy()
    )


    rd = (
        report_diag[
            report_diag[
                "anomaly_episode_id"
            ] == episode_id
        ]
        .sort_values(
            "period_index"
        )
        .reset_index(
            drop=True
        )
        .copy()
    )


    # --------------------------------------------------------
    # BASIC COUNTS
    # --------------------------------------------------------

    break_count = len(br)

    report_count = len(rd)

    financial_empty_count = int(
        rd[
            "financial_block_empty"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )

    balance_empty_with_transactions_count = int(
        rd[
            "balance_section_empty_but_transactions_present"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )


    if report_count > 2:

        interior = rd.iloc[
            1:-1
        ]

    else:

        interior = rd.iloc[
            0:0
        ]


    interior_financial_empty_count = int(
        interior[
            "financial_block_empty"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )


    interior_balance_empty_tx_count = int(
        interior[
            "balance_section_empty_but_transactions_present"
        ]
        .fillna(False)
        .astype(bool)
        .sum()
    )


    # --------------------------------------------------------
    # DELTAS
    # --------------------------------------------------------

    deltas = (
        br[
            "organization_balance_delta"
        ]
        .astype(float)
        .tolist()
    )


    max_abs_delta = (
        max(
            abs(x)
            for x in deltas
        )
        if deltas
        else 0.0
    )


    total_abs_delta = sum(
        abs(x)
        for x in deltas
    )


    net_delta = sum(
        deltas
    )


    net_reversal_exact = (
        close_zero(
            net_delta
        )
    )


    # --------------------------------------------------------
    # ALTERNATIVE VERSION INFO
    # --------------------------------------------------------

    alt_match = (
        alternative_episode_summary[
            alternative_episode_summary[
                "anomaly_episode_id"
            ] == episode_id
        ]
        if len(
            alternative_episode_summary
        )
        else pd.DataFrame()
    )


    if len(alt_match):

        alt_row = (
            alt_match.iloc[0]
        )

        alternative_report_count = int(
            alt_row[
                "alternative_report_count"
            ]
        )

        alternative_financial_state_count = int(
            alt_row[
                "alternative_financial_state_count"
            ]
        )

        exact_restoring_report_count = int(
            alt_row[
                "exact_restoring_report_count"
            ]
        )

        exact_restoring_financial_state_count = int(
            alt_row[
                "exact_restoring_financial_state_count"
            ]
        )

        has_improving_alternative = bool(
            alt_row[
                "has_improving_alternative"
            ]
        )

        has_exact_restoring_alternative = bool(
            alt_row[
                "has_exact_restoring_alternative"
            ]
        )

        alternative_report_ids_json = (
            alt_row[
                "alternative_report_ids_json"
            ]
        )

        exact_restoring_report_ids_json = (
            alt_row[
                "exact_restoring_report_ids_json"
            ]
        )

        best_continuity_improvement = float(
            alt_row[
                "best_continuity_improvement"
            ]
        )

    else:

        alternative_report_count = 0
        alternative_financial_state_count = 0
        exact_restoring_report_count = 0
        exact_restoring_financial_state_count = 0

        has_improving_alternative = False
        has_exact_restoring_alternative = False

        alternative_report_ids_json = "[]"
        exact_restoring_report_ids_json = "[]"

        best_continuity_improvement = 0.0


    # --------------------------------------------------------
    # SINGLE-BREAK EDGE STATUS
    # --------------------------------------------------------

    previous_financial_empty = False
    next_financial_empty = False

    previous_balance_empty_tx = False
    next_balance_empty_tx = False


    if break_count == 1:

        b = br.iloc[0]

        previous_id = str(
            b[
                "previous_report_id"
            ]
        )

        next_id = str(
            b[
                "next_report_id"
            ]
        )


        prev_diag = rd[
            rd[
                "report_id"
            ] == previous_id
        ]

        next_diag = rd[
            rd[
                "report_id"
            ] == next_id
        ]


        if len(prev_diag):

            previous_financial_empty = bool(
                prev_diag.iloc[0][
                    "financial_block_empty"
                ]
            )

            previous_balance_empty_tx = bool(
                prev_diag.iloc[0][
                    "balance_section_empty_but_transactions_present"
                ]
            )


        if len(next_diag):

            next_financial_empty = bool(
                next_diag.iloc[0][
                    "financial_block_empty"
                ]
            )

            next_balance_empty_tx = bool(
                next_diag.iloc[0][
                    "balance_section_empty_but_transactions_present"
                ]
            )


    # ========================================================
    # CLASSIFICATION — PRECEDENCE MATTERS
    # ========================================================

    # --------------------------------------------------------
    # A. Strongest possible public-data evidence:
    #
    # official signed version breaks continuity,
    # alternative source version restores both boundaries.
    # --------------------------------------------------------

    if has_exact_restoring_alternative:

        anomaly_class = (
            "version_integrity_restored_by_alternative"
        )

        anomaly_family = (
            "version_integrity"
        )

        classification_confidence = (
            "very_high"
        )

        recommended_review = (
            "Compare official signed report with all stored "
            "source versions and determine why financial "
            "content differs between versions."
        )


    # --------------------------------------------------------
    # B. Empty intermediate quarter,
    # same balance reappears exactly.
    # --------------------------------------------------------

    elif (
        break_count == 2
        and
        report_count == 3
        and
        interior_financial_empty_count == 1
        and
        net_reversal_exact
    ):

        anomaly_class = (
            "empty_intermediate_exact_carryover"
        )

        anomaly_family = (
            "missing_financial_block"
        )

        classification_confidence = (
            "high"
        )

        recommended_review = (
            "Check whether the intermediate report omitted "
            "an unchanged account/balance section."
        )


    # --------------------------------------------------------
    # C. Empty intermediate quarter,
    # later balance differs.
    # --------------------------------------------------------

    elif (
        break_count == 2
        and
        report_count == 3
        and
        interior_financial_empty_count == 1
        and
        not net_reversal_exact
    ):

        anomaly_class = (
            "empty_intermediate_unexplained_balance_change"
        )

        anomaly_family = (
            "missing_financial_block"
        )

        classification_confidence = (
            "high"
        )

        recommended_review = (
            "Check omitted intermediate financial activity "
            "or an incorrect opening/closing balance. "
            "Public report sequence does not explain the "
            "net balance change."
        )


    # --------------------------------------------------------
    # D. Balance section absent but transactions exist.
    # --------------------------------------------------------

    elif (
        interior_balance_empty_tx_count > 0
    ):

        anomaly_class = (
            "missing_balance_section_with_transactions"
        )

        anomaly_family = (
            "incomplete_financial_block"
        )

        classification_confidence = (
            "high"
        )

        recommended_review = (
            "Transactions are present while account/balance "
            "rows are absent; review report completeness."
        )


    # --------------------------------------------------------
    # E. Two adjacent mismatches exactly cancel.
    # No empty middle financial block.
    # --------------------------------------------------------

    elif (
        break_count == 2
        and
        net_reversal_exact
    ):

        anomaly_class = (
            "transient_exact_balance_reversal"
        )

        anomaly_family = (
            "temporary_balance_discontinuity"
        )

        classification_confidence = (
            "high"
        )

        recommended_review = (
            "Inspect the middle report: its balance differs "
            "from both neighbouring reports by equal and "
            "opposite amounts."
        )


    # --------------------------------------------------------
    # F. Two-sided but does not return to prior trajectory.
    # --------------------------------------------------------

    elif (
        break_count == 2
    ):

        anomaly_class = (
            "two_sided_nonreversing_balance_discontinuity"
        )

        anomaly_family = (
            "balance_discontinuity"
        )

        classification_confidence = (
            "medium_high"
        )

        recommended_review = (
            "Inspect the middle report and both boundaries; "
            "the two discrepancies do not offset."
        )


    # --------------------------------------------------------
    # G. Multi-quarter episode whose net effect returns to zero.
    # --------------------------------------------------------

    elif (
        break_count > 2
        and
        net_reversal_exact
    ):

        anomaly_class = (
            "multi_period_net_reversal"
        )

        anomaly_family = (
            "multi_period_balance_discontinuity"
        )

        classification_confidence = (
            "medium_high"
        )

        recommended_review = (
            "Review the entire multi-quarter sequence; "
            "several discrepancies ultimately offset."
        )


    # --------------------------------------------------------
    # H. Multi-quarter complex episode.
    # --------------------------------------------------------

    elif (
        break_count > 2
    ):

        anomaly_class = (
            "multi_period_unresolved_balance_discontinuity"
        )

        anomaly_family = (
            "multi_period_balance_discontinuity"
        )

        classification_confidence = (
            "medium"
        )

        recommended_review = (
            "Review the full multi-quarter sequence and "
            "underlying account balances."
        )


    # --------------------------------------------------------
    # I. One transition: financial block disappears.
    # --------------------------------------------------------

    elif next_financial_empty:

        anomaly_class = (
            "balance_disappears_into_empty_financial_block"
        )

        anomaly_family = (
            "missing_financial_block"
        )

        classification_confidence = (
            "medium_high"
        )

        recommended_review = (
            "Check why a previously reported non-zero "
            "balance disappears when the next report has "
            "no account or transaction data."
        )


    # --------------------------------------------------------
    # J. One transition: financial balance reappears.
    # --------------------------------------------------------

    elif previous_financial_empty:

        anomaly_class = (
            "balance_reappears_after_empty_financial_block"
        )

        anomaly_family = (
            "missing_financial_block"
        )

        classification_confidence = (
            "medium_high"
        )

        recommended_review = (
            "Check the previous report for omitted financial "
            "data because a non-zero balance appears in the "
            "following period."
        )


    # --------------------------------------------------------
    # K. Balance rows absent but transactions exist at edge.
    # --------------------------------------------------------

    elif (
        previous_balance_empty_tx
        or next_balance_empty_tx
    ):

        anomaly_class = (
            "single_boundary_missing_balance_section_with_transactions"
        )

        anomaly_family = (
            "incomplete_financial_block"
        )

        classification_confidence = (
            "medium_high"
        )

        recommended_review = (
            "Review the report where transaction rows exist "
            "without corresponding account/balance data."
        )


    # --------------------------------------------------------
    # L. Ordinary one-boundary mismatch.
    # --------------------------------------------------------

    else:

        anomaly_class = (
            "single_balance_discontinuity"
        )

        anomaly_family = (
            "balance_discontinuity"
        )

        classification_confidence = (
            "medium"
        )

        recommended_review = (
            "Compare closing balance of the earlier report "
            "with opening balance of the following report."
        )


    # ========================================================
    # REVIEW PRIORITY
    #
    # Separate from the descriptive anomaly class.
    # ========================================================

    state_exposure = bool(
        getattr(
            episode,
            "party_state_funding_exposure",
            False,
        )
    )


    if (
        anomaly_class
        ==
        "version_integrity_restored_by_alternative"
    ):

        review_priority = (
            "critical"
        )


    elif (
        state_exposure
        and
        max_abs_delta >= 10000
    ):

        review_priority = (
            "high"
        )


    elif (
        max_abs_delta >= 100000
    ):

        review_priority = (
            "high"
        )


    elif (
        max_abs_delta >= 10000
    ):

        review_priority = (
            "medium_high"
        )


    elif (
        max_abs_delta >= 1000
    ):

        review_priority = (
            "medium"
        )


    else:

        review_priority = (
            "low"
        )


    # ========================================================
    # REPORT IDS
    # ========================================================

    ordered_report_ids = (
        rd[
            "report_id"
        ]
        .astype(str)
        .tolist()
    )


    empty_report_ids = (
        rd[
            rd[
                "financial_block_empty"
            ]
            .fillna(False)
            .astype(bool)
        ][
            "report_id"
        ]
        .astype(str)
        .tolist()
    )


    classification_rows.append(
        {
            "anomaly_episode_id":
                episode_id,

            "organization_id":
                str(
                    episode.organization_id
                ),

            "root_party_id":
                str(
                    episode.root_party_id
                ),

            "party_name":
                episode.party_name,

            "organization_name":
                episode.organization_name,

            "start_year":
                int(
                    episode.start_year
                ),

            "start_quarter":
                int(
                    episode.start_quarter
                ),

            "end_year":
                int(
                    episode.end_year
                ),

            "end_quarter":
                int(
                    episode.end_quarter
                ),

            "break_count":
                break_count,

            "report_count":
                report_count,

            "anomaly_family":
                anomaly_family,

            "anomaly_class":
                anomaly_class,

            "classification_confidence":
                classification_confidence,

            "review_priority":
                review_priority,

            "max_abs_balance_delta":
                max_abs_delta,

            "total_abs_balance_delta":
                total_abs_delta,

            "net_episode_balance_delta":
                net_delta,

            "net_reversal_exact":
                net_reversal_exact,

            "financial_empty_report_count":
                financial_empty_count,

            "interior_financial_empty_count":
                interior_financial_empty_count,

            "balance_empty_with_transactions_count":
                balance_empty_with_transactions_count,

            "party_state_funding_exposure":
                state_exposure,

            "alternative_report_count":
                alternative_report_count,

            "alternative_financial_state_count":
                alternative_financial_state_count,

            "has_improving_alternative":
                has_improving_alternative,

            "has_exact_restoring_alternative":
                has_exact_restoring_alternative,

            "exact_restoring_report_count":
                exact_restoring_report_count,

            "exact_restoring_financial_state_count":
                exact_restoring_financial_state_count,

            "best_continuity_improvement":
                best_continuity_improvement,

            "report_ids_json":
                json.dumps(
                    ordered_report_ids,
                    ensure_ascii=False,
                ),

            "financial_empty_report_ids_json":
                json.dumps(
                    empty_report_ids,
                    ensure_ascii=False,
                ),

            "alternative_report_ids_json":
                alternative_report_ids_json,

            "exact_restoring_report_ids_json":
                exact_restoring_report_ids_json,

            "recommended_review":
                recommended_review,

            # Explicit methodological guardrail.
            "audit_interpretation":
                (
                    "data_quality_candidate_not_legal_conclusion"
                ),
        }
    )


classification = pd.DataFrame(
    classification_rows
)


# ============================================================
# SORT
# ============================================================

priority_rank = {
    "critical": 0,
    "high": 1,
    "medium_high": 2,
    "medium": 3,
    "low": 4,
}


classification[
    "_priority_rank"
] = classification[
    "review_priority"
].map(
    priority_rank
)


classification = (
    classification
    .sort_values(
        [
            "_priority_rank",
            "max_abs_balance_delta",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# SAVE MASTER AUDIT REGISTRY
# ============================================================

classification.drop(
    columns=[
        "_priority_rank"
    ]
).to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification.parquet",
    index=False,
)


classification.drop(
    columns=[
        "_priority_rank"
    ]
).to_csv(
    AUDIT_DIR
    / "organization_balance_anomaly_classification.csv",
    index=False,
    encoding="utf-8-sig",
)


report_diag.to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_report_diagnostics.parquet",
    index=False,
)


# State-funding review subset.
classification[
    classification[
        "party_state_funding_exposure"
    ] == True
].drop(
    columns=[
        "_priority_rank"
    ]
).to_parquet(
    AUDIT_DIR
    / "organization_balance_anomalies_state_funding_parties.parquet",
    index=False,
)


# ============================================================
# QA
# ============================================================

assert len(
    classification
) == len(
    episodes
), (
    "Classification row count "
    "does not equal episode count."
)


assert (
    classification[
        "anomaly_episode_id"
    ].nunique()
    ==
    len(
        classification
    )
), (
    "Duplicate anomaly_episode_id."
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 80)
print("MASTER ANOMALY CLASSIFICATION")
print("=" * 80)

print(
    "Episodes classified:",
    len(
        classification
    )
)

print()


print(
    "ANOMALY FAMILIES"
)

print(
    classification[
        "anomaly_family"
    ].value_counts()
)

print()


print(
    "ANOMALY CLASSES"
)

print(
    classification[
        "anomaly_class"
    ].value_counts()
)

print()


print(
    "REVIEW PRIORITY"
)

print(
    classification[
        "review_priority"
    ].value_counts()
)

print()


print(
    "State-funding party-network episodes:",
    int(
        classification[
            "party_state_funding_exposure"
        ].sum()
    )
)

print(
    "Episodes with empty financial blocks:",
    int(
        (
            classification[
                "financial_empty_report_count"
            ] > 0
        ).sum()
    )
)

print(
    "Episodes with exact restoring alternative:",
    int(
        classification[
            "has_exact_restoring_alternative"
        ].sum()
    )
)

print(
    "Distinct exact-restoring financial states:",
    int(
        classification[
            "exact_restoring_financial_state_count"
        ].sum()
    )
)


print()
print("=" * 80)
print("TOP AUDIT CASES")
print("=" * 80)

display(
    classification[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "start_year",
            "start_quarter",
            "end_year",
            "end_quarter",

            "anomaly_class",
            "classification_confidence",
            "review_priority",

            "max_abs_balance_delta",
            "net_episode_balance_delta",

            "financial_empty_report_count",

            "has_exact_restoring_alternative",
            "exact_restoring_financial_state_count",

            "party_state_funding_exposure",

            "report_ids_json",
            "exact_restoring_report_ids_json",
        ]
    ]
    .head(100)
)

Episode financial diagnostics:   0%|          | 0/294 [00:00<?, ?report/s]


MASTER ANOMALY CLASSIFICATION
Episodes classified: 131

ANOMALY FAMILIES
anomaly_family
balance_discontinuity                 67
missing_financial_block               45
temporary_balance_discontinuity       12
multi_period_balance_discontinuity     6
version_integrity                      1
Name: count, dtype: int64

ANOMALY CLASSES
anomaly_class
single_balance_discontinuity                     63
balance_reappears_after_empty_financial_block    25
balance_disappears_into_empty_financial_block    18
transient_exact_balance_reversal                 12
multi_period_unresolved_balance_discontinuity     5
two_sided_nonreversing_balance_discontinuity      4
version_integrity_restored_by_alternative         1
empty_intermediate_unexplained_balance_change     1
multi_period_net_reversal                         1
empty_intermediate_exact_carryover                1
Name: count, dtype: int64

REVIEW PRIORITY
review_priority
low            77
medium         37
medium_high     9
high            

,anomaly_episode_id,party_name,organization_name,start_year,start_quarter,end_year,end_quarter,anomaly_class,classification_confidence,review_priority,max_abs_balance_delta,net_episode_balance_delta,financial_empty_report_count,has_exact_restoring_alternative,exact_restoring_financial_state_count,party_state_funding_exposure,report_ids_json,exact_restoring_report_ids_json
0,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,2025,3,version_integrity_restored_by_alternative,very_high,critical,31164.96,-27997.21,1,True,1,True,"[""d861fd70-2ca2-11f0-b8ea-758045529e07"", ""764a...","[""317a1c90-a5a3-43e5-8e62-e205cc84fb1e"", ""c0c8..."
1,30,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,ЧЕРНІГІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПА...,2024,4,2025,1,single_balance_discontinuity,medium,high,585171.00,-585171.00,0,False,0,False,"[""90fb5f50-e060-11ef-bde1-41cc07d44e1f"", ""0b3d...",[]
2,102,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,2024,4,2025,1,balance_disappears_into_empty_financial_block,medium_high,high,366538.98,-366538.98,1,False,0,False,"[""9d9f4dc0-e060-11ef-a9cb-c3967e5ec0ca"", ""df02...",[]
3,24,"ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""","ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""",2024,2,2024,3,balance_reappears_after_empty_financial_block,medium_high,high,315740.00,315740.00,1,False,0,False,"[""d84b7780-5633-11ef-9f3c-219e709335ad"", ""c110...",[]
4,105,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Чернігівська обласна організація ПП ""Рідне місто""",2024,3,2024,4,single_balance_discontinuity,medium,high,72720.00,-72720.00,0,False,0,True,"[""48854fd0-a00d-11ef-9197-0fbca96b9f8a"", ""9d7a...",[]
5,106,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Новомосковська міська організація Партії Зелен...,2021,1,2021,2,single_balance_discontinuity,medium,high,62683.00,-62683.00,0,False,0,True,"[""5c70bc00-7d15-11ec-ba2b-a5f020a204de"", ""64ea...",[]
6,101,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,2026,1,2026,2,single_balance_discontinuity,medium,high,27761.16,-27761.16,0,False,0,True,"[""5b761b27-5a83-4b64-a34f-b3ae18edbca7"", ""04e2...",[]
7,129,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,2025,2,2025,3,single_balance_discontinuity,medium,high,14812.94,14812.94,0,False,0,True,"[""fa433233-c5b4-4e47-8e1e-bd6787b70abe"", ""3f5b...",[]
8,120,ПОЛІТИЧНА ПАРТІЯ «ПОРЯДОК»,ПОЛІТИЧНА ПАРТІЯ «ПОРЯДОК»,2024,2,2024,3,single_balance_discontinuity,medium,medium_high,61637.00,61637.00,0,False,0,False,"[""5acf8d60-5687-11ef-b4cc-1f31c784e768"", ""0bf1...",[]
9,83,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,2025,2,2025,3,balance_reappears_after_empty_financial_block,medium_high,medium_high,14535.58,14535.58,1,False,0,False,"[""44ae7255-c792-4315-9de2-c22ebce664f0"", ""ad69...",[]


In [74]:
from pathlib import Path
from decimal import Decimal, InvalidOperation

import pandas as pd


AUDIT_DIR = Path(
    "data/interim/audit"
)


# ============================================================
# LOAD
# ============================================================

classification = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification.parquet"
).copy()


org_breaks = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_breaks_with_episode_id.parquet"
).copy()


account_continuity = pd.read_parquet(
    AUDIT_DIR
    / "selected_account_continuity_iban_v2.parquet"
).copy()


for df in [
    classification,
    org_breaks,
    account_continuity,
]:
    for col in [
        "organization_id",
        "root_party_id",
        "previous_report_id",
        "next_report_id",
    ]:
        if col in df.columns:
            df[col] = df[col].astype(str)


# ============================================================
# HELPERS
# ============================================================

def dec(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        return Decimal(
            str(value)
        )

    except (
        InvalidOperation,
        ValueError,
        TypeError,
    ):
        return None


def close_money(
    left,
    right,
    tolerance=Decimal("0.01"),
):

    if left is None or right is None:
        return None

    return (
        abs(left - right)
        <= tolerance
    )


# ============================================================
# ATTACH EPISODE IDS TO ACCOUNT CONTINUITY ROWS
# ============================================================

transition_keys = (
    org_breaks[
        [
            "anomaly_episode_id",
            "organization_id",
            "previous_report_id",
            "next_report_id",
            "organization_balance_delta",
        ]
    ]
    .drop_duplicates()
)


account_anomaly_rows = (
    account_continuity.merge(
        transition_keys,
        on=[
            "organization_id",
            "previous_report_id",
            "next_report_id",
        ],
        how="inner",
        validate="many_to_one",
    )
)


# ============================================================
# ACCOUNT-LEVEL CONTRIBUTION TO ORGANIZATION DELTA
#
# Formula:
#
# same account:
#   next begin - previous end
#
# removed account:
#   0 - previous end
#
# new account:
#   next begin - 0
#
# therefore:
#
# Σ account contributions
# =
# Σ next opening balances
# -
# Σ previous closing balances
# ============================================================

contributions = []


for row in account_anomaly_rows.itertuples(
    index=False
):

    prev_end = dec(
        row.previous_end_balance
    )

    next_begin = dec(
        row.next_begin_balance
    )


    status = row.continuity_status


    if status in {
        "continuous",
        "balance_mismatch",
    }:

        if (
            prev_end is not None
            and next_begin is not None
        ):

            contribution = (
                next_begin
                - prev_end
            )

        else:

            contribution = None


    elif status in {
        "account_missing_next_with_nonzero_end",
        "account_closed_at_zero",
    }:

        if prev_end is not None:

            contribution = (
                Decimal("0")
                - prev_end
            )

        else:

            contribution = None


    elif status in {
        "new_account_with_nonzero_begin",
        "new_account_with_zero_begin",
    }:

        if next_begin is not None:

            contribution = (
                next_begin
            )

        else:

            contribution = None


    else:

        contribution = None


    contributions.append(
        float(contribution)
        if contribution is not None
        else None
    )


account_anomaly_rows[
    "account_delta_contribution"
] = contributions


# ============================================================
# RAW SPELLING CHANGE FLAG
# ============================================================

account_anomaly_rows[
    "raw_account_spelling_changed"
] = (
    account_anomaly_rows[
        "previous_account_number_raw"
    ].notna()
    &
    account_anomaly_rows[
        "next_account_number_raw"
    ].notna()
    &
    (
        account_anomaly_rows[
            "previous_account_number_raw"
        ]
        !=
        account_anomaly_rows[
            "next_account_number_raw"
        ]
    )
)


# ============================================================
# ACCOUNT-BREAK FLAGS
# ============================================================

account_anomaly_rows[
    "is_same_account_mismatch"
] = (
    account_anomaly_rows[
        "continuity_status"
    ]
    ==
    "balance_mismatch"
)


account_anomaly_rows[
    "is_nonzero_account_removed"
] = (
    account_anomaly_rows[
        "continuity_status"
    ]
    ==
    "account_missing_next_with_nonzero_end"
)


account_anomaly_rows[
    "is_nonzero_account_added"
] = (
    account_anomaly_rows[
        "continuity_status"
    ]
    ==
    "new_account_with_nonzero_begin"
)


account_anomaly_rows[
    "is_account_level_break"
] = (
    account_anomaly_rows[
        "is_same_account_mismatch"
    ]
    |
    account_anomaly_rows[
        "is_nonzero_account_removed"
    ]
    |
    account_anomaly_rows[
        "is_nonzero_account_added"
    ]
)


# ============================================================
# TRANSITION-LEVEL DECOMPOSITION
# ============================================================

transition_rows = []


group_cols = [
    "anomaly_episode_id",
    "organization_id",
    "previous_report_id",
    "next_report_id",
]


for keys, df in (
    account_anomaly_rows.groupby(
        group_cols,
        dropna=False,
    )
):

    (
        episode_id,
        organization_id,
        previous_report_id,
        next_report_id,
    ) = keys


    expected_delta = dec(
        df[
            "organization_balance_delta"
        ].iloc[0]
    )


    usable = df[
        df[
            "account_delta_contribution"
        ].notna()
    ]


    decomposition_complete = (
        len(usable)
        ==
        len(df)
    )


    if decomposition_complete:

        decomposed_delta = sum(
            (
                dec(x)
                for x in usable[
                    "account_delta_contribution"
                ]
            ),
            Decimal("0"),
        )

        decomposition_matches = (
            close_money(
                decomposed_delta,
                expected_delta,
            )
        )

    else:

        decomposed_delta = None
        decomposition_matches = False


    same_mismatch_count = int(
        df[
            "is_same_account_mismatch"
        ].sum()
    )

    removed_count = int(
        df[
            "is_nonzero_account_removed"
        ].sum()
    )

    added_count = int(
        df[
            "is_nonzero_account_added"
        ].sum()
    )


    # --------------------------------------------------------
    # ACCOUNT STRUCTURE CLASS
    # --------------------------------------------------------

    if (
        same_mismatch_count > 0
        and
        removed_count == 0
        and
        added_count == 0
    ):

        structure_class = (
            "same_account_balance_mismatch"
        )


    elif (
        same_mismatch_count == 0
        and
        removed_count > 0
        and
        added_count > 0
    ):

        structure_class = (
            "account_set_replacement"
        )


    elif (
        same_mismatch_count == 0
        and
        removed_count > 0
        and
        added_count == 0
    ):

        structure_class = (
            "nonzero_account_disappears"
        )


    elif (
        same_mismatch_count == 0
        and
        removed_count == 0
        and
        added_count > 0
    ):

        structure_class = (
            "nonzero_account_appears"
        )


    elif (
        same_mismatch_count > 0
        and
        (
            removed_count > 0
            or added_count > 0
        )
    ):

        structure_class = (
            "mixed_account_structure_and_balance_mismatch"
        )


    else:

        structure_class = (
            "other_account_pattern"
        )


    transition_rows.append(
        {
            "anomaly_episode_id":
                int(
                    episode_id
                ),

            "organization_id":
                str(
                    organization_id
                ),

            "previous_report_id":
                str(
                    previous_report_id
                ),

            "next_report_id":
                str(
                    next_report_id
                ),

            "organization_balance_delta":
                float(
                    expected_delta
                )
                if expected_delta is not None
                else None,

            "decomposed_account_delta":
                float(
                    decomposed_delta
                )
                if decomposed_delta is not None
                else None,

            "decomposition_complete":
                decomposition_complete,

            "decomposition_matches":
                bool(
                    decomposition_matches
                ),

            "account_rows":
                len(df),

            "account_break_rows":
                int(
                    df[
                        "is_account_level_break"
                    ].sum()
                ),

            "same_account_mismatch_count":
                same_mismatch_count,

            "nonzero_account_removed_count":
                removed_count,

            "nonzero_account_added_count":
                added_count,

            "raw_spelling_change_count":
                int(
                    df[
                        "raw_account_spelling_changed"
                    ].sum()
                ),

            "account_structure_class":
                structure_class,
        }
    )


transition_decomposition = pd.DataFrame(
    transition_rows
)


# ============================================================
# EPISODE-LEVEL ACCOUNT STRUCTURE
# ============================================================

episode_structure_rows = []


for episode_id, df in (
    transition_decomposition.groupby(
        "anomaly_episode_id"
    )
):

    classes = list(
        dict.fromkeys(
            df[
                "account_structure_class"
            ].astype(str)
        )
    )


    if len(classes) == 1:

        episode_account_structure = (
            classes[0]
        )

    else:

        episode_account_structure = (
            "multiple_account_patterns"
        )


    episode_structure_rows.append(
        {
            "anomaly_episode_id":
                int(
                    episode_id
                ),

            "episode_account_structure":
                episode_account_structure,

            "transition_count_account_decomposed":
                len(df),

            "same_account_mismatch_count":
                int(
                    df[
                        "same_account_mismatch_count"
                    ].sum()
                ),

            "nonzero_account_removed_count":
                int(
                    df[
                        "nonzero_account_removed_count"
                    ].sum()
                ),

            "nonzero_account_added_count":
                int(
                    df[
                        "nonzero_account_added_count"
                    ].sum()
                ),

            "raw_spelling_change_count":
                int(
                    df[
                        "raw_spelling_change_count"
                    ].sum()
                ),

            "all_transition_decompositions_match":
                bool(
                    df[
                        "decomposition_matches"
                    ].all()
                ),
        }
    )


episode_structure = pd.DataFrame(
    episode_structure_rows
)


# ============================================================
# ENRICH MASTER REGISTRY
# ============================================================

enriched = (
    classification.merge(
        episode_structure,
        on="anomaly_episode_id",
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# SAVE
# ============================================================

account_anomaly_rows.to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_account_rows.parquet",
    index=False,
)


transition_decomposition.to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_transition_decomposition.parquet",
    index=False,
)


enriched.to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_enriched.parquet",
    index=False,
)


enriched.to_csv(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_enriched.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# QA
# ============================================================

print()
print("=" * 80)
print("ACCOUNT-LEVEL DECOMPOSITION QA")
print("=" * 80)

print(
    "Organization-level broken transitions:",
    len(
        transition_decomposition
    )
)

print(
    "Transitions whose account decomposition is complete:",
    int(
        transition_decomposition[
            "decomposition_complete"
        ].sum()
    )
)

print(
    "Transitions where account sum matches organization delta:",
    int(
        transition_decomposition[
            "decomposition_matches"
        ].sum()
    )
)

print(
    "Transitions NOT matching:",
    int(
        (
            ~transition_decomposition[
                "decomposition_matches"
            ]
        ).sum()
    )
)


print()
print("=" * 80)
print("ACCOUNT STRUCTURE — TRANSITIONS")
print("=" * 80)

print(
    transition_decomposition[
        "account_structure_class"
    ].value_counts()
)


print()
print("=" * 80)
print("ACCOUNT STRUCTURE — EPISODES")
print("=" * 80)

print(
    enriched[
        "episode_account_structure"
    ].value_counts()
)


# ============================================================
# HIGH / CRITICAL + STATE-FUNDING CASES
# ============================================================

review_cases = (
    enriched[
        (
            enriched[
                "review_priority"
            ].isin(
                [
                    "critical",
                    "high",
                ]
            )
        )
        |
        (
            enriched[
                "party_state_funding_exposure"
            ] == True
        )
    ]
    .copy()
)


priority_rank = {
    "critical": 0,
    "high": 1,
    "medium_high": 2,
    "medium": 3,
    "low": 4,
}


review_cases[
    "_rank"
] = review_cases[
    "review_priority"
].map(
    priority_rank
)


print()
print("=" * 80)
print("PRIORITY / STATE-FUNDING REVIEW COHORT")
print("=" * 80)

print(
    "Episodes:",
    len(
        review_cases
    )
)


display(
    review_cases
    .sort_values(
        [
            "_rank",
            "max_abs_balance_delta",
        ],
        ascending=[
            True,
            False,
        ],
    )
    [
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "start_year",
            "start_quarter",
            "end_year",
            "end_quarter",

            "anomaly_class",
            "episode_account_structure",

            "same_account_mismatch_count",
            "nonzero_account_removed_count",
            "nonzero_account_added_count",

            "max_abs_balance_delta",
            "net_episode_balance_delta",

            "financial_empty_report_count",

            "party_state_funding_exposure",
            "review_priority",

            "has_exact_restoring_alternative",
            "all_transition_decompositions_match",
        ]
    ]
)


ACCOUNT-LEVEL DECOMPOSITION QA
Organization-level broken transitions: 163
Transitions whose account decomposition is complete: 163
Transitions where account sum matches organization delta: 163
Transitions NOT matching: 0

ACCOUNT STRUCTURE — TRANSITIONS
account_structure_class
same_account_balance_mismatch                   81
nonzero_account_appears                         45
nonzero_account_disappears                      35
account_set_replacement                          1
mixed_account_structure_and_balance_mismatch     1
Name: count, dtype: int64

ACCOUNT STRUCTURE — EPISODES
episode_account_structure
same_account_balance_mismatch                   61
nonzero_account_appears                         33
nonzero_account_disappears                      24
multiple_account_patterns                       11
account_set_replacement                          1
mixed_account_structure_and_balance_mismatch     1
Name: count, dtype: int64

PRIORITY / STATE-FUNDING REVIEW COHORT
Episodes: 29

,anomaly_episode_id,party_name,organization_name,start_year,start_quarter,end_year,end_quarter,anomaly_class,episode_account_structure,same_account_mismatch_count,nonzero_account_removed_count,nonzero_account_added_count,max_abs_balance_delta,net_episode_balance_delta,financial_empty_report_count,party_state_funding_exposure,review_priority,has_exact_restoring_alternative,all_transition_decompositions_match
0,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,2025,1,2025,3,version_integrity_restored_by_alternative,multiple_account_patterns,0,1,1,31164.96,-27997.21,1,True,critical,True,True
1,30,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,ЧЕРНІГІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПА...,2024,4,2025,1,single_balance_discontinuity,nonzero_account_disappears,0,1,0,585171.00,-585171.00,0,False,high,False,True
2,102,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,2024,4,2025,1,balance_disappears_into_empty_financial_block,nonzero_account_disappears,0,1,0,366538.98,-366538.98,1,False,high,False,True
3,24,"ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""","ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""",2024,2,2024,3,balance_reappears_after_empty_financial_block,nonzero_account_appears,0,0,1,315740.00,315740.00,1,False,high,False,True
4,105,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Чернігівська обласна організація ПП ""Рідне місто""",2024,3,2024,4,single_balance_discontinuity,same_account_balance_mismatch,1,0,0,72720.00,-72720.00,0,True,high,False,True
5,106,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Новомосковська міська організація Партії Зелен...,2021,1,2021,2,single_balance_discontinuity,same_account_balance_mismatch,1,0,0,62683.00,-62683.00,0,True,high,False,True
6,101,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,2026,1,2026,2,single_balance_discontinuity,nonzero_account_disappears,0,1,0,27761.16,-27761.16,0,True,high,False,True
7,129,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,2025,2,2025,3,single_balance_discontinuity,nonzero_account_appears,0,0,1,14812.94,14812.94,0,True,high,False,True
18,122,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»,"Кам""янець-Подільська міська організація Всеукр...",2025,3,2025,4,single_balance_discontinuity,same_account_balance_mismatch,1,0,0,8649.79,8649.79,0,True,medium,False,True
27,107,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,2021,1,2021,2,single_balance_discontinuity,same_account_balance_mismatch,1,0,0,2888.00,-2888.00,0,True,medium,False,True


In [75]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import json

import pandas as pd
from tqdm.auto import tqdm


AUDIT_DIR = Path("data/interim/audit")
RAW_DIR = Path("data/raw/report_details")


# ============================================================
# LOAD
# ============================================================

enriched = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_enriched.parquet"
).copy()

transitions = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_transition_decomposition.parquet"
).copy()

account_rows = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_account_rows.parquet"
).copy()

report_diag = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_report_diagnostics.parquet"
).copy()


for df in [
    enriched,
    transitions,
    account_rows,
    report_diag,
]:
    for col in [
        "organization_id",
        "root_party_id",
        "report_id",
        "previous_report_id",
        "next_report_id",
    ]:
        if col in df.columns:
            df[col] = df[col].astype(str)


# ============================================================
# REVIEW COHORT
#
# union:
#   critical/high
#   OR party-network state-funding exposure
# ============================================================

review = (
    enriched[
        (
            enriched["review_priority"].isin(
                ["critical", "high"]
            )
        )
        |
        (
            enriched[
                "party_state_funding_exposure"
            ] == True
        )
    ]
    .copy()
)


review_episode_ids = set(
    review["anomaly_episode_id"]
)


review_transitions = (
    transitions[
        transitions[
            "anomaly_episode_id"
        ].isin(
            review_episode_ids
        )
    ]
    .copy()
)


review_account_rows = (
    account_rows[
        account_rows[
            "anomaly_episode_id"
        ].isin(
            review_episode_ids
        )
    ]
    .copy()
)


# ============================================================
# HELPERS
# ============================================================

def dec(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    try:
        return Decimal(str(value))

    except (
        InvalidOperation,
        TypeError,
        ValueError,
    ):
        return None


def money_equal(
    left,
    right,
    tolerance=Decimal("0.01"),
):

    if left is None or right is None:
        return False

    return abs(left - right) <= tolerance


PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


CASH_IN = {
    "monetary_contributions",
    "state_funding",
    "other_incomes",
}


CASH_OUT = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


DATE_CANDIDATES = [
    "payment_date",
    "transaction_date",
    "operation_date",
    "date",
    "payment_datetime",
    "created_at",
]


def payment_date_value(row):

    for key in DATE_CANDIDATES:

        value = row.get(key)

        if value not in {
            None,
            "",
        }:
            return value

    return None


# ============================================================
# EXTRACT PAYMENT ROWS FROM ALL REPORTS USED BY REVIEW COHORT
# ============================================================

report_ids = set(
    review_transitions[
        "previous_report_id"
    ]
).union(
    set(
        review_transitions[
            "next_report_id"
        ]
    )
)


payment_rows = []
report_payment_summary_rows = []


for report_id in tqdm(
    sorted(report_ids),
    desc="Review report transactions",
    unit="report",
):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if not path.exists():
        continue


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = json.load(f)[
            "results"
        ]


    payment_info = (
        detail.get("payment_info")
        or {}
    )


    report_cash_in = Decimal("0")
    report_cash_out = Decimal("0")

    report_state_funding = Decimal("0")
    report_budget_expenses = Decimal("0")

    total_rows = 0


    for payment_type, (
        direction,
        key,
    ) in PAYMENT_PATHS.items():

        rows = (
            (
                payment_info.get(direction)
                or {}
            )
            .get(key)
            or []
        )


        category_total = Decimal("0")


        for row in rows:

            amount = dec(
                row.get("payment_amount")
            )

            if amount is None:
                continue


            category_total += amount

            total_rows += 1


            payment_rows.append(
                {
                    "report_id":
                        str(report_id),

                    "payment_type":
                        payment_type,

                    "payment_row_id":
                        row.get("id"),

                    "payment_amount":
                        float(amount),

                    "payment_date_raw":
                        payment_date_value(row),

                    "party_id":
                        row.get("party_id"),

                    "office_id":
                        row.get("office_id"),

                    # preserve useful raw counterpart fields
                    # when present without assuming their meaning
                    "payer_name":
                        row.get("payer_name"),

                    "recipient_name":
                        row.get("recipient_name"),

                    "counterparty_name":
                        row.get("counterparty_name"),

                    "payer_code":
                        row.get("payer_code"),

                    "recipient_code":
                        row.get("recipient_code"),

                    "counterparty_code":
                        row.get("counterparty_code"),
                }
            )


        if payment_type in CASH_IN:
            report_cash_in += category_total

        if payment_type in CASH_OUT:
            report_cash_out += category_total

        if payment_type == "state_funding":
            report_state_funding += category_total

        if payment_type == "budget_expenses":
            report_budget_expenses += category_total


    report_payment_summary_rows.append(
        {
            "report_id":
                str(report_id),

            "payment_row_count":
                total_rows,

            "cash_inflow":
                float(
                    report_cash_in
                ),

            "cash_outflow":
                float(
                    report_cash_out
                ),

            "net_cash_flow":
                float(
                    report_cash_in
                    - report_cash_out
                ),

            "direct_state_funding_amount":
                float(
                    report_state_funding
                ),

            "budget_expense_amount":
                float(
                    report_budget_expenses
                ),

            "has_direct_state_funding_rows":
                report_state_funding != 0,

            "has_budget_expense_rows":
                report_budget_expenses != 0,
        }
    )


payments = pd.DataFrame(
    payment_rows
)

payment_summary = pd.DataFrame(
    report_payment_summary_rows
)


# ============================================================
# CATEGORY TOTALS
# ============================================================

if len(payments):

    category_totals = (
        payments
        .groupby(
            [
                "report_id",
                "payment_type",
            ],
            as_index=False,
        )
        .agg(
            category_rows=(
                "payment_amount",
                "size",
            ),

            category_amount=(
                "payment_amount",
                "sum",
            ),
        )
    )

else:

    category_totals = pd.DataFrame(
        columns=[
            "report_id",
            "payment_type",
            "category_rows",
            "category_amount",
        ]
    )


# ============================================================
# BUILD TRANSITION DOSSIERS
# ============================================================

review_meta = (
    review[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",
            "anomaly_class",
            "episode_account_structure",
            "review_priority",
            "party_state_funding_exposure",
            "has_exact_restoring_alternative",
        ]
    ]
)


dossiers = (
    review_transitions.merge(
        review_meta,
        on="anomaly_episode_id",
        how="left",
        validate="many_to_one",
    )
)


# Previous report payment context.
prev_pay = (
    payment_summary.rename(
        columns={
            col:
                f"previous_{col}"
            for col in payment_summary.columns
            if col != "report_id"
        }
    )
    .rename(
        columns={
            "report_id":
                "previous_report_id"
        }
    )
)


next_pay = (
    payment_summary.rename(
        columns={
            col:
                f"next_{col}"
            for col in payment_summary.columns
            if col != "report_id"
        }
    )
    .rename(
        columns={
            "report_id":
                "next_report_id"
        }
    )
)


dossiers = dossiers.merge(
    prev_pay,
    on="previous_report_id",
    how="left",
)


dossiers = dossiers.merge(
    next_pay,
    on="next_report_id",
    how="left",
)


# ============================================================
# DIRECT STATE-FUNDING CONTEXT
#
# More precise than party-network exposure:
# does either adjacent report itself contain
# state_funding / budget_expenses rows?
# ============================================================

dossiers[
    "transition_direct_state_funding_context"
] = (
    dossiers[
        "previous_has_direct_state_funding_rows"
    ].fillna(False)
    |
    dossiers[
        "previous_has_budget_expense_rows"
    ].fillna(False)
    |
    dossiers[
        "next_has_direct_state_funding_rows"
    ].fillna(False)
    |
    dossiers[
        "next_has_budget_expense_rows"
    ].fillna(False)
)


# ============================================================
# SEARCH FOR AMOUNT COINCIDENCES
#
# Diagnostic only:
# exact amount match != proof of explanation.
# ============================================================

match_rows = []


for tr in (
    dossiers.itertuples(
        index=False
    )
):

    delta = dec(
        tr.organization_balance_delta
    )

    if delta is None:
        continue


    target = abs(delta)


    for side, report_id in [
        (
            "previous_report",
            str(
                tr.previous_report_id
            ),
        ),
        (
            "next_report",
            str(
                tr.next_report_id
            ),
        ),
    ]:

        # ----------------------------------------------
        # Individual transaction amount
        # ----------------------------------------------

        if len(payments):

            rp = payments[
                payments[
                    "report_id"
                ] == report_id
            ]


            for p in (
                rp.itertuples(
                    index=False
                )
            ):

                amount = abs(
                    dec(
                        p.payment_amount
                    )
                )


                if money_equal(
                    amount,
                    target,
                ):

                    match_rows.append(
                        {
                            "anomaly_episode_id":
                                tr.anomaly_episode_id,

                            "previous_report_id":
                                str(
                                    tr.previous_report_id
                                ),

                            "next_report_id":
                                str(
                                    tr.next_report_id
                                ),

                            "organization_balance_delta":
                                float(delta),

                            "match_side":
                                side,

                            "match_type":
                                "individual_payment_amount",

                            "matched_report_id":
                                report_id,

                            "payment_type":
                                p.payment_type,

                            "matched_amount":
                                float(
                                    amount
                                ),

                            "payment_row_id":
                                p.payment_row_id,

                            "payment_date_raw":
                                p.payment_date_raw,
                        }
                    )


        # ----------------------------------------------
        # Whole payment-category total
        # ----------------------------------------------

        rc = category_totals[
            category_totals[
                "report_id"
            ] == report_id
        ]


        for c in (
            rc.itertuples(
                index=False
            )
        ):

            amount = abs(
                dec(
                    c.category_amount
                )
            )


            if money_equal(
                amount,
                target,
            ):

                match_rows.append(
                    {
                        "anomaly_episode_id":
                            tr.anomaly_episode_id,

                        "previous_report_id":
                            str(
                                tr.previous_report_id
                            ),

                        "next_report_id":
                            str(
                                tr.next_report_id
                            ),

                        "organization_balance_delta":
                            float(delta),

                        "match_side":
                            side,

                        "match_type":
                            "payment_category_total",

                        "matched_report_id":
                            report_id,

                        "payment_type":
                            c.payment_type,

                        "matched_amount":
                            float(
                                amount
                            ),

                        "payment_row_id":
                            None,

                        "payment_date_raw":
                            None,
                    }
                )


matches = pd.DataFrame(
    match_rows
)


# ============================================================
# ADD MATCH COUNTS TO DOSSIER
# ============================================================

if len(matches):

    match_summary = (
        matches
        .groupby(
            [
                "anomaly_episode_id",
                "previous_report_id",
                "next_report_id",
            ],
            as_index=False,
        )
        .agg(
            exact_amount_match_count=(
                "match_type",
                "size",
            ),

            individual_payment_match_count=(
                "match_type",
                lambda s:
                    int(
                        (
                            s
                            ==
                            "individual_payment_amount"
                        ).sum()
                    ),
            ),

            category_total_match_count=(
                "match_type",
                lambda s:
                    int(
                        (
                            s
                            ==
                            "payment_category_total"
                        ).sum()
                    ),
            ),
        )
    )


    dossiers = dossiers.merge(
        match_summary,
        on=[
            "anomaly_episode_id",
            "previous_report_id",
            "next_report_id",
        ],
        how="left",
    )

else:

    dossiers[
        "exact_amount_match_count"
    ] = 0

    dossiers[
        "individual_payment_match_count"
    ] = 0

    dossiers[
        "category_total_match_count"
    ] = 0


for col in [
    "exact_amount_match_count",
    "individual_payment_match_count",
    "category_total_match_count",
]:

    dossiers[col] = (
        dossiers[col]
        .fillna(0)
        .astype(int)
    )


# ============================================================
# ACCOUNT DETAILS — ONLY ACTUAL BREAK ROWS
# ============================================================

broken_account_details = (
    review_account_rows[
        review_account_rows[
            "is_account_level_break"
        ] == True
    ]
    .copy()
)


broken_account_details = (
    broken_account_details.merge(
        review_meta,
        on="anomaly_episode_id",
        how="left",
        validate="many_to_one",
    )
)


# ============================================================
# SAVE
# ============================================================

dossiers.to_parquet(
    AUDIT_DIR
    / "priority_state_funding_transition_dossiers.parquet",
    index=False,
)

dossiers.to_csv(
    AUDIT_DIR
    / "priority_state_funding_transition_dossiers.csv",
    index=False,
    encoding="utf-8-sig",
)


broken_account_details.to_parquet(
    AUDIT_DIR
    / "priority_state_funding_account_break_details.parquet",
    index=False,
)


payments.to_parquet(
    AUDIT_DIR
    / "priority_state_funding_payment_rows.parquet",
    index=False,
)


matches.to_parquet(
    AUDIT_DIR
    / "priority_state_funding_boundary_amount_matches.parquet",
    index=False,
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 80)
print("PRIORITY / STATE-FUNDING DOSSIERS")
print("=" * 80)

print(
    "Review episodes:",
    len(review)
)

print(
    "Broken transitions:",
    len(dossiers)
)

print(
    "Broken account rows:",
    len(
        broken_account_details
    )
)

print(
    "Adjacent payment rows inspected:",
    len(payments)
)

print()


print(
    "Transitions with DIRECT state-funding context:",
    int(
        dossiers[
            "transition_direct_state_funding_context"
        ].sum()
    )
)

print(
    "Transitions with any exact payment-amount coincidence:",
    int(
        (
            dossiers[
                "exact_amount_match_count"
            ] > 0
        ).sum()
    )
)


print()
print("=" * 80)
print("ACCOUNT STRUCTURE IN REVIEW COHORT")
print("=" * 80)

print(
    dossiers[
        "account_structure_class"
    ].value_counts()
)


print()
print("=" * 80)
print("DIRECT STATE-FUNDING CONTEXT")
print("=" * 80)

display(
    dossiers[
        dossiers[
            "transition_direct_state_funding_context"
        ] == True
    ][
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "anomaly_class",
            "account_structure_class",

            "organization_balance_delta",

            "previous_report_id",
            "next_report_id",

            "previous_direct_state_funding_amount",
            "previous_budget_expense_amount",
            "next_direct_state_funding_amount",
            "next_budget_expense_amount",

            "review_priority",
        ]
    ]
    .sort_values(
        "organization_balance_delta",
        key=lambda s: s.abs(),
        ascending=False,
    )
)


print()
print("=" * 80)
print("EXACT AMOUNT COINCIDENCES — DIAGNOSTIC ONLY")
print("=" * 80)

if len(matches):

    display(
        matches.merge(
            review_meta[
                [
                    "anomaly_episode_id",
                    "party_name",
                    "organization_name",
                    "anomaly_class",
                    "review_priority",
                ]
            ],
            on="anomaly_episode_id",
            how="left",
        )
        .sort_values(
            [
                "anomaly_episode_id",
                "match_side",
                "match_type",
            ]
        )
    )

else:

    print(
        "No exact payment-amount coincidences."
    )

Review report transactions:   0%|          | 0/66 [00:00<?, ?report/s]


PRIORITY / STATE-FUNDING DOSSIERS
Review episodes: 29
Broken transitions: 37
Broken account rows: 37
Adjacent payment rows inspected: 1300

Transitions with DIRECT state-funding context: 0
Transitions with any exact payment-amount coincidence: 7

ACCOUNT STRUCTURE IN REVIEW COHORT
account_structure_class
same_account_balance_mismatch    24
nonzero_account_disappears        7
nonzero_account_appears           6
Name: count, dtype: int64

DIRECT STATE-FUNDING CONTEXT


,anomaly_episode_id,party_name,organization_name,anomaly_class,account_structure_class,organization_balance_delta,previous_report_id,next_report_id,previous_direct_state_funding_amount,previous_budget_expense_amount,next_direct_state_funding_amount,next_budget_expense_amount,review_priority



EXACT AMOUNT COINCIDENCES — DIAGNOSTIC ONLY


,anomaly_episode_id,previous_report_id,next_report_id,organization_balance_delta,match_side,match_type,matched_report_id,payment_type,matched_amount,payment_row_id,payment_date_raw,party_name,organization_name,anomaly_class,review_priority
0,16,3bcd6c20-0dfd-11ef-8a6c-27bc4724684c,26f64d00-5621-11ef-9197-0fbca96b9f8a,2.00,previous_report,individual_payment_amount,3bcd6c20-0dfd-11ef-8a6c-27bc4724684c,outgoing_expenses,2.00,8334d50f-8507-4b40-987d-a25b80c649fc,2026-02-23 12:58:22,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Кременчуцька міська організація ПП ""Рідне місто""",single_balance_discontinuity,low
1,17,a9ad4470-53dc-11ef-b4cc-1f31c784e768,6082acb0-9b6a-11ef-9197-0fbca96b9f8a,-2043.82,next_report,individual_payment_amount,6082acb0-9b6a-11ef-9197-0fbca96b9f8a,other_incomes,2043.82,175f1a67-13ed-4f1e-9d1c-d4881250016f,2026-02-23 20:34:01,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ЛЬВІВСЬКА ОБЛАСНА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПО...,single_balance_discontinuity,medium
2,17,a9ad4470-53dc-11ef-b4cc-1f31c784e768,6082acb0-9b6a-11ef-9197-0fbca96b9f8a,-2043.82,next_report,payment_category_total,6082acb0-9b6a-11ef-9197-0fbca96b9f8a,other_incomes,2043.82,None,None,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ЛЬВІВСЬКА ОБЛАСНА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПО...,single_balance_discontinuity,medium
7,32,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,-150.00,next_report,individual_payment_amount,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,outgoing_expenses,150.00,33a4af4b-324c-4b5e-b142-b79116d2053c,2025-02-05 14:29:20,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,single_balance_discontinuity,low
8,32,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,-150.00,next_report,individual_payment_amount,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,outgoing_expenses,150.00,8825e250-e913-4216-8119-6328464c5160,2025-02-05 14:29:20,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,single_balance_discontinuity,low
9,32,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,-150.00,next_report,individual_payment_amount,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,outgoing_expenses,150.00,7327248a-48f6-432f-af50-3d6043781180,2025-02-05 14:29:20,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,single_balance_discontinuity,low
10,32,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,-150.00,next_report,individual_payment_amount,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,outgoing_expenses,150.00,bd0253df-8059-42ab-8d88-5094b759c20e,2025-02-05 14:29:20,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,single_balance_discontinuity,low
3,32,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,-150.00,previous_report,individual_payment_amount,58388810-9a91-11ef-ac2d-b9ec46358bc3,outgoing_expenses,150.00,5b9281cb-e491-40d6-bb37-562d5415d317,2026-02-23 20:19:10,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,single_balance_discontinuity,low
4,32,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,-150.00,previous_report,individual_payment_amount,58388810-9a91-11ef-ac2d-b9ec46358bc3,outgoing_expenses,150.00,729c8a54-dbca-4e47-aee1-f10e4002f68d,2026-02-23 20:19:10,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,single_balance_discontinuity,low
5,32,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,-150.00,previous_report,individual_payment_amount,58388810-9a91-11ef-ac2d-b9ec46358bc3,outgoing_expenses,150.00,c747bae1-f9d3-494c-a438-e3b096ae52b7,2026-02-23 20:19:10,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,single_balance_discontinuity,low


In [76]:
from pathlib import Path
import json
import hashlib

import pandas as pd


AUDIT_DIR = Path("data/interim/audit")
RAW_DIR = Path("data/raw/report_details")


# ============================================================
# LOAD EXACT MATCHES
# ============================================================

matches = pd.read_parquet(
    AUDIT_DIR
    / "priority_state_funding_boundary_amount_matches.parquet"
).copy()


print(
    "Exact-match rows:",
    len(matches)
)

print(
    "Distinct transitions with matches:",
    matches[
        [
            "anomaly_episode_id",
            "previous_report_id",
            "next_report_id",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)


# ============================================================
# PAYMENT PATHS
# ============================================================

PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


# ============================================================
# TECHNICAL FIELDS
#
# Excluded ONLY for substantive duplicate detection.
# Raw values remain preserved.
# ============================================================

TECHNICAL_FIELDS = {
    "id",
    "report_id",
    "party_report_id",
    "party_id",
    "office_id",
    "created_at",
    "updated_at",
}


def normalize_for_hash(value):

    if isinstance(value, dict):

        return {
            k: normalize_for_hash(v)
            for k, v in sorted(
                value.items()
            )
            if k not in TECHNICAL_FIELDS
        }


    if isinstance(value, list):

        return [
            normalize_for_hash(v)
            for v in value
        ]


    return value


def substantive_hash(row):

    normalized = normalize_for_hash(
        row
    )

    payload = json.dumps(
        normalized,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
        separators=(",", ":"),
    )

    return hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()


# ============================================================
# LOAD ALL RAW PAYMENT ROWS FROM MATCHED REPORTS
# ============================================================

matched_report_ids = sorted(
    set(
        matches[
            "matched_report_id"
        ].astype(str)
    )
)


raw_rows = []


for report_id in matched_report_ids:

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if not path.exists():

        print(
            "MISSING RAW:",
            report_id
        )

        continue


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = json.load(f)[
            "results"
        ]


    payment_info = (
        detail.get("payment_info")
        or {}
    )


    for payment_type, (
        direction,
        key,
    ) in PAYMENT_PATHS.items():

        rows = (
            (
                payment_info.get(direction)
                or {}
            )
            .get(key)
            or []
        )


        for row in rows:

            raw_rows.append(
                {
                    "report_id":
                        str(report_id),

                    "payment_type":
                        payment_type,

                    "payment_row_id":
                        str(
                            row.get("id")
                        )
                        if row.get("id")
                        is not None
                        else None,

                    "payment_amount":
                        row.get(
                            "payment_amount"
                        ),

                    "substantive_hash":
                        substantive_hash(
                            row
                        ),

                    "raw_json":
                        json.dumps(
                            row,
                            ensure_ascii=False,
                            sort_keys=True,
                            default=str,
                        ),
                }
            )


raw_payments = pd.DataFrame(
    raw_rows
)


# ============================================================
# KEEP ONLY INDIVIDUAL PAYMENT ROWS THAT MATCHED
# ============================================================

matched_individual = (
    matches[
        matches[
            "match_type"
        ]
        ==
        "individual_payment_amount"
    ]
    .copy()
)


matched_individual[
    "payment_row_id"
] = (
    matched_individual[
        "payment_row_id"
    ]
    .astype(str)
)


inspection = (
    matched_individual.merge(
        raw_payments,
        left_on=[
            "matched_report_id",
            "payment_type",
            "payment_row_id",
        ],
        right_on=[
            "report_id",
            "payment_type",
            "payment_row_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ============================================================
# EXPAND ALL RAW SCALAR FIELDS
#
# This lets us discover the actual schema instead of guessing
# which field is the payment date.
# ============================================================

expanded_rows = []


for row in (
    inspection.itertuples(
        index=False
    )
):

    raw = json.loads(
        row.raw_json
    )


    out = {
        "anomaly_episode_id":
            row.anomaly_episode_id,

        "previous_report_id":
            row.previous_report_id,

        "next_report_id":
            row.next_report_id,

        "organization_balance_delta":
            row.organization_balance_delta,

        "match_side":
            row.match_side,

        "matched_report_id":
            row.matched_report_id,

        "payment_type":
            row.payment_type,

        "payment_row_id":
            row.payment_row_id,

        "matched_amount":
            row.matched_amount,

        "substantive_hash":
            row.substantive_hash,
    }


    for key, value in raw.items():

        if isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
                type(None),
            ),
        ):

            out[
                f"raw__{key}"
            ] = value


        else:

            out[
                f"raw__{key}"
            ] = json.dumps(
                value,
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            )


    expanded_rows.append(
        out
    )


expanded = pd.DataFrame(
    expanded_rows
)


# ============================================================
# DISCOVER DATE/TIME-LIKE FIELDS
# ============================================================

date_like_columns = [
    col
    for col in expanded.columns
    if (
        col.startswith("raw__")
        and
        (
            "date" in col.lower()
            or
            "time" in col.lower()
        )
    )
]


print()
print("=" * 80)
print("DATE / TIME FIELDS ACTUALLY PRESENT")
print("=" * 80)

print(
    date_like_columns
)


# ============================================================
# SUBSTANTIVE DUPLICATE ANALYSIS
# ============================================================

duplicate_summary = (
    raw_payments
    .groupby(
        [
            "report_id",
            "payment_type",
            "substantive_hash",
        ],
        as_index=False,
    )
    .agg(
        source_row_count=(
            "payment_row_id",
            "size",
        ),

        source_row_ids=(
            "payment_row_id",
            lambda s:
                " | ".join(
                    str(x)
                    for x in s
                ),
        ),

        payment_amount=(
            "payment_amount",
            "first",
        ),
    )
)


substantive_duplicates = (
    duplicate_summary[
        duplicate_summary[
            "source_row_count"
        ] > 1
    ]
    .copy()
)


# Restrict duplicate QA to matched substantive states.
matched_hashes = set(
    inspection[
        "substantive_hash"
    ].dropna()
)


matched_duplicate_groups = (
    substantive_duplicates[
        substantive_duplicates[
            "substantive_hash"
        ].isin(
            matched_hashes
        )
    ]
    .copy()
)


# ============================================================
# UNIQUE SUBSTANTIVE MATCHES
#
# Several technical IDs may point to one identical
# substantive payment state.
# ============================================================

unique_substantive_matches = (
    inspection
    .sort_values(
        [
            "anomaly_episode_id",
            "matched_report_id",
            "payment_type",
            "substantive_hash",
        ]
    )
    .drop_duplicates(
        [
            "anomaly_episode_id",
            "matched_report_id",
            "payment_type",
            "substantive_hash",
        ]
    )
    .copy()
)


# ============================================================
# SAVE
# ============================================================

expanded.to_parquet(
    AUDIT_DIR
    / "exact_boundary_payment_raw_inspection.parquet",
    index=False,
)

expanded.to_csv(
    AUDIT_DIR
    / "exact_boundary_payment_raw_inspection.csv",
    index=False,
    encoding="utf-8-sig",
)


matched_duplicate_groups.to_parquet(
    AUDIT_DIR
    / "exact_boundary_payment_substantive_duplicates.parquet",
    index=False,
)


unique_substantive_matches.to_parquet(
    AUDIT_DIR
    / "exact_boundary_payment_unique_substantive_matches.parquet",
    index=False,
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 80)
print("SUBSTANTIVE DUPLICATE QA")
print("=" * 80)

print(
    "Matched individual source rows:",
    len(
        inspection
    )
)

print(
    "Distinct matched substantive payment states:",
    len(
        unique_substantive_matches
    )
)

print(
    "Matched substantive states represented by >1 source row:",
    len(
        matched_duplicate_groups
    )
)


if len(
    matched_duplicate_groups
):

    display(
        matched_duplicate_groups
    )


print()
print("=" * 80)
print("RAW MATCHED PAYMENT INSPECTION")
print("=" * 80)


important_cols = [
    "anomaly_episode_id",
    "organization_balance_delta",
    "match_side",
    "matched_report_id",
    "payment_type",
    "payment_row_id",
    "matched_amount",
    "substantive_hash",
]


# Put semantic date/time columns early,
# but keep all other RAW scalar columns too.
ordered_cols = (
    important_cols
    +
    [
        col
        for col in date_like_columns
        if col not in important_cols
    ]
    +
    [
        col
        for col in expanded.columns
        if (
            col not in important_cols
            and
            col not in date_like_columns
        )
    ]
)


display(
    expanded[
        ordered_cols
    ]
    .sort_values(
        [
            "anomaly_episode_id",
            "match_side",
            "payment_type",
            "matched_amount",
        ]
    )
)

Exact-match rows: 23
Distinct transitions with matches: 7

DATE / TIME FIELDS ACTUALLY PRESENT
['raw__payment_instruction_date', 'raw__payment_operation_date', 'raw__refund_date', 'raw__updated_at']

SUBSTANTIVE DUPLICATE QA
Matched individual source rows: 21
Distinct matched substantive payment states: 21
Matched substantive states represented by >1 source row: 0

RAW MATCHED PAYMENT INSPECTION


,anomaly_episode_id,organization_balance_delta,match_side,matched_report_id,payment_type,payment_row_id,matched_amount,substantive_hash,raw__payment_instruction_date,raw__payment_operation_date,raw__refund_date,raw__updated_at,previous_report_id,next_report_id,raw__created_at,raw__group_code,raw__id,raw__office_id,raw__party_id,raw__payer_account_iban,raw__payer_account_type,raw__payer_address,raw__payer_bank_address,raw__payer_bank_code,raw__payer_bank_name,...,raw__payer_type,raw__payment_amount,raw__payment_code,raw__payment_currency,raw__payment_description,raw__payment_number,raw__payment_purpose,raw__payment_reason,raw__payment_type,raw__receiver_account_iban,raw__receiver_account_type,raw__receiver_address,raw__receiver_bank_address,raw__receiver_bank_code,raw__receiver_bank_name,raw__receiver_birthday,raw__receiver_code,raw__receiver_name,raw__receiver_type,raw__refund_amount,raw__refund_budget_amount,raw__refund_description,raw__refund_purpose,raw__refund_reason,raw__report_id
0,16,2.00,previous_report,3bcd6c20-0dfd-11ef-8a6c-27bc4724684c,outgoing_expenses,8334d50f-8507-4b40-987d-a25b80c649fc,2.00,59e2af46e4a7b9ef5e5a0b3794a28ddc2ff6e1725a6233...,None,2024-01-01,None,None,3bcd6c20-0dfd-11ef-8a6c-27bc4724684c,26f64d00-5621-11ef-9197-0fbca96b9f8a,2026-02-23 12:58:22,4_2,8334d50f-8507-4b40-987d-a25b80c649fc,29a20280-5718-42f8-a548-50f1c18f3e0f,a834c9eb-9b06-47a1-880c-15688f705883,None,None,None,None,None,None,...,None,2.00,None,None,None,None,РКО,договір банк.рахунку,None,UA313314670000026002300759801,None,None,None,None,None,None,09331508,"Філія АТ ""Ощадбанк""",Юридична особа,None,None,None,None,None,3bcd6c20-0dfd-11ef-8a6c-27bc4724684c
1,17,-2043.82,next_report,6082acb0-9b6a-11ef-9197-0fbca96b9f8a,other_incomes,175f1a67-13ed-4f1e-9d1c-d4881250016f,2043.82,cdcb06b87fbb41c9bb520f897ae05207dc8735949f2ebb...,None,2024-07-02,None,None,a9ad4470-53dc-11ef-b4cc-1f31c784e768,6082acb0-9b6a-11ef-9197-0fbca96b9f8a,2026-02-23 20:34:01,3_4,175f1a67-13ed-4f1e-9d1c-d4881250016f,2b048ad9-bede-483f-9557-e34907ad38e6,b11839f3-bffe-4f93-ba10-3b877ed181b9,None,None,None,None,None,None,...,Юридична особа,2043.82,None,None,Перерахування залишків коштів при закритті рах...,None,None,None,Перерахування залишків коштів при закритті рах...,UA333052990000026002011039403,None,None,None,None,"АТ КБ ""ПРИВАТБАНК""",None,None,None,None,None,None,None,None,None,6082acb0-9b6a-11ef-9197-0fbca96b9f8a
6,32,-150.00,next_report,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,outgoing_expenses,33a4af4b-324c-4b5e-b142-b79116d2053c,150.00,3fdcb6c9ed7f306ef19ac1525cebfb166f91169fb76683...,None,2024-12-25,None,None,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,2025-02-05 14:29:20,4_2,33a4af4b-324c-4b5e-b142-b79116d2053c,43b38ed0-474d-4668-aed0-15a8cab99747,7d9a4591-6cea-451f-8545-b211f5e7b9bd,None,None,None,None,None,None,...,None,150.00,None,None,None,None,Комісія за обслуговування поточного рахунку за...,Договір №90.23-СВА від 26.07.2023 року,None,UA083805820000026008010333787,None,None,None,None,None,None,[конфіденційна інформація],АТ МІБ,Юридична особа,None,None,None,None,None,3827b2d0-e3bb-11ef-82bf-3b31b6f46948
7,32,-150.00,next_report,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,outgoing_expenses,8825e250-e913-4216-8119-6328464c5160,150.00,057868c3e71c3df2f230bf20915e4da66a00acd8c171f7...,None,2024-10-25,None,None,58388810-9a91-11ef-ac2d-b9ec46358bc3,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,2025-02-05 14:29:20,4_2,8825e250-e913-4216-8119-6328464c5160,43b38ed0-474d-4668-aed0-15a8cab99747,7d9a4591-6cea-451f-8545-b211f5e7b9bd,None,None,None,None,None,None,...,None,150.00,None,None,None,None,Комісія -абонентська плата за виконання операц...,Договір №90.23-СВА від 26.07.2023 року,None,UA083805820000026008010333787,None,None,None,None,None,None,[конфіденційна інформація],АТ МІБ,Юридична особа,None,None,None,None,None,3827b2d0-e3bb-11ef-82bf-3b31b6f46948
8,32,-150.00,next_report,3827b2d0-e3bb-11ef-82bf-3b31b6f46948,outgoing_expenses,7327248a-48f6-432f-af50

In [77]:
from pathlib import Path
import re

import pandas as pd


AUDIT_DIR = Path("data/interim/audit")
REPORTS_DIR = Path("data/interim/reports")


# ============================================================
# LOAD
# ============================================================

raw_matches = pd.read_parquet(
    AUDIT_DIR
    / "exact_boundary_payment_raw_inspection.parquet"
).copy()

breaks = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_breaks_with_episode_id.parquet"
).copy()

classification = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_enriched.parquet"
).copy()

all_reports = pd.read_parquet(
    REPORTS_DIR
    / "all_reports_manifest.parquet"
).copy()


for df in [
    raw_matches,
    breaks,
    classification,
    all_reports,
]:
    for col in [
        "report_id",
        "matched_report_id",
        "previous_report_id",
        "next_report_id",
        "organization_id",
        "root_party_id",
    ]:
        if col in df.columns:
            df[col] = df[col].astype(str)


# ============================================================
# REPORT PERIOD LOOKUP
# ============================================================

report_period = (
    all_reports[
        [
            "report_id",
            "year",
            "quarter",
        ]
    ]
    .drop_duplicates(
        "report_id"
    )
    .rename(
        columns={
            "report_id":
                "matched_report_id",

            "year":
                "matched_report_year",

            "quarter":
                "matched_report_quarter",
        }
    )
)


raw_matches = raw_matches.merge(
    report_period,
    on="matched_report_id",
    how="left",
    validate="many_to_one",
)


# ============================================================
# REAL OPERATION DATE
# ============================================================

raw_matches[
    "payment_operation_date"
] = pd.to_datetime(
    raw_matches[
        "raw__payment_operation_date"
    ],
    errors="coerce",
)


raw_matches[
    "operation_year"
] = raw_matches[
    "payment_operation_date"
].dt.year


raw_matches[
    "operation_quarter"
] = raw_matches[
    "payment_operation_date"
].dt.quarter


raw_matches[
    "operation_inside_report_period"
] = (
    (
        raw_matches[
            "operation_year"
        ]
        ==
        raw_matches[
            "matched_report_year"
        ]
    )
    &
    (
        raw_matches[
            "operation_quarter"
        ]
        ==
        raw_matches[
            "matched_report_quarter"
        ]
    )
)


# ============================================================
# TEXT EVIDENCE
# ============================================================

TEXT_FIELDS = [
    "raw__payment_description",
    "raw__payment_purpose",
    "raw__payment_reason",
]


def combined_text(row):

    values = []

    for col in TEXT_FIELDS:

        value = row.get(col)

        if pd.notna(value):
            values.append(
                str(value)
            )

    return " | ".join(
        values
    ).lower()


raw_matches[
    "payment_text"
] = raw_matches.apply(
    combined_text,
    axis=1,
)


raw_matches[
    "mentions_balance"
] = raw_matches[
    "payment_text"
].str.contains(
    r"залишк",
    regex=True,
    na=False,
)


raw_matches[
    "mentions_transfer"
] = raw_matches[
    "payment_text"
].str.contains(
    r"перерах",
    regex=True,
    na=False,
)


raw_matches[
    "mentions_account_closure"
] = raw_matches[
    "payment_text"
].str.contains(
    r"закрит",
    regex=True,
    na=False,
)


raw_matches[
    "mentions_transit_account"
] = raw_matches[
    "payment_text"
].str.contains(
    r"транзит",
    regex=True,
    na=False,
)


raw_matches[
    "mentions_bank_fee"
] = raw_matches[
    "payment_text"
].str.contains(
    r"комісі|абонент",
    regex=True,
    na=False,
)


raw_matches[
    "transfer_semantic_evidence"
] = (
    raw_matches[
        "mentions_balance"
    ]
    |
    raw_matches[
        "mentions_transfer"
    ]
    |
    raw_matches[
        "mentions_account_closure"
    ]
    |
    raw_matches[
        "mentions_transit_account"
    ]
)


# ============================================================
# MECHANICAL RELATION TO THE BOUNDARY GAP
#
# delta = next opening - previous closing
#
# Strong patterns:
#
# A)
# delta < 0
# + next-period INFLOW of abs(delta)
#
# Previous closing amount is missing from next opening,
# but re-enters as a reported receipt.
#
# B)
# delta > 0
# + previous-period OUTFLOW of delta
#
# Amount leaves during previous quarter but nevertheless
# appears in next-quarter opening balance.
#
# These are diagnostic patterns, not conclusions.
# ============================================================

INFLOW_TYPES = {
    "monetary_contributions",
    "state_funding",
    "other_incomes",
}


OUTFLOW_TYPES = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


def mechanical_pattern(row):

    delta = float(
        row[
            "organization_balance_delta"
        ]
    )

    payment_type = row[
        "payment_type"
    ]

    side = row[
        "match_side"
    ]


    if (
        delta < 0
        and
        side == "next_report"
        and
        payment_type in INFLOW_TYPES
    ):

        return (
            "missing_opening_amount_reappears_as_next_period_inflow"
        )


    if (
        delta > 0
        and
        side == "previous_report"
        and
        payment_type in OUTFLOW_TYPES
    ):

        return (
            "previous_period_outflow_reappears_in_next_opening"
        )


    return (
        "amount_coincidence_without_boundary_flow_pattern"
    )


raw_matches[
    "boundary_flow_pattern"
] = raw_matches.apply(
    mechanical_pattern,
    axis=1,
)


# ============================================================
# EVIDENCE STRENGTH
# ============================================================

def evidence_strength(row):

    mechanical = (
        row[
            "boundary_flow_pattern"
        ]
        !=
        "amount_coincidence_without_boundary_flow_pattern"
    )


    semantic = bool(
        row[
            "transfer_semantic_evidence"
        ]
    )


    bank_fee = bool(
        row[
            "mentions_bank_fee"
        ]
    )


    inside_period = bool(
        row[
            "operation_inside_report_period"
        ]
    )


    if (
        mechanical
        and
        semantic
        and
        inside_period
    ):

        return "strong"


    if (
        mechanical
        and
        inside_period
        and
        not bank_fee
    ):

        return "medium"


    return "weak"


raw_matches[
    "boundary_evidence_strength"
] = raw_matches.apply(
    evidence_strength,
    axis=1,
)


# ============================================================
# COLLAPSE TO TRANSITION LEVEL
# ============================================================

transition_evidence_rows = []


group_cols = [
    "anomaly_episode_id",
    "previous_report_id",
    "next_report_id",
    "organization_balance_delta",
]


for keys, df in raw_matches.groupby(
    group_cols,
    dropna=False,
):

    (
        episode_id,
        previous_report_id,
        next_report_id,
        delta,
    ) = keys


    strong = df[
        df[
            "boundary_evidence_strength"
        ] == "strong"
    ]


    medium = df[
        df[
            "boundary_evidence_strength"
        ] == "medium"
    ]


    mechanical = df[
        df[
            "boundary_flow_pattern"
        ]
        !=
        "amount_coincidence_without_boundary_flow_pattern"
    ]


    if len(strong):

        transition_class = (
            "boundary_transfer_reclassification_candidate"
        )

        transition_evidence_strength = (
            "strong"
        )


    elif len(medium):

        transition_class = (
            "boundary_flow_reclassification_candidate"
        )

        transition_evidence_strength = (
            "medium"
        )


    else:

        transition_class = (
            "amount_coincidence_only"
        )

        transition_evidence_strength = (
            "weak"
        )


    transition_evidence_rows.append(
        {
            "anomaly_episode_id":
                int(episode_id),

            "previous_report_id":
                str(previous_report_id),

            "next_report_id":
                str(next_report_id),

            "organization_balance_delta":
                float(delta),

            "transition_boundary_class":
                transition_class,

            "transition_evidence_strength":
                transition_evidence_strength,

            "matched_source_rows":
                len(df),

            "mechanical_pattern_rows":
                len(mechanical),

            "strong_evidence_rows":
                len(strong),

            "medium_evidence_rows":
                len(medium),

            "operation_dates_json":
                (
                    df[
                        "payment_operation_date"
                    ]
                    .dropna()
                    .dt.strftime(
                        "%Y-%m-%d"
                    )
                    .drop_duplicates()
                    .to_json(
                        orient="values",
                        force_ascii=False,
                    )
                ),

            "payment_types_json":
                (
                    df[
                        "payment_type"
                    ]
                    .drop_duplicates()
                    .to_json(
                        orient="values",
                        force_ascii=False,
                    )
                ),

            "payment_text_examples":
                " || ".join(
                    df[
                        "payment_text"
                    ]
                    .dropna()
                    .drop_duplicates()
                    .tolist()[:5]
                ),
        }
    )


transition_evidence = pd.DataFrame(
    transition_evidence_rows
)


# ============================================================
# ADD PARTY / ORGANIZATION METADATA
# ============================================================

meta = classification[
    [
        "anomaly_episode_id",
        "party_name",
        "organization_name",
        "anomaly_class",
        "review_priority",
        "party_state_funding_exposure",
    ]
].drop_duplicates(
    "anomaly_episode_id"
)


transition_evidence = (
    transition_evidence.merge(
        meta,
        on="anomaly_episode_id",
        how="left",
    )
)


# ============================================================
# SAVE
# ============================================================

raw_matches.to_parquet(
    AUDIT_DIR
    / "boundary_amount_matches_classified.parquet",
    index=False,
)


transition_evidence.to_parquet(
    AUDIT_DIR
    / "boundary_transfer_reclassification_candidates.parquet",
    index=False,
)


transition_evidence.to_csv(
    AUDIT_DIR
    / "boundary_transfer_reclassification_candidates.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 80)
print("BOUNDARY PAYMENT EVIDENCE")
print("=" * 80)

print(
    "Matched payment rows:",
    len(
        raw_matches
    )
)

print(
    "Transitions:",
    len(
        transition_evidence
    )
)

print()


print(
    "EVIDENCE STRENGTH — PAYMENT ROWS"
)

print(
    raw_matches[
        "boundary_evidence_strength"
    ].value_counts()
)

print()


print(
    "TRANSITION CLASSIFICATION"
)

print(
    transition_evidence[
        "transition_boundary_class"
    ].value_counts()
)


print()
print("=" * 80)
print("STRONG / MEDIUM BOUNDARY CANDIDATES")
print("=" * 80)

display(
    transition_evidence[
        transition_evidence[
            "transition_evidence_strength"
        ].isin(
            [
                "strong",
                "medium",
            ]
        )
    ][
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "organization_balance_delta",

            "transition_boundary_class",
            "transition_evidence_strength",

            "operation_dates_json",
            "payment_types_json",

            "payment_text_examples",

            "anomaly_class",
            "review_priority",
        ]
    ]
    .sort_values(
        "organization_balance_delta",
        key=lambda s:
            s.abs(),
        ascending=False,
    )
)


print()
print("=" * 80)
print("WEAK / LIKELY INCIDENTAL AMOUNT COINCIDENCES")
print("=" * 80)

display(
    transition_evidence[
        transition_evidence[
            "transition_evidence_strength"
        ] == "weak"
    ][
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",
            "organization_balance_delta",
            "matched_source_rows",
            "payment_types_json",
            "payment_text_examples",
        ]
    ]
)


BOUNDARY PAYMENT EVIDENCE
Matched payment rows: 21
Transitions: 7

EVIDENCE STRENGTH — PAYMENT ROWS
boundary_evidence_strength
weak      17
strong     3
medium     1
Name: count, dtype: int64

TRANSITION CLASSIFICATION
transition_boundary_class
boundary_transfer_reclassification_candidate    3
amount_coincidence_only                         3
boundary_flow_reclassification_candidate        1
Name: count, dtype: int64

STRONG / MEDIUM BOUNDARY CANDIDATES


,anomaly_episode_id,party_name,organization_name,organization_balance_delta,transition_boundary_class,transition_evidence_strength,operation_dates_json,payment_types_json,payment_text_examples,anomaly_class,review_priority
5,101,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,-27761.16,boundary_transfer_reclassification_candidate,strong,"[""2026-04-24""]","[""other_incomes""]",перерахування залишку коштів на рахунок,single_balance_discontinuity,high
6,129,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,14812.94,boundary_transfer_reclassification_candidate,strong,"[""2025-05-15""]","[""outgoing_expenses""]",перерахування коштів на транзитний рахунок | в...,single_balance_discontinuity,high
1,17,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ЛЬВІВСЬКА ОБЛАСНА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПО...,-2043.82,boundary_transfer_reclassification_candidate,strong,"[""2024-07-02""]","[""other_incomes""]",перерахування залишків коштів при закритті рах...,single_balance_discontinuity,medium
0,16,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Кременчуцька міська організація ПП ""Рідне місто""",2.00,boundary_flow_reclassification_candidate,medium,"[""2024-01-01""]","[""outgoing_expenses""]",рко | договір банк.рахунку,single_balance_discontinuity,low



WEAK / LIKELY INCIDENTAL AMOUNT COINCIDENCES


,anomaly_episode_id,party_name,organization_name,organization_balance_delta,matched_source_rows,payment_types_json,payment_text_examples
2,32,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЛУГАНСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ...,-150.0,8,"[""outgoing_expenses""]",комісія -абонентська плата за виконання операц...
3,37,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ДНІПРОПЕТРОВСЬКА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПОЛ...,-150.0,6,"[""outgoing_expenses""]",комісія - абонентська плата за виконання опера...
4,89,ПОЛІТИЧНА ПАРТІЯ «ПРОПОЗИЦІЯ»,РІВНЕНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,-225.0,3,"[""outgoing_expenses""]",комісія з обслуговування рахунку за грудень **...


In [78]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd


AUDIT_DIR = Path("data/interim/audit")
RAW_DIR = Path("data/raw/report_details")


# ============================================================
# LOAD
# ============================================================

matches = pd.read_parquet(
    AUDIT_DIR
    / "boundary_amount_matches_classified.parquet"
).copy()

transition_evidence = pd.read_parquet(
    AUDIT_DIR
    / "boundary_transfer_reclassification_candidates.parquet"
).copy()


# Only rows belonging to strong / medium transition candidates.
keep_transitions = transition_evidence[
    transition_evidence[
        "transition_evidence_strength"
    ].isin(
        ["strong", "medium"]
    )
][
    [
        "anomaly_episode_id",
        "previous_report_id",
        "next_report_id",
    ]
].drop_duplicates()


inspect = matches.merge(
    keep_transitions,
    on=[
        "anomaly_episode_id",
        "previous_report_id",
        "next_report_id",
    ],
    how="inner",
)


# ============================================================
# ROBUST UKRAINIAN IBAN NORMALIZATION
# ============================================================

def valid_ua_iban(value):

    if not isinstance(value, str):
        return False

    if not re.fullmatch(
        r"UA\d{27}",
        value,
    ):
        return False

    rearranged = (
        value[4:]
        + value[:4]
    )

    remainder = 0

    for ch in rearranged:

        if ch.isdigit():
            digits = ch
        else:
            digits = str(
                ord(ch) - 55
            )

        for digit in digits:

            remainder = (
                remainder * 10
                + int(digit)
            ) % 97

    return remainder == 1


def canonical_ua_iban(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).upper()

    text = (
        text
        .replace("\u200b", "")
        .replace("\u200c", "")
        .replace("\u200d", "")
        .replace("\u2060", "")
        .replace("\ufeff", "")
    )

    pos = text.find("UA")

    if pos < 0:
        return None

    text = text[pos:]

    digits = []

    for ch in text[2:]:

        if ch.isdigit():

            digits.append(ch)

            if len(digits) == 27:
                break

        elif (
            ch.isspace()
            or ch in "-._/\\:"
        ):
            continue

        else:
            break

    if len(digits) != 27:
        return None

    candidate = (
        "UA"
        + "".join(digits)
    )

    if valid_ua_iban(
        candidate
    ):
        return candidate

    return None


# ============================================================
# LOAD REPORTED ACCOUNT SET FOR EACH ADJACENT REPORT
# ============================================================

report_ids = sorted(
    set(
        inspect[
            "previous_report_id"
        ].astype(str)
    )
    |
    set(
        inspect[
            "next_report_id"
        ].astype(str)
    )
)


report_accounts = {}


for report_id in report_ids:

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    accounts = set()

    if path.exists():

        with path.open(
            "r",
            encoding="utf-8",
        ) as f:

            detail = json.load(f)[
                "results"
            ]

        rows = (
            (
                detail.get(
                    "properties"
                )
                or {}
            )
            .get(
                "property_moneys"
            )
            or []
        )

        for row in rows:

            iban = canonical_ua_iban(
                row.get(
                    "account_number"
                )
            )

            if iban:
                accounts.add(
                    iban
                )

    report_accounts[
        report_id
    ] = accounts


# ============================================================
# PAYMENT IBANS
# ============================================================

inspect[
    "payer_iban"
] = inspect[
    "raw__payer_account_iban"
].apply(
    canonical_ua_iban
)


inspect[
    "receiver_iban"
] = inspect[
    "raw__receiver_account_iban"
].apply(
    canonical_ua_iban
)


# ============================================================
# LINK PAYMENT IBANS TO REPORTED ACCOUNT SETS
# ============================================================

rows = []


for row in inspect.itertuples(
    index=False
):

    prev_id = str(
        row.previous_report_id
    )

    next_id = str(
        row.next_report_id
    )

    prev_accounts = report_accounts.get(
        prev_id,
        set(),
    )

    next_accounts = report_accounts.get(
        next_id,
        set(),
    )

    payer = row.payer_iban
    receiver = row.receiver_iban


    payer_in_prev = (
        payer in prev_accounts
        if payer
        else False
    )

    payer_in_next = (
        payer in next_accounts
        if payer
        else False
    )

    receiver_in_prev = (
        receiver in prev_accounts
        if receiver
        else False
    )

    receiver_in_next = (
        receiver in next_accounts
        if receiver
        else False
    )


    # --------------------------------------------------------
    # ACCOUNT-TOPOLOGY INTERPRETATION
    # --------------------------------------------------------

    if (
        row.boundary_evidence_strength
        == "strong"
    ):

        if (
            row.match_side
            == "next_report"
            and
            receiver_in_next
        ):

            topology = (
                "strong_transfer_into_reported_next_account"
            )


        elif (
            row.match_side
            == "previous_report"
            and
            payer_in_prev
            and
            receiver_in_next
        ):

            topology = (
                "strong_direct_reported_account_transfer"
            )


        elif (
            row.match_side
            == "previous_report"
            and
            payer_in_prev
        ):

            topology = (
                "strong_outflow_from_reported_previous_account"
            )


        elif (
            row.match_side
            == "previous_report"
            and
            receiver is not None
            and
            receiver not in prev_accounts
            and
            receiver not in next_accounts
        ):

            topology = (
                "strong_transfer_to_unreported_or_transit_account"
            )


        else:

            topology = (
                "strong_semantic_match_without_account_link"
            )


    else:

        topology = (
            "medium_amount_flow_candidate"
        )


    rows.append(
        {
            "anomaly_episode_id":
                row.anomaly_episode_id,

            "previous_report_id":
                prev_id,

            "next_report_id":
                next_id,

            "organization_balance_delta":
                row.organization_balance_delta,

            "match_side":
                row.match_side,

            "payment_type":
                row.payment_type,

            "payment_amount":
                row.matched_amount,

            "payment_operation_date":
                row.payment_operation_date,

            "payment_text":
                row.payment_text,

            "boundary_evidence_strength":
                row.boundary_evidence_strength,

            "payer_iban":
                payer,

            "receiver_iban":
                receiver,

            "previous_report_accounts":
                json.dumps(
                    sorted(
                        prev_accounts
                    ),
                    ensure_ascii=False,
                ),

            "next_report_accounts":
                json.dumps(
                    sorted(
                        next_accounts
                    ),
                    ensure_ascii=False,
                ),

            "payer_in_previous_accounts":
                payer_in_prev,

            "payer_in_next_accounts":
                payer_in_next,

            "receiver_in_previous_accounts":
                receiver_in_prev,

            "receiver_in_next_accounts":
                receiver_in_next,

            "payment_account_topology":
                topology,
        }
    )


account_links = pd.DataFrame(
    rows
)


# ============================================================
# COLLAPSE TO TRANSITION LEVEL
# ============================================================

transition_rows = []


for keys, df in account_links.groupby(
    [
        "anomaly_episode_id",
        "previous_report_id",
        "next_report_id",
        "organization_balance_delta",
    ],
    dropna=False,
):

    (
        episode_id,
        previous_report_id,
        next_report_id,
        delta,
    ) = keys


    strong = df[
        df[
            "boundary_evidence_strength"
        ] == "strong"
    ]


    topologies = (
        df[
            "payment_account_topology"
        ]
        .drop_duplicates()
        .tolist()
    )


    account_linked = bool(
        (
            df[
                "payer_in_previous_accounts"
            ]
            |
            df[
                "payer_in_next_accounts"
            ]
            |
            df[
                "receiver_in_previous_accounts"
            ]
            |
            df[
                "receiver_in_next_accounts"
            ]
        ).any()
    )


    if (
        len(strong)
        and
        account_linked
    ):

        final_evidence = (
            "strong_account_linked"
        )


    elif len(strong):

        final_evidence = (
            "strong_semantic_only"
        )


    else:

        final_evidence = (
            "medium"
        )


    transition_rows.append(
        {
            "anomaly_episode_id":
                int(episode_id),

            "previous_report_id":
                str(previous_report_id),

            "next_report_id":
                str(next_report_id),

            "organization_balance_delta":
                float(delta),

            "boundary_account_evidence":
                final_evidence,

            "payment_account_topologies":
                " | ".join(
                    topologies
                ),

            "payment_row_count":
                len(df),

            "has_reported_account_link":
                account_linked,

            "payer_matches_previous_account":
                bool(
                    df[
                        "payer_in_previous_accounts"
                    ].any()
                ),

            "receiver_matches_next_account":
                bool(
                    df[
                        "receiver_in_next_accounts"
                    ].any()
                ),
        }
    )


transition_account_evidence = pd.DataFrame(
    transition_rows
)


# ============================================================
# ADD METADATA
# ============================================================

meta = transition_evidence[
    [
        "anomaly_episode_id",
        "party_name",
        "organization_name",
        "transition_boundary_class",
        "transition_evidence_strength",
        "anomaly_class",
        "review_priority",
    ]
].drop_duplicates(
    "anomaly_episode_id"
)


transition_account_evidence = (
    transition_account_evidence.merge(
        meta,
        on="anomaly_episode_id",
        how="left",
    )
)


# ============================================================
# SAVE
# ============================================================

account_links.to_parquet(
    AUDIT_DIR
    / "boundary_transfer_account_links.parquet",
    index=False,
)


transition_account_evidence.to_parquet(
    AUDIT_DIR
    / "boundary_transfer_account_evidence.parquet",
    index=False,
)


transition_account_evidence.to_csv(
    AUDIT_DIR
    / "boundary_transfer_account_evidence.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 80)
print("BOUNDARY TRANSFER — ACCOUNT LINKAGE")
print("=" * 80)

print(
    transition_account_evidence[
        "boundary_account_evidence"
    ].value_counts()
)


print()
print("=" * 80)
print("TRANSITION-LEVEL RESULTS")
print("=" * 80)

display(
    transition_account_evidence[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "organization_balance_delta",

            "transition_boundary_class",
            "transition_evidence_strength",
            "boundary_account_evidence",

            "payment_account_topologies",

            "payer_matches_previous_account",
            "receiver_matches_next_account",

            "anomaly_class",
            "review_priority",
        ]
    ]
    .sort_values(
        "organization_balance_delta",
        key=lambda s:
            s.abs(),
        ascending=False,
    )
)


print()
print("=" * 80)
print("PAYMENT-LEVEL ACCOUNT LINKS")
print("=" * 80)

display(
    account_links[
        [
            "anomaly_episode_id",
            "organization_balance_delta",

            "match_side",
            "payment_type",
            "payment_amount",
            "payment_operation_date",

            "payer_iban",
            "receiver_iban",

            "payer_in_previous_accounts",
            "payer_in_next_accounts",
            "receiver_in_previous_accounts",
            "receiver_in_next_accounts",

            "payment_account_topology",
            "payment_text",
        ]
    ]
)


BOUNDARY TRANSFER — ACCOUNT LINKAGE
boundary_account_evidence
strong_account_linked    3
medium                   1
Name: count, dtype: int64

TRANSITION-LEVEL RESULTS


,anomaly_episode_id,party_name,organization_name,organization_balance_delta,transition_boundary_class,transition_evidence_strength,boundary_account_evidence,payment_account_topologies,payer_matches_previous_account,receiver_matches_next_account,anomaly_class,review_priority
2,101,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,-27761.16,boundary_transfer_reclassification_candidate,strong,strong_account_linked,strong_transfer_into_reported_next_account,False,True,single_balance_discontinuity,high
3,129,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,14812.94,boundary_transfer_reclassification_candidate,strong,strong_account_linked,strong_semantic_match_without_account_link,False,False,single_balance_discontinuity,high
1,17,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ЛЬВІВСЬКА ОБЛАСНА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПО...,-2043.82,boundary_transfer_reclassification_candidate,strong,strong_account_linked,strong_transfer_into_reported_next_account,False,True,single_balance_discontinuity,medium
0,16,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Кременчуцька міська організація ПП ""Рідне місто""",2.00,boundary_flow_reclassification_candidate,medium,medium,medium_amount_flow_candidate,False,True,single_balance_discontinuity,low



PAYMENT-LEVEL ACCOUNT LINKS


,anomaly_episode_id,organization_balance_delta,match_side,payment_type,payment_amount,payment_operation_date,payer_iban,receiver_iban,payer_in_previous_accounts,payer_in_next_accounts,receiver_in_previous_accounts,receiver_in_next_accounts,payment_account_topology,payment_text
0,16,2.00,previous_report,outgoing_expenses,2.00,2024-01-01,None,UA313314670000026002300759801,False,False,True,True,medium_amount_flow_candidate,рко | договір банк.рахунку
1,17,-2043.82,next_report,other_incomes,2043.82,2024-07-02,None,UA333052990000026002011039403,False,False,True,True,strong_transfer_into_reported_next_account,перерахування залишків коштів при закритті рах...
2,101,-27761.16,next_report,other_incomes,27761.16,2026-04-24,None,UA563534890000026006111113979,False,False,False,True,strong_transfer_into_reported_next_account,перерахування залишку коштів на рахунок
3,129,14812.94,previous_report,outgoing_expenses,14812.94,2025-05-15,None,UA863052990000026003031103738,False,False,True,False,strong_semantic_match_without_account_link,перерахування коштів на транзитний рахунок | в...


In [79]:
from pathlib import Path

import pandas as pd


AUDIT_DIR = Path("data/interim/audit")


# ============================================================
# LOAD
# ============================================================

account_links = pd.read_parquet(
    AUDIT_DIR
    / "boundary_transfer_account_links.parquet"
).copy()

transition_evidence = pd.read_parquet(
    AUDIT_DIR
    / "boundary_transfer_reclassification_candidates.parquet"
).copy()

master = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_enriched.parquet"
).copy()


# ============================================================
# DIRECTIONAL ACCOUNT LINK
#
# Important:
#
# next-report inflow:
#   receiver must be a reported NEXT-period account
#
# previous-report outflow:
#   payer must be a reported PREVIOUS-period account
#
# Receiver being present in the previous report is NOT enough
# to prove directionally linked outflow.
# ============================================================

INFLOW_TYPES = {
    "monetary_contributions",
    "state_funding",
    "other_incomes",
}

OUTFLOW_TYPES = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


def directional_link(row):

    if (
        row["match_side"] == "next_report"
        and
        row["payment_type"] in INFLOW_TYPES
        and
        bool(
            row["receiver_in_next_accounts"]
        )
    ):
        return True


    if (
        row["match_side"] == "previous_report"
        and
        row["payment_type"] in OUTFLOW_TYPES
        and
        bool(
            row["payer_in_previous_accounts"]
        )
    ):
        return True


    return False


account_links[
    "directional_account_link"
] = account_links.apply(
    directional_link,
    axis=1,
)


# ============================================================
# MORE PRECISE TOPOLOGY
# ============================================================

def refined_topology(row):

    if (
        row["match_side"] == "next_report"
        and
        row["payment_type"] in INFLOW_TYPES
    ):

        if row[
            "receiver_in_next_accounts"
        ]:

            return (
                "inflow_to_reported_next_account"
            )

        return (
            "inflow_without_next_account_link"
        )


    if (
        row["match_side"] == "previous_report"
        and
        row["payment_type"] in OUTFLOW_TYPES
    ):

        if row[
            "payer_in_previous_accounts"
        ]:

            return (
                "outflow_from_reported_previous_account"
            )


        if (
            row[
                "receiver_in_previous_accounts"
            ]
            and
            not row[
                "receiver_in_next_accounts"
            ]
        ):

            return (
                "outflow_receiver_matches_previous_account_only"
            )


        if (
            row["receiver_iban"] is not None
            and
            not row[
                "receiver_in_previous_accounts"
            ]
            and
            not row[
                "receiver_in_next_accounts"
            ]
        ):

            return (
                "outflow_to_unreported_account"
            )


        return (
            "outflow_without_directional_account_link"
        )


    return (
        "other_payment_pattern"
    )


account_links[
    "refined_payment_account_topology"
] = account_links.apply(
    refined_topology,
    axis=1,
)


# ============================================================
# COLLAPSE TO TRANSITION
# ============================================================

transition_rows = []


GROUP_COLS = [
    "anomaly_episode_id",
    "previous_report_id",
    "next_report_id",
    "organization_balance_delta",
]


for keys, df in account_links.groupby(
    GROUP_COLS,
    dropna=False,
):

    (
        episode_id,
        previous_report_id,
        next_report_id,
        delta,
    ) = keys


    strong = df[
        df[
            "boundary_evidence_strength"
        ] == "strong"
    ]

    medium = df[
        df[
            "boundary_evidence_strength"
        ] == "medium"
    ]


    strong_directional = strong[
        strong[
            "directional_account_link"
        ] == True
    ]


    if len(strong_directional):

        evidence_level = (
            "strong_directional_account_link"
        )

        refined_mechanism = (
            "boundary_transfer_candidate_directionally_linked"
        )


    elif len(strong):

        evidence_level = (
            "strong_semantic_only"
        )

        refined_mechanism = (
            "boundary_transfer_candidate_semantic_only"
        )


    elif len(medium):

        evidence_level = (
            "medium_amount_flow"
        )

        refined_mechanism = (
            "boundary_flow_candidate_medium"
        )


    else:

        evidence_level = (
            "weak"
        )

        refined_mechanism = (
            "amount_coincidence_only"
        )


    transition_rows.append(
        {
            "anomaly_episode_id":
                int(episode_id),

            "previous_report_id":
                str(previous_report_id),

            "next_report_id":
                str(next_report_id),

            "organization_balance_delta":
                float(delta),

            "boundary_evidence_level":
                evidence_level,

            "refined_boundary_mechanism":
                refined_mechanism,

            "directionally_linked_payment_rows":
                int(
                    df[
                        "directional_account_link"
                    ].sum()
                ),

            "strong_directionally_linked_rows":
                len(
                    strong_directional
                ),

            "strong_semantic_rows":
                len(strong),

            "medium_rows":
                len(medium),

            "refined_account_topologies":
                " | ".join(
                    df[
                        "refined_payment_account_topology"
                    ]
                    .drop_duplicates()
                    .tolist()
                ),
        }
    )


refined = pd.DataFrame(
    transition_rows
)


# ============================================================
# ADD METADATA
# ============================================================

meta = transition_evidence[
    [
        "anomaly_episode_id",
        "party_name",
        "organization_name",
        "transition_boundary_class",
        "transition_evidence_strength",
        "anomaly_class",
        "review_priority",
    ]
].drop_duplicates(
    "anomaly_episode_id"
)


refined = refined.merge(
    meta,
    on="anomaly_episode_id",
    how="left",
)


# ============================================================
# EPISODE-LEVEL BOUNDARY EVIDENCE
# ============================================================

rank = {
    "strong_directional_account_link": 3,
    "strong_semantic_only": 2,
    "medium_amount_flow": 1,
    "weak": 0,
}


refined[
    "_boundary_rank"
] = refined[
    "boundary_evidence_level"
].map(rank)


episode_boundary = (
    refined
    .sort_values(
        [
            "anomaly_episode_id",
            "_boundary_rank",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .groupby(
        "anomaly_episode_id",
        as_index=False,
    )
    .first()
)


episode_boundary = episode_boundary[
    [
        "anomaly_episode_id",
        "boundary_evidence_level",
        "refined_boundary_mechanism",
        "refined_account_topologies",
    ]
]


# ============================================================
# ENRICH MASTER REGISTRY
# ============================================================

master_refined = master.merge(
    episode_boundary,
    on="anomaly_episode_id",
    how="left",
    validate="one_to_one",
)


master_refined[
    "has_boundary_transfer_evidence"
] = (
    master_refined[
        "boundary_evidence_level"
    ].isin(
        [
            "strong_directional_account_link",
            "strong_semantic_only",
            "medium_amount_flow",
        ]
    )
)


# Keep primary anomaly classification intact.
# Add a separate analytical mechanism layer.
master_refined[
    "analytical_mechanism"
] = master_refined[
    "anomaly_class"
]


mask = (
    master_refined[
        "refined_boundary_mechanism"
    ].notna()
)


master_refined.loc[
    mask,
    "analytical_mechanism",
] = master_refined.loc[
    mask,
    "refined_boundary_mechanism",
]


# Never overwrite the stronger version-integrity diagnosis.
version_mask = (
    master_refined[
        "anomaly_class"
    ]
    ==
    "version_integrity_restored_by_alternative"
)


master_refined.loc[
    version_mask,
    "analytical_mechanism",
] = (
    "version_integrity_restored_by_alternative"
)


# ============================================================
# SAVE
# ============================================================

account_links.to_parquet(
    AUDIT_DIR
    / "boundary_transfer_account_links_refined.parquet",
    index=False,
)


refined.drop(
    columns=[
        "_boundary_rank"
    ]
).to_parquet(
    AUDIT_DIR
    / "boundary_transfer_account_evidence_refined.parquet",
    index=False,
)


master_refined.to_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_refined.parquet",
    index=False,
)


master_refined.to_csv(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_refined.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 80)
print("REFINED BOUNDARY EVIDENCE")
print("=" * 80)

print(
    refined[
        "boundary_evidence_level"
    ].value_counts()
)


print()
print("=" * 80)
print("REFINED TRANSITION RESULTS")
print("=" * 80)

display(
    refined[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "organization_balance_delta",

            "boundary_evidence_level",
            "refined_boundary_mechanism",
            "refined_account_topologies",

            "directionally_linked_payment_rows",

            "anomaly_class",
            "review_priority",
        ]
    ]
    .sort_values(
        "organization_balance_delta",
        key=lambda s:
            s.abs(),
        ascending=False,
    )
)


REFINED BOUNDARY EVIDENCE
boundary_evidence_level
strong_directional_account_link    2
medium_amount_flow                 1
strong_semantic_only               1
Name: count, dtype: int64

REFINED TRANSITION RESULTS


,anomaly_episode_id,party_name,organization_name,organization_balance_delta,boundary_evidence_level,refined_boundary_mechanism,refined_account_topologies,directionally_linked_payment_rows,anomaly_class,review_priority
2,101,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,-27761.16,strong_directional_account_link,boundary_transfer_candidate_directionally_linked,inflow_to_reported_next_account,1,single_balance_discontinuity,high
3,129,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,14812.94,strong_semantic_only,boundary_transfer_candidate_semantic_only,outflow_receiver_matches_previous_account_only,0,single_balance_discontinuity,high
1,17,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ЛЬВІВСЬКА ОБЛАСНА ТЕРИТОРІАЛЬНА ОРГАНІЗАЦІЯ ПО...,-2043.82,strong_directional_account_link,boundary_transfer_candidate_directionally_linked,inflow_to_reported_next_account,1,single_balance_discontinuity,medium
0,16,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Кременчуцька міська організація ПП ""Рідне місто""",2.00,medium_amount_flow,boundary_flow_candidate_medium,outflow_without_directional_account_link,0,single_balance_discontinuity,low


In [80]:
from pathlib import Path
import json

import pandas as pd


AUDIT_DIR = Path("data/interim/audit")
OUT_DIR = AUDIT_DIR / "case_dossiers"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOAD
# ============================================================

master = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_refined.parquet"
).copy()

transitions = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_transition_decomposition.parquet"
).copy()

account_rows = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_account_rows.parquet"
).copy()

report_diag = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_report_diagnostics.parquet"
).copy()

boundary = pd.read_parquet(
    AUDIT_DIR
    / "boundary_transfer_account_evidence_refined.parquet"
).copy()

alt_tests = pd.read_parquet(
    AUDIT_DIR
    / "anomaly_report_version_continuity_tests.parquet"
).copy()


# ============================================================
# HIGH / CRITICAL ONLY
# ============================================================

cases = (
    master[
        master[
            "review_priority"
        ].isin(
            ["critical", "high"]
        )
    ]
    .copy()
)


priority_rank = {
    "critical": 0,
    "high": 1,
}


cases[
    "_priority_rank"
] = cases[
    "review_priority"
].map(
    priority_rank
)


cases = (
    cases
    .sort_values(
        [
            "_priority_rank",
            "max_abs_balance_delta",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print(
    "High / critical episodes:",
    len(cases)
)


# ============================================================
# HELPERS
# ============================================================

def fmt_money(value):

    if value is None:
        return "—"

    try:
        if pd.isna(value):
            return "—"
    except Exception:
        pass

    return (
        f"{float(value):,.2f}"
        .replace(",", " ")
    )


def safe_value(row, col):

    if col not in row.index:
        return None

    value = row[col]

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    return value


def first_existing(
    row,
    candidates,
):

    for col in candidates:

        value = safe_value(
            row,
            col,
        )

        if value is not None:
            return value

    return None


# ============================================================
# BUILD ONE FACT SHEET PER EPISODE
# ============================================================

summary_rows = []

all_markdown = []


for case in cases.itertuples(
    index=False
):

    episode_id = int(
        case.anomaly_episode_id
    )


    tr = (
        transitions[
            transitions[
                "anomaly_episode_id"
            ] == episode_id
        ]
        .copy()
    )


    acc = (
        account_rows[
            account_rows[
                "anomaly_episode_id"
            ] == episode_id
        ]
        .copy()
    )


    diag = (
        report_diag[
            report_diag[
                "anomaly_episode_id"
            ] == episode_id
        ]
        .sort_values(
            [
                "year",
                "quarter",
            ]
        )
        .copy()
    )


    bnd = (
        boundary[
            boundary[
                "anomaly_episode_id"
            ] == episode_id
        ]
        .copy()
    )


    alts = (
        alt_tests[
            alt_tests[
                "anomaly_episode_id"
            ] == episode_id
        ]
        .copy()
    )


    # ========================================================
    # HEADER
    # ========================================================

    lines = []

    lines.append(
        f"# Episode {episode_id}"
    )

    lines.append("")

    lines.append(
        f"**Партія:** {case.party_name}"
    )

    lines.append(
        f"**Організація:** {case.organization_name}"
    )

    lines.append(
        f"**Період:** "
        f"{case.start_year} Q{case.start_quarter}"
        f" → "
        f"{case.end_year} Q{case.end_quarter}"
    )

    lines.append(
        f"**Пріоритет:** {case.review_priority}"
    )

    lines.append(
        f"**Базовий anomaly class:** "
        f"`{case.anomaly_class}`"
    )

    lines.append(
        f"**Analytical mechanism:** "
        f"`{case.analytical_mechanism}`"
    )

    lines.append(
        f"**Максимальний абсолютний розрив:** "
        f"{fmt_money(case.max_abs_balance_delta)} грн"
    )

    lines.append(
        f"**Net episode delta:** "
        f"{fmt_money(case.net_episode_balance_delta)} грн"
    )

    lines.append(
        f"**State-funding network exposure:** "
        f"{bool(case.party_state_funding_exposure)}"
    )

    lines.append("")


    # ========================================================
    # REPORT SEQUENCE
    # ========================================================

    lines.append(
        "## 1. Послідовність звітів"
    )

    lines.append("")

    lines.append(
        "| Період | report_id | рахунки | "
        "початок | надходження | використано | "
        "кінець | payment rows | inflow | outflow |"
    )

    lines.append(
        "|---|---|---:|---:|---:|---:|---:|---:|---:|---:|"
    )


    for _, row in diag.iterrows():

        lines.append(
            "| "
            f"{int(row['year'])} Q{int(row['quarter'])}"
            " | "
            f"`{row['report_id']}`"
            " | "
            f"{int(row['account_count']) if pd.notna(row.get('account_count')) else 0}"
            " | "
            f"{fmt_money(row.get('total_begin_balance'))}"
            " | "
            f"{fmt_money(row.get('reported_income'))}"
            " | "
            f"{fmt_money(row.get('reported_used'))}"
            " | "
            f"{fmt_money(row.get('total_end_balance'))}"
            " | "
            f"{int(row['payment_rows']) if pd.notna(row.get('payment_rows')) else 0}"
            " | "
            f"{fmt_money(row.get('cash_inflow'))}"
            " | "
            f"{fmt_money(row.get('cash_outflow'))}"
            " |"
        )


    lines.append("")


    # ========================================================
    # TRANSITIONS
    # ========================================================

    lines.append(
        "## 2. Розриви між кварталами"
    )

    lines.append("")


    for _, row in tr.iterrows():

        lines.append(
            f"- `{row['previous_report_id']}` → "
            f"`{row['next_report_id']}`: "
            f"**{fmt_money(row['organization_balance_delta'])} грн**; "
            f"account structure: "
            f"`{row['account_structure_class']}`."
        )


    lines.append("")


    # ========================================================
    # ACCOUNT-LEVEL DETAILS
    # ========================================================

    lines.append(
        "## 3. Account-level декомпозиція"
    )

    lines.append("")


    broken = acc[
        acc[
            "is_account_level_break"
        ] == True
    ].copy()


    if len(broken) == 0:

        lines.append(
            "Немає account-level break rows."
        )

    else:

        for _, row in broken.iterrows():

            account = first_existing(
                row,
                [
                    "account_number_normalized",
                    "previous_account_number_normalized",
                    "next_account_number_normalized",
                    "previous_account_number_raw",
                    "next_account_number_raw",
                ],
            )

            prev_end = first_existing(
                row,
                [
                    "previous_end_balance",
                ],
            )

            next_begin = first_existing(
                row,
                [
                    "next_begin_balance",
                ],
            )

            contribution = first_existing(
                row,
                [
                    "account_delta_contribution",
                ],
            )

            lines.append(
                f"- account `{account}`; "
                f"status `{row['continuity_status']}`; "
                f"previous end = {fmt_money(prev_end)}; "
                f"next begin = {fmt_money(next_begin)}; "
                f"contribution = {fmt_money(contribution)} грн."
            )


    lines.append("")


    # ========================================================
    # BOUNDARY TRANSFER EVIDENCE
    # ========================================================

    lines.append(
        "## 4. Boundary-transfer evidence"
    )

    lines.append("")


    if len(bnd) == 0:

        lines.append(
            "Спеціального boundary-transfer evidence не виявлено."
        )

        boundary_level = None
        boundary_mechanism = None

    else:

        for _, row in bnd.iterrows():

            lines.append(
                f"- evidence: "
                f"`{row['boundary_evidence_level']}`; "
                f"mechanism: "
                f"`{row['refined_boundary_mechanism']}`; "
                f"topology: "
                f"`{row['refined_account_topologies']}`."
            )

        boundary_level = (
            bnd[
                "boundary_evidence_level"
            ].iloc[0]
        )

        boundary_mechanism = (
            bnd[
                "refined_boundary_mechanism"
            ].iloc[0]
        )


    lines.append("")


    # ========================================================
    # ALTERNATIVE SOURCE VERSIONS
    # ========================================================

    lines.append(
        "## 5. Альтернативні source versions"
    )

    lines.append("")


    alternatives = alts[
        alts[
            "candidate_is_official_selected"
        ] == False
    ].copy()


    if len(alternatives) == 0:

        lines.append(
            "Інших source instances для проблемних періодів "
            "у доступному manifest не знайдено."
        )

        alternative_count = 0

        exact_restoration = False

    else:

        alternative_count = len(
            alternatives
        )

        exact_restoration = bool(
            alternatives[
                "exact_bridge"
            ].fillna(False).any()
        )


        for _, row in alternatives.iterrows():

            lines.append(
                f"- `{row['candidate_report_id']}`; "
                f"signed={bool(row['candidate_signed'])}; "
                f"begin={fmt_money(row['candidate_total_begin'])}; "
                f"income={fmt_money(row['candidate_total_income'])}; "
                f"used={fmt_money(row['candidate_total_used'])}; "
                f"end={fmt_money(row['candidate_total_end'])}; "
                f"left delta={fmt_money(row['left_balance_delta'])}; "
                f"right delta={fmt_money(row['right_balance_delta'])}; "
                f"exact_bridge={bool(row['exact_bridge'])}."
            )


    lines.append("")


    # ========================================================
    # PROVISIONAL CASE STATUS
    #
    # Important: deliberately factual and conservative.
    # ========================================================

    lines.append(
        "## 6. Поточний статус аудиту"
    )

    lines.append("")


    if exact_restoration:

        provisional_status = (
            "version_integrity_anomaly_with_exact_restoration"
        )

        lines.append(
            "Інша збережена source version того самого "
            "звітного періоду точно відновлює обидві межі "
            "балансової тяглості."
        )


    elif (
        boundary_level
        ==
        "strong_directional_account_link"
    ):

        provisional_status = (
            "boundary_transfer_representation_candidate"
        )

        lines.append(
            "Розрив має сильний directionally account-linked "
            "зв'язок із платежем тієї самої суми. "
            "Кейс потребує перевірки правил відображення "
            "перенесення/закриття рахунків між кварталами."
        )


    elif (
        boundary_level
        ==
        "strong_semantic_only"
    ):

        provisional_status = (
            "boundary_transfer_semantic_candidate"
        )

        lines.append(
            "Сума і зміст платежу сильно пов'язані з розривом, "
            "але напрямний зв'язок із задекларованим рахунком "
            "не доведений."
        )


    elif (
        case.anomaly_class
        ==
        "balance_disappears_into_empty_financial_block"
    ):

        provisional_status = (
            "missing_financial_block_candidate"
        )

        lines.append(
            "Ненульовий залишок переходить у звіт без "
            "account/payment rows. Публічна послідовність "
            "не пояснює його відображення."
        )


    elif (
        case.anomaly_class
        ==
        "balance_reappears_after_empty_financial_block"
    ):

        provisional_status = (
            "missing_financial_block_candidate"
        )

        lines.append(
            "Ненульовий залишок з'являється після звіту без "
            "account/payment rows."
        )


    elif (
        case.episode_account_structure
        ==
        "same_account_balance_mismatch"
    ):

        provisional_status = (
            "unexplained_same_account_balance_mismatch"
        )

        lines.append(
            "Розрив виникає на тому самому рахунку; "
            "пояснення через зміну IBAN або заміну рахунків "
            "не підтверджується."
        )


    elif (
        case.episode_account_structure
        ==
        "nonzero_account_disappears"
    ):

        provisional_status = (
            "nonzero_account_disappearance_candidate"
        )

        lines.append(
            "Ненульовий рахунок перестає відображатися "
            "в наступному звіті."
        )


    elif (
        case.episode_account_structure
        ==
        "nonzero_account_appears"
    ):

        provisional_status = (
            "nonzero_account_appearance_candidate"
        )

        lines.append(
            "У наступному звіті з'являється рахунок із "
            "ненульовим opening balance."
        )


    else:

        provisional_status = (
            "manual_review_required"
        )

        lines.append(
            "Автоматична діагностика не дає достатнього "
            "пояснення; потрібен ручний аналіз."
        )


    lines.append("")

    lines.append(
        "> Усі статуси є data-quality / registry-integrity "
        "signals і не є правовим висновком про порушення."
    )

    lines.append("")


    # ========================================================
    # SAVE INDIVIDUAL MD
    # ========================================================

    dossier_text = "\n".join(
        lines
    )


    dossier_path = (
        OUT_DIR
        / f"episode_{episode_id:03d}.md"
    )


    dossier_path.write_text(
        dossier_text,
        encoding="utf-8",
    )


    all_markdown.append(
        dossier_text
    )

    all_markdown.append(
        "\n---\n"
    )


    summary_rows.append(
        {
            "anomaly_episode_id":
                episode_id,

            "party_name":
                case.party_name,

            "organization_name":
                case.organization_name,

            "review_priority":
                case.review_priority,

            "anomaly_class":
                case.anomaly_class,

            "analytical_mechanism":
                case.analytical_mechanism,

            "episode_account_structure":
                case.episode_account_structure,

            "max_abs_balance_delta":
                case.max_abs_balance_delta,

            "net_episode_balance_delta":
                case.net_episode_balance_delta,

            "boundary_evidence_level":
                boundary_level,

            "boundary_mechanism":
                boundary_mechanism,

            "alternative_report_count":
                alternative_count,

            "has_exact_restoring_alternative":
                exact_restoration,

            "provisional_case_status":
                provisional_status,

            "dossier_path":
                str(
                    dossier_path
                ),
        }
    )


# ============================================================
# SAVE COMBINED DOSSIER
# ============================================================

combined_path = (
    AUDIT_DIR
    / "high_priority_case_dossiers.md"
)

combined_path.write_text(
    "\n".join(
        all_markdown
    ),
    encoding="utf-8",
)


case_summary = pd.DataFrame(
    summary_rows
)


case_summary.to_parquet(
    AUDIT_DIR
    / "high_priority_case_summary.parquet",
    index=False,
)


case_summary.to_csv(
    AUDIT_DIR
    / "high_priority_case_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# OUTPUT
# ============================================================

print()
print("=" * 80)
print("HIGH / CRITICAL CASE DOSSIERS")
print("=" * 80)

print(
    "Cases:",
    len(
        case_summary
    )
)

print(
    "Combined dossier:",
    combined_path
)

print()


print(
    "PROVISIONAL CASE STATUS"
)

print(
    case_summary[
        "provisional_case_status"
    ].value_counts()
)


print()
display(
    case_summary[
        [
            "anomaly_episode_id",
            "party_name",
            "organization_name",

            "review_priority",

            "max_abs_balance_delta",

            "anomaly_class",
            "episode_account_structure",

            "boundary_evidence_level",

            "has_exact_restoring_alternative",

            "provisional_case_status",
        ]
    ]
)

High / critical episodes: 8

HIGH / CRITICAL CASE DOSSIERS
Cases: 8
Combined dossier: data\interim\audit\high_priority_case_dossiers.md

PROVISIONAL CASE STATUS
provisional_case_status
unexplained_same_account_balance_mismatch           2
missing_financial_block_candidate                   2
nonzero_account_disappearance_candidate             1
version_integrity_anomaly_with_exact_restoration    1
boundary_transfer_representation_candidate          1
boundary_transfer_semantic_candidate                1
Name: count, dtype: int64



,anomaly_episode_id,party_name,organization_name,review_priority,max_abs_balance_delta,anomaly_class,episode_account_structure,boundary_evidence_level,has_exact_restoring_alternative,provisional_case_status
0,48,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,critical,31164.96,version_integrity_restored_by_alternative,multiple_account_patterns,None,True,version_integrity_anomaly_with_exact_restoration
1,30,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,ЧЕРНІГІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПА...,high,585171.00,single_balance_discontinuity,nonzero_account_disappears,None,False,nonzero_account_disappearance_candidate
2,102,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,СУМСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,high,366538.98,balance_disappears_into_empty_financial_block,nonzero_account_disappears,None,False,missing_financial_block_candidate
3,24,"ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""","ПОЛІТИЧНА ПАРТІЯ ""РОЗВИТОК І ДОБРОБУТ""",high,315740.00,balance_reappears_after_empty_financial_block,nonzero_account_appears,None,False,missing_financial_block_candidate
4,105,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ МІСТО»,"Чернігівська обласна організація ПП ""Рідне місто""",high,72720.00,single_balance_discontinuity,same_account_balance_mismatch,None,False,unexplained_same_account_balance_mismatch
5,106,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Новомосковська міська організація Партії Зелен...,high,62683.00,single_balance_discontinuity,same_account_balance_mismatch,None,False,unexplained_same_account_balance_mismatch
6,101,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ»,Запорізька обласна організація Партії Зелених ...,high,27761.16,single_balance_discontinuity,nonzero_account_disappears,strong_directional_account_link,False,boundary_transfer_representation_candidate
7,129,ПОЛІТИЧНА ПАРТІЯ «ЗА МАЙБУТНЄ»,ХЕРСОНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,high,14812.94,single_balance_discontinuity,nonzero_account_appears,strong_semantic_only,False,boundary_transfer_semantic_candidate


In [81]:
from pathlib import Path
import json

import pandas as pd


AUDIT_DIR = Path("data/interim/audit")
RAW_DIR = Path("data/raw/report_details")

EPISODE_ID = 30


# ============================================================
# LOAD
# ============================================================

master = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_classification_refined.parquet"
)

transitions = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_transition_decomposition.parquet"
)

account_rows = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_account_rows.parquet"
)

report_diag = pd.read_parquet(
    AUDIT_DIR
    / "organization_balance_anomaly_report_diagnostics.parquet"
)


case = master[
    master["anomaly_episode_id"] == EPISODE_ID
].iloc[0]


tr = transitions[
    transitions["anomaly_episode_id"] == EPISODE_ID
].copy()


acc = account_rows[
    account_rows["anomaly_episode_id"] == EPISODE_ID
].copy()


diag = (
    report_diag[
        report_diag["anomaly_episode_id"] == EPISODE_ID
    ]
    .sort_values(
        ["year", "quarter"]
    )
    .copy()
)


print("=" * 90)
print("CASE")
print("=" * 90)

print("Party:", case["party_name"])
print("Organization:", case["organization_name"])
print("Anomaly class:", case["anomaly_class"])
print("Account structure:", case["episode_account_structure"])
print(
    "Max balance delta:",
    case["max_abs_balance_delta"]
)

print()


# ============================================================
# REPORT SEQUENCE
# ============================================================

print("=" * 90)
print("REPORT SEQUENCE")
print("=" * 90)

display(
    diag[
        [
            "year",
            "quarter",
            "report_id",

            "account_count",
            "total_begin_balance",
            "reported_income",
            "reported_used",
            "total_end_balance",

            "payment_rows",
            "cash_inflow",
            "cash_outflow",
            "transaction_net_flow",

            "financial_block_empty",
        ]
    ]
)


# ============================================================
# TRANSITION
# ============================================================

print()
print("=" * 90)
print("BROKEN TRANSITION")
print("=" * 90)

display(
    tr[
        [
            "previous_report_id",
            "next_report_id",

            "organization_balance_delta",

            "account_structure_class",

            "same_account_mismatch_count",
            "nonzero_account_removed_count",
            "nonzero_account_added_count",

            "decomposed_account_delta",
            "decomposition_matches",
        ]
    ]
)


# ============================================================
# ACCOUNT-LEVEL BREAK
# ============================================================

print()
print("=" * 90)
print("ACCOUNT-LEVEL BREAK")
print("=" * 90)

broken = acc[
    acc["is_account_level_break"] == True
].copy()

display(
    broken
)


# ============================================================
# RAW REPORT DETAILS
# ============================================================

report_ids = list(
    dict.fromkeys(
        tr["previous_report_id"].astype(str).tolist()
        +
        tr["next_report_id"].astype(str).tolist()
    )
)


PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


for report_id in report_ids:

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = json.load(f)["results"]


    print()
    print("=" * 90)
    print("RAW REPORT:", report_id)
    print(
        f"{detail.get('year')} Q{detail.get('quarter')}",
        "| signed:",
        detail.get("signed_date"),
    )
    print("=" * 90)


    # --------------------------------------------------------
    # MONEY / BANK ACCOUNTS
    # --------------------------------------------------------

    money_rows = (
        (
            detail.get("properties")
            or {}
        )
        .get("property_moneys")
        or []
    )


    print()
    print("PROPERTY_MONEYS:")

    if money_rows:

        money_df = pd.DataFrame(
            money_rows
        )

        wanted = [
            col
            for col in [
                "id",
                "account_number",
                "account_type",
                "account_holder",
                "account_holder_code",

                "begin_period_balance",
                "report_period_income",
                "report_period_used_funds",
                "end_period_balance",

                "created_at",
            ]
            if col in money_df.columns
        ]

        display(
            money_df[wanted]
        )

    else:

        print("NO ACCOUNT ROWS")


    # --------------------------------------------------------
    # PAYMENT CATEGORIES
    # --------------------------------------------------------

    payment_info = (
        detail.get("payment_info")
        or {}
    )


    for payment_type, (
        direction,
        key,
    ) in PAYMENT_PATHS.items():

        rows = (
            (
                payment_info.get(direction)
                or {}
            )
            .get(key)
            or []
        )

        if not rows:
            continue


        df = pd.DataFrame(rows)


        print()
        print(
            payment_type.upper(),
            "| rows:",
            len(df),
            "| sum:",
            pd.to_numeric(
                df["payment_amount"],
                errors="coerce",
            ).sum()
            if "payment_amount" in df.columns
            else "?"
        )


        wanted = [
            col
            for col in [
                "id",

                "payment_operation_date",
                "payment_instruction_date",

                "payment_amount",

                "payer_account_iban",
                "payer_name",
                "payer_code",

                "receiver_account_iban",
                "receiver_name",
                "receiver_code",

                "payment_description",
                "payment_purpose",
                "payment_reason",

                "created_at",
                "updated_at",
            ]
            if col in df.columns
        ]


        display(
            df[wanted]
            .sort_values(
                "payment_operation_date"
                if "payment_operation_date"
                in df.columns
                else df.index.name
            )
            if (
                "payment_operation_date"
                in df.columns
            )
            else df[wanted]
        )

CASE
Party: ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»
Organization: ЧЕРНІГІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ "НАШ КРАЙ"
Anomaly class: single_balance_discontinuity
Account structure: nonzero_account_disappears
Max balance delta: 585171.0

REPORT SEQUENCE


,year,quarter,report_id,account_count,total_begin_balance,reported_income,reported_used,total_end_balance,payment_rows,cash_inflow,cash_outflow,transaction_net_flow,financial_block_empty
63,2024,4,90fb5f50-e060-11ef-bde1-41cc07d44e1f,2,724011.0,0.0,0.0,724011.0,0,0.0,0.0,0.0,False
62,2025,1,0b3d8d30-2d67-11f0-8f85-d95649015426,1,138840.0,0.0,0.0,138840.0,0,0.0,0.0,0.0,False



BROKEN TRANSITION


,previous_report_id,next_report_id,organization_balance_delta,account_structure_class,same_account_mismatch_count,nonzero_account_removed_count,nonzero_account_added_count,decomposed_account_delta,decomposition_matches
33,90fb5f50-e060-11ef-bde1-41cc07d44e1f,0b3d8d30-2d67-11f0-8f85-d95649015426,-585171.0,nonzero_account_disappears,0,1,0,-585171.0,True



ACCOUNT-LEVEL BREAK


,organization_id,root_party_id,account_match_key,canonical_iban,previous_account_number_raw,next_account_number_raw,previous_report_id,previous_year,previous_quarter,previous_end_balance,next_report_id,next_year,next_quarter,next_begin_balance,balance_delta,continuity_status,party_state_funding_exposure,direct_state_funding_amount,budget_expense_amount,party_name,organization_name,anomaly_episode_id,organization_balance_delta,account_delta_contribution,raw_account_spelling_changed,is_same_account_mismatch,is_nonzero_account_removed,is_nonzero_account_added,is_account_level_break
46,4009783d-3acf-4d27-bcc3-a21e82e10021,8f775932-56e2-48a4-b866-5c501335f4eb,UA063516290000000026003240342,UA063516290000000026003240342,UA063516290000000026003240342,None,90fb5f50-e060-11ef-bde1-41cc07d44e1f,2024,4,585171.0,0b3d8d30-2d67-11f0-8f85-d95649015426,2025,1,NaN,NaN,account_missing_next_with_nonzero_end,False,0.0,0.0,ПОЛІТИЧНА ПАРТІЯ «НАШ КРАЙ»,ЧЕРНІГІВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПА...,30,-585171.0,-585171.0,False,False,True,False,True



RAW REPORT: 90fb5f50-e060-11ef-bde1-41cc07d44e1f
2024 Q4 | signed: 2025-02-03 09:30:20

PROPERTY_MONEYS:


,id,account_number,account_type,account_holder,account_holder_code,begin_period_balance,report_period_income,report_period_used_funds,end_period_balance,created_at
0,3624aa38-07da-4aea-93d8-3d9983b1d1a3,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",None,585171,0,0,585171,2025-02-03 09:30:20
1,3c307ea0-ae6b-49a7-8d31-1ab98f948144,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",None,138840,0,0,138840,2025-02-03 09:30:20
2,246c7799-c610-497c-b763-55cc2b2329b4,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",None,585171,0,0,585171,2026-02-23 14:34:38
3,4e0379c2-c77d-477f-ac80-0c425a7ca3fc,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",None,138840,0,0,138840,2026-02-23 14:34:38



RAW REPORT: 0b3d8d30-2d67-11f0-8f85-d95649015426
2025 Q1 | signed: 2025-05-12 10:12:18

PROPERTY_MONEYS:


,id,account_number,account_type,account_holder,account_holder_code,begin_period_balance,report_period_income,report_period_used_funds,end_period_balance,created_at
0,674df11b-5ee3-483d-9e24-f10f370070fe,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",None,138840,0,0,138840,2025-05-12 10:12:18
1,9eaa60f5-26f2-4c29-9cab-80b741b6f8ec,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",None,138840,0,0,138840,2026-02-23 18:58:23


In [82]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd


REPORTS_DIR = Path("data/interim/reports")
RAW_DIR = Path("data/raw/report_details")

ORGANIZATION_ID = "4009783d-3acf-4d27-bcc3-a21e82e10021"
TARGET_IBAN = "UA063516290000000026003240342"


# ============================================================
# LOAD ALL SELECTED REPORTS FOR ORGANIZATION
# ============================================================

selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()


selected["organization_id"] = (
    selected["organization_id"].astype(str)
)

selected["report_id"] = (
    selected["report_id"].astype(str)
)


org_reports = (
    selected[
        selected["organization_id"]
        ==
        ORGANIZATION_ID
    ]
    .sort_values(
        ["year", "quarter"]
    )
    .copy()
)


print(
    "Selected reports:",
    len(org_reports)
)


# ============================================================
# IBAN NORMALIZER
# ============================================================

def normalize_iban(value):

    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    ).upper()

    text = re.sub(
        r"[\s\-._/\\:]",
        "",
        text,
    )

    pos = text.find("UA")

    if pos >= 0:
        text = text[pos:]

    match = re.search(
        r"UA\d{27}",
        text,
    )

    return (
        match.group(0)
        if match
        else None
    )


# ============================================================
# EXTRACT ACCOUNT HISTORY
# ============================================================

rows = []


for report in org_reports.itertuples(
    index=False
):

    report_id = str(
        report.report_id
    )

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if not path.exists():
        continue


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        detail = json.load(f)["results"]


    money_rows = (
        (
            detail.get("properties")
            or {}
        )
        .get("property_moneys")
        or []
    )


    # substantive dedup
    seen = set()


    for m in money_rows:

        iban = normalize_iban(
            m.get("account_number")
        )

        key = (
            iban,
            m.get("begin_period_balance"),
            m.get("report_period_income"),
            m.get("report_period_used_funds"),
            m.get("end_period_balance"),
        )

        if key in seen:
            continue

        seen.add(key)


        rows.append(
            {
                "year":
                    int(report.year),

                "quarter":
                    int(report.quarter),

                "report_id":
                    report_id,

                "signed_date":
                    detail.get("signed_date"),

                "iban":
                    iban,

                "account_number_raw":
                    m.get("account_number"),

                "account_type":
                    m.get("account_type"),

                "account_holder":
                    m.get("account_holder"),

                "begin_balance":
                    m.get("begin_period_balance"),

                "income":
                    m.get("report_period_income"),

                "used":
                    m.get("report_period_used_funds"),

                "end_balance":
                    m.get("end_period_balance"),
            }
        )


history = pd.DataFrame(rows)


# ============================================================
# TARGET MEGABANK ACCOUNT
# ============================================================

mega = (
    history[
        history["iban"] == TARGET_IBAN
    ]
    .sort_values(
        ["year", "quarter"]
    )
    .copy()
)


print()
print("=" * 90)
print("MEGABANK ACCOUNT HISTORY")
print("=" * 90)

display(
    mega
)


# ============================================================
# ALL ACCOUNTS — TO SEE REPLACEMENT / PARALLEL ACCOUNTS
# ============================================================

print()
print("=" * 90)
print("ALL ACCOUNT HISTORY")
print("=" * 90)

display(
    history.sort_values(
        ["year", "quarter", "iban"]
    )
)


# ============================================================
# HOW LONG WAS 585171 CARRIED UNCHANGED?
# ============================================================

if len(mega):

    mega[
        "unchanged_585171"
    ] = (
        pd.to_numeric(
            mega["begin_balance"],
            errors="coerce",
        ).eq(585171)
        &
        pd.to_numeric(
            mega["end_balance"],
            errors="coerce",
        ).eq(585171)
        &
        pd.to_numeric(
            mega["income"],
            errors="coerce",
        ).fillna(0).eq(0)
        &
        pd.to_numeric(
            mega["used"],
            errors="coerce",
        ).fillna(0).eq(0)
    )


    print()
    print("=" * 90)
    print("UNCHANGED 585171 PERIODS")
    print("=" * 90)

    display(
        mega[
            mega["unchanged_585171"]
        ][
            [
                "year",
                "quarter",
                "report_id",
                "begin_balance",
                "income",
                "used",
                "end_balance",
                "account_holder",
            ]
        ]
    )


    print(
        "Periods carrying exactly 585171 unchanged:",
        int(
            mega["unchanged_585171"].sum()
        )
    )


# ============================================================
# FIRST / LAST APPEARANCE
# ============================================================

if len(mega):

    first = mega.iloc[0]
    last = mega.iloc[-1]

    print()
    print("=" * 90)
    print("FIRST / LAST APPEARANCE")
    print("=" * 90)

    print(
        "First:",
        f"{first['year']} Q{first['quarter']}",
        first["report_id"],
    )

    print(
        "Last:",
        f"{last['year']} Q{last['quarter']}",
        last["report_id"],
    )

Selected reports: 9

MEGABANK ACCOUNT HISTORY


,year,quarter,report_id,signed_date,iban,account_number_raw,account_type,account_holder,begin_balance,income,used,end_balance
0,2021,5,fec5bb40-d6ff-11ee-96d4-258361b278a8,2024-02-29 15:10:22,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"АТ ""Мегабанк""",0,748428,20457,727971
2,2022,5,031bf600-d700-11ee-8d4b-3dd92aa40d65,2024-02-29 15:10:50,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",727971,0,142800,585171
4,2023,5,0798f2a0-d700-11ee-8847-316909bdf249,2024-02-29 15:11:17,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171
6,2024,1,93ed3c90-0c50-11ef-96f1-37a2ca81244c,2024-05-07 17:01:37,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171
8,2024,2,993fd640-49e9-11ef-9f3c-219e709335ad,2024-07-28 19:52:14,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171
10,2024,3,6fb1a2e0-98e5-11ef-b4cc-1f31c784e768,2024-11-04 09:51:08,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171
12,2024,4,90fb5f50-e060-11ef-bde1-41cc07d44e1f,2025-02-03 09:30:20,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171



ALL ACCOUNT HISTORY


,year,quarter,report_id,signed_date,iban,account_number_raw,account_type,account_holder,begin_balance,income,used,end_balance
0,2021,5,fec5bb40-d6ff-11ee-96d4-258361b278a8,2024-02-29 15:10:22,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"АТ ""Мегабанк""",0,748428,20457,727971
1,2021,5,fec5bb40-d6ff-11ee-96d4-258361b278a8,2024-02-29 15:10:22,UA783204780000026005924904762,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",0,0,0,0
2,2022,5,031bf600-d700-11ee-8d4b-3dd92aa40d65,2024-02-29 15:10:50,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",727971,0,142800,585171
3,2022,5,031bf600-d700-11ee-8d4b-3dd92aa40d65,2024-02-29 15:10:50,UA783204780000026005924904762,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",0,142800,160,142640
4,2023,5,0798f2a0-d700-11ee-8847-316909bdf249,2024-02-29 15:11:17,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171
5,2023,5,0798f2a0-d700-11ee-8847-316909bdf249,2024-02-29 15:11:17,UA783204780000026005924904762,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",142640,0,2550,140090
6,2024,1,93ed3c90-0c50-11ef-96f1-37a2ca81244c,2024-05-07 17:01:37,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171
7,2024,1,93ed3c90-0c50-11ef-96f1-37a2ca81244c,2024-05-07 17:01:37,UA783204780000026005924904762,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",140090,0,750,139340
8,2024,2,993fd640-49e9-11ef-9f3c-219e709335ad,2024-07-28 19:52:14,UA063516290000000026003240342,UA063516290000000026003240342,Поточний рахунок,"ПАТ ""МЕГАБАНК"", Харків",585171,0,0,585171
9,2024,2,993fd640-49e9-11ef-9f3c-219e709335ad,2024-07-28 19:52:14,UA783204780000026005924904762,UA783204780000026005924904762,Поточний рахунок,"АБ ""Укргазбанк""",139340,0,500,138840



UNCHANGED 585171 PERIODS


,year,quarter,report_id,begin_balance,income,used,end_balance,account_holder
4,2023,5,0798f2a0-d700-11ee-8847-316909bdf249,585171,0,0,585171,"ПАТ ""МЕГАБАНК"", Харків"
6,2024,1,93ed3c90-0c50-11ef-96f1-37a2ca81244c,585171,0,0,585171,"ПАТ ""МЕГАБАНК"", Харків"
8,2024,2,993fd640-49e9-11ef-9f3c-219e709335ad,585171,0,0,585171,"ПАТ ""МЕГАБАНК"", Харків"
10,2024,3,6fb1a2e0-98e5-11ef-b4cc-1f31c784e768,585171,0,0,585171,"ПАТ ""МЕГАБАНК"", Харків"
12,2024,4,90fb5f50-e060-11ef-bde1-41cc07d44e1f,585171,0,0,585171,"ПАТ ""МЕГАБАНК"", Харків"


Periods carrying exactly 585171 unchanged: 5

FIRST / LAST APPEARANCE
First: 2021 Q5 fec5bb40-d6ff-11ee-96d4-258361b278a8
Last: 2024 Q4 90fb5f50-e060-11ef-bde1-41cc07d44e1f


In [83]:
from pathlib import Path
import json
from decimal import Decimal, InvalidOperation

import pandas as pd


RAW_DIR = Path("data/raw/report_details")

REPORTS = {
    "2022_annual":
        "031bf600-d700-11ee-8d4b-3dd92aa40d65",

    "2023_annual":
        "0798f2a0-d700-11ee-8847-316909bdf249",

    "2024_q4":
        "90fb5f50-e060-11ef-bde1-41cc07d44e1f",

    "2025_q1":
        "0b3d8d30-2d67-11f0-8f85-d95649015426",
}


TARGET_IBAN = (
    "UA063516290000000026003240342"
)

TARGET_AMOUNTS = {
    Decimal("585171"),
    Decimal("142800"),
}


# ============================================================
# HELPERS
# ============================================================

def as_decimal(value):

    if isinstance(value, bool):
        return None

    try:
        return Decimal(str(value))
    except (
        InvalidOperation,
        TypeError,
        ValueError,
    ):
        return None


def scalar_match(value):

    # ------------------------------------------
    # text
    # ------------------------------------------

    if isinstance(value, str):

        text = value.upper()

        if (
            "МЕГАБАНК" in text
            or
            "MEGABANK" in text
            or
            TARGET_IBAN in text
        ):
            return True


    # ------------------------------------------
    # numeric
    # ------------------------------------------

    number = as_decimal(value)

    if (
        number is not None
        and number in TARGET_AMOUNTS
    ):
        return True


    return False


def recursive_matches(
    obj,
    path="$",
):

    found = []


    if isinstance(obj, dict):

        for key, value in obj.items():

            child_path = (
                f"{path}.{key}"
            )

            if scalar_match(value):

                found.append(
                    {
                        "path":
                            child_path,

                        "value":
                            value,
                    }
                )

            found.extend(
                recursive_matches(
                    value,
                    child_path,
                )
            )


    elif isinstance(obj, list):

        for i, value in enumerate(obj):

            child_path = (
                f"{path}[{i}]"
            )

            if scalar_match(value):

                found.append(
                    {
                        "path":
                            child_path,

                        "value":
                            value,
                    }
                )

            found.extend(
                recursive_matches(
                    value,
                    child_path,
                )
            )


    return found


# ============================================================
# LOAD
# ============================================================

details = {}


for label, report_id in REPORTS.items():

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        details[label] = (
            json.load(f)["results"]
        )


# ============================================================
# SEARCH ENTIRE REPORT STRUCTURE
#
# Not just property_moneys.
# ============================================================

all_hits = []


for label, detail in details.items():

    hits = recursive_matches(
        detail
    )


    for hit in hits:

        all_hits.append(
            {
                "report_label":
                    label,

                "report_id":
                    REPORTS[label],

                "year":
                    detail.get("year"),

                "quarter":
                    detail.get("quarter"),

                "path":
                    hit["path"],

                "value":
                    hit["value"],
            }
        )


hits_df = pd.DataFrame(
    all_hits
)


print()
print("=" * 100)
print("ALL REFERENCES TO MEGABANK / IBAN / 585171 / 142800")
print("=" * 100)

display(
    hits_df
    .sort_values(
        [
            "report_label",
            "path",
        ]
    )
)


# ============================================================
# SPECIFICALLY SEARCH 2025 Q1 OUTSIDE PROPERTY_MONEYS
#
# This is the decisive reclassification check.
# ============================================================

q1 = details[
    "2025_q1"
]


q1_hits = recursive_matches(
    q1
)


q1_hits_df = pd.DataFrame(
    q1_hits
)


if len(q1_hits_df):

    q1_hits_df[
        "inside_property_moneys"
    ] = q1_hits_df[
        "path"
    ].str.contains(
        "property_moneys",
        regex=False,
    )


    print()
    print("=" * 100)
    print("2025 Q1 — POSSIBLE RECLASSIFICATION")
    print("=" * 100)

    display(
        q1_hits_df
    )


    print()
    print(
        "Matches OUTSIDE property_moneys:",
        int(
            (
                ~q1_hits_df[
                    "inside_property_moneys"
                ]
            ).sum()
        )
    )

else:

    print()
    print("=" * 100)
    print("2025 Q1 — POSSIBLE RECLASSIFICATION")
    print("=" * 100)

    print(
        "No references to Megabank, target IBAN, "
        "585171 or 142800 anywhere in report."
    )


# ============================================================
# 2022 ANNUAL — PAYMENT ROWS OF EXACTLY 142800
# ============================================================

PAYMENT_PATHS = {
    "monetary_contributions":
        ("incoming", "monetary_contributions"),

    "other_contributions":
        ("incoming", "other_contributions"),

    "state_funding":
        ("incoming", "state_funding"),

    "other_incomes":
        ("incoming", "other_incomes"),

    "budget_expenses":
        ("outgoing", "budget_expenses"),

    "outgoing_expenses":
        ("outgoing", "outgoing_expenses"),

    "return_expenses":
        ("outgoing", "return_expenses"),

    "transfer_expenses":
        ("outgoing", "transfer_expenses"),
}


detail_2022 = details[
    "2022_annual"
]


payment_info = (
    detail_2022.get(
        "payment_info"
    )
    or {}
)


payment_hits = []


for payment_type, (
    direction,
    key,
) in PAYMENT_PATHS.items():

    rows = (
        (
            payment_info.get(
                direction
            )
            or {}
        )
        .get(key)
        or []
    )


    for row in rows:

        amount = as_decimal(
            row.get(
                "payment_amount"
            )
        )


        row_text = json.dumps(
            row,
            ensure_ascii=False,
            default=str,
        ).upper()


        relevant = (
            amount
            in TARGET_AMOUNTS
            or
            "МЕГАБАНК"
            in row_text
            or
            "MEGABANK"
            in row_text
            or
            TARGET_IBAN
            in row_text
        )


        if relevant:

            payment_hits.append(
                {
                    "payment_type":
                        payment_type,

                    **row,
                }
            )


payment_hits_df = pd.DataFrame(
    payment_hits
)


print()
print("=" * 100)
print("2022 ANNUAL — RELEVANT PAYMENT ROWS")
print("=" * 100)


if len(payment_hits_df):

    wanted = [
        col
        for col in [
            "payment_type",
            "id",

            "payment_operation_date",
            "payment_instruction_date",

            "payment_amount",

            "payer_account_iban",
            "payer_name",
            "payer_code",

            "receiver_account_iban",
            "receiver_name",
            "receiver_code",

            "payment_description",
            "payment_purpose",
            "payment_reason",

            "refund_amount",
            "refund_date",
        ]
        if col in payment_hits_df.columns
    ]


    display(
        payment_hits_df[
            wanted
        ]
    )

else:

    print(
        "No matching payment rows."
    )


# ============================================================
# TOP-LEVEL STRUCTURAL CHECK 2025 Q1
#
# Show non-empty sections so we can see where a possible
# claim / receivable might live even if its amount differs.
# ============================================================

print()
print("=" * 100)
print("2025 Q1 — NON-EMPTY TOP-LEVEL FINANCIAL SECTIONS")
print("=" * 100)


for key in [
    "properties",
    "payment_info",
    "obligations",
    "organizations",
]:

    value = q1.get(key)

    if isinstance(
        value,
        dict,
    ):

        nonempty = {
            k: (
                len(v)
                if isinstance(
                    v,
                    (
                        list,
                        dict,
                    ),
                )
                else v
            )
            for k, v in value.items()
            if v not in (
                None,
                [],
                {},
                "",
            )
        }

        print()
        print(key)
        print(nonempty)

    else:

        print()
        print(
            key,
            value,
        )


ALL REFERENCES TO MEGABANK / IBAN / 585171 / 142800


,report_label,report_id,year,quarter,path,value
10,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.payment_info.outgoing.outgoing_expenses[0].r...,UA063516290000000026003240342
11,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.payment_info.outgoing.outgoing_expenses[1].r...,UA063516290000000026003240342
1,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[0].account_holder,"ПАТ ""МЕГАБАНК"", Харків"
0,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[0].account_number,UA063516290000000026003240342
2,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[0].end_period_bal...,585171
3,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[0].report_period_...,142800
4,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[1].report_period_...,142800
6,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[2].account_holder,"ПАТ ""МЕГАБАНК"", Харків"
5,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[2].account_number,UA063516290000000026003240342
7,2022_annual,031bf600-d700-11ee-8d4b-3dd92aa40d65,2022,5,$.properties.property_moneys[2].end_period_bal...,585171



2025 Q1 — POSSIBLE RECLASSIFICATION
No references to Megabank, target IBAN, 585171 or 142800 anywhere in report.

2022 ANNUAL — RELEVANT PAYMENT ROWS


,payment_type,id,payment_operation_date,payment_instruction_date,payment_amount,payer_account_iban,payer_name,payer_code,receiver_account_iban,receiver_name,receiver_code,payment_description,payment_purpose,payment_reason,refund_amount,refund_date
0,None,2581a7be-e7d3-48a6-b893-90c69fbcfb15,2022-05-24,None,70000,None,None,None,UA063516290000000026003240342,"ЧООПП ""Наш край""",39975897,None,Переведення коштів на власний в іншому банку. ...,Договір,None,None
1,None,edf1ecbb-42a0-43ea-9782-423444344c48,2022-05-24,None,72800,None,None,None,UA063516290000000026003240342,"ЧООПП ""Наш край""",39975897,None,Переведення коштів на власний в іншому банку. ...,Договір,None,None



2025 Q1 — NON-EMPTY TOP-LEVEL FINANCIAL SECTIONS

properties
{'property_moneys': 2}

payment_info
{'incoming': 4, 'outgoing': 4}

obligations []

organizations []


In [84]:
from pathlib import Path
import textwrap


# ============================================================
# FIND REPO ROOT
# ============================================================

def find_repo_root(start=Path.cwd()):
    start = start.resolve()

    for path in [start, *start.parents]:
        if (
            (path / "pyproject.toml").exists()
            and
            (path / "src" / "politdata").exists()
        ):
            return path

    raise RuntimeError(
        "Could not find politdata-pipeline repo root."
    )


ROOT = find_repo_root()

print("Repo root:", ROOT)


# ============================================================
# PATHS
# ============================================================

NORMALIZATION_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "normalization"
)

NORMALIZATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODULE_PATH = (
    NORMALIZATION_DIR
    / "accounts.py"
)

INIT_PATH = (
    NORMALIZATION_DIR
    / "__init__.py"
)

TEST_PATH = (
    ROOT
    / "tests"
    / "test_accounts.py"
)


# ============================================================
# MODULE
# ============================================================

MODULE_CODE = r'''
from __future__ import annotations

from dataclasses import dataclass
import re
import unicodedata
from typing import Any


# ============================================================
# CONSTANTS
# ============================================================

UKRAINIAN_IBAN_RE = re.compile(
    r"^UA\d{27}$"
)


ZERO_WIDTH_CHARS = {
    "\u200b",  # zero width space
    "\u200c",  # zero width non-joiner
    "\u200d",  # zero width joiner
    "\u2060",  # word joiner
    "\ufeff",  # BOM / zero width no-break space
}


SAFE_FORMATTING_CHARS = {
    "-",
    ".",
    "_",
    "/",
    "\\",
    ":",
}


# ============================================================
# RESULT OBJECT
# ============================================================

@dataclass(frozen=True)
class AccountNormalizationResult:
    """
    Result of conservative account-number normalization.

    raw:
        Original source value converted to string where possible.

    canonical:
        Checksum-valid canonical Ukrainian IBAN, or None.

    status:
        High-level normalization outcome.

    method:
        How canonical value was obtained.

    normalized_text:
        Conservative normalized representation useful for QA.
        This must NOT be used as a substitute for canonical IBAN.

    valid_iban:
        True only if canonical is a checksum-valid Ukrainian IBAN.

    candidate_count:
        Number of distinct checksum-valid IBANs detected.

    candidates:
        All distinct valid IBAN candidates detected in the source
        value. Normally zero or one. More than one is ambiguous.
    """

    raw: str | None
    canonical: str | None
    status: str
    method: str
    normalized_text: str | None
    valid_iban: bool
    candidate_count: int
    candidates: tuple[str, ...]


# ============================================================
# LOW-LEVEL NORMALIZATION
# ============================================================

def _remove_zero_width(text: str) -> str:
    return "".join(
        ch
        for ch in text
        if ch not in ZERO_WIDTH_CHARS
    )


def _unicode_normalize(text: str) -> str:
    return unicodedata.normalize(
        "NFKC",
        text,
    )


def _prepare_text(text: str) -> str:
    """
    Unicode-normalize, remove invisible formatting chars,
    uppercase, and strip outer whitespace.
    """

    text = _unicode_normalize(text)
    text = _remove_zero_width(text)
    text = text.upper()

    return text.strip()


def _conservative_display_text(
    text: str,
) -> str:
    """
    QA-friendly representation.

    Collapses whitespace but does NOT remove arbitrary punctuation
    or attempt to turn a non-IBAN identifier into an IBAN.
    """

    return re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


def _remove_safe_formatting(
    text: str,
) -> str:
    """
    Remove only formatting characters we have explicitly decided
    are safe to ignore inside a Ukrainian IBAN.
    """

    chars = []

    for ch in text:

        if ch.isspace():
            continue

        if ch in SAFE_FORMATTING_CHARS:
            continue

        chars.append(ch)

    return "".join(chars)


# ============================================================
# IBAN VALIDATION
# ============================================================

def is_valid_ua_iban(
    value: Any,
) -> bool:
    """
    Validate canonical Ukrainian IBAN using MOD-97.

    Ukrainian IBAN:
        UA + 27 digits
        total length = 29 characters.
    """

    if not isinstance(
        value,
        str,
    ):
        return False


    iban = value.upper()


    if not UKRAINIAN_IBAN_RE.fullmatch(
        iban
    ):
        return False


    rearranged = (
        iban[4:]
        + iban[:4]
    )


    remainder = 0


    for ch in rearranged:

        if ch.isdigit():
            digits = ch

        else:
            # A=10 ... Z=35
            digits = str(
                ord(ch) - 55
            )


        for digit in digits:

            remainder = (
                remainder * 10
                + int(digit)
            ) % 97


    return remainder == 1


# ============================================================
# CANDIDATE EXTRACTION
# ============================================================

def _candidate_from_ua_position(
    text: str,
    position: int,
) -> str | None:
    """
    Starting at a specific 'UA', conservatively read 27 digits.

    Safe formatting separators may occur between digits.

    If an unsupported character is encountered before 27 digits,
    extraction stops.

    If a 28th digit immediately follows the candidate, reject it
    instead of silently truncating the identifier.
    """

    if (
        position < 0
        or text[
            position:
            position + 2
        ] != "UA"
    ):
        return None


    digits = []

    i = position + 2


    while (
        i < len(text)
        and len(digits) < 27
    ):

        ch = text[i]


        if ch.isdigit():

            digits.append(ch)


        elif (
            ch.isspace()
            or ch in SAFE_FORMATTING_CHARS
        ):

            pass


        else:

            break


        i += 1


    if len(digits) != 27:
        return None


    # Reject identifiers that actually contain more than
    # 27 digits after UA.
    j = i

    while (
        j < len(text)
        and (
            text[j].isspace()
            or text[j]
            in SAFE_FORMATTING_CHARS
        )
    ):
        j += 1


    if (
        j < len(text)
        and text[j].isdigit()
    ):
        return None


    candidate = (
        "UA"
        + "".join(digits)
    )


    if is_valid_ua_iban(
        candidate
    ):
        return candidate


    return None


def _extract_valid_candidates(
    text: str,
) -> tuple[str, ...]:
    """
    Find all distinct checksum-valid Ukrainian IBANs in a value.

    This is deliberately conservative:
    only substrings beginning with explicit 'UA' are considered.
    """

    candidates = []


    for match in re.finditer(
        r"UA",
        text,
    ):

        candidate = (
            _candidate_from_ua_position(
                text,
                match.start(),
            )
        )


        if (
            candidate is not None
            and candidate not in candidates
        ):

            candidates.append(
                candidate
            )


    return tuple(
        candidates
    )


# ============================================================
# PUBLIC NORMALIZER
# ============================================================

def normalize_account_number(
    value: Any,
) -> AccountNormalizationResult:
    """
    Conservatively normalize an account-number field.

    Rules
    -----
    1. Preserve original source value.
    2. Unicode NFKC.
    3. Remove zero-width formatting characters.
    4. Uppercase.
    5. Accept only Ukrainian IBAN:
           UA + 27 digits.
    6. Validate using MOD-97.
    7. Safe formatting separators may be removed.
    8. Prefix before explicit UA may be discarded.
    9. If >1 distinct valid IBAN is detected:
           do NOT guess; return ambiguous.
    10. If validity cannot be proven:
           canonical remains None.

    The function normalizes representation, not financial meaning.
    """


    # --------------------------------------------------------
    # MISSING
    # --------------------------------------------------------

    if value is None:

        return AccountNormalizationResult(
            raw=None,
            canonical=None,
            status="missing",
            method="missing",
            normalized_text=None,
            valid_iban=False,
            candidate_count=0,
            candidates=(),
        )


    try:

        # pandas / numpy NA without importing pandas
        if value != value:

            return AccountNormalizationResult(
                raw=None,
                canonical=None,
                status="missing",
                method="missing",
                normalized_text=None,
                valid_iban=False,
                candidate_count=0,
                candidates=(),
            )

    except Exception:
        pass


    raw = str(value)


    if raw.strip() == "":

        return AccountNormalizationResult(
            raw=raw,
            canonical=None,
            status="missing",
            method="empty_string",
            normalized_text="",
            valid_iban=False,
            candidate_count=0,
            candidates=(),
        )


    prepared = _prepare_text(
        raw
    )


    display_text = (
        _conservative_display_text(
            prepared
        )
    )


    # --------------------------------------------------------
    # EXACT
    #
    # Raw source already exactly canonical.
    # --------------------------------------------------------

    raw_stripped = raw.strip()


    if (
        is_valid_ua_iban(
            raw_stripped
        )
        and raw_stripped
        ==
        raw_stripped.upper()
    ):

        canonical = (
            raw_stripped.upper()
        )

        return AccountNormalizationResult(
            raw=raw,
            canonical=canonical,
            status="valid",
            method="valid_exact",
            normalized_text=display_text,
            valid_iban=True,
            candidate_count=1,
            candidates=(
                canonical,
            ),
        )


    # --------------------------------------------------------
    # UNICODE / CASE NORMALIZATION ONLY
    # --------------------------------------------------------

    if is_valid_ua_iban(
        prepared
    ):

        return AccountNormalizationResult(
            raw=raw,
            canonical=prepared,
            status="valid",
            method="valid_normalized",
            normalized_text=display_text,
            valid_iban=True,
            candidate_count=1,
            candidates=(
                prepared,
            ),
        )


    # --------------------------------------------------------
    # SAFE FORMATTING CLEANUP
    #
    # Only use this shortcut when the normalized value itself
    # begins with UA. Prefix handling is classified separately.
    # --------------------------------------------------------

    if prepared.startswith(
        "UA"
    ):

        cleaned = (
            _remove_safe_formatting(
                prepared
            )
        )


        if is_valid_ua_iban(
            cleaned
        ):

            return AccountNormalizationResult(
                raw=raw,
                canonical=cleaned,
                status="valid",
                method=(
                    "valid_after_formatting_cleanup"
                ),
                normalized_text=display_text,
                valid_iban=True,
                candidate_count=1,
                candidates=(
                    cleaned,
                ),
            )


    # --------------------------------------------------------
    # EXPLICIT UA EXTRACTION
    #
    # Allows things such as:
    #
    #   :UA...
    #   № UA...
    #   рахунок UA...
    #
    # but never invents UA or repairs checksum errors.
    # --------------------------------------------------------

    candidates = (
        _extract_valid_candidates(
            prepared
        )
    )


    if len(candidates) == 1:

        canonical = candidates[0]

        return AccountNormalizationResult(
            raw=raw,
            canonical=canonical,
            status="valid",
            method=(
                "valid_after_prefix_removal"
            ),
            normalized_text=display_text,
            valid_iban=True,
            candidate_count=1,
            candidates=candidates,
        )


    # --------------------------------------------------------
    # AMBIGUOUS
    # --------------------------------------------------------

    if len(candidates) > 1:

        return AccountNormalizationResult(
            raw=raw,
            canonical=None,
            status=(
                "ambiguous_multiple_valid_ibans"
            ),
            method="ambiguous",
            normalized_text=display_text,
            valid_iban=False,
            candidate_count=len(
                candidates
            ),
            candidates=candidates,
        )


    # --------------------------------------------------------
    # INVALID / NON-STANDARD
    # --------------------------------------------------------

    return AccountNormalizationResult(
        raw=raw,
        canonical=None,
        status="invalid_or_nonstandard",
        method="no_valid_ua_iban",
        normalized_text=display_text,
        valid_iban=False,
        candidate_count=0,
        candidates=(),
    )


# ============================================================
# PANDAS HELPER
# ============================================================

def add_normalized_account_columns(
    df,
    source_col: str,
    prefix: str | None = None,
    *,
    copy: bool = True,
):
    """
    Add standard account-normalization columns to a pandas DataFrame.

    Example
    -------
    add_normalized_account_columns(
        df,
        source_col="payer_account_iban",
        prefix="payer_account",
    )

    Produces:
        payer_account_raw
        payer_account_canonical
        payer_account_normalization_status
        payer_account_normalization_method
        payer_account_valid_iban
        payer_account_candidate_count
        payer_account_candidates
        payer_account_normalized_text

    The original source column is preserved.
    """

    if source_col not in df.columns:

        raise KeyError(
            f"Column not found: {source_col}"
        )


    if prefix is None:

        prefix = source_col


    out = (
        df.copy()
        if copy
        else df
    )


    results = (
        out[source_col]
        .map(
            normalize_account_number
        )
    )


    out[
        f"{prefix}_raw"
    ] = results.map(
        lambda x:
            x.raw
    )


    out[
        f"{prefix}_canonical"
    ] = results.map(
        lambda x:
            x.canonical
    )


    out[
        f"{prefix}_normalization_status"
    ] = results.map(
        lambda x:
            x.status
    )


    out[
        f"{prefix}_normalization_method"
    ] = results.map(
        lambda x:
            x.method
    )


    out[
        f"{prefix}_valid_iban"
    ] = results.map(
        lambda x:
            x.valid_iban
    )


    out[
        f"{prefix}_candidate_count"
    ] = results.map(
        lambda x:
            x.candidate_count
    )


    out[
        f"{prefix}_candidates"
    ] = results.map(
        lambda x:
            list(
                x.candidates
            )
    )


    out[
        f"{prefix}_normalized_text"
    ] = results.map(
        lambda x:
            x.normalized_text
    )


    return out
'''


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ).lstrip(),
    encoding="utf-8",
)


# ============================================================
# PACKAGE EXPORTS
# ============================================================

INIT_CODE = r'''
from .accounts import (
    AccountNormalizationResult,
    add_normalized_account_columns,
    is_valid_ua_iban,
    normalize_account_number,
)

__all__ = [
    "AccountNormalizationResult",
    "add_normalized_account_columns",
    "is_valid_ua_iban",
    "normalize_account_number",
]
'''


INIT_PATH.write_text(
    textwrap.dedent(
        INIT_CODE
    ).lstrip(),
    encoding="utf-8",
)


# ============================================================
# TESTS
# ============================================================

TEST_CODE = r'''
import pandas as pd
import pytest

from politdata.normalization.accounts import (
    add_normalized_account_columns,
    is_valid_ua_iban,
    normalize_account_number,
)


IBAN_1 = "UA063516290000000026003240342"
IBAN_2 = "UA783204780000026005924904762"
IBAN_3 = "UA333052990000026002011039403"


def test_valid_ua_iban():
    assert is_valid_ua_iban(IBAN_1)
    assert is_valid_ua_iban(IBAN_2)
    assert is_valid_ua_iban(IBAN_3)


def test_invalid_checksum():
    invalid = (
        IBAN_1[:-1]
        + (
            "0"
            if IBAN_1[-1] != "0"
            else "1"
        )
    )

    assert not is_valid_ua_iban(
        invalid
    )


def test_exact():
    result = normalize_account_number(
        IBAN_1
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True
    assert result.status == "valid"
    assert result.method == "valid_exact"


@pytest.mark.parametrize(
    "raw",
    [
        IBAN_1.lower(),
        "\u200b" + IBAN_1,
        "\ufeff" + IBAN_1,
    ],
)
def test_unicode_case_normalization(
    raw,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True


@pytest.mark.parametrize(
    "raw",
    [
        "UA06 351629 000000 026003240342",
        "UA06\n351629000000026003240342",
        "UA06-351629-000000-026003240342",
        "UA06.351629.000000.026003240342",
        "UA06/351629/000000/026003240342",
    ],
)
def test_formatting_cleanup(
    raw,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True
    assert (
        result.method
        ==
        "valid_after_formatting_cleanup"
    )


@pytest.mark.parametrize(
    "raw",
    [
        ":" + IBAN_1,
        "№" + IBAN_1,
        "№ " + IBAN_1,
        "рахунок " + IBAN_1,
        "IBAN: " + IBAN_1,
    ],
)
def test_prefix_removal(
    raw,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True
    assert (
        result.method
        ==
        "valid_after_prefix_removal"
    )


def test_ambiguous_multiple_valid_ibans():
    raw = (
        f"{IBAN_1}; {IBAN_2}"
    )

    result = normalize_account_number(
        raw
    )

    assert result.canonical is None
    assert result.valid_iban is False
    assert (
        result.status
        ==
        "ambiguous_multiple_valid_ibans"
    )

    assert result.candidate_count == 2

    assert set(
        result.candidates
    ) == {
        IBAN_1,
        IBAN_2,
    }


def test_does_not_guess_invalid_identifier():
    result = normalize_account_number(
        "26003240342"
    )

    assert result.canonical is None
    assert result.valid_iban is False
    assert (
        result.status
        ==
        "invalid_or_nonstandard"
    )


def test_does_not_repair_bad_checksum():
    invalid = (
        IBAN_1[:-1]
        + (
            "0"
            if IBAN_1[-1] != "0"
            else "1"
        )
    )

    result = normalize_account_number(
        invalid
    )

    assert result.canonical is None
    assert result.valid_iban is False


@pytest.mark.parametrize(
    "raw",
    [
        None,
        "",
        "   ",
    ],
)
def test_missing(
    raw,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical is None
    assert result.status == "missing"


def test_dataframe_helper():
    df = pd.DataFrame(
        {
            "payer_account_iban": [
                IBAN_1,
                "№ " + IBAN_2,
                None,
            ]
        }
    )

    out = add_normalized_account_columns(
        df,
        source_col="payer_account_iban",
        prefix="payer_account",
    )

    assert (
        out.loc[
            0,
            "payer_account_canonical",
        ]
        ==
        IBAN_1
    )

    assert (
        out.loc[
            1,
            "payer_account_canonical",
        ]
        ==
        IBAN_2
    )

    assert pd.isna(
        out.loc[
            2,
            "payer_account_canonical",
        ]
    )

    assert (
        "payer_account_raw"
        in out.columns
    )

    assert (
        "payer_account_normalization_method"
        in out.columns
    )
'''


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ).lstrip(),
    encoding="utf-8",
)


# ============================================================
# RESULT
# ============================================================

print()
print("Created:")
print(" ", MODULE_PATH)
print(" ", INIT_PATH)
print(" ", TEST_PATH)

Repo root: C:\Users\Igor\Desktop\Projects\politdata-pipeline

Created:
  C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\accounts.py
  C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\__init__.py
  C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_accounts.py


In [85]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_accounts.py",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("Return code:", result.returncode)


C:\Users\Igor\miniforge3\envs\work\python.exe: No module named pytest

Return code: 1


In [86]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "pytest",
    ],
    check=True,
)

CompletedProcess(args=['C:\\Users\\Igor\\miniforge3\\envs\\work\\python.exe', '-m', 'pip', 'install', 'pytest'], returncode=0)

In [87]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_accounts.py",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("Return code:", result.returncode)

......FFFFF............                                                  [100%]
================================== FAILURES ===================================
__________ test_formatting_cleanup[UA06 351629 000000 026003240342] ___________

raw = 'UA06 351629 000000 026003240342'

    @pytest.mark.parametrize(
        "raw",
        [
            "UA06 351629 000000 026003240342",
            "UA06\n351629000000026003240342",
            "UA06-351629-000000-026003240342",
            "UA06.351629.000000.026003240342",
            "UA06/351629/000000/026003240342",
        ],
    )
    def test_formatting_cleanup(
        raw,
    ):
        result = normalize_account_number(
            raw
        )
    
>       assert result.canonical == IBAN_1
E       AssertionError: assert None == 'UA063516290000000026003240342'
E        +  where None = AccountNormalizationResult(raw='UA06 351629 000000 026003240342', canonical=None, status='invalid_or_nonstandard', met..._valid_ua_iban', normalize

In [89]:
from pathlib import Path
import subprocess
import sys
import textwrap


TEST_PATH = ROOT / "tests" / "test_accounts.py"


TEST_CODE = r'''
import pandas as pd
import pytest

from politdata.normalization.accounts import (
    add_normalized_account_columns,
    is_valid_ua_iban,
    normalize_account_number,
)


IBAN_1 = "UA063516290000000026003240342"
IBAN_2 = "UA783204780000026005924904762"
IBAN_3 = "UA333052990000026002011039403"


# ============================================================
# VALIDATION
# ============================================================

def test_valid_ua_iban():
    assert is_valid_ua_iban(IBAN_1)
    assert is_valid_ua_iban(IBAN_2)
    assert is_valid_ua_iban(IBAN_3)


def test_invalid_checksum():
    invalid = (
        IBAN_1[:-1]
        + (
            "0"
            if IBAN_1[-1] != "0"
            else "1"
        )
    )

    assert not is_valid_ua_iban(
        invalid
    )


# ============================================================
# EXACT / BASIC NORMALIZATION
# ============================================================

def test_exact():
    result = normalize_account_number(
        IBAN_1
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True
    assert result.status == "valid"
    assert result.method == "valid_exact"


@pytest.mark.parametrize(
    "raw",
    [
        IBAN_1.lower(),
        "\u200b" + IBAN_1,
        "\ufeff" + IBAN_1,
    ],
)
def test_unicode_case_normalization(
    raw,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True


# ============================================================
# SAFE FORMATTING CLEANUP
#
# Important:
# dirty forms are generated from a known-valid IBAN,
# so we never accidentally mistype a digit in the test fixture.
# ============================================================

@pytest.mark.parametrize(
    "separator",
    [
        " ",
        "\n",
        "\t",
        "-",
        ".",
        "/",
        "\\",
        ":",
    ],
)
def test_formatting_cleanup(
    separator,
):
    parts = [
        IBAN_1[:4],
        IBAN_1[4:10],
        IBAN_1[10:17],
        IBAN_1[17:],
    ]

    raw = separator.join(
        parts
    )

    result = normalize_account_number(
        raw
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True
    assert (
        result.method
        ==
        "valid_after_formatting_cleanup"
    )


# ============================================================
# PREFIX REMOVAL
# ============================================================

@pytest.mark.parametrize(
    "raw",
    [
        ":" + IBAN_1,
        "№" + IBAN_1,
        "№ " + IBAN_1,
        "рахунок " + IBAN_1,
        "IBAN: " + IBAN_1,
    ],
)
def test_prefix_removal(
    raw,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical == IBAN_1
    assert result.valid_iban is True
    assert (
        result.method
        ==
        "valid_after_prefix_removal"
    )


# ============================================================
# CONSERVATIVE SAFETY RULES
# ============================================================

def test_missing_digit_is_not_repaired():
    malformed = (
        IBAN_1[:10]
        + IBAN_1[11:]
    )

    result = normalize_account_number(
        malformed
    )

    assert result.canonical is None
    assert result.valid_iban is False
    assert (
        result.status
        ==
        "invalid_or_nonstandard"
    )


def test_extra_digit_is_not_truncated():
    malformed = (
        IBAN_1
        + "5"
    )

    result = normalize_account_number(
        malformed
    )

    assert result.canonical is None
    assert result.valid_iban is False


def test_does_not_guess_invalid_identifier():
    result = normalize_account_number(
        "26003240342"
    )

    assert result.canonical is None
    assert result.valid_iban is False
    assert (
        result.status
        ==
        "invalid_or_nonstandard"
    )


def test_does_not_repair_bad_checksum():
    invalid = (
        IBAN_1[:-1]
        + (
            "0"
            if IBAN_1[-1] != "0"
            else "1"
        )
    )

    result = normalize_account_number(
        invalid
    )

    assert result.canonical is None
    assert result.valid_iban is False


# ============================================================
# AMBIGUITY
# ============================================================

def test_ambiguous_multiple_valid_ibans():
    raw = (
        f"{IBAN_1}; {IBAN_2}"
    )

    result = normalize_account_number(
        raw
    )

    assert result.canonical is None
    assert result.valid_iban is False

    assert (
        result.status
        ==
        "ambiguous_multiple_valid_ibans"
    )

    assert result.candidate_count == 2

    assert set(
        result.candidates
    ) == {
        IBAN_1,
        IBAN_2,
    }


# ============================================================
# MISSING VALUES
# ============================================================

@pytest.mark.parametrize(
    "raw",
    [
        None,
        "",
        "   ",
    ],
)
def test_missing(
    raw,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical is None
    assert result.status == "missing"


# ============================================================
# DATAFRAME HELPER
# ============================================================

def test_dataframe_helper():
    df = pd.DataFrame(
        {
            "payer_account_iban": [
                IBAN_1,
                "№ " + IBAN_2,
                None,
            ]
        }
    )

    out = add_normalized_account_columns(
        df,
        source_col="payer_account_iban",
        prefix="payer_account",
    )

    assert (
        out.loc[
            0,
            "payer_account_canonical",
        ]
        ==
        IBAN_1
    )

    assert (
        out.loc[
            1,
            "payer_account_canonical",
        ]
        ==
        IBAN_2
    )

    assert pd.isna(
        out.loc[
            2,
            "payer_account_canonical",
        ]
    )

    assert (
        "payer_account_raw"
        in out.columns
    )

    assert (
        "payer_account_normalization_method"
        in out.columns
    )


def test_dataframe_helper_preserves_source_column():
    df = pd.DataFrame(
        {
            "account_number": [
                "№ " + IBAN_1,
            ]
        }
    )

    original = df[
        "account_number"
    ].copy()

    out = add_normalized_account_columns(
        df,
        source_col="account_number",
        prefix="account",
    )

    assert (
        out["account_number"]
        .equals(original)
    )

    assert (
        out.loc[
            0,
            "account_raw",
        ]
        ==
        "№ " + IBAN_1
    )

    assert (
        out.loc[
            0,
            "account_canonical",
        ]
        ==
        IBAN_1
    )
'''


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ).lstrip(),
    encoding="utf-8",
)


print("Rewritten:", TEST_PATH)


# ============================================================
# RUN TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_accounts.py",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print(result.stdout)

if result.stderr:
    print(result.stderr)

print(
    "Return code:",
    result.returncode
)

Rewritten: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_accounts.py

.............................                                            [100%]
29 passed in 0.67s

Return code: 0


In [90]:
from pathlib import Path
import json

import pandas as pd

from politdata.normalization.accounts import (
    normalize_account_number,
)


ROOT = Path(r"C:\Users\Igor\Desktop\Projects\politdata-pipeline")

SELECTED_PATH = (
    ROOT
    / "data"
    / "interim"
    / "reports"
    / "selected_reports_manifest.parquet"
)

RAW_DIR = (
    ROOT
    / "data"
    / "raw"
    / "report_details"
)


selected = pd.read_parquet(
    SELECTED_PATH
)

selected["report_id"] = (
    selected["report_id"].astype(str)
)


rows = []


for report_id in selected["report_id"]:

    path = RAW_DIR / f"{report_id}.json"

    if not path.exists():
        continue

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        detail = json.load(f)["results"]

    money_rows = (
        (
            detail.get("properties")
            or {}
        )
        .get("property_moneys")
        or []
    )

    for row in money_rows:

        raw = row.get(
            "account_number"
        )

        result = normalize_account_number(
            raw
        )

        rows.append(
            {
                "report_id":
                    report_id,

                "account_number_raw":
                    raw,

                "account_number_canonical":
                    result.canonical,

                "normalization_status":
                    result.status,

                "normalization_method":
                    result.method,

                "valid_iban":
                    result.valid_iban,

                "candidate_count":
                    result.candidate_count,
            }
        )


qa = pd.DataFrame(
    rows
)


print("=" * 80)
print("PROPERTY_MONEYS NORMALIZER REGRESSION")
print("=" * 80)

print(
    "Rows:",
    len(qa)
)

print(
    "Distinct raw spellings:",
    qa["account_number_raw"].nunique(
        dropna=True
    )
)

print(
    "Distinct canonical IBANs:",
    qa["account_number_canonical"].nunique(
        dropna=True
    )
)

print(
    "Valid IBAN rows:",
    int(
        qa["valid_iban"].sum()
    )
)

print(
    "Invalid / nonstandard rows:",
    int(
        (
            qa["normalization_status"]
            ==
            "invalid_or_nonstandard"
        ).sum()
    )
)

print(
    "Ambiguous rows:",
    int(
        (
            qa["normalization_status"]
            ==
            "ambiguous_multiple_valid_ibans"
        ).sum()
    )
)


print()
print("NORMALIZATION METHODS")

print(
    qa[
        "normalization_method"
    ].value_counts(
        dropna=False
    )
)


print()
print("INVALID / NONSTANDARD EXAMPLES")

display(
    qa[
        qa[
            "normalization_status"
        ]
        ==
        "invalid_or_nonstandard"
    ].head(50)
)


print()
print("AMBIGUOUS EXAMPLES")

display(
    qa[
        qa[
            "normalization_status"
        ]
        ==
        "ambiguous_multiple_valid_ibans"
    ].head(50)
)

PROPERTY_MONEYS NORMALIZER REGRESSION
Rows: 19138
Distinct raw spellings: 1805
Distinct canonical IBANs: 1664
Valid IBAN rows: 19138
Invalid / nonstandard rows: 0
Ambiguous rows: 0

NORMALIZATION METHODS
normalization_method
valid_exact                       18613
valid_after_formatting_cleanup      432
valid_after_prefix_removal           83
valid_normalized                     10
Name: count, dtype: int64

INVALID / NONSTANDARD EXAMPLES


,report_id,account_number_raw,account_number_canonical,normalization_status,normalization_method,valid_iban,candidate_count



AMBIGUOUS EXAMPLES


,report_id,account_number_raw,account_number_canonical,normalization_status,normalization_method,valid_iban,candidate_count


In [92]:
from pathlib import Path
import pandas as pd

AUDIT_DIR = ROOT / "data" / "interim" / "audit"

path = (
    AUDIT_DIR
    / "selected_account_snapshots_normalized.parquet"
)

snapshots = pd.read_parquet(path)

print("=" * 70)
print("EXISTING SUBSTANTIVE SNAPSHOT REGRESSION")
print("=" * 70)

print("Rows:", len(snapshots))

# знайдемо колонку canonical IBAN незалежно від
# невеликої різниці в назвах між версіями audit-коду
canonical_candidates = [
    "account_number_canonical",
    "account_number_normalized",
    "canonical_iban",
]

canonical_col = next(
    (
        col
        for col in canonical_candidates
        if col in snapshots.columns
    ),
    None,
)

if canonical_col is not None:
    print(
        "Canonical column:",
        canonical_col,
    )
    print(
        "Distinct canonical IBANs:",
        snapshots[canonical_col].nunique(
            dropna=True
        ),
    )
    print(
        "Missing canonical IBAN:",
        snapshots[canonical_col].isna().sum(),
    )
else:
    print(
        "Canonical column not found."
    )
    print(
        "Available columns:",
        snapshots.columns.tolist(),
    )

EXISTING SUBSTANTIVE SNAPSHOT REGRESSION
Rows: 12588
Canonical column: account_number_normalized
Distinct canonical IBANs: 1679
Missing canonical IBAN: 0


In [93]:
from politdata.normalization.accounts import normalize_account_number

snapshots2 = snapshots.copy()

snapshots2["account_number_canonical_new"] = (
    snapshots2["account_number_raw"]
    .map(
        lambda x:
            normalize_account_number(x).canonical
    )
)

print("Rows:", len(snapshots2))

print(
    "Old normalized IDs:",
    snapshots2[
        "account_number_normalized"
    ].nunique(dropna=True)
)

print(
    "New canonical IBANs:",
    snapshots2[
        "account_number_canonical_new"
    ].nunique(dropna=True)
)

print(
    "New missing canonical:",
    snapshots2[
        "account_number_canonical_new"
    ].isna().sum()
)

Rows: 12588
Old normalized IDs: 1679
New canonical IBANs: 1664
New missing canonical: 0


In [94]:
mapping = (
    snapshots2[
        [
            "account_number_raw",
            "account_number_normalized",
            "account_number_canonical_new",
        ]
    ]
    .drop_duplicates()
)

collapsed = (
    mapping
    .groupby(
        "account_number_canonical_new",
        dropna=False,
    )
    .agg(
        old_normalized_count=(
            "account_number_normalized",
            "nunique",
        ),
        raw_spelling_count=(
            "account_number_raw",
            "nunique",
        ),
        old_values=(
            "account_number_normalized",
            lambda s:
                " | ".join(
                    sorted(
                        set(
                            str(x)
                            for x in s
                            if x is not None
                        )
                    )
                ),
        ),
    )
    .reset_index()
)

display(
    collapsed[
        collapsed[
            "old_normalized_count"
        ] > 1
    ]
    .sort_values(
        "old_normalized_count",
        ascending=False,
    )
)

,account_number_canonical_new,old_normalized_count,raw_spelling_count,old_values
138,UA093366770000026007052508112,2,2,:UA093366770000026007052508112 | UA09336677000...
486,UA303052990000026002043302142,2,2,NOUA303052990000026002043302142 | UA3030529900...
555,UA333281680000000026000283851,2,2,:UA333281680000000026000283851 | UA33328168000...
619,UA373516290000000026007261695,2,2,:UA373516290000000026007261695 | UA37351629000...
688,UA423204780000026007212004327,2,2,NOUA423204780000026007212004327 | UA4232047800...
733,UA453052990000035705036306227,2,3,NOUA453052990000035705036306227 | UA4530529900...
765,UA473052990000026048016215059,2,3,NOUA473052990000026048016215059 | UA4730529900...
806,UA493052990000026044050219770,2,2,:UA493052990000026044050219770 | UA49305299000...
841,UA513052990000026008046304486,2,2,NOUA513052990000026008046304486 | UA5130529900...
940,UA563314010000026002300724001,2,2,:UA563314010000026002300724001 | UA56331401000...


In [96]:
from pathlib import Path
import subprocess
import sys


TEST_PATH = ROOT / "tests" / "test_accounts.py"

text = TEST_PATH.read_text(
    encoding="utf-8"
)

TEST_NAME = "test_real_world_prefix_variants"


if TEST_NAME not in text:

    addition = r'''


# ============================================================
# REAL-WORLD PREFIX VARIANTS FROM POLITDATA
# ============================================================

@pytest.mark.parametrize(
    "raw, expected",
    [
        (
            ":UA093366770000026007052508112",
            "UA093366770000026007052508112",
        ),
        (
            "NOUA303052990000026002043302142",
            "UA303052990000026002043302142",
        ),
    ],
)
def test_real_world_prefix_variants(
    raw,
    expected,
):
    result = normalize_account_number(
        raw
    )

    assert result.canonical == expected
    assert result.valid_iban is True
    assert (
        result.method
        ==
        "valid_after_prefix_removal"
    )
'''

    TEST_PATH.write_text(
        text + addition,
        encoding="utf-8",
    )

    print(
        "Added real-world prefix tests."
    )

else:
    print(
        "Real-world prefix tests already present."
    )


# ============================================================
# RUN TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_accounts.py",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print()
print(result.stdout)

if result.stderr:
    print(result.stderr)

print(
    "Return code:",
    result.returncode
)

Added real-world prefix tests.

...............................                                          [100%]
31 passed in 8.56s

Return code: 0


In [97]:
from pathlib import Path
from collections import defaultdict, Counter
import json
import random

import pandas as pd


ROOT = Path(r"C:\Users\Igor\Desktop\Projects\politdata-pipeline")

REPORTS_DIR = (
    ROOT
    / "data"
    / "interim"
    / "reports"
)

STATE_DIR = (
    ROOT
    / "data"
    / "interim"
    / "state"
)

RAW_DIR = (
    ROOT
    / "data"
    / "raw"
    / "report_details"
)

OUT_DIR = (
    ROOT
    / "data"
    / "interim"
    / "schema"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD MANIFEST + DETAIL STATE
# ============================================================

selected = pd.read_parquet(
    REPORTS_DIR
    / "selected_reports_manifest.parquet"
).copy()

selected["report_id"] = (
    selected["report_id"].astype(str)
)


state_path = (
    STATE_DIR
    / "report_detail_state.parquet"
)

state = (
    pd.read_parquet(state_path)
    if state_path.exists()
    else pd.DataFrame()
)

if len(state) and "report_id" in state.columns:
    state["report_id"] = (
        state["report_id"].astype(str)
    )


# ============================================================
# BUILD SMALL BUT DIVERSE SAMPLE
# ============================================================

sample_ids = set()


# ------------------------------------------------------------
# 1. Stratified sample by year / quarter / central-office
# ------------------------------------------------------------

group_cols = [
    col
    for col in [
        "year",
        "quarter",
        "is_party_office",
    ]
    if col in selected.columns
]


if not group_cols:
    group_cols = [
        "year",
        "quarter",
    ]


for _, group in selected.groupby(
    group_cols,
    dropna=False,
):

    # At most 3 reports per stratum.
    take = min(
        3,
        len(group),
    )

    sampled = group.sample(
        n=take,
        random_state=42,
    )

    sample_ids.update(
        sampled["report_id"]
    )


# ------------------------------------------------------------
# 2. Largest report-detail files
#
# Useful because large reports tend to expose more nested schema.
# ------------------------------------------------------------

if (
    len(state)
    and "file_size" in state.columns
):

    largest = (
        state[
            state.get(
                "status",
                "success",
            )
            == "success"
        ]
        .sort_values(
            "file_size",
            ascending=False,
        )
        .head(75)
    )

    sample_ids.update(
        largest["report_id"]
    )


# ------------------------------------------------------------
# 3. Reports known to contain paper assets
# ------------------------------------------------------------

if (
    len(state)
    and "property_paper_count"
        in state.columns
):

    paper = (
        state[
            pd.to_numeric(
                state[
                    "property_paper_count"
                ],
                errors="coerce",
            ).fillna(0) > 0
        ]
        .head(30)
    )

    sample_ids.update(
        paper["report_id"]
    )


sample_ids = sorted(
    sample_ids
)


print(
    "Schema sample reports:",
    len(sample_ids)
)


# ============================================================
# GENERIC SCHEMA PROFILER
# ============================================================

field_stats = defaultdict(
    lambda: {
        "containers": Counter(),
        "value_types": Counter(),
        "non_null_count": 0,
        "examples": [],
        "report_ids": set(),
    }
)


def typename(value):

    if value is None:
        return "null"

    if isinstance(value, bool):
        return "bool"

    if isinstance(value, int):
        return "int"

    if isinstance(value, float):
        return "float"

    if isinstance(value, str):
        return "str"

    if isinstance(value, list):
        return "list"

    if isinstance(value, dict):
        return "dict"

    return type(value).__name__


def add_example(stats, value):

    if value is None:
        return

    if isinstance(
        value,
        (dict, list),
    ):
        return

    text = str(value)

    if len(text) > 180:
        text = text[:177] + "..."

    if text not in stats["examples"]:
        if len(stats["examples"]) < 5:
            stats["examples"].append(
                text
            )


def walk(
    value,
    path,
    report_id,
):

    stats = field_stats[path]

    stats["value_types"][
        typename(value)
    ] += 1

    stats["report_ids"].add(
        report_id
    )

    if value is not None:
        stats[
            "non_null_count"
        ] += 1

    add_example(
        stats,
        value,
    )


    # --------------------------------------------------------
    # DICT
    # --------------------------------------------------------

    if isinstance(value, dict):

        stats["containers"][
            "dict"
        ] += 1

        for key, child in value.items():

            child_path = (
                f"{path}.{key}"
                if path
                else key
            )

            walk(
                child,
                child_path,
                report_id,
            )


    # --------------------------------------------------------
    # LIST
    #
    # Treat all rows as one conceptual [] path,
    # rather than [0], [1], [2]...
    # --------------------------------------------------------

    elif isinstance(value, list):

        stats["containers"][
            "list"
        ] += 1

        for child in value:

            child_path = (
                f"{path}[]"
            )

            walk(
                child,
                child_path,
                report_id,
            )


# ============================================================
# READ ONLY SAMPLE RAW
# ============================================================

read_count = 0
missing_count = 0


for report_id in sample_ids:

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if not path.exists():

        missing_count += 1
        continue


    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        payload = json.load(f)


    results = (
        payload.get("results")
        or {}
    )


    walk(
        results,
        "",
        report_id,
    )

    read_count += 1


print(
    "RAW files read:",
    read_count
)

print(
    "Missing sample RAW:",
    missing_count
)


# ============================================================
# FLATTEN INVENTORY
# ============================================================

inventory_rows = []


for path, stats in field_stats.items():

    if path == "":
        continue


    inventory_rows.append(
        {
            "json_path":
                path,

            "observed_types":
                " | ".join(
                    sorted(
                        stats[
                            "value_types"
                        ]
                    )
                ),

            "non_null_observations":
                stats[
                    "non_null_count"
                ],

            "sample_report_count":
                len(
                    stats[
                        "report_ids"
                    ]
                ),

            "examples":
                " || ".join(
                    stats[
                        "examples"
                    ]
                ),
        }
    )


inventory = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        "json_path"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# DERIVE SECTION
# ============================================================

def section_from_path(path):

    parts = path.split(".")

    if path.startswith(
        "payment_info."
    ):

        # payment_info.incoming.other_incomes[]
        if len(parts) >= 3:
            return ".".join(
                parts[:3]
            )


    if path.startswith(
        "properties."
    ):

        if len(parts) >= 2:
            return ".".join(
                parts[:2]
            )


    return parts[0]


inventory[
    "section"
] = inventory[
    "json_path"
].map(
    section_from_path
)


# ============================================================
# SAVE
# ============================================================

inventory_path = (
    OUT_DIR
    / "report_detail_schema_inventory.parquet"
)

inventory_csv = (
    OUT_DIR
    / "report_detail_schema_inventory.csv"
)


inventory.to_parquet(
    inventory_path,
    index=False,
)

inventory.to_csv(
    inventory_csv,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# HUMAN-READABLE SECTION SUMMARY
# ============================================================

section_summary = (
    inventory
    .groupby(
        "section",
        as_index=False,
    )
    .agg(
        observed_paths=(
            "json_path",
            "count",
        ),

        sample_report_count=(
            "sample_report_count",
            "max",
        ),
    )
    .sort_values(
        "section"
    )
)


section_summary.to_csv(
    OUT_DIR
    / "report_detail_section_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


print()
print("=" * 80)
print("SECTIONS FOUND")
print("=" * 80)

display(
    section_summary
)


print()
print("=" * 80)
print("PAYMENT SECTIONS")
print("=" * 80)

display(
    inventory[
        inventory[
            "section"
        ].str.startswith(
            "payment_info.",
            na=False,
        )
    ][
        [
            "section",
            "json_path",
            "observed_types",
            "examples",
        ]
    ]
)


print()
print("=" * 80)
print("PROPERTY / STATE SECTIONS")
print("=" * 80)

display(
    inventory[
        (
            inventory[
                "section"
            ].str.startswith(
                "properties.",
                na=False,
            )
        )
        |
        (
            inventory[
                "section"
            ].isin(
                [
                    "head_info",
                    "obligations",
                    "organizations",
                    "regional_offices",
                ]
            )
        )
    ][
        [
            "section",
            "json_path",
            "observed_types",
            "examples",
        ]
    ]
)

Schema sample reports: 153
RAW files read: 153
Missing sample RAW: 0

SECTIONS FOUND


,section,observed_paths,sample_report_count
0,conclusion,1,153
1,created_date,1,153
2,employees_by_civil_contract,1,153
3,employees_by_employment_contract,1,153
4,head_info,4,153
5,id,1,153
6,is_party_office,1,153
7,obligations,1,153
8,obligations[],15,10
9,organizations,1,153



PAYMENT SECTIONS


,section,json_path,observed_types,examples
33,payment_info.incoming.monetary_contributions,payment_info.incoming.monetary_contributions,list,
34,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[],dict,
35,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[]...,str,2023-12-25 10:27:10 || 2026-02-23 22:04:28 || ...
36,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[]...,str,3_1
37,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[].id,str,c046d386-c70f-4c43-9778-f9c4e646e5ce || cd154d...
38,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[]...,null | str,9b1c3511-1fcc-4091-bdcd-97930d503b65 || 3f9b18...
39,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[]...,str,79cb7831-508b-466a-8db0-db9c5f4080b2 || 7d9a45...
40,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[]...,null,
41,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[]...,null,
42,payment_info.incoming.monetary_contributions[],payment_info.incoming.monetary_contributions[]...,str,[не визначено] || Хмельницька область || Украї...



PROPERTY / STATE SECTIONS


,section,json_path,observed_types,examples
4,head_info,head_info,dict,
5,head_info,head_info.name,str,Віталій || Василь || Олександр || Сергій || Во...
6,head_info,head_info.patronymic,str,Володимирович || Костянтинович || Васильович |...
7,head_info,head_info.surname,str,Смірнов || Бичківський || Коваленко || Железня...
10,obligations,obligations,list,
26,organizations,organizations,list,
219,properties.property_intangible_asset,properties.property_intangible_asset,list,
220,properties.property_intangible_asset[],properties.property_intangible_asset[],dict,
221,properties.property_intangible_asset[],properties.property_intangible_asset[].asset_c...,int,1 || 6 || 3
222,properties.property_intangible_asset[],properties.property_intangible_asset[].asset_d...,str,Програмне забезпечення 1С:Бухгалтерія 8 для Ук...


In [99]:
from pathlib import Path
from collections import defaultdict
import json
import re
import shutil
import subprocess
import mmap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

RAW_DIR = (
    ROOT
    / "data"
    / "raw"
    / "report_details"
)

STATE_DIR = (
    ROOT
    / "data"
    / "interim"
    / "state"
)

SCHEMA_DIR = (
    ROOT
    / "data"
    / "interim"
    / "schema"
)

SCHEMA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD EXISTING SCHEMA INVENTORY
# ============================================================

inventory_path = (
    SCHEMA_DIR
    / "report_detail_schema_inventory.parquet"
)

inventory = pd.read_parquet(
    inventory_path
)


# ============================================================
# HELPER:
# DISPLAY LEAF FIELDS ALREADY FOUND
#
# Our previous "section" logic separated e.g.
# obligations[] from obligations.
# Here we use json_path directly.
# ============================================================

def show_existing_schema(
    title,
    prefix,
):
    print()
    print("=" * 90)
    print(title)
    print("=" * 90)

    part = inventory[
        inventory[
            "json_path"
        ].str.startswith(
            prefix,
            na=False,
        )
    ][
        [
            "json_path",
            "observed_types",
            "non_null_observations",
            "sample_report_count",
            "examples",
        ]
    ].copy()

    display(part)


show_existing_schema(
    "OBLIGATIONS — EXISTING SCHEMA",
    "obligations[]",
)

show_existing_schema(
    "ORGANIZATIONS — EXISTING SCHEMA",
    "organizations[]",
)

show_existing_schema(
    "REGIONAL OFFICES — EXISTING SCHEMA",
    "regional_offices[]",
)


# ============================================================
# TARGET SECTIONS THAT WERE EMPTY IN FIRST SAMPLE
# ============================================================

TARGETS = {
    "state_funding": (
        "payment_info",
        "incoming",
        "state_funding",
    ),

    "budget_expenses": (
        "payment_info",
        "outgoing",
        "budget_expenses",
    ),

    "return_expenses": (
        "payment_info",
        "outgoing",
        "return_expenses",
    ),

    "transfer_expenses": (
        "payment_info",
        "outgoing",
        "transfer_expenses",
    ),

    "property_paper": (
        "properties",
        "property_paper",
    ),
}


# We only need enough examples to understand schema.
TARGET_REPORTS_PER_SECTION = 2

# Maximum candidate files returned for one target.
# These are NOT all opened automatically:
# each file is verified and we stop after 2 real examples.
MAX_CANDIDATES_PER_TARGET = 30

# Only relevant when ripgrep is not installed.
# Hard stop prevents another accidental huge RAW traversal.
FALLBACK_MAX_FILES = 4000


# ============================================================
# LOAD DETAIL STATE
#
# Used for:
# - fast property_paper lookup
# - bounded fallback ordered by file size
# ============================================================

state_path = (
    STATE_DIR
    / "report_detail_state.parquet"
)

state = (
    pd.read_parquet(state_path)
    if state_path.exists()
    else pd.DataFrame()
)

if (
    len(state)
    and "report_id" in state.columns
):
    state["report_id"] = (
        state["report_id"]
        .astype(str)
    )


# ============================================================
# BASIC HELPERS
# ============================================================

def get_nested_list(
    obj,
    path,
):
    current = obj

    for key in path:

        if not isinstance(
            current,
            dict,
        ):
            return []

        current = current.get(
            key
        )

    if isinstance(
        current,
        list,
    ):
        return current

    return []


def read_report_results(
    path,
):
    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        payload = json.load(f)

    return (
        payload.get("results")
        or {}
    )


def typename(value):

    if value is None:
        return "null"

    if isinstance(value, bool):
        return "bool"

    if isinstance(value, int):
        return "int"

    if isinstance(value, float):
        return "float"

    if isinstance(value, str):
        return "str"

    if isinstance(value, list):
        return "list"

    if isinstance(value, dict):
        return "dict"

    return type(value).__name__


# ============================================================
# FAST TEXT SEARCH
#
# Search specifically for:
#
#   "state_funding": [
#       {
#
# rather than merely the field name.
#
# Therefore empty arrays [] do not count.
# ============================================================

RG = shutil.which(
    "rg"
)


def find_candidates_with_rg(
    key,
    max_candidates=30,
):
    """
    Native search through RAW using ripgrep.

    Important:
    - does NOT JSON-parse the corpus;
    - streams filenames;
    - terminates as soon as enough candidates are found.
    """

    pattern = (
        '"'
        + re.escape(key)
        + r'"\s*:\s*\[\s*\{'
    )

    cmd = [
        RG,
        "-l",
        "-U",
        "-P",
        "--no-messages",
        "-g",
        "*.json",
        pattern,
        str(RAW_DIR),
    ]

    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )

    paths = []

    try:

        for line in proc.stdout:

            line = line.strip()

            if not line:
                continue

            path = Path(line)

            if path.exists():
                paths.append(path)

            if (
                len(paths)
                >= max_candidates
            ):
                proc.terminate()
                break

    finally:

        if proc.stdout:
            proc.stdout.close()

        try:
            proc.wait(
                timeout=2
            )
        except subprocess.TimeoutExpired:
            proc.kill()

    return paths


# ============================================================
# FALLBACK SEARCH
#
# Only used when ripgrep is unavailable.
#
# Still:
# - NO json.load()
# - mmap byte search
# - hard file cap
# ============================================================

def fallback_file_order():

    if (
        len(state)
        and "report_id"
            in state.columns
    ):

        tmp = state.copy()

        if "status" in tmp.columns:

            tmp = tmp[
                tmp["status"]
                == "success"
            ]

        if "file_size" in tmp.columns:

            tmp = tmp.sort_values(
                "file_size",
                ascending=False,
            )

        ids = (
            tmp[
                "report_id"
            ]
            .drop_duplicates()
            .head(
                FALLBACK_MAX_FILES
            )
        )

        paths = [
            RAW_DIR
            / f"{report_id}.json"
            for report_id in ids
        ]

        return [
            p
            for p in paths
            if p.exists()
        ]

    # Last-resort bounded fallback.
    return list(
        RAW_DIR.glob(
            "*.json"
        )
    )[
        :FALLBACK_MAX_FILES
    ]


def find_candidates_fallback(
    key,
    max_candidates=30,
):

    pattern = re.compile(
        (
            rb'"'
            + re.escape(
                key.encode(
                    "utf-8"
                )
            )
            + rb'"\s*:\s*\[\s*\{'
        )
    )

    found = []

    for path in fallback_file_order():

        try:

            if path.stat().st_size == 0:
                continue

            with path.open(
                "rb"
            ) as f:

                with mmap.mmap(
                    f.fileno(),
                    0,
                    access=mmap.ACCESS_READ,
                ) as mm:

                    if pattern.search(mm):

                        found.append(
                            path
                        )

                        if (
                            len(found)
                            >= max_candidates
                        ):
                            break

        except (
            OSError,
            ValueError,
        ):
            continue

    return found


# ============================================================
# PAPER: USE STATE FIRST
#
# We already calculated property_paper_count during ingestion,
# so do not search RAW if state can tell us directly.
# ============================================================

def paper_candidates_from_state(
    limit=30,
):

    if (
        not len(state)
        or
        "property_paper_count"
        not in state.columns
        or
        "report_id"
        not in state.columns
    ):
        return []

    counts = pd.to_numeric(
        state[
            "property_paper_count"
        ],
        errors="coerce",
    ).fillna(0)

    tmp = state[
        counts > 0
    ].copy()

    if "status" in tmp.columns:

        tmp = tmp[
            tmp["status"]
            == "success"
        ]

    paths = []

    for report_id in (
        tmp["report_id"]
        .drop_duplicates()
        .head(limit)
    ):

        path = (
            RAW_DIR
            / f"{report_id}.json"
        )

        if path.exists():
            paths.append(
                path
            )

    return paths


# ============================================================
# SEARCH + VERIFY
#
# Text search only identifies candidate JSONs.
# We then JSON-load ONLY those candidates
# and confirm the requested nested section is really non-empty.
# ============================================================

found_reports = {
    name: []
    for name in TARGETS
}

rows_by_target = {
    name: []
    for name in TARGETS
}

candidate_counts = {}

json_files_actually_loaded = 0


print()
print("=" * 90)
print("TARGETED NON-EMPTY SECTION SEARCH")
print("=" * 90)

print(
    "Search engine:",
    (
        f"ripgrep: {RG}"
        if RG
        else (
            "bounded mmap fallback "
            f"(max {FALLBACK_MAX_FILES} files)"
        )
    ),
)


for name, target_path in TARGETS.items():

    # --------------------------------------------------------
    # Candidate discovery
    # --------------------------------------------------------

    if name == "property_paper":

        candidates = (
            paper_candidates_from_state(
                MAX_CANDIDATES_PER_TARGET
            )
        )

        search_method = (
            "report_detail_state.property_paper_count"
        )

        # Only if state does not provide examples,
        # fall back to text search.
        if not candidates:

            if RG:

                candidates = (
                    find_candidates_with_rg(
                        "property_paper",
                        MAX_CANDIDATES_PER_TARGET,
                    )
                )

                search_method = (
                    "ripgrep"
                )

            else:

                candidates = (
                    find_candidates_fallback(
                        "property_paper",
                        MAX_CANDIDATES_PER_TARGET,
                    )
                )

                search_method = (
                    "bounded mmap fallback"
                )

    else:

        if RG:

            candidates = (
                find_candidates_with_rg(
                    name,
                    MAX_CANDIDATES_PER_TARGET,
                )
            )

            search_method = (
                "ripgrep"
            )

        else:

            candidates = (
                find_candidates_fallback(
                    name,
                    MAX_CANDIDATES_PER_TARGET,
                )
            )

            search_method = (
                "bounded mmap fallback"
            )


    candidate_counts[
        name
    ] = len(
        candidates
    )


    # --------------------------------------------------------
    # Verify candidates by actual nested path
    # --------------------------------------------------------

    for path in candidates:

        if (
            len(
                found_reports[
                    name
                ]
            )
            >= TARGET_REPORTS_PER_SECTION
        ):
            break

        try:

            detail = (
                read_report_results(
                    path
                )
            )

            json_files_actually_loaded += 1

        except Exception as exc:

            print(
                f"Could not parse "
                f"{path.name}: "
                f"{exc}"
            )

            continue


        rows = get_nested_list(
            detail,
            target_path,
        )


        if not rows:
            continue


        found_reports[
            name
        ].append(
            path.stem
        )


        # Maximum 100 rows from each example report
        # is enough for schema/type mapping.
        for row in rows[:100]:

            if isinstance(
                row,
                dict,
            ):

                rows_by_target[
                    name
                ].append(
                    row
                )


    print(
        f"{name:20s} | "
        f"method={search_method} | "
        f"candidates={len(candidates)} | "
        f"verified_reports="
        f"{len(found_reports[name])} | "
        f"sampled_rows="
        f"{len(rows_by_target[name])}"
    )


print()
print(
    "JSON files actually loaded:",
    json_files_actually_loaded,
)


# ============================================================
# BUILD FIELD MAP FOR RARE TARGET SECTIONS
# ============================================================

field_map_rows = []


for section, rows in (
    rows_by_target.items()
):

    field_values = defaultdict(
        list
    )


    # Union of every field encountered.
    for row in rows:

        for field, value in (
            row.items()
        ):

            field_values[
                field
            ].append(
                value
            )


    for field, values in (
        field_values.items()
    ):

        observed_types = sorted(
            set(
                typename(v)
                for v in values
            )
        )


        non_null = [
            v
            for v in values
            if v is not None
        ]


        examples = []

        for value in non_null:

            if isinstance(
                value,
                (dict, list),
            ):
                continue

            text = str(
                value
            )

            if len(text) > 160:

                text = (
                    text[:157]
                    + "..."
                )

            if (
                text
                not in examples
            ):

                examples.append(
                    text
                )

            if len(examples) >= 5:
                break


        field_map_rows.append(
            {
                "section":
                    section,

                "field":
                    field,

                "observed_types":
                    " | ".join(
                        observed_types
                    ),

                "sampled_rows":
                    len(values),

                "non_null_rows":
                    len(non_null),

                "examples":
                    " || ".join(
                        examples
                    ),
            }
        )


target_field_map = pd.DataFrame(
    field_map_rows
)


if len(target_field_map):

    target_field_map = (
        target_field_map
        .sort_values(
            [
                "section",
                "field",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# SAVE TARGETED FIELD MAP
# ============================================================

target_parquet = (
    SCHEMA_DIR
    / "targeted_section_field_map.parquet"
)

target_csv = (
    SCHEMA_DIR
    / "targeted_section_field_map.csv"
)


target_field_map.to_parquet(
    target_parquet,
    index=False,
)

target_field_map.to_csv(
    target_csv,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# DISPLAY RARE SECTION SCHEMAS
# ============================================================

for section in TARGETS:

    print()
    print("=" * 90)
    print(
        section.upper()
    )
    print("=" * 90)

    if not len(
        target_field_map
    ):

        print(
            "No rows found."
        )

        continue


    part = target_field_map[
        target_field_map[
            "section"
        ] == section
    ]


    if len(part):

        display(
            part
        )

    else:

        print(
            "No non-empty example "
            "found within bounded search."
        )


# ============================================================
# PAYMENT SCHEMA COMPARISON
#
# Use:
# - first schema inventory for common payment types
# - targeted field map for rare payment types
# ============================================================

PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


payment_fields = {
    section: set()
    for section in PAYMENT_SECTIONS
}


# ------------------------------------------------------------
# Existing inventory
# ------------------------------------------------------------

for section in PAYMENT_SECTIONS:

    possible_prefixes = [
        (
            "payment_info."
            "incoming."
            f"{section}[]."
        ),
        (
            "payment_info."
            "outgoing."
            f"{section}[]."
        ),
    ]

    for prefix in (
        possible_prefixes
    ):

        paths = inventory.loc[
            inventory[
                "json_path"
            ].str.startswith(
                prefix,
                na=False,
            ),
            "json_path",
        ]

        for path in paths:

            payment_fields[
                section
            ].add(
                path.rsplit(
                    ".",
                    1,
                )[-1]
            )


# ------------------------------------------------------------
# Targeted rare section schema
# ------------------------------------------------------------

if len(target_field_map):

    for section in [
        "state_funding",
        "budget_expenses",
        "return_expenses",
        "transfer_expenses",
    ]:

        fields = (
            target_field_map.loc[
                target_field_map[
                    "section"
                ] == section,
                "field",
            ]
            .dropna()
            .astype(str)
        )

        payment_fields[
            section
        ].update(
            fields
        )


# ============================================================
# COMPARISON MATRIX
# ============================================================

all_payment_fields = sorted(
    set().union(
        *payment_fields.values()
    )
)


comparison_rows = []


for field in (
    all_payment_fields
):

    row = {
        "field": field
    }

    for section in (
        PAYMENT_SECTIONS
    ):

        row[
            section
        ] = (
            field
            in payment_fields[
                section
            ]
        )

    comparison_rows.append(
        row
    )


payment_schema_comparison = (
    pd.DataFrame(
        comparison_rows
    )
)


comparison_path = (
    SCHEMA_DIR
    / "payment_schema_comparison.csv"
)

payment_schema_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig",
)


print()
print("=" * 90)
print("PAYMENT SCHEMA COMPARISON")
print("=" * 90)

display(
    payment_schema_comparison
)


# ============================================================
# FINAL SEARCH SUMMARY
# ============================================================

summary_rows = []


for section in TARGETS:

    summary_rows.append(
        {
            "section":
                section,

            "candidate_files":
                candidate_counts.get(
                    section,
                    0,
                ),

            "verified_nonempty_reports":
                len(
                    found_reports[
                        section
                    ]
                ),

            "sampled_rows":
                len(
                    rows_by_target[
                        section
                    ]
                ),

            "report_ids":
                " | ".join(
                    found_reports[
                        section
                    ]
                ),
        }
    )


search_summary = pd.DataFrame(
    summary_rows
)


search_summary.to_csv(
    SCHEMA_DIR
    / "targeted_schema_search_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


print()
print("=" * 90)
print("TARGETED SEARCH SUMMARY")
print("=" * 90)

display(
    search_summary
)


print()
print("=" * 90)
print("DONE")
print("=" * 90)

print(
    "Files created:"
)

print(
    "-",
    target_parquet,
)

print(
    "-",
    target_csv,
)

print(
    "-",
    comparison_path,
)

print(
    "-",
    (
        SCHEMA_DIR
        / "targeted_schema_search_summary.csv"
    ),
)


OBLIGATIONS — EXISTING SCHEMA


,json_path,observed_types,non_null_observations,sample_report_count,examples
11,obligations[],dict,112,10,
12,obligations[].created_at,str,112,10,2024-03-22 16:15:21 || 2026-02-23 22:04:28 || ...
13,obligations[].end_period_remains_cost,str,112,10,243.25 || 915.72 || 2696.51 || 264.98 || 14652.52
14,obligations[].id,str,112,10,91294bbe-0628-48d8-8451-6bfb7512c0e1 || 0af50e...
15,obligations[].object_type,str,112,10,Комунальні послуги || Орендна плата та комунал...
16,obligations[].office_id,null | str,82,10,9b1c3511-1fcc-4091-bdcd-97930d503b65 || 2c612b...
17,obligations[].owning_cost,str,112,10,243.25 || 915.72 || 2696.51 || 264.98 || 14652.52
18,obligations[].owning_date,str,112,10,2019-09-30 || 2022-05-31 || 2022-06-30 || 2022...
19,obligations[].owning_reason,str,112,10,Акт наданих послуг б/н від 30.09.2019р || Акт ...
20,obligations[].party_id,str,112,10,7d9a4591-6cea-451f-8545-b211f5e7b9bd || 106ed6...



ORGANIZATIONS — EXISTING SCHEMA


,json_path,observed_types,non_null_observations,sample_report_count,examples
27,organizations[],dict,2,2,
28,organizations[].code,str,2,2,
29,organizations[].name,str,2,2,



REGIONAL OFFICES — EXISTING SCHEMA


...


TRANSFER_EXPENSES
No non-empty example found within bounded search.

PROPERTY_PAPER
No non-empty example found within bounded search.

PAYMENT SCHEMA COMPARISON


,field,monetary_contributions,other_contributions,state_funding,other_incomes,budget_expenses,outgoing_expenses,return_expenses,transfer_expenses
0,created_at,True,True,False,True,False,True,True,False
1,group_code,True,True,False,True,False,True,True,False
2,id,True,True,False,True,False,True,True,False
3,office_id,True,True,False,True,False,True,True,False
4,party_id,True,True,False,True,False,True,True,False
5,payer_account_iban,True,True,False,True,False,True,True,False
6,payer_account_type,True,True,False,True,False,True,True,False
7,payer_address,True,True,False,True,False,True,True,False
8,payer_bank_address,True,True,False,True,False,True,True,False
9,payer_bank_code,True,True,False,True,False,True,True,False



TARGETED SEARCH SUMMARY


,section,candidate_files,verified_nonempty_reports,sampled_rows,report_ids
0,state_funding,0,0,0,
1,budget_expenses,0,0,0,
2,return_expenses,4,2,4,b79590a0-9541-11ef-9197-0fbca96b9f8a | 9d6008b...
3,transfer_expenses,0,0,0,
4,property_paper,0,0,0,



DONE
Files created:
- C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\schema\targeted_section_field_map.parquet
- C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\schema\targeted_section_field_map.csv
- C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\schema\payment_schema_comparison.csv
- C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\schema\targeted_schema_search_summary.csv


In [100]:
from pathlib import Path
import pandas as pd


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)


CANDIDATES = [
    ROOT / "data/processed/normalized_v0_1/organizations.parquet",
    ROOT / "data/processed/normalized_v0_1/organization_addresses.parquet",
    ROOT / "data/processed/normalized_v0_1/organization_heads.parquet",

    ROOT / "data/interim/reports/selected_reports_manifest.parquet",
    ROOT / "data/interim/reports/report_period_selection_final.parquet",

    ROOT / "data/interim/reports/analysis_selected_reports_manifest.parquet",
    ROOT / "data/interim/reports/analysis_report_overrides.parquet",
]


print("=" * 100)
print("PIPELINE FOUNDATION SCHEMAS")
print("=" * 100)


for path in CANDIDATES:

    print()
    print("-" * 100)
    print(path.relative_to(ROOT))
    print("-" * 100)

    if not path.exists():

        print("NOT FOUND")
        continue


    df = pd.read_parquet(
        path
    )

    print(
        "Rows:",
        len(df)
    )

    print(
        "Columns:"
    )

    for col in df.columns:
        print(
            "  -",
            col,
            "|",
            df[col].dtype,
        )


    print()
    print("Sample:")

    display(
        df.head(3)
    )

PIPELINE FOUNDATION SCHEMAS

----------------------------------------------------------------------------------------------------
data\processed\normalized_v0_1\organizations.parquet
----------------------------------------------------------------------------------------------------
Rows: 9917
Columns:
  - organization_id | string
  - root_party_id | string
  - parent_id | string
  - entity_type | string
  - code | string
  - name | string
  - is_active | boolean
  - created_at | datetime64[ns]
  - updated_at | datetime64[ns]
  - web_site_url | string
  - email | string
  - phone | string
  - actual_address_same_register | boolean

Sample:


,organization_id,root_party_id,parent_id,entity_type,code,name,is_active,created_at,updated_at,web_site_url,email,phone,actual_address_same_register
0,00011adb-dc52-4942-bf91-30452f3702c4,f07e15f7-097b-481d-a00f-b021b8918585,f07e15f7-097b-481d-a00f-b021b8918585,office,36454702,ПОЛТАВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,True,2025-07-01 08:47:39.805163,2025-07-01 08:47:39.806879,<NA>,<NA>,<NA>,False
1,00037df2-83a8-411b-8393-f37d8e8b6139,1388ae5f-2819-4153-84de-5463a956d3b4,1388ae5f-2819-4153-84de-5463a956d3b4,office,26069495,ПОЛІТИЧНА ПАРТІЯ ПОЛТАВСЬКА ОБЛАСНА ПАРТІЙНА О...,True,2025-07-01 08:48:51.888559,2025-07-01 08:48:51.888559,<NA>,<NA>,<NA>,True
2,00037e06-6ba4-4984-b8d1-08927256d278,155c3deb-caa3-488b-8ecf-5e54d674a3f6,155c3deb-caa3-488b-8ecf-5e54d674a3f6,office,26601553,ОДЕСЬКА МІСЬКА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОГО ОБ'...,True,2025-07-01 08:49:53.197012,2025-10-29 13:51:09.681753,<NA>,<NA>,<NA>,True



----------------------------------------------------------------------------------------------------
data\processed\normalized_v0_1\organization_addresses.parquet
----------------------------------------------------------------------------------------------------
Rows: 11629
Columns:
  - organization_id | string
  - address_type | string
  - country | string
  - post_index | string
  - region | string
  - district | string
  - city | string
  - street | string
  - building | string
  - apartments | string
  - common | string
  - building_part_num | string
  - address_uk | string
  - address_en | string
  - post_index_not_provided | boolean
  - district_not_provided | boolean
  - common_not_provided | boolean
  - city_not_provided | boolean
  - region_not_provided | boolean
  - street_not_provided | boolean
  - building_not_provided | boolean
  - building_part_num_not_provided | boolean
  - apartments_not_provided | boolean
  - address_en_not_provided | boolean
  - address_uk_not_provi

,organization_id,address_type,country,post_index,region,district,city,street,building,apartments,common,building_part_num,address_uk,address_en,post_index_not_provided,district_not_provided,common_not_provided,city_not_provided,region_not_provided,street_not_provided,building_not_provided,building_part_num_not_provided,apartments_not_provided,address_en_not_provided,address_uk_not_provided
0,00011adb-dc52-4942-bf91-30452f3702c4,register,Україна,39600,Полтавська обл.,Кременчук,Автозаводський,Переяславська,55А,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,00011adb-dc52-4942-bf91-30452f3702c4,actual,Україна,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,00037df2-83a8-411b-8393-f37d8e8b6139,register,Україна,36000,Полтавська обл.,Полтава,Київський,Степового фронту,29,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>



----------------------------------------------------------------------------------------------------
data\processed\normalized_v0_1\organization_heads.parquet
----------------------------------------------------------------------------------------------------
Rows: 6471
Columns:
  - organization_id | string
  - head_last_name | string
  - head_first_name | string
  - head_middle_name | string

Sample:


...

,organization_id,year,quarter,report_instance_count,signed_instance_count,unsigned_instance_count,selected_report_id,selected_signed_date,selected_created_date,selection_method
0,00011adb-dc52-4942-bf91-30452f3702c4,2021,1,1,1,0,9f803ed0-f45d-11eb-b63a-0d281be7db0f,2021-08-04 09:26:00,2024-04-23 15:25:48,unique_signed
1,00011adb-dc52-4942-bf91-30452f3702c4,2021,2,1,1,0,1fa2c150-414d-11ec-b739-ddff8fe8437d,2021-11-22 09:09:44,2024-04-23 19:45:42,unique_signed
2,00011adb-dc52-4942-bf91-30452f3702c4,2021,3,1,1,0,dc3dbf70-4c58-11ec-b5d3-55c34d99ddc4,2021-11-23 14:54:05,2024-04-23 20:04:44,unique_signed



----------------------------------------------------------------------------------------------------
data\interim\reports\analysis_selected_reports_manifest.parquet
----------------------------------------------------------------------------------------------------
Rows: 78791
Columns:
  - report_id | object
  - organization_id | object
  - root_party_id | object
  - entity_type | object
  - source_party_id | object
  - party_id_matches_organization | bool
  - is_party_office | bool
  - schema_version | int64
  - report_type | object
  - year | int64
  - quarter | int64
  - period_type | object
  - signed_date | object
  - created_date | object
  - signatory_id | object
  - special_status | object
  - public_summary_version | float64
  - public_summary_generated_at | object
  - discovered_at_utc | object
  - signed_date_dt | datetime64[ns]
  - created_date_dt | datetime64[ns]
  - is_signed | bool
  - report_instance_count | int64
  - signed_instance_count | int64
  - unsigned_instance

,report_id,organization_id,root_party_id,entity_type,source_party_id,party_id_matches_organization,is_party_office,schema_version,report_type,year,quarter,period_type,signed_date,created_date,signatory_id,special_status,public_summary_version,public_summary_generated_at,discovered_at_utc,signed_date_dt,created_date_dt,is_signed,report_instance_count,signed_instance_count,unsigned_instance_count,selected_report_id,selection_method,is_selected_report,has_annual_report,has_quarterly_reports,annual_report_count,quarterly_report_count,selected_report_count,has_mixed_periodicity,include_in_annualized_analysis,official_selected_report_id,analysis_selected_report_id,analysis_selection_method,analysis_override,override_reason,continuity_exact
0,dddc6d50-d1d3-11ef-bde1-41cc07d44e1f,00011adb-dc52-4942-bf91-30452f3702c4,f07e15f7-097b-481d-a00f-b021b8918585,office,00011adb-dc52-4942-bf91-30452f3702c4,True,True,1,main,2024,4,quarterly,2025-01-14 08:52:02,2025-02-05 19:56:28,None,None,NaN,None,2026-08-29T09:28:31.042499+00:00,2025-01-14 08:52:02.000000,2025-02-05 19:56:28.000000,True,1,1,0,dddc6d50-d1d3-11ef-bde1-41cc07d44e1f,unique_signed,True,False,True,0,4,4,False,True,dddc6d50-d1d3-11ef-bde1-41cc07d44e1f,dddc6d50-d1d3-11ef-bde1-41cc07d44e1f,official_selected_signed_report,False,None,None
1,e2247690-4070-11ef-ac2d-b9ec46358bc3,00011adb-dc52-4942-bf91-30452f3702c4,f07e15f7-097b-481d-a00f-b021b8918585,office,00011adb-dc52-4942-bf91-30452f3702c4,True,True,1,main,2024,2,quarterly,2024-07-17 18:39:10,2024-07-31 19:27:14,None,None,NaN,None,2026-08-29T09:28:31.042499+00:00,2024-07-17 18:39:10.000000,2024-07-31 19:27:14.000000,True,1,1,0,e2247690-4070-11ef-ac2d-b9ec46358bc3,unique_signed,True,False,True,0,4,4,False,True,e2247690-4070-11ef-ac2d-b9ec46358bc3,e2247690-4070-11ef-ac2d-b9ec46358bc3,official_selected_signed_report,False,None,None
2,1ca530f7-4d06-4292-8a42-fbf4d81f9940,00011adb-dc52-4942-bf91-30452f3702c4,f07e15f7-097b-481d-a00f-b021b8918585,office,00011adb-dc52-4942-bf91-30452f3702c4,True,True,1,main,2026,2,quarterly,2026-07-21 15:25:19.052814,2026-07-21 14:44:59.079325,33d74d1a-4df3-4044-8603-c05ef668540d,None,2.0,2026-07-21T12:25:19+00:00,2026-08-29T09:28:31.042499+00:00,2026-07-21 15:25:19.052814,2026-07-21 14:44:59.079325,True,1,1,0,1ca530f7-4d06-4292-8a42-fbf4d81f9940,unique_signed,True,False,True,0,2,2,False,True,1ca530f7-4d06-4292-8a42-fbf4d81f9940,1ca530f7-4d06-4292-8a42-fbf4d81f9940,official_selected_signed_report,False,None,None



----------------------------------------------------------------------------------------------------
data\interim\reports\analysis_report_overrides.parquet
----------------------------------------------------------------------------------------------------
Rows: 1
Columns:
  - organization_id | object
  - year | int64
  - quarter | int64
  - official_selected_report_id | object
  - analysis_selected_report_id | object
  - analysis_selection_method | object
  - override_reason | object
  - continuity_exact | bool
  - analysis_override | bool

Sample:


,organization_id,year,quarter,official_selected_report_id,analysis_selected_report_id,analysis_selection_method,override_reason,continuity_exact,analysis_override
0,5f508361-50a4-4464-880d-01f1843cbe76,2025,2,764a691c-3237-4044-9430-44313fbf739f,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,manual_financial_continuity_override,Unsigned report contains 250 Q2 transactions; ...,True,True


In [101]:
from pathlib import Path
import textwrap
import subprocess
import sys

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

SRC_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "normalization"
)

TEST_DIR = (
    ROOT
    / "tests"
)

OUT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

SRC_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TEST_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# WRITE PERMANENT MODULE
# ============================================================

MODULE_PATH = (
    SRC_DIR
    / "reference.py"
)


MODULE_CODE = r'''
from __future__ import annotations

import re
import unicodedata

import pandas as pd


# ============================================================
# TEXT
# ============================================================

def clean_text(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text or None


# ============================================================
# REGION NORMALIZATION
# ============================================================

REGION_RULES = [
    ("вінниц", "Вінницька область"),
    ("волин", "Волинська область"),
    ("дніпропетров", "Дніпропетровська область"),
    ("донець", "Донецька область"),
    ("житомир", "Житомирська область"),
    ("закарпат", "Закарпатська область"),
    ("запоріз", "Запорізька область"),
    ("івано-франків", "Івано-Франківська область"),
    ("київськ", "Київська область"),
    ("кіровоград", "Кіровоградська область"),
    ("луган", "Луганська область"),
    ("львів", "Львівська область"),
    ("миколаїв", "Миколаївська область"),
    ("одес", "Одеська область"),
    ("полтав", "Полтавська область"),
    ("рівнен", "Рівненська область"),
    ("сум", "Сумська область"),
    ("терноп", "Тернопільська область"),
    ("харків", "Харківська область"),
    ("херсон", "Херсонська область"),
    ("хмельниц", "Хмельницька область"),
    ("черкас", "Черкаська область"),
    ("чернів", "Чернівецька область"),
    ("черніг", "Чернігівська область"),
]


def normalize_region(value):

    text = clean_text(value)

    if text is None:
        return None

    lowered = text.casefold()


    # --------------------------------------------------------
    # Kyiv city before Kyiv oblast
    # --------------------------------------------------------

    if (
        "м. київ" in lowered
        or "м.київ" in lowered
        or lowered == "київ"
        or "місто київ" in lowered
    ):
        return "м. Київ"


    if "севастопол" in lowered:
        return "м. Севастополь"


    if "крим" in lowered:
        return "Автономна Республіка Крим"


    for token, canonical in REGION_RULES:

        if token in lowered:
            return canonical


    return None


# ============================================================
# ORGANIZATION REFERENCE
# ============================================================

def build_organization_reference(
    organizations: pd.DataFrame,
    addresses: pd.DataFrame,
) -> pd.DataFrame:

    org = organizations.copy()
    addr = addresses.copy()


    required = {
        "organization_id",
        "root_party_id",
        "entity_type",
        "code",
        "name",
    }

    missing = required - set(
        org.columns
    )

    if missing:
        raise ValueError(
            "organizations missing: "
            + ", ".join(
                sorted(missing)
            )
        )


    # --------------------------------------------------------
    # Stable string identifiers
    # --------------------------------------------------------

    for col in [
        "organization_id",
        "root_party_id",
        "code",
        "name",
        "entity_type",
    ]:

        org[col] = (
            org[col]
            .astype("string")
        )


    # --------------------------------------------------------
    # Central-party lookup
    # --------------------------------------------------------

    parties = (
        org[
            org["entity_type"]
            == "party"
        ][
            [
                "organization_id",
                "code",
                "name",
            ]
        ]
        .rename(
            columns={
                "organization_id":
                    "root_party_id",

                "code":
                    "party_code",

                "name":
                    "party_name_current",
            }
        )
    )


    if (
        parties[
            "root_party_id"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate root party IDs."
        )


    # --------------------------------------------------------
    # Address selection
    #
    # Use region from:
    #   register first,
    #   actual second,
    #   other address types last.
    #
    # Populated region beats empty region.
    # --------------------------------------------------------

    addr["organization_id"] = (
        addr["organization_id"]
        .astype("string")
    )

    addr["region_source"] = (
        addr["region"]
        .astype("string")
    )


    region_present = (
        addr["region_source"]
        .notna()
        &
        (
            addr[
                "region_source"
            ]
            .str.strip()
            != ""
        )
    )


    addr[
        "_missing_region"
    ] = (
        ~region_present
    ).astype(int)


    address_priority = {
        "register": 0,
        "actual": 1,
    }


    addr[
        "_address_priority"
    ] = (
        addr[
            "address_type"
        ]
        .map(
            address_priority
        )
        .fillna(2)
    )


    selected_addr = (
        addr
        .sort_values(
            [
                "organization_id",
                "_missing_region",
                "_address_priority",
            ]
        )
        .drop_duplicates(
            "organization_id",
            keep="first",
        )
        [
            [
                "organization_id",
                "address_type",
                "region_source",
            ]
        ]
        .rename(
            columns={
                "address_type":
                    "region_source_address_type",
            }
        )
    )


    # --------------------------------------------------------
    # Main reference
    # --------------------------------------------------------

    out = (
        org
        .rename(
            columns={
                "code":
                    "organization_code",

                "name":
                    "organization_name_current",
            }
        )
        .merge(
            parties,
            on="root_party_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            selected_addr,
            on="organization_id",
            how="left",
            validate="one_to_one",
        )
    )


    out[
        "organization_level"
    ] = (
        out["entity_type"]
        .map(
            {
                "party": "central",
                "office": "office",
            }
        )
        .astype("string")
    )


    out["region"] = (
        out[
            "region_source"
        ]
        .map(
            normalize_region
        )
        .astype("string")
    )


    # Central party always analytical region = Ukraine.
    out.loc[
        out[
            "organization_level"
        ]
        == "central",
        "region",
    ] = "Україна"


    # --------------------------------------------------------
    # QA
    # --------------------------------------------------------

    if (
        out[
            "organization_id"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "organization_reference "
            "not unique by organization_id."
        )


    if (
        out[
            "party_name_current"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Some organizations have "
            "no current party name."
        )


    if (
        out[
            "party_code"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Some organizations have "
            "no root-party code."
        )


    columns = [
        "organization_id",
        "root_party_id",

        "organization_level",

        "organization_code",
        "organization_name_current",

        "party_code",
        "party_name_current",

        "region",
        "region_source",
        "region_source_address_type",
    ]


    for optional in [
        "is_active",
        "parent_id",
    ]:

        if optional in out.columns:
            columns.append(
                optional
            )


    return (
        out[
            columns
        ]
        .reset_index(
            drop=True
        )
    )


# ============================================================
# REPORT CONTEXT
# ============================================================

def build_report_context(
    analysis_manifest: pd.DataFrame,
    organization_reference: pd.DataFrame,
) -> pd.DataFrame:

    reports = (
        analysis_manifest
        .copy()
    )


    required = {
        "report_id",
        "organization_id",
        "root_party_id",
        "year",
        "quarter",

        "official_selected_report_id",
        "analysis_selected_report_id",
        "analysis_selection_method",
        "analysis_override",
    }


    missing = required - set(
        reports.columns
    )

    if missing:
        raise ValueError(
            "analysis manifest missing: "
            + ", ".join(
                sorted(missing)
            )
        )


    for col in [
        "organization_id",
        "root_party_id",
        "official_selected_report_id",
        "analysis_selected_report_id",
    ]:

        reports[col] = (
            reports[col]
            .astype("string")
        )


    # --------------------------------------------------------
    # This is the RAW file analytical normalization must read.
    # --------------------------------------------------------

    reports[
        "source_report_id"
    ] = (
        reports[
            "analysis_selected_report_id"
        ]
    )


    reports["year"] = (
        pd.to_numeric(
            reports["year"],
            errors="raise",
        )
        .astype(int)
    )


    reports["quarter"] = (
        pd.to_numeric(
            reports["quarter"],
            errors="raise",
        )
        .astype(int)
    )


    # quarter 5 = annual
    reports[
        "report_period_order"
    ] = (
        reports["year"]
        * 10
        +
        reports["quarter"]
    )


    reports[
        "period_label"
    ] = reports.apply(
        lambda r:
            (
                f"{r['year']} annual"
                if r["quarter"] == 5
                else
                f"{r['year']} Q{r['quarter']}"
            ),
        axis=1,
    ).astype("string")


    # --------------------------------------------------------
    # Latest available period PER ORGANIZATION
    # --------------------------------------------------------

    latest_order = (
        reports
        .groupby(
            "organization_id"
        )[
            "report_period_order"
        ]
        .transform(
            "max"
        )
    )


    reports[
        "is_latest_report_for_organization"
    ] = (
        reports[
            "report_period_order"
        ]
        == latest_order
    )


    reports[
        "data_recency_status"
    ] = (
        reports[
            "is_latest_report_for_organization"
        ]
        .map(
            {
                True:
                    "latest_data",

                False:
                    "historical_data",
            }
        )
        .astype("string")
    )


    # --------------------------------------------------------
    # Current reference identity
    #
    # Current name intentionally propagates backwards
    # across historical analytical data.
    # --------------------------------------------------------

    ref_cols = [
        "organization_id",

        "organization_level",

        "organization_code",
        "organization_name_current",

        "party_code",
        "party_name_current",

        "region",
        "region_source",
        "region_source_address_type",
    ]


    out = reports.merge(
        organization_reference[
            ref_cols
        ],
        on="organization_id",
        how="left",
        validate="many_to_one",
    )


    # --------------------------------------------------------
    # QA
    # --------------------------------------------------------

    if (
        out[
            "source_report_id"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Missing analytical source_report_id."
        )


    if (
        out[
            "source_report_id"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate analytical source_report_id."
        )


    latest_counts = (
        out[
            out[
                "is_latest_report_for_organization"
            ]
        ]
        .groupby(
            "organization_id"
        )
        .size()
    )


    if (
        latest_counts > 1
    ).any():
        raise ValueError(
            "More than one latest period "
            "for an organization."
        )


    return (
        out
        .sort_values(
            [
                "organization_id",
                "report_period_order",
            ]
        )
        .reset_index(
            drop=True
        )
    )
'''


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)

print(
    "Written:",
    MODULE_PATH,
)


# ============================================================
# WRITE TESTS
# ============================================================

TEST_PATH = (
    TEST_DIR
    / "test_reference.py"
)


TEST_CODE = r'''
import pandas as pd

from politdata.normalization.reference import (
    normalize_region,
    build_organization_reference,
    build_report_context,
)


def test_region_pol_tava():
    assert (
        normalize_region(
            "Полтавська обл."
        )
        == "Полтавська область"
    )


def test_region_kyiv_city():
    assert (
        normalize_region(
            "м. Київ"
        )
        == "м. Київ"
    )


def test_region_kyiv_oblast():
    assert (
        normalize_region(
            "Київська область"
        )
        == "Київська область"
    )


def test_reference_and_latest():

    organizations = pd.DataFrame(
        [
            {
                "organization_id": "p1",
                "root_party_id": "p1",
                "parent_id": None,
                "entity_type": "party",
                "code": "11111111",
                "name": "Current Party",
                "is_active": True,
            },
            {
                "organization_id": "o1",
                "root_party_id": "p1",
                "parent_id": "p1",
                "entity_type": "office",
                "code": "22222222",
                "name": "Current Office",
                "is_active": True,
            },
        ]
    )


    addresses = pd.DataFrame(
        [
            {
                "organization_id": "p1",
                "address_type": "register",
                "region": "м. Київ",
            },
            {
                "organization_id": "o1",
                "address_type": "register",
                "region": "Полтавська обл.",
            },
        ]
    )


    ref = build_organization_reference(
        organizations,
        addresses,
    )


    indexed = ref.set_index(
        "organization_id"
    )


    assert (
        indexed.loc[
            "p1",
            "region",
        ]
        == "Україна"
    )


    assert (
        indexed.loc[
            "o1",
            "region",
        ]
        == "Полтавська область"
    )


    assert (
        indexed.loc[
            "o1",
            "party_name_current",
        ]
        == "Current Party"
    )


    manifest = pd.DataFrame(
        [
            {
                "report_id": "r1",
                "organization_id": "o1",
                "root_party_id": "p1",
                "year": 2025,
                "quarter": 4,
                "official_selected_report_id": "r1",
                "analysis_selected_report_id": "r1",
                "analysis_selection_method": "official",
                "analysis_override": False,
            },
            {
                "report_id": "r2",
                "organization_id": "o1",
                "root_party_id": "p1",
                "year": 2026,
                "quarter": 1,
                "official_selected_report_id": "r2",
                "analysis_selected_report_id": "r2",
                "analysis_selection_method": "official",
                "analysis_override": False,
            },
        ]
    )


    context = build_report_context(
        manifest,
        ref,
    )


    latest = context[
        context[
            "is_latest_report_for_organization"
        ]
    ]


    assert len(latest) == 1

    assert (
        latest.iloc[0][
            "source_report_id"
        ]
        == "r2"
    )

    assert (
        latest.iloc[0][
            "data_recency_status"
        ]
        == "latest_data"
    )
'''


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)

print(
    "Written:",
    TEST_PATH,
)


# ============================================================
# RUN TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 90)
print("TESTS")
print("=" * 90)

print(
    result.stdout
)

if result.stderr:
    print(
        result.stderr
    )

print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Production tables were not built."
    )


# ============================================================
# IMPORT
# ============================================================

from politdata.normalization.reference import (
    build_organization_reference,
    build_report_context,
)


# ============================================================
# LOAD REAL FOUNDATION DATA
# ============================================================

organizations = pd.read_parquet(
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "organizations.parquet"
)


addresses = pd.read_parquet(
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "organization_addresses.parquet"
)


analysis_manifest = pd.read_parquet(
    ROOT
    / "data"
    / "interim"
    / "reports"
    / "analysis_selected_reports_manifest.parquet"
)


# ============================================================
# ORGANIZATION REFERENCE
# ============================================================

organization_reference = (
    build_organization_reference(
        organizations,
        addresses,
    )
)


ORG_REFERENCE_PATH = (
    OUT_DIR
    / "organization_reference.parquet"
)


organization_reference.to_parquet(
    ORG_REFERENCE_PATH,
    index=False,
)


# ============================================================
# REPORT CONTEXT
# ============================================================

report_context = (
    build_report_context(
        analysis_manifest,
        organization_reference,
    )
)


REPORT_CONTEXT_PATH = (
    OUT_DIR
    / "report_context.parquet"
)


report_context.to_parquet(
    REPORT_CONTEXT_PATH,
    index=False,
)


# ============================================================
# QA
# ============================================================

print()
print("=" * 90)
print("ORGANIZATION REFERENCE QA")
print("=" * 90)


print(
    "Rows:",
    len(
        organization_reference
    )
)


print(
    "Unique organization IDs:",
    organization_reference[
        "organization_id"
    ].nunique()
)


print()
print(
    "Organization levels:"
)

print(
    organization_reference[
        "organization_level"
    ]
    .value_counts(
        dropna=False
    )
)


print()
print(
    "Canonical regions:"
)

print(
    organization_reference[
        "region"
    ]
    .value_counts(
        dropna=False
    )
    .sort_index()
)


office_mask = (
    organization_reference[
        "organization_level"
    ]
    == "office"
)


missing_office_region = (
    organization_reference.loc[
        office_mask,
        "region",
    ]
    .isna()
    .sum()
)


print()
print(
    "Office rows with missing canonical region:",
    missing_office_region
)


# ------------------------------------------------------------
# Show region values not yet mapped
# ------------------------------------------------------------

unmapped = (
    organization_reference.loc[
        office_mask
        &
        organization_reference[
            "region"
        ].isna()
        &
        organization_reference[
            "region_source"
        ].notna(),
        [
            "region_source",
        ]
    ]
    .value_counts()
    .reset_index(
        name="count"
    )
)


print()
print(
    "Unmapped non-empty region values:"
)

display(
    unmapped.head(
        100
    )
)


# ============================================================
# REPORT CONTEXT QA
# ============================================================

print()
print("=" * 90)
print("REPORT CONTEXT QA")
print("=" * 90)


print(
    "Rows:",
    len(
        report_context
    )
)


print(
    "Unique analytical source reports:",
    report_context[
        "source_report_id"
    ].nunique()
)


print(
    "Organizations represented:",
    report_context[
        "organization_id"
    ].nunique()
)


print(
    "Latest reports:",
    report_context[
        "is_latest_report_for_organization"
    ].sum()
)


print(
    "Analysis overrides:",
    report_context[
        "analysis_override"
    ].sum()
)


print()
print(
    "Recency status:"
)

print(
    report_context[
        "data_recency_status"
    ]
    .value_counts(
        dropna=False
    )
)


# ============================================================
# CHECK OVERRIDE
# ============================================================

print()
print("=" * 90)
print("ANALYTICAL OVERRIDES")
print("=" * 90)


display(
    report_context[
        report_context[
            "analysis_override"
        ]
    ][
        [
            "organization_id",
            "organization_name_current",

            "party_name_current",

            "year",
            "quarter",

            "official_selected_report_id",
            "analysis_selected_report_id",

            "analysis_selection_method",
            "override_reason",
        ]
    ]
)


# ============================================================
# FILES
# ============================================================

print()
print("=" * 90)
print("FILES CREATED")
print("=" * 90)

print(
    ORG_REFERENCE_PATH
)

print(
    REPORT_CONTEXT_PATH
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\reference.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_reference.py

TESTS
...................................                                      [100%]
35 passed in 7.08s

Return code: 0

ORGANIZATION REFERENCE QA
Rows: 9917
Unique organization IDs: 9917

Organization levels:
organization_level
office     9596
central     321
Name: count, dtype: Int64

Canonical regions:
region
Івано-Франківська область    336
Автономна Республіка Крим    228
Волинська область            265
Вінницька область            377
Дніпропетровська область     442
Донецька область             438
Житомирська область          349
Закарпатська область         266
Запорізька область           241
Київська область             680
Кіровоградська область       295
Луганська область            255
Львівська область            530
Миколаївська область         356
Одеська область              388
Полтав

,region_source,count
0,,84



REPORT CONTEXT QA
Rows: 78791
Unique analytical source reports: 78791
Organizations represented: 8514
Latest reports: 8514
Analysis overrides: 1

Recency status:
data_recency_status
historical_data    70277
latest_data         8514
Name: count, dtype: Int64

ANALYTICAL OVERRIDES


,organization_id,organization_name_current,party_name_current,year,quarter,official_selected_report_id,analysis_selected_report_id,analysis_selection_method,override_reason
28679,5f508361-50a4-4464-880d-01f1843cbe76,ЛУГАНСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІ...,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,2025,2,764a691c-3237-4044-9430-44313fbf739f,317a1c90-a5a3-43e5-8e62-e205cc84fb1e,manual_financial_continuity_override,Unsigned report contains 250 Q2 transactions; ...



FILES CREATED
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\organization_reference.parquet
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\report_context.parquet


In [102]:
from pathlib import Path
import pandas as pd

from politdata.normalization.reference import (
    normalize_region,
)


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

REF_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "organization_reference.parquet"
)

ADDRESS_PATH = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "organization_addresses.parquet"
)


# ============================================================
# LOAD
# ============================================================

ref = pd.read_parquet(
    REF_PATH
)

addresses = pd.read_parquet(
    ADDRESS_PATH
)


# ============================================================
# OFFICES CURRENTLY MISSING REGION
# ============================================================

missing_ref = (
    ref[
        (ref["organization_level"] == "office")
        &
        ref["region"].isna()
    ][
        [
            "organization_id",
            "organization_code",
            "organization_name_current",
            "party_name_current",
        ]
    ]
    .copy()
)


print("=" * 90)
print("MISSING REGION DIAGNOSTIC")
print("=" * 90)

print(
    "Offices missing canonical region:",
    len(missing_ref),
)


# ============================================================
# ALL ADDRESS ROWS FOR THESE OFFICES
# ============================================================

missing_addresses = (
    addresses[
        addresses[
            "organization_id"
        ].isin(
            missing_ref[
                "organization_id"
            ]
        )
    ]
    .merge(
        missing_ref,
        on="organization_id",
        how="left",
        validate="many_to_one",
    )
    .copy()
)


print(
    "Address rows available for them:",
    len(missing_addresses),
)

print(
    "Organizations with at least one address row:",
    missing_addresses[
        "organization_id"
    ].nunique(),
)


# ============================================================
# FIELD COMPLETENESS
# ============================================================

candidate_fields = [
    col
    for col in [
        "region",
        "address_uk",
        "common",
        "district",
        "city",
        "street",
        "address_en",
    ]
    if col in missing_addresses.columns
]


completeness = []

for col in candidate_fields:

    s = (
        missing_addresses[col]
        .astype("string")
        .str.strip()
    )

    populated = (
        s.notna()
        &
        (s != "")
    )

    completeness.append(
        {
            "field": col,
            "nonempty_rows": int(
                populated.sum()
            ),
            "distinct_nonempty": int(
                s[
                    populated
                ].nunique()
            ),
        }
    )


print()
print("=" * 90)
print("ADDRESS FIELD COMPLETENESS")
print("=" * 90)

display(
    pd.DataFrame(
        completeness
    )
)


# ============================================================
# SAFE REGION RECOVERY
#
# We do NOT infer from organization name.
#
# We only pass existing address fields through the
# already-tested normalize_region().
#
# The function returns a region only if it recognizes
# an explicit oblast / Kyiv / Crimea / Sevastopol signal.
# ============================================================

RECOVERY_PRIORITY = [
    "region",
    "address_uk",
    "common",
    "district",
    "city",
]


def recover_region_from_address(row):

    for field in RECOVERY_PRIORITY:

        if field not in row.index:
            continue

        value = row[field]

        canonical = normalize_region(
            value
        )

        if canonical is not None:

            return pd.Series(
                {
                    "recovered_region":
                        canonical,

                    "recovered_from_field":
                        field,

                    "recovered_from_value":
                        str(value),
                }
            )

    return pd.Series(
        {
            "recovered_region":
                pd.NA,

            "recovered_from_field":
                pd.NA,

            "recovered_from_value":
                pd.NA,
        }
    )


recovered = (
    missing_addresses
    .apply(
        recover_region_from_address,
        axis=1,
    )
)


missing_addresses = pd.concat(
    [
        missing_addresses,
        recovered,
    ],
    axis=1,
)


# ============================================================
# IF MULTIPLE ADDRESS ROWS EXIST:
# check whether they imply conflicting regions
# ============================================================

region_candidates = (
    missing_addresses[
        missing_addresses[
            "recovered_region"
        ].notna()
    ][
        [
            "organization_id",
            "recovered_region",
        ]
    ]
    .drop_duplicates()
)


candidate_counts = (
    region_candidates
    .groupby(
        "organization_id"
    )[
        "recovered_region"
    ]
    .nunique()
)


conflicting_ids = set(
    candidate_counts[
        candidate_counts > 1
    ].index
)


print()
print("=" * 90)
print("RECOVERY RESULT")
print("=" * 90)

print(
    "Organizations recoverable from address data:",
    region_candidates[
        "organization_id"
    ].nunique(),
)

print(
    "Organizations with conflicting address-region signals:",
    len(
        conflicting_ids
    ),
)

print(
    "Still unresolved:",
    len(missing_ref)
    -
    region_candidates[
        "organization_id"
    ].nunique(),
)


# ============================================================
# RECOVERY METHODS
# ============================================================

recovery_methods = (
    missing_addresses[
        missing_addresses[
            "recovered_region"
        ].notna()
    ][
        [
            "organization_id",
            "recovered_from_field",
        ]
    ]
    .drop_duplicates()
    [
        "recovered_from_field"
    ]
    .value_counts(
        dropna=False
    )
)


print()
print(
    "Recovery fields:"
)

print(
    recovery_methods
)


# ============================================================
# SAFE ONE-REGION CANDIDATES
# ============================================================

safe_ids = set(
    candidate_counts[
        candidate_counts == 1
    ].index
)


safe_recovery = (
    missing_addresses[
        missing_addresses[
            "organization_id"
        ].isin(
            safe_ids
        )
        &
        missing_addresses[
            "recovered_region"
        ].notna()
    ][
        [
            "organization_id",
            "organization_code",
            "organization_name_current",
            "party_name_current",

            "address_type",

            "recovered_region",
            "recovered_from_field",
            "recovered_from_value",
        ]
    ]
    .drop_duplicates(
        [
            "organization_id",
            "recovered_region",
        ]
    )
    .sort_values(
        [
            "recovered_region",
            "organization_name_current",
        ]
    )
)


print()
print("=" * 90)
print("SAFE RECOVERY SAMPLE")
print("=" * 90)

display(
    safe_recovery.head(
        100
    )
)


# ============================================================
# STILL UNRESOLVED
# ============================================================

recovered_ids = set(
    region_candidates[
        "organization_id"
    ]
)


unresolved = (
    missing_ref[
        ~missing_ref[
            "organization_id"
        ].isin(
            recovered_ids
        )
    ]
    .sort_values(
        [
            "party_name_current",
            "organization_name_current",
        ]
    )
)


print()
print("=" * 90)
print("UNRESOLVED SAMPLE")
print("=" * 90)

display(
    unresolved.head(
        100
    )
)


# ============================================================
# CONFLICTS, IF ANY
# ============================================================

print()
print("=" * 90)
print("CONFLICTING REGION SIGNALS")
print("=" * 90)

if conflicting_ids:

    display(
        missing_addresses[
            missing_addresses[
                "organization_id"
            ].isin(
                conflicting_ids
            )
            &
            missing_addresses[
                "recovered_region"
            ].notna()
        ][
            [
                "organization_id",
                "organization_name_current",
                "address_type",

                "region",
                "address_uk",
                "common",
                "district",
                "city",

                "recovered_region",
                "recovered_from_field",
            ]
        ]
        .sort_values(
            [
                "organization_id",
                "address_type",
            ]
        )
    )

else:

    print(
        "No conflicts."
    )

MISSING REGION DIAGNOSTIC
Offices missing canonical region: 429
Address rows available for them: 88
Organizations with at least one address row: 85

ADDRESS FIELD COMPLETENESS


,field,nonempty_rows,distinct_nonempty
0,region,0,0
1,address_uk,0,0
2,common,0,0
3,district,1,1
4,city,1,1
5,street,16,14
6,address_en,0,0



RECOVERY RESULT
Organizations recoverable from address data: 0
Organizations with conflicting address-region signals: 0
Still unresolved: 429

Recovery fields:
Series([], Name: count, dtype: int64)

SAFE RECOVERY SAMPLE


,organization_id,organization_code,organization_name_current,party_name_current,address_type,recovered_region,recovered_from_field,recovered_from_value



UNRESOLVED SAMPLE


,organization_id,organization_code,organization_name_current,party_name_current
4974,809fa71d-cf65-4f65-a396-f24f63d44c3b,36915704,КРИМСЬКА РЕСПУБЛІКАНСЬКА ОРГАНІЗАЦІЯ ВСЕУКРАЇН...,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»
3315,57803656-dadc-4624-9237-3dfafbbc7cc9,36966481,ПОЛІТИЧНА ПАРТІЯ СЕВАСТОПОЛЬСЬКА МІСЬКА ОРГАНІ...,ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «СВОБОДА»
6564,a9a3277a-8892-4230-a702-a6ed6c4ea591,36201683,МИКОЛАЇВСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІ...,"ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
9850,fe28381b-b516-4591-859b-8773da1a0aa3,36277679,ПОЛІТИЧНА ПАРТІЯ ЗАПОРІЗЬКА ОБЛАСНА ПАРТІЙНА О...,"ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
5434,8c7738d6-7182-44af-86c3-fa18debffa1d,36200260,ПОЛІТИЧНА ПАРТІЯ КІРОВОГРАДСЬКА ОБЛАСНА ПАРТІЙ...,"ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
1052,1b3e79d6-a937-4db9-a76a-24f43eb29f22,36084771,ПОЛІТИЧНА ПАРТІЯ РІВНЕНСЬКА ОБЛАСНА ПАРТІЙНА О...,"ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
5169,85823f0a-7ada-4aa2-9da5-16f01ab5ee16,35423151,ПОЛІТИЧНА ПАРТІЯ РЕГІОНАЛЬНА ПАРТІЙНА ОРГАНІЗА...,"ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
4213,6d32005e-fd6f-4aca-90e7-7d1b6ccd8ea9,36212784,ПОЛІТИЧНА ПАРТІЯ РЕГІОНАЛЬНА ПАРТІЙНА ОРГАНІЗА...,"ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
1974,345fa341-3865-4754-8155-da776a70f3b9,36186351,"РПО ПП ""РУСИЧИ"" В М.КИЄВІ","ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
364,095bbf17-34b8-4f9a-9045-733b68b33843,36150765,"РПОПП ""РУСИЧИ""","ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""



CONFLICTING REGION SIGNALS
No conflicts.


In [103]:
from pathlib import Path
import pandas as pd

from politdata.normalization.reference import (
    normalize_region,
)


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

REF_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "organization_reference.parquet"
)

CONTEXT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "report_context.parquet"
)


ref = pd.read_parquet(
    REF_PATH
)

context = pd.read_parquet(
    CONTEXT_PATH
)


# ============================================================
# MISSING REGIONS
# ============================================================

missing = (
    ref[
        (ref["organization_level"] == "office")
        &
        ref["region"].isna()
    ]
    .copy()
)


report_org_ids = set(
    context["organization_id"]
    .astype(str)
    .unique()
)

missing[
    "has_selected_reports"
] = (
    missing["organization_id"]
    .astype(str)
    .isin(report_org_ids)
)


# ============================================================
# SAFE NAME FALLBACK
#
# Uses ONLY the already conservative normalize_region().
# Therefore:
#
# "ЛЬВІВСЬКА ОБЛАСНА..." -> Львівська область
# "КИЇВСЬКА МІСЬКА..."   -> м. Київ
#
# but:
# "БІЛОЦЕРКІВСЬКА..."    -> NA
#
# No city-to-oblast guessing here.
# ============================================================

missing[
    "region_from_explicit_name"
] = (
    missing[
        "organization_name_current"
    ]
    .map(
        normalize_region
    )
    .astype("string")
)


print("=" * 90)
print("MISSING REGION PRIORITY")
print("=" * 90)

print(
    "All missing-region offices:",
    len(missing),
)

print(
    "Missing-region offices WITH reports:",
    int(
        missing[
            "has_selected_reports"
        ].sum()
    ),
)

print(
    "Recoverable from explicit region in name:",
    int(
        missing[
            "region_from_explicit_name"
        ].notna()
        .sum()
    ),
)

print(
    "WITH reports and recoverable from name:",
    int(
        (
            missing[
                "has_selected_reports"
            ]
            &
            missing[
                "region_from_explicit_name"
            ].notna()
        ).sum()
    ),
)


remaining_report_orgs = (
    missing[
        missing[
            "has_selected_reports"
        ]
        &
        missing[
            "region_from_explicit_name"
        ].isna()
    ][
        [
            "organization_id",
            "organization_code",
            "organization_name_current",
            "party_name_current",
        ]
    ]
    .sort_values(
        [
            "party_name_current",
            "organization_name_current",
        ]
    )
)


print()
print(
    "WITH reports still unresolved after safe name fallback:",
    len(
        remaining_report_orgs
    ),
)


print()
print("=" * 90)
print("SAFE EXPLICIT-NAME RECOVERY SAMPLE")
print("=" * 90)

display(
    missing[
        missing[
            "region_from_explicit_name"
        ].notna()
    ][
        [
            "organization_code",
            "organization_name_current",
            "region_from_explicit_name",
            "has_selected_reports",
        ]
    ]
    .head(100)
)


print()
print("=" * 90)
print("REPORTING OFFICES STILL UNRESOLVED")
print("=" * 90)

display(
    remaining_report_orgs.head(
        200
    )
)

MISSING REGION PRIORITY
All missing-region offices: 429
Missing-region offices WITH reports: 310
Recoverable from explicit region in name: 233
WITH reports and recoverable from name: 173

WITH reports still unresolved after safe name fallback: 137

SAFE EXPLICIT-NAME RECOVERY SAMPLE


,organization_code,organization_name_current,region_from_explicit_name,has_selected_reports
21,33704853,Політична Партія «СПРАВЕДЛИВА УКРАЇНА» (ПЯТИХА...,Дніпропетровська область,True
33,39950139,ЛЬВІВСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІТИЧ...,Львівська область,True
57,26554575,ПОЛІТИЧНА ПАРТІЯ ПОЛТАВСЬКИЙ ОБЛАСНИЙ ОСЕРЕДОК...,Полтавська область,True
67,33825519,Донецька обласна організація Патріотичної парт...,Донецька область,False
149,42581730,"Одеська обласна партійна організація партії ""С...",Одеська область,True
306,33986086,ПОЛІТИЧНА ПАРТІЯ ЗАПОРІЗЬКА ОБЛАСНА ОРГАНІЗАЦІ...,Запорізька область,True
307,40667229,"РЕГІОНАЛЬНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ""ПЕР...",Черкаська область,True
375,39345374,Кіровоградська обласна партійна організація По...,Кіровоградська область,False
397,39999593,ПОЛІТИЧНА ПАРТІЯ КИЇВСЬКА ОБЛАСНА ПАРТІЙНА ОРГ...,Київська область,True
762,33891005,ПОЛТАВСЬКА ОБЛАСНА ПАРТІЙНА ОРГАНІЗАЦІЯ ПАРТІЇ...,Полтавська область,True



REPORTING OFFICES STILL UNRESOLVED


,organization_id,organization_code,organization_name_current,party_name_current
1974,345fa341-3865-4754-8155-da776a70f3b9,36186351,"РПО ПП ""РУСИЧИ"" В М.КИЄВІ","ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
364,095bbf17-34b8-4f9a-9045-733b68b33843,36150765,"РПОПП ""РУСИЧИ""","ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
3837,641ad9f9-2436-46a2-9100-05a40d543384,36127474,"СМПО ПП ""РУСИЧИ""","ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
5590,9036d187-f436-4702-af4e-c00d5f531d42,36279316,"ЧОПО ПП ""РУСИЧИ""","ПОЛІТИЧНА ПАРТІЯ ""БРАВО"""
5230,87332746-0d31-4812-bc2e-cbf75da75f8a,37284074,ПРИЛУЦЬКА МО ВСЕУКРАЇНСЬКОЇ КОЗАЦЬКОЇ ПАРТІЇ,"ПОЛІТИЧНА ПАРТІЯ ""ВСЕУКРАЇНСЬКА КОЗАЦЬКА ПАРТІЯ"""
2122,382a46e3-f38e-4d27-9fff-74657356cc2c,36822657,БІЛОЦЕРКІВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ П...,"ПОЛІТИЧНА ПАРТІЯ ""ЗА УКРАЇНУ!"""
503,0d235936-85f9-4135-8f63-a8b7cdf27419,37260708,БІЛЯЇВСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТ...,"ПОЛІТИЧНА ПАРТІЯ ""ЗА УКРАЇНУ!"""
2494,4232fad4-460a-40ce-8eee-ee4fc1f03a4c,36598144,БАЛАКЛІЙСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ...,"ПОЛІТИЧНА ПАРТІЯ ""ЗА УКРАЇНУ!"""
2150,38fc0714-e112-4014-9778-0a36a3a5aeab,36914826,БОГОРОДЧАНСЬКА РАЙОННА ОРГАНІЗАЩЯ ПОШТИЧНОЇ П...,"ПОЛІТИЧНА ПАРТІЯ ""ЗА УКРАЇНУ!"""
799,14de4029-c99d-416d-b3c4-2abeb46a1adc,37183979,ВАСИЛЬКІВСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІ...,"ПОЛІТИЧНА ПАРТІЯ ""ЗА УКРАЇНУ!"""


In [105]:
from pathlib import Path
import textwrap
import subprocess
import sys
import importlib

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

MODULE_PATH = (
    ROOT
    / "src"
    / "politdata"
    / "normalization"
    / "reference.py"
)

TEST_PATH = (
    ROOT
    / "tests"
    / "test_reference.py"
)

OUT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. WRITE COMPLETE CURRENT reference.py
# ============================================================

MODULE_CODE = r'''
from __future__ import annotations

import re
import unicodedata

import pandas as pd


# ============================================================
# TEXT
# ============================================================

def clean_text(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text or None


# ============================================================
# REGION NORMALIZATION
# ============================================================

REGION_RULES = [
    ("вінниц", "Вінницька область"),
    ("волин", "Волинська область"),
    ("дніпропетров", "Дніпропетровська область"),
    ("донець", "Донецька область"),
    ("житомир", "Житомирська область"),
    ("закарпат", "Закарпатська область"),
    ("запоріз", "Запорізька область"),
    ("івано-франків", "Івано-Франківська область"),
    ("київськ", "Київська область"),
    ("кіровоград", "Кіровоградська область"),
    ("луган", "Луганська область"),
    ("львів", "Львівська область"),
    ("миколаїв", "Миколаївська область"),
    ("одес", "Одеська область"),
    ("полтав", "Полтавська область"),
    ("рівнен", "Рівненська область"),
    ("сум", "Сумська область"),
    ("терноп", "Тернопільська область"),
    ("харків", "Харківська область"),
    ("херсон", "Херсонська область"),
    ("хмельниц", "Хмельницька область"),
    ("черкас", "Черкаська область"),
    ("чернів", "Чернівецька область"),
    ("черніг", "Чернігівська область"),
]


def normalize_region(value):
    """
    Conservative region normalization.

    Recognizes explicit oblast / Kyiv city /
    Crimea / Sevastopol signals.

    Does NOT infer oblast from locality names
    such as Біла Церква, Прилуки, Борислав etc.
    """

    text = clean_text(value)

    if text is None:
        return None

    lowered = text.casefold()


    # --------------------------------------------------------
    # KYIV CITY
    #
    # Must be checked before generic "київськ" oblast rule.
    #
    # Examples:
    # - м. Київ
    # - м.Києві
    # - в м. Києві
    # - Київська міська організація
    # --------------------------------------------------------

    if re.search(
        r"\bкиївськ\w*\s+міськ\w*",
        lowered,
    ):
        return "м. Київ"


    if re.search(
        r"\bм\.?\s*ки(?:їв|єв)\w*",
        lowered,
    ):
        return "м. Київ"


    if re.search(
        r"\bміст\w*\s+ки(?:їв|єв)\w*",
        lowered,
    ):
        return "м. Київ"


    if lowered == "київ":
        return "м. Київ"


    # --------------------------------------------------------
    # SEVASTOPOL
    # --------------------------------------------------------

    if "севастопол" in lowered:
        return "м. Севастополь"


    # --------------------------------------------------------
    # CRIMEA
    # --------------------------------------------------------

    if "крим" in lowered:
        return "Автономна Республіка Крим"


    # --------------------------------------------------------
    # OBLASTS
    # --------------------------------------------------------

    for token, canonical in REGION_RULES:

        if token in lowered:
            return canonical


    return None


# ============================================================
# ORGANIZATION REFERENCE
# ============================================================

def build_organization_reference(
    organizations: pd.DataFrame,
    addresses: pd.DataFrame,
) -> pd.DataFrame:

    org = organizations.copy()
    addr = addresses.copy()


    required = {
        "organization_id",
        "root_party_id",
        "entity_type",
        "code",
        "name",
    }

    missing = (
        required
        - set(org.columns)
    )

    if missing:
        raise ValueError(
            "organizations missing: "
            + ", ".join(
                sorted(missing)
            )
        )


    # --------------------------------------------------------
    # Stable identifiers
    # --------------------------------------------------------

    for col in [
        "organization_id",
        "root_party_id",
        "code",
        "name",
        "entity_type",
    ]:

        org[col] = (
            org[col]
            .astype("string")
        )


    # --------------------------------------------------------
    # CURRENT CENTRAL-PARTY REFERENCE
    #
    # Current name is intentionally propagated across
    # historical analytical data.
    #
    # Historical names will live in a separate history table.
    # --------------------------------------------------------

    parties = (
        org[
            org["entity_type"]
            == "party"
        ][
            [
                "organization_id",
                "code",
                "name",
            ]
        ]
        .rename(
            columns={
                "organization_id":
                    "root_party_id",

                "code":
                    "party_code",

                "name":
                    "party_name_current",
            }
        )
    )


    if (
        parties[
            "root_party_id"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate root party IDs."
        )


    # --------------------------------------------------------
    # ADDRESS SELECTION
    #
    # Priority:
    # 1. populated register region
    # 2. populated actual region
    # 3. other populated address type
    # 4. empty rows only if nothing else exists
    # --------------------------------------------------------

    addr["organization_id"] = (
        addr["organization_id"]
        .astype("string")
    )

    addr["region_source"] = (
        addr["region"]
        .astype("string")
    )


    region_present = (
        addr[
            "region_source"
        ].notna()
        &
        (
            addr[
                "region_source"
            ]
            .str.strip()
            != ""
        )
    )


    addr[
        "_missing_region"
    ] = (
        ~region_present
    ).astype(int)


    address_priority = {
        "register": 0,
        "actual": 1,
    }


    addr[
        "_address_priority"
    ] = (
        addr[
            "address_type"
        ]
        .map(
            address_priority
        )
        .fillna(2)
    )


    selected_addr = (
        addr
        .sort_values(
            [
                "organization_id",
                "_missing_region",
                "_address_priority",
            ]
        )
        .drop_duplicates(
            "organization_id",
            keep="first",
        )
        [
            [
                "organization_id",
                "address_type",
                "region_source",
            ]
        ]
        .rename(
            columns={
                "address_type":
                    "region_source_address_type",
            }
        )
    )


    # --------------------------------------------------------
    # MAIN REFERENCE
    # --------------------------------------------------------

    out = (
        org
        .rename(
            columns={
                "code":
                    "organization_code",

                "name":
                    "organization_name_current",
            }
        )
        .merge(
            parties,
            on="root_party_id",
            how="left",
            validate="many_to_one",
        )
        .merge(
            selected_addr,
            on="organization_id",
            how="left",
            validate="one_to_one",
        )
    )


    out[
        "organization_level"
    ] = (
        out[
            "entity_type"
        ]
        .map(
            {
                "party": "central",
                "office": "office",
            }
        )
        .astype("string")
    )


    # ========================================================
    # REGION RESOLUTION
    # ========================================================

    out[
        "region"
    ] = (
        out[
            "region_source"
        ]
        .map(
            normalize_region
        )
        .astype("string")
    )


    out[
        "region_resolution_method"
    ] = pd.Series(
        pd.NA,
        index=out.index,
        dtype="string",
    )


    out[
        "region_resolution_source"
    ] = pd.Series(
        pd.NA,
        index=out.index,
        dtype="string",
    )


    # --------------------------------------------------------
    # Address-based region
    # --------------------------------------------------------

    address_mask = (
        out[
            "organization_level"
        ]
        == "office"
    ) & (
        out["region"].notna()
    )


    out.loc[
        address_mask,
        "region_resolution_method",
    ] = (
        out.loc[
            address_mask,
            "region_source_address_type",
        ]
        .fillna("other")
        .astype("string")
        + "_address"
    )


    out.loc[
        address_mask,
        "region_resolution_source",
    ] = (
        out.loc[
            address_mask,
            "region_source",
        ]
        .astype("string")
    )


    # --------------------------------------------------------
    # SAFE NAME FALLBACK
    #
    # Only normalize_region().
    #
    # This means:
    #
    # "ЛЬВІВСЬКА ОБЛАСНА..." -> Львівська область
    # "КИЇВСЬКА МІСЬКА..."   -> м. Київ
    # "в м. Києві"           -> м. Київ
    #
    # but:
    #
    # "БІЛОЦЕРКІВСЬКА..."    -> unresolved
    # "ПРИЛУЦЬКА..."         -> unresolved
    # --------------------------------------------------------

    office_missing_region = (
        out[
            "organization_level"
        ]
        == "office"
    ) & (
        out[
            "region"
        ].isna()
    )


    region_from_name = (
        out.loc[
            office_missing_region,
            "organization_name_current",
        ]
        .map(
            normalize_region
        )
        .astype("string")
    )


    resolved_indexes = (
        region_from_name[
            region_from_name.notna()
        ]
        .index
    )


    out.loc[
        resolved_indexes,
        "region",
    ] = (
        region_from_name.loc[
            resolved_indexes
        ]
    )


    out.loc[
        resolved_indexes,
        "region_resolution_method",
    ] = (
        "explicit_organization_name"
    )


    out.loc[
        resolved_indexes,
        "region_resolution_source",
    ] = (
        out.loc[
            resolved_indexes,
            "organization_name_current",
        ]
        .astype("string")
    )


    # --------------------------------------------------------
    # Central organizations
    # --------------------------------------------------------

    central_mask = (
        out[
            "organization_level"
        ]
        == "central"
    )


    out.loc[
        central_mask,
        "region",
    ] = "Україна"


    out.loc[
        central_mask,
        "region_resolution_method",
    ] = "central_national"


    out.loc[
        central_mask,
        "region_resolution_source",
    ] = "entity_type=party"


    # --------------------------------------------------------
    # Remaining unresolved
    # --------------------------------------------------------

    unresolved_mask = (
        out[
            "organization_level"
        ]
        == "office"
    ) & (
        out[
            "region"
        ].isna()
    )


    out.loc[
        unresolved_mask,
        "region_resolution_method",
    ] = "unresolved"


    # ========================================================
    # QA
    # ========================================================

    if (
        out[
            "organization_id"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "organization_reference "
            "not unique by organization_id."
        )


    if (
        out[
            "party_name_current"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Some organizations have "
            "no current party name."
        )


    if (
        out[
            "party_code"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Some organizations have "
            "no root-party code."
        )


    columns = [
        "organization_id",
        "root_party_id",

        "organization_level",

        "organization_code",
        "organization_name_current",

        "party_code",
        "party_name_current",

        "region",

        "region_source",
        "region_source_address_type",

        "region_resolution_method",
        "region_resolution_source",
    ]


    for optional in [
        "is_active",
        "parent_id",
    ]:

        if optional in out.columns:

            columns.append(
                optional
            )


    return (
        out[
            columns
        ]
        .reset_index(
            drop=True
        )
    )


# ============================================================
# REPORT CONTEXT
# ============================================================

def build_report_context(
    analysis_manifest: pd.DataFrame,
    organization_reference: pd.DataFrame,
) -> pd.DataFrame:

    reports = (
        analysis_manifest
        .copy()
    )


    required = {
        "report_id",
        "organization_id",
        "root_party_id",
        "year",
        "quarter",

        "official_selected_report_id",
        "analysis_selected_report_id",
        "analysis_selection_method",
        "analysis_override",
    }


    missing = (
        required
        - set(reports.columns)
    )

    if missing:

        raise ValueError(
            "analysis manifest missing: "
            + ", ".join(
                sorted(missing)
            )
        )


    for col in [
        "organization_id",
        "root_party_id",
        "official_selected_report_id",
        "analysis_selected_report_id",
    ]:

        reports[col] = (
            reports[col]
            .astype("string")
        )


    # --------------------------------------------------------
    # RAW report actually used by analytical pipeline
    # --------------------------------------------------------

    reports[
        "source_report_id"
    ] = (
        reports[
            "analysis_selected_report_id"
        ]
    )


    reports[
        "year"
    ] = (
        pd.to_numeric(
            reports["year"],
            errors="raise",
        )
        .astype(int)
    )


    reports[
        "quarter"
    ] = (
        pd.to_numeric(
            reports["quarter"],
            errors="raise",
        )
        .astype(int)
    )


    # --------------------------------------------------------
    # PERIOD ORDER
    #
    # quarter=5 = annual
    # --------------------------------------------------------

    reports[
        "report_period_order"
    ] = (
        reports["year"]
        * 10
        +
        reports["quarter"]
    )


    reports[
        "period_label"
    ] = (
        reports.apply(
            lambda r:
                (
                    f"{r['year']} annual"
                    if r["quarter"] == 5
                    else
                    f"{r['year']} Q{r['quarter']}"
                ),
            axis=1,
        )
        .astype("string")
    )


    # --------------------------------------------------------
    # LATEST DATA PER ORGANIZATION
    # --------------------------------------------------------

    latest_order = (
        reports
        .groupby(
            "organization_id"
        )[
            "report_period_order"
        ]
        .transform(
            "max"
        )
    )


    reports[
        "is_latest_report_for_organization"
    ] = (
        reports[
            "report_period_order"
        ]
        == latest_order
    )


    reports[
        "data_recency_status"
    ] = (
        reports[
            "is_latest_report_for_organization"
        ]
        .map(
            {
                True:
                    "latest_data",

                False:
                    "historical_data",
            }
        )
        .astype("string")
    )


    # --------------------------------------------------------
    # CURRENT ORGANIZATION/PARTY REFERENCE
    # --------------------------------------------------------

    ref_cols = [
        "organization_id",

        "organization_level",

        "organization_code",
        "organization_name_current",

        "party_code",
        "party_name_current",

        "region",

        "region_source",
        "region_source_address_type",

        "region_resolution_method",
        "region_resolution_source",
    ]


    out = reports.merge(
        organization_reference[
            ref_cols
        ],
        on="organization_id",
        how="left",
        validate="many_to_one",
    )


    # --------------------------------------------------------
    # QA
    # --------------------------------------------------------

    if (
        out[
            "source_report_id"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Missing analytical source_report_id."
        )


    if (
        out[
            "source_report_id"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate analytical source_report_id."
        )


    latest_counts = (
        out[
            out[
                "is_latest_report_for_organization"
            ]
        ]
        .groupby(
            "organization_id"
        )
        .size()
    )


    if (
        latest_counts > 1
    ).any():

        raise ValueError(
            "More than one latest period "
            "for an organization."
        )


    return (
        out
        .sort_values(
            [
                "organization_id",
                "report_period_order",
            ]
        )
        .reset_index(
            drop=True
        )
    )
'''


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)

print(
    "Written complete module:",
    MODULE_PATH,
)


# ============================================================
# 2. ADD KYIV REGRESSION TESTS
# ============================================================

test_text = TEST_PATH.read_text(
    encoding="utf-8"
)


REGRESSION_TESTS = r'''


def test_region_kyiv_city_organization_name():

    assert (
        normalize_region(
            "КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ "
            "ПОЛІТИЧНОЇ ПАРТІЇ"
        )
        == "м. Київ"
    )


def test_region_kyiv_city_inflected():

    assert (
        normalize_region(
            "Дарницька районна в м. Києві організація"
        )
        == "м. Київ"
    )


def test_region_kyiv_oblast_organization():

    assert (
        normalize_region(
            "КИЇВСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ "
            "ПОЛІТИЧНОЇ ПАРТІЇ"
        )
        == "Київська область"
    )
'''


if (
    "test_region_kyiv_city_organization_name"
    not in test_text
):

    TEST_PATH.write_text(
        test_text
        + REGRESSION_TESTS,
        encoding="utf-8",
    )

    print(
        "Added Kyiv regression tests."
    )

else:

    print(
        "Kyiv regression tests already present."
    )


# ============================================================
# 3. RUN ALL TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 90)
print("TESTS")
print("=" * 90)

print(
    result.stdout
)

if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode,
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Reference tables were not rebuilt."
    )


# ============================================================
# 4. RELOAD MODULE IN JUPYTER
# ============================================================

import politdata.normalization.reference as reference_module

importlib.reload(
    reference_module
)


build_organization_reference = (
    reference_module
    .build_organization_reference
)

build_report_context = (
    reference_module
    .build_report_context
)


# ============================================================
# 5. LOAD FOUNDATION DATA
# ============================================================

organizations = pd.read_parquet(
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "organizations.parquet"
)


addresses = pd.read_parquet(
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "organization_addresses.parquet"
)


analysis_manifest = pd.read_parquet(
    ROOT
    / "data"
    / "interim"
    / "reports"
    / "analysis_selected_reports_manifest.parquet"
)


# ============================================================
# 6. REBUILD ORGANIZATION REFERENCE
# ============================================================

organization_reference = (
    build_organization_reference(
        organizations,
        addresses,
    )
)


ORG_REFERENCE_PATH = (
    OUT_DIR
    / "organization_reference.parquet"
)


organization_reference.to_parquet(
    ORG_REFERENCE_PATH,
    index=False,
)


# ============================================================
# 7. REBUILD REPORT CONTEXT
# ============================================================

report_context = (
    build_report_context(
        analysis_manifest,
        organization_reference,
    )
)


REPORT_CONTEXT_PATH = (
    OUT_DIR
    / "report_context.parquet"
)


report_context.to_parquet(
    REPORT_CONTEXT_PATH,
    index=False,
)


# ============================================================
# 8. REGION QA
# ============================================================

print()
print("=" * 90)
print("REGION RESOLUTION QA")
print("=" * 90)


print(
    organization_reference[
        "region_resolution_method"
    ]
    .value_counts(
        dropna=False
    )
)


office_mask = (
    organization_reference[
        "organization_level"
    ]
    == "office"
)


unresolved_offices = (
    organization_reference[
        office_mask
        &
        organization_reference[
            "region"
        ].isna()
    ]
)


print()
print(
    "All office rows still unresolved:",
    len(
        unresolved_offices
    ),
)


reporting_ids = set(
    report_context[
        "organization_id"
    ]
    .astype(str)
)


reporting_unresolved = (
    unresolved_offices[
        unresolved_offices[
            "organization_id"
        ]
        .astype(str)
        .isin(
            reporting_ids
        )
    ]
)


print(
    "Reporting offices still unresolved:",
    len(
        reporting_unresolved
    ),
)


# ============================================================
# 9. KYIV FALLBACK QA
#
# Only test rows for which organization NAME
# was actually the source of the region.
#
# Address remains authoritative when present.
# ============================================================

kyiv_name_fallback = (
    organization_reference[
        (
            organization_reference[
                "region_resolution_method"
            ]
            == "explicit_organization_name"
        )
        &
        organization_reference[
            "organization_name_current"
        ]
        .str.contains(
            r"КИЇВСЬК\w*\s+МІСЬК",
            case=False,
            regex=True,
            na=False,
        )
    ][
        [
            "organization_code",
            "organization_name_current",
            "region",
            "region_resolution_method",
        ]
    ]
)


print()
print("=" * 90)
print("KYIV NAME-FALLBACK CHECK")
print("=" * 90)


display(
    kyiv_name_fallback.head(
        100
    )
)


wrong_kyiv = (
    kyiv_name_fallback[
        kyiv_name_fallback[
            "region"
        ]
        != "м. Київ"
    ]
)


print(
    "Kyiv-city NAME fallbacks not classified as m. Київ:",
    len(
        wrong_kyiv
    ),
)


# ============================================================
# 10. REPORT CONTEXT QA
# ============================================================

print()
print("=" * 90)
print("REPORT CONTEXT QA")
print("=" * 90)


print(
    "Rows:",
    len(
        report_context
    )
)


print(
    "Unique analytical source reports:",
    report_context[
        "source_report_id"
    ].nunique()
)


print(
    "Organizations represented:",
    report_context[
        "organization_id"
    ].nunique()
)


print(
    "Latest reports:",
    int(
        report_context[
            "is_latest_report_for_organization"
        ].sum()
    )
)


print(
    "Analysis overrides:",
    int(
        report_context[
            "analysis_override"
        ].sum()
    )
)


# ============================================================
# 11. UNRESOLVED REPORTING SAMPLE
# ============================================================

print()
print("=" * 90)
print("UNRESOLVED REPORTING OFFICES")
print("=" * 90)


display(
    reporting_unresolved[
        [
            "organization_id",
            "organization_code",
            "organization_name_current",
            "party_name_current",
            "region_resolution_method",
        ]
    ]
    .head(
        150
    )
)


print()
print("=" * 90)
print("FILES UPDATED")
print("=" * 90)

print(
    ORG_REFERENCE_PATH
)

print(
    REPORT_CONTEXT_PATH
)

Written complete module: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\reference.py
Added Kyiv regression tests.

TESTS
......................................                                   [100%]
38 passed in 6.94s

Return code: 0

REGION RESOLUTION QA
region_resolution_method
register_address              9167
central_national               321
explicit_organization_name     235
unresolved                     194
Name: count, dtype: Int64

All office rows still unresolved: 194
Reporting offices still unresolved: 135

KYIV NAME-FALLBACK CHECK


,organization_code,organization_name_current,region,region_resolution_method
2574,39982693,ПОЛІТИЧНА ПАРТІЯ КИЇВСЬКА МІСЬКА 39982693 ОРГА...,м. Київ,explicit_organization_name
3196,26346859,КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ ...,м. Київ,explicit_organization_name
4768,43795245,КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАРТІЇ...,м. Київ,explicit_organization_name
6470,34192122,ПОЛІТИЧНА ПАРТІЯ КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ В...,м. Київ,explicit_organization_name
8950,40012446,КИЇВСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ ПОЛІТИЧ...,м. Київ,explicit_organization_name
9221,36175753,ПОЛІТИЧНА ПАРТІЯ КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ В...,м. Київ,explicit_organization_name


Kyiv-city NAME fallbacks not classified as m. Київ: 0

REPORT CONTEXT QA
Rows: 78791
Unique analytical source reports: 78791
Organizations represented: 8514
Latest reports: 8514
Analysis overrides: 1

UNRESOLVED REPORTING OFFICES


,organization_id,organization_code,organization_name_current,party_name_current,region_resolution_method
62,01657d0b-efe0-4838-bedf-6242d6412a88,34948502,БОРИСПІЛЬСЬКА РАЙОННА ПАРТІЙНА ОРГАНІЗАЦІЯ ПАР...,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,unresolved
70,0191d311-d83b-4dae-b3b1-4f2b5595861d,26278164,КОВЕЛЬСЬКА МІСЬКА ПАРТІЙНА ОРГАНІЗАЦІЯ «ВСЕУКР...,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКА ПАРТІЯ ДУХОВНО...,unresolved
74,01c94d7b-722a-47ca-a8ed-6ae91cb39922,26363094,"""КОРАБЕЛЬНА РАЙОННА ОРГАНІЗАЦІЯ ВСЕУКРАЇНСЬКОЇ...",ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКА ПАРТІЯ ДУХОВНО...,unresolved
99,026e2d42-4336-4f3a-af95-a5a06d4bea16,26363125,ПОЛІТИЧНА ПАРТІЯ ДОМАНІВСЬКА РАЙОННА ОРГАНІЗАЦ...,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКА ПАРТІЯ ДУХОВНО...,unresolved
187,04c55344-7690-48b9-9e03-a852188b2187,36776894,КОРСУНЬ-ШЕВЧЕНКІВСЬКА І РАЙОННА ОРГАНІЗАЦІЯ...,"ПОЛІТИЧНА ПАРТІЯ ""ЗА УКРАЇНУ!""",unresolved
261,06906a17-a140-437c-8a4a-bfc910ae9c24,33959534,КОВЕЛЬСЬКА РАЙОННА ПАРТІЙНА ОРГАНІЗАЦІЯ ПАРТІЇ...,ПОЛІТИЧНА ПАРТІЯ «КРАЇНА»,unresolved
323,0867d2d3-3327-403f-92e7-caca2b30af56,40004168,ПОЛІТИЧНА ПАРТІЯ ОЛЕВСЬКА РАЙОННА ОРГАНІЗАЦІЯ ...,ПОЛІТИЧНА ПАРТІЯ «ЛІВА ОПОЗИЦІЯ»,unresolved
332,089db16c-486f-40fb-9fb2-0844f6895d16,36976809,ЧЕРНЯХІВСЬКА РАЙОННА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПА...,"ПОЛІТИЧНА ПАРТІЯ ""ЗА УКРАЇНУ!""",unresolved
364,095bbf17-34b8-4f9a-9045-733b68b33843,36150765,"РПОПП ""РУСИЧИ""","ПОЛІТИЧНА ПАРТІЯ ""БРАВО""",unresolved
454,0bc77dc4-eb7e-450c-b54d-ba578c91afc5,26363088,ПОЛІТИЧНА ПАРТІЯ ЗАВОДСЬКА РАЙОННА ОРГАНІЗАЦІЯ...,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКА ПАРТІЯ ДУХОВНО...,unresolved



FILES UPDATED
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\organization_reference.parquet
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\report_context.parquet


In [106]:
from pathlib import Path
from collections import Counter
import json
import textwrap
import subprocess
import sys
import importlib

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

SRC_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "normalization"
)

TEST_DIR = (
    ROOT
    / "tests"
)

RAW_DIR = (
    ROOT
    / "data"
    / "raw"
    / "report_details"
)

REPORT_CONTEXT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "report_context.parquet"
)

ORG_REFERENCE_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "organization_reference.parquet"
)

FRAGMENT_ROOT = (
    ROOT
    / "data"
    / "interim"
    / "normalized_fragments"
    / "payments"
)

STATE_PATH = (
    ROOT
    / "data"
    / "interim"
    / "state"
    / "payment_normalization_state.parquet"
)

NORMALIZED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "payments"
)

ENRICHED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

for path in [
    SRC_DIR,
    TEST_DIR,
    FRAGMENT_ROOT,
    STATE_PATH.parent,
    NORMALIZED_DIR,
    ENRICHED_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 1. WRITE PERMANENT payments.py
# ============================================================

MODULE_PATH = (
    SRC_DIR
    / "payments.py"
)


MODULE_CODE = r'''
from __future__ import annotations

from dataclasses import asdict, is_dataclass
from datetime import date, datetime
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any
import json
import os
import re
import unicodedata

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from politdata.normalization.accounts import (
    normalize_account_number,
)


NORMALIZATION_VERSION = "payments_v0_1"


# ============================================================
# PAYMENT SECTIONS
# ============================================================

PAYMENT_PATHS = {
    "monetary_contributions": (
        "payment_info",
        "incoming",
        "monetary_contributions",
    ),

    "other_contributions": (
        "payment_info",
        "incoming",
        "other_contributions",
    ),

    "state_funding": (
        "payment_info",
        "incoming",
        "state_funding",
    ),

    "other_incomes": (
        "payment_info",
        "incoming",
        "other_incomes",
    ),

    "budget_expenses": (
        "payment_info",
        "outgoing",
        "budget_expenses",
    ),

    "outgoing_expenses": (
        "payment_info",
        "outgoing",
        "outgoing_expenses",
    ),

    "return_expenses": (
        "payment_info",
        "outgoing",
        "return_expenses",
    ),

    "transfer_expenses": (
        "payment_info",
        "outgoing",
        "transfer_expenses",
    ),
}


INCOMING_TYPES = {
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
}


OUTGOING_TYPES = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


# ============================================================
# SOURCE PAYMENT FIELDS OBSERVED IN POLITDATA
# ============================================================

KNOWN_SOURCE_FIELDS = {
    "created_at",
    "group_code",
    "id",
    "office_id",
    "party_id",

    "payer_account_iban",
    "payer_account_type",
    "payer_address",
    "payer_bank_address",
    "payer_bank_code",
    "payer_bank_name",
    "payer_birthday",
    "payer_code",
    "payer_name",
    "payer_type",

    "payment_amount",
    "payment_code",
    "payment_currency",
    "payment_description",
    "payment_instruction_date",
    "payment_number",
    "payment_operation_date",
    "payment_purpose",
    "payment_reason",
    "payment_type",

    "receiver_account_iban",
    "receiver_account_type",
    "receiver_address",
    "receiver_bank_address",
    "receiver_bank_code",
    "receiver_bank_name",
    "receiver_birthday",
    "receiver_code",
    "receiver_name",
    "receiver_type",

    "refund_amount",
    "refund_budget_amount",
    "refund_date",
    "refund_description",
    "refund_purpose",
    "refund_reason",

    "report_id",
    "updated_at",
}


# ============================================================
# BASIC NORMALIZATION
# ============================================================

ZERO_WIDTH_RE = re.compile(
    "[\u200b\u200c\u200d\u2060\ufeff]"
)


def clean_text(value: Any) -> str | None:

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = ZERO_WIDTH_RE.sub(
        "",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text or None


def normalize_code(value: Any) -> str | None:
    """
    Conservative identifier normalization.

    - NFKC
    - zero-width removal
    - whitespace removal

    No zero-padding.
    No digit guessing.
    """

    text = clean_text(value)

    if text is None:
        return None

    compact = re.sub(
        r"\s+",
        "",
        text,
    )

    return compact or None


def normalize_counterparty_type(
    value: Any,
) -> str | None:

    text = clean_text(value)

    if text is None:
        return None

    lowered = text.casefold()

    if (
        "фоп" in lowered
        or "фізична особа-підприємець"
        in lowered
        or "фізична особа підприємець"
        in lowered
        or "фізична особа" in lowered
    ):
        return "Фізична особа"

    if "юридична особа" in lowered:
        return "Юридична особа"

    return text


LATIN_HOMOGLYPHS = str.maketrans(
    {
        "A": "А",
        "a": "а",
        "I": "І",
        "i": "і",
        "C": "С",
        "c": "с",
        "O": "О",
        "o": "о",
    }
)


FOP_PREFIX_RE = re.compile(
    r"""^\s*
    (?:
        фоп
        |
        фізична\s+особа[\s\-–—]*підприємець
    )
    [\s:,\-–—]*
    """,
    flags=re.I | re.X,
)


def normalize_person_name(
    value: Any,
    counterparty_type: str | None = None,
) -> str | None:

    text = clean_text(value)

    if text is None:
        return None

    # Only apply person-specific transformations
    # to rows analytically recognized as natural persons.
    if counterparty_type != "Фізична особа":
        return text

    text = FOP_PREFIX_RE.sub(
        "",
        text,
    )

    # Old exploratory notebooks showed recurring
    # Latin homoglyphs inside Cyrillic personal names.
    text = text.translate(
        LATIN_HOMOGLYPHS
    )

    text = re.sub(
        r"\s*[-–—]\s*",
        "-",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text or None


# ============================================================
# DATE / TIMESTAMP / DECIMAL
# ============================================================

def parse_date(value: Any) -> date | None:

    text = clean_text(value)

    if text is None:
        return None

    parsed = pd.to_datetime(
        text,
        errors="coerce",
    )

    if pd.isna(parsed):
        return None

    return parsed.date()


def parse_timestamp(
    value: Any,
) -> datetime | None:

    text = clean_text(value)

    if text is None:
        return None

    parsed = pd.to_datetime(
        text,
        errors="coerce",
    )

    if pd.isna(parsed):
        return None

    if getattr(
        parsed,
        "tzinfo",
        None,
    ) is not None:

        parsed = (
            parsed
            .tz_convert(None)
        )

    return parsed.to_pydatetime()


def parse_decimal(
    value: Any,
) -> Decimal | None:

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass

    text = clean_text(value)

    if text is None:
        return None

    text = text.replace(
        "\u00a0",
        "",
    )

    text = text.replace(
        " ",
        "",
    )

    # Decimal comma only when no decimal point exists.
    if (
        "," in text
        and "." not in text
    ):
        text = text.replace(
            ",",
            ".",
        )

    try:
        return Decimal(text)

    except InvalidOperation:
        return None


# ============================================================
# ACCOUNT NORMALIZATION WRAPPER
#
# Robust to the exact dataclass field naming used
# in accounts.py.
# ============================================================

def _result_dict(result) -> dict:

    if result is None:
        return {}

    if isinstance(
        result,
        dict,
    ):
        return result

    if is_dataclass(result):
        return asdict(result)

    if hasattr(
        result,
        "_asdict",
    ):
        return result._asdict()

    if hasattr(
        result,
        "__dict__",
    ):
        return vars(result)

    return {}


def _first(
    mapping,
    *names,
):

    for name in names:

        if name in mapping:
            return mapping[name]

    return None


def normalize_account_fields(
    value: Any,
) -> dict:

    result = normalize_account_number(
        value
    )

    data = _result_dict(
        result
    )

    canonical = _first(
        data,
        "canonical",
        "account_canonical",
        "iban",
    )

    status = _first(
        data,
        "status",
        "account_normalization_status",
    )

    method = _first(
        data,
        "method",
        "account_normalization_method",
    )

    valid = _first(
        data,
        "valid_iban",
        "is_valid",
        "account_valid_iban",
    )

    candidate_count = _first(
        data,
        "candidate_count",
        "account_candidate_count",
    )

    candidates = _first(
        data,
        "candidates",
        "account_candidates",
    )

    normalized_text = _first(
        data,
        "normalized_text",
        "account_normalized_text",
    )

    if candidates is None:
        candidates_json = None

    elif isinstance(
        candidates,
        str,
    ):
        candidates_json = candidates

    else:
        candidates_json = json.dumps(
            candidates,
            ensure_ascii=False,
            default=str,
        )

    return {
        "raw":
            clean_text(value),

        "canonical":
            clean_text(canonical),

        "status":
            clean_text(status),

        "method":
            clean_text(method),

        "valid":
            (
                bool(valid)
                if valid is not None
                else None
            ),

        "candidate_count":
            (
                int(candidate_count)
                if candidate_count is not None
                else None
            ),

        "candidates_json":
            candidates_json,

        "normalized_text":
            clean_text(normalized_text),
    }


# ============================================================
# PYARROW SCHEMA
# ============================================================

MONEY_TYPE = pa.decimal128(
    38,
    10,
)


NORMALIZED_PAYMENT_SCHEMA = pa.schema(
    [
        # report / analytical provenance
        (
            "source_report_id",
            pa.string(),
        ),
        (
            "official_selected_report_id",
            pa.string(),
        ),
        (
            "analysis_selected_report_id",
            pa.string(),
        ),

        (
            "organization_id",
            pa.string(),
        ),
        (
            "root_party_id",
            pa.string(),
        ),

        (
            "report_year",
            pa.int32(),
        ),
        (
            "report_quarter",
            pa.int8(),
        ),
        (
            "report_period",
            pa.string(),
        ),
        (
            "report_type",
            pa.string(),
        ),

        (
            "source_is_signed",
            pa.bool_(),
        ),
        (
            "source_signed_date",
            pa.timestamp(
                "us"
            ),
        ),

        (
            "analysis_override",
            pa.bool_(),
        ),
        (
            "analysis_selection_method",
            pa.string(),
        ),

        # payment section
        (
            "source_payment_type",
            pa.string(),
        ),
        (
            "payment_direction",
            pa.string(),
        ),

        # source technical IDs
        (
            "source_row_id",
            pa.string(),
        ),
        (
            "source_party_id",
            pa.string(),
        ),
        (
            "source_office_id",
            pa.string(),
        ),
        (
            "source_report_id_in_row",
            pa.string(),
        ),

        (
            "source_created_at",
            pa.timestamp(
                "us"
            ),
        ),
        (
            "source_updated_at",
            pa.timestamp(
                "us"
            ),
        ),

        (
            "group_code",
            pa.string(),
        ),

        # payer
        (
            "payer_name_source",
            pa.string(),
        ),
        (
            "payer_name_normalized",
            pa.string(),
        ),

        (
            "payer_code_raw",
            pa.string(),
        ),
        (
            "payer_code_normalized",
            pa.string(),
        ),

        (
            "payer_type_source",
            pa.string(),
        ),
        (
            "payer_type_normalized",
            pa.string(),
        ),

        (
            "payer_address",
            pa.string(),
        ),

        (
            "payer_birthday_raw",
            pa.string(),
        ),
        (
            "payer_birthday",
            pa.date32(),
        ),

        (
            "payer_account_iban_raw",
            pa.string(),
        ),
        (
            "payer_account_iban_canonical",
            pa.string(),
        ),
        (
            "payer_account_normalization_status",
            pa.string(),
        ),
        (
            "payer_account_normalization_method",
            pa.string(),
        ),
        (
            "payer_account_valid_iban",
            pa.bool_(),
        ),
        (
            "payer_account_candidate_count",
            pa.int16(),
        ),
        (
            "payer_account_candidates",
            pa.string(),
        ),
        (
            "payer_account_normalized_text",
            pa.string(),
        ),

        (
            "payer_account_type_source",
            pa.string(),
        ),

        (
            "payer_bank_code",
            pa.string(),
        ),
        (
            "payer_bank_name",
            pa.string(),
        ),
        (
            "payer_bank_address",
            pa.string(),
        ),

        # payment
        (
            "payment_amount_raw",
            pa.string(),
        ),
        (
            "payment_amount",
            MONEY_TYPE,
        ),

        (
            "payment_code",
            pa.string(),
        ),
        (
            "payment_currency",
            pa.string(),
        ),
        (
            "payment_description",
            pa.string(),
        ),

        (
            "payment_instruction_date_raw",
            pa.string(),
        ),
        (
            "payment_instruction_date",
            pa.date32(),
        ),

        (
            "payment_number",
            pa.string(),
        ),

        (
            "payment_operation_date_raw",
            pa.string(),
        ),
        (
            "payment_operation_date",
            pa.date32(),
        ),

        (
            "payment_purpose",
            pa.string(),
        ),
        (
            "payment_reason",
            pa.string(),
        ),
        (
            "payment_type_detail_source",
            pa.string(),
        ),

        # receiver
        (
            "receiver_name_source",
            pa.string(),
        ),
        (
            "receiver_name_normalized",
            pa.string(),
        ),

        (
            "receiver_code_raw",
            pa.string(),
        ),
        (
            "receiver_code_normalized",
            pa.string(),
        ),

        (
            "receiver_type_source",
            pa.string(),
        ),
        (
            "receiver_type_normalized",
            pa.string(),
        ),

        (
            "receiver_address",
            pa.string(),
        ),

        (
            "receiver_birthday_raw",
            pa.string(),
        ),
        (
            "receiver_birthday",
            pa.date32(),
        ),

        (
            "receiver_account_iban_raw",
            pa.string(),
        ),
        (
            "receiver_account_iban_canonical",
            pa.string(),
        ),
        (
            "receiver_account_normalization_status",
            pa.string(),
        ),
        (
            "receiver_account_normalization_method",
            pa.string(),
        ),
        (
            "receiver_account_valid_iban",
            pa.bool_(),
        ),
        (
            "receiver_account_candidate_count",
            pa.int16(),
        ),
        (
            "receiver_account_candidates",
            pa.string(),
        ),
        (
            "receiver_account_normalized_text",
            pa.string(),
        ),

        (
            "receiver_account_type_source",
            pa.string(),
        ),

        (
            "receiver_bank_code",
            pa.string(),
        ),
        (
            "receiver_bank_name",
            pa.string(),
        ),
        (
            "receiver_bank_address",
            pa.string(),
        ),

        # refunds
        (
            "refund_amount_raw",
            pa.string(),
        ),
        (
            "refund_amount",
            MONEY_TYPE,
        ),

        (
            "refund_budget_amount_raw",
            pa.string(),
        ),
        (
            "refund_budget_amount",
            MONEY_TYPE,
        ),

        (
            "refund_date_raw",
            pa.string(),
        ),
        (
            "refund_date",
            pa.date32(),
        ),

        (
            "refund_description",
            pa.string(),
        ),
        (
            "refund_purpose",
            pa.string(),
        ),
        (
            "refund_reason",
            pa.string(),
        ),

        # schema drift safety
        (
            "source_extra_json",
            pa.string(),
        ),
    ]
)


# ============================================================
# HELPERS
# ============================================================

def get_nested_list(
    obj: dict,
    path: tuple,
) -> list:

    current = obj

    for key in path:

        if not isinstance(
            current,
            dict,
        ):
            return []

        current = current.get(
            key
        )

    return (
        current
        if isinstance(
            current,
            list,
        )
        else []
    )


def _safe_json(
    value,
) -> str | None:

    if not value:
        return None

    return json.dumps(
        value,
        ensure_ascii=False,
        default=str,
        sort_keys=True,
    )


# ============================================================
# NORMALIZE ONE PAYMENT ROW
# ============================================================

def normalize_payment_row(
    row: dict,
    section: str,
    detail: dict,
    context: dict,
) -> dict:

    direction = (
        "incoming"
        if section in INCOMING_TYPES
        else "outgoing"
    )

    payer_type = (
        normalize_counterparty_type(
            row.get(
                "payer_type"
            )
        )
    )

    receiver_type = (
        normalize_counterparty_type(
            row.get(
                "receiver_type"
            )
        )
    )

    payer_account = (
        normalize_account_fields(
            row.get(
                "payer_account_iban"
            )
        )
    )

    receiver_account = (
        normalize_account_fields(
            row.get(
                "receiver_account_iban"
            )
        )
    )

    extras = {
        key: value
        for key, value
        in row.items()
        if key
        not in KNOWN_SOURCE_FIELDS
    }

    signed_date = (
        detail.get(
            "signed_date"
        )
    )

    source_is_signed = (
        clean_text(
            signed_date
        )
        is not None
    )

    return {
        "source_report_id":
            clean_text(
                context.get(
                    "source_report_id"
                )
            ),

        "official_selected_report_id":
            clean_text(
                context.get(
                    "official_selected_report_id"
                )
            ),

        "analysis_selected_report_id":
            clean_text(
                context.get(
                    "analysis_selected_report_id"
                )
            ),

        "organization_id":
            clean_text(
                context.get(
                    "organization_id"
                )
            ),

        "root_party_id":
            clean_text(
                context.get(
                    "root_party_id"
                )
            ),

        "report_year":
            int(
                context.get(
                    "year"
                )
            ),

        "report_quarter":
            int(
                context.get(
                    "quarter"
                )
            ),

        "report_period":
            clean_text(
                context.get(
                    "period_label"
                )
            ),

        "report_type":
            clean_text(
                context.get(
                    "report_type"
                )
                or
                detail.get(
                    "report_type"
                )
            ),

        "source_is_signed":
            source_is_signed,

        "source_signed_date":
            parse_timestamp(
                signed_date
            ),

        "analysis_override":
            bool(
                context.get(
                    "analysis_override",
                    False,
                )
            ),

        "analysis_selection_method":
            clean_text(
                context.get(
                    "analysis_selection_method"
                )
            ),

        "source_payment_type":
            section,

        "payment_direction":
            direction,

        "source_row_id":
            clean_text(
                row.get(
                    "id"
                )
            ),

        "source_party_id":
            clean_text(
                row.get(
                    "party_id"
                )
            ),

        "source_office_id":
            clean_text(
                row.get(
                    "office_id"
                )
            ),

        "source_report_id_in_row":
            clean_text(
                row.get(
                    "report_id"
                )
            ),

        "source_created_at":
            parse_timestamp(
                row.get(
                    "created_at"
                )
            ),

        "source_updated_at":
            parse_timestamp(
                row.get(
                    "updated_at"
                )
            ),

        "group_code":
            clean_text(
                row.get(
                    "group_code"
                )
            ),

        # payer
        "payer_name_source":
            clean_text(
                row.get(
                    "payer_name"
                )
            ),

        "payer_name_normalized":
            normalize_person_name(
                row.get(
                    "payer_name"
                ),
                payer_type,
            ),

        "payer_code_raw":
            clean_text(
                row.get(
                    "payer_code"
                )
            ),

        "payer_code_normalized":
            normalize_code(
                row.get(
                    "payer_code"
                )
            ),

        "payer_type_source":
            clean_text(
                row.get(
                    "payer_type"
                )
            ),

        "payer_type_normalized":
            payer_type,

        "payer_address":
            clean_text(
                row.get(
                    "payer_address"
                )
            ),

        "payer_birthday_raw":
            clean_text(
                row.get(
                    "payer_birthday"
                )
            ),

        "payer_birthday":
            parse_date(
                row.get(
                    "payer_birthday"
                )
            ),

        "payer_account_iban_raw":
            payer_account[
                "raw"
            ],

        "payer_account_iban_canonical":
            payer_account[
                "canonical"
            ],

        "payer_account_normalization_status":
            payer_account[
                "status"
            ],

        "payer_account_normalization_method":
            payer_account[
                "method"
            ],

        "payer_account_valid_iban":
            payer_account[
                "valid"
            ],

        "payer_account_candidate_count":
            payer_account[
                "candidate_count"
            ],

        "payer_account_candidates":
            payer_account[
                "candidates_json"
            ],

        "payer_account_normalized_text":
            payer_account[
                "normalized_text"
            ],

        "payer_account_type_source":
            clean_text(
                row.get(
                    "payer_account_type"
                )
            ),

        "payer_bank_code":
            clean_text(
                row.get(
                    "payer_bank_code"
                )
            ),

        "payer_bank_name":
            clean_text(
                row.get(
                    "payer_bank_name"
                )
            ),

        "payer_bank_address":
            clean_text(
                row.get(
                    "payer_bank_address"
                )
            ),

        # payment
        "payment_amount_raw":
            clean_text(
                row.get(
                    "payment_amount"
                )
            ),

        "payment_amount":
            parse_decimal(
                row.get(
                    "payment_amount"
                )
            ),

        "payment_code":
            clean_text(
                row.get(
                    "payment_code"
                )
            ),

        "payment_currency":
            clean_text(
                row.get(
                    "payment_currency"
                )
            ),

        "payment_description":
            clean_text(
                row.get(
                    "payment_description"
                )
            ),

        "payment_instruction_date_raw":
            clean_text(
                row.get(
                    "payment_instruction_date"
                )
            ),

        "payment_instruction_date":
            parse_date(
                row.get(
                    "payment_instruction_date"
                )
            ),

        "payment_number":
            clean_text(
                row.get(
                    "payment_number"
                )
            ),

        "payment_operation_date_raw":
            clean_text(
                row.get(
                    "payment_operation_date"
                )
            ),

        "payment_operation_date":
            parse_date(
                row.get(
                    "payment_operation_date"
                )
            ),

        "payment_purpose":
            clean_text(
                row.get(
                    "payment_purpose"
                )
            ),

        "payment_reason":
            clean_text(
                row.get(
                    "payment_reason"
                )
            ),

        "payment_type_detail_source":
            clean_text(
                row.get(
                    "payment_type"
                )
            ),

        # receiver
        "receiver_name_source":
            clean_text(
                row.get(
                    "receiver_name"
                )
            ),

        "receiver_name_normalized":
            normalize_person_name(
                row.get(
                    "receiver_name"
                ),
                receiver_type,
            ),

        "receiver_code_raw":
            clean_text(
                row.get(
                    "receiver_code"
                )
            ),

        "receiver_code_normalized":
            normalize_code(
                row.get(
                    "receiver_code"
                )
            ),

        "receiver_type_source":
            clean_text(
                row.get(
                    "receiver_type"
                )
            ),

        "receiver_type_normalized":
            receiver_type,

        "receiver_address":
            clean_text(
                row.get(
                    "receiver_address"
                )
            ),

        "receiver_birthday_raw":
            clean_text(
                row.get(
                    "receiver_birthday"
                )
            ),

        "receiver_birthday":
            parse_date(
                row.get(
                    "receiver_birthday"
                )
            ),

        "receiver_account_iban_raw":
            receiver_account[
                "raw"
            ],

        "receiver_account_iban_canonical":
            receiver_account[
                "canonical"
            ],

        "receiver_account_normalization_status":
            receiver_account[
                "status"
            ],

        "receiver_account_normalization_method":
            receiver_account[
                "method"
            ],

        "receiver_account_valid_iban":
            receiver_account[
                "valid"
            ],

        "receiver_account_candidate_count":
            receiver_account[
                "candidate_count"
            ],

        "receiver_account_candidates":
            receiver_account[
                "candidates_json"
            ],

        "receiver_account_normalized_text":
            receiver_account[
                "normalized_text"
            ],

        "receiver_account_type_source":
            clean_text(
                row.get(
                    "receiver_account_type"
                )
            ),

        "receiver_bank_code":
            clean_text(
                row.get(
                    "receiver_bank_code"
                )
            ),

        "receiver_bank_name":
            clean_text(
                row.get(
                    "receiver_bank_name"
                )
            ),

        "receiver_bank_address":
            clean_text(
                row.get(
                    "receiver_bank_address"
                )
            ),

        # refunds
        "refund_amount_raw":
            clean_text(
                row.get(
                    "refund_amount"
                )
            ),

        "refund_amount":
            parse_decimal(
                row.get(
                    "refund_amount"
                )
            ),

        "refund_budget_amount_raw":
            clean_text(
                row.get(
                    "refund_budget_amount"
                )
            ),

        "refund_budget_amount":
            parse_decimal(
                row.get(
                    "refund_budget_amount"
                )
            ),

        "refund_date_raw":
            clean_text(
                row.get(
                    "refund_date"
                )
            ),

        "refund_date":
            parse_date(
                row.get(
                    "refund_date"
                )
            ),

        "refund_description":
            clean_text(
                row.get(
                    "refund_description"
                )
            ),

        "refund_purpose":
            clean_text(
                row.get(
                    "refund_purpose"
                )
            ),

        "refund_reason":
            clean_text(
                row.get(
                    "refund_reason"
                )
            ),

        "source_extra_json":
            _safe_json(
                extras
            ),
    }


# ============================================================
# NORMALIZE ONE REPORT
# ============================================================

def normalize_report_payments(
    detail: dict,
    context: dict,
) -> dict[str, list[dict]]:

    expected_report_id = clean_text(
        context.get(
            "source_report_id"
        )
    )

    actual_report_id = clean_text(
        detail.get(
            "id"
        )
    )

    if (
        actual_report_id is not None
        and
        expected_report_id is not None
        and
        actual_report_id
        != expected_report_id
    ):
        raise ValueError(
            "Report ID mismatch: "
            f"expected={expected_report_id}, "
            f"actual={actual_report_id}"
        )


    source_year = detail.get(
        "year"
    )

    if source_year is not None:

        if int(source_year) != int(
            context.get(
                "year"
            )
        ):
            raise ValueError(
                "Report year mismatch."
            )


    source_quarter = detail.get(
        "quarter"
    )

    if source_quarter is not None:

        if int(source_quarter) != int(
            context.get(
                "quarter"
            )
        ):
            raise ValueError(
                "Report quarter mismatch."
            )


    result = {}


    for section, path in (
        PAYMENT_PATHS.items()
    ):

        source_rows = (
            get_nested_list(
                detail,
                path,
            )
        )

        normalized_rows = []


        for source_row in source_rows:

            if not isinstance(
                source_row,
                dict,
            ):
                continue

            normalized_rows.append(
                normalize_payment_row(
                    source_row,
                    section,
                    detail,
                    context,
                )
            )


        result[
            section
        ] = normalized_rows


    return result


# ============================================================
# RAW READER
# ============================================================

def read_report_detail(
    path: Path,
) -> dict:

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:

        payload = json.load(
            f
        )

    results = payload.get(
        "results"
    )

    if not isinstance(
        results,
        dict,
    ):
        raise ValueError(
            "Invalid report-detail payload."
        )

    return results


# ============================================================
# PARQUET FRAGMENTS
# ============================================================

def write_fragment(
    path: Path,
    rows: list[dict],
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    table = pa.Table.from_pylist(
        rows,
        schema=NORMALIZED_PAYMENT_SCHEMA,
    )

    tmp = path.with_suffix(
        ".tmp.parquet"
    )

    pq.write_table(
        table,
        tmp,
        compression="zstd",
    )

    os.replace(
        tmp,
        path,
    )


def write_empty_normalized_table(
    path: Path,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    table = pa.Table.from_pylist(
        [],
        schema=NORMALIZED_PAYMENT_SCHEMA,
    )

    pq.write_table(
        table,
        path,
        compression="zstd",
    )
'''


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)

print(
    "Written:",
    MODULE_PATH,
)


# ============================================================
# 2. WRITE TESTS
# ============================================================

TEST_PATH = (
    TEST_DIR
    / "test_payments.py"
)


TEST_CODE = r'''
from decimal import Decimal

from politdata.normalization.payments import (
    normalize_code,
    normalize_counterparty_type,
    normalize_person_name,
    normalize_account_fields,
    normalize_report_payments,
)


def test_payment_code_normalization():

    assert (
        normalize_code(
            "  1234 5678 "
        )
        == "12345678"
    )


def test_fop_type_normalization():

    assert (
        normalize_counterparty_type(
            "ФОП"
        )
        == "Фізична особа"
    )


def test_person_name_fop_prefix():

    assert (
        normalize_person_name(
            "ФОП  Іваненко  Іван  Іванович",
            "Фізична особа",
        )
        ==
        "Іваненко Іван Іванович"
    )


def test_person_name_latin_homoglyph():

    result = normalize_person_name(
        "IВАНOВ IВАН",
        "Фізична особа",
    )

    assert "I" not in result
    assert "O" not in result


def test_account_wrapper_exact_real_iban():

    result = normalize_account_fields(
        "UA263223130000026004000045109"
    )

    assert (
        result["canonical"]
        ==
        "UA263223130000026004000045109"
    )

    assert result["valid"] is True


def test_synthetic_report_payment():

    detail = {
        "id": "r1",
        "year": 2025,
        "quarter": 2,
        "report_type": "main",
        "signed_date": None,

        "payment_info": {
            "incoming": {
                "monetary_contributions": [
                    {
                        "id": "row1",
                        "payer_code": "12345678",
                        "payer_name": "ТОВ Тест",
                        "payer_type": "Юридична особа",
                        "payment_amount": 100.25,
                        "payment_operation_date": "2025-05-01",
                    }
                ],

                "other_contributions": [],
                "state_funding": [],
                "other_incomes": [],
            },

            "outgoing": {
                "budget_expenses": [],
                "outgoing_expenses": [],
                "return_expenses": [],
                "transfer_expenses": [],
            },
        },
    }

    context = {
        "source_report_id": "r1",
        "official_selected_report_id": "r2",
        "analysis_selected_report_id": "r1",

        "organization_id": "o1",
        "root_party_id": "p1",

        "year": 2025,
        "quarter": 2,
        "period_label": "2025 Q2",
        "report_type": "main",

        "analysis_override": True,
        "analysis_selection_method": "test_override",
    }

    result = normalize_report_payments(
        detail,
        context,
    )

    rows = result[
        "monetary_contributions"
    ]

    assert len(rows) == 1

    row = rows[0]

    assert row["source_is_signed"] is False
    assert row["analysis_override"] is True

    assert (
        row["payment_amount"]
        ==
        Decimal("100.25")
    )

    assert (
        row["payer_code_normalized"]
        ==
        "12345678"
    )
'''


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)

print(
    "Written:",
    TEST_PATH,
)


# ============================================================
# 3. RUN ALL TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 90)
print("TESTS")
print("=" * 90)

print(
    result.stdout
)

if result.stderr:
    print(
        result.stderr
    )

print(
    "Return code:",
    result.returncode,
)


if result.returncode != 0:
    raise RuntimeError(
        "Tests failed. "
        "Full payment normalization was NOT started."
    )


# ============================================================
# 4. RELOAD MODULE
# ============================================================

import politdata.normalization.payments as payments_module

importlib.reload(
    payments_module
)


PAYMENT_PATHS = (
    payments_module
    .PAYMENT_PATHS
)

NORMALIZATION_VERSION = (
    payments_module
    .NORMALIZATION_VERSION
)

NORMALIZED_PAYMENT_SCHEMA = (
    payments_module
    .NORMALIZED_PAYMENT_SCHEMA
)

read_report_detail = (
    payments_module
    .read_report_detail
)

normalize_report_payments = (
    payments_module
    .normalize_report_payments
)

write_fragment = (
    payments_module
    .write_fragment
)

write_empty_normalized_table = (
    payments_module
    .write_empty_normalized_table
)


# ============================================================
# 5. LOAD REPORT CONTEXT
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH
)


print()
print("=" * 90)
print("INPUT")
print("=" * 90)

print(
    "Analysis-selected reports:",
    len(
        report_context
    ),
)

print(
    "Unique source_report_id:",
    report_context[
        "source_report_id"
    ].nunique(),
)

print(
    "Analysis overrides:",
    int(
        report_context[
            "analysis_override"
        ].sum()
    ),
)


# ============================================================
# 6. PREFLIGHT
#
# Small automatic check, including analytical override.
# No production output is written here.
# ============================================================

preflight = (
    report_context
    .head(200)
    .copy()
)


override_rows = (
    report_context[
        report_context[
            "analysis_override"
        ]
    ]
)


preflight = (
    pd.concat(
        [
            preflight,
            override_rows,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        "source_report_id"
    )
)


preflight_counts = Counter()

preflight_unknown = Counter()


for context in (
    preflight
    .to_dict(
        "records"
    )
):

    report_id = str(
        context[
            "source_report_id"
        ]
    )

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    detail = read_report_detail(
        path
    )

    normalized = (
        normalize_report_payments(
            detail,
            context,
        )
    )

    for section, rows in (
        normalized.items()
    ):

        preflight_counts[
            section
        ] += len(rows)

        for row in rows:

            if row[
                "source_extra_json"
            ]:

                extras = json.loads(
                    row[
                        "source_extra_json"
                    ]
                )

                preflight_unknown.update(
                    extras.keys()
                )


print()
print("=" * 90)
print("PREFLIGHT")
print("=" * 90)

print(
    "Reports checked:",
    len(
        preflight
    ),
)

print(
    "Rows by section:"
)

for section in PAYMENT_PATHS:

    print(
        f"  {section:24s}",
        preflight_counts[
            section
        ],
    )


print(
    "Unknown source fields:",
    dict(
        preflight_unknown
    ),
)


# ============================================================
# 7. LOAD SOURCE CONTENT HASHES
#
# We use existing report-detail state.
# No RAW scan is needed to determine unchanged reports.
# ============================================================

DETAIL_STATE_CANDIDATES = [
    ROOT
    / "data"
    / "interim"
    / "state"
    / "report_detail_state.parquet",

    ROOT
    / "data"
    / "interim"
    / "state"
    / "report_detail_excluded_state.parquet",
]


detail_states = []


for path in DETAIL_STATE_CANDIDATES:

    if path.exists():

        tmp = pd.read_parquet(
            path
        )

        if "report_id" in tmp.columns:

            detail_states.append(
                tmp
            )


if detail_states:

    detail_state = pd.concat(
        detail_states,
        ignore_index=True,
    )

    detail_state[
        "report_id"
    ] = (
        detail_state[
            "report_id"
        ]
        .astype(str)
    )

else:

    detail_state = pd.DataFrame()


fingerprint_map = {}


if len(detail_state):

    for row in (
        detail_state
        .to_dict(
            "records"
        )
    ):

        report_id = str(
            row.get(
                "report_id"
            )
        )

        fingerprint = (
            row.get(
                "content_hash"
            )
            or
            row.get(
                "raw_payload_hash"
            )
        )

        if (
            fingerprint is not None
            and
            not pd.isna(
                fingerprint
            )
        ):
            fingerprint_map[
                report_id
            ] = str(
                fingerprint
            )


def source_fingerprint(
    report_id,
    path,
):

    known = fingerprint_map.get(
        report_id
    )

    if known:
        return known

    stat = path.stat()

    return (
        f"stat:"
        f"{stat.st_size}:"
        f"{stat.st_mtime_ns}"
    )


# ============================================================
# 8. LOAD NORMALIZATION STATE
# ============================================================

if STATE_PATH.exists():

    previous_state = (
        pd.read_parquet(
            STATE_PATH
        )
    )

    previous_state[
        "source_report_id"
    ] = (
        previous_state[
            "source_report_id"
        ]
        .astype(str)
    )

    state_records = {
        str(row["source_report_id"]):
            row
        for row in previous_state
        .to_dict(
            "records"
        )
    }

else:

    state_records = {}


current_report_ids = set(
    report_context[
        "source_report_id"
    ]
    .astype(str)
)


# Remove obsolete state rows.
state_records = {
    report_id: row
    for report_id, row
    in state_records.items()
    if report_id
    in current_report_ids
}


# ============================================================
# 9. CLEAN STALE FRAGMENTS
#
# Important if an analytical override changes later.
# ============================================================

for section in PAYMENT_PATHS:

    section_dir = (
        FRAGMENT_ROOT
        / section
    )

    section_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for fragment in (
        section_dir.glob(
            "*.parquet"
        )
    ):

        if (
            fragment.stem
            not in current_report_ids
        ):
            fragment.unlink()


# ============================================================
# 10. STATE CHECKPOINT HELPER
# ============================================================

def save_state():

    rows = list(
        state_records.values()
    )

    df = pd.DataFrame(
        rows
    )

    if len(df):

        df = df.sort_values(
            "source_report_id"
        )

    tmp = STATE_PATH.with_suffix(
        ".tmp.parquet"
    )

    df.to_parquet(
        tmp,
        index=False,
    )

    tmp.replace(
        STATE_PATH
    )


# ============================================================
# 11. FULL / RESUMABLE NORMALIZATION
# ============================================================

from tqdm.auto import tqdm
from datetime import datetime


processed = 0
skipped = 0
errors = 0

unknown_fields = Counter()


contexts = (
    report_context
    .to_dict(
        "records"
    )
)


for i, context in enumerate(
    tqdm(
        contexts,
        desc="Normalize payments",
    ),
    start=1,
):

    report_id = str(
        context[
            "source_report_id"
        ]
    )

    raw_path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    # --------------------------------------------------------
    # Missing RAW
    # --------------------------------------------------------

    if not raw_path.exists():

        errors += 1

        state_records[
            report_id
        ] = {
            "source_report_id":
                report_id,

            "status":
                "error",

            "normalization_version":
                NORMALIZATION_VERSION,

            "source_fingerprint":
                None,

            "normalized_at":
                datetime.now(),

            "last_error":
                f"Missing RAW: {raw_path}",
        }

        continue


    fingerprint = (
        source_fingerprint(
            report_id,
            raw_path,
        )
    )


    previous = state_records.get(
        report_id
    )


    unchanged = (
        previous is not None
        and
        previous.get(
            "status"
        )
        == "success"
        and
        previous.get(
            "normalization_version"
        )
        == NORMALIZATION_VERSION
        and
        str(
            previous.get(
                "source_fingerprint"
            )
        )
        == str(
            fingerprint
        )
    )


    if unchanged:

        skipped += 1
        continue


    try:

        detail = read_report_detail(
            raw_path
        )

        normalized = (
            normalize_report_payments(
                detail,
                context,
            )
        )


        section_counts = {}


        # ----------------------------------------------------
        # Write / remove report fragments
        # ----------------------------------------------------

        for section, rows in (
            normalized.items()
        ):

            fragment = (
                FRAGMENT_ROOT
                / section
                / f"{report_id}.parquet"
            )

            section_counts[
                section
            ] = len(rows)


            if rows:

                write_fragment(
                    fragment,
                    rows,
                )


                for row in rows:

                    extra_json = (
                        row[
                            "source_extra_json"
                        ]
                    )

                    if extra_json:

                        extras = json.loads(
                            extra_json
                        )

                        unknown_fields.update(
                            extras.keys()
                        )


            else:

                # A changed report may previously have had rows.
                if fragment.exists():
                    fragment.unlink()


        state_row = {
            "source_report_id":
                report_id,

            "status":
                "success",

            "normalization_version":
                NORMALIZATION_VERSION,

            "source_fingerprint":
                fingerprint,

            "normalized_at":
                datetime.now(),

            "last_error":
                None,
        }


        for section in PAYMENT_PATHS:

            state_row[
                f"{section}_count"
            ] = (
                section_counts.get(
                    section,
                    0,
                )
            )


        state_records[
            report_id
        ] = state_row


        processed += 1


    except Exception as exc:

        errors += 1

        state_records[
            report_id
        ] = {
            "source_report_id":
                report_id,

            "status":
                "error",

            "normalization_version":
                NORMALIZATION_VERSION,

            "source_fingerprint":
                fingerprint,

            "normalized_at":
                datetime.now(),

            "last_error":
                repr(exc),
        }


    # --------------------------------------------------------
    # Checkpoint
    # --------------------------------------------------------

    if i % 500 == 0:

        save_state()


# Final checkpoint
save_state()


print()
print("=" * 90)
print("NORMALIZATION RUN")
print("=" * 90)

print(
    "Processed:",
    processed,
)

print(
    "Skipped unchanged:",
    skipped,
)

print(
    "Errors:",
    errors,
)

print(
    "Unknown/new source fields:",
    dict(
        unknown_fields
    ),
)


# ============================================================
# 12. ERROR QA
# ============================================================

state_df = pd.read_parquet(
    STATE_PATH
)


print()
print(
    state_df[
        "status"
    ]
    .value_counts(
        dropna=False
    )
)


error_rows = (
    state_df[
        state_df[
            "status"
        ]
        != "success"
    ]
)


if len(error_rows):

    print()
    print(
        "ERROR SAMPLE:"
    )

    display(
        error_rows[
            [
                "source_report_id",
                "last_error",
            ]
        ]
        .head(
            50
        )
    )

    raise RuntimeError(
        f"Payment normalization has "
        f"{len(error_rows)} unsuccessful reports. "
        "Canonical tables were NOT compacted."
    )


# ============================================================
# 13. COMPACT FRAGMENTS INTO 8 NORMALIZED PARQUET FILES
# ============================================================

import duckdb
import pyarrow.parquet as pq


def sql_path(
    path,
):

    return (
        Path(path)
        .resolve()
        .as_posix()
        .replace(
            "'",
            "''",
        )
    )


con = duckdb.connect()


normalized_counts = {}


for section in PAYMENT_PATHS:

    fragment_dir = (
        FRAGMENT_ROOT
        / section
    )

    out_path = (
        NORMALIZED_DIR
        / f"{section}.parquet"
    )

    fragments = list(
        fragment_dir.glob(
            "*.parquet"
        )
    )


    if not fragments:

        write_empty_normalized_table(
            out_path
        )

        normalized_counts[
            section
        ] = 0

        continue


    glob_path = sql_path(
        fragment_dir
        / "*.parquet"
    )

    output_sql_path = sql_path(
        out_path
    )


    con.execute(
        f"""
        COPY (
            SELECT *
            FROM read_parquet(
                '{glob_path}'
            )
        )
        TO '{output_sql_path}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )


    count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet(
            '{output_sql_path}'
        )
        """
    ).fetchone()[0]


    normalized_counts[
        section
    ] = count


print()
print("=" * 90)
print("NORMALIZED PAYMENT TABLES")
print("=" * 90)


for section, count in (
    normalized_counts.items()
):

    print(
        f"{section:24s}",
        f"{count:,}",
    )


# ============================================================
# 14. BUILD ENRICHED PAYMENT TABLES
#
# No public_summary.
#
# Internal transfer rule:
#
# incoming:
# payer_code == code of organization
# under same root_party_id
#
# outgoing:
# receiver_code == code of organization
# under same root_party_id
#
# Nothing else is used:
# no name
# no IBAN
# no fuzzy matching
# ============================================================

report_context_sql = sql_path(
    REPORT_CONTEXT_PATH
)

org_reference_sql = sql_path(
    ORG_REFERENCE_PATH
)


enriched_counts = {}


for section in PAYMENT_PATHS:

    normalized_path = (
        NORMALIZED_DIR
        / f"{section}.parquet"
    )

    enriched_path = (
        ENRICHED_DIR
        / f"{section}.parquet"
    )


    normalized_sql = sql_path(
        normalized_path
    )

    enriched_sql = sql_path(
        enriched_path
    )


    con.execute(
        f"""
        COPY (

            WITH

            n AS (
                SELECT *
                FROM read_parquet(
                    '{normalized_sql}'
                )
            ),

            rc AS (
                SELECT
                    source_report_id,

                    organization_id,
                    root_party_id,

                    organization_level,

                    organization_code,
                    organization_name_current,

                    party_code,
                    party_name_current,

                    region,
                    region_source,
                    region_source_address_type,
                    region_resolution_method,
                    region_resolution_source,

                    official_selected_report_id,
                    analysis_selected_report_id,

                    analysis_selection_method,
                    analysis_override

                FROM read_parquet(
                    '{report_context_sql}'
                )
            ),

            org_codes AS (
                SELECT DISTINCT
                    root_party_id,
                    trim(
                        organization_code
                    ) AS organization_code_match

                FROM read_parquet(
                    '{org_reference_sql}'
                )

                WHERE
                    organization_code
                    IS NOT NULL
                    AND
                    trim(
                        organization_code
                    ) <> ''
            ),

            joined AS (
                SELECT

                    n.*,

                    rc.organization_level,

                    rc.organization_code,
                    rc.organization_name_current,

                    rc.party_code,
                    rc.party_name_current,

                    rc.region,
                    rc.region_source,
                    rc.region_source_address_type,
                    rc.region_resolution_method,
                    rc.region_resolution_source,

                    (
                        n.source_report_id
                        =
                        n.official_selected_report_id
                    ) AS official_selected,

                    TRUE AS analysis_selected,

                    (
                        payer_match
                        .organization_code_match
                        IS NOT NULL
                    ) AS payer_same_party_code_match,

                    (
                        receiver_match
                        .organization_code_match
                        IS NOT NULL
                    ) AS receiver_same_party_code_match

                FROM n

                INNER JOIN rc
                    ON
                    n.source_report_id
                    =
                    rc.source_report_id

                LEFT JOIN org_codes
                    AS payer_match

                    ON
                    n.root_party_id
                    =
                    payer_match.root_party_id

                    AND
                    n.payer_code_normalized
                    =
                    payer_match.organization_code_match

                LEFT JOIN org_codes
                    AS receiver_match

                    ON
                    n.root_party_id
                    =
                    receiver_match.root_party_id

                    AND
                    n.receiver_code_normalized
                    =
                    receiver_match.organization_code_match
            )


            SELECT

                joined.*,

                CASE

                    WHEN
                        payment_direction
                        =
                        'incoming'
                        AND
                        payer_same_party_code_match

                    THEN
                        'internal_party_transfer'

                    ELSE
                        payer_type_normalized

                END
                AS payer_type_analytical,


                CASE

                    WHEN
                        payment_direction
                        =
                        'outgoing'
                        AND
                        receiver_same_party_code_match

                    THEN
                        'internal_party_transfer'

                    ELSE
                        receiver_type_normalized

                END
                AS receiver_type_analytical,


                CASE

                    WHEN
                        payment_direction
                        =
                        'incoming'

                    THEN
                        payer_same_party_code_match

                    WHEN
                        payment_direction
                        =
                        'outgoing'

                    THEN
                        receiver_same_party_code_match

                    ELSE
                        FALSE

                END
                AS internal_transfer,


                CASE

                    WHEN
                        (
                            payment_direction
                            =
                            'incoming'
                            AND
                            payer_same_party_code_match
                        )

                        OR

                        (
                            payment_direction
                            =
                            'outgoing'
                            AND
                            receiver_same_party_code_match
                        )

                    THEN
                        'same_root_party_organization_code'

                    ELSE
                        NULL

                END
                AS internal_transfer_rule,


                source_payment_type
                    AS analytical_payment_type,

                FALSE
                    AS was_reclassified,

                CAST(
                    NULL
                    AS VARCHAR
                )
                    AS reclassification_rule,

                CAST(
                    NULL
                    AS VARCHAR
                )
                    AS funding_source_analytical


            FROM joined

        )
        TO '{enriched_sql}'
        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )


    count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet(
            '{enriched_sql}'
        )
        """
    ).fetchone()[0]


    enriched_counts[
        section
    ] = count


# ============================================================
# 15. ENRICHMENT QA
# ============================================================

print()
print("=" * 90)
print("ENRICHED PAYMENT TABLES")
print("=" * 90)


for section, count in (
    enriched_counts.items()
):

    print(
        f"{section:24s}",
        f"{count:,}",
    )


print()
print("=" * 90)
print("INTERNAL PARTY TRANSFERS")
print("=" * 90)


internal_summary = []


for section in PAYMENT_PATHS:

    path = sql_path(
        ENRICHED_DIR
        / f"{section}.parquet"
    )

    row = con.execute(
        f"""
        SELECT
            COUNT(*) AS rows,
            COUNT(*)
                FILTER (
                    WHERE internal_transfer
                )
                AS internal_rows,

            COALESCE(
                SUM(payment_amount)
                    FILTER (
                        WHERE internal_transfer
                    ),
                0
            )
                AS internal_payment_amount

        FROM read_parquet(
            '{path}'
        )
        """
    ).fetchone()


    internal_summary.append(
        {
            "section":
                section,

            "rows":
                row[0],

            "internal_rows":
                row[1],

            "internal_payment_amount":
                row[2],
        }
    )


internal_summary = pd.DataFrame(
    internal_summary
)


display(
    internal_summary
)


# ============================================================
# 16. IBAN QA
# ============================================================

print()
print("=" * 90)
print("IBAN NORMALIZATION QA")
print("=" * 90)


iban_qa = []


for section in PAYMENT_PATHS:

    path = sql_path(
        NORMALIZED_DIR
        / f"{section}.parquet"
    )


    payer = con.execute(
        f"""
        SELECT
            COUNT(*)
                FILTER (
                    WHERE
                    payer_account_iban_raw
                    IS NOT NULL
                )
                AS raw_accounts,

            COUNT(*)
                FILTER (
                    WHERE
                    payer_account_valid_iban
                )
                AS valid_accounts,

            COUNT(*)
                FILTER (
                    WHERE
                    payer_account_normalization_status
                    =
                    'ambiguous_multiple_valid_ibans'
                )
                AS ambiguous_accounts,

            COUNT(*)
                FILTER (
                    WHERE
                    payer_account_iban_raw
                    IS NOT NULL
                    AND
                    (
                        payer_account_valid_iban
                        IS NULL
                        OR
                        NOT payer_account_valid_iban
                    )
                )
                AS invalid_accounts

        FROM read_parquet(
            '{path}'
        )
        """
    ).fetchone()


    receiver = con.execute(
        f"""
        SELECT
            COUNT(*)
                FILTER (
                    WHERE
                    receiver_account_iban_raw
                    IS NOT NULL
                )
                AS raw_accounts,

            COUNT(*)
                FILTER (
                    WHERE
                    receiver_account_valid_iban
                )
                AS valid_accounts,

            COUNT(*)
                FILTER (
                    WHERE
                    receiver_account_normalization_status
                    =
                    'ambiguous_multiple_valid_ibans'
                )
                AS ambiguous_accounts,

            COUNT(*)
                FILTER (
                    WHERE
                    receiver_account_iban_raw
                    IS NOT NULL
                    AND
                    (
                        receiver_account_valid_iban
                        IS NULL
                        OR
                        NOT receiver_account_valid_iban
                    )
                )
                AS invalid_accounts

        FROM read_parquet(
            '{path}'
        )
        """
    ).fetchone()


    iban_qa.append(
        {
            "section":
                section,

            "payer_raw":
                payer[0],

            "payer_valid":
                payer[1],

            "payer_ambiguous":
                payer[2],

            "payer_invalid":
                payer[3],

            "receiver_raw":
                receiver[0],

            "receiver_valid":
                receiver[1],

            "receiver_ambiguous":
                receiver[2],

            "receiver_invalid":
                receiver[3],
        }
    )


iban_qa = pd.DataFrame(
    iban_qa
)


display(
    iban_qa
)


# ============================================================
# 17. OVERRIDE QA
#
# Confirm the analytically preferred unsigned
# Luhansk Q2 version actually entered payment tables.
# ============================================================

print()
print("=" * 90)
print("ANALYTICAL OVERRIDE PAYMENT QA")
print("=" * 90)


override_id = (
    report_context.loc[
        report_context[
            "analysis_override"
        ],
        "source_report_id",
    ]
    .astype(str)
    .iloc[0]
)


override_summary = []


for section in PAYMENT_PATHS:

    path = sql_path(
        ENRICHED_DIR
        / f"{section}.parquet"
    )

    row = con.execute(
        f"""
        SELECT
            COUNT(*) AS rows,
            COALESCE(
                SUM(payment_amount),
                0
            ) AS amount

        FROM read_parquet(
            '{path}'
        )

        WHERE
            source_report_id
            =
            ?
        """,
        [
            override_id
        ],
    ).fetchone()


    override_summary.append(
        {
            "section":
                section,

            "rows":
                row[0],

            "payment_amount":
                row[1],
        }
    )


override_summary = pd.DataFrame(
    override_summary
)


display(
    override_summary
)


print()
print(
    "Override source_report_id:",
    override_id,
)


# ============================================================
# 18. OUTPUT FILES
# ============================================================

print()
print("=" * 90)
print("DONE")
print("=" * 90)

print(
    "Normalization state:"
)

print(
    STATE_PATH
)

print()
print(
    "Normalized payment directory:"
)

print(
    NORMALIZED_DIR
)

print()
print(
    "Enriched payment directory:"
)

print(
    ENRICHED_DIR
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\payments.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_payments.py

TESTS
............................................                             [100%]
44 passed in 1.18s

Return code: 0

INPUT
Analysis-selected reports: 78791
Unique source_report_id: 78791
Analysis overrides: 1

PREFLIGHT
Reports checked: 201
Rows by section:
  monetary_contributions   3
  other_contributions      14
  state_funding            0
  other_incomes            6
  budget_expenses          0
  outgoing_expenses        273
  return_expenses          0
  transfer_expenses        0
Unknown source fields: {}


Normalize payments:   0%|          | 0/78791 [00:00<?, ?it/s]


NORMALIZATION RUN
Processed: 78791
Skipped unchanged: 0
Errors: 0
Unknown/new source fields: {}

status
success    78791
Name: count, dtype: int64


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...


IBAN NORMALIZATION QA


,section,payer_raw,payer_valid,payer_ambiguous,payer_invalid,receiver_raw,receiver_valid,receiver_ambiguous,receiver_invalid
0,monetary_contributions,0,0,0,0,27234,27097,0,137
1,other_contributions,0,0,0,0,0,0,0,0
2,state_funding,0,0,0,0,96,96,0,0
3,other_incomes,0,0,0,0,18999,18993,0,6
4,budget_expenses,0,0,0,0,29482,29482,0,0
5,outgoing_expenses,0,0,0,0,319901,319639,0,262
6,return_expenses,0,0,0,0,137,135,0,2
7,transfer_expenses,0,0,0,0,0,0,0,0



ANALYTICAL OVERRIDE PAYMENT QA


,section,rows,payment_amount
0,monetary_contributions,0,0E-10
1,other_contributions,0,0E-10
2,state_funding,0,0E-10
3,other_incomes,3,1050000.0000000000
4,budget_expenses,0,0E-10
5,outgoing_expenses,247,1077997.2100000000
6,return_expenses,0,0E-10
7,transfer_expenses,0,0E-10



Override source_report_id: 317a1c90-a5a3-43e5-8e62-e205cc84fb1e

DONE
Normalization state:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\state\payment_normalization_state.parquet

Normalized payment directory:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\normalized_v0_1\payments

Enriched payment directory:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\payments


In [108]:
from pathlib import Path
import subprocess
import sys
import importlib

import pandas as pd
import duckdb


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

REFERENCE_MODULE_PATH = (
    ROOT
    / "src"
    / "politdata"
    / "normalization"
    / "reference.py"
)

REFERENCE_TEST_PATH = (
    ROOT
    / "tests"
    / "test_reference.py"
)

ORGANIZATIONS_PATH = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "organizations.parquet"
)

ADDRESSES_PATH = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "organization_addresses.parquet"
)

ANALYSIS_MANIFEST_PATH = (
    ROOT
    / "data"
    / "interim"
    / "reports"
    / "analysis_selected_reports_manifest.parquet"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

ORG_REFERENCE_PATH = (
    REFERENCE_DIR
    / "organization_reference.parquet"
)

REPORT_CONTEXT_PATH = (
    REFERENCE_DIR
    / "report_context.parquet"
)

NORMALIZED_PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "payments"
)

ENRICHED_PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)


# ============================================================
# PAYMENT TABLES
# ============================================================

PAYMENT_TYPES = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


# ============================================================
# 1. UPDATE reference.py
#
# Add ONE analytical party name.
#
# We do NOT create:
#   party_name_full
#   party_name_short
#
# The existing party_name_current becomes the short,
# unified analytical name.
#
# Office names are untouched.
# ============================================================

text = REFERENCE_MODULE_PATH.read_text(
    encoding="utf-8"
)


PARTY_FUNCTION = r'''

# ============================================================
# PARTY NAME NORMALIZATION
# ============================================================

def normalize_party_name_short(value):
    """
    Short unified party name for analytical datasets.

    Rules are based on the earlier exploratory cleaning:

    - remove quote characters around names;
    - remove leading "ПОЛІТИЧНА ПАРТІЯ";
    - shorten "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ" to "ВО";
    - normalize whitespace;
    - preserve remaining source capitalization.

    This function applies ONLY to root-party names.
    Office names are not changed.
    """

    text = clean_text(value)

    if text is None:
        return None


    # --------------------------------------------------------
    # Quotes
    # --------------------------------------------------------

    text = re.sub(
        r'["«»“”„‟]',
        "",
        text,
    )


    # --------------------------------------------------------
    # Generic leading legal label
    # --------------------------------------------------------

    text = re.sub(
        r"^\s*політична\s+партія\b[\s:;,\-–—]*",
        "",
        text,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # Old analytical shortening:
    # ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ -> ВО
    #
    # Support several apostrophe characters.
    # --------------------------------------------------------

    text = re.sub(
        r"\bвсеукраїнське\s+об['’ʼ`]єднання\b",
        "ВО",
        text,
        flags=re.IGNORECASE,
    )


    # --------------------------------------------------------
    # Final whitespace normalization
    # --------------------------------------------------------

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()


    return text or None
'''


FUNCTION_MARKER = (
    "\n\n# ============================================================\n"
    "# ORGANIZATION REFERENCE\n"
    "# ============================================================\n"
)


if "def normalize_party_name_short(" not in text:

    if FUNCTION_MARKER not in text:

        raise RuntimeError(
            "Could not locate ORGANIZATION REFERENCE "
            "marker in reference.py"
        )

    text = text.replace(
        FUNCTION_MARKER,
        PARTY_FUNCTION
        + FUNCTION_MARKER,
        1,
    )

    print(
        "Added normalize_party_name_short()."
    )

else:

    print(
        "normalize_party_name_short() already exists."
    )


# ============================================================
# 2. APPLY SHORT NAME TO ROOT PARTY LOOKUP
# ============================================================

NORMALIZATION_BLOCK = r'''
    # --------------------------------------------------------
    # ANALYTICAL PARTY NAME
    #
    # party_name_current in analytical/reference data
    # is intentionally the SHORT unified form.
    #
    # Full official source name remains available in
    # normalized_v0_1/organizations.parquet.
    #
    # Office names are NOT modified.
    # --------------------------------------------------------

    parties[
        "party_name_current"
    ] = (
        parties[
            "party_name_current"
        ]
        .map(
            normalize_party_name_short
        )
        .astype("string")
    )


    if (
        parties[
            "party_name_current"
        ]
        .isna()
        .any()
    ):
        raise ValueError(
            "Party-name normalization produced "
            "missing analytical party names."
        )


'''


DUPLICATE_CHECK_ANCHOR = r'''    if (
        parties[
            "root_party_id"
        ]
        .duplicated()
        .any()
    ):
        raise ValueError(
            "Duplicate root party IDs."
        )
'''


if (
    "# ANALYTICAL PARTY NAME"
    not in text
):

    if DUPLICATE_CHECK_ANCHOR not in text:

        raise RuntimeError(
            "Could not locate root-party duplicate "
            "check in reference.py"
        )

    text = text.replace(
        DUPLICATE_CHECK_ANCHOR,
        NORMALIZATION_BLOCK
        + DUPLICATE_CHECK_ANCHOR,
        1,
    )

    print(
        "Added analytical party-name normalization "
        "to build_organization_reference()."
    )

else:

    print(
        "Analytical party-name normalization "
        "already present."
    )


REFERENCE_MODULE_PATH.write_text(
    text,
    encoding="utf-8",
)


print(
    "Updated:",
    REFERENCE_MODULE_PATH,
)


# ============================================================
# 3. UPDATE TEST IMPORT
# ============================================================

test_text = REFERENCE_TEST_PATH.read_text(
    encoding="utf-8"
)


if (
    "normalize_party_name_short,"
    not in test_text
):

    if (
        "    normalize_region,\n"
        not in test_text
    ):

        raise RuntimeError(
            "Could not locate normalize_region "
            "import in test_reference.py"
        )

    test_text = test_text.replace(
        "    normalize_region,\n",
        (
            "    normalize_region,\n"
            "    normalize_party_name_short,\n"
        ),
        1,
    )


# ============================================================
# 4. ADD REGRESSION TESTS
# ============================================================

PARTY_TESTS = r'''


def test_party_name_short_political_party():

    assert (
        normalize_party_name_short(
            'ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»'
        )
        ==
        'СЛУГА НАРОДУ'
    )


def test_party_name_short_quotes():

    assert (
        normalize_party_name_short(
            'Політична партія "ГОЛОС"'
        )
        ==
        'ГОЛОС'
    )


def test_party_name_short_vo():

    assert (
        normalize_party_name_short(
            "ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»"
        )
        ==
        "ВО БАТЬКІВЩИНА"
    )


def test_party_short_name_does_not_change_office_name():

    organizations = pd.DataFrame(
        [
            {
                "organization_id": "p1",
                "root_party_id": "p1",
                "parent_id": None,
                "entity_type": "party",
                "code": "11111111",
                "name": 'ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»',
                "is_active": True,
            },

            {
                "organization_id": "o1",
                "root_party_id": "p1",
                "parent_id": "p1",
                "entity_type": "office",
                "code": "22222222",
                "name": (
                    'ПОЛІТИЧНА ПАРТІЯ '
                    'КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ «ТЕСТ»'
                ),
                "is_active": True,
            },
        ]
    )


    addresses = pd.DataFrame(
        [
            {
                "organization_id": "p1",
                "address_type": "register",
                "region": "м. Київ",
            },

            {
                "organization_id": "o1",
                "address_type": "register",
                "region": "м. Київ",
            },
        ]
    )


    result = (
        build_organization_reference(
            organizations,
            addresses,
        )
        .set_index(
            "organization_id"
        )
    )


    # One short party name across central + office.
    assert (
        result.loc[
            "p1",
            "party_name_current",
        ]
        ==
        "ТЕСТ"
    )


    assert (
        result.loc[
            "o1",
            "party_name_current",
        ]
        ==
        "ТЕСТ"
    )


    # Office name remains untouched.
    assert (
        result.loc[
            "o1",
            "organization_name_current",
        ]
        ==
        (
            'ПОЛІТИЧНА ПАРТІЯ '
            'КИЇВСЬКА МІСЬКА ОРГАНІЗАЦІЯ «ТЕСТ»'
        )
    )
'''


if (
    "test_party_name_short_political_party"
    not in test_text
):

    test_text += PARTY_TESTS

    print(
        "Added 4 party-name regression tests."
    )

else:

    print(
        "Party-name regression tests already exist."
    )


REFERENCE_TEST_PATH.write_text(
    test_text,
    encoding="utf-8",
)


# ============================================================
# 5. RUN ALL TESTS
#
# Current baseline:
# 44 passed
#
# Expected after this:
# 48 passed
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 90)
print("TESTS")
print("=" * 90)

print(
    result.stdout
)

if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode,
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Reference/payment enrichment was NOT rebuilt."
    )


# ============================================================
# 6. RELOAD REFERENCE MODULE IN JUPYTER
# ============================================================

import politdata.normalization.reference as reference_module

importlib.reload(
    reference_module
)


# ============================================================
# 7. LOAD FOUNDATION TABLES
# ============================================================

organizations = pd.read_parquet(
    ORGANIZATIONS_PATH
)

addresses = pd.read_parquet(
    ADDRESSES_PATH
)

analysis_manifest = pd.read_parquet(
    ANALYSIS_MANIFEST_PATH
)


# ============================================================
# 8. REBUILD ORGANIZATION REFERENCE
# ============================================================

organization_reference = (
    reference_module
    .build_organization_reference(
        organizations,
        addresses,
    )
)


organization_reference.to_parquet(
    ORG_REFERENCE_PATH,
    index=False,
)


# ============================================================
# 9. REBUILD REPORT CONTEXT
# ============================================================

report_context = (
    reference_module
    .build_report_context(
        analysis_manifest,
        organization_reference,
    )
)


report_context.to_parquet(
    REPORT_CONTEXT_PATH,
    index=False,
)


# ============================================================
# 10. PARTY-NAME QA
# ============================================================

party_reference = (
    organization_reference[
        [
            "root_party_id",
            "party_code",
            "party_name_current",
        ]
    ]
    .drop_duplicates()
    .copy()
)


print()
print("=" * 90)
print("PARTY NAME QA")
print("=" * 90)


print(
    "Root parties:",
    party_reference[
        "root_party_id"
    ].nunique(),
)


print(
    "Distinct short party names:",
    party_reference[
        "party_name_current"
    ].nunique(),
)


# ------------------------------------------------------------
# Remaining generic legal prefix / quotes
# ------------------------------------------------------------

remaining_generic = (
    party_reference[
        party_reference[
            "party_name_current"
        ]
        .str.contains(
            r'^\s*політична\s+партія\b|["«»“”„‟]',
            case=False,
            regex=True,
            na=False,
        )
    ]
)


print(
    "Names still containing generic prefix/quotes:",
    len(
        remaining_generic
    ),
)


if len(
    remaining_generic
):

    display(
        remaining_generic
        .head(
            100
        )
    )


# ------------------------------------------------------------
# Same short name attached to multiple root-party IDs
#
# Not automatically an error:
# stable identity remains root_party_id.
# But useful QA.
# ------------------------------------------------------------

same_short_name = (
    party_reference
    .groupby(
        "party_name_current",
        dropna=False,
    )[
        "root_party_id"
    ]
    .nunique()
    .reset_index(
        name="root_party_count"
    )
    .query(
        "root_party_count > 1"
    )
    .sort_values(
        "root_party_count",
        ascending=False,
    )
)


print(
    "Short names shared by multiple root_party_id:",
    len(
        same_short_name
    ),
)


if len(
    same_short_name
):

    display(
        same_short_name
        .head(
            100
        )
    )


print()
print(
    "Party-name sample:"
)

display(
    party_reference
    .sort_values(
        "party_name_current"
    )
    .head(
        100
    )
)


# ============================================================
# 11. VERIFY OFFICE NAMES WERE NOT CHANGED
# ============================================================

original_names = (
    organizations[
        [
            "organization_id",
            "name",
            "entity_type",
        ]
    ]
    .rename(
        columns={
            "name":
                "organization_name_source_check",
        }
    )
)


office_name_check = (
    organization_reference[
        [
            "organization_id",
            "organization_name_current",
            "organization_level",
        ]
    ]
    .merge(
        original_names,
        on="organization_id",
        how="left",
        validate="one_to_one",
    )
)


office_name_changes = (
    office_name_check[
        (
            office_name_check[
                "organization_level"
            ]
            == "office"
        )
        &
        (
            office_name_check[
                "organization_name_current"
            ].astype("string")
            !=
            office_name_check[
                "organization_name_source_check"
            ].astype("string")
        )
    ]
)


print()
print(
    "Office names changed by party-name cleaning:",
    len(
        office_name_changes
    ),
)


if len(
    office_name_changes
):

    display(
        office_name_changes.head(
            50
        )
    )

    raise RuntimeError(
        "Office names changed unexpectedly."
    )


# ============================================================
# 12. REPORT CONTEXT QA
# ============================================================

print()
print("=" * 90)
print("REPORT CONTEXT QA")
print("=" * 90)


print(
    "Rows:",
    len(
        report_context
    ),
)


print(
    "Unique source reports:",
    report_context[
        "source_report_id"
    ].nunique(),
)


print(
    "Organizations represented:",
    report_context[
        "organization_id"
    ].nunique(),
)


print(
    "Analysis overrides:",
    int(
        report_context[
            "analysis_override"
        ].sum()
    ),
)


# ============================================================
# 13. REBUILD 8 ENRICHED PAYMENT TABLES
#
# IMPORTANT:
#
# Reads ONLY:
# - normalized payment parquet
# - report_context
# - organization_reference
#
# Does NOT read report RAW JSON.
#
# Central-level organization_name_current is replaced
# in the analytical payment output by the same short
# party name, so a central-party row does NOT carry
# both a long and short version of the party name.
#
# Office names remain unchanged.
# ============================================================

def sql_path(path):

    return (
        Path(path)
        .resolve()
        .as_posix()
        .replace(
            "'",
            "''",
        )
    )


con = duckdb.connect()


report_context_sql = sql_path(
    REPORT_CONTEXT_PATH
)

organization_reference_sql = sql_path(
    ORG_REFERENCE_PATH
)


enriched_counts = {}


for payment_type in PAYMENT_TYPES:

    normalized_path = (
        NORMALIZED_PAYMENT_DIR
        / f"{payment_type}.parquet"
    )

    enriched_path = (
        ENRICHED_PAYMENT_DIR
        / f"{payment_type}.parquet"
    )


    if not normalized_path.exists():

        raise FileNotFoundError(
            normalized_path
        )


    # Explicit replacement avoids ambiguity about
    # existing target files.
    if enriched_path.exists():

        enriched_path.unlink()


    normalized_sql = sql_path(
        normalized_path
    )

    enriched_sql = sql_path(
        enriched_path
    )


    con.execute(
        f"""
        COPY (

            WITH

            n AS (

                SELECT *

                FROM read_parquet(
                    '{normalized_sql}'
                )

            ),


            rc AS (

                SELECT

                    source_report_id,

                    organization_id,
                    root_party_id,

                    organization_level,

                    organization_code,

                    CASE

                        WHEN
                            organization_level
                            =
                            'central'

                        THEN
                            party_name_current

                        ELSE
                            organization_name_current

                    END
                        AS organization_name_current,

                    party_code,
                    party_name_current,

                    region,
                    region_source,
                    region_source_address_type,
                    region_resolution_method,
                    region_resolution_source,

                    official_selected_report_id,
                    analysis_selected_report_id,

                    analysis_selection_method,
                    analysis_override

                FROM read_parquet(
                    '{report_context_sql}'
                )

            ),


            org_codes AS (

                SELECT DISTINCT

                    root_party_id,

                    trim(
                        organization_code
                    )
                        AS organization_code_match

                FROM read_parquet(
                    '{organization_reference_sql}'
                )

                WHERE
                    organization_code
                    IS NOT NULL

                    AND

                    trim(
                        organization_code
                    )
                    <> ''

            ),


            joined AS (

                SELECT

                    n.*,

                    rc.organization_level,

                    rc.organization_code,
                    rc.organization_name_current,

                    rc.party_code,
                    rc.party_name_current,

                    rc.region,
                    rc.region_source,
                    rc.region_source_address_type,
                    rc.region_resolution_method,
                    rc.region_resolution_source,


                    (
                        n.source_report_id
                        =
                        n.official_selected_report_id
                    )
                        AS official_selected,


                    TRUE
                        AS analysis_selected,


                    (
                        payer_match
                        .organization_code_match
                        IS NOT NULL
                    )
                        AS payer_same_party_code_match,


                    (
                        receiver_match
                        .organization_code_match
                        IS NOT NULL
                    )
                        AS receiver_same_party_code_match


                FROM n


                INNER JOIN rc

                    ON
                        n.source_report_id
                        =
                        rc.source_report_id


                LEFT JOIN org_codes
                    AS payer_match

                    ON
                        n.root_party_id
                        =
                        payer_match.root_party_id

                    AND

                        n.payer_code_normalized
                        =
                        payer_match.organization_code_match


                LEFT JOIN org_codes
                    AS receiver_match

                    ON
                        n.root_party_id
                        =
                        receiver_match.root_party_id

                    AND

                        n.receiver_code_normalized
                        =
                        receiver_match.organization_code_match

            )


            SELECT

                joined.*,


                CASE

                    WHEN
                        payment_direction
                        =
                        'incoming'

                        AND

                        payer_same_party_code_match

                    THEN
                        'internal_party_transfer'

                    ELSE
                        payer_type_normalized

                END
                    AS payer_type_analytical,


                CASE

                    WHEN
                        payment_direction
                        =
                        'outgoing'

                        AND

                        receiver_same_party_code_match

                    THEN
                        'internal_party_transfer'

                    ELSE
                        receiver_type_normalized

                END
                    AS receiver_type_analytical,


                CASE

                    WHEN
                        payment_direction
                        =
                        'incoming'

                    THEN
                        payer_same_party_code_match

                    WHEN
                        payment_direction
                        =
                        'outgoing'

                    THEN
                        receiver_same_party_code_match

                    ELSE
                        FALSE

                END
                    AS internal_transfer,


                CASE

                    WHEN
                        (
                            payment_direction
                            =
                            'incoming'

                            AND

                            payer_same_party_code_match
                        )

                        OR

                        (
                            payment_direction
                            =
                            'outgoing'

                            AND

                            receiver_same_party_code_match
                        )

                    THEN
                        'same_root_party_organization_code'

                    ELSE
                        NULL

                END
                    AS internal_transfer_rule,


                source_payment_type
                    AS analytical_payment_type,


                FALSE
                    AS was_reclassified,


                CAST(
                    NULL
                    AS VARCHAR
                )
                    AS reclassification_rule,


                CAST(
                    NULL
                    AS VARCHAR
                )
                    AS funding_source_analytical


            FROM joined

        )

        TO '{enriched_sql}'

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )


    count = con.execute(
        f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{enriched_sql}'
        )
        """
    ).fetchone()[0]


    enriched_counts[
        payment_type
    ] = count


# ============================================================
# 14. ROW-COUNT REGRESSION
# ============================================================

print()
print("=" * 90)
print("PAYMENT REBUILD QA")
print("=" * 90)


total_normalized = 0
total_enriched = 0


for payment_type in PAYMENT_TYPES:

    normalized_path = sql_path(
        NORMALIZED_PAYMENT_DIR
        / f"{payment_type}.parquet"
    )

    enriched_path = sql_path(
        ENRICHED_PAYMENT_DIR
        / f"{payment_type}.parquet"
    )


    normalized_count = (
        con.execute(
            f"""
            SELECT COUNT(*)

            FROM read_parquet(
                '{normalized_path}'
            )
            """
        )
        .fetchone()[0]
    )


    enriched_count = (
        con.execute(
            f"""
            SELECT COUNT(*)

            FROM read_parquet(
                '{enriched_path}'
            )
            """
        )
        .fetchone()[0]
    )


    total_normalized += (
        normalized_count
    )

    total_enriched += (
        enriched_count
    )


    print(
        f"{payment_type:24s}",
        f"normalized={normalized_count:>8,}",
        f"enriched={enriched_count:>8,}",
        (
            "OK"
            if normalized_count
            == enriched_count
            else
            "MISMATCH"
        ),
    )


print()
print(
    "Total normalized:",
    f"{total_normalized:,}",
)

print(
    "Total enriched:",
    f"{total_enriched:,}",
)


if (
    total_normalized
    != total_enriched
):

    raise RuntimeError(
        "Normalized/enriched payment totals differ."
    )


# ============================================================
# 15. INTERNAL TRANSFER REGRESSION
#
# These numbers should remain unchanged because
# party display-name shortening must NOT affect
# same-root-party code matching.
# ============================================================

internal_summary = []


for payment_type in PAYMENT_TYPES:

    path = sql_path(
        ENRICHED_PAYMENT_DIR
        / f"{payment_type}.parquet"
    )


    row = con.execute(
        f"""
        SELECT

            COUNT(*)
                AS rows,

            COUNT(*)
                FILTER (
                    WHERE internal_transfer
                )
                AS internal_rows,

            COALESCE(
                SUM(payment_amount)
                    FILTER (
                        WHERE internal_transfer
                    ),
                0
            )
                AS internal_payment_amount

        FROM read_parquet(
            '{path}'
        )
        """
    ).fetchone()


    internal_summary.append(
        {
            "section":
                payment_type,

            "rows":
                row[0],

            "internal_rows":
                row[1],

            "internal_payment_amount":
                row[2],
        }
    )


print()
print("=" * 90)
print("INTERNAL TRANSFER REGRESSION")
print("=" * 90)


display(
    pd.DataFrame(
        internal_summary
    )
)


# ============================================================
# 16. PARTY-NAME QA IN FINAL ENRICHED PAYMENTS
# ============================================================

union_parts = []


for payment_type in PAYMENT_TYPES:

    path = sql_path(
        ENRICHED_PAYMENT_DIR
        / f"{payment_type}.parquet"
    )

    union_parts.append(
        f"""
        SELECT

            '{payment_type}'
                AS section,

            root_party_id,
            party_code,
            party_name_current,

            organization_level,
            organization_name_current

        FROM read_parquet(
            '{path}'
        )
        """
    )


all_payment_names_sql = (
    "\nUNION ALL\n"
    .join(
        union_parts
    )
)


remaining_long_party_names = (
    con.execute(
        f"""
        WITH x AS (

            {all_payment_names_sql}

        )

        SELECT DISTINCT

            root_party_id,
            party_code,
            party_name_current

        FROM x

        WHERE
            regexp_matches(
                party_name_current,
                '(?i)^\\s*політична\\s+партія\\b'
            )

            OR

            regexp_matches(
                party_name_current,
                '["«»“”„‟]'
            )

        ORDER BY
            party_name_current
        """
    )
    .df()
)


central_name_mismatches = (
    con.execute(
        f"""
        WITH x AS (

            {all_payment_names_sql}

        )

        SELECT DISTINCT

            root_party_id,
            party_code,
            party_name_current,
            organization_name_current

        FROM x

        WHERE
            organization_level
            =
            'central'

            AND

            organization_name_current
            IS DISTINCT FROM
            party_name_current
        """
    )
    .df()
)


print()
print("=" * 90)
print("FINAL PARTY-NAME QA")
print("=" * 90)


print(
    "Party names still carrying generic prefix/quotes:",
    len(
        remaining_long_party_names
    ),
)


if len(
    remaining_long_party_names
):

    display(
        remaining_long_party_names
        .head(
            100
        )
    )


print(
    "Central payment rows with long organization name "
    "different from short party name:",
    len(
        central_name_mismatches
    ),
)


if len(
    central_name_mismatches
):

    display(
        central_name_mismatches
        .head(
            100
        )
    )


# ============================================================
# 17. ANALYTICAL OVERRIDE REGRESSION
# ============================================================

override_id = (
    report_context.loc[
        report_context[
            "analysis_override"
        ],
        "source_report_id",
    ]
    .astype(str)
    .iloc[0]
)


override_summary = []


for payment_type in PAYMENT_TYPES:

    path = sql_path(
        ENRICHED_PAYMENT_DIR
        / f"{payment_type}.parquet"
    )


    row = con.execute(
        f"""
        SELECT

            COUNT(*),

            COALESCE(
                SUM(payment_amount),
                0
            )

        FROM read_parquet(
            '{path}'
        )

        WHERE
            source_report_id
            =
            ?
        """,
        [
            override_id
        ],
    ).fetchone()


    override_summary.append(
        {
            "section":
                payment_type,

            "rows":
                row[0],

            "payment_amount":
                row[1],
        }
    )


print()
print("=" * 90)
print("ANALYTICAL OVERRIDE REGRESSION")
print("=" * 90)


display(
    pd.DataFrame(
        override_summary
    )
)


print(
    "Override source_report_id:",
    override_id,
)


# ============================================================
# 18. DONE
# ============================================================

print()
print("=" * 90)
print("DONE")
print("=" * 90)

print(
    "Reference:",
    ORG_REFERENCE_PATH,
)

print(
    "Report context:",
    REPORT_CONTEXT_PATH,
)

print(
    "Enriched payments:",
    ENRICHED_PAYMENT_DIR,
)

print()
print(
    "No RAW report JSON files were read."
)

normalize_party_name_short() already exists.
Added analytical party-name normalization to build_organization_reference().
Updated: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\reference.py
Party-name regression tests already exist.

TESTS
................................................                         [100%]
48 passed in 1.88s

Return code: 0

PARTY NAME QA
Root parties: 321
Distinct short party names: 320
Names still containing generic prefix/quotes: 0
Short names shared by multiple root_party_id: 1


,party_name_current,root_party_count
258,САМ ЗА СЕБЕ,2



Party-name sample:


,root_party_id,party_code,party_name_current
4478,73fc284b-40f6-4d13-81f3-2270723973ed,26059509,25
438,e75abd72-368e-4963-a386-103f702ed7bc,37195974,ЄВРОПЕЙСЬКА КОАЛІЦІЯ
464,0c30154b-5ac2-45c7-bdb4-70ad98106581,40089079,ЄВРОПЕЙСЬКА ЛІБЕРАЛЬНА ПАРТІЯ
120,e35c458c-e664-4cd5-83d1-44738d70f9b3,34494649,ЄВРОПЕЙСЬКА ПАРТІЯ УКРАЇНИ
292,7d9a4591-6cea-451f-8545-b211f5e7b9bd,21715714,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ
13,137f1977-5df1-4060-9c20-70f4d372a400,20077521,ЄВРОСОЮЗ
52,109e0303-848e-4ff3-9d5c-af300db5c491,41208188,ЄДИНА АЛЬТЕРНАТИВА
757,2723bd1c-0eb6-4438-957c-8ebaa24b197c,21714672,ЄДНІСТЬ
8408,d92ae88f-a306-4add-aee3-c5b581c2691b,43724237,ІДЕЯ НАЦІЇ
5740,9414793b-9d9e-48ee-aa5d-18f15cea020f,39965035,ІНТЕРНЕТ ПАРТІЯ УКРАЇНИ



Office names changed by party-name cleaning: 0

REPORT CONTEXT QA
Rows: 78791
Unique source reports: 78791
Organizations represented: 8514
Analysis overrides: 1


...


PAYMENT REBUILD QA
monetary_contributions   normalized=  27,234 enriched=  27,234 OK
other_contributions      normalized=   6,168 enriched=   6,168 OK
state_funding            normalized=      96 enriched=      96 OK
other_incomes            normalized=  19,007 enriched=  19,007 OK
budget_expenses          normalized=  29,482 enriched=  29,482 OK
outgoing_expenses        normalized= 319,901 enriched= 319,901 OK
return_expenses          normalized=     137 enriched=     137 OK
transfer_expenses        normalized=       3 enriched=       3 OK

Total normalized: 402,028
Total enriched: 402,028

INTERNAL TRANSFER REGRESSION


,section,rows,internal_rows,internal_payment_amount
0,monetary_contributions,27234,161,6059779.9400000000
1,other_contributions,6168,6,10131.8700000000
2,state_funding,96,0,0E-10
3,other_incomes,19007,3344,785725353.0800000000
4,budget_expenses,29482,4257,1336116787.5800000000
5,outgoing_expenses,319901,10385,1126719839.8000000000
6,return_expenses,137,14,9269943.0000000000
7,transfer_expenses,3,0,0E-10



FINAL PARTY-NAME QA
Party names still carrying generic prefix/quotes: 0
Central payment rows with long organization name different from short party name: 0

ANALYTICAL OVERRIDE REGRESSION


,section,rows,payment_amount
0,monetary_contributions,0,0E-10
1,other_contributions,0,0E-10
2,state_funding,0,0E-10
3,other_incomes,3,1050000.0000000000
4,budget_expenses,0,0E-10
5,outgoing_expenses,247,1077997.2100000000
6,return_expenses,0,0E-10
7,transfer_expenses,0,0E-10


Override source_report_id: 317a1c90-a5a3-43e5-8e62-e205cc84fb1e

DONE
Reference: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\organization_reference.parquet
Report context: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\report_context.parquet
Enriched payments: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\payments

No RAW report JSON files were read.


In [109]:
from pathlib import Path
import duckdb
import pandas as pd


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PAY_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)


STATE_FUNDING = (
    PAY_DIR
    / "state_funding.parquet"
)

BUDGET_EXPENSES = (
    PAY_DIR
    / "budget_expenses.parquet"
)

OTHER_INCOMES = (
    PAY_DIR
    / "other_incomes.parquet"
)

OUTGOING_EXPENSES = (
    PAY_DIR
    / "outgoing_expenses.parquet"
)


def sql_path(path):
    return (
        Path(path)
        .resolve()
        .as_posix()
        .replace("'", "''")
    )


sf = sql_path(STATE_FUNDING)
be = sql_path(BUDGET_EXPENSES)
oi = sql_path(OTHER_INCOMES)
oe = sql_path(OUTGOING_EXPENSES)

con = duckdb.connect()


# ============================================================
# 1. STATE FUNDING — SOURCE SEMANTICS
# ============================================================

print("=" * 90)
print("STATE FUNDING — ACCOUNT / COUNTERPARTY SEMANTICS")
print("=" * 90)


state_semantics = con.execute(
    f"""
    SELECT

        COUNT(*) AS rows,

        COUNT(DISTINCT root_party_id)
            AS parties,

        COUNT(DISTINCT organization_id)
            AS organizations,

        COUNT(*)
            FILTER (
                WHERE organization_level = 'central'
            )
            AS central_rows,

        COUNT(*)
            FILTER (
                WHERE organization_level = 'office'
            )
            AS office_rows,

        COUNT(DISTINCT receiver_account_iban_canonical)
            AS receiver_ibans,

        COUNT(DISTINCT payer_account_iban_canonical)
            AS payer_ibans,

        MIN(payment_operation_date)
            AS first_date,

        MAX(payment_operation_date)
            AS last_date,

        SUM(payment_amount)
            AS total_amount

    FROM read_parquet('{sf}')
    """
).df()

display(state_semantics)


print()
print("receiver_account_type_source:")

display(
    con.execute(
        f"""
        SELECT
            receiver_account_type_source,
            COUNT(*) AS rows,
            SUM(payment_amount) AS amount
        FROM read_parquet('{sf}')
        GROUP BY 1
        ORDER BY rows DESC
        """
    ).df()
)


print()
print("payer_account_type_source:")

display(
    con.execute(
        f"""
        SELECT
            payer_account_type_source,
            COUNT(*) AS rows,
            SUM(payment_amount) AS amount
        FROM read_parquet('{sf}')
        GROUP BY 1
        ORDER BY rows DESC
        """
    ).df()
)


# ============================================================
# 2. STATE FUNDING — PARTY LEVEL
# ============================================================

print()
print("=" * 90)
print("PARTIES WITH OBSERVED STATE-FUNDING RECEIPTS")
print("=" * 90)


state_parties = con.execute(
    f"""
    SELECT

        root_party_id,
        party_code,
        party_name_current,

        COUNT(*) AS receipt_rows,

        SUM(payment_amount)
            AS total_state_funding,

        MIN(payment_operation_date)
            AS first_receipt_date,

        MAX(payment_operation_date)
            AS last_receipt_date,

        COUNT(DISTINCT receiver_account_iban_canonical)
            AS receiver_iban_count,

        string_agg(
            DISTINCT receiver_account_iban_canonical,
            ' | '
        )
            FILTER (
                WHERE receiver_account_iban_canonical IS NOT NULL
            )
            AS receiver_ibans,

        string_agg(
            DISTINCT receiver_account_type_source,
            ' | '
        )
            FILTER (
                WHERE receiver_account_type_source IS NOT NULL
            )
            AS receiver_account_types,

        COUNT(*)
            FILTER (
                WHERE organization_level = 'central'
            )
            AS central_receipt_rows,

        COUNT(*)
            FILTER (
                WHERE organization_level = 'office'
            )
            AS office_receipt_rows

    FROM read_parquet('{sf}')

    GROUP BY
        root_party_id,
        party_code,
        party_name_current

    ORDER BY
        total_state_funding DESC
    """
).df()


display(state_parties)


# ============================================================
# 3. STATE FUNDING — ACTUAL ROW SAMPLE
# ============================================================

print()
print("=" * 90)
print("STATE FUNDING — ROW SAMPLE")
print("=" * 90)


display(
    con.execute(
        f"""
        SELECT

            party_name_current,
            party_code,

            organization_level,
            organization_name_current,

            report_year,
            report_quarter,
            payment_operation_date,
            payment_amount,

            payer_name_source,
            payer_code_raw,
            payer_type_source,

            receiver_name_source,
            receiver_code_raw,
            receiver_type_source,

            payer_account_iban_raw,
            payer_account_type_source,

            receiver_account_iban_raw,
            receiver_account_iban_canonical,
            receiver_account_type_source,

            payment_purpose,
            payment_description

        FROM read_parquet('{sf}')

        ORDER BY
            payment_amount DESC NULLS LAST

        LIMIT 100
        """
    ).df()
)


# ============================================================
# 4. BUDGET EXPENSES — ACCOUNT SEMANTICS
# ============================================================

print()
print("=" * 90)
print("BUDGET EXPENSES — ACCOUNT TYPE PROFILE")
print("=" * 90)


display(
    con.execute(
        f"""
        SELECT

            receiver_account_type_source,
            COUNT(*) AS rows,
            COUNT(DISTINCT root_party_id) AS parties,
            SUM(payment_amount) AS amount

        FROM read_parquet('{be}')

        GROUP BY 1
        ORDER BY rows DESC
        """
    ).df()
)


print()
print("payer_account_type_source:")

display(
    con.execute(
        f"""
        SELECT

            payer_account_type_source,
            COUNT(*) AS rows,
            COUNT(DISTINCT root_party_id) AS parties,
            SUM(payment_amount) AS amount

        FROM read_parquet('{be}')

        GROUP BY 1
        ORDER BY rows DESC
        """
    ).df()
)


# ============================================================
# 5. PARTIES USING budget_expenses
#    VS PARTIES WITH state_funding
# ============================================================

print()
print("=" * 90)
print("BUDGET EXPENSES VS OBSERVED STATE FUNDING")
print("=" * 90)


budget_party_status = con.execute(
    f"""
    WITH

    sf_party AS (

        SELECT

            root_party_id,

            COUNT(*) AS state_receipt_rows,

            SUM(payment_amount)
                AS total_state_funding,

            MIN(payment_operation_date)
                AS first_state_receipt_date,

            MAX(payment_operation_date)
                AS last_state_receipt_date,

            COUNT(DISTINCT receiver_account_iban_canonical)
                AS state_funding_iban_count

        FROM read_parquet('{sf}')

        GROUP BY root_party_id

    ),

    budget_party AS (

        SELECT

            root_party_id,
            party_code,
            party_name_current,

            COUNT(*) AS budget_expense_rows,

            SUM(payment_amount)
                AS budget_expense_amount,

            MIN(payment_operation_date)
                AS first_budget_expense_date,

            MAX(payment_operation_date)
                AS last_budget_expense_date,

            COUNT(DISTINCT organization_id)
                AS organizations_using_budget_section,

            COUNT(DISTINCT receiver_account_iban_canonical)
                AS budget_receiver_iban_count

        FROM read_parquet('{be}')

        GROUP BY
            root_party_id,
            party_code,
            party_name_current

    )

    SELECT

        b.*,

        COALESCE(
            s.state_receipt_rows,
            0
        )
            AS state_receipt_rows,

        COALESCE(
            s.total_state_funding,
            0
        )
            AS total_state_funding,

        s.first_state_receipt_date,
        s.last_state_receipt_date,

        COALESCE(
            s.state_funding_iban_count,
            0
        )
            AS state_funding_iban_count,

        CASE

            WHEN s.root_party_id IS NOT NULL
            THEN 'observed_state_funding'

            ELSE 'no_observed_state_funding'

        END
            AS preliminary_state_funding_status

    FROM budget_party b

    LEFT JOIN sf_party s
        USING (root_party_id)

    ORDER BY

        CASE
            WHEN s.root_party_id IS NULL
            THEN 0
            ELSE 1
        END,

        b.budget_expense_amount DESC
    """
).df()


print(
    "Parties using budget_expenses:",
    len(budget_party_status)
)

print(
    "With observed state_funding:",
    (
        budget_party_status[
            "preliminary_state_funding_status"
        ]
        ==
        "observed_state_funding"
    ).sum()
)

print(
    "WITHOUT observed state_funding:",
    (
        budget_party_status[
            "preliminary_state_funding_status"
        ]
        ==
        "no_observed_state_funding"
    ).sum()
)


display(
    budget_party_status
)


# ============================================================
# 6. HIGH-PRIORITY POSSIBLE MISCLASSIFICATION
# ============================================================

print()
print("=" * 90)
print("BUDGET EXPENSES WITH NO OBSERVED STATE FUNDING")
print("=" * 90)


no_state_budget = (
    budget_party_status[
        budget_party_status[
            "preliminary_state_funding_status"
        ]
        ==
        "no_observed_state_funding"
    ]
    .copy()
)


display(
    no_state_budget[
        [
            "root_party_id",
            "party_code",
            "party_name_current",

            "budget_expense_rows",
            "budget_expense_amount",

            "first_budget_expense_date",
            "last_budget_expense_date",

            "organizations_using_budget_section",
            "budget_receiver_iban_count",
        ]
    ]
)


# ============================================================
# 7. ROW SAMPLES FROM THE "NO STATE FUNDING" GROUP
# ============================================================

print()
print("=" * 90)
print("SAMPLE BUDGET EXPENSE ROWS — NO OBSERVED STATE FUNDING")
print("=" * 90)


if len(no_state_budget):

    ids = (
        no_state_budget[
            "root_party_id"
        ]
        .astype(str)
        .tolist()
    )

    placeholders = ",".join(
        ["?"] * len(ids)
    )

    sample = con.execute(
        f"""
        SELECT

            root_party_id,
            party_code,
            party_name_current,

            organization_level,
            organization_name_current,

            report_year,
            report_quarter,

            payment_operation_date,
            payment_amount,

            payer_name_source,
            payer_code_raw,
            payer_type_source,

            receiver_name_source,
            receiver_code_raw,
            receiver_type_source,

            payer_account_iban_raw,
            payer_account_type_source,

            receiver_account_iban_raw,
            receiver_account_iban_canonical,
            receiver_account_type_source,

            payment_purpose,
            payment_description

        FROM read_parquet('{be}')

        WHERE root_party_id IN ({placeholders})

        ORDER BY
            payment_amount DESC NULLS LAST

        LIMIT 200
        """,
        ids,
    ).df()

    display(sample)


# ============================================================
# 8. COMPARISON WITH OTHER PAYMENT SECTIONS
#
# Helps establish whether account fields have
# consistent source roles across incoming/outgoing.
# ============================================================

print()
print("=" * 90)
print("ACCOUNT-FIELD CROSS-SECTION PROFILE")
print("=" * 90)


profiles = []


for section, path in [
    ("state_funding", sf),
    ("other_incomes", oi),
    ("budget_expenses", be),
    ("outgoing_expenses", oe),
]:

    row = con.execute(
        f"""
        SELECT

            COUNT(*) AS rows,

            COUNT(*)
                FILTER (
                    WHERE payer_account_iban_raw IS NOT NULL
                )
                AS payer_account_rows,

            COUNT(*)
                FILTER (
                    WHERE receiver_account_iban_raw IS NOT NULL
                )
                AS receiver_account_rows,

            COUNT(DISTINCT payer_account_type_source)
                AS payer_account_type_count,

            COUNT(DISTINCT receiver_account_type_source)
                AS receiver_account_type_count

        FROM read_parquet('{path}')
        """
    ).fetchone()


    profiles.append(
        {
            "section": section,
            "rows": row[0],
            "payer_account_rows": row[1],
            "receiver_account_rows": row[2],
            "payer_account_type_count": row[3],
            "receiver_account_type_count": row[4],
        }
    )


display(
    pd.DataFrame(
        profiles
    )
)


print()
print("=" * 90)
print("DONE")
print("=" * 90)

print(
    "No RAW JSON files were read."
)

STATE FUNDING — ACCOUNT / COUNTERPARTY SEMANTICS


,rows,parties,organizations,central_rows,office_rows,receiver_ibans,payer_ibans,first_date,last_date,total_amount
0,96,4,4,96,0,9,0,2021-02-17,2026-04-17,3.843168e+09



receiver_account_type_source:


,receiver_account_type_source,rows,amount
0,None,96,3.843168e+09



payer_account_type_source:


...


SAMPLE BUDGET EXPENSE ROWS — NO OBSERVED STATE FUNDING


,root_party_id,party_code,party_name_current,organization_level,organization_name_current,report_year,report_quarter,payment_operation_date,payment_amount,payer_name_source,payer_code_raw,payer_type_source,receiver_name_source,receiver_code_raw,receiver_type_source,payer_account_iban_raw,payer_account_type_source,receiver_account_iban_raw,receiver_account_iban_canonical,receiver_account_type_source,payment_purpose,payment_description
0,155c3deb-caa3-488b-8ecf-5e54d674a3f6,00013215,ВО СВОБОДА,office,Калуська міська партійна організація Всеукраїн...,2025,4,2025-12-31,21733.64,None,None,None,"АТ ""ОРІАНА""",05743160,Юридична особа,None,None,UA513052990000026005035508779,UA513052990000026005035508779,None,"Орендна плата, компенсацiя витрат за комунальн...",None
1,155c3deb-caa3-488b-8ecf-5e54d674a3f6,00013215,ВО СВОБОДА,office,Івано-Франківська обласна огранізація Всеукраї...,2025,4,2025-10-21,11000.72,None,None,None,"ПП ""ТЕНДЕР ОНЛАЙН""",42414636,Юридична особа,None,None,UA783802810000000260030450601,UA783802810000000260030450601,None,Гарантійний внесок для участі в аукціоні зг. р...,None
2,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,33547558,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФО...,office,ЖИТОМИРСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАР...,2026,1,2026-03-06,10000.00,None,None,None,"ЛЕВЧЕНКО ОЛЕКСАНДР ПАВЛОВИЧ, ФОП",[конфіденційна інформація],Фізична особа/ФОП,None,None,UA463052990000026003016807628,UA463052990000026003016807628,None,"Оренда приміщення (будинку, офісу, кімнати, кв...",None
3,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,33547558,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФО...,office,ЖИТОМИРСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАР...,2026,1,2026-01-02,10000.00,None,None,None,"ЛЕВЧЕНКО ОЛЕКСАНДР ПАВЛОВИЧ, ФОП",[конфіденційна інформація],Фізична особа/ФОП,None,None,UA463052990000026003016807628,UA463052990000026003016807628,None,"Оренда приміщення (будинку, офісу, кімнати, кв...",None
4,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,33547558,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФО...,office,ЖИТОМИРСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАР...,2026,1,2026-02-06,10000.00,None,None,None,"ЛЕВЧЕНКО ОЛЕКСАНДР ПАВЛОВИЧ, ФОП",[конфіденційна інформація],Фізична особа/ФОП,None,None,UA463052990000026003016807628,UA463052990000026003016807628,None,"Оренда приміщення (будинку, офісу, кімнати, кв...",None
5,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,33547558,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФО...,office,ЖИТОМИРСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАР...,2026,1,2026-03-06,9000.00,None,None,None,Пащенко Катерина В'ячеславiвна пiдпр.,[конфіденційна інформація],Фізична особа/ФОП,None,None,UA463052990000026003016807628,UA463052990000026003016807628,None,"Оренда приміщення (будинку, офісу, кімнати, кв...",None
6,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,33547558,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФО...,office,ЖИТОМИРСЬКА ОБЛАСНА ОРГАНІЗАЦІЯ ПОЛІТИЧНОЇ ПАР...,2026,1,2026-02-02,9000.00,None,None,None,Пащенко Катерина В'ячеславiвна ФОП,[конфіденційна інформація],Фізична особа/ФОП,None,None,UA463052990000026003016807628,UA463052990000026003016807628,None,"Оренда приміщення (будинку, офісу, кімнати, кв...",None
7,993c5a91-04d5-49c4-8a37-a8a8872c9c2d,33438054,КМКС ПАРТІЯ УГОРЦІВ УКРАЇНИ,central,КМКС ПАРТІЯ УГОРЦІВ УКРАЇНИ,2025,3,2025-08-11,6172.02,None,None,None,"ТОВ ""ХОСТПРО ЛАБ""",42591377,Юридична особа,None,None,UA813005280000026005455062742,UA813005280000026005455062742,None,"Реклама в мережі Інтернет (соціальні мережі, в...",None
8,9b569362-fb23-4ebb-8a98-d0a928342eeb,00047728,ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ,office,Запорізька обласна організація Партії Зелених ...,2025,4,2025-12-19,6160.00,None,None,None,Дробна Ольга Вікторівна,[конфіденційна інформація],Фізична особа/ФОП,None,None,UA523130090000026005001029312,UA523130090000026005001029312,None,сплата заробітної плати,None
9,9b569362-fb23-4ebb-8a98-d0a928342eeb,00047728,ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ,office,Запорізька обласна організація Партії Зелених ...,2025,4,2025-12-03,6160.00,None,None,None,Дробна Ол


ACCOUNT-FIELD CROSS-SECTION PROFILE


,section,rows,payer_account_rows,receiver_account_rows,payer_account_type_count,receiver_account_type_count
0,state_funding,96,0,96,0,0
1,other_incomes,19007,0,18999,0,0
2,budget_expenses,29482,0,29482,0,0
3,outgoing_expenses,319901,0,319901,0,0



DONE
No RAW JSON files were read.


In [110]:
from pathlib import Path

import duckdb
import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PAY_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

OUT_DIR = (
    ROOT
    / "data"
    / "interim"
    / "state_funding"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


ACCOUNT_EVIDENCE_PATH = (
    OUT_DIR
    / "budget_account_funding_evidence.parquet"
)

PARTY_EVIDENCE_PATH = (
    OUT_DIR
    / "party_state_funding_evidence_candidate.parquet"
)


PAYMENT_TYPES = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
]


def sql_path(path):

    return (
        Path(path)
        .resolve()
        .as_posix()
        .replace(
            "'",
            "''",
        )
    )


paths = {
    payment_type:
        sql_path(
            PAY_DIR
            / f"{payment_type}.parquet"
        )

    for payment_type
    in PAYMENT_TYPES
}


con = duckdb.connect()


# ============================================================
# 1. BUILD ACCOUNT-LEVEL EVIDENCE
#
# IMPORTANT SEMANTIC INTERPRETATION:
#
# receiver_account_iban_canonical in these payment sections
# behaves as the PARTY / PARTY-OFFICE ACCOUNT.
#
# We therefore compare the SAME IBAN within the SAME
# root_party_id across source payment sections.
# ============================================================

account_evidence = con.execute(
    f"""
    WITH

    budget_accounts AS (

        SELECT

            root_party_id,
            party_code,
            party_name_current,

            receiver_account_iban_canonical
                AS account_iban,

            COUNT(*) AS budget_expense_rows,

            SUM(payment_amount)
                AS budget_expense_amount,

            MIN(payment_operation_date)
                AS first_budget_expense_date,

            MAX(payment_operation_date)
                AS last_budget_expense_date,

            COUNT(
                DISTINCT organization_id
            )
                AS budget_organizations

        FROM read_parquet(
            '{paths["budget_expenses"]}'
        )

        WHERE
            receiver_account_iban_canonical
            IS NOT NULL

        GROUP BY

            root_party_id,
            party_code,
            party_name_current,
            receiver_account_iban_canonical

    ),


    state_accounts AS (

        SELECT

            root_party_id,

            receiver_account_iban_canonical
                AS account_iban,

            COUNT(*) AS state_receipt_rows,

            SUM(
                COALESCE(
                    payment_amount,
                    0
                )
            )
                AS state_receipt_amount,

            COUNT(*)
                FILTER (
                    WHERE
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0
                )
                AS positive_state_receipt_rows,

            SUM(
                CASE
                    WHEN
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0

                    THEN
                        payment_amount

                    ELSE
                        0
                END
            )
                AS positive_state_receipt_amount,

            MIN(payment_operation_date)
                FILTER (
                    WHERE
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0
                )
                AS first_positive_state_receipt_date,

            MAX(payment_operation_date)
                FILTER (
                    WHERE
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0
                )
                AS last_positive_state_receipt_date

        FROM read_parquet(
            '{paths["state_funding"]}'
        )

        WHERE
            receiver_account_iban_canonical
            IS NOT NULL

        GROUP BY
            root_party_id,
            receiver_account_iban_canonical

    ),


    monetary_accounts AS (

        SELECT

            root_party_id,

            receiver_account_iban_canonical
                AS account_iban,

            COUNT(*) AS monetary_contribution_rows,

            SUM(
                COALESCE(
                    payment_amount,
                    0
                )
            )
                AS monetary_contribution_amount,

            MIN(payment_operation_date)
                AS first_monetary_contribution_date,

            MAX(payment_operation_date)
                AS last_monetary_contribution_date

        FROM read_parquet(
            '{paths["monetary_contributions"]}'
        )

        WHERE
            receiver_account_iban_canonical
            IS NOT NULL

        GROUP BY
            root_party_id,
            receiver_account_iban_canonical

    ),


    other_contribution_accounts AS (

        SELECT

            root_party_id,

            receiver_account_iban_canonical
                AS account_iban,

            COUNT(*) AS other_contribution_rows,

            SUM(
                COALESCE(
                    payment_amount,
                    0
                )
            )
                AS other_contribution_amount

        FROM read_parquet(
            '{paths["other_contributions"]}'
        )

        WHERE
            receiver_account_iban_canonical
            IS NOT NULL

        GROUP BY
            root_party_id,
            receiver_account_iban_canonical

    ),


    other_income_accounts AS (

        SELECT

            root_party_id,

            receiver_account_iban_canonical
                AS account_iban,

            COUNT(*) AS other_income_rows,

            SUM(
                COALESCE(
                    payment_amount,
                    0
                )
            )
                AS other_income_amount,

            MIN(payment_operation_date)
                AS first_other_income_date,

            MAX(payment_operation_date)
                AS last_other_income_date

        FROM read_parquet(
            '{paths["other_incomes"]}'
        )

        WHERE
            receiver_account_iban_canonical
            IS NOT NULL

        GROUP BY
            root_party_id,
            receiver_account_iban_canonical

    ),


    ordinary_expense_accounts AS (

        SELECT

            root_party_id,

            receiver_account_iban_canonical
                AS account_iban,

            COUNT(*) AS outgoing_expense_rows,

            SUM(
                COALESCE(
                    payment_amount,
                    0
                )
            )
                AS outgoing_expense_amount

        FROM read_parquet(
            '{paths["outgoing_expenses"]}'
        )

        WHERE
            receiver_account_iban_canonical
            IS NOT NULL

        GROUP BY
            root_party_id,
            receiver_account_iban_canonical

    )


    SELECT

        b.root_party_id,
        b.party_code,
        b.party_name_current,

        b.account_iban,

        b.budget_expense_rows,
        b.budget_expense_amount,

        b.first_budget_expense_date,
        b.last_budget_expense_date,

        b.budget_organizations,


        -- ====================================================
        -- STATE FUNDING EVIDENCE
        -- ====================================================

        COALESCE(
            s.state_receipt_rows,
            0
        )
            AS state_receipt_rows_same_account,

        COALESCE(
            s.state_receipt_amount,
            0
        )
            AS state_receipt_amount_same_account,

        COALESCE(
            s.positive_state_receipt_rows,
            0
        )
            AS positive_state_receipt_rows_same_account,

        COALESCE(
            s.positive_state_receipt_amount,
            0
        )
            AS positive_state_receipt_amount_same_account,

        s.first_positive_state_receipt_date,

        s.last_positive_state_receipt_date,


        -- ====================================================
        -- PRIVATE / NON-STATE INFLOW EVIDENCE
        -- ====================================================

        COALESCE(
            m.monetary_contribution_rows,
            0
        )
            AS monetary_contribution_rows_same_account,

        COALESCE(
            m.monetary_contribution_amount,
            0
        )
            AS monetary_contribution_amount_same_account,

        m.first_monetary_contribution_date,

        m.last_monetary_contribution_date,


        COALESCE(
            oc.other_contribution_rows,
            0
        )
            AS other_contribution_rows_same_account,

        COALESCE(
            oc.other_contribution_amount,
            0
        )
            AS other_contribution_amount_same_account,


        COALESCE(
            oi.other_income_rows,
            0
        )
            AS other_income_rows_same_account,

        COALESCE(
            oi.other_income_amount,
            0
        )
            AS other_income_amount_same_account,

        oi.first_other_income_date,

        oi.last_other_income_date,


        -- ====================================================
        -- ORDINARY-EXPENSE USE OF SAME ACCOUNT
        -- ====================================================

        COALESCE(
            oe.outgoing_expense_rows,
            0
        )
            AS outgoing_expense_rows_same_account,

        COALESCE(
            oe.outgoing_expense_amount,
            0
        )
            AS outgoing_expense_amount_same_account,


        -- ====================================================
        -- ANALYTICAL FLAGS
        -- ====================================================

        (
            COALESCE(
                s.positive_state_receipt_rows,
                0
            )
            > 0
        )
            AS has_positive_state_receipt_same_account,


        (
            COALESCE(
                m.monetary_contribution_rows,
                0
            )
            > 0

            OR

            COALESCE(
                oc.other_contribution_rows,
                0
            )
            > 0

            OR

            COALESCE(
                oi.other_income_rows,
                0
            )
            > 0
        )
            AS has_non_state_inflow_same_account,


        (
            COALESCE(
                oe.outgoing_expense_rows,
                0
            )
            > 0
        )
            AS also_used_in_outgoing_expenses,


        CASE

            WHEN
                COALESCE(
                    s.positive_state_receipt_rows,
                    0
                )
                > 0

            THEN
                'state_funding_account_observed'


            WHEN
                (
                    COALESCE(
                        m.monetary_contribution_rows,
                        0
                    )
                    > 0

                    OR

                    COALESCE(
                        oc.other_contribution_rows,
                        0
                    )
                    > 0

                    OR

                    COALESCE(
                        oi.other_income_rows,
                        0
                    )
                    > 0
                )

                AND

                COALESCE(
                    s.positive_state_receipt_rows,
                    0
                )
                = 0

            THEN
                'non_state_inflow_account_evidence'


            WHEN
                COALESCE(
                    oe.outgoing_expense_rows,
                    0
                )
                > 0

                AND

                COALESCE(
                    s.positive_state_receipt_rows,
                    0
                )
                = 0

            THEN
                'ordinary_expense_account_evidence'


            ELSE
                'no_account_level_funding_evidence'

        END
            AS account_funding_evidence


    FROM budget_accounts b


    LEFT JOIN state_accounts s

        ON
            b.root_party_id
            =
            s.root_party_id

        AND

            b.account_iban
            =
            s.account_iban


    LEFT JOIN monetary_accounts m

        ON
            b.root_party_id
            =
            m.root_party_id

        AND

            b.account_iban
            =
            m.account_iban


    LEFT JOIN other_contribution_accounts oc

        ON
            b.root_party_id
            =
            oc.root_party_id

        AND

            b.account_iban
            =
            oc.account_iban


    LEFT JOIN other_income_accounts oi

        ON
            b.root_party_id
            =
            oi.root_party_id

        AND

            b.account_iban
            =
            oi.account_iban


    LEFT JOIN ordinary_expense_accounts oe

        ON
            b.root_party_id
            =
            oe.root_party_id

        AND

            b.account_iban
            =
            oe.account_iban


    ORDER BY

        b.party_name_current,
        b.account_iban
    """
).df()


account_evidence.to_parquet(
    ACCOUNT_EVIDENCE_PATH,
    index=False,
)


# ============================================================
# 2. PARTY-WIDE STATE-FUNDING EVIDENCE
#
# Important:
# absence of receipt != legal proof of no entitlement.
#
# So statuses are deliberately evidence-based.
# ============================================================

party_evidence = con.execute(
    f"""
    WITH

    budget_party AS (

        SELECT

            root_party_id,
            party_code,
            party_name_current,

            COUNT(*) AS budget_expense_rows,

            SUM(
                COALESCE(
                    payment_amount,
                    0
                )
            )
                AS budget_expense_amount,

            COUNT(
                DISTINCT receiver_account_iban_canonical
            )
                AS budget_account_count

        FROM read_parquet(
            '{paths["budget_expenses"]}'
        )

        GROUP BY
            root_party_id,
            party_code,
            party_name_current

    ),


    state_party AS (

        SELECT

            root_party_id,

            COUNT(*)
                FILTER (
                    WHERE
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0
                )
                AS positive_state_receipt_rows,

            SUM(
                CASE

                    WHEN
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0

                    THEN
                        payment_amount

                    ELSE
                        0

                END
            )
                AS positive_state_receipt_amount,

            MIN(payment_operation_date)
                FILTER (
                    WHERE
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0
                )
                AS first_positive_state_receipt_date,

            MAX(payment_operation_date)
                FILTER (
                    WHERE
                        COALESCE(
                            payment_amount,
                            0
                        )
                        > 0
                )
                AS last_positive_state_receipt_date

        FROM read_parquet(
            '{paths["state_funding"]}'
        )

        GROUP BY
            root_party_id

    ),


    account_evidence AS (

        SELECT *

        FROM read_parquet(
            '{sql_path(ACCOUNT_EVIDENCE_PATH)}'
        )

    ),


    account_party AS (

        SELECT

            root_party_id,

            COUNT(*)
                AS budget_accounts_examined,

            COUNT(*)
                FILTER (
                    WHERE
                        has_positive_state_receipt_same_account
                )
                AS state_funding_accounts_observed,

            COUNT(*)
                FILTER (
                    WHERE
                        has_non_state_inflow_same_account
                )
                AS non_state_inflow_accounts_observed,

            COUNT(*)
                FILTER (
                    WHERE
                        also_used_in_outgoing_expenses
                )
                AS ordinary_expense_accounts_observed,

            SUM(
                budget_expense_rows
            )
                FILTER (
                    WHERE
                        has_non_state_inflow_same_account
                )
                AS budget_rows_on_non_state_inflow_accounts,

            SUM(
                budget_expense_amount
            )
                FILTER (
                    WHERE
                        has_non_state_inflow_same_account
                )
                AS budget_amount_on_non_state_inflow_accounts

        FROM account_evidence

        GROUP BY
            root_party_id

    )


    SELECT

        b.root_party_id,
        b.party_code,
        b.party_name_current,

        b.budget_expense_rows,
        b.budget_expense_amount,
        b.budget_account_count,


        COALESCE(
            s.positive_state_receipt_rows,
            0
        )
            AS positive_state_receipt_rows,

        COALESCE(
            s.positive_state_receipt_amount,
            0
        )
            AS positive_state_receipt_amount,

        s.first_positive_state_receipt_date,
        s.last_positive_state_receipt_date,


        COALESCE(
            a.budget_accounts_examined,
            0
        )
            AS budget_accounts_examined,

        COALESCE(
            a.state_funding_accounts_observed,
            0
        )
            AS state_funding_accounts_observed,

        COALESCE(
            a.non_state_inflow_accounts_observed,
            0
        )
            AS non_state_inflow_accounts_observed,

        COALESCE(
            a.ordinary_expense_accounts_observed,
            0
        )
            AS ordinary_expense_accounts_observed,

        COALESCE(
            a.budget_rows_on_non_state_inflow_accounts,
            0
        )
            AS budget_rows_on_non_state_inflow_accounts,

        COALESCE(
            a.budget_amount_on_non_state_inflow_accounts,
            0
        )
            AS budget_amount_on_non_state_inflow_accounts,


        CASE

            WHEN
                COALESCE(
                    s.positive_state_receipt_rows,
                    0
                )
                > 0

            THEN
                'observed_state_funding'


            WHEN
                COALESCE(
                    s.positive_state_receipt_rows,
                    0
                )
                = 0

                AND

                COALESCE(
                    a.non_state_inflow_accounts_observed,
                    0
                )
                > 0

            THEN
                'no_observed_state_funding_with_non_state_account_evidence'


            WHEN
                COALESCE(
                    s.positive_state_receipt_rows,
                    0
                )
                = 0

                AND

                COALESCE(
                    a.ordinary_expense_accounts_observed,
                    0
                )
                > 0

            THEN
                'no_observed_state_funding_with_ordinary_account_evidence'


            ELSE
                'no_observed_state_funding_only'

        END
            AS state_funding_evidence_status


    FROM budget_party b


    LEFT JOIN state_party s
        USING (
            root_party_id
        )


    LEFT JOIN account_party a
        USING (
            root_party_id
        )


    ORDER BY

        CASE

            WHEN
                COALESCE(
                    s.positive_state_receipt_rows,
                    0
                )
                > 0

            THEN 1

            ELSE 0

        END,

        b.budget_expense_amount DESC
    """
).df()


party_evidence.to_parquet(
    PARTY_EVIDENCE_PATH,
    index=False,
)


# ============================================================
# 3. ACCOUNT-LEVEL QA
# ============================================================

print()
print("=" * 90)
print("BUDGET ACCOUNT FUNDING EVIDENCE")
print("=" * 90)


print(
    "Budget accounts:",
    len(
        account_evidence
    )
)


print()
print(
    account_evidence[
        "account_funding_evidence"
    ]
    .value_counts(
        dropna=False
    )
)


display(
    account_evidence[
        [
            "party_name_current",
            "party_code",
            "account_iban",

            "budget_expense_rows",
            "budget_expense_amount",

            "positive_state_receipt_rows_same_account",
            "positive_state_receipt_amount_same_account",

            "monetary_contribution_rows_same_account",
            "monetary_contribution_amount_same_account",

            "other_contribution_rows_same_account",
            "other_contribution_amount_same_account",

            "other_income_rows_same_account",
            "other_income_amount_same_account",

            "outgoing_expense_rows_same_account",
            "outgoing_expense_amount_same_account",

            "account_funding_evidence",
        ]
    ]
)


# ============================================================
# 4. PARTY-LEVEL QA
# ============================================================

print()
print("=" * 90)
print("PARTY STATE-FUNDING EVIDENCE")
print("=" * 90)


print(
    party_evidence[
        "state_funding_evidence_status"
    ]
    .value_counts(
        dropna=False
    )
)


display(
    party_evidence
)


# ============================================================
# 5. HIGH-CONFIDENCE RECLASSIFICATION CANDIDATES
#
# This is still ONLY a preview.
#
# Candidate if:
# - no positive state-funding receipt anywhere for party
# - AND the same budget-expense account is demonstrably
#   also used for non-state inflows.
# ============================================================

high_confidence = (
    party_evidence[
        party_evidence[
            "state_funding_evidence_status"
        ]
        ==
        (
            "no_observed_state_funding_"
            "with_non_state_account_evidence"
        )
    ]
    .copy()
)


print()
print("=" * 90)
print("HIGH-CONFIDENCE RECLASSIFICATION CANDIDATES")
print("=" * 90)


print(
    "Parties:",
    len(
        high_confidence
    )
)


print(
    "Budget-expense rows:",
    int(
        high_confidence[
            "budget_rows_on_non_state_inflow_accounts"
        ].sum()
    ),
)


print(
    "Budget-expense amount:",
    high_confidence[
        "budget_amount_on_non_state_inflow_accounts"
    ].sum()
)


display(
    high_confidence[
        [
            "root_party_id",
            "party_code",
            "party_name_current",

            "budget_expense_rows",
            "budget_expense_amount",

            "budget_account_count",

            "positive_state_receipt_rows",

            "non_state_inflow_accounts_observed",

            "budget_rows_on_non_state_inflow_accounts",
            "budget_amount_on_non_state_inflow_accounts",

            "state_funding_evidence_status",
        ]
    ]
)


# ============================================================
# 6. PARTIES WHERE ABSENCE-ONLY IS STILL THE EVIDENCE
#
# These should NOT be automatically reclassified yet.
# ============================================================

weak_evidence = (
    party_evidence[
        party_evidence[
            "state_funding_evidence_status"
        ]
        ==
        "no_observed_state_funding_only"
    ]
)


print()
print("=" * 90)
print("NO-STATE-RECEIPT ONLY — DO NOT AUTO-RECLASSIFY YET")
print("=" * 90)


print(
    "Parties:",
    len(
        weak_evidence
    )
)


display(
    weak_evidence
)


# ============================================================
# 7. IMPORTANT CONTROL:
#
# Do any accounts appear BOTH as:
# - positive state-funding account
# - non-state inflow account
#
# If yes, account type != exclusive source of funds.
# We must know this before final classification.
# ============================================================

mixed_accounts = (
    account_evidence[
        (
            account_evidence[
                "has_positive_state_receipt_same_account"
            ]
        )
        &
        (
            account_evidence[
                "has_non_state_inflow_same_account"
            ]
        )
    ]
)


print()
print("=" * 90)
print("MIXED STATE + NON-STATE INFLOW ACCOUNTS")
print("=" * 90)


print(
    "Mixed accounts:",
    len(
        mixed_accounts
    )
)


display(
    mixed_accounts[
        [
            "party_name_current",
            "party_code",
            "account_iban",

            "positive_state_receipt_amount_same_account",

            "monetary_contribution_amount_same_account",
            "other_contribution_amount_same_account",
            "other_income_amount_same_account",

            "budget_expense_rows",
            "budget_expense_amount",
        ]
    ]
)


# ============================================================
# 8. FILES
# ============================================================

print()
print("=" * 90)
print("FILES")
print("=" * 90)

print(
    ACCOUNT_EVIDENCE_PATH
)

print(
    PARTY_EVIDENCE_PATH
)

print()
print(
    "No RAW JSON files were read."
)

print(
    "No enriched payment files were modified."
)


BUDGET ACCOUNT FUNDING EVIDENCE
Budget accounts: 36

account_funding_evidence
non_state_inflow_account_evidence    28
state_funding_account_observed        7
ordinary_expense_account_evidence     1
Name: count, dtype: int64


,party_name_current,party_code,account_iban,budget_expense_rows,budget_expense_amount,positive_state_receipt_rows_same_account,positive_state_receipt_amount_same_account,monetary_contribution_rows_same_account,monetary_contribution_amount_same_account,other_contribution_rows_same_account,other_contribution_amount_same_account,other_income_rows_same_account,other_income_amount_same_account,outgoing_expense_rows_same_account,outgoing_expense_amount_same_account,account_funding_evidence
0,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,21715714,UA163805820000026009050316815,826,5.388045e+07,0,0.000000e+00,0,0.00,0,0.0,1,3290.03,0,0.00,non_state_inflow_account_evidence
1,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,21715714,UA393805820000026007030316815,3252,1.604308e+08,16,2.138817e+08,0,0.00,0,0.0,2,196952.04,0,0.00,state_funding_account_observed
2,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,21715714,UA553805820000026001070316815,179,3.433348e+07,2,3.433348e+07,0,0.00,0,0.0,0,0.00,0,0.00,state_funding_account_observed
3,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,21715714,UA763805820000026008040316815,122,1.328807e+08,0,0.000000e+00,0,0.00,0,0.0,46,1418394.34,0,0.00,non_state_inflow_account_evidence
4,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,21715714,UA923805820000026002080316815,7398,4.500527e+08,26,4.790841e+08,0,0.00,0,0.0,5,128471.04,0,0.00,state_funding_account_observed
5,ВО БАТЬКІВЩИНА,20069956,UA303808380000026008700622652,111,2.080065e+05,0,0.000000e+00,0,0.00,0,0.0,1,1284.76,0,0.00,non_state_inflow_account_evidence
6,ВО БАТЬКІВЩИНА,20069956,UA363516290000000002604826373,1405,8.854623e+07,5,8.638362e+07,0,0.00,0,0.0,17,2162604.76,0,0.00,state_funding_account_observed
7,ВО БАТЬКІВЩИНА,20069956,UA723808380000026048700741786,7463,3.794849e+08,17,3.914542e+08,0,0.00,0,0.0,24,1561543.59,0,0.00,state_funding_account_observed
8,ВО БАТЬКІВЩИНА,20069956,UA803052990000026000046213832,44,4.906651e+05,0,0.000000e+00,120,1397939.16,0,0.0,4,5237.89,361,1164391.11,non_state_inflow_account_evidence
9,ВО СВОБОДА,00013215,UA503052990000026005025506060,3,2.210400e+03,0,0.000000e+00,3,5340.80,0,0.0,0,0.00,7,3270.40,non_state_inflow_account_evidence



PARTY STATE-FUNDING EVIDENCE
state_funding_evidence_status
no_observed_state_funding_with_non_state_account_evidence    13
observed_state_funding                                        4
Name: count, dtype: int64


,root_party_id,party_code,party_name_current,budget_expense_rows,budget_expense_amount,budget_account_count,positive_state_receipt_rows,positive_state_receipt_amount,first_positive_state_receipt_date,last_positive_state_receipt_date,budget_accounts_examined,state_funding_accounts_observed,non_state_inflow_accounts_observed,ordinary_expense_accounts_observed,budget_rows_on_non_state_inflow_accounts,budget_amount_on_non_state_inflow_accounts,state_funding_evidence_status
0,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,33547558,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФО...,59,1.259139e+05,1,0,0.000000e+00,NaT,NaT,1,0,1,1,59.0,1.259139e+05,no_observed_state_funding_with_non_state_accou...
1,9b569362-fb23-4ebb-8a98-d0a928342eeb,00047728,ПАРТІЯ ЗЕЛЕНИХ УКРАЇНИ,24,5.906700e+04,1,0,0.000000e+00,NaT,NaT,1,0,1,1,24.0,5.906700e+04,no_observed_state_funding_with_non_state_accou...
2,a834c9eb-9b06-47a1-880c-15688f705883,36681860,РІДНЕ МІСТО,32,4.616434e+04,2,0,0.000000e+00,NaT,NaT,2,0,2,2,32.0,4.616434e+04,no_observed_state_funding_with_non_state_accou...
3,155c3deb-caa3-488b-8ecf-5e54d674a3f6,00013215,ВО СВОБОДА,14,4.245476e+04,3,0,0.000000e+00,NaT,NaT,3,0,3,3,14.0,4.245476e+04,no_observed_state_funding_with_non_state_accou...
4,5080e980-1875-4f56-aff3-7625fd2de3f0,33308363,КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА ПАРТІЯ УКРАЇНИ СИЛА...,10,2.500750e+04,1,0,0.000000e+00,NaT,NaT,1,0,1,1,10.0,2.500750e+04,no_observed_state_funding_with_non_state_accou...
5,f07e15f7-097b-481d-a00f-b021b8918585,36088708,ЗА МАЙБУТНЄ,33,1.153100e+04,2,0,0.000000e+00,NaT,NaT,2,0,1,2,30.0,1.027000e+04,no_observed_state_funding_with_non_state_accou...
6,82c1b72c-f67e-4eaf-ac09-b025ee67adcc,39550414,ПРОПОЗИЦІЯ,17,7.229000e+03,1,0,0.000000e+00,NaT,NaT,1,0,1,1,17.0,7.229000e+03,no_observed_state_funding_with_non_state_accou...
7,23a2ed39-d9ba-4503-861a-83c7efd63a05,39870881,ГРОМАДА РОЗВИТОК РУШІЙ,5,6.565470e+03,1,0,0.000000e+00,NaT,NaT,1,0,1,1,5.0,6.565470e+03,no_observed_state_funding_with_non_state_accou...
8,993c5a91-04d5-49c4-8a37-a8a8872c9c2d,33438054,КМКС ПАРТІЯ УГОРЦІВ УКРАЇНИ,1,6.172020e+03,1,0,0.000000e+00,NaT,NaT,1,0,1,1,1.0,6.172020e+03,no_observed_state_funding_with_non_state_accou...
9,f5334c01-faf3-44f4-ae43-6bfceffe8b5e,40212900,ЗА ОДЕЩИНУ,22,3.300000e+03,1,0,0.000000e+00,NaT,NaT,1,0,1,1,22.0,3.300000e+03,no_observed_state_funding_with_non_state_accou...



HIGH-CONFIDENCE RECLASSIFICATION CANDIDATES
Parties: 13
Budget-expense rows: 220
Budget-expense amount: 335401.03


...


NO-STATE-RECEIPT ONLY — DO NOT AUTO-RECLASSIFY YET
Parties: 0


,root_party_id,party_code,party_name_current,budget_expense_rows,budget_expense_amount,budget_account_count,positive_state_receipt_rows,positive_state_receipt_amount,first_positive_state_receipt_date,last_positive_state_receipt_date,budget_accounts_examined,state_funding_accounts_observed,non_state_inflow_accounts_observed,ordinary_expense_accounts_observed,budget_rows_on_non_state_inflow_accounts,budget_amount_on_non_state_inflow_accounts,state_funding_evidence_status



MIXED STATE + NON-STATE INFLOW ACCOUNTS
Mixed accounts: 6


,party_name_current,party_code,account_iban,positive_state_receipt_amount_same_account,monetary_contribution_amount_same_account,other_contribution_amount_same_account,other_income_amount_same_account,budget_expense_rows,budget_expense_amount
1,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,21715714,UA393805820000026007030316815,2.138817e+08,0.0,0.0,196952.04,3252,1.604308e+08
4,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,21715714,UA923805820000026002080316815,4.790841e+08,0.0,0.0,128471.04,7398,4.500527e+08
6,ВО БАТЬКІВЩИНА,20069956,UA363516290000000002604826373,8.638362e+07,0.0,0.0,2162604.76,1405,8.854623e+07
7,ВО БАТЬКІВЩИНА,20069956,UA723808380000026048700741786,3.914542e+08,0.0,0.0,1561543.59,7463,3.794849e+08
13,ГОЛОС,39651598,UA813808050000000026002651529,1.189793e+08,0.0,0.0,407690.58,1637,1.256447e+08
28,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,2.519052e+09,0.0,0.0,2748378.06,5834,2.521767e+09



FILES
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\state_funding\budget_account_funding_evidence.parquet
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\state_funding\party_state_funding_evidence_candidate.parquet

No RAW JSON files were read.
No enriched payment files were modified.


In [111]:
from pathlib import Path
from dataclasses import asdict, is_dataclass
from concurrent.futures import ThreadPoolExecutor
import json
import re
import subprocess
import sys
import textwrap
import unicodedata

import duckdb
import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

RAW_DIR = (
    ROOT
    / "data"
    / "raw"
    / "report_details"
)

REPORT_CONTEXT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "report_context.parquet"
)

DETAIL_STATE_PATH = (
    ROOT
    / "data"
    / "interim"
    / "state"
    / "report_detail_state.parquet"
)

NORMALIZED_PROPERTY_DIR = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "properties"
)

ENRICHED_PROPERTY_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "properties"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

NORMALIZED_PROPERTY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ENRICHED_PROPERTY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


PROPERTY_MONEYS_PATH = (
    NORMALIZED_PROPERTY_DIR
    / "property_moneys.parquet"
)

ENRICHED_PROPERTY_MONEYS_PATH = (
    ENRICHED_PROPERTY_DIR
    / "property_moneys.parquet"
)

REPORT_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "report_account_reference.parquet"
)

ORGANIZATION_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "organization_account_reference.parquet"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)


PAYMENT_TYPES = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


# ============================================================
# 1. WRITE PERMANENT MODULE
# ============================================================

MODULE_PATH = (
    ROOT
    / "src"
    / "politdata"
    / "normalization"
    / "property_moneys.py"
)


MODULE_CODE = r'''
from __future__ import annotations

from dataclasses import asdict, is_dataclass
import json
import re
import unicodedata

from politdata.normalization.accounts import (
    normalize_account_number,
)


NORMALIZATION_VERSION = "property_moneys_v0_1"


ZERO_WIDTH_RE = re.compile(
    r"[\u200b\u200c\u200d\u2060\ufeff]"
)


ACCOUNT_NUMBER_KEYS = (
    "account_number",
    "accountNumber",
    "iban",
    "IBAN",
    "account",
    "number",
    "bank_account",
    "bankAccount",
    "bank_account_number",
    "bankAccountNumber",
)


ACCOUNT_TYPE_KEYS = (
    "account_type",
    "accountType",
    "type",
    "account_kind",
    "accountKind",
    "kind",
    "money_type",
    "moneyType",
    "account_purpose",
    "accountPurpose",
    "purpose",
)


def clean_text(value):

    if value is None:
        return None

    text = unicodedata.normalize(
        "NFKC",
        str(value),
    )

    text = ZERO_WIDTH_RE.sub(
        "",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text or None


def _is_scalar(value):

    return (
        value is None
        or isinstance(
            value,
            (
                str,
                int,
                float,
                bool,
            ),
        )
    )


def extract_property_moneys(detail):
    """
    Extract account/money snapshot rows from a report detail.

    Expected primary location:
        detail["properties"]["moneys"]

    Conservative fallbacks are supported for schema drift.
    """

    if not isinstance(
        detail,
        dict,
    ):
        return []


    properties = detail.get(
        "properties"
    )

    if not isinstance(
        properties,
        dict,
    ):
        return []


    preferred_keys = (
        "moneys",
        "money",
        "property_moneys",
        "propertyMoneys",
        "monies",
    )


    for key in preferred_keys:

        value = properties.get(
            key
        )

        if isinstance(
            value,
            list,
        ):
            return value


    # Schema-drift fallback:
    # only search immediately under "properties".
    for key, value in properties.items():

        normalized_key = (
            str(key)
            .lower()
            .replace(
                "_",
                "",
            )
        )

        if (
            isinstance(
                value,
                list,
            )
            and
            (
                "money"
                in normalized_key
                or
                "monies"
                in normalized_key
            )
        ):
            return value


    return []


def get_account_number_source(row):

    if not isinstance(
        row,
        dict,
    ):
        return None


    for key in ACCOUNT_NUMBER_KEYS:

        value = row.get(
            key
        )

        value = clean_text(
            value
        )

        if value:
            return value


    # Conservative schema-drift fallback.
    for key, value in row.items():

        key_norm = (
            str(key)
            .lower()
            .replace(
                "_",
                "",
            )
        )

        if (
            "iban"
            in key_norm
            or
            "accountnumber"
            in key_norm
        ):

            value = clean_text(
                value
            )

            if value:
                return value


    return None


def get_account_type_source(row):

    if not isinstance(
        row,
        dict,
    ):
        return None


    for key in ACCOUNT_TYPE_KEYS:

        value = row.get(
            key
        )

        if isinstance(
            value,
            (
                dict,
                list,
            ),
        ):
            continue

        value = clean_text(
            value
        )

        if value:
            return value


    return None


def classify_account_type(
    account_type_source,
    source_row=None,
):
    """
    Conservative analytical account classification.

    state_funding_account:
        source explicitly describes state/budget financing.

    ordinary_account:
        source explicitly describes a current/ordinary
        party account without state-funding wording.

    unknown:
        insufficient evidence.

    IMPORTANT:
    This describes the declared account category,
    not exclusive provenance of every hryvnia on it.
    """

    parts = []


    if account_type_source is not None:

        parts.append(
            clean_text(
                account_type_source
            )
            or ""
        )


    if isinstance(
        source_row,
        dict,
    ):

        # Look for explicit boolean-like state/budget flags.
        for key, value in source_row.items():

            key_text = (
                str(key)
                .lower()
                .replace(
                    "_",
                    "",
                )
            )


            if (
                (
                    "state"
                    in key_text
                    or
                    "budget"
                    in key_text
                    or
                    "держ"
                    in key_text
                )
                and
                value is True
            ):
                return (
                    "state_funding_account"
                )


            if _is_scalar(
                value
            ):

                value_text = clean_text(
                    value
                )

                if value_text:

                    parts.append(
                        value_text
                    )


    text = (
        " | ".join(
            parts
        )
        .lower()
    )


    # Explicit state-financing language.
    if re.search(
        r"""
        держав\w*
        .*?
        (
            фінанс
            |
            бюджет
        )
        """,
        text,
        flags=re.IGNORECASE | re.VERBOSE,
    ):

        return (
            "state_funding_account"
        )


    if re.search(
        r"""
        (
            держбюдж
            |
            бюджетн\w*\s+кошт
            |
            державн\w*\s+кошт
        )
        """,
        text,
        flags=re.IGNORECASE | re.VERBOSE,
    ):

        return (
            "state_funding_account"
        )


    # Explicit ordinary/current account wording.
    if re.search(
        r"""
        (
            поточн\w*\s+рах
            |
            розрахунк\w*\s+рах
            |
            звичайн\w*\s+рах
        )
        """,
        text,
        flags=re.IGNORECASE | re.VERBOSE,
    ):

        return (
            "ordinary_account"
        )


    return "unknown"


def normalize_account_result(value):

    result = normalize_account_number(
        value
    )


    if is_dataclass(
        result
    ):

        data = asdict(
            result
        )

    elif isinstance(
        result,
        dict,
    ):

        data = dict(
            result
        )

    elif hasattr(
        result,
        "_asdict",
    ):

        data = dict(
            result._asdict()
        )

    elif hasattr(
        result,
        "__dict__",
    ):

        data = dict(
            vars(
                result
            )
        )

    else:

        data = {
            "value":
                result
        }


    canonical = None

    for key in (
        "canonical",
        "canonical_iban",
        "iban_canonical",
        "normalized",
        "normalized_account",
        "account_number_canonical",
    ):

        candidate = data.get(
            key
        )

        if candidate:

            canonical = str(
                candidate
            )

            break


    status = None

    for key in (
        "status",
        "normalization_status",
        "iban_status",
    ):

        candidate = data.get(
            key
        )

        if candidate is not None:

            status = str(
                candidate
            )

            break


    method = None

    for key in (
        "method",
        "normalization_method",
        "iban_method",
    ):

        candidate = data.get(
            key
        )

        if candidate is not None:

            method = str(
                candidate
            )

            break


    return {
        "account_iban_canonical":
            canonical,

        "account_iban_status":
            status,

        "account_iban_normalization_method":
            method,

        "account_normalization_result_json":
            json.dumps(
                data,
                ensure_ascii=False,
                default=str,
                sort_keys=True,
            ),
    }


def normalize_property_money_row(
    row,
    *,
    source_report_id,
    organization_id,
    root_party_id,
    report_year,
    report_quarter,
):

    if not isinstance(
        row,
        dict,
    ):

        raise TypeError(
            "property_moneys row must be a dict"
        )


    account_number_source = (
        get_account_number_source(
            row
        )
    )


    account_type_source = (
        get_account_type_source(
            row
        )
    )


    normalized_account = (
        normalize_account_result(
            account_number_source
        )
    )


    account_type_analytical = (
        classify_account_type(
            account_type_source,
            row,
        )
    )


    result = {
        "source_report_id":
            str(
                source_report_id
            ),

        "organization_id":
            str(
                organization_id
            ),

        "root_party_id":
            str(
                root_party_id
            ),

        "report_year":
            report_year,

        "report_quarter":
            report_quarter,

        "party_account_iban_source":
            account_number_source,

        "party_account_iban":
            normalized_account[
                "account_iban_canonical"
            ],

        "party_account_iban_status":
            normalized_account[
                "account_iban_status"
            ],

        "party_account_iban_normalization_method":
            normalized_account[
                "account_iban_normalization_method"
            ],

        "party_account_type_source":
            account_type_source,

        "party_account_type_analytical":
            account_type_analytical,

        "party_account_type_resolution_method":
            (
                "property_moneys_declared_type"
                if
                account_type_analytical
                !=
                "unknown"
                else
                "property_moneys_unresolved_type"
            ),

        "source_row_json":
            json.dumps(
                row,
                ensure_ascii=False,
                default=str,
                sort_keys=True,
            ),
    }


    # Preserve all top-level source fields.
    # Scalar values get real columns.
    # Nested values are serialized.
    for key, value in row.items():

        safe_key = re.sub(
            r"[^0-9A-Za-zА-Яа-яІіЇїЄєҐґ_]+",
            "_",
            str(key),
        ).strip(
            "_"
        )


        if not safe_key:

            continue


        column = (
            "source__"
            + safe_key
        )


        if _is_scalar(
            value
        ):

            result[
                column
            ] = value

        else:

            result[
                column
            ] = json.dumps(
                value,
                ensure_ascii=False,
                default=str,
                sort_keys=True,
            )


    return result
'''


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)

print(
    "Written:",
    MODULE_PATH
)


# ============================================================
# 2. WRITE TESTS
# ============================================================

TEST_PATH = (
    ROOT
    / "tests"
    / "test_property_moneys.py"
)


TEST_CODE = r'''
from politdata.normalization.property_moneys import (
    extract_property_moneys,
    get_account_number_source,
    get_account_type_source,
    classify_account_type,
    normalize_property_money_row,
)


def test_extract_property_moneys():

    detail = {
        "properties": {
            "moneys": [
                {
                    "account_number":
                        "UA263223130000026004000045109"
                }
            ]
        }
    }

    rows = extract_property_moneys(
        detail
    )

    assert len(rows) == 1


def test_get_account_number():

    row = {
        "account_number":
            "UA263223130000026004000045109"
    }

    assert (
        get_account_number_source(
            row
        )
        ==
        "UA263223130000026004000045109"
    )


def test_get_account_type_source():

    row = {
        "account_type":
            "Поточний рахунок"
    }

    assert (
        get_account_type_source(
            row
        )
        ==
        "Поточний рахунок"
    )


def test_state_funding_account():

    assert (
        classify_account_type(
            (
                "Рахунок для отримання коштів "
                "державного фінансування"
            )
        )
        ==
        "state_funding_account"
    )


def test_ordinary_account():

    assert (
        classify_account_type(
            "Поточний рахунок"
        )
        ==
        "ordinary_account"
    )


def test_normalized_property_money_row():

    row = {
        "account_number":
            "UA263223130000026004000045109",

        "account_type":
            "Поточний рахунок",
    }


    result = normalize_property_money_row(
        row,
        source_report_id="r1",
        organization_id="o1",
        root_party_id="p1",
        report_year=2025,
        report_quarter=1,
    )


    assert (
        result[
            "party_account_iban"
        ]
        ==
        "UA263223130000026004000045109"
    )

    assert (
        result[
            "party_account_type_analytical"
        ]
        ==
        "ordinary_account"
    )
'''


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)

print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 3. RUN TESTS
#
# Current baseline: 48
# Expected: 54
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 90)
print("TESTS")
print("=" * 90)

print(
    result.stdout
)

if result.stderr:
    print(
        result.stderr
    )

print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. Production extraction not started."
    )


# ============================================================
# 4. IMPORT MODULE
# ============================================================

from politdata.normalization.property_moneys import (
    extract_property_moneys,
    normalize_property_money_row,
)


# ============================================================
# 5. LOAD REPORT CONTEXT
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH
)


assert (
    report_context[
        "source_report_id"
    ].nunique()
    ==
    len(
        report_context
    )
)


context_by_report = (
    report_context
    .set_index(
        "source_report_id"
    )
    .to_dict(
        orient="index"
    )
)


selected_ids = (
    report_context[
        "source_report_id"
    ]
    .astype(str)
    .tolist()
)


print()
print("=" * 90)
print("INPUT")
print("=" * 90)

print(
    "Analysis-selected reports:",
    len(
        selected_ids
    )
)


# ============================================================
# 6. TRY TO USE DETAIL STATE TO AVOID READING EMPTY RAW
# ============================================================

candidate_ids = selected_ids

count_column = None


if DETAIL_STATE_PATH.exists():

    detail_state = pd.read_parquet(
        DETAIL_STATE_PATH
    )


    count_candidates = [
        "property_moneys_count",
        "property_money_count",
        "property_moneys_rows",
        "property_money_rows",
        "money_property_count",
        "properties_moneys_count",
    ]


    for candidate in count_candidates:

        if candidate in detail_state.columns:

            count_column = candidate

            break


    if count_column is None:

        # Flexible schema search.
        for column in detail_state.columns:

            normalized = (
                column
                .lower()
                .replace(
                    "_",
                    "",
                )
            )

            if (
                "money"
                in normalized
                and
                (
                    "count"
                    in normalized
                    or
                    "row"
                    in normalized
                )
            ):

                count_column = column

                break


    report_id_col = None

    for candidate in (
        "report_id",
        "source_report_id",
        "id",
    ):

        if candidate in detail_state.columns:

            report_id_col = candidate

            break


    if (
        count_column is not None
        and
        report_id_col is not None
    ):

        state_subset = (
            detail_state[
                [
                    report_id_col,
                    count_column,
                ]
            ]
            .copy()
        )


        state_subset[
            report_id_col
        ] = (
            state_subset[
                report_id_col
            ]
            .astype(str)
        )


        selected_set = set(
            selected_ids
        )


        candidate_ids = (
            state_subset.loc[
                (
                    state_subset[
                        report_id_col
                    ].isin(
                        selected_set
                    )
                )
                &
                (
                    pd.to_numeric(
                        state_subset[
                            count_column
                        ],
                        errors="coerce",
                    )
                    .fillna(0)
                    > 0
                ),
                report_id_col,
            ]
            .drop_duplicates()
            .tolist()
        )


print()
print("=" * 90)
print("RAW READ PLAN")
print("=" * 90)

print(
    "Count column:",
    count_column
)

print(
    "Reports requiring RAW read:",
    len(
        candidate_ids
    )
)

print(
    "Total selected reports:",
    len(
        selected_ids
    )
)


# ============================================================
# 7. RAW READER
#
# Only used because property_moneys has not yet been
# production-normalized.
#
# If state contains section counts, only nonempty reports
# are opened.
# ============================================================

try:
    import orjson

    def load_json(path):

        with open(
            path,
            "rb",
        ) as f:

            return orjson.loads(
                f.read()
            )

except ImportError:

    def load_json(path):

        with open(
            path,
            "r",
            encoding="utf-8",
        ) as f:

            return json.load(
                f
            )


def get_context_value(
    context,
    *names,
):

    for name in names:

        if name in context:

            value = context[
                name
            ]

            if pd.notna(
                value
            ):

                return value

    return None


def process_report(
    report_id,
):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    if not path.exists():

        return {
            "report_id":
                report_id,

            "status":
                "missing",

            "rows":
                [],
        }


    payload = load_json(
        path
    )


    detail = (
        payload.get(
            "results"
        )
        if isinstance(
            payload,
            dict,
        )
        else None
    )


    if not isinstance(
        detail,
        dict,
    ):

        return {
            "report_id":
                report_id,

            "status":
                "invalid_wrapper",

            "rows":
                [],
        }


    context = context_by_report[
        report_id
    ]


    organization_id = (
        get_context_value(
            context,
            "organization_id",
        )
    )


    root_party_id = (
        get_context_value(
            context,
            "root_party_id",
        )
    )


    report_year = (
        get_context_value(
            context,
            "year",
            "report_year",
        )
    )


    report_quarter = (
        get_context_value(
            context,
            "quarter",
            "report_quarter",
        )
    )


    source_rows = (
        extract_property_moneys(
            detail
        )
    )


    normalized_rows = []


    for source_row in source_rows:

        normalized_rows.append(
            normalize_property_money_row(
                source_row,
                source_report_id=
                    report_id,

                organization_id=
                    organization_id,

                root_party_id=
                    root_party_id,

                report_year=
                    report_year,

                report_quarter=
                    report_quarter,
            )
        )


    return {
        "report_id":
            report_id,

        "status":
            "success",

        "rows":
            normalized_rows,
    }


# ============================================================
# 8. EXTRACT PROPERTY_MONEYS
#
# This is the only phase that may open report RAW.
# ============================================================

all_rows = []

missing = []
invalid = []


with ThreadPoolExecutor(
    max_workers=16
) as executor:

    results = executor.map(
        process_report,
        candidate_ids,
    )


    for result in tqdm(
        results,
        total=len(
            candidate_ids
        ),
        desc="Property moneys",
    ):

        if (
            result[
                "status"
            ]
            ==
            "success"
        ):

            all_rows.extend(
                result[
                    "rows"
                ]
            )

        elif (
            result[
                "status"
            ]
            ==
            "missing"
        ):

            missing.append(
                result[
                    "report_id"
                ]
            )

        else:

            invalid.append(
                result[
                    "report_id"
                ]
            )


if missing:

    raise RuntimeError(
        f"Missing RAW detail files: {len(missing)}"
    )


if invalid:

    raise RuntimeError(
        f"Invalid RAW detail files: {len(invalid)}"
    )


property_moneys = pd.DataFrame(
    all_rows
)


print()
print("=" * 90)
print("PROPERTY_MONEYS EXTRACTION")
print("=" * 90)

print(
    "Rows:",
    len(
        property_moneys
    )
)

print(
    "Reports represented:",
    property_moneys[
        "source_report_id"
    ].nunique()
    if len(property_moneys)
    else 0
)

print(
    "Organizations represented:",
    property_moneys[
        "organization_id"
    ].nunique()
    if len(property_moneys)
    else 0
)


if len(
    property_moneys
) < 1000:

    raise RuntimeError(
        "Unexpectedly few property_moneys rows. "
        "Extractor/schema should be checked before writing output."
    )


# ============================================================
# 9. CORE ACCOUNT QA
# ============================================================

print()
print("=" * 90)
print("ACCOUNT NORMALIZATION QA")
print("=" * 90)


print(
    "Rows:",
    len(
        property_moneys
    )
)

print(
    "Canonical IBAN rows:",
    property_moneys[
        "party_account_iban"
    ].notna().sum()
)

print(
    "Distinct canonical IBAN:",
    property_moneys[
        "party_account_iban"
    ].nunique()
)


print()
print(
    "IBAN status:"
)

print(
    property_moneys[
        "party_account_iban_status"
    ]
    .value_counts(
        dropna=False
    )
)


# Historical regression was ~19,138 rows
# and 1,664 distinct canonical IBAN.
# Do not hard-fail on exact counts because report selection
# can evolve, but large deviations should be visible here.


# ============================================================
# 10. ACCOUNT TYPE SOURCE QA
# ============================================================

print()
print("=" * 90)
print("ACCOUNT TYPE SOURCE QA")
print("=" * 90)


print(
    property_moneys[
        "party_account_type_analytical"
    ]
    .value_counts(
        dropna=False
    )
)


type_source_qa = (
    property_moneys[
        [
            "party_account_type_source",
            "party_account_type_analytical",
        ]
    ]
    .value_counts(
        dropna=False
    )
    .reset_index(
        name="rows"
    )
    .sort_values(
        "rows",
        ascending=False,
    )
)


display(
    type_source_qa.head(
        100
    )
)


# ============================================================
# 11. WRITE NORMALIZED PROPERTY_MONEYS
# ============================================================

property_moneys.to_parquet(
    PROPERTY_MONEYS_PATH,
    index=False,
)


print(
    "Written:",
    PROPERTY_MONEYS_PATH
)


# ============================================================
# 12. ENRICH PROPERTY_MONEYS WITH REPORT CONTEXT
# ============================================================

context_columns = [
    "source_report_id",
    "organization_id",
    "root_party_id",
    "organization_level",
    "organization_code",
    "organization_name_current",
    "party_code",
    "party_name_current",
    "region",
    "analysis_override",
]


available_context_columns = [
    column
    for column
    in context_columns
    if column
    in report_context.columns
]


enriched_property_moneys = (
    property_moneys
    .merge(
        report_context[
            available_context_columns
        ],
        on=[
            "source_report_id",
            "organization_id",
            "root_party_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


enriched_property_moneys.to_parquet(
    ENRICHED_PROPERTY_MONEYS_PATH,
    index=False,
)


print(
    "Written:",
    ENRICHED_PROPERTY_MONEYS_PATH
)


# ============================================================
# 13. BUILD REPORT-LEVEL ACCOUNT REFERENCE
#
# This is the preferred reference for payments:
#
# same report
# + same organization
# + same IBAN
#
# so historical account-type changes are preserved.
# ============================================================

con = duckdb.connect()


def sql_path(path):

    return (
        Path(path)
        .resolve()
        .as_posix()
        .replace(
            "'",
            "''",
        )
    )


pm_sql = sql_path(
    ENRICHED_PROPERTY_MONEYS_PATH
)

report_ref_sql = sql_path(
    REPORT_ACCOUNT_REFERENCE_PATH
)

org_ref_sql = sql_path(
    ORGANIZATION_ACCOUNT_REFERENCE_PATH
)


con.execute(
    f"""
    COPY (

        SELECT

            source_report_id,
            organization_id,
            root_party_id,

            party_account_iban,

            first(
                party_account_iban_source
            )
                AS party_account_iban_source,

            first(
                party_account_type_source
            )
                AS party_account_type_source,

            CASE

                WHEN
                    COUNT(
                        DISTINCT
                        party_account_type_analytical
                    )
                    FILTER (
                        WHERE
                            party_account_type_analytical
                            <>
                            'unknown'
                    )
                    > 1

                THEN
                    'conflicting_declared_types'


                WHEN
                    MAX(
                        CASE
                            WHEN
                                party_account_type_analytical
                                =
                                'state_funding_account'

                            THEN 1
                            ELSE 0
                        END
                    )
                    = 1

                THEN
                    'state_funding_account'


                WHEN
                    MAX(
                        CASE
                            WHEN
                                party_account_type_analytical
                                =
                                'ordinary_account'

                            THEN 1
                            ELSE 0
                        END
                    )
                    = 1

                THEN
                    'ordinary_account'


                ELSE
                    'unknown'

            END
                AS party_account_type_analytical,


            CASE

                WHEN
                    COUNT(
                        DISTINCT
                        party_account_type_analytical
                    )
                    FILTER (
                        WHERE
                            party_account_type_analytical
                            <>
                            'unknown'
                    )
                    > 1

                THEN
                    'property_moneys_same_report_conflict'

                ELSE
                    'property_moneys_exact_report_org_iban'

            END
                AS party_account_type_resolution_method,


            COUNT(*)
                AS snapshot_rows

        FROM read_parquet(
            '{pm_sql}'
        )

        WHERE
            party_account_iban
            IS NOT NULL

        GROUP BY

            source_report_id,
            organization_id,
            root_party_id,
            party_account_iban

    )

    TO '{report_ref_sql}'

    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)


# ============================================================
# 14. BUILD ORGANIZATION-LEVEL ACCOUNT HISTORY
#
# Useful for audits and fallback/reference,
# but payment enrichment prefers exact report-level match.
# ============================================================

con.execute(
    f"""
    COPY (

        SELECT

            organization_id,
            root_party_id,
            party_account_iban,

            MIN(report_year)
                AS first_observed_year,

            MAX(report_year)
                AS last_observed_year,

            COUNT(
                DISTINCT source_report_id
            )
                AS reports_observed,

            COUNT(
                DISTINCT
                party_account_type_source
            )
                FILTER (
                    WHERE
                        party_account_type_source
                        IS NOT NULL
                )
                AS declared_type_source_count,

            COUNT(
                DISTINCT
                party_account_type_analytical
            )
                FILTER (
                    WHERE
                        party_account_type_analytical
                        <>
                        'unknown'
                )
                AS analytical_type_count,

            MAX(
                CASE
                    WHEN
                        party_account_type_analytical
                        =
                        'state_funding_account'

                    THEN 1
                    ELSE 0
                END
            )
                =
                1
                AS ever_declared_state_funding_account,

            MAX(
                CASE
                    WHEN
                        party_account_type_analytical
                        =
                        'ordinary_account'

                    THEN 1
                    ELSE 0
                END
            )
                =
                1
                AS ever_declared_ordinary_account,

            CASE

                WHEN
                    COUNT(
                        DISTINCT
                        party_account_type_analytical
                    )
                    FILTER (
                        WHERE
                            party_account_type_analytical
                            <>
                            'unknown'
                    )
                    > 1

                THEN
                    'historical_type_conflict'


                WHEN
                    MAX(
                        CASE
                            WHEN
                                party_account_type_analytical
                                =
                                'state_funding_account'

                            THEN 1
                            ELSE 0
                        END
                    )
                    = 1

                THEN
                    'state_funding_account'


                WHEN
                    MAX(
                        CASE
                            WHEN
                                party_account_type_analytical
                                =
                                'ordinary_account'

                            THEN 1
                            ELSE 0
                        END
                    )
                    = 1

                THEN
                    'ordinary_account'


                ELSE
                    'unknown'

            END
                AS historical_account_type_analytical

        FROM read_parquet(
            '{pm_sql}'
        )

        WHERE
            party_account_iban
            IS NOT NULL

        GROUP BY

            organization_id,
            root_party_id,
            party_account_iban

    )

    TO '{org_ref_sql}'

    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
    """
)


print(
    "Written:",
    REPORT_ACCOUNT_REFERENCE_PATH
)

print(
    "Written:",
    ORGANIZATION_ACCOUNT_REFERENCE_PATH
)


# ============================================================
# 15. ACCOUNT REFERENCE QA
# ============================================================

report_account_qa = con.execute(
    f"""
    SELECT

        COUNT(*) AS rows,

        COUNT(DISTINCT party_account_iban)
            AS ibans,

        COUNT(*)
            FILTER (
                WHERE
                    party_account_type_analytical
                    =
                    'state_funding_account'
            )
            AS state_funding_rows,

        COUNT(*)
            FILTER (
                WHERE
                    party_account_type_analytical
                    =
                    'ordinary_account'
            )
            AS ordinary_rows,

        COUNT(*)
            FILTER (
                WHERE
                    party_account_type_analytical
                    =
                    'unknown'
            )
            AS unknown_rows,

        COUNT(*)
            FILTER (
                WHERE
                    party_account_type_analytical
                    =
                    'conflicting_declared_types'
            )
            AS conflict_rows

    FROM read_parquet(
        '{report_ref_sql}'
    )
    """
).df()


print()
print("=" * 90)
print("REPORT ACCOUNT REFERENCE QA")
print("=" * 90)

display(
    report_account_qa
)


# ============================================================
# 16. ADD PARTY ACCOUNT FIELDS TO ALL ENRICHED PAYMENT FILES
#
# Source API columns are preserved.
#
# New analytical columns:
#
#   party_account_iban
#   party_account_type_source
#   party_account_type_analytical
#   party_account_type_resolution_method
#
# Preferred exact match:
#   source_report_id + organization_id + IBAN
# ============================================================

payment_account_qa = []


for payment_type in PAYMENT_TYPES:

    payment_path = (
        PAYMENT_DIR
        / f"{payment_type}.parquet"
    )

    if not payment_path.exists():

        raise FileNotFoundError(
            payment_path
        )


    payment_sql = sql_path(
        payment_path
    )


    tmp_path = (
        PAYMENT_DIR
        / f"{payment_type}.account_enrichment.tmp.parquet"
    )

    tmp_sql = sql_path(
        tmp_path
    )


    # Avoid duplicate columns if cell is rerun.
    payment_columns = [
        row[0]
        for row
        in con.execute(
            f"""
            DESCRIBE
            SELECT *
            FROM read_parquet(
                '{payment_sql}'
            )
            """
        ).fetchall()
    ]


    columns_to_replace = [
        column
        for column
        in (
            "party_account_iban",
            "party_account_type_source",
            "party_account_type_analytical",
            "party_account_type_resolution_method",
        )
        if column
        in payment_columns
    ]


    if columns_to_replace:

        exclude_sql = (
            "EXCLUDE ("
            + ", ".join(
                columns_to_replace
            )
            + ")"
        )

    else:

        exclude_sql = ""


    con.execute(
        f"""
        COPY (

            SELECT

                p.*
                {exclude_sql},

                p.receiver_account_iban_canonical
                    AS party_account_iban,


                r.party_account_type_source
                    AS party_account_type_source,


                COALESCE(
                    r.party_account_type_analytical,
                    'unknown'
                )
                    AS party_account_type_analytical,


                CASE

                    WHEN
                        r.party_account_iban
                        IS NOT NULL

                    THEN
                        r.party_account_type_resolution_method

                    WHEN
                        p.receiver_account_iban_canonical
                        IS NULL

                    THEN
                        'no_party_account_in_payment'

                    ELSE
                        'no_same_report_property_moneys_match'

                END
                    AS party_account_type_resolution_method


            FROM read_parquet(
                '{payment_sql}'
            ) p


            LEFT JOIN read_parquet(
                '{report_ref_sql}'
            ) r

                ON
                    p.source_report_id
                    =
                    r.source_report_id

                AND

                    p.organization_id
                    =
                    r.organization_id

                AND

                    p.receiver_account_iban_canonical
                    =
                    r.party_account_iban

        )

        TO '{tmp_sql}'

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )


    # Validate row count before replacement.
    old_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet(
            '{payment_sql}'
        )
        """
    ).fetchone()[0]


    new_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet(
            '{tmp_sql}'
        )
        """
    ).fetchone()[0]


    if old_count != new_count:

        raise RuntimeError(
            f"{payment_type}: row count changed "
            f"{old_count} -> {new_count}"
        )


    tmp_path.replace(
        payment_path
    )


    final_sql = sql_path(
        payment_path
    )


    qa = con.execute(
        f"""
        SELECT

            COUNT(*) AS rows,

            COUNT(*)
                FILTER (
                    WHERE
                        party_account_iban
                        IS NOT NULL
                )
                AS rows_with_account,

            COUNT(*)
                FILTER (
                    WHERE
                        party_account_type_analytical
                        =
                        'state_funding_account'
                )
                AS state_funding_account_rows,

            COUNT(*)
                FILTER (
                    WHERE
                        party_account_type_analytical
                        =
                        'ordinary_account'
                )
                AS ordinary_account_rows,

            COUNT(*)
                FILTER (
                    WHERE
                        party_account_iban
                        IS NOT NULL

                        AND

                        party_account_type_analytical
                        =
                        'unknown'
                )
                AS unknown_account_type_rows,

            COUNT(*)
                FILTER (
                    WHERE
                        party_account_type_resolution_method
                        =
                        'no_same_report_property_moneys_match'
                )
                AS no_same_report_match

        FROM read_parquet(
            '{final_sql}'
        )
        """
    ).fetchone()


    payment_account_qa.append(
        {
            "section":
                payment_type,

            "rows":
                qa[0],

            "rows_with_account":
                qa[1],

            "state_funding_account_rows":
                qa[2],

            "ordinary_account_rows":
                qa[3],

            "unknown_account_type_rows":
                qa[4],

            "no_same_report_match":
                qa[5],
        }
    )


print()
print("=" * 90)
print("PAYMENT ACCOUNT ENRICHMENT QA")
print("=" * 90)

display(
    pd.DataFrame(
        payment_account_qa
    )
)


# ============================================================
# 17. STATE_FUNDING CROSS-CHECK
#
# Direct state-funding receipts should mostly/ideally
# map to declared state-funding accounts.
#
# We do NOT force classification here.
# ============================================================

state_funding_sql = sql_path(
    PAYMENT_DIR
    / "state_funding.parquet"
)


state_crosscheck = con.execute(
    f"""
    SELECT

        party_name_current,

        party_account_iban,

        party_account_type_source,
        party_account_type_analytical,
        party_account_type_resolution_method,

        COUNT(*) AS receipt_rows,

        SUM(payment_amount)
            AS receipt_amount

    FROM read_parquet(
        '{state_funding_sql}'
    )

    GROUP BY

        party_name_current,
        party_account_iban,
        party_account_type_source,
        party_account_type_analytical,
        party_account_type_resolution_method

    ORDER BY
        receipt_amount DESC NULLS LAST
    """
).df()


print()
print("=" * 90)
print("STATE FUNDING ACCOUNT-TYPE CROSSCHECK")
print("=" * 90)

display(
    state_crosscheck
)


# ============================================================
# 18. BUDGET EXPENSES:
# CENTRAL VS OFFICE + ACCOUNT TYPE
#
# This is the table we need for the NEXT classification step.
# Nothing is reclassified yet.
# ============================================================

budget_sql = sql_path(
    PAYMENT_DIR
    / "budget_expenses.parquet"
)


budget_profile = con.execute(
    f"""
    SELECT

        organization_level,

        party_account_type_analytical,

        COUNT(*) AS rows,

        COUNT(
            DISTINCT root_party_id
        )
            AS parties,

        COUNT(
            DISTINCT organization_id
        )
            AS organizations,

        COUNT(
            DISTINCT party_account_iban
        )
            AS accounts,

        SUM(payment_amount)
            AS amount

    FROM read_parquet(
        '{budget_sql}'
    )

    GROUP BY
        organization_level,
        party_account_type_analytical

    ORDER BY
        organization_level,
        rows DESC
    """
).df()


print()
print("=" * 90)
print("BUDGET EXPENSE PROFILE BY OWNER + ACCOUNT TYPE")
print("=" * 90)

display(
    budget_profile
)


# ============================================================
# 19. INTERNAL TRANSFERS FROM CENTRAL:
# ACCOUNT TYPE OF SOURCE ACCOUNT
#
# This directly implements the analytical need you described:
#
# central -> office from state account
# central -> office from ordinary account
#
# Nothing is classified yet.
# ============================================================

transfer_parts = []


for payment_type in (
    "budget_expenses",
    "outgoing_expenses",
):

    path_sql = sql_path(
        PAYMENT_DIR
        / f"{payment_type}.parquet"
    )


    transfer_parts.append(
        f"""
        SELECT

            '{payment_type}'
                AS source_section,

            root_party_id,
            party_name_current,

            organization_id,
            organization_name_current,

            receiver_code_normalized,

            party_account_iban,
            party_account_type_source,
            party_account_type_analytical,

            payment_operation_date,
            payment_amount

        FROM read_parquet(
            '{path_sql}'
        )

        WHERE
            organization_level
            =
            'central'

            AND

            internal_transfer
        """
    )


central_internal_transfers = (
    con.execute(
        "\nUNION ALL\n".join(
            transfer_parts
        )
    )
    .df()
)


print()
print("=" * 90)
print("CENTRAL INTERNAL TRANSFERS BY SOURCE ACCOUNT TYPE")
print("=" * 90)


if len(
    central_internal_transfers
):

    display(
        central_internal_transfers
        .groupby(
            [
                "source_section",
                "party_account_type_analytical",
            ],
            dropna=False,
        )
        .agg(
            rows=(
                "payment_amount",
                "size",
            ),

            amount=(
                "payment_amount",
                "sum",
            ),

            accounts=(
                "party_account_iban",
                "nunique",
            ),
        )
        .reset_index()
    )


    display(
        central_internal_transfers
        .sort_values(
            "payment_amount",
            ascending=False,
        )
        .head(
            100
        )
    )


# ============================================================
# 20. DONE
# ============================================================

print()
print("=" * 90)
print("DONE")
print("=" * 90)

print(
    "Normalized property_moneys:",
    PROPERTY_MONEYS_PATH
)

print(
    "Enriched property_moneys:",
    ENRICHED_PROPERTY_MONEYS_PATH
)

print(
    "Report account reference:",
    REPORT_ACCOUNT_REFERENCE_PATH
)

print(
    "Organization account reference:",
    ORGANIZATION_ACCOUNT_REFERENCE_PATH
)

print()
print(
    "Budget expenses were NOT reclassified."
)

print(
    "Internal-transfer funding source was NOT yet assigned."
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\property_moneys.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_property_moneys.py

TESTS
......................................................                   [100%]
54 passed in 1.22s

Return code: 0

INPUT
Analysis-selected reports: 78791

RAW READ PLAN
Count column: None
Reports requiring RAW read: 78791
Total selected reports: 78791


Property moneys:   0%|          | 0/78791 [00:00<?, ?it/s]


PROPERTY_MONEYS EXTRACTION
Rows: 19140
Reports represented: 8947
Organizations represented: 936

ACCOUNT NORMALIZATION QA
Rows: 19140
Canonical IBAN rows: 19140
Distinct canonical IBAN: 1664

IBAN status:
party_account_iban_status
valid    19140
Name: count, dtype: int64

ACCOUNT TYPE SOURCE QA
party_account_type_analytical
ordinary_account         16805
unknown                   2272
state_funding_account       63
Name: count, dtype: int64


,party_account_type_source,party_account_type_analytical,rows
0,Поточний рахунок,ordinary_account,16340
1,Рахунок для соціальних виплат,unknown,472
2,поточний рахунок,ordinary_account,411
3,Транзитний рахунок,unknown,195
4,поточний,unknown,166
5,Вкладний (депозитний) рахунок,unknown,155
6,Інше - Транзитний рахунок,unknown,113
7,Соціальний рахунок,unknown,90
8,Картковий рахунок,unknown,83
9,Інше-Транзитний рахунок,unknown,60


Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\normalized_v0_1\properties\property_moneys.parquet
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\properties\property_moneys.parquet
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\report_account_reference.parquet
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\organization_account_reference.parquet

REPORT ACCOUNT REFERENCE QA


...

,organization_level,party_account_type_analytical,rows,parties,organizations,accounts,amount
0,central,state_funding_account,18501,3,3,7,3.311804e+09
1,central,unknown,9531,3,3,6,6.222468e+08
2,central,ordinary_account,399,6,6,7,1.637221e+07
3,office,ordinary_account,833,10,15,17,3.999944e+06
4,office,unknown,218,1,2,2,1.482644e+06



CENTRAL INTERNAL TRANSFERS BY SOURCE ACCOUNT TYPE


,source_section,party_account_type_analytical,rows,amount,accounts
0,budget_expenses,ordinary_account,1,3.000000e+02,1
1,budget_expenses,state_funding_account,4029,1.239324e+09,6
2,budget_expenses,unknown,163,9.528419e+07,4
3,outgoing_expenses,ordinary_account,2745,7.976400e+08,60
4,outgoing_expenses,unknown,48,1.295544e+08,19


,source_section,root_party_id,party_name_current,organization_id,organization_name_current,receiver_code_normalized,party_account_iban,party_account_type_source,party_account_type_analytical,payment_operation_date,payment_amount
1340,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2026-05-25,1.000000e+08
5442,outgoing_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA723223130000026000000047424,Поточний рахунок,ordinary_account,2023-12-27,4.439600e+07
1349,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2025-01-20,4.329776e+07
3095,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2024-06-07,4.085845e+07
3155,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2024-02-12,4.085845e+07
2185,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2024-09-06,4.085845e+07
2376,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2023-02-22,3.963879e+07
2409,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2023-11-01,3.963879e+07
2393,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2023-07-27,3.963879e+07
2383,budget_expenses,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,СЛУГА НАРОДУ,40422142,UA023805260000026007001059741,Для зарахування коштів з державного бюджету на...,state_funding_account,2023-05-05,3.963879e+07



DONE
Normalized property_moneys: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\normalized_v0_1\properties\property_moneys.parquet
Enriched property_moneys: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\properties\property_moneys.parquet
Report account reference: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\report_account_reference.parquet
Organization account reference: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\organization_account_reference.parquet

Budget expenses were NOT reclassified.
Internal-transfer funding source was NOT yet assigned.


In [112]:
from pathlib import Path
import pandas as pd


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PATH = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "properties"
    / "property_moneys.parquet"
)


df = pd.read_parquet(
    PATH,
    columns=[
        "party_account_iban",
        "party_account_type_source",
        "organization_id",
        "root_party_id",
    ],
)


x = (
    df[
        df[
            "party_account_type_source"
        ]
        .fillna("")
        .str.contains(
            r"держ|бюдж|фінанс|відшкодуван",
            case=False,
            regex=True,
        )
    ]
    .groupby(
        "party_account_type_source",
        dropna=False,
    )
    .agg(
        rows=(
            "party_account_iban",
            "size",
        ),
        ibans=(
            "party_account_iban",
            "nunique",
        ),
        parties=(
            "root_party_id",
            "nunique",
        ),
        organizations=(
            "organization_id",
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "parties",
            "rows",
        ],
        ascending=False,
    )
)


pd.set_option(
    "display.max_colwidth",
    None,
)

pd.set_option(
    "display.max_rows",
    500,
)


display(
    x
)


print()
print("=" * 100)
print("FULL SOURCE VALUES")
print("=" * 100)

for value in x[
    "party_account_type_source"
]:

    print(
        repr(value)
    )

,party_account_type_source,rows,ibans,parties,organizations
7,Рахунок для відшкодування витрат пов'язаних із фінансуванням передвиборної агітації,2,2,2,2
0,Бюджетний рахунок,17,2,1,1
19,рахунок для отримання коштів з Державного бюджету України на фінансування статутної діяльності,14,3,1,1
1,Для зарахування коштів з державного бюджету на статутну діяльність,13,1,1,1
5,Поточний рахунок для зарахування коштів з державного бюджету на статутну діяльність,13,1,1,1
12,Рахунок для отримання коштів державного фінансування статутної діяльності політичної партії,13,3,1,1
2,"Для отримання відшкодув.витрат, пов'яз. із фінанс.передвибор.агітації",12,1,1,1
4,Поточний рахунок для виплат страхових відшкодувань,11,1,1,1
18,"рахунок для відшкодування витрат, пов’язаних із фінансуванням передвиборної агітації",11,2,1,1
16,Спеціальний рахунок для страхових відшкодувань,9,2,1,1



FULL SOURCE VALUES
"Рахунок для відшкодування витрат пов'язаних із фінансуванням передвиборної агітації"
'Бюджетний рахунок'
'рахунок для отримання коштів з Державного бюджету України на фінансування статутної діяльності'
'Для зарахування коштів з державного бюджету на статутну діяльність'
'Поточний рахунок для зарахування коштів з державного бюджету на статутну діяльність'
'Рахунок для отримання коштів державного фінансування статутної діяльності політичної партії'
"Для отримання відшкодув.витрат, пов'яз. із фінанс.передвибор.агітації"
'Поточний рахунок для виплат страхових відшкодувань'
'рахунок для відшкодування витрат, пов’язаних із фінансуванням передвиборної агітації'
'Спеціальний рахунок для страхових відшкодувань'
'Рахунок для отримання коштів з державного бюджету на фінансування статутної діяльності'
'Рахунок для відшкодування витрат пов"язаних з фінансуванням передвиборчої агітації'
'Рахунок для відшкодування витрат, пов’язаних із фінансуванням передвиборної агітації'
'Рахун

In [115]:
from pathlib import Path
import re

import pandas as pd


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

STATE_FUNDING_PATH = (
    PAYMENT_DIR
    / "state_funding.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "state_funding_account_reference.parquet"
)


# ============================================================
# 1. LOAD EXISTING ENRICHED STATE-FUNDING TABLE
# ============================================================

sf = pd.read_parquet(
    STATE_FUNDING_PATH
)


account_col = (
    "party_account_iban"
    if "party_account_iban" in sf.columns
    else "receiver_account_iban_canonical"
)


# ============================================================
# 2. FIND SOURCE FIELD:
#    "Форма державного фінансування"
#
# Exclude our derived account-type columns so that they
# cannot be mistaken for the source field.
# ============================================================

EXCLUDED = {
    "party_account_type_source",
    "party_account_type_declared",
    "party_account_type_declared_code",
    "party_account_type",
    "party_account_type_analytical",
    "party_account_type_resolution_method",
    "party_account_funding_class",
}


FORM_PATTERN = re.compile(
    r"статут|передвибор|відшкодув|державн\w*\s+фінанс",
    flags=re.IGNORECASE,
)


candidates = []


for col in sf.columns:

    if col in EXCLUDED:
        continue

    series = (
        sf[col]
        .dropna()
        .astype("string")
        .str.strip()
    )

    if series.empty:
        continue

    hits = series.str.contains(
        FORM_PATTERN,
        na=False,
    )

    if not hits.any():
        continue

    unique_count = int(
        series.nunique()
    )

    matched_rows = int(
        hits.sum()
    )

    match_ratio = float(
        hits.mean()
    )

    col_lower = col.lower()

    name_score = sum(
        token in col_lower
        for token in (
            "form",
            "fund",
            "financ",
            "state",
            "type",
        )
    )

    # Descriptive free-text fields get a penalty.
    if any(
        token in col_lower
        for token in (
            "purpose",
            "description",
            "name",
        )
    ):
        name_score -= 2


    candidates.append(
        {
            "column": col,
            "name_score": name_score,
            "matched_rows": matched_rows,
            "match_ratio": match_ratio,
            "unique_values": unique_count,
        }
    )


candidate_df = pd.DataFrame(
    candidates
)


if candidate_df.empty:

    raise RuntimeError(
        "Could not detect the source field "
        "'Форма державного фінансування'."
    )


candidate_df = candidate_df.sort_values(
    [
        "name_score",
        "match_ratio",
        "matched_rows",
        "unique_values",
    ],
    ascending=[
        False,
        False,
        False,
        True,
    ],
)


display(
    candidate_df
)


form_col = candidate_df.iloc[0][
    "column"
]


print()
print(
    "Detected funding-form column:",
    form_col
)


# ============================================================
# 3. SHOW EXACT SOURCE VALUES
# ============================================================

print()
print("=" * 100)
print("SOURCE STATE-FUNDING FORMS")
print("=" * 100)


source_forms = (
    sf[
        form_col
    ]
    .dropna()
    .astype("string")
    .str.strip()
    .value_counts()
)


print(
    source_forms.to_string()
)


# ============================================================
# 4. CANONICALIZE THE TWO LEGAL FORMS
#
# Preserve exact source wording separately.
# ============================================================

def classify_state_funding_form(
    value,
):

    if pd.isna(value):
        return "unknown"

    text = str(
        value
    ).strip().lower()


    if (
        "передвибор" in text
        or
        (
            "відшкодув" in text
            and
            "агіта" in text
        )
    ):

        return (
            "state_campaign_reimbursement"
        )


    if (
        "статут" in text
        or
        "державного фінансування" in text
    ):

        return (
            "state_statutory_funding"
        )


    return "unknown"


FORM_LABELS = {

    "state_statutory_funding":
        (
            "Державне фінансування "
            "статутної діяльності політичної партії"
        ),

    "state_campaign_reimbursement":
        (
            "Відшкодування витрат політичної партії, "
            "пов'язаних з фінансуванням "
            "передвиборної агітації"
        ),

    "unknown":
        "Невизначена форма державного фінансування",
}


sf[
    "state_funding_form_source"
] = (
    sf[
        form_col
    ]
    .astype("string")
)


sf[
    "state_funding_form_code"
] = (
    sf[
        "state_funding_form_source"
    ]
    .map(
        classify_state_funding_form
    )
)


sf[
    "state_funding_form"
] = (
    sf[
        "state_funding_form_code"
    ]
    .map(
        FORM_LABELS
    )
)


# ============================================================
# 5. PRIMARY EVIDENCE RULE
#
# Only a POSITIVE receipt establishes the account as
# a confirmed state-funding receiving account.
# ============================================================

positive = (
    sf[
        sf[
            account_col
        ].notna()
        &
        (
            pd.to_numeric(
                sf[
                    "payment_amount"
                ],
                errors="coerce",
            )
            .fillna(0)
            > 0
        )
    ]
    .copy()
)


if positive.empty:

    raise RuntimeError(
        "No positive state-funding receipts found."
    )


# Every positive receipt should have a recognized form.
unknown_positive = (
    positive[
        positive[
            "state_funding_form_code"
        ]
        ==
        "unknown"
    ]
)


print()
print("=" * 100)
print("FORM CLASSIFICATION CONTROL")
print("=" * 100)

print(
    "Positive receipt rows:",
    len(
        positive
    )
)

print(
    "Unclassified positive receipt rows:",
    len(
        unknown_positive
    )
)


if len(
    unknown_positive
):

    display(
        unknown_positive[
            [
                "party_name_current",
                account_col,
                form_col,
                "payment_amount",
            ]
        ]
    )

    raise RuntimeError(
        "Some positive state-funding receipts "
        "have an unknown funding form."
    )


# ============================================================
# 6. BUILD ONE ROW PER ORGANIZATION + IBAN
#
# If the same account ever receives both legal forms,
# we preserve that fact rather than forcing one label.
# ============================================================

account_ref = (
    positive
    .groupby(
        [
            "root_party_id",
            "organization_id",
            account_col,
        ],
        dropna=False,
    )
    .agg(
        party_name_current=(
            "party_name_current",
            "first",
        ),

        organization_name_current=(
            "organization_name_current",
            "first",
        ),

        organization_level=(
            "organization_level",
            "first",
        ),

        first_state_receipt_date=(
            "payment_operation_date",
            "min",
        ),

        last_state_receipt_date=(
            "payment_operation_date",
            "max",
        ),

        positive_state_receipt_rows=(
            "payment_amount",
            "size",
        ),

        positive_state_receipt_amount=(
            "payment_amount",
            "sum",
        ),

        state_funding_form_count=(
            "state_funding_form_code",
            "nunique",
        ),

        state_funding_forms_observed=(
            "state_funding_form_code",
            lambda s:
                " | ".join(
                    sorted(
                        set(
                            s.dropna()
                        )
                    )
                ),
        ),

        state_funding_source_forms_observed=(
            "state_funding_form_source",
            lambda s:
                " | ".join(
                    sorted(
                        set(
                            s.dropna()
                            .astype(str)
                        )
                    )
                ),
        ),
    )
    .reset_index()
    .rename(
        columns={
            account_col:
                "party_account_iban"
        }
    )
)


account_ref[
    "state_funding_account_confirmed"
] = True


account_ref[
    "state_funding_form_code"
] = (
    account_ref
    .apply(
        lambda row:
            (
                row[
                    "state_funding_forms_observed"
                ]
                if
                row[
                    "state_funding_form_count"
                ]
                ==
                1
                else
                "multiple_state_funding_forms"
            ),
        axis=1,
    )
)


account_ref[
    "state_funding_account_evidence"
] = (
    "positive_transaction_in_state_funding_section"
)


# ============================================================
# 7. SAVE
# ============================================================

account_ref.to_parquet(
    STATE_ACCOUNT_REFERENCE_PATH,
    index=False,
)


# Also retain the funding-form fields directly in the
# state_funding analytical payment table.
sf.to_parquet(
    STATE_FUNDING_PATH,
    index=False,
)


# ============================================================
# 8. FINAL QA
# ============================================================

print()
print("=" * 100)
print("CONFIRMED STATE-FUNDING ACCOUNTS")
print("=" * 100)


summary = (
    account_ref[
        "state_funding_form_code"
    ]
    .value_counts(
        dropna=False
    )
)


print(
    summary.to_string()
)


print()
print(
    "Confirmed accounts:",
    len(
        account_ref
    )
)

print(
    "Parties:",
    account_ref[
        "root_party_id"
    ].nunique()
)

print(
    "Organizations:",
    account_ref[
        "organization_id"
    ].nunique()
)


mixed = (
    account_ref[
        account_ref[
            "state_funding_form_count"
        ]
        >
        1
    ]
)


print(
    "Accounts with >1 state-funding form:",
    len(
        mixed
    )
)


if len(
    mixed
):

    display(
        mixed
    )


print()
print(
    "Written:",
    STATE_ACCOUNT_REFERENCE_PATH
)

print()
print(
    "No RAW JSON files were read."
)

,column,name_score,matched_rows,match_ratio,unique_values
0,payment_type_detail_source,1,96,1.0000,1
1,receiver_bank_name,-2,6,0.0625,13



Detected funding-form column: payment_type_detail_source

SOURCE STATE-FUNDING FORMS
payment_type_detail_source
Державне фінансування статутної діяльності політичної партії    96

FORM CLASSIFICATION CONTROL
Positive receipt rows: 93
Unclassified positive receipt rows: 0

CONFIRMED STATE-FUNDING ACCOUNTS
state_funding_form_code
state_statutory_funding    7

Confirmed accounts: 7
Parties: 4
Organizations: 4
Accounts with >1 state-funding form: 0

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\processed\enriched_v0_1\reference\state_funding_account_reference.parquet

No RAW JSON files were read.


In [116]:
from pathlib import Path

import duckdb
import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

ORG_REFERENCE_PATH = (
    REFERENCE_DIR
    / "organization_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "state_funding_account_reference.parquet"
)


PAYMENT_TYPES = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


INCOMING_TYPES = {
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
}


# ============================================================
# HELPERS
# ============================================================

def sql_path(path):

    return (
        Path(path)
        .resolve()
        .as_posix()
        .replace(
            "'",
            "''",
        )
    )


con = duckdb.connect()


org_ref_sql = sql_path(
    ORG_REFERENCE_PATH
)

state_account_sql = sql_path(
    STATE_ACCOUNT_REFERENCE_PATH
)


# ============================================================
# 1. CONFIRMED STATE ACCOUNT MAP
#
# This is now the ONLY authoritative analytical rule
# for identifying a political state-funding account:
#
# positive transaction observed in state_funding.
# ============================================================

state_accounts = pd.read_parquet(
    STATE_ACCOUNT_REFERENCE_PATH
)


print()
print("=" * 100)
print("CONFIRMED STATE-FUNDING ACCOUNT REFERENCE")
print("=" * 100)

print(
    "Accounts:",
    len(
        state_accounts
    )
)

print(
    "Parties:",
    state_accounts[
        "root_party_id"
    ].nunique()
)

print(
    "Organizations:",
    state_accounts[
        "organization_id"
    ].nunique()
)


display(
    state_accounts[
        [
            "party_name_current",
            "organization_name_current",
            "party_account_iban",

            "state_funding_form_code",
            "state_funding_source_forms_observed",

            "first_state_receipt_date",
            "last_state_receipt_date",

            "positive_state_receipt_rows",
            "positive_state_receipt_amount",
        ]
    ]
)


# ============================================================
# 2. UNIQUE SAME-PARTY ORGANIZATION CODE MAP
#
# Needed only to distinguish:
#
# central -> office
# office -> central
# office -> office
# same organization / own account
#
# Strict EDRPOU matching only.
# ============================================================

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW org_code_map AS

    SELECT

        root_party_id,
        organization_code,

        first(
            organization_id
        )
            AS organization_id,

        first(
            organization_name_current
        )
            AS organization_name_current,

        first(
            organization_level
        )
            AS organization_level

    FROM read_parquet(
        '{org_ref_sql}'
    )

    WHERE
        organization_code
        IS NOT NULL

        AND

        trim(
            organization_code
        )
        <> ''

    GROUP BY

        root_party_id,
        organization_code

    HAVING

        COUNT(
            DISTINCT organization_id
        )
        =
        1
    """
)


# ============================================================
# 3. STATE ACCOUNT TEMP VIEW
# ============================================================

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW confirmed_state_accounts AS

    SELECT

        root_party_id,
        organization_id,
        party_account_iban,

        state_funding_form_code,

        state_funding_source_forms_observed
            AS state_funding_form_source,

        first_state_receipt_date,
        last_state_receipt_date,

        positive_state_receipt_rows,
        positive_state_receipt_amount

    FROM read_parquet(
        '{state_account_sql}'
    )

    WHERE
        state_funding_account_confirmed
    """
)


# ============================================================
# 4. REBUILD ALL ENRICHED PAYMENT TABLES
#
# Adds / refreshes:
#
# payment_direction
#
# party_account_iban
# party_account_type_source        <- property_moneys source text
# party_account_type               <- human analytical type
# party_account_type_analytical
#
# state_funding_account_confirmed
# state_funding_form_source
# state_funding_form_code
#
# internal counterparty organization
# internal transfer direction
# internal transfer funding source
#
# analytical_payment_type
# was_reclassified
# reclassification_rule
# funding_source_analytical
# ============================================================

qa_rows = []


for payment_type in PAYMENT_TYPES:

    path = (
        PAYMENT_DIR
        / f"{payment_type}.parquet"
    )

    path_sql = sql_path(
        path
    )


    columns = [
        row[0]
        for row
        in con.execute(
            f"""
            DESCRIBE

            SELECT *

            FROM read_parquet(
                '{path_sql}'
            )
            """
        ).fetchall()
    ]


    # --------------------------------------------------------
    # Known available/fallback expressions
    # --------------------------------------------------------

    if (
        "party_account_iban"
        in columns
    ):

        account_expr = (
            "p.party_account_iban"
        )

    else:

        account_expr = (
            "p.receiver_account_iban_canonical"
        )


    if (
        "party_account_type_source"
        in columns
    ):

        source_account_type_expr = (
            "p.party_account_type_source"
        )

    else:

        source_account_type_expr = (
            "NULL::VARCHAR"
        )


    if (
        "source_payment_type"
        in columns
    ):

        source_payment_type_expr = (
            "p.source_payment_type"
        )

    else:

        source_payment_type_expr = (
            f"'{payment_type}'"
        )


    direction = (
        "incoming"
        if
        payment_type
        in
        INCOMING_TYPES
        else
        "outgoing"
    )


    if (
        direction
        ==
        "incoming"
    ):

        counterparty_code_expr = (
            "p.payer_code_normalized"
        )

    else:

        counterparty_code_expr = (
            "p.receiver_code_normalized"
        )


    # --------------------------------------------------------
    # For state_funding itself we already have the exact
    # source funding-form field. Preserve it as fallback
    # even for the 3 zero-amount rows.
    # --------------------------------------------------------

    if (
        payment_type
        ==
        "state_funding"

        and

        "state_funding_form_source"
        in columns
    ):

        state_form_source_fallback = (
            "p.state_funding_form_source"
        )

    else:

        state_form_source_fallback = (
            "NULL::VARCHAR"
        )


    if (
        payment_type
        ==
        "state_funding"

        and

        "state_funding_form_code"
        in columns
    ):

        state_form_code_fallback = (
            "p.state_funding_form_code"
        )

    else:

        state_form_code_fallback = (
            "NULL::VARCHAR"
        )


    # --------------------------------------------------------
    # Replace derived columns safely if rerunning cell.
    # --------------------------------------------------------

    replace_columns = [
        "payment_direction",

        "party_account_iban",
        "party_account_type",
        "party_account_type_analytical",
        "party_account_type_resolution_method",

        "state_funding_account_confirmed",
        "state_funding_form_source",
        "state_funding_form_code",

        "internal_counterparty_organization_id",
        "internal_counterparty_organization_name",
        "internal_counterparty_organization_level",

        "internal_transfer_source_organization_id",
        "internal_transfer_source_organization_level",

        "internal_transfer_destination_organization_id",
        "internal_transfer_destination_organization_level",

        "internal_transfer_direction",
        "internal_transfer_funding_source",

        "analytical_payment_type",
        "was_reclassified",
        "reclassification_rule",
        "funding_source_analytical",
    ]


    existing_replace_columns = [
        column
        for column
        in replace_columns
        if column
        in columns
    ]


    exclude_sql = (
        (
            "EXCLUDE ("
            + ", ".join(
                existing_replace_columns
            )
            + ")"
        )
        if
        existing_replace_columns
        else
        ""
    )


    # --------------------------------------------------------
    # Output file
    # --------------------------------------------------------

    tmp_path = (
        PAYMENT_DIR
        / f"{payment_type}.final_funding.tmp.parquet"
    )

    tmp_sql = sql_path(
        tmp_path
    )


    # --------------------------------------------------------
    # Payment-type-specific analytical rules
    # --------------------------------------------------------

    if (
        payment_type
        ==
        "budget_expenses"
    ):

        analytical_payment_type_sql = """
            CASE

                WHEN
                    organization_level
                    =
                    'central'

                    AND

                    state_funding_account_confirmed

                THEN
                    'budget_expenses'

                ELSE
                    'outgoing_expenses'

            END
        """


        was_reclassified_sql = """
            NOT (
                organization_level
                =
                'central'

                AND

                state_funding_account_confirmed
            )
        """


        reclassification_rule_sql = """
            CASE

                WHEN
                    organization_level
                    =
                    'office'

                THEN
                    'office_budget_expense_treated_as_ordinary_or_mixed'


                WHEN
                    organization_level
                    =
                    'central'

                    AND

                    NOT state_funding_account_confirmed

                THEN
                    'central_budget_expense_without_confirmed_state_account'


                ELSE NULL

            END
        """


        funding_source_sql = """
            CASE

                WHEN
                    organization_level
                    =
                    'central'

                    AND

                    state_funding_account_confirmed

                THEN
                    state_funding_form_code


                WHEN
                    organization_level
                    =
                    'office'

                THEN
                    'mixed_or_unknown'


                WHEN
                    party_account_type_analytical
                    =
                    'ordinary_account'

                THEN
                    'private_or_non_state'


                ELSE
                    'unknown'

            END
        """

    elif (
        payment_type
        ==
        "state_funding"
    ):

        analytical_payment_type_sql = (
            "'state_funding'"
        )

        was_reclassified_sql = (
            "FALSE"
        )

        reclassification_rule_sql = (
            "NULL::VARCHAR"
        )

        funding_source_sql = """
            COALESCE(
                state_funding_form_code,
                'unknown'
            )
        """

    else:

        analytical_payment_type_sql = (
            source_payment_type_expr
        )

        was_reclassified_sql = (
            "FALSE"
        )

        reclassification_rule_sql = (
            "NULL::VARCHAR"
        )

        funding_source_sql = (
            "NULL::VARCHAR"
        )


    # ========================================================
    # BUILD
    # ========================================================

    con.execute(
        f"""
        COPY (

            WITH

            base AS (

                SELECT

                    p.*
                    {exclude_sql},


                    -- ========================================
                    -- DIRECTION
                    -- ========================================

                    '{direction}'
                        AS payment_direction,


                    -- ========================================
                    -- PARTY ACCOUNT
                    --
                    -- In PolitData payment sections this is
                    -- the account of the reporting party /
                    -- party office.
                    -- ========================================

                    {account_expr}
                        AS party_account_iban,


                    -- ========================================
                    -- CONFIRMED STATE ACCOUNT
                    --
                    -- ONLY positive state_funding receipt
                    -- establishes this flag.
                    -- ========================================

                    (
                        sf.party_account_iban
                        IS NOT NULL
                    )
                        AS state_funding_account_confirmed,


                    COALESCE(
                        sf.state_funding_form_source,
                        {state_form_source_fallback}
                    )
                        AS state_funding_form_source,


                    COALESCE(
                        sf.state_funding_form_code,
                        {state_form_code_fallback}
                    )
                        AS state_funding_form_code,


                    -- ========================================
                    -- HUMAN ACCOUNT TYPE
                    --
                    -- Confirmed state-funding form has
                    -- highest priority.
                    --
                    -- Otherwise retain property_moneys
                    -- source wording.
                    -- ========================================

                    CASE

                        WHEN
                            sf.party_account_iban
                            IS NOT NULL

                        THEN
                            sf.state_funding_form_source


                        WHEN
                            {source_account_type_expr}
                            IS NOT NULL

                        THEN
                            {source_account_type_expr}


                        ELSE
                            'Тип рахунку не визначено'

                    END
                        AS party_account_type,


                    -- ========================================
                    -- SIMPLE MACHINE ACCOUNT TYPE
                    -- ========================================

                    CASE

                        WHEN
                            sf.party_account_iban
                            IS NOT NULL

                        THEN
                            sf.state_funding_form_code


                        WHEN
                            regexp_matches(
                                lower(
                                    COALESCE(
                                        {source_account_type_expr},
                                        ''
                                    )
                                ),
                                'поточн|розрахунк'
                            )

                        THEN
                            'ordinary_account'


                        WHEN
                            {source_account_type_expr}
                            IS NOT NULL

                        THEN
                            'other_declared_account'


                        ELSE
                            'unknown'

                    END
                        AS party_account_type_analytical,


                    CASE

                        WHEN
                            sf.party_account_iban
                            IS NOT NULL

                        THEN
                            'positive_transaction_in_state_funding_section'


                        WHEN
                            {source_account_type_expr}
                            IS NOT NULL

                        THEN
                            'property_moneys_source_type'


                        ELSE
                            'unresolved'

                    END
                        AS party_account_type_resolution_method,


                    -- ========================================
                    -- SAME-PARTY COUNTERPARTY ORGANIZATION
                    --
                    -- Strict EDRPOU only.
                    -- ========================================

                    CASE
                        WHEN p.internal_transfer
                        THEN c.organization_id
                        ELSE NULL
                    END
                        AS internal_counterparty_organization_id,


                    CASE
                        WHEN p.internal_transfer
                        THEN c.organization_name_current
                        ELSE NULL
                    END
                        AS internal_counterparty_organization_name,


                    CASE
                        WHEN p.internal_transfer
                        THEN c.organization_level
                        ELSE NULL
                    END
                        AS internal_counterparty_organization_level


                FROM read_parquet(
                    '{path_sql}'
                ) p


                LEFT JOIN confirmed_state_accounts sf

                    ON
                        p.root_party_id
                        =
                        sf.root_party_id

                    AND

                        p.organization_id
                        =
                        sf.organization_id

                    AND

                        {account_expr}
                        =
                        sf.party_account_iban


                LEFT JOIN org_code_map c

                    ON
                        p.root_party_id
                        =
                        c.root_party_id

                    AND

                        {counterparty_code_expr}
                        =
                        c.organization_code

            ),


            directed AS (

                SELECT

                    *,


                    -- ========================================
                    -- PHYSICAL SOURCE ORGANIZATION
                    -- ========================================

                    CASE

                        WHEN
                            NOT internal_transfer

                        THEN NULL


                        WHEN
                            payment_direction
                            =
                            'outgoing'

                        THEN
                            organization_id


                        ELSE
                            internal_counterparty_organization_id

                    END
                        AS internal_transfer_source_organization_id,


                    CASE

                        WHEN
                            NOT internal_transfer

                        THEN NULL


                        WHEN
                            payment_direction
                            =
                            'outgoing'

                        THEN
                            organization_level


                        ELSE
                            internal_counterparty_organization_level

                    END
                        AS internal_transfer_source_organization_level,


                    -- ========================================
                    -- PHYSICAL DESTINATION ORGANIZATION
                    -- ========================================

                    CASE

                        WHEN
                            NOT internal_transfer

                        THEN NULL


                        WHEN
                            payment_direction
                            =
                            'outgoing'

                        THEN
                            internal_counterparty_organization_id


                        ELSE
                            organization_id

                    END
                        AS internal_transfer_destination_organization_id,


                    CASE

                        WHEN
                            NOT internal_transfer

                        THEN NULL


                        WHEN
                            payment_direction
                            =
                            'outgoing'

                        THEN
                            internal_counterparty_organization_level


                        ELSE
                            organization_level

                    END
                        AS internal_transfer_destination_organization_level


                FROM base

            ),


            transfer_classified AS (

                SELECT

                    *,


                    -- ========================================
                    -- TRANSFER DIRECTION
                    -- ========================================

                    CASE

                        WHEN
                            NOT internal_transfer

                        THEN NULL


                        WHEN
                            internal_counterparty_organization_id
                            IS NULL

                        THEN
                            'same_party_counterparty_unresolved'


                        WHEN
                            internal_transfer_source_organization_id
                            =
                            internal_transfer_destination_organization_id

                        THEN
                            'intra_organization'


                        WHEN
                            internal_transfer_source_organization_level
                            =
                            'central'

                            AND

                            internal_transfer_destination_organization_level
                            =
                            'office'

                        THEN
                            'central_to_office'


                        WHEN
                            internal_transfer_source_organization_level
                            =
                            'office'

                            AND

                            internal_transfer_destination_organization_level
                            =
                            'central'

                        THEN
                            'office_to_central'


                        WHEN
                            internal_transfer_source_organization_level
                            =
                            'office'

                            AND

                            internal_transfer_destination_organization_level
                            =
                            'office'

                        THEN
                            'office_to_office'


                        ELSE
                            'other_same_party_transfer'

                    END
                        AS internal_transfer_direction


                FROM directed

            ),


            funding_classified AS (

                SELECT

                    *,


                    -- ========================================
                    -- FUNDING SOURCE OF INTERNAL TRANSFER
                    --
                    -- Only OUTGOING central -> office exposes
                    -- the source account of the centre.
                    -- ========================================

                    CASE

                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'outgoing'

                            AND

                            state_funding_account_confirmed

                        THEN
                            state_funding_form_code


                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'outgoing'

                            AND

                            party_account_type_analytical
                            =
                            'ordinary_account'

                        THEN
                            'private_or_non_state'


                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'outgoing'

                        THEN
                            'unknown'


                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'incoming'

                        THEN
                            'unknown_source_account'


                        WHEN
                            internal_transfer_direction
                            IN (
                                'office_to_central',
                                'office_to_office'
                            )

                        THEN
                            'mixed_or_unknown'


                        ELSE NULL

                    END
                        AS internal_transfer_funding_source


                FROM transfer_classified

            ),


            final AS (

                SELECT

                    *,

                    {analytical_payment_type_sql}
                        AS analytical_payment_type,

                    {was_reclassified_sql}
                        AS was_reclassified,

                    {reclassification_rule_sql}
                        AS reclassification_rule,

                    {funding_source_sql}
                        AS funding_source_analytical


                FROM funding_classified

            )


            SELECT *

            FROM final

        )

        TO '{tmp_sql}'

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )


    # ========================================================
    # ROW-COUNT CONTROL BEFORE REPLACEMENT
    # ========================================================

    old_count = con.execute(
        f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{path_sql}'
        )
        """
    ).fetchone()[0]


    new_count = con.execute(
        f"""
        SELECT COUNT(*)

        FROM read_parquet(
            '{tmp_sql}'
        )
        """
    ).fetchone()[0]


    if (
        old_count
        !=
        new_count
    ):

        raise RuntimeError(
            f"{payment_type}: row count changed "
            f"{old_count} -> {new_count}"
        )


    tmp_path.replace(
        path
    )


    qa_rows.append(
        {
            "section":
                payment_type,

            "rows":
                new_count,
        }
    )


# ============================================================
# 5. GLOBAL ROW CONTROL
# ============================================================

qa = pd.DataFrame(
    qa_rows
)


print()
print("=" * 100)
print("PAYMENT ROW CONTROL")
print("=" * 100)

display(
    qa
)


print(
    "Total payment rows:",
    int(
        qa[
            "rows"
        ].sum()
    )
)


# ============================================================
# 6. FINAL BUDGET EXPENSE CLASSIFICATION
# ============================================================

budget_sql = sql_path(
    PAYMENT_DIR
    / "budget_expenses.parquet"
)


budget_profile = con.execute(
    f"""
    SELECT

        organization_level,

        state_funding_account_confirmed,
        state_funding_form_code,

        party_account_type,

        analytical_payment_type,
        funding_source_analytical,

        reclassification_rule,

        COUNT(*) AS rows,

        COUNT(
            DISTINCT root_party_id
        )
            AS parties,

        COUNT(
            DISTINCT organization_id
        )
            AS organizations,

        COUNT(
            DISTINCT party_account_iban
        )
            AS accounts,

        SUM(payment_amount)
            AS amount

    FROM read_parquet(
        '{budget_sql}'
    )

    GROUP BY

        organization_level,

        state_funding_account_confirmed,
        state_funding_form_code,

        party_account_type,

        analytical_payment_type,
        funding_source_analytical,

        reclassification_rule

    ORDER BY
        amount DESC NULLS LAST
    """
).df()


print()
print("=" * 100)
print("FINAL BUDGET EXPENSE CLASSIFICATION")
print("=" * 100)

display(
    budget_profile
)


# ============================================================
# 7. BUDGET RECLASSIFICATION TOTALS
# ============================================================

reclassification_qa = con.execute(
    f"""
    SELECT

        reclassification_rule,

        COUNT(*) AS rows,

        SUM(payment_amount)
            AS amount

    FROM read_parquet(
        '{budget_sql}'
    )

    WHERE
        was_reclassified

    GROUP BY
        reclassification_rule

    ORDER BY
        amount DESC
    """
).df()


print()
print("=" * 100)
print("BUDGET RECLASSIFICATION TOTALS")
print("=" * 100)

display(
    reclassification_qa
)


# ============================================================
# 8. UNION ONLY COMMON INTERNAL-TRANSFER FIELDS
# ============================================================

transfer_queries = []


for payment_type in PAYMENT_TYPES:

    path_sql = sql_path(
        PAYMENT_DIR
        / f"{payment_type}.parquet"
    )


    transfer_queries.append(
        f"""
        SELECT

            '{payment_type}'
                AS section,

            root_party_id,
            party_name_current,

            organization_id,
            organization_name_current,
            organization_level,

            payment_direction,

            internal_counterparty_organization_id,
            internal_counterparty_organization_name,
            internal_counterparty_organization_level,

            internal_transfer_source_organization_id,
            internal_transfer_source_organization_level,

            internal_transfer_destination_organization_id,
            internal_transfer_destination_organization_level,

            internal_transfer_direction,
            internal_transfer_funding_source,

            party_account_iban,
            party_account_type,
            party_account_type_analytical,

            state_funding_account_confirmed,
            state_funding_form_source,
            state_funding_form_code,

            payment_operation_date,
            payment_amount

        FROM read_parquet(
            '{path_sql}'
        )

        WHERE
            internal_transfer
        """
    )


transfer_union_sql = (
    "\nUNION ALL\n"
    .join(
        transfer_queries
    )
)


# ============================================================
# 9. INTERNAL TRANSFER DIRECTIONS
# ============================================================

direction_summary = con.execute(
    f"""
    WITH transfers AS (

        {transfer_union_sql}

    )

    SELECT

        internal_transfer_direction,

        payment_direction,

        COUNT(*) AS rows,

        SUM(payment_amount)
            AS amount

    FROM transfers

    GROUP BY

        internal_transfer_direction,
        payment_direction

    ORDER BY
        rows DESC
    """
).df()


print()
print("=" * 100)
print("INTERNAL TRANSFER DIRECTIONS")
print("=" * 100)

display(
    direction_summary
)


# ============================================================
# 10. CENTRAL -> OFFICE OUTGOING
#
# This is the main classification we wanted.
# ============================================================

central_to_office = con.execute(
    f"""
    WITH transfers AS (

        {transfer_union_sql}

    )

    SELECT

        internal_transfer_funding_source,

        state_funding_form_source,

        party_account_type,

        COUNT(*) AS rows,

        COUNT(
            DISTINCT root_party_id
        )
            AS parties,

        COUNT(
            DISTINCT party_account_iban
        )
            AS source_accounts,

        COUNT(
            DISTINCT
            internal_transfer_destination_organization_id
        )
            AS destination_offices,

        SUM(payment_amount)
            AS amount

    FROM transfers

    WHERE

        internal_transfer_direction
        =
        'central_to_office'

        AND

        payment_direction
        =
        'outgoing'

    GROUP BY

        internal_transfer_funding_source,
        state_funding_form_source,
        party_account_type

    ORDER BY
        amount DESC NULLS LAST
    """
).df()


print()
print("=" * 100)
print("CENTRAL -> OFFICE OUTGOING FUNDING SOURCE")
print("=" * 100)

display(
    central_to_office
)


# ============================================================
# 11. SAMPLE CENTRAL -> OFFICE
# ============================================================

sample = con.execute(
    f"""
    WITH transfers AS (

        {transfer_union_sql}

    )

    SELECT

        party_name_current,

        organization_name_current
            AS source_organization,

        internal_counterparty_organization_name
            AS destination_organization,

        party_account_iban,
        party_account_type,

        state_funding_account_confirmed,

        state_funding_form_source,
        state_funding_form_code,

        internal_transfer_funding_source,

        payment_operation_date,
        payment_amount

    FROM transfers

    WHERE

        internal_transfer_direction
        =
        'central_to_office'

        AND

        payment_direction
        =
        'outgoing'

    ORDER BY
        payment_amount DESC NULLS LAST

    LIMIT 100
    """
).df()


print()
print("=" * 100)
print("CENTRAL -> OFFICE SAMPLE")
print("=" * 100)

display(
    sample
)


# ============================================================
# 12. UNRESOLVED INTERNAL COUNTERPARTIES
# ============================================================

unresolved = con.execute(
    f"""
    WITH transfers AS (

        {transfer_union_sql}

    )

    SELECT

        section,
        payment_direction,

        COUNT(*) AS rows,

        SUM(payment_amount)
            AS amount

    FROM transfers

    WHERE

        internal_transfer_direction
        =
        'same_party_counterparty_unresolved'

    GROUP BY

        section,
        payment_direction

    ORDER BY
        rows DESC
    """
).df()


print()
print("=" * 100)
print("UNRESOLVED SAME-PARTY COUNTERPARTIES")
print("=" * 100)

display(
    unresolved
)


# ============================================================
# 13. DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Confirmed state-funding accounts:",
    len(
        state_accounts
    )
)

print(
    "No RAW JSON files were read."
)

print(
    "Normalized payment files were NOT changed."
)

print(
    "Enriched payment files now use positive "
    "state_funding receipts as the authoritative "
    "state-account marker."
)


CONFIRMED STATE-FUNDING ACCOUNT REFERENCE
Accounts: 7
Parties: 4
Organizations: 4


,party_name_current,organization_name_current,party_account_iban,state_funding_form_code,state_funding_source_forms_observed,first_state_receipt_date,last_state_receipt_date,positive_state_receipt_rows,positive_state_receipt_amount
0,ВО БАТЬКІВЩИНА,ВО БАТЬКІВЩИНА,UA363516290000000002604826373,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,2021-02-17,2022-01-21,5,86383625.0000000000
1,ВО БАТЬКІВЩИНА,ВО БАТЬКІВЩИНА,UA723808380000026048700741786,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,2022-06-30,2026-04-17,17,391454215.0000000000
2,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,UA393805820000026007030316815,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,2021-02-17,2022-10-11,16,213881715.0000000000
3,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,UA553805820000026001070316815,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,2023-02-21,2023-02-21,2,34333475.0000000000
4,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,UA923805820000026002080316815,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,2023-05-31,2026-04-17,26,479084075.0000000000
5,СЛУГА НАРОДУ,СЛУГА НАРОДУ,UA023805260000026007001059741,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,2021-02-17,2026-04-17,22,2519051560.0000000000
6,ГОЛОС,ГОЛОС,UA813808050000000026002651529,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,2021-02-17,2024-04-18,5,118979325.0000000000


BinderException: Binder Error: Referenced table "p" not found!
Candidate tables: "funding_classified"

LINE 560:                     p.source_payment_type
                              ^

In [117]:
from pathlib import Path

import duckdb
import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

ORG_REFERENCE_PATH = (
    REFERENCE_DIR
    / "organization_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "state_funding_account_reference.parquet"
)


PAYMENT_TYPES = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


INCOMING_TYPES = {
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
}


# ============================================================
# HELPERS
# ============================================================

def sql_path(path):

    return (
        Path(path)
        .resolve()
        .as_posix()
        .replace("'", "''")
    )


con = duckdb.connect()


org_ref_sql = sql_path(
    ORG_REFERENCE_PATH
)

state_account_sql = sql_path(
    STATE_ACCOUNT_REFERENCE_PATH
)


# ============================================================
# 1. CLEAN FAILED TEMP FILES, IF ANY
# ============================================================

for payment_type in PAYMENT_TYPES:

    tmp_path = (
        PAYMENT_DIR
        / f"{payment_type}.final_funding.tmp.parquet"
    )

    if tmp_path.exists():
        tmp_path.unlink()


# ============================================================
# 2. UNIQUE SAME-PARTY ORGANIZATION CODE MAP
#
# Strict EDRPOU matching only.
# ============================================================

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW org_code_map AS

    SELECT

        root_party_id,
        organization_code,

        first(organization_id)
            AS organization_id,

        first(organization_name_current)
            AS organization_name_current,

        first(organization_level)
            AS organization_level

    FROM read_parquet(
        '{org_ref_sql}'
    )

    WHERE
        organization_code IS NOT NULL

        AND

        trim(organization_code) <> ''

    GROUP BY

        root_party_id,
        organization_code

    HAVING

        COUNT(
            DISTINCT organization_id
        ) = 1
    """
)


# ============================================================
# 3. CONFIRMED STATE-FUNDING ACCOUNTS
#
# ONLY positive receipt in state_funding establishes
# a confirmed state-funding account.
# ============================================================

con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW confirmed_state_accounts AS

    SELECT

        root_party_id,
        organization_id,
        party_account_iban,

        state_funding_form_code,

        state_funding_source_forms_observed
            AS state_funding_form_source,

        first_state_receipt_date,
        last_state_receipt_date,

        positive_state_receipt_rows,
        positive_state_receipt_amount

    FROM read_parquet(
        '{state_account_sql}'
    )

    WHERE
        state_funding_account_confirmed
    """
)


state_accounts = con.execute(
    """
    SELECT *
    FROM confirmed_state_accounts
    ORDER BY
        root_party_id,
        party_account_iban
    """
).df()


print()
print("=" * 100)
print("CONFIRMED STATE-FUNDING ACCOUNTS")
print("=" * 100)

print(
    "Accounts:",
    len(state_accounts)
)

print(
    "Parties:",
    state_accounts[
        "root_party_id"
    ].nunique()
)


# Hard regression from previous successful step.
assert len(state_accounts) == 7
assert (
    state_accounts[
        "root_party_id"
    ].nunique()
    ==
    4
)


# ============================================================
# 4. REBUILD ALL 8 ENRICHED PAYMENT TABLES
#
# IMPORTANT:
# The prior error was caused by using p.source_payment_type
# inside a later CTE where alias p no longer exists.
#
# Here analytical_payment_type uses the unqualified column
# source_payment_type after it has passed through the CTEs.
# ============================================================

qa_rows = []


for payment_type in PAYMENT_TYPES:

    path = (
        PAYMENT_DIR
        / f"{payment_type}.parquet"
    )

    path_sql = sql_path(path)


    columns = [
        row[0]
        for row
        in con.execute(
            f"""
            DESCRIBE
            SELECT *
            FROM read_parquet(
                '{path_sql}'
            )
            """
        ).fetchall()
    ]


    # --------------------------------------------------------
    # Existing party account
    # --------------------------------------------------------

    if "party_account_iban" in columns:

        account_expr = (
            "p.party_account_iban"
        )

    else:

        account_expr = (
            "p.receiver_account_iban_canonical"
        )


    # --------------------------------------------------------
    # Source account type from property_moneys
    # --------------------------------------------------------

    if "party_account_type_source" in columns:

        source_account_type_expr = (
            "p.party_account_type_source"
        )

    else:

        source_account_type_expr = (
            "NULL::VARCHAR"
        )


    # --------------------------------------------------------
    # Existing source payment type
    #
    # IMPORTANT:
    # final CTE must NOT refer to alias p.
    # --------------------------------------------------------

    if "source_payment_type" in columns:

        analytical_source_type_expr = (
            "source_payment_type"
        )

    else:

        analytical_source_type_expr = (
            f"'{payment_type}'"
        )


    # --------------------------------------------------------
    # Payment direction
    # --------------------------------------------------------

    if payment_type in INCOMING_TYPES:

        direction = "incoming"

        counterparty_code_expr = (
            "p.payer_code_normalized"
        )

    else:

        direction = "outgoing"

        counterparty_code_expr = (
            "p.receiver_code_normalized"
        )


    # --------------------------------------------------------
    # state_funding has source funding-form fields already
    # --------------------------------------------------------

    if (
        payment_type == "state_funding"
        and
        "state_funding_form_source" in columns
    ):

        state_form_source_fallback = (
            "p.state_funding_form_source"
        )

    else:

        state_form_source_fallback = (
            "NULL::VARCHAR"
        )


    if (
        payment_type == "state_funding"
        and
        "state_funding_form_code" in columns
    ):

        state_form_code_fallback = (
            "p.state_funding_form_code"
        )

    else:

        state_form_code_fallback = (
            "NULL::VARCHAR"
        )


    # --------------------------------------------------------
    # Drop previously derived fields before rebuilding them.
    # Makes rerun idempotent.
    # --------------------------------------------------------

    replace_columns = [
        "payment_direction",

        "party_account_iban",
        "party_account_type",
        "party_account_type_analytical",
        "party_account_type_resolution_method",

        "state_funding_account_confirmed",
        "state_funding_form_source",
        "state_funding_form_code",

        "internal_counterparty_organization_id",
        "internal_counterparty_organization_name",
        "internal_counterparty_organization_level",

        "internal_transfer_source_organization_id",
        "internal_transfer_source_organization_level",

        "internal_transfer_destination_organization_id",
        "internal_transfer_destination_organization_level",

        "internal_transfer_direction",
        "internal_transfer_funding_source",

        "analytical_payment_type",
        "was_reclassified",
        "reclassification_rule",
        "funding_source_analytical",
    ]


    existing_replace_columns = [
        column
        for column in replace_columns
        if column in columns
    ]


    if existing_replace_columns:

        exclude_sql = (
            "EXCLUDE ("
            + ", ".join(existing_replace_columns)
            + ")"
        )

    else:

        exclude_sql = ""


    # --------------------------------------------------------
    # Analytical payment classification
    # --------------------------------------------------------

    if payment_type == "budget_expenses":

        analytical_payment_type_sql = """
            CASE

                WHEN
                    organization_level = 'central'

                    AND

                    state_funding_account_confirmed

                THEN
                    'budget_expenses'

                ELSE
                    'outgoing_expenses'

            END
        """


        was_reclassified_sql = """
            NOT (
                organization_level = 'central'

                AND

                state_funding_account_confirmed
            )
        """


        reclassification_rule_sql = """
            CASE

                WHEN
                    organization_level = 'office'

                THEN
                    'office_budget_expense_treated_as_ordinary_or_mixed'


                WHEN
                    organization_level = 'central'

                    AND

                    NOT state_funding_account_confirmed

                THEN
                    'central_budget_expense_without_confirmed_state_account'


                ELSE NULL

            END
        """


        funding_source_sql = """
            CASE

                WHEN
                    organization_level = 'central'

                    AND

                    state_funding_account_confirmed

                THEN
                    state_funding_form_code


                WHEN
                    organization_level = 'office'

                THEN
                    'mixed_or_unknown'


                WHEN
                    party_account_type_analytical
                    =
                    'ordinary_account'

                THEN
                    'private_or_non_state'


                ELSE
                    'unknown'

            END
        """


    elif payment_type == "state_funding":

        analytical_payment_type_sql = (
            "'state_funding'"
        )

        was_reclassified_sql = (
            "FALSE"
        )

        reclassification_rule_sql = (
            "NULL::VARCHAR"
        )

        funding_source_sql = """
            COALESCE(
                state_funding_form_code,
                'unknown'
            )
        """


    else:

        # THIS is the local fix for the BinderException.
        analytical_payment_type_sql = (
            analytical_source_type_expr
        )

        was_reclassified_sql = (
            "FALSE"
        )

        reclassification_rule_sql = (
            "NULL::VARCHAR"
        )

        funding_source_sql = (
            "NULL::VARCHAR"
        )


    # --------------------------------------------------------
    # TEMP OUTPUT
    # --------------------------------------------------------

    tmp_path = (
        PAYMENT_DIR
        / f"{payment_type}.final_funding.tmp.parquet"
    )

    tmp_sql = sql_path(tmp_path)


    # ========================================================
    # BUILD
    # ========================================================

    con.execute(
        f"""
        COPY (

            WITH

            base AS (

                SELECT

                    p.*
                    {exclude_sql},


                    '{direction}'
                        AS payment_direction,


                    -- ========================================
                    -- PARTY / OFFICE ACCOUNT
                    -- ========================================

                    {account_expr}
                        AS party_account_iban,


                    -- ========================================
                    -- CONFIRMED STATE-FUNDING ACCOUNT
                    -- ========================================

                    (
                        sf.party_account_iban
                        IS NOT NULL
                    )
                        AS state_funding_account_confirmed,


                    COALESCE(
                        sf.state_funding_form_source,
                        {state_form_source_fallback}
                    )
                        AS state_funding_form_source,


                    COALESCE(
                        sf.state_funding_form_code,
                        {state_form_code_fallback}
                    )
                        AS state_funding_form_code,


                    -- ========================================
                    -- HUMAN ACCOUNT TYPE
                    -- ========================================

                    CASE

                        WHEN
                            sf.state_funding_form_code
                            =
                            'state_statutory_funding'

                        THEN
                            'Рахунок державного фінансування статутної діяльності'


                        WHEN
                            sf.state_funding_form_code
                            =
                            'state_campaign_reimbursement'

                        THEN
                            'Рахунок відшкодування витрат на передвиборну агітацію'


                        WHEN
                            {source_account_type_expr}
                            IS NOT NULL

                        THEN
                            {source_account_type_expr}


                        ELSE
                            'Тип рахунку не визначено'

                    END
                        AS party_account_type,


                    -- ========================================
                    -- MACHINE ACCOUNT TYPE
                    -- ========================================

                    CASE

                        WHEN
                            sf.state_funding_form_code
                            =
                            'state_statutory_funding'

                        THEN
                            'state_statutory_funding_account'


                        WHEN
                            sf.state_funding_form_code
                            =
                            'state_campaign_reimbursement'

                        THEN
                            'state_campaign_reimbursement_account'


                        WHEN
                            regexp_matches(
                                lower(
                                    COALESCE(
                                        {source_account_type_expr},
                                        ''
                                    )
                                ),
                                'поточн|розрахунк'
                            )

                        THEN
                            'ordinary_account'


                        WHEN
                            {source_account_type_expr}
                            IS NOT NULL

                        THEN
                            'other_declared_account'


                        ELSE
                            'unknown'

                    END
                        AS party_account_type_analytical,


                    CASE

                        WHEN
                            sf.party_account_iban
                            IS NOT NULL

                        THEN
                            'positive_transaction_in_state_funding_section'


                        WHEN
                            {source_account_type_expr}
                            IS NOT NULL

                        THEN
                            'property_moneys_source_type'


                        ELSE
                            'unresolved'

                    END
                        AS party_account_type_resolution_method,


                    -- ========================================
                    -- INTERNAL COUNTERPARTY
                    -- Strict same-root EDRPOU map only.
                    -- ========================================

                    CASE
                        WHEN p.internal_transfer
                        THEN c.organization_id
                        ELSE NULL
                    END
                        AS internal_counterparty_organization_id,


                    CASE
                        WHEN p.internal_transfer
                        THEN c.organization_name_current
                        ELSE NULL
                    END
                        AS internal_counterparty_organization_name,


                    CASE
                        WHEN p.internal_transfer
                        THEN c.organization_level
                        ELSE NULL
                    END
                        AS internal_counterparty_organization_level


                FROM read_parquet(
                    '{path_sql}'
                ) p


                LEFT JOIN confirmed_state_accounts sf

                    ON
                        p.root_party_id
                        =
                        sf.root_party_id

                    AND

                        p.organization_id
                        =
                        sf.organization_id

                    AND

                        {account_expr}
                        =
                        sf.party_account_iban


                LEFT JOIN org_code_map c

                    ON
                        p.root_party_id
                        =
                        c.root_party_id

                    AND

                        {counterparty_code_expr}
                        =
                        c.organization_code

            ),


            directed AS (

                SELECT

                    *,


                    CASE

                        WHEN NOT internal_transfer
                        THEN NULL

                        WHEN payment_direction = 'outgoing'
                        THEN organization_id

                        ELSE
                            internal_counterparty_organization_id

                    END
                        AS internal_transfer_source_organization_id,


                    CASE

                        WHEN NOT internal_transfer
                        THEN NULL

                        WHEN payment_direction = 'outgoing'
                        THEN organization_level

                        ELSE
                            internal_counterparty_organization_level

                    END
                        AS internal_transfer_source_organization_level,


                    CASE

                        WHEN NOT internal_transfer
                        THEN NULL

                        WHEN payment_direction = 'outgoing'
                        THEN
                            internal_counterparty_organization_id

                        ELSE
                            organization_id

                    END
                        AS internal_transfer_destination_organization_id,


                    CASE

                        WHEN NOT internal_transfer
                        THEN NULL

                        WHEN payment_direction = 'outgoing'
                        THEN
                            internal_counterparty_organization_level

                        ELSE
                            organization_level

                    END
                        AS internal_transfer_destination_organization_level


                FROM base

            ),


            transfer_classified AS (

                SELECT

                    *,


                    CASE

                        WHEN NOT internal_transfer
                        THEN NULL


                        WHEN
                            internal_counterparty_organization_id
                            IS NULL

                        THEN
                            'same_party_counterparty_unresolved'


                        WHEN
                            internal_transfer_source_organization_id
                            =
                            internal_transfer_destination_organization_id

                        THEN
                            'intra_organization'


                        WHEN
                            internal_transfer_source_organization_level
                            =
                            'central'

                            AND

                            internal_transfer_destination_organization_level
                            =
                            'office'

                        THEN
                            'central_to_office'


                        WHEN
                            internal_transfer_source_organization_level
                            =
                            'office'

                            AND

                            internal_transfer_destination_organization_level
                            =
                            'central'

                        THEN
                            'office_to_central'


                        WHEN
                            internal_transfer_source_organization_level
                            =
                            'office'

                            AND

                            internal_transfer_destination_organization_level
                            =
                            'office'

                        THEN
                            'office_to_office'


                        ELSE
                            'other_same_party_transfer'

                    END
                        AS internal_transfer_direction


                FROM directed

            ),


            funding_classified AS (

                SELECT

                    *,


                    CASE

                        -- ------------------------------------
                        -- CENTRE -> OFFICE:
                        -- source account is visible in the
                        -- outgoing central row.
                        -- ------------------------------------

                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'outgoing'

                            AND

                            state_funding_account_confirmed

                        THEN
                            state_funding_form_code


                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'outgoing'

                            AND

                            party_account_type_analytical
                            =
                            'ordinary_account'

                        THEN
                            'private_or_non_state'


                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'outgoing'

                        THEN
                            'unknown'


                        -- Incoming office row does not expose
                        -- the account at the centre.
                        WHEN
                            internal_transfer_direction
                            =
                            'central_to_office'

                            AND

                            payment_direction
                            =
                            'incoming'

                        THEN
                            'unknown_source_account'


                        -- Office-originating transfers are not
                        -- traced to a clean funding source.
                        WHEN
                            internal_transfer_direction
                            IN (
                                'office_to_central',
                                'office_to_office'
                            )

                        THEN
                            'mixed_or_unknown'


                        ELSE NULL

                    END
                        AS internal_transfer_funding_source


                FROM transfer_classified

            ),


            final AS (

                SELECT

                    *,

                    {analytical_payment_type_sql}
                        AS analytical_payment_type,

                    {was_reclassified_sql}
                        AS was_reclassified,

                    {reclassification_rule_sql}
                        AS reclassification_rule,

                    {funding_source_sql}
                        AS funding_source_analytical


                FROM funding_classified

            )


            SELECT *
            FROM final

        )

        TO '{tmp_sql}'

        (
            FORMAT PARQUET,
            COMPRESSION ZSTD
        )
        """
    )


    # ========================================================
    # ROW-COUNT CONTROL
    # ========================================================

    old_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet(
            '{path_sql}'
        )
        """
    ).fetchone()[0]


    new_count = con.execute(
        f"""
        SELECT COUNT(*)
        FROM read_parquet(
            '{tmp_sql}'
        )
        """
    ).fetchone()[0]


    if old_count != new_count:

        raise RuntimeError(
            f"{payment_type}: row count changed "
            f"{old_count} -> {new_count}"
        )


    tmp_path.replace(path)


    qa_rows.append(
        {
            "section":
                payment_type,

            "rows":
                new_count,
        }
    )


# ============================================================
# 5. GLOBAL PAYMENT CONTROL
# ============================================================

qa = pd.DataFrame(
    qa_rows
)


print()
print("=" * 100)
print("PAYMENT ROW CONTROL")
print("=" * 100)

display(qa)


total_rows = int(
    qa["rows"].sum()
)


print(
    "Total payment rows:",
    f"{total_rows:,}"
)


if total_rows != 402_028:

    raise RuntimeError(
        f"Expected 402,028 payment rows, got {total_rows:,}"
    )


# ============================================================
# 6. FINAL BUDGET CLASSIFICATION
# ============================================================

budget_sql = sql_path(
    PAYMENT_DIR
    / "budget_expenses.parquet"
)


budget_profile = con.execute(
    f"""
    SELECT

        organization_level,

        state_funding_account_confirmed,
        state_funding_form_code,

        party_account_type,

        analytical_payment_type,
        funding_source_analytical,

        reclassification_rule,

        COUNT(*) AS rows,

        COUNT(
            DISTINCT root_party_id
        )
            AS parties,

        COUNT(
            DISTINCT organization_id
        )
            AS organizations,

        COUNT(
            DISTINCT party_account_iban
        )
            AS accounts,

        SUM(payment_amount)
            AS amount

    FROM read_parquet(
        '{budget_sql}'
    )

    GROUP BY

        organization_level,

        state_funding_account_confirmed,
        state_funding_form_code,

        party_account_type,

        analytical_payment_type,
        funding_source_analytical,

        reclassification_rule

    ORDER BY
        amount DESC NULLS LAST
    """
).df()


print()
print("=" * 100)
print("FINAL BUDGET EXPENSE CLASSIFICATION")
print("=" * 100)

display(
    budget_profile
)


# ============================================================
# 7. RECLASSIFICATION TOTALS
# ============================================================

reclassification_qa = con.execute(
    f"""
    SELECT

        reclassification_rule,

        COUNT(*) AS rows,

        SUM(payment_amount)
            AS amount

    FROM read_parquet(
        '{budget_sql}'
    )

    WHERE
        was_reclassified

    GROUP BY
        reclassification_rule

    ORDER BY
        amount DESC
    """
).df()


print()
print("=" * 100)
print("BUDGET RECLASSIFICATION TOTALS")
print("=" * 100)

display(
    reclassification_qa
)


# ============================================================
# 8. UNION INTERNAL TRANSFERS
# ============================================================

transfer_queries = []


for payment_type in PAYMENT_TYPES:

    path_sql = sql_path(
        PAYMENT_DIR
        / f"{payment_type}.parquet"
    )


    transfer_queries.append(
        f"""
        SELECT

            '{payment_type}'
                AS section,

            root_party_id,
            party_name_current,

            organization_id,
            organization_name_current,
            organization_level,

            payment_direction,

            internal_counterparty_organization_id,
            internal_counterparty_organization_name,
            internal_counterparty_organization_level,

            internal_transfer_source_organization_id,
            internal_transfer_source_organization_level,

            internal_transfer_destination_organization_id,
            internal_transfer_destination_organization_level,

            internal_transfer_direction,
            internal_transfer_funding_source,

            party_account_iban,
            party_account_type,
            party_account_type_analytical,

            state_funding_account_confirmed,
            state_funding_form_source,
            state_funding_form_code,

            payment_operation_date,
            payment_amount

        FROM read_parquet(
            '{path_sql}'
        )

        WHERE
            internal_transfer
        """
    )


transfer_union_sql = (
    "\nUNION ALL\n"
    .join(
        transfer_queries
    )
)


# ============================================================
# 9. INTERNAL TRANSFER DIRECTIONS
# ============================================================

direction_summary = con.execute(
    f"""
    WITH transfers AS (

        {transfer_union_sql}

    )

    SELECT

        internal_transfer_direction,

        payment_direction,

        COUNT(*) AS rows,

        SUM(payment_amount)
            AS amount

    FROM transfers

    GROUP BY

        internal_transfer_direction,
        payment_direction

    ORDER BY
        rows DESC
    """
).df()


print()
print("=" * 100)
print("INTERNAL TRANSFER DIRECTIONS")
print("=" * 100)

display(
    direction_summary
)


# ============================================================
# 10. CENTRAL -> OFFICE FUNDING SOURCE
# ============================================================

central_to_office = con.execute(
    f"""
    WITH transfers AS (

        {transfer_union_sql}

    )

    SELECT

        internal_transfer_funding_source,

        state_funding_form_source,

        party_account_type,

        COUNT(*) AS rows,

        COUNT(
            DISTINCT root_party_id
        )
            AS parties,

        COUNT(
            DISTINCT party_account_iban
        )
            AS source_accounts,

        COUNT(
            DISTINCT
            internal_transfer_destination_organization_id
        )
            AS destination_offices,

        SUM(payment_amount)
            AS amount

    FROM transfers

    WHERE

        internal_transfer_direction
        =
        'central_to_office'

        AND

        payment_direction
        =
        'outgoing'

    GROUP BY

        internal_transfer_funding_source,
        state_funding_form_source,
        party_account_type

    ORDER BY
        amount DESC NULLS LAST
    """
).df()


print()
print("=" * 100)
print("CENTRAL -> OFFICE OUTGOING FUNDING SOURCE")
print("=" * 100)

display(
    central_to_office
)


# ============================================================
# 11. UNRESOLVED SAME-PARTY COUNTERPARTIES
# ============================================================

unresolved = con.execute(
    f"""
    WITH transfers AS (

        {transfer_union_sql}

    )

    SELECT

        section,
        payment_direction,

        COUNT(*) AS rows,

        SUM(payment_amount)
            AS amount

    FROM transfers

    WHERE
        internal_transfer_direction
        =
        'same_party_counterparty_unresolved'

    GROUP BY

        section,
        payment_direction

    ORDER BY
        rows DESC
    """
).df()


print()
print("=" * 100)
print("UNRESOLVED SAME-PARTY COUNTERPARTIES")
print("=" * 100)

display(
    unresolved
)


# ============================================================
# 12. DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Confirmed state-funding accounts:",
    len(state_accounts)
)

print(
    "No RAW JSON files were read."
)

print(
    "Normalized payment files were NOT modified."
)

print(
    "Enriched payment files were rebuilt successfully."
)


CONFIRMED STATE-FUNDING ACCOUNTS
Accounts: 7
Parties: 4


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


PAYMENT ROW CONTROL


,section,rows
0,monetary_contributions,27234
1,other_contributions,6168
2,state_funding,96
3,other_incomes,19007
4,budget_expenses,29482
5,outgoing_expenses,319901
6,return_expenses,137
7,transfer_expenses,3


Total payment rows: 402,028

FINAL BUDGET EXPENSE CLASSIFICATION


...


CENTRAL -> OFFICE OUTGOING FUNDING SOURCE


,internal_transfer_funding_source,state_funding_form_source,party_account_type,rows,parties,source_accounts,destination_offices,amount
0,private_or_non_state,None,Поточний рахунок,2227,16,19,91,6.749580e+08
1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,Рахунок державного фінансування статутної діяльності,3344,3,5,55,2.264752e+08
2,unknown,None,Рахунок для отримання коштів з державного бюджету на фінансування статутної діяльності,250,1,1,26,1.517083e+07
3,unknown,None,Рахунок для отримання коштів державного фінансування статутної діяльності політичної партії,49,1,1,26,2.609253e+06
4,unknown,None,"Рахунок для відшкодування витрат, пов""язаних з фінансуванням передвиборчої агітації",2,1,1,2,5.254610e+05
5,unknown,None,Рахунок для відшкодування витрат пов'язаних із фінансуванням передвиборної агітації,16,1,1,9,3.161992e+05
6,private_or_non_state,None,поточний рахунок,5,1,1,4,3.076000e+04
7,private_or_non_state,None,поточний,4,1,1,3,2.507500e+04



UNRESOLVED SAME-PARTY COUNTERPARTIES


,section,payment_direction,rows,amount



DONE
Confirmed state-funding accounts: 7
No RAW JSON files were read.
Normalized payment files were NOT modified.
Enriched payment files were rebuilt successfully.


In [118]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import json
import re
import subprocess
import sys
import textwrap
from datetime import datetime, timezone

import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

RAW_DIR = (
    ROOT
    / "data"
    / "raw"
    / "report_details"
)

REPORT_CONTEXT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "report_context.parquet"
)

DETAIL_STATE_PATH = (
    ROOT
    / "data"
    / "interim"
    / "state"
    / "report_detail_state.parquet"
)

SECTION_STATE_PATH = (
    ROOT
    / "data"
    / "interim"
    / "state"
    / "report_section_normalization_state.parquet"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

ENRICHED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
)

MODULE_PATH = (
    ROOT
    / "src"
    / "politdata"
    / "normalization"
    / "report_sections.py"
)

TEST_PATH = (
    ROOT
    / "tests"
    / "test_report_sections.py"
)


# ============================================================
# OUTPUT TABLES
# ============================================================

TABLE_PATHS = {

    # Properties
    "realty": (
        NORMALIZED_ROOT
        / "properties"
        / "realty.parquet"
    ),

    "transport": (
        NORMALIZED_ROOT
        / "properties"
        / "transport.parquet"
    ),

    "movable": (
        NORMALIZED_ROOT
        / "properties"
        / "movable.parquet"
    ),

    "intangible": (
        NORMALIZED_ROOT
        / "properties"
        / "intangible.parquet"
    ),

    "paper": (
        NORMALIZED_ROOT
        / "properties"
        / "paper.parquet"
    ),

    # Obligations
    "obligations": (
        NORMALIZED_ROOT
        / "obligations"
        / "obligations.parquet"
    ),

    # Report-state tables
    "head_info": (
        NORMALIZED_ROOT
        / "report_state"
        / "head_info.parquet"
    ),

    "employee_counts": (
        NORMALIZED_ROOT
        / "report_state"
        / "employee_counts.parquet"
    ),

    "organizations": (
        NORMALIZED_ROOT
        / "report_state"
        / "organizations.parquet"
    ),

    "regional_offices": (
        NORMALIZED_ROOT
        / "report_state"
        / "regional_offices.parquet"
    ),
}


ENRICHED_TABLE_PATHS = {

    name: (
        ENRICHED_ROOT
        / path.relative_to(
            NORMALIZED_ROOT
        )
    )

    for name, path
    in TABLE_PATHS.items()
}


for path in (
    list(
        TABLE_PATHS.values()
    )
    +
    list(
        ENRICHED_TABLE_PATHS.values()
    )
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


SECTION_STATE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. WRITE PERMANENT NORMALIZATION MODULE
# ============================================================

MODULE_CODE = r'''
from __future__ import annotations

import json
import re


NORMALIZATION_VERSION = (
    "report_sections_v0_1"
)


SECTION_PATHS = {

    "realty":
        ("properties", "realty"),

    "transport":
        ("properties", "transport"),

    "movable":
        ("properties", "movable"),

    "intangible":
        ("properties", "intangible"),

    "paper":
        ("properties", "paper"),

    "obligations":
        ("obligations",),

    "head_info":
        ("head_info",),

    "organizations":
        ("organizations",),

    "regional_offices":
        ("regional_offices",),
}


def get_nested(
    obj,
    path,
):
    """
    Conservative path lookup.
    """

    value = obj

    for key in path:

        if not isinstance(
            value,
            dict,
        ):
            return None

        value = value.get(
            key
        )

    return value


def coerce_rows(
    value,
):
    """
    Convert a report section into zero or more row objects.

    - list -> one row per item
    - dict -> one row
    - scalar -> {"value": scalar}
    - None -> []

    Conservative support for wrapper dictionaries with a
    single list-like member such as results/data/items/list.
    """

    if value is None:
        return []


    if isinstance(
        value,
        list,
    ):

        rows = []

        for item in value:

            if isinstance(
                item,
                dict,
            ):
                rows.append(
                    item
                )

            else:
                rows.append(
                    {
                        "value":
                            item
                    }
                )

        return rows


    if isinstance(
        value,
        dict,
    ):

        wrapper_keys = (
            "results",
            "data",
            "items",
            "list",
        )

        list_members = [
            key
            for key
            in wrapper_keys
            if isinstance(
                value.get(key),
                list,
            )
        ]


        if (
            len(value) == 1
            and
            len(list_members) == 1
        ):

            return coerce_rows(
                value[
                    list_members[0]
                ]
            )


        return [
            value
        ]


    return [
        {
            "value":
                value
        }
    ]


def extract_section_rows(
    detail,
    section,
):
    """
    Extract one known structural section from report detail.
    """

    if section not in SECTION_PATHS:

        raise KeyError(
            section
        )


    value = get_nested(
        detail,
        SECTION_PATHS[
            section
        ],
    )


    return coerce_rows(
        value
    )


def safe_source_key(
    key,
):
    """
    Stable column-safe source field name.
    Unicode letters are preserved.
    """

    text = str(
        key
    ).strip()

    text = re.sub(
        r"[^\w]+",
        "_",
        text,
        flags=re.UNICODE,
    )

    text = text.strip(
        "_"
    )

    return (
        text
        or
        "field"
    )


def scalar_or_json(
    value,
):
    """
    Preserve scalars as scalars and nested structures as JSON.
    """

    if isinstance(
        value,
        (
            dict,
            list,
        ),
    ):

        return json.dumps(
            value,
            ensure_ascii=False,
            sort_keys=True,
            default=str,
        )

    return value


def normalize_source_row(
    source_row,
    *,
    source_report_id,
    source_section,
    source_row_index,
    organization_id,
    root_party_id,
    report_year,
    report_quarter,
    source_is_signed,
    source_signed_date,
    report_schema_version_source,
    report_type_source,
    is_party_office_source,
):
    """
    Structural normalization.

    All source top-level fields are preserved under source__*.
    The full source row is also retained as JSON.
    """

    if not isinstance(
        source_row,
        dict,
    ):

        source_row = {
            "value":
                source_row
        }


    result = {

        "source_report_id":
            str(
                source_report_id
            ),

        "source_section":
            source_section,

        "source_row_index":
            int(
                source_row_index
            ),

        "organization_id":
            (
                str(
                    organization_id
                )
                if
                organization_id
                is not None
                else
                None
            ),

        "root_party_id":
            (
                str(
                    root_party_id
                )
                if
                root_party_id
                is not None
                else
                None
            ),

        "report_year":
            report_year,

        "report_quarter":
            report_quarter,

        "source_is_signed":
            bool(
                source_is_signed
            ),

        "source_signed_date":
            source_signed_date,

        "report_schema_version_source":
            report_schema_version_source,

        "report_type_source":
            report_type_source,

        "is_party_office_source":
            is_party_office_source,

        "source_row_json":
            json.dumps(
                source_row,
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            ),
    }


    for key, value in source_row.items():

        base_column = (
            "source__"
            +
            safe_source_key(
                key
            )
        )

        column = base_column

        suffix = 2

        while column in result:

            column = (
                f"{base_column}_{suffix}"
            )

            suffix += 1


        result[
            column
        ] = scalar_or_json(
            value
        )


    return result
'''


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    MODULE_PATH
)


# ============================================================
# 2. WRITE TESTS
# ============================================================

TEST_CODE = r'''
from politdata.normalization.report_sections import (
    extract_section_rows,
    normalize_source_row,
)


def test_extract_property_list():

    detail = {
        "properties": {
            "realty": [
                {
                    "id": "x"
                },
                {
                    "id": "y"
                },
            ]
        }
    }

    rows = extract_section_rows(
        detail,
        "realty",
    )

    assert len(rows) == 2


def test_extract_head_info_dict():

    detail = {
        "head_info": {
            "name": "TEST"
        }
    }

    rows = extract_section_rows(
        detail,
        "head_info",
    )

    assert len(rows) == 1

    assert (
        rows[0]["name"]
        ==
        "TEST"
    )


def test_empty_section():

    detail = {
        "properties": {
            "transport": []
        }
    }

    assert (
        extract_section_rows(
            detail,
            "transport",
        )
        ==
        []
    )


def test_scalar_list_item():

    detail = {
        "organizations": [
            "123"
        ]
    }

    rows = extract_section_rows(
        detail,
        "organizations",
    )

    assert (
        rows
        ==
        [
            {
                "value": "123"
            }
        ]
    )


def test_normalize_preserves_source_fields():

    result = normalize_source_row(
        {
            "id": "a",
            "amount": 10,
            "nested": {
                "x": 1
            },
        },
        source_report_id="r1",
        source_section="obligations",
        source_row_index=0,
        organization_id="o1",
        root_party_id="p1",
        report_year=2025,
        report_quarter=1,
        source_is_signed=True,
        source_signed_date="2025-01-01",
        report_schema_version_source="1",
        report_type_source="main",
        is_party_office_source=False,
    )

    assert (
        result[
            "source__id"
        ]
        ==
        "a"
    )

    assert (
        result[
            "source__amount"
        ]
        ==
        10
    )

    assert (
        '"x": 1'
        in
        result[
            "source__nested"
        ]
    )
'''


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


# ============================================================
# 3. RUN ALL TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)

if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "RAW production extraction was not started."
    )


# ============================================================
# 4. IMPORT MODULE
# ============================================================

from politdata.normalization.report_sections import (
    NORMALIZATION_VERSION,
    extract_section_rows,
    normalize_source_row,
)


# ============================================================
# 5. LOAD REPORT CONTEXT
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH
)


if (
    report_context[
        "source_report_id"
    ].nunique()
    !=
    len(
        report_context
    )
):

    raise RuntimeError(
        "report_context source_report_id is not unique."
    )


report_context[
    "source_report_id"
] = (
    report_context[
        "source_report_id"
    ]
    .astype(str)
)


context_by_report = (
    report_context
    .set_index(
        "source_report_id"
    )
    .to_dict(
        orient="index"
    )
)


selected_ids = (
    report_context[
        "source_report_id"
    ]
    .tolist()
)


print()
print("=" * 100)
print("INPUT")
print("=" * 100)

print(
    "Analysis-selected reports:",
    f"{len(selected_ids):,}"
)


# ============================================================
# 6. CONTEXT HELPERS
# ============================================================

def first_context_value(
    context,
    *names,
):

    for name in names:

        if name not in context:
            continue

        value = context[
            name
        ]

        if pd.notna(
            value
        ):

            return value

    return None


# ============================================================
# 7. JSON READER
# ============================================================

try:

    import orjson


    def load_json(
        path,
    ):

        with open(
            path,
            "rb",
        ) as f:

            return orjson.loads(
                f.read()
            )


except ImportError:


    def load_json(
        path,
    ):

        with open(
            path,
            "r",
            encoding="utf-8",
        ) as f:

            return json.load(
                f
            )


# ============================================================
# 8. SECTIONS EXTRACTED IN THIS ONE PASS
# ============================================================

ROW_SECTIONS = [
    "realty",
    "transport",
    "movable",
    "intangible",
    "paper",
    "obligations",
    "head_info",
    "organizations",
    "regional_offices",
]


ALL_TABLE_NAMES = (
    ROW_SECTIONS
    +
    [
        "employee_counts"
    ]
)


# ============================================================
# 9. PROCESS ONE REPORT
# ============================================================

def process_report(
    report_id,
):

    raw_path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    if not raw_path.exists():

        return {
            "report_id":
                report_id,

            "status":
                "missing",

            "rows":
                {},

            "counts":
                {},
        }


    stat = raw_path.stat()


    try:

        payload = load_json(
            raw_path
        )

    except Exception as exc:

        return {
            "report_id":
                report_id,

            "status":
                "json_error",

            "error":
                repr(exc),

            "rows":
                {},

            "counts":
                {},
        }


    detail = (
        payload.get(
            "results"
        )
        if isinstance(
            payload,
            dict,
        )
        else
        None
    )


    if not isinstance(
        detail,
        dict,
    ):

        return {
            "report_id":
                report_id,

            "status":
                "invalid_wrapper",

            "rows":
                {},

            "counts":
                {},
        }


    context = context_by_report[
        report_id
    ]


    organization_id = (
        first_context_value(
            context,
            "organization_id",
        )
    )


    root_party_id = (
        first_context_value(
            context,
            "root_party_id",
        )
    )


    report_year = (
        first_context_value(
            context,
            "report_year",
            "year",
        )
    )


    report_quarter = (
        first_context_value(
            context,
            "report_quarter",
            "quarter",
        )
    )


    signed_date = detail.get(
        "signed_date"
    )


    source_is_signed = bool(
        signed_date
    )


    common_kwargs = {

        "source_report_id":
            report_id,

        "organization_id":
            organization_id,

        "root_party_id":
            root_party_id,

        "report_year":
            report_year,

        "report_quarter":
            report_quarter,

        "source_is_signed":
            source_is_signed,

        "source_signed_date":
            signed_date,

        "report_schema_version_source":
            detail.get(
                "schema_version"
            ),

        "report_type_source":
            detail.get(
                "report_type"
            ),

        "is_party_office_source":
            detail.get(
                "is_party_office"
            ),
    }


    output_rows = {}
    counts = {}


    # --------------------------------------------------------
    # Normal row sections
    # --------------------------------------------------------

    for section in ROW_SECTIONS:

        source_rows = (
            extract_section_rows(
                detail,
                section,
            )
        )


        normalized_rows = []


        for row_index, source_row in enumerate(
            source_rows
        ):

            normalized_rows.append(
                normalize_source_row(
                    source_row,
                    source_section=
                        section,

                    source_row_index=
                        row_index,

                    **common_kwargs,
                )
            )


        output_rows[
            section
        ] = normalized_rows


        counts[
            section
        ] = len(
            normalized_rows
        )


    # --------------------------------------------------------
    # Employee counts:
    # one report-level row
    # --------------------------------------------------------

    civil_value = detail.get(
        "employees_by_civil_contract"
    )

    employment_value = detail.get(
        "employees_by_employment_contract"
    )


    def scalar_or_json(
        value,
    ):

        if isinstance(
            value,
            (
                dict,
                list,
            ),
        ):

            return json.dumps(
                value,
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            )

        return value


    employee_record = {

        "source_report_id":
            str(
                report_id
            ),

        "source_section":
            "employee_counts",

        "source_row_index":
            0,

        "organization_id":
            (
                str(
                    organization_id
                )
                if
                organization_id
                is not None
                else
                None
            ),

        "root_party_id":
            (
                str(
                    root_party_id
                )
                if
                root_party_id
                is not None
                else
                None
            ),

        "report_year":
            report_year,

        "report_quarter":
            report_quarter,

        "source_is_signed":
            source_is_signed,

        "source_signed_date":
            signed_date,

        "report_schema_version_source":
            detail.get(
                "schema_version"
            ),

        "report_type_source":
            detail.get(
                "report_type"
            ),

        "is_party_office_source":
            detail.get(
                "is_party_office"
            ),

        "employees_by_civil_contract_source":
            scalar_or_json(
                civil_value
            ),

        "employees_by_employment_contract_source":
            scalar_or_json(
                employment_value
            ),
    }


    output_rows[
        "employee_counts"
    ] = [
        employee_record
    ]


    counts[
        "employee_counts"
    ] = 1


    return {

        "report_id":
            report_id,

        "status":
            "success",

        "rows":
            output_rows,

        "counts":
            counts,

        "raw_size":
            stat.st_size,

        "raw_mtime_ns":
            stat.st_mtime_ns,
    }


# ============================================================
# 10. ONE PRODUCTION RAW PASS
# ============================================================

tables = {
    name: []
    for name
    in ALL_TABLE_NAMES
}


state_rows = []

missing = []
invalid = []


print()
print("=" * 100)
print("ONE-TIME PRODUCTION RAW PASS")
print("=" * 100)

print(
    "RAW files to open:",
    f"{len(selected_ids):,}"
)

print(
    "Payments will NOT be read or modified."
)

print(
    "property_moneys will NOT be re-extracted."
)


with ThreadPoolExecutor(
    max_workers=16
) as executor:

    results = executor.map(
        process_report,
        selected_ids,
    )


    for item in tqdm(
        results,
        total=len(
            selected_ids
        ),
        desc="Report structural sections",
    ):

        status = item[
            "status"
        ]


        if status == "missing":

            missing.append(
                item[
                    "report_id"
                ]
            )

            continue


        if status != "success":

            invalid.append(
                {
                    "report_id":
                        item[
                            "report_id"
                        ],

                    "status":
                        status,

                    "error":
                        item.get(
                            "error"
                        ),
                }
            )

            continue


        for name in ALL_TABLE_NAMES:

            tables[
                name
            ].extend(
                item[
                    "rows"
                ].get(
                    name,
                    [],
                )
            )


        state_record = {

            "source_report_id":
                item[
                    "report_id"
                ],

            "raw_size":
                item[
                    "raw_size"
                ],

            "raw_mtime_ns":
                item[
                    "raw_mtime_ns"
                ],

            "normalization_version":
                NORMALIZATION_VERSION,
        }


        for name in ALL_TABLE_NAMES:

            state_record[
                f"{name}_count"
            ] = (
                item[
                    "counts"
                ].get(
                    name,
                    0,
                )
            )


        state_rows.append(
            state_record
        )


# ============================================================
# 11. HARD RAW COVERAGE QA
# ============================================================

print()
print("=" * 100)
print("RAW COVERAGE QA")
print("=" * 100)

print(
    "Missing RAW:",
    len(
        missing
    )
)

print(
    "Invalid RAW:",
    len(
        invalid
    )
)


if missing:

    print(
        "First missing IDs:",
        missing[:20],
    )


if invalid:

    display(
        pd.DataFrame(
            invalid
        ).head(
            50
        )
    )


if (
    missing
    or
    invalid
):

    raise RuntimeError(
        "RAW coverage is incomplete. "
        "No normalized section parquet has been written."
    )


if len(
    state_rows
) != len(
    selected_ids
):

    raise RuntimeError(
        "Processed-report count differs from selected manifest."
    )


# ============================================================
# 12. DATAFRAME STABILIZER
#
# Dynamic source schemas occasionally mix scalar types.
# If a source column contains incompatible Python types,
# cast only that source column to pandas string.
# ============================================================

BASE_COLUMNS = [
    "source_report_id",
    "source_section",
    "source_row_index",
    "organization_id",
    "root_party_id",
    "report_year",
    "report_quarter",
    "source_is_signed",
    "source_signed_date",
    "report_schema_version_source",
    "report_type_source",
    "is_party_office_source",
]


def stabilize_dataframe(
    rows,
):

    if not rows:

        return pd.DataFrame(
            columns=BASE_COLUMNS
        )


    df = pd.DataFrame(
        rows
    )


    for column in df.columns:

        if (
            not column.startswith(
                "source__"
            )
        ):

            continue


        nonnull = (
            df[
                column
            ]
            .dropna()
        )


        if nonnull.empty:

            continue


        type_names = {
            type(value).__name__
            for value
            in nonnull.head(
                10000
            )
        }


        # Mixed source scalar representations:
        # preserve them textually rather than letting
        # Arrow guess or fail.
        if len(
            type_names
        ) > 1:

            df[
                column
            ] = (
                df[
                    column
                ]
                .astype(
                    "string"
                )
            )


    return df


normalized_dfs = {

    name:
        stabilize_dataframe(
            rows
        )

    for name, rows
    in tables.items()
}


# ============================================================
# 13. NORMALIZED EXTRACTION QA
# ============================================================

qa_records = []


for name, df in normalized_dfs.items():

    qa_records.append(
        {

            "section":
                name,

            "rows":
                len(
                    df
                ),

            "reports":
                (
                    df[
                        "source_report_id"
                    ].nunique()
                    if
                    "source_report_id"
                    in df.columns
                    and
                    len(df)
                    else
                    0
                ),

            "organizations":
                (
                    df[
                        "organization_id"
                    ].nunique()
                    if
                    "organization_id"
                    in df.columns
                    and
                    len(df)
                    else
                    0
                ),

            "columns":
                len(
                    df.columns
                ),

            "source_columns":
                sum(
                    column.startswith(
                        "source__"
                    )
                    for column
                    in df.columns
                ),
        }
    )


section_qa = (
    pd.DataFrame(
        qa_records
    )
    .sort_values(
        "section"
    )
)


print()
print("=" * 100)
print("NORMALIZED SECTION QA")
print("=" * 100)

display(
    section_qa
)


# ============================================================
# 14. SOURCE SCHEMA SUMMARY
#
# No hardcoded "unknown fields":
# source columns are preserved dynamically.
# ============================================================

schema_records = []


for name, df in normalized_dfs.items():

    for column in df.columns:

        if not column.startswith(
            "source__"
        ):

            continue


        schema_records.append(
            {
                "section":
                    name,

                "source_column":
                    column,

                "non_null_rows":
                    int(
                        df[
                            column
                        ].notna().sum()
                    ),

                "dtype":
                    str(
                        df[
                            column
                        ].dtype
                    ),
            }
        )


schema_qa = pd.DataFrame(
    schema_records
)


print()
print("=" * 100)
print("SOURCE SCHEMA SUMMARY")
print("=" * 100)


if len(
    schema_qa
):

    display(
        schema_qa.sort_values(
            [
                "section",
                "non_null_rows",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )

else:

    print(
        "No dynamic source fields found."
    )


# ============================================================
# 15. BUILD LATEST-REPORT MAP
#
# Latest AVAILABLE analysis-selected report
# per organization.
#
# Quarter order:
# Q1 < Q2 < Q3 < Q4 < annual(Q5)
# ============================================================

ctx = report_context.copy()


year_col = (
    "report_year"
    if
    "report_year"
    in ctx.columns
    else
    "year"
)


quarter_col = (
    "report_quarter"
    if
    "report_quarter"
    in ctx.columns
    else
    "quarter"
)


ctx[
    "_year_rank"
] = pd.to_numeric(
    ctx[
        year_col
    ],
    errors="coerce",
)


ctx[
    "_quarter_rank"
] = (
    pd.to_numeric(
        ctx[
            quarter_col
        ],
        errors="coerce",
    )
    .fillna(0)
)


ctx = ctx.sort_values(
    [
        "organization_id",
        "_year_rank",
        "_quarter_rank",
        "source_report_id",
    ]
)


latest_by_org = (
    ctx
    .groupby(
        "organization_id",
        dropna=False,
    )
    .tail(1)
)


latest_report_ids = set(
    latest_by_org[
        "source_report_id"
    ].astype(str)
)


print()
print("=" * 100)
print("LATEST REPORT MAP")
print("=" * 100)

print(
    "Organizations:",
    len(
        latest_by_org
    )
)

print(
    "Latest report IDs:",
    len(
        latest_report_ids
    )
)


# ============================================================
# 16. REPORT PERIOD HELPER
# ============================================================

def report_period_label(
    year,
    quarter,
):

    if pd.isna(
        year
    ):

        return None


    try:

        year = int(
            year
        )

    except Exception:

        return None


    try:

        quarter = int(
            quarter
        )

    except Exception:

        quarter = None


    if quarter == 5:

        return str(
            year
        )


    if quarter in (
        1,
        2,
        3,
        4,
    ):

        return (
            f"{year}Q{quarter}"
        )


    return str(
        year
    )


# ============================================================
# 17. ENRICHMENT FUNCTION
#
# Adds current organization/party identity,
# region, override provenance and latest/historical status.
# ============================================================

CONTEXT_CANDIDATES = [
    "organization_name_current",
    "organization_code",
    "organization_level",
    "region",

    "party_name_current",
    "party_code",

    "analysis_override",
    "analysis_selection_method",

    "official_selected_report_id",
    "analysis_selected_report_id",

    "continuity_exact",
]


available_context_columns = [
    column
    for column
    in CONTEXT_CANDIDATES
    if column
    in report_context.columns
]


merge_context = (
    report_context[
        [
            "source_report_id",
            *available_context_columns,
        ]
    ]
    .copy()
)


def enrich_table(
    df,
):

    enriched = (
        df
        .merge(
            merge_context,
            on="source_report_id",
            how="left",
            validate="many_to_one",
        )
    )


    enriched[
        "analysis_selected"
    ] = True


    if (
        "analysis_override"
        in enriched.columns
    ):

        enriched[
            "official_selected"
        ] = (
            ~
            enriched[
                "analysis_override"
            ]
            .fillna(
                False
            )
            .astype(bool)
        )

    else:

        enriched[
            "official_selected"
        ] = True


    enriched[
        "is_latest_data"
    ] = (
        enriched[
            "source_report_id"
        ]
        .astype(str)
        .isin(
            latest_report_ids
        )
    )


    enriched[
        "data_recency_status"
    ] = (
        enriched[
            "is_latest_data"
        ]
        .map(
            {
                True:
                    "latest_data",

                False:
                    "historical_data",
            }
        )
    )


    enriched[
        "report_period"
    ] = [
        report_period_label(
            year,
            quarter,
        )
        for year, quarter
        in zip(
            enriched[
                "report_year"
            ],
            enriched[
                "report_quarter"
            ],
        )
    ]


    return enriched


enriched_dfs = {

    name:
        enrich_table(
            df
        )

    for name, df
    in normalized_dfs.items()
}


# ============================================================
# 18. ENRICHED QA
# ============================================================

enriched_qa_records = []


for name, df in enriched_dfs.items():

    enriched_qa_records.append(
        {

            "section":
                name,

            "rows":
                len(
                    df
                ),

            "latest_rows":
                int(
                    df[
                        "is_latest_data"
                    ].sum()
                )
                if
                len(df)
                else
                0,

            "historical_rows":
                int(
                    (
                        ~
                        df[
                            "is_latest_data"
                        ]
                    ).sum()
                )
                if
                len(df)
                else
                0,

            "analysis_override_rows":
                int(
                    df[
                        "analysis_override"
                    ]
                    .fillna(
                        False
                    )
                    .sum()
                )
                if
                (
                    len(df)
                    and
                    "analysis_override"
                    in df.columns
                )
                else
                0,
        }
    )


enriched_qa = pd.DataFrame(
    enriched_qa_records
)


print()
print("=" * 100)
print("ENRICHED SECTION QA")
print("=" * 100)

display(
    enriched_qa
)


# ============================================================
# 19. WRITE NORMALIZED + ENRICHED PARQUET
#
# Temp file -> replace.
# ============================================================

for name in ALL_TABLE_NAMES:

    normalized_path = (
        TABLE_PATHS[
            name
        ]
    )

    enriched_path = (
        ENRICHED_TABLE_PATHS[
            name
        ]
    )


    normalized_tmp = (
        normalized_path
        .with_suffix(
            ".tmp.parquet"
        )
    )

    enriched_tmp = (
        enriched_path
        .with_suffix(
            ".tmp.parquet"
        )
    )


    normalized_dfs[
        name
    ].to_parquet(
        normalized_tmp,
        index=False,
    )


    enriched_dfs[
        name
    ].to_parquet(
        enriched_tmp,
        index=False,
    )


    normalized_tmp.replace(
        normalized_path
    )


    enriched_tmp.replace(
        enriched_path
    )


# ============================================================
# 20. NORMALIZATION STATE
#
# This records section counts so future work no longer needs
# another full RAW scan merely to discover whether a section
# is populated.
# ============================================================

section_state = pd.DataFrame(
    state_rows
)


section_state[
    "normalized_at"
] = pd.Timestamp.now(
    tz="UTC"
)


# Attach existing report-detail hashes if available.
if DETAIL_STATE_PATH.exists():

    detail_state = pd.read_parquet(
        DETAIL_STATE_PATH
    )


    report_id_column = next(
        (
            column
            for column
            in (
                "report_id",
                "source_report_id",
                "id",
            )
            if column
            in detail_state.columns
        ),
        None,
    )


    hash_candidates = [
        column
        for column
        in (
            "raw_content_hash",
            "semantic_content_hash",
            "content_hash",
            "raw_hash",
        )
        if column
        in detail_state.columns
    ]


    if (
        report_id_column
        is not None
    ):

        hash_df = (
            detail_state[
                [
                    report_id_column,
                    *hash_candidates,
                ]
            ]
            .copy()
        )


        hash_df[
            report_id_column
        ] = (
            hash_df[
                report_id_column
            ]
            .astype(str)
        )


        hash_df = (
            hash_df
            .drop_duplicates(
                subset=[
                    report_id_column
                ],
                keep="last",
            )
            .rename(
                columns={
                    report_id_column:
                        "source_report_id"
                }
            )
        )


        section_state = (
            section_state
            .merge(
                hash_df,
                on="source_report_id",
                how="left",
                validate="one_to_one",
            )
        )


section_state.to_parquet(
    SECTION_STATE_PATH,
    index=False,
)


# ============================================================
# 21. STATE COUNT SUMMARY
# ============================================================

count_columns = [
    f"{name}_count"
    for name
    in ALL_TABLE_NAMES
]


state_summary = []


for name in ALL_TABLE_NAMES:

    column = (
        f"{name}_count"
    )


    state_summary.append(
        {

            "section":
                name,

            "reports_with_rows":
                int(
                    (
                        section_state[
                            column
                        ]
                        >
                        0
                    ).sum()
                ),

            "total_rows":
                int(
                    section_state[
                        column
                    ].sum()
                ),

            "max_rows_per_report":
                int(
                    section_state[
                        column
                    ].max()
                ),
        }
    )


state_summary = pd.DataFrame(
    state_summary
)


print()
print("=" * 100)
print("SECTION COVERAGE MATRIX")
print("=" * 100)

display(
    state_summary.sort_values(
        "section"
    )
)


# ============================================================
# 22. REGRESSION: PAYMENTS / PROPERTY_MONEYS UNTOUCHED
# ============================================================

payment_dir = (
    ENRICHED_ROOT
    / "payments"
)


payment_counts = {}


for name in [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]:

    path = (
        payment_dir
        / f"{name}.parquet"
    )

    payment_counts[
        name
    ] = len(
        pd.read_parquet(
            path,
            columns=[
                "source_report_id"
            ],
        )
    )


payment_total = sum(
    payment_counts.values()
)


property_moneys_path = (
    NORMALIZED_ROOT
    / "properties"
    / "property_moneys.parquet"
)


property_moneys_rows = len(
    pd.read_parquet(
        property_moneys_path,
        columns=[
            "source_report_id"
        ],
    )
)


print()
print("=" * 100)
print("UNCHANGED EXISTING TABLES CONTROL")
print("=" * 100)

print(
    "Payment rows:",
    f"{payment_total:,}"
)

print(
    "property_moneys rows:",
    f"{property_moneys_rows:,}"
)


if payment_total != 402_028:

    raise RuntimeError(
        f"Payment regression: "
        f"expected 402,028, got {payment_total:,}"
    )


if property_moneys_rows != 19_140:

    raise RuntimeError(
        f"property_moneys regression: "
        f"expected 19,140, got {property_moneys_rows:,}"
    )


# ============================================================
# 23. FILES
# ============================================================

print()
print("=" * 100)
print("OUTPUT FILES")
print("=" * 100)


for name in ALL_TABLE_NAMES:

    print()
    print(
        name
    )

    print(
        "  normalized:",
        TABLE_PATHS[
            name
        ]
    )

    print(
        "  enriched:  ",
        ENRICHED_TABLE_PATHS[
            name
        ]
    )


print()
print(
    "state:",
    SECTION_STATE_PATH
)


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "All analysis-selected RAW report details were read "
    "once for production structural normalization."
)

print(
    "Payments were not modified."
)

print(
    "property_moneys was not modified."
)

print(
    "Future section-coverage checks should use "
    "report_section_normalization_state.parquet, "
    "not another full RAW scan."
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\normalization\report_sections.py

TESTS
.................................................................        [100%]
65 passed in 12.38s

Return code: 0

INPUT
Analysis-selected reports: 78,791

ONE-TIME PRODUCTION RAW PASS
RAW files to open: 78,791
Payments will NOT be read or modified.
property_moneys will NOT be re-extracted.


Report structural sections:   0%|          | 0/78791 [00:00<?, ?it/s]


RAW COVERAGE QA
Missing RAW: 0
Invalid RAW: 0

NORMALIZED SECTION QA


,section,rows,reports,organizations,columns,source_columns
9,employee_counts,78791,78791,8514,14,0
6,head_info,78791,78791,8514,16,3
3,intangible,0,0,0,12,0
2,movable,0,0,0,12,0
5,obligations,26132,2378,407,27,14
7,organizations,308,110,19,15,2
4,paper,0,0,0,12,0
0,realty,0,0,0,12,0
8,regional_offices,102678,4500,503,29,16
1,transport,0,0,0,12,0



SOURCE SCHEMA SUMMARY


,section,source_column,non_null_rows,dtype
14,head_info,source__name,78791,object
15,head_info,source__surname,78791,object
16,head_info,source__patronymic,75290,object
0,obligations,source__id,26132,object
1,obligations,source__party_id,26132,object
3,obligations,source__report_status,26132,int64
4,obligations,source__object_type,26132,object
5,obligations,source__person_type,26132,object
6,obligations,source__person_name,26132,object
7,obligations,source__person_code,26132,object



LATEST REPORT MAP
Organizations: 8514
Latest report IDs: 8514

ENRICHED SECTION QA


,section,rows,latest_rows,historical_rows,analysis_override_rows
0,realty,0,0,0,0
1,transport,0,0,0,0
2,movable,0,0,0,0
3,intangible,0,0,0,0
4,paper,0,0,0,0
5,obligations,26132,1334,24798,3
6,head_info,78791,8514,70277,1
7,organizations,308,42,266,0
8,regional_offices,102678,8404,94274,0
9,employee_counts,78791,8514,70277,1


ArrowInvalid: ("Could not convert '0' with type str: tried to convert to int64", 'Conversion failed for column employees_by_civil_contract_source with type object')

In [119]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import importlib
import json
import re

import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# PRECONDITION
# ============================================================

required_globals = [
    "normalized_dfs",
    "enriched_dfs",
    "state_rows",
    "selected_ids",
    "report_context",
    "context_by_report",
    "RAW_DIR",
    "TABLE_PATHS",
    "ENRICHED_TABLE_PATHS",
    "SECTION_STATE_PATH",
    "MODULE_PATH",
    "load_json",
    "first_context_value",
    "normalize_source_row",
    "NORMALIZATION_VERSION",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "This continuation cell requires the same kernel. "
        f"Missing variables: {missing_globals}"
    )


# ============================================================
# 1. FIX employee_counts TYPE MIX
#
# API has values such as 0 and "0".
# Normalize them to nullable integer.
# ============================================================

EMPLOYEE_COLUMNS = [
    "employees_by_civil_contract_source",
    "employees_by_employment_contract_source",
]


for layer_name, df in [
    (
        "normalized",
        normalized_dfs["employee_counts"],
    ),
    (
        "enriched",
        enriched_dfs["employee_counts"],
    ),
]:

    for column in EMPLOYEE_COLUMNS:

        original = df[column]

        numeric = pd.to_numeric(
            original,
            errors="coerce",
        )

        bad = (
            original.notna()
            &
            numeric.isna()
        )

        if bad.any():

            print(
                f"{layer_name} / {column}: "
                "non-numeric source values:"
            )

            print(
                original[
                    bad
                ]
                .astype(str)
                .value_counts()
                .head(50)
            )

            raise RuntimeError(
                f"Unexpected non-numeric employee values "
                f"in {column}."
            )

        df[column] = (
            numeric.astype(
                "Int64"
            )
        )


print()
print("=" * 100)
print("EMPLOYEE COUNT TYPE QA")
print("=" * 100)

for column in EMPLOYEE_COLUMNS:

    print()
    print(column)

    print(
        normalized_dfs[
            "employee_counts"
        ][column]
        .value_counts(
            dropna=False
        )
        .head(20)
    )


# ============================================================
# 2. WRITE employee_counts NOW
#
# The previous cell failed before this final table was written.
# ============================================================

for path, df in [
    (
        TABLE_PATHS[
            "employee_counts"
        ],
        normalized_dfs[
            "employee_counts"
        ],
    ),
    (
        ENRICHED_TABLE_PATHS[
            "employee_counts"
        ],
        enriched_dfs[
            "employee_counts"
        ],
    ),
]:

    tmp = path.with_suffix(
        ".tmp.parquet"
    )

    df.to_parquet(
        tmp,
        index=False,
    )

    tmp.replace(
        path
    )


print()
print(
    "employee_counts written successfully."
)


# ============================================================
# 3. VERIFY ALREADY-WRITTEN NON-PROPERTY TABLES
#
# Do NOT re-read RAW for them.
# ============================================================

GOOD_EXISTING = [
    "obligations",
    "head_info",
    "organizations",
    "regional_offices",
    "employee_counts",
]


existing_qa = []


for name in GOOD_EXISTING:

    normalized_path = (
        TABLE_PATHS[name]
    )

    enriched_path = (
        ENRICHED_TABLE_PATHS[name]
    )

    if not normalized_path.exists():
        raise FileNotFoundError(
            normalized_path
        )

    if not enriched_path.exists():
        raise FileNotFoundError(
            enriched_path
        )


    n_rows = len(
        pd.read_parquet(
            normalized_path,
            columns=[
                "source_report_id"
            ],
        )
    )

    e_rows = len(
        pd.read_parquet(
            enriched_path,
            columns=[
                "source_report_id"
            ],
        )
    )


    if n_rows != e_rows:
        raise RuntimeError(
            f"{name}: normalized/enriched mismatch "
            f"{n_rows} != {e_rows}"
        )


    existing_qa.append(
        {
            "section": name,
            "rows": n_rows,
        }
    )


print()
print("=" * 100)
print("PERSISTED NON-PROPERTY SECTIONS")
print("=" * 100)

display(
    pd.DataFrame(
        existing_qa
    )
)


# ============================================================
# 4. CHEAP PROPERTY-KEY PROBE
#
# Read only a small number of RAW files.
# We need exact keys under detail["properties"].
# ============================================================

probe_ids = selected_ids[:50]


property_key_stats = {}


for report_id in probe_ids:

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )

    payload = load_json(
        path
    )

    detail = payload.get(
        "results",
        {}
    )

    properties = detail.get(
        "properties"
    )

    if not isinstance(
        properties,
        dict,
    ):
        continue


    for key, value in properties.items():

        record = property_key_stats.setdefault(
            key,
            {
                "reports_seen": 0,
                "list_values": 0,
                "dict_values": 0,
                "other_values": 0,
                "nonempty_values": 0,
            },
        )


        record[
            "reports_seen"
        ] += 1


        if isinstance(
            value,
            list,
        ):

            record[
                "list_values"
            ] += 1

            if len(value):
                record[
                    "nonempty_values"
                ] += 1


        elif isinstance(
            value,
            dict,
        ):

            record[
                "dict_values"
            ] += 1

            if len(value):
                record[
                    "nonempty_values"
                ] += 1


        else:

            record[
                "other_values"
            ] += 1

            if value not in (
                None,
                "",
            ):
                record[
                    "nonempty_values"
                ] += 1


property_key_qa = (
    pd.DataFrame.from_dict(
        property_key_stats,
        orient="index",
    )
    .reset_index(
        names="property_key"
    )
    .sort_values(
        "property_key"
    )
)


print()
print("=" * 100)
print("PROPERTY KEYS FOUND IN RAW")
print("=" * 100)

display(
    property_key_qa
)


# ============================================================
# 5. AUTOMATIC CANONICAL KEY DETECTION
# ============================================================

def normalize_key(
    value,
):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


PROPERTY_KEY_PATTERNS = {

    "realty": [
        "realty",
        "realties",
        "realestate",
        "realestates",
    ],

    "transport": [
        "transport",
        "transports",
        "vehicle",
        "vehicles",
    ],

    "movable": [
        "movable",
        "movables",
    ],

    "intangible": [
        "intangible",
        "intangibles",
    ],

    "paper": [
        "paper",
        "papers",
        "security",
        "securities",
    ],
}


detected_keys = {}


all_property_keys = list(
    property_key_stats
)


for canonical, patterns in (
    PROPERTY_KEY_PATTERNS.items()
):

    matches = []


    for source_key in all_property_keys:

        normalized = normalize_key(
            source_key
        )


        if any(
            pattern
            in normalized
            for pattern
            in patterns
        ):

            matches.append(
                source_key
            )


    if len(matches) != 1:

        print()
        print(
            f"{canonical}: candidates = {matches}"
        )

        raise RuntimeError(
            "Property key could not be resolved "
            f"uniquely for {canonical}. "
            "Stop here; do NOT run another full RAW pass."
        )


    detected_keys[
        canonical
    ] = matches[0]


print()
print("=" * 100)
print("DETECTED PROPERTY SECTION MAP")
print("=" * 100)

for canonical, source_key in (
    detected_keys.items()
):

    print(
        f"{canonical:12s} -> properties[{source_key!r}]"
    )


# ============================================================
# 6. PATCH PERMANENT MODULE FOR FUTURE SYNC
# ============================================================

module_text = MODULE_PATH.read_text(
    encoding="utf-8"
)


for canonical, source_key in (
    detected_keys.items()
):

    pattern = (
        rf'"{canonical}"\s*:\s*'
        rf'\("properties",\s*"[^"]+"\)'
    )

    replacement = (
        f'"{canonical}": '
        f'("properties", "{source_key}")'
    )


    module_text, replacements = re.subn(
        pattern,
        replacement,
        module_text,
        count=1,
    )


    if replacements != 1:

        raise RuntimeError(
            f"Could not patch SECTION_PATHS for {canonical}"
        )


MODULE_PATH.write_text(
    module_text,
    encoding="utf-8",
)


import politdata.normalization.report_sections as rs_module

importlib.reload(
    rs_module
)


print()
print(
    "Permanent SECTION_PATHS updated."
)


# ============================================================
# 7. PROPERTY-ONLY RAW EXTRACTION
#
# Unfortunately the first pass used wrong paths, so the
# property rows themselves were not retained.
#
# We now re-open RAW ONLY to recover these five sections.
# No obligations/head_info/etc are processed again.
# ============================================================

PROPERTY_SECTIONS = [
    "realty",
    "transport",
    "movable",
    "intangible",
    "paper",
]


def coerce_rows(
    value,
):

    return rs_module.coerce_rows(
        value
    )


def process_property_report(
    report_id,
):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    payload = load_json(
        path
    )


    detail = payload.get(
        "results"
    )


    if not isinstance(
        detail,
        dict,
    ):

        return {
            "report_id": report_id,
            "status": "invalid",
            "rows": {},
            "counts": {},
        }


    properties = detail.get(
        "properties"
    )


    if not isinstance(
        properties,
        dict,
    ):

        properties = {}


    context = context_by_report[
        report_id
    ]


    organization_id = (
        first_context_value(
            context,
            "organization_id",
        )
    )


    root_party_id = (
        first_context_value(
            context,
            "root_party_id",
        )
    )


    report_year = (
        first_context_value(
            context,
            "report_year",
            "year",
        )
    )


    report_quarter = (
        first_context_value(
            context,
            "report_quarter",
            "quarter",
        )
    )


    signed_date = detail.get(
        "signed_date"
    )


    common_kwargs = {

        "source_report_id":
            report_id,

        "organization_id":
            organization_id,

        "root_party_id":
            root_party_id,

        "report_year":
            report_year,

        "report_quarter":
            report_quarter,

        "source_is_signed":
            bool(
                signed_date
            ),

        "source_signed_date":
            signed_date,

        "report_schema_version_source":
            detail.get(
                "schema_version"
            ),

        "report_type_source":
            detail.get(
                "report_type"
            ),

        "is_party_office_source":
            detail.get(
                "is_party_office"
            ),
    }


    output = {}
    counts = {}


    for section in PROPERTY_SECTIONS:

        source_key = (
            detected_keys[
                section
            ]
        )


        source_rows = coerce_rows(
            properties.get(
                source_key
            )
        )


        normalized_rows = []


        for row_index, source_row in enumerate(
            source_rows
        ):

            normalized_rows.append(
                normalize_source_row(
                    source_row,
                    source_section=
                        section,

                    source_row_index=
                        row_index,

                    **common_kwargs,
                )
            )


        output[
            section
        ] = normalized_rows


        counts[
            section
        ] = len(
            normalized_rows
        )


    return {
        "report_id":
            report_id,

        "status":
            "success",

        "rows":
            output,

        "counts":
            counts,
    }


property_rows = {
    section: []
    for section
    in PROPERTY_SECTIONS
}


property_count_by_report = {}


print()
print("=" * 100)
print("PROPERTY-ONLY CORRECTIVE RAW PASS")
print("=" * 100)

print(
    "Reports:",
    f"{len(selected_ids):,}"
)


with ThreadPoolExecutor(
    max_workers=16
) as executor:

    results = executor.map(
        process_property_report,
        selected_ids,
    )


    for item in tqdm(
        results,
        total=len(
            selected_ids
        ),
        desc="Property sections only",
    ):

        if (
            item[
                "status"
            ]
            !=
            "success"
        ):

            raise RuntimeError(
                "Invalid report encountered: "
                f"{item['report_id']}"
            )


        property_count_by_report[
            item[
                "report_id"
            ]
        ] = item[
            "counts"
        ]


        for section in PROPERTY_SECTIONS:

            property_rows[
                section
            ].extend(
                item[
                    "rows"
                ][
                    section
                ]
            )


# ============================================================
# 8. ROBUST PARQUET TYPE STABILIZATION
# ============================================================

def stabilize_for_parquet(
    df,
):

    df = df.copy()


    for column in df.columns:

        if str(
            df[
                column
            ].dtype
        ) != "object":

            continue


        nonnull = (
            df[
                column
            ]
            .dropna()
        )


        if nonnull.empty:
            continue


        python_types = (
            nonnull
            .map(type)
            .nunique()
        )


        if python_types > 1:

            df[
                column
            ] = (
                df[
                    column
                ]
                .astype(
                    "string"
                )
            )


    return df


property_dfs = {}


for section in PROPERTY_SECTIONS:

    df = pd.DataFrame(
        property_rows[
            section
        ]
    )


    if df.empty:

        df = pd.DataFrame(
            columns=[
                "source_report_id",
                "source_section",
                "source_row_index",
                "organization_id",
                "root_party_id",
                "report_year",
                "report_quarter",
                "source_is_signed",
                "source_signed_date",
                "report_schema_version_source",
                "report_type_source",
                "is_party_office_source",
            ]
        )


    property_dfs[
        section
    ] = stabilize_for_parquet(
        df
    )


# ============================================================
# 9. PROPERTY EXTRACTION QA
# ============================================================

property_qa = []


for section, df in (
    property_dfs.items()
):

    property_qa.append(
        {
            "section":
                section,

            "rows":
                len(df),

            "reports":
                (
                    df[
                        "source_report_id"
                    ].nunique()
                    if
                    len(df)
                    else
                    0
                ),

            "organizations":
                (
                    df[
                        "organization_id"
                    ].nunique()
                    if
                    len(df)
                    else
                    0
                ),

            "columns":
                len(
                    df.columns
                ),

            "source_columns":
                sum(
                    col.startswith(
                        "source__"
                    )
                    for col
                    in df.columns
                ),
        }
    )


print()
print("=" * 100)
print("CORRECTED PROPERTY SECTION QA")
print("=" * 100)

display(
    pd.DataFrame(
        property_qa
    )
)


# ============================================================
# 10. REBUILD LATEST REPORT MAP
# ============================================================

ctx = report_context.copy()


year_col = (
    "report_year"
    if
    "report_year"
    in ctx.columns
    else
    "year"
)

quarter_col = (
    "report_quarter"
    if
    "report_quarter"
    in ctx.columns
    else
    "quarter"
)


ctx[
    "_year_rank"
] = pd.to_numeric(
    ctx[
        year_col
    ],
    errors="coerce",
)


ctx[
    "_quarter_rank"
] = (
    pd.to_numeric(
        ctx[
            quarter_col
        ],
        errors="coerce",
    )
    .fillna(0)
)


latest_by_org = (
    ctx
    .sort_values(
        [
            "organization_id",
            "_year_rank",
            "_quarter_rank",
            "source_report_id",
        ]
    )
    .groupby(
        "organization_id",
        dropna=False,
    )
    .tail(1)
)


latest_report_ids = set(
    latest_by_org[
        "source_report_id"
    ]
    .astype(str)
)


# ============================================================
# 11. ENRICH PROPERTY TABLES
# ============================================================

CONTEXT_COLUMNS = [
    "organization_name_current",
    "organization_code",
    "organization_level",
    "region",

    "party_name_current",
    "party_code",

    "analysis_override",
    "analysis_selection_method",

    "official_selected_report_id",
    "analysis_selected_report_id",

    "continuity_exact",
]


available_context_columns = [
    col
    for col in CONTEXT_COLUMNS
    if col in report_context.columns
]


merge_context = (
    report_context[
        [
            "source_report_id",
            *available_context_columns,
        ]
    ]
    .copy()
)


def report_period_label(
    year,
    quarter,
):

    if pd.isna(year):
        return None


    try:
        year = int(year)

    except Exception:
        return None


    try:
        quarter = int(quarter)

    except Exception:
        quarter = None


    if quarter == 5:
        return str(year)


    if quarter in (
        1,
        2,
        3,
        4,
    ):

        return (
            f"{year}Q{quarter}"
        )


    return str(year)


def enrich_property_table(
    df,
):

    enriched = (
        df
        .merge(
            merge_context,
            on="source_report_id",
            how="left",
            validate="many_to_one",
        )
    )


    enriched[
        "analysis_selected"
    ] = True


    if (
        "analysis_override"
        in enriched.columns
    ):

        enriched[
            "official_selected"
        ] = (
            ~
            enriched[
                "analysis_override"
            ]
            .fillna(False)
            .astype(bool)
        )

    else:

        enriched[
            "official_selected"
        ] = True


    enriched[
        "is_latest_data"
    ] = (
        enriched[
            "source_report_id"
        ]
        .astype(str)
        .isin(
            latest_report_ids
        )
    )


    enriched[
        "data_recency_status"
    ] = enriched[
        "is_latest_data"
    ].map(
        {
            True:
                "latest_data",

            False:
                "historical_data",
        }
    )


    enriched[
        "report_period"
    ] = [
        report_period_label(
            year,
            quarter,
        )

        for year, quarter

        in zip(
            enriched[
                "report_year"
            ],
            enriched[
                "report_quarter"
            ],
        )
    ]


    return stabilize_for_parquet(
        enriched
    )


property_enriched_dfs = {

    section:
        enrich_property_table(
            df
        )

    for section, df
    in property_dfs.items()
}


# ============================================================
# 12. WRITE CORRECTED PROPERTY TABLES
# ============================================================

for section in PROPERTY_SECTIONS:

    normalized_path = (
        TABLE_PATHS[
            section
        ]
    )

    enriched_path = (
        ENRICHED_TABLE_PATHS[
            section
        ]
    )


    normalized_tmp = (
        normalized_path
        .with_suffix(
            ".tmp.parquet"
        )
    )

    enriched_tmp = (
        enriched_path
        .with_suffix(
            ".tmp.parquet"
        )
    )


    property_dfs[
        section
    ].to_parquet(
        normalized_tmp,
        index=False,
    )


    property_enriched_dfs[
        section
    ].to_parquet(
        enriched_tmp,
        index=False,
    )


    normalized_tmp.replace(
        normalized_path
    )

    enriched_tmp.replace(
        enriched_path
    )


# ============================================================
# 13. PATCH SECTION STATE WITH REAL PROPERTY COUNTS
# ============================================================

state_index = {
    row[
        "source_report_id"
    ]:
        row

    for row
    in state_rows
}


for report_id, counts in (
    property_count_by_report.items()
):

    state_record = (
        state_index[
            report_id
        ]
    )


    for section in PROPERTY_SECTIONS:

        state_record[
            f"{section}_count"
        ] = counts[
            section
        ]


section_state = pd.DataFrame(
    state_rows
)


section_state[
    "normalized_at"
] = pd.Timestamp.now(
    tz="UTC"
)


section_state.to_parquet(
    SECTION_STATE_PATH,
    index=False,
)


# ============================================================
# 14. FINAL COVERAGE MATRIX
# ============================================================

ALL_SECTION_NAMES = [
    "realty",
    "transport",
    "movable",
    "intangible",
    "paper",
    "obligations",
    "head_info",
    "organizations",
    "regional_offices",
    "employee_counts",
]


coverage = []


for section in ALL_SECTION_NAMES:

    count_col = (
        f"{section}_count"
    )


    coverage.append(
        {
            "section":
                section,

            "reports_with_rows":
                int(
                    (
                        section_state[
                            count_col
                        ]
                        >
                        0
                    ).sum()
                ),

            "total_rows":
                int(
                    section_state[
                        count_col
                    ].sum()
                ),

            "max_rows_per_report":
                int(
                    section_state[
                        count_col
                    ].max()
                ),
        }
    )


coverage = (
    pd.DataFrame(
        coverage
    )
    .sort_values(
        "section"
    )
)


print()
print("=" * 100)
print("FINAL SECTION COVERAGE MATRIX")
print("=" * 100)

display(
    coverage
)


# ============================================================
# 15. FINAL EXISTING-DATA REGRESSION
# ============================================================

payment_dir = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)


payment_total = 0


for name in [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]:

    payment_total += len(
        pd.read_parquet(
            payment_dir
            / f"{name}.parquet",

            columns=[
                "source_report_id"
            ],
        )
    )


property_moneys_path = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "properties"
    / "property_moneys.parquet"
)


property_moneys_rows = len(
    pd.read_parquet(
        property_moneys_path,
        columns=[
            "source_report_id"
        ],
    )
)


print()
print("=" * 100)
print("REGRESSION CONTROL")
print("=" * 100)

print(
    "Payments:",
    f"{payment_total:,}"
)

print(
    "property_moneys:",
    f"{property_moneys_rows:,}"
)


assert payment_total == 402_028

assert property_moneys_rows == 19_140


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Existing non-property sections were preserved."
)

print(
    "employee_counts serialization was fixed."
)

print(
    "Property section keys were detected from RAW "
    "instead of guessed."
)

print(
    "Only the five property sections required "
    "the corrective RAW pass."
)

print(
    "report_section_normalization_state.parquet "
    "now contains real section counts."
)

normalized / employees_by_civil_contract_source: non-numeric source values:
employees_by_civil_contract_source
--    8880
        78
Name: count, dtype: int64


RuntimeError: Unexpected non-numeric employee values in employees_by_civil_contract_source.

In [120]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import importlib
import json
import re

import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# PRECONDITION
# ============================================================

required_globals = [
    "normalized_dfs",
    "enriched_dfs",
    "state_rows",
    "selected_ids",
    "report_context",
    "context_by_report",
    "RAW_DIR",
    "TABLE_PATHS",
    "ENRICHED_TABLE_PATHS",
    "SECTION_STATE_PATH",
    "MODULE_PATH",
    "load_json",
    "first_context_value",
    "normalize_source_row",
    "NORMALIZATION_VERSION",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "This continuation requires the same Jupyter kernel. "
        f"Missing variables: {missing_globals}"
    )


# ============================================================
# 1. FIX EMPLOYEE COUNTS
#
# IMPORTANT:
# Preserve exact source representation:
#
# employees_by_*_source
#
# Add normalized nullable integer:
#
# employees_by_civil_contract
# employees_by_employment_contract
#
# Source markers such as "--" and "" become NA only in the
# normalized numerical column.
# ============================================================

EMPLOYEE_MAP = {
    "employees_by_civil_contract_source":
        "employees_by_civil_contract",

    "employees_by_employment_contract_source":
        "employees_by_employment_contract",
}


MISSING_MARKERS = {
    "",
    "-",
    "--",
    "—",
    "–",
    "null",
    "none",
    "nan",
}


def add_normalized_employee_counts(df):

    df = df.copy()

    for source_col, normalized_col in EMPLOYEE_MAP.items():

        # Preserve source safely as text for parquet.
        source = (
            df[source_col]
            .astype("string")
        )

        df[source_col] = source


        stripped = (
            source
            .str.strip()
        )


        missing_mask = (
            stripped
            .str.lower()
            .isin(MISSING_MARKERS)
        )


        cleaned = (
            stripped
            .mask(
                missing_mask,
                pd.NA,
            )
        )


        numeric = pd.to_numeric(
            cleaned,
            errors="coerce",
        )


        unexpected_mask = (
            cleaned.notna()
            &
            numeric.isna()
        )


        if unexpected_mask.any():

            print()
            print(
                f"Unexpected values in {source_col}:"
            )

            print(
                cleaned[
                    unexpected_mask
                ]
                .value_counts()
                .head(100)
            )

            raise RuntimeError(
                f"Unexpected non-numeric employee values "
                f"in {source_col}."
            )


        # Employee count must be integer if present.
        fractional_mask = (
            numeric.notna()
            &
            (
                numeric
                !=
                numeric.round()
            )
        )


        if fractional_mask.any():

            print(
                numeric[
                    fractional_mask
                ]
                .value_counts()
            )

            raise RuntimeError(
                f"Fractional employee counts found "
                f"in {source_col}."
            )


        df[normalized_col] = (
            numeric.astype(
                "Int64"
            )
        )


        print()
        print(
            source_col
        )

        print(
            "  source missing markers:",
            int(
                missing_mask.sum()
            )
        )

        print(
            "  normalized numeric:",
            int(
                df[
                    normalized_col
                ].notna().sum()
            )
        )

        print(
            "  normalized NA:",
            int(
                df[
                    normalized_col
                ].isna().sum()
            )
        )


    return df


print()
print("=" * 100)
print("EMPLOYEE COUNT NORMALIZATION")
print("=" * 100)


normalized_dfs[
    "employee_counts"
] = add_normalized_employee_counts(
    normalized_dfs[
        "employee_counts"
    ]
)


enriched_dfs[
    "employee_counts"
] = add_normalized_employee_counts(
    enriched_dfs[
        "employee_counts"
    ]
)


# ============================================================
# 2. WRITE employee_counts
# ============================================================

for path, df in [
    (
        TABLE_PATHS[
            "employee_counts"
        ],
        normalized_dfs[
            "employee_counts"
        ],
    ),
    (
        ENRICHED_TABLE_PATHS[
            "employee_counts"
        ],
        enriched_dfs[
            "employee_counts"
        ],
    ),
]:

    tmp = path.with_suffix(
        ".tmp.parquet"
    )

    df.to_parquet(
        tmp,
        index=False,
    )

    tmp.replace(
        path
    )


print()
print(
    "employee_counts written successfully."
)


# ============================================================
# 3. VERIFY ALREADY-PERSISTED NON-PROPERTY TABLES
#
# These came from the successful first RAW pass.
# Do NOT re-read RAW for them.
# ============================================================

GOOD_EXISTING = [
    "obligations",
    "head_info",
    "organizations",
    "regional_offices",
    "employee_counts",
]


existing_qa = []


for name in GOOD_EXISTING:

    normalized_path = (
        TABLE_PATHS[name]
    )

    enriched_path = (
        ENRICHED_TABLE_PATHS[name]
    )


    if not normalized_path.exists():
        raise FileNotFoundError(
            normalized_path
        )

    if not enriched_path.exists():
        raise FileNotFoundError(
            enriched_path
        )


    n_rows = len(
        pd.read_parquet(
            normalized_path,
            columns=[
                "source_report_id"
            ],
        )
    )

    e_rows = len(
        pd.read_parquet(
            enriched_path,
            columns=[
                "source_report_id"
            ],
        )
    )


    if n_rows != e_rows:

        raise RuntimeError(
            f"{name}: normalized/enriched mismatch "
            f"{n_rows:,} != {e_rows:,}"
        )


    existing_qa.append(
        {
            "section":
                name,

            "rows":
                n_rows,
        }
    )


print()
print("=" * 100)
print("PERSISTED NON-PROPERTY SECTIONS")
print("=" * 100)

display(
    pd.DataFrame(
        existing_qa
    )
)


# ============================================================
# 4. CHEAP PROPERTY-KEY PROBE
#
# The previous extraction produced zero for:
# realty, transport, movable, intangible, paper.
#
# That means our guessed nested paths were wrong.
#
# Read only a small sample first to discover exact keys.
# ============================================================

probe_ids = selected_ids[:100]


property_key_stats = {}


for report_id in probe_ids:

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    payload = load_json(
        path
    )


    detail = payload.get(
        "results",
        {}
    )


    properties = detail.get(
        "properties"
    )


    if not isinstance(
        properties,
        dict,
    ):
        continue


    for key, value in properties.items():

        rec = property_key_stats.setdefault(
            key,
            {
                "reports_seen": 0,
                "list_values": 0,
                "dict_values": 0,
                "other_values": 0,
                "nonempty_values": 0,
            },
        )


        rec[
            "reports_seen"
        ] += 1


        if isinstance(
            value,
            list,
        ):

            rec[
                "list_values"
            ] += 1

            if len(value):
                rec[
                    "nonempty_values"
                ] += 1


        elif isinstance(
            value,
            dict,
        ):

            rec[
                "dict_values"
            ] += 1

            if len(value):
                rec[
                    "nonempty_values"
                ] += 1


        else:

            rec[
                "other_values"
            ] += 1

            if value not in (
                None,
                "",
            ):
                rec[
                    "nonempty_values"
                ] += 1


property_key_qa = (
    pd.DataFrame.from_dict(
        property_key_stats,
        orient="index",
    )
    .reset_index(
        names="property_key"
    )
    .sort_values(
        "property_key"
    )
)


print()
print("=" * 100)
print("PROPERTY KEYS FOUND IN RAW")
print("=" * 100)

display(
    property_key_qa
)


if property_key_qa.empty:

    raise RuntimeError(
        "No dictionary keys found under detail['properties']. "
        "Do NOT start another full RAW pass."
    )


# ============================================================
# 5. AUTOMATIC PROPERTY KEY DETECTION
# ============================================================

def normalize_key(value):

    return re.sub(
        r"[^a-z0-9]",
        "",
        str(value).lower(),
    )


PROPERTY_KEY_PATTERNS = {

    "realty": [
        "realty",
        "realties",
        "realestate",
        "realestates",
    ],

    "transport": [
        "transport",
        "transports",
        "vehicle",
        "vehicles",
    ],

    "movable": [
        "movable",
        "movables",
    ],

    "intangible": [
        "intangible",
        "intangibles",
    ],

    "paper": [
        "paper",
        "papers",
        "security",
        "securities",
    ],
}


detected_keys = {}


all_property_keys = list(
    property_key_stats
)


for canonical, patterns in (
    PROPERTY_KEY_PATTERNS.items()
):

    matches = []


    for source_key in all_property_keys:

        normalized = normalize_key(
            source_key
        )


        if any(
            pattern in normalized
            for pattern in patterns
        ):

            matches.append(
                source_key
            )


    print(
        f"{canonical:12s}: {matches}"
    )


    if len(matches) != 1:

        raise RuntimeError(
            "Could not uniquely resolve property key "
            f"for {canonical}: {matches}. "
            "Do NOT run another full RAW pass."
        )


    detected_keys[
        canonical
    ] = matches[0]


print()
print("=" * 100)
print("DETECTED PROPERTY SECTION MAP")
print("=" * 100)

for canonical, source_key in (
    detected_keys.items()
):

    print(
        f"{canonical:12s} -> properties[{source_key!r}]"
    )


# ============================================================
# 6. PATCH PERMANENT report_sections.py
#
# Future sync will use the correct paths.
# ============================================================

module_text = MODULE_PATH.read_text(
    encoding="utf-8"
)


for canonical, source_key in (
    detected_keys.items()
):

    pattern = (
        rf'"{canonical}"\s*:\s*'
        rf'\("properties",\s*"[^"]+"\)'
    )

    replacement = (
        f'"{canonical}": '
        f'("properties", "{source_key}")'
    )


    module_text, replacements = re.subn(
        pattern,
        replacement,
        module_text,
        count=1,
    )


    if replacements != 1:

        raise RuntimeError(
            f"Could not patch SECTION_PATHS "
            f"for {canonical}."
        )


MODULE_PATH.write_text(
    module_text,
    encoding="utf-8",
)


import politdata.normalization.report_sections as rs_module

importlib.reload(
    rs_module
)


print()
print(
    "Permanent SECTION_PATHS updated."
)


# ============================================================
# 7. PROPERTY-ONLY CORRECTIVE RAW PASS
#
# IMPORTANT:
#
# This does NOT redo:
# - payments
# - property_moneys
# - obligations
# - head_info
# - organizations
# - regional_offices
# - employee_counts
#
# It extracts ONLY these five incorrectly addressed sections.
# ============================================================

PROPERTY_SECTIONS = [
    "realty",
    "transport",
    "movable",
    "intangible",
    "paper",
]


def process_property_report(
    report_id,
):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    payload = load_json(
        path
    )


    detail = payload.get(
        "results"
    )


    if not isinstance(
        detail,
        dict,
    ):

        return {
            "report_id":
                report_id,

            "status":
                "invalid",

            "rows":
                {},

            "counts":
                {},
        }


    properties = detail.get(
        "properties"
    )


    if not isinstance(
        properties,
        dict,
    ):

        properties = {}


    context = context_by_report[
        report_id
    ]


    organization_id = (
        first_context_value(
            context,
            "organization_id",
        )
    )


    root_party_id = (
        first_context_value(
            context,
            "root_party_id",
        )
    )


    report_year = (
        first_context_value(
            context,
            "report_year",
            "year",
        )
    )


    report_quarter = (
        first_context_value(
            context,
            "report_quarter",
            "quarter",
        )
    )


    signed_date = detail.get(
        "signed_date"
    )


    common_kwargs = {

        "source_report_id":
            report_id,

        "organization_id":
            organization_id,

        "root_party_id":
            root_party_id,

        "report_year":
            report_year,

        "report_quarter":
            report_quarter,

        "source_is_signed":
            bool(
                signed_date
            ),

        "source_signed_date":
            signed_date,

        "report_schema_version_source":
            detail.get(
                "schema_version"
            ),

        "report_type_source":
            detail.get(
                "report_type"
            ),

        "is_party_office_source":
            detail.get(
                "is_party_office"
            ),
    }


    output = {}
    counts = {}


    for section in PROPERTY_SECTIONS:

        source_key = (
            detected_keys[
                section
            ]
        )


        source_rows = (
            rs_module.coerce_rows(
                properties.get(
                    source_key
                )
            )
        )


        rows = []


        for row_index, source_row in enumerate(
            source_rows
        ):

            rows.append(
                normalize_source_row(
                    source_row,

                    source_section=
                        section,

                    source_row_index=
                        row_index,

                    **common_kwargs,
                )
            )


        output[
            section
        ] = rows


        counts[
            section
        ] = len(rows)


    return {
        "report_id":
            report_id,

        "status":
            "success",

        "rows":
            output,

        "counts":
            counts,
    }


property_rows = {
    section: []
    for section in PROPERTY_SECTIONS
}


property_count_by_report = {}


print()
print("=" * 100)
print("PROPERTY-ONLY CORRECTIVE RAW PASS")
print("=" * 100)

print(
    "Reports opened:",
    f"{len(selected_ids):,}"
)

print(
    "Only 5 property sections are processed."
)


with ThreadPoolExecutor(
    max_workers=16
) as executor:

    results = executor.map(
        process_property_report,
        selected_ids,
    )


    for item in tqdm(
        results,
        total=len(
            selected_ids
        ),
        desc="Property sections only",
    ):

        if item[
            "status"
        ] != "success":

            raise RuntimeError(
                "Invalid report encountered: "
                f"{item['report_id']}"
            )


        property_count_by_report[
            item[
                "report_id"
            ]
        ] = item[
            "counts"
        ]


        for section in PROPERTY_SECTIONS:

            property_rows[
                section
            ].extend(
                item[
                    "rows"
                ][
                    section
                ]
            )


# ============================================================
# 8. ROBUST PARQUET STABILIZATION
# ============================================================

BASE_COLUMNS = [
    "source_report_id",
    "source_section",
    "source_row_index",
    "organization_id",
    "root_party_id",
    "report_year",
    "report_quarter",
    "source_is_signed",
    "source_signed_date",
    "report_schema_version_source",
    "report_type_source",
    "is_party_office_source",
]


def stabilize_for_parquet(df):

    df = df.copy()


    for column in df.columns:

        if str(
            df[column].dtype
        ) != "object":

            continue


        nonnull = (
            df[column]
            .dropna()
        )


        if nonnull.empty:
            continue


        # Detect mixed underlying Python scalar types across
        # the whole column, not only an early sample.
        python_types = {
            type(value).__name__
            for value in nonnull
        }


        if len(
            python_types
        ) > 1:

            df[column] = (
                df[column]
                .astype(
                    "string"
                )
            )


    return df


property_dfs = {}


for section in PROPERTY_SECTIONS:

    df = pd.DataFrame(
        property_rows[
            section
        ]
    )


    if df.empty:

        df = pd.DataFrame(
            columns=BASE_COLUMNS
        )


    property_dfs[
        section
    ] = stabilize_for_parquet(
        df
    )


# ============================================================
# 9. CORRECTED PROPERTY QA
# ============================================================

property_qa = []


for section, df in (
    property_dfs.items()
):

    property_qa.append(
        {
            "section":
                section,

            "rows":
                len(df),

            "reports":
                (
                    df[
                        "source_report_id"
                    ].nunique()
                    if len(df)
                    else 0
                ),

            "organizations":
                (
                    df[
                        "organization_id"
                    ].nunique()
                    if len(df)
                    else 0
                ),

            "columns":
                len(
                    df.columns
                ),

            "source_columns":
                sum(
                    column.startswith(
                        "source__"
                    )
                    for column in df.columns
                ),
        }
    )


print()
print("=" * 100)
print("CORRECTED PROPERTY SECTION QA")
print("=" * 100)

display(
    pd.DataFrame(
        property_qa
    )
)


# ============================================================
# 10. LATEST REPORT MAP
# ============================================================

ctx = report_context.copy()


year_col = (
    "report_year"
    if "report_year" in ctx.columns
    else "year"
)


quarter_col = (
    "report_quarter"
    if "report_quarter" in ctx.columns
    else "quarter"
)


ctx[
    "_year_rank"
] = pd.to_numeric(
    ctx[
        year_col
    ],
    errors="coerce",
)


ctx[
    "_quarter_rank"
] = (
    pd.to_numeric(
        ctx[
            quarter_col
        ],
        errors="coerce",
    )
    .fillna(0)
)


latest_by_org = (
    ctx
    .sort_values(
        [
            "organization_id",
            "_year_rank",
            "_quarter_rank",
            "source_report_id",
        ]
    )
    .groupby(
        "organization_id",
        dropna=False,
    )
    .tail(1)
)


latest_report_ids = set(
    latest_by_org[
        "source_report_id"
    ]
    .astype(str)
)


# ============================================================
# 11. ENRICH PROPERTY TABLES
# ============================================================

CONTEXT_COLUMNS = [
    "organization_name_current",
    "organization_code",
    "organization_level",
    "region",

    "party_name_current",
    "party_code",

    "analysis_override",
    "analysis_selection_method",

    "official_selected_report_id",
    "analysis_selected_report_id",

    "continuity_exact",
]


available_context_columns = [
    column
    for column in CONTEXT_COLUMNS
    if column in report_context.columns
]


merge_context = (
    report_context[
        [
            "source_report_id",
            *available_context_columns,
        ]
    ]
    .copy()
)


def report_period_label(
    year,
    quarter,
):

    if pd.isna(year):
        return None


    try:
        year = int(year)
    except Exception:
        return None


    try:
        quarter = int(quarter)
    except Exception:
        quarter = None


    if quarter == 5:
        return str(year)


    if quarter in (
        1,
        2,
        3,
        4,
    ):
        return (
            f"{year}Q{quarter}"
        )


    return str(year)


def enrich_property_table(df):

    enriched = (
        df
        .merge(
            merge_context,
            on="source_report_id",
            how="left",
            validate="many_to_one",
        )
    )


    enriched[
        "analysis_selected"
    ] = True


    if (
        "analysis_override"
        in enriched.columns
    ):

        enriched[
            "official_selected"
        ] = (
            ~
            enriched[
                "analysis_override"
            ]
            .fillna(False)
            .astype(bool)
        )

    else:

        enriched[
            "official_selected"
        ] = True


    enriched[
        "is_latest_data"
    ] = (
        enriched[
            "source_report_id"
        ]
        .astype(str)
        .isin(
            latest_report_ids
        )
    )


    enriched[
        "data_recency_status"
    ] = (
        enriched[
            "is_latest_data"
        ]
        .map(
            {
                True:
                    "latest_data",

                False:
                    "historical_data",
            }
        )
    )


    enriched[
        "report_period"
    ] = [
        report_period_label(
            year,
            quarter,
        )
        for year, quarter
        in zip(
            enriched[
                "report_year"
            ],
            enriched[
                "report_quarter"
            ],
        )
    ]


    return stabilize_for_parquet(
        enriched
    )


property_enriched_dfs = {

    section:
        enrich_property_table(
            df
        )

    for section, df
    in property_dfs.items()
}


# ============================================================
# 12. WRITE CORRECTED PROPERTY TABLES
# ============================================================

for section in PROPERTY_SECTIONS:

    normalized_path = (
        TABLE_PATHS[
            section
        ]
    )

    enriched_path = (
        ENRICHED_TABLE_PATHS[
            section
        ]
    )


    normalized_tmp = (
        normalized_path
        .with_suffix(
            ".tmp.parquet"
        )
    )

    enriched_tmp = (
        enriched_path
        .with_suffix(
            ".tmp.parquet"
        )
    )


    property_dfs[
        section
    ].to_parquet(
        normalized_tmp,
        index=False,
    )


    property_enriched_dfs[
        section
    ].to_parquet(
        enriched_tmp,
        index=False,
    )


    normalized_tmp.replace(
        normalized_path
    )

    enriched_tmp.replace(
        enriched_path
    )


# ============================================================
# 13. PATCH SECTION STATE COUNTS
# ============================================================

state_index = {
    row[
        "source_report_id"
    ]:
        row
    for row in state_rows
}


for report_id, counts in (
    property_count_by_report.items()
):

    state_record = (
        state_index[
            report_id
        ]
    )


    for section in PROPERTY_SECTIONS:

        state_record[
            f"{section}_count"
        ] = counts[
            section
        ]


section_state = pd.DataFrame(
    state_rows
)


section_state[
    "normalized_at"
] = pd.Timestamp.now(
    tz="UTC"
)


section_state.to_parquet(
    SECTION_STATE_PATH,
    index=False,
)


# ============================================================
# 14. FINAL SECTION COVERAGE MATRIX
# ============================================================

ALL_SECTION_NAMES = [
    "realty",
    "transport",
    "movable",
    "intangible",
    "paper",
    "obligations",
    "head_info",
    "organizations",
    "regional_offices",
    "employee_counts",
]


coverage = []


for section in ALL_SECTION_NAMES:

    count_col = (
        f"{section}_count"
    )


    coverage.append(
        {
            "section":
                section,

            "reports_with_rows":
                int(
                    (
                        section_state[
                            count_col
                        ] > 0
                    ).sum()
                ),

            "total_rows":
                int(
                    section_state[
                        count_col
                    ].sum()
                ),

            "max_rows_per_report":
                int(
                    section_state[
                        count_col
                    ].max()
                ),
        }
    )


coverage = (
    pd.DataFrame(
        coverage
    )
    .sort_values(
        "section"
    )
)


print()
print("=" * 100)
print("FINAL SECTION COVERAGE MATRIX")
print("=" * 100)

display(
    coverage
)


# ============================================================
# 15. EXISTING DATA REGRESSION
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)


payment_dir = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)


payment_total = 0


for name in [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]:

    payment_total += len(
        pd.read_parquet(
            payment_dir
            / f"{name}.parquet",

            columns=[
                "source_report_id"
            ],
        )
    )


property_moneys_path = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "properties"
    / "property_moneys.parquet"
)


property_moneys_rows = len(
    pd.read_parquet(
        property_moneys_path,
        columns=[
            "source_report_id"
        ],
    )
)


print()
print("=" * 100)
print("REGRESSION CONTROL")
print("=" * 100)

print(
    "Payments:",
    f"{payment_total:,}"
)

print(
    "property_moneys:",
    f"{property_moneys_rows:,}"
)


assert payment_total == 402_028
assert property_moneys_rows == 19_140


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Employee source values were preserved."
)

print(
    '"--" and blank employee values are normalized to NA.'
)

print(
    "Existing non-property extraction was preserved."
)

print(
    "Only the five property sections required "
    "a corrective RAW pass."
)

print(
    "Future section coverage should use "
    "report_section_normalization_state.parquet."
)


EMPLOYEE COUNT NORMALIZATION

employees_by_civil_contract_source
  source missing markers: 8958
  normalized numeric: 69833
  normalized NA: 8958

employees_by_employment_contract_source
  source missing markers: 8557
  normalized numeric: 70234
  normalized NA: 8557

employees_by_civil_contract_source
  source missing markers: 8958
  normalized numeric: 69833
  normalized NA: 8958

employees_by_employment_contract_source
  source missing markers: 8557
  normalized numeric: 70234
  normalized NA: 8557

employee_counts written successfully.

PERSISTED NON-PROPERTY SECTIONS


,section,rows
0,obligations,26132
1,head_info,78791
2,organizations,308
3,regional_offices,102678
4,employee_counts,78791



PROPERTY KEYS FOUND IN RAW


,property_key,reports_seen,list_values,dict_values,other_values,nonempty_values
0,property_intangible_asset,100,100,0,0,0
2,property_moneys,100,100,0,0,16
1,property_movable,100,100,0,0,0
3,property_object,100,100,0,0,6
4,property_paper,100,100,0,0,0
5,property_transport,100,100,0,0,0


realty      : []


RuntimeError: Could not uniquely resolve property key for realty: []. Do NOT run another full RAW pass.

In [122]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import importlib
import json
import re

import pandas as pd
from tqdm.auto import tqdm


# ============================================================
# PRECONDITION
# ============================================================

required_globals = [
    "selected_ids",
    "report_context",
    "context_by_report",
    "RAW_DIR",
    "TABLE_PATHS",
    "ENRICHED_TABLE_PATHS",
    "SECTION_STATE_PATH",
    "MODULE_PATH",
    "state_rows",
    "load_json",
    "first_context_value",
    "normalize_source_row",
]

missing_globals = [
    name
    for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Run this in the same Jupyter kernel. "
        f"Missing variables: {missing_globals}"
    )


# ============================================================
# 1. FINAL RAW PROPERTY KEY MAP
#
# Confirmed from the actual RAW structure.
# ============================================================

detected_keys = {

    "realty":
        "property_object",

    "transport":
        "property_transport",

    "movable":
        "property_movable",

    "intangible":
        "property_intangible_asset",

    "paper":
        "property_paper",
}


print()
print("=" * 100)
print("FINAL PROPERTY SECTION MAP")
print("=" * 100)

for canonical, source_key in detected_keys.items():

    print(
        f"{canonical:12s} -> properties[{source_key!r}]"
    )


# ============================================================
# 2. PATCH PERMANENT report_sections.py
# ============================================================

module_text = MODULE_PATH.read_text(
    encoding="utf-8"
)


for canonical, source_key in detected_keys.items():

    pattern = (
        rf'"{canonical}"\s*:\s*'
        rf'\("properties",\s*"[^"]+"\)'
    )

    replacement = (
        f'"{canonical}": '
        f'("properties", "{source_key}")'
    )


    module_text, replacements = re.subn(
        pattern,
        replacement,
        module_text,
        count=1,
    )


    if replacements != 1:

        raise RuntimeError(
            f"Could not patch SECTION_PATHS "
            f"for {canonical}."
        )


MODULE_PATH.write_text(
    module_text,
    encoding="utf-8",
)


import politdata.normalization.report_sections as rs_module

importlib.reload(
    rs_module
)


print()
print(
    "Permanent SECTION_PATHS patched."
)


# ============================================================
# 3. REGRESSION CONTROL OF THE FIVE PATHS
# ============================================================

for section, raw_key in detected_keys.items():

    assert (
        rs_module.SECTION_PATHS[
            section
        ]
        ==
        (
            "properties",
            raw_key,
        )
    )


# ============================================================
# 4. PROPERTY-ONLY PROCESSOR
#
# This pass opens RAW, but extracts ONLY:
#
# - property_object
# - property_transport
# - property_movable
# - property_intangible_asset
# - property_paper
#
# It does NOT redo payments, money accounts, obligations,
# heads, employees, organizations or regional_offices.
# ============================================================

PROPERTY_SECTIONS = [
    "realty",
    "transport",
    "movable",
    "intangible",
    "paper",
]


def process_property_report(report_id):

    path = (
        RAW_DIR
        / f"{report_id}.json"
    )


    try:

        payload = load_json(
            path
        )

    except Exception as exc:

        return {
            "report_id":
                report_id,

            "status":
                "json_error",

            "error":
                repr(exc),

            "rows":
                {},

            "counts":
                {},
        }


    detail = (
        payload.get(
            "results"
        )
        if isinstance(
            payload,
            dict,
        )
        else None
    )


    if not isinstance(
        detail,
        dict,
    ):

        return {
            "report_id":
                report_id,

            "status":
                "invalid_wrapper",

            "rows":
                {},

            "counts":
                {},
        }


    properties = detail.get(
        "properties"
    )


    if not isinstance(
        properties,
        dict,
    ):

        properties = {}


    context = context_by_report[
        report_id
    ]


    organization_id = (
        first_context_value(
            context,
            "organization_id",
        )
    )


    root_party_id = (
        first_context_value(
            context,
            "root_party_id",
        )
    )


    report_year = (
        first_context_value(
            context,
            "report_year",
            "year",
        )
    )


    report_quarter = (
        first_context_value(
            context,
            "report_quarter",
            "quarter",
        )
    )


    signed_date = detail.get(
        "signed_date"
    )


    common_kwargs = {

        "source_report_id":
            report_id,

        "organization_id":
            organization_id,

        "root_party_id":
            root_party_id,

        "report_year":
            report_year,

        "report_quarter":
            report_quarter,

        "source_is_signed":
            bool(
                signed_date
            ),

        "source_signed_date":
            signed_date,

        "report_schema_version_source":
            detail.get(
                "schema_version"
            ),

        "report_type_source":
            detail.get(
                "report_type"
            ),

        "is_party_office_source":
            detail.get(
                "is_party_office"
            ),
    }


    output = {}
    counts = {}


    for section in PROPERTY_SECTIONS:

        source_key = (
            detected_keys[
                section
            ]
        )


        source_rows = (
            rs_module.coerce_rows(
                properties.get(
                    source_key
                )
            )
        )


        rows = []


        for row_index, source_row in enumerate(
            source_rows
        ):

            rows.append(
                normalize_source_row(
                    source_row,

                    source_section=
                        section,

                    source_row_index=
                        row_index,

                    **common_kwargs,
                )
            )


        output[
            section
        ] = rows


        counts[
            section
        ] = len(
            rows
        )


    return {
        "report_id":
            report_id,

        "status":
            "success",

        "rows":
            output,

        "counts":
            counts,
    }


# ============================================================
# 5. RUN CORRECTIVE PROPERTY PASS
# ============================================================

property_rows = {
    section: []
    for section in PROPERTY_SECTIONS
}


property_count_by_report = {}

errors = []


print()
print("=" * 100)
print("PROPERTY-ONLY CORRECTIVE RAW PASS")
print("=" * 100)

print(
    "Reports:",
    f"{len(selected_ids):,}"
)

print(
    "Only five property sections are extracted."
)


with ThreadPoolExecutor(
    max_workers=16
) as executor:

    results = executor.map(
        process_property_report,
        selected_ids,
    )


    for item in tqdm(
        results,
        total=len(selected_ids),
        desc="Property sections",
    ):

        if item[
            "status"
        ] != "success":

            errors.append(
                {
                    "report_id":
                        item[
                            "report_id"
                        ],

                    "status":
                        item[
                            "status"
                        ],

                    "error":
                        item.get(
                            "error"
                        ),
                }
            )

            continue


        property_count_by_report[
            item[
                "report_id"
            ]
        ] = item[
            "counts"
        ]


        for section in PROPERTY_SECTIONS:

            property_rows[
                section
            ].extend(
                item[
                    "rows"
                ][
                    section
                ]
            )


if errors:

    display(
        pd.DataFrame(
            errors
        ).head(100)
    )

    raise RuntimeError(
        f"Property extraction errors: {len(errors)}"
    )


if (
    len(
        property_count_by_report
    )
    !=
    len(
        selected_ids
    )
):

    raise RuntimeError(
        "Property extraction report coverage mismatch."
    )


# ============================================================
# 6. PARQUET TYPE STABILIZATION
#
# Source columns may contain heterogeneous primitive types.
# Nested dict/list values were already serialized as JSON.
# ============================================================

BASE_COLUMNS = [
    "source_report_id",
    "source_section",
    "source_row_index",
    "organization_id",
    "root_party_id",
    "report_year",
    "report_quarter",
    "source_is_signed",
    "source_signed_date",
    "report_schema_version_source",
    "report_type_source",
    "is_party_office_source",
]


def stabilize_for_parquet(df):

    df = df.copy()


    for column in df.columns:

        if str(
            df[
                column
            ].dtype
        ) != "object":

            continue


        nonnull = (
            df[
                column
            ]
            .dropna()
        )


        if nonnull.empty:
            continue


        python_types = {
            type(value).__name__
            for value in nonnull
        }


        if len(
            python_types
        ) > 1:

            df[
                column
            ] = (
                df[
                    column
                ]
                .astype(
                    "string"
                )
            )


    return df


property_dfs = {}


for section in PROPERTY_SECTIONS:

    df = pd.DataFrame(
        property_rows[
            section
        ]
    )


    if df.empty:

        df = pd.DataFrame(
            columns=BASE_COLUMNS
        )


    property_dfs[
        section
    ] = stabilize_for_parquet(
        df
    )


# ============================================================
# 7. NORMALIZED PROPERTY QA
# ============================================================

property_qa = []


for section, df in property_dfs.items():

    property_qa.append(
        {
            "section":
                section,

            "rows":
                len(df),

            "reports":
                (
                    df[
                        "source_report_id"
                    ].nunique()
                    if len(df)
                    else 0
                ),

            "organizations":
                (
                    df[
                        "organization_id"
                    ].nunique()
                    if len(df)
                    else 0
                ),

            "columns":
                len(
                    df.columns
                ),

            "source_columns":
                sum(
                    column.startswith(
                        "source__"
                    )
                    for column in df.columns
                ),
        }
    )


property_qa = pd.DataFrame(
    property_qa
)


print()
print("=" * 100)
print("CORRECTED PROPERTY SECTION QA")
print("=" * 100)

display(
    property_qa
)


# ============================================================
# 8. SOURCE SCHEMA QA FOR PROPERTY TABLES
# ============================================================

schema_rows = []


for section, df in property_dfs.items():

    for column in df.columns:

        if not column.startswith(
            "source__"
        ):
            continue


        schema_rows.append(
            {
                "section":
                    section,

                "column":
                    column,

                "non_null_rows":
                    int(
                        df[
                            column
                        ].notna().sum()
                    ),

                "dtype":
                    str(
                        df[
                            column
                        ].dtype
                    ),
            }
        )


property_schema_qa = pd.DataFrame(
    schema_rows
)


print()
print("=" * 100)
print("PROPERTY SOURCE SCHEMA SUMMARY")
print("=" * 100)


if len(property_schema_qa):

    display(
        property_schema_qa.sort_values(
            [
                "section",
                "non_null_rows",
            ],
            ascending=[
                True,
                False,
            ],
        )
    )

else:

    print(
        "No populated property source columns."
    )


# ============================================================
# 9. LATEST REPORT MAP
# ============================================================

ctx = report_context.copy()


year_col = (
    "report_year"
    if "report_year" in ctx.columns
    else "year"
)


quarter_col = (
    "report_quarter"
    if "report_quarter" in ctx.columns
    else "quarter"
)


ctx[
    "_year_rank"
] = pd.to_numeric(
    ctx[
        year_col
    ],
    errors="coerce",
)


ctx[
    "_quarter_rank"
] = (
    pd.to_numeric(
        ctx[
            quarter_col
        ],
        errors="coerce",
    )
    .fillna(0)
)


latest_by_org = (
    ctx
    .sort_values(
        [
            "organization_id",
            "_year_rank",
            "_quarter_rank",
            "source_report_id",
        ]
    )
    .groupby(
        "organization_id",
        dropna=False,
    )
    .tail(1)
)


latest_report_ids = set(
    latest_by_org[
        "source_report_id"
    ]
    .astype(str)
)


# ============================================================
# 10. ENRICH PROPERTY TABLES
# ============================================================

CONTEXT_COLUMNS = [
    "organization_name_current",
    "organization_code",
    "organization_level",
    "region",

    "party_name_current",
    "party_code",

    "analysis_override",
    "analysis_selection_method",

    "official_selected_report_id",
    "analysis_selected_report_id",

    "continuity_exact",
]


available_context_columns = [
    column
    for column in CONTEXT_COLUMNS
    if column in report_context.columns
]


merge_context = (
    report_context[
        [
            "source_report_id",
            *available_context_columns,
        ]
    ]
    .copy()
)


def report_period_label(
    year,
    quarter,
):

    if pd.isna(year):
        return None


    try:
        year = int(year)
    except Exception:
        return None


    try:
        quarter = int(quarter)
    except Exception:
        quarter = None


    if quarter == 5:
        return str(year)


    if quarter in (
        1,
        2,
        3,
        4,
    ):
        return (
            f"{year}Q{quarter}"
        )


    return str(year)


def enrich_property_table(df):

    enriched = (
        df
        .merge(
            merge_context,
            on="source_report_id",
            how="left",
            validate="many_to_one",
        )
    )


    enriched[
        "analysis_selected"
    ] = True


    if (
        "analysis_override"
        in enriched.columns
    ):

        enriched[
            "official_selected"
        ] = (
            ~
            enriched[
                "analysis_override"
            ]
            .fillna(False)
            .astype(bool)
        )

    else:

        enriched[
            "official_selected"
        ] = True


    enriched[
        "is_latest_data"
    ] = (
        enriched[
            "source_report_id"
        ]
        .astype(str)
        .isin(
            latest_report_ids
        )
    )


    enriched[
        "data_recency_status"
    ] = (
        enriched[
            "is_latest_data"
        ]
        .map(
            {
                True:
                    "latest_data",

                False:
                    "historical_data",
            }
        )
    )


    enriched[
        "report_period"
    ] = [
        report_period_label(
            year,
            quarter,
        )
        for year, quarter in zip(
            enriched[
                "report_year"
            ],
            enriched[
                "report_quarter"
            ],
        )
    ]


    return stabilize_for_parquet(
        enriched
    )


property_enriched_dfs = {

    section:
        enrich_property_table(
            df
        )

    for section, df in property_dfs.items()
}


# ============================================================
# 11. WRITE PROPERTY TABLES
# ============================================================

for section in PROPERTY_SECTIONS:

    normalized_path = (
        TABLE_PATHS[
            section
        ]
    )

    enriched_path = (
        ENRICHED_TABLE_PATHS[
            section
        ]
    )


    normalized_tmp = (
        normalized_path
        .with_suffix(
            ".tmp.parquet"
        )
    )

    enriched_tmp = (
        enriched_path
        .with_suffix(
            ".tmp.parquet"
        )
    )


    property_dfs[
        section
    ].to_parquet(
        normalized_tmp,
        index=False,
    )


    property_enriched_dfs[
        section
    ].to_parquet(
        enriched_tmp,
        index=False,
    )


    normalized_tmp.replace(
        normalized_path
    )

    enriched_tmp.replace(
        enriched_path
    )


# ============================================================
# 12. UPDATE SECTION STATE
# ============================================================

state_index = {
    row[
        "source_report_id"
    ]:
        row
    for row in state_rows
}


for report_id, counts in property_count_by_report.items():

    record = state_index[
        report_id
    ]


    for section in PROPERTY_SECTIONS:

        record[
            f"{section}_count"
        ] = counts[
            section
        ]


section_state = pd.DataFrame(
    state_rows
)


section_state[
    "normalized_at"
] = pd.Timestamp.now(
    tz="UTC"
)


section_state.to_parquet(
    SECTION_STATE_PATH,
    index=False,
)


# ============================================================
# 13. FINAL COVERAGE MATRIX
# ============================================================

ALL_SECTION_NAMES = [
    "realty",
    "transport",
    "movable",
    "intangible",
    "paper",
    "obligations",
    "head_info",
    "organizations",
    "regional_offices",
    "employee_counts",
]


coverage = []


for section in ALL_SECTION_NAMES:

    count_col = (
        f"{section}_count"
    )


    coverage.append(
        {
            "section":
                section,

            "reports_with_rows":
                int(
                    (
                        section_state[
                            count_col
                        ]
                        > 0
                    ).sum()
                ),

            "total_rows":
                int(
                    section_state[
                        count_col
                    ].sum()
                ),

            "max_rows_per_report":
                int(
                    section_state[
                        count_col
                    ].max()
                ),
        }
    )


coverage = (
    pd.DataFrame(
        coverage
    )
    .sort_values(
        "section"
    )
)


print()
print("=" * 100)
print("FINAL SECTION COVERAGE MATRIX")
print("=" * 100)

display(
    coverage
)


# ============================================================
# 14. PERSISTED ROW CONTROL
# ============================================================

persisted = []


for section in ALL_SECTION_NAMES:

    n_path = (
        TABLE_PATHS[
            section
        ]
    )

    e_path = (
        ENRICHED_TABLE_PATHS[
            section
        ]
    )


    n_rows = len(
        pd.read_parquet(
            n_path,
            columns=[
                "source_report_id"
            ],
        )
    )


    e_rows = len(
        pd.read_parquet(
            e_path,
            columns=[
                "source_report_id"
            ],
        )
    )


    if n_rows != e_rows:

        raise RuntimeError(
            f"{section}: normalized/enriched row mismatch "
            f"{n_rows:,} != {e_rows:,}"
        )


    persisted.append(
        {
            "section":
                section,

            "normalized_rows":
                n_rows,

            "enriched_rows":
                e_rows,
        }
    )


print()
print("=" * 100)
print("PERSISTED SECTION CONTROL")
print("=" * 100)

display(
    pd.DataFrame(
        persisted
    )
)


# ============================================================
# 15. EXISTING PAYMENT / MONEY REGRESSION
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)


payment_dir = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)


payment_total = 0


for name in [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]:

    payment_total += len(
        pd.read_parquet(
            payment_dir
            / f"{name}.parquet",

            columns=[
                "source_report_id"
            ],
        )
    )


property_moneys_path = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "properties"
    / "property_moneys.parquet"
)


property_moneys_rows = len(
    pd.read_parquet(
        property_moneys_path,
        columns=[
            "source_report_id"
        ],
    )
)


print()
print("=" * 100)
print("REGRESSION CONTROL")
print("=" * 100)

print(
    "Payments:",
    f"{payment_total:,}"
)

print(
    "property_moneys:",
    f"{property_moneys_rows:,}"
)


assert payment_total == 402_028
assert property_moneys_rows == 19_140


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Correct property key map is now permanent."
)

print(
    "employee_counts were not rebuilt."
)

print(
    "Payments and property_moneys were not modified."
)

print(
    "report_section_normalization_state.parquet "
    "now contains corrected property section counts."
)


FINAL PROPERTY SECTION MAP
realty       -> properties['property_object']
transport    -> properties['property_transport']
movable      -> properties['property_movable']
intangible   -> properties['property_intangible_asset']
paper        -> properties['property_paper']

Permanent SECTION_PATHS patched.

PROPERTY-ONLY CORRECTIVE RAW PASS
Reports: 78,791
Only five property sections are extracted.


Property sections:   0%|          | 0/78791 [00:00<?, ?it/s]


CORRECTED PROPERTY SECTION QA


,section,rows,reports,organizations,columns,source_columns
0,realty,11325,4071,569,28,15
1,transport,1325,253,20,27,14
2,movable,693,103,10,26,13
3,intangible,8614,1849,169,25,12
4,paper,0,0,0,12,0



PROPERTY SOURCE SCHEMA SUMMARY


...


FINAL SECTION COVERAGE MATRIX


,section,reports_with_rows,total_rows,max_rows_per_report
9,employee_counts,78791,78791,1
6,head_info,78791,78791,1
3,intangible,1849,8614,94
2,movable,103,693,34
5,obligations,2378,26132,247
7,organizations,110,308,33
4,paper,0,0,0
0,realty,4071,11325,44
8,regional_offices,4500,102678,667
1,transport,253,1325,34



PERSISTED SECTION CONTROL


,section,normalized_rows,enriched_rows
0,realty,11325,11325
1,transport,1325,1325
2,movable,693,693
3,intangible,8614,8614
4,paper,0,0
5,obligations,26132,26132
6,head_info,78791,78791
7,organizations,308,308
8,regional_offices,102678,102678
9,employee_counts,78791,78791



REGRESSION CONTROL
Payments: 402,028
property_moneys: 19,140

DONE
Correct property key map is now permanent.
employee_counts were not rebuilt.
Payments and property_moneys were not modified.
report_section_normalization_state.parquet now contains corrected property section counts.


In [123]:
from pathlib import Path
import ast
import re
from collections import Counter, defaultdict

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

SRC = ROOT / "src" / "politdata"
TESTS = ROOT / "tests"


# ============================================================
# HELPERS
# ============================================================

def rel(path):
    return str(
        Path(path).relative_to(ROOT)
    ).replace("\\", "/")


def read_text(path):
    return path.read_text(
        encoding="utf-8"
    )


def module_name_from_path(path):
    relative = (
        path.relative_to(
            ROOT / "src"
        )
        .with_suffix("")
    )

    return ".".join(
        relative.parts
    )


# ============================================================
# 1. SOURCE MODULE INVENTORY
# ============================================================

module_rows = []
symbol_rows = []
import_rows = []


python_files = sorted(
    SRC.rglob("*.py")
)


for path in python_files:

    text = read_text(path)
    lines = text.splitlines()

    try:
        tree = ast.parse(text)
        parse_ok = True

    except SyntaxError as exc:

        module_rows.append(
            {
                "file": rel(path),
                "module": module_name_from_path(path),
                "lines": len(lines),
                "nonblank_lines": sum(
                    bool(x.strip())
                    for x in lines
                ),
                "functions": None,
                "classes": None,
                "imports": None,
                "parse_ok": False,
                "syntax_error": str(exc),
            }
        )

        continue


    functions = []
    classes = []
    imports = []


    for node in tree.body:

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            ),
        ):
            functions.append(node)

            symbol_rows.append(
                {
                    "file": rel(path),
                    "module": module_name_from_path(path),
                    "symbol": node.name,
                    "type": "function",
                    "line": node.lineno,
                    "end_line": getattr(
                        node,
                        "end_lineno",
                        None,
                    ),
                    "approx_lines": (
                        getattr(
                            node,
                            "end_lineno",
                            node.lineno,
                        )
                        -
                        node.lineno
                        +
                        1
                    ),
                    "public": not node.name.startswith("_"),
                }
            )


        elif isinstance(
            node,
            ast.ClassDef,
        ):
            classes.append(node)

            symbol_rows.append(
                {
                    "file": rel(path),
                    "module": module_name_from_path(path),
                    "symbol": node.name,
                    "type": "class",
                    "line": node.lineno,
                    "end_line": getattr(
                        node,
                        "end_lineno",
                        None,
                    ),
                    "approx_lines": (
                        getattr(
                            node,
                            "end_lineno",
                            node.lineno,
                        )
                        -
                        node.lineno
                        +
                        1
                    ),
                    "public": not node.name.startswith("_"),
                }
            )


        elif isinstance(
            node,
            ast.Import,
        ):

            for alias in node.names:

                imports.append(
                    alias.name
                )


        elif isinstance(
            node,
            ast.ImportFrom,
        ):

            if node.module:

                imports.append(
                    node.module
                )


    current_module = module_name_from_path(
        path
    )


    for imported in imports:

        if imported.startswith(
            "politdata"
        ):

            import_rows.append(
                {
                    "from_module":
                        current_module,

                    "to_module":
                        imported,
                }
            )


    module_rows.append(
        {
            "file": rel(path),
            "module": current_module,
            "lines": len(lines),
            "nonblank_lines": sum(
                bool(x.strip())
                for x in lines
            ),
            "functions": len(functions),
            "classes": len(classes),
            "imports": len(imports),
            "parse_ok": True,
            "syntax_error": None,
        }
    )


modules_df = (
    pd.DataFrame(module_rows)
    .sort_values(
        "lines",
        ascending=False,
    )
)


symbols_df = (
    pd.DataFrame(symbol_rows)
    .sort_values(
        [
            "file",
            "line",
        ]
    )
)


imports_df = pd.DataFrame(
    import_rows
)


print()
print("=" * 110)
print("SOURCE MODULE INVENTORY")
print("=" * 110)

display(
    modules_df[
        [
            "file",
            "lines",
            "nonblank_lines",
            "functions",
            "classes",
            "imports",
            "parse_ok",
        ]
    ]
)


print()
print(
    "Source modules:",
    len(modules_df),
)

print(
    "Total source lines:",
    int(
        modules_df["lines"].sum()
    ),
)


# ============================================================
# 2. LARGEST FUNCTIONS / CLASSES
#
# Useful for spotting code that should be decomposed.
# ============================================================

print()
print("=" * 110)
print("LARGEST FUNCTIONS / CLASSES")
print("=" * 110)


if len(symbols_df):

    display(
        symbols_df[
            [
                "file",
                "symbol",
                "type",
                "line",
                "approx_lines",
                "public",
            ]
        ]
        .sort_values(
            "approx_lines",
            ascending=False,
        )
        .head(40)
    )

else:

    print(
        "No top-level functions/classes found."
    )


# ============================================================
# 3. DUPLICATE SYMBOL NAMES
#
# Not automatically wrong.
# It tells us where generic helpers may have multiplied.
# ============================================================

print()
print("=" * 110)
print("DUPLICATE FUNCTION / CLASS NAMES")
print("=" * 110)


if len(symbols_df):

    duplicate_symbols = (
        symbols_df
        .groupby(
            [
                "symbol",
                "type",
            ],
            as_index=False,
        )
        .agg(
            occurrences=(
                "file",
                "size",
            ),

            files=(
                "file",
                lambda values:
                    " | ".join(
                        sorted(
                            set(values)
                        )
                    ),
            ),
        )
    )


    duplicate_symbols = (
        duplicate_symbols[
            duplicate_symbols[
                "occurrences"
            ] > 1
        ]
        .sort_values(
            [
                "occurrences",
                "symbol",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )


    display(
        duplicate_symbols
    )

else:

    duplicate_symbols = pd.DataFrame()


# ============================================================
# 4. INTERNAL MODULE DEPENDENCIES
# ============================================================

print()
print("=" * 110)
print("INTERNAL POLITDATA IMPORTS")
print("=" * 110)


if len(imports_df):

    dependency_df = (
        imports_df
        .drop_duplicates()
        .sort_values(
            [
                "from_module",
                "to_module",
            ]
        )
    )

    display(
        dependency_df
    )

else:

    dependency_df = pd.DataFrame()

    print(
        "No internal politdata imports found."
    )


# ============================================================
# 5. CODE RESPONSIBILITY / RISK PATTERN SCAN
#
# This is textual only.
#
# It helps identify:
# - RAW readers
# - parquet writers
# - hardcoded project paths
# - SQL embedded in Python
# - orchestration mixed into low-level modules
# ============================================================

PATTERNS = {

    "raw_report_access": [
        r"report_details",
        r"data[/\\]raw",
        r"\.json",
    ],

    "filesystem_json_scan": [
        r"rglob\s*\(",
        r"glob\s*\(",
        r"iterdir\s*\(",
    ],

    "direct_file_open": [
        r"\bopen\s*\(",
        r"\.read_text\s*\(",
        r"\.write_text\s*\(",
    ],

    "parquet_read": [
        r"read_parquet\s*\(",
    ],

    "parquet_write": [
        r"to_parquet\s*\(",
        r"COPY\s*\(",
        r"FORMAT\s+PARQUET",
    ],

    "duckdb": [
        r"\bduckdb\b",
    ],

    "http_requests": [
        r"\brequests\b",
        r"\.get\s*\(",
        r"\.post\s*\(",
    ],

    "hardcoded_windows_path": [
        r"[A-Za-z]:\\",
    ],

    "pytest_or_test_logic": [
        r"\bpytest\b",
        r"\bassert\b",
    ],

    "state_funding_logic": [
        r"state_funding",
        r"state statutory",
        r"statutory",
    ],

    "internal_transfer_logic": [
        r"internal_transfer",
        r"same_root",
    ],

    "region_logic": [
        r"region",
        r"oblast",
        r"м\. Київ",
    ],
}


pattern_rows = []


for path in python_files:

    text = read_text(path)


    for category, regexes in (
        PATTERNS.items()
    ):

        count = 0


        for pattern in regexes:

            count += len(
                re.findall(
                    pattern,
                    text,
                    flags=re.IGNORECASE,
                )
            )


        if count:

            pattern_rows.append(
                {
                    "file":
                        rel(path),

                    "category":
                        category,

                    "matches":
                        count,
                }
            )


patterns_df = pd.DataFrame(
    pattern_rows
)


print()
print("=" * 110)
print("CODE RESPONSIBILITY / RISK MAP")
print("=" * 110)


if len(patterns_df):

    pivot = (
        patterns_df
        .pivot_table(
            index="file",
            columns="category",
            values="matches",
            aggfunc="sum",
            fill_value=0,
        )
        .reset_index()
    )


    display(
        pivot
    )

else:

    pivot = pd.DataFrame()

    print(
        "No patterns found."
    )


# ============================================================
# 6. TEST INVENTORY
# ============================================================

test_rows = []
test_symbol_rows = []


for path in sorted(
    TESTS.glob(
        "test_*.py"
    )
):

    text = read_text(
        path
    )


    try:

        tree = ast.parse(
            text
        )

        parse_ok = True

    except SyntaxError:

        tree = None
        parse_ok = False


    tests = []


    if tree is not None:

        for node in tree.body:

            if (
                isinstance(
                    node,
                    (
                        ast.FunctionDef,
                        ast.AsyncFunctionDef,
                    ),
                )
                and
                node.name.startswith(
                    "test_"
                )
            ):

                tests.append(
                    node.name
                )

                test_symbol_rows.append(
                    {
                        "file":
                            rel(path),

                        "test":
                            node.name,

                        "line":
                            node.lineno,
                    }
                )


    test_rows.append(
        {
            "file":
                rel(path),

            "lines":
                len(
                    text.splitlines()
                ),

            "test_functions":
                len(tests),

            "parse_ok":
                parse_ok,
        }
    )


tests_df = (
    pd.DataFrame(
        test_rows
    )
    .sort_values(
        "file"
    )
)


print()
print("=" * 110)
print("TEST INVENTORY")
print("=" * 110)

display(
    tests_df
)


print()
print(
    "Test files:",
    len(tests_df),
)

print(
    "Test functions:",
    int(
        tests_df[
            "test_functions"
        ].sum()
    ),
)


# ============================================================
# 7. TEST COVERAGE BY MODULE NAME — HEURISTIC ONLY
#
# This does not pretend to be code coverage.
# It shows whether a source module has an obvious matching
# test module.
# ============================================================

test_stems = {
    path.stem.removeprefix(
        "test_"
    )
    for path
    in TESTS.glob(
        "test_*.py"
    )
}


coverage_rows = []


for path in python_files:

    if path.name == "__init__.py":
        continue


    stem = path.stem


    coverage_rows.append(
        {
            "source_file":
                rel(path),

            "module_stem":
                stem,

            "matching_test_file":
                stem
                in
                test_stems,
        }
    )


test_match_df = pd.DataFrame(
    coverage_rows
)


print()
print("=" * 110)
print("SOURCE MODULES WITHOUT OBVIOUS MATCHING TEST FILE")
print("=" * 110)

display(
    test_match_df[
        ~
        test_match_df[
            "matching_test_file"
        ]
    ]
)


# ============================================================
# 8. HIGH-LEVEL REFACTOR CANDIDATES
#
# Purely mechanical signals.
# No changes are made.
# ============================================================

refactor_rows = []


pattern_lookup = defaultdict(
    set
)


if len(patterns_df):

    for row in (
        patterns_df
        .itertuples(
            index=False
        )
    ):

        pattern_lookup[
            row.file
        ].add(
            row.category
        )


for row in modules_df.itertuples(
    index=False
):

    reasons = []


    if row.lines >= 500:

        reasons.append(
            "large_module"
        )


    categories = (
        pattern_lookup[
            row.file
        ]
    )


    if (
        "raw_report_access"
        in categories
        and
        "parquet_write"
        in categories
    ):

        reasons.append(
            "raw_and_persistence_mixed"
        )


    if (
        "http_requests"
        in categories
        and
        "parquet_write"
        in categories
    ):

        reasons.append(
            "api_and_persistence_mixed"
        )


    if (
        "hardcoded_windows_path"
        in categories
    ):

        reasons.append(
            "hardcoded_path"
        )


    if (
        "pytest_or_test_logic"
        in categories
        and
        not row.file.startswith(
            "tests/"
        )
    ):

        reasons.append(
            "assertions_inside_source"
        )


    if reasons:

        refactor_rows.append(
            {
                "file":
                    row.file,

                "lines":
                    row.lines,

                "reasons":
                    " | ".join(
                        reasons
                    ),
            }
        )


refactor_df = (
    pd.DataFrame(
        refactor_rows
    )
)


print()
print("=" * 110)
print("MECHANICAL REFACTOR CANDIDATES")
print("=" * 110)


if len(refactor_df):

    display(
        refactor_df.sort_values(
            [
                "lines",
                "file",
            ],
            ascending=[
                False,
                True,
            ],
        )
    )

else:

    print(
        "No obvious candidates by these simple rules."
    )


# ============================================================
# 9. DONE
# ============================================================

print()
print("=" * 110)
print("DONE")
print("=" * 110)

print(
    "Nothing was modified."
)

print(
    "RAW JSON files were NOT read."
)

print(
    "Parquet datasets were NOT read."
)

print(
    "No API requests were made."
)


SOURCE MODULE INVENTORY


,file,lines,nonblank_lines,functions,classes,imports,parse_ok
6,src/politdata/normalization/payments.py,1861,1529,17,0,14,True
11,src/politdata/report_details.py,1110,873,15,0,11,True
8,src/politdata/normalization/reference.py,1070,848,5,0,4,True
12,src/politdata/report_discovery.py,975,763,7,0,9,True
7,src/politdata/normalization/property_moneys.py,872,654,9,0,6,True
13,src/politdata/reports.py,824,646,6,0,5,True
14,src/politdata/sync.py,776,604,5,0,9,True
5,src/politdata/normalization/accounts.py,760,560,10,1,5,True
9,src/politdata/normalization/report_sections.py,379,283,6,0,3,True
3,src/politdata/discovery.py,336,264,5,0,3,True



Source modules: 15
Total source lines: 9600

LARGEST FUNCTIONS / CLASSES


,file,symbol,type,line,approx_lines,public
34,src/politdata/normalization/payments.py,normalize_payment_row,function,1032,631,True
95,src/politdata/sync.py,run_organization_sync,function,164,613,True
51,src/politdata/normalization/reference.py,build_organization_reference,function,220,580,True
77,src/politdata/report_details.py,run_report_detail_batch,function,737,374,True
83,src/politdata/report_discovery.py,run_report_discovery_batch,function,478,340,True
52,src/politdata/normalization/reference.py,build_report_context,function,806,265,True
20,src/politdata/normalization/accounts.py,normalize_account_number,function,368,262,True
45,src/politdata/normalization/property_moneys.py,classify_account_type,function,313,259,True
86,src/politdata/reports.py,fetch_all_reports,function,164,185,True
8,src/politdata/discovery.py,compare_manifests,function,107,167,True



DUPLICATE FUNCTION / CLASS NAMES


...


Test files: 5
Test functions: 49

SOURCE MODULES WITHOUT OBVIOUS MATCHING TEST FILE


,source_file,module_stem,matching_test_file
0,src/politdata/api.py,api,False
1,src/politdata/change_detection.py,change_detection,False
2,src/politdata/discovery.py,discovery,False
8,src/politdata/refresh.py,refresh,False
9,src/politdata/report_details.py,report_details,False
10,src/politdata/report_discovery.py,report_discovery,False
11,src/politdata/reports.py,reports,False
12,src/politdata/sync.py,sync,False



MECHANICAL REFACTOR CANDIDATES


,file,lines,reasons
0,src/politdata/normalization/payments.py,1861,large_module
1,src/politdata/report_details.py,1110,large_module | raw_and_persistence_mixed | api_and_persistence_mixed
2,src/politdata/normalization/reference.py,1070,large_module
3,src/politdata/report_discovery.py,975,large_module | raw_and_persistence_mixed | api_and_persistence_mixed
4,src/politdata/normalization/property_moneys.py,872,large_module
5,src/politdata/reports.py,824,large_module | raw_and_persistence_mixed | api_and_persistence_mixed
6,src/politdata/sync.py,776,large_module
7,src/politdata/normalization/accounts.py,760,large_module
8,src/politdata/discovery.py,336,api_and_persistence_mixed
9,src/politdata/refresh.py,329,raw_and_persistence_mixed | api_and_persistence_mixed



DONE
Nothing was modified.
RAW JSON files were NOT read.
Parquet datasets were NOT read.
No API requests were made.


In [124]:
from pathlib import Path
import subprocess
import sys
import textwrap


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

TESTS = ROOT / "tests"

TESTS.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. change_detection
# ============================================================

(TESTS / "test_change_detection.py").write_text(
    textwrap.dedent(
        r'''
        from politdata.change_detection import (
            canonicalize_record,
            organization_content_hash,
            deep_diff,
            classify_record_change,
        )


        def test_updated_at_is_excluded_from_content_hash():

            old = {
                "id": "x",
                "name": "Party",
                "updated_at": "2025-01-01",
            }

            new = {
                "id": "x",
                "name": "Party",
                "updated_at": "2026-01-01",
            }

            assert (
                organization_content_hash(old)
                ==
                organization_content_hash(new)
            )


        def test_meaningful_change_changes_hash():

            old = {
                "id": "x",
                "name": "Old",
            }

            new = {
                "id": "x",
                "name": "New",
            }

            assert (
                organization_content_hash(old)
                !=
                organization_content_hash(new)
            )


        def test_classification_distinguishes_technical_change():

            old = {
                "id": "x",
                "name": "Party",
                "updated_at": "2025-01-01",
            }

            new = {
                "id": "x",
                "name": "Party",
                "updated_at": "2026-01-01",
            }

            result = classify_record_change(
                old,
                new,
            )

            assert (
                result["content_changed"]
                is False
            )

            # Full diff still records the source-field change.
            assert (
                "updated_at"
                in
                result["changed_fields"]
            )


        def test_deep_diff_nested_field():

            differences = deep_diff(
                {
                    "a": {
                        "b": 1
                    }
                },
                {
                    "a": {
                        "b": 2
                    }
                },
            )

            assert differences == [
                {
                    "field": "a.b",
                    "old": 1,
                    "new": 2,
                }
            ]
        '''
    ),
    encoding="utf-8",
)


# ============================================================
# 2. discovery
# ============================================================

(TESTS / "test_discovery.py").write_text(
    textwrap.dedent(
        r'''
        import pandas as pd

        from politdata.discovery import (
            build_organization_manifest,
            compare_manifests,
        )


        def test_build_organization_manifest_party_and_office():

            parties = [
                {
                    "id": "party-1",
                    "code": "12345678",
                    "name": "PARTY",
                    "is_active": True,
                    "created_at": "2020-01-01",
                    "updated_at": "2025-01-01",
                    "regional_offices": [
                        {
                            "id": "office-1",
                            "code": "7654321",
                            "name": "OFFICE",
                            "is_active": True,
                        }
                    ],
                }
            ]

            df = build_organization_manifest(
                parties,
                discovered_at_utc=
                    "2026-01-01T00:00:00+00:00",
            )

            assert len(df) == 2

            party = (
                df.loc[
                    df["organization_id"]
                    ==
                    "party-1"
                ]
                .iloc[0]
            )

            office = (
                df.loc[
                    df["organization_id"]
                    ==
                    "office-1"
                ]
                .iloc[0]
            )

            assert (
                party["entity_type"]
                ==
                "party"
            )

            assert (
                office["entity_type"]
                ==
                "office"
            )

            assert (
                office["root_party_id"]
                ==
                "party-1"
            )

            # Important production rule:
            # organization codes are identifiers,
            # not numbers to be padded.
            assert (
                office["code"]
                ==
                "7654321"
            )


        def test_compare_manifests_separates_index_and_refresh_changes():

            previous = pd.DataFrame(
                [
                    {
                        "organization_id": "a",
                        "root_party_id": "a",
                        "parent_id": None,
                        "entity_type": "party",
                        "code": "1",
                        "name": "OLD",
                        "is_active": True,
                        "source_updated_at": "2025-01-01",
                    },
                    {
                        "organization_id": "b",
                        "root_party_id": "b",
                        "parent_id": None,
                        "entity_type": "party",
                        "code": "2",
                        "name": "B",
                        "is_active": True,
                        "source_updated_at": "2025-01-01",
                    },
                ]
            )

            current = previous.copy()

            current.loc[
                current["organization_id"] == "a",
                "name",
            ] = "NEW"

            current.loc[
                current["organization_id"] == "b",
                "source_updated_at",
            ] = "2026-01-01"

            (
                summary,
                new_df,
                disappeared_df,
                index_changed_df,
                refresh_candidates_df,
            ) = compare_manifests(
                current,
                previous,
            )

            assert summary["new"] == 0
            assert summary["disappeared"] == 0

            assert (
                set(
                    index_changed_df[
                        "organization_id"
                    ]
                )
                ==
                {"a"}
            )

            assert (
                set(
                    refresh_candidates_df[
                        "organization_id"
                    ]
                )
                ==
                {"a", "b"}
            )
        '''
    ),
    encoding="utf-8",
)


# ============================================================
# 3. reports
# ============================================================

(TESTS / "test_reports.py").write_text(
    textwrap.dedent(
        r'''
        import pandas as pd

        from politdata.reports import (
            classify_period_type,
            reports_to_manifest,
            add_periodicity_flags,
        )


        def test_quarter_five_is_annual():

            assert (
                classify_period_type(1)
                ==
                "quarterly"
            )

            assert (
                classify_period_type(4)
                ==
                "quarterly"
            )

            assert (
                classify_period_type(5)
                ==
                "annual"
            )

            assert (
                classify_period_type(None)
                ==
                "missing"
            )


        def test_reports_to_manifest_preserves_source_period():

            organization = {
                "organization_id": "org-1",
                "root_party_id": "party-1",
                "entity_type": "office",
            }

            reports = [
                {
                    "id": "report-1",
                    "party_id": "org-1",
                    "is_party_office": True,
                    "year": 2025,
                    "quarter": 5,
                    "public_summary": {
                        "v": 2,
                        "generated_at":
                            "2025-01-01",
                    },
                }
            ]

            df = reports_to_manifest(
                organization,
                reports,
                discovered_at_utc=
                    "2026-01-01T00:00:00+00:00",
            )

            assert len(df) == 1

            row = df.iloc[0]

            assert row["quarter"] == 5
            assert row["period_type"] == "annual"

            assert (
                row[
                    "party_id_matches_organization"
                ]
                is True
                or
                bool(
                    row[
                        "party_id_matches_organization"
                    ]
                )
                is True
            )


        def test_annual_preference_does_not_delete_quarterlies():

            df = pd.DataFrame(
                [
                    {
                        "report_id": "q1",
                        "organization_id": "o1",
                        "year": 2025,
                        "period_type": "quarterly",
                    },
                    {
                        "report_id": "q2",
                        "organization_id": "o1",
                        "year": 2025,
                        "period_type": "quarterly",
                    },
                    {
                        "report_id": "annual",
                        "organization_id": "o1",
                        "year": 2025,
                        "period_type": "annual",
                    },
                    {
                        "report_id": "only-q1",
                        "organization_id": "o2",
                        "year": 2025,
                        "period_type": "quarterly",
                    },
                ]
            )

            result = add_periodicity_flags(
                df
            )

            # Nothing physically disappears.
            assert len(result) == 4

            selected_o1 = set(
                result.loc[
                    (
                        result["organization_id"]
                        ==
                        "o1"
                    )
                    &
                    result[
                        "include_by_annual_preference"
                    ],
                    "report_id",
                ]
            )

            assert selected_o1 == {
                "annual"
            }

            selected_o2 = set(
                result.loc[
                    (
                        result["organization_id"]
                        ==
                        "o2"
                    )
                    &
                    result[
                        "include_by_annual_preference"
                    ],
                    "report_id",
                ]
            )

            assert selected_o2 == {
                "only-q1"
            }
        '''
    ),
    encoding="utf-8",
)


# ============================================================
# 4. report_details
# ============================================================

(TESTS / "test_report_details.py").write_text(
    textwrap.dedent(
        r'''
        import pytest

        from politdata.report_details import (
            raw_payload_hash,
            report_detail_content_hash,
            validate_report_detail_payload,
            property_paper_count,
        )


        def test_public_summary_changes_raw_hash_but_not_semantic_hash():

            first = {
                "code": 0,
                "results": {
                    "id": "r1",
                    "year": 2025,
                    "quarter": 1,
                    "public_summary": {
                        "v": 1,
                        "text": "old",
                    },
                },
            }

            second = {
                "code": 0,
                "results": {
                    "id": "r1",
                    "year": 2025,
                    "quarter": 1,
                    "public_summary": {
                        "v": 2,
                        "text": "new",
                    },
                },
            }

            assert (
                raw_payload_hash(first)
                !=
                raw_payload_hash(second)
            )

            assert (
                report_detail_content_hash(first)
                ==
                report_detail_content_hash(second)
            )


        def test_semantic_hash_changes_when_report_content_changes():

            first = {
                "code": 0,
                "results": {
                    "id": "r1",
                    "year": 2025,
                },
            }

            second = {
                "code": 0,
                "results": {
                    "id": "r1",
                    "year": 2026,
                },
            }

            assert (
                report_detail_content_hash(first)
                !=
                report_detail_content_hash(second)
            )


        def test_validate_report_detail_payload():

            payload = {
                "code": 0,
                "results": {
                    "id": "r1",
                },
            }

            results = (
                validate_report_detail_payload(
                    payload,
                    "r1",
                )
            )

            assert results["id"] == "r1"


        def test_validate_report_detail_rejects_wrong_id():

            payload = {
                "code": 0,
                "results": {
                    "id": "wrong",
                },
            }

            with pytest.raises(
                ValueError
            ):

                validate_report_detail_payload(
                    payload,
                    "expected",
                )


        def test_property_paper_count_uses_source_section():

            payload = {
                "results": {
                    "properties": {
                        "property_paper": [
                            {"id": "a"},
                            {"id": "b"},
                        ]
                    }
                }
            }

            assert (
                property_paper_count(
                    payload
                )
                ==
                2
            )
        '''
    ),
    encoding="utf-8",
)


# ============================================================
# 5. RUN COMPLETE TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print(
    result.stdout
)

if result.stderr:
    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Characterization tests failed. "
        "Do not start source refactoring yet."
    )


print()
print(
    "Safety net extended successfully."
)

.......................................................................F [ 91%]
.......                                                                  [100%]
================================== FAILURES ===================================
_________________________ test_extract_property_list __________________________

    def test_extract_property_list():
    
        detail = {
            "properties": {
                "realty": [
                    {
                        "id": "x"
                    },
                    {
                        "id": "y"
                    },
                ]
            }
        }
    
        rows = extract_section_rows(
            detail,
            "realty",
        )
    
>       assert len(rows) == 2
E       assert 0 == 2
E        +  where 0 = len([])

tests\test_report_sections.py:28: AssertionError
=========================== short test summary info ===========================
FAILED tests/test_report_sections.py::test_extract

RuntimeError: Characterization tests failed. Do not start source refactoring yet.

In [125]:
from pathlib import Path
import subprocess
import sys


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

TEST_PATH = (
    ROOT
    / "tests"
    / "test_report_sections.py"
)


# ============================================================
# 1. PATCH STALE TEST FIXTURE
#
# Old exploratory assumption:
#   properties["realty"]
#
# Confirmed actual PolitData RAW:
#   properties["property_object"]
# ============================================================

text = TEST_PATH.read_text(
    encoding="utf-8"
)


old = '''        "properties": {
            "realty": [
'''

new = '''        "properties": {
            "property_object": [
'''


if old not in text:

    raise RuntimeError(
        "Expected old realty fixture was not found. "
        "Test file was not modified."
    )


text = text.replace(
    old,
    new,
    1,
)


# ============================================================
# 2. ADD EXPLICIT API SECTION-MAP REGRESSION TEST
# ============================================================

mapping_test = r'''


def test_confirmed_property_section_paths():

    from politdata.normalization.report_sections import (
        SECTION_PATHS,
    )

    assert SECTION_PATHS["realty"] == (
        "properties",
        "property_object",
    )

    assert SECTION_PATHS["transport"] == (
        "properties",
        "property_transport",
    )

    assert SECTION_PATHS["movable"] == (
        "properties",
        "property_movable",
    )

    assert SECTION_PATHS["intangible"] == (
        "properties",
        "property_intangible_asset",
    )

    assert SECTION_PATHS["paper"] == (
        "properties",
        "property_paper",
    )
'''


if (
    "def test_confirmed_property_section_paths"
    not in text
):

    text = (
        text.rstrip()
        +
        mapping_test
        +
        "\n"
    )


TEST_PATH.write_text(
    text,
    encoding="utf-8",
)


print(
    "Updated:",
    TEST_PATH
)


# ============================================================
# 3. RUN COMPLETE TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print(result.stdout)


if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests still fail. "
        "Do not start refactoring yet."
    )


print()
print(
    "Safety net is green. "
    "Confirmed PolitData property paths are now "
    "encoded in regression tests."
)

Updated: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_report_sections.py

........................................................................ [ 90%]
........                                                                 [100%]
80 passed in 1.01s

Return code: 0

Safety net is green. Confirmed PolitData property paths are now encoded in regression tests.


In [126]:
from pathlib import Path
import subprocess
import sys
import textwrap


ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

ENRICHMENT_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "enrichment"
)

TESTS_DIR = (
    ROOT
    / "tests"
)

ENRICHMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. PACKAGE __init__.py
# ============================================================

(
    ENRICHMENT_DIR
    / "__init__.py"
).write_text(
    "",
    encoding="utf-8",
)


# ============================================================
# 2. PURE PAYMENT ENRICHMENT RULES
#
# IMPORTANT:
#
# This module contains analytical/business rules only.
#
# It does NOT:
# - read RAW
# - read/write parquet
# - call API
# - perform joins
# - normalize source payment rows
#
# Batch orchestration will be added separately after these
# rules are frozen and tested.
# ============================================================

MODULE = r'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional


STATE_STATUTORY_FUNDING = (
    "state_statutory_funding"
)

STATE_CAMPAIGN_REIMBURSEMENT = (
    "state_campaign_reimbursement"
)


STATE_FUNDING_ACCOUNT_EVIDENCE = (
    "positive_transaction_in_state_funding_section"
)


STATE_ACCOUNT_HUMAN_LABELS = {

    STATE_STATUTORY_FUNDING:
        (
            "Рахунок державного фінансування "
            "статутної діяльності"
        ),

    STATE_CAMPAIGN_REIMBURSEMENT:
        (
            "Рахунок відшкодування витрат "
            "на передвиборну агітацію"
        ),
}


STATE_ACCOUNT_MACHINE_TYPES = {

    STATE_STATUTORY_FUNDING:
        "state_statutory_funding_account",

    STATE_CAMPAIGN_REIMBURSEMENT:
        "state_campaign_reimbursement_account",
}


ORDINARY_ACCOUNT_MARKERS = (
    "поточн",
    "розрахунк",
)


@dataclass(frozen=True)
class AccountClassification:
    """
    Analytical classification of the reporting
    party/office account attached to a payment row.
    """

    party_account_type: str

    party_account_type_analytical: str

    party_account_type_resolution_method: str


@dataclass(frozen=True)
class PaymentClassification:
    """
    Final analytical classification of a payment row.
    """

    analytical_payment_type: str

    was_reclassified: bool

    reclassification_rule: Optional[str]

    funding_source_analytical: Optional[str]


def _clean_optional_text(
    value,
) -> Optional[str]:

    if value is None:
        return None

    text = str(
        value
    ).strip()

    return (
        text
        if text
        else None
    )


def is_ordinary_account_type(
    party_account_type_source,
) -> bool:
    """
    Conservative textual classification used only when
    the account is NOT confirmed from state_funding receipts.

    This does not establish funding origin.
    """

    text = _clean_optional_text(
        party_account_type_source
    )

    if text is None:
        return False

    lowered = text.lower()

    return any(
        marker in lowered
        for marker
        in ORDINARY_ACCOUNT_MARKERS
    )


def classify_party_account(
    *,
    state_funding_account_confirmed: bool,
    state_funding_form_code=None,
    party_account_type_source=None,
) -> AccountClassification:
    """
    State-funding evidence has priority over property_moneys
    account labels.

    A confirmed state account means that a positive receipt
    was observed in the actual state_funding section.

    property_moneys remains auxiliary account metadata.
    """

    state_form = _clean_optional_text(
        state_funding_form_code
    )

    source_type = _clean_optional_text(
        party_account_type_source
    )


    if state_funding_account_confirmed:

        if state_form in STATE_ACCOUNT_HUMAN_LABELS:

            return AccountClassification(

                party_account_type=
                    STATE_ACCOUNT_HUMAN_LABELS[
                        state_form
                    ],

                party_account_type_analytical=
                    STATE_ACCOUNT_MACHINE_TYPES[
                        state_form
                    ],

                party_account_type_resolution_method=
                    STATE_FUNDING_ACCOUNT_EVIDENCE,
            )


        # This situation should normally be caught by QA:
        # confirmed state account without a known form.
        return AccountClassification(

            party_account_type=(
                source_type
                or
                "Тип рахунку не визначено"
            ),

            party_account_type_analytical=
                "other_declared_account"
                if source_type
                else
                "unknown",

            party_account_type_resolution_method=
                STATE_FUNDING_ACCOUNT_EVIDENCE,
        )


    if is_ordinary_account_type(
        source_type
    ):

        return AccountClassification(

            party_account_type=
                source_type,

            party_account_type_analytical=
                "ordinary_account",

            party_account_type_resolution_method=
                "property_moneys_source_type",
        )


    if source_type:

        return AccountClassification(

            party_account_type=
                source_type,

            party_account_type_analytical=
                "other_declared_account",

            party_account_type_resolution_method=
                "property_moneys_source_type",
        )


    return AccountClassification(

        party_account_type=
            "Тип рахунку не визначено",

        party_account_type_analytical=
            "unknown",

        party_account_type_resolution_method=
            "unresolved",
    )


def classify_internal_transfer_direction(
    *,
    internal_transfer: bool,
    internal_counterparty_organization_id=None,
    source_organization_id=None,
    source_organization_level=None,
    destination_organization_id=None,
    destination_organization_level=None,
) -> Optional[str]:
    """
    Classify physical transfer direction after a same-root
    organization has already been matched by strict EDRPOU.

    No fuzzy name or IBAN matching is performed here.
    """

    if not internal_transfer:
        return None


    if internal_counterparty_organization_id is None:
        return (
            "same_party_counterparty_unresolved"
        )


    if (
        source_organization_id is not None
        and
        destination_organization_id is not None
        and
        source_organization_id
        ==
        destination_organization_id
    ):

        return (
            "intra_organization"
        )


    if (
        source_organization_level
        ==
        "central"
        and
        destination_organization_level
        ==
        "office"
    ):

        return (
            "central_to_office"
        )


    if (
        source_organization_level
        ==
        "office"
        and
        destination_organization_level
        ==
        "central"
    ):

        return (
            "office_to_central"
        )


    if (
        source_organization_level
        ==
        "office"
        and
        destination_organization_level
        ==
        "office"
    ):

        return (
            "office_to_office"
        )


    return (
        "other_same_party_transfer"
    )


def classify_internal_transfer_funding_source(
    *,
    internal_transfer_direction=None,
    payment_direction=None,
    state_funding_account_confirmed: bool = False,
    state_funding_form_code=None,
    party_account_type_analytical=None,
) -> Optional[str]:
    """
    Funding source can be established for central -> office
    only from the outgoing central row, because that row
    exposes the centre's source account.

    The corresponding incoming office row does not reveal
    that source account by itself.
    """

    if (
        internal_transfer_direction
        ==
        "central_to_office"
        and
        payment_direction
        ==
        "outgoing"
    ):

        if state_funding_account_confirmed:

            return (
                _clean_optional_text(
                    state_funding_form_code
                )
                or
                "unknown"
            )


        if (
            party_account_type_analytical
            ==
            "ordinary_account"
        ):

            return (
                "private_or_non_state"
            )


        return (
            "unknown"
        )


    if (
        internal_transfer_direction
        ==
        "central_to_office"
        and
        payment_direction
        ==
        "incoming"
    ):

        return (
            "unknown_source_account"
        )


    if (
        internal_transfer_direction
        in {
            "office_to_central",
            "office_to_office",
        }
    ):

        return (
            "mixed_or_unknown"
        )


    return None


def classify_payment(
    *,
    source_payment_type: str,
    organization_level=None,
    state_funding_account_confirmed: bool = False,
    state_funding_form_code=None,
    party_account_type_analytical=None,
) -> PaymentClassification:
    """
    Final source-preserving analytical payment classification.

    Source section remains unchanged elsewhere.

    Rules
    -----
    state_funding:
        stays state_funding and receives its state funding form.

    budget_expenses:
        central + confirmed state account
            -> stays budget_expenses

        all other cases
            -> analytical outgoing_expenses

    other payment sections:
        retain their source payment type.
    """

    source_type = _clean_optional_text(
        source_payment_type
    )


    if source_type is None:

        raise ValueError(
            "source_payment_type is required"
        )


    state_form = _clean_optional_text(
        state_funding_form_code
    )


    # --------------------------------------------------------
    # Incoming state financing
    # --------------------------------------------------------

    if source_type == "state_funding":

        return PaymentClassification(

            analytical_payment_type=
                "state_funding",

            was_reclassified=
                False,

            reclassification_rule=
                None,

            funding_source_analytical=
                state_form
                or
                "unknown",
        )


    # --------------------------------------------------------
    # Source budget_expenses
    # --------------------------------------------------------

    if source_type == "budget_expenses":

        if (
            organization_level
            ==
            "central"
            and
            state_funding_account_confirmed
        ):

            return PaymentClassification(

                analytical_payment_type=
                    "budget_expenses",

                was_reclassified=
                    False,

                reclassification_rule=
                    None,

                funding_source_analytical=
                    state_form
                    or
                    "unknown",
            )


        if (
            organization_level
            ==
            "office"
        ):

            return PaymentClassification(

                analytical_payment_type=
                    "outgoing_expenses",

                was_reclassified=
                    True,

                reclassification_rule=(
                    "office_budget_expense_"
                    "treated_as_ordinary_or_mixed"
                ),

                funding_source_analytical=
                    "mixed_or_unknown",
            )


        if (
            party_account_type_analytical
            ==
            "ordinary_account"
        ):

            funding_source = (
                "private_or_non_state"
            )

        else:

            funding_source = (
                "unknown"
            )


        return PaymentClassification(

            analytical_payment_type=
                "outgoing_expenses",

            was_reclassified=
                True,

            reclassification_rule=(
                "central_budget_expense_without_"
                "confirmed_state_account"
            ),

            funding_source_analytical=
                funding_source,
        )


    # --------------------------------------------------------
    # All other source sections
    # --------------------------------------------------------

    return PaymentClassification(

        analytical_payment_type=
            source_type,

        was_reclassified=
            False,

        reclassification_rule=
            None,

        funding_source_analytical=
            None,
    )
'''


MODULE_PATH = (
    ENRICHMENT_DIR
    / "payments.py"
)


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    MODULE_PATH
)


# ============================================================
# 3. CHARACTERIZATION / BUSINESS-RULE TESTS
# ============================================================

TEST_CODE = r'''
from politdata.enrichment.payments import (
    STATE_STATUTORY_FUNDING,
    classify_party_account,
    classify_internal_transfer_direction,
    classify_internal_transfer_funding_source,
    classify_payment,
)


def test_confirmed_state_account_overrides_declared_current_type():

    result = classify_party_account(
        state_funding_account_confirmed=True,
        state_funding_form_code=
            STATE_STATUTORY_FUNDING,
        party_account_type_source=
            "Поточний рахунок",
    )

    assert (
        result.party_account_type_analytical
        ==
        "state_statutory_funding_account"
    )

    assert (
        result.party_account_type_resolution_method
        ==
        "positive_transaction_in_state_funding_section"
    )


def test_current_account_is_ordinary_but_not_state():

    result = classify_party_account(
        state_funding_account_confirmed=False,
        party_account_type_source=
            "Поточний рахунок",
    )

    assert (
        result.party_account_type_analytical
        ==
        "ordinary_account"
    )


def test_unknown_account_type_is_preserved_as_declared_other():

    result = classify_party_account(
        state_funding_account_confirmed=False,
        party_account_type_source=
            "Інше-Транзитний рахунок",
    )

    assert (
        result.party_account_type_analytical
        ==
        "other_declared_account"
    )

    assert (
        result.party_account_type
        ==
        "Інше-Транзитний рахунок"
    )


def test_missing_account_type_is_unresolved():

    result = classify_party_account(
        state_funding_account_confirmed=False,
        party_account_type_source=None,
    )

    assert (
        result.party_account_type_analytical
        ==
        "unknown"
    )

    assert (
        result.party_account_type_resolution_method
        ==
        "unresolved"
    )


def test_internal_transfer_direction_central_to_office():

    result = classify_internal_transfer_direction(
        internal_transfer=True,
        internal_counterparty_organization_id=
            "office-1",
        source_organization_id=
            "central-1",
        source_organization_level=
            "central",
        destination_organization_id=
            "office-1",
        destination_organization_level=
            "office",
    )

    assert result == "central_to_office"


def test_internal_transfer_direction_intra_organization():

    result = classify_internal_transfer_direction(
        internal_transfer=True,
        internal_counterparty_organization_id=
            "central-1",
        source_organization_id=
            "central-1",
        source_organization_level=
            "central",
        destination_organization_id=
            "central-1",
        destination_organization_level=
            "central",
    )

    assert result == "intra_organization"


def test_internal_transfer_direction_unresolved_counterparty():

    result = classify_internal_transfer_direction(
        internal_transfer=True,
        internal_counterparty_organization_id=None,
        source_organization_id=
            "central-1",
        source_organization_level=
            "central",
        destination_organization_id=None,
        destination_organization_level=None,
    )

    assert (
        result
        ==
        "same_party_counterparty_unresolved"
    )


def test_central_to_office_state_transfer():

    result = (
        classify_internal_transfer_funding_source(
            internal_transfer_direction=
                "central_to_office",
            payment_direction=
                "outgoing",
            state_funding_account_confirmed=
                True,
            state_funding_form_code=
                STATE_STATUTORY_FUNDING,
            party_account_type_analytical=
                "state_statutory_funding_account",
        )
    )

    assert (
        result
        ==
        STATE_STATUTORY_FUNDING
    )


def test_central_to_office_private_transfer():

    result = (
        classify_internal_transfer_funding_source(
            internal_transfer_direction=
                "central_to_office",
            payment_direction=
                "outgoing",
            state_funding_account_confirmed=
                False,
            party_account_type_analytical=
                "ordinary_account",
        )
    )

    assert result == "private_or_non_state"


def test_central_to_office_incoming_does_not_infer_source():

    result = (
        classify_internal_transfer_funding_source(
            internal_transfer_direction=
                "central_to_office",
            payment_direction=
                "incoming",
            state_funding_account_confirmed=
                False,
            party_account_type_analytical=
                "ordinary_account",
        )
    )

    assert result == "unknown_source_account"


def test_office_to_office_is_mixed_or_unknown():

    result = (
        classify_internal_transfer_funding_source(
            internal_transfer_direction=
                "office_to_office",
            payment_direction=
                "outgoing",
        )
    )

    assert result == "mixed_or_unknown"


def test_state_funding_payment_keeps_source_type():

    result = classify_payment(
        source_payment_type=
            "state_funding",
        organization_level=
            "central",
        state_funding_account_confirmed=
            True,
        state_funding_form_code=
            STATE_STATUTORY_FUNDING,
        party_account_type_analytical=
            "state_statutory_funding_account",
    )

    assert (
        result.analytical_payment_type
        ==
        "state_funding"
    )

    assert (
        result.funding_source_analytical
        ==
        STATE_STATUTORY_FUNDING
    )

    assert result.was_reclassified is False


def test_central_budget_expense_on_confirmed_state_account_stays_budget():

    result = classify_payment(
        source_payment_type=
            "budget_expenses",
        organization_level=
            "central",
        state_funding_account_confirmed=
            True,
        state_funding_form_code=
            STATE_STATUTORY_FUNDING,
        party_account_type_analytical=
            "state_statutory_funding_account",
    )

    assert (
        result.analytical_payment_type
        ==
        "budget_expenses"
    )

    assert result.was_reclassified is False

    assert (
        result.funding_source_analytical
        ==
        STATE_STATUTORY_FUNDING
    )


def test_central_budget_expense_on_current_account_becomes_outgoing():

    result = classify_payment(
        source_payment_type=
            "budget_expenses",
        organization_level=
            "central",
        state_funding_account_confirmed=
            False,
        party_account_type_analytical=
            "ordinary_account",
    )

    assert (
        result.analytical_payment_type
        ==
        "outgoing_expenses"
    )

    assert result.was_reclassified is True

    assert (
        result.reclassification_rule
        ==
        "central_budget_expense_without_confirmed_state_account"
    )

    assert (
        result.funding_source_analytical
        ==
        "private_or_non_state"
    )


def test_office_budget_expense_becomes_outgoing_mixed():

    result = classify_payment(
        source_payment_type=
            "budget_expenses",
        organization_level=
            "office",
        state_funding_account_confirmed=
            False,
        party_account_type_analytical=
            "ordinary_account",
    )

    assert (
        result.analytical_payment_type
        ==
        "outgoing_expenses"
    )

    assert result.was_reclassified is True

    assert (
        result.funding_source_analytical
        ==
        "mixed_or_unknown"
    )

    assert (
        result.reclassification_rule
        ==
        "office_budget_expense_treated_as_ordinary_or_mixed"
    )


def test_other_payment_type_is_not_reclassified():

    result = classify_payment(
        source_payment_type=
            "outgoing_expenses",
        organization_level=
            "central",
        state_funding_account_confirmed=
            False,
        party_account_type_analytical=
            "ordinary_account",
    )

    assert (
        result.analytical_payment_type
        ==
        "outgoing_expenses"
    )

    assert result.was_reclassified is False

    assert (
        result.funding_source_analytical
        is None
    )
'''


TEST_PATH = (
    TESTS_DIR
    / "test_payment_enrichment.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 4. RUN COMPLETE SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Payment enrichment rule extraction failed. "
        "No parquet data were modified."
    )


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Payment analytical business rules now live in:"
)

print(
    MODULE_PATH
)

print()
print(
    "No RAW files were read."
)

print(
    "No parquet files were read or modified."
)

print(
    "normalization/payments.py was not modified."
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payments.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_payment_enrichment.py

TESTS
........................................................................ [ 75%]
........................                                                 [100%]
96 passed in 2.77s

Return code: 0

DONE
Payment analytical business rules now live in:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payments.py

No RAW files were read.
No parquet files were read or modified.
normalization/payments.py was not modified.


In [127]:
from pathlib import Path
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

ENRICHMENT_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "enrichment"
)

TESTS_DIR = (
    ROOT
    / "tests"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)


# ============================================================
# 1. WRITE FRAME-LEVEL ENRICHMENT / PARITY MODULE
#
# This layer applies the already-tested pure business rules
# to a DataFrame.
#
# IMPORTANT:
# - no RAW access
# - no API
# - no joins yet
# - no writes to production parquet
#
# It assumes prerequisite facts have already been resolved:
# - confirmed state account
# - funding form
# - internal counterparty organization
# - physical transfer source/destination
# ============================================================

MODULE_CODE = r'''
from __future__ import annotations

from pathlib import Path

import pandas as pd

from politdata.enrichment.payments import (
    classify_party_account,
    classify_internal_transfer_direction,
    classify_internal_transfer_funding_source,
    classify_payment,
)


PAYMENT_SECTIONS = (
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
)


DERIVED_COLUMNS = (
    "party_account_type",
    "party_account_type_analytical",
    "party_account_type_resolution_method",
    "internal_transfer_direction",
    "internal_transfer_funding_source",
    "analytical_payment_type",
    "was_reclassified",
    "reclassification_rule",
    "funding_source_analytical",
)


def _none_if_missing(value):

    if pd.isna(value):
        return None

    return value


def _bool_value(value) -> bool:

    if pd.isna(value):
        return False

    return bool(value)


def _source_payment_type(
    row,
    section: str,
):

    if (
        "source_payment_type"
        in row.index
        and
        pd.notna(
            row["source_payment_type"]
        )
    ):
        return str(
            row["source_payment_type"]
        )

    return section


def derive_payment_enrichment(
    df: pd.DataFrame,
    *,
    section: str,
) -> pd.DataFrame:
    """
    Recalculate analytical payment fields from resolved
    prerequisite facts.

    This is deliberately not responsible for resolving:
    - organization identities
    - EDRPOU counterparty matching
    - confirmed state-account joins

    Those belong to the future batch orchestration layer.
    """

    if section not in PAYMENT_SECTIONS:

        raise ValueError(
            f"Unknown payment section: {section}"
        )


    output = []


    for row in df.to_dict(
        orient="records"
    ):

        source_type = (
            row.get(
                "source_payment_type"
            )
        )

        if source_type is None:
            source_type = section


        account = classify_party_account(

            state_funding_account_confirmed=
                _bool_value(
                    row.get(
                        "state_funding_account_confirmed"
                    )
                ),

            state_funding_form_code=
                _none_if_missing(
                    row.get(
                        "state_funding_form_code"
                    )
                ),

            party_account_type_source=
                _none_if_missing(
                    row.get(
                        "party_account_type_source"
                    )
                ),
        )


        transfer_direction = (
            classify_internal_transfer_direction(

                internal_transfer=
                    _bool_value(
                        row.get(
                            "internal_transfer"
                        )
                    ),

                internal_counterparty_organization_id=
                    _none_if_missing(
                        row.get(
                            "internal_counterparty_organization_id"
                        )
                    ),

                source_organization_id=
                    _none_if_missing(
                        row.get(
                            "internal_transfer_source_organization_id"
                        )
                    ),

                source_organization_level=
                    _none_if_missing(
                        row.get(
                            "internal_transfer_source_organization_level"
                        )
                    ),

                destination_organization_id=
                    _none_if_missing(
                        row.get(
                            "internal_transfer_destination_organization_id"
                        )
                    ),

                destination_organization_level=
                    _none_if_missing(
                        row.get(
                            "internal_transfer_destination_organization_level"
                        )
                    ),
            )
        )


        transfer_funding = (
            classify_internal_transfer_funding_source(

                internal_transfer_direction=
                    transfer_direction,

                payment_direction=
                    _none_if_missing(
                        row.get(
                            "payment_direction"
                        )
                    ),

                state_funding_account_confirmed=
                    _bool_value(
                        row.get(
                            "state_funding_account_confirmed"
                        )
                    ),

                state_funding_form_code=
                    _none_if_missing(
                        row.get(
                            "state_funding_form_code"
                        )
                    ),

                party_account_type_analytical=
                    account.party_account_type_analytical,
            )
        )


        payment = classify_payment(

            source_payment_type=
                source_type,

            organization_level=
                _none_if_missing(
                    row.get(
                        "organization_level"
                    )
                ),

            state_funding_account_confirmed=
                _bool_value(
                    row.get(
                        "state_funding_account_confirmed"
                    )
                ),

            state_funding_form_code=
                _none_if_missing(
                    row.get(
                        "state_funding_form_code"
                    )
                ),

            party_account_type_analytical=
                account.party_account_type_analytical,
        )


        output.append(
            {
                "party_account_type":
                    account.party_account_type,

                "party_account_type_analytical":
                    account.party_account_type_analytical,

                "party_account_type_resolution_method":
                    account.party_account_type_resolution_method,

                "internal_transfer_direction":
                    transfer_direction,

                "internal_transfer_funding_source":
                    transfer_funding,

                "analytical_payment_type":
                    payment.analytical_payment_type,

                "was_reclassified":
                    payment.was_reclassified,

                "reclassification_rule":
                    payment.reclassification_rule,

                "funding_source_analytical":
                    payment.funding_source_analytical,
            }
        )


    return pd.DataFrame(
        output,
        index=df.index,
    )


def _comparison_series(
    series: pd.Series,
) -> pd.Series:
    """
    Normalize null representation for parity comparison only.
    """

    return (
        series
        .astype("string")
        .fillna("<NULL>")
    )


def validate_payment_enrichment_frame(
    df: pd.DataFrame,
    *,
    section: str,
):
    """
    Compare recalculated analytical fields against the
    currently persisted enriched dataset.

    Returns:
        summary_df,
        mismatch_samples_df
    """

    recalculated = (
        derive_payment_enrichment(
            df,
            section=section,
        )
    )


    summaries = []
    samples = []


    for column in DERIVED_COLUMNS:

        if column not in df.columns:

            summaries.append(
                {
                    "section":
                        section,

                    "column":
                        column,

                    "rows":
                        len(df),

                    "mismatches":
                        None,

                    "status":
                        "missing_existing_column",
                }
            )

            continue


        old = _comparison_series(
            df[column]
        )

        new = _comparison_series(
            recalculated[column]
        )


        mismatch = (
            old
            !=
            new
        )


        mismatch_count = int(
            mismatch.sum()
        )


        summaries.append(
            {
                "section":
                    section,

                "column":
                    column,

                "rows":
                    len(df),

                "mismatches":
                    mismatch_count,

                "status":
                    (
                        "match"
                        if mismatch_count == 0
                        else
                        "mismatch"
                    ),
            }
        )


        if mismatch_count:

            sample_indices = (
                mismatch[
                    mismatch
                ]
                .index[:20]
            )


            for idx in sample_indices:

                sample = {
                    "section":
                        section,

                    "row_index":
                        idx,

                    "column":
                        column,

                    "existing":
                        old.loc[idx],

                    "recalculated":
                        new.loc[idx],
                }


                for context_column in (
                    "source_report_id",
                    "party_name_current",
                    "organization_name_current",
                    "organization_level",
                    "source_payment_type",
                    "payment_direction",
                    "party_account_iban",
                    "party_account_type_source",
                    "state_funding_account_confirmed",
                    "state_funding_form_code",
                    "internal_transfer",
                ):

                    if context_column in df.columns:

                        sample[
                            context_column
                        ] = df.loc[
                            idx,
                            context_column,
                        ]


                samples.append(
                    sample
                )


    return (
        pd.DataFrame(
            summaries
        ),
        pd.DataFrame(
            samples
        ),
    )


def validate_payment_directory(
    payment_dir,
):
    """
    Full read-only parity check across all 8 enriched payment
    parquet files.
    """

    payment_dir = Path(
        payment_dir
    )


    summaries = []
    samples = []


    for section in PAYMENT_SECTIONS:

        path = (
            payment_dir
            / f"{section}.parquet"
        )


        if not path.exists():

            raise FileNotFoundError(
                path
            )


        df = pd.read_parquet(
            path
        )


        summary, mismatch_samples = (
            validate_payment_enrichment_frame(
                df,
                section=section,
            )
        )


        summaries.append(
            summary
        )


        if len(
            mismatch_samples
        ):

            samples.append(
                mismatch_samples
            )


    summary_df = pd.concat(
        summaries,
        ignore_index=True,
    )


    if samples:

        sample_df = pd.concat(
            samples,
            ignore_index=True,
        )

    else:

        sample_df = pd.DataFrame()


    return (
        summary_df,
        sample_df,
    )
'''


MODULE_PATH = (
    ENRICHMENT_DIR
    / "payment_batch.py"
)


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    MODULE_PATH
)


# ============================================================
# 2. SMALL UNIT TESTS FOR FRAME LAYER
# ============================================================

TEST_CODE = r'''
import pandas as pd

from politdata.enrichment.payment_batch import (
    derive_payment_enrichment,
    validate_payment_enrichment_frame,
)


def test_frame_derivation_state_budget():

    df = pd.DataFrame(
        [
            {
                "source_payment_type":
                    "budget_expenses",

                "organization_level":
                    "central",

                "state_funding_account_confirmed":
                    True,

                "state_funding_form_code":
                    "state_statutory_funding",

                "party_account_type_source":
                    "Поточний рахунок",

                "internal_transfer":
                    False,

                "payment_direction":
                    "outgoing",
            }
        ]
    )


    result = derive_payment_enrichment(
        df,
        section="budget_expenses",
    )


    row = result.iloc[0]


    assert (
        row[
            "party_account_type_analytical"
        ]
        ==
        "state_statutory_funding_account"
    )

    assert (
        row[
            "analytical_payment_type"
        ]
        ==
        "budget_expenses"
    )

    assert (
        row[
            "funding_source_analytical"
        ]
        ==
        "state_statutory_funding"
    )


def test_frame_derivation_office_budget():

    df = pd.DataFrame(
        [
            {
                "source_payment_type":
                    "budget_expenses",

                "organization_level":
                    "office",

                "state_funding_account_confirmed":
                    False,

                "state_funding_form_code":
                    None,

                "party_account_type_source":
                    "Поточний рахунок",

                "internal_transfer":
                    False,

                "payment_direction":
                    "outgoing",
            }
        ]
    )


    result = derive_payment_enrichment(
        df,
        section="budget_expenses",
    )


    row = result.iloc[0]


    assert (
        row[
            "analytical_payment_type"
        ]
        ==
        "outgoing_expenses"
    )

    assert (
        bool(
            row[
                "was_reclassified"
            ]
        )
        is True
    )

    assert (
        row[
            "funding_source_analytical"
        ]
        ==
        "mixed_or_unknown"
    )


def test_parity_validator_reports_zero_mismatch():

    df = pd.DataFrame(
        [
            {
                "source_payment_type":
                    "outgoing_expenses",

                "organization_level":
                    "central",

                "state_funding_account_confirmed":
                    False,

                "state_funding_form_code":
                    None,

                "party_account_type_source":
                    "Поточний рахунок",

                "internal_transfer":
                    False,

                "payment_direction":
                    "outgoing",

                "party_account_type":
                    "Поточний рахунок",

                "party_account_type_analytical":
                    "ordinary_account",

                "party_account_type_resolution_method":
                    "property_moneys_source_type",

                "internal_transfer_direction":
                    None,

                "internal_transfer_funding_source":
                    None,

                "analytical_payment_type":
                    "outgoing_expenses",

                "was_reclassified":
                    False,

                "reclassification_rule":
                    None,

                "funding_source_analytical":
                    None,
            }
        ]
    )


    summary, samples = (
        validate_payment_enrichment_frame(
            df,
            section="outgoing_expenses",
        )
    )


    assert (
        summary[
            "mismatches"
        ].fillna(1).sum()
        ==
        0
    )

    assert samples.empty
'''


TEST_PATH = (
    TESTS_DIR
    / "test_payment_batch.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 3. RUN COMPLETE UNIT TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("UNIT TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Frame-level payment enrichment tests failed."
    )


# ============================================================
# 4. FULL PRODUCTION PARITY CHECK
#
# Reads existing enriched parquet only.
# Does NOT modify it.
# ============================================================

from politdata.enrichment.payment_batch import (
    validate_payment_directory,
)


print()
print("=" * 100)
print("FULL PAYMENT ENRICHMENT PARITY")
print("=" * 100)

print(
    "Reading existing enriched payment parquet files..."
)


summary, mismatch_samples = (
    validate_payment_directory(
        PAYMENT_DIR
    )
)


# ============================================================
# 5. SUMMARY BY DERIVED COLUMN
# ============================================================

column_summary = (
    summary
    .groupby(
        "column",
        as_index=False,
    )
    .agg(
        rows=(
            "rows",
            "sum",
        ),

        mismatches=(
            "mismatches",
            "sum",
        ),

        sections=(
            "section",
            "nunique",
        ),
    )
)


column_summary[
    "match_ratio"
] = (
    1
    -
    (
        column_summary[
            "mismatches"
        ]
        /
        column_summary[
            "rows"
        ]
    )
)


print()
print("=" * 100)
print("PARITY BY COLUMN")
print("=" * 100)

display(
    column_summary
)


# ============================================================
# 6. SUMMARY BY SECTION
# ============================================================

section_summary = (
    summary
    .groupby(
        "section",
        as_index=False,
    )
    .agg(
        comparisons=(
            "column",
            "size",
        ),

        mismatches=(
            "mismatches",
            "sum",
        ),
    )
)


print()
print("=" * 100)
print("PARITY BY PAYMENT SECTION")
print("=" * 100)

display(
    section_summary
)


# ============================================================
# 7. MISMATCH SAMPLES
# ============================================================

total_mismatches = int(
    summary[
        "mismatches"
    ]
    .fillna(0)
    .sum()
)


print()
print(
    "Total field-level mismatches:",
    f"{total_mismatches:,}"
)


if total_mismatches:

    print()
    print("=" * 100)
    print("MISMATCH SAMPLES")
    print("=" * 100)

    display(
        mismatch_samples.head(
            100
        )
    )

    raise RuntimeError(
        "New pure-function enrichment does not yet have "
        "full parity with the verified production dataset."
    )


# ============================================================
# 8. ROW CONTROL
# ============================================================

payment_rows = 0


for path in PAYMENT_DIR.glob(
    "*.parquet"
):

    payment_rows += len(
        pd.read_parquet(
            path,
            columns=[
                "source_report_id"
            ],
        )
    )


print()
print(
    "Production payment rows:",
    f"{payment_rows:,}"
)


if payment_rows != 402_028:

    raise RuntimeError(
        f"Payment baseline changed: {payment_rows:,}"
    )


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Pure-function payment enrichment has full parity "
    "with the current production dataset."
)

print(
    "No RAW files were read."
)

print(
    "No production parquet files were modified."
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_batch.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_payment_batch.py

UNIT TESTS
........................................................................ [ 72%]
...........................                                              [100%]
99 passed in 0.97s

Return code: 0

FULL PAYMENT ENRICHMENT PARITY
Reading existing enriched payment parquet files...

PARITY BY COLUMN


,column,rows,mismatches,sections,match_ratio
0,analytical_payment_type,402028,0,8,1.0
1,funding_source_analytical,402028,0,8,1.0
2,internal_transfer_direction,402028,0,8,1.0
3,internal_transfer_funding_source,402028,0,8,1.0
4,party_account_type,402028,0,8,1.0
5,party_account_type_analytical,402028,0,8,1.0
6,party_account_type_resolution_method,402028,0,8,1.0
7,reclassification_rule,402028,0,8,1.0
8,was_reclassified,402028,0,8,1.0



PARITY BY PAYMENT SECTION


,section,comparisons,mismatches
0,budget_expenses,9,0
1,monetary_contributions,9,0
2,other_contributions,9,0
3,other_incomes,9,0
4,outgoing_expenses,9,0
5,return_expenses,9,0
6,state_funding,9,0
7,transfer_expenses,9,0



Total field-level mismatches: 0

Production payment rows: 402,028

DONE
Pure-function payment enrichment has full parity with the current production dataset.
No RAW files were read.
No production parquet files were modified.


In [128]:
from pathlib import Path
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

ENRICHMENT_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "enrichment"
)

TESTS_DIR = (
    ROOT
    / "tests"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

ORG_REFERENCE_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "organization_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "state_funding_account_reference.parquet"
)


# ============================================================
# 1. PRODUCTION RESOLUTION MODULE
#
# Responsibilities:
#
# - payment direction by source section
# - party account field resolution
# - confirmed state-account join
# - state-funding form resolution
# - strict same-root EDRPOU organization matching
# - physical transfer source/destination
#
# NOT responsible for:
#
# - business classification of account/payment
# - RAW/API access
# - parquet persistence
# ============================================================

MODULE_CODE = r'''
from __future__ import annotations

from typing import Optional

import pandas as pd

from politdata.enrichment.payments import (
    classify_internal_transfer_direction,
)


INCOMING_SECTIONS = {
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
}


OUTGOING_SECTIONS = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


PAYMENT_SECTIONS = (
    INCOMING_SECTIONS
    |
    OUTGOING_SECTIONS
)


RESOLVED_COLUMNS = (
    "payment_direction",
    "party_account_iban",

    "state_funding_account_confirmed",
    "state_funding_form_source",
    "state_funding_form_code",

    "internal_counterparty_organization_id",
    "internal_counterparty_organization_name",
    "internal_counterparty_organization_level",

    "internal_transfer_source_organization_id",
    "internal_transfer_source_organization_level",

    "internal_transfer_destination_organization_id",
    "internal_transfer_destination_organization_level",

    "internal_transfer_direction",
)


def _clean_optional_text(
    value,
) -> Optional[str]:

    if value is None:
        return None

    if pd.isna(value):
        return None

    text = str(
        value
    ).strip()

    return (
        text
        if text
        else None
    )


def classify_state_funding_form(
    value,
) -> Optional[str]:
    """
    Classify the source field
    'Форма державного фінансування'.

    This is separate from account-type labels.
    """

    text = _clean_optional_text(
        value
    )

    if text is None:
        return None

    lowered = text.lower()


    if (
        "статут" in lowered
        and
        (
            "держав" in lowered
            or
            "бюджет" in lowered
        )
    ):

        return (
            "state_statutory_funding"
        )


    if (
        "відшкодуван" in lowered
        and
        (
            "агітац" in lowered
            or
            "передвибор" in lowered
        )
    ):

        return (
            "state_campaign_reimbursement"
        )


    return None


def payment_direction_for_section(
    section: str,
) -> str:

    if section in INCOMING_SECTIONS:
        return "incoming"

    if section in OUTGOING_SECTIONS:
        return "outgoing"

    raise ValueError(
        f"Unknown payment section: {section}"
    )


def party_account_column(
    columns,
) -> str:
    """
    Prefer canonical analytical column if already present.

    Otherwise use canonical account field produced by
    payment normalization.
    """

    columns = set(
        columns
    )


    candidates = (
        "party_account_iban",
        "receiver_account_iban_canonical",
    )


    for column in candidates:

        if column in columns:
            return column


    raise KeyError(
        "No canonical party-account column found. "
        f"Tried: {candidates}"
    )


def counterparty_code_column(
    section: str,
) -> str:

    direction = (
        payment_direction_for_section(
            section
        )
    )


    if direction == "incoming":
        return "payer_code_normalized"

    return "receiver_code_normalized"


def build_unique_organization_code_map(
    organization_reference: pd.DataFrame,
) -> pd.DataFrame:
    """
    Strict same-root organization matching by EDRPOU only.

    If the same code maps to more than one organization
    within the same root party, it is excluded instead of
    being guessed/fanned out.
    """

    required = {
        "root_party_id",
        "organization_id",
        "organization_code",
        "organization_name_current",
        "organization_level",
    }


    missing = (
        required
        -
        set(
            organization_reference.columns
        )
    )


    if missing:

        raise KeyError(
            "organization_reference missing columns: "
            f"{sorted(missing)}"
        )


    df = (
        organization_reference[
            [
                "root_party_id",
                "organization_id",
                "organization_code",
                "organization_name_current",
                "organization_level",
            ]
        ]
        .copy()
    )


    df[
        "organization_code"
    ] = (
        df[
            "organization_code"
        ]
        .astype("string")
        .str.strip()
    )


    df = df[
        df[
            "organization_code"
        ].notna()
        &
        (
            df[
                "organization_code"
            ]
            !=
            ""
        )
    ]


    counts = (
        df
        .groupby(
            [
                "root_party_id",
                "organization_code",
            ],
            dropna=False,
        )[
            "organization_id"
        ]
        .nunique()
        .rename(
            "_organization_count"
        )
        .reset_index()
    )


    unique_keys = (
        counts[
            counts[
                "_organization_count"
            ]
            ==
            1
        ][
            [
                "root_party_id",
                "organization_code",
            ]
        ]
    )


    result = (
        df
        .merge(
            unique_keys,
            on=[
                "root_party_id",
                "organization_code",
            ],
            how="inner",
            validate="many_to_one",
        )
        .drop_duplicates(
            subset=[
                "root_party_id",
                "organization_code",
            ]
        )
        .reset_index(
            drop=True
        )
    )


    return result


def prepare_state_account_reference(
    state_account_reference: pd.DataFrame,
) -> pd.DataFrame:
    """
    Normalize the canonical confirmed-state-account
    reference for joining to payment rows.
    """

    required = {
        "root_party_id",
        "organization_id",
        "party_account_iban",
        "state_funding_form_code",
    }


    missing = (
        required
        -
        set(
            state_account_reference.columns
        )
    )


    if missing:

        raise KeyError(
            "state account reference missing columns: "
            f"{sorted(missing)}"
        )


    df = (
        state_account_reference
        .copy()
    )


    if (
        "state_funding_source_forms_observed"
        in
        df.columns
    ):

        source_column = (
            "state_funding_source_forms_observed"
        )

    elif (
        "state_funding_form_source"
        in
        df.columns
    ):

        source_column = (
            "state_funding_form_source"
        )

    else:

        raise KeyError(
            "No state-funding source-form column found."
        )


    result = (
        df[
            [
                "root_party_id",
                "organization_id",
                "party_account_iban",
                "state_funding_form_code",
                source_column,
            ]
        ]
        .rename(
            columns={
                source_column:
                    "state_funding_form_source",
            }
        )
        .copy()
    )


    duplicate_keys = (
        result
        .duplicated(
            subset=[
                "root_party_id",
                "organization_id",
                "party_account_iban",
            ],
            keep=False,
        )
    )


    if duplicate_keys.any():

        raise ValueError(
            "Confirmed state-account reference "
            "contains duplicate join keys."
        )


    return result


def resolve_payment_facts(
    payments: pd.DataFrame,
    *,
    section: str,
    organization_reference: pd.DataFrame,
    state_account_reference: pd.DataFrame,
) -> pd.DataFrame:
    """
    Resolve all technical facts needed by analytical
    payment enrichment.

    Row count and row order are preserved.
    """

    if section not in PAYMENT_SECTIONS:

        raise ValueError(
            f"Unknown payment section: {section}"
        )


    required_payment_columns = {
        "root_party_id",
        "organization_id",
        "organization_level",
        "internal_transfer",
    }


    code_column = (
        counterparty_code_column(
            section
        )
    )


    required_payment_columns.add(
        code_column
    )


    missing = (
        required_payment_columns
        -
        set(
            payments.columns
        )
    )


    if missing:

        raise KeyError(
            f"{section} missing payment columns: "
            f"{sorted(missing)}"
        )


    account_column = (
        party_account_column(
            payments.columns
        )
    )


    direction = (
        payment_direction_for_section(
            section
        )
    )


    df = payments.copy()


    # Avoid collisions when validating an already-enriched
    # production table.
    existing_derived = [
        column
        for column
        in RESOLVED_COLUMNS
        if column in df.columns
    ]


    if existing_derived:

        df = df.drop(
            columns=
                existing_derived
        )


    df[
        "payment_direction"
    ] = direction


    df[
        "party_account_iban"
    ] = (
        payments[
            account_column
        ]
    )


    # --------------------------------------------------------
    # CONFIRMED STATE ACCOUNT
    # --------------------------------------------------------

    state_ref = (
        prepare_state_account_reference(
            state_account_reference
        )
        .rename(
            columns={
                "state_funding_form_source":
                    "_state_funding_form_source_ref",

                "state_funding_form_code":
                    "_state_funding_form_code_ref",
            }
        )
    )


    before_rows = len(
        df
    )


    df = df.merge(
        state_ref,

        on=[
            "root_party_id",
            "organization_id",
            "party_account_iban",
        ],

        how="left",
        validate="many_to_one",
        sort=False,
    )


    if len(df) != before_rows:

        raise RuntimeError(
            "State-account join changed payment row count."
        )


    df[
        "state_funding_account_confirmed"
    ] = (
        df[
            "_state_funding_form_code_ref"
        ]
        .notna()
    )


    # --------------------------------------------------------
    # FUNDING FORM
    #
    # On confirmed accounts the reference wins.
    #
    # For source state_funding rows, retain the source
    # transaction form even if the row itself does not
    # establish a confirmed account.
    # --------------------------------------------------------

    if (
        section
        ==
        "state_funding"
        and
        "payment_type_detail_source"
        in payments.columns
    ):

        source_forms = (
            payments[
                "payment_type_detail_source"
            ]
        )

        source_codes = (
            source_forms
            .map(
                classify_state_funding_form
            )
        )

    else:

        source_forms = pd.Series(
            pd.NA,
            index=df.index,
            dtype="string",
        )

        source_codes = pd.Series(
            pd.NA,
            index=df.index,
            dtype="string",
        )


    df[
        "state_funding_form_source"
    ] = (
        df[
            "_state_funding_form_source_ref"
        ]
        .combine_first(
            source_forms
        )
    )


    df[
        "state_funding_form_code"
    ] = (
        df[
            "_state_funding_form_code_ref"
        ]
        .combine_first(
            source_codes
        )
    )


    df = df.drop(
        columns=[
            "_state_funding_form_source_ref",
            "_state_funding_form_code_ref",
        ]
    )


    # --------------------------------------------------------
    # SAME-PARTY ORGANIZATION MATCH
    # --------------------------------------------------------

    org_map = (
        build_unique_organization_code_map(
            organization_reference
        )
        .rename(
            columns={
                "organization_id":
                    "_counterparty_organization_id",

                "organization_name_current":
                    "_counterparty_organization_name",

                "organization_level":
                    "_counterparty_organization_level",

                "organization_code":
                    "_counterparty_code",
            }
        )
    )


    counterparty_codes = (
        payments[
            code_column
        ]
        .astype("string")
        .str.strip()
    )


    df[
        "_counterparty_code"
    ] = counterparty_codes


    before_rows = len(
        df
    )


    df = df.merge(
        org_map,

        on=[
            "root_party_id",
            "_counterparty_code",
        ],

        how="left",
        validate="many_to_one",
        sort=False,
    )


    if len(df) != before_rows:

        raise RuntimeError(
            "Organization-code join changed payment row count."
        )


    internal_mask = (
        payments[
            "internal_transfer"
        ]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )


    # Only internal transfers may receive an internal
    # counterparty organization.
    df[
        "internal_counterparty_organization_id"
    ] = (
        df[
            "_counterparty_organization_id"
        ]
        .where(
            internal_mask
        )
    )


    df[
        "internal_counterparty_organization_name"
    ] = (
        df[
            "_counterparty_organization_name"
        ]
        .where(
            internal_mask
        )
    )


    df[
        "internal_counterparty_organization_level"
    ] = (
        df[
            "_counterparty_organization_level"
        ]
        .where(
            internal_mask
        )
    )


    df = df.drop(
        columns=[
            "_counterparty_code",
            "_counterparty_organization_id",
            "_counterparty_organization_name",
            "_counterparty_organization_level",
        ]
    )


    # --------------------------------------------------------
    # PHYSICAL SOURCE / DESTINATION
    # --------------------------------------------------------

    if direction == "outgoing":

        df[
            "internal_transfer_source_organization_id"
        ] = (
            df[
                "organization_id"
            ]
            .where(
                internal_mask
            )
        )


        df[
            "internal_transfer_source_organization_level"
        ] = (
            df[
                "organization_level"
            ]
            .where(
                internal_mask
            )
        )


        df[
            "internal_transfer_destination_organization_id"
        ] = (
            df[
                "internal_counterparty_organization_id"
            ]
        )


        df[
            "internal_transfer_destination_organization_level"
        ] = (
            df[
                "internal_counterparty_organization_level"
            ]
        )


    else:

        df[
            "internal_transfer_source_organization_id"
        ] = (
            df[
                "internal_counterparty_organization_id"
            ]
        )


        df[
            "internal_transfer_source_organization_level"
        ] = (
            df[
                "internal_counterparty_organization_level"
            ]
        )


        df[
            "internal_transfer_destination_organization_id"
        ] = (
            df[
                "organization_id"
            ]
            .where(
                internal_mask
            )
        )


        df[
            "internal_transfer_destination_organization_level"
        ] = (
            df[
                "organization_level"
            ]
            .where(
                internal_mask
            )
        )


    # --------------------------------------------------------
    # TRANSFER DIRECTION
    #
    # Apply pure business function only to internal rows.
    # Typically ~18k rather than all ~402k.
    # --------------------------------------------------------

    df[
        "internal_transfer_direction"
    ] = pd.NA


    internal_indices = (
        df.index[
            internal_mask
        ]
    )


    for idx in internal_indices:

        df.at[
            idx,
            "internal_transfer_direction",
        ] = (
            classify_internal_transfer_direction(

                internal_transfer=True,

                internal_counterparty_organization_id=
                    _clean_optional_text(
                        df.at[
                            idx,
                            "internal_counterparty_organization_id",
                        ]
                    ),

                source_organization_id=
                    _clean_optional_text(
                        df.at[
                            idx,
                            "internal_transfer_source_organization_id",
                        ]
                    ),

                source_organization_level=
                    _clean_optional_text(
                        df.at[
                            idx,
                            "internal_transfer_source_organization_level",
                        ]
                    ),

                destination_organization_id=
                    _clean_optional_text(
                        df.at[
                            idx,
                            "internal_transfer_destination_organization_id",
                        ]
                    ),

                destination_organization_level=
                    _clean_optional_text(
                        df.at[
                            idx,
                            "internal_transfer_destination_organization_level",
                        ]
                    ),
            )
        )


    return df


def compare_resolved_facts(
    existing: pd.DataFrame,
    recalculated: pd.DataFrame,
    *,
    section: str,
):
    """
    Compare technical resolution against current verified
    production enrichment.
    """

    summaries = []
    samples = []


    for column in RESOLVED_COLUMNS:

        if column not in existing.columns:

            summaries.append(
                {
                    "section":
                        section,

                    "column":
                        column,

                    "rows":
                        len(existing),

                    "mismatches":
                        None,

                    "status":
                        "missing_existing_column",
                }
            )

            continue


        old = (
            existing[
                column
            ]
            .astype("string")
            .fillna("<NULL>")
        )


        new = (
            recalculated[
                column
            ]
            .astype("string")
            .fillna("<NULL>")
        )


        mismatch = (
            old
            !=
            new
        )


        mismatch_count = int(
            mismatch.sum()
        )


        summaries.append(
            {
                "section":
                    section,

                "column":
                    column,

                "rows":
                    len(existing),

                "mismatches":
                    mismatch_count,

                "status":
                    (
                        "match"
                        if mismatch_count == 0
                        else
                        "mismatch"
                    ),
            }
        )


        if mismatch_count:

            for idx in (
                mismatch[
                    mismatch
                ]
                .index[:20]
            ):

                sample = {
                    "section":
                        section,

                    "row_index":
                        idx,

                    "column":
                        column,

                    "existing":
                        old.loc[idx],

                    "recalculated":
                        new.loc[idx],
                }


                for context_column in (
                    "source_report_id",
                    "party_name_current",
                    "organization_name_current",
                    "organization_level",
                    "source_payment_type",
                    "payment_direction",
                    "party_account_iban",
                    "payer_code_normalized",
                    "receiver_code_normalized",
                    "internal_transfer",
                ):

                    if (
                        context_column
                        in
                        existing.columns
                    ):

                        sample[
                            context_column
                        ] = (
                            existing.at[
                                idx,
                                context_column,
                            ]
                        )


                samples.append(
                    sample
                )


    return (
        pd.DataFrame(
            summaries
        ),
        pd.DataFrame(
            samples
        ),
    )
'''


MODULE_PATH = (
    ENRICHMENT_DIR
    / "payment_resolution.py"
)


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    MODULE_PATH
)


# ============================================================
# 2. TESTS
# ============================================================

TEST_CODE = r'''
import pandas as pd

from politdata.enrichment.payment_resolution import (
    classify_state_funding_form,
    payment_direction_for_section,
    build_unique_organization_code_map,
    resolve_payment_facts,
)


def test_state_funding_form_statutory():

    assert (
        classify_state_funding_form(
            "Державне фінансування статутної "
            "діяльності політичної партії"
        )
        ==
        "state_statutory_funding"
    )


def test_state_funding_form_campaign_reimbursement():

    assert (
        classify_state_funding_form(
            "Відшкодування витрат, пов'язаних "
            "з фінансуванням передвиборної агітації"
        )
        ==
        "state_campaign_reimbursement"
    )


def test_payment_direction():

    assert (
        payment_direction_for_section(
            "other_incomes"
        )
        ==
        "incoming"
    )

    assert (
        payment_direction_for_section(
            "outgoing_expenses"
        )
        ==
        "outgoing"
    )


def test_ambiguous_same_root_code_is_excluded():

    organizations = pd.DataFrame(
        [
            {
                "root_party_id": "p",
                "organization_id": "o1",
                "organization_code": "123",
                "organization_name_current": "A",
                "organization_level": "office",
            },
            {
                "root_party_id": "p",
                "organization_id": "o2",
                "organization_code": "123",
                "organization_name_current": "B",
                "organization_level": "office",
            },
            {
                "root_party_id": "p",
                "organization_id": "o3",
                "organization_code": "456",
                "organization_name_current": "C",
                "organization_level": "office",
            },
        ]
    )


    result = (
        build_unique_organization_code_map(
            organizations
        )
    )


    assert (
        set(
            result[
                "organization_code"
            ]
        )
        ==
        {"456"}
    )


def test_resolution_central_to_office_state_account():

    payments = pd.DataFrame(
        [
            {
                "root_party_id": "p",
                "organization_id": "central",
                "organization_level": "central",

                "receiver_code_normalized": "222",
                "internal_transfer": True,

                "receiver_account_iban_canonical":
                    "UA000000000000000000000000001",
            }
        ]
    )


    organizations = pd.DataFrame(
        [
            {
                "root_party_id": "p",
                "organization_id": "central",
                "organization_code": "111",
                "organization_name_current": "CENTRAL",
                "organization_level": "central",
            },
            {
                "root_party_id": "p",
                "organization_id": "office",
                "organization_code": "222",
                "organization_name_current": "OFFICE",
                "organization_level": "office",
            },
        ]
    )


    state_accounts = pd.DataFrame(
        [
            {
                "root_party_id": "p",
                "organization_id": "central",
                "party_account_iban":
                    "UA000000000000000000000000001",

                "state_funding_form_code":
                    "state_statutory_funding",

                "state_funding_source_forms_observed":
                    "Державне фінансування статутної "
                    "діяльності політичної партії",
            }
        ]
    )


    result = resolve_payment_facts(
        payments,
        section="outgoing_expenses",
        organization_reference=
            organizations,
        state_account_reference=
            state_accounts,
    )


    row = result.iloc[0]


    assert (
        bool(
            row[
                "state_funding_account_confirmed"
            ]
        )
        is True
    )

    assert (
        row[
            "internal_counterparty_organization_id"
        ]
        ==
        "office"
    )

    assert (
        row[
            "internal_transfer_direction"
        ]
        ==
        "central_to_office"
    )
'''


TEST_PATH = (
    TESTS_DIR
    / "test_payment_resolution.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 3. RUN TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("UNIT TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:
    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Payment resolution tests failed."
    )


# ============================================================
# 4. LOAD REFERENCES ONCE
# ============================================================

from politdata.enrichment.payment_resolution import (
    resolve_payment_facts,
    compare_resolved_facts,
)


organization_reference = pd.read_parquet(
    ORG_REFERENCE_PATH
)


state_account_reference = pd.read_parquet(
    STATE_ACCOUNT_REFERENCE_PATH
)


print()
print("=" * 100)
print("REFERENCE INPUT")
print("=" * 100)

print(
    "Organizations:",
    f"{len(organization_reference):,}"
)

print(
    "Confirmed state accounts:",
    f"{len(state_account_reference):,}"
)


# ============================================================
# 5. FULL READ-ONLY RESOLUTION PARITY
# ============================================================

PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


summaries = []
sample_frames = []


print()
print("=" * 100)
print("FULL PAYMENT RESOLUTION PARITY")
print("=" * 100)


for section in PAYMENT_SECTIONS:

    path = (
        PAYMENT_DIR
        / f"{section}.parquet"
    )


    existing = pd.read_parquet(
        path
    )


    resolved = resolve_payment_facts(
        existing,
        section=section,
        organization_reference=
            organization_reference,
        state_account_reference=
            state_account_reference,
    )


    if len(resolved) != len(existing):

        raise RuntimeError(
            f"{section}: row count changed "
            f"{len(existing):,} -> {len(resolved):,}"
        )


    summary, samples = (
        compare_resolved_facts(
            existing,
            resolved,
            section=section,
        )
    )


    summaries.append(
        summary
    )


    if len(samples):
        sample_frames.append(
            samples
        )


    print(
        f"{section:24s}",
        f"{len(existing):>8,}",
        "rows",
    )


summary = pd.concat(
    summaries,
    ignore_index=True,
)


if sample_frames:

    mismatch_samples = pd.concat(
        sample_frames,
        ignore_index=True,
    )

else:

    mismatch_samples = (
        pd.DataFrame()
    )


# ============================================================
# 6. PARITY BY COLUMN
# ============================================================

column_summary = (
    summary
    .groupby(
        "column",
        as_index=False,
    )
    .agg(
        rows=(
            "rows",
            "sum",
        ),

        mismatches=(
            "mismatches",
            "sum",
        ),

        sections=(
            "section",
            "nunique",
        ),
    )
)


column_summary[
    "match_ratio"
] = (
    1
    -
    (
        column_summary[
            "mismatches"
        ]
        /
        column_summary[
            "rows"
        ]
    )
)


print()
print("=" * 100)
print("RESOLUTION PARITY BY COLUMN")
print("=" * 100)

display(
    column_summary
)


# ============================================================
# 7. PARITY BY SECTION
# ============================================================

section_summary = (
    summary
    .groupby(
        "section",
        as_index=False,
    )
    .agg(
        comparisons=(
            "column",
            "size",
        ),

        mismatches=(
            "mismatches",
            "sum",
        ),
    )
)


print()
print("=" * 100)
print("RESOLUTION PARITY BY PAYMENT SECTION")
print("=" * 100)

display(
    section_summary
)


# ============================================================
# 8. MISMATCHES
# ============================================================

total_mismatches = int(
    summary[
        "mismatches"
    ]
    .fillna(0)
    .sum()
)


print()
print(
    "Total field-level mismatches:",
    f"{total_mismatches:,}"
)


if total_mismatches:

    print()
    print("=" * 100)
    print("MISMATCH SAMPLES")
    print("=" * 100)

    display(
        mismatch_samples.head(
            100
        )
    )

    raise RuntimeError(
        "Resolution layer does not yet reproduce "
        "the verified production dataset."
    )


# ============================================================
# 9. BASELINE CONTROL
# ============================================================

total_rows = 0


for section in PAYMENT_SECTIONS:

    total_rows += len(
        pd.read_parquet(
            PAYMENT_DIR
            / f"{section}.parquet",
            columns=[
                "source_report_id"
            ],
        )
    )


print()
print(
    "Production payment rows:",
    f"{total_rows:,}"
)


if total_rows != 402_028:

    raise RuntimeError(
        f"Payment baseline changed: {total_rows:,}"
    )


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Technical payment-resolution logic now lives "
    "in src/politdata/enrichment/payment_resolution.py."
)

print(
    "No RAW files were read."
)

print(
    "No production parquet files were modified."
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_resolution.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_payment_resolution.py

UNIT TESTS
........................................................................ [ 69%]
................................                                         [100%]
104 passed in 1.16s

Return code: 0

REFERENCE INPUT
Organizations: 9,917
Confirmed state accounts: 7

FULL PAYMENT RESOLUTION PARITY


C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_resolution.py:674: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(


monetary_contributions     27,234 rows


C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_resolution.py:674: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  .combine_first(


other_contributions         6,168 rows
state_funding                  96 rows
other_incomes              19,007 rows
budget_expenses            29,482 rows


...

transfer_expenses               3 rows

RESOLUTION PARITY BY COLUMN


,column,rows,mismatches,sections,match_ratio
0,internal_counterparty_organization_id,402028,0,8,1.0
1,internal_counterparty_organization_level,402028,0,8,1.0
2,internal_counterparty_organization_name,402028,0,8,1.0
3,internal_transfer_destination_organization_id,402028,0,8,1.0
4,internal_transfer_destination_organization_level,402028,0,8,1.0
5,internal_transfer_direction,402028,0,8,1.0
6,internal_transfer_source_organization_id,402028,0,8,1.0
7,internal_transfer_source_organization_level,402028,0,8,1.0
8,party_account_iban,402028,0,8,1.0
9,payment_direction,402028,0,8,1.0



RESOLUTION PARITY BY PAYMENT SECTION


,section,comparisons,mismatches
0,budget_expenses,13,0
1,monetary_contributions,13,0
2,other_contributions,13,0
3,other_incomes,13,0
4,outgoing_expenses,13,0
5,return_expenses,13,0
6,state_funding,13,0
7,transfer_expenses,13,0



Total field-level mismatches: 0

Production payment rows: 402,028

DONE
Technical payment-resolution logic now lives in src/politdata/enrichment/payment_resolution.py.
No RAW files were read.
No production parquet files were modified.


In [129]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

ENRICHMENT_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "enrichment"
)

TESTS_DIR = (
    ROOT
    / "tests"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

ORG_REFERENCE_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "organization_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "state_funding_account_reference.parquet"
)

VALIDATION_DIR = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "payment_enrichment_rebuild_v0_1"
)

RESOLUTION_PATH = (
    ENRICHMENT_DIR
    / "payment_resolution.py"
)

BATCH_PATH = (
    ENRICHMENT_DIR
    / "payment_batch.py"
)


# ============================================================
# 1. REMOVE pandas FutureWarning
#
# No behavior change:
# reference value still has priority,
# source transaction form remains fallback.
# ============================================================

text = RESOLUTION_PATH.read_text(
    encoding="utf-8"
)


old_block = '''    df[
        "state_funding_form_source"
    ] = (
        df[
            "_state_funding_form_source_ref"
        ]
        .combine_first(
            source_forms
        )
    )


    df[
        "state_funding_form_code"
    ] = (
        df[
            "_state_funding_form_code_ref"
        ]
        .combine_first(
            source_codes
        )
    )
'''


new_block = '''    state_source_ref = (
        df[
            "_state_funding_form_source_ref"
        ]
    )


    df[
        "state_funding_form_source"
    ] = (
        state_source_ref.where(
            state_source_ref.notna(),
            source_forms,
        )
    )


    state_code_ref = (
        df[
            "_state_funding_form_code_ref"
        ]
    )


    df[
        "state_funding_form_code"
    ] = (
        state_code_ref.where(
            state_code_ref.notna(),
            source_codes,
        )
    )
'''


if old_block in text:

    text = text.replace(
        old_block,
        new_block,
        1,
    )

    RESOLUTION_PATH.write_text(
        text,
        encoding="utf-8",
    )

    print(
        "FutureWarning source patched:",
        RESOLUTION_PATH
    )

elif ".combine_first(" in text:

    raise RuntimeError(
        "Expected combine_first block was not found "
        "in the known form. Stop instead of guessing."
    )

else:

    print(
        "FutureWarning patch already present."
    )


# ============================================================
# 2. ADD COMPACT DIRECTORY ORCHESTRATION
#    TO EXISTING payment_batch.py
#
# Do NOT create another large module.
# ============================================================

batch_text = BATCH_PATH.read_text(
    encoding="utf-8"
)


ORCHESTRATION_MARKER = (
    "# PAYMENT DIRECTORY ORCHESTRATION V0.1"
)


ORCHESTRATION_CODE = r'''


# ============================================================
# PAYMENT DIRECTORY ORCHESTRATION V0.1
# ============================================================

def rebuild_payment_enrichment_frame(
    payments: pd.DataFrame,
    *,
    section: str,
    organization_reference: pd.DataFrame,
    state_account_reference: pd.DataFrame,
) -> pd.DataFrame:
    """
    Rebuild the complete technical + analytical payment
    enrichment for one already-normalized/base-enriched
    payment frame.

    Recomputed fields:
    - payment direction
    - party account
    - confirmed state-account facts
    - internal organization counterparties
    - physical transfer direction
    - analytical account classification
    - analytical payment classification
    - analytical funding source

    Existing values in those fields are not trusted:
    they are recalculated and overwritten.

    Other source/base-enrichment columns are preserved.
    """

    from politdata.enrichment.payment_resolution import (
        resolve_payment_facts,
    )


    original_columns = list(
        payments.columns
    )


    base = (
        payments
        .reset_index(
            drop=True
        )
        .copy()
    )


    resolved = resolve_payment_facts(
        base,
        section=section,
        organization_reference=
            organization_reference,
        state_account_reference=
            state_account_reference,
    )


    derived = derive_payment_enrichment(
        resolved,
        section=section,
    )


    result = resolved.copy()


    for column in DERIVED_COLUMNS:

        result[
            column
        ] = derived[
            column
        ].to_numpy()


    # If rebuilding an already-enriched dataset, restore
    # its exact column order.
    #
    # If a future input lacks some newly-derived fields,
    # append them at the end instead.
    extra_columns = [
        column
        for column
        in result.columns
        if column not in original_columns
    ]


    target_columns = (
        [
            column
            for column
            in original_columns
            if column in result.columns
        ]
        +
        extra_columns
    )


    result = result[
        target_columns
    ]


    if len(result) != len(payments):

        raise RuntimeError(
            f"{section}: payment enrichment changed "
            f"row count {len(payments)} -> {len(result)}"
        )


    return result


def enrich_payment_directory(
    input_dir,
    output_dir,
    *,
    organization_reference,
    state_account_reference,
    overwrite: bool = False,
) -> pd.DataFrame:
    """
    Rebuild all eight payment files into a separate directory.

    This function does not access RAW or API.

    `organization_reference` and `state_account_reference`
    may be DataFrames or parquet paths.
    """

    input_dir = Path(
        input_dir
    )

    output_dir = Path(
        output_dir
    )


    if isinstance(
        organization_reference,
        (
            str,
            Path,
        ),
    ):

        organization_reference = (
            pd.read_parquet(
                organization_reference
            )
        )


    if isinstance(
        state_account_reference,
        (
            str,
            Path,
        ),
    ):

        state_account_reference = (
            pd.read_parquet(
                state_account_reference
            )
        )


    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    summary = []


    for section in PAYMENT_SECTIONS:

        input_path = (
            input_dir
            / f"{section}.parquet"
        )

        output_path = (
            output_dir
            / f"{section}.parquet"
        )


        if not input_path.exists():

            raise FileNotFoundError(
                input_path
            )


        if (
            output_path.exists()
            and
            not overwrite
        ):

            raise FileExistsError(
                output_path
            )


        source = pd.read_parquet(
            input_path
        )


        rebuilt = (
            rebuild_payment_enrichment_frame(
                source,
                section=section,
                organization_reference=
                    organization_reference,
                state_account_reference=
                    state_account_reference,
            )
        )


        temp_path = (
            output_path
            .with_suffix(
                ".tmp.parquet"
            )
        )


        rebuilt.to_parquet(
            temp_path,
            index=False,
        )


        temp_path.replace(
            output_path
        )


        summary.append(
            {
                "section":
                    section,

                "rows":
                    len(rebuilt),

                "columns":
                    len(rebuilt.columns),

                "output":
                    str(output_path),
            }
        )


    return pd.DataFrame(
        summary
    )
'''


if ORCHESTRATION_MARKER not in batch_text:

    batch_text = (
        batch_text.rstrip()
        +
        "\n"
        +
        textwrap.dedent(
            ORCHESTRATION_CODE
        )
        +
        "\n"
    )


    BATCH_PATH.write_text(
        batch_text,
        encoding="utf-8",
    )


    print(
        "Directory orchestration added:",
        BATCH_PATH
    )

else:

    print(
        "Directory orchestration already present."
    )


# ============================================================
# 3. ADD INTEGRATION TEST
#
# Key point:
# stale/wrong enrichment fields must be ignored
# and recomputed from prerequisite facts.
# ============================================================

TEST_PATH = (
    TESTS_DIR
    / "test_payment_batch.py"
)


test_text = TEST_PATH.read_text(
    encoding="utf-8"
)


INTEGRATION_TEST_NAME = (
    "test_rebuild_overwrites_stale_payment_enrichment"
)


INTEGRATION_TEST = r'''


def test_rebuild_overwrites_stale_payment_enrichment():

    from politdata.enrichment.payment_batch import (
        rebuild_payment_enrichment_frame,
    )


    payments = pd.DataFrame(
        [
            {
                "root_party_id":
                    "party",

                "organization_id":
                    "central",

                "organization_level":
                    "central",

                "receiver_code_normalized":
                    "222",

                "receiver_account_iban_canonical":
                    "UA000000000000000000000000001",

                "party_account_type_source":
                    "Поточний рахунок",

                "internal_transfer":
                    True,

                "source_payment_type":
                    "budget_expenses",

                # -------------------------------
                # Deliberately WRONG stale fields
                # -------------------------------

                "payment_direction":
                    "incoming",

                "party_account_iban":
                    "WRONG",

                "state_funding_account_confirmed":
                    False,

                "state_funding_form_source":
                    None,

                "state_funding_form_code":
                    None,

                "internal_counterparty_organization_id":
                    "WRONG",

                "internal_counterparty_organization_name":
                    "WRONG",

                "internal_counterparty_organization_level":
                    "central",

                "internal_transfer_source_organization_id":
                    "WRONG",

                "internal_transfer_source_organization_level":
                    "office",

                "internal_transfer_destination_organization_id":
                    "WRONG",

                "internal_transfer_destination_organization_level":
                    "central",

                "internal_transfer_direction":
                    "office_to_central",

                "internal_transfer_funding_source":
                    "unknown",

                "party_account_type":
                    "WRONG",

                "party_account_type_analytical":
                    "unknown",

                "party_account_type_resolution_method":
                    "unresolved",

                "analytical_payment_type":
                    "outgoing_expenses",

                "was_reclassified":
                    True,

                "reclassification_rule":
                    "WRONG",

                "funding_source_analytical":
                    "unknown",
            }
        ]
    )


    organizations = pd.DataFrame(
        [
            {
                "root_party_id":
                    "party",

                "organization_id":
                    "central",

                "organization_code":
                    "111",

                "organization_name_current":
                    "CENTRAL",

                "organization_level":
                    "central",
            },
            {
                "root_party_id":
                    "party",

                "organization_id":
                    "office",

                "organization_code":
                    "222",

                "organization_name_current":
                    "OFFICE",

                "organization_level":
                    "office",
            },
        ]
    )


    state_accounts = pd.DataFrame(
        [
            {
                "root_party_id":
                    "party",

                "organization_id":
                    "central",

                "party_account_iban":
                    "UA000000000000000000000000001",

                "state_funding_form_code":
                    "state_statutory_funding",

                "state_funding_source_forms_observed":
                    (
                        "Державне фінансування статутної "
                        "діяльності політичної партії"
                    ),
            }
        ]
    )


    result = (
        rebuild_payment_enrichment_frame(
            payments,
            section="budget_expenses",
            organization_reference=
                organizations,
            state_account_reference=
                state_accounts,
        )
    )


    row = result.iloc[0]


    assert (
        row[
            "payment_direction"
        ]
        ==
        "outgoing"
    )

    assert (
        row[
            "party_account_iban"
        ]
        ==
        "UA000000000000000000000000001"
    )

    assert (
        bool(
            row[
                "state_funding_account_confirmed"
            ]
        )
        is True
    )

    assert (
        row[
            "internal_counterparty_organization_id"
        ]
        ==
        "office"
    )

    assert (
        row[
            "internal_transfer_direction"
        ]
        ==
        "central_to_office"
    )

    assert (
        row[
            "internal_transfer_funding_source"
        ]
        ==
        "state_statutory_funding"
    )

    assert (
        row[
            "party_account_type_analytical"
        ]
        ==
        "state_statutory_funding_account"
    )

    assert (
        row[
            "analytical_payment_type"
        ]
        ==
        "budget_expenses"
    )

    assert (
        bool(
            row[
                "was_reclassified"
            ]
        )
        is False
    )

    assert (
        row[
            "funding_source_analytical"
        ]
        ==
        "state_statutory_funding"
    )
'''


if INTEGRATION_TEST_NAME not in test_text:

    test_text = (
        test_text.rstrip()
        +
        "\n"
        +
        textwrap.dedent(
            INTEGRATION_TEST
        )
        +
        "\n"
    )


    TEST_PATH.write_text(
        test_text,
        encoding="utf-8",
    )


    print(
        "Integration test added:",
        TEST_PATH
    )

else:

    print(
        "Integration test already present."
    )


# ============================================================
# 4. RUN COMPLETE TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Temporary payment rebuild was not started."
    )


# ============================================================
# 5. FRESH IMPORT AFTER SOURCE PATCHES
# ============================================================

import politdata.enrichment.payment_resolution as payment_resolution
import politdata.enrichment.payment_batch as payment_batch


importlib.reload(
    payment_resolution
)

importlib.reload(
    payment_batch
)


# ============================================================
# 6. PREPARE CLEAN VALIDATION DIRECTORY
#
# Safe to delete: this is NOT production.
# ============================================================

if VALIDATION_DIR.exists():

    shutil.rmtree(
        VALIDATION_DIR
    )


VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 7. TEMPORARY FULL PAYMENT REBUILD
#
# Reads current payment rows as the base dataset,
# but does NOT trust any existing resolution /
# analytical enrichment fields.
#
# They are all regenerated.
# ============================================================

print()
print("=" * 100)
print("TEMPORARY FULL PAYMENT ENRICHMENT REBUILD")
print("=" * 100)


rebuild_summary = (
    payment_batch.enrich_payment_directory(
        PAYMENT_DIR,
        VALIDATION_DIR,

        organization_reference=
            ORG_REFERENCE_PATH,

        state_account_reference=
            STATE_ACCOUNT_REFERENCE_PATH,

        overwrite=True,
    )
)


display(
    rebuild_summary
)


# ============================================================
# 8. FULL OUTPUT PARITY
#
# Compare every column, not just the 22 rebuilt fields.
# ============================================================

PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


def count_series_mismatches(
    left,
    right,
):

    if len(left) != len(right):

        raise ValueError(
            "Series lengths differ."
        )


    both_missing = (
        left.isna()
        &
        right.isna()
    )


    try:

        equal = left.eq(
            right
        )

        equal = equal.fillna(
            False
        )

    except Exception:

        equal = (
            left
            .astype("string")
            .fillna("<NULL>")
            ==
            right
            .astype("string")
            .fillna("<NULL>")
        )


    final_equal = (
        equal
        |
        both_missing
    )


    return int(
        (
            ~final_equal
        ).sum()
    )


parity_rows = []
mismatch_samples = []


print()
print("=" * 100)
print("FULL DIRECTORY PARITY")
print("=" * 100)


for section in PAYMENT_SECTIONS:

    production_path = (
        PAYMENT_DIR
        / f"{section}.parquet"
    )

    rebuilt_path = (
        VALIDATION_DIR
        / f"{section}.parquet"
    )


    production = pd.read_parquet(
        production_path
    )

    rebuilt = pd.read_parquet(
        rebuilt_path
    )


    if len(production) != len(rebuilt):

        raise RuntimeError(
            f"{section}: row count mismatch "
            f"{len(production):,} != {len(rebuilt):,}"
        )


    production_columns = list(
        production.columns
    )

    rebuilt_columns = list(
        rebuilt.columns
    )


    missing_columns = sorted(
        set(production_columns)
        -
        set(rebuilt_columns)
    )

    extra_columns = sorted(
        set(rebuilt_columns)
        -
        set(production_columns)
    )


    if (
        missing_columns
        or
        extra_columns
    ):

        raise RuntimeError(
            f"{section}: schema mismatch. "
            f"Missing={missing_columns}; "
            f"extra={extra_columns}"
        )


    # Compare in production column order.
    rebuilt = rebuilt[
        production_columns
    ]


    section_mismatches = 0
    mismatching_columns = 0


    for column in production_columns:

        mismatch_count = (
            count_series_mismatches(
                production[
                    column
                ],
                rebuilt[
                    column
                ],
            )
        )


        section_mismatches += (
            mismatch_count
        )


        if mismatch_count:

            mismatching_columns += 1


            old = (
                production[
                    column
                ]
                .astype("string")
                .fillna("<NULL>")
            )

            new = (
                rebuilt[
                    column
                ]
                .astype("string")
                .fillna("<NULL>")
            )


            mask = (
                old
                !=
                new
            )


            for idx in (
                mask[
                    mask
                ]
                .index[:5]
            ):

                mismatch_samples.append(
                    {
                        "section":
                            section,

                        "row_index":
                            int(idx),

                        "column":
                            column,

                        "production":
                            old.loc[
                                idx
                            ],

                        "rebuilt":
                            new.loc[
                                idx
                            ],
                    }
                )


    parity_rows.append(
        {
            "section":
                section,

            "rows":
                len(production),

            "columns":
                len(
                    production_columns
                ),

            "mismatching_columns":
                mismatching_columns,

            "field_level_mismatches":
                section_mismatches,
        }
    )


parity_df = pd.DataFrame(
    parity_rows
)


display(
    parity_df
)


# ============================================================
# 9. FINAL CONTROLS
# ============================================================

total_rows = int(
    parity_df[
        "rows"
    ].sum()
)


total_mismatches = int(
    parity_df[
        "field_level_mismatches"
    ].sum()
)


print()
print(
    "Rows rebuilt:",
    f"{total_rows:,}"
)

print(
    "Total field-level mismatches:",
    f"{total_mismatches:,}"
)


if total_rows != 402_028:

    raise RuntimeError(
        f"Expected 402,028 rows, got {total_rows:,}"
    )


if total_mismatches:

    print()
    print("=" * 100)
    print("MISMATCH SAMPLES")
    print("=" * 100)

    display(
        pd.DataFrame(
            mismatch_samples
        ).head(
            100
        )
    )


    raise RuntimeError(
        "Temporary end-to-end payment rebuild "
        "does not yet reproduce production exactly."
    )


# ============================================================
# 10. CONFIRM PRODUCTION FILES WERE NOT TOUCHED
# ============================================================

production_total = 0


for section in PAYMENT_SECTIONS:

    production_total += len(
        pd.read_parquet(
            PAYMENT_DIR
            / f"{section}.parquet",

            columns=[
                "source_report_id"
            ],
        )
    )


assert production_total == 402_028


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Temporary payment rebuild has full parity "
    "with production."
)

print(
    "Validation output:"
)

print(
    VALIDATION_DIR
)

print()
print(
    "Production payment parquet files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

FutureWarning source patched: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_resolution.py
Directory orchestration added: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_batch.py
Integration test added: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_payment_batch.py

TESTS
........................................F............................... [ 68%]
.................................                                        [100%]
================================== FAILURES ===================================
______________ test_rebuild_overwrites_stale_payment_enrichment _______________

    def test_rebuild_overwrites_stale_payment_enrichment():
    
        from politdata.enrichment.payment_batch import (
            rebuild_payment_enrichment_frame,
        )
    
    
        payments = pd.DataFrame(
            [
                {
                    "root_party_id":
                        "party

RuntimeError: Tests failed. Temporary payment rebuild was not started.

In [130]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

RESOLUTION_PATH = (
    ROOT
    / "src"
    / "politdata"
    / "enrichment"
    / "payment_resolution.py"
)

PAYMENT_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

ORG_REFERENCE_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "organization_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "state_funding_account_reference.parquet"
)

VALIDATION_DIR = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "payment_enrichment_rebuild_v0_1"
)


# ============================================================
# 1. FIX ACCOUNT SOURCE PRIORITY
#
# Correct architecture:
#
# receiver_account_iban_canonical
#     = normalized/source-derived fact
#     = authoritative rebuild input
#
# party_account_iban
#     = analytical/enriched field
#     = fallback only
#
# A rebuild must NOT trust stale enrichment.
# ============================================================

text = RESOLUTION_PATH.read_text(
    encoding="utf-8"
)


old = '''    candidates = (
        "party_account_iban",
        "receiver_account_iban_canonical",
    )
'''


new = '''    candidates = (
        "receiver_account_iban_canonical",
        "party_account_iban",
    )
'''


if old in text:

    text = text.replace(
        old,
        new,
        1,
    )

    RESOLUTION_PATH.write_text(
        text,
        encoding="utf-8",
    )

    print(
        "Account-source priority patched:",
        RESOLUTION_PATH
    )

elif new in text:

    print(
        "Account-source priority already patched."
    )

else:

    raise RuntimeError(
        "Expected party_account_column() candidate block "
        "was not found. Stop instead of guessing."
    )


# ============================================================
# 2. RUN COMPLETE TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)

if result.stderr:
    print(
        result.stderr
    )

print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests still fail. "
        "Temporary rebuild was not started."
    )


# ============================================================
# 3. FRESH IMPORT
# ============================================================

import politdata.enrichment.payment_resolution as payment_resolution
import politdata.enrichment.payment_batch as payment_batch


importlib.reload(
    payment_resolution
)

importlib.reload(
    payment_batch
)


# ============================================================
# 4. CLEAN VALIDATION DIRECTORY
#
# Production data are NOT touched.
# ============================================================

if VALIDATION_DIR.exists():

    shutil.rmtree(
        VALIDATION_DIR
    )


VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5. TEMPORARY FULL PAYMENT REBUILD
# ============================================================

print()
print("=" * 100)
print("TEMPORARY FULL PAYMENT ENRICHMENT REBUILD")
print("=" * 100)


rebuild_summary = (
    payment_batch.enrich_payment_directory(
        PAYMENT_DIR,
        VALIDATION_DIR,

        organization_reference=
            ORG_REFERENCE_PATH,

        state_account_reference=
            STATE_ACCOUNT_REFERENCE_PATH,

        overwrite=True,
    )
)


display(
    rebuild_summary
)


# ============================================================
# 6. FULL OUTPUT PARITY
#
# Compare ALL persisted columns, not only rebuilt fields.
# ============================================================

PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


def count_series_mismatches(
    left,
    right,
):

    if len(left) != len(right):

        raise ValueError(
            "Series lengths differ."
        )


    both_missing = (
        left.isna()
        &
        right.isna()
    )


    try:

        equal = (
            left.eq(
                right
            )
            .fillna(
                False
            )
        )

    except Exception:

        equal = (
            left
            .astype("string")
            .fillna("<NULL>")
            ==
            right
            .astype("string")
            .fillna("<NULL>")
        )


    return int(
        (
            ~(
                equal
                |
                both_missing
            )
        ).sum()
    )


parity_rows = []
mismatch_samples = []


print()
print("=" * 100)
print("FULL DIRECTORY PARITY")
print("=" * 100)


for section in PAYMENT_SECTIONS:

    production_path = (
        PAYMENT_DIR
        / f"{section}.parquet"
    )

    rebuilt_path = (
        VALIDATION_DIR
        / f"{section}.parquet"
    )


    production = pd.read_parquet(
        production_path
    )

    rebuilt = pd.read_parquet(
        rebuilt_path
    )


    # --------------------------------------------------------
    # ROW COUNT
    # --------------------------------------------------------

    if len(production) != len(rebuilt):

        raise RuntimeError(
            f"{section}: row count mismatch "
            f"{len(production):,} != {len(rebuilt):,}"
        )


    # --------------------------------------------------------
    # SCHEMA
    # --------------------------------------------------------

    production_columns = list(
        production.columns
    )

    rebuilt_columns = list(
        rebuilt.columns
    )


    missing_columns = sorted(
        set(production_columns)
        -
        set(rebuilt_columns)
    )

    extra_columns = sorted(
        set(rebuilt_columns)
        -
        set(production_columns)
    )


    if (
        missing_columns
        or
        extra_columns
    ):

        raise RuntimeError(
            f"{section}: schema mismatch. "
            f"Missing={missing_columns}; "
            f"extra={extra_columns}"
        )


    rebuilt = rebuilt[
        production_columns
    ]


    # --------------------------------------------------------
    # FIELD PARITY
    # --------------------------------------------------------

    section_mismatches = 0
    mismatching_columns = 0


    for column in production_columns:

        mismatch_count = (
            count_series_mismatches(
                production[
                    column
                ],
                rebuilt[
                    column
                ],
            )
        )


        section_mismatches += (
            mismatch_count
        )


        if mismatch_count:

            mismatching_columns += 1


            old = (
                production[
                    column
                ]
                .astype("string")
                .fillna("<NULL>")
            )

            new = (
                rebuilt[
                    column
                ]
                .astype("string")
                .fillna("<NULL>")
            )


            mismatch_mask = (
                old
                !=
                new
            )


            for idx in (
                mismatch_mask[
                    mismatch_mask
                ]
                .index[:10]
            ):

                mismatch_samples.append(
                    {
                        "section":
                            section,

                        "row_index":
                            int(idx),

                        "column":
                            column,

                        "production":
                            old.loc[
                                idx
                            ],

                        "rebuilt":
                            new.loc[
                                idx
                            ],
                    }
                )


    parity_rows.append(
        {
            "section":
                section,

            "rows":
                len(production),

            "columns":
                len(
                    production_columns
                ),

            "mismatching_columns":
                mismatching_columns,

            "field_level_mismatches":
                section_mismatches,
        }
    )


parity_df = pd.DataFrame(
    parity_rows
)


display(
    parity_df
)


# ============================================================
# 7. TOTAL CONTROL
# ============================================================

total_rows = int(
    parity_df[
        "rows"
    ].sum()
)


total_mismatches = int(
    parity_df[
        "field_level_mismatches"
    ].sum()
)


print()
print(
    "Rows rebuilt:",
    f"{total_rows:,}"
)

print(
    "Total field-level mismatches:",
    f"{total_mismatches:,}"
)


if total_rows != 402_028:

    raise RuntimeError(
        f"Expected 402,028 rows, got {total_rows:,}"
    )


if total_mismatches:

    print()
    print("=" * 100)
    print("MISMATCH SAMPLES")
    print("=" * 100)

    display(
        pd.DataFrame(
            mismatch_samples
        ).head(
            100
        )
    )


    raise RuntimeError(
        "Temporary end-to-end rebuild does not yet "
        "reproduce production exactly."
    )


# ============================================================
# 8. CONFIRM PRODUCTION UNCHANGED
# ============================================================

production_total = 0


for section in PAYMENT_SECTIONS:

    production_total += len(
        pd.read_parquet(
            PAYMENT_DIR
            / f"{section}.parquet",

            columns=[
                "source_report_id"
            ],
        )
    )


assert production_total == 402_028


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Full payment enrichment rebuild has exact "
    "production parity."
)

print(
    "Validation output:"
)

print(
    VALIDATION_DIR
)

print()
print(
    "Production payment files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

Account-source priority patched: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_resolution.py

TESTS
........................................................................ [ 68%]
.................................                                        [100%]
105 passed in 1.27s

Return code: 0

TEMPORARY FULL PAYMENT ENRICHMENT REBUILD


,section,rows,columns,output
0,monetary_contributions,27234,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\monetary_contributions.parquet
1,other_contributions,6168,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\other_contributions.parquet
2,state_funding,96,127,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\state_funding.parquet
3,other_incomes,19007,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\other_incomes.parquet
4,budget_expenses,29482,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\budget_expenses.parquet
5,outgoing_expenses,319901,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\outgoing_expenses.parquet
6,return_expenses,137,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\return_expenses.parquet
7,transfer_expenses,3,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1\transfer_expenses.parquet



FULL DIRECTORY PARITY


,section,rows,columns,mismatching_columns,field_level_mismatches
0,monetary_contributions,27234,126,0,0
1,other_contributions,6168,126,0,0
2,state_funding,96,127,0,0
3,other_incomes,19007,126,0,0
4,budget_expenses,29482,126,0,0
5,outgoing_expenses,319901,126,0,0
6,return_expenses,137,126,0,0
7,transfer_expenses,3,126,0,0



Rows rebuilt: 402,028
Total field-level mismatches: 0

DONE
Full payment enrichment rebuild has exact production parity.
Validation output:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_enrichment_rebuild_v0_1

Production payment files were NOT modified.
RAW files were NOT read.
API was NOT called.


In [131]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

NORMALIZED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "payments"
)

ENRICHED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REPORT_CONTEXT_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "report_context.parquet"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)


PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


# ============================================================
# THESE ARE ALREADY REPRODUCED BY THE NEW PAYMENT MODULES
# ============================================================

PAYMENT_DERIVED_COLUMNS = {

    # Resolution
    "payment_direction",
    "party_account_iban",

    "state_funding_account_confirmed",
    "state_funding_form_source",
    "state_funding_form_code",

    "internal_counterparty_organization_id",
    "internal_counterparty_organization_name",
    "internal_counterparty_organization_level",

    "internal_transfer_source_organization_id",
    "internal_transfer_source_organization_level",

    "internal_transfer_destination_organization_id",
    "internal_transfer_destination_organization_level",

    "internal_transfer_direction",

    # Analytical rules
    "party_account_type",
    "party_account_type_analytical",
    "party_account_type_resolution_method",

    "internal_transfer_funding_source",

    "analytical_payment_type",
    "was_reclassified",
    "reclassification_rule",
    "funding_source_analytical",
}


# ============================================================
# HELPERS
# ============================================================

def parquet_columns(path):

    schema = pq.ParquetFile(
        path
    ).schema_arrow

    return list(
        schema.names
    )


def row_count(path):

    return (
        pq.ParquetFile(
            path
        )
        .metadata
        .num_rows
    )


def normalized_compare_series(series):

    return (
        series
        .astype("string")
        .fillna("<NULL>")
    )


def mismatch_count(left, right):

    if len(left) != len(right):
        raise ValueError(
            "Series lengths differ."
        )

    both_missing = (
        left.isna()
        &
        right.isna()
    )

    try:

        equal = (
            left.eq(right)
            .fillna(False)
        )

    except Exception:

        equal = (
            normalized_compare_series(left)
            ==
            normalized_compare_series(right)
        )

    return int(
        (
            ~(
                equal
                |
                both_missing
            )
        ).sum()
    )


# ============================================================
# 1. REPORT CONTEXT
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH
)


if (
    report_context[
        "source_report_id"
    ].nunique()
    !=
    len(report_context)
):

    raise RuntimeError(
        "report_context source_report_id is not unique."
    )


context_columns = set(
    report_context.columns
)


print()
print("=" * 110)
print("REPORT CONTEXT")
print("=" * 110)

print(
    "Rows:",
    f"{len(report_context):,}"
)

print(
    "Columns:",
    len(report_context.columns)
)

print()
print(
    sorted(
        report_context.columns
    )
)


# ============================================================
# 2. SCHEMA DELTA normalized -> enriched
# ============================================================

schema_rows = []
enriched_only_all = set()
normalized_only_all = set()


for section in PAYMENT_SECTIONS:

    normalized_path = (
        NORMALIZED_DIR
        / f"{section}.parquet"
    )

    enriched_path = (
        ENRICHED_DIR
        / f"{section}.parquet"
    )


    if not normalized_path.exists():

        raise FileNotFoundError(
            normalized_path
        )


    if not enriched_path.exists():

        raise FileNotFoundError(
            enriched_path
        )


    normalized_columns = parquet_columns(
        normalized_path
    )

    enriched_columns = parquet_columns(
        enriched_path
    )


    normalized_set = set(
        normalized_columns
    )

    enriched_set = set(
        enriched_columns
    )


    enriched_only = sorted(
        enriched_set
        -
        normalized_set
    )

    normalized_only = sorted(
        normalized_set
        -
        enriched_set
    )


    enriched_only_all.update(
        enriched_only
    )

    normalized_only_all.update(
        normalized_only
    )


    schema_rows.append(
        {
            "section":
                section,

            "normalized_rows":
                row_count(
                    normalized_path
                ),

            "enriched_rows":
                row_count(
                    enriched_path
                ),

            "normalized_columns":
                len(
                    normalized_columns
                ),

            "enriched_columns":
                len(
                    enriched_columns
                ),

            "common_columns":
                len(
                    normalized_set
                    &
                    enriched_set
                ),

            "enriched_only":
                len(
                    enriched_only
                ),

            "normalized_only":
                len(
                    normalized_only
                ),
        }
    )


schema_df = pd.DataFrame(
    schema_rows
)


print()
print("=" * 110)
print("NORMALIZED -> ENRICHED SCHEMA DELTA")
print("=" * 110)

display(
    schema_df
)


if not (
    schema_df[
        "normalized_rows"
    ]
    ==
    schema_df[
        "enriched_rows"
    ]
).all():

    raise RuntimeError(
        "Normalized/enriched row counts differ."
    )


# ============================================================
# 3. ALL ENRICHED-ONLY COLUMNS
# ============================================================

enriched_only_table = []


for column in sorted(
    enriched_only_all
):

    if column in PAYMENT_DERIVED_COLUMNS:

        category = (
            "payment_resolution_or_analytical"
        )

    elif column in context_columns:

        category = (
            "report_context"
        )

    else:

        category = (
            "other_base_enrichment"
        )


    enriched_only_table.append(
        {
            "column":
                column,

            "category":
                category,
        }
    )


enriched_only_df = pd.DataFrame(
    enriched_only_table
)


print()
print("=" * 110)
print("ALL ENRICHED-ONLY COLUMNS")
print("=" * 110)

display(
    enriched_only_df
)


if normalized_only_all:

    print()
    print("=" * 110)
    print("NORMALIZED-ONLY COLUMNS")
    print("=" * 110)

    display(
        pd.DataFrame(
            {
                "column":
                    sorted(
                        normalized_only_all
                    )
            }
        )
    )


# ============================================================
# 4. VERIFY ALL COMMON COLUMNS ARE UNCHANGED
#
# This establishes that enrichment is additive rather than
# silently transforming normalized source fields.
# ============================================================

common_parity_rows = []


print()
print("=" * 110)
print("COMMON COLUMN PARITY")
print("=" * 110)


for section in PAYMENT_SECTIONS:

    normalized = pd.read_parquet(
        NORMALIZED_DIR
        / f"{section}.parquet"
    )

    enriched = pd.read_parquet(
        ENRICHED_DIR
        / f"{section}.parquet"
    )


    if len(normalized) != len(enriched):

        raise RuntimeError(
            f"{section}: row count mismatch."
        )


    common_columns = [
        column
        for column
        in normalized.columns
        if column
        in enriched.columns
    ]


    section_mismatches = 0
    mismatching_columns = []


    for column in common_columns:

        mismatches = mismatch_count(
            normalized[
                column
            ],
            enriched[
                column
            ],
        )


        section_mismatches += (
            mismatches
        )


        if mismatches:

            mismatching_columns.append(
                f"{column} ({mismatches})"
            )


    common_parity_rows.append(
        {
            "section":
                section,

            "common_columns":
                len(
                    common_columns
                ),

            "mismatching_columns":
                len(
                    mismatching_columns
                ),

            "field_level_mismatches":
                section_mismatches,

            "details":
                " | ".join(
                    mismatching_columns[:20]
                ),
        }
    )


common_parity_df = pd.DataFrame(
    common_parity_rows
)


display(
    common_parity_df
)


# ============================================================
# 5. VERIFY REPORT_CONTEXT COLUMNS
#
# Any enriched-only column with the same name in
# report_context should be a direct many-to-one join.
# ============================================================

context_added_columns = sorted(
    (
        enriched_only_all
        &
        context_columns
    )
    -
    PAYMENT_DERIVED_COLUMNS
)


context_parity_rows = []


print()
print("=" * 110)
print("REPORT_CONTEXT JOIN PARITY")
print("=" * 110)


if context_added_columns:

    context_lookup = (
        report_context[
            [
                "source_report_id",
                *context_added_columns,
            ]
        ]
        .copy()
    )


    for section in PAYMENT_SECTIONS:

        normalized = pd.read_parquet(
            NORMALIZED_DIR
            / f"{section}.parquet",
            columns=[
                "source_report_id"
            ],
        )


        enriched = pd.read_parquet(
            ENRICHED_DIR
            / f"{section}.parquet",
            columns=[
                "source_report_id",
                *context_added_columns,
            ],
        )


        expected = (
            normalized
            .merge(
                context_lookup,
                on="source_report_id",
                how="left",
                validate="many_to_one",
                sort=False,
            )
        )


        section_mismatches = 0


        for column in context_added_columns:

            section_mismatches += (
                mismatch_count(
                    enriched[
                        column
                    ],
                    expected[
                        column
                    ],
                )
            )


        context_parity_rows.append(
            {
                "section":
                    section,

                "context_columns":
                    len(
                        context_added_columns
                    ),

                "field_level_mismatches":
                    section_mismatches,
            }
        )


    display(
        pd.DataFrame(
            context_parity_rows
        )
    )


else:

    print(
        "No enriched-only report_context columns."
    )


# ============================================================
# 6. INSPECT ONLY REMAINING BASE-ENRICHMENT COLUMNS
#
# These are the columns we still need to explain before
# normalized -> enriched can be productionized.
# ============================================================

remaining_columns = sorted(
    enriched_only_all
    -
    PAYMENT_DERIVED_COLUMNS
    -
    context_columns
)


print()
print("=" * 110)
print("REMAINING BASE-ENRICHMENT COLUMNS")
print("=" * 110)


remaining_summary = []


for column in remaining_columns:

    sections_with_column = []


    total_rows = 0
    non_null_rows = 0
    unique_values = set()
    examples = []


    for section in PAYMENT_SECTIONS:

        path = (
            ENRICHED_DIR
            / f"{section}.parquet"
        )


        columns = parquet_columns(
            path
        )


        if column not in columns:
            continue


        sections_with_column.append(
            section
        )


        df = pd.read_parquet(
            path,
            columns=[
                column
            ],
        )


        total_rows += len(
            df
        )


        non_null = (
            df[
                column
            ]
            .dropna()
        )


        non_null_rows += len(
            non_null
        )


        for value in (
            non_null
            .astype("string")
            .drop_duplicates()
            .head(20)
        ):

            value = str(
                value
            )

            unique_values.add(
                value
            )


            if len(examples) < 10:

                examples.append(
                    value
                )


    remaining_summary.append(
        {
            "column":
                column,

            "sections":
                len(
                    sections_with_column
                ),

            "rows":
                total_rows,

            "non_null_rows":
                non_null_rows,

            "sample_unique_values":
                len(
                    unique_values
                ),

            "examples":
                " | ".join(
                    examples
                ),
        }
    )


remaining_df = pd.DataFrame(
    remaining_summary
)


if len(
    remaining_df
):

    display(
        remaining_df
    )

else:

    print(
        "None. All enrichment columns are already "
        "explained by report_context + payment modules."
    )


# ============================================================
# 7. REFERENCE TABLE SCHEMAS
#
# Read schema only.
# We do NOT use older account references as authoritative
# state-funding evidence.
# ============================================================

reference_rows = []


for filename in [
    "organization_reference.parquet",
    "report_context.parquet",
    "report_account_reference.parquet",
    "organization_account_reference.parquet",
    "state_funding_account_reference.parquet",
]:

    path = (
        REFERENCE_DIR
        / filename
    )


    if not path.exists():
        continue


    columns = parquet_columns(
        path
    )


    reference_rows.append(
        {
            "file":
                filename,

            "rows":
                row_count(
                    path
                ),

            "columns":
                len(
                    columns
                ),

            "column_names":
                " | ".join(
                    columns
                ),
        }
    )


print()
print("=" * 110)
print("REFERENCE TABLE SCHEMAS")
print("=" * 110)

display(
    pd.DataFrame(
        reference_rows
    )
)


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

total_common_mismatches = int(
    common_parity_df[
        "field_level_mismatches"
    ].sum()
)


total_context_mismatches = (
    int(
        pd.DataFrame(
            context_parity_rows
        )[
            "field_level_mismatches"
        ].sum()
    )
    if context_parity_rows
    else
    0
)


print()
print("=" * 110)
print("SUMMARY")
print("=" * 110)

print(
    "Payment rows:",
    f"{schema_df['normalized_rows'].sum():,}"
)

print(
    "Common-column mismatches:",
    f"{total_common_mismatches:,}"
)

print(
    "Report-context join mismatches:",
    f"{total_context_mismatches:,}"
)

print(
    "Remaining unexplained base-enrichment columns:",
    len(
        remaining_columns
    )
)

print()
print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No parquet files were modified."
)


REPORT CONTEXT
Rows: 78,791
Columns: 56

['analysis_override', 'analysis_selected_report_id', 'analysis_selection_method', 'annual_report_count', 'continuity_exact', 'created_date', 'created_date_dt', 'data_recency_status', 'discovered_at_utc', 'entity_type', 'has_annual_report', 'has_mixed_periodicity', 'has_quarterly_reports', 'include_in_annualized_analysis', 'is_latest_report_for_organization', 'is_party_office', 'is_selected_report', 'is_signed', 'official_selected_report_id', 'organization_code', 'organization_id', 'organization_level', 'organization_name_current', 'override_reason', 'party_code', 'party_id_matches_organization', 'party_name_current', 'period_label', 'period_type', 'public_summary_generated_at', 'public_summary_version', 'quarter', 'quarterly_report_count', 'region', 'region_resolution_method', 'region_resolution_source', 'region_source', 'region_source_address_type', 'report_id', 'report_instance_count', 'report_period_order', 'report_type', 'root_party_id', 's

,section,normalized_rows,enriched_rows,normalized_columns,enriched_columns,common_columns,enriched_only,normalized_only
0,monetary_contributions,27234,27234,87,126,87,39,0
1,other_contributions,6168,6168,87,126,87,39,0
2,state_funding,96,96,87,127,87,40,0
3,other_incomes,19007,19007,87,126,87,39,0
4,budget_expenses,29482,29482,87,126,87,39,0
5,outgoing_expenses,319901,319901,87,126,87,39,0
6,return_expenses,137,137,87,126,87,39,0
7,transfer_expenses,3,3,87,126,87,39,0



ALL ENRICHED-ONLY COLUMNS


,column,category
0,analysis_selected,other_base_enrichment
1,analytical_payment_type,payment_resolution_or_analytical
2,funding_source_analytical,payment_resolution_or_analytical
3,internal_counterparty_organization_id,payment_resolution_or_analytical
4,internal_counterparty_organization_level,payment_resolution_or_analytical
5,internal_counterparty_organization_name,payment_resolution_or_analytical
6,internal_transfer,other_base_enrichment
7,internal_transfer_destination_organization_id,payment_resolution_or_analytical
8,internal_transfer_destination_organization_level,payment_resolution_or_analytical
9,internal_transfer_direction,payment_resolution_or_analytical



COMMON COLUMN PARITY


...


REMAINING BASE-ENRICHMENT COLUMNS


,column,sections,rows,non_null_rows,sample_unique_values,examples
0,analysis_selected,8,402028,402028,1,True | True | True | True | True | True | True | True
1,internal_transfer,8,402028,402028,2,False | True | False | True | False | True | False | True | False | False
2,internal_transfer_rule,8,402028,18167,1,same_root_party_organization_code | same_root_party_organization_code | same_root_party_organization_code | same_root_party_organization_code | same_root_party_organization_code | same_root_party_organization_code
3,official_selected,8,402028,402028,2,True | True | True | True | False | True | True | False | True | True
4,party_account_type_source,8,402028,395199,45,Поточний рахунок | UAH (гривня) | 14305909 | поточний рахунок | поточний | Поточний | касове обслуговування | Картковий рахунок | розрахунковий рахунок | Нараховані доходи за розрахунково-касове обслуговування
5,payer_same_party_code_match,8,402028,402028,2,False | True | False | True | False | True | False | False | False | False
6,payer_type_analytical,8,402028,52549,3,Фізична особа | Юридична особа | internal_party_transfer | Фізична особа | Юридична особа | internal_party_transfer | internal_party_transfer | Юридична особа | Фізична особа | Фізична особа
7,receiver_same_party_code_match,8,402028,402028,2,False | False | False | False | True | False | False | True | False | True
8,receiver_type_analytical,8,402028,349523,3,internal_party_transfer | Юридична особа | Фізична особа | Фізична особа | Юридична особа | internal_party_transfer | Фізична особа | Юридична особа | internal_party_transfer | Фізична особа
9,state_funding_form,1,96,96,1,Державне фінансування статутної діяльності політичної партії



REFERENCE TABLE SCHEMAS


,file,rows,columns,column_names
0,organization_reference.parquet,9917,14,organization_id | root_party_id | organization_level | organization_code | organization_name_current | party_code | party_name_current | region | region_source | region_source_address_type | region_resolution_method | region_resolution_source | is_active | parent_id
1,report_context.parquet,78791,56,report_id | organization_id | root_party_id | entity_type | source_party_id | party_id_matches_organization | is_party_office | schema_version | report_type | year | quarter | period_type | signed_date | created_date | signatory_id | special_status | public_summary_version | public_summary_generated_at | discovered_at_utc | signed_date_dt | created_date_dt | is_signed | report_instance_count | signed_instance_count | unsigned_instance_count | selected_report_id | selection_method | is_selected_report | has_annual_report | has_quarterly_reports | annual_report_count | quarterly_report_count | selected_report_count | has_mixed_periodicity | include_in_annualized_analysis | official_selected_report_id | analysis_selected_report_id | analysis_selection_method | analysis_override | override_reason | continuity_exact | source_report_id | report_period_order | period_label | is_latest_report_for_organization | data_recency_status | organization_level | organization_code | organization_name_current | party_code | party_name_current | region | region_source | region_source_address_type | region_resolution_method | region_resolution_source
2,report_account_reference.parquet,12586,9,source_report_id | organization_id | root_party_id | party_account_iban | party_account_iban_source | party_account_type_source | party_account_type_analytical | party_account_type_resolution_method | snapshot_rows
3,organization_account_reference.parquet,1668,25,organization_id | root_party_id | party_account_iban | first_observed_year | last_observed_year | reports_observed | has_explicit_statutory | has_explicit_campaign_reimbursement | has_explicit_ordinary | has_generic_budget_account | has_social_insurance_account | has_deposit_account | has_transit_account | has_card_account | has_escrow_account | has_other_special_account | observed_source_account_types | positive_state_receipt_rows | positive_state_receipt_amount | first_positive_state_receipt_date | last_positive_state_receipt_date | party_account_type_analytical | party_account_type | party_account_type_resolution_method | party_account_funding_class
4,state_funding_account_reference.parquet,7,16,root_party_id | organization_id | party_account_iban | party_name_current | organization_name_current | organization_level | first_state_receipt_date | last_state_receipt_date | positive_state_receipt_rows | positive_state_receipt_amount | state_funding_form_count | state_funding_forms_observed | state_funding_source_forms_observed | state_funding_account_confirmed | state_funding_form_code | state_funding_account_evidence



SUMMARY
Payment rows: 402,028
Common-column mismatches: 10,109,508
Report-context join mismatches: 2,351,797
Remaining unexplained base-enrichment columns: 10

RAW files were NOT read.
API was NOT called.
No parquet files were modified.


In [132]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

NORMALIZED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "payments"
)

ENRICHED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

REPORT_CONTEXT_PATH = (
    REFERENCE_DIR
    / "report_context.parquet"
)

ORG_REFERENCE_PATH = (
    REFERENCE_DIR
    / "organization_reference.parquet"
)

REPORT_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "report_account_reference.parquet"
)


PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


INCOMING_SECTIONS = {
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
}


# ============================================================
# HELPERS
# ============================================================

def compare_series(left, right):

    if len(left) != len(right):

        raise ValueError(
            "Series lengths differ."
        )

    both_missing = (
        left.isna()
        &
        right.isna()
    )

    try:

        equal = (
            left.eq(right)
            .fillna(False)
        )

    except Exception:

        equal = (
            left
            .astype("string")
            .fillna("<NULL>")
            ==
            right
            .astype("string")
            .fillna("<NULL>")
        )

    return int(
        (
            ~(
                equal
                |
                both_missing
            )
        ).sum()
    )


def normalized_code(series):

    return (
        series
        .astype("string")
        .str.strip()
    )


def choose_row_key(
    normalized,
    enriched,
):

    # --------------------------------------------------------
    # Preferred production-row identity
    # --------------------------------------------------------

    if (
        "source_row_id"
        in normalized.columns
        and
        "source_row_id"
        in enriched.columns
    ):

        n_key = (
            normalized[
                "source_row_id"
            ]
            .astype("string")
        )

        e_key = (
            enriched[
                "source_row_id"
            ]
            .astype("string")
        )

        if (
            n_key.notna().all()
            and
            e_key.notna().all()
            and
            n_key.is_unique
            and
            e_key.is_unique
            and
            set(n_key)
            ==
            set(e_key)
        ):

            return (
                "source_row_id",
                n_key,
                e_key,
            )


    # --------------------------------------------------------
    # Conservative fallback
    # --------------------------------------------------------

    required = {
        "source_report_id",
        "source_row_id",
    }

    if (
        required
        <=
        set(normalized.columns)
        and
        required
        <=
        set(enriched.columns)
    ):

        n_key = (
            normalized[
                "source_report_id"
            ]
            .astype("string")
            +
            "||"
            +
            normalized[
                "source_row_id"
            ]
            .astype("string")
        )

        e_key = (
            enriched[
                "source_report_id"
            ]
            .astype("string")
            +
            "||"
            +
            enriched[
                "source_row_id"
            ]
            .astype("string")
        )

        if (
            n_key.is_unique
            and
            e_key.is_unique
            and
            set(n_key)
            ==
            set(e_key)
        ):

            return (
                "source_report_id + source_row_id",
                n_key,
                e_key,
            )


    raise RuntimeError(
        "No safe stable row key found."
    )


# ============================================================
# REFERENCE INPUTS
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH
)

organization_reference = pd.read_parquet(
    ORG_REFERENCE_PATH
)

report_account_reference = pd.read_parquet(
    REPORT_ACCOUNT_REFERENCE_PATH
)


# ============================================================
# BUILD STRICT SAME-ROOT ORGANIZATION-CODE MAP
# ============================================================

org_codes = (
    organization_reference[
        [
            "root_party_id",
            "organization_id",
            "organization_code",
            "organization_name_current",
            "organization_level",
        ]
    ]
    .copy()
)


org_codes[
    "organization_code"
] = normalized_code(
    org_codes[
        "organization_code"
    ]
)


org_codes = org_codes[
    org_codes[
        "organization_code"
    ].notna()
    &
    (
        org_codes[
            "organization_code"
        ]
        !=
        ""
    )
]


code_counts = (
    org_codes
    .groupby(
        [
            "root_party_id",
            "organization_code",
        ],
        as_index=False,
    )[
        "organization_id"
    ]
    .nunique()
    .rename(
        columns={
            "organization_id":
                "_organization_count"
        }
    )
)


unique_org_codes = (
    org_codes
    .merge(
        code_counts[
            code_counts[
                "_organization_count"
            ]
            ==
            1
        ][
            [
                "root_party_id",
                "organization_code",
            ]
        ],
        on=[
            "root_party_id",
            "organization_code",
        ],
        how="inner",
        validate="many_to_one",
    )
    .drop_duplicates(
        subset=[
            "root_party_id",
            "organization_code",
        ]
    )
)


same_root_key_set = set(
    zip(
        unique_org_codes[
            "root_party_id"
        ].astype("string"),

        unique_org_codes[
            "organization_code"
        ].astype("string"),
    )
)


# ============================================================
# 1. KEY AUDIT + KEYED COMMON-COLUMN PARITY
# ============================================================

key_rows = []
common_rows = []
context_rows = []

remaining_rows = []

type_pairs_payer = []
type_pairs_receiver = []


CONTEXT_COLUMNS = [
    "organization_code",
    "organization_level",
    "organization_name_current",
    "party_code",
    "party_name_current",
    "region",
    "region_resolution_method",
    "region_resolution_source",
    "region_source",
    "region_source_address_type",
]


print()
print("=" * 110)
print("KEYED NORMALIZED -> ENRICHED AUDIT")
print("=" * 110)


for section in PAYMENT_SECTIONS:

    normalized = pd.read_parquet(
        NORMALIZED_DIR
        / f"{section}.parquet"
    )

    enriched = pd.read_parquet(
        ENRICHED_DIR
        / f"{section}.parquet"
    )


    # --------------------------------------------------------
    # STABLE ROW KEY
    # --------------------------------------------------------

    (
        key_name,
        normalized_key,
        enriched_key,
    ) = choose_row_key(
        normalized,
        enriched,
    )


    normalized = (
        normalized
        .assign(
            _audit_row_key=
                normalized_key
        )
        .set_index(
            "_audit_row_key",
            drop=True,
        )
        .sort_index()
    )


    enriched = (
        enriched
        .assign(
            _audit_row_key=
                enriched_key
        )
        .set_index(
            "_audit_row_key",
            drop=True,
        )
        .sort_index()
    )


    key_rows.append(
        {
            "section":
                section,

            "rows":
                len(normalized),

            "key":
                key_name,

            "same_key_set":
                normalized.index.equals(
                    enriched.index
                ),
        }
    )


    # --------------------------------------------------------
    # COMMON COLUMN PARITY AFTER ALIGNMENT
    # --------------------------------------------------------

    common_columns = [
        column
        for column
        in normalized.columns
        if column
        in enriched.columns
    ]


    mismatching_columns = []
    field_mismatches = 0


    for column in common_columns:

        mismatches = compare_series(
            normalized[
                column
            ],
            enriched[
                column
            ],
        )


        field_mismatches += (
            mismatches
        )


        if mismatches:

            mismatching_columns.append(
                f"{column} ({mismatches})"
            )


    common_rows.append(
        {
            "section":
                section,

            "common_columns":
                len(
                    common_columns
                ),

            "mismatching_columns":
                len(
                    mismatching_columns
                ),

            "field_level_mismatches":
                field_mismatches,

            "details":
                " | ".join(
                    mismatching_columns[:20]
                ),
        }
    )


    # --------------------------------------------------------
    # REPORT CONTEXT — KEYED PARITY
    # --------------------------------------------------------

    expected_context = (
        normalized[
            [
                "source_report_id"
            ]
        ]
        .reset_index()
        .merge(
            report_context[
                [
                    "source_report_id",
                    *CONTEXT_COLUMNS,
                ]
            ],
            on="source_report_id",
            how="left",
            validate="many_to_one",
            sort=False,
        )
        .set_index(
            "_audit_row_key"
        )
        .reindex(
            normalized.index
        )
    )


    context_mismatches = 0
    context_bad_columns = []


    for column in CONTEXT_COLUMNS:

        mismatches = compare_series(
            enriched[
                column
            ],
            expected_context[
                column
            ],
        )


        context_mismatches += (
            mismatches
        )


        if mismatches:

            context_bad_columns.append(
                f"{column} ({mismatches})"
            )


    context_rows.append(
        {
            "section":
                section,

            "context_columns":
                len(
                    CONTEXT_COLUMNS
                ),

            "mismatching_columns":
                len(
                    context_bad_columns
                ),

            "field_level_mismatches":
                context_mismatches,

            "details":
                " | ".join(
                    context_bad_columns
                ),
        }
    )


    # ========================================================
    # 2. TEST REMAINING BASE-ENRICHMENT FIELDS
    # ========================================================

    # --------------------------------------------------------
    # analysis_selected
    #
    # All normalized payment rows come from the
    # analysis-selected corpus.
    # --------------------------------------------------------

    predicted_analysis_selected = pd.Series(
        True,
        index=normalized.index,
    )


    # --------------------------------------------------------
    # official_selected
    # --------------------------------------------------------

    predicted_official_selected = (
        normalized[
            "source_report_id"
        ].astype("string")
        ==
        normalized[
            "official_selected_report_id"
        ].astype("string")
    )


    # --------------------------------------------------------
    # SAME-ROOT EDRPOU MATCHES
    # --------------------------------------------------------

    payer_codes = normalized_code(
        normalized[
            "payer_code_normalized"
        ]
    )


    receiver_codes = normalized_code(
        normalized[
            "receiver_code_normalized"
        ]
    )


    roots = (
        normalized[
            "root_party_id"
        ]
        .astype("string")
    )


    predicted_payer_match = pd.Series(
        [
            (
                root,
                code,
            )
            in
            same_root_key_set

            if (
                pd.notna(root)
                and
                pd.notna(code)
            )
            else
            False

            for root, code
            in zip(
                roots,
                payer_codes,
            )
        ],
        index=normalized.index,
    )


    predicted_receiver_match = pd.Series(
        [
            (
                root,
                code,
            )
            in
            same_root_key_set

            if (
                pd.notna(root)
                and
                pd.notna(code)
            )
            else
            False

            for root, code
            in zip(
                roots,
                receiver_codes,
            )
        ],
        index=normalized.index,
    )


    if section in INCOMING_SECTIONS:

        predicted_internal_transfer = (
            predicted_payer_match
        )

    else:

        predicted_internal_transfer = (
            predicted_receiver_match
        )


    predicted_internal_rule = pd.Series(
        pd.NA,
        index=normalized.index,
        dtype="string",
    )


    predicted_internal_rule.loc[
        predicted_internal_transfer
    ] = (
        "same_root_party_organization_code"
    )


    # --------------------------------------------------------
    # PARTY ACCOUNT TYPE SOURCE
    #
    # Report-relative property_moneys account metadata.
    # Not authoritative for state-funding origin.
    # --------------------------------------------------------

    account_base = (
        normalized[
            [
                "source_report_id",
                "organization_id",
                "receiver_account_iban_canonical",
            ]
        ]
        .reset_index()
        .rename(
            columns={
                "receiver_account_iban_canonical":
                    "party_account_iban"
            }
        )
    )


    account_ref = (
        report_account_reference[
            [
                "source_report_id",
                "organization_id",
                "party_account_iban",
                "party_account_type_source",
            ]
        ]
        .drop_duplicates(
            subset=[
                "source_report_id",
                "organization_id",
                "party_account_iban",
            ]
        )
    )


    expected_account_type = (
        account_base
        .merge(
            account_ref,
            on=[
                "source_report_id",
                "organization_id",
                "party_account_iban",
            ],
            how="left",
            validate="many_to_one",
            sort=False,
        )
        .set_index(
            "_audit_row_key"
        )
        .reindex(
            normalized.index
        )[
            "party_account_type_source"
        ]
    )


    # --------------------------------------------------------
    # state_funding_form
    # --------------------------------------------------------

    if (
        section
        ==
        "state_funding"
    ):

        predicted_state_funding_form = (
            normalized[
                "payment_type_detail_source"
            ]
            if
            "payment_type_detail_source"
            in normalized.columns
            else
            pd.Series(
                pd.NA,
                index=normalized.index,
            )
        )

    else:

        predicted_state_funding_form = pd.Series(
            pd.NA,
            index=normalized.index,
        )


    predictions = {

        "analysis_selected":
            predicted_analysis_selected,

        "official_selected":
            predicted_official_selected,

        "payer_same_party_code_match":
            predicted_payer_match,

        "receiver_same_party_code_match":
            predicted_receiver_match,

        "internal_transfer":
            predicted_internal_transfer,

        "internal_transfer_rule":
            predicted_internal_rule,

        "party_account_type_source":
            expected_account_type,

        "state_funding_form":
            predicted_state_funding_form,
    }


    for column, predicted in (
        predictions.items()
    ):

        if column not in enriched.columns:
            continue


        mismatches = compare_series(
            enriched[
                column
            ],
            predicted,
        )


        remaining_rows.append(
            {
                "section":
                    section,

                "column":
                    column,

                "rows":
                    len(enriched),

                "mismatches":
                    mismatches,

                "match_ratio":
                    (
                        1
                        -
                        mismatches
                        /
                        len(enriched)
                    ),
            }
        )


    # --------------------------------------------------------
    # TYPE MAPPING DIAGNOSTIC
    #
    # We do NOT guess the rule yet.
    # Show exact normalized -> analytical mappings,
    # distinguishing internal transfers.
    # --------------------------------------------------------

    if (
        "payer_type_analytical"
        in enriched.columns
    ):

        payer_pairs = pd.DataFrame(
            {
                "section":
                    section,

                "internal_transfer":
                    enriched[
                        "internal_transfer"
                    ],

                "type_normalized":
                    normalized[
                        "payer_type_normalized"
                    ],

                "type_analytical":
                    enriched[
                        "payer_type_analytical"
                    ],
            }
        )

        type_pairs_payer.append(
            payer_pairs
        )


    if (
        "receiver_type_analytical"
        in enriched.columns
    ):

        receiver_pairs = pd.DataFrame(
            {
                "section":
                    section,

                "internal_transfer":
                    enriched[
                        "internal_transfer"
                    ],

                "type_normalized":
                    normalized[
                        "receiver_type_normalized"
                    ],

                "type_analytical":
                    enriched[
                        "receiver_type_analytical"
                    ],
            }
        )

        type_pairs_receiver.append(
            receiver_pairs
        )


# ============================================================
# OUTPUT 1 — ROW IDENTITY
# ============================================================

key_df = pd.DataFrame(
    key_rows
)


print()
print("=" * 110)
print("ROW KEY AUDIT")
print("=" * 110)

display(
    key_df
)


# ============================================================
# OUTPUT 2 — REAL COMMON-COLUMN PARITY
# ============================================================

common_df = pd.DataFrame(
    common_rows
)


print()
print("=" * 110)
print("KEYED COMMON-COLUMN PARITY")
print("=" * 110)

display(
    common_df
)


print(
    "Total keyed common-column mismatches:",
    f"{int(common_df['field_level_mismatches'].sum()):,}"
)


# ============================================================
# OUTPUT 3 — REAL REPORT CONTEXT PARITY
# ============================================================

context_df = pd.DataFrame(
    context_rows
)


print()
print("=" * 110)
print("KEYED REPORT-CONTEXT PARITY")
print("=" * 110)

display(
    context_df
)


print(
    "Total keyed report-context mismatches:",
    f"{int(context_df['field_level_mismatches'].sum()):,}"
)


# ============================================================
# OUTPUT 4 — REMAINING FIELD RULES
# ============================================================

remaining_df = pd.DataFrame(
    remaining_rows
)


print()
print("=" * 110)
print("REMAINING BASE-ENRICHMENT RULE PARITY")
print("=" * 110)

display(
    remaining_df
)


if len(
    remaining_df
):

    remaining_summary = (
        remaining_df
        .groupby(
            "column",
            as_index=False,
        )
        .agg(
            rows=(
                "rows",
                "sum",
            ),

            mismatches=(
                "mismatches",
                "sum",
            ),
        )
    )


    remaining_summary[
        "match_ratio"
    ] = (
        1
        -
        remaining_summary[
            "mismatches"
        ]
        /
        remaining_summary[
            "rows"
        ]
    )


    print()
    print(
        "=" * 110
    )

    print(
        "REMAINING RULE SUMMARY"
    )

    print(
        "=" * 110
    )


    display(
        remaining_summary
    )


# ============================================================
# OUTPUT 5 — PAYER TYPE MAPPING
# ============================================================

if type_pairs_payer:

    payer_mapping = (
        pd.concat(
            type_pairs_payer,
            ignore_index=True,
        )
        .groupby(
            [
                "internal_transfer",
                "type_normalized",
                "type_analytical",
            ],
            dropna=False,
            as_index=False,
        )
        .size()
        .sort_values(
            "size",
            ascending=False,
        )
    )


    print()
    print("=" * 110)
    print("PAYER TYPE MAPPING")
    print("=" * 110)

    display(
        payer_mapping.head(
            50
        )
    )


# ============================================================
# OUTPUT 6 — RECEIVER TYPE MAPPING
# ============================================================

if type_pairs_receiver:

    receiver_mapping = (
        pd.concat(
            type_pairs_receiver,
            ignore_index=True,
        )
        .groupby(
            [
                "internal_transfer",
                "type_normalized",
                "type_analytical",
            ],
            dropna=False,
            as_index=False,
        )
        .size()
        .sort_values(
            "size",
            ascending=False,
        )
    )


    print()
    print("=" * 110)
    print("RECEIVER TYPE MAPPING")
    print("=" * 110)

    display(
        receiver_mapping.head(
            50
        )
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 110)
print("DONE")
print("=" * 110)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No parquet files were modified."
)


KEYED NORMALIZED -> ENRICHED AUDIT

ROW KEY AUDIT


,section,rows,key,same_key_set
0,monetary_contributions,27234,source_row_id,True
1,other_contributions,6168,source_row_id,True
2,state_funding,96,source_row_id,True
3,other_incomes,19007,source_row_id,True
4,budget_expenses,29482,source_row_id,True
5,outgoing_expenses,319901,source_row_id,True
6,return_expenses,137,source_row_id,True
7,transfer_expenses,3,source_row_id,True



KEYED COMMON-COLUMN PARITY


,section,common_columns,mismatching_columns,field_level_mismatches,details
0,monetary_contributions,87,0,0,
1,other_contributions,87,0,0,
2,state_funding,87,0,0,
3,other_incomes,87,0,0,
4,budget_expenses,87,0,0,
5,outgoing_expenses,87,0,0,
6,return_expenses,87,0,0,
7,transfer_expenses,87,0,0,


Total keyed common-column mismatches: 0

KEYED REPORT-CONTEXT PARITY


...


PAYER TYPE MAPPING


,internal_transfer,type_normalized,type_analytical,size
2,False,NaN,NaN,334837
0,False,Фізична особа,Фізична особа,31980
1,False,Юридична особа,Юридична особа,17044
6,True,NaN,NaN,14642
4,True,Юридична особа,internal_party_transfer,3511
3,True,Фізична особа,Фізична особа,8
5,True,Юридична особа,Юридична особа,6



RECEIVER TYPE MAPPING


,internal_transfer,type_normalized,type_analytical,size
1,False,Юридична особа,Юридична особа,231301
0,False,Фізична особа,Фізична особа,103566
2,False,NaN,NaN,48994
3,True,Юридична особа,internal_party_transfer,14656
4,True,NaN,NaN,3511



DONE
RAW files were NOT read.
API was NOT called.
No parquet files were modified.


In [133]:
from pathlib import Path

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

ENRICHED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

REPORT_CONTEXT_PATH = (
    REFERENCE_DIR
    / "report_context.parquet"
)

ORG_REFERENCE_PATH = (
    REFERENCE_DIR
    / "organization_reference.parquet"
)


PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


INCOMING_SECTIONS = {
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
}


OUTGOING_SECTIONS = {
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
}


# ============================================================
# HELPERS
# ============================================================

def equal_series(left, right):

    left = (
        left
        .astype("string")
        .fillna("<NULL>")
    )

    right = (
        right
        .astype("string")
        .fillna("<NULL>")
    )

    return (
        left
        ==
        right
    )


def mismatch_count(left, right):

    return int(
        (
            ~equal_series(
                left,
                right,
            )
        ).sum()
    )


# ============================================================
# REFERENCES
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH,
    columns=[
        "source_report_id",
        "organization_id",
        "organization_name_current",
        "party_name_current",
    ],
)


organization_reference = pd.read_parquet(
    ORG_REFERENCE_PATH,
    columns=[
        "organization_id",
        "organization_code",
        "organization_level",
        "organization_name_current",
        "party_name_current",
    ],
)


if (
    report_context[
        "source_report_id"
    ].duplicated().any()
):

    raise RuntimeError(
        "report_context source_report_id is not unique."
    )


if (
    organization_reference[
        "organization_id"
    ].duplicated().any()
):

    raise RuntimeError(
        "organization_reference organization_id is not unique."
    )


# ============================================================
# 1. EXACT ANALYTICAL TYPE RULES
#
# Rule:
#
# incoming:
#   payer is the counterparty
#   same-party payer -> internal_party_transfer
#
# outgoing:
#   receiver is the counterparty
#   same-party receiver -> internal_party_transfer
#
# The non-counterparty side keeps its normalized type.
# ============================================================

type_rows = []


for section in PAYMENT_SECTIONS:

    df = pd.read_parquet(
        ENRICHED_DIR
        / f"{section}.parquet",
        columns=[
            "payer_type_normalized",
            "payer_type_analytical",
            "receiver_type_normalized",
            "receiver_type_analytical",
            "payer_same_party_code_match",
            "receiver_same_party_code_match",
        ],
    )


    expected_payer = (
        df[
            "payer_type_normalized"
        ]
        .astype("string")
        .copy()
    )


    expected_receiver = (
        df[
            "receiver_type_normalized"
        ]
        .astype("string")
        .copy()
    )


    if section in INCOMING_SECTIONS:

        payer_internal = (
            df[
                "payer_same_party_code_match"
            ]
            .fillna(False)
            .astype(bool)
        )


        expected_payer.loc[
            payer_internal
        ] = (
            "internal_party_transfer"
        )


    if section in OUTGOING_SECTIONS:

        receiver_internal = (
            df[
                "receiver_same_party_code_match"
            ]
            .fillna(False)
            .astype(bool)
        )


        expected_receiver.loc[
            receiver_internal
        ] = (
            "internal_party_transfer"
        )


    payer_mismatches = mismatch_count(
        df[
            "payer_type_analytical"
        ],
        expected_payer,
    )


    receiver_mismatches = mismatch_count(
        df[
            "receiver_type_analytical"
        ],
        expected_receiver,
    )


    type_rows.append(
        {
            "section":
                section,

            "rows":
                len(df),

            "payer_type_mismatches":
                payer_mismatches,

            "receiver_type_mismatches":
                receiver_mismatches,

            "total_mismatches":
                (
                    payer_mismatches
                    +
                    receiver_mismatches
                ),
        }
    )


type_df = pd.DataFrame(
    type_rows
)


print()
print("=" * 110)
print("ANALYTICAL COUNTERPARTY TYPE RULE PARITY")
print("=" * 110)

display(
    type_df
)


print(
    "Total type-rule mismatches:",
    f"{int(type_df['total_mismatches'].sum()):,}"
)


# ============================================================
# 2. ORGANIZATION NAME DIAGNOSTIC
#
# Compare three candidates:
#
# A. current enriched payment name
# B. report_context name
# C. organization_reference name
#
# Also compare each with short party_name_current.
# ============================================================

name_frames = []


for section in PAYMENT_SECTIONS:

    payments = pd.read_parquet(
        ENRICHED_DIR
        / f"{section}.parquet",
        columns=[
            "source_row_id",
            "source_report_id",
            "organization_id",
            "organization_level",
            "organization_code",
            "organization_name_current",
            "party_name_current",
        ],
    )


    check = (
        payments
        .rename(
            columns={
                "organization_name_current":
                    "payment_organization_name",

                "party_name_current":
                    "payment_party_name",
            }
        )
        .merge(
            report_context.rename(
                columns={
                    "organization_id":
                        "context_organization_id",

                    "organization_name_current":
                        "context_organization_name",

                    "party_name_current":
                        "context_party_name",
                }
            ),
            on="source_report_id",
            how="left",
            validate="many_to_one",
            sort=False,
        )
        .merge(
            organization_reference.rename(
                columns={
                    "organization_code":
                        "reference_organization_code",

                    "organization_level":
                        "reference_organization_level",

                    "organization_name_current":
                        "reference_organization_name",

                    "party_name_current":
                        "reference_party_name",
                }
            ),
            on="organization_id",
            how="left",
            validate="many_to_one",
            sort=False,
        )
    )


    check[
        "section"
    ] = section


    check[
        "payment_equals_context"
    ] = equal_series(
        check[
            "payment_organization_name"
        ],
        check[
            "context_organization_name"
        ],
    )


    check[
        "payment_equals_reference"
    ] = equal_series(
        check[
            "payment_organization_name"
        ],
        check[
            "reference_organization_name"
        ],
    )


    check[
        "context_equals_reference"
    ] = equal_series(
        check[
            "context_organization_name"
        ],
        check[
            "reference_organization_name"
        ],
    )


    check[
        "payment_org_equals_party_name"
    ] = equal_series(
        check[
            "payment_organization_name"
        ],
        check[
            "payment_party_name"
        ],
    )


    check[
        "context_org_equals_party_name"
    ] = equal_series(
        check[
            "context_organization_name"
        ],
        check[
            "context_party_name"
        ],
    )


    name_frames.append(
        check
    )


names = pd.concat(
    name_frames,
    ignore_index=True,
)


# ============================================================
# 3. ONLY THE 84k NAME MISMATCHES
# ============================================================

mismatches = names[
    ~names[
        "payment_equals_context"
    ]
].copy()


print()
print("=" * 110)
print("ORGANIZATION NAME MISMATCHES BY LEVEL")
print("=" * 110)


by_level = (
    mismatches
    .groupby(
        "organization_level",
        dropna=False,
        as_index=False,
    )
    .agg(
        rows=(
            "source_row_id",
            "size",
        ),

        organizations=(
            "organization_id",
            "nunique",
        ),

        reports=(
            "source_report_id",
            "nunique",
        ),

        payment_equals_reference=(
            "payment_equals_reference",
            "sum",
        ),

        context_equals_reference=(
            "context_equals_reference",
            "sum",
        ),

        payment_org_equals_party_name=(
            "payment_org_equals_party_name",
            "sum",
        ),

        context_org_equals_party_name=(
            "context_org_equals_party_name",
            "sum",
        ),
    )
)


display(
    by_level
)


# ============================================================
# 4. OVERALL NAME-SOURCE PATTERN
# ============================================================

pattern_summary = pd.DataFrame(
    [
        {
            "metric":
                "name mismatch rows",

            "rows":
                len(mismatches),
        },
        {
            "metric":
                "payment name == organization_reference",

            "rows":
                int(
                    mismatches[
                        "payment_equals_reference"
                    ].sum()
                ),
        },
        {
            "metric":
                "context name == organization_reference",

            "rows":
                int(
                    mismatches[
                        "context_equals_reference"
                    ].sum()
                ),
        },
        {
            "metric":
                "payment organization name == short party name",

            "rows":
                int(
                    mismatches[
                        "payment_org_equals_party_name"
                    ].sum()
                ),
        },
        {
            "metric":
                "context organization name == short party name",

            "rows":
                int(
                    mismatches[
                        "context_org_equals_party_name"
                    ].sum()
                ),
        },
    ]
)


print()
print("=" * 110)
print("NAME-SOURCE PATTERN")
print("=" * 110)

display(
    pattern_summary
)


# ============================================================
# 5. UNIQUE ORGANIZATION EXAMPLES
# ============================================================

examples = (
    mismatches[
        [
            "organization_id",
            "organization_code",
            "organization_level",
            "payment_organization_name",
            "context_organization_name",
            "reference_organization_name",
            "payment_party_name",
        ]
    ]
    .drop_duplicates()
    .head(50)
)


print()
print("=" * 110)
print("UNIQUE ORGANIZATION NAME EXAMPLES")
print("=" * 110)

display(
    examples
)


# ============================================================
# 6. TRANSACTION COUNTS BY UNIQUE NAME PAIR
# ============================================================

name_pairs = (
    mismatches
    .groupby(
        [
            "organization_level",
            "payment_organization_name",
            "context_organization_name",
            "reference_organization_name",
            "payment_party_name",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .sort_values(
        "size",
        ascending=False,
    )
)


print()
print("=" * 110)
print("MOST COMMON NAME PAIRS")
print("=" * 110)

display(
    name_pairs.head(
        50
    )
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 110)
print("DONE")
print("=" * 110)

print(
    "Total organization-name mismatches:",
    f"{len(mismatches):,}"
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No parquet files were modified."
)


ANALYTICAL COUNTERPARTY TYPE RULE PARITY


,section,rows,payer_type_mismatches,receiver_type_mismatches,total_mismatches
0,monetary_contributions,27234,0,0,0
1,other_contributions,6168,0,0,0
2,state_funding,96,0,0,0
3,other_incomes,19007,0,0,0
4,budget_expenses,29482,0,0,0
5,outgoing_expenses,319901,0,0,0
6,return_expenses,137,0,0,0
7,transfer_expenses,3,0,0,0


Total type-rule mismatches: 0

ORGANIZATION NAME MISMATCHES BY LEVEL


,organization_level,rows,organizations,reports,payment_equals_reference,context_equals_reference,payment_org_equals_party_name,context_org_equals_party_name
0,central,84511,221,1450,0,84511,84511,0



NAME-SOURCE PATTERN


...


UNIQUE ORGANIZATION NAME EXAMPLES


,organization_id,organization_code,organization_level,payment_organization_name,context_organization_name,reference_organization_name,payment_party_name
50,8d8131a2-d14a-4a06-9f23-0eb2943e0419,42622242,central,РІДНЕ ЗАКАРПАТТЯ,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ ЗАКАРПАТТЯ»,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ ЗАКАРПАТТЯ»,РІДНЕ ЗАКАРПАТТЯ
51,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,33547558,central,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА»,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА»,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА
77,cefb6684-da05-4a85-a02c-de65d25d8707,38010759,central,БДЖОЛА,ПОЛІТИЧНА ПАРТІЯ «БДЖОЛА»,ПОЛІТИЧНА ПАРТІЯ «БДЖОЛА»,БДЖОЛА
79,e923d163-59e0-44fe-8891-8f62c7bdb3c7,43958579,central,ПОРЯДОК ДЛЯ УКРАЇНИ,ПОЛІТИЧНА ПАРТІЯ «ПОРЯДОК ДЛЯ УКРАЇНИ»,ПОЛІТИЧНА ПАРТІЯ «ПОРЯДОК ДЛЯ УКРАЇНИ»,ПОРЯДОК ДЛЯ УКРАЇНИ
80,dbac3b1c-adb3-4ae5-92cb-cc307ff3ab3d,00042458,central,КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛІСТІВ,ПОЛІТИЧНА ПАРТІЯ «КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛІСТІВ»,ПОЛІТИЧНА ПАРТІЯ «КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛІСТІВ»,КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛІСТІВ
130,79cb7831-508b-466a-8db0-db9c5f4080b2,40152983,central,ПАРТІЯ ВОЛОДИМИРА БУРЯКА ЄДНАННЯ,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ВОЛОДИМИРА БУРЯКА «ЄДНАННЯ»,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ВОЛОДИМИРА БУРЯКА «ЄДНАННЯ»,ПАРТІЯ ВОЛОДИМИРА БУРЯКА ЄДНАННЯ
148,7ed50708-02f2-40b7-979a-d4ba7ce7ca1a,40194253,central,ВО ПЛАТФОРМА ГРОМАД,ПОЛІТИЧНА ПАРТІЯ «ВО «ПЛАТФОРМА ГРОМАД»,ПОЛІТИЧНА ПАРТІЯ «ВО «ПЛАТФОРМА ГРОМАД»,ВО ПЛАТФОРМА ГРОМАД
149,2971b6fb-8db3-4800-8d53-252c8e2984ff,21707382,central,ГРОМАДСЬКА СИЛА,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДСЬКА СИЛА»,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДСЬКА СИЛА»,ГРОМАДСЬКА СИЛА
167,4dc49830-ea98-44cf-b4c7-e226b6fe4f18,39169428,central,ЛІБЕРТАРІАНСЬКА ПАРТІЯ 5.10,ПОЛІТИЧНА ПАРТІЯ «ЛІБЕРТАРІАНСЬКА ПАРТІЯ 5.10»,ПОЛІТИЧНА ПАРТІЯ «ЛІБЕРТАРІАНСЬКА ПАРТІЯ 5.10»,ЛІБЕРТАРІАНСЬКА ПАРТІЯ 5.10
173,82c1b72c-f67e-4eaf-ac09-b025ee67adcc,39550414,central,ПРОПОЗИЦІЯ,ПОЛІТИЧНА ПАРТІЯ «ПРОПОЗИЦІЯ»,ПОЛІТИЧНА ПАРТІЯ «ПРОПОЗИЦІЯ»,ПРОПОЗИЦІЯ



MOST COMMON NAME PAIRS


,organization_level,payment_organization_name,context_organization_name,reference_organization_name,payment_party_name,size
105,central,НАРОДОВЛАДДЯ,ПОЛІТИЧНА ПАРТІЯ «НАРОДОВЛАДДЯ»,ПОЛІТИЧНА ПАРТІЯ «НАРОДОВЛАДДЯ»,НАРОДОВЛАДДЯ,13998
2,central,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,13674
186,central,СЛУГА НАРОДУ,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,СЛУГА НАРОДУ,13161
21,central,ВО БАТЬКІВЩИНА,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»,ВО БАТЬКІВЩИНА,9700
39,central,ГОЛОС,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,ГОЛОС,3600
183,central,СИЛА ЛЮДЕЙ,ПОЛІТИЧНА ПАРТІЯ «СИЛА ЛЮДЕЙ»,ПОЛІТИЧНА ПАРТІЯ «СИЛА ЛЮДЕЙ»,СИЛА ЛЮДЕЙ,3071
9,central,БДЖОЛА,ПОЛІТИЧНА ПАРТІЯ «БДЖОЛА»,ПОЛІТИЧНА ПАРТІЯ «БДЖОЛА»,БДЖОЛА,3024
54,central,ДЕМОКРАТИЧНА СОКИРА,ПОЛІТИЧНА ПАРТІЯ «ДЕМОКРАТИЧНА СОКИРА»,ПОЛІТИЧНА ПАРТІЯ «ДЕМОКРАТИЧНА СОКИРА»,ДЕМОКРАТИЧНА СОКИРА,2257
200,central,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА»,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА»,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА,1877
81,central,КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА ПАРТІЯ УКРАЇНИ СИЛА І ЧЕСТЬ,ПОЛІТИЧНА ПАРТІЯ «КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА ПАРТІЯ УКРАЇНИ СИЛА І ЧЕСТЬ»,ПОЛІТИЧНА ПАРТІЯ «КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА ПАРТІЯ УКРАЇНИ СИЛА І ЧЕСТЬ»,КОНСЕРВАТИВНО-ДЕМОКРАТИЧНА ПАРТІЯ УКРАЇНИ СИЛА І ЧЕСТЬ,1466



DONE
Total organization-name mismatches: 84,511
RAW files were NOT read.
API was NOT called.
No parquet files were modified.


In [134]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

ENRICHMENT_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "enrichment"
)

TESTS_DIR = (
    ROOT
    / "tests"
)

NORMALIZED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "payments"
)

CURRENT_ENRICHED_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "payments"
)

REFERENCE_DIR = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

REPORT_CONTEXT_PATH = (
    REFERENCE_DIR
    / "report_context.parquet"
)

ORG_REFERENCE_PATH = (
    REFERENCE_DIR
    / "organization_reference.parquet"
)

REPORT_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "report_account_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_DIR
    / "state_funding_account_reference.parquet"
)

VALIDATION_DIR = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "payment_from_normalized_v0_1"
)


# ============================================================
# 1. CREATE BASE PAYMENT ENRICHMENT MODULE
# ============================================================

BASE_MODULE = r'''
from __future__ import annotations

from typing import Optional

import pandas as pd

from politdata.enrichment.payment_resolution import (
    INCOMING_SECTIONS,
    OUTGOING_SECTIONS,
    PAYMENT_SECTIONS,
    build_unique_organization_code_map,
)


REPORT_CONTEXT_COLUMNS = (
    "organization_code",
    "organization_level",
    "organization_name_current",
    "party_code",
    "party_name_current",
    "region",
    "region_resolution_method",
    "region_resolution_source",
    "region_source",
    "region_source_address_type",
)


BASE_ENRICHMENT_COLUMNS = (
    "analysis_selected",
    "official_selected",

    "payer_same_party_code_match",
    "receiver_same_party_code_match",

    "payer_type_analytical",
    "receiver_type_analytical",

    "internal_transfer",
    "internal_transfer_rule",

    "party_account_type_source",
)


INTERNAL_TRANSFER_RULE = (
    "same_root_party_organization_code"
)


def _clean_code_series(
    series: pd.Series,
) -> pd.Series:

    return (
        series
        .astype("string")
        .str.strip()
    )


def _same_root_code_matches(
    payments: pd.DataFrame,
    *,
    code_column: str,
    organization_reference: pd.DataFrame,
) -> pd.Series:
    """
    Strict same-root EDRPOU match.

    No fuzzy names.
    No IBAN inference.
    No code padding.
    """

    org_map = (
        build_unique_organization_code_map(
            organization_reference
        )
    )


    valid_keys = set(
        zip(
            org_map[
                "root_party_id"
            ].astype("string"),

            org_map[
                "organization_code"
            ].astype("string"),
        )
    )


    roots = (
        payments[
            "root_party_id"
        ]
        .astype("string")
    )


    codes = (
        _clean_code_series(
            payments[
                code_column
            ]
        )
    )


    values = []


    for root, code in zip(
        roots,
        codes,
    ):

        if (
            pd.isna(root)
            or
            pd.isna(code)
        ):

            values.append(
                False
            )

        else:

            values.append(
                (
                    root,
                    code,
                )
                in
                valid_keys
            )


    return pd.Series(
        values,
        index=payments.index,
        dtype="bool",
    )


def _prepare_report_context(
    report_context: pd.DataFrame,
) -> pd.DataFrame:

    required = {
        "source_report_id",
        *REPORT_CONTEXT_COLUMNS,
    }


    missing = (
        required
        -
        set(
            report_context.columns
        )
    )


    if missing:

        raise KeyError(
            "report_context missing columns: "
            f"{sorted(missing)}"
        )


    if (
        report_context[
            "source_report_id"
        ]
        .duplicated()
        .any()
    ):

        raise ValueError(
            "report_context source_report_id "
            "must be unique."
        )


    return (
        report_context[
            [
                "source_report_id",
                *REPORT_CONTEXT_COLUMNS,
            ]
        ]
        .copy()
    )


def _prepare_report_account_reference(
    report_account_reference: pd.DataFrame,
) -> pd.DataFrame:

    required = {
        "source_report_id",
        "organization_id",
        "party_account_iban",
        "party_account_type_source",
    }


    missing = (
        required
        -
        set(
            report_account_reference.columns
        )
    )


    if missing:

        raise KeyError(
            "report_account_reference missing columns: "
            f"{sorted(missing)}"
        )


    ref = (
        report_account_reference[
            [
                "source_report_id",
                "organization_id",
                "party_account_iban",
                "party_account_type_source",
            ]
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Ensure duplicate keys, if any, do not contain
    # conflicting declared account types.
    # --------------------------------------------------------

    conflicts = (
        ref
        .groupby(
            [
                "source_report_id",
                "organization_id",
                "party_account_iban",
            ],
            dropna=False,
        )[
            "party_account_type_source"
        ]
        .nunique(
            dropna=True
        )
    )


    if (
        conflicts
        >
        1
    ).any():

        raise ValueError(
            "report_account_reference contains "
            "conflicting account types for the same key."
        )


    return (
        ref
        .drop_duplicates(
            subset=[
                "source_report_id",
                "organization_id",
                "party_account_iban",
            ]
        )
        .reset_index(
            drop=True
        )
    )


def add_base_payment_enrichment(
    payments: pd.DataFrame,
    *,
    section: str,
    report_context: pd.DataFrame,
    organization_reference: pd.DataFrame,
    report_account_reference: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add enrichment that is logically prior to payment
    resolution/business classification.

    Input:
        normalized payment rows.

    Adds:
        current organization/party context,
        region,
        official/analysis selection flags,
        strict same-party EDRPOU matches,
        counterparty analytical types,
        internal-transfer flag/rule,
        report-relative declared account type,
        source state-funding form.

    Important naming rule:
        organization_name_current
            = full organization name

        party_name_current
            = short/unified party name

    Existing stale derived values are discarded.
    """

    if section not in PAYMENT_SECTIONS:

        raise ValueError(
            f"Unknown payment section: {section}"
        )


    required = {
        "source_report_id",
        "official_selected_report_id",
        "organization_id",
        "root_party_id",
        "payer_code_normalized",
        "receiver_code_normalized",
        "payer_type_normalized",
        "receiver_type_normalized",
        "receiver_account_iban_canonical",
    }


    missing = (
        required
        -
        set(
            payments.columns
        )
    )


    if missing:

        raise KeyError(
            f"{section} missing normalized columns: "
            f"{sorted(missing)}"
        )


    df = (
        payments
        .reset_index(
            drop=True
        )
        .copy()
    )


    # --------------------------------------------------------
    # Remove any stale copies if function is accidentally
    # called on already-enriched rows.
    # --------------------------------------------------------

    stale_columns = [
        column
        for column
        in (
            *REPORT_CONTEXT_COLUMNS,
            *BASE_ENRICHMENT_COLUMNS,
            "state_funding_form",
        )
        if column in df.columns
    ]


    if stale_columns:

        df = df.drop(
            columns=
                stale_columns
        )


    # --------------------------------------------------------
    # CURRENT REPORT / ORGANIZATION CONTEXT
    # --------------------------------------------------------

    context = (
        _prepare_report_context(
            report_context
        )
    )


    before_rows = len(
        df
    )


    df = df.merge(
        context,
        on="source_report_id",
        how="left",
        validate="many_to_one",
        sort=False,
    )


    if len(df) != before_rows:

        raise RuntimeError(
            "report_context join changed row count."
        )


    if (
        df[
            "organization_level"
        ]
        .isna()
        .any()
    ):

        raise RuntimeError(
            "Some payment rows did not resolve "
            "report context."
        )


    # --------------------------------------------------------
    # REPORT SELECTION FLAGS
    # --------------------------------------------------------

    df[
        "analysis_selected"
    ] = True


    df[
        "official_selected"
    ] = (
        df[
            "source_report_id"
        ].astype("string")
        ==
        df[
            "official_selected_report_id"
        ].astype("string")
    )


    # --------------------------------------------------------
    # STRICT SAME-ROOT EDRPOU MATCHES
    # --------------------------------------------------------

    df[
        "payer_same_party_code_match"
    ] = (
        _same_root_code_matches(
            df,
            code_column=
                "payer_code_normalized",
            organization_reference=
                organization_reference,
        )
    )


    df[
        "receiver_same_party_code_match"
    ] = (
        _same_root_code_matches(
            df,
            code_column=
                "receiver_code_normalized",
            organization_reference=
                organization_reference,
        )
    )


    # --------------------------------------------------------
    # COUNTERPARTY ANALYTICAL TYPES
    # --------------------------------------------------------

    df[
        "payer_type_analytical"
    ] = (
        df[
            "payer_type_normalized"
        ]
        .astype("string")
    )


    df[
        "receiver_type_analytical"
    ] = (
        df[
            "receiver_type_normalized"
        ]
        .astype("string")
    )


    if section in INCOMING_SECTIONS:

        internal_mask = (
            df[
                "payer_same_party_code_match"
            ]
        )


        df.loc[
            internal_mask,
            "payer_type_analytical",
        ] = (
            "internal_party_transfer"
        )


    else:

        internal_mask = (
            df[
                "receiver_same_party_code_match"
            ]
        )


        df.loc[
            internal_mask,
            "receiver_type_analytical",
        ] = (
            "internal_party_transfer"
        )


    # --------------------------------------------------------
    # INTERNAL TRANSFER FLAG / RULE
    # --------------------------------------------------------

    df[
        "internal_transfer"
    ] = (
        internal_mask
        .fillna(False)
        .astype(bool)
    )


    df[
        "internal_transfer_rule"
    ] = pd.Series(
        pd.NA,
        index=df.index,
        dtype="string",
    )


    df.loc[
        df[
            "internal_transfer"
        ],
        "internal_transfer_rule",
    ] = (
        INTERNAL_TRANSFER_RULE
    )


    # --------------------------------------------------------
    # REPORT-RELATIVE ACCOUNT TYPE SOURCE
    #
    # Auxiliary metadata only.
    # It is NOT authoritative evidence of funding origin.
    # --------------------------------------------------------

    account_ref = (
        _prepare_report_account_reference(
            report_account_reference
        )
    )


    account_join = (
        account_ref
        .rename(
            columns={
                "party_account_iban":
                    "receiver_account_iban_canonical",
            }
        )
    )


    before_rows = len(
        df
    )


    df = df.merge(
        account_join,
        on=[
            "source_report_id",
            "organization_id",
            "receiver_account_iban_canonical",
        ],
        how="left",
        validate="many_to_one",
        sort=False,
    )


    if len(df) != before_rows:

        raise RuntimeError(
            "report account join changed row count."
        )


    # --------------------------------------------------------
    # SOURCE STATE-FUNDING FORM
    #
    # Only the source state_funding section contains this
    # human-readable field.
    # --------------------------------------------------------

    if section == "state_funding":

        if (
            "payment_type_detail_source"
            not in df.columns
        ):

            raise KeyError(
                "state_funding missing "
                "payment_type_detail_source"
            )


        df[
            "state_funding_form"
        ] = (
            df[
                "payment_type_detail_source"
            ]
        )


    return df
'''


BASE_PATH = (
    ENRICHMENT_DIR
    / "payment_base.py"
)


BASE_PATH.write_text(
    textwrap.dedent(
        BASE_MODULE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    BASE_PATH
)


# ============================================================
# 2. TEST BASE ENRICHMENT
# ============================================================

TEST_CODE = r'''
import pandas as pd

from politdata.enrichment.payment_base import (
    add_base_payment_enrichment,
)


def _references():

    context = pd.DataFrame(
        [
            {
                "source_report_id": "r1",

                "organization_code": "111",
                "organization_level": "central",

                "organization_name_current":
                    "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»",

                "party_code": "111",

                "party_name_current":
                    "ТЕСТ",

                "region": "Україна",
                "region_resolution_method":
                    "central_national",
                "region_resolution_source":
                    "organization_level",
                "region_source":
                    None,
                "region_source_address_type":
                    None,
            }
        ]
    )


    organizations = pd.DataFrame(
        [
            {
                "root_party_id": "p",
                "organization_id": "central",
                "organization_code": "111",

                "organization_name_current":
                    "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»",

                "organization_level": "central",
            },
            {
                "root_party_id": "p",
                "organization_id": "office",
                "organization_code": "222",

                "organization_name_current":
                    "ТЕСТОВИЙ ОСЕРЕДОК",

                "organization_level": "office",
            },
        ]
    )


    accounts = pd.DataFrame(
        [
            {
                "source_report_id": "r1",
                "organization_id": "central",

                "party_account_iban":
                    "UA000000000000000000000000001",

                "party_account_type_source":
                    "Поточний рахунок",
            }
        ]
    )


    return (
        context,
        organizations,
        accounts,
    )


def test_base_keeps_full_organization_name_and_short_party_name():

    context, organizations, accounts = (
        _references()
    )


    payments = pd.DataFrame(
        [
            {
                "source_report_id": "r1",
                "official_selected_report_id": "r1",

                "organization_id": "central",
                "root_party_id": "p",

                "payer_code_normalized": None,
                "receiver_code_normalized": "222",

                "payer_type_normalized": None,
                "receiver_type_normalized":
                    "Юридична особа",

                "receiver_account_iban_canonical":
                    "UA000000000000000000000000001",
            }
        ]
    )


    result = add_base_payment_enrichment(
        payments,
        section="outgoing_expenses",
        report_context=context,
        organization_reference=
            organizations,
        report_account_reference=
            accounts,
    )


    row = result.iloc[0]


    assert (
        row[
            "organization_name_current"
        ]
        ==
        "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»"
    )

    assert (
        row[
            "party_name_current"
        ]
        ==
        "ТЕСТ"
    )


def test_outgoing_internal_counterparty_type():

    context, organizations, accounts = (
        _references()
    )


    payments = pd.DataFrame(
        [
            {
                "source_report_id": "r1",
                "official_selected_report_id": "r1",

                "organization_id": "central",
                "root_party_id": "p",

                "payer_code_normalized": None,
                "receiver_code_normalized": "222",

                "payer_type_normalized": None,
                "receiver_type_normalized":
                    "Юридична особа",

                "receiver_account_iban_canonical":
                    "UA000000000000000000000000001",
            }
        ]
    )


    result = add_base_payment_enrichment(
        payments,
        section="outgoing_expenses",
        report_context=context,
        organization_reference=
            organizations,
        report_account_reference=
            accounts,
    )


    row = result.iloc[0]


    assert (
        bool(
            row[
                "receiver_same_party_code_match"
            ]
        )
        is True
    )

    assert (
        row[
            "receiver_type_analytical"
        ]
        ==
        "internal_party_transfer"
    )

    assert (
        bool(
            row[
                "internal_transfer"
            ]
        )
        is True
    )

    assert (
        row[
            "internal_transfer_rule"
        ]
        ==
        "same_root_party_organization_code"
    )

    assert (
        row[
            "party_account_type_source"
        ]
        ==
        "Поточний рахунок"
    )


def test_analysis_and_official_selection_flags():

    context, organizations, accounts = (
        _references()
    )


    payments = pd.DataFrame(
        [
            {
                "source_report_id": "r1",

                "official_selected_report_id":
                    "official-other",

                "organization_id": "central",
                "root_party_id": "p",

                "payer_code_normalized": None,
                "receiver_code_normalized": None,

                "payer_type_normalized": None,
                "receiver_type_normalized": None,

                "receiver_account_iban_canonical":
                    None,
            }
        ]
    )


    result = add_base_payment_enrichment(
        payments,
        section="other_incomes",
        report_context=context,
        organization_reference=
            organizations,
        report_account_reference=
            accounts,
    )


    row = result.iloc[0]


    assert (
        bool(
            row[
                "analysis_selected"
            ]
        )
        is True
    )

    assert (
        bool(
            row[
                "official_selected"
            ]
        )
        is False
    )
'''


TEST_PATH = (
    TESTS_DIR
    / "test_payment_base.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 3. ADD NORMALIZED -> ENRICHED ORCHESTRATION
# ============================================================

BATCH_PATH = (
    ENRICHMENT_DIR
    / "payment_batch.py"
)


batch_text = BATCH_PATH.read_text(
    encoding="utf-8"
)


MARKER = (
    "# NORMALIZED PAYMENT PIPELINE V0.1"
)


PIPELINE_CODE = r'''


# ============================================================
# NORMALIZED PAYMENT PIPELINE V0.1
# ============================================================

def rebuild_payment_from_normalized_frame(
    normalized: pd.DataFrame,
    *,
    section: str,
    report_context: pd.DataFrame,
    organization_reference: pd.DataFrame,
    report_account_reference: pd.DataFrame,
    state_account_reference: pd.DataFrame,
) -> pd.DataFrame:
    """
    Complete normalized -> enriched transformation
    for one payment section.
    """

    from politdata.enrichment.payment_base import (
        add_base_payment_enrichment,
    )


    base = add_base_payment_enrichment(
        normalized,
        section=section,
        report_context=
            report_context,
        organization_reference=
            organization_reference,
        report_account_reference=
            report_account_reference,
    )


    return (
        rebuild_payment_enrichment_frame(
            base,
            section=section,
            organization_reference=
                organization_reference,
            state_account_reference=
                state_account_reference,
        )
    )


def enrich_normalized_payment_directory(
    input_dir,
    output_dir,
    *,
    report_context,
    organization_reference,
    report_account_reference,
    state_account_reference,
    overwrite: bool = False,
) -> pd.DataFrame:
    """
    Build all enriched payment parquet files directly
    from normalized_v0_1/payments.

    No RAW access.
    No API access.
    """

    input_dir = Path(
        input_dir
    )

    output_dir = Path(
        output_dir
    )


    def load_if_path(value):

        if isinstance(
            value,
            (
                str,
                Path,
            ),
        ):

            return pd.read_parquet(
                value
            )

        return value


    report_context = load_if_path(
        report_context
    )

    organization_reference = load_if_path(
        organization_reference
    )

    report_account_reference = load_if_path(
        report_account_reference
    )

    state_account_reference = load_if_path(
        state_account_reference
    )


    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )


    rows = []


    for section in PAYMENT_SECTIONS:

        input_path = (
            input_dir
            / f"{section}.parquet"
        )

        output_path = (
            output_dir
            / f"{section}.parquet"
        )


        if not input_path.exists():

            raise FileNotFoundError(
                input_path
            )


        if (
            output_path.exists()
            and
            not overwrite
        ):

            raise FileExistsError(
                output_path
            )


        normalized = pd.read_parquet(
            input_path
        )


        enriched = (
            rebuild_payment_from_normalized_frame(
                normalized,
                section=section,

                report_context=
                    report_context,

                organization_reference=
                    organization_reference,

                report_account_reference=
                    report_account_reference,

                state_account_reference=
                    state_account_reference,
            )
        )


        if len(enriched) != len(normalized):

            raise RuntimeError(
                f"{section}: row count changed."
            )


        temp_path = (
            output_path
            .with_suffix(
                ".tmp.parquet"
            )
        )


        enriched.to_parquet(
            temp_path,
            index=False,
        )


        temp_path.replace(
            output_path
        )


        rows.append(
            {
                "section":
                    section,

                "rows":
                    len(enriched),

                "columns":
                    len(
                        enriched.columns
                    ),

                "output":
                    str(
                        output_path
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )
'''


if MARKER not in batch_text:

    BATCH_PATH.write_text(
        (
            batch_text.rstrip()
            +
            "\n"
            +
            textwrap.dedent(
                PIPELINE_CODE
            )
            +
            "\n"
        ),
        encoding="utf-8",
    )

    print(
        "Normalized payment pipeline added:",
        BATCH_PATH
    )

else:

    print(
        "Normalized payment pipeline already present."
    )


# ============================================================
# 4. RUN ALL TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:
    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Full normalized -> enriched rebuild "
        "was not started."
    )


# ============================================================
# 5. FRESH IMPORT
# ============================================================

import politdata.enrichment.payment_base as payment_base
import politdata.enrichment.payment_batch as payment_batch


importlib.reload(
    payment_base
)

importlib.reload(
    payment_batch
)


# ============================================================
# 6. CLEAN VALIDATION DIRECTORY
# ============================================================

if VALIDATION_DIR.exists():

    shutil.rmtree(
        VALIDATION_DIR
    )


VALIDATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 7. FULL NORMALIZED -> ENRICHED REBUILD
# ============================================================

print()
print("=" * 100)
print("NORMALIZED -> ENRICHED PAYMENT REBUILD")
print("=" * 100)


rebuild_summary = (
    payment_batch
    .enrich_normalized_payment_directory(
        NORMALIZED_DIR,
        VALIDATION_DIR,

        report_context=
            REPORT_CONTEXT_PATH,

        organization_reference=
            ORG_REFERENCE_PATH,

        report_account_reference=
            REPORT_ACCOUNT_REFERENCE_PATH,

        state_account_reference=
            STATE_ACCOUNT_REFERENCE_PATH,

        overwrite=True,
    )
)


display(
    rebuild_summary
)


# ============================================================
# 8. KEYED COMPARISON WITH CURRENT PRODUCTION
#
# Current production contains one known old defect:
#
# central organization_name_current
#     = short party_name_current
#
# New rebuild intentionally corrects this.
# ============================================================

PAYMENT_SECTIONS = [
    "monetary_contributions",
    "other_contributions",
    "state_funding",
    "other_incomes",
    "budget_expenses",
    "outgoing_expenses",
    "return_expenses",
    "transfer_expenses",
]


def series_mismatch_mask(
    left,
    right,
):

    left = (
        left
        .astype("string")
        .fillna("<NULL>")
    )

    right = (
        right
        .astype("string")
        .fillna("<NULL>")
    )

    return (
        left
        !=
        right
    )


comparison_rows = []

unexpected_samples = []

name_fix_rows = []


for section in PAYMENT_SECTIONS:

    current = pd.read_parquet(
        CURRENT_ENRICHED_DIR
        / f"{section}.parquet"
    )

    rebuilt = pd.read_parquet(
        VALIDATION_DIR
        / f"{section}.parquet"
    )


    if len(current) != len(rebuilt):

        raise RuntimeError(
            f"{section}: row count mismatch."
        )


    if (
        not
        current[
            "source_row_id"
        ].astype("string").is_unique
    ):

        raise RuntimeError(
            f"{section}: current source_row_id "
            "is not unique."
        )


    if (
        not
        rebuilt[
            "source_row_id"
        ].astype("string").is_unique
    ):

        raise RuntimeError(
            f"{section}: rebuilt source_row_id "
            "is not unique."
        )


    current = (
        current
        .assign(
            _key=
                current[
                    "source_row_id"
                ].astype("string")
        )
        .set_index(
            "_key"
        )
        .sort_index()
    )


    rebuilt = (
        rebuilt
        .assign(
            _key=
                rebuilt[
                    "source_row_id"
                ].astype("string")
        )
        .set_index(
            "_key"
        )
        .sort_index()
    )


    if not current.index.equals(
        rebuilt.index
    ):

        raise RuntimeError(
            f"{section}: source_row_id sets differ."
        )


    current_columns = set(
        current.columns
    )

    rebuilt_columns = set(
        rebuilt.columns
    )


    if current_columns != rebuilt_columns:

        raise RuntimeError(
            f"{section}: schema set differs. "
            f"Missing={sorted(current_columns - rebuilt_columns)}; "
            f"Extra={sorted(rebuilt_columns - current_columns)}"
        )


    section_unexpected = 0
    section_name_fixes = 0
    mismatching_columns = []


    for column in sorted(
        current_columns
    ):

        mask = series_mismatch_mask(
            current[
                column
            ],
            rebuilt[
                column
            ],
        )


        count = int(
            mask.sum()
        )


        if count == 0:
            continue


        if (
            column
            ==
            "organization_name_current"
        ):

            # -----------------------------------------------
            # Expected controlled correction.
            # -----------------------------------------------

            fix_current = (
                current.loc[
                    mask
                ]
            )

            fix_rebuilt = (
                rebuilt.loc[
                    mask
                ]
            )


            # All known old defects were central parties.
            if not (
                fix_current[
                    "organization_level"
                ]
                .astype("string")
                ==
                "central"
            ).all():

                raise RuntimeError(
                    f"{section}: organization-name "
                    "difference outside central organizations."
                )


            # Existing bad value = short party name.
            if not (
                fix_current[
                    "organization_name_current"
                ]
                .astype("string")
                ==
                fix_current[
                    "party_name_current"
                ]
                .astype("string")
            ).all():

                raise RuntimeError(
                    f"{section}: unexpected old "
                    "organization-name pattern."
                )


            section_name_fixes += (
                count
            )


            name_fix_rows.append(
                {
                    "section":
                        section,

                    "corrected_rows":
                        count,
                }
            )


        else:

            section_unexpected += (
                count
            )

            mismatching_columns.append(
                f"{column} ({count})"
            )


            for idx in (
                mask[
                    mask
                ]
                .index[:5]
            ):

                unexpected_samples.append(
                    {
                        "section":
                            section,

                        "source_row_id":
                            idx,

                        "column":
                            column,

                        "current":
                            str(
                                current.at[
                                    idx,
                                    column
                                ]
                            ),

                        "rebuilt":
                            str(
                                rebuilt.at[
                                    idx,
                                    column
                                ]
                            ),
                    }
                )


    comparison_rows.append(
        {
            "section":
                section,

            "rows":
                len(current),

            "columns":
                len(
                    current_columns
                ),

            "expected_name_corrections":
                section_name_fixes,

            "unexpected_field_mismatches":
                section_unexpected,

            "unexpected_columns":
                " | ".join(
                    mismatching_columns
                ),
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


print()
print("=" * 100)
print("NORMALIZED -> ENRICHED PARITY")
print("=" * 100)

display(
    comparison_df
)


# ============================================================
# 9. FINAL BASELINE
# ============================================================

total_rows = int(
    comparison_df[
        "rows"
    ].sum()
)


total_name_corrections = int(
    comparison_df[
        "expected_name_corrections"
    ].sum()
)


total_unexpected = int(
    comparison_df[
        "unexpected_field_mismatches"
    ].sum()
)


print()
print(
    "Rows rebuilt:",
    f"{total_rows:,}"
)

print(
    "Expected organization-name corrections:",
    f"{total_name_corrections:,}"
)

print(
    "Unexpected field-level mismatches:",
    f"{total_unexpected:,}"
)


if total_rows != 402_028:

    raise RuntimeError(
        f"Expected 402,028 rows, got {total_rows:,}"
    )


if total_name_corrections != 84_511:

    raise RuntimeError(
        "Expected exactly 84,511 known "
        "organization-name corrections, got "
        f"{total_name_corrections:,}"
    )


if total_unexpected:

    print()
    print("=" * 100)
    print("UNEXPECTED MISMATCH SAMPLES")
    print("=" * 100)

    display(
        pd.DataFrame(
            unexpected_samples
        ).head(
            100
        )
    )


    raise RuntimeError(
        "Normalized -> enriched rebuild still has "
        "unexpected differences."
    )


# ============================================================
# 10. EXAMPLES OF CONTROLLED NAME CORRECTION
# ============================================================

examples = []


for section in PAYMENT_SECTIONS:

    current = pd.read_parquet(
        CURRENT_ENRICHED_DIR
        / f"{section}.parquet",
        columns=[
            "source_row_id",
            "organization_id",
            "organization_level",
            "organization_name_current",
            "party_name_current",
        ],
    )


    rebuilt = pd.read_parquet(
        VALIDATION_DIR
        / f"{section}.parquet",
        columns=[
            "source_row_id",
            "organization_id",
            "organization_level",
            "organization_name_current",
            "party_name_current",
        ],
    )


    joined = (
        current
        .merge(
            rebuilt,
            on="source_row_id",
            how="inner",
            suffixes=(
                "_old",
                "_new",
            ),
            validate="one_to_one",
        )
    )


    mask = (
        joined[
            "organization_name_current_old"
        ].astype("string")
        !=
        joined[
            "organization_name_current_new"
        ].astype("string")
    )


    if mask.any():

        sample = (
            joined.loc[
                mask,
                [
                    "organization_id_old",
                    "organization_name_current_old",
                    "organization_name_current_new",
                    "party_name_current_new",
                ],
            ]
            .drop_duplicates()
            .head(10)
        )


        examples.append(
            sample
        )


if examples:

    print()
    print("=" * 100)
    print("CONTROLLED NAME CORRECTION EXAMPLES")
    print("=" * 100)

    display(
        pd.concat(
            examples,
            ignore_index=True,
        )
        .drop_duplicates()
        .head(30)
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Payments can now be rebuilt directly from "
    "normalized_v0_1/payments."
)

print(
    "Validation output:"
)

print(
    VALIDATION_DIR
)

print()
print(
    "Production enriched payment parquet files "
    "were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_base.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_payment_base.py
Normalized payment pipeline added: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\payment_batch.py

TESTS
........................................................................ [ 66%]
....................................                                     [100%]
108 passed in 5.85s

Return code: 0

NORMALIZED -> ENRICHED PAYMENT REBUILD


,section,rows,columns,output
0,monetary_contributions,27234,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\monetary_contributions.parquet
1,other_contributions,6168,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\other_contributions.parquet
2,state_funding,96,127,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\state_funding.parquet
3,other_incomes,19007,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\other_incomes.parquet
4,budget_expenses,29482,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\budget_expenses.parquet
5,outgoing_expenses,319901,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\outgoing_expenses.parquet
6,return_expenses,137,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\return_expenses.parquet
7,transfer_expenses,3,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1\transfer_expenses.parquet



NORMALIZED -> ENRICHED PARITY


,section,rows,columns,expected_name_corrections,unexpected_field_mismatches,unexpected_columns
0,monetary_contributions,27234,126,21009,0,
1,other_contributions,6168,126,2156,0,
2,state_funding,96,127,96,0,
3,other_incomes,19007,126,1437,0,
4,budget_expenses,29482,126,28431,0,
5,outgoing_expenses,319901,126,31324,0,
6,return_expenses,137,126,55,0,
7,transfer_expenses,3,126,3,0,



Rows rebuilt: 402,028
Expected organization-name corrections: 84,511
Unexpected field-level mismatches: 0

CONTROLLED NAME CORRECTION EXAMPLES


,organization_id_old,organization_name_current_old,organization_name_current_new,party_name_current_new
0,8d8131a2-d14a-4a06-9f23-0eb2943e0419,РІДНЕ ЗАКАРПАТТЯ,ПОЛІТИЧНА ПАРТІЯ «РІДНЕ ЗАКАРПАТТЯ»,РІДНЕ ЗАКАРПАТТЯ
1,a6c4a2af-c24e-4b17-974f-9bbed4e88b65,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА,ПОЛІТИЧНА ПАРТІЯ «УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА»,УДАР (УКРАЇНСЬКИЙ ДЕМОКРАТИЧНИЙ АЛЬЯНС ЗА РЕФОРМИ) ВІТАЛІЯ КЛИЧКА
2,cefb6684-da05-4a85-a02c-de65d25d8707,БДЖОЛА,ПОЛІТИЧНА ПАРТІЯ «БДЖОЛА»,БДЖОЛА
3,e923d163-59e0-44fe-8891-8f62c7bdb3c7,ПОРЯДОК ДЛЯ УКРАЇНИ,ПОЛІТИЧНА ПАРТІЯ «ПОРЯДОК ДЛЯ УКРАЇНИ»,ПОРЯДОК ДЛЯ УКРАЇНИ
4,dbac3b1c-adb3-4ae5-92cb-cc307ff3ab3d,КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛІСТІВ,ПОЛІТИЧНА ПАРТІЯ «КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛІСТІВ»,КОНГРЕС УКРАЇНСЬКИХ НАЦІОНАЛІСТІВ
5,79cb7831-508b-466a-8db0-db9c5f4080b2,ПАРТІЯ ВОЛОДИМИРА БУРЯКА ЄДНАННЯ,ПОЛІТИЧНА ПАРТІЯ «ПАРТІЯ ВОЛОДИМИРА БУРЯКА «ЄДНАННЯ»,ПАРТІЯ ВОЛОДИМИРА БУРЯКА ЄДНАННЯ
6,7ed50708-02f2-40b7-979a-d4ba7ce7ca1a,ВО ПЛАТФОРМА ГРОМАД,ПОЛІТИЧНА ПАРТІЯ «ВО «ПЛАТФОРМА ГРОМАД»,ВО ПЛАТФОРМА ГРОМАД
7,2971b6fb-8db3-4800-8d53-252c8e2984ff,ГРОМАДСЬКА СИЛА,ПОЛІТИЧНА ПАРТІЯ «ГРОМАДСЬКА СИЛА»,ГРОМАДСЬКА СИЛА
8,4dc49830-ea98-44cf-b4c7-e226b6fe4f18,ЛІБЕРТАРІАНСЬКА ПАРТІЯ 5.10,ПОЛІТИЧНА ПАРТІЯ «ЛІБЕРТАРІАНСЬКА ПАРТІЯ 5.10»,ЛІБЕРТАРІАНСЬКА ПАРТІЯ 5.10
9,82c1b72c-f67e-4eaf-ac09-b025ee67adcc,ПРОПОЗИЦІЯ,ПОЛІТИЧНА ПАРТІЯ «ПРОПОЗИЦІЯ»,ПРОПОЗИЦІЯ



DONE
Payments can now be rebuilt directly from normalized_v0_1/payments.
Validation output:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\payment_from_normalized_v0_1

Production enriched payment parquet files were NOT modified.
RAW files were NOT read.
API was NOT called.


In [135]:
from pathlib import Path
from itertools import combinations

import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

ENRICHED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
)

REPORT_CONTEXT_PATH = (
    ENRICHED_ROOT
    / "reference"
    / "report_context.parquet"
)


# ============================================================
# CURRENT VERIFIED BASELINE
# ============================================================

EXPECTED_ROWS = {
    "realty": 11_325,
    "transport": 1_325,
    "movable": 693,
    "intangible": 8_614,
    "paper": 0,
    "obligations": 26_132,
    "head_info": 78_791,
    "employee_counts": 78_791,
    "organizations": 308,
    "regional_offices": 102_678,
}


SECTIONS = list(
    EXPECTED_ROWS
)


# ============================================================
# HELPERS
# ============================================================

def parquet_rows(path):

    return (
        pq.ParquetFile(path)
        .metadata
        .num_rows
    )


def parquet_columns(path):

    return list(
        pq.ParquetFile(
            path
        ).schema_arrow.names
    )


def find_section_file(
    root,
    section,
    expected_rows,
):

    candidates = list(
        root.rglob(
            f"{section}.parquet"
        )
    )


    if not candidates:

        raise FileNotFoundError(
            f"No {section}.parquet under {root}"
        )


    exact = [
        path
        for path
        in candidates
        if parquet_rows(path)
        ==
        expected_rows
    ]


    if len(exact) == 1:

        return exact[0]


    print()
    print(
        f"Ambiguous section: {section}"
    )

    for path in candidates:

        print(
            " ",
            parquet_rows(path),
            path,
        )


    raise RuntimeError(
        f"Could not uniquely resolve {section}"
    )


def comparable_string(
    series,
):

    return (
        series
        .astype("string")
        .fillna("<NULL>")
    )


def mismatch_count(
    left,
    right,
):

    if len(left) != len(right):

        raise ValueError(
            "Series lengths differ."
        )


    both_missing = (
        left.isna()
        &
        right.isna()
    )


    try:

        equal = (
            left.eq(
                right
            )
            .fillna(False)
        )

    except Exception:

        equal = (
            comparable_string(left)
            ==
            comparable_string(right)
        )


    return int(
        (
            ~(
                equal
                |
                both_missing
            )
        ).sum()
    )


def make_key(
    df,
    columns,
):

    values = []


    for column in columns:

        values.append(
            df[
                column
            ]
            .astype("string")
            .fillna("<NULL>")
        )


    key = values[0]


    for value in values[1:]:

        key = (
            key
            +
            "||"
            +
            value
        )


    return key


def find_safe_row_key(
    normalized,
    enriched,
):

    if len(normalized) == 0:

        return (
            "empty_section",
            (),
        )


    common = (
        set(normalized.columns)
        &
        set(enriched.columns)
    )


    # --------------------------------------------------------
    # Strong known candidates first
    # --------------------------------------------------------

    candidates = [
        ("source_row_id",),
        ("source_report_id", "source_row_id"),
        ("source_report_id", "source__id"),
        ("source_report_id", "source_id"),
        ("source_report_id", "source_row_index"),
        ("source_report_id", "row_index"),
        ("source_report_id", "section_row_index"),
        ("source_report_id",),
    ]


    # --------------------------------------------------------
    # Add identifier-like candidates dynamically
    # --------------------------------------------------------

    identifier_columns = [
        column
        for column
        in sorted(common)
        if (
            column.endswith("_id")
            or
            column.endswith("_index")
            or
            column.startswith("source__")
        )
    ]


    for column in identifier_columns:

        candidates.append(
            (column,)
        )


    if (
        "source_report_id"
        in common
    ):

        for column in identifier_columns:

            if column != "source_report_id":

                candidates.append(
                    (
                        "source_report_id",
                        column,
                    )
                )


    # Deduplicate candidate tuples
    seen = set()
    unique_candidates = []


    for candidate in candidates:

        if candidate in seen:
            continue

        seen.add(
            candidate
        )

        unique_candidates.append(
            candidate
        )


    for columns in unique_candidates:

        if not all(
            column in common
            for column in columns
        ):
            continue


        n_key = make_key(
            normalized,
            columns,
        )

        e_key = make_key(
            enriched,
            columns,
        )


        if (
            n_key.is_unique
            and
            e_key.is_unique
            and
            set(n_key)
            ==
            set(e_key)
        ):

            return (
                " + ".join(columns),
                columns,
            )


    return (
        None,
        None,
    )


# ============================================================
# 1. LOCATE CURRENT SECTION FILES
# ============================================================

paths = {}


for section in SECTIONS:

    normalized_path = (
        find_section_file(
            NORMALIZED_ROOT,
            section,
            EXPECTED_ROWS[
                section
            ],
        )
    )


    enriched_path = (
        find_section_file(
            ENRICHED_ROOT,
            section,
            EXPECTED_ROWS[
                section
            ],
        )
    )


    paths[
        section
    ] = {
        "normalized":
            normalized_path,

        "enriched":
            enriched_path,
    }


print()
print("=" * 110)
print("SECTION FILES")
print("=" * 110)


file_rows = []


for section in SECTIONS:

    file_rows.append(
        {
            "section":
                section,

            "rows":
                EXPECTED_ROWS[
                    section
                ],

            "normalized":
                str(
                    paths[
                        section
                    ][
                        "normalized"
                    ].relative_to(
                        ROOT
                    )
                ),

            "enriched":
                str(
                    paths[
                        section
                    ][
                        "enriched"
                    ].relative_to(
                        ROOT
                    )
                ),
        }
    )


display(
    pd.DataFrame(
        file_rows
    )
)


# ============================================================
# 2. REPORT CONTEXT
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH
)


if (
    len(report_context)
    !=
    78_791
):

    raise RuntimeError(
        "Unexpected report_context row count."
    )


if (
    report_context[
        "source_report_id"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "report_context source_report_id "
        "is not unique."
    )


context_columns = set(
    report_context.columns
)


# ============================================================
# 3. SCHEMA + SAFE KEY AUDIT
# ============================================================

schema_rows = []

loaded = {}


for section in SECTIONS:

    normalized = pd.read_parquet(
        paths[
            section
        ][
            "normalized"
        ]
    )

    enriched = pd.read_parquet(
        paths[
            section
        ][
            "enriched"
        ]
    )


    loaded[
        section
    ] = (
        normalized,
        enriched,
    )


    if (
        len(normalized)
        !=
        EXPECTED_ROWS[
            section
        ]
    ):

        raise RuntimeError(
            f"{section}: normalized baseline changed."
        )


    if (
        len(enriched)
        !=
        EXPECTED_ROWS[
            section
        ]
    ):

        raise RuntimeError(
            f"{section}: enriched baseline changed."
        )


    key_name, key_columns = (
        find_safe_row_key(
            normalized,
            enriched,
        )
    )


    normalized_set = set(
        normalized.columns
    )

    enriched_set = set(
        enriched.columns
    )


    schema_rows.append(
        {
            "section":
                section,

            "rows":
                len(normalized),

            "normalized_columns":
                len(
                    normalized.columns
                ),

            "enriched_columns":
                len(
                    enriched.columns
                ),

            "common_columns":
                len(
                    normalized_set
                    &
                    enriched_set
                ),

            "enriched_only":
                len(
                    enriched_set
                    -
                    normalized_set
                ),

            "normalized_only":
                len(
                    normalized_set
                    -
                    enriched_set
                ),

            "safe_row_key":
                key_name,
        }
    )


schema_df = pd.DataFrame(
    schema_rows
)


print()
print("=" * 110)
print("SCHEMA / ROW KEY AUDIT")
print("=" * 110)

display(
    schema_df
)


# ============================================================
# 4. KEYED COMMON-COLUMN PARITY
# ============================================================

common_rows = []


for section in SECTIONS:

    normalized, enriched = (
        loaded[
            section
        ]
    )


    if len(normalized) == 0:

        common_rows.append(
            {
                "section":
                    section,

                "common_columns":
                    len(
                        set(
                            normalized.columns
                        )
                        &
                        set(
                            enriched.columns
                        )
                    ),

                "mismatching_columns":
                    0,

                "field_level_mismatches":
                    0,

                "details":
                    "",
            }
        )

        continue


    key_name, key_columns = (
        find_safe_row_key(
            normalized,
            enriched,
        )
    )


    if key_columns is None:

        common_rows.append(
            {
                "section":
                    section,

                "common_columns":
                    None,

                "mismatching_columns":
                    None,

                "field_level_mismatches":
                    None,

                "details":
                    "NO SAFE ROW KEY",
            }
        )

        continue


    normalized = (
        normalized
        .assign(
            _audit_key=
                make_key(
                    normalized,
                    key_columns,
                )
        )
        .set_index(
            "_audit_key"
        )
        .sort_index()
    )


    enriched = (
        enriched
        .assign(
            _audit_key=
                make_key(
                    enriched,
                    key_columns,
                )
        )
        .set_index(
            "_audit_key"
        )
        .sort_index()
    )


    common_columns = [
        column
        for column
        in normalized.columns
        if column
        in enriched.columns
    ]


    bad_columns = []
    total_mismatches = 0


    for column in common_columns:

        mismatches = mismatch_count(
            normalized[
                column
            ],
            enriched[
                column
            ],
        )


        total_mismatches += (
            mismatches
        )


        if mismatches:

            bad_columns.append(
                f"{column} ({mismatches})"
            )


    common_rows.append(
        {
            "section":
                section,

            "common_columns":
                len(
                    common_columns
                ),

            "mismatching_columns":
                len(
                    bad_columns
                ),

            "field_level_mismatches":
                total_mismatches,

            "details":
                " | ".join(
                    bad_columns[:20]
                ),
        }
    )


common_df = pd.DataFrame(
    common_rows
)


print()
print("=" * 110)
print("KEYED COMMON-COLUMN PARITY")
print("=" * 110)

display(
    common_df
)


# ============================================================
# 5. ALL ENRICHED-ONLY COLUMNS
# ============================================================

enriched_only_presence = {}


for section in SECTIONS:

    normalized, enriched = (
        loaded[
            section
        ]
    )


    extra = (
        set(
            enriched.columns
        )
        -
        set(
            normalized.columns
        )
    )


    for column in extra:

        enriched_only_presence.setdefault(
            column,
            []
        ).append(
            section
        )


extra_rows = []


for column, sections in sorted(
    enriched_only_presence.items()
):

    if column in context_columns:

        category = (
            "report_context"
        )

    elif column in {
        "is_latest_data",
        "data_recency_status",
    }:

        category = (
            "snapshot_recency"
        )

    else:

        category = (
            "other_enrichment"
        )


    extra_rows.append(
        {
            "column":
                column,

            "category":
                category,

            "section_count":
                len(
                    sections
                ),

            "sections":
                " | ".join(
                    sections
                ),
        }
    )


extra_df = pd.DataFrame(
    extra_rows
)


print()
print("=" * 110)
print("ENRICHED-ONLY COLUMNS")
print("=" * 110)

display(
    extra_df
)


# ============================================================
# 6. REPORT CONTEXT DIRECT-JOIN PARITY
# ============================================================

context_added = sorted(
    set(
        enriched_only_presence
    )
    &
    context_columns
)


context_rows = []


for section in SECTIONS:

    normalized, enriched = (
        loaded[
            section
        ]
    )


    available = [
        column
        for column
        in context_added
        if column
        in enriched.columns
    ]


    if (
        len(normalized) == 0
        or
        not available
    ):

        context_rows.append(
            {
                "section":
                    section,

                "context_columns":
                    len(
                        available
                    ),

                "mismatching_columns":
                    0,

                "field_level_mismatches":
                    0,

                "details":
                    "",
            }
        )

        continue


    if (
        "source_report_id"
        not in normalized.columns
    ):

        context_rows.append(
            {
                "section":
                    section,

                "context_columns":
                    len(
                        available
                    ),

                "mismatching_columns":
                    None,

                "field_level_mismatches":
                    None,

                "details":
                    "NO source_report_id",
            }
        )

        continue


    key_name, key_columns = (
        find_safe_row_key(
            normalized,
            enriched,
        )
    )


    if key_columns is None:

        context_rows.append(
            {
                "section":
                    section,

                "context_columns":
                    len(
                        available
                    ),

                "mismatching_columns":
                    None,

                "field_level_mismatches":
                    None,

                "details":
                    "NO SAFE ROW KEY",
            }
        )

        continue


    n = (
        normalized
        .assign(
            _audit_key=
                make_key(
                    normalized,
                    key_columns,
                )
        )
    )


    e = (
        enriched
        .assign(
            _audit_key=
                make_key(
                    enriched,
                    key_columns,
                )
        )
        .set_index(
            "_audit_key"
        )
        .sort_index()
    )


    expected = (
        n[
            [
                "_audit_key",
                "source_report_id",
            ]
        ]
        .merge(
            report_context[
                [
                    "source_report_id",
                    *available,
                ]
            ],
            on="source_report_id",
            how="left",
            validate="many_to_one",
            sort=False,
        )
        .set_index(
            "_audit_key"
        )
        .sort_index()
    )


    bad_columns = []
    total_mismatches = 0


    for column in available:

        mismatches = mismatch_count(
            e[
                column
            ],
            expected[
                column
            ],
        )


        total_mismatches += (
            mismatches
        )


        if mismatches:

            bad_columns.append(
                f"{column} ({mismatches})"
            )


    context_rows.append(
        {
            "section":
                section,

            "context_columns":
                len(
                    available
                ),

            "mismatching_columns":
                len(
                    bad_columns
                ),

            "field_level_mismatches":
                total_mismatches,

            "details":
                " | ".join(
                    bad_columns[:20]
                ),
        }
    )


context_df = pd.DataFrame(
    context_rows
)


print()
print("=" * 110)
print("REPORT_CONTEXT DIRECT-JOIN PARITY")
print("=" * 110)

display(
    context_df
)


# ============================================================
# 7. SNAPSHOT / LATEST-DATA PARITY
# ============================================================

recency_rows = []


for section in SECTIONS:

    normalized, enriched = (
        loaded[
            section
        ]
    )


    recency_columns = [
        column
        for column
        in (
            "is_latest_data",
            "data_recency_status",
        )
        if column
        in enriched.columns
    ]


    if (
        len(normalized) == 0
        or
        not recency_columns
    ):

        recency_rows.append(
            {
                "section":
                    section,

                "recency_columns":
                    len(
                        recency_columns
                    ),

                "field_level_mismatches":
                    0,

                "details":
                    "",
            }
        )

        continue


    key_name, key_columns = (
        find_safe_row_key(
            normalized,
            enriched,
        )
    )


    if key_columns is None:

        recency_rows.append(
            {
                "section":
                    section,

                "recency_columns":
                    len(
                        recency_columns
                    ),

                "field_level_mismatches":
                    None,

                "details":
                    "NO SAFE ROW KEY",
            }
        )

        continue


    n = (
        normalized
        .assign(
            _audit_key=
                make_key(
                    normalized,
                    key_columns,
                )
        )
    )


    e = (
        enriched
        .assign(
            _audit_key=
                make_key(
                    enriched,
                    key_columns,
                )
        )
        .set_index(
            "_audit_key"
        )
        .sort_index()
    )


    context_needed = [
        "source_report_id",
        "is_latest_report_for_organization",
        "data_recency_status",
    ]


    expected = (
        n[
            [
                "_audit_key",
                "source_report_id",
            ]
        ]
        .merge(
            report_context[
                context_needed
            ],
            on="source_report_id",
            how="left",
            validate="many_to_one",
            sort=False,
        )
        .set_index(
            "_audit_key"
        )
        .sort_index()
    )


    mismatches = 0
    details = []


    if (
        "is_latest_data"
        in recency_columns
    ):

        count = mismatch_count(
            e[
                "is_latest_data"
            ],
            expected[
                "is_latest_report_for_organization"
            ],
        )


        mismatches += count


        if count:

            details.append(
                f"is_latest_data ({count})"
            )


    if (
        "data_recency_status"
        in recency_columns
    ):

        count = mismatch_count(
            e[
                "data_recency_status"
            ],
            expected[
                "data_recency_status"
            ],
        )


        mismatches += count


        if count:

            details.append(
                f"data_recency_status ({count})"
            )


    recency_rows.append(
        {
            "section":
                section,

            "recency_columns":
                len(
                    recency_columns
                ),

            "field_level_mismatches":
                mismatches,

            "details":
                " | ".join(
                    details
                ),
        }
    )


recency_df = pd.DataFrame(
    recency_rows
)


print()
print("=" * 110)
print("SNAPSHOT / LATEST-DATA PARITY")
print("=" * 110)

display(
    recency_df
)


# ============================================================
# 8. REMAINING UNEXPLAINED ENRICHMENT
# ============================================================

explained = (
    context_columns
    |
    {
        "is_latest_data",
        "data_recency_status",
    }
)


remaining = (
    extra_df[
        ~extra_df[
            "column"
        ].isin(
            explained
        )
    ]
    if len(extra_df)
    else
    pd.DataFrame()
)


print()
print("=" * 110)
print("REMAINING UNEXPLAINED ENRICHMENT")
print("=" * 110)


if len(
    remaining
):

    display(
        remaining
    )

else:

    print(
        "None."
    )


# ============================================================
# 9. SECTION SCHEMA DETAILS
# ============================================================

schema_detail_rows = []


for section in SECTIONS:

    normalized, enriched = (
        loaded[
            section
        ]
    )


    schema_detail_rows.append(
        {
            "section":
                section,

            "normalized_columns":
                " | ".join(
                    normalized.columns
                ),

            "enriched_only_columns":
                " | ".join(
                    sorted(
                        set(
                            enriched.columns
                        )
                        -
                        set(
                            normalized.columns
                        )
                    )
                ),
        }
    )


print()
print("=" * 110)
print("SECTION SCHEMA DETAILS")
print("=" * 110)

display(
    pd.DataFrame(
        schema_detail_rows
    )
)


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

common_total = (
    common_df[
        "field_level_mismatches"
    ]
    .fillna(0)
    .sum()
)


context_total = (
    context_df[
        "field_level_mismatches"
    ]
    .fillna(0)
    .sum()
)


recency_total = (
    recency_df[
        "field_level_mismatches"
    ]
    .fillna(0)
    .sum()
)


print()
print("=" * 110)
print("SUMMARY")
print("=" * 110)

print(
    "Sections:",
    len(
        SECTIONS
    )
)

print(
    "Normalized rows total:",
    f"{sum(EXPECTED_ROWS.values()):,}"
)

print(
    "Keyed common-column mismatches:",
    f"{int(common_total):,}"
)

print(
    "Report-context mismatches:",
    f"{int(context_total):,}"
)

print(
    "Snapshot-recency mismatches:",
    f"{int(recency_total):,}"
)

print(
    "Remaining unexplained enrichment columns:",
    (
        len(remaining)
        if isinstance(
            remaining,
            pd.DataFrame
        )
        else
        0
    )
)

print()
print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No parquet files were modified."
)


SECTION FILES


,section,rows,normalized,enriched
0,realty,11325,data\processed\normalized_v0_1\properties\realty.parquet,data\processed\enriched_v0_1\properties\realty.parquet
1,transport,1325,data\processed\normalized_v0_1\properties\transport.parquet,data\processed\enriched_v0_1\properties\transport.parquet
2,movable,693,data\processed\normalized_v0_1\properties\movable.parquet,data\processed\enriched_v0_1\properties\movable.parquet
3,intangible,8614,data\processed\normalized_v0_1\properties\intangible.parquet,data\processed\enriched_v0_1\properties\intangible.parquet
4,paper,0,data\processed\normalized_v0_1\properties\paper.parquet,data\processed\enriched_v0_1\properties\paper.parquet
5,obligations,26132,data\processed\normalized_v0_1\obligations\obligations.parquet,data\processed\enriched_v0_1\obligations\obligations.parquet
6,head_info,78791,data\processed\normalized_v0_1\report_state\head_info.parquet,data\processed\enriched_v0_1\report_state\head_info.parquet
7,employee_counts,78791,data\processed\normalized_v0_1\report_state\employee_counts.parquet,data\processed\enriched_v0_1\report_state\employee_counts.parquet
8,organizations,308,data\processed\normalized_v0_1\report_state\organizations.parquet,data\processed\enriched_v0_1\report_state\organizations.parquet
9,regional_offices,102678,data\processed\normalized_v0_1\report_state\regional_offices.parquet,data\processed\enriched_v0_1\report_state\regional_offices.parquet



SCHEMA / ROW KEY AUDIT


,section,rows,normalized_columns,enriched_columns,common_columns,enriched_only,normalized_only,safe_row_key
0,realty,11325,28,44,28,16,0,source_report_id + source__id
1,transport,1325,27,43,27,16,0,source_report_id + source__id
2,movable,693,26,42,26,16,0,source_report_id + source__id
3,intangible,8614,25,41,25,16,0,source_report_id + source__id
4,paper,0,12,28,12,16,0,empty_section
5,obligations,26132,27,43,27,16,0,source_report_id + source__id
6,head_info,78791,16,32,16,16,0,source_report_id + source_row_index
7,employee_counts,78791,16,32,16,16,0,source_report_id + source_row_index
8,organizations,308,15,31,15,16,0,source_report_id + source_row_index
9,regional_offices,102678,29,45,29,16,0,source_report_id + source_row_index



KEYED COMMON-COLUMN PARITY


...


REMAINING UNEXPLAINED ENRICHMENT


,column,category,section_count,sections
1,analysis_selected,other_enrichment,10,realty | transport | movable | intangible | paper | obligations | head_info | employee_counts | organizations | regional_offices
7,official_selected,other_enrichment,10,realty | transport | movable | intangible | paper | obligations | head_info | employee_counts | organizations | regional_offices
15,report_period,other_enrichment,10,realty | transport | movable | intangible | paper | obligations | head_info | employee_counts | organizations | regional_offices



SECTION SCHEMA DETAILS


,section,normalized_columns,enriched_only_columns
0,realty,source_report_id | source_section | source_row_index | organization_id | root_party_id | report_year | report_quarter | source_is_signed | source_signed_date | report_schema_version_source | report_type_source | is_party_office_source | source_row_json | source__id | source__party_id | source__office_id | source__report_status | source__object_type | source__object_number | source__owning_date | source__owning_cost | source__owner_code | source__owner_name | source__total_area | source__object_address | source__object_rights | source__substraction_date | source__created_at,analysis_override | analysis_selected | analysis_selected_report_id | analysis_selection_method | continuity_exact | data_recency_status | is_latest_data | official_selected | official_selected_report_id | organization_code | organization_level | organization_name_current | party_code | party_name_current | region | report_period
1,transport,source_report_id | source_section | source_row_index | organization_id | root_party_id | report_year | report_quarter | source_is_signed | source_signed_date | report_schema_version_source | report_type_source | is_party_office_source | source_row_json | source__id | source__party_id | source__office_id | source__report_status | source__transport_type | source__owning_date | source__owning_cost | source__object_number | source__transport_brand | source__transport_model | source__production_year | source__object_rights | source__substraction_date | source__created_at,analysis_override | analysis_selected | analysis_selected_report_id | analysis_selection_method | continuity_exact | data_recency_status | is_latest_data | official_selected | official_selected_report_id | organization_code | organization_level | organization_name_current | party_code | party_name_current | region | report_period
2,movable,source_report_id | source_section | source_row_index | organization_id | root_party_id | report_year | report_quarter | source_is_signed | source_signed_date | report_schema_version_source | report_type_source | is_party_office_source | source_row_json | source__id | source__party_id | source__office_id | source__report_status | source__movable_type | source__owning_date | source__owning_cost | source__description | source__manufacturer_name | source__trade_mark | source__movable_rights | source__substraction_date | source__created_at,analysis_override | analysis_selected | analysis_selected_report_id | analysis_selection_method | continuity_exact | data_recency_status | is_latest_data | official_selected | official_selected_report_id | organization_code | organization_level | organization_name_current | party_code | party_name_current | region | report_period
3,intangible,source_report_id | source_section | source_row_index | organization_id | root_party_id | report_year | report_quarter | source_is_signed | source_signed_date | report_schema_version_source | report_type_source | is_party_office_source | source_row_json | source__id | source__party_id | source__office_id | source__report_status | source__asset_type | source__asset_count | source__asset_description | source__asset_rights | source__owning_date | source__owning_cost | source__substraction_date | source__created_at,analysis_override | analysis_selected | analysis_selected_report_id | analysis_selection_method | continuity_exact | data_recency_status | is_latest_data | official_selected | official_selected_report_id | organization_code | organization_level | organization_name_current | party_code | party_name_current | region | report_period
4,paper,source_report_id | source_section | source_row_index | organization_id | root_party_id | report_year | report_quarter | source_is_signed | source_signed_date | report_schema_version_source | report_type_source | is_party_office_source,analysis_override | analysis_selected | analysis_selected_report_id | analysis_selection_method | continuity_exact | data_rec


SUMMARY
Sections: 10
Normalized rows total: 308,657
Keyed common-column mismatches: 0
Report-context mismatches: 0
Snapshot-recency mismatches: 0
Remaining unexplained enrichment columns: 3

RAW files were NOT read.
API was NOT called.
No parquet files were modified.


In [137]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

ENRICHMENT_DIR = (
    ROOT
    / "src"
    / "politdata"
    / "enrichment"
)

TESTS_DIR = (
    ROOT
    / "tests"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

CURRENT_ENRICHED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
)

REPORT_CONTEXT_PATH = (
    CURRENT_ENRICHED_ROOT
    / "reference"
    / "report_context.parquet"
)

VALIDATION_ROOT = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "report_sections_from_normalized_v0_1"
)


# ============================================================
# SECTION MAP
# ============================================================

SECTION_PATHS = {
    "realty":
        Path("properties/realty.parquet"),

    "transport":
        Path("properties/transport.parquet"),

    "movable":
        Path("properties/movable.parquet"),

    "intangible":
        Path("properties/intangible.parquet"),

    "paper":
        Path("properties/paper.parquet"),

    "obligations":
        Path("obligations/obligations.parquet"),

    "head_info":
        Path("report_state/head_info.parquet"),

    "employee_counts":
        Path("report_state/employee_counts.parquet"),

    "organizations":
        Path("report_state/organizations.parquet"),

    "regional_offices":
        Path("report_state/regional_offices.parquet"),
}


EXPECTED_ROWS = {
    "realty": 11_325,
    "transport": 1_325,
    "movable": 693,
    "intangible": 8_614,
    "paper": 0,
    "obligations": 26_132,
    "head_info": 78_791,
    "employee_counts": 78_791,
    "organizations": 308,
    "regional_offices": 102_678,
}


# ============================================================
# HELPERS
# ============================================================

def report_period_label(
    year,
    quarter,
):

    if pd.isna(year):
        return None

    try:
        year = int(year)

    except Exception:
        return None

    try:
        quarter = int(quarter)

    except Exception:
        quarter = None


    # quarter=5 means annual
    if quarter == 5:
        return str(year)


    if quarter in (
        1,
        2,
        3,
        4,
    ):
        return (
            f"{year}Q{quarter}"
        )


    return str(year)


def compare_series(
    left,
    right,
):

    if len(left) != len(right):
        raise ValueError(
            "Series lengths differ."
        )


    left = (
        left
        .astype("string")
        .fillna("<NULL>")
    )

    right = (
        right
        .astype("string")
        .fillna("<NULL>")
    )


    return int(
        (
            left != right
        ).sum()
    )


def make_key(
    df,
    columns,
):

    result = (
        df[
            columns[0]
        ]
        .astype("string")
        .fillna("<NULL>")
    )


    for column in columns[1:]:

        result = (
            result
            +
            "||"
            +
            df[
                column
            ]
            .astype("string")
            .fillna("<NULL>")
        )


    return result


def section_key_columns(
    section,
):

    if section in {
        "realty",
        "transport",
        "movable",
        "intangible",
        "obligations",
    }:

        return (
            "source_report_id",
            "source__id",
        )


    return (
        "source_report_id",
        "source_row_index",
    )


# ============================================================
# 1. VERIFY THE THREE RULES AGAIN
#
# The only change from the previous attempt:
#
# report_period is derived from
# report_year + report_quarter,
# NOT report_context.period_label.
# ============================================================

report_context = pd.read_parquet(
    REPORT_CONTEXT_PATH
)


if (
    report_context[
        "source_report_id"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "report_context source_report_id "
        "is not unique."
    )


context_lookup = (
    report_context[
        [
            "source_report_id",
            "official_selected_report_id",
        ]
    ]
    .copy()
)


rule_rows = []


print()
print("=" * 100)
print("VERIFY FINAL THREE ENRICHMENT RULES")
print("=" * 100)


for section, relative_path in (
    SECTION_PATHS.items()
):

    enriched = pd.read_parquet(
        CURRENT_ENRICHED_ROOT
        / relative_path
    )


    if len(enriched) == 0:

        rule_rows.append(
            {
                "section":
                    section,

                "rows":
                    0,

                "analysis_selected_mismatches":
                    0,

                "official_selected_mismatches":
                    0,

                "report_period_mismatches":
                    0,

                "total_mismatches":
                    0,
            }
        )

        continue


    expected_context = (
        enriched[
            [
                "source_report_id"
            ]
        ]
        .merge(
            context_lookup,
            on="source_report_id",
            how="left",
            validate="many_to_one",
            sort=False,
        )
    )


    expected_analysis = pd.Series(
        True,
        index=enriched.index,
    )


    expected_official = (
        enriched[
            "source_report_id"
        ]
        .astype("string")
        ==
        expected_context[
            "official_selected_report_id"
        ]
        .astype("string")
    )


    expected_period = pd.Series(
        [
            report_period_label(
                year,
                quarter,
            )
            for year, quarter
            in zip(
                enriched[
                    "report_year"
                ],
                enriched[
                    "report_quarter"
                ],
            )
        ],
        index=enriched.index,
        dtype="string",
    )


    analysis_mismatch = (
        compare_series(
            enriched[
                "analysis_selected"
            ],
            expected_analysis,
        )
    )


    official_mismatch = (
        compare_series(
            enriched[
                "official_selected"
            ],
            expected_official,
        )
    )


    period_mismatch = (
        compare_series(
            enriched[
                "report_period"
            ],
            expected_period,
        )
    )


    rule_rows.append(
        {
            "section":
                section,

            "rows":
                len(enriched),

            "analysis_selected_mismatches":
                analysis_mismatch,

            "official_selected_mismatches":
                official_mismatch,

            "report_period_mismatches":
                period_mismatch,

            "total_mismatches":
                (
                    analysis_mismatch
                    +
                    official_mismatch
                    +
                    period_mismatch
                ),
        }
    )


rule_df = pd.DataFrame(
    rule_rows
)


display(
    rule_df
)


rule_mismatches = int(
    rule_df[
        "total_mismatches"
    ].sum()
)


print(
    "Total rule mismatches:",
    f"{rule_mismatches:,}"
)


if rule_mismatches != 0:

    raise RuntimeError(
        "Final report-section rules still do not "
        "have exact parity. "
        "No module was written."
    )


# ============================================================
# 2. CREATE GENERIC REPORT-SECTION ENRICHMENT MODULE
# ============================================================

MODULE_CODE = r'''
from __future__ import annotations

from pathlib import Path

import pandas as pd


SECTION_PATHS = {
    "realty":
        Path("properties/realty.parquet"),

    "transport":
        Path("properties/transport.parquet"),

    "movable":
        Path("properties/movable.parquet"),

    "intangible":
        Path("properties/intangible.parquet"),

    "paper":
        Path("properties/paper.parquet"),

    "obligations":
        Path("obligations/obligations.parquet"),

    "head_info":
        Path("report_state/head_info.parquet"),

    "employee_counts":
        Path("report_state/employee_counts.parquet"),

    "organizations":
        Path("report_state/organizations.parquet"),

    "regional_offices":
        Path("report_state/regional_offices.parquet"),
}


CONTEXT_COLUMNS = (
    "organization_name_current",
    "organization_code",
    "organization_level",
    "region",

    "party_name_current",
    "party_code",

    "analysis_override",
    "analysis_selection_method",

    "official_selected_report_id",
    "analysis_selected_report_id",

    "continuity_exact",
)


FINAL_ENRICHMENT_COLUMNS = (
    "organization_name_current",
    "organization_code",
    "organization_level",
    "region",

    "party_name_current",
    "party_code",

    "analysis_override",
    "analysis_selection_method",

    "official_selected_report_id",
    "analysis_selected_report_id",

    "continuity_exact",

    "analysis_selected",
    "official_selected",

    "is_latest_data",
    "data_recency_status",

    "report_period",
)


def report_period_label(
    year,
    quarter,
):
    """
    Canonical analytical period label.

    Quarterly:
        2025Q1 ... 2025Q4

    Annual (quarter=5):
        2025
    """

    if pd.isna(year):
        return None


    try:
        year = int(year)

    except Exception:
        return None


    try:
        quarter = int(quarter)

    except Exception:
        quarter = None


    if quarter == 5:
        return str(year)


    if quarter in (
        1,
        2,
        3,
        4,
    ):

        return (
            f"{year}Q{quarter}"
        )


    return str(year)


def _prepare_report_context(
    report_context: pd.DataFrame,
) -> pd.DataFrame:

    required = {
        "source_report_id",
        "is_latest_report_for_organization",
        *CONTEXT_COLUMNS,
    }


    missing = (
        required
        -
        set(
            report_context.columns
        )
    )


    if missing:

        raise KeyError(
            "report_context missing columns: "
            f"{sorted(missing)}"
        )


    if (
        report_context[
            "source_report_id"
        ]
        .duplicated()
        .any()
    ):

        raise ValueError(
            "report_context source_report_id "
            "must be unique."
        )


    return (
        report_context[
            [
                "source_report_id",
                *CONTEXT_COLUMNS,
                "is_latest_report_for_organization",
            ]
        ]
        .copy()
    )


def enrich_report_section_frame(
    normalized: pd.DataFrame,
    *,
    report_context: pd.DataFrame,
) -> pd.DataFrame:
    """
    Generic normalized -> enriched transformation
    for all ten report-state / snapshot sections.

    Existing normalized fields are preserved unchanged.
    """

    required = {
        "source_report_id",
        "report_year",
        "report_quarter",
    }


    missing = (
        required
        -
        set(
            normalized.columns
        )
    )


    if missing:

        raise KeyError(
            "normalized section missing columns: "
            f"{sorted(missing)}"
        )


    original_columns = list(
        normalized.columns
    )


    # --------------------------------------------------------
    # Safe rebuild if called on stale enriched input.
    # --------------------------------------------------------

    stale = [
        column
        for column
        in FINAL_ENRICHMENT_COLUMNS
        if column
        in normalized.columns
    ]


    base = (
        normalized
        .drop(
            columns=stale,
            errors="ignore",
        )
        .reset_index(
            drop=True
        )
        .copy()
    )


    context = (
        _prepare_report_context(
            report_context
        )
    )


    before_rows = len(
        base
    )


    result = base.merge(
        context,
        on="source_report_id",
        how="left",
        validate="many_to_one",
        sort=False,
    )


    if len(result) != before_rows:

        raise RuntimeError(
            "report_context join changed "
            "section row count."
        )


    # --------------------------------------------------------
    # ANALYTICAL SELECTION
    # --------------------------------------------------------

    result[
        "analysis_selected"
    ] = True


    result[
        "official_selected"
    ] = (
        result[
            "source_report_id"
        ]
        .astype("string")
        ==
        result[
            "official_selected_report_id"
        ]
        .astype("string")
    )


    # --------------------------------------------------------
    # SNAPSHOT RECENCY
    # --------------------------------------------------------

    result[
        "is_latest_data"
    ] = (
        result[
            "is_latest_report_for_organization"
        ]
        .fillna(False)
        .astype(bool)
    )


    result[
        "data_recency_status"
    ] = (
        result[
            "is_latest_data"
        ]
        .map(
            {
                True:
                    "latest_data",

                False:
                    "historical_data",
            }
        )
    )


    result = result.drop(
        columns=[
            "is_latest_report_for_organization"
        ]
    )


    # --------------------------------------------------------
    # CANONICAL PERIOD LABEL
    # --------------------------------------------------------

    result[
        "report_period"
    ] = [
        report_period_label(
            year,
            quarter,
        )
        for year, quarter
        in zip(
            result[
                "report_year"
            ],
            result[
                "report_quarter"
            ],
        )
    ]


    # --------------------------------------------------------
    # Stable output schema:
    # normalized fields first, enrichment fields after.
    # --------------------------------------------------------

    output_columns = (
        [
            column
            for column
            in original_columns
            if column
            not in FINAL_ENRICHMENT_COLUMNS
        ]
        +
        list(
            FINAL_ENRICHMENT_COLUMNS
        )
    )


    return result[
        output_columns
    ]


def enrich_report_sections_directory(
    normalized_root,
    output_root,
    *,
    report_context,
    overwrite: bool = False,
) -> pd.DataFrame:
    """
    Rebuild all ten enriched report sections from
    normalized parquet.

    No RAW access.
    No API access.
    """

    normalized_root = Path(
        normalized_root
    )

    output_root = Path(
        output_root
    )


    if isinstance(
        report_context,
        (
            str,
            Path,
        ),
    ):

        report_context = (
            pd.read_parquet(
                report_context
            )
        )


    rows = []


    for section, relative_path in (
        SECTION_PATHS.items()
    ):

        input_path = (
            normalized_root
            / relative_path
        )

        output_path = (
            output_root
            / relative_path
        )


        if not input_path.exists():

            raise FileNotFoundError(
                input_path
            )


        if (
            output_path.exists()
            and
            not overwrite
        ):

            raise FileExistsError(
                output_path
            )


        normalized = pd.read_parquet(
            input_path
        )


        enriched = (
            enrich_report_section_frame(
                normalized,
                report_context=
                    report_context,
            )
        )


        if len(enriched) != len(normalized):

            raise RuntimeError(
                f"{section}: row count changed."
            )


        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )


        temp_path = (
            output_path
            .with_suffix(
                ".tmp.parquet"
            )
        )


        enriched.to_parquet(
            temp_path,
            index=False,
        )


        temp_path.replace(
            output_path
        )


        rows.append(
            {
                "section":
                    section,

                "rows":
                    len(enriched),

                "columns":
                    len(
                        enriched.columns
                    ),

                "output":
                    str(
                        output_path
                    ),
            }
        )


    return pd.DataFrame(
        rows
    )
'''


MODULE_PATH = (
    ENRICHMENT_DIR
    / "report_sections.py"
)


MODULE_PATH.write_text(
    textwrap.dedent(
        MODULE_CODE
    ),
    encoding="utf-8",
)


print()
print(
    "Written:",
    MODULE_PATH
)


# ============================================================
# 3. TESTS
# ============================================================

TEST_CODE = r'''
import pandas as pd

from politdata.enrichment.report_sections import (
    report_period_label,
    enrich_report_section_frame,
)


def _context():

    return pd.DataFrame(
        [
            {
                "source_report_id":
                    "r1",

                "organization_name_current":
                    "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»",

                "organization_code":
                    "12345678",

                "organization_level":
                    "central",

                "region":
                    "Україна",

                "party_name_current":
                    "ТЕСТ",

                "party_code":
                    "12345678",

                "analysis_override":
                    False,

                "analysis_selection_method":
                    "single_signed",

                "official_selected_report_id":
                    "r1",

                "analysis_selected_report_id":
                    "r1",

                "continuity_exact":
                    False,

                "is_latest_report_for_organization":
                    True,
            }
        ]
    )


def test_report_period_quarters():

    assert (
        report_period_label(
            2025,
            1,
        )
        ==
        "2025Q1"
    )

    assert (
        report_period_label(
            2025,
            4,
        )
        ==
        "2025Q4"
    )


def test_report_period_annual():

    assert (
        report_period_label(
            2025,
            5,
        )
        ==
        "2025"
    )


def test_generic_report_section_enrichment():

    normalized = pd.DataFrame(
        [
            {
                "source_report_id":
                    "r1",

                "source_row_index":
                    0,

                "report_year":
                    2025,

                "report_quarter":
                    1,

                "source__id":
                    "asset-1",

                "source__object_type":
                    "Квартира",
            }
        ]
    )


    result = (
        enrich_report_section_frame(
            normalized,
            report_context=
                _context(),
        )
    )


    row = result.iloc[0]


    assert (
        row[
            "source__object_type"
        ]
        ==
        "Квартира"
    )

    assert (
        row[
            "organization_name_current"
        ]
        ==
        "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»"
    )

    assert (
        row[
            "party_name_current"
        ]
        ==
        "ТЕСТ"
    )

    assert (
        bool(
            row[
                "analysis_selected"
            ]
        )
        is True
    )

    assert (
        bool(
            row[
                "official_selected"
            ]
        )
        is True
    )

    assert (
        bool(
            row[
                "is_latest_data"
            ]
        )
        is True
    )

    assert (
        row[
            "data_recency_status"
        ]
        ==
        "latest_data"
    )

    assert (
        row[
            "report_period"
        ]
        ==
        "2025Q1"
    )


def test_nonofficial_analytical_report():

    context = _context()

    context.loc[
        0,
        "official_selected_report_id",
    ] = "official-other"


    normalized = pd.DataFrame(
        [
            {
                "source_report_id":
                    "r1",

                "source_row_index":
                    0,

                "report_year":
                    2025,

                "report_quarter":
                    2,
            }
        ]
    )


    result = (
        enrich_report_section_frame(
            normalized,
            report_context=
                context,
        )
    )


    row = result.iloc[0]


    assert (
        bool(
            row[
                "analysis_selected"
            ]
        )
        is True
    )

    assert (
        bool(
            row[
                "official_selected"
            ]
        )
        is False
    )


def test_historical_data():

    context = _context()

    context.loc[
        0,
        "is_latest_report_for_organization",
    ] = False


    normalized = pd.DataFrame(
        [
            {
                "source_report_id":
                    "r1",

                "source_row_index":
                    0,

                "report_year":
                    2024,

                "report_quarter":
                    5,
            }
        ]
    )


    result = (
        enrich_report_section_frame(
            normalized,
            report_context=
                context,
        )
    )


    row = result.iloc[0]


    assert (
        bool(
            row[
                "is_latest_data"
            ]
        )
        is False
    )

    assert (
        row[
            "data_recency_status"
        ]
        ==
        "historical_data"
    )

    assert (
        row[
            "report_period"
        ]
        ==
        "2024"
    )


def test_empty_section_preserves_enriched_schema():

    normalized = pd.DataFrame(
        {
            "source_report_id":
                pd.Series(
                    dtype="string"
                ),

            "source_section":
                pd.Series(
                    dtype="string"
                ),

            "source_row_index":
                pd.Series(
                    dtype="Int64"
                ),

            "report_year":
                pd.Series(
                    dtype="Int64"
                ),

            "report_quarter":
                pd.Series(
                    dtype="Int64"
                ),
        }
    )


    result = (
        enrich_report_section_frame(
            normalized,
            report_context=
                _context(),
        )
    )


    assert result.empty

    assert (
        "report_period"
        in result.columns
    )

    assert (
        "is_latest_data"
        in result.columns
    )
'''


TEST_PATH = (
    TESTS_DIR
    / "test_report_section_enrichment.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 4. RUN COMPLETE TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:
    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Report-section enrichment tests failed. "
        "Validation rebuild was not started."
    )


# ============================================================
# 5. FRESH IMPORT
# ============================================================

import politdata.enrichment.report_sections as report_sections


importlib.reload(
    report_sections
)


# ============================================================
# 6. CLEAN VALIDATION DIRECTORY
# ============================================================

if VALIDATION_ROOT.exists():

    shutil.rmtree(
        VALIDATION_ROOT
    )


VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 7. FULL NORMALIZED -> ENRICHED REBUILD
# ============================================================

print()
print("=" * 100)
print("REPORT SECTION NORMALIZED -> ENRICHED REBUILD")
print("=" * 100)


rebuild_summary = (
    report_sections
    .enrich_report_sections_directory(
        NORMALIZED_ROOT,
        VALIDATION_ROOT,

        report_context=
            REPORT_CONTEXT_PATH,

        overwrite=True,
    )
)


display(
    rebuild_summary
)


# ============================================================
# 8. FULL KEYED PRODUCTION PARITY
# ============================================================

comparison_rows = []
mismatch_samples = []


for section, relative_path in (
    SECTION_PATHS.items()
):

    current = pd.read_parquet(
        CURRENT_ENRICHED_ROOT
        / relative_path
    )

    rebuilt = pd.read_parquet(
        VALIDATION_ROOT
        / relative_path
    )


    expected_rows = (
        EXPECTED_ROWS[
            section
        ]
    )


    if len(current) != expected_rows:
        raise RuntimeError(
            f"{section}: current baseline changed."
        )


    if len(rebuilt) != expected_rows:
        raise RuntimeError(
            f"{section}: rebuilt row count changed."
        )


    current_columns = set(
        current.columns
    )

    rebuilt_columns = set(
        rebuilt.columns
    )


    missing = sorted(
        current_columns
        -
        rebuilt_columns
    )

    extra = sorted(
        rebuilt_columns
        -
        current_columns
    )


    if (
        missing
        or
        extra
    ):

        raise RuntimeError(
            f"{section}: schema mismatch. "
            f"Missing={missing}; "
            f"extra={extra}"
        )


    if len(current) == 0:

        comparison_rows.append(
            {
                "section":
                    section,

                "rows":
                    0,

                "columns":
                    len(
                        current.columns
                    ),

                "mismatching_columns":
                    0,

                "field_level_mismatches":
                    0,
            }
        )

        continue


    key_columns = (
        section_key_columns(
            section
        )
    )


    current_key = (
        make_key(
            current,
            key_columns,
        )
    )

    rebuilt_key = (
        make_key(
            rebuilt,
            key_columns,
        )
    )


    if (
        not current_key.is_unique
        or
        not rebuilt_key.is_unique
    ):

        raise RuntimeError(
            f"{section}: row key is not unique."
        )


    current = (
        current
        .assign(
            _key=current_key
        )
        .set_index(
            "_key"
        )
        .sort_index()
    )


    rebuilt = (
        rebuilt
        .assign(
            _key=rebuilt_key
        )
        .set_index(
            "_key"
        )
        .sort_index()
    )


    if not current.index.equals(
        rebuilt.index
    ):

        raise RuntimeError(
            f"{section}: row-key sets differ."
        )


    bad_columns = []
    total_mismatches = 0


    for column in sorted(
        current_columns
    ):

        count = compare_series(
            current[
                column
            ],
            rebuilt[
                column
            ],
        )


        total_mismatches += (
            count
        )


        if count:

            bad_columns.append(
                f"{column} ({count})"
            )


            old = (
                current[
                    column
                ]
                .astype("string")
                .fillna("<NULL>")
            )

            new = (
                rebuilt[
                    column
                ]
                .astype("string")
                .fillna("<NULL>")
            )


            mask = (
                old != new
            )


            for idx in (
                mask[
                    mask
                ]
                .index[:5]
            ):

                mismatch_samples.append(
                    {
                        "section":
                            section,

                        "row_key":
                            idx,

                        "column":
                            column,

                        "current":
                            old.loc[idx],

                        "rebuilt":
                            new.loc[idx],
                    }
                )


    comparison_rows.append(
        {
            "section":
                section,

            "rows":
                len(current),

            "columns":
                len(
                    current_columns
                ),

            "mismatching_columns":
                len(
                    bad_columns
                ),

            "field_level_mismatches":
                total_mismatches,
        }
    )


comparison_df = pd.DataFrame(
    comparison_rows
)


print()
print("=" * 100)
print("FULL REPORT SECTION PARITY")
print("=" * 100)

display(
    comparison_df
)


# ============================================================
# 9. FINAL CONTROL
# ============================================================

total_rows = int(
    comparison_df[
        "rows"
    ].sum()
)


total_mismatches = int(
    comparison_df[
        "field_level_mismatches"
    ].sum()
)


print()
print(
    "Rows rebuilt:",
    f"{total_rows:,}"
)

print(
    "Field-level mismatches:",
    f"{total_mismatches:,}"
)


if total_rows != 308_657:

    raise RuntimeError(
        f"Expected 308,657 rows, "
        f"got {total_rows:,}"
    )


if total_mismatches:

    print()
    print("=" * 100)
    print("MISMATCH SAMPLES")
    print("=" * 100)

    display(
        pd.DataFrame(
            mismatch_samples
        ).head(
            100
        )
    )


    raise RuntimeError(
        "Report-section rebuild does not have "
        "full production parity."
    )


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "All ten report sections can now be rebuilt "
    "directly from normalized_v0_1."
)

print(
    "Validation output:"
)

print(
    VALIDATION_ROOT
)

print()
print(
    "Production enriched files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)


VERIFY FINAL THREE ENRICHMENT RULES


,section,rows,analysis_selected_mismatches,official_selected_mismatches,report_period_mismatches,total_mismatches
0,realty,11325,0,0,0,0
1,transport,1325,0,0,0,0
2,movable,693,0,0,0,0
3,intangible,8614,0,0,0,0
4,paper,0,0,0,0,0
5,obligations,26132,0,0,0,0
6,head_info,78791,0,0,0,0
7,employee_counts,78791,0,0,0,0
8,organizations,308,0,0,0,0
9,regional_offices,102678,0,0,0,0


Total rule mismatches: 0

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\enrichment\report_sections.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_report_section_enrichment.py

TESTS
........................................................................ [ 63%]
..........................................                               [100%]
114 passed in 1.83s

Return code: 0

REPORT SECTION NORMALIZED -> ENRICHED REBUILD


,section,rows,columns,output
0,realty,11325,44,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\properties\realty.parquet
1,transport,1325,43,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\properties\transport.parquet
2,movable,693,42,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\properties\movable.parquet
3,intangible,8614,41,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\properties\intangible.parquet
4,paper,0,28,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\properties\paper.parquet
5,obligations,26132,43,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\obligations\obligations.parquet
6,head_info,78791,32,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\report_state\head_info.parquet
7,employee_counts,78791,32,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\report_state\employee_counts.parquet
8,organizations,308,31,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\report_state\organizations.parquet
9,regional_offices,102678,45,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1\report_state\regional_offices.parquet



FULL REPORT SECTION PARITY


,section,rows,columns,mismatching_columns,field_level_mismatches
0,realty,11325,44,0,0
1,transport,1325,43,0,0
2,movable,693,42,0,0
3,intangible,8614,41,0,0
4,paper,0,28,0,0
5,obligations,26132,43,0,0
6,head_info,78791,32,0,0
7,employee_counts,78791,32,0,0
8,organizations,308,31,0,0
9,regional_offices,102678,45,0,0



Rows rebuilt: 308,657
Field-level mismatches: 0

DONE
All ten report sections can now be rebuilt directly from normalized_v0_1.
Validation output:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\report_sections_from_normalized_v0_1

Production enriched files were NOT modified.
RAW files were NOT read.
API was NOT called.


In [138]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

SRC_DIR = (
    ROOT
    / "src"
    / "politdata"
)

TESTS_DIR = (
    ROOT
    / "tests"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

REFERENCE_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

VALIDATION_ROOT = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "full_enrichment_pipeline_v0_1"
)


# ============================================================
# 1. CENTRAL QA MODULE
# ============================================================

QA_CODE = r'''
from __future__ import annotations

from pathlib import Path
from typing import Mapping

import pandas as pd
import pyarrow.parquet as pq


PAYMENT_EXPECTED_ROWS = {
    "monetary_contributions": 27_234,
    "other_contributions": 6_168,
    "state_funding": 96,
    "other_incomes": 19_007,
    "budget_expenses": 29_482,
    "outgoing_expenses": 319_901,
    "return_expenses": 137,
    "transfer_expenses": 3,
}


REPORT_SECTION_EXPECTED_ROWS = {
    "realty": 11_325,
    "transport": 1_325,
    "movable": 693,
    "intangible": 8_614,
    "paper": 0,
    "obligations": 26_132,
    "head_info": 78_791,
    "employee_counts": 78_791,
    "organizations": 308,
    "regional_offices": 102_678,
}


PAYMENT_PATHS = {
    section:
        Path("payments")
        / f"{section}.parquet"

    for section
    in PAYMENT_EXPECTED_ROWS
}


REPORT_SECTION_PATHS = {
    "realty":
        Path("properties/realty.parquet"),

    "transport":
        Path("properties/transport.parquet"),

    "movable":
        Path("properties/movable.parquet"),

    "intangible":
        Path("properties/intangible.parquet"),

    "paper":
        Path("properties/paper.parquet"),

    "obligations":
        Path("obligations/obligations.parquet"),

    "head_info":
        Path("report_state/head_info.parquet"),

    "employee_counts":
        Path("report_state/employee_counts.parquet"),

    "organizations":
        Path("report_state/organizations.parquet"),

    "regional_offices":
        Path("report_state/regional_offices.parquet"),
}


REFERENCE_IDENTITY_COLUMNS = (
    "organization_code",
    "organization_level",
    "organization_name_current",
    "party_code",
    "party_name_current",
    "region",
)


def parquet_row_count(
    path,
) -> int:

    return int(
        pq.ParquetFile(
            path
        )
        .metadata
        .num_rows
    )


def validate_expected_counts(
    actual: Mapping[str, int],
    expected: Mapping[str, int],
) -> None:
    """
    Raise if any expected dataset count changed.
    """

    missing = (
        set(expected)
        -
        set(actual)
    )


    extra = (
        set(actual)
        -
        set(expected)
    )


    if missing or extra:

        raise RuntimeError(
            "Dataset set changed. "
            f"Missing={sorted(missing)}; "
            f"extra={sorted(extra)}"
        )


    bad = {
        name: {
            "expected": int(expected[name]),
            "actual": int(actual[name]),
        }

        for name in expected

        if (
            int(actual[name])
            !=
            int(expected[name])
        )
    }


    if bad:

        raise RuntimeError(
            "Regression row-count baseline changed: "
            f"{bad}"
        )


def collect_enriched_row_counts(
    output_root,
) -> pd.DataFrame:

    output_root = Path(
        output_root
    )


    rows = []


    for section, relative_path in (
        PAYMENT_PATHS.items()
    ):

        path = (
            output_root
            / relative_path
        )


        if not path.exists():

            raise FileNotFoundError(
                path
            )


        rows.append(
            {
                "layer":
                    "payments",

                "section":
                    section,

                "rows":
                    parquet_row_count(
                        path
                    ),

                "expected_rows":
                    PAYMENT_EXPECTED_ROWS[
                        section
                    ],
            }
        )


    for section, relative_path in (
        REPORT_SECTION_PATHS.items()
    ):

        path = (
            output_root
            / relative_path
        )


        if not path.exists():

            raise FileNotFoundError(
                path
            )


        rows.append(
            {
                "layer":
                    "report_sections",

                "section":
                    section,

                "rows":
                    parquet_row_count(
                        path
                    ),

                "expected_rows":
                    REPORT_SECTION_EXPECTED_ROWS[
                        section
                    ],
            }
        )


    result = pd.DataFrame(
        rows
    )


    payment_actual = {
        row[
            "section"
        ]:
            int(
                row[
                    "rows"
                ]
            )

        for row
        in rows

        if (
            row[
                "layer"
            ]
            ==
            "payments"
        )
    }


    section_actual = {
        row[
            "section"
        ]:
            int(
                row[
                    "rows"
                ]
            )

        for row
        in rows

        if (
            row[
                "layer"
            ]
            ==
            "report_sections"
        )
    }


    validate_expected_counts(
        payment_actual,
        PAYMENT_EXPECTED_ROWS,
    )


    validate_expected_counts(
        section_actual,
        REPORT_SECTION_EXPECTED_ROWS,
    )


    result[
        "matches_baseline"
    ] = (
        result[
            "rows"
        ]
        ==
        result[
            "expected_rows"
        ]
    )


    return result


def _compare_series(
    left,
    right,
) -> int:

    left = (
        left
        .astype("string")
        .fillna("<NULL>")
    )

    right = (
        right
        .astype("string")
        .fillna("<NULL>")
    )


    return int(
        (
            left
            !=
            right
        ).sum()
    )


def validate_payment_reference_identity(
    payment_root,
    organization_reference,
) -> pd.DataFrame:
    """
    Confirm universal organization/party identity fields
    in enriched payments exactly match the current
    organization reference.

    This specifically protects the contract:

        organization_name_current
            = full organization name

        party_name_current
            = short unified party name
    """

    payment_root = Path(
        payment_root
    )


    if isinstance(
        organization_reference,
        (
            str,
            Path,
        ),
    ):

        organization_reference = (
            pd.read_parquet(
                organization_reference
            )
        )


    required = {
        "organization_id",
        *REFERENCE_IDENTITY_COLUMNS,
    }


    missing = (
        required
        -
        set(
            organization_reference.columns
        )
    )


    if missing:

        raise KeyError(
            "organization_reference missing columns: "
            f"{sorted(missing)}"
        )


    if (
        organization_reference[
            "organization_id"
        ]
        .duplicated()
        .any()
    ):

        raise ValueError(
            "organization_reference organization_id "
            "must be unique."
        )


    reference = (
        organization_reference[
            [
                "organization_id",
                *REFERENCE_IDENTITY_COLUMNS,
            ]
        ]
        .rename(
            columns={
                column:
                    f"_reference_{column}"

                for column
                in REFERENCE_IDENTITY_COLUMNS
            }
        )
    )


    rows = []


    for section in PAYMENT_EXPECTED_ROWS:

        path = (
            payment_root
            / f"{section}.parquet"
        )


        payments = pd.read_parquet(
            path,
            columns=[
                "organization_id",
                *REFERENCE_IDENTITY_COLUMNS,
            ],
        )


        joined = payments.merge(
            reference,
            on="organization_id",
            how="left",
            validate="many_to_one",
            sort=False,
        )


        unresolved = int(
            joined[
                "_reference_organization_level"
            ]
            .isna()
            .sum()
        )


        if unresolved:

            raise RuntimeError(
                f"{section}: {unresolved:,} payment rows "
                "did not resolve organization_reference."
            )


        for column in REFERENCE_IDENTITY_COLUMNS:

            mismatches = (
                _compare_series(
                    joined[
                        column
                    ],
                    joined[
                        f"_reference_{column}"
                    ],
                )
            )


            rows.append(
                {
                    "section":
                        section,

                    "column":
                        column,

                    "rows":
                        len(joined),

                    "mismatches":
                        mismatches,
                }
            )


    result = pd.DataFrame(
        rows
    )


    total_mismatches = int(
        result[
            "mismatches"
        ].sum()
    )


    if total_mismatches:

        raise RuntimeError(
            "Payment organization-reference identity "
            f"has {total_mismatches:,} field-level mismatches."
        )


    return result


def validate_enriched_output(
    output_root,
    *,
    organization_reference,
):
    """
    Top-level QA for the currently consolidated
    enrichment layers.
    """

    output_root = Path(
        output_root
    )


    counts = (
        collect_enriched_row_counts(
            output_root
        )
    )


    payment_identity = (
        validate_payment_reference_identity(
            output_root
            / "payments",

            organization_reference=
                organization_reference,
        )
    )


    return {
        "row_counts":
            counts,

        "payment_reference_identity":
            payment_identity,
    }
'''


QA_PATH = (
    SRC_DIR
    / "qa.py"
)


QA_PATH.write_text(
    textwrap.dedent(
        QA_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    QA_PATH
)


# ============================================================
# 2. SHORT ENRICHMENT ORCHESTRATOR
# ============================================================

PIPELINE_CODE = r'''
from __future__ import annotations

from pathlib import Path

from politdata.enrichment.payment_batch import (
    enrich_normalized_payment_directory,
)

from politdata.enrichment.report_sections import (
    enrich_report_sections_directory,
)

from politdata.qa import (
    validate_enriched_output,
)


def rebuild_enriched_data_layers(
    normalized_root,
    output_root,
    *,
    reference_root,
    overwrite: bool = False,
):
    """
    Rebuild the currently productionized enrichment layers
    from normalized parquet plus the verified reference layer.

    Included:
        - all eight payment sections
        - all ten report-state/snapshot sections

    Required prebuilt references:
        - report_context
        - organization_reference
        - report_account_reference
        - state_funding_account_reference

    This function intentionally does NOT:
        - read RAW
        - call the API
        - rebuild the reference layer
        - promote output to production automatically
    """

    normalized_root = Path(
        normalized_root
    )

    output_root = Path(
        output_root
    )

    reference_root = Path(
        reference_root
    )


    report_context_path = (
        reference_root
        / "report_context.parquet"
    )

    organization_reference_path = (
        reference_root
        / "organization_reference.parquet"
    )

    report_account_reference_path = (
        reference_root
        / "report_account_reference.parquet"
    )

    state_account_reference_path = (
        reference_root
        / "state_funding_account_reference.parquet"
    )


    required_references = (
        report_context_path,
        organization_reference_path,
        report_account_reference_path,
        state_account_reference_path,
    )


    for path in required_references:

        if not path.exists():

            raise FileNotFoundError(
                path
            )


    # --------------------------------------------------------
    # PAYMENTS
    # --------------------------------------------------------

    payment_summary = (
        enrich_normalized_payment_directory(
            normalized_root
            / "payments",

            output_root
            / "payments",

            report_context=
                report_context_path,

            organization_reference=
                organization_reference_path,

            report_account_reference=
                report_account_reference_path,

            state_account_reference=
                state_account_reference_path,

            overwrite=
                overwrite,
        )
    )


    # --------------------------------------------------------
    # REPORT SECTIONS
    # --------------------------------------------------------

    section_summary = (
        enrich_report_sections_directory(
            normalized_root,
            output_root,

            report_context=
                report_context_path,

            overwrite=
                overwrite,
        )
    )


    # --------------------------------------------------------
    # CENTRAL REGRESSION QA
    # --------------------------------------------------------

    qa = validate_enriched_output(
        output_root,

        organization_reference=
            organization_reference_path,
    )


    return {
        "payments":
            payment_summary,

        "report_sections":
            section_summary,

        "qa":
            qa,
    }
'''


PIPELINE_PATH = (
    SRC_DIR
    / "pipeline.py"
)


PIPELINE_PATH.write_text(
    textwrap.dedent(
        PIPELINE_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    PIPELINE_PATH
)


# ============================================================
# 3. SMALL CHARACTERIZATION TESTS
# ============================================================

TEST_CODE = r'''
import pytest

from politdata.qa import (
    PAYMENT_EXPECTED_ROWS,
    REPORT_SECTION_EXPECTED_ROWS,
    validate_expected_counts,
)


def test_payment_regression_baseline_total():

    assert (
        sum(
            PAYMENT_EXPECTED_ROWS.values()
        )
        ==
        402_028
    )


def test_report_section_regression_baseline_total():

    assert (
        sum(
            REPORT_SECTION_EXPECTED_ROWS.values()
        )
        ==
        308_657
    )


def test_validate_expected_counts_accepts_exact():

    validate_expected_counts(
        {
            "a": 1,
            "b": 2,
        },
        {
            "a": 1,
            "b": 2,
        },
    )


def test_validate_expected_counts_rejects_change():

    with pytest.raises(
        RuntimeError
    ):

        validate_expected_counts(
            {
                "a": 1,
                "b": 99,
            },
            {
                "a": 1,
                "b": 2,
            },
        )
'''


TEST_PATH = (
    TESTS_DIR
    / "test_pipeline_qa.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 4. RUN COMPLETE TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Consolidated rebuild was not started."
    )


# ============================================================
# 5. FRESH IMPORT
# ============================================================

import politdata.qa as qa
import politdata.pipeline as pipeline


importlib.reload(
    qa
)

importlib.reload(
    pipeline
)


# ============================================================
# 6. CLEAN ONLY VALIDATION OUTPUT
# ============================================================

if VALIDATION_ROOT.exists():

    shutil.rmtree(
        VALIDATION_ROOT
    )


VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 7. ONE-CALL CONSOLIDATED REBUILD
# ============================================================

print()
print("=" * 100)
print("CONSOLIDATED ENRICHMENT REBUILD")
print("=" * 100)


result = (
    pipeline
    .rebuild_enriched_data_layers(
        NORMALIZED_ROOT,
        VALIDATION_ROOT,

        reference_root=
            REFERENCE_ROOT,

        overwrite=True,
    )
)


# ============================================================
# 8. OUTPUT SUMMARIES
# ============================================================

print()
print("=" * 100)
print("PAYMENTS")
print("=" * 100)

display(
    result[
        "payments"
    ]
)


print()
print("=" * 100)
print("REPORT SECTIONS")
print("=" * 100)

display(
    result[
        "report_sections"
    ]
)


print()
print("=" * 100)
print("REGRESSION ROW COUNTS")
print("=" * 100)

display(
    result[
        "qa"
    ][
        "row_counts"
    ]
)


print()
print("=" * 100)
print("PAYMENT ORGANIZATION REFERENCE QA")
print("=" * 100)


payment_reference_qa = (
    result[
        "qa"
    ][
        "payment_reference_identity"
    ]
)


reference_summary = (
    payment_reference_qa
    .groupby(
        "column",
        as_index=False,
    )
    .agg(
        rows=(
            "rows",
            "sum",
        ),

        mismatches=(
            "mismatches",
            "sum",
        ),

        sections=(
            "section",
            "nunique",
        ),
    )
)


display(
    reference_summary
)


# ============================================================
# 9. FINAL CONTROLS
# ============================================================

payment_rows = int(
    result[
        "payments"
    ][
        "rows"
    ].sum()
)


section_rows = int(
    result[
        "report_sections"
    ][
        "rows"
    ].sum()
)


reference_mismatches = int(
    payment_reference_qa[
        "mismatches"
    ].sum()
)


print()
print(
    "Payment rows:",
    f"{payment_rows:,}"
)

print(
    "Report-section rows:",
    f"{section_rows:,}"
)

print(
    "Payment reference mismatches:",
    f"{reference_mismatches:,}"
)


if payment_rows != 402_028:

    raise RuntimeError(
        "Payment regression baseline changed."
    )


if section_rows != 308_657:

    raise RuntimeError(
        "Report-section regression baseline changed."
    )


if reference_mismatches != 0:

    raise RuntimeError(
        "Payment identity/reference QA failed."
    )


print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Consolidated enrichment orchestration is green."
)

print()
print(
    "Production modules:"
)

print(
    "  src/politdata/pipeline.py"
)

print(
    "  src/politdata/qa.py"
)

print()
print(
    "Validation output:"
)

print(
    VALIDATION_ROOT
)

print()
print(
    "Production parquet files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\qa.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\pipeline.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_pipeline_qa.py

TESTS
........................................................................ [ 61%]
..............................................                           [100%]
118 passed in 8.99s

Return code: 0

CONSOLIDATED ENRICHMENT REBUILD

PAYMENTS


,section,rows,columns,output
0,monetary_contributions,27234,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\monetary_contributions.parquet
1,other_contributions,6168,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\other_contributions.parquet
2,state_funding,96,127,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\state_funding.parquet
3,other_incomes,19007,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\other_incomes.parquet
4,budget_expenses,29482,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\budget_expenses.parquet
5,outgoing_expenses,319901,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\outgoing_expenses.parquet
6,return_expenses,137,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\return_expenses.parquet
7,transfer_expenses,3,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\payments\transfer_expenses.parquet



REPORT SECTIONS


,section,rows,columns,output
0,realty,11325,44,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\properties\realty.parquet
1,transport,1325,43,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\properties\transport.parquet
2,movable,693,42,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\properties\movable.parquet
3,intangible,8614,41,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\properties\intangible.parquet
4,paper,0,28,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\properties\paper.parquet
5,obligations,26132,43,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\obligations\obligations.parquet
6,head_info,78791,32,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\report_state\head_info.parquet
7,employee_counts,78791,32,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\report_state\employee_counts.parquet
8,organizations,308,31,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\report_state\organizations.parquet
9,regional_offices,102678,45,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1\report_state\regional_offices.parquet



REGRESSION ROW COUNTS


,layer,section,rows,expected_rows,matches_baseline
0,payments,monetary_contributions,27234,27234,True
1,payments,other_contributions,6168,6168,True
2,payments,state_funding,96,96,True
3,payments,other_incomes,19007,19007,True
4,payments,budget_expenses,29482,29482,True
5,payments,outgoing_expenses,319901,319901,True
6,payments,return_expenses,137,137,True
7,payments,transfer_expenses,3,3,True
8,report_sections,realty,11325,11325,True
9,report_sections,transport,1325,1325,True



PAYMENT ORGANIZATION REFERENCE QA


,column,rows,mismatches,sections
0,organization_code,402028,0,8
1,organization_level,402028,0,8
2,organization_name_current,402028,0,8
3,party_code,402028,0,8
4,party_name_current,402028,0,8
5,region,402028,0,8



Payment rows: 402,028
Report-section rows: 308,657
Payment reference mismatches: 0

DONE
Consolidated enrichment orchestration is green.

Production modules:
  src/politdata/pipeline.py
  src/politdata/qa.py

Validation output:
C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_enrichment_pipeline_v0_1

Production parquet files were NOT modified.
RAW files were NOT read.
API was NOT called.


In [139]:
from pathlib import Path
from collections import defaultdict, deque
import ast

import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

SRC_ROOT = (
    ROOT
    / "src"
    / "politdata"
)

TEST_ROOT = (
    ROOT
    / "tests"
)

NOTEBOOK_ROOT = (
    ROOT
    / "notebooks"
)

DATA_ROOT = (
    ROOT
    / "data"
)


# ============================================================
# HELPERS
# ============================================================

def module_name_from_path(path):

    relative = (
        path
        .relative_to(
            SRC_ROOT.parent
        )
        .with_suffix("")
    )

    parts = list(
        relative.parts
    )

    if (
        parts
        and
        parts[-1]
        ==
        "__init__"
    ):

        parts = parts[:-1]

    return ".".join(
        parts
    )


def safe_parse(path):

    text = path.read_text(
        encoding="utf-8"
    )

    try:

        tree = ast.parse(
            text
        )

    except SyntaxError as exc:

        raise RuntimeError(
            f"Could not parse {path}: {exc}"
        )


    return (
        text,
        tree,
    )


def internal_imports(tree):

    imports = set()


    for node in ast.walk(
        tree
    ):

        if isinstance(
            node,
            ast.Import
        ):

            for alias in node.names:

                if (
                    alias.name
                    ==
                    "politdata"
                    or
                    alias.name.startswith(
                        "politdata."
                    )
                ):

                    imports.add(
                        alias.name
                    )


        elif isinstance(
            node,
            ast.ImportFrom
        ):

            if (
                node.module
                and
                (
                    node.module
                    ==
                    "politdata"
                    or
                    node.module.startswith(
                        "politdata."
                    )
                )
            ):

                imports.add(
                    node.module
                )


    return sorted(
        imports
    )


def function_inventory(tree):

    rows = []


    for node in ast.walk(
        tree
    ):

        if isinstance(
            node,
            (
                ast.FunctionDef,
                ast.AsyncFunctionDef,
            )
        ):

            end_lineno = getattr(
                node,
                "end_lineno",
                node.lineno,
            )


            rows.append(
                {
                    "function":
                        node.name,

                    "start_line":
                        node.lineno,

                    "end_line":
                        end_lineno,

                    "lines":
                        (
                            end_lineno
                            -
                            node.lineno
                            +
                            1
                        ),
                }
            )


    return rows


def parquet_row_count(path):

    try:

        return int(
            pq.ParquetFile(
                path
            )
            .metadata
            .num_rows
        )

    except Exception:

        return None


# ============================================================
# 1. SOURCE MODULE INVENTORY
# ============================================================

module_rows = []

import_graph = {}

function_rows = []


source_files = sorted(
    SRC_ROOT.rglob(
        "*.py"
    )
)


for path in source_files:

    text, tree = safe_parse(
        path
    )


    module = module_name_from_path(
        path
    )


    imports = internal_imports(
        tree
    )


    import_graph[
        module
    ] = imports


    functions = function_inventory(
        tree
    )


    classes = [
        node
        for node in ast.walk(
            tree
        )
        if isinstance(
            node,
            ast.ClassDef
        )
    ]


    layer = "other"


    if (
        module
        in {
            "politdata.pipeline",
            "politdata.qa",
        }
    ):

        layer = (
            "orchestration_qa"
        )


    elif module.startswith(
        "politdata.enrichment"
    ):

        layer = (
            "enrichment"
        )


    elif module.startswith(
        "politdata.normalization"
    ):

        layer = (
            "normalization"
        )


    elif module in {
        "politdata.api",
        "politdata.discovery",
        "politdata.change_detection",
        "politdata.refresh",
        "politdata.sync",
        "politdata.report_discovery",
        "politdata.reports",
        "politdata.report_details",
    }:

        layer = (
            "api_discovery_sync"
        )


    module_rows.append(
        {
            "module":
                module,

            "layer":
                layer,

            "lines":
                len(
                    text.splitlines()
                ),

            "functions":
                len(
                    functions
                ),

            "classes":
                len(
                    classes
                ),

            "internal_imports":
                len(
                    imports
                ),

            "path":
                str(
                    path.relative_to(
                        ROOT
                    )
                ),
        }
    )


    for function in functions:

        function_rows.append(
            {
                "module":
                    module,

                **function,
            }
        )


modules_df = (
    pd.DataFrame(
        module_rows
    )
    .sort_values(
        [
            "layer",
            "lines",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 110)
print("SOURCE MODULE INVENTORY")
print("=" * 110)

display(
    modules_df
)


print(
    "Source modules:",
    len(
        modules_df
    )
)

print(
    "Source lines:",
    f"{int(modules_df['lines'].sum()):,}"
)


# ============================================================
# 2. TEST INVENTORY / DIRECT MODULE COVERAGE
# ============================================================

test_rows = []

tested_modules = defaultdict(
    set
)


for path in sorted(
    TEST_ROOT.glob(
        "test_*.py"
    )
):

    text, tree = safe_parse(
        path
    )


    tests = [
        node
        for node in ast.walk(
            tree
        )
        if (
            isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                )
            )
            and
            node.name.startswith(
                "test_"
            )
        )
    ]


    imports = internal_imports(
        tree
    )


    for module in imports:

        tested_modules[
            module
        ].add(
            path.name
        )


    test_rows.append(
        {
            "test_file":
                path.name,

            "tests":
                len(
                    tests
                ),

            "lines":
                len(
                    text.splitlines()
                ),

            "politdata_imports":
                " | ".join(
                    imports
                ),
        }
    )


tests_df = pd.DataFrame(
    test_rows
)


print()
print("=" * 110)
print("TEST INVENTORY")
print("=" * 110)

display(
    tests_df
)


print(
    "Test functions:",
    f"{int(tests_df['tests'].sum()):,}"
)


# ============================================================
# 3. STATIC REACHABILITY FROM NEW pipeline.py
# ============================================================

all_modules = set(
    import_graph
)


def normalize_import_target(
    imported,
):

    if imported in all_modules:

        return imported


    # Import may point to a package rather than exact module.
    matches = [
        module
        for module in all_modules
        if module.startswith(
            imported
            +
            "."
        )
    ]


    return (
        imported
        if imported in all_modules
        else None
    )


def reachable_from(
    root_module,
):

    visited = set()

    queue = deque(
        [
            root_module
        ]
    )


    while queue:

        module = queue.popleft()


        if module in visited:
            continue


        visited.add(
            module
        )


        for imported in (
            import_graph.get(
                module,
                []
            )
        ):

            target = normalize_import_target(
                imported
            )


            if (
                target
                and
                target
                not in visited
            ):

                queue.append(
                    target
                )


    return visited


pipeline_reachable = (
    reachable_from(
        "politdata.pipeline"
    )
)


coverage_rows = []


for row in module_rows:

    module = row[
        "module"
    ]


    direct_test_files = sorted(
        tested_modules.get(
            module,
            set(),
        )
    )


    coverage_rows.append(
        {
            "module":
                module,

            "layer":
                row[
                    "layer"
                ],

            "lines":
                row[
                    "lines"
                ],

            "reachable_from_pipeline":
                (
                    module
                    in
                    pipeline_reachable
                ),

            "directly_imported_by_tests":
                bool(
                    direct_test_files
                ),

            "test_files":
                " | ".join(
                    direct_test_files
                ),
        }
    )


coverage_df = (
    pd.DataFrame(
        coverage_rows
    )
    .sort_values(
        [
            "reachable_from_pipeline",
            "directly_imported_by_tests",
            "lines",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)


print()
print("=" * 110)
print("MODULE REACHABILITY / TEST COVERAGE")
print("=" * 110)

display(
    coverage_df
)


# ============================================================
# 4. LARGEST FUNCTIONS OUTSIDE NEW ENRICHMENT STACK
# ============================================================

functions_df = pd.DataFrame(
    function_rows
)


if len(
    functions_df
):

    functions_df[
        "consolidated_enrichment"
    ] = (
        functions_df[
            "module"
        ].eq(
            "politdata.pipeline"
        )
        |
        functions_df[
            "module"
        ].eq(
            "politdata.qa"
        )
        |
        functions_df[
            "module"
        ].str.startswith(
            "politdata.enrichment."
        )
    )


    remaining_large = (
        functions_df[
            ~functions_df[
                "consolidated_enrichment"
            ]
        ]
        .sort_values(
            "lines",
            ascending=False,
        )
        .head(
            30
        )
    )


    print()
    print("=" * 110)
    print("LARGEST FUNCTIONS OUTSIDE CONSOLIDATED ENRICHMENT")
    print("=" * 110)

    display(
        remaining_large
    )


# ============================================================
# 5. SEARCH FOR PRODUCERS OF CRITICAL REFERENCE ARTIFACTS
#
# This is the key question before the next refactor:
# are these references reproducibly built in src,
# or is some logic still notebook-only?
# ============================================================

SEARCH_TERMS = [
    "organization_reference",
    "report_context",
    "report_account_reference",
    "organization_account_reference",
    "state_funding_account_reference",

    "build_organization_reference",
    "build_report_context",

    "positive_state_receipt",
    "state_funding_account_confirmed",
]


producer_rows = []


for path in source_files:

    text = path.read_text(
        encoding="utf-8"
    )


    module = module_name_from_path(
        path
    )


    lines = text.splitlines()


    for term in SEARCH_TERMS:

        matches = [
            index + 1
            for index, line
            in enumerate(
                lines
            )
            if term in line
        ]


        if matches:

            producer_rows.append(
                {
                    "term":
                        term,

                    "module":
                        module,

                    "matches":
                        len(
                            matches
                        ),

                    "first_lines":
                        ", ".join(
                            str(value)
                            for value
                            in matches[:10]
                        ),
                }
            )


producer_df = pd.DataFrame(
    producer_rows
)


print()
print("=" * 110)
print("REFERENCE PRODUCER SEARCH")
print("=" * 110)


if len(
    producer_df
):

    display(
        producer_df.sort_values(
            [
                "term",
                "module",
            ]
        )
    )

else:

    print(
        "No producer-related terms found in src."
    )


# ============================================================
# 6. CRITICAL EXISTING PARQUET ARTIFACTS
#
# Metadata only — no full parquet reads.
# ============================================================

CRITICAL_NAMES = {
    "analysis_selected_reports_manifest.parquet",
    "selected_reports_manifest.parquet",
    "report_period_selection_final.parquet",

    "report_detail_state.parquet",
    "report_section_normalization_state.parquet",

    "organization_reference.parquet",
    "report_context.parquet",

    "report_account_reference.parquet",
    "organization_account_reference.parquet",
    "state_funding_account_reference.parquet",

    "property_moneys.parquet",
}


artifact_rows = []


for path in DATA_ROOT.rglob(
    "*.parquet"
):

    if path.name not in CRITICAL_NAMES:
        continue


    artifact_rows.append(
        {
            "file":
                path.name,

            "rows":
                parquet_row_count(
                    path
                ),

            "size_mb":
                round(
                    path.stat().st_size
                    /
                    1024
                    /
                    1024,
                    2,
                ),

            "path":
                str(
                    path.relative_to(
                        ROOT
                    )
                ),
        }
    )


artifacts_df = (
    pd.DataFrame(
        artifact_rows
    )
    .sort_values(
        [
            "file",
            "path",
        ]
    )
    if artifact_rows
    else
    pd.DataFrame()
)


print()
print("=" * 110)
print("CRITICAL DATA ARTIFACTS")
print("=" * 110)


if len(
    artifacts_df
):

    display(
        artifacts_df
    )

else:

    print(
        "No critical parquet artifacts found."
    )


# ============================================================
# 7. NOTEBOOK INVENTORY
#
# Do not parse/execute notebook cells.
# Just identify what remains as development history.
# ============================================================

notebook_rows = []


if NOTEBOOK_ROOT.exists():

    for path in sorted(
        NOTEBOOK_ROOT.rglob(
            "*.ipynb"
        )
    ):

        notebook_rows.append(
            {
                "notebook":
                    str(
                        path.relative_to(
                            ROOT
                        )
                    ),

                "size_mb":
                    round(
                        path.stat().st_size
                        /
                        1024
                        /
                        1024,
                        2,
                    ),
            }
        )


notebooks_df = pd.DataFrame(
    notebook_rows
)


print()
print("=" * 110)
print("NOTEBOOK INVENTORY")
print("=" * 110)


if len(
    notebooks_df
):

    display(
        notebooks_df
    )

else:

    print(
        "No notebooks found."
    )


# ============================================================
# 8. STATIC REMAINDER SUMMARY
# ============================================================

remaining_modules = coverage_df[
    ~coverage_df[
        "reachable_from_pipeline"
    ]
].copy()


priority_order = {
    "normalization": 1,
    "api_discovery_sync": 2,
    "other": 3,
    "orchestration_qa": 4,
    "enrichment": 5,
}


remaining_modules[
    "_priority"
] = (
    remaining_modules[
        "layer"
    ]
    .map(
        priority_order
    )
    .fillna(
        99
    )
)


remaining_modules = (
    remaining_modules
    .sort_values(
        [
            "_priority",
            "lines",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .drop(
        columns=[
            "_priority"
        ]
    )
)


print()
print("=" * 110)
print("MODULES NOT YET REACHED BY NEW ENRICHMENT ENTRY POINT")
print("=" * 110)

display(
    remaining_modules
)


# ============================================================
# 9. HIGH-LEVEL COUNTS
# ============================================================

print()
print("=" * 110)
print("SUMMARY")
print("=" * 110)

print(
    "Source modules:",
    len(
        modules_df
    )
)

print(
    "Modules reachable from politdata.pipeline:",
    len(
        pipeline_reachable
        &
        set(
            modules_df[
                "module"
            ]
        )
    )
)

print(
    "Modules outside new pipeline reachability:",
    len(
        remaining_modules
    )
)

print(
    "Directly test-imported modules:",
    int(
        coverage_df[
            "directly_imported_by_tests"
        ].sum()
    )
)

print(
    "Critical parquet artifacts found:",
    len(
        artifacts_df
    )
)

print()
print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No production files were modified."
)


SOURCE MODULE INVENTORY


,module,layer,lines,functions,classes,internal_imports,path
0,politdata.report_details,api_discovery_sync,1110,15,0,0,src\politdata\report_details.py
1,politdata.report_discovery,api_discovery_sync,975,7,0,0,src\politdata\report_discovery.py
2,politdata.reports,api_discovery_sync,824,6,0,0,src\politdata\reports.py
3,politdata.sync,api_discovery_sync,776,6,0,0,src\politdata\sync.py
4,politdata.discovery,api_discovery_sync,336,5,0,0,src\politdata\discovery.py
5,politdata.refresh,api_discovery_sync,329,4,0,0,src\politdata\refresh.py
6,politdata.change_detection,api_discovery_sync,170,4,0,0,src\politdata\change_detection.py
7,politdata.api,api_discovery_sync,125,3,0,0,src\politdata\api.py
8,politdata.enrichment.payment_resolution,enrichment,1153,9,0,1,src\politdata\enrichment\payment_resolution.py
9,politdata.enrichment.payment_batch,enrichment,1056,12,0,3,src\politdata\enrichment\payment_batch.py


Source modules: 23
Source lines: 14,367

TEST INVENTORY


,test_file,tests,lines,politdata_imports
0,test_accounts.py,15,409,politdata.normalization.accounts
1,test_change_detection.py,4,103,politdata.change_detection
2,test_discovery.py,2,156,politdata.discovery
3,test_payment_base.py,3,299,politdata.enrichment.payment_base
4,test_payment_batch.py,4,480,politdata.enrichment.payment_batch
5,test_payment_enrichment.py,16,374,politdata.enrichment.payments
6,test_payment_resolution.py,5,196,politdata.enrichment.payment_resolution
7,test_payments.py,6,150,politdata.normalization.payments
8,test_pipeline_qa.py,4,62,politdata.qa
9,test_property_moneys.py,12,193,politdata.normalization.property_moneys


Test functions: 102

MODULE REACHABILITY / TEST COVERAGE


...


NOTEBOOK INVENTORY


,notebook,size_mb
0,notebooks\exploration\.ipynb_checkpoints\01_politdata_api_exploration-checkpoint.ipynb,6.06
1,notebooks\exploration\01_politdata_api_exploration.ipynb,7.03



MODULES NOT YET REACHED BY NEW ENRICHMENT ENTRY POINT


,module,layer,lines,reachable_from_pipeline,directly_imported_by_tests,test_files
12,politdata.normalization.payments,normalization,1861,False,True,test_payments.py
14,politdata.normalization.reference,normalization,1070,False,True,test_reference.py
13,politdata.normalization.property_moneys,normalization,872,False,True,test_property_moneys.py
11,politdata.normalization.accounts,normalization,760,False,True,test_accounts.py
15,politdata.normalization.report_sections,normalization,379,False,True,test_report_sections.py
10,politdata.normalization,normalization,13,False,False,
19,politdata.report_details,api_discovery_sync,1110,False,True,test_report_details.py
20,politdata.report_discovery,api_discovery_sync,975,False,False,
21,politdata.reports,api_discovery_sync,824,False,True,test_reports.py
22,politdata.sync,api_discovery_sync,776,False,False,



SUMMARY
Source modules: 23
Modules reachable from politdata.pipeline: 7
Modules outside new pipeline reachability: 16
Directly test-imported modules: 15
Critical parquet artifacts found: 12

RAW files were NOT read.
API was NOT called.
No production files were modified.


In [140]:
from pathlib import Path
import inspect

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

ENRICHED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
)

REFERENCE_ROOT = (
    ENRICHED_ROOT
    / "reference"
)

PROPERTY_MONEYS_PATH = (
    ENRICHED_ROOT
    / "properties"
    / "property_moneys.parquet"
)

STATE_FUNDING_PATH = (
    NORMALIZED_ROOT
    / "payments"
    / "state_funding.parquet"
)

REPORT_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_ROOT
    / "report_account_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_ROOT
    / "state_funding_account_reference.parquet"
)

ORGANIZATION_REFERENCE_PATH = (
    REFERENCE_ROOT
    / "organization_reference.parquet"
)


# ============================================================
# HELPERS
# ============================================================

def clean_string(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value or None


def first_non_null(series):
    for value in series:
        if pd.notna(value):
            return value
    return pd.NA


def compare_reference_frames(
    current,
    candidate,
    *,
    key_columns,
    label,
):
    current = current.copy()
    candidate = candidate.copy()

    if current.duplicated(key_columns).any():
        raise RuntimeError(
            f"{label}: current reference key is not unique."
        )

    if candidate.duplicated(key_columns).any():
        raise RuntimeError(
            f"{label}: candidate reference key is not unique."
        )

    current_keys = set(
        map(
            tuple,
            current[key_columns]
            .astype("string")
            .fillna("<NULL>")
            .to_numpy(),
        )
    )

    candidate_keys = set(
        map(
            tuple,
            candidate[key_columns]
            .astype("string")
            .fillna("<NULL>")
            .to_numpy(),
        )
    )

    print()
    print("=" * 110)
    print(label)
    print("=" * 110)

    print("Current rows:", f"{len(current):,}")
    print("Candidate rows:", f"{len(candidate):,}")
    print(
        "Missing candidate keys:",
        f"{len(current_keys - candidate_keys):,}"
    )
    print(
        "Extra candidate keys:",
        f"{len(candidate_keys - current_keys):,}"
    )

    if current_keys != candidate_keys:
        return None

    merged = current.merge(
        candidate,
        on=key_columns,
        how="outer",
        validate="one_to_one",
        suffixes=("_current", "_candidate"),
    )

    common_value_columns = [
        column
        for column in current.columns
        if (
            column not in key_columns
            and
            column in candidate.columns
        )
    ]

    rows = []

    for column in common_value_columns:
        left = (
            merged[f"{column}_current"]
            .astype("string")
            .fillna("<NULL>")
        )

        right = (
            merged[f"{column}_candidate"]
            .astype("string")
            .fillna("<NULL>")
        )

        mismatch = left != right

        rows.append(
            {
                "column": column,
                "rows": len(merged),
                "mismatches": int(mismatch.sum()),
                "match_ratio": (
                    1
                    -
                    int(mismatch.sum())
                    /
                    len(merged)
                    if len(merged)
                    else 1.0
                ),
            }
        )

    result = pd.DataFrame(rows)

    display(result)

    print(
        "Total field-level mismatches:",
        f"{int(result['mismatches'].sum()):,}"
    )

    return result


# ============================================================
# 1. CONFIRM EXISTING REFERENCE BUILDERS
# ============================================================

from politdata.normalization.reference import (
    build_organization_reference,
    build_report_context,
)

print()
print("=" * 110)
print("EXISTING REFERENCE BUILDERS")
print("=" * 110)

print(
    "build_organization_reference",
    inspect.signature(build_organization_reference),
)

print(
    "build_report_context",
    inspect.signature(build_report_context),
)


# ============================================================
# 2. REPORT ACCOUNT REFERENCE
#
# Test whether it is simply a report-relative aggregation
# of enriched property_moneys.
#
# No RAW involved.
# ============================================================

property_moneys = pd.read_parquet(
    PROPERTY_MONEYS_PATH
)

current_report_account = pd.read_parquet(
    REPORT_ACCOUNT_REFERENCE_PATH
)


required_account_columns = {
    "source_report_id",
    "organization_id",
    "root_party_id",
    "party_account_iban",
    "party_account_iban_source",
    "party_account_type_source",
    "party_account_type_analytical",
    "party_account_type_resolution_method",
}


missing = (
    required_account_columns
    -
    set(property_moneys.columns)
)

if missing:
    raise RuntimeError(
        "property_moneys missing required columns: "
        f"{sorted(missing)}"
    )


report_account_keys = [
    "source_report_id",
    "organization_id",
    "root_party_id",
    "party_account_iban",
]


# ------------------------------------------------------------
# Confirm that descriptive fields are functionally unique
# inside each report/account key.
# ------------------------------------------------------------

descriptive_columns = [
    "party_account_iban_source",
    "party_account_type_source",
    "party_account_type_analytical",
    "party_account_type_resolution_method",
]


conflict_rows = []

for column in descriptive_columns:
    conflicts = (
        property_moneys
        .groupby(
            report_account_keys,
            dropna=False,
        )[column]
        .nunique(dropna=True)
    )

    count = int(
        (conflicts > 1).sum()
    )

    conflict_rows.append(
        {
            "column": column,
            "conflicting_keys": count,
        }
    )


conflict_df = pd.DataFrame(
    conflict_rows
)

print()
print("=" * 110)
print("REPORT ACCOUNT FUNCTIONAL DEPENDENCIES")
print("=" * 110)

display(conflict_df)


if int(
    conflict_df[
        "conflicting_keys"
    ].sum()
) != 0:
    raise RuntimeError(
        "Report-account descriptive fields are not "
        "functionally unique. Stop instead of guessing."
    )


candidate_report_account = (
    property_moneys
    .groupby(
        report_account_keys,
        dropna=False,
        as_index=False,
    )
    .agg(
        party_account_iban_source=(
            "party_account_iban_source",
            first_non_null,
        ),

        party_account_type_source=(
            "party_account_type_source",
            first_non_null,
        ),

        party_account_type_analytical=(
            "party_account_type_analytical",
            first_non_null,
        ),

        party_account_type_resolution_method=(
            "party_account_type_resolution_method",
            first_non_null,
        ),

        snapshot_rows=(
            "party_account_iban",
            "size",
        ),
    )
)


report_account_parity = compare_reference_frames(
    current_report_account,
    candidate_report_account,
    key_columns=report_account_keys,
    label="REPORT ACCOUNT REFERENCE PARITY",
)


# ============================================================
# 3. STATE FUNDING ACCOUNT REFERENCE
#
# Build only from:
#   normalized state_funding transactions
#   + organization_reference identity
#
# Important:
# positive receipt in actual state_funding section
# is the evidence.
# ============================================================

from politdata.enrichment.payment_resolution import (
    classify_state_funding_form,
)


state_funding = pd.read_parquet(
    STATE_FUNDING_PATH
)

organization_reference = pd.read_parquet(
    ORGANIZATION_REFERENCE_PATH
)

current_state_account = pd.read_parquet(
    STATE_ACCOUNT_REFERENCE_PATH
)


required_state_columns = {
    "organization_id",
    "root_party_id",
    "receiver_account_iban_canonical",
    "payment_amount",
    "payment_operation_date",
    "payment_type_detail_source",
}


missing = (
    required_state_columns
    -
    set(state_funding.columns)
)

if missing:
    raise RuntimeError(
        "normalized state_funding missing columns: "
        f"{sorted(missing)}"
    )


state = state_funding.copy()

state[
    "payment_amount"
] = pd.to_numeric(
    state[
        "payment_amount"
    ],
    errors="coerce",
)


state = state[
    state[
        "payment_amount"
    ].gt(0)
    &
    state[
        "receiver_account_iban_canonical"
    ].notna()
].copy()


state = state.rename(
    columns={
        "receiver_account_iban_canonical":
            "party_account_iban",
    }
)


state[
    "_state_funding_form_code"
] = (
    state[
        "payment_type_detail_source"
    ]
    .map(
        classify_state_funding_form
    )
)


if (
    state[
        "_state_funding_form_code"
    ]
    .isna()
    .any()
):
    raise RuntimeError(
        "A positive state_funding transaction has "
        "an unclassified funding form."
    )


org_identity = (
    organization_reference[
        [
            "organization_id",
            "party_name_current",
            "organization_name_current",
            "organization_level",
        ]
    ]
    .copy()
)


if (
    org_identity[
        "organization_id"
    ]
    .duplicated()
    .any()
):
    raise RuntimeError(
        "organization_reference organization_id "
        "is not unique."
    )


state = state.merge(
    org_identity,
    on="organization_id",
    how="left",
    validate="many_to_one",
)


state_keys = [
    "root_party_id",
    "organization_id",
    "party_account_iban",
]


def unique_join(series):
    values = sorted(
        {
            str(value).strip()
            for value in series
            if pd.notna(value)
            and str(value).strip()
        }
    )

    if not values:
        return pd.NA

    return " | ".join(values)


def unique_single(series):
    values = [
        value
        for value in pd.unique(
            series.dropna()
        )
    ]

    if len(values) == 0:
        return pd.NA

    if len(values) > 1:
        raise RuntimeError(
            f"Expected one unique value, got {values}"
        )

    return values[0]


candidate_state_account = (
    state
    .groupby(
        state_keys,
        dropna=False,
        as_index=False,
    )
    .agg(
        party_name_current=(
            "party_name_current",
            unique_single,
        ),

        organization_name_current=(
            "organization_name_current",
            unique_single,
        ),

        organization_level=(
            "organization_level",
            unique_single,
        ),

        first_state_receipt_date=(
            "payment_operation_date",
            "min",
        ),

        last_state_receipt_date=(
            "payment_operation_date",
            "max",
        ),

        positive_state_receipt_rows=(
            "payment_amount",
            "size",
        ),

        positive_state_receipt_amount=(
            "payment_amount",
            "sum",
        ),

        state_funding_form_count=(
            "_state_funding_form_code",
            "nunique",
        ),

        state_funding_forms_observed=(
            "_state_funding_form_code",
            unique_join,
        ),

        state_funding_source_forms_observed=(
            "payment_type_detail_source",
            unique_join,
        ),

        state_funding_form_code=(
            "_state_funding_form_code",
            unique_single,
        ),
    )
)


candidate_state_account[
    "state_funding_account_confirmed"
] = True

candidate_state_account[
    "state_funding_account_evidence"
] = (
    "positive_transaction_in_state_funding_section"
)


# ============================================================
# 4. STATE REFERENCE PARITY
# ============================================================

state_account_parity = compare_reference_frames(
    current_state_account,
    candidate_state_account,
    key_columns=state_keys,
    label="STATE FUNDING ACCOUNT REFERENCE PARITY",
)


# ============================================================
# 5. SHOW ANY DIFFERENT STATE-FORM REPRESENTATION
#
# Only seven rows, so if one textual aggregation field differs
# we can see the exact representation immediately.
# ============================================================

if state_account_parity is not None:
    state_mismatches = int(
        state_account_parity[
            "mismatches"
        ].sum()
    )
else:
    state_mismatches = None


if state_mismatches:
    print()
    print("=" * 110)
    print("CURRENT STATE FUNDING REFERENCE")
    print("=" * 110)

    display(
        current_state_account[
            [
                "root_party_id",
                "organization_id",
                "party_account_iban",
                "state_funding_form_count",
                "state_funding_forms_observed",
                "state_funding_source_forms_observed",
                "state_funding_form_code",
                "state_funding_account_evidence",
            ]
        ]
    )

    print()
    print("=" * 110)
    print("CANDIDATE STATE FUNDING REFERENCE")
    print("=" * 110)

    display(
        candidate_state_account[
            [
                "root_party_id",
                "organization_id",
                "party_account_iban",
                "state_funding_form_count",
                "state_funding_forms_observed",
                "state_funding_source_forms_observed",
                "state_funding_form_code",
                "state_funding_account_evidence",
            ]
        ]
    )


# ============================================================
# 6. REFERENCE-LAYER STATUS
# ============================================================

report_account_mismatches = (
    int(
        report_account_parity[
            "mismatches"
        ].sum()
    )
    if report_account_parity is not None
    else None
)


print()
print("=" * 110)
print("REFERENCE GAP STATUS")
print("=" * 110)


status = pd.DataFrame(
    [
        {
            "artifact":
                "organization_reference",

            "producer_in_src":
                True,

            "required_by_pipeline":
                True,

            "status":
                "existing build_organization_reference",
        },
        {
            "artifact":
                "report_context",

            "producer_in_src":
                True,

            "required_by_pipeline":
                True,

            "status":
                "existing build_report_context",
        },
        {
            "artifact":
                "report_account_reference",

            "producer_in_src":
                False,

            "required_by_pipeline":
                True,

            "candidate_mismatches":
                report_account_mismatches,

            "status":
                (
                    "deterministic candidate confirmed"
                    if report_account_mismatches == 0
                    else
                    "needs local repair"
                ),
        },
        {
            "artifact":
                "state_funding_account_reference",

            "producer_in_src":
                False,

            "required_by_pipeline":
                True,

            "candidate_mismatches":
                state_mismatches,

            "status":
                (
                    "deterministic candidate confirmed"
                    if state_mismatches == 0
                    else
                    "needs local repair"
                ),
        },
        {
            "artifact":
                "organization_account_reference",

            "producer_in_src":
                False,

            "required_by_pipeline":
                False,

            "status":
                "analytical artifact; non-blocking",
        },
    ]
)


display(status)


print()
print(
    "Positive state_funding rows:",
    f"{len(state):,}"
)

print(
    "Confirmed state-funding accounts:",
    f"{len(candidate_state_account):,}"
)

print()
print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No parquet files were modified."
)


EXISTING REFERENCE BUILDERS
build_organization_reference (organizations: 'pd.DataFrame', addresses: 'pd.DataFrame') -> 'pd.DataFrame'
build_report_context (analysis_manifest: 'pd.DataFrame', organization_reference: 'pd.DataFrame') -> 'pd.DataFrame'

REPORT ACCOUNT FUNCTIONAL DEPENDENCIES


,column,conflicting_keys
0,party_account_iban_source,0
1,party_account_type_source,2
2,party_account_type_analytical,1
3,party_account_type_resolution_method,1


RuntimeError: Report-account descriptive fields are not functionally unique. Stop instead of guessing.

In [141]:
from pathlib import Path

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

ENRICHED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
)

REFERENCE_ROOT = (
    ENRICHED_ROOT
    / "reference"
)

PROPERTY_MONEYS_PATH = (
    ENRICHED_ROOT
    / "properties"
    / "property_moneys.parquet"
)

STATE_FUNDING_PATH = (
    NORMALIZED_ROOT
    / "payments"
    / "state_funding.parquet"
)

REPORT_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_ROOT
    / "report_account_reference.parquet"
)

STATE_ACCOUNT_REFERENCE_PATH = (
    REFERENCE_ROOT
    / "state_funding_account_reference.parquet"
)

ORGANIZATION_REFERENCE_PATH = (
    REFERENCE_ROOT
    / "organization_reference.parquet"
)


# ============================================================
# HELPERS
# ============================================================

def string_series(series):

    return (
        series
        .astype("string")
        .fillna("<NULL>")
    )


def mismatch_count(
    left,
    right,
):

    return int(
        (
            string_series(left)
            !=
            string_series(right)
        ).sum()
    )


def unique_join(series):

    values = sorted(
        {
            str(value).strip()
            for value in series
            if (
                pd.notna(value)
                and
                str(value).strip()
            )
        }
    )

    if not values:
        return pd.NA

    return " | ".join(
        values
    )


def single_unique(series):

    values = list(
        pd.unique(
            series.dropna()
        )
    )

    if not values:
        return pd.NA

    if len(values) != 1:

        raise RuntimeError(
            f"Expected one value, got {values}"
        )

    return values[0]


def compare_reference(
    current,
    candidate,
    *,
    keys,
    label,
):

    current = current.copy()
    candidate = candidate.copy()


    if current.duplicated(keys).any():

        raise RuntimeError(
            f"{label}: current key not unique."
        )


    if candidate.duplicated(keys).any():

        raise RuntimeError(
            f"{label}: candidate key not unique."
        )


    left_keys = set(
        map(
            tuple,
            current[
                keys
            ]
            .astype("string")
            .fillna("<NULL>")
            .to_numpy(),
        )
    )


    right_keys = set(
        map(
            tuple,
            candidate[
                keys
            ]
            .astype("string")
            .fillna("<NULL>")
            .to_numpy(),
        )
    )


    missing = len(
        left_keys
        -
        right_keys
    )

    extra = len(
        right_keys
        -
        left_keys
    )


    common_columns = [
        column
        for column
        in current.columns
        if (
            column
            not in keys
            and
            column in candidate.columns
        )
    ]


    if (
        missing
        or
        extra
    ):

        return {
            "strategy":
                label,

            "current_rows":
                len(current),

            "candidate_rows":
                len(candidate),

            "missing_keys":
                missing,

            "extra_keys":
                extra,

            "compared_columns":
                len(
                    common_columns
                ),

            "field_mismatches":
                None,

            "mismatching_columns":
                "KEY SET DIFFERENCE",
        }


    joined = current.merge(
        candidate,
        on=keys,
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_current",
            "_candidate",
        ),
    )


    mismatches = 0
    bad_columns = []


    for column in common_columns:

        count = mismatch_count(
            joined[
                f"{column}_current"
            ],
            joined[
                f"{column}_candidate"
            ],
        )


        mismatches += count


        if count:

            bad_columns.append(
                f"{column} ({count})"
            )


    return {
        "strategy":
            label,

        "current_rows":
            len(current),

        "candidate_rows":
            len(candidate),

        "missing_keys":
            0,

        "extra_keys":
            0,

        "compared_columns":
            len(
                common_columns
            ),

        "field_mismatches":
            mismatches,

        "mismatching_columns":
            " | ".join(
                bad_columns
            ),
    }


# ============================================================
# 1. LOAD ONLY PROCESSED DATA
# ============================================================

property_moneys = pd.read_parquet(
    PROPERTY_MONEYS_PATH
)

current_report_ref = pd.read_parquet(
    REPORT_ACCOUNT_REFERENCE_PATH
)

state_funding = pd.read_parquet(
    STATE_FUNDING_PATH
)

current_state_ref = pd.read_parquet(
    STATE_ACCOUNT_REFERENCE_PATH
)

organization_reference = pd.read_parquet(
    ORGANIZATION_REFERENCE_PATH
)


print()
print("=" * 110)
print("INPUT")
print("=" * 110)

print(
    "property_moneys:",
    f"{len(property_moneys):,}"
)

print(
    "current report_account_reference:",
    f"{len(current_report_ref):,}"
)

print(
    "normalized state_funding:",
    f"{len(state_funding):,}"
)

print(
    "current state_funding_account_reference:",
    f"{len(current_state_ref):,}"
)


# ============================================================
# 2. REPORT ACCOUNT CONFLICTS
# ============================================================

KEYS = [
    "source_report_id",
    "organization_id",
    "root_party_id",
    "party_account_iban",
]


VALUE_COLUMNS = [
    "party_account_iban_source",
    "party_account_type_source",
    "party_account_type_analytical",
    "party_account_type_resolution_method",
]


required = set(
    KEYS
    +
    VALUE_COLUMNS
)


missing = (
    required
    -
    set(
        property_moneys.columns
    )
)


if missing:

    raise RuntimeError(
        "property_moneys missing: "
        f"{sorted(missing)}"
    )


conflict_flags = None


for column in VALUE_COLUMNS:

    counts = (
        property_moneys
        .groupby(
            KEYS,
            dropna=False,
        )[
            column
        ]
        .nunique(
            dropna=True
        )
    )


    flag = (
        counts
        >
        1
    )


    conflict_flags = (
        flag
        if conflict_flags is None
        else
        (
            conflict_flags
            |
            flag
        )
    )


conflict_index = (
    conflict_flags[
        conflict_flags
    ]
    .index
)


print()
print("=" * 110)
print("REPORT ACCOUNT CONFLICT KEYS")
print("=" * 110)

print(
    "Conflict keys:",
    len(
        conflict_index
    )
)


# ============================================================
# 3. SHOW EXACT CONFLICTING SOURCE ROWS + CURRENT RESULT
# ============================================================

property_with_order = (
    property_moneys
    .reset_index(
        names="_physical_row_order"
    )
)


conflict_key_df = pd.DataFrame(
    list(
        conflict_index
    ),
    columns=KEYS,
)


conflict_source = (
    property_with_order
    .merge(
        conflict_key_df,
        on=KEYS,
        how="inner",
        validate="many_to_many",
    )
)


show_columns = [
    *KEYS,

    "_physical_row_order",

    *[
        column
        for column
        in [
            "source_row_index",
            "source__id",
            "report_year",
            "report_quarter",
            "party_account_iban_source",
            "party_account_type_source",
            "party_account_type_analytical",
            "party_account_type_resolution_method",
        ]
        if column
        in conflict_source.columns
    ],
]


print()
print("=" * 110)
print("CONFLICTING PROPERTY_MONEYS ROWS")
print("=" * 110)

display(
    conflict_source[
        show_columns
    ]
    .sort_values(
        KEYS
        +
        (
            ["source_row_index"]
            if
            "source_row_index"
            in conflict_source.columns
            else
            ["_physical_row_order"]
        )
    )
)


print()
print("=" * 110)
print("CURRENT REPORT ACCOUNT VALUES FOR CONFLICT KEYS")
print("=" * 110)

display(
    current_report_ref
    .merge(
        conflict_key_df,
        on=KEYS,
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        KEYS
    )
)


# ============================================================
# 4. CANDIDATE REPORT-ACCOUNT REDUCERS
#
# We test several deterministic strategies against ALL
# 12,586 current reference rows.
# ============================================================

work = (
    property_with_order
    .copy()
)


# Classification priorities are ONLY used to discover
# the historical reducer.
# They do not redefine state-funding evidence.
priority_map = {
    "state_statutory_funding_account": 50,
    "state_campaign_reimbursement_account": 50,
    "state_funding_account": 50,

    "ordinary_account": 40,

    "budget_account_unspecified": 30,
    "social_insurance_account": 30,
    "deposit_account": 30,
    "transit_account": 30,
    "card_account": 30,
    "escrow_account": 30,
    "other_special_account": 30,

    "unknown": 10,
}


work[
    "_analytical_priority"
] = (
    work[
        "party_account_type_analytical"
    ]
    .map(
        priority_map
    )
    .fillna(
        0
    )
)


if (
    "source_row_index"
    in work.columns
):

    work[
        "_source_order"
    ] = pd.to_numeric(
        work[
            "source_row_index"
        ],
        errors="coerce",
    )

else:

    work[
        "_source_order"
    ] = (
        work[
            "_physical_row_order"
        ]
    )


def candidate_from_representative(
    ordered,
    *,
    keep,
):

    selected = (
        ordered
        .drop_duplicates(
            subset=KEYS,
            keep=keep,
        )
        [
            KEYS
            +
            VALUE_COLUMNS
        ]
        .copy()
    )


    counts = (
        property_moneys
        .groupby(
            KEYS,
            dropna=False,
            as_index=False,
        )
        .size()
        .rename(
            columns={
                "size":
                    "snapshot_rows"
            }
        )
    )


    return selected.merge(
        counts,
        on=KEYS,
        how="left",
        validate="one_to_one",
    )


candidates = {}


# ------------------------------------------------------------
# Physical/source order
# ------------------------------------------------------------

candidates[
    "first_physical_row"
] = (
    candidate_from_representative(
        work.sort_values(
            "_physical_row_order"
        ),
        keep="first",
    )
)


candidates[
    "last_physical_row"
] = (
    candidate_from_representative(
        work.sort_values(
            "_physical_row_order"
        ),
        keep="last",
    )
)


candidates[
    "first_source_row_index"
] = (
    candidate_from_representative(
        work.sort_values(
            [
                "_source_order",
                "_physical_row_order",
            ]
        ),
        keep="first",
    )
)


candidates[
    "last_source_row_index"
] = (
    candidate_from_representative(
        work.sort_values(
            [
                "_source_order",
                "_physical_row_order",
            ]
        ),
        keep="last",
    )
)


# ------------------------------------------------------------
# Prefer more specifically classified row
# ------------------------------------------------------------

candidates[
    "priority_then_first"
] = (
    candidate_from_representative(
        work.sort_values(
            [
                "_analytical_priority",
                "_source_order",
                "_physical_row_order",
            ],
            ascending=[
                False,
                True,
                True,
            ],
        ),
        keep="first",
    )
)


candidates[
    "priority_then_last"
] = (
    candidate_from_representative(
        work.sort_values(
            [
                "_analytical_priority",
                "_source_order",
                "_physical_row_order",
            ],
            ascending=[
                False,
                False,
                False,
            ],
        ),
        keep="first",
    )
)


# ------------------------------------------------------------
# Independent first/last non-null aggregation.
# This tests whether the historical builder resolved fields
# independently rather than selecting one source row.
# ------------------------------------------------------------

def first_non_null(series):

    non_null = series.dropna()

    if len(non_null):
        return non_null.iloc[0]

    return pd.NA


def last_non_null(series):

    non_null = series.dropna()

    if len(non_null):
        return non_null.iloc[-1]

    return pd.NA


candidates[
    "independent_first_non_null"
] = (
    work
    .sort_values(
        [
            "_source_order",
            "_physical_row_order",
        ]
    )
    .groupby(
        KEYS,
        dropna=False,
        as_index=False,
    )
    .agg(
        party_account_iban_source=(
            "party_account_iban_source",
            first_non_null,
        ),

        party_account_type_source=(
            "party_account_type_source",
            first_non_null,
        ),

        party_account_type_analytical=(
            "party_account_type_analytical",
            first_non_null,
        ),

        party_account_type_resolution_method=(
            "party_account_type_resolution_method",
            first_non_null,
        ),

        snapshot_rows=(
            "party_account_iban",
            "size",
        ),
    )
)


candidates[
    "independent_last_non_null"
] = (
    work
    .sort_values(
        [
            "_source_order",
            "_physical_row_order",
        ]
    )
    .groupby(
        KEYS,
        dropna=False,
        as_index=False,
    )
    .agg(
        party_account_iban_source=(
            "party_account_iban_source",
            last_non_null,
        ),

        party_account_type_source=(
            "party_account_type_source",
            last_non_null,
        ),

        party_account_type_analytical=(
            "party_account_type_analytical",
            last_non_null,
        ),

        party_account_type_resolution_method=(
            "party_account_type_resolution_method",
            last_non_null,
        ),

        snapshot_rows=(
            "party_account_iban",
            "size",
        ),
    )
)


# ============================================================
# 5. SCORE ALL REPORT-ACCOUNT STRATEGIES
# ============================================================

strategy_rows = []


for name, candidate in (
    candidates.items()
):

    strategy_rows.append(
        compare_reference(
            current_report_ref,
            candidate,
            keys=KEYS,
            label=name,
        )
    )


strategy_df = (
    pd.DataFrame(
        strategy_rows
    )
    .sort_values(
        [
            "field_mismatches",
            "strategy",
        ],
        na_position="last",
    )
    .reset_index(
        drop=True
    )
)


print()
print("=" * 110)
print("REPORT ACCOUNT REDUCER STRATEGY PARITY")
print("=" * 110)

display(
    strategy_df
)


exact_strategies = (
    strategy_df[
        strategy_df[
            "field_mismatches"
        ]
        .eq(0)
        &
        strategy_df[
            "missing_keys"
        ]
        .eq(0)
        &
        strategy_df[
            "extra_keys"
        ]
        .eq(0)
    ][
        "strategy"
    ]
    .tolist()
)


print(
    "Exact reducer strategies:",
    exact_strategies
)


# ============================================================
# 6. DOES CURRENT CONFLICT RESULT CORRESPOND TO ONE
#    ACTUAL PROPERTY_MONEYS SOURCE ROW?
# ============================================================

current_conflicts = (
    current_report_ref
    .merge(
        conflict_key_df,
        on=KEYS,
        how="inner",
        validate="one_to_one",
    )
)


match_rows = []


for _, current_row in (
    current_conflicts.iterrows()
):

    mask = pd.Series(
        True,
        index=property_with_order.index,
    )


    for key in KEYS:

        if pd.isna(
            current_row[
                key
            ]
        ):

            mask &= (
                property_with_order[
                    key
                ].isna()
            )

        else:

            mask &= (
                property_with_order[
                    key
                ].astype("string")
                ==
                str(
                    current_row[
                        key
                    ]
                )
            )


    source_group = (
        property_with_order[
            mask
        ]
    )


    for _, source_row in (
        source_group.iterrows()
    ):

        same = True


        for column in VALUE_COLUMNS:

            left = (
                "<NULL>"
                if pd.isna(
                    current_row[
                        column
                    ]
                )
                else
                str(
                    current_row[
                        column
                    ]
                )
            )


            right = (
                "<NULL>"
                if pd.isna(
                    source_row[
                        column
                    ]
                )
                else
                str(
                    source_row[
                        column
                    ]
                )
            )


            if left != right:

                same = False
                break


        if same:

            match_rows.append(
                {
                    **{
                        key:
                            current_row[
                                key
                            ]
                        for key
                        in KEYS
                    },

                    "matching_physical_row":
                        int(
                            source_row[
                                "_physical_row_order"
                            ]
                        ),

                    "matching_source_row_index":
                        (
                            source_row[
                                "source_row_index"
                            ]
                            if
                            "source_row_index"
                            in source_row.index
                            else
                            pd.NA
                        ),
                }
            )


print()
print("=" * 110)
print("CURRENT CONFLICT VALUES MATCH SOURCE ROW")
print("=" * 110)


if match_rows:

    display(
        pd.DataFrame(
            match_rows
        )
    )

else:

    print(
        "Current conflict values are composite; "
        "they do not correspond to one source row."
    )


# ============================================================
# 7. STATE FUNDING ACCOUNT REFERENCE
#
# Independent from property_moneys classification.
# Evidence = positive transaction in actual state_funding
# section.
# ============================================================

required_state = {
    "root_party_id",
    "organization_id",
    "receiver_account_iban_canonical",
    "payment_amount",
    "payment_operation_date",
    "payment_type_detail_source",
}


missing = (
    required_state
    -
    set(
        state_funding.columns
    )
)


if missing:

    raise RuntimeError(
        "state_funding missing: "
        f"{sorted(missing)}"
    )


state = state_funding.copy()


state[
    "payment_amount"
] = pd.to_numeric(
    state[
        "payment_amount"
    ],
    errors="coerce",
)


state = state[
    state[
        "payment_amount"
    ].gt(0)
    &
    state[
        "receiver_account_iban_canonical"
    ].notna()
].copy()


state = state.rename(
    columns={
        "receiver_account_iban_canonical":
            "party_account_iban"
    }
)


def classify_state_form(
    value,
):

    if pd.isna(value):
        return None


    text = str(
        value
    ).strip().lower()


    if (
        "статут"
        in text
    ):

        return (
            "state_statutory_funding"
        )


    if (
        "відшкод"
        in text
        or
        "вибор"
        in text
    ):

        return (
            "state_campaign_reimbursement"
        )


    return None


state[
    "_state_funding_form_code"
] = (
    state[
        "payment_type_detail_source"
    ]
    .map(
        classify_state_form
    )
)


unclassified = state[
    state[
        "_state_funding_form_code"
    ].isna()
]


if len(unclassified):

    print()
    print("=" * 110)
    print("UNCLASSIFIED POSITIVE STATE FUNDING ROWS")
    print("=" * 110)

    display(
        unclassified[
            [
                "source_report_id",
                "payment_type_detail_source",
                "payment_amount",
            ]
        ]
    )


    raise RuntimeError(
        "Positive state-funding form not classified."
    )


org_identity = (
    organization_reference[
        [
            "organization_id",
            "party_name_current",
            "organization_name_current",
            "organization_level",
        ]
    ]
    .copy()
)


if (
    org_identity[
        "organization_id"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "organization_reference organization_id "
        "is not unique."
    )


state = state.merge(
    org_identity,
    on="organization_id",
    how="left",
    validate="many_to_one",
)


STATE_KEYS = [
    "root_party_id",
    "organization_id",
    "party_account_iban",
]


candidate_state_ref = (
    state
    .groupby(
        STATE_KEYS,
        dropna=False,
        as_index=False,
    )
    .agg(
        party_name_current=(
            "party_name_current",
            single_unique,
        ),

        organization_name_current=(
            "organization_name_current",
            single_unique,
        ),

        organization_level=(
            "organization_level",
            single_unique,
        ),

        first_state_receipt_date=(
            "payment_operation_date",
            "min",
        ),

        last_state_receipt_date=(
            "payment_operation_date",
            "max",
        ),

        positive_state_receipt_rows=(
            "payment_amount",
            "size",
        ),

        positive_state_receipt_amount=(
            "payment_amount",
            "sum",
        ),

        state_funding_form_count=(
            "_state_funding_form_code",
            "nunique",
        ),

        state_funding_forms_observed=(
            "_state_funding_form_code",
            unique_join,
        ),

        state_funding_source_forms_observed=(
            "payment_type_detail_source",
            unique_join,
        ),

        state_funding_form_code=(
            "_state_funding_form_code",
            single_unique,
        ),
    )
)


candidate_state_ref[
    "state_funding_account_confirmed"
] = True


candidate_state_ref[
    "state_funding_account_evidence"
] = (
    "positive_transaction_in_state_funding_section"
)


state_parity = compare_reference(
    current_state_ref,
    candidate_state_ref,
    keys=STATE_KEYS,
    label="state_funding_reference",
)


state_parity_df = pd.DataFrame(
    [
        state_parity
    ]
)


print()
print("=" * 110)
print("STATE FUNDING ACCOUNT REFERENCE PARITY")
print("=" * 110)

display(
    state_parity_df
)


# ============================================================
# 8. IF STATE REFERENCE DIFFERS, SHOW ONLY 7 ROWS
# ============================================================

state_mismatches = (
    state_parity[
        "field_mismatches"
    ]
)


if (
    state_mismatches
    not in (
        None,
        0,
    )
):

    print()
    print("=" * 110)
    print("CURRENT STATE REFERENCE")
    print("=" * 110)

    display(
        current_state_ref
        .sort_values(
            STATE_KEYS
        )
    )


    print()
    print("=" * 110)
    print("CANDIDATE STATE REFERENCE")
    print("=" * 110)

    display(
        candidate_state_ref
        .sort_values(
            STATE_KEYS
        )
    )


# ============================================================
# 9. FINAL STATUS
# ============================================================

print()
print("=" * 110)
print("REFERENCE BUILDER DIAGNOSTIC STATUS")
print("=" * 110)

print(
    "Report-account conflict keys:",
    len(
        conflict_index
    )
)

print(
    "Exact report-account reducer strategies:",
    exact_strategies
)

print(
    "Positive state_funding rows:",
    f"{len(state):,}"
)

print(
    "Confirmed state-funding accounts:",
    f"{len(candidate_state_ref):,}"
)

print(
    "State-reference field mismatches:",
    state_mismatches,
)

print()
print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No parquet files were modified."
)


INPUT
property_moneys: 19,140
current report_account_reference: 12,586
normalized state_funding: 96
current state_funding_account_reference: 7

REPORT ACCOUNT CONFLICT KEYS
Conflict keys: 2

CONFLICTING PROPERTY_MONEYS ROWS


,source_report_id,organization_id,root_party_id,party_account_iban,_physical_row_order,source__id,report_year,report_quarter,party_account_iban_source,party_account_type_source,party_account_type_analytical,party_account_type_resolution_method
4,4ce818b0-0d18-11ef-8a6c-27bc4724684c,9b1c3511-1fcc-4091-bdcd-97930d503b65,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553282090000026047000002354,11036,f93fd57d-f171-4743-a9f1-c98d0625b18a,2024,1,UA553282090000026047000002354,Поточний рахунок(для ФСС),ordinary_account,property_moneys_declared_type
5,4ce818b0-0d18-11ef-8a6c-27bc4724684c,9b1c3511-1fcc-4091-bdcd-97930d503b65,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553282090000026047000002354,11037,caf2af74-5277-4748-a464-dd2f15155af0,2024,1,UA553282090000026047000002354,Поточний рахунок,ordinary_account,property_moneys_declared_type
6,4ce818b0-0d18-11ef-8a6c-27bc4724684c,9b1c3511-1fcc-4091-bdcd-97930d503b65,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553282090000026047000002354,11039,e691a25c-bc1f-4511-b9c3-11244b55f743,2024,1,UA553282090000026047000002354,Поточний рахунок(для ФСС),ordinary_account,property_moneys_declared_type
7,4ce818b0-0d18-11ef-8a6c-27bc4724684c,9b1c3511-1fcc-4091-bdcd-97930d503b65,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553282090000026047000002354,11040,421b657f-c54a-41ce-84a4-8843d91f84a9,2024,1,UA553282090000026047000002354,Поточний рахунок,ordinary_account,property_moneys_declared_type
0,574f8ce0-e511-11ee-a478-6126d3bbe2ef,48f8120a-a5c7-4ff2-b63b-d3942098b448,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA663052990000026003026212388,5147,ad87ffe4-2d65-4ab7-98df-677ff2a9d080,2021,1,UA663052990000026003026212388,Поточний рахунок,ordinary_account,property_moneys_declared_type
1,574f8ce0-e511-11ee-a478-6126d3bbe2ef,48f8120a-a5c7-4ff2-b63b-d3942098b448,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA663052990000026003026212388,5148,360aa949-679f-4035-82b2-0006f8c182c4,2021,1,UA663052990000026003026212388,Рахунок для соц.виплат,unknown,property_moneys_unresolved_type
2,574f8ce0-e511-11ee-a478-6126d3bbe2ef,48f8120a-a5c7-4ff2-b63b-d3942098b448,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA663052990000026003026212388,5149,f03fb692-01f4-433e-b1d4-8d7c6cfde84e,2021,1,UA663052990000026003026212388,Поточний рахунок,ordinary_account,property_moneys_declared_type
3,574f8ce0-e511-11ee-a478-6126d3bbe2ef,48f8120a-a5c7-4ff2-b63b-d3942098b448,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA663052990000026003026212388,5150,fa45bfbd-7242-48da-a2f4-e7d51d1a1296,2021,1,UA663052990000026003026212388,Рахунок для соц.виплат,unknown,property_moneys_unresolved_type



CURRENT REPORT ACCOUNT VALUES FOR CONFLICT KEYS


,source_report_id,organization_id,root_party_id,party_account_iban,party_account_iban_source,party_account_type_source,party_account_type_analytical,party_account_type_resolution_method,snapshot_rows
1,4ce818b0-0d18-11ef-8a6c-27bc4724684c,9b1c3511-1fcc-4091-bdcd-97930d503b65,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553282090000026047000002354,UA553282090000026047000002354,Поточний рахунок(для ФСС),ordinary_account,property_moneys_exact_report_org_iban,4
0,574f8ce0-e511-11ee-a478-6126d3bbe2ef,48f8120a-a5c7-4ff2-b63b-d3942098b448,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA663052990000026003026212388,UA663052990000026003026212388,Поточний рахунок,ordinary_account,property_moneys_exact_report_org_iban,4



REPORT ACCOUNT REDUCER STRATEGY PARITY


...


CURRENT STATE REFERENCE


,root_party_id,organization_id,party_account_iban,party_name_current,organization_name_current,organization_level,first_state_receipt_date,last_state_receipt_date,positive_state_receipt_rows,positive_state_receipt_amount,state_funding_form_count,state_funding_forms_observed,state_funding_source_forms_observed,state_funding_account_confirmed,state_funding_form_code,state_funding_account_evidence
0,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,UA363516290000000002604826373,ВО БАТЬКІВЩИНА,ВО БАТЬКІВЩИНА,central,2021-02-17,2022-01-21,5,86383625.0000000000,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,True,state_statutory_funding,positive_transaction_in_state_funding_section
1,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,UA723808380000026048700741786,ВО БАТЬКІВЩИНА,ВО БАТЬКІВЩИНА,central,2022-06-30,2026-04-17,17,391454215.0000000000,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,True,state_statutory_funding,positive_transaction_in_state_funding_section
2,7d9a4591-6cea-451f-8545-b211f5e7b9bd,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA393805820000026007030316815,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,central,2021-02-17,2022-10-11,16,213881715.0000000000,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,True,state_statutory_funding,positive_transaction_in_state_funding_section
3,7d9a4591-6cea-451f-8545-b211f5e7b9bd,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553805820000026001070316815,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,central,2023-02-21,2023-02-21,2,34333475.0000000000,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,True,state_statutory_funding,positive_transaction_in_state_funding_section
4,7d9a4591-6cea-451f-8545-b211f5e7b9bd,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA923805820000026002080316815,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,central,2023-05-31,2026-04-17,26,479084075.0000000000,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,True,state_statutory_funding,positive_transaction_in_state_funding_section
5,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,UA023805260000026007001059741,СЛУГА НАРОДУ,СЛУГА НАРОДУ,central,2021-02-17,2026-04-17,22,2519051560.0000000000,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,True,state_statutory_funding,positive_transaction_in_state_funding_section
6,b11839f3-bffe-4f93-ba10-3b877ed181b9,b11839f3-bffe-4f93-ba10-3b877ed181b9,UA813808050000000026002651529,ГОЛОС,ГОЛОС,central,2021-02-17,2024-04-18,5,118979325.0000000000,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,True,state_statutory_funding,positive_transaction_in_state_funding_section



CANDIDATE STATE REFERENCE


,root_party_id,organization_id,party_account_iban,party_name_current,organization_name_current,organization_level,first_state_receipt_date,last_state_receipt_date,positive_state_receipt_rows,positive_state_receipt_amount,state_funding_form_count,state_funding_forms_observed,state_funding_source_forms_observed,state_funding_form_code,state_funding_account_confirmed,state_funding_account_evidence
0,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,UA363516290000000002604826373,ВО БАТЬКІВЩИНА,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»,central,2021-02-17,2022-01-21,5,8.638362e+07,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,state_statutory_funding,True,positive_transaction_in_state_funding_section
1,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,UA723808380000026048700741786,ВО БАТЬКІВЩИНА,ПОЛІТИЧНА ПАРТІЯ «ВСЕУКРАЇНСЬКЕ ОБ'ЄДНАННЯ «БАТЬКІВЩИНА»,central,2022-06-30,2026-04-17,17,3.914542e+08,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,state_statutory_funding,True,positive_transaction_in_state_funding_section
2,7d9a4591-6cea-451f-8545-b211f5e7b9bd,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA393805820000026007030316815,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,central,2021-02-17,2022-10-11,16,2.138817e+08,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,state_statutory_funding,True,positive_transaction_in_state_funding_section
3,7d9a4591-6cea-451f-8545-b211f5e7b9bd,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553805820000026001070316815,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,central,2023-02-21,2023-02-21,2,3.433348e+07,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,state_statutory_funding,True,positive_transaction_in_state_funding_section
4,7d9a4591-6cea-451f-8545-b211f5e7b9bd,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA923805820000026002080316815,ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ,ПОЛІТИЧНА ПАРТІЯ «ЄВРОПЕЙСЬКА СОЛІДАРНІСТЬ»,central,2023-05-31,2026-04-17,26,4.790841e+08,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,state_statutory_funding,True,positive_transaction_in_state_funding_section
5,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,UA023805260000026007001059741,СЛУГА НАРОДУ,ПОЛІТИЧНА ПАРТІЯ «СЛУГА НАРОДУ»,central,2021-02-17,2026-04-17,22,2.519052e+09,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,state_statutory_funding,True,positive_transaction_in_state_funding_section
6,b11839f3-bffe-4f93-ba10-3b877ed181b9,b11839f3-bffe-4f93-ba10-3b877ed181b9,UA813808050000000026002651529,ГОЛОС,ПОЛІТИЧНА ПАРТІЯ «ГОЛОС»,central,2021-02-17,2024-04-18,5,1.189793e+08,1,state_statutory_funding,Державне фінансування статутної діяльності політичної партії,state_statutory_funding,True,positive_transaction_in_state_funding_section



REFERENCE BUILDER DIAGNOSTIC STATUS
Report-account conflict keys: 2
Exact report-account reducer strategies: []
Positive state_funding rows: 93
Confirmed state-funding accounts: 7
State-reference field mismatches: 14

RAW files were NOT read.
API was NOT called.
No parquet files were modified.


In [142]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import importlib
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

SRC_ROOT = (
    ROOT
    / "src"
    / "politdata"
)

TEST_ROOT = (
    ROOT
    / "tests"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

INTERIM_ROOT = (
    ROOT
    / "data"
    / "interim"
)

CURRENT_REFERENCE_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

VALIDATION_ROOT = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "reference_layer_v0_1"
)


ORGANIZATIONS_PATH = (
    NORMALIZED_ROOT
    / "organizations.parquet"
)

ADDRESSES_PATH = (
    NORMALIZED_ROOT
    / "organization_addresses.parquet"
)

ANALYSIS_MANIFEST_PATH = (
    INTERIM_ROOT
    / "reports"
    / "analysis_selected_reports_manifest.parquet"
)

PROPERTY_MONEYS_PATH = (
    NORMALIZED_ROOT
    / "properties"
    / "property_moneys.parquet"
)

STATE_FUNDING_PATH = (
    NORMALIZED_ROOT
    / "payments"
    / "state_funding.parquet"
)


# ============================================================
# HELPERS
# ============================================================

def string_values(series):

    return (
        series
        .astype("string")
        .fillna("<NULL>")
    )


def string_mismatch_count(
    left,
    right,
):

    return int(
        (
            string_values(left)
            !=
            string_values(right)
        ).sum()
    )


def decimal_value(value):

    if pd.isna(value):
        return None

    if isinstance(
        value,
        Decimal,
    ):
        return value

    text = str(
        value
    ).strip()

    if not text:
        return None

    text = (
        text
        .replace(
            "\u00a0",
            ""
        )
        .replace(
            " ",
            ""
        )
        .replace(
            ",",
            "."
        )
    )

    try:
        return Decimal(
            text
        )

    except InvalidOperation as exc:

        raise ValueError(
            f"Cannot parse Decimal: {value!r}"
        ) from exc


def decimal_mismatch_count(
    left,
    right,
):

    mismatches = 0

    for a, b in zip(
        left,
        right,
    ):

        da = decimal_value(
            a
        )

        db = decimal_value(
            b
        )

        if (
            da is None
            and
            db is None
        ):
            continue

        if da != db:
            mismatches += 1

    return mismatches


def first_non_null(
    series,
):

    for value in series:

        if pd.notna(value):
            return value

    return pd.NA


def unique_single(
    series,
):

    values = list(
        pd.unique(
            series.dropna()
        )
    )

    if not values:
        return pd.NA

    if len(values) != 1:

        raise RuntimeError(
            f"Expected one unique value, got {values}"
        )

    return values[0]


def unique_join(
    series,
):

    values = sorted(
        {
            str(value).strip()
            for value in series
            if (
                pd.notna(value)
                and
                str(value).strip()
            )
        }
    )

    if not values:
        return pd.NA

    return " | ".join(
        values
    )


def compare_by_key(
    current,
    candidate,
    *,
    keys,
    decimal_columns=(),
    ignore_columns=(),
):

    current = current.copy()
    candidate = candidate.copy()

    if current.duplicated(keys).any():

        raise RuntimeError(
            f"Current key not unique: {keys}"
        )

    if candidate.duplicated(keys).any():

        raise RuntimeError(
            f"Candidate key not unique: {keys}"
        )


    current_key = (
        current[
            keys
        ]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1,
        )
    )

    candidate_key = (
        candidate[
            keys
        ]
        .astype("string")
        .fillna("<NULL>")
        .agg(
            "||".join,
            axis=1,
        )
    )


    if set(
        current_key
    ) != set(
        candidate_key
    ):

        raise RuntimeError(
            "Reference key sets differ. "
            f"Missing={len(set(current_key) - set(candidate_key))}; "
            f"extra={len(set(candidate_key) - set(current_key))}"
        )


    current = (
        current
        .assign(
            _key=current_key
        )
        .set_index(
            "_key"
        )
        .sort_index()
    )

    candidate = (
        candidate
        .assign(
            _key=candidate_key
        )
        .set_index(
            "_key"
        )
        .sort_index()
    )


    common_columns = [
        column
        for column in current.columns
        if (
            column in candidate.columns
            and
            column not in keys
            and
            column not in ignore_columns
        )
    ]


    rows = []

    for column in common_columns:

        if column in decimal_columns:

            count = decimal_mismatch_count(
                current[
                    column
                ],
                candidate[
                    column
                ],
            )

        else:

            count = string_mismatch_count(
                current[
                    column
                ],
                candidate[
                    column
                ],
            )


        rows.append(
            {
                "column":
                    column,

                "mismatches":
                    count,
            }
        )


    return pd.DataFrame(
        rows
    )


# ============================================================
# 1. LOAD INPUTS
# ============================================================

organizations = pd.read_parquet(
    ORGANIZATIONS_PATH
)

addresses = pd.read_parquet(
    ADDRESSES_PATH
)

analysis_manifest = pd.read_parquet(
    ANALYSIS_MANIFEST_PATH
)

property_moneys = pd.read_parquet(
    PROPERTY_MONEYS_PATH
)

state_funding = pd.read_parquet(
    STATE_FUNDING_PATH
)


current_org_ref = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "organization_reference.parquet"
)

current_report_context = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "report_context.parquet"
)

current_report_account = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "report_account_reference.parquet"
)

current_state_account = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "state_funding_account_reference.parquet"
)


# ============================================================
# 2. EXISTING ORGANIZATION + REPORT CONTEXT BUILDERS
# ============================================================

from politdata.normalization.reference import (
    build_organization_reference,
    build_report_context,
)


candidate_org_ref = (
    build_organization_reference(
        organizations,
        addresses,
    )
)


candidate_report_context = (
    build_report_context(
        analysis_manifest,
        candidate_org_ref,
    )
)


org_parity = compare_by_key(
    current_org_ref,
    candidate_org_ref,
    keys=[
        "organization_id"
    ],
)


context_parity = compare_by_key(
    current_report_context,
    candidate_report_context,
    keys=[
        "source_report_id"
    ],
)


print()
print("=" * 110)
print("ORGANIZATION REFERENCE PARITY")
print("=" * 110)

display(
    org_parity
)

print(
    "Total mismatches:",
    f"{int(org_parity['mismatches'].sum()):,}"
)


print()
print("=" * 110)
print("REPORT CONTEXT PARITY")
print("=" * 110)

display(
    context_parity
)

print(
    "Total mismatches:",
    f"{int(context_parity['mismatches'].sum()):,}"
)


if int(
    org_parity[
        "mismatches"
    ].sum()
) != 0:

    raise RuntimeError(
        "organization_reference builder "
        "does not reproduce production."
    )


if int(
    context_parity[
        "mismatches"
    ].sum()
) != 0:

    raise RuntimeError(
        "report_context builder "
        "does not reproduce production."
    )


# ============================================================
# 3. REPORT ACCOUNT REFERENCE
#
# Historical rule recovered:
#
# - key = exact report + org + root party + canonical IBAN
# - source/descriptive values = first non-null occurrence
#   in normalized property_moneys order
# - resolution method belongs to REFERENCE RESOLUTION,
#   not to the property_moneys row:
#
#   property_moneys_exact_report_org_iban
# ============================================================

REPORT_ACCOUNT_KEYS = [
    "source_report_id",
    "organization_id",
    "root_party_id",
    "party_account_iban",
]


required = {
    *REPORT_ACCOUNT_KEYS,

    "party_account_iban_source",
    "party_account_type_source",
    "party_account_type_analytical",
}


missing = (
    required
    -
    set(
        property_moneys.columns
    )
)


if missing:

    raise RuntimeError(
        "normalized property_moneys missing: "
        f"{sorted(missing)}"
    )


candidate_report_account = (
    property_moneys
    .groupby(
        REPORT_ACCOUNT_KEYS,
        dropna=False,
        sort=False,
        as_index=False,
    )
    .agg(
        party_account_iban_source=(
            "party_account_iban_source",
            first_non_null,
        ),

        party_account_type_source=(
            "party_account_type_source",
            first_non_null,
        ),

        party_account_type_analytical=(
            "party_account_type_analytical",
            first_non_null,
        ),

        snapshot_rows=(
            "party_account_iban",
            "size",
        ),
    )
)


candidate_report_account[
    "party_account_type_resolution_method"
] = (
    "property_moneys_exact_report_org_iban"
)


# Exact production column order
candidate_report_account = (
    candidate_report_account[
        list(
            current_report_account.columns
        )
    ]
)


report_account_parity = compare_by_key(
    current_report_account,
    candidate_report_account,
    keys=REPORT_ACCOUNT_KEYS,
)


print()
print("=" * 110)
print("REPORT ACCOUNT REFERENCE PARITY")
print("=" * 110)

display(
    report_account_parity
)


report_account_mismatches = int(
    report_account_parity[
        "mismatches"
    ].sum()
)


print(
    "Total mismatches:",
    f"{report_account_mismatches:,}"
)


if report_account_mismatches != 0:

    raise RuntimeError(
        "Recovered report_account_reference rule "
        "does not have exact production parity."
    )


# ============================================================
# 4. STATE FUNDING REFERENCE
#
# IMPORTANT:
# - payment amount remains exact Decimal
# - positive row in actual state_funding section
#   = confirmed account evidence
# - organization_name_current uses FULL current
#   organization name (corrected behavior)
# ============================================================

def classify_state_funding_form(
    value,
):

    if pd.isna(value):
        return None

    text = str(
        value
    ).strip().lower()


    if "статут" in text:

        return (
            "state_statutory_funding"
        )


    if (
        "відшкод" in text
        or
        "вибор" in text
    ):

        return (
            "state_campaign_reimbursement"
        )


    return None


state = state_funding.copy()


state[
    "_amount_decimal"
] = (
    state[
        "payment_amount"
    ]
    .map(
        decimal_value
    )
)


positive_mask = (
    state[
        "_amount_decimal"
    ]
    .map(
        lambda value:
            (
                value is not None
                and
                value > 0
            )
    )
    &
    state[
        "receiver_account_iban_canonical"
    ]
    .notna()
)


state = state[
    positive_mask
].copy()


state = state.rename(
    columns={
        "receiver_account_iban_canonical":
            "party_account_iban"
    }
)


state[
    "_state_funding_form_code"
] = (
    state[
        "payment_type_detail_source"
    ]
    .map(
        classify_state_funding_form
    )
)


if (
    state[
        "_state_funding_form_code"
    ]
    .isna()
    .any()
):

    raise RuntimeError(
        "Positive state_funding row has "
        "unclassified funding form."
    )


org_identity = (
    candidate_org_ref[
        [
            "organization_id",
            "party_name_current",
            "organization_name_current",
            "organization_level",
        ]
    ]
    .copy()
)


state = state.merge(
    org_identity,
    on="organization_id",
    how="left",
    validate="many_to_one",
)


STATE_KEYS = [
    "root_party_id",
    "organization_id",
    "party_account_iban",
]


candidate_state_account = (
    state
    .groupby(
        STATE_KEYS,
        dropna=False,
        sort=False,
        as_index=False,
    )
    .agg(
        party_name_current=(
            "party_name_current",
            unique_single,
        ),

        organization_name_current=(
            "organization_name_current",
            unique_single,
        ),

        organization_level=(
            "organization_level",
            unique_single,
        ),

        first_state_receipt_date=(
            "payment_operation_date",
            "min",
        ),

        last_state_receipt_date=(
            "payment_operation_date",
            "max",
        ),

        positive_state_receipt_rows=(
            "_amount_decimal",
            "size",
        ),

        positive_state_receipt_amount=(
            "_amount_decimal",
            "sum",
        ),

        state_funding_form_count=(
            "_state_funding_form_code",
            "nunique",
        ),

        state_funding_forms_observed=(
            "_state_funding_form_code",
            unique_join,
        ),

        state_funding_source_forms_observed=(
            "payment_type_detail_source",
            unique_join,
        ),

        state_funding_form_code=(
            "_state_funding_form_code",
            unique_single,
        ),
    )
)


# Match canonical financial scale used by processed data.
candidate_state_account[
    "positive_state_receipt_amount"
] = (
    candidate_state_account[
        "positive_state_receipt_amount"
    ]
    .map(
        lambda value:
            (
                value.quantize(
                    Decimal(
                        "0.0000000000"
                    )
                )
                if
                isinstance(
                    value,
                    Decimal,
                )
                else
                value
            )
    )
)


candidate_state_account[
    "state_funding_account_confirmed"
] = True


candidate_state_account[
    "state_funding_account_evidence"
] = (
    "positive_transaction_in_state_funding_section"
)


candidate_state_account = (
    candidate_state_account[
        list(
            current_state_account.columns
        )
    ]
)


# ============================================================
# 5. STATE REFERENCE PARITY
#
# Ignore organization_name_current for first comparison,
# because current production has the already-confirmed
# short-name defect.
# ============================================================

state_parity = compare_by_key(
    current_state_account,
    candidate_state_account,
    keys=STATE_KEYS,

    decimal_columns=[
        "positive_state_receipt_amount"
    ],

    ignore_columns=[
        "organization_name_current"
    ],
)


print()
print("=" * 110)
print("STATE FUNDING REFERENCE PARITY — EXCLUDING KNOWN NAME FIX")
print("=" * 110)

display(
    state_parity
)


state_other_mismatches = int(
    state_parity[
        "mismatches"
    ].sum()
)


print(
    "Non-name mismatches:",
    f"{state_other_mismatches:,}"
)


if state_other_mismatches != 0:

    raise RuntimeError(
        "state_funding_account_reference has "
        "unexpected non-name differences."
    )


# ============================================================
# 6. VERIFY EXACTLY SEVEN CONTROLLED NAME FIXES
# ============================================================

old = (
    current_state_account
    .sort_values(
        STATE_KEYS
    )
    .reset_index(
        drop=True
    )
)


new = (
    candidate_state_account
    .sort_values(
        STATE_KEYS
    )
    .reset_index(
        drop=True
    )
)


name_mask = (
    string_values(
        old[
            "organization_name_current"
        ]
    )
    !=
    string_values(
        new[
            "organization_name_current"
        ]
    )
)


name_fix_count = int(
    name_mask.sum()
)


print()
print("=" * 110)
print("CONTROLLED STATE-REFERENCE NAME CORRECTIONS")
print("=" * 110)

display(
    pd.DataFrame(
        {
            "organization_id":
                old.loc[
                    name_mask,
                    "organization_id"
                ],

            "old_organization_name":
                old.loc[
                    name_mask,
                    "organization_name_current"
                ],

            "new_organization_name":
                new.loc[
                    name_mask,
                    "organization_name_current"
                ],

            "party_name_current":
                new.loc[
                    name_mask,
                    "party_name_current"
                ],
        }
    )
)


print(
    "Controlled name corrections:",
    name_fix_count
)


if name_fix_count != 7:

    raise RuntimeError(
        "Expected exactly 7 known state-reference "
        f"name corrections, got {name_fix_count}."
    )


# Old broken value must equal short party name.
if not (
    string_values(
        old.loc[
            name_mask,
            "organization_name_current"
        ]
    )
    .reset_index(
        drop=True
    )
    ==
    string_values(
        old.loc[
            name_mask,
            "party_name_current"
        ]
    )
    .reset_index(
        drop=True
    )
).all():

    raise RuntimeError(
        "State-reference name difference is not "
        "the known short-party-name defect."
    )


# ============================================================
# 7. WRITE PRODUCTION REFERENCE MODULE
# ============================================================

REFERENCE_CODE = r'''
from __future__ import annotations

from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

from politdata.normalization.reference import (
    build_organization_reference,
    build_report_context,
)


REPORT_ACCOUNT_RESOLUTION_METHOD = (
    "property_moneys_exact_report_org_iban"
)

STATE_FUNDING_ACCOUNT_EVIDENCE = (
    "positive_transaction_in_state_funding_section"
)


def _first_non_null(
    series: pd.Series,
):

    for value in series:

        if pd.notna(value):
            return value

    return pd.NA


def _unique_single(
    series: pd.Series,
):

    values = list(
        pd.unique(
            series.dropna()
        )
    )

    if not values:
        return pd.NA

    if len(values) != 1:

        raise ValueError(
            f"Expected one unique value, got {values}"
        )

    return values[0]


def _unique_join(
    series: pd.Series,
):

    values = sorted(
        {
            str(value).strip()
            for value in series
            if (
                pd.notna(value)
                and
                str(value).strip()
            )
        }
    )

    if not values:
        return pd.NA

    return " | ".join(
        values
    )


def _to_decimal(
    value,
):

    if pd.isna(value):
        return None

    if isinstance(
        value,
        Decimal,
    ):
        return value

    text = (
        str(value)
        .strip()
        .replace(
            "\u00a0",
            ""
        )
        .replace(
            " ",
            ""
        )
        .replace(
            ",",
            "."
        )
    )

    if not text:
        return None

    try:

        return Decimal(
            text
        )

    except InvalidOperation as exc:

        raise ValueError(
            f"Cannot parse Decimal: {value!r}"
        ) from exc


def classify_state_funding_form(
    value,
):
    """
    Classify the source UI value
    'Форма державного фінансування'.

    Actual transaction membership in state_funding,
    not this textual classification, is the account
    confirmation evidence.
    """

    if pd.isna(value):
        return None

    text = (
        str(value)
        .strip()
        .lower()
    )


    if "статут" in text:

        return (
            "state_statutory_funding"
        )


    if (
        "відшкод" in text
        or
        "вибор" in text
    ):

        return (
            "state_campaign_reimbursement"
        )


    return None


def build_report_account_reference(
    property_moneys: pd.DataFrame,
) -> pd.DataFrame:
    """
    Build report-relative account metadata from normalized
    property_moneys.

    Exact key:
        report + organization + root party + canonical IBAN.

    Descriptive conflicts are resolved by the first non-null
    source value in normalized row order.

    The resolution method describes THIS reference lookup,
    not the upstream property_moneys classifier.
    """

    keys = [
        "source_report_id",
        "organization_id",
        "root_party_id",
        "party_account_iban",
    ]


    required = {
        *keys,

        "party_account_iban_source",
        "party_account_type_source",
        "party_account_type_analytical",
    }


    missing = (
        required
        -
        set(
            property_moneys.columns
        )
    )


    if missing:

        raise KeyError(
            "property_moneys missing columns: "
            f"{sorted(missing)}"
        )


    result = (
        property_moneys
        .groupby(
            keys,
            dropna=False,
            sort=False,
            as_index=False,
        )
        .agg(
            party_account_iban_source=(
                "party_account_iban_source",
                _first_non_null,
            ),

            party_account_type_source=(
                "party_account_type_source",
                _first_non_null,
            ),

            party_account_type_analytical=(
                "party_account_type_analytical",
                _first_non_null,
            ),

            snapshot_rows=(
                "party_account_iban",
                "size",
            ),
        )
    )


    result[
        "party_account_type_resolution_method"
    ] = (
        REPORT_ACCOUNT_RESOLUTION_METHOD
    )


    return result[
        [
            "source_report_id",
            "organization_id",
            "root_party_id",
            "party_account_iban",
            "party_account_iban_source",
            "party_account_type_source",
            "party_account_type_analytical",
            "party_account_type_resolution_method",
            "snapshot_rows",
        ]
    ]


def build_state_funding_account_reference(
    state_funding: pd.DataFrame,
    organization_reference: pd.DataFrame,
) -> pd.DataFrame:
    """
    Confirm state-funding receiving accounts.

    Evidence:
        positive transaction in the actual state_funding
        transaction section.

    property_moneys account labels are NOT used as evidence.
    """

    required = {
        "root_party_id",
        "organization_id",
        "receiver_account_iban_canonical",
        "payment_amount",
        "payment_operation_date",
        "payment_type_detail_source",
    }


    missing = (
        required
        -
        set(
            state_funding.columns
        )
    )


    if missing:

        raise KeyError(
            "state_funding missing columns: "
            f"{sorted(missing)}"
        )


    org_required = {
        "organization_id",
        "party_name_current",
        "organization_name_current",
        "organization_level",
    }


    missing = (
        org_required
        -
        set(
            organization_reference.columns
        )
    )


    if missing:

        raise KeyError(
            "organization_reference missing columns: "
            f"{sorted(missing)}"
        )


    if (
        organization_reference[
            "organization_id"
        ]
        .duplicated()
        .any()
    ):

        raise ValueError(
            "organization_reference organization_id "
            "must be unique."
        )


    state = (
        state_funding
        .copy()
    )


    state[
        "_amount_decimal"
    ] = (
        state[
            "payment_amount"
        ]
        .map(
            _to_decimal
        )
    )


    positive = (
        state[
            "_amount_decimal"
        ]
        .map(
            lambda value:
                (
                    value is not None
                    and
                    value > 0
                )
        )
        &
        state[
            "receiver_account_iban_canonical"
        ]
        .notna()
    )


    state = (
        state[
            positive
        ]
        .copy()
    )


    state = state.rename(
        columns={
            "receiver_account_iban_canonical":
                "party_account_iban"
        }
    )


    state[
        "_state_funding_form_code"
    ] = (
        state[
            "payment_type_detail_source"
        ]
        .map(
            classify_state_funding_form
        )
    )


    if (
        state[
            "_state_funding_form_code"
        ]
        .isna()
        .any()
    ):

        raise ValueError(
            "Positive state_funding transaction has "
            "an unclassified funding form."
        )


    identity = (
        organization_reference[
            [
                "organization_id",
                "party_name_current",
                "organization_name_current",
                "organization_level",
            ]
        ]
        .copy()
    )


    state = state.merge(
        identity,
        on="organization_id",
        how="left",
        validate="many_to_one",
        sort=False,
    )


    if (
        state[
            "organization_level"
        ]
        .isna()
        .any()
    ):

        raise ValueError(
            "Some state_funding rows did not resolve "
            "organization_reference."
        )


    keys = [
        "root_party_id",
        "organization_id",
        "party_account_iban",
    ]


    result = (
        state
        .groupby(
            keys,
            dropna=False,
            sort=False,
            as_index=False,
        )
        .agg(
            party_name_current=(
                "party_name_current",
                _unique_single,
            ),

            organization_name_current=(
                "organization_name_current",
                _unique_single,
            ),

            organization_level=(
                "organization_level",
                _unique_single,
            ),

            first_state_receipt_date=(
                "payment_operation_date",
                "min",
            ),

            last_state_receipt_date=(
                "payment_operation_date",
                "max",
            ),

            positive_state_receipt_rows=(
                "_amount_decimal",
                "size",
            ),

            positive_state_receipt_amount=(
                "_amount_decimal",
                "sum",
            ),

            state_funding_form_count=(
                "_state_funding_form_code",
                "nunique",
            ),

            state_funding_forms_observed=(
                "_state_funding_form_code",
                _unique_join,
            ),

            state_funding_source_forms_observed=(
                "payment_type_detail_source",
                _unique_join,
            ),

            state_funding_form_code=(
                "_state_funding_form_code",
                _unique_single,
            ),
        )
    )


    result[
        "positive_state_receipt_amount"
    ] = (
        result[
            "positive_state_receipt_amount"
        ]
        .map(
            lambda value:
                (
                    value.quantize(
                        Decimal(
                            "0.0000000000"
                        )
                    )
                    if
                    isinstance(
                        value,
                        Decimal,
                    )
                    else
                    value
                )
        )
    )


    result[
        "state_funding_account_confirmed"
    ] = True


    result[
        "state_funding_account_evidence"
    ] = (
        STATE_FUNDING_ACCOUNT_EVIDENCE
    )


    return result[
        [
            "root_party_id",
            "organization_id",
            "party_account_iban",
            "party_name_current",
            "organization_name_current",
            "organization_level",
            "first_state_receipt_date",
            "last_state_receipt_date",
            "positive_state_receipt_rows",
            "positive_state_receipt_amount",
            "state_funding_form_count",
            "state_funding_forms_observed",
            "state_funding_source_forms_observed",
            "state_funding_account_confirmed",
            "state_funding_form_code",
            "state_funding_account_evidence",
        ]
    ]


def _load_frame(
    value,
):

    if isinstance(
        value,
        (
            str,
            Path,
        ),
    ):

        return pd.read_parquet(
            value
        )

    return value


def _write_parquet_atomic(
    frame: pd.DataFrame,
    path: Path,
):

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temp = path.with_suffix(
        ".tmp.parquet"
    )


    frame.to_parquet(
        temp,
        index=False,
    )


    temp.replace(
        path
    )


def rebuild_reference_layer(
    *,
    organizations,
    addresses,
    analysis_manifest,
    property_moneys,
    state_funding,
    output_root,
    overwrite: bool = False,
):
    """
    Rebuild the four references required by the
    consolidated enrichment pipeline.

    Inputs are processed/normalized parquet or DataFrames.

    No RAW access.
    No API access.
    """

    output_root = Path(
        output_root
    )


    inputs = {
        "organizations":
            _load_frame(
                organizations
            ),

        "addresses":
            _load_frame(
                addresses
            ),

        "analysis_manifest":
            _load_frame(
                analysis_manifest
            ),

        "property_moneys":
            _load_frame(
                property_moneys
            ),

        "state_funding":
            _load_frame(
                state_funding
            ),
    }


    organization_reference = (
        build_organization_reference(
            inputs[
                "organizations"
            ],
            inputs[
                "addresses"
            ],
        )
    )


    report_context = (
        build_report_context(
            inputs[
                "analysis_manifest"
            ],
            organization_reference,
        )
    )


    report_account_reference = (
        build_report_account_reference(
            inputs[
                "property_moneys"
            ]
        )
    )


    state_funding_account_reference = (
        build_state_funding_account_reference(
            inputs[
                "state_funding"
            ],
            organization_reference,
        )
    )


    frames = {
        "organization_reference.parquet":
            organization_reference,

        "report_context.parquet":
            report_context,

        "report_account_reference.parquet":
            report_account_reference,

        "state_funding_account_reference.parquet":
            state_funding_account_reference,
    }


    for filename, frame in (
        frames.items()
    ):

        path = (
            output_root
            / filename
        )


        if (
            path.exists()
            and
            not overwrite
        ):

            raise FileExistsError(
                path
            )


        _write_parquet_atomic(
            frame,
            path,
        )


    return {
        "organization_reference":
            organization_reference,

        "report_context":
            report_context,

        "report_account_reference":
            report_account_reference,

        "state_funding_account_reference":
            state_funding_account_reference,
    }
'''


REFERENCE_PATH = (
    SRC_ROOT
    / "reference.py"
)


REFERENCE_PATH.write_text(
    textwrap.dedent(
        REFERENCE_CODE
    ),
    encoding="utf-8",
)


print()
print(
    "Written:",
    REFERENCE_PATH
)


# ============================================================
# 8. TESTS
# ============================================================

TEST_CODE = r'''
from decimal import Decimal

import pandas as pd

from politdata.reference import (
    REPORT_ACCOUNT_RESOLUTION_METHOD,
    STATE_FUNDING_ACCOUNT_EVIDENCE,
    build_report_account_reference,
    build_state_funding_account_reference,
)


def test_report_account_reference_resolves_duplicate_descriptions():

    source = pd.DataFrame(
        [
            {
                "source_report_id": "r1",
                "organization_id": "o1",
                "root_party_id": "p1",
                "party_account_iban": "UA1",

                "party_account_iban_source":
                    "UA1",

                "party_account_type_source":
                    "Поточний рахунок",

                "party_account_type_analytical":
                    "ordinary_account",
            },
            {
                "source_report_id": "r1",
                "organization_id": "o1",
                "root_party_id": "p1",
                "party_account_iban": "UA1",

                "party_account_iban_source":
                    "UA1",

                "party_account_type_source":
                    "Інший опис",

                "party_account_type_analytical":
                    "unknown",
            },
        ]
    )


    result = (
        build_report_account_reference(
            source
        )
    )


    assert len(result) == 1

    row = result.iloc[0]


    assert (
        row[
            "party_account_type_source"
        ]
        ==
        "Поточний рахунок"
    )

    assert (
        row[
            "party_account_type_analytical"
        ]
        ==
        "ordinary_account"
    )

    assert (
        row[
            "party_account_type_resolution_method"
        ]
        ==
        REPORT_ACCOUNT_RESOLUTION_METHOD
    )

    assert (
        int(
            row[
                "snapshot_rows"
            ]
        )
        ==
        2
    )


def test_state_account_requires_positive_state_transaction():

    transactions = pd.DataFrame(
        [
            {
                "root_party_id": "p1",
                "organization_id": "p1",

                "receiver_account_iban_canonical":
                    "UA1",

                "payment_amount":
                    Decimal("100.00"),

                "payment_operation_date":
                    "2025-01-01",

                "payment_type_detail_source":
                    (
                        "Державне фінансування "
                        "статутної діяльності "
                        "політичної партії"
                    ),
            },
            {
                "root_party_id": "p1",
                "organization_id": "p1",

                "receiver_account_iban_canonical":
                    "UA2",

                "payment_amount":
                    Decimal("0.00"),

                "payment_operation_date":
                    "2025-01-01",

                "payment_type_detail_source":
                    (
                        "Державне фінансування "
                        "статутної діяльності "
                        "політичної партії"
                    ),
            },
        ]
    )


    organizations = pd.DataFrame(
        [
            {
                "organization_id":
                    "p1",

                "party_name_current":
                    "ТЕСТ",

                "organization_name_current":
                    "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»",

                "organization_level":
                    "central",
            }
        ]
    )


    result = (
        build_state_funding_account_reference(
            transactions,
            organizations,
        )
    )


    assert len(result) == 1

    row = result.iloc[0]


    assert (
        row[
            "party_account_iban"
        ]
        ==
        "UA1"
    )

    assert (
        row[
            "organization_name_current"
        ]
        ==
        "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»"
    )

    assert (
        row[
            "party_name_current"
        ]
        ==
        "ТЕСТ"
    )

    assert (
        row[
            "positive_state_receipt_amount"
        ]
        ==
        Decimal(
            "100.0000000000"
        )
    )

    assert (
        bool(
            row[
                "state_funding_account_confirmed"
            ]
        )
        is True
    )

    assert (
        row[
            "state_funding_account_evidence"
        ]
        ==
        STATE_FUNDING_ACCOUNT_EVIDENCE
    )
'''


TEST_PATH = (
    TEST_ROOT
    / "test_reference_builders.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 9. RUN COMPLETE TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 110)
print("TESTS")
print("=" * 110)

print(
    result.stdout
)


if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Reference-layer validation rebuild "
        "was not started."
    )


# ============================================================
# 10. FRESH IMPORT
# ============================================================

import politdata.reference as reference


importlib.reload(
    reference
)


# ============================================================
# 11. REBUILD FOUR REQUIRED REFERENCES TO VALIDATION
# ============================================================

if VALIDATION_ROOT.exists():

    shutil.rmtree(
        VALIDATION_ROOT
    )


VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print()
print("=" * 110)
print("REFERENCE LAYER REBUILD")
print("=" * 110)


rebuilt = (
    reference
    .rebuild_reference_layer(
        organizations=
            ORGANIZATIONS_PATH,

        addresses=
            ADDRESSES_PATH,

        analysis_manifest=
            ANALYSIS_MANIFEST_PATH,

        property_moneys=
            PROPERTY_MONEYS_PATH,

        state_funding=
            STATE_FUNDING_PATH,

        output_root=
            VALIDATION_ROOT,

        overwrite=True,
    )
)


summary = pd.DataFrame(
    [
        {
            "artifact":
                "organization_reference",

            "rows":
                len(
                    rebuilt[
                        "organization_reference"
                    ]
                ),
        },
        {
            "artifact":
                "report_context",

            "rows":
                len(
                    rebuilt[
                        "report_context"
                    ]
                ),
        },
        {
            "artifact":
                "report_account_reference",

            "rows":
                len(
                    rebuilt[
                        "report_account_reference"
                    ]
                ),
        },
        {
            "artifact":
                "state_funding_account_reference",

            "rows":
                len(
                    rebuilt[
                        "state_funding_account_reference"
                    ]
                ),
        },
    ]
)


display(
    summary
)


# ============================================================
# 12. FINAL VALIDATION FROM WRITTEN PARQUET
# ============================================================

validation_org = pd.read_parquet(
    VALIDATION_ROOT
    / "organization_reference.parquet"
)

validation_context = pd.read_parquet(
    VALIDATION_ROOT
    / "report_context.parquet"
)

validation_report_account = pd.read_parquet(
    VALIDATION_ROOT
    / "report_account_reference.parquet"
)

validation_state = pd.read_parquet(
    VALIDATION_ROOT
    / "state_funding_account_reference.parquet"
)


org_final = compare_by_key(
    current_org_ref,
    validation_org,
    keys=[
        "organization_id"
    ],
)


context_final = compare_by_key(
    current_report_context,
    validation_context,
    keys=[
        "source_report_id"
    ],
)


report_account_final = compare_by_key(
    current_report_account,
    validation_report_account,
    keys=REPORT_ACCOUNT_KEYS,
)


state_final = compare_by_key(
    current_state_account,
    validation_state,
    keys=STATE_KEYS,

    decimal_columns=[
        "positive_state_receipt_amount"
    ],

    ignore_columns=[
        "organization_name_current"
    ],
)


final_checks = pd.DataFrame(
    [
        {
            "artifact":
                "organization_reference",

            "rows":
                len(
                    validation_org
                ),

            "unexpected_mismatches":
                int(
                    org_final[
                        "mismatches"
                    ].sum()
                ),

            "expected_controlled_changes":
                0,
        },
        {
            "artifact":
                "report_context",

            "rows":
                len(
                    validation_context
                ),

            "unexpected_mismatches":
                int(
                    context_final[
                        "mismatches"
                    ].sum()
                ),

            "expected_controlled_changes":
                0,
        },
        {
            "artifact":
                "report_account_reference",

            "rows":
                len(
                    validation_report_account
                ),

            "unexpected_mismatches":
                int(
                    report_account_final[
                        "mismatches"
                    ].sum()
                ),

            "expected_controlled_changes":
                0,
        },
        {
            "artifact":
                "state_funding_account_reference",

            "rows":
                len(
                    validation_state
                ),

            "unexpected_mismatches":
                int(
                    state_final[
                        "mismatches"
                    ].sum()
                ),

            "expected_controlled_changes":
                7,
        },
    ]
)


print()
print("=" * 110)
print("FINAL REFERENCE-LAYER VALIDATION")
print("=" * 110)

display(
    final_checks
)


total_unexpected = int(
    final_checks[
        "unexpected_mismatches"
    ].sum()
)


# Written state file must still contain exactly
# the seven known corrected names.
state_old = (
    current_state_account
    .sort_values(
        STATE_KEYS
    )
    .reset_index(
        drop=True
    )
)

state_new = (
    validation_state
    .sort_values(
        STATE_KEYS
    )
    .reset_index(
        drop=True
    )
)


written_name_fixes = int(
    (
        string_values(
            state_old[
                "organization_name_current"
            ]
        )
        !=
        string_values(
            state_new[
                "organization_name_current"
            ]
        )
    ).sum()
)


print()
print(
    "Unexpected field mismatches:",
    f"{total_unexpected:,}"
)

print(
    "Controlled state-reference name corrections:",
    written_name_fixes
)


if total_unexpected != 0:

    raise RuntimeError(
        "Reference-layer rebuild has "
        "unexpected differences."
    )


if written_name_fixes != 7:

    raise RuntimeError(
        "Expected exactly 7 controlled "
        "state-reference name corrections."
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 110)
print("DONE")
print("=" * 110)

print(
    "Required reference layer is now reproducible "
    "from processed/normalized inputs."
)

print()
print(
    "Production module:"
)

print(
    "  src/politdata/reference.py"
)

print()
print(
    "Validation output:"
)

print(
    VALIDATION_ROOT
)

print()
print(
    "Expected reference counts:"
)

print(
    "  organization_reference: 9,917"
)

print(
    "  report_context: 78,791"
)

print(
    "  report_account_reference: 12,586"
)

print(
    "  state_funding_account_reference: 7"
)

print()
print(
    "Production parquet files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)


ORGANIZATION REFERENCE PARITY


,column,mismatches
0,root_party_id,0
1,organization_level,0
2,organization_code,0
3,organization_name_current,0
4,party_code,0
5,party_name_current,0
6,region,0
7,region_source,0
8,region_source_address_type,0
9,region_resolution_method,0


Total mismatches: 0

REPORT CONTEXT PARITY


,column,mismatches
0,report_id,0
1,organization_id,0
2,root_party_id,0
3,entity_type,0
4,source_party_id,0
5,party_id_matches_organization,0
6,is_party_office,0
7,schema_version,0
8,report_type,0
9,year,0


Total mismatches: 0

REPORT ACCOUNT REFERENCE PARITY


,column,mismatches
0,party_account_iban_source,0
1,party_account_type_source,0
2,party_account_type_analytical,1454
3,party_account_type_resolution_method,0
4,snapshot_rows,0


Total mismatches: 1,454


RuntimeError: Recovered report_account_reference rule does not have exact production parity.

In [143]:
from pathlib import Path

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

NORMALIZED_PATH = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
    / "properties"
    / "property_moneys.parquet"
)

ENRICHED_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "properties"
    / "property_moneys.parquet"
)

ORG_ACCOUNT_REF_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "organization_account_reference.parquet"
)

REPORT_ACCOUNT_REF_PATH = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
    / "report_account_reference.parquet"
)


# ============================================================
# HELPERS
# ============================================================

def as_string(series):

    return (
        series
        .astype("string")
        .fillna("<NULL>")
    )


def mismatch_count(left, right):

    return int(
        (
            as_string(left)
            !=
            as_string(right)
        ).sum()
    )


def make_key(df, columns):

    result = (
        df[
            columns[0]
        ]
        .astype("string")
        .fillna("<NULL>")
    )

    for column in columns[1:]:

        result = (
            result
            +
            "||"
            +
            df[
                column
            ]
            .astype("string")
            .fillna("<NULL>")
        )

    return result


def first_non_null(series):

    for value in series:

        if pd.notna(value):
            return value

    return pd.NA


# ============================================================
# 1. LOAD
# ============================================================

normalized = pd.read_parquet(
    NORMALIZED_PATH
)

enriched = pd.read_parquet(
    ENRICHED_PATH
)

organization_account = pd.read_parquet(
    ORG_ACCOUNT_REF_PATH
)

current_report_account = pd.read_parquet(
    REPORT_ACCOUNT_REF_PATH
)


print()
print("=" * 110)
print("INPUT")
print("=" * 110)

print(
    "normalized property_moneys:",
    f"{len(normalized):,}",
    "rows /",
    len(normalized.columns),
    "columns",
)

print(
    "enriched property_moneys:",
    f"{len(enriched):,}",
    "rows /",
    len(enriched.columns),
    "columns",
)

print(
    "organization_account_reference:",
    f"{len(organization_account):,}"
)

print(
    "report_account_reference:",
    f"{len(current_report_account):,}"
)


if len(normalized) != 19_140:

    raise RuntimeError(
        "Unexpected normalized property_moneys baseline."
    )


if len(enriched) != 19_140:

    raise RuntimeError(
        "Unexpected enriched property_moneys baseline."
    )


# ============================================================
# 2. FIND SAFE ROW KEY
# ============================================================

candidate_keys = [
    [
        "source_report_id",
        "source__id",
    ],
    [
        "source_report_id",
        "source_row_index",
    ],
    [
        "source_report_id",
        "organization_id",
        "party_account_iban",
        "source__id",
    ],
]


KEY_COLUMNS = None


for columns in candidate_keys:

    if not all(
        column in normalized.columns
        and
        column in enriched.columns
        for column in columns
    ):
        continue


    nk = make_key(
        normalized,
        columns,
    )

    ek = make_key(
        enriched,
        columns,
    )


    if (
        nk.is_unique
        and
        ek.is_unique
        and
        set(nk)
        ==
        set(ek)
    ):

        KEY_COLUMNS = columns

        break


if KEY_COLUMNS is None:

    raise RuntimeError(
        "No safe property_moneys row key found."
    )


print()
print(
    "Safe row key:",
    " + ".join(
        KEY_COLUMNS
    )
)


# ============================================================
# 3. ALIGN NORMALIZED / ENRICHED
# ============================================================

normalized = (
    normalized
    .assign(
        _audit_key=
            make_key(
                normalized,
                KEY_COLUMNS,
            )
    )
    .set_index(
        "_audit_key"
    )
    .sort_index()
)


enriched = (
    enriched
    .assign(
        _audit_key=
            make_key(
                enriched,
                KEY_COLUMNS,
            )
    )
    .set_index(
        "_audit_key"
    )
    .sort_index()
)


if not normalized.index.equals(
    enriched.index
):

    raise RuntimeError(
        "property_moneys row-key sets differ."
    )


# ============================================================
# 4. SCHEMA DELTA
# ============================================================

normalized_columns = set(
    normalized.columns
)

enriched_columns = set(
    enriched.columns
)


enriched_only = sorted(
    enriched_columns
    -
    normalized_columns
)

normalized_only = sorted(
    normalized_columns
    -
    enriched_columns
)


print()
print("=" * 110)
print("SCHEMA DELTA")
print("=" * 110)

print(
    "Common columns:",
    len(
        normalized_columns
        &
        enriched_columns
    )
)

print(
    "Enriched-only:",
    len(
        enriched_only
    )
)

print(
    "Normalized-only:",
    len(
        normalized_only
    )
)


display(
    pd.DataFrame(
        {
            "enriched_only":
                pd.Series(
                    enriched_only,
                    dtype="string",
                )
        }
    )
)


if normalized_only:

    print(
        "Normalized-only columns:",
        normalized_only,
    )


# ============================================================
# 5. COMMON-COLUMN PARITY
# ============================================================

common_rows = []


for column in sorted(
    normalized_columns
    &
    enriched_columns
):

    count = mismatch_count(
        normalized[
            column
        ],
        enriched[
            column
        ],
    )


    common_rows.append(
        {
            "column":
                column,

            "mismatches":
                count,
        }
    )


common_df = (
    pd.DataFrame(
        common_rows
    )
    .sort_values(
        [
            "mismatches",
            "column",
        ],
        ascending=[
            False,
            True,
        ],
    )
)


print()
print("=" * 110)
print("COMMON-COLUMN DIFFERENCES")
print("=" * 110)

display(
    common_df[
        common_df[
            "mismatches"
        ]
        >
        0
    ]
)


print(
    "Total common-field mismatches:",
    f"{int(common_df['mismatches'].sum()):,}"
)


# ============================================================
# 6. EXACT 1,454 ANALYTICAL-TYPE CHANGES
# ============================================================

analytical_mask = (
    as_string(
        normalized[
            "party_account_type_analytical"
        ]
    )
    !=
    as_string(
        enriched[
            "party_account_type_analytical"
        ]
    )
)


analytical_changes = pd.DataFrame(
    {
        "organization_id":
            normalized.loc[
                analytical_mask,
                "organization_id"
            ],

        "root_party_id":
            normalized.loc[
                analytical_mask,
                "root_party_id"
            ],

        "party_account_iban":
            normalized.loc[
                analytical_mask,
                "party_account_iban"
            ],

        "party_account_type_source":
            normalized.loc[
                analytical_mask,
                "party_account_type_source"
            ],

        "normalized_type":
            normalized.loc[
                analytical_mask,
                "party_account_type_analytical"
            ],

        "enriched_type":
            enriched.loc[
                analytical_mask,
                "party_account_type_analytical"
            ],

        "normalized_method":
            (
                normalized.loc[
                    analytical_mask,
                    "party_account_type_resolution_method"
                ]
                if
                "party_account_type_resolution_method"
                in normalized.columns
                else
                pd.NA
            ),

        "enriched_method":
            (
                enriched.loc[
                    analytical_mask,
                    "party_account_type_resolution_method"
                ]
                if
                "party_account_type_resolution_method"
                in enriched.columns
                else
                pd.NA
            ),
    }
)


print()
print("=" * 110)
print("ANALYTICAL ACCOUNT-TYPE CHANGES")
print("=" * 110)

print(
    "Changed rows:",
    f"{len(analytical_changes):,}"
)


mapping = (
    analytical_changes
    .groupby(
        [
            "normalized_type",
            "enriched_type",
            "enriched_method",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
    .sort_values(
        "size",
        ascending=False,
    )
)


display(
    mapping
)


# ============================================================
# 7. TEST ORGANIZATION_ACCOUNT_REFERENCE AS THE SOURCE
# ============================================================

org_ref_required = {
    "organization_id",
    "root_party_id",
    "party_account_iban",
    "party_account_type_analytical",
    "party_account_type_resolution_method",
}


missing = (
    org_ref_required
    -
    set(
        organization_account.columns
    )
)


if missing:

    raise RuntimeError(
        "organization_account_reference missing: "
        f"{sorted(missing)}"
    )


org_ref = (
    organization_account[
        [
            "organization_id",
            "root_party_id",
            "party_account_iban",
            "party_account_type_analytical",
            "party_account_type_resolution_method",
        ]
    ]
    .copy()
)


if (
    org_ref
    .duplicated(
        [
            "organization_id",
            "root_party_id",
            "party_account_iban",
        ]
    )
    .any()
):

    raise RuntimeError(
        "organization_account_reference key "
        "is not unique."
    )


base_for_join = (
    normalized
    .reset_index()
    [
        [
            "_audit_key",
            "organization_id",
            "root_party_id",
            "party_account_iban",
        ]
    ]
)


expected_from_org_ref = (
    base_for_join
    .merge(
        org_ref,
        on=[
            "organization_id",
            "root_party_id",
            "party_account_iban",
        ],
        how="left",
        validate="many_to_one",
        sort=False,
    )
    .set_index(
        "_audit_key"
    )
    .reindex(
        enriched.index
    )
)


type_mismatch_to_org_ref = (
    mismatch_count(
        enriched[
            "party_account_type_analytical"
        ],
        expected_from_org_ref[
            "party_account_type_analytical"
        ],
    )
)


method_mismatch_to_org_ref = (
    mismatch_count(
        enriched[
            "party_account_type_resolution_method"
        ],
        expected_from_org_ref[
            "party_account_type_resolution_method"
        ],
    )
    if
    "party_account_type_resolution_method"
    in enriched.columns
    else
    None
)


print()
print("=" * 110)
print("ORGANIZATION ACCOUNT REFERENCE PARITY")
print("=" * 110)

print(
    "party_account_type_analytical mismatches:",
    f"{type_mismatch_to_org_ref:,}"
)

print(
    "party_account_type_resolution_method mismatches:",
    (
        f"{method_mismatch_to_org_ref:,}"
        if
        method_mismatch_to_org_ref is not None
        else
        "N/A"
    )
)


# ============================================================
# 8. HOW MANY ENRICHED VALUES ACTUALLY DIFFER
#    FROM NORMALIZED BECAUSE OF ORG REFERENCE?
# ============================================================

comparison = pd.DataFrame(
    {
        "normalized":
            normalized[
                "party_account_type_analytical"
            ],

        "enriched":
            enriched[
                "party_account_type_analytical"
            ],

        "org_reference":
            expected_from_org_ref[
                "party_account_type_analytical"
            ],
    }
)


comparison[
    "normalized_equals_enriched"
] = (
    as_string(
        comparison[
            "normalized"
        ]
    )
    ==
    as_string(
        comparison[
            "enriched"
        ]
    )
)


comparison[
    "enriched_equals_org_reference"
] = (
    as_string(
        comparison[
            "enriched"
        ]
    )
    ==
    as_string(
        comparison[
            "org_reference"
        ]
    )
)


pattern = (
    comparison
    .groupby(
        [
            "normalized_equals_enriched",
            "enriched_equals_org_reference",
        ],
        dropna=False,
        as_index=False,
    )
    .size()
)


print()
print("=" * 110)
print("PROPERTY_MONEYS ENRICHMENT PATTERN")
print("=" * 110)

display(
    pattern
)


# ============================================================
# 9. REPORT ACCOUNT REFERENCE FROM ENRICHED PROPERTY_MONEYS
#
# This should reproduce the current 12,586-row reference
# except that reference-level resolution method is set
# explicitly.
# ============================================================

REPORT_KEYS = [
    "source_report_id",
    "organization_id",
    "root_party_id",
    "party_account_iban",
]


candidate_report_account = (
    enriched
    .reset_index(
        drop=True
    )
    .groupby(
        REPORT_KEYS,
        dropna=False,
        sort=False,
        as_index=False,
    )
    .agg(
        party_account_iban_source=(
            "party_account_iban_source",
            first_non_null,
        ),

        party_account_type_source=(
            "party_account_type_source",
            first_non_null,
        ),

        party_account_type_analytical=(
            "party_account_type_analytical",
            first_non_null,
        ),

        snapshot_rows=(
            "party_account_iban",
            "size",
        ),
    )
)


candidate_report_account[
    "party_account_type_resolution_method"
] = (
    "property_moneys_exact_report_org_iban"
)


candidate_report_account = (
    candidate_report_account[
        list(
            current_report_account.columns
        )
    ]
)


current_key = make_key(
    current_report_account,
    REPORT_KEYS,
)

candidate_key = make_key(
    candidate_report_account,
    REPORT_KEYS,
)


if (
    not current_key.is_unique
    or
    not candidate_key.is_unique
):

    raise RuntimeError(
        "report_account_reference key not unique."
    )


if set(
    current_key
) != set(
    candidate_key
):

    raise RuntimeError(
        "report_account_reference key sets differ."
    )


current_aligned = (
    current_report_account
    .assign(
        _key=current_key
    )
    .set_index(
        "_key"
    )
    .sort_index()
)


candidate_aligned = (
    candidate_report_account
    .assign(
        _key=candidate_key
    )
    .set_index(
        "_key"
    )
    .sort_index()
)


report_rows = []


for column in current_report_account.columns:

    if column in REPORT_KEYS:
        continue


    count = mismatch_count(
        current_aligned[
            column
        ],
        candidate_aligned[
            column
        ],
    )


    report_rows.append(
        {
            "column":
                column,

            "mismatches":
                count,
        }
    )


report_parity = pd.DataFrame(
    report_rows
)


print()
print("=" * 110)
print("REPORT ACCOUNT REFERENCE FROM ENRICHED PROPERTY_MONEYS")
print("=" * 110)

display(
    report_parity
)


print(
    "Total report-account mismatches:",
    f"{int(report_parity['mismatches'].sum()):,}"
)


# ============================================================
# 10. EXAMPLES OF THE 1,454 CHANGES
# ============================================================

print()
print("=" * 110)
print("EXAMPLES OF PROPERTY_MONEYS ANALYTICAL-TYPE CHANGES")
print("=" * 110)

display(
    analytical_changes
    .drop_duplicates()
    .head(
        50
    )
)


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 110)
print("SUMMARY")
print("=" * 110)

print(
    "Property_moneys rows:",
    f"{len(enriched):,}"
)

print(
    "Normalized -> enriched analytical-type changes:",
    f"{int(analytical_mask.sum()):,}"
)

print(
    "Enriched type mismatches vs organization_account_reference:",
    f"{type_mismatch_to_org_ref:,}"
)

print(
    "Enriched method mismatches vs organization_account_reference:",
    (
        f"{method_mismatch_to_org_ref:,}"
        if
        method_mismatch_to_org_ref is not None
        else
        "N/A"
    )
)

print(
    "Report-account mismatches from enriched property_moneys:",
    f"{int(report_parity['mismatches'].sum()):,}"
)

print()
print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

print(
    "No parquet files were modified."
)


INPUT
normalized property_moneys: 19,140 rows / 28 columns
enriched property_moneys: 19,140 rows / 33 columns
organization_account_reference: 1,668
report_account_reference: 12,586

Safe row key: source_report_id + source__id

SCHEMA DELTA
Common columns: 26
Enriched-only: 7
Normalized-only: 2


,enriched_only
0,analysis_override
1,organization_code
2,organization_level
3,organization_name_current
4,party_code
5,party_name_current
6,region


Normalized-only columns: ['party_account_type_declared', 'party_account_type_declared_code']

COMMON-COLUMN DIFFERENCES


,column,mismatches
5,party_account_type_analytical,2143


Total common-field mismatches: 2,143

ANALYTICAL ACCOUNT-TYPE CHANGES
Changed rows: 2,143


...


REPORT ACCOUNT REFERENCE FROM ENRICHED PROPERTY_MONEYS


,column,mismatches
0,party_account_iban_source,0
1,party_account_type_source,2
2,party_account_type_analytical,1
3,party_account_type_resolution_method,0
4,snapshot_rows,0


Total report-account mismatches: 3

EXAMPLES OF PROPERTY_MONEYS ANALYTICAL-TYPE CHANGES


,organization_id,root_party_id,party_account_iban,party_account_type_source,normalized_type,enriched_type,normalized_method,enriched_method
_audit_key,,,,,,,,
002f5e70-2bfd-11f0-a300-eb47575da91a||2b285cdf-03c0-4867-a40e-4136e21c612a,f80ac50a-35f3-4c5c-8f52-251ae9adb903,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,UA053223130000029249001834223,Транзитний рахунок,transit_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
006891e0-e77f-11ee-8847-316909bdf249||256a932a-96ee-433e-b252-fa93e0ef46ef,9b1c3511-1fcc-4091-bdcd-97930d503b65,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA553282090000026047000002354,Рахунок для соціальних виплат,social_insurance_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
018e7ab0-e6c3-11ee-8d4b-3dd92aa40d65||15efcdc7-403a-47ed-ade4-76b133264817,991b7269-b467-4fa1-9fb9-b32cd68ccd2f,7d9a4591-6cea-451f-8545-b211f5e7b9bd,UA113387830000026041055102340,Рахунок для соціальних виплат,social_insurance_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
01b8d0d0-eaba-11ee-96f1-37a2ca81244c||64b23078-1f7b-44ed-8dff-79b76592e3ea,088627e6-1904-4e4a-8e7a-a7c83a1d6448,94655276-fc7b-4a8f-a36b-d3eabb8d27fa,UA423223130000029243000110944,Інше - Транзитний рахунок,transit_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
0255ec70-25b1-11f0-a300-eb47575da91a||50fd56e5-9dda-4494-acf5-46877498f430,edc80e40-f5d3-47b7-9242-f1bbf6593538,b11839f3-bffe-4f93-ba10-3b877ed181b9,UA713003350000000026001745243,поточний,ordinary_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
0255ec70-25b1-11f0-a300-eb47575da91a||6b8d3477-af05-4ba1-b28e-521496fea621,edc80e40-f5d3-47b7-9242-f1bbf6593538,b11839f3-bffe-4f93-ba10-3b877ed181b9,UA603003350000000026002678096,поточний,ordinary_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
028c5724-0c77-44c8-ad94-0a46f537a412||2627a68d-fa42-47ad-b9a7-2fba2a900ec3,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,UA723808380000026048700741786,Бюджетний рахунок,budget_account_unspecified,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
028c5724-0c77-44c8-ad94-0a46f537a412||a29eff80-fe84-410f-86a6-eedefa10126b,1e52ae20-1244-42ac-927d-7e014c80fc14,1e52ae20-1244-42ac-927d-7e014c80fc14,UA223204780000026046903289478,Рахунок для соціальних виплат,social_insurance_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type
029003f0-d96d-11ef-be11-253fcd32c4c9||7ddc6635-bfa7-4c47-9e95-c5edce80a7ff,f785a0c5-6fb8-4774-965b-793d3842fc49,f785a0c5-6fb8-4774-965b-793d3842fc49,UA413545070000026000305701942,Картковий рахунок,card_account,unknown,property_moneys_unresolved_type,property_moneys_unresolved_type



SUMMARY
Property_moneys rows: 19,140
Normalized -> enriched analytical-type changes: 2,143
Enriched type mismatches vs organization_account_reference: 2,164
Enriched method mismatches vs organization_account_reference: 19,140
Report-account mismatches from enriched property_moneys: 3

RAW files were NOT read.
API was NOT called.
No parquet files were modified.


In [144]:
from pathlib import Path
from decimal import Decimal, InvalidOperation
import importlib
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

SRC_ROOT = ROOT / "src" / "politdata"
TEST_ROOT = ROOT / "tests"

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

INTERIM_ROOT = (
    ROOT
    / "data"
    / "interim"
)

CURRENT_REFERENCE_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "enriched_v0_1"
    / "reference"
)

VALIDATION_ROOT = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "reference_layer_v0_1"
)


ORGANIZATIONS_PATH = (
    NORMALIZED_ROOT
    / "organizations.parquet"
)

ADDRESSES_PATH = (
    NORMALIZED_ROOT
    / "organization_addresses.parquet"
)

ANALYSIS_MANIFEST_PATH = (
    INTERIM_ROOT
    / "reports"
    / "analysis_selected_reports_manifest.parquet"
)

PROPERTY_MONEYS_PATH = (
    NORMALIZED_ROOT
    / "properties"
    / "property_moneys.parquet"
)

STATE_FUNDING_PATH = (
    NORMALIZED_ROOT
    / "payments"
    / "state_funding.parquet"
)


# ============================================================
# HELPERS
# ============================================================

def as_string(series):

    return (
        series
        .astype("string")
        .fillna("<NULL>")
    )


def mismatch_count(left, right):

    return int(
        (
            as_string(left)
            !=
            as_string(right)
        ).sum()
    )


def to_decimal(value):

    if pd.isna(value):
        return None

    if isinstance(value, Decimal):
        return value

    text = (
        str(value)
        .strip()
        .replace("\u00a0", "")
        .replace(" ", "")
        .replace(",", ".")
    )

    if not text:
        return None

    try:

        return Decimal(text)

    except InvalidOperation as exc:

        raise ValueError(
            f"Cannot parse Decimal: {value!r}"
        ) from exc


def decimal_mismatch_count(left, right):

    total = 0

    for a, b in zip(left, right):

        da = to_decimal(a)
        db = to_decimal(b)

        if da is None and db is None:
            continue

        if da != db:
            total += 1

    return total


def first_non_null(series):

    for value in series:

        if pd.notna(value):
            return value

    return pd.NA


def unique_single(series):

    values = list(
        pd.unique(
            series.dropna()
        )
    )

    if not values:
        return pd.NA

    if len(values) != 1:

        raise RuntimeError(
            f"Expected one unique value: {values}"
        )

    return values[0]


def unique_join(series):

    values = sorted(
        {
            str(value).strip()
            for value in series
            if (
                pd.notna(value)
                and str(value).strip()
            )
        }
    )

    if not values:
        return pd.NA

    return " | ".join(values)


def keyed_align(
    current,
    candidate,
    keys,
):

    current = current.copy()
    candidate = candidate.copy()

    if current.duplicated(keys).any():
        raise RuntimeError(
            f"Current key not unique: {keys}"
        )

    if candidate.duplicated(keys).any():
        raise RuntimeError(
            f"Candidate key not unique: {keys}"
        )

    ck = (
        current[keys]
        .astype("string")
        .fillna("<NULL>")
        .agg("||".join, axis=1)
    )

    nk = (
        candidate[keys]
        .astype("string")
        .fillna("<NULL>")
        .agg("||".join, axis=1)
    )

    if set(ck) != set(nk):

        raise RuntimeError(
            "Reference key sets differ. "
            f"missing={len(set(ck) - set(nk))}; "
            f"extra={len(set(nk) - set(ck))}"
        )

    return (
        current
        .assign(_key=ck)
        .set_index("_key")
        .sort_index(),

        candidate
        .assign(_key=nk)
        .set_index("_key")
        .sort_index(),
    )


def compare_all(
    current,
    candidate,
    *,
    keys,
    decimal_columns=(),
    ignore_columns=(),
):

    old, new = keyed_align(
        current,
        candidate,
        keys,
    )

    rows = []

    for column in current.columns:

        if (
            column in keys
            or column in ignore_columns
            or column not in candidate.columns
        ):
            continue

        if column in decimal_columns:

            count = decimal_mismatch_count(
                old[column],
                new[column],
            )

        else:

            count = mismatch_count(
                old[column],
                new[column],
            )

        rows.append(
            {
                "column":
                    column,

                "mismatches":
                    count,
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# 1. LOAD PROCESSED INPUTS
# ============================================================

organizations = pd.read_parquet(
    ORGANIZATIONS_PATH
)

addresses = pd.read_parquet(
    ADDRESSES_PATH
)

analysis_manifest = pd.read_parquet(
    ANALYSIS_MANIFEST_PATH
)

property_moneys = pd.read_parquet(
    PROPERTY_MONEYS_PATH
)

state_funding = pd.read_parquet(
    STATE_FUNDING_PATH
)


current_org = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "organization_reference.parquet"
)

current_context = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "report_context.parquet"
)

current_report_account = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "report_account_reference.parquet"
)

current_state_account = pd.read_parquet(
    CURRENT_REFERENCE_ROOT
    / "state_funding_account_reference.parquet"
)


# ============================================================
# 2. EXISTING BUILDERS
# ============================================================

from politdata.normalization.reference import (
    build_organization_reference,
    build_report_context,
)


candidate_org = (
    build_organization_reference(
        organizations,
        addresses,
    )
)


candidate_context = (
    build_report_context(
        analysis_manifest,
        candidate_org,
    )
)


org_check = compare_all(
    current_org,
    candidate_org,
    keys=[
        "organization_id"
    ],
)


context_check = compare_all(
    current_context,
    candidate_context,
    keys=[
        "source_report_id"
    ],
)


print()
print("=" * 110)
print("ORGANIZATION REFERENCE")
print("=" * 110)

display(org_check)

print(
    "Mismatches:",
    int(
        org_check[
            "mismatches"
        ].sum()
    )
)


print()
print("=" * 110)
print("REPORT CONTEXT")
print("=" * 110)

display(context_check)

print(
    "Mismatches:",
    int(
        context_check[
            "mismatches"
        ].sum()
    )
)


if int(org_check["mismatches"].sum()) != 0:

    raise RuntimeError(
        "organization_reference parity failed."
    )


if int(context_check["mismatches"].sum()) != 0:

    raise RuntimeError(
        "report_context parity failed."
    )


# ============================================================
# 3. REPORT ACCOUNT REFERENCE
#
# IMPORTANT:
# Use CURRENT NORMALIZED property_moneys classification.
#
# Therefore old report-account analytical type is allowed
# to change where the normalized classifier has improved.
# ============================================================

REPORT_KEYS = [
    "source_report_id",
    "organization_id",
    "root_party_id",
    "party_account_iban",
]


candidate_report_account = (
    property_moneys
    .groupby(
        REPORT_KEYS,
        dropna=False,
        sort=False,
        as_index=False,
    )
    .agg(
        party_account_iban_source=(
            "party_account_iban_source",
            first_non_null,
        ),

        party_account_type_source=(
            "party_account_type_source",
            first_non_null,
        ),

        party_account_type_analytical=(
            "party_account_type_analytical",
            first_non_null,
        ),

        snapshot_rows=(
            "party_account_iban",
            "size",
        ),
    )
)


candidate_report_account[
    "party_account_type_resolution_method"
] = (
    "property_moneys_exact_report_org_iban"
)


candidate_report_account = (
    candidate_report_account[
        list(
            current_report_account.columns
        )
    ]
)


old_report, new_report = keyed_align(
    current_report_account,
    candidate_report_account,
    REPORT_KEYS,
)


report_rows = []

for column in current_report_account.columns:

    if column in REPORT_KEYS:
        continue

    count = mismatch_count(
        old_report[column],
        new_report[column],
    )

    report_rows.append(
        {
            "column":
                column,

            "mismatches":
                count,
        }
    )


report_check = pd.DataFrame(
    report_rows
)


print()
print("=" * 110)
print("REPORT ACCOUNT REFERENCE")
print("=" * 110)

display(report_check)


report_type_updates = int(
    report_check.loc[
        report_check[
            "column"
        ]
        ==
        "party_account_type_analytical",
        "mismatches",
    ].iloc[0]
)


other_report_mismatches = int(
    report_check.loc[
        report_check[
            "column"
        ]
        !=
        "party_account_type_analytical",
        "mismatches",
    ].sum()
)


print(
    "Controlled analytical-type updates:",
    f"{report_type_updates:,}"
)

print(
    "Unexpected other mismatches:",
    f"{other_report_mismatches:,}"
)


if report_type_updates != 1_454:

    raise RuntimeError(
        "Expected exactly 1,454 controlled "
        "report-account analytical-type updates."
    )


if other_report_mismatches != 0:

    raise RuntimeError(
        "Unexpected report-account differences."
    )


# ============================================================
# 4. STATE FUNDING ACCOUNT REFERENCE
# ============================================================

def classify_state_form(value):

    if pd.isna(value):
        return None

    text = (
        str(value)
        .strip()
        .lower()
    )

    if "статут" in text:

        return (
            "state_statutory_funding"
        )

    if (
        "відшкод" in text
        or
        "вибор" in text
    ):

        return (
            "state_campaign_reimbursement"
        )

    return None


state = state_funding.copy()


state[
    "_amount_decimal"
] = (
    state[
        "payment_amount"
    ]
    .map(
        to_decimal
    )
)


positive = (
    state[
        "_amount_decimal"
    ]
    .map(
        lambda value:
            (
                value is not None
                and value > 0
            )
    )
    &
    state[
        "receiver_account_iban_canonical"
    ]
    .notna()
)


state = (
    state[
        positive
    ]
    .copy()
)


state = state.rename(
    columns={
        "receiver_account_iban_canonical":
            "party_account_iban"
    }
)


state[
    "_form_code"
] = (
    state[
        "payment_type_detail_source"
    ]
    .map(
        classify_state_form
    )
)


if state["_form_code"].isna().any():

    raise RuntimeError(
        "Unclassified positive state_funding row."
    )


identity = (
    candidate_org[
        [
            "organization_id",
            "party_name_current",
            "organization_name_current",
            "organization_level",
        ]
    ]
)


state = state.merge(
    identity,
    on="organization_id",
    how="left",
    validate="many_to_one",
)


STATE_KEYS = [
    "root_party_id",
    "organization_id",
    "party_account_iban",
]


candidate_state = (
    state
    .groupby(
        STATE_KEYS,
        dropna=False,
        sort=False,
        as_index=False,
    )
    .agg(
        party_name_current=(
            "party_name_current",
            unique_single,
        ),

        organization_name_current=(
            "organization_name_current",
            unique_single,
        ),

        organization_level=(
            "organization_level",
            unique_single,
        ),

        first_state_receipt_date=(
            "payment_operation_date",
            "min",
        ),

        last_state_receipt_date=(
            "payment_operation_date",
            "max",
        ),

        positive_state_receipt_rows=(
            "_amount_decimal",
            "size",
        ),

        positive_state_receipt_amount=(
            "_amount_decimal",
            "sum",
        ),

        state_funding_form_count=(
            "_form_code",
            "nunique",
        ),

        state_funding_forms_observed=(
            "_form_code",
            unique_join,
        ),

        state_funding_source_forms_observed=(
            "payment_type_detail_source",
            unique_join,
        ),

        state_funding_form_code=(
            "_form_code",
            unique_single,
        ),
    )
)


candidate_state[
    "positive_state_receipt_amount"
] = (
    candidate_state[
        "positive_state_receipt_amount"
    ]
    .map(
        lambda value:
            (
                value.quantize(
                    Decimal(
                        "0.0000000000"
                    )
                )
                if isinstance(
                    value,
                    Decimal,
                )
                else value
            )
    )
)


candidate_state[
    "state_funding_account_confirmed"
] = True


candidate_state[
    "state_funding_account_evidence"
] = (
    "positive_transaction_in_state_funding_section"
)


candidate_state = (
    candidate_state[
        list(
            current_state_account.columns
        )
    ]
)


state_check = compare_all(
    current_state_account,
    candidate_state,
    keys=STATE_KEYS,

    decimal_columns=[
        "positive_state_receipt_amount"
    ],

    ignore_columns=[
        "organization_name_current"
    ],
)


print()
print("=" * 110)
print("STATE FUNDING REFERENCE — NON-NAME PARITY")
print("=" * 110)

display(state_check)


state_other_mismatches = int(
    state_check[
        "mismatches"
    ].sum()
)


if state_other_mismatches != 0:

    raise RuntimeError(
        "Unexpected state reference differences."
    )


old_state, new_state = keyed_align(
    current_state_account,
    candidate_state,
    STATE_KEYS,
)


name_mask = (
    as_string(
        old_state[
            "organization_name_current"
        ]
    )
    !=
    as_string(
        new_state[
            "organization_name_current"
        ]
    )
)


state_name_updates = int(
    name_mask.sum()
)


print(
    "Controlled organization-name updates:",
    state_name_updates
)


if state_name_updates != 7:

    raise RuntimeError(
        "Expected exactly 7 controlled "
        "state-reference name updates."
    )


# ============================================================
# 5. WRITE FINAL REFERENCE MODULE
# ============================================================

REFERENCE_CODE = r'''
from __future__ import annotations

from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd

from politdata.normalization.reference import (
    build_organization_reference,
    build_report_context,
)


REPORT_ACCOUNT_RESOLUTION_METHOD = (
    "property_moneys_exact_report_org_iban"
)

STATE_FUNDING_ACCOUNT_EVIDENCE = (
    "positive_transaction_in_state_funding_section"
)


def _first_non_null(series):

    for value in series:

        if pd.notna(value):
            return value

    return pd.NA


def _unique_single(series):

    values = list(
        pd.unique(
            series.dropna()
        )
    )

    if not values:
        return pd.NA

    if len(values) != 1:

        raise ValueError(
            f"Expected one unique value: {values}"
        )

    return values[0]


def _unique_join(series):

    values = sorted(
        {
            str(value).strip()
            for value in series
            if (
                pd.notna(value)
                and str(value).strip()
            )
        }
    )

    if not values:
        return pd.NA

    return " | ".join(values)


def _to_decimal(value):

    if pd.isna(value):
        return None

    if isinstance(value, Decimal):
        return value

    text = (
        str(value)
        .strip()
        .replace("\u00a0", "")
        .replace(" ", "")
        .replace(",", ".")
    )

    if not text:
        return None

    try:

        return Decimal(text)

    except InvalidOperation as exc:

        raise ValueError(
            f"Cannot parse Decimal: {value!r}"
        ) from exc


def classify_state_funding_form(value):

    if pd.isna(value):
        return None

    text = (
        str(value)
        .strip()
        .lower()
    )

    if "статут" in text:

        return (
            "state_statutory_funding"
        )

    if (
        "відшкод" in text
        or
        "вибор" in text
    ):

        return (
            "state_campaign_reimbursement"
        )

    return None


def build_report_account_reference(
    property_moneys,
):

    keys = [
        "source_report_id",
        "organization_id",
        "root_party_id",
        "party_account_iban",
    ]

    required = {
        *keys,
        "party_account_iban_source",
        "party_account_type_source",
        "party_account_type_analytical",
    }

    missing = (
        required
        -
        set(
            property_moneys.columns
        )
    )

    if missing:

        raise KeyError(
            f"property_moneys missing: {sorted(missing)}"
        )

    result = (
        property_moneys
        .groupby(
            keys,
            dropna=False,
            sort=False,
            as_index=False,
        )
        .agg(
            party_account_iban_source=(
                "party_account_iban_source",
                _first_non_null,
            ),

            party_account_type_source=(
                "party_account_type_source",
                _first_non_null,
            ),

            party_account_type_analytical=(
                "party_account_type_analytical",
                _first_non_null,
            ),

            snapshot_rows=(
                "party_account_iban",
                "size",
            ),
        )
    )

    result[
        "party_account_type_resolution_method"
    ] = (
        REPORT_ACCOUNT_RESOLUTION_METHOD
    )

    return result[
        [
            "source_report_id",
            "organization_id",
            "root_party_id",
            "party_account_iban",
            "party_account_iban_source",
            "party_account_type_source",
            "party_account_type_analytical",
            "party_account_type_resolution_method",
            "snapshot_rows",
        ]
    ]


def build_state_funding_account_reference(
    state_funding,
    organization_reference,
):

    state = state_funding.copy()

    state[
        "_amount_decimal"
    ] = (
        state[
            "payment_amount"
        ]
        .map(
            _to_decimal
        )
    )

    positive = (
        state[
            "_amount_decimal"
        ]
        .map(
            lambda value:
                (
                    value is not None
                    and value > 0
                )
        )
        &
        state[
            "receiver_account_iban_canonical"
        ]
        .notna()
    )

    state = state[
        positive
    ].copy()

    state = state.rename(
        columns={
            "receiver_account_iban_canonical":
                "party_account_iban"
        }
    )

    state[
        "_form_code"
    ] = (
        state[
            "payment_type_detail_source"
        ]
        .map(
            classify_state_funding_form
        )
    )

    if state["_form_code"].isna().any():

        raise ValueError(
            "Unclassified positive state_funding row."
        )

    identity = (
        organization_reference[
            [
                "organization_id",
                "party_name_current",
                "organization_name_current",
                "organization_level",
            ]
        ]
    )

    state = state.merge(
        identity,
        on="organization_id",
        how="left",
        validate="many_to_one",
        sort=False,
    )

    keys = [
        "root_party_id",
        "organization_id",
        "party_account_iban",
    ]

    result = (
        state
        .groupby(
            keys,
            dropna=False,
            sort=False,
            as_index=False,
        )
        .agg(
            party_name_current=(
                "party_name_current",
                _unique_single,
            ),

            organization_name_current=(
                "organization_name_current",
                _unique_single,
            ),

            organization_level=(
                "organization_level",
                _unique_single,
            ),

            first_state_receipt_date=(
                "payment_operation_date",
                "min",
            ),

            last_state_receipt_date=(
                "payment_operation_date",
                "max",
            ),

            positive_state_receipt_rows=(
                "_amount_decimal",
                "size",
            ),

            positive_state_receipt_amount=(
                "_amount_decimal",
                "sum",
            ),

            state_funding_form_count=(
                "_form_code",
                "nunique",
            ),

            state_funding_forms_observed=(
                "_form_code",
                _unique_join,
            ),

            state_funding_source_forms_observed=(
                "payment_type_detail_source",
                _unique_join,
            ),

            state_funding_form_code=(
                "_form_code",
                _unique_single,
            ),
        )
    )

    result[
        "positive_state_receipt_amount"
    ] = (
        result[
            "positive_state_receipt_amount"
        ]
        .map(
            lambda value:
                (
                    value.quantize(
                        Decimal(
                            "0.0000000000"
                        )
                    )
                    if isinstance(
                        value,
                        Decimal,
                    )
                    else value
                )
        )
    )

    result[
        "state_funding_account_confirmed"
    ] = True

    result[
        "state_funding_account_evidence"
    ] = (
        STATE_FUNDING_ACCOUNT_EVIDENCE
    )

    return result


def _load(value):

    if isinstance(
        value,
        (
            str,
            Path,
        ),
    ):

        return pd.read_parquet(
            value
        )

    return value


def _atomic_write(frame, path):

    path = Path(path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temp = path.with_suffix(
        ".tmp.parquet"
    )

    frame.to_parquet(
        temp,
        index=False,
    )

    temp.replace(path)


def rebuild_reference_layer(
    *,
    organizations,
    addresses,
    analysis_manifest,
    property_moneys,
    state_funding,
    output_root,
    overwrite=False,
):

    output_root = Path(
        output_root
    )

    organizations = _load(
        organizations
    )

    addresses = _load(
        addresses
    )

    analysis_manifest = _load(
        analysis_manifest
    )

    property_moneys = _load(
        property_moneys
    )

    state_funding = _load(
        state_funding
    )

    organization_reference = (
        build_organization_reference(
            organizations,
            addresses,
        )
    )

    report_context = (
        build_report_context(
            analysis_manifest,
            organization_reference,
        )
    )

    report_account_reference = (
        build_report_account_reference(
            property_moneys
        )
    )

    state_funding_account_reference = (
        build_state_funding_account_reference(
            state_funding,
            organization_reference,
        )
    )

    frames = {
        "organization_reference.parquet":
            organization_reference,

        "report_context.parquet":
            report_context,

        "report_account_reference.parquet":
            report_account_reference,

        "state_funding_account_reference.parquet":
            state_funding_account_reference,
    }

    for filename, frame in frames.items():

        path = (
            output_root
            / filename
        )

        if (
            path.exists()
            and
            not overwrite
        ):

            raise FileExistsError(path)

        _atomic_write(
            frame,
            path,
        )

    return frames
'''


REFERENCE_PATH = (
    SRC_ROOT
    / "reference.py"
)


REFERENCE_PATH.write_text(
    textwrap.dedent(
        REFERENCE_CODE
    ),
    encoding="utf-8",
)


print()
print(
    "Written:",
    REFERENCE_PATH
)


# ============================================================
# 6. SMALL TESTS
# ============================================================

TEST_CODE = r'''
from decimal import Decimal

import pandas as pd

from politdata.reference import (
    build_report_account_reference,
    build_state_funding_account_reference,
)


def test_report_account_uses_normalized_analytical_type():

    source = pd.DataFrame(
        [
            {
                "source_report_id": "r1",
                "organization_id": "o1",
                "root_party_id": "p1",
                "party_account_iban": "UA1",

                "party_account_iban_source":
                    "UA1",

                "party_account_type_source":
                    "Транзитний рахунок",

                "party_account_type_analytical":
                    "transit_account",
            }
        ]
    )

    result = (
        build_report_account_reference(
            source
        )
    )

    assert (
        result.iloc[0][
            "party_account_type_analytical"
        ]
        ==
        "transit_account"
    )


def test_positive_state_receipt_confirms_account():

    transactions = pd.DataFrame(
        [
            {
                "root_party_id": "p1",
                "organization_id": "p1",

                "receiver_account_iban_canonical":
                    "UA1",

                "payment_amount":
                    Decimal("100.00"),

                "payment_operation_date":
                    "2025-01-01",

                "payment_type_detail_source":
                    (
                        "Державне фінансування "
                        "статутної діяльності "
                        "політичної партії"
                    ),
            }
        ]
    )

    orgs = pd.DataFrame(
        [
            {
                "organization_id": "p1",

                "party_name_current":
                    "ТЕСТ",

                "organization_name_current":
                    "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»",

                "organization_level":
                    "central",
            }
        ]
    )

    result = (
        build_state_funding_account_reference(
            transactions,
            orgs,
        )
    )

    row = result.iloc[0]

    assert (
        row[
            "organization_name_current"
        ]
        ==
        "ПОЛІТИЧНА ПАРТІЯ «ТЕСТ»"
    )

    assert (
        row[
            "positive_state_receipt_amount"
        ]
        ==
        Decimal(
            "100.0000000000"
        )
    )

    assert (
        bool(
            row[
                "state_funding_account_confirmed"
            ]
        )
        is True
    )
'''


TEST_PATH = (
    TEST_ROOT
    / "test_reference_builders.py"
)


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


# ============================================================
# 7. TEST SUITE
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 110)
print("TESTS")
print("=" * 110)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed."
    )


# ============================================================
# 8. VALIDATION REBUILD
# ============================================================

import politdata.reference as reference

importlib.reload(reference)


if VALIDATION_ROOT.exists():

    shutil.rmtree(
        VALIDATION_ROOT
    )


VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


rebuilt = (
    reference
    .rebuild_reference_layer(
        organizations=
            ORGANIZATIONS_PATH,

        addresses=
            ADDRESSES_PATH,

        analysis_manifest=
            ANALYSIS_MANIFEST_PATH,

        property_moneys=
            PROPERTY_MONEYS_PATH,

        state_funding=
            STATE_FUNDING_PATH,

        output_root=
            VALIDATION_ROOT,

        overwrite=True,
    )
)


# ============================================================
# 9. FINAL COUNTS
# ============================================================

summary = pd.DataFrame(
    [
        {
            "artifact":
                filename.replace(
                    ".parquet",
                    ""
                ),

            "rows":
                len(frame),
        }

        for filename, frame
        in rebuilt.items()
    ]
)


print()
print("=" * 110)
print("REFERENCE LAYER REBUILD")
print("=" * 110)

display(summary)


print()
print("=" * 110)
print("FINAL CONTROL")
print("=" * 110)

print(
    "organization_reference:",
    len(
        rebuilt[
            "organization_reference.parquet"
        ]
    )
)

print(
    "report_context:",
    len(
        rebuilt[
            "report_context.parquet"
        ]
    )
)

print(
    "report_account_reference:",
    len(
        rebuilt[
            "report_account_reference.parquet"
        ]
    )
)

print(
    "state_funding_account_reference:",
    len(
        rebuilt[
            "state_funding_account_reference.parquet"
        ]
    )
)

print()
print(
    "Controlled report-account analytical-type updates:",
    report_type_updates
)

print(
    "Controlled state-reference name updates:",
    state_name_updates
)

print()
print(
    "Production parquet files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)


ORGANIZATION REFERENCE


,column,mismatches
0,root_party_id,0
1,organization_level,0
2,organization_code,0
3,organization_name_current,0
4,party_code,0
5,party_name_current,0
6,region,0
7,region_source,0
8,region_source_address_type,0
9,region_resolution_method,0


Mismatches: 0

REPORT CONTEXT


,column,mismatches
0,report_id,0
1,organization_id,0
2,root_party_id,0
3,entity_type,0
4,source_party_id,0
5,party_id_matches_organization,0
6,is_party_office,0
7,schema_version,0
8,report_type,0
9,year,0


Mismatches: 0

REPORT ACCOUNT REFERENCE


...

Controlled analytical-type updates: 1,454
Unexpected other mismatches: 0

STATE FUNDING REFERENCE — NON-NAME PARITY


,column,mismatches
0,party_name_current,0
1,organization_level,0
2,first_state_receipt_date,0
3,last_state_receipt_date,0
4,positive_state_receipt_rows,0
5,positive_state_receipt_amount,0
6,state_funding_form_count,0
7,state_funding_forms_observed,0
8,state_funding_source_forms_observed,0
9,state_funding_account_confirmed,0


Controlled organization-name updates: 7

Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\reference.py

TESTS
........................................................................ [ 60%]
................................................                         [100%]
120 passed in 2.18s

Return code: 0

REFERENCE LAYER REBUILD


,artifact,rows
0,organization_reference,9917
1,report_context,78791
2,report_account_reference,12586
3,state_funding_account_reference,7



FINAL CONTROL
organization_reference: 9917
report_context: 78791
report_account_reference: 12586
state_funding_account_reference: 7

Controlled report-account analytical-type updates: 1454
Controlled state-reference name updates: 7

Production parquet files were NOT modified.
RAW files were NOT read.
API was NOT called.


In [145]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import textwrap

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PIPELINE_PATH = (
    ROOT
    / "src"
    / "politdata"
    / "pipeline.py"
)

TEST_PATH = (
    ROOT
    / "tests"
    / "test_pipeline_full.py"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

INTERIM_ROOT = (
    ROOT
    / "data"
    / "interim"
)

VALIDATION_ROOT = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "full_processed_pipeline_v0_1"
)


# ============================================================
# 1. ADD TOP-LEVEL COMPOSITION TO pipeline.py
# ============================================================

pipeline_text = PIPELINE_PATH.read_text(
    encoding="utf-8"
)

MARKER = "# FULL PROCESSED ANALYTICS PIPELINE V0.1"


PIPELINE_EXTENSION = r'''


# ============================================================
# FULL PROCESSED ANALYTICS PIPELINE V0.1
# ============================================================

def rebuild_processed_analytics(
    normalized_root,
    interim_root,
    output_root,
    *,
    overwrite: bool = False,
):
    """
    Rebuild the currently productionized analytical layers
    from normalized data plus the analysis-selected manifest.

    Flow:

        normalized organization tables
        analysis-selected report manifest
        normalized property_moneys
        normalized state_funding
                ↓
        reference layer
                ↓
        payment enrichment
        report-section enrichment
                ↓
        QA

    This function intentionally does NOT:
        - call the PolitData API
        - read RAW report JSON
        - perform report discovery/selection
        - normalize RAW data
        - promote validation output into production
    """

    from politdata.reference import (
        rebuild_reference_layer,
    )


    normalized_root = Path(
        normalized_root
    )

    interim_root = Path(
        interim_root
    )

    output_root = Path(
        output_root
    )


    reference_root = (
        output_root
        / "reference"
    )


    # --------------------------------------------------------
    # Required processed inputs
    # --------------------------------------------------------

    organizations_path = (
        normalized_root
        / "organizations.parquet"
    )

    addresses_path = (
        normalized_root
        / "organization_addresses.parquet"
    )

    analysis_manifest_path = (
        interim_root
        / "reports"
        / "analysis_selected_reports_manifest.parquet"
    )

    property_moneys_path = (
        normalized_root
        / "properties"
        / "property_moneys.parquet"
    )

    state_funding_path = (
        normalized_root
        / "payments"
        / "state_funding.parquet"
    )


    required_inputs = (
        organizations_path,
        addresses_path,
        analysis_manifest_path,
        property_moneys_path,
        state_funding_path,
    )


    for path in required_inputs:

        if not path.exists():

            raise FileNotFoundError(
                path
            )


    # --------------------------------------------------------
    # 1. REFERENCE LAYER
    # --------------------------------------------------------

    references = (
        rebuild_reference_layer(
            organizations=
                organizations_path,

            addresses=
                addresses_path,

            analysis_manifest=
                analysis_manifest_path,

            property_moneys=
                property_moneys_path,

            state_funding=
                state_funding_path,

            output_root=
                reference_root,

            overwrite=
                overwrite,
        )
    )


    # --------------------------------------------------------
    # 2. ENRICHMENT + QA
    # --------------------------------------------------------

    enrichment = (
        rebuild_enriched_data_layers(
            normalized_root,
            output_root,

            reference_root=
                reference_root,

            overwrite=
                overwrite,
        )
    )


    # --------------------------------------------------------
    # 3. REFERENCE SUMMARY
    # --------------------------------------------------------

    reference_summary = pd.DataFrame(
        [
            {
                "artifact":
                    filename.replace(
                        ".parquet",
                        ""
                    ),

                "rows":
                    len(
                        frame
                    ),
            }

            for filename, frame
            in references.items()
        ]
    )


    return {
        "references":
            reference_summary,

        "payments":
            enrichment[
                "payments"
            ],

        "report_sections":
            enrichment[
                "report_sections"
            ],

        "qa":
            enrichment[
                "qa"
            ],
    }
'''


if MARKER not in pipeline_text:

    PIPELINE_PATH.write_text(
        (
            pipeline_text.rstrip()
            +
            "\n"
            +
            textwrap.dedent(
                PIPELINE_EXTENSION
            )
            +
            "\n"
        ),
        encoding="utf-8",
    )

    print(
        "Added full processed pipeline:",
        PIPELINE_PATH
    )

else:

    print(
        "Full processed pipeline already present."
    )


# ============================================================
# 2. SMALL TEST
# ============================================================

TEST_CODE = r'''
from pathlib import Path

import politdata.pipeline as pipeline


def test_full_processed_pipeline_entry_point_exists():

    assert callable(
        pipeline.rebuild_processed_analytics
    )


def test_full_processed_pipeline_has_explicit_inputs():

    names = (
        pipeline
        .rebuild_processed_analytics
        .__code__
        .co_varnames
    )

    assert "normalized_root" in names
    assert "interim_root" in names
    assert "output_root" in names
'''


TEST_PATH.write_text(
    textwrap.dedent(
        TEST_CODE
    ),
    encoding="utf-8",
)


print(
    "Written:",
    TEST_PATH
)


# ============================================================
# 3. ALL TESTS
# ============================================================

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    result.stdout
)


if result.stderr:

    print(
        result.stderr
    )


print(
    "Return code:",
    result.returncode
)


if result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Full validation pipeline was not started."
    )


# ============================================================
# 4. FRESH IMPORT
# ============================================================

import politdata.pipeline as pipeline

importlib.reload(
    pipeline
)


# ============================================================
# 5. CLEAN ONLY VALIDATION OUTPUT
# ============================================================

if VALIDATION_ROOT.exists():

    shutil.rmtree(
        VALIDATION_ROOT
    )


VALIDATION_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 6. RUN COMPLETE PROCESSED ANALYTICS PIPELINE
# ============================================================

print()
print("=" * 100)
print("FULL PROCESSED ANALYTICS PIPELINE")
print("=" * 100)


result = (
    pipeline
    .rebuild_processed_analytics(
        NORMALIZED_ROOT,
        INTERIM_ROOT,
        VALIDATION_ROOT,

        overwrite=True,
    )
)


# ============================================================
# 7. REFERENCE OUTPUT
# ============================================================

print()
print("=" * 100)
print("REFERENCES")
print("=" * 100)

display(
    result[
        "references"
    ]
)


# ============================================================
# 8. PAYMENT OUTPUT
# ============================================================

print()
print("=" * 100)
print("PAYMENTS")
print("=" * 100)

display(
    result[
        "payments"
    ]
)


# ============================================================
# 9. REPORT SECTIONS
# ============================================================

print()
print("=" * 100)
print("REPORT SECTIONS")
print("=" * 100)

display(
    result[
        "report_sections"
    ]
)


# ============================================================
# 10. QA
# ============================================================

print()
print("=" * 100)
print("ROW COUNT QA")
print("=" * 100)

display(
    result[
        "qa"
    ][
        "row_counts"
    ]
)


payment_identity = (
    result[
        "qa"
    ][
        "payment_reference_identity"
    ]
)


print()
print("=" * 100)
print("PAYMENT IDENTITY QA")
print("=" * 100)

display(
    payment_identity
    .groupby(
        "column",
        as_index=False,
    )
    .agg(
        rows=(
            "rows",
            "sum",
        ),

        mismatches=(
            "mismatches",
            "sum",
        ),

        sections=(
            "section",
            "nunique",
        ),
    )
)


# ============================================================
# 11. FINAL BASELINE ASSERTIONS
# ============================================================

reference_counts = dict(
    zip(
        result[
            "references"
        ][
            "artifact"
        ],
        result[
            "references"
        ][
            "rows"
        ],
    )
)


EXPECTED_REFERENCES = {
    "organization_reference":
        9_917,

    "report_context":
        78_791,

    "report_account_reference":
        12_586,

    "state_funding_account_reference":
        7,
}


if reference_counts != EXPECTED_REFERENCES:

    raise RuntimeError(
        "Reference baseline changed. "
        f"Actual={reference_counts}"
    )


payment_rows = int(
    result[
        "payments"
    ][
        "rows"
    ].sum()
)


section_rows = int(
    result[
        "report_sections"
    ][
        "rows"
    ].sum()
)


identity_mismatches = int(
    payment_identity[
        "mismatches"
    ].sum()
)


if payment_rows != 402_028:

    raise RuntimeError(
        "Payment baseline changed."
    )


if section_rows != 308_657:

    raise RuntimeError(
        "Report-section baseline changed."
    )


if identity_mismatches != 0:

    raise RuntimeError(
        "Payment identity QA failed."
    )


# ============================================================
# 12. CONFIRM NEW REFERENCES ARE ACTUALLY USED
# ============================================================

new_report_account = pd.read_parquet(
    VALIDATION_ROOT
    / "reference"
    / "report_account_reference.parquet"
)


new_state_account = pd.read_parquet(
    VALIDATION_ROOT
    / "reference"
    / "state_funding_account_reference.parquet"
)


if len(new_report_account) != 12_586:

    raise RuntimeError(
        "Unexpected report_account_reference size."
    )


if len(new_state_account) != 7:

    raise RuntimeError(
        "Unexpected state_funding_account_reference size."
    )


# Current normalized account classification must be present.
normalized_pm = pd.read_parquet(
    NORMALIZED_ROOT
    / "properties"
    / "property_moneys.parquet",
    columns=[
        "source_report_id",
        "organization_id",
        "root_party_id",
        "party_account_iban",
        "party_account_type_analytical",
    ],
)


# We don't compare against legacy reference here:
# 1,454 analytical-type differences are the intentional
# migration already validated in the previous step.


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Complete processed analytical rebuild is green."
)

print()
print(
    "Reference rows:",
    reference_counts
)

print(
    "Payment rows:",
    f"{payment_rows:,}"
)

print(
    "Report-section rows:",
    f"{section_rows:,}"
)

print(
    "Payment identity mismatches:",
    f"{identity_mismatches:,}"
)

print()
print(
    "Entry point:"
)

print(
    "  politdata.pipeline.rebuild_processed_analytics(...)"
)

print()
print(
    "Validation output:"
)

print(
    VALIDATION_ROOT
)

print()
print(
    "Production parquet files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

Added full processed pipeline: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\pipeline.py
Written: C:\Users\Igor\Desktop\Projects\politdata-pipeline\tests\test_pipeline_full.py

TESTS
........................................................................ [ 59%]
..................................................                       [100%]
122 passed in 5.66s

Return code: 0

FULL PROCESSED ANALYTICS PIPELINE


NameError: name 'pd' is not defined

In [146]:
from pathlib import Path
import importlib
import subprocess
import sys

import pandas as pd


# ============================================================
# PATHS
# ============================================================

ROOT = Path(
    r"C:\Users\Igor\Desktop\Projects\politdata-pipeline"
)

PIPELINE_PATH = (
    ROOT
    / "src"
    / "politdata"
    / "pipeline.py"
)

NORMALIZED_ROOT = (
    ROOT
    / "data"
    / "processed"
    / "normalized_v0_1"
)

INTERIM_ROOT = (
    ROOT
    / "data"
    / "interim"
)

VALIDATION_ROOT = (
    ROOT
    / "data"
    / "interim"
    / "refactor_validation"
    / "full_processed_pipeline_v0_1"
)


# ============================================================
# 1. PATCH MISSING PANDAS IMPORT
# ============================================================

text = PIPELINE_PATH.read_text(
    encoding="utf-8"
)


if "import pandas as pd" not in text:

    lines = text.splitlines()

    insert_at = 0

    # Keep __future__ import first if present.
    for i, line in enumerate(lines):

        if line.startswith(
            "from __future__ import"
        ):
            insert_at = i + 1
            continue

        if (
            i >= insert_at
            and line.strip()
        ):
            break


    lines.insert(
        insert_at,
        ""
    )

    lines.insert(
        insert_at + 1,
        "import pandas as pd"
    )


    PIPELINE_PATH.write_text(
        "\n".join(lines) + "\n",
        encoding="utf-8",
    )

    print(
        "Patched:",
        PIPELINE_PATH
    )

else:

    print(
        "pandas import already present."
    )


# ============================================================
# 2. TEST SUITE
# ============================================================

test_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests",
        "-q",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)


print()
print("=" * 100)
print("TESTS")
print("=" * 100)

print(
    test_result.stdout
)


if test_result.stderr:

    print(
        test_result.stderr
    )


print(
    "Return code:",
    test_result.returncode
)


if test_result.returncode != 0:

    raise RuntimeError(
        "Tests failed. "
        "Final pipeline run was not started."
    )


# ============================================================
# 3. FRESH IMPORT
# ============================================================

import politdata.pipeline as pipeline


importlib.reload(
    pipeline
)


# ============================================================
# 4. RE-RUN ONLY THE FAILED FINAL PHASE
#
# Do NOT delete validation output.
# overwrite=True safely rebuilds the same validation files.
# ============================================================

print()
print("=" * 100)
print("FULL PROCESSED ANALYTICS PIPELINE")
print("=" * 100)


result = (
    pipeline
    .rebuild_processed_analytics(
        NORMALIZED_ROOT,
        INTERIM_ROOT,
        VALIDATION_ROOT,

        overwrite=True,
    )
)


# ============================================================
# 5. SUMMARIES
# ============================================================

print()
print("=" * 100)
print("REFERENCES")
print("=" * 100)

display(
    result[
        "references"
    ]
)


print()
print("=" * 100)
print("PAYMENTS")
print("=" * 100)

display(
    result[
        "payments"
    ]
)


print()
print("=" * 100)
print("REPORT SECTIONS")
print("=" * 100)

display(
    result[
        "report_sections"
    ]
)


print()
print("=" * 100)
print("ROW COUNT QA")
print("=" * 100)

display(
    result[
        "qa"
    ][
        "row_counts"
    ]
)


# ============================================================
# 6. FINAL ASSERTIONS
# ============================================================

reference_counts = dict(
    zip(
        result[
            "references"
        ][
            "artifact"
        ],

        result[
            "references"
        ][
            "rows"
        ],
    )
)


EXPECTED_REFERENCES = {
    "organization_reference":
        9_917,

    "report_context":
        78_791,

    "report_account_reference":
        12_586,

    "state_funding_account_reference":
        7,
}


payment_rows = int(
    result[
        "payments"
    ][
        "rows"
    ].sum()
)


section_rows = int(
    result[
        "report_sections"
    ][
        "rows"
    ].sum()
)


payment_identity = (
    result[
        "qa"
    ][
        "payment_reference_identity"
    ]
)


identity_mismatches = int(
    payment_identity[
        "mismatches"
    ].sum()
)


if reference_counts != EXPECTED_REFERENCES:

    raise RuntimeError(
        "Reference baseline changed: "
        f"{reference_counts}"
    )


if payment_rows != 402_028:

    raise RuntimeError(
        "Payment baseline changed."
    )


if section_rows != 308_657:

    raise RuntimeError(
        "Report-section baseline changed."
    )


if identity_mismatches != 0:

    raise RuntimeError(
        "Payment identity QA failed."
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 100)
print("DONE")
print("=" * 100)

print(
    "Reference rows:",
    reference_counts
)

print(
    "Payment rows:",
    f"{payment_rows:,}"
)

print(
    "Report-section rows:",
    f"{section_rows:,}"
)

print(
    "Payment identity mismatches:",
    f"{identity_mismatches:,}"
)

print()
print(
    "Entry point:"
)

print(
    "  politdata.pipeline.rebuild_processed_analytics(...)"
)

print()
print(
    "Production parquet files were NOT modified."
)

print(
    "RAW files were NOT read."
)

print(
    "API was NOT called."
)

Patched: C:\Users\Igor\Desktop\Projects\politdata-pipeline\src\politdata\pipeline.py

TESTS
........................................................................ [ 59%]
..................................................                       [100%]
122 passed in 1.60s

Return code: 0

FULL PROCESSED ANALYTICS PIPELINE

REFERENCES


,artifact,rows
0,organization_reference,9917
1,report_context,78791
2,report_account_reference,12586
3,state_funding_account_reference,7



PAYMENTS


,section,rows,columns,output
0,monetary_contributions,27234,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\monetary_contributions.parquet
1,other_contributions,6168,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\other_contributions.parquet
2,state_funding,96,127,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\state_funding.parquet
3,other_incomes,19007,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\other_incomes.parquet
4,budget_expenses,29482,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\budget_expenses.parquet
5,outgoing_expenses,319901,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\outgoing_expenses.parquet
6,return_expenses,137,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\return_expenses.parquet
7,transfer_expenses,3,126,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\payments\transfer_expenses.parquet



REPORT SECTIONS


,section,rows,columns,output
0,realty,11325,44,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\properties\realty.parquet
1,transport,1325,43,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\properties\transport.parquet
2,movable,693,42,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\properties\movable.parquet
3,intangible,8614,41,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\properties\intangible.parquet
4,paper,0,28,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\properties\paper.parquet
5,obligations,26132,43,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\obligations\obligations.parquet
6,head_info,78791,32,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\report_state\head_info.parquet
7,employee_counts,78791,32,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\report_state\employee_counts.parquet
8,organizations,308,31,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\report_state\organizations.parquet
9,regional_offices,102678,45,C:\Users\Igor\Desktop\Projects\politdata-pipeline\data\interim\refactor_validation\full_processed_pipeline_v0_1\report_state\regional_offices.parquet



ROW COUNT QA


,layer,section,rows,expected_rows,matches_baseline
0,payments,monetary_contributions,27234,27234,True
1,payments,other_contributions,6168,6168,True
2,payments,state_funding,96,96,True
3,payments,other_incomes,19007,19007,True
4,payments,budget_expenses,29482,29482,True
5,payments,outgoing_expenses,319901,319901,True
6,payments,return_expenses,137,137,True
7,payments,transfer_expenses,3,3,True
8,report_sections,realty,11325,11325,True
9,report_sections,transport,1325,1325,True



DONE
Reference rows: {'organization_reference': 9917, 'report_context': 78791, 'report_account_reference': 12586, 'state_funding_account_reference': 7}
Payment rows: 402,028
Report-section rows: 308,657
Payment identity mismatches: 0

Entry point:
  politdata.pipeline.rebuild_processed_analytics(...)

Production parquet files were NOT modified.
RAW files were NOT read.
API was NOT called.
